# Matmul with grouped program ordering for L2 reuse

A matmul launched on a 1-D grid has to turn each program id into an output tile `(pid_m, pid_n)`. Row-major order (`pid_m = pid // num_pid_n`) runs a whole row of tiles before moving on, so every program in flight needs a different column panel of $B$ and the L2 cache thrashes. **Grouped ordering** walks `GROUP_M` tile-rows at a time, column by column, so programs that run together share both their $A$ rows and their $B$ columns.

With `num_pid_m = cdiv(M, BM)` and `num_pid_n = cdiv(N, BN)`, the mapping is:

```
group_size   = GROUP_M * num_pid_n                      # programs per group
group_id     = pid // group_size
first_pid_m  = group_id * GROUP_M
group_rows   = min(num_pid_m - first_pid_m, GROUP_M)    # the last group may be shorter
pid_m        = first_pid_m + (pid % group_size) % group_rows
pid_n        = (pid % group_size) // group_rows
```

Write `grouped_matmul(a, b, BM, BN, BK, GROUP_M)`.

## Inputs
- `a`: `float32` CUDA `(M, K)`, `b`: `float32` CUDA `(K, N)`, any sizes $\ge 1$. Use their strides.
- `BM`, `BN`, `BK`: Python ints, powers of two $\ge 16$. `GROUP_M`: Python int $\ge 1$ (it may exceed `num_pid_m`).

## Output
A tuple `(order, c)`:
- `order`: `int32` `(num_pid_m, num_pid_n)`; `order[pid_m, pid_n]` is the program id that computed that tile. Each program writes its own `pid` there, which makes the schedule visible to the tests.
- `c`: `float32` `(M, N)` $= a b$, accumulated in float32 with `tl.dot(..., input_precision="ieee")`.

## Requirements
- A 1-D grid of `num_pid_m * num_pid_n` programs using the mapping above. Mask every load (`other=0.0`) and the store of `c`.

This problem runs in **Colab with a GPU runtime**. Expected outputs come from a PyTorch reference (float64 matmul, Python mapping).

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import triton
import triton.language as tl

def grouped_matmul(a, b, BM, BN, BK, GROUP_M):
    """a (M, K), b (K, N) float32 CUDA. Return (order, a @ b) where order[pid_m, pid_n] = pid that computed the tile."""
    raise NotImplementedError


@triton.jit
def _grouped_matmul_kernel(a_ptr, b_ptr, c_ptr, order_ptr, M, N, K,
                           stride_am, stride_ak, stride_bk, stride_bn, stride_cm, stride_cn,
                           BM: tl.constexpr, BN: tl.constexpr, BK: tl.constexpr, GROUP_M: tl.constexpr):
    pid = tl.program_id(axis=0)
    # TODO: pid -> (pid_m, pid_n) with grouped ordering, then the usual tiled matmul
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0]], "dtype": "float32"}, {"__nd__": [[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0]], "dtype": "float32"}, 16, 16, 16, 2], "e": {"__tup__": [{"__nd__": [[0, 2, 4], [1, 3, 5], [6, 8, 10], [7, 9, 11]], "dtype": "int32"}, {"__nd__": [[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0], [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0]], "dtype": "float32"}, {"__nd__": [[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]], "dtype": "float32"}, 16, 16, 16, 1], "e": {"__tup__": [{"__nd__": [[0, 1], [2, 3], [4, 5]], "dtype": "int32"}, {"__nd__": [[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.8999031186103821, -0.4583752751350403, -1.3140853643417358, 0.6586976647377014, 0.9554258584976196, 0.5246310234069824, -2.558269500732422, -1.5353412628173828, 0.6136307120323181, 1.494365930557251, -1.4945822954177856, 0.9703254699707031, 1.1190851926803589, 0.017410578206181526, -1.339133381843567, -0.6403135657310486, -0.36200037598609924, 0.6909022927284241, 0.8993356227874756, -0.986848771572113, -0.10213501006364822, -0.03492661938071251, -0.1748647689819336, 0.9459331631660461, -0.2716423571109772, -0.0700131207704544, -1.4837714433670044, 0.14615555107593536, -0.9387549757957458, 1.8856043815612793, 0.14862462878227234, 0.1838734894990921, 0.0986730232834816], [-1.1710097789764404, -0.4353877604007721, 0.11089904606342316, 0.9117034077644348, 0.21966463327407837, -1.3220744132995605, -0.21426066756248474, -0.13816185295581818, -1.6847881078720093, 0.5395525693893433, 0.7637683749198914, -0.9124293923377991, 2.1853322982788086, 0.324837327003479, -0.5884791612625122, 1.9458446502685547, -0.5886527895927429, 0.14952750504016876, -1.812504529953003, -1.097822904586792, 1.750251054763794, -1.0016318559646606, -0.8418654203414917, -0.26036927103996277, -0.8247355222702026, -1.2520310878753662, -1.0220783948898315, -0.17003101110458374, 1.5131309032440186, 0.7696735262870789, 2.263970136642456, -0.8698101043701172, 0.17317888140678406], [0.25686517357826233, 0.5076766014099121, 0.7218186855316162, -1.7291629314422607, 1.4759763479232788, -0.013240644708275795, -0.4972066581249237, 0.25351643562316895, -0.5871797204017639, -1.2591904401779175, -0.37412479519844055, -1.9415563344955444, -1.127770185470581, 0.7236483693122864, 0.4756447672843933, -1.151097297668457, -2.2454750537872314, 0.17502057552337646, 0.36982715129852295, -0.05980798602104187, -1.0016217231750488, 0.05884239077568054, -0.36620163917541504, 0.1824212670326233, 0.5660898685455322, -0.04666045308113098, -0.2123861461877823, 0.5310394167900085, 0.26019051671028137, -0.8776059150695801, -0.10265259444713593, 0.6619710326194763, 0.566676676273346], [-0.32494738698005676, -0.16475775837898254, -1.0546441078186035, 0.22098210453987122, 1.0684579610824585, 0.4071141481399536, -0.46759316325187683, 0.6242401003837585, -0.004906778689473867, 0.6311420798301697, -1.2471394538879395, -0.4167402982711792, 0.7652712464332581, 1.1058533191680908, -0.7801048755645752, -0.5019945502281189, 1.8491978645324707, 0.3650195896625519, -0.153121218085289, 0.48895737528800964, -0.5500261783599854, -1.6647868156433105, 0.026143142953515053, -0.34950289130210876, -0.6110222935676575, 0.007791154086589813, -0.8798443078994751, -0.4935612678527832, -1.8657894134521484, 1.1923428773880005, 1.4240317344665527, -1.1406865119934082, -1.5961689949035645], [0.3878454864025116, 0.7894242405891418, -0.4591422379016876, -1.1156907081604004, 1.8323572874069214, -0.6859242916107178, 0.36959096789360046, 0.2881930470466614, 0.3097412884235382, 1.2666064500808716, 1.3789180517196655, -1.4739190340042114, -0.12562070786952972, 1.0913655757904053, 0.17375719547271729, -0.6511379480361938, 0.7755193114280701, -0.5232162475585938, -0.18898096680641174, 0.8250610828399658, 0.5654955506324768, 1.880777359008789, 0.6875735521316528, 0.003970490302890539, 1.0332063436508179, -0.04454539716243744, 0.06852097064256668, 0.13934198021888733, -0.41248008608818054, 0.26471850275993347, 0.1666335165500641, 0.17127445340156555, -0.9556545615196228], [0.5732484459877014, 0.10141777247190475, -0.15393570065498352, -0.1486535668373108, -0.6692723631858826, -0.6993617415428162, 1.8623509407043457, 0.17674580216407776, 0.7380908131599426, 0.6799888610839844, -0.7537494897842407, -0.14892086386680603, -0.5416708588600159, 0.517824649810791, 0.35509219765663147, -0.9670684933662415, 0.9322242140769958, 0.9030817151069641, -1.3903950452804565, 0.3607134521007538, 0.2591469883918762, -1.7361066341400146, -1.160036325454712, 0.5330931544303894, -1.2110098600387573, -1.9539152383804321, 0.03853512182831764, -1.9655067920684814, -0.20679813623428345, 0.09442973881959915, -0.10538601130247116, 0.5674481987953186, -0.0816906988620758], [-0.2834911048412323, -1.1433733701705933, 1.1357825994491577, -1.1710913181304932, 0.3445030450820923, -0.09653632342815399, -0.4260581433773041, -1.315321683883667, -1.6966569423675537, -0.09251436591148376, 0.6657434105873108, 0.05454370751976967, -1.1645617485046387, 0.9531631469726562, -2.605194568634033, -0.13056007027626038, 0.9154669642448425, 1.0347535610198975, -0.14629866182804108, 1.2095682621002197, 1.728361964225769, -0.27340182662010193, 1.104597806930542, 0.06376875191926956, 1.0828369855880737, -1.1844961643218994, 0.7466009259223938, 1.2563724517822266, 1.1375545263290405, 0.09971880912780762, -1.9823657274246216, 0.2861904799938202, -2.318258762359619], [0.14128269255161285, 0.20135077834129333, 0.6472941637039185, -0.5949511528015137, 0.6877020597457886, -0.3357521891593933, -1.1458567380905151, 0.17348985373973846, 2.262974739074707, 0.35219985246658325, 0.47316688299179077, 1.6408905982971191, 1.1046124696731567, 1.102184772491455, -1.4408499002456665, -0.7911708354949951, 2.235787868499756, 0.2224586308002472, -0.8280426263809204, 1.261252999305725, 0.3924678862094879, -1.9293192625045776, -0.6627941131591797, -1.1360650062561035, -1.2008289098739624, -0.7337416410446167, -1.3539620637893677, -0.45091667771339417, 0.23689255118370056, -0.20710228383541107, -0.2102249711751938, 0.9449697732925415, -0.2974686622619629], [0.1296825259923935, -0.8770400285720825, -1.1862388849258423, 0.51100093126297, -0.972798228263855, 0.13335035741329193, 0.8983229994773865, -0.4918181598186493, 2.117896318435669, 1.154932975769043, -0.774941623210907, 2.13230562210083, 0.5126007199287415, 1.0842838287353516, -0.4711441099643707, 0.3590772747993469, -0.23570898175239563, 0.3278779685497284, -0.6708052158355713, -0.4871984124183655, -0.7413039207458496, -2.210026502609253, -0.8632765412330627, 0.270134299993515, -1.2500736713409424, -1.0433697700500488, 1.3556078672409058, -1.7961822748184204, -0.24617260694503784, 0.4725686311721802, 0.6061264872550964, 0.3723907768726349, 0.2877083122730255], [-0.07896576821804047, -1.1026699542999268, -2.212771415710449, 1.6808733940124512, -0.41940876841545105, 0.10577099025249481, -1.0259913206100464, 0.02224990352988243, -1.428452730178833, -0.8158507943153381, 0.19532310962677002, 1.770680546760559, 0.49967917799949646, 2.0179903507232666, 0.27029886841773987, 0.20965175330638885, 1.4535316228866577, 0.012672136537730694, -1.4806346893310547, 0.5149244070053101, 0.3988657593727112, 0.18802908062934875, 0.45602163672447205, -0.340660959482193, -0.08864977955818176, -1.288060188293457, 0.7480973601341248, -0.684158980846405, 1.2231075763702393, 0.10027065128087997, -0.8516147136688232, 0.6974550485610962, 0.3599795401096344], [0.43590420484542847, 0.05157991871237755, -0.5600208044052124, 1.4960922002792358, -1.552392840385437, -1.041038990020752, 1.2759385108947754, 1.7062774896621704, -1.9293212890625, 1.7610541582107544, 0.6014499068260193, -2.00620174407959, -0.6632086634635925, -1.1056885719299316, 0.07043773680925369, -0.2917822599411011, 1.2332061529159546, 0.18745777010917664, 0.8555373549461365, 0.997447669506073, -1.104976773262024, -0.14799921214580536, 0.47859784960746765, 0.2995857894420624, -0.657784104347229, 0.4879797101020813, -1.1764971017837524, -0.6216272711753845, 0.5223340392112732, 0.6654309630393982, -1.0732537508010864, 0.2159944325685501, 0.5570297241210938], [0.9360799789428711, 0.6457073092460632, 0.16403627395629883, -1.0509146451950073, -1.6835135221481323, -0.8572655320167542, -0.6963451504707336, -0.4555371403694153, 0.25345197319984436, 0.5250223278999329, 1.9980356693267822, -1.6953502893447876, -0.9134610295295715, -1.7069451808929443, -1.1620583534240723, -1.446288824081421, -0.9352262616157532, -0.9528447389602661, -1.2973151206970215, -0.749412477016449, 0.6836749911308289, 0.6057178974151611, -0.15176734328269958, 0.7778471112251282, -2.188178300857544, -1.0612519979476929, 0.7438154816627502, -0.44453996419906616, -0.6817123889923096, -0.45675763487815857, -0.4625297784805298, 0.704009473323822, 1.447244644165039], [-1.3964546918869019, -1.1432619094848633, -0.6047202944755554, 0.23877355456352234, 0.6882261633872986, -0.08775756508111954, 0.3833174705505371, -0.30538854002952576, 0.37029939889907837, -2.07387638092041, 0.7120949625968933, 0.045047517865896225, -1.1478885412216187, -0.06779567152261734, -1.2625044584274292, 1.1089059114456177, 1.073300838470459, 0.0848100483417511, -1.7510160207748413, -2.063983678817749, 0.0727948397397995, -1.1467972993850708, 2.0426647663116455, -1.507251262664795, -1.5780147314071655, 0.9184601306915283, 2.0231704711914062, -1.669063687324524, 0.803791880607605, 0.38384389877319336, -0.4328260123729706, -1.3933830261230469, -0.7584786415100098], [-0.4709814190864563, 0.5679681301116943, 0.5609595775604248, -0.37571558356285095, -2.0546727180480957, -0.6836041212081909, -0.5735498070716858, -1.2105238437652588, 0.29197072982788086, 1.272192120552063, 0.665062665939331, -0.41499340534210205, -1.666552186012268, 0.4583747088909149, 1.1336168050765991, 0.20790615677833557, 0.163664311170578, 0.3044619560241699, -0.13469940423965454, -1.089009165763855, 1.8226481676101685, 1.1215150356292725, 1.3628349304199219, 0.009091629646718502, 0.7252418994903564, -2.0238122940063477, -0.1765352487564087, -0.1620151400566101, -0.8795292973518372, -0.922248363494873, -1.7394911050796509, 0.3160697817802429, 0.14133892953395844], [0.45134201645851135, -0.6857211589813232, 0.8102511763572693, 2.148977518081665, -0.5179665088653564, 0.6146741509437561, -2.6837375164031982, 1.754376769065857, 1.4886592626571655, 0.06065455451607704, 0.7225856184959412, -1.5005122423171997, 1.131286859512329, -1.0014513731002808, -0.3416120111942291, -0.5470161437988281, 0.9327576160430908, 0.22310537099838257, 0.24521410465240479, 1.0738639831542969, -1.2151941061019897, 0.4829792380332947, 0.9331173896789551, 2.1121885776519775, 1.2370816469192505, -1.0399430990219116, 0.9409905672073364, -1.5432658195495605, -0.42526742815971375, -0.36796969175338745, 0.6471886038780212, -0.48888903856277466, -0.1507345587015152], [0.34696587920188904, 0.015107192099094391, 0.31262239813804626, -0.5860698819160461, 0.03495031222701073, 1.0742604732513428, 0.7831284403800964, 0.41479969024658203, 0.3535030484199524, -1.6817735433578491, 1.1966789960861206, 1.6724965572357178, 1.3022451400756836, -1.2942326068878174, -0.6232863664627075, -0.6558471322059631, 0.23779083788394928, -0.5155795812606812, 0.18819868564605713, 0.6934308409690857, 0.06432077288627625, 1.6394416093826294, 1.1536107063293457, 0.26563167572021484, -0.6244698762893677, 0.08083276450634003, -0.5325934290885925, -1.2438243627548218, 0.15908955037593842, 1.307270884513855, -0.40861353278160095, -0.9304059147834778, 0.08489415794610977], [-0.6191263198852539, 0.9043327569961548, -0.24656984210014343, 0.07291946560144424, -0.187511146068573, 0.09801522642374039, -0.8479596972465515, 0.14680016040802002, -0.6895408630371094, -0.7100067138671875, 0.29836875200271606, -1.6988108158111572, -0.7515841722488403, -0.2632159888744354, -0.23131847381591797, 0.47007840871810913, 1.118277668952942, 0.05540956184267998, 0.024942995980381966, -0.12039241939783096, 0.04569206386804581, -0.08779995143413544, 2.937380313873291, -0.7223324775695801, 0.9342005848884583, 0.9290662407875061, 0.6426178812980652, -0.10218202322721481, 0.5072893500328064, 1.085374116897583, -0.8222247958183289, 1.181255578994751, 0.27646568417549133], [1.5324430465698242, 0.19275467097759247, 0.225307896733284, 0.2398947924375534, 1.2878522872924805, -0.29720813035964966, 1.4777028560638428, 0.2831805348396301, -1.604684829711914, -0.2553133964538574, 1.4359402656555176, -0.03268507122993469, -1.3366377353668213, 0.47181636095046997, -0.1753959208726883, 0.6676793098449707, -0.4114547371864319, 1.2054307460784912, 0.5911117196083069, 0.12867727875709534, -1.2888044118881226, -0.3759533762931824, 1.6365784406661987, 2.461146593093872, 0.8043172955513, -0.04201611131429672, 0.8933999538421631, -0.18157339096069336, -1.2520462274551392, -0.2233319878578186, -1.5462781190872192, -0.11772065609693527, 0.21148231625556946], [0.8776394724845886, 2.142606735229492, -0.23554906249046326, 1.0518031120300293, -0.08785352855920792, 0.8981180191040039, 0.17291632294654846, 0.061453528702259064, -0.12160354107618332, -0.8075142502784729, -0.8923971652984619, 2.1207849979400635, -0.4519121050834656, -0.1141229122877121, -0.699027955532074, -1.239908218383789, 0.6202710866928101, -0.8065798282623291, -1.3927572965621948, -1.1554834842681885, -0.1550513058900833, 0.8607157468795776, -0.5249208211898804, 2.4977715015411377, -1.2541661262512207, 1.0243734121322632, -1.7396317720413208, -0.16639791429042816, 0.15475785732269287, -1.0757285356521606, 0.035045016556978226, 0.11347280442714691, -0.22673399746418], [1.1841027736663818, -0.18521659076213837, 1.355492115020752, -0.9944339990615845, 0.23443466424942017, -0.09827530384063721, -0.6036983132362366, 0.30331557989120483, -0.6957701444625854, 0.5019524097442627, -1.223759651184082, 0.004288239870220423, -0.4606814682483673, 1.5303953886032104, -0.01700490713119507, -1.1904680728912354, -2.1827242374420166, -0.5535383224487305, -0.3185899257659912, 1.7100979089736938, 2.3178536891937256, 0.013037207536399364, -0.3308594524860382, 1.3968191146850586, -1.1285423040390015, -0.4913652241230011, 1.455488920211792, -0.04435897618532181, 0.03992754966020584, 0.10061267018318176, -0.8511046767234802, -0.6384986042976379, -0.06595558673143387], [-1.65879225730896, -0.3902221918106079, 0.703824520111084, -0.19587136805057526, -2.3835530281066895, 0.12895779311656952, 0.7925776243209839, -0.2523709535598755, 1.3760493993759155, 0.3249971270561218, 0.14536908268928528, -0.38748422265052795, -1.5404058694839478, -1.0179729461669922, -0.575279176235199, -1.827839732170105, 0.600877583026886, -1.0819159746170044, -1.6162034273147583, 0.0171206071972847, -0.780884325504303, -0.8399583697319031, -0.5029340982437134, 1.6197550296783447, -0.6615617871284485, 0.4144236445426941, 0.2150653451681137, -0.13213010132312775, -0.21591006219387054, -0.5756670236587524, 0.8215908408164978, 0.47491052746772766, -1.1652508974075317], [1.6954865455627441, 1.3738999366760254, 1.534897804260254, 1.7814605236053467, -0.7402556538581848, 0.13519427180290222, 0.4030706584453583, -0.7339342832565308, 0.2864648103713989, -0.5419684648513794, 1.4307308197021484, -0.13700881600379944, -1.7071185111999512, -1.4293266534805298, -0.8566039800643921, -0.7225134968757629, 1.216440200805664, -0.8717086315155029, -1.494749903678894, 0.4862147867679596, 0.6418257355690002, 1.2611799240112305, -0.9460441470146179, -0.3007318675518036, -0.5822508931159973, 0.36769694089889526, 1.2012457847595215, -0.45043936371803284, -1.3434103727340698, 0.08650703728199005, -1.3707393407821655, -2.541722536087036, 0.28454267978668213], [0.9492698311805725, -1.7157375812530518, -0.006869330070912838, -1.038689136505127, -0.6974465250968933, 0.058501001447439194, -0.37954795360565186, -1.1146408319473267, 1.9081367254257202, -0.15841223299503326, 1.0266318321228027, -0.6831233501434326, 0.8704742789268494, 0.5657032132148743, -0.8921152353286743, -1.1389124393463135, 0.5938212275505066, -0.031848639249801636, -1.4403527975082397, -1.1494170427322388, -1.4370230436325073, 0.8036302924156189, 0.5587061643600464, -0.7483652234077454, 1.8650603294372559, -0.2549581527709961, 0.7426502704620361, 1.556613564491272, -0.8623237609863281, -0.6465769410133362, -1.0332574844360352, -0.14799988269805908, -1.8309932947158813], [1.3821425437927246, 0.7643731236457825, -0.0799730122089386, -0.4333830177783966, 0.6291009187698364, -1.0556741952896118, -0.8429441452026367, -0.1958383321762085, 0.5132637619972229, -1.3493880033493042, -2.1075692176818848, -1.2414186000823975, 0.3184157907962799, 0.6318437457084656, 1.1161473989486694, 1.088407039642334, -2.3566079139709473, 1.9803273677825928, -1.195037603378296, 1.3977832794189453, 0.1043664887547493, -0.3818780481815338, 0.5468370914459229, -0.7387341260910034, -1.9393256902694702, -0.2706186771392822, -1.1248379945755005, 0.534800112247467, -0.03687746077775955, -0.23443321883678436, -0.8537653088569641, -1.1263427734375, 1.00685453414917], [-0.1887790411710739, 0.7323111295700073, 0.6601270437240601, -0.7505186796188354, -0.27323904633522034, 1.2058132886886597, -0.8728693127632141, -0.08649203181266785, 0.8504368662834167, 1.2942299842834473, -0.7061967849731445, -1.2057045698165894, -0.03805288299918175, -0.8950231075286865, 0.7048787474632263, 0.6489371061325073, -1.7708303928375244, -0.24275313317775726, -0.8262097835540771, -0.033612336963415146, 1.1155413389205933, 0.43447861075401306, 0.8978976011276245, 0.0027777496725320816, -2.6910126209259033, -2.9569592475891113, -0.31769460439682007, 0.744182825088501, -0.3085881173610687, 0.17100951075553894, -1.5078818798065186, 0.8876871466636658, -2.4846179485321045], [-1.3491706848144531, -1.546278476715088, 1.0564754009246826, -1.4004573822021484, 1.209554672241211, -0.1911104917526245, -1.5221372842788696, -1.0479254722595215, 0.5509051084518433, -0.9176278710365295, 1.0885967016220093, -0.7233878970146179, 1.569963812828064, -0.751186728477478, 0.9699144959449768, 0.04679206758737564, -0.10635776817798615, 0.2956945598125458, 0.9242297410964966, 0.2530613839626312, -0.6040695309638977, -0.002265071962028742, 0.2286248803138733, -0.5424455404281616, 0.8305144906044006, -0.96439129114151, -1.3908461332321167, -0.06986145675182343, -1.0823297500610352, 0.9213810563087463, 1.228393316268921, -1.919921875, 0.04241737723350525], [-0.3029714822769165, 0.6972513794898987, -0.19374139606952667, 0.8257158994674683, -0.05092581734061241, 0.7098743915557861, -0.9867958426475525, 1.4491692781448364, -1.5946506261825562, -1.4568476676940918, 0.3270825743675232, 0.964439868927002, 0.2709067463874817, -2.6652159690856934, -1.3865383863449097, 0.24557794630527496, 0.06523028761148453, 0.12412471324205399, -0.2938699424266815, -0.2669796049594879, 0.13816572725772858, 1.2594645023345947, 0.6779926419258118, -0.5075498819351196, 0.6025964021682739, 0.2775305509567261, -0.9836982488632202, 0.8962817788124084, 0.11050381511449814, -0.34924063086509705, 0.37133851647377014, -2.9414453506469727, 1.601850152015686], [-0.31375640630722046, -0.44220975041389465, -0.9816311597824097, 0.30627667903900146, 0.7132523059844971, 1.1218878030776978, -0.6482441425323486, 1.9496804475784302, -0.4865599572658539, 0.010171583853662014, 2.970968008041382, 0.7807389497756958, -1.9831628799438477, 0.1812298446893692, 0.6112141609191895, -0.4037332832813263, 0.2940548360347748, 1.3554388284683228, -0.6453508734703064, -0.49437010288238525, 0.7139834761619568, -0.38205093145370483, -0.22394582629203796, 0.44720879197120667, 1.068036675453186, 1.0667312145233154, 2.111229181289673, 0.8928000926971436, 0.4409791827201843, -2.1375954151153564, -0.001538573415018618, -0.65567547082901, -1.6248083114624023], [0.02314361184835434, 0.45823681354522705, -0.6269470453262329, 0.7082387208938599, -1.1235982179641724, -0.7026557922363281, 1.1051366329193115, 0.5595276951789856, 0.4839227795600891, -0.34300899505615234, 0.7244430184364319, -1.2352527379989624, -0.05574558302760124, 0.08569443225860596, -0.26980888843536377, 1.271958589553833, 1.0944594144821167, 0.18767233192920685, 0.004210630431771278, 0.2315317541360855, 0.853624165058136, 0.34284618496894836, 0.24331901967525482, 0.4315861761569977, 1.1104165315628052, 0.5473667979240417, -0.4876408576965332, 0.5577390193939209, -0.9523491263389587, 1.203368902206421, 2.287093162536621, 0.15321217477321625, 0.9807662963867188], [1.265201210975647, -0.5547460913658142, 0.854436457157135, 1.3333178758621216, -0.26146236062049866, 0.8939447402954102, 0.5393257141113281, -0.9474541544914246, 0.8511086702346802, 1.8847929239273071, 0.24231652915477753, 0.48922330141067505, -0.8519030213356018, -0.9525532126426697, -0.4582779109477997, 0.3089849054813385, 0.4178001880645752, 0.43185341358184814, 0.9222221970558167, 0.5821750164031982, -0.13951098918914795, 0.8311170339584351, -0.7039746046066284, 1.8731147050857544, -0.7294390797615051, 0.04164654389023781, -0.9582239389419556, 1.4264137744903564, 0.5558668971061707, -0.7786097526550293, -0.794025719165802, 0.9583896994590759, -1.3961421251296997], [0.2524065375328064, -0.9657363891601562, 0.37706106901168823, 1.1542261838912964, 0.02713397517800331, 0.5405804514884949, 0.3851805031299591, -0.8401992917060852, -0.4251210391521454, -0.7389296293258667, -1.179717779159546, -2.665703773498535, 1.4356073141098022, -0.491498202085495, 0.46894386410713196, 0.4630524814128876, -0.6732829809188843, 0.44945430755615234, 0.6006954908370972, -0.36795246601104736, -0.6248364448547363, 0.41569629311561584, 0.13297678530216217, -0.5198304653167725, 0.898234486579895, -1.3164219856262207, 1.0312918424606323, 0.10628420859575272, -1.1473209857940674, 0.41652318835258484, -0.7588562369346619, 0.03826719522476196, -0.15688541531562805], [-0.6990015506744385, 1.4016379117965698, -0.08112846314907074, -0.20107074081897736, 0.8807601928710938, 0.4938201904296875, -0.04028857499361038, 0.22751930356025696, -1.2422022819519043, 1.3364770412445068, -1.5608032941818237, 1.522441029548645, 0.5861697793006897, -0.4113505184650421, -0.715015709400177, -1.796675682067871, -1.5641100406646729, 1.4067883491516113, 0.7288343906402588, 0.7648867964744568, 0.5806168913841248, -1.5306833982467651, 0.6830989122390747, -0.8719853758811951, -1.8474117517471313, -0.663332462310791, -0.5502601861953735, 1.2292487621307373, -1.1766968965530396, -1.116884469985962, -0.22315442562103271, 0.9783003926277161, -2.2504019737243652], [-0.026549089699983597, 0.9600598812103271, 1.7288378477096558, 0.4192456305027008, 0.7869210243225098, -0.2834038734436035, -0.2535204291343689, 0.006409502122551203, -0.9464609622955322, 0.11732052266597748, 0.23467959463596344, 0.9794695973396301, 1.7882970571517944, 0.33955255150794983, 0.4407752454280853, 0.38020560145378113, -1.8028666973114014, 0.478577196598053, 0.034354954957962036, -0.8846963047981262, 0.4098007082939148, -1.2382882833480835, 0.8156172633171082, -0.6123212575912476, -1.0886752605438232, -0.35756537318229675, -0.10970030725002289, -1.2354501485824585, -2.0281107425689697, 1.2981054782867432, -1.284261703491211, -0.659424364566803, -0.35577183961868286], [-1.3108446598052979, -0.4365514814853668, 0.3914957344532013, 2.1902589797973633, 1.455220103263855, 0.9997928738594055, -1.157912015914917, 0.960088849067688, -0.4760870337486267, 1.9024279117584229, -0.8372438549995422, -1.1885597705841064, 0.2792993485927582, 1.240312099456787, -0.8726211190223694, -1.1538082361221313, 0.29970884323120117, -1.222969651222229, -0.34200412034988403, -1.1053526401519775, -0.7955606579780579, 0.33235424757003784, 0.5915045738220215, 0.462802916765213, -1.1798439025878906, 0.05416422709822655, 0.24310925602912903, -1.1842577457427979, -0.4464147686958313, 0.6097335815429688, 1.0225083827972412, 1.207926630973816, -0.9957178831100464], [-0.8028660416603088, -0.08736041188240051, -0.5841209292411804, -1.1596986055374146, -0.19880369305610657, 0.039243679493665695, 1.016602635383606, 0.0021594504360109568, -1.1832267045974731, 1.138392448425293, -0.7801547646522522, 0.9778569340705872, -0.05609746649861336, -0.6341299414634705, 1.9264453649520874, -0.4241088330745697, -2.276127815246582, -0.9432900547981262, -0.6659168601036072, 0.8874530792236328, -0.33722952008247375, -0.6951429843902588, -0.7948446273803711, -1.3862978219985962, -0.1582811176776886, 0.3939249813556671, 0.4637182354927063, -0.09862518310546875, -1.7092335224151611, -0.5752314925193787, 0.33637985587120056, 0.2500740587711334, 1.3591388463974], [1.2648379802703857, 1.0045390129089355, 1.777077078819275, 0.8857879638671875, 0.24294352531433105, 1.485949993133545, -0.5768786668777466, 0.8962158560752869, -1.8457896709442139, -0.09838689118623734, 0.9826818108558655, 1.8390215635299683, -1.9730743169784546, -1.2194558382034302, -0.22949755191802979, -1.3033612966537476, -0.468979150056839, 0.03609301894903183, 1.4470247030258179, -0.892238974571228, 0.5959594249725342, -1.5505070686340332, 1.156083583831787, 0.45588386058807373, 0.5596024394035339, -0.34148871898651123, 1.210231065750122, 0.7127923965454102, -1.7253141403198242, 1.3332270383834839, 2.0510940551757812, 0.5522165298461914, -0.1679176539182663], [0.2850044369697571, 2.28482723236084, -0.2683456540107727, -1.4254059791564941, 1.4899109601974487, 0.6456810235977173, -0.0008382191881537437, 0.5424531102180481, -0.5704626441001892, 1.505360722541809, 0.4919479489326477, -0.5281388163566589, -1.7758952379226685, -1.4390945434570312, -1.0993258953094482, 0.5244581699371338, 0.12036565691232681, -0.5530344843864441, 0.21207955479621887, 0.985340416431427, 0.0844932273030281, 1.2224406003952026, -0.9929238557815552, -0.16473734378814697, 0.8481938242912292, -0.3639611303806305, -0.14604413509368896, -2.2999157905578613, 0.49820196628570557, 0.5129743218421936, -0.6230924725532532, -0.49698272347450256, 0.1296909749507904], [0.9923047423362732, -2.2392148971557617, -2.616438388824463, 0.6562344431877136, 1.5428861379623413, 0.062160421162843704, -0.40183451771736145, -0.1819300800561905, -0.01330346055328846, 0.11034388840198517, 1.6083965301513672, -1.5473889112472534, 0.7326810956001282, 0.2484724223613739, 0.5079293847084045, -2.182305097579956, 0.8615733981132507, -2.32364821434021, -1.1983821392059326, -0.8048321008682251, -0.4721190929412842, 0.09090957790613174, -0.9109734892845154, 0.6957644820213318, 0.40681755542755127, 0.7599347233772278, 0.3173638880252838, -0.3694393038749695, 1.2629344463348389, -0.25047004222869873, -0.330195814371109, 0.4560188353061676, 1.0703057050704956], [0.8246799111366272, -0.31268805265426636, 0.8544406294822693, 0.32552021741867065, -1.805362582206726, -0.005000210367143154, -0.2121913731098175, -0.5081697702407837, -0.4858408570289612, -0.6500990986824036, -0.5463030934333801, 0.00952125247567892, 1.1016736030578613, -0.7539150714874268, 2.0666449069976807, -1.006325364112854, 1.6236298084259033, -0.6658770442008972, -0.0837690532207489, -0.4862793982028961, 1.8097050189971924, -0.3744311034679413, 0.36108261346817017, -1.3231927156448364, -0.30176061391830444, -1.089979887008667, -0.05505608394742012, -2.0198400020599365, -1.8138787746429443, 1.4165327548980713, -0.2320563644170761, -0.9649589657783508, 1.0070873498916626], [-0.7442213892936707, 0.710289478302002, -0.2076931893825531, -1.116620421409607, 0.03305329382419586, -0.9776239991188049, -1.3432561159133911, -2.6633312702178955, 2.1998233795166016, -1.7679386138916016, -1.0564184188842773, -2.084948778152466, -1.9199737310409546, 0.8787534236907959, 1.5946288108825684, -1.158696174621582, -0.12082263082265854, 0.4983322024345398, 0.27485665678977966, 0.45786553621292114, 1.1695462465286255, -1.6934912204742432, 0.5562804937362671, 1.2203669548034668, 0.9696736335754395, 0.059353966265916824, 0.48526284098625183, 0.48054903745651245, 1.360451579093933, -0.1990768164396286, -0.1440541297197342, -1.8188444375991821, -0.0011682405602186918], [-0.4306592345237732, 1.2280662059783936, -0.7029686570167542, 0.44393065571784973, 0.25271859765052795, -0.3569774925708771, 0.9649531245231628, 0.04803653806447983, 0.5708579421043396, -0.1743210107088089, -0.3089892566204071, -1.7813185453414917, 0.34780484437942505, -0.19395321607589722, -1.9013041257858276, -0.3555911183357239, -0.545335590839386, 0.8513015508651733, -0.4159545302391052, 0.22912654280662537, -0.3120248019695282, -1.2175706624984741, 0.024887073785066605, -0.41345661878585815, -1.2927535772323608, -0.9435593485832214, -1.2080312967300415, -0.3977668583393097, 0.487494558095932, 2.5930190086364746, 0.1864939033985138, 0.2801334857940674, 0.562183141708374], [-1.180060625076294, -1.2601020336151123, 0.8675949573516846, 0.5094172954559326, 0.938113272190094, 0.14628809690475464, -1.6217007637023926, 0.07194868475198746, -2.0524168014526367, 0.5233405232429504, 0.2769964933395386, -0.8159721493721008, -0.8666795492172241, -0.9094231724739075, -0.8983151912689209, 0.3509356677532196, 0.5026132464408875, -0.7427237033843994, 0.13776817917823792, 1.419980764389038, -0.4051627516746521, 0.258024662733078, -0.9395707249641418, -2.0066866874694824, -0.3979945182800293, -0.7039204835891724, 1.2575139999389648, -0.8063749074935913, -0.11140891164541245, -0.12228339910507202, -0.274550199508667, 2.056575059890747, 0.0007191899348981678], [-1.0769264698028564, -1.0662426948547363, -0.7396999597549438, -1.038934588432312, 0.08246397227048874, -0.6776779294013977, -0.7065669298171997, -1.1987769603729248, 0.6441663503646851, -0.10321833193302155, -0.6512730121612549, 0.7108118534088135, 0.6076623797416687, 1.2074064016342163, -0.9303717017173767, 1.4292367696762085, 0.7266244888305664, 0.1793835312128067, 0.8136462569236755, 0.7081722617149353, -0.9408215880393982, -1.330755352973938, -1.789583444595337, -0.920840859413147, -0.9281929731369019, 0.6046157479286194, -0.49416065216064453, -0.8374435901641846, 0.5765594244003296, 0.37432289123535156, 0.3760901391506195, 1.4074956178665161, -0.3568952679634094], [-0.2807236611843109, 1.914574384689331, 1.3255313634872437, -1.792946696281433, -0.09823252260684967, 0.32115840911865234, 2.112454414367676, -0.01061192061752081, 0.5304145216941833, -2.3519225120544434, 0.04478298872709274, -0.860512375831604, 1.9200011491775513, -1.3756797313690186, 0.008779442869126797, -2.1717987060546875, 0.7402613162994385, -0.9781771898269653, 0.5742835402488708, 0.8983065485954285, -0.9302676916122437, 1.3348839282989502, 1.5149152278900146, 0.594433069229126, -1.1432583332061768, 0.09890557825565338, -0.5026213526725769, -0.23091302812099457, 0.03197353705763817, -0.26873883605003357, 0.25820693373680115, -0.5280998945236206, 1.431689977645874], [-2.526808261871338, -0.6436015963554382, 0.1617695391178131, -1.284970998764038, -0.412663072347641, -0.2437162697315216, -1.4966198205947876, -0.6292728185653687, -0.04020804166793823, -1.8384084701538086, -0.06322100013494492, 1.3774735927581787, -0.0024771252647042274, 1.507949948310852, 1.1397091150283813, -0.21432091295719147, 0.515099823474884, 1.0319112539291382, -0.6079912185668945, 2.072019100189209, 0.6979082822799683, 0.255918949842453, -0.021805427968502045, -0.19099211692810059, -0.45064279437065125, -0.1313873827457428, 0.41438692808151245, -0.634353518486023, -1.1304303407669067, 1.5309072732925415, 1.590489387512207, -0.10195150971412659, -0.21800914406776428], [-1.0979174375534058, -0.023040322586894035, -2.2529568672180176, 0.8131577968597412, 0.9166340231895447, -1.255744218826294, 0.1477547436952591, -1.6343077421188354, -0.27313435077667236, -0.60235196352005, 0.1792914867401123, 1.3425698280334473, 1.0641388893127441, 0.1138862892985344, 0.5967855453491211, -1.7611325979232788, -0.44235482811927795, -0.14458729326725006, -0.1922607570886612, 0.21417218446731567, 0.5368334054946899, -0.021568486467003822, 0.4227941632270813, -0.5910213589668274, -1.8227453231811523, -0.5585258603096008, 0.5273871421813965, 0.8045322299003601, -1.885867714881897, -1.4262117147445679, -0.617684543132782, -0.2348073571920395, 1.9817605018615723], [-0.45795929431915283, 1.5475995540618896, 1.3176418542861938, -0.8948923349380493, 0.8774781823158264, 0.6532900333404541, -0.860852837562561, -0.2434271275997162, -0.5453047752380371, -0.31026092171669006, -0.6048600077629089, -1.192702054977417, -0.20955362915992737, -0.6233299970626831, 0.29834872484207153, -0.6292323470115662, 0.7884424924850464, -0.5981305241584778, 0.20838885009288788, -0.3676600456237793, 1.6633070707321167, -0.018315935507416725, 0.31944897770881653, -0.6153111457824707, 1.487412929534912, -2.3068196773529053, -2.049015760421753, -1.137660264968872, -1.6164379119873047, -0.8252266645431519, -0.6303850412368774, -0.028483035042881966, -1.4813190698623657], [0.6765457391738892, 0.7064183950424194, 0.6991832852363586, 1.1151520013809204, -0.670414388179779, -3.7679977416992188, 0.5549207329750061, 1.4494202136993408, -0.27025195956230164, 0.739774227142334, 0.03226511552929878, 1.3589082956314087, -0.12282413244247437, 0.38222771883010864, 2.450981855392456, -2.2131237983703613, -0.2488706111907959, 1.4983537197113037, -1.6479604244232178, 0.09928475320339203, 0.8983843326568604, -1.7217707633972168, -0.3000392019748688, 0.912118673324585, 0.6295017600059509, 1.4861860275268555, -1.713024616241455, 0.6913756728172302, -0.4565344452857971, -0.6052491068840027, 1.447505235671997, -1.737714409828186, 0.3383203148841858], [1.1490925550460815, -1.3965381383895874, 0.7470375299453735, -0.4334375262260437, -1.8461668491363525, 0.980688750743866, -1.8626978397369385, 1.383813738822937, -0.711910605430603, -0.2723172605037689, 0.39477482438087463, 0.7907900214195251, 0.9467736482620239, -0.8245117664337158, -1.489797830581665, -0.34852996468544006, -0.6639066338539124, 2.2239181995391846, 0.21247488260269165, 0.2475304901599884, 0.6061961054801941, -1.4521735906600952, -0.5143530964851379, -1.4138424396514893, 1.9953510761260986, -0.6346685886383057, 0.2459847331047058, 0.7155243754386902, 0.3113636076450348, 0.3626692593097687, 0.7222578525543213, 0.76004958152771, -0.4648844599723816], [0.5418962240219116, 1.5209969282150269, -0.4043193459510803, 2.213106870651245, -1.053531289100647, -0.2759469151496887, 0.8472574949264526, 0.1256166249513626, 1.580712914466858, 1.057334303855896, -0.02894216775894165, 0.4695548713207245, -0.11845706403255463, -1.211267352104187, 1.9116370677947998, 0.05762818083167076, 0.5327285528182983, -2.755430221557617, -0.9666861295700073, -0.509304404258728, -0.4227880835533142, 0.14325475692749023, -0.44659146666526794, -1.4629899263381958, 0.46694308519363403, 0.6687488555908203, -0.3058018386363983, -0.2474314272403717, 0.20040200650691986, 0.878034770488739, -0.9595992565155029, 1.4995898008346558, -0.7835356593132019], [-0.08539485931396484, 0.3021692931652069, 0.8105026483535767, -0.8883257508277893, 0.16308173537254333, -0.8294118642807007, 1.4996343851089478, -0.0625474750995636, 1.5083374977111816, 1.039313793182373, 0.3828728497028351, -0.23036427795886993, 0.3595101237297058, -0.34171175956726074, 0.352510541677475, 1.0944455862045288, -1.0603115558624268, 1.1162198781967163, 0.14490365982055664, 0.22394491732120514, 0.6653387546539307, -0.14154288172721863, -0.14901958405971527, 0.19632019102573395, -1.8851860761642456, -0.835516631603241, -0.20279796421527863, 0.9377351999282837, -0.8374294638633728, 0.6385728120803833, 0.12492170184850693, 0.4615686535835266, 0.04279348999261856], [0.30709534883499146, 0.2576027512550354, -0.8271039724349976, 3.2502903938293457, -0.19153210520744324, 0.21993862092494965, 0.18383772671222687, 0.16326306760311127, -1.5419727563858032, -2.391817808151245, -1.1703927516937256, -0.36263731122016907, 1.0515587329864502, 1.8410060405731201, 1.0539880990982056, 2.069675922393799, 1.0463762283325195, -0.6795795559883118, -0.7563943266868591, 0.34195154905319214, -0.46421507000923157, -0.21407008171081543, -1.2966785430908203, 0.5747382044792175, 1.3843786716461182, -2.5175297260284424, 0.5084304213523865, -0.2783489227294922, -1.240092396736145, 0.1603882610797882, -0.8913378119468689, -0.08792746812105179, 0.02707221917808056], [0.7074143886566162, 0.9234904050827026, 1.2984203100204468, -0.22021719813346863, -0.45979249477386475, -1.5067737102508545, -0.505081057548523, -0.5878749489784241, 1.0580514669418335, 0.08533702045679092, 1.4125248193740845, 0.29294946789741516, -0.5791202187538147, 1.099886178970337, -0.035009145736694336, -0.6068607568740845, 0.24506793916225433, -0.2784549593925476, 0.12435968965291977, 0.9012701511383057, 1.127774715423584, -0.07393535226583481, -1.0988141298294067, -1.4582359790802002, 1.1060227155685425, -0.44124269485473633, -0.5298418402671814, -0.9505932927131653, -0.639820396900177, -0.7663716673851013, 1.2306078672409058, 1.4097703695297241, 0.7234137654304504], [-0.719792366027832, -0.5610424876213074, -0.35913681983947754, -0.9751121401786804, 0.5336808562278748, -1.0024164915084839, -0.16312481462955475, 0.0009374450310133398, 0.031466878950595856, -1.7540545463562012, 0.6035351157188416, -2.1358070373535156, -0.12762746214866638, 0.11102542281150818, 0.15819939970970154, 0.5185714960098267, 1.39644193649292, -1.8108388185501099, -0.990541398525238, -2.3956949710845947, -0.8946470618247986, 0.647798478603363, -0.41940346360206604, 0.3882126212120056, 0.35430118441581726, 2.3599650859832764, -0.23136983811855316, -0.4936714768409729, -0.36463090777397156, 1.022636890411377, -1.0653650760650635, -0.2703433930873871, 0.9643120169639587], [-0.2583121657371521, -0.526416003704071, 0.09918398410081863, -0.35272854566574097, 0.3935942053794861, -0.6569238305091858, 1.5015417337417603, -1.5121046304702759, -2.145592451095581, -1.065731167793274, 0.7704298496246338, -0.37895503640174866, 1.637758731842041, -1.077773928642273, 0.43484577536582947, -0.05734919384121895, 1.444725751876831, 0.0979139655828476, -0.06382862478494644, -0.3386208117008209, 0.9276825785636902, -0.37600061297416687, 1.1527037620544434, 0.08764442056417465, -0.7881914973258972, 0.3407512903213501, -0.2344415783882141, -0.4014061987400055, 1.1215636730194092, 0.33226141333580017, -0.8036428689956665, 1.5453014373779297, -0.4230853319168091], [-1.2981587648391724, 0.4916646182537079, 0.02014278434216976, -1.212384581565857, -1.024172306060791, -1.1282484531402588, 1.3797791004180908, 0.9831665754318237, -0.7361054420471191, 0.0031751771457493305, 0.6935985088348389, 0.6855688095092773, 1.1869772672653198, 2.457927942276001, -0.897288978099823, -0.2037564069032669, -0.743502140045166, 1.3265676498413086, 0.11827308684587479, 1.0294450521469116, -0.1681685447692871, 0.037038106471300125, 1.4155890941619873, -0.024147426709532738, 0.12740236520767212, -1.256752848625183, 1.732814908027649, -0.658704400062561, 0.33684584498405457, -0.21729344129562378, -0.16970239579677582, -1.0838935375213623, -0.7002350687980652], [-1.8287628889083862, 0.43022847175598145, 1.3552838563919067, -1.6214100122451782, -0.4875483512878418, 0.4143708050251007, 0.5146179795265198, -0.3156066834926605, -1.1812398433685303, -1.8035709857940674, -1.6437913179397583, 2.114628553390503, -0.6022742390632629, -2.1312084197998047, -0.6494404077529907, 0.6762714385986328, 0.3991154730319977, -1.4838532209396362, 0.1026792898774147, -1.2792972326278687, -1.2131582498550415, 0.6953432559967041, 1.0922316312789917, -2.2884328365325928, 0.6799390912055969, 0.18415388464927673, 0.40973392128944397, -0.12326642125844955, 0.3602060079574585, -0.18213045597076416, 0.3457046449184418, -0.3708389401435852, 0.8817909955978394], [-1.5168603658676147, 0.7180557250976562, -0.8634173274040222, -1.9698121547698975, -1.1364762783050537, -0.6883548498153687, 0.9740656018257141, 1.4669350385665894, 0.17532455921173096, -1.044979214668274, 0.7143205404281616, 2.689962387084961, -2.578665018081665, 1.376423954963684, 0.37085410952568054, -1.2945469617843628, -0.20865505933761597, -0.8598510026931763, -0.11789493262767792, -0.24957609176635742, 0.28290244936943054, -0.784021258354187, -1.3176707029342651, 0.09295748174190521, 0.030073143541812897, 0.5392067432403564, 1.8996367454528809, -0.545200526714325, 0.9804509878158569, -0.39754199981689453, -1.0288605690002441, 1.2147682905197144, -1.8796896934509277], [0.8849246501922607, -0.7431023120880127, 0.02508961781859398, 0.021814610809087753, 0.12624585628509521, -1.771342396736145, -1.3708492517471313, -1.378656029701233, -2.095831871032715, -0.0691559687256813, -0.41842585802078247, -0.3496246635913849, 0.08096820116043091, -0.14532925188541412, 1.437666654586792, 0.5193088054656982, -1.4708075523376465, 0.4889136254787445, -1.2152122259140015, 0.9030336141586304, -1.2275418043136597, -0.8061491847038269, -0.3852047622203827, 0.4848991632461548, -0.41770437359809875, 0.4540065824985504, 2.6646270751953125, -0.6805050373077393, 1.0918060541152954, -1.6112881898880005, -0.2455417662858963, -0.6864802837371826, -1.2117727994918823], [1.2038301229476929, -0.2591373324394226, -0.28814172744750977, -1.6071385145187378, -1.43028724193573, -1.4036364555358887, -0.5248169898986816, 0.34404894709587097, -0.3734819293022156, -0.2872104346752167, 0.43879565596580505, -0.8108862638473511, 0.005336306523531675, -0.3300354480743408, -0.12913525104522705, 0.7253822684288025, 0.7893970012664795, -0.237039715051651, 1.173269271850586, 1.6224768161773682, -0.9495989680290222, 0.5134251117706299, 0.5040676593780518, -1.1526801586151123, 0.4949263334274292, 0.49130168557167053, -2.070441246032715, 0.8422613739967346, -1.4758999347686768, -0.8901575207710266, -1.911316990852356, -0.33437785506248474, -0.47482815384864807], [0.8722385764122009, -0.4562698304653168, -1.37643563747406, 1.4179246425628662, -0.17544561624526978, -0.3007773458957672, 0.9156194925308228, -0.9888858795166016, -1.8128583431243896, 0.2562413513660431, -0.2672420144081116, -1.227473497390747, -0.402447372674942, 0.6951013803482056, 1.3002431392669678, -2.2080814838409424, 0.10351710021495819, 0.20042270421981812, -1.2821789979934692, -0.729926586151123, 0.40107157826423645, 0.11248844861984253, 1.5171936750411987, -0.05625048279762268, 1.162052869796753, -1.5830533504486084, 0.13167443871498108, -0.47474050521850586, -1.6676690578460693, 0.7157323360443115, 1.5018305778503418, 0.26352810859680176, -1.7888318300247192], [0.08597050607204437, -0.31659382581710815, -1.4124858379364014, 0.8543019890785217, 1.2984063625335693, 0.13837644457817078, 0.8985907435417175, 1.06361985206604, 0.7507938742637634, 1.4183677434921265, 1.023157000541687, 0.6626275181770325, 1.6546343564987183, -0.8233789205551147, 2.005817413330078, -0.6204504370689392, 0.06315172463655472, -0.23119528591632843, 0.6624128818511963, 1.2157474756240845, 0.7406305074691772, -0.21093013882637024, -1.0813395977020264, 2.304866075515747, -1.3469252586364746, -0.08770667761564255, 0.7094663381576538, -0.5715884566307068, 1.368421196937561, -1.3708343505859375, 0.525062620639801, -2.228431463241577, 1.952713966369629], [2.638108253479004, -0.9800573587417603, -0.27161484956741333, 0.7141802906990051, -0.062568299472332, 1.5313307046890259, 0.8119798302650452, -1.1985061168670654, -0.8411993980407715, -0.731806218624115, -0.46005794405937195, 0.019985433667898178, -0.10329987853765488, 0.8273008465766907, 0.9678579568862915, 0.9882715940475464, -1.7561088800430298, -0.029074518010020256, 0.2534950077533722, -0.5133402347564697, 1.834403157234192, 1.673601746559143, 0.49277228116989136, -0.5848450660705566, -0.5493894815444946, 0.47072291374206543, -0.44195806980133057, 0.0907803401350975, 0.26928067207336426, 0.4881858825683594, 2.25488018989563, -1.3602641820907593, -1.5005815029144287], [-0.09589898586273193, -0.5117193460464478, -1.2149391174316406, 0.24449960887432098, -0.23208093643188477, 0.1697874367237091, -0.6839438676834106, -0.663449227809906, -2.0876529216766357, -0.697304904460907, -0.6779983043670654, 0.030023908242583275, -0.8261248469352722, 0.9288634657859802, -0.021958010271191597, -0.2878284156322479, 1.7334332466125488, 1.1394572257995605, 0.4025708734989166, -0.8080047965049744, 0.6093148589134216, 2.4937400817871094, 0.7076113820075989, -0.393223375082016, -0.9662273526191711, -1.1197113990783691, -1.2541718482971191, -0.29387161135673523, -2.061673641204834, -1.3096446990966797, 0.196504145860672, 0.8062432408332825, -1.014363169670105], [1.085580825805664, 1.144005298614502, 0.4837956130504608, -0.5743128657341003, -0.5039026141166687, 0.8465922474861145, -0.9432703852653503, -0.6749463677406311, 0.346348375082016, 1.0394617319107056, 0.5261312127113342, 1.860133171081543, 0.5382035374641418, 0.7095589637756348, -0.4374868869781494, 1.7808033227920532, -1.1994032859802246, 0.2578565776348114, 0.003710675286129117, 1.5860525369644165, -1.7977465391159058, 0.6933899521827698, 0.07299965620040894, 0.7408111691474915, -0.06671296805143356, -2.286343812942505, -0.6706362962722778, -0.4070431590080261, -0.18183347582817078, 1.3550323247909546, -0.14667661488056183, -0.1749134659767151, -0.5882591605186462], [-0.3224141299724579, -0.980094850063324, 1.8164191246032715, 1.2524516582489014, -0.1938246488571167, -1.971239686012268, 0.605668306350708, -0.6217864155769348, -0.4065493941307068, 0.3026559054851532, -0.5929405689239502, 0.21755218505859375, -0.82145094871521, 0.1995057612657547, 0.9547842144966125, 0.22981497645378113, -0.6596660614013672, -0.5865942239761353, -1.6826391220092773, -0.4429577589035034, -1.0463193655014038, -0.03405516967177391, -1.2811641693115234, 0.644148051738739, 1.1679718494415283, 2.295997142791748, 0.49946463108062744, 1.1549549102783203, 0.6597863435745239, -0.4566473662853241, -0.27927330136299133, 0.6340168714523315, -0.5666196942329407], [-0.4418620467185974, -0.8041290044784546, -0.8639131188392639, 0.9051325917243958, -1.2173289060592651, -1.7691930532455444, -1.3990808725357056, 1.7209707498550415, -0.7921659350395203, 2.2339189052581787, -0.6516728401184082, -0.7325198650360107, 2.583237648010254, 0.7035468816757202, 0.0368322879076004, 0.6401119232177734, -1.3064908981323242, -0.6730167269706726, 0.06121606379747391, -1.7836315631866455, 1.8410627841949463, -0.7834659218788147, 0.21497222781181335, -0.2512907087802887, 2.9512500762939453, 0.8258538842201233, 0.8299887776374817, -0.8573771715164185, -1.9652012586593628, 1.6658973693847656, -0.9953504800796509, -0.12151817977428436, -0.5166004300117493], [1.321007251739502, -1.113547682762146, -0.9945765137672424, -0.15878714621067047, -1.2645045518875122, 0.705176055431366, -0.6546556949615479, -2.190046787261963, -1.2402119636535645, 0.9216005206108093, -0.22152233123779297, -0.2380335032939911, -2.119309425354004, -0.05773083493113518, -0.02946457266807556, -1.432288646697998, 1.2479020357131958, -0.7588814496994019, 0.7944391965866089, 1.2127660512924194, 1.0010355710983276, 0.2142678052186966, 1.2218976020812988, -0.44252803921699524, -0.8713173270225525, -0.7309157848358154, 0.6863226890563965, 0.22883771359920502, -0.5854735374450684, -1.8622945547103882, 0.13047955930233002, -0.7632626295089722, 1.2823892831802368], [-0.5077354311943054, 0.7208672761917114, 1.323121428489685, -2.210054397583008, -0.46826601028442383, 1.2202192544937134, -0.15905001759529114, 0.09857494384050369, -0.7462381720542908, 0.3020547926425934, -0.28252890706062317, -0.048897720873355865, 1.230759859085083, -1.9943397045135498, 0.8300361633300781, 3.7575411796569824, -0.46699193120002747, -0.8483969569206238, -1.5228526592254639, -0.9330482482910156, 0.9225545525550842, -0.4808100163936615, 0.10179721564054489, -2.2259693145751953, 0.05433781072497368, -0.26724621653556824, 0.09602277725934982, 0.20396873354911804, 0.20479841530323029, -1.0090302228927612, -0.013549759984016418, -0.9637280106544495, -0.08468464016914368], [1.1313281059265137, -0.6971340775489807, -1.0856798887252808, 1.4179487228393555, 1.6853034496307373, -0.3449119031429291, 0.7436985373497009, 1.4514081478118896, 0.09272009134292603, 1.586503267288208, -0.17085234820842743, 0.26527172327041626, -1.000540018081665, -0.9464293718338013, -0.14191733300685883, -1.1170077323913574, -0.6841689348220825, -0.8874569535255432, 1.2730120420455933, -1.107324481010437, 0.6400001645088196, -0.628736138343811, -0.2108902782201767, -0.49769070744514465, -1.1219767332077026, 2.0365638732910156, 0.4585055410861969, -0.8124166131019592, -1.4190102815628052, -1.0102483034133911, 0.07199376821517944, -0.34911245107650757, 0.03120686672627926], [-1.2435638904571533, -0.5446205735206604, -0.21722254157066345, -0.934386134147644, -0.1185380145907402, -0.5104355216026306, 0.6091113686561584, -0.04846624284982681, 1.269591212272644, 1.8234503269195557, -0.12896761298179626, -0.996695339679718, -0.2065039873123169, 1.108787178993225, 0.6029279828071594, -0.5478240847587585, -0.5212636590003967, -0.23756884038448334, -0.9419698715209961, -1.0419690608978271, -1.16730535030365, -0.29786044359207153, -1.102216362953186, 0.43141257762908936, 0.9966529011726379, 0.7729189991950989, -0.6717371344566345, 1.8373103141784668, -0.37766000628471375, 1.1194566488265991, -1.077149510383606, -0.11799504607915878, 1.8300775289535522], [0.022798944264650345, -0.06551290303468704, 0.6136327385902405, -1.609268307685852, 1.218510389328003, -0.6036399006843567, -0.3058144748210907, 1.687220573425293, 0.6324081420898438, -0.7054689526557922, -0.6047933101654053, -0.3377457559108734, -0.37432658672332764, 0.31795868277549744, 1.0969336032867432, -0.917602002620697, -0.03657042235136032, 0.5233070850372314, 0.9182479381561279, -0.8860645294189453, -1.6849993467330933, 0.3372024595737457, 1.5494998693466187, 0.5091332793235779, -0.3024965524673462, -1.1873646974563599, 1.4508296251296997, -1.3742297887802124, 0.34418976306915283, 1.310017704963684, 0.20195774734020233, 0.1554163098335266, 1.0745983123779297], [-1.5146942138671875, -0.6049176454544067, -0.7351235151290894, 1.084187388420105, 0.8891127705574036, -0.35399699211120605, 1.950961709022522, 0.9877452850341797, -0.13326673209667206, -0.2075616717338562, 0.9928106069564819, 0.831153392791748, 0.08828262239694595, 0.16622860729694366, 0.6299760341644287, 0.2295009344816208, 1.2608602046966553, 0.591141402721405, -0.43176138401031494, -1.450405478477478, -1.0189845561981201, -0.739018440246582, -0.30841341614723206, 0.9905819296836853, -0.6460029482841492, -0.14006198942661285, -0.21028642356395721, 1.0718525648117065, -0.19503161311149597, -0.5414747595787048, -0.37144431471824646, -0.3351280987262726, 0.5114478468894958], [0.474945992231369, 0.37564346194267273, 0.5163120627403259, 0.4124102294445038, -0.27548131346702576, 0.5331630110740662, 0.41746777296066284, -0.5865136384963989, -1.3342688083648682, -0.5657451152801514, -0.4520595967769623, -1.2742491960525513, -2.032310724258423, -0.16981783509254456, 0.6690875291824341, -0.008969045244157314, 0.10773415863513947, -1.0828536748886108, 0.0085885189473629, 0.701532781124115, -0.0630626529455185, -0.73561030626297, 0.7569798231124878, -0.6334829926490784, -0.15557312965393066, -1.120652675628662, 0.1579175442457199, 0.36152297258377075, 0.012591136619448662, -0.8080368041992188, -0.46157991886138916, 0.21639692783355713, -0.09482179582118988], [-0.22171662747859955, 1.8458296060562134, 2.478273391723633, 0.1350417584180832, 0.9387320876121521, -0.10986661911010742, 0.8777616024017334, 1.5539265871047974, 0.34063130617141724, -0.35331013798713684, 0.20970883965492249, 0.37985625863075256, -0.3012240529060364, 0.20893658697605133, -0.7905332446098328, 0.7114876508712769, 0.7298285961151123, -0.2809576988220215, 0.27354779839515686, 1.4109400510787964, -0.5502831935882568, 0.779044508934021, -0.04133983701467514, 0.4400462508201599, -0.5696076154708862, 2.109607219696045, -0.6122036576271057, 1.0780926942825317, -0.7593865990638733, -1.1546369791030884, 0.01904722861945629, -0.20769155025482178, -0.6423287391662598], [-0.7379605770111084, -0.5655580759048462, 0.984368622303009, 0.16269102692604065, 0.2727546691894531, 0.5281720161437988, 0.5240306854248047, 1.1009448766708374, 0.5318252444267273, -1.5854336023330688, -0.352181613445282, 0.7004758715629578, 1.3182944059371948, 0.05420364439487457, 0.18721118569374084, -0.22967472672462463, 0.7112058401107788, 1.1726067066192627, 0.04680094122886658, 0.3065390884876251, -0.7644605040550232, -0.9042505621910095, 0.42656898498535156, 1.1022566556930542, -0.3991999626159668, -0.2379390150308609, 0.08623664081096649, -0.360666424036026, -0.2719908356666565, 0.9519915580749512, 1.3257395029067993, 0.6603543162345886, -0.2601564824581146], [1.022500991821289, 0.7164359092712402, -1.0427571535110474, 0.48402711749076843, 0.8427267670631409, 0.21236921846866608, 2.0899717807769775, 0.9459696412086487, 0.9011909365653992, 0.3307592272758484, 0.9071002006530762, -0.9317687749862671, -1.2943469285964966, 0.7911126613616943, -1.8438796997070312, 0.9963704347610474, -0.644330620765686, 0.819264829158783, -1.5665993690490723, -0.5940564274787903, 1.2286591529846191, 1.493628740310669, 1.3109318017959595, 0.5686963200569153, -1.4720616340637207, 0.6178896427154541, -0.18978016078472137, -0.18092814087867737, -0.3438883423805237, 1.7818210124969482, 1.6354472637176514, -0.14776656031608582, 0.25841113924980164], [-0.04556020721793175, -0.9596010446548462, -0.03289930522441864, 1.037282109260559, -0.8466551899909973, -0.47087839245796204, 1.8749223947525024, 0.4546196162700653, -1.516983151435852, 0.754109799861908, 1.6816319227218628, -0.14924365282058716, -0.6934044361114502, 0.39501163363456726, 1.0393571853637695, -0.21611429750919342, -0.2467990219593048, -2.6200971603393555, -1.9792057275772095, -1.309715747833252, 0.14208142459392548, 1.006695032119751, 0.70172518491745, 0.9722415804862976, -0.8592473268508911, -0.4835117757320404, -1.098766803741455, -0.7759919166564941, 1.0016374588012695, 0.16032874584197998, -0.2587509751319885, -0.5948218703269958, 0.5651354193687439], [1.0164674520492554, 1.1092535257339478, -0.4605601727962494, 1.7539442777633667, 0.6694806218147278, -0.676331102848053, -0.07376173138618469, 1.2833938598632812, 1.4656051397323608, -0.450529545545578, -1.0701922178268433, -0.6440037488937378, 0.34168320894241333, -0.6756145358085632, 0.29671505093574524, 1.5783675909042358, 0.8688259124755859, 0.4775187373161316, -0.6283763647079468, 1.4007364511489868, -1.2984215021133423, -0.0011815647594630718, 0.5408336520195007, 1.2541484832763672, -1.8811681270599365, -1.575905442237854, -0.5601462125778198, 2.255281686782837, -0.04707985743880272, 0.5321341753005981, 1.566293716430664, -1.4594324827194214, -0.47837722301483154], [0.10580122470855713, 0.6754146218299866, 0.2063751518726349, -0.665755569934845, -0.14942653477191925, 0.33964693546295166, -0.5366119742393494, 0.055525023490190506, -0.44056177139282227, -0.5111770033836365, -1.7284820079803467, 0.2210964858531952, -0.8771241307258606, -0.04530470445752144, -0.7823072671890259, -1.2890735864639282, -0.862220048904419, -0.6011978387832642, -0.020903071388602257, -0.10028854012489319, -0.5430030226707458, 0.01554049551486969, 0.10107069462537766, 0.08623089641332626, 1.115495204925537, -0.056531719863414764, 0.2520125210285187, 0.08887916803359985, -0.03696179389953613, -0.42896780371665955, -0.7431235313415527, -1.3119257688522339, 0.33971694111824036]], "dtype": "float32"}, {"__nd__": [[-0.29489365220069885, -0.22762563824653625, 0.7245655655860901, 0.7632038593292236, 0.6924785375595093, 1.2822744846343994, 0.33860987424850464, -0.7426949143409729, 0.7297540903091431, 0.6514515280723572, 1.1964656114578247, -0.17519479990005493, 0.6640900373458862, -1.2036164999008179, 0.5920833945274353, 0.6605721712112427, 1.4935624599456787, 1.4716242551803589, 0.3876299262046814, 1.2174279689788818, 0.552365243434906, 0.2945398688316345, 1.5597779750823975, 0.4132893681526184, 0.7978946566581726, -0.16306520998477936, -0.3453797996044159, 0.6241661906242371, -0.8171179294586182, 1.9362022876739502, 1.5046889781951904, -1.3194129467010498, 1.2019692659378052, -1.2878085374832153, -0.14486850798130035, 0.6767037510871887, -0.31237009167671204, 0.9981086254119873, -0.19530817866325378, -1.8152176141738892, 0.9754126071929932, -0.1876583695411682, 1.437214970588684, -0.3460627794265747, -1.0855785608291626, -0.2759855091571808, 0.49442726373672485, -2.136962652206421, 1.8169363737106323, -0.3671073615550995], [0.4259355962276459, 0.21883925795555115, 1.5929914712905884, 0.15204942226409912, -2.1361606121063232, 0.7942264080047607, 0.2651371359825134, 0.14422249794006348, -1.2239391803741455, 0.8267810940742493, -0.9230107069015503, 1.123164415359497, 1.4918309450149536, 0.3459322452545166, 0.3403257131576538, 0.1072925478219986, 2.093668222427368, -0.47250956296920776, 1.3616018295288086, -1.2418612241744995, -0.8002734184265137, -1.8122563362121582, 0.03704371303319931, -1.5730395317077637, 0.574286937713623, 0.4278097450733185, -1.0498722791671753, -0.5534213781356812, -1.0351301431655884, -0.4824362099170685, 0.6159270405769348, -0.3318521976470947, -0.7905750870704651, 0.013681449927389622, -1.450927734375, -0.21188265085220337, -0.1774272620677948, -1.429725170135498, 0.21854358911514282, -0.12850825488567352, 0.948599636554718, -0.5846222639083862, -0.9605821371078491, 0.8891125321388245, 0.7549342513084412, 0.1392548829317093, -0.902070164680481, -0.7359170913696289, 0.5266828536987305, -0.6730678677558899], [1.1058584451675415, 0.020072191953659058, -1.0594167709350586, 1.3594410419464111, 1.607681155204773, 0.4355831444263458, -0.22123484313488007, -0.18188869953155518, -0.3594989478588104, 0.037390079349279404, 0.5025942325592041, 0.2918994724750519, -0.4856294095516205, 1.3691530227661133, -0.21753084659576416, 2.0714495182037354, -0.9945359826087952, 0.9617483615875244, 0.014530928805470467, 0.947394609451294, -0.5744219422340393, -1.0477172136306763, -1.7359687089920044, -0.5857177972793579, 0.13518157601356506, -0.8992565870285034, 0.44859060645103455, 1.0830693244934082, 0.8345542550086975, -0.06412634253501892, -1.2746319770812988, 0.6428699493408203, -0.5165280103683472, -0.7929400205612183, -0.2462347447872162, 0.3125182092189789, 0.3748677670955658, -0.11998848617076874, -0.9997764229774475, -1.0564124584197998, 0.5960147380828857, 0.26356688141822815, -0.41538000106811523, -1.13937246799469, -1.6378142833709717, 0.1507745087146759, 1.005835771560669, 0.1357908993959427, -1.0524604320526123, 1.5208725929260254], [-1.9510256052017212, -0.7427549958229065, -0.7636708617210388, 0.19022785127162933, -0.7258796095848083, -2.1682193279266357, 0.9184992909431458, -0.9415436387062073, 0.2763286530971527, -0.780536413192749, 1.4110639095306396, -1.5974417924880981, 1.974958062171936, -1.0856789350509644, 2.157703399658203, 1.1058357954025269, 2.17228627204895, 1.866078495979309, -0.24190212786197662, -0.9003875851631165, 1.5390464067459106, 0.47608843445777893, -1.2149338722229004, -1.7279694080352783, -0.4448949098587036, 0.5574769973754883, -1.2485790252685547, -0.7256300449371338, 0.6946454644203186, -0.9892962574958801, -0.2594333589076996, 1.338673710823059, -1.4280219078063965, 1.1843323707580566, 0.31568512320518494, 1.02118980884552, -3.09879469871521, 1.5253710746765137, -0.7671810984611511, -0.9793148636817932, 0.2448863536119461, 1.2907251119613647, 0.06015590578317642, -1.6980342864990234, 1.4137508869171143, 0.15094105899333954, 1.046862244606018, 1.0392628908157349, -0.32319456338882446, -1.2166028022766113], [0.29989761114120483, 0.5807323455810547, 0.5006057024002075, -0.3911826014518738, 1.3174676895141602, 0.3667182922363281, 0.17916399240493774, 1.5232495069503784, 0.6737160682678223, 0.8948745131492615, -0.33569005131721497, -0.41660773754119873, -1.5355099439620972, 1.6363356113433838, -1.6590888500213623, 1.0900300741195679, -1.398013949394226, 0.7475332021713257, 0.7482397556304932, 0.7281690835952759, -0.6074103116989136, -0.2798879146575928, 0.9788053035736084, -0.5724311470985413, 0.8041585683822632, -1.0203309059143066, -0.9856090545654297, 1.569303035736084, 2.149243116378784, 1.5257282257080078, -0.3930453658103943, -0.5309750437736511, 0.9312905669212341, 0.5511982440948486, 0.11713720858097076, 0.0029402594082057476, 1.3937196731567383, 0.49271219968795776, -0.02808060124516487, -0.20819970965385437, -0.2503089904785156, -1.241460919380188, 1.2265700101852417, 0.007894786074757576, -1.6700396537780762, -0.7983174920082092, -0.4583938717842102, 0.05360901355743408, 1.570939064025879, -1.1148244142532349], [-0.8898813724517822, -0.9318233132362366, 0.04167100787162781, -0.1704513281583786, -3.1053059101104736, -0.10652477294206619, -0.6032898426055908, -0.399217814207077, -0.7045793533325195, 0.6333852410316467, -1.212041974067688, 0.29239121079444885, -0.18179109692573547, 1.3543646335601807, -2.4340851306915283, -0.20719781517982483, 0.9852440357208252, 0.45120495557785034, -0.08513346314430237, -0.28626638650894165, -0.6989208459854126, 0.02041679620742798, -1.4590871334075928, -0.18408291041851044, -0.34965381026268005, 0.7216296195983887, -0.14063112437725067, -1.7219278812408447, 0.797718346118927, -0.3235187232494354, 0.8392775654792786, 0.9606361985206604, 1.2364064455032349, 1.118633508682251, -1.210412859916687, -0.6867432594299316, 0.33344703912734985, -0.0530235581099987, 0.5973663926124573, -1.2463345527648926, 0.2534125745296478, -0.7202317118644714, 1.14622962474823, -0.24326181411743164, 0.5368741154670715, -1.0494822263717651, 1.4175360202789307, 0.043993156403303146, -0.9838204383850098, 0.758858323097229], [1.5793057680130005, -0.11655687540769577, 1.9637420177459717, -0.6975983381271362, 0.1137014850974083, -0.4138297140598297, 0.43363022804260254, 0.7406242489814758, -1.0598641633987427, 0.229350283741951, -0.9108262658119202, 0.45690053701400757, -0.8669142723083496, -1.0755616426467896, 0.13132734596729279, -0.408875048160553, -1.1348172426223755, -1.1636073589324951, -1.4616849422454834, -1.4716285467147827, 0.7339748740196228, -0.6436705589294434, 0.07269831001758575, -2.344944715499878, -0.14216892421245575, 0.0062080384232103825, 0.05107325315475464, -0.24638426303863525, 0.44922059774398804, -2.09222412109375, -0.670089602470398, 1.245070219039917, -0.2361678183078766, -0.1035064160823822, 3.6640148162841797, 1.7282503843307495, 0.6571133732795715, -0.1402338296175003, -0.6737102270126343, 0.07283665984869003, -0.7320036888122559, 0.3420828878879547, -0.5979408621788025, 0.013463586568832397, -0.7292788028717041, -0.1334851235151291, -0.1413317620754242, -1.2083656787872314, -0.1230081245303154, -0.8350988030433655], [-0.42145299911499023, -0.2895812690258026, -1.3793940544128418, 0.35791879892349243, 0.24459724128246307, -0.8562648892402649, -1.0982460975646973, 0.6396048069000244, 1.3979591131210327, -1.053088903427124, 0.10936755686998367, 1.4218318462371826, 0.9678647518157959, 2.4593923091888428, -0.4806312620639801, -0.6072132587432861, -1.1947648525238037, 0.06972446292638779, 1.2912312746047974, -1.0667694807052612, 0.10151305794715881, -0.3949592113494873, -0.8640904426574707, -0.6457409858703613, -0.5671964287757874, -1.7500213384628296, -0.5317729115486145, -0.41117021441459656, -0.6758841872215271, -0.4630023241043091, 0.9152907133102417, -0.9752108454704285, 0.13157658278942108, 0.9673910737037659, -1.231221079826355, 1.7662363052368164, -1.823956847190857, -0.3850262761116028, 0.5621654987335205, -0.16995283961296082, -0.26935964822769165, 1.415971040725708, 1.1519259214401245, -0.933907151222229, -0.5798577070236206, 1.9327162504196167, 0.6194162368774414, 0.38547512888908386, 1.1952109336853027, -0.3759394586086273], [-0.4196530282497406, -0.07429429143667221, 0.9007154703140259, 0.6696605086326599, -0.7281674742698669, 1.30608332157135, -0.20198164880275726, 2.9270763397216797, -0.41142794489860535, 0.8790708184242249, -1.9834808111190796, -0.4311593174934387, -1.5148568153381348, -1.0679436922073364, 0.7250745892524719, -1.6210922002792358, 0.9958165287971497, 0.7441652417182922, -0.7882323265075684, 0.5373652577400208, -0.6139528155326843, 0.6588292121887207, 0.23751311004161835, 0.7906549572944641, -2.3296992778778076, -2.031126022338867, 0.589658260345459, -0.17000220715999603, -1.1054574251174927, 1.569218397140503, -1.9500060081481934, 2.072145462036133, 1.0490366220474243, -0.43739596009254456, 1.663769006729126, 0.4400269091129303, -0.4579814374446869, 0.21640408039093018, -1.8854526281356812, 0.4417535364627838, -0.6903690695762634, 0.47022807598114014, -0.3895149230957031, -0.4373016059398651, 0.2726907730102539, 1.0872281789779663, -0.01909545063972473, 0.607519268989563, -2.464831829071045, -1.7101333141326904], [1.657670021057129, -0.8716546893119812, -1.6895188093185425, -0.7895626425743103, -0.9290140271186829, 1.508525013923645, 1.4036284685134888, -0.23755072057247162, -0.5631338357925415, -0.5158030986785889, 1.3873507976531982, -1.198851227760315, -1.6062389612197876, -1.487395167350769, -0.3538319766521454, 1.117097020149231, -1.6440178155899048, 1.0789886713027954, -0.5433033108711243, -0.3179700970649719, 0.12425066530704498, -0.05559340864419937, -2.264462471008301, 2.429957389831543, -0.2926662266254425, -0.7138704061508179, -0.918107807636261, 0.22878092527389526, 0.11786015331745148, -0.5116540789604187, -0.8582608699798584, 1.7566680908203125, -0.5813036561012268, 0.4918835759162903, -0.2022688090801239, -0.8058778047561646, -0.63057941198349, 1.4461710453033447, -1.6421838998794556, -0.1111140251159668, -1.1836652755737305, 1.154997706413269, 0.9427019357681274, 0.07050982862710953, 0.578978955745697, -1.533183217048645, 1.0563734769821167, 1.025160312652588, 0.7964121103286743, -1.3578532934188843], [1.417523741722107, -0.16682256758213043, -2.263535737991333, 0.9802421927452087, 0.6105175018310547, -1.5634444952011108, 2.191577911376953, 0.7925490140914917, 0.8642694354057312, 0.22017601132392883, 1.0302220582962036, -2.156918525695801, -0.2931259274482727, -0.144461527466774, 0.41816744208335876, 1.680561900138855, -0.46954694390296936, 0.4291965663433075, -1.074721336364746, -0.4903208911418915, 0.8734135627746582, 0.1788935512304306, -1.1942930221557617, -0.9327466487884521, -0.25638964772224426, -1.0826427936553955, -0.38776034116744995, -1.165462851524353, -1.1282427310943604, -0.4040403664112091, -0.0850614458322525, 1.4715311527252197, -0.7339051961898804, 1.182477593421936, -2.1911840438842773, 0.12064320594072342, -1.871583342552185, -0.5847625136375427, 0.5013070106506348, 0.0446026585996151, -0.4075547456741333, 0.949201762676239, -1.700259804725647, -0.13947871327400208, 1.0736174583435059, -0.7368273735046387, -0.28714582324028015, 1.184221625328064, -0.7876341342926025, 1.2922176122665405], [0.058138009160757065, -0.423669695854187, -0.8614294528961182, 0.6368367075920105, 1.6361660957336426, -2.243959665298462, 2.631810426712036, -0.3393356502056122, -0.09503275156021118, -0.8179579973220825, -1.2120676040649414, 0.6494922637939453, -1.7295782566070557, 0.2986627221107483, -0.7859243750572205, -0.10322856158018112, -0.6283469200134277, 0.7088385820388794, 1.322974681854248, 0.29603174328804016, -1.0060392618179321, -0.08539615571498871, -1.3594313859939575, -2.4176902770996094, 0.12756936252117157, 0.38068047165870667, -0.91008460521698, 0.002672482980415225, -0.04392004758119583, -0.8426980972290039, -0.9130293726921082, -0.025842798873782158, 0.5816428065299988, 1.0986531972885132, 0.39145514369010925, 0.7380170226097107, -1.4038331508636475, -1.3520005941390991, 0.6645359992980957, -1.2096714973449707, -0.6510354280471802, 0.2867007851600647, 0.8583133816719055, 0.8945786952972412, -0.3836454749107361, 0.468523234128952, 2.0307819843292236, 0.37245655059814453, 1.060547113418579, -1.0211269855499268], [-1.100699782371521, 1.4606682062149048, 0.5242165327072144, 0.8691370487213135, -1.6984246969223022, -2.638676881790161, 0.03080044686794281, -0.4149455726146698, -0.2504291832447052, -0.5746155381202698, 1.0549793243408203, 0.47744378447532654, 1.7973991632461548, -0.9096495509147644, -0.3227140009403229, -0.13451696932315826, -0.42088234424591064, 2.856851577758789, 1.0292963981628418, 0.5245506167411804, -0.5371926426887512, -0.9079551696777344, -2.0850069522857666, -1.6557585000991821, 1.48731529712677, -0.4345776438713074, 0.3613467514514923, 0.8643251061439514, -2.197402000427246, 0.3389036953449249, 1.1320533752441406, -1.2814130783081055, -0.6208911538124084, -1.5836894512176514, -1.9882844686508179, 0.0022190995514392853, -0.38683080673217773, 1.0104882717132568, 1.0024493932724, -0.16466820240020752, 1.130163311958313, 0.6545385718345642, 1.7192281484603882, 1.4566140174865723, -0.6041373014450073, 1.9719364643096924, 0.8945356607437134, -0.7342209219932556, -1.778987169265747, 0.3683503568172455], [0.2274673730134964, -1.1464083194732666, 0.31683114171028137, 2.335076093673706, 0.9457080960273743, -1.8271437883377075, -0.22689183056354523, -0.2992511987686157, -0.45797455310821533, -0.3293705880641937, -0.3317406475543976, 0.8416281342506409, -0.7243439555168152, -0.15906254947185516, -0.8317772150039673, -1.1626076698303223, -1.3342163562774658, -0.44414588809013367, 0.037974096834659576, -0.18865545094013214, 2.063314437866211, 0.6440221667289734, -0.8066678643226624, 2.288508415222168, 1.3564568758010864, -1.171634554862976, 1.4285507202148438, -1.152898907661438, -0.026333967223763466, 0.33161717653274536, 0.6162164211273193, -0.5590810179710388, 0.5057956576347351, 0.30211424827575684, -0.30958321690559387, 0.27419760823249817, 0.30397137999534607, 0.18316225707530975, -0.7922418117523193, -1.181965708732605, -0.13090930879116058, 0.9758108854293823, 0.7478390336036682, 2.1813831329345703, 2.349515914916992, -1.1602091789245605, -1.218732476234436, -1.103071689605713, 0.47750595211982727, -0.9114797711372375], [0.8208231925964355, -1.3531502485275269, -0.36717158555984497, -0.6692163348197937, -0.8183422088623047, 0.9733860492706299, 0.454222708940506, -1.137001872062683, -1.0501372814178467, 1.0033860206604004, 0.04216624051332474, 0.03060969151556492, -0.23344601690769196, -0.7699293494224548, 2.883819580078125, 0.508497953414917, -0.6910054087638855, 0.7385743856430054, 0.2378004491329193, 0.0003857812553178519, -0.17077453434467316, -0.5855147838592529, -0.4461878538131714, -1.7479370832443237, 1.7832345962524414, 2.2117466926574707, 1.2401021718978882, -1.005082607269287, -0.6470393538475037, 1.494524359703064, 0.8221380710601807, 0.4370344877243042, -1.1000499725341797, -1.0477491617202759, 1.255324125289917, 0.11983102560043335, -0.935347855091095, -2.0659537315368652, 1.5308603048324585, -0.41651588678359985, -1.2884187698364258, -0.05240415036678314, -1.3184007406234741, 0.5026088356971741, -1.2561591863632202, -0.2778390049934387, -2.145725727081299, -0.5164003968238831, -1.1357921361923218, -0.3676931858062744], [0.9510999917984009, 0.861781895160675, 0.3359469175338745, 1.244585394859314, -0.7295312285423279, -0.9887392520904541, 1.148601770401001, 0.0012989359674975276, -1.7451603412628174, -1.815014123916626, -0.28310659527778625, 1.8824976682662964, -0.06262164562940598, -0.8988449573516846, 0.6487563848495483, 0.6088430285453796, 1.8563889265060425, -1.7182183265686035, 2.000383138656616, 0.009843943640589714, 0.5807315707206726, 2.2699813842773438, 0.7363042831420898, 0.08203806728124619, 0.43421801924705505, -0.44385215640068054, -1.0568619966506958, -1.516505479812622, -1.4657427072525024, -0.7875732779502869, -1.8867067098617554, -0.7557993531227112, 2.044092893600464, 0.359360009431839, -2.387854814529419, -0.345485657453537, 0.15020103752613068, 0.34058114886283875, 0.625632643699646, -1.5233278274536133, 0.3228168189525604, 0.5766950845718384, 0.1084413230419159, -0.06159384548664093, -0.36275380849838257, -0.5115762948989868, 0.2390671968460083, -0.1434325873851776, -0.8470759391784668, -0.9222463965415955], [-2.0242156982421875, 0.7371552586555481, 1.1880767345428467, -0.581394374370575, 1.6014389991760254, 1.100539207458496, -0.05674423649907112, -1.0440618991851807, -0.37988579273223877, -1.9199150800704956, 0.06974037736654282, 0.2909139096736908, -0.1565067023038864, 1.2828341722488403, -1.1116100549697876, -1.8374346494674683, 0.1655832976102829, -1.763810157775879, 1.458625078201294, -0.32280653715133667, 0.5343409776687622, 0.07247921079397202, 0.00240227859467268, -0.9826515316963196, -1.4164223670959473, -0.18427687883377075, -0.33124032616615295, 0.7507110834121704, 0.0648527592420578, 3.563298225402832, 0.949830949306488, -1.6927876472473145, 0.3487931787967682, -0.05120568722486496, 2.6270344257354736, -0.1337510347366333, -0.3148338496685028, -1.1405142545700073, 0.2304186224937439, 0.3848130404949188, 0.6951789259910583, -0.4257126748561859, -0.8432270288467407, 1.296546220779419, -0.21474522352218628, -0.9033794403076172, 0.6301528811454773, 1.6115275621414185, -0.6332997679710388, 0.30823302268981934], [0.4004955291748047, -2.2941722869873047, 0.7246740460395813, -1.342728614807129, -1.0246773958206177, 0.48517483472824097, 0.04159156605601311, 1.73343825340271, -0.7444857954978943, 0.9491186141967773, -1.7668023109436035, -0.984320342540741, -0.82405024766922, 1.4308844804763794, 0.49168938398361206, 0.06905114650726318, -0.40329599380493164, -0.4204418957233429, 0.3097444176673889, -0.2689228057861328, 1.4489576816558838, 0.7856079936027527, 0.9471412301063538, 0.8131369948387146, -0.792937695980072, -0.48119235038757324, -0.580651044845581, 0.9371213316917419, 2.05790114402771, -1.8183590173721313, 0.43523627519607544, 0.5625103712081909, -0.21339797973632812, -0.45415738224983215, -0.9195059537887573, 0.17652449011802673, 1.2584058046340942, 0.7772361636161804, -2.9645304679870605, -2.466975688934326, -0.2907254695892334, 1.4538545608520508, 0.2231423556804657, 0.8517555594444275, 0.3945610225200653, -0.04022981598973274, -1.5782973766326904, -0.5692254900932312, -0.6765251159667969, 1.2131667137145996], [0.40207353234291077, 0.10512244701385498, 0.2715143859386444, 0.38285601139068604, -0.07153027504682541, -0.6734410524368286, -2.2633697986602783, -1.562956690788269, -0.6956251859664917, -1.6425645351409912, -0.4042893350124359, 0.6252009272575378, 0.31482192873954773, 0.14000007510185242, 0.4763847589492798, -0.15480248630046844, 0.6526604890823364, -1.1937259435653687, 0.5352431535720825, -0.4138876497745514, -1.4445794820785522, 1.2144170999526978, 0.4054468870162964, -1.911102533340454, -1.024153709411621, 1.3246814012527466, 0.4732552170753479, -2.2607622146606445, -0.23188894987106323, 1.0528916120529175, -0.2823069095611572, -0.3435870409011841, -0.09265919029712677, 0.8146039247512817, 0.04438291862607002, 0.511604905128479, 1.0389039516448975, -0.05748613923788071, -1.5582164525985718, 0.6185095906257629, 1.544312834739685, -0.3276195824146271, 0.17975029349327087, 0.21151147782802582, -1.0759713649749756, 0.38520580530166626, -0.4685728847980499, -0.15503093600273132, -0.4414218068122864, -2.4265310764312744], [-0.9510751366615295, -0.5035111308097839, 0.6238594055175781, -0.8673586249351501, -1.6202940940856934, 0.062088873237371445, -1.186166524887085, -0.4850134253501892, 0.5505208969116211, 0.9274881482124329, -0.1481567919254303, -0.8234906792640686, -0.3723835051059723, -1.845357894897461, 1.100740909576416, 0.8810153603553772, -0.002623653970658779, 0.6124134063720703, -1.4372907876968384, -0.7699332237243652, 1.3551331758499146, 0.8439428806304932, -1.1965917348861694, 0.6047784686088562, -1.3623746633529663, -0.6742733716964722, -0.2198168784379959, 0.5312091708183289, 0.4965271055698395, 1.0624805688858032, -0.5517519116401672, 1.724762201309204, 2.162310838699341, -0.11051683127880096, -1.5263540744781494, -1.0760961771011353, 0.3452169895172119, -0.3347579538822174, 0.7947940230369568, -0.7388173341751099, -0.14604303240776062, -0.8924309611320496, -0.702032208442688, -1.77492094039917, 1.6046111583709717, 0.7742415070533752, 0.32492026686668396, -1.5936734676361084, -1.4467658996582031, -0.2679276466369629], [0.08887839317321777, 0.486747682094574, -0.9904703497886658, -0.08090400695800781, -0.6087045669555664, 0.5146785974502563, 1.2874619960784912, 1.0321069955825806, 1.1789354085922241, -0.44243523478507996, 1.0979571342468262, -0.6152939796447754, -0.5412255525588989, -0.5066548585891724, 0.3310129940509796, 0.450759619474411, 0.4042391777038574, 0.777944028377533, -1.6679843664169312, -0.7540982961654663, 0.07207559049129486, 0.23821303248405457, -1.5235519409179688, -2.834602117538452, 0.8113420009613037, -0.371684193611145, -0.14082327485084534, 0.15598678588867188, -0.7522332072257996, -1.3620445728302002, -0.37569186091423035, -0.7001377940177917, 1.2189351320266724, -1.3860772848129272, 1.0451960563659668, -1.5135915279388428, -1.0728355646133423, -1.2743911743164062, 0.33978497982025146, 1.122406244277954, 0.6892522573471069, -0.6707316040992737, -0.6173789501190186, 0.3471987843513489, -0.306484580039978, -1.5812734365463257, -0.4740085303783417, -0.45739102363586426, -0.36191651225090027, -0.9897106289863586], [0.03456631675362587, 1.0068877935409546, 0.9901683330535889, -1.7632108926773071, -2.043210029602051, -0.7302117943763733, 2.3583998680114746, -0.12064532935619354, 1.317313313484192, 0.5067875385284424, -0.8583232760429382, -1.231028437614441, -0.6065053343772888, 0.18404312431812286, -0.5671141147613525, -0.17289474606513977, 1.0806645154953003, 0.6885601282119751, -0.010809788480401039, 0.06210758909583092, 0.30858495831489563, -1.0192981958389282, -0.291686475276947, 2.3683764934539795, 1.0145875215530396, 0.8986137509346008, -0.8877911567687988, -0.13251850008964539, -0.7545868158340454, 2.034396171569824, 1.5814673900604248, 1.0819729566574097, -0.32791149616241455, 0.34058713912963867, -0.693020761013031, -0.41850805282592773, -0.4396449029445648, -0.15532584488391876, 0.07502425462007523, -1.075295090675354, -1.1419671773910522, 1.3774865865707397, -0.6555546522140503, 0.12731966376304626, 0.14919936656951904, 0.41699886322021484, 0.801291823387146, 3.087733030319214, -1.967136263847351, -0.9586904048919678], [1.6990491151809692, -1.6908495426177979, -2.1583290100097656, 1.2315696477890015, 0.6144797205924988, 0.6308344602584839, -1.1030566692352295, -2.0396950244903564, -1.2315764427185059, 3.1737704277038574, 1.0972310304641724, -0.651327908039093, 2.589353084564209, -0.12645192444324493, -0.733984112739563, -0.4447003901004791, -1.6352204084396362, 0.8739778399467468, 0.16411912441253662, 0.8902482390403748, -0.5627276301383972, 1.3324308395385742, -1.6777002811431885, 1.0164928436279297, 0.8641127943992615, -0.6487624049186707, -0.7091285586357117, 0.565910816192627, -1.2900338172912598, -0.3599904179573059, -0.7019692063331604, 1.5109970569610596, -0.8248934149742126, -0.3169247806072235, -0.24441945552825928, -0.03825821354985237, 0.7952659726142883, -0.1652614176273346, -0.23448796570301056, -0.41918742656707764, 1.243301272392273, -0.5376797318458557, 1.579474925994873, -1.3879152536392212, -0.02550435997545719, 0.43106499314308167, -0.11495061963796616, 1.3409146070480347, 0.28697487711906433, -1.325626254081726], [0.8257302641868591, 0.015898263081908226, -1.0439766645431519, -0.3167405128479004, 0.9022573232650757, 1.350619912147522, 1.3031965494155884, 0.5110377669334412, 0.03950054571032524, 0.8826654553413391, 0.6325737237930298, -0.5793702006340027, -0.7923598885536194, 0.2497047483921051, -0.09346086531877518, -0.39633703231811523, -1.2053470611572266, 0.43532949686050415, -0.5452427268028259, 0.4893653690814972, -0.3822953999042511, -0.3119625151157379, 0.8756542205810547, 1.624958872795105, -0.2284456342458725, 0.007414609659463167, -1.369726538658142, 0.40484797954559326, -0.05271105840802193, 1.1864213943481445, -1.0409959554672241, 0.6146268844604492, 1.8834606409072876, 0.6883020401000977, 0.7080949544906616, -0.23436854779720306, 0.7710458040237427, -1.3887121677398682, 1.5176633596420288, 2.326855182647705, 0.04452645033597946, 0.3372933566570282, 0.28869467973709106, -0.8523188829421997, -1.365804672241211, -1.8711243867874146, 0.7854153513908386, 0.2953707277774811, 1.9025086164474487, -1.9221235513687134], [-0.38497933745384216, 0.31259357929229736, -0.6277384757995605, 0.39448922872543335, -0.38517990708351135, 0.40253040194511414, -2.051766872406006, -1.1210989952087402, -1.6314711570739746, -0.26929548382759094, 0.4677882492542267, -0.7009813785552979, 0.5358653664588928, 0.7267922759056091, -0.6825883388519287, 0.17337901890277863, 0.6321101188659668, -0.30569660663604736, -1.1334599256515503, 0.7635881900787354, 2.633904218673706, -1.082214117050171, -0.5668436884880066, 0.2964678704738617, 1.0823205709457397, -0.6341980695724487, -1.0165845155715942, 1.6083532571792603, -0.047822579741477966, 0.24267315864562988, 2.807063102722168, -1.290223240852356, -1.3550776243209839, 0.6651382446289062, -1.4036744832992554, 0.42101186513900757, 0.8094975352287292, 0.39794817566871643, 1.660935401916504, 0.7019261717796326, -0.013973316177725792, 1.2127724885940552, -0.23943153023719788, -0.2233610898256302, 0.5575894713401794, -1.5181576013565063, -0.28715214133262634, -0.45258164405822754, 0.380522221326828, -0.6722116470336914], [-0.46946296095848083, -0.11020923405885696, 0.49035611748695374, -1.3400346040725708, -0.5221797823905945, 0.10261697322130203, -1.3498848676681519, 1.2300807237625122, 0.6810116171836853, 1.7957606315612793, -0.05276034027338028, 0.23544643819332123, -1.2966738939285278, 2.13960337638855, 0.9595680236816406, -0.08622090518474579, 0.019455045461654663, 0.8094500303268433, 0.8753104209899902, 1.906605839729309, 0.45513179898262024, -0.6490171551704407, -1.4912357330322266, -0.4829266667366028, 0.251412570476532, -0.43936917185783386, 0.7207179069519043, 1.1647554636001587, 0.03174485266208649, -0.8073902130126953, -0.29842087626457214, -0.15358257293701172, 0.3084877133369446, -0.15620721876621246, -1.8959981203079224, -0.39409494400024414, 0.6452283263206482, -0.627346932888031, -0.3807252049446106, 2.6497926712036133, -0.31007444858551025, -2.7115371227264404, -0.5048575401306152, -0.8735126852989197, -0.9590675830841064, 0.5231752395629883, -0.3524523973464966, 0.10181336104869843, -1.3948580026626587, -0.5918375849723816], [-2.262822151184082, 2.156470537185669, -1.2465171813964844, -0.5837334394454956, 0.38781803846359253, 0.5623561143875122, -1.9372867345809937, -0.35438302159309387, 0.42990803718566895, -0.3002009987831116, -0.39242637157440186, -0.8695212602615356, 0.3480885922908783, -0.5723089575767517, -0.4769570827484131, 1.754491925239563, -1.9854826927185059, -0.41845500469207764, -0.0695427879691124, 0.7170159816741943, -0.5426803231239319, -0.15873974561691284, 0.1625450849533081, -1.6427712440490723, -0.35976698994636536, 1.1381419897079468, 1.0746616125106812, 0.8566325902938843, 1.3095371723175049, 0.2988162040710449, -0.19490614533424377, -0.6808376908302307, 0.3308844566345215, 1.6061105728149414, 1.236817479133606, -0.19533294439315796, 0.1076713353395462, 0.9446232318878174, 1.06084406375885, 1.330500602722168, -0.3338034749031067, -0.6151017546653748, 0.23462006449699402, 0.5346651077270508, -0.6818456649780273, -1.1438515186309814, 0.28193625807762146, -1.5950977802276611, -2.4353907108306885, -0.29843002557754517], [1.6595933437347412, -0.4622820019721985, -0.47850871086120605, -0.4227825105190277, 0.2984481751918793, -1.1699106693267822, 0.10567401349544525, -1.0472159385681152, -1.6841188669204712, -0.37310791015625, -0.11294993758201599, 1.5016872882843018, -1.5569384098052979, -0.06332091242074966, 0.0554133802652359, 1.7240574359893799, 1.817244291305542, 1.4335006475448608, -1.1260862350463867, -0.15105916559696198, 3.2155938148498535, 1.6254605054855347, -0.3911444842815399, -0.781146228313446, 0.43410637974739075, -0.9875398874282837, 1.0202765464782715, -0.6201590895652771, 0.8611266016960144, -1.8285696506500244, 0.9328784942626953, 0.2900281250476837, 1.707410216331482, -1.014646053314209, -0.049045808613300323, -0.7862211465835571, 0.8087397217750549, -0.13845643401145935, 1.3661569356918335, 1.684255838394165, 0.3427085876464844, 0.7393924593925476, 0.7639765739440918, 1.074670433998108, -0.8179081082344055, -1.2729685306549072, 0.7478671073913574, -0.2758277654647827, -1.001766562461853, -0.03670923411846161], [-0.7025889158248901, -0.5753227472305298, 0.29576170444488525, -2.3653311729431152, 0.0991709977388382, 0.7668958306312561, -0.13582603633403778, 0.10684617608785629, -0.9430207014083862, 0.1773575395345688, 0.8785650134086609, 0.06626737117767334, 0.7994179725646973, 0.933002233505249, -0.7612165808677673, -0.06740135699510574, 0.4980207085609436, -0.1485130339860916, 0.1385861039161682, -0.20327775180339813, -0.16777074337005615, 0.8190354704856873, -0.521034300327301, -1.1536626815795898, 2.5530474185943604, 1.7250263690948486, 0.9327877163887024, -1.0466697216033936, -2.0998568534851074, -0.685957133769989, 0.5229571461677551, 0.7834556102752686, -0.6687882542610168, 0.0832347422838211, 1.0543309450149536, 0.41502445936203003, 0.0663740485906601, 1.68538236618042, -1.211496353149414, -1.3490861654281616, 0.16668039560317993, 1.3427386283874512, 1.1687078475952148, 0.35855376720428467, -0.20234571397304535, -1.1258273124694824, -0.6164645552635193, -0.3361782431602478, 0.3565721809864044, -0.6747019290924072], [-0.6791474223136902, 0.557927131652832, -0.3058668076992035, -1.5166490077972412, -0.06123563274741173, -0.41388365626335144, -0.44280508160591125, 1.4303609132766724, 0.2963597774505615, -1.6049634218215942, -0.16352947056293488, 0.20739614963531494, 0.482186883687973, -1.051658034324646, 0.9139016270637512, 0.9496085047721863, -1.9995919466018677, -1.4915637969970703, -0.6397563815116882, 0.6146972179412842, -0.2695263624191284, 0.5772497057914734, 0.768519937992096, -0.5960761308670044, -1.3519046306610107, 0.8407791256904602, 0.20258431136608124, 0.6788392663002014, -0.4202357232570648, -0.8186793923377991, -0.13317088782787323, -0.24104364216327667, -0.11921615153551102, 0.3768041431903839, -1.7257896661758423, -1.0477110147476196, -0.3662351071834564, -0.7992873787879944, 0.7005767822265625, 0.7140751481056213, 1.8259481191635132, 1.44326913356781, -1.6470386981964111, 0.8437207341194153, -0.029386445879936218, -0.28822967410087585, -0.744920015335083, 0.41289836168289185, 0.5271305441856384, -0.4549640417098999], [-0.5876159071922302, -1.9939687252044678, 1.3093149662017822, -0.03998443856835365, 0.06919459253549576, -1.1868267059326172, -1.057979702949524, -0.7116740345954895, -1.036529541015625, -0.13342086970806122, 0.3180595934391022, 0.32170528173446655, 0.45481234788894653, -0.9656554460525513, -0.13994990289211273, -0.051144346594810486, 0.003995598293840885, -0.276848703622818, 0.04516400769352913, -2.386679172515869, -0.8349555730819702, -0.0004781889438163489, -0.7096040844917297, -1.3220257759094238, -0.509190559387207, -1.9548219442367554, -0.3269571363925934, -0.5775059461593628, -0.6575673818588257, 1.0341880321502686, 1.2508761882781982, -0.46780216693878174, -1.3991048336029053, 0.6038889288902283, 0.7962014675140381, -1.9853712320327759, -0.653784990310669, -0.9695145487785339, 2.1755552291870117, -0.6427586674690247, 1.5768868923187256, 0.6725166440010071, 1.3221638202667236, -0.32253751158714294, 0.695327877998352, 0.6436320543289185, -0.6524344086647034, -0.33590078353881836, -1.1896487474441528, 0.5052012205123901], [-2.3389458656311035, -0.7501124739646912, 0.36563462018966675, 0.44065961241722107, -0.7214556932449341, 0.9552919268608093, 0.08952955901622772, 0.655512273311615, 0.9831815361976624, -0.6653455495834351, 0.5016297101974487, -1.7170302867889404, -1.0428667068481445, 1.2984980344772339, 1.0741047859191895, 2.1831443309783936, -0.4692382514476776, -0.08406099677085876, -0.0939297303557396, -1.0629802942276, 0.5848077535629272, -0.07820317149162292, 0.5378762483596802, -0.09526724368333817, 1.0348715782165527, 0.1035667210817337, -0.143172949552536, -0.32504501938819885, -0.2998296320438385, -0.5883328914642334, 0.43580400943756104, -1.2040092945098877, 0.6574526429176331, 0.45168066024780273, 0.5213439464569092, -1.308923602104187, -0.06038028001785278, -0.9933180809020996, 0.932930052280426, -0.7105221152305603, -0.7694498896598816, 0.2853214740753174, -0.12134234607219696, -0.8893519043922424, -1.8012206554412842, -0.915924072265625, 0.5503101944923401, 0.0347757525742054, -0.6521821618080139, -1.6184197664260864], [-0.3056657016277313, 0.24076202511787415, 1.2973673343658447, 1.5656354427337646, -1.0067064762115479, -0.7638749480247498, 1.0487165451049805, 0.4238404333591461, -0.5194843411445618, -0.7078648805618286, -1.531586766242981, 0.7571381330490112, -1.1508220434188843, -0.009252703748643398, -0.8307551741600037, 0.7709883451461792, 0.14813627302646637, 0.15589065849781036, -1.0210883617401123, 0.8576659560203552, -0.8891359567642212, 0.002860646927729249, -1.5042762756347656, 1.358158826828003, 0.968702495098114, -0.4805198907852173, 0.08254794031381607, -0.5731602311134338, 1.6095243692398071, -0.014022227376699448, -0.11691784113645554, 0.1461201012134552, -0.44051575660705566, -0.2448863387107849, -0.26302433013916016, 0.47125348448753357, 0.10232546925544739, -1.3578596115112305, -1.75978422164917, -1.2530014514923096, -0.3272581994533539, -0.029248028993606567, -0.9310663342475891, -0.10413903743028641, 1.1752604246139526, 0.7514644861221313, -0.7268681526184082, -1.6504814624786377, 0.10705269128084183, 2.842708110809326]], "dtype": "float32"}, 16, 16, 16, 3], "e": {"__tup__": [{"__nd__": [[0, 3, 6, 9], [1, 4, 7, 10], [2, 5, 8, 11], [12, 14, 16, 18], [13, 15, 17, 19]], "dtype": "int32"}, {"__nd__": [[-4.198798656463623, -1.8445621728897095, 0.4558938443660736, -0.35646310448646545, -3.200796365737915, -1.2712056636810303, 4.642147064208984, 2.8651349544525146, 1.940441370010376, -4.1238250732421875, 0.5729866027832031, -2.523942232131958, -3.0619521141052246, -3.7900099754333496, -3.023660898208618, -2.417747974395752, 2.576920509338379, 7.710851669311523, 3.070490837097168, 7.384742259979248, -5.566464900970459, 4.456188201904297, 4.4141411781311035, 13.034791946411133, -5.081598281860352, -0.4941246509552002, -5.984727382659912, 4.289219379425049, 2.304433822631836, 4.649763107299805, -0.3385769724845886, -3.2737202644348145, 3.736825942993164, -0.8162312507629395, -9.743209838867188, -7.972615718841553, 1.9673830270767212, 4.690666198730469, -5.27589750289917, 0.3810555636882782, 7.359824180603027, 1.9839919805526733, 9.170445442199707, 4.862717628479004, 0.9057326912879944, -0.9471084475517273, 5.232356071472168, 5.7033843994140625, 10.47515869140625, -7.444707870483398], [3.1767446994781494, 0.46911484003067017, -1.8470638990402222, 1.8322817087173462, 1.6074858903884888, -15.055277824401855, 11.959298133850098, 2.896442174911499, -2.945420265197754, -12.126047134399414, 13.262587547302246, 4.158372402191162, 8.925004959106445, -9.037442207336426, 0.6491439938545227, 4.499372959136963, -0.8663967251777649, 1.240821361541748, 0.8836493492126465, -9.619668960571289, 0.7482147812843323, 3.613767385482788, -6.079751491546631, -8.76408863067627, 9.940945625305176, -6.051880836486816, -1.963992953300476, -2.3318920135498047, -9.880975723266602, -13.587263107299805, -1.0131065845489502, -5.658042907714844, -8.15388298034668, -4.665931224822998, -6.398260593414307, -6.597636699676514, -9.495681762695312, 6.0108466148376465, 4.9110636711120605, -6.553220272064209, 7.882357597351074, 12.447785377502441, 4.58107328414917, 6.687122821807861, 6.3666911125183105, -0.8003543019294739, -4.512881278991699, -1.3458973169326782, 3.229146957397461, 10.579100608825684], [6.272463798522949, -2.99098539352417, 4.150546073913574, 1.5608614683151245, 0.3736974895000458, 8.041839599609375, -11.287567138671875, 2.3234033584594727, 0.7899281978607178, 11.052648544311523, -3.8844704627990723, 2.0401394367218018, -3.0172276496887207, 10.50180435180664, -2.8573081493377686, 4.7749152183532715, -1.6802979707717896, -0.8470450639724731, -3.7035672664642334, 3.055920124053955, 0.5883185267448425, -4.735847473144531, 9.919820785522461, 11.847356796264648, 9.397431373596191, -0.31028619408607483, 6.086443901062012, 0.28342723846435547, 8.304533004760742, 1.0967557430267334, 6.859018802642822, -2.513812780380249, -0.29979705810546875, -2.4075963497161865, -4.65366792678833, 0.6168898940086365, 14.696805953979492, -0.41652408242225647, -0.6129888296127319, 0.2884792983531952, -2.772721767425537, -4.707339286804199, 1.068672776222229, -2.478987455368042, -4.543585777282715, -1.085597276687622, -10.696917533874512, -9.582895278930664, 6.05845308303833, 5.565613269805908], [-5.318134307861328, -3.9454691410064697, 4.313870906829834, 0.6444984674453735, 4.494771480560303, -2.775470018386841, -10.864839553833008, -0.3569881021976471, 0.1250630021095276, -4.293254375457764, 0.5545570254325867, 5.59026575088501, 4.363687992095947, -1.586259126663208, -4.237278938293457, -12.499208450317383, -7.12470817565918, -4.435277462005615, 7.177371025085449, -4.528628349304199, -0.3856833875179291, 1.8804473876953125, 2.771197557449341, 1.9431300163269043, -14.878189086914062, -10.058332443237305, -1.9740581512451172, 6.098017692565918, 4.243284702301025, 8.044074058532715, 1.248831033706665, -7.204885482788086, 1.474118947982788, 0.5668666362762451, -0.6375651359558105, -3.842442274093628, 0.8381559252738953, 1.5976184606552124, 2.6807518005371094, 3.2287490367889404, 8.614151000976562, -3.36674427986145, 6.839380264282227, 4.980250835418701, 6.42421817779541, 4.9403252601623535, -0.3421785533428192, 1.600650429725647, 5.970564365386963, -0.917786180973053], [5.511142730712891, 1.9657528400421143, -0.7860404253005981, -3.5795743465423584, -0.9029756784439087, 6.9975457191467285, 0.3326376974582672, 1.8834748268127441, 3.8957276344299316, 6.532886981964111, 4.22463846206665, -8.55568790435791, -3.1636300086975098, -1.0068697929382324, -3.4948649406433105, 1.160313606262207, -6.804132461547852, 1.068894386291504, -4.765301704406738, -1.6691633462905884, 7.562566757202148, -4.3321332931518555, -1.1048393249511719, 9.207735061645508, 2.6260244846343994, -7.636305809020996, -3.197737216949463, 5.81318473815918, -3.6148970127105713, 12.638150215148926, 7.6599812507629395, 1.455776333808899, 0.3968105614185333, 0.38329851627349854, -2.384176731109619, -4.799731254577637, 2.6226115226745605, -1.1792112588882446, 4.5228095054626465, 4.557593822479248, -2.176255226135254, 1.2810412645339966, -2.5506584644317627, 2.167569637298584, 1.462938666343689, -7.172698497772217, -4.450160503387451, 6.452826499938965, -0.7064288854598999, -9.34756088256836], [-4.011655330657959, -3.763094425201416, 6.826965808868408, 0.045539889484643936, 5.331864356994629, 11.242490768432617, 4.531974792480469, 8.855740547180176, 3.0249075889587402, -3.201820135116577, -2.130814552307129, -2.874289035797119, -2.7611098289489746, -7.998486518859863, 4.07078742980957, -6.646401882171631, -10.275959014892578, -7.705374240875244, -3.9756972789764404, -8.510031700134277, -3.4472153186798096, -3.5408263206481934, 8.718327522277832, 0.8835426568984985, -8.274166107177734, -0.44918566942214966, 0.4747336506843567, 2.7609920501708984, 3.9211814403533936, 1.2418813705444336, -6.015344142913818, 0.6317141056060791, 0.33849942684173584, -2.771746873855591, 22.032711029052734, 4.4992194175720215, -3.667963743209839, -0.8667669296264648, -7.4975128173828125, -6.713934898376465, -4.8268046379089355, 3.36379337310791, -3.8942577838897705, 2.117260217666626, 2.8909969329833984, -0.7909936904907227, -3.4035611152648926, -8.864657402038574, 8.464197158813477, 2.849015712738037], [4.247399806976318, 2.5741991996765137, -13.377102851867676, -3.5051627159118652, 13.8912935256958, 6.6550188064575195, -3.227126121520996, -4.222043514251709, 1.610870122909546, -0.11182919889688492, 7.800259113311768, -7.818023681640625, -2.518630027770996, 5.691372871398926, -11.830777168273926, 6.030878067016602, -6.621187686920166, -5.780496597290039, -7.756078720092773, 2.9608657360076904, 15.080805778503418, 9.184890747070312, 2.652094602584839, 4.433106899261475, 1.0135327577590942, -0.5142660737037659, -0.23715290427207947, 9.157779693603516, 5.759876251220703, -8.277412414550781, -1.066925048828125, -1.2646758556365967, 11.301640510559082, 0.17124296724796295, -0.7461246252059937, -5.592195987701416, 12.345757484436035, 8.802454948425293, -1.002119779586792, 2.400953531265259, 3.562380313873291, 0.5723018646240234, 2.4203341007232666, 3.326533555984497, -0.2086915224790573, -19.31015968322754, 4.875187873840332, 2.1165308952331543, 5.175041198730469, -2.5993528366088867], [-8.921566009521484, -2.221346378326416, 3.9949803352355957, 8.921751976013184, 8.386492729187012, -0.24007129669189453, 4.213922023773193, 9.928621292114258, 4.453914642333984, -5.9779534339904785, -2.1148743629455566, 0.5730690360069275, -7.558824062347412, 2.1046323776245117, -5.254204750061035, -6.577392101287842, -2.0871193408966064, 3.2510879039764404, 4.507628917694092, -2.2337117195129395, -2.0335419178009033, 2.539968967437744, -4.424317836761475, -1.6464077234268188, -9.423489570617676, -14.497672080993652, 1.3583955764770508, 4.29947566986084, -2.71406888961792, 7.625426769256592, -6.7411932945251465, -2.674706220626831, 7.810436725616455, -4.641721725463867, 3.504272699356079, -1.185675024986267, -5.1666083335876465, -0.41746756434440613, -8.192384719848633, -7.865808010101318, 1.4360791444778442, -0.9331105947494507, 2.916069984436035, 5.410048961639404, 4.991606712341309, 5.507784366607666, 5.540956974029541, -0.3530021011829376, 0.7397364974021912, 2.9096014499664307], [-8.940932273864746, -2.170531749725342, 2.431001901626587, 6.849073886871338, 3.9006338119506836, -3.0293667316436768, 4.229309558868408, 8.516178131103516, -1.574466586112976, -8.058737754821777, -6.698477268218994, 0.345297634601593, -5.155144214630127, -13.571744918823242, 0.701861560344696, -6.994152069091797, -9.902189254760742, -3.6917800903320312, 1.4244400262832642, -2.560657262802124, -8.98354721069336, 4.133042335510254, 2.493602991104126, -0.6976284980773926, -10.739402770996094, -1.8265116214752197, 2.0394415855407715, -2.7516653537750244, 0.541692852973938, -3.240180730819702, -13.299043655395508, 2.0614078044891357, 2.5867340564727783, 5.67501163482666, 14.323736190795898, 3.78035831451416, -7.012566566467285, 4.719563007354736, -6.9801530838012695, -6.33664608001709, -3.8789284229278564, 4.6367268562316895, 5.480587959289551, 4.978937149047852, 6.12795352935791, 2.5396969318389893, 3.9142284393310547, -8.460288047790527, 1.4235255718231201, -3.43634033203125], [-15.909850120544434, -0.5872607827186584, -4.432537078857422, 2.0202243328094482, 2.946669340133667, -11.787654876708984, 9.45663070678711, -7.949148654937744, 4.5556864738464355, -7.837524890899658, 4.257485866546631, -4.656518936157227, 6.617761135101318, -2.4879202842712402, -2.868069887161255, -3.3892722129821777, -2.1133222579956055, -0.0942767858505249, 3.5212557315826416, -2.159292221069336, 7.995702266693115, 6.093034267425537, -3.480555534362793, -0.6073987483978271, 9.049355506896973, 7.843914985656738, -0.9209873676300049, -3.1526362895965576, -2.833878993988037, 1.6936992406845093, 6.912566661834717, -6.140338897705078, 0.8341684341430664, 5.53614616394043, 2.118894100189209, 1.3594179153442383, -11.558164596557617, 1.9650849103927612, 5.72512149810791, -11.835958480834961, -2.8509180545806885, 6.731292247772217, 3.6662824153900146, 7.464817523956299, 11.469907760620117, -5.012752532958984, 2.57995343208313, 0.9851551651954651, 4.29671573638916, -0.5852189064025879], [1.5579845905303955, -4.225535869598389, -2.166656255722046, -8.429678916931152, -2.733377456665039, 4.7007551193237305, -1.2974143028259277, -6.092902183532715, 3.167922019958496, -4.10452938079834, 10.536855697631836, -4.0862650871276855, 9.90681266784668, -1.2276382446289062, 10.175984382629395, 1.2491140365600586, -1.809860348701477, -5.259519577026367, -1.2890338897705078, -5.9518046379089355, 5.370853900909424, 1.2663124799728394, -1.4583804607391357, 5.099196434020996, -4.839858531951904, 5.162474155426025, -4.759609222412109, -3.050398826599121, -1.7619173526763916, -3.369004011154175, 1.1868146657943726, 5.4831438064575195, -7.751512050628662, 1.7171909809112549, -1.3855756521224976, 6.452304840087891, -5.868738174438477, 2.654479503631592, -8.801986694335938, 0.03679925948381424, 0.049483396112918854, 4.725059509277344, -6.631834506988525, -9.462562561035156, 2.6951346397399902, 2.864140510559082, -0.7252111434936523, 4.686940670013428, 8.727936744689941, -0.038983866572380066], [3.4268338680267334, 6.675106048583984, -5.658786773681641, 2.427361488342285, -0.4773571193218231, 11.777788162231445, 12.801080703735352, 7.396533966064453, 13.576322555541992, 4.908905029296875, 4.680127143859863, -10.485923767089844, 0.24391485750675201, -6.82457971572876, 1.9347046613693237, 6.808735370635986, -0.16249050199985504, 2.821734666824341, -10.547541618347168, 0.08601651340723038, -10.287907600402832, -5.297634601593018, 4.782081604003906, 12.704024314880371, -3.498521566390991, -0.27400001883506775, 0.19440197944641113, -0.3913777470588684, -2.0088508129119873, -0.009941005147993565, -5.665214538574219, 6.171350955963135, -1.2490559816360474, -3.390442132949829, 3.1197187900543213, -4.170474529266357, -5.6875901222229, -3.321934938430786, -3.273540496826172, 5.769810676574707, -3.4579997062683105, -2.4058353900909424, -9.649746894836426, -5.987072944641113, 1.2917797565460205, 0.5797351002693176, 1.6079487800598145, 2.4451324939727783, 1.6592199802398682, 13.531460762023926], [-4.444633483886719, 5.982789516448975, -5.151710510253906, -0.1710875928401947, 15.258231163024902, -0.9768348336219788, -3.6849279403686523, 7.667444705963135, 3.0116074085235596, 4.482259273529053, -2.710914373397827, 0.7069764137268066, 8.500514030456543, 6.57940149307251, -7.637604236602783, -9.263684272766113, -9.142203330993652, -12.647921562194824, 8.018539428710938, 6.247918605804443, -8.53365421295166, 6.543740272521973, 5.733796119689941, -7.988497734069824, -4.587406158447266, 1.2183889150619507, 3.872734785079956, 3.8145642280578613, 0.859456479549408, -7.059873580932617, -12.854418754577637, -0.7512324452400208, -3.2919139862060547, 5.507606506347656, 9.10594654083252, 4.924710273742676, -0.5646438598632812, 5.796230792999268, -5.759064197540283, 2.8219735622406006, 1.5480802059173584, -8.726592063903809, -0.5214298367500305, 1.171334147453308, 1.603872299194336, 1.7102524042129517, -3.924266815185547, 7.052799224853516, -2.7332019805908203, 6.76054048538208], [12.883990287780762, -1.927893042564392, -11.419878005981445, 7.024315357208252, -1.4849642515182495, 11.631367683410645, 12.550997734069824, -5.798480987548828, -3.5140955448150635, 1.8127353191375732, 2.554521083831787, -8.335662841796875, -0.7290894389152527, -6.454005241394043, 4.088860034942627, 0.45293259620666504, 2.5233922004699707, -2.860517978668213, -7.902331829071045, -0.41456684470176697, 3.7383177280426025, 0.6559332609176636, -1.792070984840393, 12.420551300048828, 4.088077068328857, 4.833265781402588, -1.9241360425949097, -4.176107883453369, -3.4967281818389893, -1.5226179361343384, -2.688711166381836, 6.7378249168396, -4.5370612144470215, -4.180785179138184, 6.293123722076416, -1.4252372980117798, -2.6814754009246826, -4.611494541168213, -6.326733112335205, -2.3300766944885254, -6.232123374938965, 5.829848289489746, -10.560087203979492, 2.328420400619507, 5.183136463165283, -10.92303466796875, -3.411313772201538, 8.952923774719238, -0.26978814601898193, -1.648446798324585], [-14.066665649414062, 0.5762979388237, -11.867863655090332, 4.361660480499268, -4.051148891448975, 3.3384735584259033, -9.251824378967285, -5.289632797241211, 5.129152774810791, 2.112373113632202, 8.378829002380371, -12.748841285705566, 16.396026611328125, 0.4818325638771057, 0.2628139853477478, -1.7681024074554443, 0.4501013159751892, 10.069575309753418, 1.2703694105148315, 2.614614486694336, -2.5240721702575684, -1.3325176239013672, -4.7898268699646, 11.726125717163086, -11.395428657531738, -6.665358543395996, -8.92811393737793, 4.931344509124756, -3.7610957622528076, 21.00935173034668, 3.846865177154541, 4.401314735412598, -4.095486640930176, 7.375114917755127, -9.916364669799805, 1.8413366079330444, -9.370052337646484, 6.529569625854492, 3.450418472290039, -1.3685353994369507, 4.668326377868652, 7.9393134117126465, 4.848515033721924, -14.585824012756348, 4.6630401611328125, 6.250609874725342, 8.158148765563965, 9.40768051147461, -5.7654852867126465, -0.8413798809051514], [-0.791800320148468, 5.527041435241699, 0.46911856532096863, -2.172982931137085, -2.1929616928100586, -7.770853042602539, 6.295472621917725, 2.8524539470672607, 6.730791091918945, 5.0459136962890625, -4.018162727355957, -1.0972187519073486, 5.4964704513549805, 1.4487295150756836, -6.115744113922119, -5.198085784912109, -3.236837863922119, 3.6756255626678467, 1.614651083946228, 3.8560791015625, -11.933349609375, -4.0608978271484375, -5.113424301147461, -8.306787490844727, -3.19671368598938, 2.7539443969726562, -4.272031784057617, 2.0242371559143066, -8.385175704956055, 4.928938865661621, -1.2484349012374878, 5.31405782699585, 0.631809413433075, 0.7690402865409851, -5.625345706939697, 3.658771276473999, -3.723616600036621, -6.102327346801758, 2.7949557304382324, -2.8806204795837402, 4.691932201385498, -1.4846066236495972, -2.1221389770507812, -2.6091134548187256, -2.175943374633789, 11.127193450927734, 6.855587959289551, 7.496734619140625, -2.4138877391815186, 1.573498249053955], [-4.529379367828369, -0.12956856191158295, -5.396348476409912, -0.855425238609314, -2.581177234649658, 7.936522483825684, -12.687950134277344, -4.971659183502197, -2.6945440769195557, 5.583312034606934, 4.239325046539307, -3.439424753189087, 12.971451759338379, 8.57438850402832, -0.3051259517669678, 2.586606740951538, -1.4910359382629395, -7.245790004730225, 2.118957281112671, 4.013510704040527, 3.142058849334717, 3.874732494354248, -1.2750358581542969, 2.236773729324341, 3.5172364711761475, 2.6931796073913574, -0.9077752828598022, 2.7510969638824463, -3.6045007705688477, -2.071877956390381, 3.547778367996216, -2.9757697582244873, -4.426617622375488, 1.7508541345596313, -8.49837589263916, -3.8012752532958984, 4.692138195037842, -2.103111505508423, 1.0038738250732422, 3.5501441955566406, 6.502270221710205, -4.22135591506958, -4.276326656341553, -4.365811347961426, 0.3019137382507324, -3.727113962173462, -5.930357933044434, 5.4488749504089355, -2.720881938934326, -1.8566416501998901], [12.283859252929688, -1.9947881698608398, -8.579008102416992, 5.878957748413086, 10.58907699584961, 3.964677333831787, 1.2577779293060303, -3.1473283767700195, -1.213565707206726, 8.879985809326172, 1.2586724758148193, -5.19145393371582, 0.03969838470220566, 4.067786693572998, -0.9281026124954224, 7.032365798950195, -10.088871002197266, -3.315708875656128, -0.37687504291534424, 6.2824602127075195, 6.93218469619751, 1.7118083238601685, 8.50301456451416, 7.569383144378662, 0.8026006817817688, -0.47736024856567383, -9.278645515441895, 4.392916679382324, 10.573122024536133, -1.804635763168335, -3.4155609607696533, 2.3336169719696045, 5.1693010330200195, 7.262579917907715, -4.307020664215088, 9.358807563781738, 8.821450233459473, -0.3051556348800659, -1.1738990545272827, -0.161067396402359, -1.4189014434814453, -0.9872390031814575, 0.43954986333847046, -6.168848514556885, -5.493556022644043, -9.250930786132812, -0.6499433517456055, -4.393924713134766, 14.968050003051758, -5.353387832641602], [-1.7336493730545044, -2.7481157779693604, 7.0488152503967285, -1.618397831916809, 1.4311004877090454, 0.5098119378089905, 16.972780227661133, 2.4713404178619385, 4.597461223602295, 7.102327823638916, -2.172407627105713, 3.425600290298462, -2.3622002601623535, 9.616998672485352, -4.4446330070495605, -9.824207305908203, 10.627840995788574, 7.21986198425293, 7.79764986038208, -1.2849024534225464, -6.170779228210449, -10.86777114868164, 1.1020915508270264, -1.0739710330963135, 3.061922550201416, 2.4767887592315674, -8.911835670471191, -0.4128491282463074, -2.100079298019409, 2.8594396114349365, 0.8069786429405212, 0.24351848661899567, 3.655402183532715, 1.0655509233474731, 7.43086576461792, 3.317739486694336, -6.0424089431762695, -9.354445457458496, 3.29707407951355, 1.6678458452224731, -1.4807931184768677, -4.996588230133057, 3.527442216873169, -1.4902492761611938, -2.7697627544403076, 0.5003728270530701, 10.363914489746094, 6.927758693695068, 14.113200187683105, -5.004886627197266], [3.641265392303467, 2.5954830646514893, -8.282018661499023, 0.7590442299842834, 2.800121545791626, 6.852852821350098, -0.044955648481845856, 1.8167635202407837, 10.179816246032715, 6.266049861907959, 3.8582777976989746, -0.8294203877449036, -5.018420219421387, -5.825364112854004, -1.4114139080047607, 7.169740676879883, -11.35218334197998, 8.710206031799316, -9.118019104003906, 4.410049915313721, -2.993759870529175, -1.4985847473144531, -2.6464426517486572, 6.075387001037598, 4.835216045379639, 1.2724599838256836, 5.317185878753662, 4.7713847160339355, 5.217724323272705, -1.4215905666351318, -5.528629779815674, 1.9235398769378662, 12.447035789489746, -4.777517318725586, 0.5254608988761902, -3.9808566570281982, 3.3794333934783936, 0.025446657091379166, 0.5464035272598267, 2.9696524143218994, -0.7173861861228943, -5.142251014709473, 3.8372743129730225, -1.4184377193450928, -2.0335185527801514, -5.867354869842529, 0.7563440203666687, -13.292303085327148, 6.877793788909912, -3.144733190536499], [-1.8616673946380615, -3.0000052452087402, -3.4392406940460205, -5.452400207519531, 5.541123867034912, 11.197046279907227, -1.744138240814209, 4.632070064544678, 2.4270503520965576, 4.959301471710205, 0.6224707365036011, -5.466538906097412, -4.682164192199707, -0.3983951807022095, 2.1478025913238525, -6.265211582183838, -4.834742546081543, -3.5646913051605225, -8.978311538696289, -5.227049827575684, -2.7363548278808594, -6.086396217346191, -0.006556609645485878, 4.460738658905029, -14.402661323547363, -2.707258701324463, 3.03644061088562, 1.9131090641021729, 0.8807733058929443, -1.214429497718811, -7.948602676391602, 10.731094360351562, -1.6138532161712646, 3.1756505966186523, 18.02535057067871, -2.4481706619262695, -2.861187219619751, -5.017859935760498, 4.134732246398926, 15.540127754211426, -6.008035659790039, -2.4098639488220215, -7.14903450012207, -8.578370094299316, 1.2145882844924927, -1.0996116399765015, 6.206450939178467, 7.465726375579834, -4.470914363861084, 3.218090295791626], [-1.419999599456787, 11.96137523651123, -1.1880618333816528, -2.2440621852874756, 1.6227431297302246, 4.151102066040039, 11.153100967407227, 2.053523302078247, 9.735947608947754, 2.80851149559021, -0.6578671336174011, -6.154848575592041, 2.018028736114502, -5.784066200256348, 2.9338181018829346, 3.3146238327026367, 11.737234115600586, 2.894099712371826, -5.113376617431641, 5.1914801597595215, 0.5408981442451477, -8.215075492858887, 0.008355002850294113, -3.6792044639587402, -11.419717788696289, 2.5587589740753174, -4.713357448577881, 6.493539333343506, 7.123818874359131, 5.789633750915527, -4.110968589782715, 7.847397327423096, 0.11250226944684982, 0.2904890477657318, 2.534515857696533, 5.1125874519348145, -11.061124801635742, -0.433878630399704, -3.83681321144104, 3.369391918182373, -0.9854277968406677, -6.678452968597412, -15.602636337280273, -6.123408794403076, 6.316428184509277, -0.050390854477882385, 7.311790943145752, 4.315057277679443, -3.1846113204956055, 11.107952117919922], [0.7177532911300659, 4.500468730926514, -5.405251979827881, 3.4995813369750977, 6.383138179779053, 3.0243234634399414, -4.308793544769287, -1.3234665393829346, -0.6615198254585266, 6.478017807006836, 1.4245166778564453, -7.573880195617676, -1.79453444480896, -1.1002060174942017, -6.530298709869385, -4.583789348602295, 0.4958864152431488, 6.699830055236816, -5.51167106628418, 10.775510787963867, 13.909311294555664, -0.5965126156806946, 3.981311082839966, 15.31564712524414, -2.271318197250366, -7.650788307189941, 5.823651313781738, 10.085577011108398, -1.9293115139007568, 7.341663360595703, 8.807671546936035, 0.4082731604576111, -0.22529065608978271, -4.112950801849365, 1.2889131307601929, 1.9330639839172363, 5.113522052764893, 11.40367603302002, 2.4007534980773926, 6.604481220245361, -4.387928485870361, 4.192009449005127, 3.381455659866333, 5.078459739685059, 0.6944190859794617, -3.41074538230896, 5.797626495361328, 7.0667595863342285, -9.138879776000977, 3.0809552669525146], [8.175801277160645, -6.088130474090576, 10.78611946105957, 2.542720317840576, -7.849118232727051, 2.9839625358581543, 0.10710689425468445, 6.502018451690674, -2.071849822998047, 13.4287748336792, -7.2770185470581055, 9.34598159790039, 2.5382120609283447, -6.9755706787109375, 9.90926742553711, -0.7597991228103638, 7.597123146057129, 6.508472442626953, 2.4368720054626465, 5.43453311920166, 1.1695997714996338, 7.133337497711182, 9.161824226379395, 12.23498821258545, 6.567338466644287, -1.7355939149856567, 4.816192150115967, 1.6322870254516602, 2.851456880569458, -4.928305149078369, -5.276710510253906, 2.8177475929260254, 7.8845672607421875, -16.814027786254883, -9.519547462463379, 0.9712685346603394, 8.340890884399414, 3.423060417175293, -11.369582176208496, -13.768243789672852, 0.5825262069702148, -4.108974933624268, 3.040074110031128, 0.021477751433849335, 2.63610577583313, 11.334478378295898, -12.755555152893066, -15.420822143554688, 2.2711641788482666, 7.609182834625244], [10.693429946899414, -1.253301739692688, -9.783769607543945, -1.3776870965957642, -9.462523460388184, 12.220914840698242, 9.143001556396484, 2.278362512588501, 2.510274887084961, 5.606658935546875, 3.6251862049102783, -2.37890625, 3.2292275428771973, -9.183658599853516, 3.535219192504883, 5.316372394561768, 1.0171453952789307, 6.045166015625, -1.7372028827667236, -5.1645355224609375, -7.202908515930176, 4.778091907501221, 5.511541366577148, 7.189025402069092, -1.0840281248092651, 5.136326789855957, 0.3015616834163666, -6.381925106048584, -6.974056243896484, -6.17916202545166, -10.911523818969727, 10.60452651977539, 9.228450775146484, -7.169863700866699, 0.38138192892074585, -7.597085475921631, -1.1161000728607178, 4.08538293838501, 0.17924010753631592, -4.824906349182129, -1.6279919147491455, 2.8943519592285156, 3.5965495109558105, -1.5625959634780884, -5.860020160675049, -2.9601829051971436, 4.818133354187012, 6.88327169418335, -0.7575717568397522, -6.521759033203125], [8.307073593139648, 1.2156307697296143, -1.6883169412612915, 3.971317768096924, 2.5301661491394043, -6.564779281616211, -5.287283420562744, -1.338228702545166, -4.693310260772705, 0.3288629353046417, -1.4095525741577148, 1.3055179119110107, 1.4403202533721924, -3.085474967956543, -1.8905203342437744, -2.100895404815674, -4.316071510314941, 1.1618374586105347, -1.3809096813201904, 5.150452613830566, -5.960487365722656, 1.5042426586151123, -0.7236444354057312, 2.3508734703063965, -4.668361663818359, -5.6441650390625, 2.538987874984741, 3.581270456314087, -0.48366135358810425, 11.725384712219238, 0.8498093485832214, 0.91969895362854, -6.240122318267822, -5.447652816772461, -11.312312126159668, -4.518519401550293, 6.150829315185547, -2.1529242992401123, 4.164863586425781, 0.6593236327171326, 6.627388954162598, 1.1777124404907227, -3.0304315090179443, 2.63848876953125, 0.7172259092330933, 8.208868980407715, -6.749894142150879, 5.270866394042969, -7.544900417327881, 12.275339126586914], [3.234677791595459, 4.335944175720215, -1.8423523902893066, -2.6581718921661377, -4.563503265380859, -13.398497581481934, 4.499215602874756, -6.550384521484375, -0.527612030506134, 0.056335851550102234, -2.0399515628814697, 8.271965980529785, 11.24013900756836, 8.560323715209961, -9.695206642150879, -2.57983136177063, 15.086394309997559, 4.6897149085998535, 5.1756391525268555, 2.010157823562622, -3.2984390258789062, -2.016763210296631, -7.608974933624268, -6.901877403259277, 0.500146746635437, -0.9308081865310669, -10.099959373474121, 0.933661937713623, 1.050957441329956, -5.167642593383789, 7.19712495803833, -0.3789638876914978, -5.765573501586914, 1.18807852268219, -16.03704833984375, 4.457052230834961, -4.980732440948486, -1.4983294010162354, 3.34209942817688, -1.8467532396316528, 6.880316734313965, -0.27665266394615173, 2.2332847118377686, -4.279304504394531, 4.793895721435547, 11.593687057495117, 6.490087985992432, 7.567368507385254, 3.9120476245880127, 15.1382474899292], [1.5736039876937866, -5.820786476135254, -17.872163772583008, -4.902061462402344, 6.497232437133789, 0.7905784845352173, 0.1271039843559265, 1.2643928527832031, 4.244414806365967, 6.564251899719238, 1.7764440774917603, -8.788580894470215, -5.812572479248047, 15.167868614196777, -4.37870979309082, 3.631047487258911, -2.731771230697632, 1.7096410989761353, 0.07909221202135086, -2.757322311401367, 11.790936470031738, 0.5890374183654785, -3.750662326812744, -8.586968421936035, -0.3710732161998749, -3.810011148452759, -2.1075706481933594, -1.6400507688522339, 6.428933143615723, -4.972403049468994, 6.074885845184326, 2.053893804550171, 2.264092206954956, 12.657242774963379, -1.5036460161209106, 3.2304792404174805, -6.475281715393066, 0.7052111029624939, 7.001397609710693, 6.235411167144775, -9.74303150177002, -0.3649331331253052, 2.7687039375305176, -1.015868067741394, 0.7053123712539673, -12.710700988769531, 0.13139164447784424, 3.8814704418182373, -1.139397144317627, -0.841296374797821], [-1.1432181596755981, -2.4113922119140625, 6.326379776000977, 1.8864378929138184, -5.004815578460693, -5.201342582702637, -1.8506871461868286, 1.1914485692977905, -5.316175937652588, -6.11738920211792, 0.7008442878723145, 1.6324292421340942, 3.0268824100494385, -6.475406169891357, 6.911616802215576, -0.9159307479858398, 5.851959705352783, -6.635762691497803, -4.890009880065918, -8.16528606414795, 9.0701265335083, 3.064424514770508, -2.2507197856903076, -3.3683369159698486, -5.740245342254639, -10.626363754272461, -4.553867816925049, -2.2489495277404785, -5.627164363861084, -0.247538760304451, 4.658352851867676, -2.5564520359039307, -1.8933300971984863, 0.1854352355003357, -2.1138622760772705, -4.892760753631592, -4.570774078369141, -8.226536750793457, 6.870131492614746, 3.9621291160583496, 7.23783016204834, 6.606996536254883, -7.164674282073975, -1.8710262775421143, 6.0099873542785645, 0.31006258726119995, -4.955512046813965, 0.318590372800827, -7.186646938323975, -0.870364785194397], [4.9872236251831055, -5.449955463409424, -2.641836643218994, -7.253640174865723, -1.6137864589691162, 7.44240140914917, 12.0269775390625, -0.16300134360790253, -0.4856436848640442, 0.3886567950248718, 3.8465652465820312, -8.28463363647461, -11.669708251953125, -2.3517630100250244, 4.540367603302002, 6.088781356811523, 9.827463150024414, 7.889795780181885, -3.88508939743042, -1.6983106136322021, 6.736783981323242, 5.5617523193359375, 0.9110983610153198, 4.155306339263916, -5.9028849601745605, 1.6750508546829224, -6.104198455810547, -4.1015305519104, 1.3761125802993774, 0.7288345098495483, -7.150437831878662, 12.99583911895752, 11.400318145751953, 1.2079193592071533, 6.7167582511901855, -1.1629364490509033, -0.9285928010940552, 5.25243616104126, -6.4870758056640625, -0.8834754228591919, -4.07639741897583, 4.145691871643066, 2.8361716270446777, -7.1517333984375, -7.417852401733398, -10.50800895690918, 13.371829986572266, 8.163103103637695, -0.2638755738735199, -13.610800743103027], [-4.639188289642334, 6.560766220092773, 2.277331590652466, 1.2512315511703491, -8.639138221740723, -0.5113391876220703, -11.031044960021973, -6.630662441253662, -3.065135955810547, -2.084520101547241, 2.0642850399017334, -2.6617534160614014, 11.485798835754395, -6.8528151512146, 4.0043416023254395, 4.826296806335449, 3.1015119552612305, 1.3209081888198853, -3.504338264465332, 3.1820743083953857, 4.056102275848389, 0.679524302482605, 6.986190319061279, 2.7518250942230225, 0.8436275720596313, 6.600930213928223, 2.5009729862213135, 2.3328380584716797, 5.245429515838623, 1.4247634410858154, 5.4246296882629395, -2.998232841491699, -4.090102672576904, -3.772881031036377, -1.7063660621643066, 1.389048457145691, 5.988716125488281, 10.709452629089355, 0.4134746789932251, -1.8034664392471313, 3.630427360534668, 2.875934362411499, 0.46874648332595825, -1.3706154823303223, -2.6468889713287354, 0.9786263108253479, -1.82244873046875, -3.9636764526367188, -8.434239387512207, 2.9844517707824707], [6.5074462890625, -10.259149551391602, -4.031999111175537, -1.6832937002182007, -2.195737838745117, -0.4806387424468994, -2.6654579639434814, -3.038700819015503, 1.1966445446014404, 6.466146945953369, 0.22080101072788239, 1.1665923595428467, -0.8699928522109985, 2.7384021282196045, -1.3867895603179932, 4.1246724128723145, -4.471907615661621, 9.01340103149414, 3.4182844161987305, -8.936108589172363, -3.6622097492218018, -0.5888000726699829, -2.4805946350097656, -8.87332534790039, -2.8700110912323, -0.6916701793670654, -1.8283196687698364, 1.8847826719284058, 8.535861015319824, -16.654151916503906, -4.001286029815674, 1.935417652130127, 3.1708052158355713, -3.0228371620178223, 0.20190460979938507, -3.9139773845672607, 4.474120616912842, 2.60382080078125, -2.974480152130127, -1.5158123970031738, 1.11014986038208, -6.772959232330322, 13.187478065490723, 1.3802719116210938, -4.452184677124023, 3.8210415840148926, 5.488129138946533, -3.48992657661438, 7.515865802764893, -6.099030494689941], [8.713306427001953, 2.6366586685180664, -9.416827201843262, 12.331001281738281, 2.2091994285583496, -8.964512825012207, 4.935050010681152, 2.055636405944824, 2.3826560974121094, 0.7860482335090637, 2.3030846118927, 3.020522117614746, 7.43391227722168, -2.766376256942749, 3.9057071208953857, 7.0060014724731445, -10.91458511352539, 6.90289306640625, 6.838817596435547, 5.989927291870117, -9.941319465637207, -4.159729480743408, -4.2624592781066895, -7.901145935058594, 1.8426287174224854, 0.7564578056335449, -1.7121964693069458, 4.512300968170166, 1.12312912940979, -11.259427070617676, -9.662243843078613, -2.004546642303467, -4.776124000549316, -3.8331260681152344, -14.619726181030273, 3.8001205921173096, -3.096184253692627, -0.04571724310517311, -3.6133573055267334, -4.916691303253174, 5.088993549346924, -1.9899321794509888, 0.4565947651863098, 2.653280735015869, -3.3638906478881836, 7.9266886711120605, -0.4848031997680664, -3.108318567276001, 6.404208183288574, 3.4609222412109375], [-10.231484413146973, -2.841709613800049, -7.566923141479492, 0.42247286438941956, 0.7798576951026917, -1.6061102151870728, -3.4724926948547363, -1.4343825578689575, 8.216567993164062, -2.1158738136291504, 9.630592346191406, -7.323880195617676, 6.149771213531494, 5.487400054931641, -6.25552225112915, 0.7598732709884644, -11.85509967803955, 5.043175220489502, 4.6283345222473145, -4.061814785003662, -6.137273788452148, -4.018924236297607, -6.429747581481934, 9.490527153015137, -2.686843156814575, -2.92679500579834, -3.538902759552002, 0.8335922360420227, 3.4780964851379395, 5.014480113983154, -0.881446897983551, 1.0019993782043457, -7.181986331939697, 10.344056129455566, 0.6668845415115356, -4.355769157409668, -6.921759605407715, 6.860791206359863, 0.16736871004104614, 1.3984037637710571, 1.2377698421478271, 2.240969181060791, 9.593015670776367, -5.188197612762451, 1.9559122323989868, -1.1843236684799194, 7.025191783905029, 14.14880084991455, 4.205492973327637, -7.815062999725342], [5.381433010101318, -1.2623564004898071, 1.0541960000991821, 1.1412734985351562, -7.4146037101745605, -2.9571421146392822, 0.38285183906555176, -2.3478012084960938, -0.07129319757223129, 2.247361421585083, -4.826244831085205, 4.080798149108887, -8.173115730285645, -7.676884174346924, 4.805675506591797, 7.1708598136901855, -7.8649091720581055, 2.7511239051818848, -4.529841423034668, -1.7971913814544678, -5.021643161773682, -7.1613335609436035, -6.29081916809082, -2.3950679302215576, 4.1145219802856445, 3.464404821395874, 4.393545150756836, -0.6925584077835083, 7.9548020362854, -9.415247917175293, -1.387484073638916, 1.3826158046722412, -3.4709243774414062, -1.3555655479431152, -0.5648497939109802, -1.4628934860229492, 0.12383826076984406, -6.8927321434021, 6.822899341583252, 1.0650262832641602, -10.336977005004883, -7.816004276275635, -3.21242094039917, -1.1827168464660645, 0.3472837209701538, 6.400142192840576, -2.556411027908325, -11.566658020019531, -0.8028510212898254, 8.02178955078125], [1.3161009550094604, -8.503939628601074, -15.715400695800781, 6.16741943359375, 9.970029830932617, -1.4082911014556885, -6.497744560241699, -10.07927417755127, -0.053657758980989456, -1.6857143640518188, 4.63014554977417, -1.8014582395553589, 4.856356620788574, 6.736907482147217, -1.9436243772506714, 15.897674560546875, -5.820927619934082, 0.3931594491004944, -0.4803500473499298, -3.876462697982788, -6.440596580505371, -1.9047456979751587, -3.727696418762207, -19.82933807373047, -6.84506893157959, -0.8479316234588623, -7.195948123931885, -0.21594145894050598, 10.044347763061523, -7.168522357940674, 3.1317873001098633, -4.3704752922058105, -4.2518792152404785, 10.629740715026855, -3.078080415725708, -3.5522124767303467, -5.286981105804443, -10.414161682128906, 11.409745216369629, 1.8702934980392456, 12.174086570739746, -2.1480796337127686, 3.2895209789276123, -7.252318859100342, -7.2670979499816895, -5.94404935836792, 5.723448753356934, -2.06479811668396, 9.927927017211914, 0.6155765056610107], [1.6990517377853394, 7.993746280670166, 4.308217525482178, -9.01064682006836, -8.201292037963867, 14.168460845947266, 2.46551513671875, 6.835062026977539, 5.265652179718018, -0.24808810651302338, -3.3164360523223877, -3.0450117588043213, -1.5799065828323364, 4.424188137054443, -8.352622032165527, 0.989409863948822, 1.4126547574996948, -11.301045417785645, 2.0547497272491455, -2.128504991531372, -9.095752716064453, -8.365751266479492, 5.572552680969238, 7.092196941375732, -3.2442781925201416, 3.210662364959717, -11.505534172058105, 2.3148813247680664, 0.5924518704414368, 6.922226905822754, 0.06523635238409042, 0.8305908441543579, 0.8207217454910278, 8.09835433959961, -9.232758522033691, -1.4922682046890259, 2.7669124603271484, -0.47776278853416443, -2.8590149879455566, -2.9694736003875732, -2.30891489982605, -3.7163426876068115, -7.143592834472656, -3.985185384750366, 2.530341148376465, -4.093924522399902, -0.5102778077125549, 3.2065317630767822, 10.51445198059082, -1.9792499542236328], [-12.765496253967285, 6.455822944641113, -1.5221531391143799, -4.968423366546631, 4.552242279052734, 1.752358078956604, 2.4660067558288574, 2.054158926010132, 12.299821853637695, 1.4703928232192993, 10.975502967834473, -11.866087913513184, -1.1779437065124512, 0.09884548932313919, -3.8449478149414062, -1.7121862173080444, -4.718781471252441, 8.504451751708984, -7.573092460632324, 6.044620037078857, 2.785649061203003, -4.308071136474609, 1.085239291191101, 6.168722629547119, 8.31895637512207, 2.5342252254486084, 4.502288818359375, 2.740232229232788, -1.4422097206115723, 15.293645858764648, 11.240681648254395, -3.500528573989868, -3.288691282272339, 2.5577199459075928, 8.544258117675781, 1.5120679140090942, -6.011586666107178, 4.791816711425781, 6.930287837982178, 10.888945579528809, -6.847204208374023, -2.726167678833008, 1.093074083328247, -0.22112879157066345, 0.6458568572998047, -6.406002998352051, -0.8725126385688782, 2.008401393890381, 5.653820514678955, 1.912681221961975], [-6.674399375915527, 5.083491802215576, -0.6137648820877075, 5.9465436935424805, -0.8076704144477844, 0.8728383183479309, 1.2420779466629028, -7.694472789764404, 3.657057762145996, -6.596136093139648, 3.6893417835235596, 0.2412172108888626, 11.837380409240723, -10.226634979248047, 7.214227676391602, -4.210207462310791, -3.209743022918701, 0.0015432718209922314, -0.8063946962356567, 2.367056131362915, -11.16236686706543, -6.491700172424316, -5.045114040374756, -12.04253101348877, -2.584135055541992, 8.858400344848633, 3.3642232418060303, 3.3622281551361084, -3.7507810592651367, 9.431495666503906, 3.215756416320801, -7.302795886993408, -10.393131256103516, -10.036499977111816, 7.3867926597595215, 0.3222234547138214, -10.74608039855957, -9.629436492919922, 2.2181451320648193, -2.7867684364318848, 6.845575332641602, -3.362149238586426, -10.90721321105957, 3.4641599655151367, 1.1626770496368408, 7.674863338470459, -3.2334821224212646, -1.4790889024734497, -3.2710492610931396, 11.931337356567383], [4.333641529083252, -3.135607957839966, 3.5912320613861084, -4.035399436950684, 4.438969135284424, 21.044902801513672, -15.41476821899414, 3.254768133163452, -10.308979034423828, 15.165264129638672, -4.889789581298828, 0.0605170913040638, -0.7099753022193909, -4.2138991355896, 5.898068904876709, -9.130613327026367, 3.811002731323242, -6.419939994812012, -10.879781723022461, 7.153942584991455, 0.7411705851554871, 4.574840545654297, 10.996025085449219, 4.929665565490723, -0.23881785571575165, 4.1322526931762695, 11.913728713989258, 2.8942689895629883, 2.5847525596618652, 8.474645614624023, -3.596127986907959, 4.619028568267822, 1.888718605041504, -9.378678321838379, 13.630915641784668, -4.923861980438232, 15.152806282043457, -3.7846267223358154, -4.490556240081787, 14.158323287963867, 3.2650907039642334, -9.830167770385742, -8.738037109375, 3.300668954849243, 3.967771053314209, -10.175686836242676, -20.833925247192383, -10.086400985717773, -2.90097975730896, -1.0899944305419922], [-1.827530860900879, -1.6539112329483032, 9.684802055358887, -3.257453203201294, -5.356090545654297, 0.12435682862997055, -3.2493679523468018, 12.957036018371582, 0.09468727558851242, -2.4186837673187256, -2.6156489849090576, 1.7682448625564575, 7.45171594619751, -5.30113410949707, 1.9327558279037476, -1.585738182067871, -1.2199331521987915, -7.384810447692871, -4.2665324211120605, -6.534758567810059, -2.774298906326294, 3.3040363788604736, 7.989781856536865, 2.3249645233154297, -8.54201602935791, -3.8124825954437256, -1.1129870414733887, 0.08720774203538895, -0.008521683514118195, -11.117271423339844, -4.876689434051514, 3.1082732677459717, -2.4414193630218506, -2.189342975616455, -2.722548723220825, -1.2056958675384521, 1.8713587522506714, 4.0544867515563965, -10.247123718261719, -2.959901809692383, 8.586373329162598, 5.094531536102295, -3.0583693981170654, 0.29471492767333984, 7.276549339294434, 5.882258415222168, -6.588963508605957, -3.5550570487976074, 6.416419982910156, 3.941162109375], [-14.324264526367188, 6.530851364135742, -5.047629356384277, -2.582458019256592, 0.29567185044288635, 0.09275103360414505, -5.1340651512146, -4.647597789764404, 11.121050834655762, -11.189447402954102, 5.648777008056641, -8.514251708984375, 0.059090375900268555, 5.060251235961914, -2.147036552429199, 14.894058227539062, -3.706425666809082, -5.186978340148926, 1.0478893518447876, -3.2990782260894775, -0.7756751775741577, 1.1750893592834473, -0.1071292981505394, 1.8114356994628906, -2.280056953430176, 4.364235877990723, -0.38199958205223083, 1.2579853534698486, 10.156094551086426, 2.214268684387207, -0.3040195107460022, -5.0065693855285645, -0.5073358416557312, 8.310968399047852, -6.07553243637085, -7.098123550415039, -1.9460957050323486, 5.637416362762451, 2.690711736679077, -5.525752067565918, -3.0989692211151123, -3.4337329864501953, -3.2270328998565674, -7.17595100402832, 0.06727217882871628, -1.6419018507003784, 5.610742092132568, 4.0595316886901855, -4.969950199127197, 7.679847717285156], [-9.96964168548584, 1.7378367185592651, 11.106584548950195, 1.1611673831939697, 3.296772003173828, -4.549530982971191, -4.096221446990967, 6.707250118255615, -0.3653393089771271, -12.525829315185547, -5.4896674156188965, 5.247346878051758, -9.304023742675781, -0.6789161562919617, 0.9835476279258728, -4.962535381317139, -1.3049499988555908, -11.237862586975098, 7.4467902183532715, -1.1187701225280762, -2.323312759399414, 7.775488376617432, 6.863640308380127, 2.016486883163452, -5.3076019287109375, -1.1369441747665405, 6.414928436279297, -3.2551181316375732, -1.3746836185455322, 2.233086347579956, -7.562441825866699, -7.862185001373291, 7.147347450256348, 1.2318549156188965, -0.9917777180671692, -5.05824089050293, 5.27689266204834, 2.54673433303833, -4.45161771774292, -2.662163734436035, -0.11932399868965149, -2.644228219985962, 0.10014001280069351, 7.19244384765625, 1.7520339488983154, 2.6663625240325928, -1.7962727546691895, -4.014367580413818, -4.442472457885742, -1.2857991456985474], [2.763409376144409, 5.55690336227417, 14.704792976379395, -0.6396557688713074, -5.208177089691162, 1.3818706274032593, -5.442145347595215, -0.9374381303787231, -1.0830270051956177, 13.577861785888672, -7.711569309234619, 5.472364902496338, 7.80903959274292, 3.5172815322875977, -6.189433574676514, -7.6386237144470215, -2.016526699066162, 2.9739468097686768, -1.288492202758789, 0.40146374702453613, -15.204350471496582, -12.333480834960938, -3.9149134159088135, -6.910109519958496, 0.4702640175819397, 0.99894779920578, 3.3549647331237793, 2.6951117515563965, -5.177736759185791, 11.002471923828125, 2.6317875385284424, 5.045849800109863, -4.771658420562744, -8.670554161071777, 6.96163272857666, 3.891584634780884, 8.57215690612793, -11.88485050201416, 1.028962254524231, 3.7884328365325928, 6.285707950592041, -9.201560974121094, -4.766537189483643, -2.3812501430511475, -6.081347465515137, 16.22867202758789, -1.117168664932251, -1.1954030990600586, -8.572403907775879, 8.71028995513916], [-6.409765243530273, -3.73681378364563, 0.9395950436592102, -0.5619625449180603, 1.044366478919983, -8.13973331451416, -3.286778688430786, 2.4486358165740967, 2.0395052433013916, -0.7196912169456482, -9.228702545166016, 0.6579455733299255, -4.943772315979004, -1.491966962814331, 2.3703761100769043, -3.7454030513763428, -12.110221862792969, -7.271995544433594, -0.6873661875724792, -2.925554037094116, -1.80076265335083, 1.964375376701355, -3.158342123031616, -1.2439253330230713, -6.589209079742432, -1.597784399986267, 5.485133647918701, 2.535431385040283, 3.050900459289551, 5.046630859375, -1.4143811464309692, -1.597869873046875, 2.677186965942383, 0.6379873752593994, -4.645677089691162, -11.735689163208008, 0.7617964148521423, -14.961797714233398, 10.780091285705566, -0.6897469758987427, 1.5338810682296753, -1.7143466472625732, -8.79709243774414, 6.374689102172852, 8.398406028747559, 4.460268020629883, -7.828246116638184, 0.3615603446960449, -12.590618133544922, 6.897653579711914], [0.20783443748950958, 2.6235289573669434, 2.862515449523926, 5.264477252960205, -0.017197297886013985, -11.56498908996582, 11.798851013183594, -1.697144865989685, 4.718299865722656, 4.102814197540283, -5.429706573486328, -3.5030641555786133, -4.644590854644775, -9.865458488464355, 2.423628807067871, -0.23847085237503052, -1.266244649887085, 9.871703147888184, -4.723357677459717, -0.24037568271160126, -3.636467218399048, -0.8201249241828918, -2.90771484375, -4.368000030517578, 2.212491989135742, 2.7155959606170654, 2.511230945587158, -0.06267551332712173, 10.169927597045898, -1.4580305814743042, -3.7872438430786133, 2.9694697856903076, -1.7234787940979004, -3.8688488006591797, 10.045881271362305, 0.6332783699035645, -2.9605512619018555, -5.4961442947387695, -1.6214683055877686, 3.5833661556243896, -6.409091949462891, -8.853309631347656, -2.2299859523773193, 6.19337797164917, 5.881902694702148, 6.931511402130127, -2.16139817237854, -3.129265069961548, -2.074312448501587, 5.511701583862305], [5.569871425628662, 2.2048161029815674, -0.9977854490280151, 6.825439453125, -1.5609450340270996, 11.168617248535156, -1.9970980882644653, -5.60792875289917, -0.10435563325881958, 0.9238651990890503, 4.148379802703857, -0.611919105052948, 7.852595329284668, 5.069023132324219, -5.2089619636535645, -4.84799861907959, 5.445859909057617, -2.7372326850891113, 0.6813322305679321, -4.339264869689941, -4.326872825622559, -9.3666410446167, 3.3898708820343018, -0.010151669383049011, 0.8140647411346436, -0.5192916393280029, -6.737855911254883, 3.982008218765259, -1.1127355098724365, 8.864534378051758, 5.888975620269775, -7.1554274559021, -4.274520397186279, -6.418558120727539, 0.001398661406710744, -2.8372178077697754, 3.348437547683716, -5.0587568283081055, 4.497039318084717, -1.4239658117294312, 4.612743854522705, -4.218109607696533, -3.762744426727295, -0.28835394978523254, -1.415042519569397, -2.3670196533203125, -1.5817853212356567, 6.204980373382568, 8.226344108581543, 1.541022777557373], [12.246336936950684, -15.399290084838867, -4.310369491577148, -0.2655485272407532, 13.60206127166748, -0.9080454707145691, 6.704675674438477, 1.65079665184021, -1.6079376935958862, 8.143418312072754, 7.227043151855469, 1.3646332025527954, -2.598940849304199, -2.281564235687256, 17.43950843811035, -1.808866262435913, -6.423736095428467, 12.307995796203613, -3.3529374599456787, -2.256255865097046, 9.896093368530273, -8.843096733093262, -9.011090278625488, -8.880097389221191, 3.392660617828369, -10.6293363571167, -0.48668187856674194, 11.222761154174805, 1.2617889642715454, -6.561703205108643, 3.449049234390259, 1.4644207954406738, -12.953835487365723, -11.031054496765137, 7.599069118499756, 7.206199645996094, -11.000375747680664, -9.602920532226562, -0.7209827303886414, 5.107595920562744, -3.4602231979370117, 4.4174346923828125, -3.1755542755126953, -0.40873655676841736, 1.6251870393753052, 6.351861000061035, -10.330162048339844, -6.976644992828369, 12.036186218261719, 4.11166524887085], [-8.861919403076172, -5.686025142669678, -9.421830177307129, 3.1126723289489746, -1.5662212371826172, -5.684755802154541, -11.535852432250977, -3.19950532913208, 0.3904380798339844, -8.571465492248535, 4.179407119750977, -1.235825538635254, 5.718071937561035, 6.566066741943359, -3.2586207389831543, 7.20152473449707, 2.676327705383301, 3.4789416790008545, -1.296540379524231, -0.4844086468219757, 8.22053337097168, 4.635218620300293, -4.291275978088379, -5.122518062591553, -2.704932689666748, -6.333137512207031, 0.40786027908325195, 3.7332661151885986, 0.11270283162593842, -11.94386100769043, 12.218335151672363, -9.946356773376465, -1.0313968658447266, -0.617415726184845, -16.668184280395508, -2.3883748054504395, -2.173229455947876, 7.7050395011901855, 2.304309368133545, -8.690143585205078, 7.2695512771606445, 9.58486557006836, 8.976449012756348, -0.37753811478614807, 1.4968100786209106, 2.096527576446533, 5.294309616088867, -7.557353973388672, -1.8109086751937866, 11.75455093383789], [-8.246939659118652, 3.5540568828582764, 0.3836626410484314, -2.820652723312378, -7.641499996185303, 3.7170305252075195, 6.103061676025391, -1.1357754468917847, 0.5636625289916992, -3.854299306869507, 5.547791481018066, -3.259139060974121, 4.933258056640625, -8.539401054382324, 15.935104370117188, 3.2641043663024902, 10.332877159118652, 4.733959674835205, 1.1367555856704712, -2.291395425796509, 0.9544110298156738, -7.005283832550049, -4.268693447113037, -13.07174015045166, 1.1612467765808105, 8.391971588134766, -0.43691661953926086, -3.5661284923553467, -12.931553840637207, 0.5847750306129456, 0.144536092877388, 3.193190097808838, -7.897246837615967, 0.42441174387931824, 7.03910493850708, 3.473325729370117, -17.96186065673828, -2.1478748321533203, 5.747069835662842, 3.3642892837524414, -5.537321090698242, 1.3445181846618652, -9.626091003417969, -4.695962429046631, -2.2368826866149902, 1.9288488626480103, 4.978389263153076, 6.905176639556885, -4.461233139038086, -11.965585708618164], [12.964637756347656, -2.2471868991851807, 2.138711929321289, 1.2860174179077148, -2.4100046157836914, 1.5893259048461914, 8.92611026763916, 11.595800399780273, -2.5136077404022217, -0.12602457404136658, -5.151689052581787, 1.6926203966140747, -10.039441108703613, -9.677386283874512, 7.628518104553223, 6.240850925445557, -4.465386390686035, 0.8299767971038818, -4.2420806884765625, -4.453725337982178, -2.0665690898895264, 4.901176929473877, 2.350529909133911, -1.2188341617584229, -5.847402095794678, -6.792741775512695, 1.0584399700164795, -2.670668125152588, -0.3207339644432068, -8.50228500366211, -13.359615325927734, 8.555380821228027, 6.92631721496582, -6.335441589355469, 3.3780529499053955, -2.2775533199310303, 1.556654930114746, -2.356548309326172, -6.958903789520264, -2.296269178390503, -1.6885662078857422, 4.5883049964904785, -2.8663151264190674, 1.5022000074386597, -5.424832344055176, 2.088099479675293, -2.2166330814361572, -2.918673276901245, -5.822834014892578, -2.4412591457366943], [-14.448314666748047, 7.928800106048584, 5.153792381286621, 11.119940757751465, -2.2305121421813965, -15.181795120239258, 1.879604697227478, -12.793810844421387, -3.5993785858154297, -16.176225662231445, 2.150759696960449, 5.755411624908447, 14.947366714477539, -8.506365776062012, 6.0903496742248535, -1.2996125221252441, 11.726507186889648, -4.43388032913208, 5.128735542297363, -5.062916278839111, 13.506815910339355, 0.9628480672836304, 8.222454071044922, -6.077628135681152, 4.9589033126831055, 7.9774088859558105, -4.299968242645264, -4.998255252838135, 2.11072039604187, 5.73391580581665, 9.217726707458496, -14.005107879638672, 1.7970224618911743, 3.4024150371551514, -0.612552285194397, 7.028502464294434, -8.57503604888916, 1.7039016485214233, 14.617274284362793, -9.272172927856445, 2.9187874794006348, 8.407533645629883, -2.0531654357910156, 4.68388557434082, 7.824920654296875, -2.903883695602417, 1.4273526668548584, -7.548396110534668, 3.4487688541412354, -0.7059274315834045], [-3.555290699005127, -2.515711784362793, 4.219647407531738, 11.291772842407227, 1.9442893266677856, 0.5699464678764343, 2.515272617340088, 2.666829824447632, 3.201260566711426, -3.911785364151001, 0.9352616667747498, -6.466495513916016, -5.345510005950928, -4.132239818572998, 6.046350002288818, 4.964794635772705, 4.2356181144714355, -0.092046819627285, -5.589866638183594, -5.65770959854126, 3.417249917984009, -4.879462718963623, -4.330559730529785, -1.3782368898391724, -0.03756087273359299, -9.740785598754883, 0.6330398321151733, 0.17740537226200104, -2.651395082473755, 6.717885971069336, 2.98350191116333, -3.218928337097168, -4.963791370391846, -2.354163646697998, 0.6338889002799988, -3.3248860836029053, -6.667430400848389, -6.060545444488525, -2.131709337234497, -7.097989559173584, -1.1719233989715576, 1.6939059495925903, -9.043013572692871, -0.5514038801193237, 7.409817695617676, 0.8616531491279602, -4.689253330230713, -4.770143508911133, -4.973730564117432, 3.6272313594818115], [-2.263699531555176, 14.408329963684082, 4.570504665374756, -0.23203341662883759, 7.7264885902404785, 2.879486322402954, -2.240501880645752, 7.766084671020508, 5.134337425231934, -1.2436367273330688, -0.26422443985939026, 4.924182891845703, -0.3948831856250763, 11.22215747833252, -1.3625035285949707, -7.320310592651367, -2.079106330871582, -9.38085651397705, 4.431633472442627, 12.697299003601074, -1.2411068677902222, -6.132152557373047, 8.829124450683594, 7.496356010437012, 3.717627763748169, 1.0445301532745361, 5.484816551208496, 5.149136066436768, -2.8398594856262207, 8.9677095413208, 2.134956121444702, -10.75940990447998, -3.260874032974243, -1.5940665006637573, -1.8819711208343506, 2.6533989906311035, 4.349942684173584, -5.777118682861328, 4.099752902984619, 17.48641014099121, -1.3046377897262573, -7.563698768615723, -12.207012176513672, 2.786571979522705, -5.854510307312012, 0.5018808841705322, -7.042499542236328, 8.127457618713379, 0.15038755536079407, 6.893161773681641], [-1.6876744031906128, 3.828538656234741, 1.1597553491592407, -5.666706085205078, 5.027136325836182, 1.0981650352478027, 3.693809986114502, -3.274085521697998, 1.3816170692443848, 0.9791563749313354, 9.524660110473633, -4.9002766609191895, 5.752609729766846, 2.864114999771118, 2.0146400928497314, 4.356712341308594, -7.129418849945068, -1.464622139930725, 1.089102029800415, 0.35657525062561035, -1.720151424407959, -0.700388491153717, 0.3917129933834076, -15.887533187866211, 10.54676628112793, 10.018317222595215, 1.4034281969070435, 4.480504512786865, -5.2405686378479, -3.827507734298706, 0.5837368369102478, -5.779520511627197, -2.8435168266296387, -6.657482624053955, 7.745543956756592, -2.2773706912994385, 3.959766149520874, -2.999434471130371, 3.9956305027008057, 2.180098533630371, 4.107741355895996, -5.3827033042907715, -3.099837064743042, 2.452714443206787, -11.873151779174805, -4.185131072998047, -2.072345733642578, 0.7929414510726929, -0.7926295399665833, 1.3346977233886719], [6.8638787269592285, -0.7330929636955261, -5.602597236633301, 5.071852207183838, 4.0372538566589355, -9.906305313110352, -4.803155422210693, -2.7929177284240723, -3.396078586578369, 2.1730129718780518, -2.2887046337127686, 1.5609720945358276, 7.076969146728516, -4.176751136779785, -6.613278388977051, -4.900688171386719, -17.487560272216797, -5.029679298400879, -0.7162440419197083, -4.742632865905762, 3.1708881855010986, 0.6458832621574402, -6.314147472381592, 1.629481315612793, 1.6004935503005981, -3.853142499923706, 2.7873027324676514, 0.8431099653244019, -3.4231629371643066, -8.960405349731445, -0.8721503019332886, 2.786400556564331, -3.816399335861206, 3.291632652282715, -2.7698917388916016, 5.5794677734375, 3.369528293609619, 4.115908622741699, -2.7966625690460205, -4.5502495765686035, 0.26549285650253296, 8.389545440673828, 4.425247669219971, 8.462697982788086, 9.948233604431152, 3.255802869796753, -3.872579574584961, -6.020954132080078, -1.5103882551193237, -0.5866754055023193], [-0.07388714700937271, 8.372282028198242, 5.124342441558838, 0.3098280727863312, 4.624091625213623, -4.701767921447754, -6.246520042419434, -9.602396965026855, -7.84201192855835, -3.4645073413848877, -9.896048545837402, 14.288759231567383, 4.3046979904174805, 11.154333114624023, -12.770670890808105, -3.1847879886627197, 2.429046154022217, -11.471014976501465, 9.978299140930176, 3.8982911109924316, -14.38459587097168, -5.454631805419922, -2.003338098526001, -11.896661758422852, 4.059582233428955, 8.400712966918945, 1.3594533205032349, 1.1204677820205688, 0.8841687440872192, -4.845010757446289, -0.012084435671567917, -6.73193883895874, -8.901668548583984, 2.3135690689086914, 3.253092050552368, 2.8229801654815674, 6.843143939971924, -6.080657958984375, 5.201731204986572, -1.4435936212539673, 2.571185350418091, -7.753364086151123, -1.0715702772140503, 0.5611453056335449, -6.732982635498047, 9.935611724853516, 5.358853340148926, 4.956839561462402, -3.709073066711426, 10.822715759277344], [-3.0240390300750732, 1.778576135635376, -5.901758670806885, -4.237540245056152, 13.58084774017334, 3.884359121322632, -2.1156082153320312, 7.044333457946777, 7.120254039764404, -2.2743232250213623, -7.743998050689697, 1.9459317922592163, -10.56653118133545, 10.763042449951172, -0.6975687742233276, -5.560621738433838, -12.43018913269043, -15.058356285095215, 0.7128351926803589, -5.410131454467773, -1.139453649520874, -5.5990729331970215, 5.914764404296875, -12.622069358825684, -1.5325881242752075, 7.229885578155518, 8.244342803955078, -4.3767876625061035, -1.4685674905776978, -8.024916648864746, -2.802027940750122, -3.985699415206909, 3.7234063148498535, 11.106642723083496, 14.665340423583984, 5.39874792098999, -2.7886877059936523, -10.388134002685547, 7.055478572845459, 11.692869186401367, -11.78774642944336, -3.348759174346924, -7.481870651245117, 6.540976047515869, -2.7553870677948, -6.1208038330078125, -1.9623396396636963, -2.464296579360962, 4.5539727210998535, -9.225851058959961], [-2.188067674636841, 2.6514649391174316, -6.523137092590332, -5.777058124542236, 7.352765083312988, 6.126102447509766, -4.236794471740723, -8.215970993041992, 2.1199381351470947, 8.933171272277832, 9.178860664367676, -5.2879509925842285, 5.537437438964844, -5.871676921844482, 8.070577621459961, 8.993022918701172, -5.790642261505127, 4.255439758300781, 4.641449928283691, 7.709428787231445, 1.2951301336288452, -0.23388151824474335, 7.5477800369262695, 3.2486472129821777, 10.715707778930664, 11.327484130859375, 5.2289934158325195, 7.819242000579834, 3.570370674133301, 1.855678677558899, -0.8668060302734375, -2.9555506706237793, -1.2608294486999512, -1.1457420587539673, -1.7409199476242065, -0.6207059025764465, 4.229914665222168, 11.039360046386719, 5.7632832527160645, -0.5509073138237, -5.610509395599365, -6.478507041931152, 4.7941789627075195, -1.7027329206466675, -3.797663927078247, -3.871938467025757, -4.489057540893555, -11.951539993286133, 0.6510311961174011, 4.20766019821167], [9.383588790893555, 2.1931936740875244, 2.1200177669525146, 5.058317184448242, 0.4139504134654999, 1.8131290674209595, -3.9333267211914062, -8.415520668029785, 0.7986757159233093, 0.537771463394165, 1.4687299728393555, 4.83802604675293, 1.4234187602996826, -1.4183273315429688, 6.060990333557129, -5.38886022567749, 9.941227912902832, -2.3999581336975098, 2.6014750003814697, 4.907039165496826, 9.714367866516113, 4.45009708404541, 4.670312404632568, 13.133193969726562, -3.4295401573181152, -3.3212034702301025, 0.8818992376327515, 0.4879981577396393, -5.891167163848877, 6.994332790374756, 3.225097894668579, -3.031028985977173, 5.436845779418945, -9.455378532409668, -12.934547424316406, 3.2295591831207275, 6.067341327667236, -0.9086688160896301, -2.3288068771362305, 1.516087293624878, -0.07135265320539474, -4.645020008087158, -5.652686595916748, -1.9031983613967896, 0.7249014973640442, 8.618364334106445, 0.057581931352615356, 1.9897559881210327, 0.5623649954795837, 2.451927900314331], [-0.7948789596557617, -9.223797798156738, -4.721069812774658, -2.3962643146514893, 3.136075258255005, 3.179546356201172, -3.0708518028259277, -12.3184814453125, 0.9306116700172424, 6.964282035827637, 12.092625617980957, -12.814125061035156, 13.398231506347656, -11.066240310668945, 5.266453742980957, -0.6434836983680725, -8.290282249450684, 1.257319688796997, -8.721025466918945, -7.489369869232178, 8.76054859161377, -5.057088375091553, 4.648099422454834, 1.2623016834259033, 2.697956085205078, 3.825435161590576, -3.2164182662963867, 6.66432523727417, 2.6357243061065674, 2.306490182876587, 14.854211807250977, -2.407177686691284, -14.008954048156738, -0.6864216923713684, 12.207045555114746, -3.7863306999206543, -4.014787673950195, 0.7379130125045776, 12.131134033203125, 0.5975871682167053, 2.6508142948150635, 4.40017557144165, 1.3038177490234375, 0.5123229026794434, 5.332324028015137, -7.524384021759033, -6.120172500610352, 1.953565239906311, 7.903897285461426, -2.4656968116760254], [1.717079758644104, 0.8020488619804382, -0.3373245298862457, -6.275822639465332, -5.162097930908203, -3.249509811401367, 11.533781051635742, 3.3461360931396484, 1.4924442768096924, 2.0265047550201416, 0.4696545898914337, -1.9753187894821167, -5.216402053833008, -8.674291610717773, 1.1863234043121338, -2.1624112129211426, -4.806336879730225, 13.052840232849121, -2.9725279808044434, -3.001725196838379, -10.08270263671875, -2.079648733139038, -11.22347354888916, -10.775979995727539, 1.2568340301513672, 1.7690041065216064, -1.8660469055175781, -5.393326759338379, -1.0702651739120483, 9.715046882629395, -4.91895866394043, 10.8698091506958, 1.9394690990447998, 1.541249394416809, 9.858657836914062, 5.955959796905518, -9.520724296569824, -1.913283109664917, -1.6903893947601318, 1.9687652587890625, -6.22773551940918, 0.1411805897951126, 3.7952167987823486, -0.5554418563842773, -0.027292631566524506, 2.5053603649139404, -0.214938223361969, -6.181882381439209, -0.10487707704305649, -2.7004075050354004], [5.686284065246582, -7.495841979980469, 3.6103053092956543, -3.349429130554199, -5.84769344329834, -7.619365215301514, 3.8141181468963623, -6.684810161590576, -1.1360938549041748, 6.746100425720215, 4.78121280670166, 1.1283693313598633, 4.986311912536621, -11.601053237915039, 3.7042601108551025, 0.5239584445953369, 9.115194320678711, 6.388235569000244, -1.6114603281021118, -2.151789426803589, 1.4442641735076904, 5.287683963775635, 0.030935920774936676, -7.43275785446167, 9.020383834838867, 4.69237756729126, 1.0037286281585693, -6.874035358428955, -7.572697162628174, -1.4030084609985352, 6.5275421142578125, 1.2659070491790771, 1.593469500541687, -4.758761882781982, -0.8283013105392456, -5.5999627113342285, -2.589942216873169, 3.3705615997314453, 6.082159042358398, -9.479177474975586, 7.521107196807861, 0.3985733091831207, 8.584341049194336, 0.5953531265258789, -0.444801390171051, -3.174147129058838, -2.6346209049224854, -3.3255715370178223, -2.9361934661865234, -5.941372871398926], [-0.6513653993606567, -4.758984088897705, 4.285569190979004, 0.6245077252388, 0.3499217927455902, -2.926124095916748, 8.994579315185547, -10.832072257995605, 5.877998352050781, -1.3973946571350098, -0.4694060683250427, -4.162403583526611, 1.427237629890442, 4.539401531219482, -3.1071834564208984, -8.773216247558594, 5.279751300811768, -5.230515956878662, 6.1406402587890625, -7.113023281097412, 3.7089407444000244, 0.7861641049385071, 7.357938766479492, 10.124826431274414, 1.0070955753326416, 4.634708881378174, -5.861837387084961, -3.065964937210083, 4.479753017425537, 9.031612396240234, 9.47851848602295, -6.927090167999268, -0.753625214099884, -0.6265837550163269, 5.477672576904297, -4.376797199249268, 0.32476475834846497, -4.8857808113098145, 1.131193995475769, -8.14006519317627, -1.046157717704773, -0.743168294429779, 0.5913986563682556, 5.2372727394104, 3.1471059322357178, -1.5663988590240479, 0.7070788145065308, 13.913697242736816, 2.554457902908325, -0.24931229650974274], [6.49594783782959, -2.49157977104187, -1.822313904762268, 6.504256248474121, -6.018948554992676, -5.31975793838501, 10.728902816772461, -3.0074353218078613, -5.703115463256836, -2.4967825412750244, -3.555764675140381, 1.5174936056137085, 0.8911176919937134, -12.338312149047852, -1.6216647624969482, 3.6744332313537598, 1.7637603282928467, 0.545108437538147, 4.2291412353515625, -0.3173549771308899, -2.051619529724121, 4.706845283508301, 1.6993836164474487, 13.267890930175781, -2.679011821746826, 0.07157967239618301, -7.931906223297119, -5.155673980712891, -7.254039764404297, 2.9301345348358154, -4.364387035369873, 6.925294876098633, 6.93287992477417, 4.261104583740234, -18.60232162475586, -2.2418625354766846, -1.0018402338027954, 2.288116693496704, 2.3095154762268066, -17.469343185424805, 2.5558366775512695, 11.392587661743164, 3.71073842048645, 2.1277012825012207, 7.254749774932861, 0.6045380234718323, 7.428746700286865, -0.13225895166397095, 4.924159526824951, -3.882647752761841], [1.0197277069091797, 0.5537453889846802, -3.3347840309143066, -5.062121391296387, 10.267680168151855, 1.517621397972107, 1.588024377822876, 0.8243626356124878, -1.1049273014068604, 1.671862006187439, 5.896444320678711, -1.9388338327407837, -9.310205459594727, 3.256462335586548, 10.673603057861328, 9.884786605834961, -1.5108963251113892, 5.026651382446289, -2.4547178745269775, 7.102092266082764, 13.442169189453125, -5.107044219970703, -1.1723105907440186, 0.5569367408752441, 7.860632419586182, 1.0168969631195068, 4.388060569763184, 8.622732162475586, 5.26237678527832, -7.0576605796813965, -1.9748761653900146, 0.9639356732368469, -3.1601812839508057, -0.7689430117607117, 5.049471855163574, 2.779554605484009, -0.1054571121931076, 3.0709786415100098, 3.695004463195801, 8.975836753845215, -9.703944206237793, 1.0230439901351929, -4.205191135406494, -5.093254566192627, -7.8631415367126465, -6.144786357879639, 1.6673550605773926, -0.49697113037109375, -2.16011643409729, -1.822180151939392], [-2.792379140853882, 10.095024108886719, -18.90222930908203, 8.213753700256348, -3.187854051589966, -6.992347240447998, -7.2340850830078125, -2.7189526557922363, 3.8993589878082275, -12.588837623596191, 15.902328491210938, -1.645668864250183, 12.250316619873047, -6.021470546722412, 6.85944128036499, 2.6658716201782227, -10.295153617858887, 6.239818572998047, 0.15957151353359222, 7.485649108886719, 6.502644062042236, -4.319451808929443, -9.839760780334473, 1.6773202419281006, 4.969790935516357, -3.686655044555664, -2.060504674911499, 9.930136680603027, -9.376360893249512, -10.148506164550781, 5.946760177612305, -15.179553985595703, -10.165274620056152, -0.8110594153404236, -19.14019012451172, -0.06905724853277206, -9.1731595993042, 6.884396076202393, 7.25885534286499, 14.888833045959473, 3.162895679473877, 8.268059730529785, 2.1445298194885254, 3.5885961055755615, 1.1462836265563965, 0.23179733753204346, -0.20778420567512512, 0.8067615032196045, 4.560770034790039, -8.900467872619629], [0.5230047702789307, -4.0595479011535645, -1.9515032768249512, 0.5830773711204529, 1.9150725603103638, 9.291980743408203, 0.45924222469329834, -17.260639190673828, 1.262003779411316, 4.840405464172363, 3.520758867263794, -6.930523872375488, -1.7006807327270508, -10.464648246765137, -4.06167459487915, -1.508440375328064, 4.610418319702148, -0.06056901812553406, -9.555112838745117, -0.026337431743741035, -1.042011022567749, 6.251521110534668, -2.6749651432037354, 9.83044147491455, -2.146973133087158, 6.608392238616943, 2.2215068340301514, -3.8488237857818604, 8.483695983886719, 10.477764129638672, 1.2424627542495728, 5.246779918670654, 2.7398746013641357, -1.8857160806655884, 14.530128479003906, -5.938313007354736, 2.467325448989868, -1.2415424585342407, -3.930006742477417, -0.7397783398628235, -0.590414822101593, -11.521117210388184, -0.300082802772522, -2.6925601959228516, 8.645736694335938, -7.83436918258667, 1.6775034666061401, -2.211146593093872, 0.8162671327590942, 5.930039405822754], [9.906122207641602, 8.458735466003418, -1.0394947528839111, 4.676346302032471, -8.651140213012695, -0.038691066205501556, 3.6378660202026367, -3.4520933628082275, -10.4935302734375, -3.42513370513916, 0.02134575881063938, 15.255446434020996, 3.8266682624816895, -0.08815070241689682, -5.3159871101379395, 4.1821675300598145, 8.433927536010742, -1.8747906684875488, 10.409334182739258, 1.750431776046753, -7.577165126800537, -0.29400575160980225, -6.381822109222412, -10.130290985107422, 10.170062065124512, 0.42302241921424866, 0.6823693513870239, -2.793128252029419, -10.234220504760742, -9.688610076904297, -3.517221450805664, -6.55063009262085, 1.408167839050293, -8.26384449005127, -12.082923889160156, -3.8646304607391357, 1.5493775606155396, 0.3068847358226776, 8.458213806152344, -6.492759704589844, 1.030929446220398, -4.713919162750244, 2.108664035797119, 3.2436327934265137, -6.588798999786377, 2.8856067657470703, 2.4039719104766846, -3.081742525100708, -8.606696128845215, 13.179573059082031], [0.24712194502353668, 0.1052241176366806, -4.188264846801758, -0.9355999827384949, 5.036436080932617, 1.7022342681884766, -1.1488940715789795, 3.8563625812530518, 11.737353324890137, 2.063204288482666, 3.7379298210144043, -3.5670504570007324, -4.376739501953125, 2.3335466384887695, 3.057690382003784, 0.6444595456123352, -3.749741554260254, 6.621071815490723, 2.0407752990722656, 1.1746643781661987, -7.917446136474609, -3.042069673538208, -0.9894025921821594, -7.432076930999756, -5.995368957519531, -2.561007022857666, -2.8887264728546143, 1.3242545127868652, 7.900956153869629, -0.6973584890365601, -5.958949565887451, 0.20628218352794647, -1.6247025728225708, 4.915273666381836, 8.202522277832031, 6.426573753356934, -6.858205318450928, 3.2567451000213623, -7.574031352996826, 9.32363510131836, -4.503550052642822, -11.179065704345703, 5.113336563110352, -8.40500545501709, -6.881605625152588, 4.4277520179748535, 3.0019588470458984, 1.234958291053772, 8.497117042541504, -9.182930946350098], [9.440110206604004, 0.13874880969524384, 0.6791629791259766, -0.5249629020690918, -1.1468307971954346, 2.5519673824310303, 1.9838335514068604, 9.377341270446777, -6.236215114593506, -2.1596474647521973, -5.914833068847656, 4.043615818023682, -15.86182975769043, -0.3890549838542938, -0.0997096598148346, 0.37642234563827515, -6.555277347564697, -0.26920536160469055, -9.749691009521484, 6.583113670349121, 9.440861701965332, -1.8052476644515991, -1.9969502687454224, 17.54747772216797, 0.5986030101776123, -5.764683723449707, 6.988162040710449, 1.5731725692749023, 6.603676795959473, -6.194761753082275, -0.6756212711334229, 5.104706287384033, -1.171840786933899, -2.6763803958892822, -0.16453929245471954, 1.5993636846542358, 5.681178569793701, -0.9651432037353516, -5.857329845428467, 10.83935260772705, -9.322661399841309, 6.244585990905762, -6.705958366394043, 6.390661239624023, 2.8381130695343018, -4.855770587921143, -4.889254570007324, -0.4292369782924652, 1.2441335916519165, 4.073668956756592], [-0.7289626598358154, -0.07397683709859848, -1.2045528888702393, 1.5678117275238037, 8.335832595825195, 7.855318546295166, -10.853639602661133, 2.180175304412842, 0.5437037348747253, 3.6545283794403076, -6.922930717468262, 1.9096189737319946, 4.544663429260254, 7.571731090545654, -5.566575050354004, -2.9853272438049316, -19.65143394470215, -7.499048233032227, 5.561001300811768, 5.201643943786621, -14.523045539855957, -2.311241626739502, 7.914275169372559, 6.417850017547607, -0.10230369865894318, 2.2733144760131836, 3.203122138977051, 2.9133849143981934, 2.046549081802368, 12.360711097717285, 0.6423019170761108, -2.629502296447754, -6.097207546234131, 3.8174664974212646, 5.111789226531982, 6.067927837371826, 6.426703929901123, -3.6250412464141846, -3.748750686645508, -4.209126949310303, 1.4201207160949707, 3.2607381343841553, 2.3516082763671875, 1.111698031425476, -7.539802074432373, 6.068811893463135, -8.29319953918457, -0.5997510552406311, 4.666804313659668, -0.2684628367424011], [5.338715553283691, -1.9729840755462646, -0.9370919466018677, 0.38074859976768494, 9.492973327636719, -9.246268272399902, 9.660627365112305, 3.314082145690918, -4.145918846130371, -4.936459541320801, -3.88960599899292, 3.6027207374572754, -6.358410358428955, 5.829962253570557, -1.2083334922790527, -3.4366960525512695, -5.512931823730469, -2.6111981868743896, 2.6136653423309326, -4.193055152893066, 4.701576232910156, 0.9630638360977173, 0.3388897478580475, -8.393538475036621, -1.8613885641098022, -2.268026351928711, -1.6402759552001953, -3.660274028778076, 6.691788196563721, -6.1836724281311035, -3.8184075355529785, 2.366516590118408, -1.7107958793640137, 4.894645690917969, 12.242149353027344, 10.018370628356934, -3.7586817741394043, -3.0321271419525146, -0.9319410920143127, 3.2823829650878906, -6.472471237182617, 5.296675205230713, -0.5633077621459961, 4.523348331451416, -2.6935784816741943, -1.3070645332336426, 0.8684877753257751, 4.915328025817871, 3.7705979347229004, 1.1384037733078003], [2.3611061573028564, -2.661480665206909, -0.9733438491821289, 1.9700357913970947, 2.1811580657958984, 8.129646301269531, -5.670943260192871, -11.593064308166504, -2.927185535430908, 3.1263601779937744, 3.8872621059417725, 0.5848197937011719, 6.1319966316223145, -2.579862117767334, 2.075899124145508, 4.344658374786377, 5.124406337738037, -5.220381736755371, -4.061263561248779, -3.666156768798828, 2.5797221660614014, 2.3553354740142822, 4.379773139953613, -0.5921562910079956, 2.394414186477661, 6.8605217933654785, 1.9028723239898682, -4.739051342010498, 5.055851459503174, -0.20797847211360931, 0.44740191102027893, 1.4249037504196167, 0.17048239707946777, -0.7858436107635498, 8.021652221679688, 0.11706393957138062, 2.9438605308532715, -0.5644354224205017, 2.5358047485351562, -2.534942388534546, 0.8440411686897278, -6.197225570678711, -2.9386279582977295, -6.101178169250488, 0.7963078022003174, -5.848118305206299, -2.0774922370910645, -4.7378129959106445, 4.160387992858887, 3.0814294815063477], [6.278403282165527, 0.2102314978837967, 4.0838165283203125, 2.310561418533325, 1.7754722833633423, -2.8644351959228516, -0.8803638219833374, 2.5045394897460938, 0.18446248769760132, 5.060113430023193, -4.040578365325928, 7.5086750984191895, -6.3126091957092285, 11.983236312866211, -0.7112410664558411, 2.0838205814361572, 4.740999221801758, 3.7105484008789062, 7.388823986053467, -1.119244933128357, 3.8689658641815186, -4.590874671936035, -7.264133930206299, -4.419793128967285, -4.4438066482543945, -11.7903413772583, -3.931018829345703, 2.23065447807312, 2.439196825027466, 1.2266114950180054, -5.5613555908203125, 3.5506935119628906, 7.886139869689941, 0.5715184807777405, -6.999143123626709, 3.0738232135772705, 2.188977003097534, -5.80620002746582, -0.4729641079902649, 2.9621591567993164, 0.04916401952505112, -6.683420181274414, -0.8755677342414856, -7.0621657371521, -5.624358177185059, 6.026504039764404, 6.976727485656738, 3.461174249649048, -5.22894811630249, -4.205789089202881], [-7.024819850921631, -4.815248489379883, 2.136199712753296, 2.011916160583496, 4.0556254386901855, -6.019098281860352, -6.809210300445557, 3.3666045665740967, -1.70626962184906, -1.09268057346344, -4.140768527984619, 2.880375385284424, 2.6972761154174805, 6.046446323394775, -0.732460081577301, -2.8078553676605225, -10.107955932617188, -0.8003783226013184, 4.425044059753418, -2.910846710205078, -4.2414045333862305, 0.051345184445381165, -0.49775800108909607, -9.353358268737793, -6.572365760803223, -5.049496173858643, -0.032589949667453766, 2.78969407081604, 1.2769312858581543, 1.6889551877975464, -1.2174677848815918, -2.3906307220458984, 0.8141658306121826, 1.6824653148651123, 2.5931947231292725, 1.2181189060211182, 0.5710723996162415, -5.4077982902526855, 4.881011486053467, -2.780595302581787, 5.673730373382568, 3.4770615100860596, 4.013298511505127, -1.166563630104065, -5.411258220672607, 7.228232383728027, 1.2612909078598022, -1.3769408464431763, -4.190206527709961, 1.2420456409454346], [6.480617046356201, -5.313638687133789, 4.630534648895264, -3.031846046447754, -3.480587959289551, -1.3351093530654907, 10.04780101776123, 15.984121322631836, 4.522993087768555, 6.941293239593506, -2.9473862648010254, -2.7900307178497314, -1.7638179063796997, -4.223688125610352, -2.008881092071533, -0.9725713133811951, -2.895819664001465, -3.5007660388946533, -4.595147132873535, -6.470229625701904, 2.459787607192993, 4.437721252441406, 2.672238826751709, 6.71919584274292, -4.892813682556152, -13.007686614990234, -8.366929054260254, -0.7090523838996887, -2.230851173400879, -8.95308780670166, -5.1568989753723145, 8.075886726379395, 4.793313503265381, 3.4258413314819336, -0.6175416111946106, -3.176313877105713, -3.6010525226593018, -1.070044755935669, -4.696845054626465, -3.4513614177703857, 2.998692035675049, 5.501501083374023, 0.7042309641838074, -3.38130521774292, 5.639341831207275, -1.784886121749878, -2.908367156982422, 4.6563191413879395, 3.131385087966919, -6.031609058380127], [10.4983549118042, -0.6165856719017029, -9.471038818359375, 0.6440796256065369, 7.504335403442383, -4.60810661315918, 19.96346664428711, -4.270130634307861, 4.5626702308654785, 1.223665714263916, 12.719233512878418, -3.6205296516418457, 4.037433624267578, -6.598538398742676, 1.0636214017868042, -0.2682299017906189, -7.230733394622803, 2.556584596633911, -6.432126998901367, -2.869830846786499, -0.36520496010780334, -4.675827980041504, -7.682724952697754, 2.923835277557373, 11.203733444213867, 5.123296737670898, -1.8570722341537476, -6.262099266052246, -9.320246696472168, -3.929090738296509, -1.747977614402771, 8.963696479797363, -9.540008544921875, 1.8971136808395386, 9.729182243347168, 6.245238304138184, -13.809087753295898, -3.127023458480835, 5.090717315673828, 0.2407330870628357, -6.6242356300354, 6.980926990509033, -4.681048393249512, -4.146761417388916, 2.9058148860931396, -4.133779525756836, 2.4139366149902344, 10.004252433776855, 11.55420207977295, 2.2018494606018066], [2.1574928760528564, -4.915730953216553, 7.125469207763672, -2.2537295818328857, -0.8177052736282349, -3.106855869293213, 2.466968297958374, -1.703747034072876, -7.498098373413086, 0.021787725389003754, -1.7012163400650024, 9.353147506713867, 7.479092597961426, -11.283160209655762, 7.250050067901611, -2.8292202949523926, 11.785477638244629, 4.121494770050049, 5.976284980773926, -7.744691371917725, 3.922534465789795, 10.53653335571289, 7.599917888641357, -0.5165606141090393, -10.23163890838623, -8.482756614685059, -5.9170331954956055, -3.3787662982940674, -2.966569185256958, 7.845217704772949, -3.89290189743042, 5.977557182312012, 8.98593521118164, -3.3438773155212402, 2.2589871883392334, 0.36893266439437866, -4.67208194732666, 1.5954039096832275, 4.330978870391846, -5.4973554611206055, 7.859648704528809, 7.819817543029785, 7.289809703826904, -3.033264398574829, 1.040144681930542, 7.7373504638671875, 2.640275001525879, 0.4877474308013916, -0.5275107622146606, -5.0576958656311035], [1.2325783967971802, 3.217622756958008, 0.5606986880302429, -0.10795672237873077, 1.413899540901184, 4.715059280395508, -8.597673416137695, -4.556903839111328, -1.1298134326934814, 5.017053604125977, -4.041538715362549, 5.534261703491211, 2.8439958095550537, 4.623527526855469, -8.282089233398438, -5.054727554321289, 2.140191078186035, -0.7498729228973389, -0.909209132194519, 6.166732311248779, -2.749828815460205, -5.566918849945068, 2.720883369445801, 7.252089500427246, 1.4040740728378296, 3.387981414794922, 0.953847348690033, 4.783253192901611, 6.24863338470459, 0.23322619497776031, 3.977294445037842, -1.9841502904891968, -1.334259271621704, 0.07243704050779343, -0.5379993319511414, 3.8173861503601074, 7.652133941650391, 1.8737375736236572, -0.8366102576255798, 4.310024738311768, 0.4618988633155823, -5.265503406524658, 2.717756509780884, -0.6087567210197449, 1.9792648553848267, 1.3030458688735962, 1.0727394819259644, -4.069010257720947, 8.706727027893066, 3.108886241912842]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.8277092576026917, -0.43716323375701904, -0.5140575170516968, 0.7431458234786987, -0.7322806119918823, -0.7001209855079651, -1.672414779663086, -0.2502804398536682, 0.5541940331459045, 0.24066466093063354, -1.0132958889007568, -1.2005659341812134, 1.2048346996307373, 2.0394437313079834, -0.19672048091888428, -0.15594099462032318, -2.3966851234436035, 0.7022680640220642, 0.4630788266658783, 1.7880549430847168, 0.08623840659856796, -0.8958799839019775, -0.18807265162467957, 1.4568904638290405, 0.10398590564727783, 0.13824066519737244, 1.0937821865081787, 0.5546332597732544, 0.49395355582237244, -0.4891314208507538, 0.8432779312133789, 0.3045929968357086, -0.08861952275037766, -1.0188319683074951, 1.381442666053772, 0.4536643624305725, -0.8233689665794373, 0.2561010718345642, -0.332714706659317, -0.12107150256633759], [-0.1754635125398636, 2.07251238822937, 0.818844735622406, -0.4182963967323303, 1.6149795055389404, 0.3843151926994324, -2.135021209716797, -0.5053561925888062, -0.13369451463222504, 0.7428860664367676, -1.3842723369598389, 0.08880794793367386, 1.9102952480316162, 1.0803035497665405, 0.8498811721801758, -1.2814507484436035, 0.950386643409729, 0.5974435210227966, -0.0949757769703865, 0.6172016263008118, 0.03261159732937813, 1.551149845123291, 0.2565300464630127, -0.4790799915790558, 0.6019493341445923, -2.5323486328125, 0.890527606010437, 1.9034582376480103, -0.38839423656463623, 0.003434923477470875, -0.1970464140176773, -0.5614762902259827, 0.014710926450788975, 1.5724496841430664, -1.0381510257720947, -1.4914911985397339, -1.047613501548767, -1.5570505857467651, -0.9614403247833252, -0.094561368227005], [1.8061909675598145, 1.159663200378418, 0.28131425380706787, -0.9146504998207092, 1.0471019744873047, -0.12017960846424103, 0.4743921458721161, -1.4445693492889404, 0.5174800753593445, 0.6184548139572144, 1.0363788604736328, 0.3666855990886688, 0.9899292588233948, 0.15871688723564148, -0.1153831034898758, 0.5556033849716187, -0.07883309572935104, 1.3630274534225464, -0.8533170223236084, -0.01614494062960148, -0.7510843276977539, -0.34902632236480713, 0.866935670375824, -0.36092904210090637, -1.260677456855774, 0.5738793611526489, 0.23505324125289917, -0.9842745661735535, 1.9724491834640503, -0.566318690776825, 0.48875656723976135, 0.008625525049865246, 0.8355092406272888, 0.8810750842094421, -0.5710356831550598, -1.2825847864151, 0.7306497693061829, -0.6461235880851746, -1.6412079334259033, -2.161285400390625], [1.6899752616882324, 1.3248885869979858, 0.48294082283973694, 0.09542959183454514, -0.813988208770752, -1.0022016763687134, 0.2187187820672989, 1.3051912784576416, 0.923140287399292, 0.8372040390968323, -1.556792974472046, 1.3683680295944214, 0.45409631729125977, -0.9841788411140442, 1.0207520723342896, 2.010296583175659, 1.946933388710022, -1.191525936126709, 0.12144388258457184, -0.3847058415412903, 1.677195429801941, -0.06525666266679764, -0.6026773452758789, -0.17228065431118011, 0.1553896963596344, -0.8302900791168213, -1.3458906412124634, 0.36025020480155945, -1.6290380954742432, 0.9183518886566162, 0.23417431116104126, 0.23794154822826385, 1.3983896970748901, 0.4288792014122009, -1.2457090616226196, -0.8532958626747131, 1.7401264905929565, 0.7271178364753723, 0.1980314552783966, 1.1418461799621582], [1.4576762914657593, -0.19083155691623688, 0.37601813673973083, -0.22260625660419464, -1.8103649616241455, -3.0281593799591064, 1.4617379903793335, -0.6878136396408081, 1.7843877077102661, -2.1313414573669434, 1.1297353506088257, 2.7384490966796875, -0.7565467357635498, 0.23373563587665558, -0.7796242833137512, 0.04767397791147232, -0.9482946395874023, -1.8571900129318237, 0.06039995700120926, 0.7954912781715393, -0.7729319930076599, 1.3143620491027832, -1.5312530994415283, 0.3862897753715515, -1.0801112651824951, 0.5026304125785828, 0.8133854269981384, -1.5848621129989624, 0.4279917776584625, -0.576248824596405, 0.008467976935207844, -0.09487579762935638, 2.308452844619751, -0.8342383503913879, -0.4542168080806732, -0.304206520318985, -1.2202787399291992, 1.9567086696624756, -0.15225525200366974, 0.7217538356781006], [-1.7488386631011963, -0.2122252732515335, 0.09164931625127792, -1.1075609922409058, 0.38315868377685547, 0.10845569521188736, -1.4854000806808472, 0.9723572134971619, -0.4380717873573303, 0.7899146676063538, 0.3178020715713501, 0.4178259074687958, 0.7108185887336731, 1.7293399572372437, 0.5223357677459717, 0.7960121631622314, -0.9197636246681213, -0.7194663286209106, 1.4351952075958252, 0.7083175778388977, -1.7628153562545776, 0.9686563014984131, -0.6364453434944153, 0.6932663321495056, 1.0115070343017578, -0.19261635839939117, 0.5670584440231323, -1.2659229040145874, -0.3728887736797333, -0.41530683636665344, 0.10583879798650742, -0.5033165812492371, -1.1701397895812988, 1.3310825824737549, 0.11083976924419403, -1.3167635202407837, 2.617859363555908, -0.47602495551109314, -0.822422444820404, 1.304918885231018], [-1.2999851703643799, -0.887698769569397, -0.725159764289856, 0.749086856842041, -0.39542320370674133, -0.14151355624198914, -2.060767412185669, 0.3990132212638855, 0.15459652245044708, -0.6434407830238342, 0.5917026400566101, 0.9288092851638794, 1.1518789529800415, -1.5077553987503052, 0.6674280166625977, 2.2548470497131348, -0.3592239022254944, -0.6954652667045593, -0.19648176431655884, -1.0615947246551514, 1.1506835222244263, 0.6308456659317017, 2.0116541385650635, -0.35697075724601746, -0.18446899950504303, 0.16862830519676208, -1.2299748659133911, 0.2621655762195587, -1.188506841659546, -0.2963516116142273, 0.8089778423309326, 0.1366206258535385, 0.775034487247467, -1.1565546989440918, -0.40124234557151794, -1.2941137552261353, -0.3726158142089844, -0.2086125910282135, 0.6616613268852234, -0.42989587783813477], [-0.37607550621032715, -0.2518135607242584, -0.8244867324829102, 0.2844659984111786, -0.43555834889411926, -1.681104302406311, 0.7066280841827393, 0.22794048488140106, 1.4414403438568115, 0.5683965086936951, -1.1068216562271118, 1.0641433000564575, -2.694533586502075, 0.16583208739757538, 0.7456055879592896, -0.3519720137119293, 0.15752941370010376, -1.557235598564148, -0.33581310510635376, 0.3904436230659485, -0.4694134294986725, -0.06800797581672668, -0.5793206095695496, -0.06994400173425674, 1.0885459184646606, -0.6118496060371399, 1.2366031408309937, 0.0822744220495224, -0.07052911072969437, 0.6485756039619446, 1.3115166425704956, 0.5755025744438171, -0.47775134444236755, -0.6518181562423706, -1.1798397302627563, 1.910982608795166, -0.5838778018951416, -0.8370386958122253, 0.46694278717041016, 0.4839010536670685], [1.0025559663772583, 1.1668211221694946, -1.5279299020767212, 1.2453728914260864, -1.0903639793395996, 1.036863088607788, -0.34084081649780273, 0.2250257283449173, -0.5477524399757385, -0.12080924212932587, -0.9417667984962463, 0.08330743759870529, 1.0047798156738281, -0.006559513974934816, -1.5431814193725586, 0.7628589272499084, 0.0981869325041771, 0.533287525177002, 0.562391996383667, 0.31462225317955017, -0.5623667240142822, 0.32490283250808716, 0.5011147260665894, 0.9540308117866516, -0.10108482092618942, -1.0221225023269653, 1.7369346618652344, 1.3488627672195435, -0.17356035113334656, 3.0140111446380615, -0.8578866124153137, -1.5688849687576294, 0.711506724357605, 0.38983824849128723, 0.7501474618911743, 1.9755651950836182, 0.12367844581604004, 0.2970261871814728, -0.30914101004600525, 1.2589048147201538], [0.010770917870104313, 0.6505719423294067, 0.909869372844696, -1.6717945337295532, 0.7017760872840881, 0.2631687819957733, 2.066472291946411, -0.33636578917503357, 0.902146577835083, 0.26850220561027527, -1.192186713218689, -0.5218479633331299, 1.649485468864441, -1.06048583984375, -0.4264789819717407, 1.5862222909927368, -2.2255513668060303, -1.3067318201065063, -0.25600484013557434, 1.2563016414642334, -2.290797710418701, -1.018538236618042, -1.2047263383865356, -1.6912115812301636, -1.036863088607788, 1.7973803281784058, 0.6871635913848877, 0.23915651440620422, 0.1811959594488144, -1.8825175762176514, -0.7858487963676453, -1.2904843091964722, -0.5032013654708862, -1.341043472290039, -1.6197562217712402, -0.7272545695304871, 0.8484418392181396, 0.9283809661865234, 0.3860040605068207, 0.6363003253936768], [-0.2280847281217575, 1.7586228847503662, 1.0209691524505615, 1.8335715532302856, 0.34192904829978943, 0.27075666189193726, -1.5049656629562378, 0.8648931980133057, -1.2477365732192993, -0.6454417705535889, -0.19006940722465515, -0.7814934849739075, 0.926879346370697, -0.12671571969985962, -1.1090741157531738, 1.3095349073410034, -0.6505703330039978, -0.2630775272846222, 1.4689581394195557, 0.1161932572722435, 1.1769404411315918, 1.7771846055984497, 1.4838130474090576, 0.43447425961494446, 0.2059205323457718, 0.9841245412826538, -0.5013439655303955, 0.08484815061092377, 1.5882457494735718, 0.6942358613014221, 0.8744193315505981, -0.5601195693016052, -0.35173919796943665, -2.3475799560546875, 1.229046106338501, 1.2342643737792969, 0.7767070531845093, -1.4913909435272217, 0.33965328335762024, 1.6807154417037964], [-0.3146990239620209, 0.5839623212814331, -0.20747512578964233, 0.3123409152030945, 0.2743520140647888, -0.06353042274713516, -1.1337568759918213, 0.6076449155807495, -0.34570562839508057, 0.7355226874351501, 0.4321286082267761, 0.28177958726882935, -0.25079023838043213, 0.7118085026741028, -2.2047157287597656, -1.1806689500808716, 0.8340710401535034, 0.09810991585254669, 0.5057129859924316, 1.540401816368103, 0.7136369347572327, -1.3451181650161743, 0.34612223505973816, -0.705748438835144, -1.830642580986023, 0.7921937108039856, 1.2689977884292603, 1.1398954391479492, 2.0905792713165283, -0.538127064704895, -0.2322266399860382, 0.41054850816726685, -0.015103469602763653, -0.2324559986591339, 0.259591668844223, 0.6100406050682068, 1.2558308839797974, -0.358769953250885, 0.14323392510414124, 1.4904165267944336], [0.05966435745358467, -0.9364458918571472, 1.1530667543411255, 0.13803648948669434, 1.849117636680603, 1.377917766571045, -0.8752222061157227, 0.8383851647377014, 0.39399364590644836, 0.47437381744384766, 1.458461880683899, -0.27394193410873413, -0.09934087097644806, -1.1215381622314453, -0.08458264172077179, 0.4205075204372406, 0.07508087903261185, -1.1281366348266602, -0.4799598455429077, -2.6924729347229004, 0.9171205163002014, -0.556195855140686, 0.7716796398162842, -0.589608371257782, -0.721470057964325, 1.7573795318603516, 0.04891211539506912, 2.338364362716675, 0.6240823268890381, -0.9215217232704163, -0.3671850562095642, 0.09364904463291168, -1.907318353652954, -1.0797979831695557, 1.7528631687164307, 1.27618408203125, -0.4690597355365753, 0.6435851454734802, 1.7241528034210205, -1.2294820547103882], [0.29648348689079285, 1.0383546352386475, -0.07750599086284637, 0.7704665064811707, 0.9282940030097961, -1.5538727045059204, 1.0284162759780884, 1.263260006904602, 0.24031466245651245, -1.3415066003799438, 0.44855907559394836, -0.4814506769180298, 0.9866454601287842, 0.28834059834480286, 1.1126511096954346, 1.350921630859375, -0.4852193593978882, -0.1380699723958969, 0.18598362803459167, -1.0382897853851318, -0.8868938684463501, -0.42145195603370667, 1.622083306312561, 0.9782228469848633, 1.5666061639785767, -2.3014678955078125, 0.10071117430925369, 1.2496060132980347, 0.5529727339744568, -1.7267848253250122, 0.3862934112548828, 0.34255465865135193, 2.481147050857544, -0.03191803768277168, -2.4534785747528076, -0.2891044020652771, -0.5157498121261597, 1.4799367189407349, -0.22204825282096863, -0.5719646215438843], [-0.2737775146961212, -0.6374572515487671, 1.9322909116744995, -0.034715138375759125, 0.6586096286773682, 0.6063557863235474, 1.0808720588684082, -0.911363959312439, 0.608424961566925, 1.9794782400131226, -0.35513314604759216, 0.03177337348461151, -0.24576327204704285, -1.5609474182128906, 0.07532462477684021, 1.826335072517395, -0.9749610424041748, -0.4069940447807312, 0.8444125056266785, -0.38245120644569397, -0.1157383844256401, -0.7863340973854065, 1.0368752479553223, -1.0021882057189941, -0.8868524432182312, -0.5149103999137878, 1.3784652948379517, 1.878701090812683, -0.16797253489494324, 0.8601852059364319, 2.5827383995056152, 1.214883804321289, 0.5528185963630676, -1.071024775505066, -1.0935488939285278, -0.09742359071969986, 0.7226366400718689, 0.753433346748352, 0.6489998698234558, -1.3269141912460327], [-0.5559860467910767, 0.9026747941970825, 1.847939372062683, -0.5997872948646545, 0.20651446282863617, 1.3144304752349854, 1.0922012329101562, 0.264759361743927, -0.7171360850334167, -0.004576528910547495, 0.13683414459228516, -1.5669602155685425, -2.1762242317199707, -0.326617568731308, -1.3346483707427979, -0.8993481397628784, -0.08944518119096756, 0.5200627446174622, -0.574501633644104, 0.2835707366466522, 1.2602039575576782, 0.386200487613678, 0.41911551356315613, -0.7378063797950745, 0.2028297781944275, 1.2370309829711914, -1.3371933698654175, 0.09444739669561386, 0.364641934633255, -0.17873765528202057, 0.5096559524536133, -1.8402833938598633, -1.189488410949707, 2.167576313018799, -0.30452844500541687, -1.0174833536148071, 0.4608880281448364, 0.9778062105178833, 0.4891843795776367, 1.2553244829177856], [0.9295943975448608, 1.1281052827835083, 0.15780286490917206, -0.9241790175437927, -0.495524138212204, 1.3890223503112793, -0.04685007780790329, 0.46792763471603394, -0.1703261286020279, 1.387450933456421, 0.5502406358718872, -2.067652940750122, -1.148400902748108, -0.8921011090278625, 1.1618155241012573, 1.6713365316390991, -0.4224424660205841, 0.04649841785430908, 0.463575541973114, -0.08872223645448685, -0.7259487509727478, -0.5107312798500061, 2.2371718883514404, -0.5495654344558716, -0.5428544282913208, 0.9850358963012695, 0.6661985516548157, 0.35585588216781616, -0.4162273108959198, -0.632625162601471, 0.9105641841888428, -0.12171360850334167, 1.0821584463119507, 0.00039026146987453103, 1.0364190340042114, -0.5021300315856934, 0.6839936375617981, 0.25070905685424805, 1.5640913248062134, 1.5301566123962402], [0.5457803010940552, 0.518609881401062, 0.0025572918821126223, -0.920051634311676, 0.5226525068283081, -0.6019607186317444, -2.018411874771118, -1.1955890655517578, 0.536644458770752, -1.3477920293807983, -1.0384467840194702, 1.5973769426345825, -0.5827511548995972, -0.9586367011070251, 0.856432318687439, 0.6474496126174927, -0.35612916946411133, -0.684002697467804, -0.640083372592926, -1.0240933895111084, 1.1350356340408325, 0.6382095217704773, 0.4162405729293823, 2.903069019317627, 0.18538792431354523, -0.7199545502662659, 1.043853998184204, -1.9387528896331787, 0.025453481823205948, 0.451205849647522, 1.0444865226745605, 0.2001810222864151, 0.8677982091903687, -1.2258752584457397, 0.3305976986885071, 0.6916252374649048, -1.5128178596496582, 0.05040311813354492, -0.6119596362113953, -0.004065188579261303], [-0.28125864267349243, -1.353285551071167, -0.30672019720077515, 0.3294171690940857, -0.7563528418540955, -0.4417758882045746, -0.3730677664279938, 0.9236100912094116, 0.7939465641975403, -1.1596095561981201, -0.03551016375422478, 0.1489894539117813, -0.23167604207992554, 0.039582252502441406, -0.9694807529449463, -0.2913375198841095, 0.8576592206954956, -0.39089834690093994, 0.2086479663848877, 0.738278865814209, 0.6081814765930176, 1.726683497428894, -0.4169676899909973, 0.12993383407592773, -1.10029137134552, -0.6305108666419983, 1.2446571588516235, -0.7890781760215759, 2.5220906734466553, 0.22886155545711517, 0.4459245502948761, -0.8366793990135193, -0.5561423301696777, -0.9123261570930481, -0.8632451295852661, 0.446188360452652, -0.7774487733840942, 2.7997756004333496, 0.4499501585960388, -0.21370014548301697], [0.9596097469329834, -0.7746561765670776, 1.0693519115447998, 1.2830722332000732, -0.27606329321861267, -0.0825953483581543, -2.387059450149536, 1.2728887796401978, 0.3665566146373749, 0.8389058709144592, -0.8234984874725342, 0.7866398096084595, 1.2587435245513916, 0.3172808289527893, 0.1914137303829193, 1.3412957191467285, -0.35898661613464355, 0.3322286307811737, -0.7395948767662048, -0.34829819202423096, 0.2906559109687805, 0.7327268123626709, -0.43339914083480835, -0.032088130712509155, -0.9819590449333191, -0.6183879971504211, -0.2813457250595093, 0.019394321367144585, -0.0032459774520248175, -0.4133109748363495, 0.9081853032112122, -0.45726868510246277, -0.584888219833374, 0.7654758095741272, -0.6389152407646179, -0.9001184105873108, -1.4736191034317017, -0.8791859745979309, -0.4903890788555145, 1.0700184106826782]], "dtype": "float32"}, {"__nd__": [[2.0850980281829834, 2.6727638244628906, 0.3668134808540344, -1.3328737020492554, -1.975158452987671, 1.872833251953125, -1.8000736236572266, -1.2157427072525024, -1.076509952545166, 0.8232683539390564, 0.10612107068300247, -1.9528816938400269, -0.393399178981781, -0.5055912733078003, 0.5888087749481201, 0.05895645543932915, -1.874473214149475, -0.18908511102199554, 0.8319835066795349, -0.0227140374481678, -0.1145421713590622, 0.16763941943645477, -0.8132068514823914, 0.6073718070983887, -0.3210306167602539, -1.6400835514068604, 0.7457373738288879, 0.8078919649124146, -1.5524039268493652, -0.29919424653053284, 0.3256857991218567, -0.32016968727111816, 0.9807774424552917, 1.763764500617981, -0.14336107671260834, 0.2381313443183899, 0.1929374486207962, -0.59837806224823, 0.4647541344165802, -0.8200318813323975, -0.6082965135574341, -1.1691399812698364, -0.21548520028591156, 1.2865101099014282, -1.249795913696289, -1.543552279472351, -0.2839665412902832, 0.18454323709011078, 0.3528023064136505, -2.187344551086426, 1.154404878616333, 0.8393645882606506, -1.3790169954299927, -0.38396984338760376, 0.9257561564445496, 1.5330836772918701, 0.031803201884031296, 0.11724888533353806, 0.40849265456199646, 0.6744706630706787, 0.9322047829627991, 0.21576137840747833, -2.0696938037872314, -0.0740073025226593, -1.8670004606246948, -2.2225489616394043, 2.154747486114502, 0.7767680287361145, 1.355428695678711, -0.5032809376716614], [1.3775016069412231, 0.4537794888019562, -0.07391732931137085, -0.0027761515229940414, -2.5370566844940186, 0.5079435706138611, -0.3319561183452606, -0.38803938031196594, 0.021959858015179634, 0.39355191588401794, -0.9190229773521423, 0.14420592784881592, -0.8836538195610046, -0.018996719270944595, 0.34692028164863586, 0.6577188372612, -1.8649884462356567, 0.7496073842048645, -0.23683977127075195, -0.43828853964805603, 0.745760977268219, 0.2134813368320465, 0.9349930882453918, -0.7231748700141907, -0.8711258769035339, -0.06866888701915741, -1.0149362087249756, 0.3288643956184387, 0.8170731067657471, -0.8747732043266296, -0.373084157705307, 1.2636908292770386, -0.13157138228416443, -0.337820440530777, 1.0569486618041992, -0.6279898881912231, 0.9932927489280701, -1.1451181173324585, -2.1122334003448486, -0.4192636013031006, 0.5147877931594849, 1.828912377357483, 0.02068641223013401, -0.7683727741241455, -0.07290423661470413, -0.3093319237232208, -0.26388436555862427, -0.513763964176178, -0.36411261558532715, -1.2846505641937256, 0.2885182201862335, 0.35197004675865173, -0.761770486831665, -0.1736295521259308, -0.39090949296951294, -0.9428280591964722, -1.359370231628418, -0.5196812152862549, -0.5272858142852783, 1.1005467176437378, 0.17424888908863068, 1.3651138544082642, -0.8838122487068176, 0.05320873111486435, -0.49330997467041016, 0.39334559440612793, -0.41013064980506897, 0.744485080242157, -1.6125855445861816, 0.14383380115032196], [0.4464554786682129, 0.8242036700248718, 0.8512179255485535, 0.8740065097808838, -0.9134808778762817, -0.16869518160820007, 0.5192636251449585, -1.023606300354004, 2.190321922302246, 1.14448082447052, 0.5350333452224731, -0.9667453169822693, -1.0580638647079468, -2.2010903358459473, 0.07798468321561813, 1.8597149848937988, -1.3805193901062012, 1.0879616737365723, -0.8143323659896851, 1.9490737915039062, 0.21196547150611877, -1.3072513341903687, -0.8261590600013733, 1.6460539102554321, 1.095367431640625, 0.09473760426044464, 1.4668688774108887, -0.005540103185921907, -0.3776092529296875, -0.0564529225230217, -2.1081724166870117, -0.6760526299476624, -0.11911032348871231, 0.02380002662539482, -1.1922467947006226, 1.8536449670791626, -0.8499791622161865, -1.2753762006759644, 0.6010007262229919, -0.5244473814964294, 0.016032187268137932, 0.15967923402786255, 0.0355568565428257, -2.1901443004608154, 0.2843003273010254, -0.5774217844009399, 0.1639580875635147, 1.2886680364608765, -0.1921708881855011, 0.5237479209899902, 1.2354230880737305, 1.3563247919082642, 1.365385890007019, -1.0583962202072144, 0.27658501267433167, 0.8486860990524292, -3.467604637145996, -0.5924422144889832, -1.9868556261062622, 2.156107187271118, 0.4803948402404785, -1.826351523399353, 0.02019844576716423, 0.6968889832496643, 0.023632800206542015, -0.302543044090271, -0.32653045654296875, 0.5859342813491821, -0.8288165330886841, 0.2267707884311676], [2.226658821105957, -1.605089783668518, 0.08036268502473831, 1.9836400747299194, -0.0530085563659668, -1.310105800628662, 0.36093443632125854, 0.11409059911966324, -1.73443603515625, -1.0760666131973267, -1.4950202703475952, -0.5012762546539307, 2.1305816173553467, 1.3989518880844116, -1.1373882293701172, -0.22216686606407166, -2.3960001468658447, -0.19529388844966888, -0.6102783679962158, 0.08616728335618973, 1.2731072902679443, -0.1745033860206604, 0.16704805195331573, -0.10755216330289841, 0.013703415170311928, 1.8950449228286743, 0.3510226011276245, 0.048544567078351974, -0.11732892692089081, -0.5809670686721802, -0.01846781000494957, -0.34581202268600464, 0.6244943737983704, 0.0072934431955218315, -0.854189932346344, 0.7242521047592163, -2.129606246948242, 0.5472646355628967, -0.34681475162506104, -0.7433812022209167, 1.0443171262741089, 0.9320998787879944, -0.08689213544130325, -0.22881434857845306, -1.0376205444335938, 0.846889078617096, -2.121450901031494, -0.5761076211929321, -2.379081964492798, -0.12556162476539612, -0.6995493173599243, 1.5498688220977783, 0.18638651072978973, 0.30179670453071594, 0.804990291595459, -1.1689027547836304, 1.1097909212112427, 1.3019022941589355, 0.2532607614994049, 0.9389157891273499, 0.07110065221786499, -2.488090991973877, 1.3930020332336426, 0.7555659413337708, -1.2545355558395386, -0.1253974735736847, 0.02849642001092434, -1.2646883726119995, 1.2747331857681274, -0.6356675028800964], [1.351024866104126, 0.4419185519218445, 1.3215892314910889, -0.4457889795303345, 1.3230561017990112, 0.2882055640220642, 0.8752288222312927, -0.018561724573373795, -0.44052669405937195, 1.1700682640075684, 0.4812484681606293, -0.7489339709281921, -0.1536409854888916, 0.9044121503829956, 0.1388629823923111, -0.31486931443214417, 0.6563006043434143, -1.726969838142395, -0.7915902137756348, -1.1388881206512451, -0.9209718704223633, -0.20167399942874908, 1.3545230627059937, -0.5080669522285461, -1.0542163848876953, -0.9672793745994568, -0.5869001150131226, -0.5214324593544006, 0.7549601197242737, -1.4764807224273682, -0.550322949886322, 0.13942185044288635, 1.2102090120315552, 2.091555595397949, -0.4610937535762787, 0.35064610838890076, -0.3627181947231293, -1.5216481685638428, 1.055195689201355, 0.1891162097454071, -0.9747977256774902, -0.8400007486343384, -1.1813147068023682, -0.10593412816524506, 1.4403115510940552, 0.6467636823654175, -0.6159198880195618, -0.9748803973197937, -0.2895357012748718, -0.3510175049304962, -0.9283972978591919, -2.2037129402160645, -0.3960105776786804, 0.7686980962753296, 1.6738855838775635, -0.42049428820610046, -0.03958633914589882, 0.7715886831283569, -0.5561782121658325, 0.7823039889335632, -1.5745174884796143, -1.2202110290527344, -0.8752236366271973, 0.062218137085437775, -0.30849355459213257, 0.328181654214859, -0.03964166343212128, 1.0172048807144165, 1.3897860050201416, 0.6965205669403076], [-0.41320106387138367, 0.6417480111122131, 0.8038228154182434, 0.048151396214962006, -0.12069276720285416, 0.4979976713657379, 0.29296064376831055, -1.7260934114456177, -1.397423505783081, -1.8475488424301147, 0.6875718235969543, 0.7161420583724976, -0.6102203130722046, 0.6980676054954529, 0.17975136637687683, 0.5254415273666382, 0.5096674561500549, -0.11407601833343506, 0.6303573250770569, -0.1585003137588501, 0.4303087592124939, 0.8165830969810486, -0.31781721115112305, -0.09671612083911896, -0.35507625341415405, -0.649226725101471, 0.13535068929195404, -1.7858892679214478, -2.0493874549865723, 1.7938439846038818, -0.9502183198928833, 0.7404763698577881, -0.9561804533004761, -0.22176308929920197, 2.4788551330566406, -1.9021482467651367, 0.8061060905456543, 0.5315218567848206, 0.8200269341468811, -0.22452548146247864, 0.18059778213500977, 0.2501384913921356, -0.8308793902397156, 0.9407126307487488, -1.0608420372009277, -2.222834348678589, 0.4422459006309509, -0.5792306661605835, 1.3595319986343384, -0.15013886988162994, -2.015510320663452, -0.7601553201675415, 0.46063899993896484, -1.2026278972625732, -1.0211831331253052, -0.10145614296197891, 1.6367931365966797, -1.2364848852157593, 1.2654300928115845, 0.8289097547531128, 0.2386673390865326, -0.9513953328132629, 1.4042999744415283, 0.4677097201347351, 1.14752197265625, 0.21006512641906738, 0.013359091244637966, 0.20079956948757172, -0.652015209197998, -1.2317901849746704], [-0.7738243937492371, -2.149587869644165, 1.9284344911575317, 0.2781268358230591, -1.5921083688735962, 0.29261305928230286, -0.7054240703582764, 0.9347184896469116, 1.2246809005737305, -1.1098086833953857, 0.8649235367774963, 1.2565076351165771, 1.3990942239761353, 0.3965991139411926, -0.42187973856925964, -0.35499435663223267, -0.7307186126708984, -0.4291057288646698, -0.9675933122634888, 0.447100430727005, -1.3265180587768555, -2.6558127403259277, -1.3451647758483887, -0.038454569876194, 0.539569616317749, -0.2072206735610962, 0.6081919074058533, -0.4204123318195343, 0.7286632061004639, -0.7176520228385925, -0.7419347167015076, -0.15919972956180573, -1.1769062280654907, -0.4954063594341278, -0.20434680581092834, 0.5588781833648682, -0.16039647161960602, 0.35775262117385864, -1.006560206413269, -0.6368533968925476, -0.24407000839710236, -1.6096160411834717, 1.5302581787109375, -2.0818238258361816, -0.07056284695863724, -0.413961797952652, -0.5289332270622253, -0.23131144046783447, 1.2929526567459106, 0.6182065606117249, -0.4897776246070862, -0.1393800973892212, 1.8085201978683472, 0.6220486760139465, 1.032950758934021, 1.3460092544555664, -1.2928622961044312, 1.186232089996338, 1.2781156301498413, -1.766940951347351, 0.9406410455703735, 0.6955577731132507, -0.7860601544380188, -1.5319355726242065, 1.064923882484436, 0.7014485597610474, -0.20528803765773773, 1.0955358743667603, 1.308656930923462, -0.8783389925956726], [0.6448330283164978, 0.7327595949172974, 0.9820383787155151, -1.5926709175109863, -0.18976998329162598, -2.1149697303771973, 0.11593158543109894, -0.42151758074760437, 0.05774121358990669, -1.5339505672454834, 0.36580023169517517, -0.35113903880119324, -0.30835652351379395, 1.448643684387207, -0.5980699062347412, 1.8098667860031128, 0.4675021469593048, -0.28778889775276184, -1.4930334091186523, -0.6240761876106262, 0.7440525889396667, -0.13558782637119293, -0.3464633822441101, -1.5475008487701416, -1.2276469469070435, 0.18657661974430084, 1.367075800895691, -0.3081885576248169, -1.3142844438552856, 0.35740432143211365, 1.043432593345642, 1.397399663925171, 0.27179625630378723, -2.0742526054382324, -1.8981562852859497, 2.838268280029297, 0.683628261089325, 0.41719913482666016, -0.4127011001110077, 0.3952673077583313, -0.5046959519386292, -0.7047301530838013, -1.6730629205703735, -0.7127963304519653, 0.5007957220077515, -0.6949547529220581, 0.6166484355926514, 0.14676032960414886, -0.23575738072395325, 0.11054803431034088, -0.031940020620822906, -0.022563129663467407, -1.7154089212417603, -0.3411325216293335, 0.7000875473022461, 2.0129880905151367, 1.0699135065078735, -1.967200756072998, -0.15714780986309052, -0.8753332495689392, -1.0806586742401123, 0.8091142773628235, -0.2833382785320282, -0.6733196377754211, 1.8472988605499268, 2.2328968048095703, 0.14081119000911713, -0.5095331072807312, -1.8094154596328735, 1.0969955921173096], [-0.39383187890052795, 0.8384184837341309, 0.7005000710487366, -0.6042508482933044, 0.14003023505210876, 0.854315996170044, -0.2955067455768585, 2.0858957767486572, 0.16115006804466248, 0.23296497762203217, 1.2026854753494263, -0.8811586499214172, -0.6491857171058655, -0.6035438179969788, 0.25287091732025146, -1.0871864557266235, -1.1090511083602905, -0.8663045763969421, 2.2877037525177, -1.360094428062439, -0.13344749808311462, 0.017439477145671844, -1.8361090421676636, -1.4295333623886108, 0.8745023608207703, -1.740095615386963, -2.8446156978607178, 0.18584004044532776, 1.076222538948059, 1.4809939861297607, 1.044053077697754, -0.06772755831480026, 0.731387197971344, 0.0012689536670222878, -1.0019729137420654, -1.3828635215759277, 1.4857234954833984, 2.372230291366577, -1.1484885215759277, 0.21108564734458923, 0.23627126216888428, -2.4533677101135254, 1.5296213626861572, -1.2025896310806274, 0.5845800042152405, 0.14016835391521454, -1.0278605222702026, -0.4816436171531677, -0.1831725686788559, -0.45607703924179077, 0.41328155994415283, 0.036625321954488754, -1.212277889251709, -0.31561392545700073, -1.64320707321167, 0.49899548292160034, 0.8642928600311279, -0.6844402551651001, -0.10132422298192978, -0.5059205889701843, 1.1837739944458008, 0.8151512145996094, -0.3458191454410553, 0.5330387949943542, -0.15408895909786224, -0.4135763347148895, 0.5008264183998108, -0.8057782649993896, 0.5142449140548706, 1.0426764488220215], [0.5127596259117126, -1.2017489671707153, 0.7229970693588257, 0.8211948275566101, 0.9599484801292419, 0.5496016144752502, -1.062652826309204, 0.2879432737827301, 0.12725333869457245, 0.1714269369840622, -0.15207624435424805, 0.007268228568136692, -2.047072649002075, 0.10115447640419006, 0.2269585132598877, -1.0355783700942993, -1.2654038667678833, 0.28528833389282227, 0.41993096470832825, -0.2060178965330124, -0.6502509713172913, 1.5688457489013672, 1.1230862140655518, 0.5951850414276123, 0.8497157692909241, -0.5868121385574341, 0.8080899119377136, 0.38101017475128174, -0.8768452405929565, -0.21919018030166626, 0.26526790857315063, -1.215962290763855, 0.2394786775112152, 0.06995191425085068, 0.266370564699173, -0.5412341952323914, 1.1735153198242188, 2.8225724697113037, 0.7196919322013855, 0.20374520123004913, -0.7841286063194275, 0.3705730438232422, -0.16245409846305847, 0.10330215841531754, 1.1738697290420532, -0.24721047282218933, -0.12085007131099701, -0.4139300286769867, 0.31758585572242737, 0.3470650315284729, -0.47694674134254456, -0.18511086702346802, 0.5330585241317749, -1.499718189239502, -0.855362057685852, 1.7436904907226562, 2.445511817932129, -1.388648271560669, -1.33440101146698, -1.631497859954834, -0.982694685459137, 0.15229591727256775, 0.018442116677761078, 0.7919120192527771, -0.29383641481399536, -1.2623523473739624, 0.21276652812957764, 1.8547102212905884, 0.40489745140075684, -0.36171984672546387], [-1.111391544342041, 1.7399797439575195, 1.70579195022583, -0.3597234785556793, -1.1644713878631592, 0.13186873495578766, 0.4068291485309601, 0.5515672564506531, -0.2844773828983307, 1.3498626947402954, 0.4398581385612488, 0.564218282699585, -0.3484295606613159, 0.21691463887691498, -0.5585595965385437, -1.6047255992889404, 1.3881269693374634, 1.2066398859024048, -0.3715479373931885, 1.6871544122695923, -0.35336989164352417, 0.051080476492643356, -0.024962861090898514, -0.6507844924926758, 1.8332369327545166, -0.067175954580307, 1.3004206418991089, -1.0599461793899536, -1.2316213846206665, 1.1888339519500732, -0.4319838583469391, 0.8091236352920532, -1.7251646518707275, 1.7963933944702148, 1.6185619831085205, 0.8129748106002808, -0.12960337102413177, -1.8326106071472168, -0.6797130107879639, 0.8956472277641296, 1.758044958114624, -0.35186144709587097, 0.008265255019068718, 0.3679424226284027, -0.08503264933824539, 2.0626440048217773, -0.0037014561239629984, 1.5316542387008667, -0.2714602053165436, 1.1085506677627563, 0.33795514702796936, -0.9392845034599304, 0.3358834981918335, 2.2750375270843506, 0.6724454164505005, -0.40251556038856506, -0.23360300064086914, 0.2410593181848526, -0.15750686824321747, -0.6917137503623962, -1.161821961402893, -1.2548906803131104, 2.480722427368164, 0.2739003002643585, -2.3441336154937744, -1.4346213340759277, 0.5810766220092773, -1.9163459539413452, -0.5919109582901001, 0.170929953455925], [0.2760136127471924, -0.3984447121620178, -0.1184661015868187, -0.9499393701553345, 1.2105165719985962, -1.626306176185608, -0.9348533749580383, -0.7881023287773132, 0.3968943953514099, 0.3457278609275818, -1.2533220052719116, -0.7878350615501404, -1.3115222454071045, 0.1591726541519165, -0.38672012090682983, 0.6603729128837585, -0.7048208117485046, -0.24566389620304108, 0.09780246019363403, 1.8303014039993286, 0.39419567584991455, 0.7767348289489746, -0.37408143281936646, 0.5416340231895447, -0.9731485843658447, 0.48466479778289795, 0.20590521395206451, 0.5483015775680542, 0.23035088181495667, -0.560158371925354, 0.5837802886962891, 0.3141638934612274, -0.8258086442947388, -1.4860731363296509, 0.33076027035713196, -0.457311749458313, 0.05944797396659851, 0.555932879447937, -0.3585776090621948, -0.40757548809051514, -0.23279564082622528, -1.5462478399276733, 0.8542478084564209, -0.07099601626396179, 0.8178693056106567, 0.29092955589294434, -0.3497565984725952, 1.1652796268463135, 0.4701234996318817, 0.46837183833122253, -0.4853198230266571, 0.52093505859375, -1.4238739013671875, 1.4190378189086914, 2.5224626064300537, -1.0963730812072754, 0.15621578693389893, 0.6982500553131104, -1.8039746284484863, 0.20225924253463745, 0.33694055676460266, 0.19296976923942566, 0.09625782817602158, 0.24697963893413544, -0.6870889663696289, -0.8146517872810364, 0.7177279591560364, 0.786690354347229, 0.21649393439292908, -0.7802314758300781], [0.28943026065826416, 0.5572245717048645, 0.954452395439148, 0.7735992670059204, -1.314761996269226, -0.93638014793396, 0.049034491181373596, -0.27890077233314514, 1.5404211282730103, 0.07808845490217209, 0.0868656262755394, 0.05312669277191162, 0.6456570625305176, 0.027791501954197884, -2.3148481845855713, 2.0349485874176025, 0.23966829478740692, 0.42835018038749695, -0.6051558256149292, -0.0038612426724284887, 0.4194839894771576, -0.8096935749053955, 0.9061692357063293, 0.34585434198379517, -3.2650675773620605, -0.6528272032737732, -0.1539628654718399, -0.4163820445537567, 0.7206942439079285, -0.3033151626586914, -0.1901128888130188, -0.8473642468452454, -0.3509562313556671, -0.05822982266545296, 0.31801092624664307, 2.025122880935669, 1.4632713794708252, 1.0967451333999634, 0.5562528967857361, 0.6618943810462952, -0.12408585101366043, 2.8831145763397217, 0.2890775501728058, -0.522932231426239, 1.0620992183685303, -0.3692452013492584, 1.0750157833099365, 0.6828508377075195, 1.5610642433166504, -1.4352010488510132, -0.26361480355262756, 0.11447491496801376, -0.052481915801763535, 0.4994382858276367, 0.7039594650268555, 0.2937205731868744, 0.7063030004501343, -1.4193435907363892, -0.4110919237136841, 1.7407938241958618, 0.027225598692893982, -1.0186245441436768, 1.0310698747634888, -0.7134324908256531, -0.9132281541824341, 0.962261438369751, 0.8858211636543274, -0.2538401484489441, 1.7809805870056152, 0.21704646944999695], [0.49769166111946106, 0.14644575119018555, 0.8683226704597473, 0.5347753763198853, 0.6045018434524536, 0.9449782967567444, -0.41262009739875793, -0.37936997413635254, 3.3844683170318604, -0.04727819934487343, 0.23254601657390594, -0.5442861318588257, 0.9605587124824524, 1.3005222082138062, 0.6727029085159302, -0.5361359119415283, 2.1609458923339844, -0.2739124000072479, -0.12664546072483063, -0.33568477630615234, -0.7400486469268799, -0.6552751660346985, 1.6334185600280762, 0.7532209753990173, -0.3351599872112274, 0.3841952085494995, -0.9331059455871582, 0.29098376631736755, 0.016071751713752747, 2.4093379974365234, -0.4387708008289337, -0.4841739535331726, -1.3081748485565186, 0.39270228147506714, -0.19507189095020294, 0.061372555792331696, -1.8018254041671753, 1.5362253189086914, -0.34923285245895386, 0.9069385528564453, 0.40931040048599243, 0.42970359325408936, 0.660942792892456, 0.6753584146499634, 1.2904350757598877, 0.6973778605461121, -2.2147159576416016, 0.03603136166930199, -1.179612398147583, 0.5501375794410706, 1.9984681606292725, -0.4585285484790802, -1.6222989559173584, -0.622413694858551, -2.0964701175689697, 0.8194953203201294, -0.6315962076187134, -1.3845382928848267, 0.4468449354171753, 0.09535059332847595, 0.3875683844089508, 0.5680106282234192, 0.5095880031585693, 1.7758762836456299, -0.3950616419315338, 1.2547121047973633, -0.33301252126693726, 0.9645686149597168, -0.8052039742469788, -0.32508739829063416], [-2.3609931468963623, -0.28935110569000244, -0.35256075859069824, -0.13472773134708405, -0.9918180704116821, -0.43689319491386414, -0.47242841124534607, -1.040303111076355, -1.1281064748764038, -0.6255194544792175, 0.895254909992218, 1.6666253805160522, -0.0795164406299591, -2.5551235675811768, -0.026164861395955086, 0.601041853427887, -1.3783951997756958, 0.1744520664215088, 0.19216963648796082, 1.2862948179244995, -0.7525246739387512, 1.6358627080917358, 0.4422389268875122, -0.6266348958015442, -0.6869161128997803, 0.12049148976802826, 0.9294725060462952, -0.10347910225391388, -0.7835367321968079, -0.9964900016784668, -1.2607636451721191, 0.887509286403656, 0.9954971671104431, -0.9502460956573486, 0.011576410382986069, -2.2391223907470703, 2.5003085136413574, 1.1774200201034546, 0.2238454669713974, 1.019740104675293, -0.19390814006328583, 0.5418007969856262, -0.055931609123945236, -0.5964322686195374, -1.3390192985534668, 1.4117034673690796, 0.47758424282073975, -0.3889204263687134, 0.6106300354003906, -0.4562484323978424, -0.38643428683280945, -0.5748535990715027, 0.4056434631347656, -0.5990669131278992, -0.9185624122619629, -1.8013907670974731, -0.9520278573036194, 0.32421791553497314, -0.40875062346458435, 1.4599610567092896, 1.1979167461395264, -0.8049061298370361, -1.8020983934402466, 1.182795524597168, 0.7273520827293396, 0.012033252976834774, -1.1566340923309326, 0.2548290193080902, 0.4152505397796631, -0.17475122213363647], [-0.03973911330103874, -0.10920204222202301, -1.4839515686035156, 0.9470715522766113, 1.1201486587524414, 0.2444034069776535, -0.6712763905525208, -1.3282897472381592, 3.192432165145874, -0.5392500162124634, -0.3495247960090637, -0.6480036973953247, -0.1075221598148346, 2.622624158859253, -0.32713890075683594, 0.46045994758605957, -0.3198500871658325, -0.622113823890686, -0.546775221824646, -0.14415860176086426, 1.2115232944488525, 0.03903697431087494, 0.19101770222187042, 0.33918875455856323, 0.42132681608200073, 0.2732911705970764, 0.5550609827041626, 1.0864765644073486, 0.36792999505996704, -0.7352631688117981, -0.5909485816955566, -1.913561463356018, 1.1378284692764282, 0.4762137234210968, -0.3792227506637573, -1.3636873960494995, -0.06228348985314369, -0.5244929194450378, 0.12489921599626541, -0.6314524412155151, -0.2974832355976105, -0.08534348756074905, -0.979587197303772, -0.581879734992981, -0.2733287215232849, 0.546602189540863, 0.6315075755119324, -1.2371339797973633, -0.7291518449783325, -1.5682286024093628, -2.639827251434326, -0.9921295642852783, 0.6288851499557495, 1.9062002897262573, 1.9182089567184448, 0.3737029433250427, 1.0577747821807861, 1.024393916130066, 0.31028661131858826, -0.29837971925735474, 1.093001365661621, -0.3551842272281647, 0.2071915864944458, 0.05026599392294884, -0.22419989109039307, -0.5026038885116577, 0.1772446483373642, -0.390140563249588, -0.1809779703617096, 1.1866328716278076], [2.258291244506836, 2.084818124771118, -0.18759846687316895, -1.554081916809082, -1.010676383972168, -0.7212377786636353, 0.5685429573059082, 1.3371546268463135, -0.5673259496688843, -0.4228360950946808, -0.33073440194129944, -0.4391044080257416, 0.11206324398517609, -2.2055504322052, 0.05033137649297714, -2.3813161849975586, -3.5429675579071045, 0.5098833441734314, 1.6506197452545166, 1.7737298011779785, -0.9215187430381775, -0.30532804131507874, 1.4006541967391968, 1.192572832107544, 0.35457679629325867, 1.5741479396820068, 1.336687445640564, -0.14985468983650208, 0.2708500325679779, -0.005228765774518251, 0.3567679226398468, -0.23374386131763458, -0.08167829364538193, 0.08645781874656677, 1.1564966440200806, 0.1905835121870041, 1.6894603967666626, -0.3209328055381775, -0.529151439666748, -0.8376772999763489, 0.5622466802597046, 1.02858567237854, 2.4938387870788574, -0.03523929789662361, 1.1997123956680298, 2.3267345428466797, 0.9619269371032715, -0.07346519827842712, -0.39196300506591797, -2.3698954582214355, 0.4960106611251831, 0.3935980796813965, -0.08900552988052368, 0.5945647954940796, 0.4221624433994293, 0.23348668217658997, -0.011260943487286568, -1.120560884475708, 0.07257089763879776, 0.5039306282997131, -0.38201457262039185, 0.2358212023973465, 0.22947779297828674, 1.6445151567459106, -0.545268177986145, 2.5012242794036865, 0.5262479186058044, 1.300431251525879, -0.39257073402404785, -2.154600143432617], [1.33756685256958, -0.6842649579048157, 0.008463931269943714, -1.498746395111084, 0.47545313835144043, 1.9191614389419556, 0.28416141867637634, -1.4522253274917603, -1.116978406906128, -0.021894706413149834, 0.9942949414253235, -0.36599424481391907, 1.3564225435256958, -1.527300238609314, 0.3591661751270294, -1.2054948806762695, -0.691754937171936, -0.34775400161743164, 0.4914230704307556, -1.1264828443527222, 1.4063698053359985, -1.099577784538269, -0.23702624440193176, -0.18977361917495728, 1.356138825416565, 0.6011810898780823, 0.9531124234199524, 0.3494047522544861, -0.5745034217834473, -0.7130845785140991, -0.838853657245636, 1.127086877822876, -1.233534574508667, -0.8642869591712952, -0.3226979672908783, -0.4645378589630127, -0.3750862181186676, 1.2429739236831665, -0.6500008702278137, -0.41570910811424255, -0.7678437829017639, -0.5812945365905762, 1.3052688837051392, 0.2139451950788498, -1.0491095781326294, 0.8559850454330444, 1.5544297695159912, 0.3976323902606964, -0.2642357349395752, -0.7426775693893433, 0.9963453412055969, -1.4302819967269897, 0.7706234455108643, 1.704546332359314, -0.07270032167434692, 1.1278681755065918, -0.18128594756126404, 0.7679879665374756, -0.08983474969863892, -1.2742445468902588, -0.031408537179231644, 0.42140448093414307, 0.7220390439033508, 0.6712164282798767, -0.09799414873123169, 0.1767456978559494, 0.8972131013870239, -0.8681837320327759, 0.08687976747751236, 1.6838151216506958], [0.04049953073263168, 0.6089691519737244, -1.9031401872634888, 2.003865957260132, -0.8130072355270386, -0.4460652470588684, 1.760499358177185, -0.8937281370162964, -0.30819350481033325, -2.1570277214050293, -1.1853773593902588, -0.10120408236980438, 0.04907361418008804, 0.2929381728172302, -0.7143484950065613, -1.839002013206482, 1.5416916608810425, -0.4096807539463043, -0.7374477386474609, 0.5746737718582153, -0.8173245787620544, -1.0269825458526611, 1.845281958580017, 0.6782819628715515, -0.3018801212310791, -0.6630101799964905, -0.11614498496055603, 0.3345932960510254, 1.1702576875686646, -0.7446957230567932, 0.10483676940202713, -1.4219449758529663, -0.03476995229721069, 0.20071114599704742, -0.14202100038528442, -0.49817201495170593, -0.4962832033634186, -0.4739629030227661, -0.28572964668273926, -1.857115626335144, 1.9394346475601196, 0.00511568458750844, 1.5743829011917114, -0.09239626675844193, -0.7926651835441589, -0.8436810374259949, 1.9090908765792847, -1.6296186447143555, 0.7964950203895569, -1.595828652381897, -0.19486117362976074, 1.1135172843933105, -0.008394564501941204, -0.6965034008026123, -0.7378952503204346, -0.7460854053497314, 0.4926929473876953, -0.7971093058586121, 0.9437264204025269, 0.9655888676643372, -1.7991596460342407, 1.4013442993164062, -1.74420166015625, -0.20490016043186188, 0.24137718975543976, 0.14870771765708923, 1.8088189363479614, 0.17048987746238708, 0.019199848175048828, 1.2235931158065796], [-0.1440751701593399, -0.06486578285694122, 0.2960744798183441, -1.4925462007522583, 0.7079954743385315, 0.5394129157066345, -2.155836582183838, 0.21119698882102966, -0.3823504149913788, 0.5242404341697693, -0.6945290565490723, 0.6439391374588013, 0.07151070982217789, 1.8071765899658203, 0.6453135013580322, -0.3406580686569214, -0.8524554371833801, -1.8805261850357056, 0.11566787958145142, 1.5989737510681152, -1.924802541732788, 1.4488462209701538, 1.5064315795898438, -0.9173319935798645, -0.47511541843414307, -0.1440081000328064, 1.4307729005813599, -0.6174241900444031, 0.20547513663768768, -0.27101802825927734, -1.0317671298980713, -0.492048978805542, -0.7799707055091858, 0.2227853685617447, 2.106384754180908, -0.1750839352607727, -1.0478142499923706, -0.43339264392852783, -0.9566906690597534, -1.4998506307601929, -0.12896767258644104, 1.0352646112442017, -0.6991934180259705, -0.3713527023792267, -0.10860953480005264, 0.17385107278823853, -0.9340203404426575, 0.9371576905250549, 0.14612433314323425, 1.1122575998306274, -1.9159784317016602, 0.1659604012966156, 0.9456046223640442, -0.00783211924135685, -1.56912362575531, 1.7291146516799927, -0.06700707972049713, 1.3952580690383911, -2.0508193969726562, 0.9374798536300659, -1.2146258354187012, -0.5775865316390991, -0.03056166134774685, 0.2464565634727478, 0.21057668328285217, -0.03622313588857651, 0.6236785650253296, -0.07422031462192535, 1.1866196393966675, -0.45603278279304504], [1.6123982667922974, -0.26193344593048096, -0.02246084064245224, 0.7933541536331177, -0.614436149597168, -0.20357467234134674, 0.9677976965904236, -1.4253708124160767, -0.361169695854187, -0.2307543307542801, -1.1645551919937134, -0.15940557420253754, 1.7641881704330444, -1.0947113037109375, -0.14803080260753632, -0.874274492263794, -2.3897125720977783, 0.8196143507957458, -0.8602834343910217, -0.7885820865631104, -0.18213917315006256, -0.5754677057266235, -0.007196344435214996, -0.2545405924320221, 0.13707438111305237, -0.6133846044540405, 0.6056404113769531, 0.2617022395133972, -0.49604660272598267, -0.6500841975212097, -0.6950111985206604, -0.12208587676286697, 1.2519975900650024, 0.6028143167495728, 0.851337194442749, -0.11651705950498581, 0.33096426725387573, 1.453383445739746, -1.3872302770614624, -0.3329983353614807, 1.0313091278076172, -0.5437090396881104, -2.05950665473938, 2.349867820739746, 0.25798219442367554, -1.469677209854126, 1.0964363813400269, -0.017848515883088112, 1.7155243158340454, 0.8224004507064819, 1.7763121128082275, -1.3731271028518677, -0.6095489263534546, -2.106318712234497, -0.9205277562141418, -0.9338206052780151, -0.2954706847667694, -0.5586636662483215, 1.8314611911773682, -0.21212846040725708, -2.394015073776245, -0.4111219346523285, 0.9101552963256836, -0.8629969358444214, -0.08548121899366379, -0.5782530307769775, 0.30402544140815735, 0.7049210071563721, -0.8063345551490784, 0.33550742268562317], [0.4934980869293213, -1.6836026906967163, 0.5531863570213318, -0.8320940732955933, -0.48952868580818176, 0.5290991067886353, -0.0333014540374279, 1.0293927192687988, 0.6947815418243408, -2.2108235359191895, 0.272339791059494, -1.7585190534591675, 0.4660826027393341, -0.5149959921836853, 0.6004643440246582, 0.30938220024108887, -0.1668969839811325, -2.379852771759033, -0.934755802154541, -0.14517533779144287, 1.0117796659469604, -0.4555515646934509, -0.504378080368042, -1.4339739084243774, 2.1576459407806396, -0.2929539680480957, -0.5601922273635864, -0.4764561653137207, -0.10892939567565918, -0.9054820537567139, 0.34343111515045166, -2.679811477661133, -1.1872543096542358, 0.20690880715847015, 1.0696624517440796, 0.5078275203704834, -0.09926766902208328, 1.600055456161499, -0.20570023357868195, -0.20481842756271362, -0.26238125562667847, -0.419754296541214, 0.04709396883845329, -0.8050892949104309, 1.0817383527755737, -1.1671409606933594, -1.297170877456665, 1.3912088871002197, 0.25007858872413635, 0.7883578538894653, -0.990673303604126, 1.246144413948059, -1.3939985036849976, 0.7515562772750854, 0.9735942482948303, -0.1892968863248825, -1.9053304195404053, -0.18174073100090027, 1.4591599702835083, -0.020416859537363052, -1.0898692607879639, 1.1625113487243652, 2.4017951488494873, 1.6441901922225952, 0.5885197520256042, -0.09249625355005264, -0.11928823590278625, 0.8446840047836304, 0.8051104545593262, 0.4376782476902008], [-0.42220234870910645, 0.6280560493469238, 0.4606426954269409, 0.8519623279571533, -0.9667774438858032, 2.4725000858306885, 1.7189282178878784, -1.474492073059082, -0.3568890690803528, 0.24704964458942413, 0.03379594534635544, -1.0598353147506714, 0.4696548879146576, 0.6294484734535217, -0.06258724629878998, -0.9508728384971619, 0.07408116012811661, 0.8823204636573792, 1.9756748676300049, 0.24186304211616516, 0.8613426089286804, 0.7670099139213562, 1.001837968826294, -1.2338956594467163, 1.7421764135360718, 0.6976189613342285, -1.1226422786712646, 0.27291956543922424, -0.5575341582298279, -0.2906922996044159, 0.6675100922584534, -1.1138683557510376, -0.007336026057600975, 0.15236559510231018, -0.48446527123451233, -0.48722973465919495, 0.790945827960968, -0.679418683052063, 0.8629192113876343, -0.08342543244361877, 0.9864001870155334, 0.7462831139564514, 0.5810195207595825, -0.19313795864582062, 1.2244408130645752, 2.4011125564575195, -1.2702523469924927, -1.32418692111969, 0.043983399868011475, 2.5173091888427734, -1.8272812366485596, -0.5172780156135559, -0.9411382079124451, 0.8316216468811035, -0.17097976803779602, 0.10073570162057877, -0.5711592435836792, -1.2858612537384033, 0.31414636969566345, -0.4680486023426056, 1.336600422859192, -0.7687689065933228, 0.6147221922874451, -0.16935226321220398, 0.14809001982212067, -0.10860789567232132, 1.0769340991973877, -1.6823523044586182, -0.5788229703903198, 0.8050779104232788], [-1.0657830238342285, 0.1178336814045906, 0.6880485415458679, 1.0754780769348145, -0.1387188583612442, 0.15714557468891144, -0.41369470953941345, 0.5552850365638733, -0.6433817744255066, -0.08671879023313522, -0.2120666801929474, -0.09002033621072769, 1.7304271459579468, -0.6005355715751648, -0.11141137033700943, 1.4105685949325562, -0.6935920715332031, 0.08270400017499924, -0.0857856273651123, -1.2546610832214355, -0.46917274594306946, -1.7624337673187256, 0.5339840054512024, 0.6288599371910095, 0.21194815635681152, -0.8013015389442444, -0.29011455178260803, 0.5883454084396362, 1.5655858516693115, 0.40832263231277466, -2.020433187484741, -0.3188422918319702, 1.1343690156936646, -0.8923438787460327, 0.006932623218744993, 1.0116032361984253, 0.7404350638389587, -1.698319435119629, -0.689485490322113, 1.203761100769043, -0.2578622102737427, -0.6563969254493713, -0.2915245592594147, 0.24298593401908875, 0.3128099739551544, 1.051524043083191, 0.37820640206336975, -0.6528393030166626, 0.6684989929199219, 0.6036516427993774, 0.2879334092140198, 0.6441166400909424, -0.508413553237915, -0.912264347076416, -0.2651650905609131, 1.317600131034851, 1.1075704097747803, 1.1395986080169678, 1.1421340703964233, -0.6621925234794617, 0.510866641998291, -0.44834133982658386, -0.39990147948265076, -0.4432697594165802, 1.4361281394958496, -0.201649010181427, 1.042862892150879, -0.5600792169570923, -2.7840306758880615, -0.1932140588760376], [-2.5527307987213135, 0.33472198247909546, -0.1668413132429123, -0.5693341493606567, 0.15144960582256317, -0.9127938747406006, -2.4038259983062744, -0.3789249658584595, -1.7637548446655273, -0.6227508783340454, -0.18206770718097687, -0.2809329032897949, -0.7952303290367126, 0.6290260553359985, 0.6906354427337646, -0.5424806475639343, 0.5121368169784546, 0.37611812353134155, -1.0136951208114624, -0.5207625031471252, -0.40320825576782227, -0.8802111744880676, -0.019711386412382126, 0.43310534954071045, 0.4528869390487671, -0.2768307328224182, 0.3850143849849701, -2.182095527648926, 0.043105531483888626, 0.5284362435340881, 0.0782979428768158, -0.06815361231565475, -0.4106152355670929, -0.5546054840087891, -0.17821529507637024, -0.4238491356372833, -2.0083730220794678, -1.768474817276001, -0.5471705198287964, -0.6218112111091614, 0.38724493980407715, 0.4772305488586426, 0.16943879425525665, 0.7475669980049133, 0.889835774898529, 1.9359766244888306, 0.6142483353614807, -1.3484138250350952, -1.2357081174850464, 2.5169284343719482, 0.25539830327033997, 0.7937415242195129, -0.4008444547653198, 0.4045107364654541, -1.4360703229904175, -1.601118564605713, 0.34475743770599365, 1.2330012321472168, -0.23561686277389526, 0.32682961225509644, -0.3881165087223053, 1.2328521013259888, -0.10824783891439438, -1.3588223457336426, 2.2728195190429688, -0.9097006916999817, 1.3241336345672607, 0.042633213102817535, 0.1822144240140915, 0.4353739619255066], [-0.14502887427806854, -0.6753343939781189, -0.5628121495246887, -0.022623280063271523, -0.7760487198829651, 1.177900791168213, -1.42164945602417, 0.6728270649909973, -0.6471058130264282, 0.5255551338195801, -0.517952561378479, 0.8071964979171753, -1.1356168985366821, 1.5000650882720947, -0.2728368043899536, -1.0578091144561768, 0.6304627060890198, -1.1237024068832397, -0.5460718274116516, 1.3200799226760864, 0.6684293150901794, -0.2990837097167969, -0.699491024017334, 0.523946225643158, -0.9087852239608765, -1.2663458585739136, -1.6101115942001343, 0.91238933801651, 0.4573710262775421, 1.1058157682418823, 0.05605659261345863, 1.3217910528182983, -1.2328081130981445, -0.5053418278694153, 1.1566417217254639, 1.1285005807876587, 0.28367599844932556, 0.5985116958618164, -0.212117537856102, 0.0437912717461586, 1.0643181800842285, -1.7637207508087158, -0.3966217637062073, -0.0628429725766182, 0.33314183354377747, -0.9063357710838318, 2.0369913578033447, -1.1061519384384155, 0.06519834697246552, 0.6842378377914429, 0.7526978254318237, 0.9526932835578918, -0.6658565402030945, 0.06648173183202744, 0.36821192502975464, -0.06835149973630905, -0.06076599285006523, -0.3804909884929657, -0.21753908693790436, -0.7286021709442139, 1.0072438716888428, -0.9548628926277161, -0.2399837076663971, 0.3776617646217346, -0.4989333152770996, -0.24854716658592224, 1.6066550016403198, 0.20647092163562775, 1.1743139028549194, 0.1475559026002884], [2.2796247005462646, 0.5564283728599548, 0.5056384205818176, 0.2245904505252838, 0.6813902258872986, -0.18801477551460266, -1.8589617013931274, -1.0215167999267578, -2.8348183631896973, -2.0644733905792236, 2.0344114303588867, 1.1598665714263916, 2.5011487007141113, -0.7682463526725769, -1.1100612878799438, 0.4468945264816284, 0.27833592891693115, 0.5219970345497131, -0.9131054282188416, -1.536738634109497, 1.0190349817276, 0.8302682638168335, 2.4334323406219482, 0.116155244410038, 3.296583890914917, 0.04085089638829231, 0.14003099501132965, 0.3295910656452179, 0.6420074105262756, -1.5967093706130981, -0.885112464427948, -0.6535990834236145, 2.4378762245178223, -1.0908697843551636, -0.6392540335655212, 0.8886602520942688, 0.942479133605957, -1.0680162906646729, 1.9644780158996582, 0.11881304532289505, 1.105741024017334, 0.991860032081604, 0.5347906947135925, -0.5409600138664246, -1.5906994342803955, -0.8863663673400879, 0.3145420253276825, -1.184556007385254, 0.5981714725494385, -0.9303840398788452, 0.7018967270851135, -0.2000168263912201, -0.5177063345909119, -0.005377970170229673, -0.1496669352054596, 0.4575062692165375, -0.21806295216083527, -1.5416975021362305, 0.7798487544059753, -0.3161563575267792, -2.9996724128723145, 1.1771154403686523, 1.1720691919326782, -1.42130708694458, -0.6114386320114136, -0.018857182934880257, 1.4498953819274902, 0.8993534445762634, -0.9685867428779602, -0.28276732563972473], [0.7314795255661011, -1.6019634008407593, 1.385502576828003, -0.5689979791641235, 0.2854747772216797, 1.2966066598892212, -0.8381891250610352, 0.17500261962413788, 0.006027581170201302, 0.8720093965530396, -1.0846573114395142, 0.1683686077594757, 0.32276177406311035, -0.38410481810569763, 0.5920903086662292, 1.5063270330429077, 0.6080256104469299, 0.7853367924690247, -1.582865595817566, -0.7175677418708801, -1.3542646169662476, 1.1495567560195923, 0.3745271563529968, 0.03599359840154648, 0.6191831827163696, -1.2620059251785278, 0.4026302695274353, -0.527283787727356, -1.026094675064087, 0.6835550665855408, 0.5356741547584534, -1.8444877862930298, -0.23532740771770477, 1.5992300510406494, 1.2757439613342285, 0.21113061904907227, 0.21292538940906525, 1.1811269521713257, -1.621106743812561, 0.6169980764389038, -0.2825905978679657, 0.0328470803797245, -1.2628110647201538, 1.7319976091384888, 0.952045202255249, -1.4987531900405884, -0.4840964078903198, -0.196431502699852, -0.7327600121498108, -0.5087536573410034, 0.3281940221786499, -0.41925248503685, -0.4190903306007385, -0.9821982979774475, 0.7112870812416077, -0.04864524304866791, 0.36918187141418457, -0.41753289103507996, -1.664416790008545, -1.1104389429092407, 0.9294427037239075, 0.09185943752527237, 0.9925876259803772, -1.2020431756973267, -0.4934542775154114, -0.6199666261672974, -0.9893078207969666, -0.9172034859657288, -0.32199716567993164, -0.3881458342075348], [-0.47009795904159546, -0.36108601093292236, -1.3968324661254883, 1.4736343622207642, -0.0741904228925705, -0.43372154235839844, -0.6972594857215881, 1.1095844507217407, 0.03319726511836052, -1.0517191886901855, -2.4062070846557617, 1.9903373718261719, 1.1485300064086914, -0.14587071537971497, -0.4262557923793793, -0.07704517990350723, -0.028758326545357704, -1.3511680364608765, -0.2957921326160431, -1.987646222114563, -0.298019140958786, 0.11449426412582397, 1.1766167879104614, -1.5876508951187134, 1.323006272315979, -0.3718750774860382, -0.2654281258583069, -0.09851600229740143, -0.7546308636665344, -0.08641711622476578, 0.2749064564704895, -0.3840329945087433, -1.658310890197754, -0.8580965995788574, 1.0519920587539673, -1.467100977897644, 0.3593783974647522, 1.594077229499817, -0.08761713653802872, 0.19688762724399567, -0.342146098613739, 0.6674664616584778, 0.3407857418060303, -0.366608589887619, -0.29141268134117126, 2.143644332885742, 1.6157501935958862, -0.3560062348842621, -0.9199876189231873, 0.7677261829376221, 1.5042898654937744, -0.08333103358745575, -1.5493336915969849, -0.1944812685251236, 0.11702635139226913, 0.45875877141952515, 0.5397542119026184, -2.1401402950286865, 0.6504780054092407, 0.7611957788467407, 0.8778930902481079, -0.65944904088974, -0.003328672144562006, 1.0077170133590698, 1.1040465831756592, 2.0367271900177, -0.475725919008255, -1.8536057472229004, 0.49536243081092834, 0.6998342275619507], [-0.6383841037750244, 2.352841854095459, 0.7070732712745667, 1.918355941772461, 0.28254786133766174, 0.181104376912117, -1.066089153289795, 1.0335346460342407, 0.7301533222198486, -2.228980541229248, -1.1030865907669067, -2.1233315467834473, 0.05767300724983215, -1.1426228284835815, 1.4985899925231934, -0.18096663057804108, -0.5074804425239563, 2.3407552242279053, -1.5281121730804443, -0.09656616300344467, 0.3691624701023102, -0.3930843472480774, 0.18619076907634735, 0.8205493688583374, -0.6782922744750977, -0.45622631907463074, -0.7214857339859009, 0.014197789132595062, 1.049493432044983, 0.6707140207290649, 1.3234734535217285, 1.0261937379837036, 1.2632052898406982, -0.945482075214386, -0.10428670793771744, 0.34891393780708313, -0.3430003523826599, -1.2466931343078613, 1.6020289659500122, 0.5775207281112671, 0.8452780842781067, -0.08740496635437012, -0.9747108817100525, -0.30536237359046936, 1.929614543914795, 0.6016175150871277, 2.298915386199951, -0.1934422254562378, 0.41781482100486755, -0.08016592264175415, -0.935230016708374, 1.150066614151001, -0.7145572900772095, -1.8518192768096924, -2.3199501037597656, 2.846038341522217, 2.47430419921875, 2.144737720489502, 0.7553079724311829, 0.48922958970069885, 1.0311627388000488, 0.7393538355827332, 1.3591899871826172, -1.1574101448059082, -0.8072368502616882, -0.41575512290000916, -1.7197027206420898, -1.1508558988571167, 1.841310977935791, 1.8541069030761719], [0.29674065113067627, -1.5691885948181152, -0.4027461111545563, -1.5721129179000854, 1.1750625371932983, -2.4129810333251953, -2.0849103927612305, 1.7131956815719604, 0.15231648087501526, -0.4651757776737213, 1.706803321838379, -0.8817654848098755, 0.1421533077955246, 1.2613776922225952, 1.1758129596710205, 0.13567321002483368, 1.4719443321228027, 1.7200630903244019, 1.6434890031814575, 0.1649996042251587, -1.906849980354309, 0.7457695603370667, -0.535515308380127, 1.231110692024231, -1.6356066465377808, -0.7918795943260193, -1.042510986328125, -1.44212806224823, -1.7224966287612915, -0.13305003941059113, -0.30615219473838806, -1.3683863878250122, -1.845118761062622, 0.37490352988243103, -0.24637296795845032, -0.189091756939888, 0.4671521484851837, -0.5512995719909668, -1.0935777425765991, -0.722172737121582, 1.274505853652954, 1.0718426704406738, -1.4877359867095947, 0.38264453411102295, -1.8261682987213135, -0.04167108237743378, -2.153646945953369, -0.2049959897994995, -1.3005794286727905, -1.1254481077194214, 0.04774310812354088, -1.2163439989089966, 0.9300553798675537, -1.5894544124603271, -0.4297384023666382, -3.31866192817688, -0.5428176522254944, 1.0267412662506104, 0.5845681428909302, -0.021868843585252762, -2.073150396347046, 0.18572546541690826, 0.11695560812950134, 0.18210679292678833, 0.39689961075782776, -0.9673024415969849, 0.4656801223754883, 0.9441999197006226, -0.27983149886131287, -0.45341140031814575], [-1.5953530073165894, 1.0715177059173584, -0.12060504406690598, 0.39353683590888977, -0.06882255524396896, 1.0891073942184448, 0.3741447925567627, -0.7695006728172302, 0.3206258714199066, 1.2124370336532593, -1.0211156606674194, 0.07376037538051605, 0.5109211802482605, -0.22015637159347534, -1.9678125381469727, -0.31839001178741455, -0.3794640600681305, -2.0860862731933594, -0.1072680801153183, 1.2370827198028564, -0.018792303279042244, -1.2330862283706665, 1.2479811906814575, -0.09932219982147217, 0.23706082999706268, 0.5839447975158691, -1.368751883506775, -1.2576172351837158, 0.27972733974456787, 0.17707638442516327, -0.057150375097990036, 1.1795189380645752, 0.08150725811719894, 0.3324219584465027, -1.4584101438522339, 0.27845773100852966, 0.647488534450531, -0.7814022302627563, 1.8085064888000488, -1.2725934982299805, 1.0041025876998901, 0.24639791250228882, -1.0585336685180664, 0.7890924215316772, 0.7194098234176636, 0.5231953859329224, 1.3285479545593262, 0.2778262794017792, 0.4277343451976776, 0.5301883816719055, 0.6954048275947571, 1.122246503829956, 0.5208264589309692, -0.7264543771743774, 0.09232562780380249, -0.8818853497505188, 0.8273704051971436, -0.9278144240379333, 0.34638160467147827, -1.4764500856399536, 0.758579671382904, 0.14331857860088348, -0.49674686789512634, -1.508273959159851, -0.40597179532051086, 0.2410571426153183, 0.9164498448371887, -0.08942605555057526, 0.6071053147315979, -0.272225558757782], [-0.03998695686459541, -1.4240984916687012, 1.2999202013015747, 0.8563091158866882, -0.1813359260559082, 0.9492132067680359, -0.2406189739704132, 1.438356637954712, -0.7024730443954468, -1.7546781301498413, 0.7500275373458862, 0.1041160374879837, 0.4227138161659241, 0.6476534008979797, -0.878829836845398, 0.7897257804870605, 1.3265550136566162, 0.23966409265995026, 0.3275817632675171, 1.014807105064392, -1.1565945148468018, 1.2346611022949219, -1.0933159589767456, 1.6302120685577393, -0.17337459325790405, -0.2848508059978485, -0.1719411164522171, -0.056986674666404724, 1.717917561531067, 0.1160164624452591, -0.7337120771408081, 1.9104218482971191, 0.011295527219772339, -0.199677973985672, 1.094290852546692, 0.7045323252677917, -0.19077366590499878, 0.21981391310691833, 1.7948367595672607, 0.5166721343994141, -0.31168174743652344, 0.1780727207660675, 0.8508645296096802, -0.012376775965094566, -0.4462644159793854, 1.3086360692977905, -0.7247142791748047, 0.8466644883155823, -2.6653454303741455, -1.0006901025772095, -0.31510624289512634, -0.5575198531150818, 0.3401992917060852, 2.064213752746582, 0.7625582814216614, -0.3253912329673767, 0.09942755103111267, -1.3024741411209106, 0.5082759857177734, -1.1463775634765625, -0.1630144715309143, -0.24461308121681213, -1.5851385593414307, 0.31294456124305725, -0.05619794875383377, 2.385061264038086, -1.5681160688400269, 1.8931511640548706, 0.6656240224838257, -1.6675878763198853], [-0.33275237679481506, -2.9697630405426025, 1.2764793634414673, -1.9474986791610718, -0.37121105194091797, 0.4916924834251404, 0.8561903834342957, -0.4538915157318115, 0.4492330253124237, -0.9812260270118713, 1.291412353515625, 0.6067607402801514, -1.3116984367370605, -0.7351610660552979, -0.284533828496933, 2.0309839248657227, 0.9643575549125671, -1.8399949073791504, 1.837459683418274, -0.732742190361023, 0.3860560953617096, -0.15426769852638245, 0.4534689784049988, -0.28595614433288574, -0.8778231143951416, -0.006109577603638172, 0.03348139300942421, 1.2406398057937622, -0.9813918471336365, -1.0527762174606323, 1.1841530799865723, -0.1349211484193802, -2.0920557975769043, -1.061203956604004, -1.7381771802902222, -1.2321058511734009, -1.4646575450897217, 1.761276125907898, -0.8852013349533081, -1.1146186590194702, 2.6289920806884766, -0.49063751101493835, 0.419974148273468, -0.5169654488563538, -0.9442557692527771, 0.6283194422721863, 1.9375513792037964, -0.6144128441810608, -1.0179402828216553, 0.8296866416931152, 1.6951051950454712, -0.3631415367126465, 0.4591749310493469, -0.29004141688346863, -1.2635046243667603, -1.3431103229522705, 0.4486519992351532, 0.7034425735473633, 0.9123783707618713, -0.05411975458264351, -1.333357572555542, -0.11542432755231857, 0.816146731376648, -0.38661760091781616, -0.4496946930885315, 0.834000825881958, -0.9491202235221863, -0.7023788094520569, -1.0025341510772705, 1.0377601385116577], [-0.8048997521400452, -0.26946204900741577, 1.1457351446151733, -0.778425395488739, 1.1954505443572998, -0.6128268837928772, 0.11370459944009781, -1.3487194776535034, 0.7749868035316467, -1.55853271484375, -0.4154419004917145, -1.167047142982483, -0.6111412644386292, -0.8320674896240234, -0.480337530374527, -0.3052077293395996, -1.253352403640747, 1.8503278493881226, 0.1309313327074051, -0.8756036758422852, 0.25498202443122864, -0.45500174164772034, -1.537900447845459, -1.3020150661468506, 1.5371111631393433, 0.694943904876709, -0.47583794593811035, 2.3250982761383057, -0.3908231854438782, 0.5628311634063721, 1.0576192140579224, 0.07884260267019272, -1.343044638633728, 0.019969239830970764, 0.4734455347061157, -0.6000180244445801, 0.5139465928077698, -0.5913829803466797, -0.10209953784942627, 1.1275430917739868, -0.9911660552024841, 0.5884270668029785, -0.6454065442085266, 0.9478386044502258, 0.7278352975845337, -0.77677983045578, 2.362229585647583, 0.6374277472496033, 0.48092028498649597, 0.297683447599411, -0.4190966784954071, -0.8967857360839844, 0.6297850608825684, 1.0206648111343384, -0.11123280227184296, 1.382651448249817, 0.23791290819644928, -0.48433545231819153, -1.0472806692123413, -1.3668467998504639, 0.6519150137901306, 0.5540313124656677, -1.7215654850006104, 0.4473518133163452, -0.12331268936395645, -0.9495335817337036, -2.163816213607788, 0.6140503883361816, 0.45586222410202026, 1.1774406433105469], [-0.14428037405014038, 0.9348194003105164, 2.7604928016662598, -0.7984036803245544, -0.7818652391433716, 2.1959071159362793, 0.9356593489646912, -0.7877044081687927, -1.2982878684997559, -0.7553954124450684, -1.0601835250854492, 0.47903725504875183, 0.6573916673660278, -1.8584061861038208, 0.4527602195739746, 0.6982101798057556, 0.6773503422737122, 0.31275486946105957, 1.6923208236694336, 0.14804059267044067, 0.15789489448070526, -0.8072845935821533, -1.6362837553024292, 1.0412927865982056, -0.10539046674966812, -0.3820647895336151, 0.1995851993560791, -0.2670189440250397, 0.5713584423065186, -0.09005743265151978, -0.5359631776809692, 0.2071048468351364, 0.37369269132614136, -0.6401896476745605, 1.8969919681549072, 0.25342652201652527, 0.11338088661432266, 0.41908931732177734, -1.5317137241363525, 0.6405224800109863, -0.021210668608546257, -0.5670639872550964, -0.8593438863754272, -0.13155154883861542, -2.009763717651367, 0.6383091807365417, -0.05452568084001541, -0.7905638217926025, 1.1853781938552856, -0.8660032153129578, -1.1514426469802856, -0.3950519859790802, -0.510940432548523, -1.07916259765625, 0.26003411412239075, 1.0983896255493164, -1.369622826576233, -0.9987757802009583, -0.037695303559303284, -0.928203821182251, 0.5687654614448547, 1.1224277019500732, -1.5949245691299438, 0.9421014189720154, -0.7616243958473206, 0.45742735266685486, 1.5198205709457397, -0.8618493676185608, -0.7337480783462524, 0.11037804186344147], [-2.397109270095825, -0.07785163819789886, 0.15084350109100342, -0.7562969923019409, -1.5702989101409912, 1.061653971672058, -0.9157474040985107, 0.5490899085998535, 0.8685299158096313, -1.722694993019104, -0.14554716646671295, -0.4665551483631134, 0.30858293175697327, 1.8024929761886597, -0.49789807200431824, 0.42459043860435486, 0.16828685998916626, -0.5908886790275574, 1.1485223770141602, 0.1867581158876419, 0.3462722599506378, 0.17227865755558014, -1.4120701551437378, -0.839078426361084, 0.03868807107210159, 0.5169333219528198, -1.670350432395935, -0.8458612561225891, -0.19895727932453156, -0.3093803822994232, -0.5626093149185181, -0.7902058959007263, -0.5753894448280334, 0.042829059064388275, -0.02773940935730934, -0.6294649243354797, -0.15410351753234863, -1.3870915174484253, 0.30029627680778503, 1.0713344812393188, -0.20922274887561798, -0.5400494337081909, -0.4969332218170166, 0.27515190839767456, -0.8263818621635437, -0.499021053314209, -0.47083979845046997, 0.07497306168079376, -0.5243374109268188, 1.750773549079895, 0.1447359323501587, 0.23474179208278656, -0.9278759360313416, 0.605278491973877, 0.375457227230072, 0.47584712505340576, 0.8866193294525146, -0.3925503194332123, 1.0818092823028564, -0.05804560333490372, 0.07236815243959427, -0.9884893298149109, 0.428257554769516, 0.7341750264167786, 0.4841792583465576, -0.06215016916394234, -0.028649775311350822, 1.1281628608703613, 0.9963968396186829, -0.5090944766998291], [-1.029170274734497, 1.0327637195587158, 1.4395607709884644, -0.43852075934410095, -0.9306878447532654, -0.7996318340301514, 0.8226134181022644, -0.42137280106544495, 0.39485353231430054, -2.4669575691223145, 0.7463653683662415, -0.12409964948892593, 1.3669153451919556, 0.40624499320983887, 0.006589402444660664, 0.7110879421234131, 0.1195206567645073, -0.8238638043403625, -0.3845605254173279, 0.28626126050949097, 1.1153188943862915, -0.9247187972068787, -0.29399344325065613, 1.7048697471618652, 0.32468464970588684, -0.7818419933319092, -0.7779103517532349, -1.7214186191558838, -0.20177890360355377, 0.1685919463634491, -2.299051284790039, -1.2453912496566772, -0.1739024817943573, -0.11598765105009079, 0.1548151969909668, -2.9967920780181885, 0.8579975962638855, -1.2464616298675537, -0.49855226278305054, 0.15581871569156647, 1.1388543844223022, 1.3313292264938354, -0.25807201862335205, -0.7262607216835022, -0.42380091547966003, 0.2542918920516968, 0.2344033569097519, 1.7522231340408325, 0.005713304039090872, 1.1399048566818237, 0.009918268769979477, 0.11584564298391342, 1.5636317729949951, -0.4049302041530609, -0.3821682929992676, -1.152870535850525, 0.0675937831401825, 1.2693647146224976, 0.23799115419387817, -1.5189564228057861, -0.8888655304908752, 0.9143125414848328, -1.3566995859146118, 0.21817295253276825, -1.3863744735717773, 1.5542210340499878, 1.1671525239944458, -0.35616979002952576, 0.7663818001747131, -1.144040822982788], [0.3704150915145874, -1.633056402206421, -0.1673170030117035, 0.36408865451812744, 0.1120203509926796, -1.142240285873413, -1.0189927816390991, 0.4709406793117523, -1.0527565479278564, -0.0965019166469574, -0.09140557795763016, 0.6747688055038452, 1.06868577003479, -0.7935543656349182, -0.04123418778181076, 1.2700631618499756, -0.08953435719013214, 1.2833468914031982, -0.8221092820167542, -1.4305740594863892, -0.5539798736572266, 0.32257533073425293, -0.022160528227686882, -0.6301864385604858, -1.8576724529266357, 0.3340699076652527, -0.639860212802887, -0.04294991120696068, -0.5864004492759705, 0.1600489467382431, -2.219802141189575, -1.699904203414917, -0.565126359462738, -1.094895362854004, 0.14202599227428436, 1.1512762308120728, -0.48379454016685486, 0.5357025861740112, 1.097241759300232, 0.4727626144886017, 0.22292709350585938, 1.5896059274673462, 0.16931575536727905, 2.0047061443328857, -0.6567842364311218, -1.235592246055603, 0.44807103276252747, -1.2022422552108765, -0.7937237620353699, 0.03620556369423866, 0.18992897868156433, -0.2672426104545593, -0.11100775003433228, 0.1405043751001358, 0.47785496711730957, 0.5370975136756897, -0.4650322496891022, 0.05897559970617294, -0.7235539555549622, 0.68526691198349, -1.0691277980804443, 0.4324953556060791, -0.0627380758523941, -1.3089455366134644, -0.5718945860862732, -0.9570289254188538, -0.32899364829063416, 0.875920295715332, -0.5589437484741211, -0.5411231517791748], [0.6497879028320312, 1.1681050062179565, 0.5969669222831726, -0.4875859022140503, 0.4788987934589386, -0.4299287796020508, -1.2664819955825806, 0.9415683150291443, 1.2912487983703613, 0.5939888954162598, 0.42518723011016846, -0.2624174654483795, 1.1246728897094727, 0.6684484481811523, 1.0251718759536743, 2.341144323348999, -0.22655509412288666, 0.3817678391933441, -1.0719630718231201, 0.38390472531318665, -0.6674644947052002, -0.16893114149570465, 1.311453104019165, -0.7929648160934448, -0.6958339214324951, -0.46568456292152405, -1.2517460584640503, -1.27717125415802, -1.3701927661895752, 1.3642830848693848, -1.0100924968719482, -0.45644253492355347, -0.02263643406331539, 0.39339402318000793, 0.8018644452095032, -0.6908888220787048, -1.2194432020187378, -0.39568567276000977, 0.845304548740387, -0.018860939890146255, 0.8548625111579895, -0.9734957218170166, -0.8929722905158997, 1.3862168788909912, -0.15174219012260437, 0.4078148305416107, 0.13085125386714935, -0.5798578858375549, -0.10315442085266113, -0.8918983340263367, -1.7591888904571533, 0.12373191863298416, 1.932847023010254, 1.152708649635315, 1.741357684135437, -0.8313584327697754, -1.8616597652435303, -0.24237248301506042, 0.8344513177871704, 0.04479898512363434, -1.5920791625976562, -0.3854620158672333, -0.8553760647773743, 0.4414701461791992, -0.6386474967002869, 0.25379717350006104, 0.8181407451629639, -0.03251338750123978, -1.1060199737548828, -0.7068474292755127]], "dtype": "float32"}, 16, 32, 16, 8], "e": {"__tup__": [{"__nd__": [[0, 2, 4], [1, 3, 5]], "dtype": "int32"}, {"__nd__": [[2.1589815616607666, 0.4685523509979248, -0.8624899983406067, 5.940581321716309, 9.703187942504883, 3.792839527130127, -9.572509765625, -6.1964335441589355, -1.2566498517990112, 1.3023386001586914, -2.2164576053619385, -0.4946921467781067, 11.15820026397705, 6.801636695861816, -0.9966147541999817, 3.1466073989868164, 7.678967475891113, 1.012125849723816, -1.599633812904358, -12.276084899902344, -2.0891001224517822, 1.625963568687439, 7.026156425476074, 0.6014751195907593, -0.09947105497121811, -6.370028018951416, -5.374011516571045, 5.009059906005859, 2.0965261459350586, 6.13986873626709, -4.161497592926025, -4.819042205810547, 3.9803221225738525, 1.72193443775177, -0.5761986374855042, 1.4688420295715332, -3.3767812252044678, 3.9200539588928223, -1.3719909191131592, 3.769988775253296, -2.790390729904175, 9.238029479980469, -8.138988494873047, 7.803352355957031, -4.78355598449707, -4.667749404907227, -4.156467914581299, -1.9063295125961304, -0.9451242685317993, -1.3643579483032227, 4.75712251663208, -0.7947162389755249, -2.9473721981048584, -9.452823638916016, -13.510665893554688, 6.815619945526123, 5.870882034301758, -3.2111825942993164, -4.116883754730225, -1.4193520545959473, -0.516434907913208, 1.512306809425354, -6.447659015655518, -0.6830111145973206, -0.9715695977210999, -5.937047481536865, 6.531528472900391, -5.198397636413574, -0.3384585976600647, 6.1555352210998535], [17.9708194732666, -3.033982038497925, 1.0755877494812012, -5.115009307861328, 2.4252259731292725, 1.1802889108657837, 1.3948181867599487, -4.871364116668701, -0.18148724734783173, 6.465871334075928, 4.449647903442383, -5.535041332244873, -6.042175769805908, -12.938589096069336, 4.025463104248047, 9.187909126281738, -4.647310256958008, -1.6799012422561646, -0.7761593461036682, -6.8603129386901855, -3.4433069229125977, 11.82166576385498, 22.473316192626953, -2.760881185531616, -3.636401653289795, -0.3838552236557007, 5.861691474914551, -1.0787122249603271, -1.498833179473877, -9.033559799194336, 5.728443145751953, -6.374721050262451, 3.7497191429138184, 4.7267560958862305, -1.188352346420288, 0.626839280128479, 1.4572070837020874, 10.726675987243652, 0.06335347890853882, -3.529086112976074, -2.164313793182373, 16.482114791870117, 3.973504066467285, -0.06315214931964874, 11.628043174743652, -0.25238630175590515, -7.6225128173828125, 0.7416948080062866, -3.0971105098724365, -9.424899101257324, 5.234004497528076, -2.3293659687042236, -7.714303493499756, -6.167344570159912, -8.856982231140137, -7.245677947998047, -4.247441291809082, -8.020452499389648, -8.335598945617676, 19.22811508178711, -10.323894500732422, 1.4204461574554443, 12.139899253845215, 3.4745407104492188, 1.7793021202087402, 4.565288543701172, -8.250762939453125, 6.603871822357178, -5.0245585441589355, 2.0367777347564697], [3.2806739807128906, -0.39738929271698, -2.5511553287506104, -1.685734510421753, -7.523678779602051, 14.62404727935791, -1.3959275484085083, 0.02830878458917141, 4.466756820678711, 8.459157943725586, 4.772566318511963, 1.2715970277786255, -7.782576560974121, 2.624891757965088, -5.335122585296631, -9.605917930603027, -0.9364882707595825, -9.497532844543457, 14.176042556762695, 0.24386130273342133, 2.2608156204223633, 4.037810802459717, 0.7810322642326355, -0.8271468877792358, 5.866065502166748, -3.72571063041687, -1.0319154262542725, 7.937671184539795, 1.2315938472747803, -8.414655685424805, 4.009069442749023, 5.693395137786865, -7.88132381439209, 6.44966459274292, -1.0087733268737793, -5.033755302429199, 8.608311653137207, 3.3636133670806885, 1.7633734941482544, -1.9570122957229614, -3.501329183578491, -1.2418060302734375, 10.408573150634766, -11.005030632019043, -1.1664650440216064, 8.699024200439453, -1.4841315746307373, 4.695998668670654, -2.8967771530151367, -2.8946924209594727, 10.070110321044922, -6.689657688140869, -5.8890604972839355, 9.448714256286621, 2.363706350326538, 2.2991878986358643, 2.5024523735046387, -4.694816589355469, -0.27602991461753845, 2.335224151611328, 9.461088180541992, -8.031342506408691, 2.531597852706909, 6.883124351501465, -6.175378799438477, 0.39812812209129333, 1.2398478984832764, 1.1020466089248657, 10.588443756103516, 3.0676891803741455], [4.673976898193359, 5.021129131317139, -2.155313491821289, -3.699808120727539, -11.127352714538574, -6.992122650146484, -7.000022888183594, 3.736891031265259, 12.31763744354248, -5.546152591705322, -2.27392578125, -13.110930442810059, -5.928380012512207, 0.7546960115432739, 0.7692773938179016, 10.526698112487793, -23.29507827758789, 2.7776317596435547, 0.9153459668159485, 10.381329536437988, -2.07855486869812, 2.940431833267212, -4.995396614074707, 7.74461030960083, -16.683794021606445, -1.7366801500320435, 0.5402908325195312, -0.8088757395744324, 1.465133786201477, -5.95758056640625, -0.564411997795105, -4.301743984222412, 12.261696815490723, -1.6664540767669678, -6.154228210449219, -3.222062826156616, 7.770338535308838, 1.385919213294983, -2.088378667831421, -5.701272010803223, 1.290539264678955, -3.0254852771759033, -2.493837594985962, 0.07401926815509796, 3.5956790447235107, -3.4523470401763916, -0.4597127139568329, 1.0957918167114258, -3.449415922164917, -13.493770599365234, -3.384063720703125, 7.242285251617432, -0.9938752055168152, -2.812096118927002, 7.923293590545654, -3.6845650672912598, 4.342247486114502, 3.2865891456604004, 1.8557265996932983, 3.521728992462158, 0.2184344083070755, 2.6885104179382324, -10.264605522155762, -1.6437658071517944, -2.7071237564086914, 2.5145039558410645, -0.0979071781039238, 14.753844261169434, 1.7969591617584229, -8.04943561553955], [-0.3618444502353668, 2.105343818664551, 0.5323671698570251, -0.4254462420940399, -3.5597312450408936, -7.955593585968018, -10.124402046203613, 16.99241828918457, 9.315378189086914, -0.7778099775314331, 2.851097345352173, -1.8643367290496826, 6.068209648132324, 4.0411376953125, -2.9417519569396973, 3.8314638137817383, 2.4164772033691406, -8.543654441833496, -3.329622268676758, 13.85036563873291, -2.3668723106384277, -5.3815107345581055, -12.134140968322754, 7.072325706481934, 5.42965841293335, -5.292773723602295, -6.598952770233154, 3.9767773151397705, 15.231415748596191, 0.9694029092788696, -4.645256996154785, 0.35308748483657837, -1.0245915651321411, -2.2627956867218018, -0.6495346426963806, 3.5509867668151855, -4.028472423553467, -5.625644683837891, -2.051307439804077, -2.4399571418762207, 3.2769107818603516, -13.492472648620605, 12.964140892028809, -13.390110969543457, -4.662912368774414, 2.603754758834839, -13.031147003173828, 18.034955978393555, -5.125705718994141, -2.5691049098968506, 5.5779595375061035, 13.601164817810059, -0.4796740710735321, 15.078073501586914, 14.000533103942871, -3.7590816020965576, -17.302772521972656, 8.49999713897705, 0.31404295563697815, -5.04571533203125, 0.8133611083030701, 8.564380645751953, -12.157898902893066, 1.502183437347412, -11.664169311523438, 1.6384198665618896, 6.2314558029174805, 3.8869054317474365, 6.2296223640441895, -9.913959503173828], [-16.34156608581543, -1.4872000217437744, -5.963249683380127, -3.7977123260498047, 7.151336193084717, -8.978418350219727, -6.142561912536621, -0.29275932908058167, 16.74110984802246, -7.528202056884766, 6.787728309631348, -0.824846088886261, -9.49671459197998, 17.16206932067871, -0.7241941690444946, 8.511823654174805, 18.716949462890625, -9.895468711853027, -2.484279155731201, 0.5688406229019165, 1.1219637393951416, 2.5669150352478027, 12.201924324035645, -6.579507350921631, -4.126323699951172, 1.9490376710891724, -5.6376776695251465, -2.9512932300567627, -1.195346474647522, 2.154202699661255, 0.022253604605793953, -8.310948371887207, -7.4009809494018555, -4.861264228820801, -6.6646409034729, -3.5947799682617188, -7.296976566314697, -5.225700855255127, 4.407089710235596, 2.9566543102264404, 3.3880889415740967, 1.9469126462936401, -1.2907359600067139, -4.997628688812256, 3.7813668251037598, 1.8838485479354858, 0.7499036192893982, -1.9394359588623047, -2.1153151988983154, 10.319077491760254, -4.700263500213623, 2.2917733192443848, -2.408186197280884, 5.013334274291992, -5.072725296020508, -3.305558443069458, 4.334110736846924, -2.028904438018799, 0.824784517288208, 7.686004638671875, -12.133289337158203, -0.08160009980201721, 6.202622890472412, 7.194899082183838, 10.207999229431152, 1.5779693126678467, 0.6257462501525879, 4.3547139167785645, -6.31273078918457, 6.753201007843018], [-3.44460391998291, -3.353666067123413, -11.749687194824219, 7.694021224975586, 5.440328121185303, -9.449121475219727, 8.126397132873535, -2.339562177658081, 2.8686468601226807, 3.7800559997558594, -2.573077440261841, -7.096046447753906, -1.9969267845153809, 9.799920082092285, -6.291288375854492, 0.25959065556526184, 1.3849506378173828, 6.792078018188477, -0.8817460536956787, 5.566135406494141, 7.077273368835449, 10.118350982666016, -3.122744083404541, -2.8310155868530273, -6.684789180755615, 4.368859767913818, -2.9623560905456543, -0.3156013488769531, -1.2509056329727173, -1.2968077659606934, 3.5124118328094482, -5.305668354034424, 5.842727184295654, 2.988922595977783, -2.3306033611297607, 1.3609412908554077, 7.145274639129639, 0.4453662633895874, 5.152318954467773, 2.1026124954223633, 1.5178632736206055, 5.305017948150635, -7.545436859130859, 5.978857517242432, 5.920043468475342, 3.3280348777770996, -4.491733074188232, 0.23260436952114105, -3.8516175746917725, 1.42548406124115, -13.88851547241211, -3.724575996398926, -2.458559989929199, 9.341875076293945, 11.094541549682617, -16.398805618286133, 6.4607672691345215, 1.500245451927185, -0.8021216988563538, 1.9645670652389526, 2.3813741207122803, -9.77351188659668, 10.52420711517334, -2.392327070236206, -1.0463184118270874, -6.127713680267334, -1.8040722608566284, -6.3984694480896, 0.3902645409107208, 2.971193790435791], [-1.6860946416854858, -2.180072784423828, -1.2809596061706543, -3.2797913551330566, 9.552814483642578, -5.2706732749938965, -14.214754104614258, 12.001822471618652, -9.244913101196289, 0.05623847246170044, -1.3366260528564453, 1.820494532585144, 1.5474900007247925, -3.5128884315490723, 9.4699068069458, -3.011414051055908, -1.8088897466659546, 1.0500890016555786, 3.187734365463257, -0.6176300048828125, -9.66698932647705, 4.8383049964904785, -0.4856605529785156, 1.353600263595581, 3.7599680423736572, -0.17214272916316986, -3.98860502243042, -3.6719305515289307, 4.906778335571289, -1.0747796297073364, 4.1388115882873535, -1.5484604835510254, 10.304950714111328, -7.173099517822266, -5.006406784057617, -1.4619258642196655, -2.0006940364837646, 2.1175286769866943, -3.7671854496002197, -2.3933627605438232, 1.030116081237793, -9.517471313476562, -0.15473535656929016, -1.864154577255249, -4.022446632385254, 5.856365203857422, -13.682367324829102, -9.062990188598633, -3.900620222091675, -0.26764488220214844, -2.1053667068481445, 5.23356819152832, -4.938602447509766, -10.329309463500977, -4.2772216796875, -3.3934500217437744, -2.743926763534546, 5.526432991027832, -1.722410798072815, -5.083507061004639, -1.91959547996521, 13.427191734313965, -7.933553695678711, -2.985569715499878, 5.1810431480407715, -4.656104564666748, 2.8725109100341797, 3.4157018661499023, -3.870070219039917, -3.911360025405884], [13.17043399810791, 8.184992790222168, 11.636183738708496, 5.215207576751709, -1.6372690200805664, 8.13367748260498, -5.996913909912109, -4.015572547912598, -5.990275859832764, -22.752735137939453, -6.973179340362549, -9.11208724975586, 12.399131774902344, 0.19273516535758972, 4.2210373878479, 11.444498062133789, -2.779641628265381, 11.315361022949219, -4.113176345825195, -11.519783020019531, 8.791898727416992, 0.5865352153778076, 4.635865688323975, 3.1871535778045654, 1.6582807302474976, -1.7840160131454468, -1.1854223012924194, 3.9479641914367676, 4.785130023956299, 0.40674063563346863, 4.331188678741455, -4.756077289581299, 8.828174591064453, -6.5438361167907715, 10.414078712463379, 0.15228316187858582, -3.4946560859680176, -3.1551284790039062, 0.5777415037155151, 2.856851577758789, 2.688899278640747, 6.6797404289245605, -2.3361711502075195, 3.9757707118988037, -2.729771137237549, -3.5909907817840576, 5.8474602699279785, -6.843217849731445, 1.450738787651062, -12.83979320526123, -14.249889373779297, 4.539425849914551, -6.155734539031982, -1.3416146039962769, -2.117969512939453, 18.355918884277344, 12.8038330078125, 0.7434105277061462, 7.605381488800049, -1.8556530475616455, -0.1581907570362091, 10.040115356445312, 4.402640342712402, -3.9024505615234375, -7.192473411560059, 2.944371223449707, 1.30653977394104, -6.992549896240234, -1.7694214582443237, 3.3475022315979004], [-3.855891466140747, -4.539668560028076, -3.992384672164917, 1.1106036901474, -3.2044546604156494, 1.264259934425354, -12.024367332458496, 5.02231502532959, 13.272317886352539, 9.57758903503418, 8.43547248840332, 13.949450492858887, -7.831148624420166, 20.464563369750977, -3.4184842109680176, 12.966740608215332, 10.925451278686523, -12.483409881591797, -8.011419296264648, 1.8108336925506592, -1.1926000118255615, 3.8073906898498535, -4.271991729736328, -1.1053669452667236, -10.849030494689941, -12.322624206542969, -5.358577728271484, -3.504932403564453, 4.92471170425415, -2.1645941734313965, -10.07843017578125, -6.579311370849609, 1.8180010318756104, 3.007004737854004, 1.738492727279663, -0.5550642609596252, 4.095632076263428, 0.21806490421295166, 3.6885924339294434, -0.4954076111316681, -11.11283016204834, -2.199331760406494, 1.0421133041381836, -16.25800323486328, -5.580674648284912, -19.626585006713867, -6.960272789001465, -1.2380919456481934, 6.1162214279174805, -5.158151626586914, -11.045282363891602, 0.1272895783185959, 10.459620475769043, 6.385830879211426, 12.98315715789795, 5.460808753967285, -7.806421756744385, -1.4372446537017822, -11.177323341369629, 5.069458961486816, 5.975833892822266, -3.1119587421417236, -10.47985553741455, -4.992788791656494, 1.1798259019851685, 0.042708296328783035, 5.078308582305908, 6.915356636047363, 12.049710273742676, -4.926539421081543], [10.125309944152832, 6.240235805511475, -6.0910773277282715, 14.8046875, -3.7590439319610596, -1.4249248504638672, -0.6055802702903748, -3.883375406265259, 4.451615333557129, -7.043719291687012, -19.012638092041016, -10.705257415771484, 9.062211036682129, 11.14118766784668, 0.882183313369751, 4.197720050811768, -4.044835567474365, 8.86315631866455, -12.400105476379395, -4.48391580581665, 6.421631813049316, -3.165959358215332, 4.399431228637695, -8.413410186767578, -0.6643143892288208, 0.8551123142242432, -6.042441368103027, 0.05900193005800247, -1.7662947177886963, 0.3108643591403961, -2.4226932525634766, -10.465091705322266, 0.28800976276397705, 2.8872158527374268, 11.657852172851562, 11.469657897949219, -5.735113620758057, -8.466005325317383, -1.5213358402252197, 0.5875288248062134, 0.5979101657867432, 9.276737213134766, -15.422574043273926, 6.261700630187988, 2.1905174255371094, -3.5315048694610596, 0.9177924394607544, -8.543560028076172, -1.7347666025161743, -2.370059013366699, -13.437522888183594, 4.843963623046875, -9.461992263793945, -0.25408172607421875, 9.097514152526855, 2.905954360961914, -6.887099742889404, -10.072933197021484, 4.260859489440918, 11.618346214294434, -2.5921173095703125, -6.668661117553711, 4.729620456695557, 6.024364471435547, 2.66782808303833, -2.7208597660064697, 4.703481674194336, -5.513480186462402, -3.4941518306732178, 10.2664794921875], [16.051254272460938, 4.769871234893799, 1.5925766229629517, -0.7853639125823975, 2.0633597373962402, 4.847771167755127, -1.4357972145080566, 2.647728443145752, -2.5153820514678955, 3.6877212524414062, -11.689924240112305, 5.729477405548096, 6.5855326652526855, 7.411558151245117, -2.993823289871216, -2.2597992420196533, 0.5106760859489441, -1.5748448371887207, 0.534769594669342, -2.3163652420043945, -3.5650932788848877, 6.903176784515381, 13.643757820129395, -5.877449035644531, 3.494609832763672, 1.1220334768295288, -3.4403884410858154, 0.9560199975967407, -4.621445655822754, 4.126043319702148, 1.1754428148269653, -0.20807574689388275, -4.924132823944092, 2.8431246280670166, 10.46981143951416, 4.672013282775879, -2.081841230392456, 3.0045833587646484, -0.41309359669685364, -0.5483824014663696, 4.350442886352539, 3.021695852279663, -3.7803702354431152, 9.25460433959961, -0.3851984143257141, -1.8729863166809082, 2.0870535373687744, -1.3154325485229492, -2.735952138900757, -1.330906629562378, 5.637722492218018, -2.336726427078247, -5.66956901550293, -1.5516798496246338, 3.057504415512085, 8.399353981018066, 2.516991376876831, -19.033845901489258, -4.062869071960449, -2.5530877113342285, -11.336905479431152, -4.192492961883545, 3.0800347328186035, 3.734253168106079, -8.663371086120605, 8.065321922302246, 5.287740707397461, -1.7157108783721924, -4.629147529602051, -4.020906448364258], [3.9145820140838623, 4.9776611328125, 7.477022647857666, 2.536268711090088, 0.18573088943958282, -0.029363391920924187, 11.411208152770996, -9.88564682006836, -6.358692646026611, 6.976928234100342, -5.6782145500183105, 1.3589426279067993, -4.635570049285889, -7.562714576721191, -4.679188251495361, -0.6438727974891663, -0.9228723049163818, 9.756427764892578, -4.8154296875, -7.416308403015137, 6.276574611663818, 0.6683163642883301, -8.450390815734863, -3.9975273609161377, 6.25137996673584, -5.464648246765137, -0.8045820593833923, 2.215749502182007, -12.301412582397461, 5.243993759155273, -0.37239521741867065, -6.2191081047058105, 2.709728717803955, 8.686201095581055, 3.5752170085906982, 4.643752098083496, 12.878900527954102, 1.5275042057037354, 0.11073658615350723, 10.257458686828613, -2.2220118045806885, -2.5961570739746094, -13.436657905578613, 9.497249603271484, 3.7085177898406982, -12.172264099121094, 9.41773796081543, -5.607515811920166, 7.177971839904785, 1.734474539756775, 1.854148030281067, -8.759897232055664, -6.112097263336182, -6.50689172744751, 11.979964256286621, 1.1020582914352417, 1.7039928436279297, -13.42945671081543, -8.368705749511719, -4.599552631378174, 5.194118499755859, -8.423324584960938, 2.6967852115631104, -2.4125893115997314, -7.338791370391846, -10.150141716003418, 1.013473629951477, -6.9688591957092285, -3.415255069732666, 3.822129487991333], [-3.4337172508239746, -3.8144824504852295, 4.63051176071167, 4.676661014556885, -9.7992582321167, -1.3985403776168823, 3.9277570247650146, 0.8217724561691284, -0.29385367035865784, -0.9293506741523743, 6.615840911865234, 3.7390403747558594, 12.848250389099121, 7.949111461639404, -7.905457019805908, 11.626928329467773, 9.1473388671875, -4.536730766296387, -2.985969305038452, -3.336022138595581, -2.859844446182251, -2.4367852210998535, 5.596346855163574, 3.67032527923584, -1.181652307510376, -0.8497047424316406, 2.611534833908081, -10.76793384552002, 6.282679080963135, -8.669231414794922, -9.141502380371094, -3.774257183074951, 7.109412670135498, 2.1360840797424316, -11.81734848022461, 1.0004624128341675, 1.1132234334945679, -8.178732872009277, -2.658947467803955, -0.11604741960763931, 1.2807555198669434, 8.831436157226562, 5.768773078918457, -9.076276779174805, -3.4375436305999756, 20.189037322998047, -17.590282440185547, -2.4268758296966553, -16.159780502319336, -1.9655168056488037, 0.19339117407798767, -3.8511502742767334, -1.4966400861740112, 11.48621654510498, 10.202068328857422, -15.164783477783203, -5.716792583465576, -0.4745127260684967, 3.2970314025878906, -0.8177298903465271, 2.2451090812683105, 0.32716628909111023, -7.672516345977783, -7.224710464477539, 4.66813325881958, 14.602045059204102, 4.3503265380859375, -5.687722682952881, -5.814728260040283, -3.849604845046997], [-0.6568050980567932, -6.24676513671875, 0.3882503807544708, 11.352693557739258, 4.4753737449646, 1.7967756986618042, -5.045385837554932, -0.4980604648590088, 2.262784957885742, -3.3264503479003906, 4.848506927490234, -1.7265807390213013, 0.06956925243139267, 3.7760891914367676, -1.9724360704421997, 0.5744830965995789, 2.4868359565734863, 7.6180291175842285, -0.10855032503604889, 2.850550651550293, -5.034306049346924, 7.357455730438232, -0.8017335534095764, 10.864917755126953, 6.233534336090088, -6.720487594604492, -2.147399663925171, -7.009936332702637, -1.8132197856903076, -8.37563705444336, -7.819285869598389, -15.116938591003418, 7.954256534576416, 5.934758186340332, -5.390555381774902, -3.6326229572296143, 9.3076810836792, -2.891115427017212, 9.950592994689941, -4.800289630889893, 2.3229446411132812, 4.18653678894043, -6.572176456451416, -5.691344261169434, -5.16347599029541, -8.430585861206055, -6.9430766105651855, -5.022855281829834, -0.6125189661979675, -4.199720859527588, -9.054701805114746, -4.032394886016846, 9.819768905639648, -10.806065559387207, 3.2253246307373047, -3.4452760219573975, 5.26371955871582, 0.5326443910598755, -4.517204284667969, -4.226273059844971, -1.2052068710327148, -3.4120540618896484, 1.2493082284927368, -9.560487747192383, -2.9825522899627686, -10.250510215759277, 1.275667428970337, 5.401738166809082, 8.23205280303955, -1.601877212524414], [1.943859577178955, -10.991217613220215, 3.6901798248291016, -8.847275733947754, -6.26511812210083, 1.467834711074829, 2.771552801132202, 1.819414496421814, 4.9967474937438965, -0.46043920516967773, 5.729613780975342, 1.6424696445465088, -5.9770426750183105, 2.565523862838745, 13.127214431762695, 3.114793300628662, -0.16335342824459076, -0.8489375710487366, -1.2170488834381104, -2.6440227031707764, 0.028883686289191246, -4.922534465789795, -5.610202789306641, -4.238142967224121, 2.625070095062256, -3.2352569103240967, 0.9438955187797546, -4.341073989868164, -15.431012153625488, 3.6600418090820312, -5.111910343170166, -3.1062934398651123, -16.734586715698242, 1.022979497909546, 2.9850170612335205, -6.260700702667236, -12.043437004089355, 0.017251526936888695, -10.905216217041016, -5.537250518798828, 8.153585433959961, -6.160221576690674, -5.2927565574646, -0.35261270403862, -1.7813301086425781, -8.470742225646973, 3.035987138748169, -0.3784502148628235, -2.967590093612671, 15.14576530456543, 6.025087356567383, -3.464207410812378, 9.790288925170898, -6.534291744232178, -6.461164474487305, -3.075302839279175, -13.421006202697754, 5.612392902374268, 4.659951210021973, 0.47405412793159485, -9.514103889465332, -3.906827688217163, 7.012203693389893, 2.088285446166992, 5.543332099914551, 0.7251213788986206, -4.806827068328857, -0.1990976482629776, -8.872539520263672, 1.7363320589065552], [-3.075092077255249, -0.6222304701805115, 0.4851343035697937, -0.882729709148407, -5.482019424438477, 9.790058135986328, -5.87277889251709, -5.627798557281494, -3.055661201477051, -4.955413818359375, 9.983012199401855, 2.384465456008911, -3.8602802753448486, 7.271534442901611, 2.353158473968506, 3.4770779609680176, 1.642397403717041, 8.747570991516113, 4.846768379211426, -0.5517415404319763, -0.6640735268592834, 12.108495712280273, 0.9042783975601196, -5.249403476715088, 5.940452575683594, -3.8491227626800537, -4.556493282318115, 0.4920722544193268, -12.838822364807129, 1.8362116813659668, -8.188750267028809, -4.800701141357422, -1.0699259042739868, 2.7318012714385986, 4.416645050048828, -9.132476806640625, 9.217276573181152, -4.752439498901367, 8.527501106262207, 1.043028473854065, 3.874804973602295, 4.913607597351074, -5.80514669418335, 5.186766624450684, -10.24421215057373, -4.479498863220215, 2.979053497314453, -10.869365692138672, -4.068100452423096, -3.6756482124328613, -12.466822624206543, -7.882061958312988, 9.0364351272583, 4.052233695983887, 1.2122901678085327, 0.41208139061927795, -2.0385451316833496, -9.282519340515137, 1.659555435180664, -5.950165748596191, -1.517765760421753, -1.0986274480819702, -9.101673126220703, -0.09781348705291748, -0.8649017214775085, -5.763725757598877, 0.5163895487785339, 4.453121185302734, -6.867652416229248, -0.7625616192817688], [0.46321481466293335, 11.946346282958984, -6.6370768547058105, 4.969523906707764, 6.353171348571777, -4.412048816680908, -0.6148208975791931, -2.9008092880249023, -8.26513385772705, 0.35687220096588135, 0.5234558582305908, -8.476107597351074, 5.017248630523682, -12.090717315673828, 2.7718312740325928, 3.81752872467041, -7.258758068084717, 6.031035900115967, 5.068877696990967, -4.050711631774902, 3.7908241748809814, 0.28291282057762146, -1.068486213684082, 5.978896617889404, 3.182074785232544, -3.7932279109954834, -7.109020709991455, 5.563472747802734, 11.632232666015625, -6.0271100997924805, -5.264853000640869, 3.6632437705993652, 18.413850784301758, -3.8799591064453125, -0.9389914274215698, -4.19977331161499, 10.588690757751465, -13.05983829498291, 4.919644355773926, 3.5558817386627197, -3.0488998889923096, -2.2110190391540527, -4.124420642852783, 1.3466747999191284, -1.8020411729812622, 7.438105583190918, -0.7141607403755188, -0.9385907649993896, 5.254393100738525, -6.050200939178467, -2.4013562202453613, -0.26440104842185974, -7.438165664672852, -2.665369987487793, 2.4274790287017822, -9.536767959594727, -4.847349166870117, 5.425017833709717, 5.8996992111206055, 4.1656012535095215, 0.5442311763763428, 4.296478748321533, -9.780349731445312, 0.535280704498291, 2.6899683475494385, -5.85971736907959, 4.593615531921387, 2.2262134552001953, -10.037294387817383, 1.6638997793197632], [8.065759658813477, 5.826895236968994, -1.9816360473632812, 0.7795203924179077, 1.3993983268737793, -11.967304229736328, 2.019510269165039, 9.025930404663086, -2.50494122505188, -16.443878173828125, -2.0851826667785645, 1.3284450769424438, 17.725112915039062, -1.8537030220031738, 0.693680465221405, -1.3432589769363403, -0.708996057510376, -7.471678256988525, -2.9421448707580566, -6.1052775382995605, 4.065484046936035, -3.333035469055176, 2.3918938636779785, -1.07744562625885, 9.470510482788086, -1.3241808414459229, -0.42934951186180115, -5.553135395050049, 0.24966296553611755, -0.41394466161727905, -4.6464924812316895, -11.842123985290527, 0.6109687089920044, -5.9905548095703125, 3.5540881156921387, -2.8878767490386963, 3.44561505317688, 3.895780086517334, -3.0197741985321045, -0.6750626564025879, 3.68717098236084, 3.651322364807129, 2.788191556930542, -5.7484331130981445, -3.8042449951171875, 1.8268545866012573, -2.9440619945526123, 8.275899887084961, 1.0420031547546387, 0.7190520167350769, 2.9837729930877686, 3.8537044525146484, -5.955939769744873, -3.849417209625244, -0.2263151854276657, 2.8644344806671143, -2.1307265758514404, -2.9454734325408936, 8.142969131469727, -0.16890040040016174, -11.073307991027832, 6.249089241027832, 5.973906517028809, 5.526391983032227, -1.1837245225906372, 13.10193920135498, 7.195611476898193, -7.826257705688477, 0.3642425239086151, -2.3658056259155273], [17.331050872802734, -1.8035084009170532, -5.994750022888184, -1.532362937927246, 10.086435317993164, -9.968948364257812, -1.3753697872161865, -3.9246904850006104, 9.33692455291748, 7.423714637756348, 2.1648054122924805, -9.6114501953125, -4.303825855255127, 1.0725908279418945, 0.4447575807571411, 13.1405029296875, -5.924966335296631, -1.4414995908737183, -0.49046260118484497, -2.4929733276367188, 6.217188358306885, 7.419571876525879, 4.781930923461914, -0.3327195942401886, -8.177833557128906, 0.11461144685745239, 5.7391557693481445, 5.792022228240967, -10.653925895690918, -0.8544068336486816, 4.709488868713379, -6.959640979766846, 4.507580280303955, 0.27812445163726807, -9.767930030822754, 5.1022491455078125, -1.1638197898864746, 14.982271194458008, 2.3523647785186768, -2.6899898052215576, -2.299098014831543, 1.226657748222351, -7.1655426025390625, 1.0688625574111938, 0.7677960395812988, -4.33549690246582, -5.228569030761719, 2.9473133087158203, -4.583582401275635, -11.691823959350586, -0.036423634737730026, 1.1448911428451538, -3.1512742042541504, -3.072906494140625, 6.340910911560059, -3.1257879734039307, 2.1728034019470215, -3.1528749465942383, -4.5748772621154785, 10.020442962646484, -3.7639429569244385, -8.462716102600098, 8.172255516052246, 6.909217834472656, -3.665311813354492, -3.5922658443450928, 0.5339229702949524, -2.8390655517578125, -4.782887935638428, 4.630208969116211]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.06944452226161957]], "dtype": "float32"}, {"__nd__": [[0.07622247189283371]], "dtype": "float32"}, 16, 16, 16, 4], "e": {"__tup__": [{"__nd__": [[0]], "dtype": "int32"}, {"__nd__": [[0.005293233320116997]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-1.6537500619888306, -0.4274130165576935, 0.8562620878219604, -1.0109565258026123, -0.3658285140991211, 1.2741838693618774, 1.1216728687286377, 0.3507866859436035, 1.0892198085784912, 1.8637112379074097, -1.532790184020996, -1.2994399070739746, -1.8560892343521118, -0.668485164642334, -1.1751433610916138, 0.08281990885734558, -1.9162863492965698, -0.608836829662323, -0.9929649829864502, -1.2117431163787842, -0.24460576474666595, 0.979509711265564, 0.5811188817024231, 0.27953267097473145, 1.097720980644226, -0.036493152379989624, 0.8669057488441467, -1.2854284048080444, -0.4269864559173584, 0.021518591791391373, -0.13033731281757355, -0.14107897877693176, -0.5136457085609436, 0.013372328132390976, -0.6026788353919983, -0.11207947880029678, 0.07180707156658173, -0.33738335967063904, 0.2780008614063263, -1.9354324340820312, -0.3215479850769043, -0.1744881570339203, -1.6148300170898438, -0.43283775448799133, 2.1487977504730225, 0.23135894536972046, -0.05572584643959999, 0.06974141299724579, 1.13506019115448, -0.8590519428253174, 0.5592306852340698, 0.7376121282577515, 0.7269995808601379, -0.5098750591278076, 1.0249706506729126, 0.965287446975708, -2.6137256622314453, -0.014858915470540524, -0.22996832430362701, 0.7362406849861145, 0.6185839176177979, -1.360425353050232, -0.0999433621764183, -0.9970879554748535], [-0.5900167226791382, -0.5924866795539856, 0.20837675034999847, -1.2420120239257812, 0.3980320692062378, -1.1078174114227295, -0.6900467276573181, -1.2513986825942993, 0.3869415819644928, -1.2487812042236328, 0.7676643133163452, -0.7035082578659058, -0.6674354672431946, 1.297192931175232, -1.7618216276168823, 0.5524778962135315, -0.5349332690238953, 0.21677926182746887, 0.8255569338798523, -0.5247205495834351, -0.16517877578735352, -1.373883843421936, 0.397774875164032, 0.23881791532039642, 0.5839869379997253, 0.3442881405353546, 0.5964995622634888, -1.450986385345459, 1.272254228591919, 1.1521461009979248, -1.3647252321243286, -0.6701651215553284, -1.2062016725540161, 0.2445482313632965, 0.321230411529541, -1.9317991733551025, 0.6712756752967834, -0.17382927238941193, -0.22331330180168152, -1.153573989868164, -1.0190579891204834, -0.1792765110731125, 0.9607180953025818, -0.07020413130521774, 1.307187795639038, -0.8685872554779053, -0.5753854513168335, -1.609528660774231, 1.0080933570861816, -1.6022512912750244, 0.03820056468248367, 0.36820071935653687, -0.49221065640449524, 0.3276534080505371, -0.38920363783836365, -0.329581081867218, -0.4508781135082245, -0.45299211144447327, 0.6590209007263184, 0.3803202211856842, 0.20435695350170135, -0.2577323615550995, 0.1496373862028122, -0.06361223012208939], [-0.8096859455108643, 1.1646249294281006, 0.3269283175468445, 0.42841771245002747, 0.9001417756080627, 0.32641443610191345, -1.830308198928833, -0.49467751383781433, -1.8678653240203857, -0.33775365352630615, 1.0921822786331177, 0.7090123891830444, -1.2443784475326538, -0.5147417783737183, 0.9310961365699768, -1.0233380794525146, 0.4808003604412079, -0.8171783685684204, -1.401787519454956, 0.8078743815422058, -0.3187079429626465, -0.4712984263896942, 0.5496507287025452, -0.21708977222442627, 0.053550176322460175, -0.6521735787391663, 0.3188130855560303, -0.3922572135925293, 0.1975812017917633, 1.0207500457763672, -0.9698506593704224, -0.8899236917495728, 0.9914593696594238, -0.884381115436554, 0.4708156883716583, -0.0712578296661377, -0.09241536259651184, 0.7797442674636841, 0.9750664830207825, 0.3017346262931824, 0.7010027170181274, -0.8835291862487793, -0.3151910901069641, -0.8221455812454224, -1.038490891456604, -0.6038610339164734, -1.093428134918213, -1.6798175573349, -0.13903498649597168, 0.12929752469062805, 1.634770393371582, 2.455183506011963, 1.3633126020431519, -1.2903530597686768, -0.5800521373748779, 0.6181479692459106, 1.2227787971496582, -1.4895954132080078, -1.1120442152023315, -0.7288857102394104, -0.24124419689178467, 0.9562284350395203, -1.7427161931991577, 1.4941238164901733], [-1.5135977268218994, 0.5880029797554016, 2.6004533767700195, 0.6025485396385193, -0.9412236213684082, 0.5810254216194153, 0.5610688328742981, -0.5131603479385376, -1.461044430732727, -0.6704360842704773, -0.09542573243379593, 0.12191596627235413, 0.5961014032363892, -0.5739267468452454, 0.4973251223564148, 0.8283689022064209, -0.8741931319236755, 0.21678972244262695, -0.02883736602962017, 1.3365956544876099, 0.38391682505607605, 0.6554083228111267, 0.1764838546514511, 1.1928634643554688, -0.8768047094345093, -0.7914597392082214, -1.0341547727584839, 0.0414743646979332, -0.6787872910499573, -1.3030438423156738, 0.6601332426071167, 0.6454576253890991, -2.2209510803222656, -1.7704156637191772, -0.13675114512443542, 1.2243057489395142, 0.8880713582038879, 0.09256661683320999, -1.110275387763977, 1.4795215129852295, 0.8998793959617615, -0.4364912807941437, -0.6822623610496521, -1.16097092628479, 0.986305296421051, -0.48466581106185913, 0.9610456824302673, 0.061736635863780975, -0.7031830549240112, 0.5857332348823547, -0.5855982899665833, -1.6612281799316406, -1.5429965257644653, -0.6281660795211792, -1.2945692539215088, -0.830669105052948, 0.1047571450471878, -0.22933721542358398, 1.6027297973632812, -0.5975714325904846, -0.004361962433904409, 0.3962728977203369, -0.18533839285373688, 0.27623000741004944], [0.4166838228702545, 1.9626625776290894, 0.7530367374420166, -0.40445390343666077, 1.193289041519165, -0.7597149014472961, 0.3226364254951477, -1.6019439697265625, -0.4433860778808594, 0.7056628465652466, -0.2699902355670929, 0.027541177347302437, -0.512828528881073, -1.6209216117858887, 1.6270769834518433, -1.3286300897598267, 1.0354430675506592, 0.14734092354774475, -0.5264033079147339, -1.9635816812515259, 0.8490915894508362, 0.17715269327163696, -0.2051781862974167, 0.08175981789827347, 0.8089094758033752, 1.450427770614624, -1.4393497705459595, 0.11866585165262222, -2.201629400253296, -0.8549988269805908, 1.0540684461593628, 0.6068093180656433, 0.2586899399757385, 0.5126243233680725, 0.846413791179657, -0.7999319434165955, -0.7071583271026611, 1.2612510919570923, -0.8369940519332886, 0.09397516399621964, -1.422494888305664, 0.4839118421077728, -1.793463945388794, 0.20669396221637726, 1.753823161125183, -1.1639994382858276, 0.35284221172332764, -0.08218705654144287, -0.1997276395559311, -0.5749725103378296, 1.0222584009170532, 0.2770999073982239, 0.8785882592201233, 0.29082030057907104, 0.48208266496658325, -0.5405625700950623, -0.43295347690582275, -2.6252193450927734, 0.3591744899749756, 0.9073582291603088, -0.27401983737945557, -1.6551364660263062, 1.7557569742202759, 1.9878394603729248], [-0.2722161114215851, 0.15771980583667755, 0.018919093534350395, 0.9745885729789734, 0.8800183534622192, 0.28560909628868103, 1.0652880668640137, 0.07693855464458466, -1.0547523498535156, -0.02105814218521118, 0.8958985805511475, -0.7475829720497131, -0.5021466612815857, 0.12750910222530365, -0.47867274284362793, -0.3127177655696869, -0.9397487044334412, -1.3850632905960083, -0.39268022775650024, 0.07733870297670364, -0.0513063482940197, 1.5170745849609375, 0.31005600094795227, 0.31708812713623047, 0.3053382337093353, -0.29828840494155884, -0.5187252163887024, 0.4547334909439087, -0.0472324900329113, 1.9176992177963257, 0.2375006079673767, -1.5615051984786987, -0.37418845295906067, 1.8645480871200562, -1.137711763381958, -0.3690042495727539, -0.35921627283096313, 0.20699059963226318, 0.3336663544178009, -0.8075066804885864, -0.757714569568634, 0.36263614892959595, 1.634285569190979, 0.25425416231155396, -1.526407241821289, -0.6597191691398621, -0.07230282574892044, -0.8125987648963928, -0.07784027606248856, -0.3542684018611908, 0.02437490224838257, 0.5017791986465454, -2.197801113128662, 0.8541887998580933, 1.1723909378051758, 0.7364023327827454, -0.01421385444700718, 0.2598029375076294, 2.1610562801361084, 1.4093194007873535, -0.018822714686393738, 0.8257951140403748, 0.31065675616264343, -1.2925808429718018], [-0.7970237135887146, 0.4129348695278168, -1.3076626062393188, 0.6887454986572266, 0.575278103351593, 1.6092519760131836, 0.017384227365255356, -1.6939457654953003, -0.3782947361469269, -0.5909169912338257, -0.5143182873725891, -0.06261099129915237, -1.7867445945739746, 1.0981532335281372, 0.49595123529434204, -0.4646267294883728, -0.1828591376543045, 0.061147790402173996, -0.07329436391592026, -1.0181059837341309, -0.921370267868042, -1.3447450399398804, 0.4849357306957245, -0.7990272641181946, -0.5053874850273132, -1.5548467636108398, -0.03281436488032341, -0.9718114733695984, -0.8821970820426941, -2.134857416152954, 0.9522486925125122, 0.10819532722234726, 0.2360609471797943, -2.0840182304382324, -1.884204387664795, 1.4873034954071045, -0.4974699020385742, -0.1673143059015274, -0.05120202153921127, 0.11194189637899399, -0.00649813748896122, -1.1482971906661987, 0.8178497552871704, -0.6458467841148376, -0.20798125863075256, -1.4420146942138672, -1.7256686687469482, 0.8306668996810913, -1.4130992889404297, -1.721554160118103, -1.0723246335983276, 0.378785640001297, -0.2912886142730713, -0.6760950088500977, -1.060102939605713, -0.33190762996673584, -0.9864497184753418, -0.8168679475784302, -0.5591674447059631, -1.0157755613327026, -0.567794919013977, 0.801035463809967, -0.45461276173591614, 1.6623010635375977], [-0.5870884656906128, -2.6277379989624023, -0.08844684064388275, 0.6719110012054443, -0.06473632156848907, -1.892642855644226, -0.3231038749217987, -1.7665256261825562, 0.8580392599105835, 0.5304247736930847, 0.27292314171791077, 1.0685604810714722, -0.8255398273468018, 0.013615889474749565, -0.3631298840045929, -0.9504479765892029, -0.3816438317298889, -1.524263620376587, -1.7544747591018677, -0.9611214995384216, -0.037163496017456055, 0.25953802466392517, -0.08661375194787979, 2.2346205711364746, 0.9776848554611206, -0.47701531648635864, 0.8263320922851562, -1.005403757095337, 0.48044243454933167, -0.5254747867584229, 0.3102215826511383, -1.1315577030181885, 1.3476336002349854, -0.2011861652135849, 0.6842817068099976, 0.38579297065734863, -2.6614348888397217, 0.3738093376159668, 0.30389368534088135, -1.6337145566940308, 1.4113495349884033, 0.5116433501243591, -1.0374218225479126, 0.7313273549079895, -0.443685919046402, -0.42894884943962097, 0.7047721147537231, -0.22293011844158173, -1.1107780933380127, 0.03494589403271675, 0.3720880150794983, 0.7571253776550293, 0.3908412456512451, -0.3699364960193634, 0.70328688621521, -1.519095778465271, -0.44140464067459106, -0.28969183564186096, 0.7011076807975769, 0.16139622032642365, -0.0009976079454645514, 0.6124628186225891, -0.9646650552749634, -0.6641655564308167], [0.23817497491836548, -0.09632843732833862, 0.08275580406188965, -0.618524968624115, -0.7715355753898621, -0.3684723973274231, 1.311316728591919, 0.36316415667533875, 0.3344014883041382, -1.5983766317367554, -0.8099855780601501, -1.1323156356811523, -0.40909427404403687, -1.1415095329284668, 0.5460498332977295, 0.08516452461481094, -0.09639070183038712, 0.7722142338752747, -0.44583842158317566, -1.3297133445739746, 0.14644959568977356, -0.8800665140151978, -1.446388602256775, -1.00503671169281, -0.6300265789031982, -0.6009039282798767, 0.16456544399261475, -0.3272649645805359, -0.14538390934467316, -0.9480281472206116, -1.0480141639709473, -0.07915555685758591, 1.4011805057525635, -0.7422923445701599, 2.6297638416290283, -0.23762765526771545, -1.2253801822662354, -0.40424445271492004, -0.02154519222676754, 0.9008948802947998, 0.9159940481185913, -0.7349756956100464, -0.8297791481018066, -0.21886250376701355, -0.25814884901046753, 0.6681212186813354, 0.05786646902561188, 0.6159993410110474, -0.7964404821395874, 0.8199850916862488, -0.3245046138763428, -0.9445955157279968, -1.9501609802246094, -2.895263433456421, -0.5398381352424622, -0.3965272903442383, -1.6360639333724976, -1.9447120428085327, -0.36738428473472595, 0.6118091940879822, 0.19062650203704834, 2.0047478675842285, -0.43785762786865234, -1.7406421899795532], [0.538936197757721, -0.14846794307231903, 1.5749108791351318, -0.8808534145355225, 1.3619471788406372, -0.8278377056121826, -1.2051900625228882, -0.6168670654296875, -1.7284255027770996, -1.0300081968307495, 0.28483468294143677, -1.757996916770935, -0.6577789187431335, -0.29920482635498047, 0.6305592060089111, 1.144449234008789, 0.17744538187980652, 0.5562980771064758, 0.9354936480522156, 0.8785611391067505, -0.10730026662349701, 1.9299907684326172, 1.0134861469268799, 0.5678386092185974, 0.012979025952517986, -1.5775936841964722, 0.0356585793197155, -0.9259161353111267, 0.9289919137954712, 0.28423553705215454, 0.1848224401473999, -0.7898216843605042, -0.4235180914402008, -1.2881503105163574, -0.7301387190818787, 0.2970917522907257, 0.8495899438858032, -0.5864861607551575, -0.8259982466697693, 1.145329236984253, -1.9043083190917969, -0.2887876033782959, 0.455424964427948, 0.7462840676307678, 0.46460315585136414, -0.9796565771102905, -0.9529938101768494, 0.25970447063446045, -0.7160426378250122, 1.1458837985992432, 1.0537950992584229, -0.9392752647399902, 1.1262152194976807, -0.616151750087738, 0.6606099009513855, 1.7268471717834473, -0.28790006041526794, -1.0076006650924683, -0.4122888147830963, 1.051368236541748, 0.20174413919448853, -2.5675110816955566, -0.4417691230773926, 0.15324419736862183], [1.499774694442749, 1.775849461555481, 1.2691822052001953, -0.3019898533821106, -0.8200145363807678, 1.0948666334152222, -0.14749178290367126, -1.4842333793640137, -1.1059318780899048, 1.5301949977874756, -1.1198011636734009, -1.6462138891220093, -1.9737364053726196, -1.4022648334503174, -0.4298378527164459, 1.0236833095550537, 1.3535614013671875, -0.7117483615875244, -1.1920320987701416, -0.687292754650116, 0.28548330068588257, 0.1351311206817627, 0.9049732685089111, -2.017061471939087, 2.018040895462036, -2.282590627670288, -0.408812552690506, -1.0735775232315063, 0.8066277503967285, 1.4877370595932007, -0.05463465675711632, 0.27823618054389954, 1.5529102087020874, 1.736811637878418, -1.3333450555801392, -0.44910168647766113, -0.023348985239863396, -0.6466782093048096, -2.6458709239959717, 0.10769779980182648, 0.09165707230567932, -0.4344644546508789, -0.041253793984651566, -0.8418391346931458, -1.6691166162490845, 1.1918271780014038, 1.0731401443481445, 2.884631872177124, 0.01808822900056839, -0.00015621018246747553, 0.26914462447166443, 0.975104808807373, -0.5858946442604065, -0.14099663496017456, 0.16775226593017578, 1.0768150091171265, -1.0798190832138062, -0.014544883742928505, -1.6455893516540527, 2.260863780975342, 1.2614243030548096, -0.614548921585083, 0.31900858879089355, 0.2313554286956787], [-1.3015786409378052, 1.4724149703979492, 0.5477567911148071, -0.036055419594049454, 0.19059410691261292, -1.731704592704773, 0.13600072264671326, 1.8337385654449463, 1.4603376388549805, 2.5786287784576416, -0.7414077520370483, 1.4722613096237183, -0.0532732717692852, 0.5175474882125854, 0.8043010830879211, -0.4411078989505768, 0.05610746890306473, -1.2795578241348267, -0.8814493417739868, -0.41398561000823975, 0.9874811768531799, -0.6580967307090759, 0.45072993636131287, 0.7737000584602356, -0.797280490398407, 0.36721935868263245, -0.4266416132450104, -0.7556228041648865, -0.9666985273361206, -1.0557652711868286, -0.08353358507156372, 1.7234890460968018, -0.5066438913345337, -1.9465192556381226, -0.9320257902145386, 0.6610256433486938, -1.7496752738952637, 1.6782317161560059, -0.8539604544639587, 1.3994266986846924, 0.2618565857410431, -1.7265405654907227, -0.6558769941329956, 0.6340780258178711, 1.3476145267486572, 1.752793312072754, 0.18517877161502838, -1.0347909927368164, -1.892870545387268, 0.4603668451309204, 0.5405884981155396, 0.19218307733535767, -0.872097373008728, -0.2120080590248108, 2.010612726211548, -0.42413902282714844, -1.0623432397842407, -1.1574524641036987, -0.7847273945808411, 1.7627439498901367, 0.18572326004505157, -0.9710010886192322, -0.15757150948047638, -1.4154078960418701], [-0.9389414191246033, 0.8586532473564148, -0.9536478519439697, 1.4026331901550293, -0.801718533039093, -0.22788755595684052, 0.8638091683387756, -0.14773938059806824, 1.3940098285675049, -1.521504521369934, 0.9079343676567078, -0.5736343264579773, 0.040092457085847855, 0.9808877110481262, -1.023071050643921, 0.9823772311210632, -1.4210011959075928, 1.0840511322021484, 0.5765699148178101, 0.05162734165787697, -0.23036813735961914, 0.44424912333488464, -1.2768160104751587, -1.4111088514328003, -0.40309855341911316, -0.16664613783359528, 0.4229128956794739, -1.3721333742141724, -0.04431099444627762, 0.7740611433982849, -0.053824882954359055, 0.5431603193283081, 0.3523256182670593, 1.5468273162841797, 0.8732019662857056, -0.3203423023223877, -0.9601350426673889, 2.357797622680664, -1.2296950817108154, 0.16502217948436737, 0.2509765923023224, 1.9139312505722046, 0.8976900577545166, -0.863908052444458, -0.6548640131950378, 0.15273533761501312, 0.913069486618042, 1.1760611534118652, -1.5295088291168213, -0.3901248574256897, -0.722743034362793, -2.0653278827667236, 0.4868890047073364, -1.1638554334640503, 0.059346266090869904, -0.20238618552684784, 0.3858749568462372, -0.23874981701374054, -0.9418673515319824, -0.5103363990783691, -0.22561690211296082, -1.7408713102340698, -1.496909260749817, 0.4453689455986023], [0.6625651121139526, -0.4433211088180542, 0.44764384627342224, 0.4197987914085388, 1.061331868171692, 0.16334712505340576, 0.446850448846817, 1.266608476638794, -0.6259452700614929, -0.258556991815567, -1.4347718954086304, -1.3279855251312256, -1.2335529327392578, -1.3729219436645508, -0.08100682497024536, -0.3216901421546936, 1.9436538219451904, 1.2808692455291748, -0.11205561459064484, 1.0009570121765137, -0.21549224853515625, -0.04315852001309395, 1.2050033807754517, 1.0537605285644531, -0.9417451620101929, -0.2791809141635895, -0.9524105787277222, -0.8650596141815186, 0.29248708486557007, 1.209139108657837, -0.5326158404350281, 0.11807545274496078, -1.3260328769683838, -0.34483739733695984, 0.4229625463485718, -1.2176827192306519, 0.6003639101982117, -0.5505265593528748, -1.3170374631881714, -1.0288807153701782, -0.10862318426370621, -0.47064757347106934, -1.3058183193206787, 0.6148233413696289, 0.342324435710907, -0.2907104194164276, -0.11936210840940475, 0.8187763690948486, -0.5408757925033569, 0.00835030060261488, 0.26851433515548706, -1.4107890129089355, -0.4524790942668915, 1.5564494132995605, 1.4695909023284912, 0.5362366437911987, -0.5271303653717041, -0.263479620218277, 0.016773030161857605, 0.31537503004074097, 0.17650234699249268, -0.5869859457015991, 1.067976474761963, -0.8607656359672546], [0.7631015777587891, 1.4559338092803955, 0.4287804067134857, -0.28625938296318054, -2.6311051845550537, 1.1936184167861938, 1.4053318500518799, -0.056618768721818924, 0.33227023482322693, 0.5392254590988159, 0.3061976134777069, -0.7595224380493164, -0.3928467631340027, -1.270014762878418, 0.9001995921134949, 0.6659714579582214, 1.98585045337677, -1.182450532913208, -0.19620610773563385, -0.11143781244754791, -0.3337741196155548, 0.13646583259105682, -0.11269709467887878, -0.6348387598991394, -0.21566730737686157, -0.5881375670433044, 1.2117100954055786, -0.49229657649993896, -1.3204714059829712, -0.2260860949754715, -1.197218656539917, 0.07512439787387848, 0.12357188761234283, -0.7278842926025391, 1.0523900985717773, -0.7627328038215637, 0.5464094281196594, 0.49573925137519836, -0.9952153563499451, 0.19806016981601715, 0.5484898090362549, -0.288287490606308, 0.5082020163536072, 0.36439135670661926, -0.7580965161323547, 0.9314829707145691, -0.8097376823425293, -0.17494820058345795, 0.2446538805961609, -0.9209794402122498, -0.3994826376438141, 1.0761418342590332, -0.7174903750419617, 1.4020599126815796, -1.466352939605713, 0.010600385256111622, 1.4958144426345825, -1.1131680011749268, 0.7792298197746277, -0.29731810092926025, -0.3659686744213104, -0.4261714816093445, -1.8619122505187988, -0.09007216989994049], [-1.6904629468917847, -1.2163262367248535, 0.08176849782466888, -2.140324592590332, 0.7139079570770264, -0.8798079490661621, 0.3279481530189514, 0.10442864894866943, 1.2034924030303955, -0.08480630069971085, -0.27476558089256287, 0.845099687576294, -1.1445338726043701, -0.4713309109210968, -1.6362913846969604, -1.4373526573181152, -1.1232725381851196, 2.2524099349975586, 2.4082489013671875, 0.1320929378271103, -0.3303682804107666, -0.6551176309585571, 1.8003506660461426, 0.8380928039550781, -1.2607603073120117, 0.15172454714775085, 2.142789363861084, -0.5768862962722778, -0.22168096899986267, -0.27530795335769653, 1.6438074111938477, -0.18021364510059357, -0.6019604206085205, 0.8629665970802307, -0.8141658306121826, 1.6626849174499512, 0.3920193910598755, 0.4021841585636139, 1.1462870836257935, 1.168611764907837, 0.5211985111236572, -0.18360960483551025, 1.1529978513717651, -1.6535656452178955, -0.07286595553159714, -0.6018948554992676, -3.1850826740264893, -0.6950860023498535, 0.3605163097381592, 0.6465437412261963, 0.6029210090637207, -1.8374927043914795, -0.6065263152122498, 0.4224570095539093, -0.09379478543996811, 1.9483762979507446, -1.1695209741592407, -0.25443825125694275, 1.040144920349121, 0.405944287776947, 0.40857386589050293, -1.4146863222122192, -1.2453469038009644, -0.6436789631843567], [-0.3571300506591797, -0.3116327226161957, -0.04493436962366104, 0.726892352104187, 0.37684422731399536, 0.4646076261997223, -0.5617304444313049, -0.573874294757843, -0.3196546137332916, 0.10583218932151794, 0.06260602921247482, 1.303484559059143, -0.7509085536003113, -0.4367200434207916, 0.20892246067523956, -0.19696733355522156, -0.25846704840660095, 0.8166622519493103, 0.728489339351654, 0.6935450434684753, -0.007912646979093552, 0.48240184783935547, 0.6600342392921448, 0.3006336987018585, -0.9930435419082642, -1.017773985862732, -0.008633793331682682, -1.0099542140960693, -0.8188064694404602, 0.43640458583831787, 0.6591635346412659, -1.6171598434448242, 0.13217675685882568, -0.035406142473220825, 0.25676429271698, -0.45860958099365234, -0.39318400621414185, 0.5316866636276245, -1.047277808189392, 0.5257636308670044, 0.4961681365966797, -0.39551419019699097, -0.615029513835907, 0.038718659430742264, 0.645556628704071, 0.3238248825073242, 1.1675302982330322, 1.5667304992675781, 1.5668644905090332, -0.966923177242279, 0.31754443049430847, -1.0868337154388428, 0.27661633491516113, -0.580205500125885, 1.0159653425216675, 0.6279154419898987, 1.283399224281311, 2.8670599460601807, -2.8346750736236572, -0.12539105117321014, -0.253776490688324, 0.3483615219593048, 0.11433950811624527, 0.2705041766166687], [0.8714864253997803, -0.230600506067276, -0.1721847802400589, -0.683174729347229, 0.06825036555528641, 0.5454875230789185, -0.7256373763084412, -0.05606833100318909, -0.2901877462863922, -1.0747114419937134, 0.5802562832832336, 0.7854791283607483, -0.02701530046761036, 1.241782784461975, 0.2694283425807953, -0.7085946798324585, -2.631322145462036, -0.4152401089668274, -0.44639503955841064, 0.34006771445274353, 0.2747352421283722, -1.5715162754058838, 0.39663010835647583, -1.066361427307129, -2.8237221240997314, 0.9209019541740417, 0.1327550709247589, -0.40475085377693176, -0.3800571858882904, 1.0418877601623535, -1.3605760335922241, -1.027356743812561, -1.1096727848052979, 1.1022136211395264, 0.5081177353858948, -0.11489449441432953, 0.7061768174171448, -0.6569194793701172, -1.0423524379730225, 0.9599167108535767, -1.072861671447754, -0.8256291151046753, -1.56143057346344, 1.1486451625823975, -0.5887597799301147, 0.7597559690475464, 0.33708420395851135, -0.6303848624229431, 0.9393934011459351, -0.06450194120407104, 0.724471926689148, 0.4152720272541046, -0.30458810925483704, -0.6735740900039673, -1.116087555885315, 0.7620155811309814, -0.04319137707352638, -0.9804452061653137, 1.2467771768569946, -0.16092677414417267, -0.5149791836738586, 0.38398829102516174, 1.5483970642089844, -0.7364452481269836], [-1.8531826734542847, 0.5751078724861145, 0.019955476745963097, -0.3801290988922119, 0.5607262849807739, -1.5232127904891968, -0.30101388692855835, 0.22821247577667236, -0.8121591806411743, 0.8979605436325073, -0.0027319358196109533, -1.3947466611862183, -0.39520934224128723, -0.4326056241989136, 0.007151959929615259, 0.4038235545158386, 0.38131090998649597, 1.9164105653762817, 2.0418660640716553, 1.0462286472320557, -0.3980411887168884, -1.320983648300171, 1.403955101966858, 0.05671094357967377, 0.6523199081420898, -0.43254324793815613, 1.103031873703003, -0.5704027414321899, -0.34546783566474915, -0.2589680254459381, -0.2021854817867279, 0.25235509872436523, 0.19566094875335693, -0.39799046516418457, -0.26634615659713745, -0.4144361913204193, 0.8477921485900879, 1.0543828010559082, -0.0009899898432195187, -0.28664812445640564, -1.1506948471069336, -1.7135777473449707, 0.19022251665592194, -0.17822612822055817, -1.1200000047683716, -0.5354771614074707, -0.0035796973388642073, 0.13033874332904816, 1.776381254196167, -0.021896036341786385, -1.1090670824050903, -1.2661736011505127, 0.27640411257743835, -0.2969399094581604, -2.1907455921173096, 1.2212927341461182, 0.21036195755004883, 0.8236870765686035, -0.7108004689216614, -0.025189513340592384, 1.7727519273757935, 1.1724658012390137, 1.4003496170043945, 1.5639631748199463], [0.9819134473800659, 0.051106978207826614, -0.766471266746521, 0.26446112990379333, 4.528099060058594, -0.5473648309707642, -0.06171491742134094, 0.24994495511054993, 0.9005752205848694, -0.6621305346488953, -0.04632720723748207, 2.378188133239746, 1.3570541143417358, 0.2627449631690979, -1.2042906284332275, 0.42776286602020264, 1.4653527736663818, -0.07582750171422958, -0.2316378951072693, 0.32459092140197754, 1.543845534324646, 0.7959599494934082, -1.1510162353515625, -0.40200650691986084, -0.6927193403244019, -1.2378662824630737, -0.7157668471336365, -0.581606924533844, 0.254505455493927, -0.5901006460189819, -0.5468810200691223, -0.02249983511865139, 0.7334126830101013, -0.39492714405059814, -1.3086826801300049, -0.7732113003730774, 0.3780534267425537, 0.9857895970344543, -0.8240052461624146, 1.2957459688186646, 1.7460805177688599, 0.9840703010559082, -0.1761072874069214, -0.9585814476013184, -0.5982276201248169, -0.6098280549049377, -1.0165977478027344, -0.4497460722923279, -0.8595421314239502, -0.07068663835525513, 0.5270912647247314, -1.6199394464492798, -0.596868097782135, -0.6890312433242798, 0.16227658092975616, 1.0465278625488281, 0.5762686133384705, 0.31326207518577576, 0.5726742744445801, -0.9676131010055542, 0.2345767468214035, 0.20687884092330933, -1.0116175413131714, 0.2280246913433075], [0.49844977259635925, 1.9910709857940674, -0.2894696295261383, 0.26758506894111633, -0.23370207846164703, -0.2631383240222931, -1.0487736463546753, -0.4914109706878662, -1.4870761632919312, -0.8239482641220093, 1.6464641094207764, 0.5925680994987488, -0.36356088519096375, 0.8135951161384583, -1.0852253437042236, -0.02828109636902809, 2.066141366958618, -1.2257006168365479, 0.43271902203559875, -0.3017316162586212, 0.14545831084251404, -0.14128045737743378, -1.5727953910827637, 1.9212511777877808, -0.5274868011474609, -0.23876842856407166, 0.5538238286972046, 0.644307017326355, -1.68035888671875, -0.795757532119751, -0.1702987253665924, -2.42583966255188, 0.11270911991596222, -0.04508213698863983, 0.5975645780563354, 0.2077649086713791, 0.17120292782783508, 0.27548912167549133, -1.4312478303909302, -1.3671998977661133, -1.2518949508666992, -2.3404572010040283, 0.7697360515594482, -2.268510103225708, 0.19192321598529816, -1.6960968971252441, 0.7666382789611816, 0.0496959462761879, 0.02926958166062832, 1.2790285348892212, 0.2374524176120758, -1.320900559425354, 1.3521322011947632, -0.3056032955646515, 1.609988808631897, -0.7447879314422607, -0.25868573784828186, 0.6760613918304443, 0.8054176568984985, 1.2031936645507812, 0.8589191436767578, 1.5883809328079224, 1.55233633518219, 0.9857364296913147], [0.5131049752235413, 0.5103688836097717, 0.7834703922271729, -1.6245561838150024, -0.688304603099823, -1.347267746925354, -0.39729881286621094, -0.22568486630916595, 1.0327671766281128, 0.10954615473747253, -0.5932667851448059, -0.7667669653892517, 0.5646756291389465, -0.031397540122270584, -0.8650144338607788, 0.6216845512390137, -0.3678467571735382, 1.1159766912460327, -0.1379466950893402, -0.010189197957515717, 0.44085752964019775, 1.5182859897613525, -1.6728456020355225, -1.5846326351165771, -0.4072133004665375, 1.2781420946121216, 0.013818939216434956, 0.17249426245689392, -0.10687470436096191, -1.397630214691162, 0.7108827233314514, -0.19342520833015442, 0.5130883455276489, -0.7150583863258362, -0.7587121725082397, 1.180458426475525, -0.44047024846076965, 0.6454337239265442, 0.9139595031738281, -1.4365310668945312, 1.1323449611663818, 0.08160267025232315, -2.3133351802825928, -0.3252706825733185, -0.7016269564628601, 1.7096823453903198, 1.1657869815826416, 1.9240986108779907, 1.2308735847473145, 1.130040168762207, -0.7534153461456299, -0.8588849306106567, -2.6376194953918457, 0.14204610884189606, -0.06236693635582924, -0.29289761185646057, -0.7266058325767517, 1.5947542190551758, 0.5286194682121277, -0.578622579574585, 0.5938574075698853, 0.20746169984340668, -0.4606676697731018, 0.45059284567832947], [0.28030094504356384, -0.8169374465942383, -0.13353581726551056, -0.04518749937415123, 1.6148849725723267, -0.10625304281711578, 1.0483051538467407, 0.4438502788543701, 1.5384266376495361, 0.09901917725801468, 2.8182144165039062, -1.3372184038162231, -0.23375080525875092, 0.4733431339263916, 0.7891275882720947, -0.5157313942909241, 0.24417509138584137, 0.6511634588241577, 1.5706489086151123, 0.8293654918670654, 1.441846489906311, 0.4678781032562256, -0.9702507853507996, -0.46921300888061523, 0.6181798577308655, 0.001064202981069684, -0.005273886024951935, 0.24928295612335205, 0.16694758832454681, 0.15995587408542633, 0.25664207339286804, 0.5045489072799683, -1.6649513244628906, 0.3379359841346741, -0.6776567697525024, -1.2506482601165771, -0.015952954068779945, 0.9629379510879517, -2.316134452819824, -0.21030667424201965, 0.13430264592170715, -2.2390902042388916, -0.8470133543014526, -0.39377760887145996, -0.5199034214019775, 0.875892698764801, -0.07336746901273727, 0.4961390197277069, 0.575797975063324, 0.44057875871658325, 1.2908276319503784, 1.9311217069625854, 0.8626510500907898, 0.17046146094799042, -0.45738255977630615, -0.9842987060546875, -0.6740337014198303, -0.34648749232292175, 1.1383068561553955, -0.3182818591594696, 0.12041434645652771, 1.176568627357483, 0.7174041867256165, 0.09099540114402771], [0.18193483352661133, 0.8152327537536621, -0.8307231664657593, 0.2477046102285385, -0.7200594544410706, 0.031063780188560486, -0.5863063931465149, 1.2949328422546387, 0.36272719502449036, -0.3385457396507263, -1.0989502668380737, 0.07461177557706833, 0.9054735898971558, 0.06830354779958725, -2.0592141151428223, -0.35544338822364807, -0.3413890600204468, 0.2581612765789032, 0.1978921890258789, 0.5082757472991943, 0.26519525051116943, 0.8206202983856201, -1.2698469161987305, 0.01689152605831623, -0.7054096460342407, -0.057969823479652405, -0.29669177532196045, 0.47772231698036194, 0.641867458820343, -1.7030041217803955, 0.06932715326547623, -2.137014389038086, -0.917722761631012, 0.46996092796325684, 0.6133231520652771, 0.14839911460876465, -0.5928117036819458, -0.7043001651763916, 0.7719364166259766, -0.05457274615764618, -1.2621556520462036, -2.6704177856445312, -0.1883394420146942, 0.30255264043807983, -2.459839344024658, 1.5824111700057983, 0.6304231286048889, 1.0463393926620483, -0.6699048280715942, 0.1208510622382164, 0.6011061072349548, -1.0443376302719116, -0.33984947204589844, -1.6634916067123413, 0.3492553234100342, 0.42279529571533203, 2.2306737899780273, -0.5033565163612366, -0.7289785146713257, 0.5852304100990295, -1.5231233835220337, -0.9493889212608337, 0.47508320212364197, 0.4013362526893616], [1.8221406936645508, -1.0684322118759155, -1.0598770380020142, 0.2975975275039673, -1.181586742401123, -2.131183385848999, -0.6026066541671753, 0.8607448935508728, 0.9299806952476501, -0.7226256132125854, -0.9741036295890808, 0.1067657619714737, -1.8645374774932861, -0.9446126818656921, 1.1431149244308472, -0.2206648290157318, -1.747001051902771, -1.0233287811279297, -0.5070479512214661, -0.3817223310470581, -0.08060560375452042, -0.902556300163269, 0.14401020109653473, 0.4588204622268677, -1.1145586967468262, 0.1932554841041565, 1.4042904376983643, 0.28483474254608154, -0.6666746735572815, 0.256344199180603, -1.0694612264633179, -0.41615718603134155, 0.01076308824121952, -0.4219094514846802, -1.081505298614502, 1.1075447797775269, -1.5874502658843994, 1.4638559818267822, 1.1752634048461914, 0.17994001507759094, 0.21534018218517303, -0.13418473303318024, 0.8428825736045837, 0.44397449493408203, 0.4810653626918793, 0.26529422402381897, -1.5711283683776855, -0.4666357934474945, 0.4322364032268524, 0.33944761753082275, 1.0232292413711548, -0.223502516746521, -0.180852472782135, -1.317579984664917, -0.4328678548336029, 1.0945720672607422, -0.7846588492393494, -0.132017582654953, -2.191601037979126, 0.25805339217185974, 0.2400965690612793, -0.1861032247543335, -1.6458015441894531, -1.057261347770691], [0.805824875831604, -0.5786218047142029, -0.38615643978118896, 0.8670237064361572, -0.5172384977340698, 0.9025017619132996, 0.07657209038734436, -1.1158514022827148, 0.1262367069721222, -1.1927168369293213, -0.15967202186584473, -0.7704101204872131, 1.009202480316162, -0.13661538064479828, 0.44154515862464905, 0.23095828294754028, -1.1420791149139404, 0.5493685007095337, -2.312056303024292, -0.7030104398727417, 1.1318517923355103, 0.2872443199157715, -1.8689714670181274, 1.2503944635391235, -0.8083896040916443, -1.0701618194580078, 1.653591513633728, -0.29977676272392273, -0.4107392132282257, -1.1939032077789307, -0.543559730052948, 0.41950297355651855, 1.9177500009536743, -0.3169621229171753, -0.10783857852220535, -0.9298937320709229, -1.6713835000991821, -0.06455688178539276, 0.14402908086776733, -0.6204595565795898, 0.631580114364624, 0.31581535935401917, 0.132600799202919, -0.1888633370399475, 1.0991981029510498, 0.9422498941421509, 2.577348470687866, 1.932466745376587, -0.12694001197814941, -0.3124018609523773, -0.23010489344596863, -0.05776219442486763, -1.3830504417419434, -1.8291946649551392, -1.1685535907745361, 0.7142876982688904, -2.675248146057129, -0.6650707721710205, -1.1088554859161377, -0.8063421845436096, 0.5647501349449158, -0.8323883414268494, 0.16237857937812805, -0.4275156557559967], [2.4226670265197754, -0.08581352233886719, 0.5647186040878296, 0.018392767757177353, -0.3385154604911804, 0.5949903726577759, -1.0220005512237549, -0.09552726149559021, -0.06551650911569595, 0.5297324657440186, -0.9726383686065674, -0.9490318894386292, -0.47418496012687683, -1.4360028505325317, 0.051768958568573, 2.3080341815948486, -0.9437879920005798, 0.9449293613433838, 0.2947165071964264, 1.7026537656784058, -0.6035447120666504, -1.779922366142273, -0.6146323680877686, -1.6698201894760132, 0.911136269569397, -0.18406222760677338, -0.46383458375930786, 0.48738667368888855, -1.1114444732666016, 0.4752330482006073, -0.6573379635810852, 0.1256977766752243, -0.6734502911567688, 0.4054957628250122, 2.621453046798706, 1.2082555294036865, 0.8734456300735474, 1.0366623401641846, 1.35677969455719, 1.2062079906463623, 0.8746020793914795, -1.2704955339431763, 1.5686428546905518, 0.04612761363387108, -0.12808506190776825, -1.3921353816986084, 0.523612380027771, 0.22069592773914337, 0.1892150640487671, 0.3254474401473999, 0.550961434841156, -1.534791350364685, 1.8871583938598633, -0.7574371099472046, 0.08064468204975128, 0.5664638876914978, 0.610823392868042, -0.6060823202133179, -0.18116679787635803, 0.03496994078159332, -2.0373833179473877, 0.8053544163703918, 0.6291490793228149, 2.1288907527923584], [1.7862378358840942, 0.18569785356521606, 2.3733389377593994, 0.05941574275493622, -0.7756116390228271, 0.1571151316165924, 0.36425331234931946, 0.6486042141914368, -0.8676347732543945, -1.1362626552581787, 1.358392357826233, 0.035601671785116196, 1.2920317649841309, 0.033771585673093796, 1.3449469804763794, -0.20131030678749084, -0.0307808555662632, -0.1866472065448761, -1.4923100471496582, 0.7265682816505432, 0.1980307698249817, -0.1554373949766159, -0.24318447709083557, -1.7925734519958496, 0.8214755058288574, -0.2512892186641693, -2.61006760597229, -0.25885820388793945, -1.9063661098480225, 1.1486340761184692, 0.711338222026825, -2.069706678390503, -0.9075145721435547, 0.012475772760808468, -0.3459470570087433, -1.2003670930862427, -0.5934295654296875, 0.5843662619590759, -1.106019139289856, 0.9351005554199219, 0.717764675617218, 2.0202090740203857, 0.5746579170227051, 1.729636311531067, 1.1420954465866089, 0.8200594782829285, -0.3459573984146118, 0.013601982966065407, -0.5169364809989929, -1.0548053979873657, -0.7038385272026062, 0.18413609266281128, -0.8577828407287598, 0.8157011866569519, -0.5951441526412964, 0.03540642186999321, -0.028180068358778954, -0.06603702902793884, -0.8211561441421509, -1.0308607816696167, 0.9352630972862244, -0.013513528741896152, 2.364879846572876, -0.01618942618370056], [-1.8201481103897095, -0.4495912790298462, -1.6597511768341064, 1.0011589527130127, -0.5890416502952576, 0.7811460494995117, 1.738860845565796, 0.2223903089761734, 0.2145162969827652, 0.2856127917766571, -0.8031604886054993, -0.36986473202705383, 0.80005943775177, -1.8296215534210205, -0.04846058040857315, 1.7682757377624512, 0.24398569762706757, 0.8978433609008789, -1.549232840538025, 0.22126919031143188, -0.8651995062828064, 2.0896565914154053, -1.2357286214828491, 1.0180760622024536, -0.5893309116363525, -1.0195436477661133, -0.8293412327766418, -0.523572564125061, -0.6660016775131226, 0.5028786659240723, -0.3934696614742279, 0.320615291595459, -1.5459953546524048, -0.28175631165504456, -0.010955951176583767, -0.4522067904472351, 1.4877766370773315, 0.05181273818016052, 1.341505527496338, 2.012573480606079, 0.041926298290491104, -1.0329327583312988, -0.4175379276275635, 0.46691858768463135, -1.5255502462387085, 0.6791440844535828, -1.9195094108581543, 1.0452370643615723, 0.038619063794612885, -0.676798939704895, -0.29244503378868103, 0.112189881503582, 0.29229629039764404, 1.7239694595336914, -2.0120184421539307, 0.1495516300201416, -0.6580609679222107, 1.6216877698898315, -2.792405128479004, 0.11248350888490677, 0.3371530771255493, -0.024509208276867867, -0.06578691303730011, -0.3664630055427551], [1.2656115293502808, 1.3412508964538574, 1.288353681564331, 0.3393727242946625, -1.5057889223098755, 0.003693582257255912, 0.12478924542665482, -0.01622169278562069, 1.4401872158050537, 0.533873975276947, -0.9328035116195679, -0.6116886138916016, -1.453771948814392, 1.7370879650115967, 1.4959826469421387, 1.4371259212493896, -0.21007314324378967, 1.6136314868927002, -0.45301783084869385, 0.285170316696167, 0.01018517930060625, 0.18895994126796722, 0.7704911231994629, 0.770155668258667, -1.5271821022033691, -0.034512460231781006, -0.4650816023349762, 0.5367650389671326, 1.9927643537521362, 0.0761418417096138, -0.2408500760793686, 0.36749398708343506, -0.5773681402206421, -0.5989954471588135, -0.48365065455436707, 0.18906374275684357, -2.031277894973755, 0.3944627046585083, -0.11658800393342972, -0.31195545196533203, -0.14426790177822113, -0.3442799746990204, 0.4883994460105896, 0.8711374402046204, -1.8540488481521606, 0.2903780937194824, -0.5914223790168762, -0.5852323770523071, 1.8613085746765137, 0.8660931587219238, 0.8840749859809875, 0.40712180733680725, -1.0072407722473145, -1.9476579427719116, 0.051731813699007034, 2.0780999660491943, -1.1102588176727295, 0.29968374967575073, 0.07342828065156937, -0.11833076179027557, -1.6732776165008545, -0.02812301740050316, -0.1311914324760437, -0.4372314214706421], [-0.35694220662117004, 0.06123284995555878, 1.054787278175354, -1.0773686170578003, -0.9509587287902832, -1.2195677757263184, 0.6969624161720276, -2.080958604812622, -0.6867640614509583, 1.3693801164627075, 1.0081007480621338, -0.34011805057525635, -0.6594576239585876, 0.4856531620025635, -0.8394131064414978, -0.057695962488651276, -1.251795768737793, -1.663352608680725, -0.2275361865758896, -1.187495231628418, 0.5569999814033508, 0.1843366175889969, -3.0921356678009033, -1.0134357213974, 1.8508083820343018, 0.7913557291030884, 2.8667619228363037, 0.08017779886722565, -1.2405081987380981, 0.7052721381187439, -0.4899485111236572, -1.3805657625198364, 1.0908246040344238, 0.38605406880378723, 0.2992815375328064, 1.80801260471344, -2.2023444175720215, 0.6539175510406494, 0.6959123015403748, -1.682610034942627, 1.2445719242095947, -1.2920573949813843, -0.17187392711639404, -0.4074011743068695, 1.2073017358779907, -0.7388037443161011, -0.8735748529434204, 0.6701216697692871, 0.3305201530456543, -0.3412814736366272, -0.7473475933074951, -0.14120352268218994, -0.5340683460235596, 0.627963125705719, 0.08823644369840622, 0.021656813099980354, 0.2016616314649582, -0.4915536940097809, -1.153472661972046, -0.7987843751907349, -0.29216083884239197, -1.7722028493881226, -0.04668111726641655, -0.506627082824707], [0.3574577569961548, -0.497010737657547, -0.9403583407402039, 1.0974853038787842, -0.42028263211250305, -1.375827670097351, -0.7709831595420837, 0.3287920355796814, -1.352785587310791, 0.9783014059066772, -1.500785231590271, -1.1990076303482056, 0.3689987063407898, -0.5717056393623352, 0.6098632216453552, 0.10312783718109131, -0.9542528986930847, -1.0936717987060547, 2.284083366394043, -0.828725278377533, -0.5583553910255432, -1.851714015007019, -0.573697566986084, 1.444739818572998, -0.4658609926700592, 0.044338520616292953, -0.8944250345230103, -0.8483310341835022, 0.9151071906089783, 0.9585349559783936, 0.6949580907821655, 0.596796989440918, 0.26206329464912415, 1.2043590545654297, -1.0487524271011353, 1.6481728553771973, -1.1704126596450806, -1.1543477773666382, 0.8897722363471985, 0.9445759057998657, 1.6488919258117676, 0.09211380034685135, 1.2336442470550537, -0.3974166512489319, -0.44879940152168274, -0.8926771283149719, -1.520471453666687, 1.6326982975006104, -0.07439852505922318, 1.8030482530593872, -0.4621196985244751, 0.18626050651073456, 0.7817714214324951, -1.1229842901229858, 0.3302643895149231, -0.5040808916091919, -0.5131139159202576, 0.21636591851711273, -0.6280289888381958, -1.4368804693222046, 0.3037209212779999, 1.4902020692825317, -1.2857387065887451, 0.23192989826202393], [1.230545997619629, 0.051775697618722916, -0.24838677048683167, -0.3890337646007538, 0.5579243898391724, -2.3219947814941406, -0.15936417877674103, 1.0843549966812134, -0.4027692973613739, -1.116687536239624, 1.6170848608016968, 1.2344564199447632, 0.8454408049583435, 0.3017062246799469, 1.4126099348068237, -0.8046709299087524, -0.7838191986083984, 0.7654950022697449, 1.0885331630706787, 0.2048632651567459, -0.9909752607345581, -0.1180635392665863, -2.060927152633667, 0.5229699015617371, 1.030038595199585, -0.501446008682251, 0.18701577186584473, 1.2256749868392944, 0.8744609355926514, -0.3922298550605774, 0.20336385071277618, 0.3786879777908325, 0.08958959579467773, -0.7634917497634888, 1.0525131225585938, 0.274582177400589, -0.11632551997900009, 0.9587435722351074, -1.6096173524856567, 0.0027499543502926826, -0.8768853545188904, 0.4596981704235077, -0.17403662204742432, -0.7192109227180481, -0.41370129585266113, -2.2362899780273438, -0.45870184898376465, -0.31318363547325134, -0.9580084085464478, -0.497598797082901, -2.5610694885253906, 0.38056328892707825, 0.2776150107383728, 2.2238643169403076, -0.9068543314933777, 1.4092719554901123, -0.6846181750297546, 1.5876384973526, -0.1894347071647644, -0.8789272904396057, 0.6653140187263489, -3.2759037017822266, -1.1787816286087036, -0.5875270366668701], [-0.5605406761169434, 1.2558754682540894, 0.4647124707698822, 0.42709624767303467, -1.9702397584915161, 0.8380990028381348, 0.7724612951278687, -0.24736745655536652, -0.18968795239925385, -1.0222935676574707, 0.7556647062301636, 0.3012895882129669, -0.6170680522918701, 1.5616397857666016, -1.0441811084747314, 0.08137936890125275, -2.1785192489624023, 0.22714489698410034, -1.747177004814148, -0.24649517238140106, -1.6310417652130127, -1.3935585021972656, 0.35230231285095215, -0.09009313583374023, 0.7155565619468689, 0.14642731845378876, 1.0617965459823608, -1.1381555795669556, 0.6969247460365295, 0.2241160273551941, -1.0821539163589478, -1.6353687047958374, -1.3164033889770508, 0.3410271406173706, -0.8549069166183472, -0.774869441986084, 1.470336675643921, 0.8695546388626099, -1.3834418058395386, -1.5634130239486694, 0.6330335140228271, 1.8228832483291626, 1.133437156677246, -1.1078224182128906, -1.4934149980545044, 1.2237154245376587, 0.43979355692863464, -0.3632234036922455, -1.108177661895752, -0.9969732761383057, -0.46110498905181885, -0.7588750720024109, -0.670078694820404, -1.0017364025115967, -0.4094180166721344, 0.3356165289878845, -0.17076483368873596, 1.4438811540603638, -1.4379537105560303, -0.8698294758796692, -0.6779597401618958, 1.364478349685669, 1.592569351196289, -0.4812394976615906], [0.8861101865768433, 0.4477650821208954, 1.2428505420684814, 0.09486168622970581, 0.34237873554229736, 1.6716015338897705, 0.6712148785591125, 0.16893362998962402, -1.166555643081665, -0.5703045725822449, 0.9760453701019287, 1.0851472616195679, -0.08303854614496231, 1.2416085004806519, 0.004367851186543703, 0.4460260272026062, -0.5648086667060852, 2.0298707485198975, 0.40782734751701355, 0.1682930290699005, -0.07730137556791306, 0.20123343169689178, 0.7264956831932068, 0.19746403396129608, 1.4930484294891357, 1.921041488647461, 0.6625503301620483, -1.2688038349151611, 0.03652124106884003, 2.0576353073120117, 0.5501890182495117, -0.6545595526695251, 1.4213277101516724, 1.2351298332214355, -0.03851025551557541, 0.6644005179405212, 0.32683172821998596, -1.3655771017074585, 2.552260637283325, -0.4809618294239044, -0.30608081817626953, 0.8892715573310852, 2.3357791900634766, -1.9249118566513062, -0.00332133611664176, 0.4402633607387543, 1.9316577911376953, -0.3182193338871002, -0.8371680378913879, -0.452157199382782, 0.17271865904331207, 1.0412321090698242, 0.5126670002937317, -0.7325825095176697, 1.6188091039657593, -0.39683595299720764, -0.37960338592529297, -0.33556288480758667, 2.2701663970947266, 0.21889442205429077, 0.4249227046966553, -0.5481521487236023, 0.7986307144165039, -0.8616434931755066], [0.3002222776412964, 0.45309978723526, -0.7027101516723633, 0.32526424527168274, 0.1526394635438919, -0.19121481478214264, 0.923373281955719, 0.7629517912864685, 0.5506743788719177, 1.2962563037872314, 0.1742505431175232, 0.24917636811733246, -1.0390734672546387, -0.460113525390625, -0.16562215983867645, -0.47016727924346924, 1.489797592163086, 0.01852835714817047, -1.6981539726257324, -0.3199811279773712, 0.045768383890390396, -1.9119313955307007, 0.6232866644859314, 1.704653263092041, 1.5249370336532593, -1.1198458671569824, 1.9778846502304077, -0.6250505447387695, -1.5133519172668457, 0.5182169079780579, -1.6401371955871582, -1.5372430086135864, -1.1530251502990723, 0.862864077091217, 0.8059313297271729, -0.20151536166667938, -0.9079384803771973, 1.2392712831497192, 0.739727795124054, -0.6165040731430054, -1.524040699005127, 2.112051010131836, -0.5313743352890015, 0.9199309945106506, 2.77270770072937, -1.2161730527877808, 1.2101370096206665, -0.6100457906723022, 0.5987197160720825, -0.024649545550346375, -0.1020653024315834, -3.1881446838378906, -2.448406219482422, -0.6951296925544739, -0.1762157380580902, 0.6245748400688171, -0.46265316009521484, -0.40802785754203796, -0.19676275551319122, -0.7154894471168518, -1.087436556816101, -0.9596808552742004, -0.455511212348938, -0.5928913354873657], [-0.7248556613922119, 0.031064096838235855, 0.7807395458221436, 1.7581331729888916, 0.7599861025810242, 0.4866653084754944, -1.7553269863128662, 0.6983761191368103, 0.9598606824874878, 1.4599127769470215, 0.17593912780284882, -0.9087165594100952, 1.960181474685669, -1.0376667976379395, 1.9519014358520508, -0.018273405730724335, -0.9659863114356995, -0.07067859172821045, -0.8730925917625427, 0.6378980875015259, 1.742813229560852, 0.7417003512382507, 0.6962320804595947, -2.4158308506011963, 1.7804211378097534, 0.2102494090795517, -0.1019933819770813, -1.1110721826553345, -0.6201949715614319, -1.0392552614212036, 1.509294033050537, -0.20597364008426666, 0.5813263058662415, -0.4769374430179596, -0.506543755531311, 0.6241638660430908, 0.30809396505355835, -0.07624661922454834, 0.38720840215682983, 0.7485634684562683, 1.2404961585998535, -0.26461121439933777, 0.9270989894866943, -0.4288703501224518, -0.4983886480331421, 2.1294119358062744, 0.36645835638046265, -1.2763389348983765, 0.8324966430664062, 0.2713034451007843, -0.992795467376709, -0.3866860568523407, 0.22523394227027893, -1.807186245918274, 0.2050931602716446, 0.002115225652232766, 0.9608064293861389, 1.0856571197509766, 0.399176687002182, 0.9835801720619202, 0.43948692083358765, 0.2873584032058716, 0.3094642460346222, -1.4858214855194092], [0.7891826033592224, -0.21934297680854797, 1.0960969924926758, 0.09995561093091965, -1.211320161819458, -0.3664090633392334, -0.26118186116218567, -0.7984284162521362, 2.894765853881836, 0.6133800745010376, 0.5452911257743835, 0.755517840385437, 1.1525814533233643, 0.5714293718338013, 2.8632709980010986, -0.30070286989212036, -0.515053391456604, -0.2426343709230423, 0.25223785638809204, 0.3118222951889038, 0.07829922437667847, -0.8250125050544739, -0.13436277210712433, 0.32210519909858704, 0.014099306426942348, 0.5704144835472107, -0.014237562194466591, 0.49203065037727356, 1.1120861768722534, 1.9352062940597534, -0.9687658548355103, 0.10756649821996689, -1.7126743793487549, -0.12840856611728668, -0.4526478350162506, 0.42119044065475464, -0.061588749289512634, 1.3362741470336914, 0.6736571192741394, 0.3811626434326172, 1.2221592664718628, -0.7599076628684998, 0.04093950241804123, 0.25967058539390564, 2.3243203163146973, -0.7025310397148132, 0.6549809575080872, -1.401168942451477, -0.256612628698349, -0.13210415840148926, -0.784892201423645, 1.5319602489471436, 2.1225249767303467, 1.1799182891845703, -1.8496705293655396, 2.472831964492798, -0.029309572651982307, 0.9577406048774719, 0.16645516455173492, 0.04565737023949623, -0.3477080762386322, 0.3337518572807312, -1.62258780002594, 0.1289805918931961], [0.3848545551300049, 0.5896903276443481, -0.7715246677398682, 0.2542855143547058, 0.42748698592185974, 0.9163482785224915, 0.8216834664344788, -0.045069269835948944, -0.02945464849472046, 1.3893074989318848, 1.812006950378418, 1.5142086744308472, -1.350914478302002, 0.25219789147377014, 0.5116147398948669, 0.4294796884059906, 0.8573004603385925, -0.0878540426492691, 1.065557599067688, 0.3850100636482239, -0.8003702759742737, 1.1681146621704102, 2.1522037982940674, 0.841168224811554, -0.40308496356010437, -1.8052668571472168, 1.4131314754486084, 0.7085573077201843, 0.49681541323661804, 0.04733016714453697, 0.5550697445869446, 0.9047254323959351, -0.8101711273193359, -0.6454972624778748, -0.22124382853507996, 0.4010276198387146, 0.7166630625724792, 0.6233394742012024, 0.6846868991851807, 0.04322420805692673, -0.6291787028312683, -0.1853201538324356, -1.3060396909713745, -0.8299652338027954, -2.06961727142334, 0.7702249884605408, 0.9661646485328674, 1.0215651988983154, -0.366111695766449, 1.1043024063110352, 0.6171583533287048, -0.35247257351875305, -0.17421624064445496, -0.3607092797756195, -0.03921618312597275, 0.229551762342453, 0.3770058751106262, 0.7106847167015076, -0.11701023578643799, 0.7573708891868591, 0.1840975433588028, -0.5313616991043091, 1.8014789819717407, -0.7273930907249451], [-1.108605980873108, 1.517749309539795, -0.9581322073936462, 0.4648233950138092, -1.212065577507019, 1.5461509227752686, -0.333726167678833, 1.0142567157745361, 0.17957323789596558, -2.65181040763855, 0.8919550180435181, -0.9022703170776367, 0.6024171113967896, -0.45594316720962524, 0.16184158623218536, 0.49888455867767334, 2.8524839878082275, 0.07161015272140503, -0.6843708157539368, -0.8915217518806458, -1.8706547021865845, -0.501434862613678, -0.17760802805423737, -0.02143721655011177, 1.654242753982544, -0.7517386674880981, -0.9822805523872375, 0.30996018648147583, 0.43456006050109863, -2.0658071041107178, -0.08386315405368805, 0.14111219346523285, 0.39730390906333923, -1.0497087240219116, 0.15017005801200867, 0.43639740347862244, 0.7762191295623779, 0.002253386890515685, 0.12211831659078598, 0.8544104099273682, -0.5703039765357971, -1.213854193687439, -1.108174443244934, 2.023015022277832, -1.5523852109909058, -1.2298966646194458, -0.2440527081489563, 0.7579118013381958, 1.086207628250122, 0.5146442651748657, -0.6214272975921631, 0.8652706742286682, -0.7347277998924255, -0.21146278083324432, 0.13351206481456757, 1.2554199695587158, 0.8943196535110474, 1.0669015645980835, -2.4134066104888916, 1.3298207521438599, 0.07098232209682465, -0.3105771541595459, -0.6097124218940735, 0.19383279979228973], [-3.0423786640167236, 1.5037579536437988, 1.6938538551330566, 0.8986001014709473, 0.6342250108718872, 1.2521131038665771, -0.9190546870231628, 1.1367467641830444, 1.1682629585266113, 1.0351430177688599, 0.8644383549690247, 0.9182700514793396, -1.2942713499069214, -0.26271557807922363, 1.266007423400879, 1.543901801109314, 1.7293308973312378, 0.4979183077812195, -1.0490574836730957, -0.8666189908981323, 2.7214126586914062, -1.762593388557434, -1.3984555006027222, 0.39151591062545776, 0.2815034091472626, 2.8613040447235107, 1.2054190635681152, -0.9065613150596619, -0.5301028490066528, -0.7490296363830566, -1.3837617635726929, -1.2020949125289917, -1.832297444343567, -0.37810018658638, 0.9720914363861084, -0.9176777601242065, 0.481893926858902, -0.5274640917778015, -0.3894028961658478, -1.0408098697662354, -0.12663137912750244, 1.285605549812317, 0.629474401473999, -0.06050698831677437, 1.268182396888733, -0.03854791447520256, 0.05957181379199028, -0.4423351287841797, -1.080341100692749, -0.7152531743049622, 0.5576445460319519, -1.6138668060302734, 0.06912606954574585, 0.6278581619262695, 1.1757005453109741, -0.1334577202796936, 1.1399483680725098, 1.0620193481445312, -0.4677916169166565, -0.11774156987667084, 1.3680529594421387, 2.268242120742798, -0.8735920190811157, -2.057271957397461], [-1.5107362270355225, -0.9182109236717224, 0.468440443277359, -0.49752864241600037, -2.2983999252319336, 0.27737659215927124, 0.26365926861763, 0.07593555003404617, -0.268051415681839, 0.9760565757751465, -1.200302004814148, -0.37361952662467957, 1.738953709602356, -1.1523436307907104, -0.14882288873195648, -0.1761784553527832, 0.14792503416538239, 1.3029096126556396, 0.28824833035469055, -0.4126616418361664, 0.2253018021583557, -0.9480178356170654, -0.8388245105743408, -0.6018637418746948, -0.22032153606414795, -1.193625807762146, -0.40553614497184753, -0.9535717964172363, 0.6802579164505005, -0.9459497332572937, 0.167949378490448, -0.9384728670120239, -0.9032581448554993, 0.04970955103635788, 0.8543328642845154, -0.878869891166687, -0.8190522789955139, 1.806492805480957, -1.1792768239974976, 0.740482747554779, -0.09313556551933289, 0.11681561172008514, 0.5637350082397461, 1.7862178087234497, -0.2328934669494629, 1.2373416423797607, 1.166254997253418, -1.358709454536438, 1.4639314413070679, -1.1646977663040161, -0.18450644612312317, -0.4483366906642914, -1.061050534248352, -2.335982084274292, -1.3207048177719116, 0.9129051566123962, 0.209712415933609, -0.4438384175300598, -0.6790496706962585, -0.697559118270874, -0.6465696692466736, 0.4058360159397125, 0.5461564660072327, 0.2712286114692688], [-1.6621955633163452, -0.34966644644737244, 0.3863847553730011, 1.4591461420059204, -0.39939403533935547, -0.27329906821250916, 0.7983536720275879, -1.0813769102096558, -0.8971377611160278, 1.9103325605392456, -0.22326843440532684, -0.6606063842773438, 0.607881486415863, -0.2762224078178406, -1.1538002490997314, -0.24258896708488464, 1.9041160345077515, -0.6708327531814575, 0.5533804297447205, 0.22553256154060364, 0.4956527352333069, -1.2219586372375488, 1.0922809839248657, 0.8360683917999268, 1.4047346115112305, 0.4467516541481018, -0.11318591237068176, -1.0517127513885498, -1.3382203578948975, -0.21864360570907593, 0.6051406264305115, -0.25532791018486023, 0.8657100200653076, -0.4443505108356476, -0.27113252878189087, 1.3595298528671265, -0.6944662928581238, -1.04314386844635, -0.2965269684791565, -0.5572881698608398, 0.44818437099456787, 1.2007662057876587, -0.26852136850357056, 0.6470427513122559, 0.8157436847686768, 0.7489946484565735, 0.09087208658456802, -0.714225709438324, -1.244458556175232, -1.626702070236206, -1.9816592931747437, 1.8697903156280518, 0.5433071255683899, -0.6811546087265015, 1.612043857574463, 0.04124664142727852, 0.6083692908287048, 1.0996406078338623, 0.4950721263885498, 1.257340669631958, -1.3236992359161377, 0.4417547583580017, 0.13885930180549622, 0.36462658643722534], [0.20020100474357605, -0.04028285667300224, 0.8802167177200317, 2.1705219745635986, -0.44009825587272644, 0.9582212567329407, -0.11537644267082214, 0.553862452507019, -0.2519795298576355, 0.2215922623872757, 0.17453987896442413, -1.1533124446868896, -0.057873740792274475, 1.461186170578003, -0.6247990727424622, 0.9015837907791138, -0.02692735567688942, 1.8529984951019287, -1.7635254859924316, -1.285510778427124, 0.9746770262718201, -1.4594969749450684, 1.6189535856246948, -0.8076702952384949, 0.4792686998844147, -1.5233699083328247, 0.43773210048675537, -0.7649684548377991, -0.3397396206855774, -1.2568315267562866, 1.548142433166504, 0.02270001731812954, -0.12647926807403564, 0.002415600698441267, -0.5405815839767456, 1.892061710357666, 0.6401572823524475, 0.15440210700035095, 0.44281432032585144, 0.6443873047828674, -0.6778934597969055, -0.8244379162788391, -0.008318272419273853, 1.112656593322754, -0.8185245990753174, -0.5130899548530579, -0.5316728353500366, -0.32891833782196045, 2.133863925933838, 0.3093043267726898, -0.2848067283630371, -0.07712723314762115, 1.6550878286361694, 0.9846739768981934, 0.6993865966796875, -0.321900874376297, -0.4692322015762329, -0.884645938873291, -1.7358074188232422, 0.789888858795166, -0.4124329388141632, 0.016346165910363197, -0.5898130536079407, -1.111649513244629], [0.4991186261177063, 1.2283239364624023, -0.04435718059539795, 0.9438774585723877, -0.7850056290626526, -0.2858290374279022, 2.06927227973938, 2.0041685104370117, -1.436381220817566, 1.4321513175964355, 0.23680183291435242, -0.3713141977787018, 0.7202736139297485, 0.3001329004764557, 0.3297620415687561, -0.7275322079658508, 0.30535608530044556, -1.3772677183151245, -0.29037296772003174, -0.28537875413894653, 0.8740251660346985, 0.2744891941547394, 0.22315441071987152, -1.4961521625518799, 0.44387155771255493, 0.49965226650238037, -0.07802771031856537, 0.2491654008626938, -1.356975793838501, 0.019771995022892952, -0.3609410226345062, 1.4225459098815918, 0.44047388434410095, 2.709690570831299, 0.6710537672042847, 0.2898348867893219, -0.42852094769477844, -1.7654651403427124, 0.5843261480331421, -0.8124788403511047, -2.541724443435669, 0.05877448245882988, -1.2272772789001465, -1.2641472816467285, 1.6172025203704834, -0.010187997482717037, 1.7664462327957153, 0.7976797819137573, -0.4133094847202301, 1.0143146514892578, 1.2947450876235962, 0.1138550192117691, 0.5737236142158508, -0.6697453260421753, -0.8723469376564026, 2.7714316844940186, 0.09837505221366882, 0.2318485826253891, -0.4870348870754242, 0.03429100662469864, -1.0899779796600342, 0.41861492395401, -0.5756540298461914, -1.051544427871704], [-0.881551206111908, -0.9161311388015747, -1.0506058931350708, 0.25694945454597473, -0.22163762152194977, -1.8840677738189697, -1.778388261795044, 0.39584505558013916, -1.6581941843032837, -0.7210900783538818, 0.3932461142539978, -0.02564993128180504, 0.8693593740463257, -0.46629583835601807, 0.5670414566993713, 0.5938529968261719, -0.08371739089488983, -0.33951830863952637, 1.6613409519195557, -1.2741007804870605, -2.4271340370178223, -0.49547091126441956, -0.39803674817085266, -0.5461317896842957, -3.4024431705474854, 0.8974737524986267, -0.42420050501823425, -0.753812313079834, 0.25143182277679443, -0.47024285793304443, -0.09609375894069672, -2.1872549057006836, -0.3786037862300873, -1.3634876012802124, -0.8940707445144653, -0.6074232459068298, 0.5810543894767761, -0.9067612886428833, 0.43064963817596436, 0.7435541749000549, 1.196918249130249, 0.43956923484802246, 0.5225597023963928, -0.1729862093925476, -0.22916579246520996, 0.5564014315605164, -0.9585438966751099, 0.01885545253753662, 2.5709033012390137, -0.8518858551979065, 0.7083045244216919, -0.35817739367485046, 0.4381670653820038, -0.005486877169460058, -0.48341891169548035, 1.0271869897842407, -1.90349280834198, 0.07081438601016998, 0.4306257665157318, -0.2512041926383972, -0.23879049718379974, -0.16618898510932922, -1.4825795888900757, 0.5486708283424377], [1.7177342176437378, -0.5352605581283569, -0.5295563340187073, -0.2674768567085266, -0.6564047932624817, 0.911371648311615, 1.7142698764801025, 0.21922144293785095, 0.5946996212005615, -1.0072559118270874, -1.6350582838058472, -0.755022406578064, 1.2435877323150635, -0.7568370699882507, 0.7919870615005493, 0.7955837249755859, 1.2590142488479614, 1.3438748121261597, -2.811931848526001, 0.16718757152557373, 0.18164584040641785, 1.3918125629425049, -0.9644896984100342, -0.455477237701416, 2.2200584411621094, 0.35889437794685364, 2.290231704711914, -0.9570331573486328, -1.929283618927002, 1.0828431844711304, -0.682428240776062, -0.8323537111282349, -0.1092766523361206, 0.5705132484436035, 0.775504469871521, -0.09176847338676453, 0.31672000885009766, -0.7797441482543945, 0.2814173102378845, 0.4135304391384125, -0.49861183762550354, 0.389242023229599, 0.1951119601726532, -1.0907875299453735, 1.1746169328689575, -0.33957117795944214, -1.1598740816116333, -0.8570137023925781, 0.9044889211654663, 0.5763401985168457, 0.29043254256248474, -0.21704769134521484, 0.5525850057601929, -1.4828294515609741, 1.3499277830123901, 0.9994875192642212, 0.26487278938293457, -0.8734527230262756, -0.4182984530925751, 0.6628946661949158, 0.6910164952278137, 0.19482338428497314, 1.4402016401290894, 0.18290507793426514], [0.6276447176933289, -0.4706684350967407, 0.2929410934448242, 0.22823059558868408, 0.38445910811424255, 1.3893147706985474, -0.819977879524231, 1.4462792873382568, 0.4973459243774414, -0.7145399451255798, 1.200840711593628, 1.357319951057434, 0.7548007965087891, -0.34186190366744995, -0.030717000365257263, 0.06670314818620682, 1.2360061407089233, -0.263264536857605, 0.9871453642845154, -1.1918338537216187, -0.6937476396560669, 0.1219513788819313, 2.3793041706085205, -1.1169198751449585, -1.3432462215423584, 0.30887454748153687, -0.25682780146598816, 0.47351667284965515, 0.8527318835258484, -0.2413366138935089, -0.8794381022453308, -1.0425677299499512, -0.607743501663208, 0.49076828360557556, -0.6404445171356201, -0.8572055697441101, -0.8262742757797241, 1.2639039754867554, -0.36036035418510437, -2.057544231414795, -0.5016615986824036, 0.9361851811408997, 0.22663068771362305, -0.6206637024879456, 1.114399790763855, 1.5032368898391724, 2.282186985015869, 0.5400609970092773, 0.23598720133304596, -0.2451082468032837, 0.44310393929481506, 1.844663381576538, -1.5507338047027588, -1.4336916208267212, 1.4161396026611328, 1.6291170120239258, 0.2626504898071289, -0.04565435275435448, 1.2847421169281006, -0.39728641510009766, 0.5005553960800171, 1.5423779487609863, 0.27825963497161865, 0.2533591091632843], [0.10734337568283081, 1.2078114748001099, -0.023566197603940964, -1.5027410984039307, -1.8209161758422852, 0.8269162178039551, 0.38765302300453186, 0.8334869146347046, -2.8593106269836426, 0.3428937792778015, 1.6782296895980835, -2.222038507461548, 0.4306189715862274, -1.1514109373092651, -0.5401790142059326, -1.3408045768737793, 1.1176735162734985, -0.7326687574386597, -0.4931548833847046, 2.5839791297912598, 0.5543655157089233, 0.1624433398246765, -1.2155545949935913, 0.677493155002594, 2.768998384475708, -0.6899412274360657, 1.2328592538833618, 0.0403057262301445, 1.0263577699661255, 1.548303246498108, 0.4518735706806183, 0.6742918491363525, 0.7974798679351807, -0.26213663816452026, 1.9060696363449097, 0.5543789267539978, -1.3389856815338135, 0.07141824811697006, 0.37122607231140137, -0.5620158910751343, 0.4874689280986786, 0.8487169742584229, 0.29174375534057617, 1.7340195178985596, 0.3311026692390442, -0.18750470876693726, 0.14445355534553528, 0.574809193611145, 0.24407045543193817, 1.5748571157455444, -0.5476647019386292, 0.6744484305381775, 1.0480997562408447, -0.831892192363739, -0.2800067663192749, 0.14345212280750275, -0.5390263199806213, -0.651154637336731, 0.25528445839881897, -0.529449462890625, 0.6242361068725586, -0.7437646985054016, -0.7655271887779236, 0.1618347018957138], [0.5499083399772644, 0.29149723052978516, 0.6504536271095276, 0.016853883862495422, 1.0743181705474854, 0.15603910386562347, 1.2129749059677124, -0.5721049308776855, 0.015724821016192436, 0.39139267802238464, -0.14000774919986725, -1.4868589639663696, -0.5039132237434387, -0.18012148141860962, 0.879288375377655, 0.7233598828315735, 1.536181926727295, 0.5261237025260925, -0.3492657542228699, -1.3523318767547607, -0.3984143137931824, 0.7751617431640625, 0.3497232496738434, 0.3161110579967499, -1.018755316734314, 1.1686521768569946, 0.7634015083312988, 0.4073675870895386, 0.15758894383907318, -1.434715986251831, -1.4213075637817383, 0.7583489418029785, -0.2565523386001587, 0.4590304493904114, 0.5244531631469727, -1.0411443710327148, 1.6376854181289673, 1.0216621160507202, -0.12904009222984314, -0.8112831711769104, 0.01604948565363884, -0.43615663051605225, 0.10061818361282349, -0.22272132337093353, -0.35647091269493103, 0.3701465427875519, -0.7638804316520691, 1.1633168458938599, 1.1683592796325684, -1.1341520547866821, 0.6560880541801453, -0.4413277804851532, 0.8237267136573792, -1.0810056924819946, 0.7277446389198303, -0.24561618268489838, 0.3796994388103485, 0.9930242896080017, 0.09688006341457367, 0.9890801310539246, -0.10598664730787277, -0.15714381635189056, -1.0672388076782227, 0.4461987316608429], [0.8915403485298157, -0.5238789319992065, 0.8839012384414673, -0.3536188304424286, 1.6310042142868042, 0.5882189273834229, -0.8131844401359558, -0.44394320249557495, 0.9577182531356812, -0.7000404000282288, 0.5258699655532837, -0.4771181046962738, 0.4406589865684509, -1.38503897190094, -0.1814012974500656, -0.6760832667350769, -0.4790396988391876, -0.8888053297996521, 0.05895806476473808, -0.32705551385879517, -1.863373875617981, -1.2830023765563965, -0.5447397828102112, 2.637047052383423, 0.820248544216156, 0.4071250557899475, 0.23695604503154755, -0.7143684029579163, -0.5548110604286194, -1.3229749202728271, -0.1265760362148285, 0.6664904356002808, 0.30348658561706543, 1.5393704175949097, 2.2033824920654297, -1.4581794738769531, 0.16533513367176056, 0.7921120524406433, -0.4572386145591736, -0.24450623989105225, -0.4630470871925354, 0.19668258726596832, 0.09815730154514313, -0.6033609509468079, -0.1759224534034729, 0.14981992542743683, 0.33143171668052673, -0.40242817997932434, -1.5466364622116089, 1.4256267547607422, -0.3898407518863678, 0.030426036566495895, -0.06307920813560486, 0.022822486236691475, -1.4980831146240234, -0.48818013072013855, -1.4012656211853027, 1.4344909191131592, -1.6347362995147705, 0.032655034214258194, 1.9228825569152832, -0.7015183568000793, 2.5627007484436035, 0.47304096817970276], [0.1915811449289322, -0.6257995367050171, 1.835057258605957, -0.032303500920534134, -0.7324961423873901, 0.43250569701194763, -1.05222749710083, -0.6497929096221924, -1.0763782262802124, -1.8655129671096802, -0.888221025466919, 0.49307936429977417, -0.6993638873100281, 1.1597256660461426, -0.041837312281131744, 0.06396220624446869, 1.8447901010513306, -0.3621601462364197, -0.030345255509018898, -0.012834899127483368, -0.8167586326599121, 0.5045657157897949, 0.1559632569551468, 1.0940747261047363, -0.08194214105606079, -0.08568555861711502, 0.37976911664009094, 0.6460665464401245, -1.8049702644348145, 0.9850090146064758, -0.7673841714859009, -0.6746307015419006, 1.070778250694275, 0.9715349078178406, 0.34304291009902954, -1.007346272468567, 0.8938677310943604, 0.42909693717956543, 0.8410179018974304, 0.23938967287540436, 1.1478201150894165, -0.34701818227767944, 1.5555970668792725, -0.37607699632644653, -0.896994948387146, -0.5830869674682617, 0.26581069827079773, -0.5773043632507324, 2.5336737632751465, -0.44483402371406555, 1.145308017730713, -1.093220829963684, -1.9279143810272217, 0.27926069498062134, -1.2242077589035034, 0.5396543741226196, -1.1144148111343384, 0.4736331105232239, 0.11439167708158493, -1.4367742538452148, 0.7602621912956238, -0.117156021296978, 1.7587567567825317, 0.534080982208252], [0.627578616142273, -1.198391318321228, 0.6974188685417175, 0.10542916506528854, -0.4041925072669983, 1.0507875680923462, 0.2617175877094269, 0.8582590818405151, 0.5727453827857971, 0.6339438557624817, 0.9896792769432068, -0.0868181437253952, -0.5480989217758179, 0.7912032008171082, -0.40462929010391235, 0.47097694873809814, 1.0037034749984741, -1.1003620624542236, 0.9629514813423157, -1.2029134035110474, 0.08458102494478226, -0.20301786065101624, -1.2500258684158325, -0.8001523613929749, -1.5879004001617432, 0.7218127846717834, -1.100258231163025, -1.712011456489563, -0.700795590877533, 0.037012092769145966, -0.31843653321266174, -0.22583705186843872, 1.3277684450149536, -0.17281731963157654, -1.2224152088165283, -1.5678009986877441, 0.26813527941703796, 0.8753563165664673, -0.764665424823761, -0.15875349938869476, -0.9583292603492737, -1.0602538585662842, -1.5128473043441772, -0.5403291583061218, -0.5014292001724243, -0.10618210583925247, 0.8087253570556641, 0.5956453084945679, -0.6938188672065735, -0.6351234316825867, 3.6545045375823975, -1.4439287185668945, -0.25639796257019043, 0.8227496147155762, -1.0497466325759888, -0.9097151160240173, 0.35602372884750366, -0.06692267209291458, 0.5323976874351501, -0.5536643862724304, 0.8271318078041077, 0.9779579639434814, 0.8641343712806702, 2.0515334606170654], [0.32142677903175354, 0.9789279103279114, -1.5036234855651855, -1.7829679250717163, 0.49360963702201843, -0.10681542009115219, 1.007216215133667, -0.29867294430732727, 0.47827795147895813, 0.6993534564971924, -0.40062347054481506, 1.7299466133117676, 0.8632400631904602, -0.5188496112823486, 0.7220160365104675, -0.789903998374939, 1.3634556531906128, 0.34042009711265564, -0.37283921241760254, -0.8880863785743713, -0.6544814705848694, -0.3112649917602539, -0.28925666213035583, 0.9128548502922058, -1.2494014501571655, -0.5340626239776611, -1.6868520975112915, -0.9170207977294922, 0.243898406624794, 0.4457174837589264, -1.6249687671661377, -1.7534737586975098, 0.3258424401283264, -0.8476507663726807, 0.4409147799015045, 0.7490931153297424, -0.6334284543991089, -0.8312855958938599, -0.2596563994884491, -0.9770264625549316, -0.42573726177215576, -0.8772796392440796, 0.23961323499679565, 0.23142454028129578, 0.0265517458319664, 0.4031783640384674, -0.9292460083961487, 1.726276159286499, 1.5515620708465576, -0.17648805677890778, 0.5041124820709229, 1.0181649923324585, -0.3141331374645233, -0.2748218774795532, 0.6076909303665161, 1.0696830749511719, -0.955377995967865, -0.25174427032470703, -0.6433267593383789, 1.8951340913772583, -1.2624685764312744, 0.3738151490688324, 0.3517579138278961, 0.9563905000686646], [0.6163815259933472, -0.6350777745246887, 0.4692700505256653, -1.6114646196365356, 0.8873435258865356, 0.34222933650016785, -0.14005199074745178, 1.2984071969985962, -0.8153167366981506, -0.4378899335861206, 1.1100136041641235, -1.8749607801437378, -1.4560141563415527, 0.03757321089506149, -0.1739087849855423, -1.517307162284851, 1.0583921670913696, 0.9598608613014221, 0.49386143684387207, 0.5641010999679565, 2.295834541320801, -0.4223703145980835, 0.3953622281551361, -0.5419747233390808, -0.43613114953041077, -0.36800292134284973, 0.8131293058395386, -1.6036609411239624, 1.8258683681488037, -0.44004538655281067, 0.9258294105529785, 1.4665272235870361, 1.0781697034835815, 0.6800140738487244, -1.0152586698532104, -0.23296231031417847, 0.677893877029419, -1.1538249254226685, -0.08221051096916199, -0.4786030352115631, 1.606022834777832, 0.7001023888587952, -0.4037758409976959, -1.2690421342849731, 0.12217433750629425, -0.8843285441398621, -0.08750486373901367, -1.2604957818984985, -0.5640835762023926, -1.1461416482925415, 0.20967550575733185, 0.10264238715171814, -1.993618130683899, 1.7198492288589478, -1.1251497268676758, -0.6516932845115662, -1.4523742198944092, 1.3658764362335205, -1.11669921875, 0.7179707884788513, 0.7036399245262146, -0.46461230516433716, 0.8973227739334106, -2.158907413482666], [-0.45613357424736023, -1.758130431175232, -0.8821253180503845, 1.70339035987854, 1.2190290689468384, 0.3320518434047699, 0.28025245666503906, 1.8850045204162598, 1.1909465789794922, -0.5526818633079529, 0.6203141808509827, 1.2568199634552002, -0.2438773661851883, 0.3734855055809021, 0.5388668775558472, 0.7946230173110962, -0.3419228792190552, -0.3844951093196869, -1.5261679887771606, 0.563836395740509, -1.7649481296539307, 0.06677424907684326, 0.44356876611709595, -0.16982541978359222, -0.9747982025146484, 0.6648755073547363, 1.1598871946334839, -0.22076527774333954, 0.8568893671035767, -0.9130309224128723, 0.26870909333229065, 0.5039705038070679, -0.2369457185268402, 0.48243874311447144, -1.5934456586837769, 1.5935306549072266, -0.44357046484947205, 0.14762501418590546, -2.313765525817871, -0.6849156618118286, -0.4626518785953522, -0.148110494017601, -0.7617818713188171, 0.17924846708774567, -0.4713347256183624, -0.6290545463562012, -0.3023841381072998, -0.13163046538829803, 1.070876121520996, -1.2244967222213745, 0.4217594265937805, 1.4363387823104858, 1.0948739051818848, -0.2822338044643402, 0.2503948509693146, 0.8423480987548828, -1.367169976234436, 0.8187119364738464, -0.7992361187934875, -0.6515666842460632, -0.16937290132045746, -0.048156045377254486, 1.2839688062667847, 0.22692330181598663], [0.057389672845602036, 1.2810351848602295, -1.1738489866256714, -0.3327978551387787, -0.753389835357666, -0.39403489232063293, 0.047173574566841125, -0.03201829642057419, 2.6602470874786377, -0.569698691368103, -0.27760276198387146, -0.7716040015220642, 0.8815982341766357, -0.7113020420074463, -0.7179627418518066, -0.41917452216148376, 0.18540945649147034, -0.5329447388648987, 0.22586674988269806, 0.3248191177845001, -0.9114817976951599, 0.5317498445510864, -1.3477418422698975, -0.4655071198940277, -0.03201751038432121, 2.792090892791748, -0.9185317754745483, 1.2246119976043701, -0.27998101711273193, -1.5706712007522583, 0.41138753294944763, 0.1413717269897461, 1.423938274383545, -1.6777775287628174, -0.6726048588752747, 1.4456590414047241, 0.0718759223818779, 1.091371774673462, -0.2844756543636322, 1.0884099006652832, -0.2154223769903183, 0.20120832324028015, -0.35309526324272156, -1.0920957326889038, -0.4475751221179962, -1.1296436786651611, -0.19734595715999603, -1.184019684791565, 0.6191976070404053, 0.8149452209472656, 0.35620903968811035, -0.4386627674102783, 1.723706603050232, 0.07874419540166855, -0.6523458361625671, -1.383902907371521, 0.06512793153524399, -0.7795573472976685, 0.018100298941135406, 2.1773149967193604, -1.7531418800354004, -0.34441065788269043, -0.42508596181869507, -0.23151268064975739], [-0.26177600026130676, -1.2340322732925415, -0.04610295593738556, -0.8631006479263306, -0.1450241357088089, -0.7599934935569763, -1.057794213294983, -2.347471237182617, 1.2986600399017334, 0.8590212464332581, 1.2130706310272217, -1.2086920738220215, -1.0498210191726685, -1.376110553741455, 0.41412603855133057, -1.8889107704162598, 1.6665327548980713, -0.7722508311271667, 0.5016369223594666, -1.5204620361328125, -0.6762974262237549, 0.03982318565249443, 0.17288514971733093, -0.8694491982460022, -0.13832387328147888, -0.5383340120315552, 2.0260422229766846, 0.5089855194091797, 0.630169153213501, -0.654385507106781, -0.21257241070270538, 1.4672963619232178, 0.14192970097064972, 0.9363475441932678, 0.4843825697898865, 0.4830925166606903, 0.2502492070198059, -0.36086273193359375, 0.5221124887466431, -0.7339433431625366, 0.813656747341156, -0.25303563475608826, -0.13047738373279572, 1.657570242881775, -1.3792369365692139, -1.1428457498550415, -0.7759315371513367, -0.8280497789382935, -1.5602176189422607, -0.7846124172210693, -0.35349714756011963, 2.095612049102783, -0.6983758807182312, -1.7957472801208496, -0.9935098886489868, 2.7179415225982666, -1.9182313680648804, 0.2541595995426178, -0.1241014152765274, 0.3860056698322296, -1.0792148113250732, 1.4168843030929565, 0.10623510181903839, 0.7635831832885742], [-1.103320837020874, 1.5660468339920044, -1.5139166116714478, 0.7193096876144409, 0.6477003693580627, -0.7675926089286804, -1.4402992725372314, 1.054105520248413, -0.12005815654993057, 2.51967191696167, -0.4658309519290924, 1.1934044361114502, 2.3971147537231445, 0.12797099351882935, 0.9360376596450806, 1.2465715408325195, -0.9733068346977234, -0.400945782661438, -0.7245347499847412, 0.6087667346000671, -1.0419574975967407, -0.9941193461418152, -0.1576734483242035, 0.0045476206578314304, 0.7285580039024353, -1.1512446403503418, -0.5803140997886658, -0.1518869251012802, -0.7681871056556702, -0.2352684736251831, -0.03625034913420677, -0.14473244547843933, -0.8736667037010193, 0.8979460000991821, -0.5118810534477234, 1.100062608718872, 1.1430548429489136, -1.3800827264785767, 1.802454948425293, 1.4254142045974731, -0.8275171518325806, -0.8461654186248779, 0.3741832375526428, -1.564599871635437, -0.7850694060325623, 0.23531371355056763, -0.008252006955444813, 0.904103696346283, 0.5717993974685669, -0.009411728009581566, -0.5312141180038452, 0.28549063205718994, -0.6189544796943665, -0.2843257486820221, 1.40513277053833, 0.8794937133789062, -0.9982858896255493, -0.13404236733913422, 1.8410214185714722, 1.320512056350708, 1.4419184923171997, -1.8085342645645142, -1.1436954736709595, -0.9861688017845154], [0.11179624497890472, -0.9216994047164917, -1.1894774436950684, 0.4345437288284302, -0.9202832579612732, 0.5181456804275513, 0.3335256278514862, 0.6587130427360535, -2.525219440460205, -0.8469070792198181, 0.1605534851551056, -1.4238814115524292, 0.515470027923584, -0.18364715576171875, 0.5927519202232361, -1.4228837490081787, -0.2875743508338928, -0.9649417996406555, 0.4815314710140228, 1.7731703519821167, 1.8659133911132812, 0.2629724442958832, 0.36997586488723755, 0.6034106612205505, -1.1754705905914307, -0.13566111028194427, -0.17191006243228912, 0.5795065760612488, -0.03791322559118271, 0.31553152203559875, -0.20559395849704742, -0.3466578722000122, 0.34660741686820984, -0.25982996821403503, 0.4030613899230957, -0.8505783677101135, 0.10952527076005936, -0.7412058115005493, -0.06291978806257248, 0.19990478456020355, -0.6985172033309937, -0.9486731886863708, -0.8133881092071533, 0.40180933475494385, 0.6072019934654236, 1.2638167142868042, -1.0433745384216309, 0.04144391790032387, 0.5002108216285706, 0.7062760591506958, 0.8458139896392822, -0.4340236485004425, 0.9538764953613281, 0.1652967631816864, 0.7938051819801331, -0.057936206459999084, 0.627017617225647, 0.15518610179424286, -1.542018175125122, -0.8458916544914246, -0.41311922669410706, -0.02465633861720562, 1.1499807834625244, -1.1129957437515259], [-1.1081925630569458, 0.7778357863426208, -1.047042727470398, -0.9054626226425171, -1.4651004076004028, -0.46886971592903137, -1.8133310079574585, 0.9229480028152466, -0.9916163086891174, -0.1355687826871872, 1.6344102621078491, 0.724578320980072, -0.785995364189148, 0.9576352834701538, -0.1281728744506836, -0.13619087636470795, 0.1926700919866562, -0.5920110940933228, 0.878276526927948, -0.0718315839767456, -0.45237842202186584, -1.4719960689544678, -0.4251477122306824, 0.35420069098472595, 0.540000319480896, 1.2174793481826782, -1.0819385051727295, 1.306330919265747, 0.11111792176961899, 0.697217583656311, 0.6709890365600586, 0.7486141324043274, -0.04733799770474434, -0.11781343072652817, 0.20853495597839355, -1.5770989656448364, -0.17899401485919952, -0.36729681491851807, 0.24896878004074097, -1.3926581144332886, -0.12167003005743027, 0.5158858299255371, 1.6037282943725586, 0.6059061288833618, -0.179937481880188, 0.5927020311355591, -0.1525336652994156, 0.4609048664569855, 0.35014691948890686, 1.9252995252609253, 2.689263343811035, -0.5395178198814392, -0.3254295885562897, 0.838394284248352, 0.6594832539558411, 0.62326979637146, 1.8555870056152344, -0.5499058961868286, -0.4287521243095398, -0.8672704100608826, -0.42956802248954773, 0.4693804085254669, 0.5837408900260925, 2.121180772781372], [-0.9798105359077454, 0.8854969143867493, -0.3764079511165619, 1.2514824867248535, -0.07131654769182205, -0.8471415042877197, 1.8389670848846436, 0.8202831149101257, 0.1066749095916748, 1.7926570177078247, 0.6394784450531006, 0.24378283321857452, 0.3306727409362793, -1.0808589458465576, -0.7925599217414856, -0.03670885041356087, 2.2036094665527344, 2.3547871112823486, 1.0686380863189697, -0.8621094822883606, -2.860426187515259, 0.40538451075553894, -1.9582873582839966, 0.06781048327684402, 0.10847540944814682, -0.15271329879760742, -0.39177510142326355, -0.9874116778373718, 0.8944041132926941, -0.007593767251819372, 1.1288241147994995, 0.023678360506892204, -0.9831333160400391, 0.715524435043335, 0.6647661328315735, 0.8603225946426392, -1.9161036014556885, 0.7380024790763855, 1.4524292945861816, 0.7807742357254028, -0.02529144659638405, 0.19635885953903198, -0.5196585059165955, 0.09693990647792816, -0.5178092122077942, 1.1221646070480347, -0.24265238642692566, -0.08258455246686935, -0.6735198497772217, -0.6143296360969543, 0.015742741525173187, 0.24506963789463043, 0.03589072450995445, -0.3031995892524719, 1.5386245250701904, -0.9088364839553833, 0.30680957436561584, 0.2823789417743683, -0.23387983441352844, -0.054692912846803665, 0.1885894536972046, 1.8350993394851685, 0.6334384083747864, -2.311121940612793], [-0.6243067383766174, 0.503572404384613, 1.672895908355713, -1.3736292123794556, 1.2397381067276, -0.4716123044490814, -2.0391736030578613, 0.0856531411409378, 1.7054438591003418, 0.2651154100894928, 0.7175903916358948, -1.0847870111465454, 0.11235353350639343, -0.6535214781761169, 0.27277907729148865, 1.3176223039627075, -0.40230023860931396, -1.2651152610778809, -1.9598560333251953, 0.983733594417572, -1.8049484491348267, 0.32011404633522034, 0.5988112092018127, -0.13383930921554565, 1.7205449342727661, 0.3648245632648468, -0.6177648901939392, -0.2642115354537964, 0.2588241696357727, -1.3814373016357422, 0.8653422594070435, -0.3093428611755371, 1.6611638069152832, -0.8563734292984009, 1.859047532081604, 1.2636637687683105, 1.7973322868347168, -0.1734662801027298, 0.2565367519855499, -0.6955748796463013, -1.4778028726577759, 1.0691756010055542, -0.08868959546089172, 1.3585313558578491, -0.9723363518714905, 0.7569658160209656, -1.5147489309310913, 1.1016815900802612, -0.7069809436798096, 1.8942151069641113, 0.3279566764831543, 0.25654253363609314, -1.3739595413208008, 0.1091965064406395, 0.22020097076892853, -1.9555916786193848, -0.771622896194458, 0.5795787572860718, -0.02837945893406868, 0.32724860310554504, -1.9831011295318604, 0.14866295456886292, 0.12416800856590271, -0.7315869331359863], [-0.9439712762832642, 0.32762885093688965, -0.494078665971756, 1.6690902709960938, -0.6577025651931763, -1.0377655029296875, 1.48403799533844, -0.15512172877788544, 0.02466898038983345, 0.26903036236763, 0.7373945116996765, 1.5292004346847534, -1.4777467250823975, -0.9624742269515991, -0.3321616053581238, 0.13341695070266724, -0.6044629812240601, 0.13686016201972961, 1.1657627820968628, 0.3410431742668152, -1.380246639251709, 0.5506439805030823, -1.7021218538284302, 0.1895066797733307, -1.1580171585083008, 0.617702066898346, 0.6478862762451172, 0.7805115580558777, -0.8103960752487183, 1.0946476459503174, 0.37511569261550903, -0.4352729320526123, -0.4119781255722046, 1.8740198612213135, 1.2930883169174194, -0.18383991718292236, 0.21968451142311096, -0.37854722142219543, -1.1470037698745728, -0.7088143825531006, 1.8040494918823242, 1.7330403327941895, 1.8211050033569336, 0.6667999029159546, 0.058732155710458755, 0.21499229967594147, 1.0227042436599731, 0.9898709058761597, 0.711127758026123, -0.5867255330085754, -1.4003117084503174, 0.5915313363075256, -0.05825451388955116, -0.9512874484062195, 0.37432777881622314, -0.04984413459897041, -0.3538450598716736, -0.7311168313026428, 0.3523755669593811, -0.376375287771225, -0.9514910578727722, -0.3891845941543579, -0.9398230314254761, 1.2991890907287598], [-0.9040823578834534, -0.6083871722221375, 0.7568459510803223, 1.870312213897705, -1.01410710811615, -1.4311798810958862, -0.17613866925239563, -1.0567758083343506, 1.2603654861450195, 0.13802409172058105, 2.06850004196167, -0.8544338941574097, -1.0813614130020142, -2.6560583114624023, 0.9147379994392395, -1.4190999269485474, 2.7937099933624268, 0.9662119746208191, -0.012346011586487293, -0.2748475670814514, -0.09565110504627228, 0.00976366177201271, -0.1914430409669876, 1.1169095039367676, -0.38120079040527344, 0.6911590099334717, -0.8566555380821228, 1.2211568355560303, -0.5815926194190979, 0.6729851365089417, -1.344648003578186, -0.1398286074399948, -1.6443185806274414, 0.48888278007507324, -1.2923798561096191, 0.8986998796463013, 1.4954782724380493, 0.5727301836013794, 2.476827383041382, -0.7983734607696533, 0.8032050132751465, -0.1980319321155548, 0.8694624304771423, -0.05818140506744385, -0.8080138564109802, -1.2001217603683472, -0.671501100063324, 1.1153967380523682, 0.14134986698627472, 0.32160744071006775, -0.695215106010437, 0.870932936668396, 0.5338497757911682, -0.6727134585380554, 1.718084454536438, 1.5075093507766724, -0.7914236187934875, -0.0912993997335434, -0.22064208984375, -0.35472118854522705, -0.09942441433668137, 2.8608596324920654, 1.1970384120941162, -0.595740795135498], [-1.222450852394104, 0.1705036759376526, -2.605774164199829, 1.338616967201233, 0.35448509454727173, -0.26170170307159424, 2.700212001800537, -1.397812008857727, 0.27006545662879944, 1.1861572265625, -0.6502308249473572, -0.9803791642189026, -0.3471647799015045, -0.5723398327827454, -0.5702088475227356, -1.448667049407959, 2.7710654735565186, 0.1794401854276657, 0.8951717615127563, -0.28503531217575073, 0.7108233571052551, 0.4092843532562256, -0.08058591187000275, -1.4407292604446411, 0.34179437160491943, 0.8412719964981079, 1.027909755706787, -2.06520676612854, 0.08311709761619568, 0.6407034397125244, 0.3312705159187317, 0.2267787754535675, 1.09926176071167, 0.8301357626914978, 1.3325748443603516, -0.5405079126358032, 0.904979407787323, -0.5152260661125183, 0.21973247826099396, -0.4895334541797638, 2.243563413619995, -1.3721740245819092, 0.2927381098270416, 1.609036922454834, 0.17887544631958008, -0.3279156982898712, -0.40474840998649597, 0.25579971075057983, 0.3788977563381195, -0.04659204185009003, 0.5397241115570068, -1.3982008695602417, -0.9322419762611389, 1.0089881420135498, -0.07072248309850693, -0.27365726232528687, 0.8948601484298706, 0.34230998158454895, -0.7914880514144897, 0.4563583731651306, 0.1800042986869812, 0.3030463755130768, 0.10959155857563019, -0.2081562876701355], [-0.4241795539855957, -0.7647191882133484, 0.07178223878145218, -0.08003272116184235, -0.7022271156311035, -1.956663727760315, 0.5200225114822388, -0.4853290915489197, -0.46465155482292175, 0.691432774066925, -0.7367307543754578, 0.5294610261917114, 0.07564615458250046, 0.19121053814888, -0.014881176874041557, -0.23071518540382385, 0.40569961071014404, -1.2298450469970703, 1.8548617362976074, -0.02797984704375267, -1.0555087327957153, -0.13692541420459747, 0.7058911323547363, -1.4468731880187988, -0.7362598776817322, 0.025977380573749542, -2.6473073959350586, 0.5744472146034241, -0.061658892780542374, -0.7878937125205994, -0.8519385457038879, -0.8803542256355286, -0.019026009365916252, 0.18010953068733215, -0.09837932884693146, 0.5697281956672668, 0.6165921092033386, -0.6324190497398376, 1.550666093826294, 0.7490363121032715, 0.42683497071266174, 0.7961742877960205, 0.07679175585508347, -0.30593597888946533, -1.010642170906067, 0.49674680829048157, 0.8027575016021729, -0.9175498485565186, 0.046182285994291306, 1.552729606628418, 0.7599107027053833, -0.3102363348007202, 1.8968462944030762, -0.4562617838382721, 0.0815458744764328, 0.9306552410125732, -0.33532869815826416, -0.07400917261838913, -0.3595576882362366, -0.5231126546859741, -0.10750345140695572, 0.2811613082885742, 0.8515191078186035, -1.6890472173690796], [-0.8012802600860596, -0.6962264776229858, 1.5931450128555298, -0.6976523399353027, -0.6854451298713684, 0.8327200412750244, -0.4141201674938202, -0.33548250794410706, -0.4572877883911133, 1.42807137966156, -0.3725074827671051, 0.6293309926986694, -1.0372074842453003, -1.6845535039901733, -0.01617010124027729, -0.2565743625164032, 0.31969979405403137, 2.023590564727783, -1.0930612087249756, -0.9912711381912231, 0.4346255362033844, -0.20583781599998474, 0.260061651468277, -0.03822362422943115, 1.0673469305038452, -1.929919958114624, 0.8482518196105957, -0.6182344555854797, 0.7707468867301941, 0.3475954234600067, 2.322364568710327, -1.2314435243606567, -0.4341075122356415, -0.8958830833435059, 0.8148927092552185, 1.031728744506836, 1.0528448820114136, -1.5534617900848389, 0.36106187105178833, -0.10074242949485779, 0.5899817943572998, -0.6892057657241821, -0.4163775146007538, -1.1320122480392456, -0.25666704773902893, -0.661316454410553, 1.6946821212768555, -0.6137552857398987, 0.7275798320770264, 0.2355337142944336, 1.3809633255004883, 0.04583369195461273, -0.14016404747962952, 0.054347969591617584, 0.359233558177948, 0.6964830160140991, 0.4355873465538025, 0.6174837946891785, -0.79367595911026, 0.724422037601471, -0.6582679152488708, -1.4616023302078247, 1.2590086460113525, -0.4558728039264679], [1.2571966648101807, -1.7326778173446655, 0.8500989079475403, 0.7817459106445312, 0.020029742270708084, 1.7265349626541138, -0.5848844647407532, -0.05824033170938492, 2.1396937370300293, -1.1811903715133667, -0.6173928380012512, -0.7953323125839233, 0.03160000219941139, -0.6349095106124878, -0.5391369462013245, -0.3097072243690491, 1.3188440799713135, -0.6175576448440552, -0.7268558144569397, -0.06694969534873962, -0.18182063102722168, -0.685796320438385, 0.03590623661875725, 1.6653534173965454, -0.5610991716384888, -1.7243765592575073, -0.14918938279151917, 0.963094174861908, -0.33785727620124817, 0.2092341184616089, 1.388624668121338, 0.5498565435409546, -0.8102056980133057, -1.135209321975708, 0.6892885565757751, -0.13777421414852142, -1.684183955192566, -0.4553166627883911, 0.5808738470077515, 0.026180613785982132, -1.3096647262573242, 1.158116340637207, 0.1549922674894333, -0.45487332344055176, -1.0086532831192017, 0.2817896008491516, 0.3087831139564514, 0.7851797342300415, 0.12802565097808838, 1.1147522926330566, -0.2665661871433258, -1.566049337387085, 0.1440305858850479, 0.20208461582660675, -0.6344966292381287, -0.3035184144973755, 0.139312744140625, 0.5725396871566772, 0.6932547688484192, 0.178948312997818, -2.4883675575256348, 1.5848875045776367, -0.6370900273323059, 0.4896005094051361], [-1.5833104848861694, 2.0831165313720703, 0.7272587418556213, 0.9896682500839233, 0.686884880065918, 0.1495089828968048, 0.9549791812896729, 0.5470250844955444, 0.0675751194357872, -1.583014726638794, 0.7004441618919373, 0.07292897999286652, 0.7556688189506531, 2.338594436645508, -1.4924663305282593, -1.1154080629348755, -0.407462477684021, -1.393570899963379, -0.5042240023612976, 0.8273309469223022, 0.916926383972168, -1.3246142864227295, 0.5362414717674255, -1.3594852685928345, -0.7067953944206238, 1.611960768699646, 0.41503462195396423, -1.5446792840957642, 0.17396028339862823, -0.10819967836141586, 1.8987014293670654, -1.2772667407989502, -0.5974779725074768, 1.463078260421753, 0.4873213469982147, -0.09165437519550323, 0.13660185039043427, 0.7817419171333313, 0.5087664127349854, -1.228440523147583, 0.9845712780952454, -1.8516829013824463, 1.1235820055007935, -0.7416420578956604, 0.5588607788085938, -1.1377860307693481, -0.7023640275001526, -0.19317986071109772, 0.8087364435195923, -0.5982218384742737, 0.5914639234542847, 0.6926661133766174, 0.9207069277763367, -0.889634907245636, -1.1274020671844482, 1.3934686183929443, 0.4642236828804016, -1.0300583839416504, -1.243377923965454, 0.6978793740272522, 0.6207650899887085, 1.8934063911437988, -0.7050851583480835, -0.9927200675010681], [-1.7440953254699707, -0.14920000731945038, -1.2001569271087646, 0.8019065856933594, -0.34981536865234375, 1.6991853713989258, 0.7634080648422241, -0.6853184700012207, -1.0221152305603027, -0.5815185904502869, 0.1484137624502182, 0.44420325756073, -0.6630057096481323, 0.827312707901001, -0.756716251373291, 1.8032671213150024, 1.0781700611114502, -0.5790590047836304, -0.1739163100719452, 0.47642314434051514, 0.280734121799469, -1.147181510925293, -0.7547219395637512, -1.5937700271606445, -0.8122183084487915, -0.9939756393432617, -0.16805291175842285, 1.2663387060165405, -0.49034634232521057, 0.7136639952659607, 0.9327658414840698, 0.9049814343452454, 0.5218791961669922, -0.15120674669742584, -0.5889275670051575, -1.046534776687622, 1.0744819641113281, 0.6514521241188049, 0.7330442667007446, -1.100239634513855, -1.3421001434326172, -1.3982890844345093, 0.52046138048172, -0.39408060908317566, -1.7296892404556274, -1.5164076089859009, 0.9320486187934875, -1.630194067955017, -0.03921865299344063, -1.3799041509628296, -1.0924016237258911, 1.5654890537261963, 0.9105688333511353, -0.05603336915373802, 0.6465495824813843, -0.45571985840797424, -0.25749143958091736, 0.06927335262298584, 0.37816184759140015, 0.5632774233818054, -0.03466523811221123, -0.3565828204154968, 0.4823382496833801, 0.03808040916919708], [-0.4947200119495392, 0.6735790967941284, 1.7708637714385986, -0.9240479469299316, 0.8279619812965393, -1.2177597284317017, -0.12197237461805344, -1.2505398988723755, 0.1661505252122879, 0.7083947062492371, 0.6531080603599548, -0.9431464672088623, -1.7182528972625732, -0.43051162362098694, 0.9596714973449707, 0.5606896877288818, -0.2358458787202835, 1.2223405838012695, -1.7168529033660889, 2.355422019958496, -0.300676167011261, -0.4789960980415344, 0.2900583744049072, 1.1021450757980347, 1.1078059673309326, -1.372750163078308, -1.2754261493682861, -1.0166220664978027, -0.41084206104278564, -0.5464734435081482, 1.8004804849624634, -0.6171566247940063, 1.370822548866272, -0.08710487931966782, 0.4454346299171448, 1.1068408489227295, 1.4357497692108154, -0.5843785405158997, 1.3438807725906372, -0.4582517147064209, 0.329447478055954, 0.5598844289779663, 0.9192423224449158, -0.8134185671806335, -1.7985846996307373, 0.2690526843070984, -0.49306803941726685, -0.20407113432884216, -0.1619809865951538, -0.8487797379493713, 2.2602317333221436, 1.207953929901123, -0.13196510076522827, 2.8472821712493896, -0.35610488057136536, -0.23176085948944092, -0.9728811383247375, 0.05575192719697952, -1.162065029144287, 0.8092717528343201, -1.1988736391067505, 1.0810877084732056, -1.2913410663604736, -0.4459993243217468], [1.3596723079681396, 0.9647109508514404, -2.380518913269043, 0.4703948497772217, -0.3176696300506592, 2.507413864135742, 0.46503007411956787, -0.4147797226905823, -0.7274100184440613, -0.9256003499031067, 3.3434712886810303, -0.36025071144104004, 2.25248384475708, 1.0039629936218262, 1.6412700414657593, 0.5292674899101257, 0.38165083527565, 0.11467945575714111, -0.012528138235211372, 0.00234623858705163, -0.9703661799430847, -0.31750738620758057, 1.532967448234558, -1.8174293041229248, 1.0881094932556152, 1.5570296049118042, 0.7195404767990112, 0.09240653365850449, -1.06850266456604, 1.152591586112976, -0.6553120017051697, -0.5348818302154541, -0.7649633884429932, 0.16377045214176178, -0.7733150124549866, -0.24665535986423492, 1.1083674430847168, -1.1541022062301636, -0.3091001808643341, 0.8508620262145996, 1.2586443424224854, 0.27496814727783203, 0.2103714942932129, 0.7121446132659912, 0.9687218070030212, -0.7581115961074829, -0.24935497343540192, -0.37768375873565674, -0.5024508237838745, 0.013733875006437302, 0.9592373967170715, 0.16365858912467957, 0.6991173028945923, -0.10939629375934601, -0.5234084129333496, -0.5690014362335205, -1.0436002016067505, -1.2004237174987793, 0.02406139113008976, -1.4935858249664307, -0.818080723285675, -1.2539159059524536, -2.015263319015503, -1.8642560243606567], [-0.7447884678840637, -0.2581753730773926, 0.8200429677963257, 0.8970603942871094, 0.46909892559051514, 0.43581777811050415, -1.2921642065048218, -0.4068869650363922, 1.2441948652267456, 2.263576030731201, 0.6533619165420532, 0.8544022440910339, -1.7478177547454834, 0.5592261552810669, 1.957550048828125, 0.37098920345306396, 0.32584285736083984, 1.9840459823608398, -0.7468361258506775, 0.4591374397277832, 1.2901304960250854, 0.7598081231117249, 0.24256835877895355, -1.8496428728103638, -1.750546932220459, 1.53803551197052, -1.2604961395263672, 1.3534554243087769, -1.1954559087753296, 0.6495906114578247, 0.13286682963371277, 0.9172079563140869, 0.5358219742774963, -0.3083058297634125, -1.5730879306793213, -2.6780877113342285, 0.4004214406013489, 0.4755100607872009, 0.12060588598251343, 0.09246017783880234, -1.4112813472747803, -0.2081575244665146, -0.22358886897563934, -0.034943629056215286, 0.15808911621570587, -0.841448962688446, -1.4396530389785767, -0.20374980568885803, -0.8658607602119446, -1.010608434677124, -0.6751843690872192, 1.1384741067886353, -0.5542109608650208, 0.10079951584339142, 0.5665322542190552, -1.577641248703003, 0.12887045741081238, -0.4708751440048218, -0.023747604340314865, -0.0020151352509856224, -0.30449849367141724, 0.34143370389938354, 1.2895396947860718, 1.2092021703720093], [-0.6879873871803284, -0.024814827367663383, 1.7356191873550415, -0.5357710123062134, -0.7552692890167236, 0.07924345135688782, -0.18007998168468475, 0.6188278794288635, -1.5699599981307983, -0.8959129452705383, -0.015237848274409771, 0.4212370216846466, -0.4477600157260895, 0.062188826501369476, 0.437334269285202, 2.3644840717315674, 0.2719847857952118, 0.8217916488647461, -0.6260242462158203, 0.9558201432228088, -1.4283668994903564, -0.05979185178875923, 0.05594127997756004, -0.5479453206062317, 1.4028409719467163, 0.2653826177120209, -1.1548914909362793, 0.610025942325592, 0.31276392936706543, -0.8531847596168518, 1.5447149276733398, 1.0218480825424194, 1.5312368869781494, 0.92954421043396, -0.18027275800704956, 0.8153901696205139, 0.6244218349456787, 1.2653074264526367, 0.31858351826667786, -0.6644607186317444, 0.8146886229515076, 0.34357285499572754, -0.5717531442642212, -1.4203424453735352, 0.03144931048154831, 0.6321541666984558, 0.03435257077217102, 0.5341607332229614, -0.4614744484424591, 2.0977823734283447, 0.4306185841560364, 0.9716458320617676, 0.05844969302415848, 2.193601131439209, 1.197364091873169, 0.4224851131439209, -0.7905207872390747, 0.7400985360145569, 0.5341318249702454, 1.8606981039047241, -0.966820240020752, 0.1060071811079979, 2.0752387046813965, 0.2819693684577942], [1.2473785877227783, 0.4231792986392975, 0.5414047837257385, 2.6661365032196045, 0.2624339163303375, 1.2535368204116821, -1.3665450811386108, -0.9787470102310181, 1.0555673837661743, 0.6033262014389038, -0.6175394654273987, -1.1731107234954834, -0.8338024616241455, 0.27530619502067566, -0.2779839336872101, -0.5891465544700623, -0.698337197303772, 0.7843254208564758, 1.0940661430358887, 0.24835483729839325, 0.5890392661094666, -0.9772985577583313, -1.5027813911437988, -0.8540111184120178, 0.039995741099119186, 1.564400553703308, 0.08789065480232239, 2.462298631668091, 0.8228502869606018, -0.3035077452659607, -0.4506959319114685, -0.6139414310455322, -0.4152548909187317, 1.619211196899414, 0.9568041563034058, 1.2484370470046997, 1.8559046983718872, 1.717301607131958, -0.9562055468559265, 2.9231016635894775, -1.8765143156051636, -1.1407272815704346, -1.1536080837249756, 1.1772648096084595, -1.0474145412445068, 1.0980464220046997, 0.631456732749939, -0.4009464681148529, -0.6979992389678955, 0.17986294627189636, -0.034680597484111786, 1.5715821981430054, -2.4050495624542236, -0.14170955121517181, -1.1459941864013672, -0.583297610282898, -0.42605361342430115, -0.23398253321647644, -1.1375868320465088, -1.036319613456726, 0.07447806745767593, -0.4809738099575043, 0.9499805569648743, 2.559404134750366], [1.2744876146316528, 0.6356781721115112, -0.22761045396327972, -1.3239662647247314, 0.33578574657440186, -1.289513111114502, -0.11195898056030273, -1.003172516822815, -0.30195778608322144, 0.03767399862408638, -1.0419204235076904, 0.9818891286849976, 0.5892415046691895, -1.5080783367156982, -0.836560845375061, -1.4055572748184204, -0.967517077922821, 0.9329987168312073, 0.30431628227233887, 0.8457531929016113, 1.4914937019348145, 1.507902979850769, -0.9884315729141235, 1.8675442934036255, 0.8250849843025208, 1.125870943069458, -1.264987826347351, -0.7787761092185974, 0.747006356716156, -0.33921319246292114, -0.5822184681892395, -0.7590010166168213, 0.08879393339157104, 0.6134829521179199, 0.6026777029037476, 2.1467435359954834, 0.4480178654193878, 1.3097413778305054, 0.8973742127418518, 0.00558155681937933, 1.1400498151779175, -1.1009660959243774, -0.3049545884132385, 0.10685395449399948, -0.6054580807685852, -1.9943100214004517, -0.08213550597429276, -0.24967873096466064, 0.21331928670406342, -1.1419402360916138, 1.0268412828445435, -3.041304349899292, 0.8227821588516235, -0.0034754136577248573, -0.15686838328838348, 0.018800506368279457, -0.5940115451812744, -1.1262259483337402, 0.8563719391822815, -0.021234918385744095, -0.3633631467819214, -0.05208136513829231, -0.6966245770454407, 0.4691278636455536], [0.7919309735298157, -0.32203102111816406, 0.24709270894527435, 1.745758056640625, -1.2025223970413208, 0.6711863279342651, -0.05468789488077164, 1.6026111841201782, -1.089857578277588, 0.46678677201271057, 0.7851249575614929, 0.561622679233551, -0.2923485040664673, -1.3143253326416016, -0.80462247133255, -1.0025572776794434, -1.3006384372711182, -1.021561861038208, 0.09651435166597366, -0.7992585897445679, -0.7100520133972168, -0.10523291677236557, -0.757277250289917, 0.41795825958251953, -0.2083899825811386, 0.8123515248298645, -0.5052034258842468, -0.7786543369293213, 0.028239531442523003, -0.33129721879959106, -1.1141356229782104, -1.778883457183838, 0.4799254834651947, -0.190511554479599, 0.20018704235553741, -0.6905288696289062, -1.0496159791946411, -1.6735076904296875, -0.3365844190120697, 0.265604168176651, -1.2745939493179321, 1.3089691400527954, 0.4394604563713074, -0.17955103516578674, -0.27521660923957825, 0.9027703404426575, 1.1460456848144531, -0.054656900465488434, -0.6502789855003357, -2.4952311515808105, -1.5659972429275513, 0.617555558681488, 0.8651894330978394, 1.5994806289672852, -0.19030441343784332, 1.5005093812942505, 1.9810521602630615, 0.2767557203769684, 0.02400786057114601, -0.3568476736545563, 0.9805665016174316, 0.6993932127952576, 0.1343212127685547, 0.267647922039032], [-0.39806753396987915, 0.7829737663269043, -1.0042835474014282, -0.24743643403053284, -0.22920344769954681, 0.8210688233375549, -0.47806939482688904, -1.1607822179794312, 0.10995551198720932, -1.0515789985656738, 1.2301597595214844, 2.135704278945923, 0.4104008376598358, 0.23024095594882965, 0.21889054775238037, 0.4186477065086365, 1.5197153091430664, 0.03831403702497482, 0.15806308388710022, -0.574364185333252, -0.49134454131126404, -0.01898742839694023, -1.6028238534927368, -0.7052839994430542, -0.2663556635379791, 0.3461412191390991, -0.45315149426460266, 0.3881455063819885, 0.4498709738254547, 0.9986981749534607, 0.3956809937953949, -0.7260002493858337, -0.06892286241054535, -0.0027012138161808252, -0.04445314407348633, -1.564906358718872, 0.4217619001865387, 0.28082844614982605, -1.6765352487564087, -0.11702710390090942, -0.5987797975540161, -0.21672213077545166, -0.27765658497810364, -0.9922688603401184, -0.7561701536178589, 2.584489583969116, -0.8659266233444214, 0.41335102915763855, -0.6488955616950989, 0.8266472816467285, -1.53566312789917, -2.887479782104492, -0.07248985022306442, 0.38415250182151794, 0.052361976355314255, -1.6838458776474, 0.27210307121276855, 0.03706821799278259, -0.46450528502464294, 0.2211877405643463, 0.48498061299324036, -2.5539350509643555, -0.7159649133682251, 1.2014071941375732], [0.5721702575683594, -0.9796525835990906, 1.4306343793869019, -0.3209420144557953, -1.2586485147476196, -0.5358318090438843, -1.272711157798767, 1.1664092540740967, 0.8554638028144836, -0.6809660196304321, -0.3627684712409973, 1.538313388824463, -0.8706623911857605, -1.6080031394958496, 1.3267379999160767, -0.10513381659984589, 0.20664186775684357, 2.5948715209960938, -0.4562956690788269, -1.2685976028442383, -0.9327258467674255, -0.39308440685272217, 1.2552013397216797, -0.1137031689286232, -1.0818300247192383, 0.37904876470565796, 1.2661242485046387, -1.392706274986267, 1.7270841598510742, 0.7981560826301575, 1.3978049755096436, 0.6858354210853577, -0.6754341721534729, 0.65522301197052, -1.691373586654663, 2.021972417831421, 0.6471613645553589, -0.31976914405822754, -0.07344327867031097, 0.9388967156410217, -0.6100152134895325, -0.0670367106795311, -0.9289357662200928, -2.4704716205596924, 0.5588350296020508, -1.4053481817245483, -1.8175313472747803, -0.5835034847259521, 0.979269802570343, -0.9704271554946899, 0.7558965682983398, 0.2662416696548462, 0.7370585799217224, -0.08059144020080566, 0.0143855856731534, -0.3911247253417969, -0.46494513750076294, -0.06762894988059998, 0.40242254734039307, -0.2592327296733856, -1.680421233177185, -0.3616240918636322, -2.5546109676361084, 2.159872531890869], [0.11824372410774231, -0.41484907269477844, 0.16643860936164856, 1.7075345516204834, 0.7285947203636169, 1.0656174421310425, -0.1922595500946045, 0.30306676030158997, 0.3606102168560028, -2.4519383907318115, -1.7239973545074463, -0.12286125868558884, 0.6267532110214233, 0.8886464834213257, 0.44631531834602356, 0.16271013021469116, 2.4323363304138184, -0.18694138526916504, -0.6734205484390259, 0.3593777120113373, -0.13271814584732056, 0.2289668321609497, 3.3250601291656494, 1.7541179656982422, -1.2499780654907227, -0.30711767077445984, -0.5767387747764587, 0.7433305382728577, -0.8991454243659973, 0.7390005588531494, 0.36488592624664307, 1.3129746913909912, -0.3443339765071869, -0.005690055899322033, 0.02333875000476837, 1.028847098350525, -0.7836323380470276, -0.01792605035007, -1.486826777458191, 1.306686282157898, 0.09501495212316513, -0.6147383451461792, -1.3204984664916992, -1.9768531322479248, -0.38738250732421875, -0.5101918578147888, -1.0630896091461182, -0.13064897060394287, 0.2510433793067932, -0.6955761909484863, 1.2158384323120117, -0.34051278233528137, 0.9501394629478455, 2.055032253265381, -1.0638434886932373, -0.34485650062561035, 0.9677503705024719, -0.25452539324760437, -1.091497540473938, 0.47966453433036804, -0.20750834047794342, -0.860751211643219, 1.9059224128723145, -0.10985850542783737], [0.3154602348804474, -0.11301849037408829, -0.20844174921512604, -0.3954753279685974, 0.24947915971279144, 1.871102213859558, -1.2173435688018799, 0.04359957575798035, -0.2422773540019989, -0.9947719573974609, 0.17910747230052948, 1.452211856842041, -0.17130756378173828, -0.45508867502212524, 0.01480232272297144, -0.3064922094345093, -1.1261810064315796, -1.6767796277999878, 0.5606868267059326, 0.048954304307699203, -0.6751373410224915, -0.01299987081438303, 0.03567953407764435, 0.16147373616695404, 0.45761802792549133, 0.6702934503555298, 0.8111808896064758, 0.801820695400238, 0.22150905430316925, -0.1797369420528412, -0.4246148467063904, 1.4908970594406128, -1.4640696048736572, -0.08340374380350113, 0.23714421689510345, -0.809413731098175, 0.7787894010543823, 1.2117189168930054, 0.15472450852394104, 1.6440963745117188, 1.1435868740081787, 0.5047681331634521, 1.6016218662261963, -0.4979030191898346, 0.6158470511436462, -0.9644393920898438, -1.5990933179855347, 0.49960339069366455, -0.8843212127685547, 1.6819664239883423, 1.4205442667007446, 0.012928755953907967, -0.5350579619407654, -1.8765029907226562, -0.8157432079315186, 0.17208118736743927, 3.107726812362671, -0.7920467853546143, -0.12319665402173996, -0.6037619113922119, -0.019281810149550438, -1.0393553972244263, -1.836633324623108, -0.6151173114776611], [-0.20551590621471405, -1.7084213495254517, 0.7332836985588074, 1.4231261014938354, -0.16929759085178375, 1.5565781593322754, -0.03136681765317917, 1.5386830568313599, -0.3302071988582611, 0.711557924747467, -0.28293442726135254, 1.3323463201522827, 1.2662293910980225, -0.20584355294704437, -2.5057296752929688, -0.855402410030365, 0.4336896240711212, 0.0865364745259285, 1.285743236541748, 1.599364161491394, -0.9861806631088257, -1.0572643280029297, 0.6284163594245911, -0.997057318687439, 0.2677123248577118, 0.19134078919887543, 1.0323309898376465, 1.7096896171569824, 0.7221904993057251, 2.0259203910827637, 0.550314724445343, -0.752358078956604, -0.08262449502944946, -1.1022995710372925, 1.0042238235473633, 0.11065813899040222, -1.5993869304656982, -0.5451757311820984, -0.9008399248123169, -0.6417496204376221, -0.11474853754043579, 0.3738795518875122, 0.408584326505661, 0.22984136641025543, 1.4785654544830322, -1.300204873085022, -0.8817747831344604, -0.7918386459350586, -0.7034593224525452, -0.2949204742908478, 0.468222439289093, 0.5826916694641113, 0.40395283699035645, -1.5325742959976196, -0.4093533754348755, 0.5101607441902161, 2.0382015705108643, -0.6271598935127258, 0.6609817147254944, 1.4435005187988281, 0.09403301775455475, -2.394721031188965, -2.0280818939208984, -0.5551096200942993], [-0.6819602251052856, 0.2949841320514679, -0.6805810928344727, 0.4893081784248352, 0.04919013753533363, -0.15085789561271667, -0.532895565032959, -0.31563106179237366, 0.5814460515975952, 0.20115862786769867, -0.16109509766101837, 0.6944609880447388, 0.849702000617981, -0.5844008326530457, 1.4145804643630981, 0.15002234280109406, 1.268263339996338, -0.9073572158813477, 0.8658241033554077, 0.17323313653469086, -1.1780672073364258, 0.6557906270027161, 2.27595591545105, -0.47897329926490784, -0.27551954984664917, 0.6015567779541016, 0.07910507917404175, 0.7426432967185974, 0.24209535121917725, -0.1306675374507904, 0.605018138885498, 0.7046347260475159, 1.1122183799743652, 0.07249539345502853, 0.5027087330818176, -1.5671675205230713, -1.101426601409912, -1.3136191368103027, -1.344188928604126, -0.0884602963924408, -0.2260124683380127, -0.33858343958854675, 2.4516284465789795, 0.39751341938972473, -1.4758304357528687, 2.7266838550567627, 1.0972763299942017, -0.2750687301158905, 1.0579845905303955, -0.32135725021362305, 0.8568229675292969, -0.2419203370809555, 0.6373105049133301, -0.02955358475446701, 0.500063955783844, -0.2612166404724121, 0.7653011679649353, -1.2886474132537842, -1.6965140104293823, -0.8459064960479736, 0.5462213158607483, -0.022057579830288887, -0.9810523986816406, 0.04113566875457764], [-1.2494381666183472, 2.0713207721710205, 0.42222368717193604, 0.026556797325611115, -1.4552277326583862, -0.9021154046058655, -1.2611027956008911, 2.0642318725585938, 1.217913269996643, 0.5594647526741028, -1.6814618110656738, 1.1890171766281128, -0.9984115958213806, -0.08790688961744308, 0.022316008806228638, -0.22821590304374695, 0.9970403909683228, -0.29999491572380066, 1.3335095643997192, 0.6895601749420166, 0.39096903800964355, 0.6926942467689514, -0.3173215091228485, -0.973514974117279, -1.541707158088684, 0.12218619138002396, 0.03373827412724495, -0.3778446912765503, 0.3145875334739685, -0.23622773587703705, 0.8914685249328613, 0.4191938638687134, -0.598984956741333, -0.22348444163799286, 0.8869422674179077, -1.108436942100525, 0.7820351719856262, -0.4175219237804413, 0.18521197140216827, 1.0907484292984009, -1.0952982902526855, 0.15872600674629211, 2.9112548828125, -0.752118706703186, 1.3610460758209229, -0.7404026985168457, 0.5569101572036743, 1.0206172466278076, -1.098857045173645, 0.10847975313663483, 0.49832627177238464, -1.2613029479980469, -0.7719871997833252, -0.22696945071220398, -0.4170098304748535, -0.3150184750556946, -1.4384446144104004, -0.5756218433380127, -0.9143586158752441, 0.4386526346206665, 1.3209409713745117, -0.7466919422149658, 1.0112286806106567, -1.5474857091903687], [-0.8577980399131775, -0.26656368374824524, 0.4158121943473816, 0.6396334171295166, 0.1250762939453125, 0.1687881350517273, 1.6419826745986938, 0.4175404906272888, 0.784818172454834, 0.8696168661117554, 1.901149034500122, -0.9416170120239258, -0.39762839674949646, 0.5398362278938293, -0.41445523500442505, -2.1737122535705566, 0.7244579195976257, -0.34578070044517517, -0.4403177499771118, -1.4928641319274902, 0.06932530552148819, -1.056269884109497, 0.394428014755249, 1.1966679096221924, -1.6413782835006714, -1.6815133094787598, -0.15316982567310333, -0.627774715423584, 0.6568999290466309, -0.3032493591308594, -1.0141716003417969, -1.1430617570877075, 0.44678670167922974, 2.8358683586120605, 0.03677598014473915, -0.1285439431667328, -1.5654168128967285, -0.839023232460022, 0.9417906403541565, -0.44401222467422485, 0.17484678328037262, 0.6020558476448059, -0.5276976823806763, -0.3750387132167816, -2.8818447589874268, 1.387218713760376, 0.1656758040189743, -0.5148892998695374, 0.7550580501556396, 0.006921226624399424, -1.0615613460540771, 0.13856174051761627, 1.1610337495803833, -1.93878173828125, 0.34486135840415955, -0.9680647253990173, -0.900711715221405, -0.3589261472225189, -1.2773271799087524, 0.004824578296393156, -0.046091146767139435, -0.06605903804302216, -0.47183263301849365, 0.1394178718328476], [0.15103566646575928, 0.3561820983886719, -0.004988584667444229, 0.11276306957006454, -0.7738811373710632, -1.2332390546798706, 1.6799311637878418, -1.123875617980957, -0.7765319347381592, 1.1035268306732178, -1.006324052810669, 1.0087471008300781, 0.31762099266052246, 1.2341464757919312, -0.8612860441207886, 0.5262769460678101, 0.6757984757423401, 0.77534419298172, 0.1990073025226593, -0.36663761734962463, -0.6358815431594849, -0.6473003029823303, 0.2351042926311493, 0.23892053961753845, -0.0733189806342125, -0.03298221528530121, 1.0145643949508667, -0.2913835644721985, 1.4349443912506104, 0.8989914655685425, -0.8567671775817871, -1.944488763809204, -0.0917152538895607, -0.15416908264160156, 0.5434013605117798, -0.1304432600736618, 1.00551176071167, -0.44795092940330505, -0.8239149451255798, 0.5436502695083618, -0.4113617539405823, -0.8834244012832642, 1.5769866704940796, -1.1073323488235474, -0.254941463470459, -0.4468497633934021, 0.23477256298065186, -0.7880491018295288, 1.3495066165924072, -1.9818346500396729, 0.6631917357444763, -0.28553324937820435, 0.9588140249252319, -0.31910863518714905, -0.20466887950897217, 0.814949631690979, 0.7214257121086121, -1.0999102592468262, 1.2484151124954224, -0.5924899578094482, 1.8354510068893433, 0.9537194967269897, 0.3789156377315521, 0.5052117705345154], [0.1778957098722458, -0.9638076424598694, -0.6356808543205261, -0.10989280790090561, -0.07362924516201019, 0.7437847852706909, 0.051958635449409485, -1.434375286102295, -1.3721486330032349, 0.19918949902057648, 1.0241020917892456, 0.854279100894928, -1.0299181938171387, -0.03757037967443466, -0.6951369643211365, 1.5080281496047974, 0.32333022356033325, 0.14280378818511963, -0.5875697731971741, 0.5994328856468201, -0.24501177668571472, 1.82699716091156, 1.203961968421936, -0.9588716626167297, 0.577462375164032, -0.8215224146842957, -1.5668754577636719, 0.2523952126502991, 0.1676401048898697, 0.8254136443138123, -1.678810477256775, -1.132125735282898, -0.8332351446151733, -1.2355678081512451, 0.2569684684276581, 0.6427686214447021, 1.6540195941925049, -0.18286187946796417, -1.1309834718704224, -1.128326416015625, 0.23698790371418, 1.471571445465088, 0.7276602387428284, -0.06883788108825684, 0.9545378088951111, -1.445146083831787, -0.5525350570678711, -0.9900660514831543, -0.26546671986579895, 0.3961372375488281, -0.8264543414115906, -2.42301607131958, -0.25771525502204895, 0.4829973876476288, 1.469416618347168, -0.04011835530400276, -0.8299242854118347, -0.657436728477478, -0.2858465611934662, 0.3398120105266571, -0.6684836149215698, -1.1962952613830566, 0.8865395784378052, 0.030824970453977585], [-0.5615600943565369, -1.0222728252410889, 1.2613155841827393, -1.2917602062225342, -1.0014381408691406, 1.7130507230758667, 0.26053258776664734, 0.9656006097793579, -0.9924517869949341, -0.8076432943344116, 0.39666348695755005, -0.28751590847969055, 2.590184450149536, 1.596214771270752, 1.3285350799560547, -2.9240200519561768, -3.4276299476623535, 0.8004654049873352, -0.24135005474090576, 1.1728919744491577, -0.11656522750854492, -0.27881667017936707, -1.0015270709991455, 0.6958001255989075, -1.1230289936065674, -0.6158177852630615, -0.2501307725906372, 2.2602367401123047, 0.37523216009140015, -0.44540101289749146, -0.5312137603759766, 0.14534834027290344, -0.5451674461364746, 0.2737436294555664, 1.478783369064331, -1.4134795665740967, -0.5981148481369019, 2.1365272998809814, -1.692763090133667, 2.907059907913208, -0.5701807141304016, -0.08651579171419144, -0.29838815331459045, -1.7636619806289673, -1.083188772201538, -1.2861446142196655, -1.799867868423462, 0.5571219325065613, -0.20879700779914856, 0.022363772615790367, 2.4660239219665527, -0.26785120368003845, -0.535723865032196, 0.33730414509773254, 1.3085263967514038, 0.40589439868927, -0.5206329226493835, 0.6218340992927551, -0.4816449284553528, -0.30338406562805176, 0.11550971120595932, -0.16223081946372986, 1.7701256275177002, 0.23613816499710083], [0.6428195834159851, 0.2239404320716858, -1.10137939453125, 1.590919852256775, -1.0208052396774292, 0.6935575008392334, 0.7051896452903748, -0.2794593572616577, -0.8929509520530701, -0.7058388590812683, 0.2915537357330322, -0.2988869249820709, -2.1386687755584717, -1.7731893062591553, -1.6424646377563477, -1.4169102907180786, -0.12378641217947006, -0.11727261543273926, -0.576755166053772, -1.8268946409225464, 0.8318467140197754, -0.5623821020126343, 2.4277384281158447, 1.8632771968841553, -0.22107090055942535, -0.5975211262702942, 1.6783415079116821, 0.3946274220943451, -1.7167649269104004, 0.7969144582748413, 0.1214810162782669, -0.5687045454978943, -0.010986288078129292, -0.0726483091711998, 0.6699908375740051, -1.6747380495071411, 0.458637535572052, -1.2608816623687744, 0.7399235367774963, 1.4392094612121582, -0.3240715563297272, -0.07445597648620605, 1.296383261680603, -1.7851519584655762, -0.3094748556613922, -0.013059699907898903, -1.7341485023498535, -0.23901621997356415, -0.608657717704773, -1.0811491012573242, 0.00771175604313612, 0.6840169429779053, 1.120850682258606, -0.5409008860588074, 1.3001306056976318, 1.0738816261291504, -0.3648315966129303, -1.785995602607727, 0.3084147274494171, 0.17200888693332672, 0.7582923173904419, 0.6652496457099915, 0.8041543364524841, 0.7359490394592285], [-0.46423617005348206, -0.07522563636302948, 0.16124045848846436, 0.4337606430053711, 1.4816889762878418, 0.2530559301376343, -0.7523564100265503, -0.9350461363792419, -0.5472654104232788, -0.260921448469162, -2.532057046890259, -1.6269837617874146, 0.9441959261894226, -0.44184479117393494, 0.22053954005241394, -0.437601774930954, -0.4968124032020569, -0.9882537722587585, 0.21184773743152618, -0.2974299192428589, 0.7426332235336304, -0.7164829969406128, -1.1681562662124634, -1.0939213037490845, 0.7277955412864685, -1.779485821723938, 1.2700691223144531, 1.4010486602783203, -0.22083601355552673, 0.14843039214611053, -0.8716719746589661, -0.21362406015396118, 0.3936479091644287, 0.5706775784492493, -0.9151181578636169, 1.2949961423873901, 0.37089839577674866, 0.16651758551597595, -1.8711868524551392, -0.7281670570373535, 0.4406574070453644, 0.15559208393096924, -0.527563214302063, 0.24822348356246948, -1.0693780183792114, 0.31819918751716614, -0.6229131817817688, 1.4089165925979614, 2.123777151107788, 0.04620255157351494, 1.2048238515853882, 2.0017988681793213, -1.1114518642425537, 0.13321740925312042, 0.23097948729991913, -0.3273056149482727, 0.706824004650116, 0.15079998970031738, 0.5940754413604736, 0.28047722578048706, 0.5817556381225586, -0.7835142612457275, -1.340806245803833, -1.0501022338867188], [0.6865017414093018, -0.5035357475280762, -1.193661093711853, 2.1822996139526367, 1.374463438987732, -0.21229714155197144, 0.4758991003036499, -1.3103853464126587, -0.1893427073955536, -1.922036051750183, 0.5108209848403931, 0.5642085075378418, 0.8161568641662598, 1.0146291255950928, 0.968079686164856, -0.8130358457565308, -0.3397047221660614, -0.3606933355331421, -0.34572869539260864, -0.4450537860393524, 0.10607261955738068, 0.6108393669128418, -0.5820186734199524, -0.756271481513977, -0.512600839138031, -1.2111655473709106, 0.06704381853342056, -1.3994437456130981, 1.9118410348892212, -0.06428667902946472, -0.6197690367698669, 0.23124809563159943, -0.14805790781974792, 0.12062126398086548, -1.6269556283950806, -1.041038990020752, 0.39142853021621704, 1.3154298067092896, -2.2438228130340576, -1.0562036037445068, 2.04071044921875, -0.42727160453796387, -0.6814464330673218, 1.5621678829193115, -1.4895116090774536, -0.3672183156013489, -0.7886616587638855, -0.8650888204574585, 2.650148630142212, 0.1857793629169464, -0.14325648546218872, -1.262410283088684, 0.3836348056793213, -0.248857319355011, 0.6046287417411804, -1.679242491722107, 0.033745571970939636, -0.20006941258907318, 0.025283506140112877, 0.6105366349220276, -1.9109488725662231, -1.673851728439331, 0.47868412733078003, -2.952470064163208], [0.6649857759475708, 0.24790355563163757, 0.3213534355163574, -0.11803290247917175, -1.6006578207015991, -0.5391583442687988, 0.1791469305753708, -0.06905116885900497, -0.657360315322876, 2.096630573272705, -0.6663800477981567, 0.48080307245254517, 0.8105447292327881, -0.001134607009589672, 1.3897019624710083, 0.7622564435005188, -1.3612523078918457, 1.7719645500183105, 0.4398041367530823, -0.3415379822254181, -0.6646782159805298, 0.2624807357788086, 0.8133839964866638, 0.20531263947486877, 1.2428752183914185, 0.018962545320391655, -0.8288195133209229, 0.6616595387458801, 0.9117007255554199, -1.0849767923355103, 0.8559705018997192, -0.02406843565404415, 1.0921281576156616, 0.31788158416748047, 1.0016294717788696, 0.11432646214962006, 0.7899506092071533, 0.9204381704330444, 1.5300283432006836, -1.4288886785507202, -1.1265431642532349, 0.31495141983032227, 0.10081510990858078, 1.9966565370559692, 1.8656983375549316, -0.9980350136756897, 0.10484151542186737, -0.47490084171295166, 0.5049684047698975, 1.0007096529006958, -0.5420506000518799, -0.06925984472036362, -0.22304703295230865, 1.2488592863082886, 1.483040690422058, 0.43930014967918396, 0.24271723628044128, -0.317620187997818, 1.4412610530853271, -0.28763675689697266, -1.950005054473877, -1.2086611986160278, -0.6262622475624084, -0.4468451738357544], [-1.012300729751587, 0.28737109899520874, 0.44755473732948303, -0.3314485549926758, 0.5944420099258423, -1.5226117372512817, 0.09396938234567642, -1.6787245273590088, 1.6555176973342896, 0.24240164458751678, -1.1193783283233643, -0.5552759170532227, -1.4641988277435303, -0.6854211091995239, 1.0043630599975586, 0.301578551530838, -0.3301270008087158, -1.1967002153396606, 0.40074217319488525, 1.428067922592163, -0.27768567204475403, 0.12270250171422958, -0.8636223077774048, 0.3658476769924164, -1.0737955570220947, 0.8638274669647217, 0.9288004040718079, -0.7146891951560974, -0.5875287055969238, -1.1304905414581299, 0.8656005859375, -1.5771312713623047, 0.3740067780017853, 0.5503873825073242, -0.15297196805477142, 0.34682804346084595, -1.624842882156372, 0.48692384362220764, -0.21156033873558044, 0.9337130188941956, -0.839034914970398, -0.9646997451782227, 0.6130580306053162, 0.13900604844093323, 1.4063469171524048, -1.2663750648498535, 1.887808918952942, 0.5390524864196777, -0.6987515687942505, 0.2935014069080353, 0.3516758978366852, -1.163645625114441, -0.9694482088088989, 1.4474399089813232, -0.7768707871437073, -0.2030486762523651, 1.1828080415725708, 0.775733232498169, 0.21850928664207458, 0.1480262577533722, -1.769161581993103, 0.15863895416259766, -1.1707462072372437, -0.6138434410095215], [0.8484327793121338, -1.3427306413650513, 0.042731206864118576, -0.2834191620349884, 0.396429181098938, -0.3028489351272583, 1.2040367126464844, -1.5359165668487549, 0.6406696438789368, 0.5926120281219482, -1.00651216506958, 0.2069578915834427, 0.08197883516550064, -0.43802645802497864, -0.5153903365135193, 2.1209635734558105, -0.5088400840759277, 1.279363751411438, -0.3086516559123993, 0.12153422087430954, 0.23217104375362396, 0.38262999057769775, 0.6427330374717712, -0.9432715177536011, 0.1414552927017212, 1.19524347782135, -0.7564148306846619, 1.3404666185379028, -0.4249318540096283, -1.0911160707473755, -0.47164231538772583, -0.9156490564346313, -1.0951340198516846, -1.0050703287124634, 0.019228367134928703, 1.4870212078094482, 1.508674144744873, -0.1298253983259201, 0.5158957839012146, 1.9186314344406128, 0.17609535157680511, -0.22883373498916626, -0.6861960887908936, -0.765197217464447, -0.08845333755016327, 1.0398344993591309, -0.4306497275829315, 0.07039424777030945, 0.6686019897460938, -0.5959669351577759, 1.6359373331069946, 0.8707573413848877, 0.6390408873558044, 1.4935163259506226, 2.0752522945404053, 0.3664247691631317, 1.1062729358673096, 0.630953311920166, -0.04949628561735153, -1.3621147871017456, -1.0169998407363892, 0.19628994166851044, -0.9213117957115173, -0.9958127737045288], [-0.15525813400745392, -0.29363954067230225, -0.07654834538698196, 2.68949294090271, 1.1513999700546265, 0.06435059756040573, -0.2684056758880615, 1.3613133430480957, 0.5361668467521667, 0.9477344155311584, 0.5567829012870789, -0.9587773084640503, -2.973151922225952, 2.033193588256836, -0.25651466846466064, 0.769877016544342, 0.3870275914669037, -0.3043491840362549, -0.1503302901983261, -0.6553099155426025, 0.8999369740486145, -0.8129075169563293, 1.3499364852905273, 0.0691542699933052, 0.4456942081451416, -0.9382490515708923, -1.3670810461044312, 1.7613047361373901, 1.578635334968567, 0.5172615051269531, -0.28905221819877625, -0.1535780429840088, 0.006724509876221418, 2.8377528190612793, 0.008354807272553444, 0.5704886317253113, -0.6061753630638123, 0.24755524098873138, -0.768993616104126, -0.5894567370414734, -1.3177647590637207, -0.5222880840301514, -1.4220110177993774, 0.049772296100854874, -0.9939709305763245, -1.3212372064590454, -1.4958144426345825, -1.2160882949829102, 1.5282721519470215, -0.023768696933984756, 0.3536891043186188, -1.0486048460006714, -0.5634444355964661, 0.8227040767669678, -0.716592013835907, -1.7655125856399536, 1.466854453086853, 0.30832087993621826, 0.8806549906730652, 0.5771833658218384, -0.5410946011543274, 0.8524878621101379, -0.45720309019088745, -2.2890090942382812], [0.6228150725364685, 1.2055840492248535, -0.05307198315858841, -0.4985889494419098, 0.9313837289810181, 2.451900005340576, 0.19549985229969025, 0.5279973745346069, 1.0157002210617065, -0.5920429229736328, 0.3797908127307892, -0.3250808119773865, -1.7490752935409546, 1.1905133724212646, -2.2318308353424072, -0.4439624845981598, -0.9510850310325623, 1.348041296005249, -1.1955996751785278, -2.340012550354004, -1.1068744659423828, 1.1071504354476929, -0.798142671585083, 0.3457113206386566, 1.674962043762207, 0.055780451744794846, -0.12613984942436218, -0.044395994395017624, 0.4348593056201935, -2.6398167610168457, 0.2892918884754181, 1.4963525533676147, -0.6719527840614319, 1.8576749563217163, -1.6240214109420776, 2.1794655323028564, -0.8878088593482971, -0.013205108232796192, -0.2566836178302765, -0.7127594351768494, 0.7205953598022461, 0.11105812340974808, -0.2985022962093353, 1.4100797176361084, -1.2692689895629883, 0.07202871888875961, 0.4821135401725769, -0.19181422889232635, 0.8551062941551208, -0.3865639269351959, 0.4254577159881592, 0.2712551951408386, -0.2446572184562683, -0.4112681746482849, 0.28727713227272034, -0.7725890278816223, -0.5259912610054016, 1.28055739402771, -0.06711183488368988, 0.9849379658699036, 0.37178266048431396, 0.7638642191886902, -1.493865728378296, -0.4589337408542633], [-0.48130831122398376, 0.864222526550293, 0.3367101550102234, -0.5331099033355713, 1.1370617151260376, -0.4049496054649353, -0.6532543301582336, -1.171942949295044, -0.6257140636444092, -2.2525153160095215, -0.7979467511177063, 0.38341984152793884, 0.18571041524410248, 0.26187992095947266, -1.5566596984863281, 0.8808377981185913, -0.6757953763008118, 1.9737005233764648, -0.8332393765449524, -1.4466307163238525, -2.0446276664733887, -0.4274064302444458, -2.0054516792297363, 0.24335446953773499, 0.20894576609134674, -0.280711829662323, -1.8346649408340454, 0.19722141325473785, 1.5558571815490723, 1.2049174308776855, -0.41942480206489563, 0.9050500392913818, -0.38112619519233704, 0.7133423686027527, 0.20257055759429932, 1.958729863166809, 0.21341869235038757, -0.2499941736459732, 0.42648449540138245, -0.7401673197746277, 1.1233068704605103, -1.273006558418274, 0.6855695843696594, 1.616970419883728, -0.2365991324186325, -0.6656901240348816, -0.21595677733421326, -1.7579108476638794, 0.5826454162597656, 0.09778064489364624, 1.0162383317947388, 0.07564913481473923, -0.889851987361908, -1.721138596534729, 0.36159852147102356, -0.23027800023555756, 0.4711947739124298, -0.14988426864147186, -1.3481252193450928, 0.10931193828582764, 0.524242103099823, 0.015524066984653473, -1.706836462020874, 0.6418095827102661], [-0.3891323208808899, -0.46935394406318665, -1.30380380153656, -0.26045501232147217, 0.3485962152481079, -0.5623186826705933, -1.014408826828003, -1.055294394493103, 0.3446664810180664, -1.05629301071167, -0.2565155327320099, -0.06249729171395302, -1.208832025527954, 1.0194315910339355, 0.2702384293079376, -0.023810824379324913, 0.832691490650177, -0.03446396067738533, 0.07717056572437286, 0.06455352157354355, 0.7958584427833557, 1.3705393075942993, 0.8461942076683044, -1.0259220600128174, -1.5445406436920166, -1.3576297760009766, -1.9047605991363525, -1.328781247138977, 0.7215976715087891, 2.9387168884277344, 1.1922640800476074, 0.20161227881908417, 1.657836675643921, -1.4824110269546509, 0.8507936596870422, -0.9906642436981201, -0.4174126088619232, 0.8107655048370361, 0.04276283085346222, -1.1464475393295288, 0.5794917941093445, 0.8033329248428345, 0.2661373019218445, 2.0032730102539062, -1.9436719417572021, -2.241607904434204, 0.37521716952323914, -0.3953782320022583, 0.40396028757095337, 0.9862180948257446, 0.0021855186205357313, -1.0535625219345093, -0.23907919228076935, -0.2565329074859619, 0.47734951972961426, 0.06301392614841461, 1.4181442260742188, 0.24969631433486938, -1.499660849571228, 0.2068854719400406, 0.7381889820098877, 0.2299100160598755, -0.23346605896949768, -0.2700929045677185], [1.7918516397476196, 0.36090654134750366, 0.2386365532875061, -2.264092206954956, 1.0895047187805176, 1.032676100730896, 1.1340895891189575, -0.42623811960220337, -0.7265564203262329, -1.7042189836502075, -2.5197176933288574, -0.026928823441267014, -0.7295626401901245, -1.602435827255249, -1.5238059759140015, -1.116385817527771, -0.31207793951034546, -0.8870473504066467, 1.4660853147506714, -1.0543479919433594, -0.049068886786699295, -0.17059326171875, -0.3826986849308014, -0.2923290729522705, 1.6923598051071167, 0.6599352955818176, -0.8829944133758545, 0.2977524995803833, -1.2938035726547241, -0.9463376998901367, 0.3817140460014343, -1.88336181640625, 0.314797967672348, -1.5210586786270142, 0.853248655796051, -1.804401159286499, -0.37696388363838196, 0.06154009699821472, -0.8789152503013611, 0.3454974889755249, -1.8240294456481934, 1.9369256496429443, 1.1345268487930298, 0.24633629620075226, -0.2169266641139984, -0.0033349222503602505, -0.6841743588447571, 1.0200613737106323, 0.4174579679965973, -1.9560383558273315, -0.8279024362564087, 2.4837868213653564, 0.22590315341949463, -0.9615967869758606, -1.2298908233642578, 0.6604843735694885, 0.22709143161773682, 0.25230416655540466, -0.9307721257209778, -0.18006184697151184, -1.0035020112991333, -0.7247927784919739, 0.7875633239746094, -0.009778445586562157], [2.160784959793091, -0.5855413675308228, -0.17647404968738556, 1.3447476625442505, 1.9225714206695557, -0.37194469571113586, -1.1885393857955933, -0.2404138594865799, 0.7838898301124573, 1.0964492559432983, 1.6893326044082642, -0.5598757863044739, -0.8859888315200806, 0.9143788814544678, -0.15403854846954346, -1.788442611694336, -0.5061243772506714, -0.5025807619094849, 1.8611873388290405, 0.9562788009643555, -1.2961986064910889, 0.7016597390174866, 0.6096901893615723, -1.6097195148468018, -0.2752353250980377, -0.7226750254631042, 0.690866231918335, 3.0498127937316895, -0.10982150584459305, 1.5626089572906494, 0.7610514163970947, -0.7516739368438721, -0.4567192494869232, 1.412112832069397, -0.37591543793678284, 0.9582463502883911, -0.6095597147941589, -0.9933733940124512, -0.18625099956989288, 1.1162680387496948, -0.3955463469028473, 1.3168106079101562, -1.2556936740875244, -1.8480345010757446, -0.44157594442367554, 1.0053666830062866, 0.35741615295410156, 0.4939022660255432, 1.7919119596481323, -0.6283974051475525, 1.3222143650054932, -0.8153300285339355, 0.48310232162475586, -1.39517343044281, 1.8822170495986938, 0.4019058048725128, 0.6983893513679504, 1.1023553609848022, 0.7292256951332092, -0.2813410758972168, 0.6829112768173218, -0.6362713575363159, -0.3797855079174042, 0.7045124173164368], [-1.5417685508728027, -0.7715063095092773, 0.035656265914440155, 1.4384881258010864, 0.6329207420349121, 0.7496027946472168, -0.8117754459381104, 0.4525175392627716, -1.257557988166809, 1.125552773475647, -0.4915676414966583, -0.037554048001766205, -0.11707129329442978, -0.7572054266929626, -0.8576181530952454, -0.09755736589431763, -0.08917830884456635, 0.954338014125824, -1.2609517574310303, 0.27649858593940735, 1.0203850269317627, 0.7238742709159851, -1.3706660270690918, -0.11166999489068985, -0.6873457431793213, -0.41577765345573425, 1.2664053440093994, 0.3905143439769745, 0.10845724493265152, -0.5768641233444214, 0.20757895708084106, -0.44716876745224, 0.9406945705413818, 2.821932792663574, 0.7029278874397278, 1.0694187879562378, 1.1568048000335693, 0.4204852283000946, 0.2616208791732788, -0.3929455280303955, 0.596930742263794, 1.6054033041000366, -0.873744785785675, -0.9779797792434692, 0.42876890301704407, 0.45284974575042725, 0.7116327881813049, 0.7760298848152161, -1.2259513139724731, -0.009340568445622921, -0.36425790190696716, 1.4355608224868774, 1.5920826196670532, -0.16347675025463104, 0.4369570016860962, 1.245761513710022, -0.869915246963501, -0.9140652418136597, -1.086499571800232, -0.14843443036079407, 2.833416223526001, -0.6972147226333618, 1.2997645139694214, 1.559980034828186], [-1.0949255228042603, 0.8389906883239746, -0.8162219524383545, 1.631060004234314, 0.45674335956573486, -0.5896191000938416, 0.7097570300102234, 0.6721582412719727, 1.0800307989120483, 0.5267423391342163, 0.7641843557357788, -1.4972869157791138, -0.8849824666976929, 0.32431676983833313, -0.8663661479949951, 0.7971698045730591, 0.7133501768112183, 0.522179126739502, -1.3773486614227295, -0.3151969909667969, 1.1343234777450562, -0.6562994122505188, -0.08351994305849075, 0.4696126878261566, -0.06952652335166931, -1.286498785018921, -2.8760814666748047, -1.5623167753219604, -0.34098365902900696, -0.05703084170818329, 0.45043936371803284, 0.7636275291442871, -0.6965510249137878, 0.06550703942775726, 0.7527629733085632, 0.8297440409660339, 1.193517804145813, -0.7857999205589294, 0.3226340413093567, 0.6223483681678772, -1.0329499244689941, -1.933146357536316, 1.0158305168151855, -0.9349638819694519, -0.2795797288417816, -1.2859734296798706, -0.2744401693344116, -0.7696375250816345, 0.30994927883148193, -0.1444716900587082, 1.1813653707504272, 0.21465836465358734, -0.2134132981300354, 0.5974366068840027, -0.10522452741861343, 1.415907621383667, -1.0470943450927734, 0.8768486976623535, -0.6966108083724976, -0.2751924693584442, 1.3987033367156982, 0.03103853203356266, 0.1621754765510559, -0.5812902450561523], [1.7556630373001099, 0.5892373323440552, -0.9748266339302063, 1.4871097803115845, 0.3449150621891022, -0.3160647451877594, -1.1506903171539307, -0.11499650776386261, -0.337458997964859, -0.2402692288160324, -0.01843942329287529, -1.3086737394332886, -0.08034881949424744, -1.5292264223098755, -0.6848180890083313, 1.2909566164016724, 0.21059772372245789, -1.1297541856765747, -0.5083792209625244, 0.9941102862358093, -2.319037914276123, -0.40490543842315674, -0.2380586713552475, -0.3645806908607483, -0.06762547045946121, 0.5549679398536682, 0.11790341138839722, 1.1990668773651123, 0.020118622109293938, -1.1512224674224854, -0.15086013078689575, -0.0509384423494339, 0.44834810495376587, 0.4904542863368988, -2.0866284370422363, 0.4445764422416687, 1.3999614715576172, -2.5184402465820312, -1.0230644941329956, 0.4057564437389374, -0.24209637939929962, -0.23218603432178497, 0.34517499804496765, -0.13612055778503418, -0.6657006144523621, 0.09210482239723206, 1.096602201461792, -1.3808200359344482, -1.0320802927017212, 2.431110143661499, -0.06170014664530754, 0.04713570326566696, -0.7843238115310669, 2.2222495079040527, 0.9192696809768677, 0.13540105521678925, -0.6303296089172363, -0.1716175228357315, -0.45642098784446716, 0.28911301493644714, 1.1158891916275024, 0.43962717056274414, -0.562442421913147, 1.5327060222625732], [0.21603980660438538, 1.6177079677581787, 1.8454055786132812, -0.6689066290855408, -0.44137731194496155, 0.3350389897823334, -0.00846436619758606, 0.13400092720985413, 0.5811858773231506, -1.2496144771575928, 0.819563627243042, 1.5589599609375, 0.4405066967010498, -2.223400354385376, -0.4633715748786926, -0.22916528582572937, -0.3058392405509949, -0.008629444986581802, 1.233298659324646, 0.5404771566390991, -0.28250670433044434, 2.837977170944214, -0.9006819128990173, 0.17462454736232758, -0.5372267365455627, 1.241405963897705, 0.13147962093353271, -0.5725314617156982, -1.3521385192871094, 0.0874059721827507, -0.2896321415901184, -0.4928123652935028, -0.5890526175498962, 1.7123299837112427, 0.8591312766075134, -1.2632019519805908, 1.006912350654602, 0.7207159996032715, -1.3163537979125977, -0.3537522852420807, -0.1526191681623459, 1.125023365020752, 0.29590851068496704, -0.6078055500984192, -0.04451233893632889, 1.2024389505386353, -1.1966915130615234, -0.2681625485420227, 0.5378357172012329, 0.45378410816192627, 2.151200771331787, 1.4607559442520142, 0.35376179218292236, 0.6015051603317261, 1.3386338949203491, -0.1967145949602127, -0.25693151354789734, -0.7180506587028503, 0.1439327895641327, -1.30370032787323, -0.9228889346122742, -0.555827260017395, -1.4355696439743042, 1.4397748708724976], [-0.28127455711364746, -1.3776029348373413, -0.7821853756904602, 1.1584314107894897, -1.8443211317062378, 0.4811846613883972, -1.1199417114257812, -0.31089314818382263, 1.4274030923843384, -0.4974246621131897, -0.6846231818199158, -0.8450751900672913, 0.7812429070472717, -0.5866410136222839, -0.9684877395629883, -0.3277006149291992, -0.27505236864089966, 1.030362844467163, -0.6460412740707397, -1.214412808418274, 0.4125615060329437, -1.8931689262390137, 0.3755904734134674, 0.8509675860404968, -0.14403435587882996, -1.3913716077804565, 0.8351430892944336, -0.11850942671298981, -0.006141699384897947, -0.23808744549751282, 0.522176206111908, 0.4007943570613861, 0.1691569983959198, 1.0799193382263184, 2.0385656356811523, 0.5647778511047363, -0.12277209013700485, -1.2154372930526733, -0.37353914976119995, -1.1781866550445557, -1.1515092849731445, 0.9668610095977783, 0.8500126004219055, 0.13169978559017181, -0.4389320909976959, -1.590679407119751, 1.1139484643936157, 1.7275328636169434, -0.4886763393878937, -1.1948696374893188, -0.647910475730896, 1.3273534774780273, -1.2594854831695557, 1.1544684171676636, -0.34631338715553284, -0.4315309524536133, 0.21320058405399323, -1.163784384727478, -0.9824802875518799, -1.1061931848526, 0.7716779708862305, 0.47190868854522705, -0.5389009714126587, -1.0885014533996582], [1.3610920906066895, 0.4262106120586395, 2.6371543407440186, 0.6538894772529602, -0.47705137729644775, 0.7055191397666931, -0.6320503950119019, 0.6768575310707092, 0.42403873801231384, -0.003330598585307598, -0.8725524544715881, 0.5113964080810547, 0.4002298414707184, -0.7770388126373291, -0.8438812494277954, 0.5920924544334412, -1.5332081317901611, -0.6680158376693726, 1.2838306427001953, -2.278513193130493, 0.9486759305000305, 1.4161901473999023, -0.09210314601659775, 1.9084309339523315, 3.818175792694092, -0.36271020770072937, -1.7054035663604736, 2.2103307247161865, 0.29797160625457764, 1.1627780199050903, 1.4658875465393066, -0.6146692633628845, -0.9351674914360046, -0.013432417064905167, 0.9418964385986328, -0.004460510797798634, 0.4951280653476715, 1.2439310550689697, 0.7690967321395874, 1.30802583694458, -2.1162750720977783, 0.19462384283542633, 2.0154409408569336, 0.6417952179908752, -0.7582452297210693, -0.8707838654518127, 1.1863192319869995, -1.6136211156845093, 0.1404862254858017, -0.1548488587141037, -0.6770359873771667, 0.24394318461418152, -0.779315710067749, 1.2597140073776245, 0.3803369700908661, 0.3815006613731384, -0.03456459939479828, 1.347581148147583, 0.2159801721572876, -0.42071500420570374, -2.5116631984710693, 1.4051942825317383, -0.6078661680221558, -0.3350786566734314], [0.16369256377220154, 1.9138624668121338, 0.21743158996105194, -1.8334591388702393, 0.5004594922065735, -0.9557145237922668, 0.5365001559257507, 0.14571815729141235, -0.36233943700790405, 1.2592958211898804, 0.9090408086776733, 1.1544859409332275, 0.0010766556952148676, 1.119612216949463, -0.700010359287262, -0.8620640635490417, 0.808557391166687, 0.9140306115150452, 0.21759167313575745, 0.7821832895278931, 0.9984415173530579, 1.859384298324585, 1.2124983072280884, -0.9791850447654724, -1.684874176979065, 1.5084508657455444, 1.4573500156402588, 0.2908209264278412, -0.5782980918884277, 0.44979098439216614, 0.2875022292137146, -0.17506393790245056, 0.49086564779281616, 1.1654202938079834, -1.8382149934768677, 0.04750295355916023, -0.6044162511825562, 0.7727349400520325, -0.9590772390365601, 0.32424396276474, -0.8589096665382385, 0.4982645809650421, 0.10404769331216812, 1.015643835067749, 0.0002977183321490884, -2.5645878314971924, -0.3334643244743347, -1.0639969110488892, -0.73698490858078, 1.0952489376068115, -1.6712191104888916, -1.76228666305542, 0.105118528008461, 1.7349443435668945, -0.05772646516561508, 0.10516024380922318, -1.3205410242080688, -0.6007177829742432, -1.4191218614578247, -0.031347982585430145, 0.2902936637401581, -1.7901962995529175, 1.059205412864685, -2.0315563678741455], [0.12842372059822083, -0.6240931749343872, -1.4786237478256226, 2.3421030044555664, -1.0680440664291382, -0.47668132185935974, 1.6645851135253906, 0.40976351499557495, -1.8167264461517334, -1.5385948419570923, -0.02410835213959217, -0.10190441459417343, 0.18317192792892456, 0.17957651615142822, 1.2128865718841553, 0.6162052154541016, 0.10406845062971115, 0.9090516567230225, 0.16679249703884125, -0.8106604814529419, 0.735153317451477, 0.5757245421409607, 0.7266482710838318, -1.0950379371643066, -0.14753375947475433, -1.2296712398529053, 0.6219797134399414, -1.1121859550476074, 0.38990023732185364, -0.4839164614677429, -1.4854483604431152, 0.14864085614681244, -0.3939979374408722, 0.41607654094696045, -1.0055519342422485, -0.01908975839614868, -2.3389334678649902, -2.527646780014038, -0.06897031515836716, -1.2820942401885986, 1.8346612453460693, -1.7500611543655396, 1.1050889492034912, -0.313960462808609, 1.0920416116714478, -2.436913013458252, 1.9380117654800415, -0.5714656114578247, -0.5505056977272034, -1.0565011501312256, -0.4131198823451996, -0.3581258952617645, -0.4245781898498535, 0.12347790598869324, 0.13342095911502838, 0.49187326431274414, 0.5651874542236328, -1.1054332256317139, 0.8909394145011902, 0.46117934584617615, 1.8852899074554443, -1.8723863363265991, -0.2610611617565155, -2.853172779083252], [2.1439480781555176, -0.16122958064079285, 0.17927119135856628, 0.09718182682991028, -0.2566766142845154, -0.30110326409339905, -0.8743054866790771, 0.7751107811927795, -2.602107524871826, -1.3269989490509033, 1.0649815797805786, 0.840086042881012, -1.0462385416030884, 0.8584954142570496, 0.5123063325881958, -2.1446585655212402, -0.9499323964118958, -0.06613769382238388, 0.5803629159927368, 1.682026982307434, 0.28338155150413513, -0.8935893177986145, -0.9263719916343689, -0.017816245555877686, -0.991998553276062, -1.2439595460891724, -0.05401879549026489, 0.28278985619544983, -1.0565509796142578, -1.7640775442123413, 0.5037670731544495, 1.4716306924819946, 0.8668228387832642, 2.337724208831787, -0.0653780996799469, 1.642181396484375, -1.3705809116363525, 0.33243122696876526, -0.1469767540693283, -0.41202759742736816, -0.6452663540840149, 0.08305956423282623, 1.061611294746399, -1.0804930925369263, -1.2641422748565674, 0.06215887889266014, -0.39142635464668274, -0.9361523389816284, -0.9815237522125244, 0.10969121009111404, 0.4135451018810272, 1.0169330835342407, 0.18324416875839233, 0.7825540900230408, -0.682563304901123, -0.4372726082801819, -0.14882872998714447, -1.3747837543487549, 1.7683295011520386, -0.08132724463939667, -0.4947807788848877, -1.2345670461654663, -0.9088903665542603, -0.17194367945194244], [0.2548378109931946, 2.310991048812866, -1.0122425556182861, 0.4429140090942383, -2.0124380588531494, 0.4532494843006134, -0.43179214000701904, -0.8115993738174438, -0.39360275864601135, -0.4254329204559326, 0.5849595069885254, 0.36335471272468567, 0.3132248520851135, 0.10029440373182297, 0.015947388485074043, -1.043923020362854, 0.16288898885250092, -0.2570713460445404, 1.0101810693740845, 0.20113565027713776, -0.08309353142976761, 0.37741169333457947, -0.39847713708877563, -1.3425509929656982, -0.6857481598854065, 1.1415382623672485, -0.7363259196281433, -0.2843281924724579, 0.25577685236930847, 2.4795267581939697, -1.675615668296814, -1.4111090898513794, 0.4782940447330475, -0.6765301823616028, 0.2015245258808136, -0.6851044297218323, -0.1760634183883667, -0.19856904447078705, -1.3372302055358887, 0.009043470025062561, -0.7236249446868896, -0.7106516361236572, 0.6477786302566528, -0.8318676948547363, -1.7577496767044067, 0.36333754658699036, 0.41885799169540405, -0.42907559871673584, 0.9384974837303162, -0.2590083181858063, 1.6978332996368408, 0.13328145444393158, 0.9590063095092773, 1.063935399055481, -1.5720553398132324, -1.501844048500061, 1.0049384832382202, -1.8108134269714355, -0.1133880689740181, -0.08561883121728897, -0.8084619045257568, -0.3527023196220398, 0.1710730791091919, 0.05256088450551033], [3.0576388835906982, -0.6603098511695862, 1.1160097122192383, -1.0180641412734985, 1.1253423690795898, -0.6283416152000427, 0.091053806245327, -0.19502249360084534, -0.6789650917053223, 0.45766493678092957, 0.15355587005615234, 0.4870021939277649, -0.42345598340034485, -0.42793381214141846, -0.41653090715408325, -0.3959310054779053, 1.3277004957199097, 0.9016005396842957, -1.0518165826797485, 0.6999713778495789, -0.03695399686694145, -0.2592690885066986, 1.2764610052108765, 0.4304753243923187, -0.11785035580396652, 2.0821855068206787, -1.277220606803894, -0.24143768846988678, 0.8208662867546082, -1.0043123960494995, 1.8654769659042358, 0.6414003968238831, 0.1345292031764984, -0.4288330674171448, 0.24145667254924774, -0.9903261065483093, -0.17487655580043793, 0.12770500779151917, -0.7338810563087463, 0.7623225450515747, -1.48902428150177, 0.3057056963443756, 1.442368745803833, 1.3139822483062744, -0.0656701847910881, 0.24375060200691223, -0.3371262848377228, 1.4444191455841064, -0.7625695466995239, -1.1696752309799194, 1.1227262020111084, -0.7249253988265991, 1.1830896139144897, 0.655250072479248, -0.1598290503025055, -0.5230727791786194, -0.5771331787109375, 0.27641716599464417, 0.27870258688926697, -0.17813044786453247, 0.6006900072097778, -0.5649502277374268, 0.1832202970981598, -0.05476602539420128], [2.24306583404541, -0.030179131776094437, 1.9233818054199219, -0.11333867907524109, 0.17754144966602325, 0.3213414251804352, 0.2601664960384369, -0.8357275128364563, 0.5794060230255127, -0.1709897369146347, -0.32697200775146484, -0.5404452085494995, -0.9028246402740479, 0.4244031608104706, -0.28513240814208984, -0.6386144161224365, -0.21471284329891205, 0.7930806279182434, -0.08677202463150024, -0.42060816287994385, 0.32245588302612305, -1.6562190055847168, 0.5067712068557739, -0.51996910572052, 0.36629730463027954, 1.9577887058258057, 1.0347181558609009, 0.7560901045799255, -0.3331200182437897, -0.6441741585731506, -0.45803338289260864, -0.8077399134635925, 1.3426520824432373, -1.209455966949463, -1.223546028137207, 0.5050978064537048, 0.7529075741767883, -1.058508276939392, -0.6671267747879028, 0.7011266946792603, 0.04684049263596535, -1.42310631275177, -0.16090185940265656, -1.6374865770339966, -0.7705625891685486, -0.6116631627082825, -0.02049158327281475, -0.40565025806427, -0.7574063539505005, -1.1437146663665771, 2.1444571018218994, -0.23128415644168854, -0.6624578237533569, -0.35046133399009705, -1.681511402130127, 0.04993325099349022, -0.7758941650390625, -0.18471524119377136, -0.029751472175121307, 0.015341886319220066, 0.2739339768886566, 0.7521232962608337, 0.28437694907188416, 0.40621548891067505], [-0.024804705753922462, 0.5132901668548584, 0.38860636949539185, -0.1816728264093399, -1.1869659423828125, -0.36656513810157776, 0.26915907859802246, 0.3860887289047241, -0.3569888174533844, -0.17096459865570068, 0.7251277565956116, 0.6932465434074402, -1.4937846660614014, 0.5899564623832703, -0.22037941217422485, -1.8817813396453857, -2.455824375152588, 0.2812381088733673, 1.0010416507720947, 1.129144549369812, 0.5331259965896606, 0.09511423110961914, 0.8404790163040161, -0.3252541720867157, 0.3708086311817169, 1.0986335277557373, -0.5920867919921875, 1.2040364742279053, 0.36742010712623596, 0.411494642496109, 0.5107793211936951, 0.6880608797073364, -1.4417829513549805, 0.0006633663433603942, -0.46023935079574585, 0.405132532119751, 0.4237845540046692, -0.7489288449287415, 1.5681037902832031, -0.8651566505432129, -0.5626874566078186, 0.3063395023345947, -2.275787830352783, 0.8007818460464478, -0.7422981262207031, 1.4525907039642334, 1.7148178815841675, 0.8622615933418274, 0.724496603012085, -1.0958200693130493, -0.8274185061454773, -0.5051106214523315, -0.1449328064918518, -1.0166306495666504, 1.292114496231079, 0.521583080291748, -0.5664100646972656, -1.132868766784668, 0.8702975511550903, 0.3111509382724762, -1.0053256750106812, -2.538930892944336, 2.1656723022460938, -0.8060099482536316], [1.3616666793823242, 1.1523480415344238, 1.7054916620254517, -0.2944720387458801, 0.33850616216659546, -1.051272988319397, 2.6417343616485596, 0.23857977986335754, 0.7967625856399536, 1.3104519844055176, -0.8523914217948914, 0.34132710099220276, -0.866649329662323, 0.17389729619026184, -0.10408628731966019, -0.44957855343818665, 0.6597117185592651, -0.6078253984451294, -1.2541106939315796, -1.5534436702728271, -0.03594084829092026, 1.1906490325927734, 0.6789054870605469, 0.15483063459396362, -0.48511365056037903, 0.7699633836746216, 0.22153005003929138, -1.3065180778503418, -0.6829450130462646, 0.536081075668335, -0.4166729152202606, 0.25032132863998413, 1.833741545677185, -0.2982957661151886, 1.2264683246612549, -0.10234726965427399, 0.8238246440887451, 1.0523370504379272, -1.3390570878982544, -0.9449054598808289, 0.4937697649002075, -0.7111477851867676, -0.8050144910812378, -0.8555657863616943, 0.03201783820986748, 1.0080955028533936, 0.9653390049934387, -1.329884648323059, -1.239327073097229, 0.5673204064369202, 1.4193259477615356, 0.19709105789661407, 1.8162423372268677, -1.124632477760315, 0.8932505249977112, 0.4154292941093445, 0.8098368048667908, -2.360410213470459, 0.7656429409980774, -0.8344572186470032, -1.2185330390930176, -0.21597744524478912, 1.196771264076233, -0.32836663722991943], [0.5102306604385376, 0.1449420303106308, 0.746155321598053, 0.18004707992076874, 0.9954747557640076, 0.13304126262664795, -0.9842492938041687, -1.4426791667938232, 0.13728682696819305, -0.5037129521369934, -0.9292266368865967, 0.5696766972541809, -0.4241110682487488, -0.8182953596115112, -0.3907053470611572, -2.0326642990112305, 0.8805475234985352, 0.5219101905822754, -0.8830311894416809, -0.994610071182251, -0.6838671565055847, 0.09379181265830994, 0.30898118019104004, -0.43679821491241455, 0.16164830327033997, 0.8590416312217712, 0.35488906502723694, -0.6100798845291138, -1.1497265100479126, 1.0852677822113037, 0.09136655181646347, 0.19463570415973663, 0.15496332943439484, -1.7974023818969727, 0.9392367005348206, -1.3033350706100464, 0.6680089235305786, 0.8093662858009338, -1.3081291913986206, 0.6339474320411682, 0.40776026248931885, -1.6356667280197144, -1.6007987260818481, 0.2603442668914795, 0.7780705690383911, -0.6698072552680969, -2.6990957260131836, -0.7865837216377258, 0.872580349445343, -0.7030400633811951, -0.7737511992454529, 1.60299813747406, 0.7277466654777527, 0.9672888517379761, 0.4929467737674713, 0.20001770555973053, -0.0031040951143950224, 0.050033099949359894, 0.2066912204027176, -0.6236152648925781, -1.6301077604293823, -0.318988561630249, -0.8403995037078857, 0.7794976830482483], [-1.4327749013900757, -1.073394775390625, -0.8260132670402527, -1.290260910987854, 0.3886719048023224, 1.141887903213501, 0.016928905621170998, -0.5146916508674622, -1.3376173973083496, -0.1753753274679184, -0.6124323606491089, -1.5945121049880981, -1.6706459522247314, 0.7780007123947144, 0.053245969116687775, 1.9736272096633911, 0.546207070350647, -2.0777487754821777, -0.2451213151216507, -0.17461253702640533, -0.5461740493774414, -0.8646180033683777, -0.5514003038406372, 0.3128788471221924, 0.5028719902038574, -1.478613018989563, 0.6422809362411499, 0.5974945425987244, -0.7562350630760193, 1.3613860607147217, -0.15566295385360718, 0.9952348470687866, 2.3112294673919678, 1.3150279521942139, -2.0186007022857666, 0.0424342080950737, -1.3090901374816895, -1.5631123781204224, -0.6731067895889282, 1.192800521850586, -0.003196872305124998, 0.9605153799057007, -0.428812712430954, 0.35779157280921936, 0.8974881172180176, -0.7778850197792053, 0.8702963590621948, 2.221435785293579, 0.04424549266695976, -0.7379714250564575, -0.8369898796081543, -0.16847850382328033, 0.08053643256425858, -0.7998078465461731, 1.2878323793411255, 1.1613824367523193, -0.5071538686752319, -0.123019739985466, -0.3281252682209015, 0.30750572681427, 0.14641913771629333, -0.08860138803720474, -0.6235833764076233, 1.0589051246643066], [1.6840612888336182, -0.9171088337898254, 0.894820511341095, 0.15951110422611237, -1.0486745834350586, 0.19654230773448944, 0.7210617661476135, 0.2036619782447815, 0.7007734775543213, 1.3252679109573364, -0.06125161424279213, -1.0598928928375244, -1.0232315063476562, 1.0227338075637817, 0.08492852747440338, -0.027578648179769516, -0.27352097630500793, 0.47355398535728455, 1.3157118558883667, -1.5066943168640137, -0.04653027281165123, 0.7793734073638916, -0.20873849093914032, 0.2642582356929779, 0.466309130191803, -0.33730557560920715, -1.3674367666244507, 0.2563607096672058, -0.5533087253570557, 0.9529104828834534, -2.580367088317871, 0.44745972752571106, 0.32097768783569336, -0.4162885248661041, 1.4379072189331055, 0.7602179646492004, -0.07256995141506195, -0.08776476979255676, -0.10123101621866226, -0.39665985107421875, 0.08689673244953156, 0.642427384853363, 0.3134649395942688, -0.292582631111145, 0.5877929329872131, 0.1683908998966217, -0.2083006501197815, -0.31150561571121216, -0.010285230353474617, -0.6678115129470825, 0.9214940071105957, -1.8016456365585327, 0.1365925520658493, 1.2668229341506958, -0.08712305873632431, 2.519055128097534, -0.03301692008972168, 1.219732642173767, -0.5761860609054565, -0.5485206246376038, -0.7545017600059509, 0.719192624092102, 2.603480815887451, -0.43250882625579834], [1.2523703575134277, 0.36128509044647217, 0.5481140613555908, 0.75645512342453, 1.135840892791748, 0.49224480986595154, -0.7857608795166016, -0.20614702999591827, -1.2615944147109985, 1.4051103591918945, -0.27805978059768677, -1.5767486095428467, -0.1796690970659256, 0.136479914188385, 0.6477475762367249, 1.0461139678955078, 0.11819043755531311, -1.679195523262024, 0.12706151604652405, -0.4608258605003357, -0.7719389796257019, -0.24352115392684937, -0.7774087190628052, 0.6321128010749817, -0.17461340129375458, 0.14230279624462128, -0.08254998177289963, -0.28074806928634644, 0.0207002405077219, 0.9701736569404602, 1.221613883972168, 1.4960790872573853, -0.04293004423379898, -0.4506867527961731, 1.2507787942886353, 0.830979585647583, 0.9903246760368347, -1.1051676273345947, -0.375125914812088, -0.515527606010437, -0.27376556396484375, 0.2820892632007599, 0.8424488306045532, 0.6971551179885864, -1.832633376121521, -1.3555543422698975, -0.4383648931980133, 1.5028780698776245, -1.3763070106506348, 1.8368216753005981, 0.8733047246932983, 0.7565663456916809, -1.611190676689148, 2.397033214569092, 1.5066503286361694, -0.42667603492736816, -0.5125632286071777, -0.3738650977611542, 0.4147445559501648, 1.3576983213424683, 0.8137016296386719, 0.6447469592094421, 0.8086296319961548, -0.9887027144432068], [0.18468166887760162, 0.7193228602409363, -0.9165789484977722, 0.5102687478065491, -0.192417174577713, 0.26975104212760925, -0.14690162241458893, -1.3432228565216064, 0.8429371118545532, -0.6957684755325317, 0.880823016166687, -1.5539333820343018, -1.8200308084487915, -0.13100144267082214, -0.023032205179333687, 1.9132553339004517, -0.14580880105495453, 1.1577329635620117, 0.5856553316116333, 1.30171537399292, 1.2557679414749146, -1.7121608257293701, 0.00023683886684011668, -0.45467281341552734, 0.38699087500572205, 0.0682302713394165, -0.5646861791610718, 0.6024578809738159, 0.30371996760368347, 1.9252088069915771, -0.9304523468017578, 0.40737468004226685, -0.47058314085006714, 1.646132230758667, 1.3298075199127197, 0.7875903844833374, 1.2852717638015747, 2.1415693759918213, 0.3099912106990814, -1.6599146127700806, 1.249572515487671, -1.2484694719314575, -0.1302345097064972, -1.234870195388794, 0.4917562007904053, 0.24922163784503937, 0.27835696935653687, 0.8670992255210876, -0.5800866484642029, -0.3302381634712219, -1.501409888267517, -2.1979236602783203, 0.771609902381897, -0.9183444380760193, 0.3681459426879883, -0.8489437699317932, -0.10141322016716003, 1.1999495029449463, -0.29974818229675293, 1.4434194564819336, 0.07277949899435043, -0.41036394238471985, 0.0074604107066988945, 0.010329140350222588], [1.8767634630203247, -0.8236798644065857, -0.5713992118835449, 0.12531208992004395, 1.3182226419448853, 0.34194090962409973, -1.1614885330200195, -0.7334617972373962, 0.5429067611694336, 0.2858477830886841, -0.9910068511962891, -0.7918798327445984, -0.1276569813489914, -0.4278227686882019, 1.203048586845398, 0.2510303258895874, 0.3381192982196808, -0.6384795308113098, 0.3175402581691742, -0.32825446128845215, -0.011304674670100212, -0.38485416769981384, 1.6328712701797485, 0.6756364107131958, -0.6779835224151611, -0.05325795337557793, -1.2710469961166382, -0.5248138308525085, 0.9613511562347412, -0.44964098930358887, -0.640466570854187, -1.1038804054260254, 0.5032966732978821, 0.5326505303382874, 0.43368133902549744, -1.1367753744125366, -1.60610032081604, 0.5746976733207703, -0.2943819463253021, -1.0590537786483765, -0.6054172515869141, -0.6121187806129456, 1.0528991222381592, -0.16785287857055664, 0.470237672328949, -1.2201846837997437, -1.7386388778686523, 0.4832080006599426, -1.7052114009857178, -1.4031438827514648, 1.4235162734985352, 2.1885361671447754, -0.49416103959083557, -0.7551481127738953, 0.9496021866798401, -0.18654875457286835, -0.6083051562309265, 0.49168211221694946, 0.13167501986026764, 0.027675846591591835, 0.6854051351547241, 0.15217669308185577, -0.6291036605834961, 1.0767890214920044], [-0.8935871124267578, -0.19189095497131348, -0.2691514492034912, -0.6778218150138855, -0.516182005405426, -0.1959105134010315, 0.4985189437866211, 0.1804790496826172, 2.3115782737731934, -0.16897056996822357, -0.603585422039032, 1.9513514041900635, 1.60630464553833, -0.2676895558834076, 1.0194076299667358, 1.8181254863739014, -0.2771548926830292, 0.4609341621398926, 0.7399344444274902, -1.699013113975525, 1.029930830001831, 0.4964653253555298, 1.5443835258483887, -0.6264384984970093, -1.5144603252410889, 0.35595372319221497, -1.553572416305542, -0.6172037720680237, 1.1438814401626587, 0.24128980934619904, 0.31021541357040405, 0.5212536454200745, -0.1030861958861351, -0.2292434722185135, 0.3920240104198456, -1.3430730104446411, -0.4956567585468292, 0.3753354549407959, 1.5546507835388184, 0.08906564861536026, -0.5385567545890808, -0.978496789932251, 0.41329339146614075, -0.5098661780357361, -0.5126596093177795, 0.6512343287467957, -2.0425400733947754, -0.2948690354824066, 0.6275118589401245, -1.2189348936080933, -0.7038323283195496, 0.9503761529922485, -0.9387057423591614, 0.22558677196502686, -1.390505313873291, -0.2972820997238159, 1.6158326864242554, 0.1640956550836563, -0.9593057632446289, -0.20445439219474792, 1.0084606409072876, -0.3071247637271881, -0.619313657283783, 1.8391587734222412], [-0.3516821265220642, -0.411441445350647, -0.775521457195282, 1.7322003841400146, -0.16586418449878693, -1.1982526779174805, 0.13549372553825378, -0.5225821733474731, 0.058568406850099564, -1.2500560283660889, 1.333876609802246, -0.9797513484954834, 0.14062143862247467, -2.1047298908233643, -0.2611576318740845, -1.406964659690857, -0.6028401851654053, 0.6250366568565369, -1.0542491674423218, -1.0216833353042603, 1.065671443939209, -0.6557065844535828, 1.5658094882965088, 1.212996006011963, 0.12911193072795868, -0.5910003185272217, 0.663038432598114, -0.6489027738571167, 0.6662630438804626, -1.5948141813278198, -0.5041405558586121, -0.6821818351745605, -0.3418695032596588, -0.7180176973342896, -0.07808968424797058, 0.4812767207622528, -2.188918113708496, -1.5853925943374634, 1.5749045610427856, -1.902740716934204, -0.06129249185323715, 0.17138850688934326, 0.45040491223335266, -1.204940676689148, 4.8707021051086485e-05, -0.0190042182803154, -1.264652967453003, 1.070892572402954, -0.16973209381103516, -2.181816577911377, -0.9151440858840942, 0.07562973350286484, 1.5173090696334839, -0.053646914660930634, 0.659615695476532, -1.5121986865997314, 0.8059390783309937, -0.42200997471809387, -1.3847779035568237, -1.3568837642669678, 0.16843213140964508, -0.0949687585234642, 0.9550638198852539, 0.42550572752952576], [-0.3766272962093353, -0.37466123700141907, -0.14999642968177795, -0.37763118743896484, -0.5158254504203796, 1.1615997552871704, -1.1232068538665771, 1.1037429571151733, 1.287217140197754, -0.13594502210617065, 0.09493949264287949, 0.901482105255127, -0.5462535619735718, 1.325412631034851, 0.2226783037185669, 0.40389248728752136, -1.2223402261734009, 0.3514080047607422, -0.2209915965795517, -0.8994660973548889, 0.04561743512749672, -1.1874945163726807, -0.018174689263105392, -0.18673747777938843, -0.264749675989151, 0.14040574431419373, -0.5709447264671326, -0.035160522907972336, 0.23462916910648346, -0.04329892620444298, -0.3692696690559387, -1.090750813484192, -1.7032047510147095, 2.136514663696289, 0.527929961681366, -0.35287725925445557, -2.565406560897827, -0.6321181058883667, -0.6202636957168579, 1.3945010900497437, 0.9507632851600647, 0.2050701230764389, -0.35406461358070374, 1.5363644361495972, 1.230766773223877, -0.19375553727149963, -0.4903627038002014, -0.5295312404632568, 0.5479856729507446, -0.1259451061487198, -1.4149281978607178, -0.2981167733669281, 0.8824052810668945, 0.2558247447013855, -0.07823286950588226, -2.394785165786743, 1.4311261177062988, 0.09858254343271255, 0.30724290013313293, 0.8007985949516296, 0.5382064580917358, -0.10690979659557343, 1.0014714002609253, -0.5707564949989319], [0.35469114780426025, -0.7030869126319885, -1.5524786710739136, 0.09636842459440231, 0.39605915546417236, -1.4353917837142944, -1.0589025020599365, -0.5275905728340149, -0.13854198157787323, 0.6338041424751282, -1.3639956712722778, -0.19093778729438782, 0.4962471127510071, 0.19410237669944763, -0.6151231527328491, -0.32437750697135925, -1.0134400129318237, -0.9063469767570496, 1.8509166240692139, -0.6595608592033386, -1.2679095268249512, -0.9265102744102478, 0.6250320076942444, 0.4840651750564575, -1.3547680377960205, -0.3351418375968933, -1.9833687543869019, 0.33348244428634644, 0.08992650359869003, -0.667194664478302, 0.4465293288230896, -1.7725123167037964, 0.8059428930282593, -1.0764154195785522, -0.6323701739311218, -0.9772066473960876, -0.804014265537262, -0.8487676978111267, 0.3910079300403595, -0.40759262442588806, -1.2594738006591797, 0.18110834062099457, -0.09870124608278275, -0.3024890124797821, -0.1801557093858719, -0.3741947114467621, 0.3520980179309845, 1.3169702291488647, 1.0998384952545166, -0.4205057919025421, -1.3531968593597412, -0.13746744394302368, -0.9546943306922913, 0.07904642075300217, -0.7178388237953186, -0.375598281621933, -0.8287873864173889, 0.11090996116399765, -1.1140577793121338, 0.9161932468414307, -1.2829514741897583, 0.5837653279304504, 0.6348704695701599, 0.0599789135158062], [-0.043724045157432556, -1.0574556589126587, -0.9346153736114502, 0.2858351469039917, 1.6803425550460815, -0.6650437116622925, 0.6893836855888367, 0.9127371907234192, -0.6959517002105713, 0.13625651597976685, -0.451231449842453, 2.2183618545532227, 0.08804001659154892, -0.179277241230011, -1.6078516244888306, -0.7100348472595215, -0.5729082822799683, -0.38535961508750916, -0.44311878085136414, -0.4888903796672821, 0.6328632235527039, -0.5562623143196106, -1.6494405269622803, -0.291252076625824, -2.5955636501312256, -0.18815486133098602, 0.6101452112197876, -0.7207827568054199, 1.3293901681900024, 0.9253624081611633, -0.7832103967666626, 1.2728468179702759, 0.17364193499088287, 0.1345885843038559, -0.3735083043575287, -0.10423362255096436, -0.3270062208175659, -0.7157096266746521, -0.8903025388717651, 0.12082822620868683, -0.16721533238887787, 1.3734890222549438, -0.7970662117004395, -0.4349973499774933, 0.5387543439865112, -0.8666158318519592, -0.06716315448284149, 1.3234319686889648, 1.3357784748077393, 0.6697839498519897, -0.7610583901405334, -0.6630332469940186, 1.1162768602371216, -1.308450698852539, -0.5157987475395203, -0.5941063761711121, -1.6935856342315674, 0.11799167841672897, -0.8337015509605408, 0.5059701204299927, 0.01598208025097847, -0.4194169342517853, -0.3649997115135193, 0.018266139551997185], [-0.6747820973396301, -2.435701370239258, 0.4525472819805145, -0.31598353385925293, -0.22244837880134583, -0.42038172483444214, -1.734627604484558, -0.4446709156036377, -0.15049642324447632, 0.8111044764518738, -0.6426600217819214, 0.5865259766578674, 0.5247359871864319, -1.661856770515442, 0.5620366930961609, -1.9318119287490845, -1.4817103147506714, -0.576765239238739, 0.8727607727050781, -0.9696584343910217, -0.19723768532276154, -0.45784756541252136, 0.4671020805835724, 0.6200299859046936, -0.6755119562149048, 0.11148267239332199, -1.0673869848251343, -0.06505060195922852, -0.33255553245544434, -0.2973095178604126, 0.8937531113624573, -0.19927483797073364, -2.3786544799804688, 0.27735841274261475, -1.5876450538635254, -0.4796123802661896, -2.4308888912200928, -1.9531755447387695, -1.2203028202056885, 1.0535780191421509, -0.02384001575410366, -0.5890194773674011, -0.15920263528823853, -2.0296669006347656, -0.2255694419145584, -1.7322118282318115, -0.6099796295166016, 1.9411746263504028, 0.7790974378585815, 0.08953641355037689, -0.8005132079124451, 0.8798929452896118, -1.3956795930862427, 0.3989715576171875, 1.293250560760498, -0.06622577458620071, 0.9215342402458191, 3.8947510719299316, 0.6638232469558716, 1.0855594873428345, 0.4483441412448883, -1.2848199605941772, 1.3339059352874756, 0.052325718104839325], [2.4664547443389893, 0.4191471040248871, 1.1274733543395996, -0.47026482224464417, -0.5561394095420837, 0.6275542378425598, -0.19607235491275787, 0.5675443410873413, 0.6347482204437256, -1.5373330116271973, -0.3698470890522003, 1.7898468971252441, 0.2682938575744629, 1.8670378923416138, 1.503719449043274, -1.3734506368637085, 0.4499194324016571, 0.4532826244831085, -0.5712937116622925, -1.59163236618042, -0.4857983887195587, 0.18849903345108032, 0.25749263167381287, 1.045735239982605, -0.40639379620552063, -0.47516152262687683, 0.5126277804374695, -0.052459120750427246, 0.534547746181488, 0.5017030239105225, 0.8046631217002869, 0.2019135057926178, 0.2630562484264374, -0.7107700705528259, -0.0466771125793457, 0.55942702293396, 0.822043240070343, 0.48656731843948364, -0.4371318817138672, 0.6944904327392578, 0.4704338014125824, 0.11329019069671631, 0.5431002378463745, -0.02410225011408329, 1.1700767278671265, -1.1994147300720215, -0.01962142065167427, -0.00907337199896574, -0.44719743728637695, 1.090129017829895, -1.0090917348861694, -0.880103588104248, -0.28448185324668884, -0.9492024779319763, -0.41440558433532715, 0.7132774591445923, 0.5934153199195862, 0.12990404665470123, 0.1664937138557434, -0.8786046504974365, -1.8214993476867676, 0.0542723722755909, -1.1032241582870483, 0.2416090965270996], [-0.019391382113099098, -0.7940725088119507, 0.1165260523557663, -0.053212497383356094, 1.1153112649917603, -0.3806772232055664, 1.0741806030273438, 0.9389386177062988, -0.10614654421806335, -0.2623653709888458, 1.1575652360916138, 2.103388547897339, -0.39241522550582886, -0.4847502112388611, -1.2994180917739868, 0.10680920630693436, 1.2596700191497803, -1.0450776815414429, -1.3316011428833008, -0.22609621286392212, -0.7051240801811218, -0.05560854449868202, 0.5053719282150269, -0.26373091340065, 0.5537893772125244, -0.8809172511100769, 2.31112003326416, 1.2000188827514648, -0.5010499358177185, 0.48034489154815674, 0.09787842631340027, 0.03706684708595276, -0.20587359368801117, 0.5560054779052734, 0.6339462399482727, 0.036241281777620316, 0.10432861745357513, -1.4596443176269531, -1.4688618183135986, -0.17916330695152283, -0.1685096025466919, 0.2176014631986618, 1.0274555683135986, 0.24134372174739838, -1.1827073097229004, -1.0658321380615234, -0.8300603628158569, 0.8385171890258789, 0.6366214752197266, -0.5034611821174622, 0.7555537223815918, -0.7946024537086487, 0.7554347515106201, 0.4908409118652344, 0.6978777647018433, 0.4307062327861786, -0.1298528015613556, -1.1437880992889404, -1.2113642692565918, 0.030557487159967422, 0.0704483613371849, 0.38148438930511475, 1.0902800559997559, -1.1581774950027466], [0.490568608045578, 0.11082614213228226, 1.6781444549560547, 1.77326500415802, 0.5619459748268127, -0.12536872923374176, -0.9331201910972595, -1.3120503425598145, 0.13473565876483917, 1.872381329536438, -0.4096710979938507, 0.9487718939781189, 0.9602159857749939, 1.0033886432647705, -0.8663676977157593, -0.44444912672042847, 0.7819109559059143, -0.9783844351768494, 0.05182979628443718, 0.5326167941093445, -0.2847009003162384, 2.1347687244415283, -1.3161388635635376, 0.24815082550048828, 0.701230525970459, 0.3630238473415375, -1.3575047254562378, 0.8947655558586121, 0.14598287642002106, 0.1179022565484047, -1.76154363155365, -3.629258155822754, 0.43150272965431213, 0.8780675530433655, -1.0818442106246948, -1.130632758140564, -0.5319852232933044, -0.4359551966190338, 0.7012668251991272, 0.6728267073631287, 0.5207541584968567, -3.096696138381958, -1.6430904865264893, -0.9571670889854431, 0.3577239215373993, -0.3614168167114258, 1.7938799858093262, -1.138236165046692, 0.33186599612236023, -0.030964884907007217, 2.250257730484009, 0.19770731031894684, 0.5159918665885925, 0.6645206809043884, -0.13383233547210693, 0.09127609431743622, 0.6898574233055115, 2.051098585128784, 0.24753138422966003, 0.2808508276939392, -2.0957067012786865, 0.8321660161018372, -0.9711331725120544, -0.4580230116844177]], "dtype": "float32"}, {"__nd__": [[0.18683065474033356, -2.547105073928833, 2.188209295272827, -2.9772298336029053, 0.5052353739738464, -0.3641848564147949, 0.21155428886413574, 1.1319997310638428, -0.838808536529541, -1.1145431995391846, -0.27265506982803345, 0.3131217658519745, -2.0726635456085205, -0.11736136674880981, -0.07604145258665085, 0.3449048101902008, 0.36789241433143616, 0.5038065910339355, 0.40638497471809387, -0.5657163262367249, -0.9997144937515259, 1.4095793962478638, -0.6252074241638184, 0.15656159818172455, 0.6998900771141052, 0.1912502944469452, -0.28566285967826843, -1.3359581232070923, 1.2875840663909912, 0.7515042424201965, -0.13502588868141174, 0.9832943677902222, 0.4025346040725708, 1.3735626935958862, 1.1158030033111572, -0.1738462895154953, -1.7634786367416382, 2.0259766578674316, 0.5009103417396545, -0.9207556843757629, 1.4766405820846558, 0.1545269787311554, -1.1074960231781006, 0.3648494482040405, 1.3713269233703613, 2.5592892169952393, 0.2515753507614136, 0.009573744609951973, -1.2384490966796875, 0.8293868899345398, -1.569629430770874, 1.165738582611084, 1.2222462892532349, -1.3912173509597778, 0.14978274703025818, -0.6899198889732361, 0.5890461802482605, -1.4649641513824463, -0.14580388367176056, -1.1487659215927124, 1.1228829622268677, 0.21602770686149597, -1.9279866218566895, 0.5312024354934692, 0.018023261800408363, 1.6086052656173706, -2.1864285469055176, -1.06804621219635, 0.6956365704536438, 0.5248381495475769, -0.25835371017456055, 1.0767704248428345, -1.18975031375885, -0.10699732601642609, 0.20144233107566833, 0.7641730308532715, -0.31205183267593384, -0.21769197285175323, 1.7438685894012451, 0.25707441568374634, 0.744998574256897, -0.14981834590435028, 0.9360592365264893, -0.5783786177635193, -0.8447331190109253, 0.6006035804748535, 1.211738109588623, -1.7005053758621216, -0.11263945698738098, -0.5640031099319458, -1.2774808406829834, 0.5340853929519653, -0.18756462633609772, 0.8771249651908875, 0.5831727385520935, 2.730196475982666, -0.9084885716438293, 1.0591613054275513, 0.31338879466056824, -0.5703891515731812, 1.9971387386322021, 0.40735411643981934, -0.1321018785238266, 0.050215285271406174, 0.37285682559013367, -0.9145424365997314, -0.6437334418296814, 0.06604073196649551, -1.946822166442871, -0.2626226842403412, -0.8211739659309387, 0.5519856810569763, 0.6504243016242981, 0.8492632508277893, -0.4967687427997589, -0.8381749391555786, 2.4907824993133545, 2.452516794204712, 0.8603317141532898, 1.1529592275619507, 0.4288848340511322, -0.31166645884513855, 1.224313497543335, -0.666532039642334, 0.0872960239648819, -0.037772852927446365, 0.25430232286453247, 0.3681240677833557, -1.008091926574707], [0.31783327460289, 0.17222703993320465, -0.07765281200408936, 0.16935867071151733, 1.7047100067138672, -2.5377395153045654, -0.4191281199455261, -0.014357690699398518, 0.7675962448120117, 0.11962952464818954, -0.2868804633617401, -2.005905866622925, -0.5214062929153442, 1.0451291799545288, -2.2898924350738525, 0.877207338809967, -0.3678020238876343, -0.8940147161483765, 1.0902924537658691, 0.7606456875801086, -0.3713293671607971, -0.7084997296333313, 0.9797632694244385, 1.9321237802505493, -0.32286885380744934, -0.32575297355651855, 1.4802645444869995, 2.1136438846588135, 0.43023961782455444, 1.0443726778030396, -0.6826373338699341, -0.09730887413024902, 0.41638121008872986, -0.8151024580001831, -2.0633697509765625, 0.5151689052581787, 1.0249412059783936, 1.2253092527389526, -0.3825396001338959, -0.8503367304801941, -0.887724757194519, 0.16136164963245392, 0.06653788685798645, -1.7720052003860474, 0.35011377930641174, -2.4668521881103516, 1.9313654899597168, 0.5070515871047974, 0.03630569949746132, -0.0793173536658287, -0.7350209951400757, -1.0802671909332275, -1.2702672481536865, -1.2976078987121582, 0.9418087005615234, -0.5582008361816406, 1.2638943195343018, 0.4038221836090088, 1.3533374071121216, 0.2101602405309677, 0.7129351496696472, -0.42829644680023193, -1.5437042713165283, 1.4823819398880005, 1.0160729885101318, -0.03641674295067787, 0.036154408007860184, 0.14976896345615387, 0.42420899868011475, 1.2222286462783813, -0.1078987792134285, 0.014234217815101147, 0.29610273241996765, -0.289993017911911, -0.503191351890564, -1.4659557342529297, 1.51889169216156, -0.3298755884170532, 1.0943100452423096, -0.7337341904640198, 0.35133665800094604, 0.8355968594551086, -0.3117578327655792, 0.10331103950738907, -1.4748971462249756, -1.1132228374481201, 0.9204492568969727, 0.6118268370628357, 1.225220799446106, -0.9163643717765808, 0.07245902717113495, 0.1550740897655487, 0.3336600959300995, 1.969333529472351, 0.34151211380958557, -1.2611799240112305, 0.3926769196987152, -0.09502124786376953, -1.3851300477981567, 1.0643483400344849, 0.02485482580959797, -0.6954725384712219, -0.6192420721054077, 0.8091660141944885, 1.8016327619552612, 1.129518747329712, -0.7992802262306213, 0.8039170503616333, -0.4879661500453949, -1.531781554222107, 0.45608627796173096, -0.3855203092098236, 0.24370591342449188, 0.23223048448562622, -1.1641284227371216, -0.8083723783493042, -0.6954365968704224, 0.6731856465339661, -0.25031229853630066, -1.028525948524475, 1.0796853303909302, -0.4828285872936249, 1.1578011512756348, -0.2893967926502228, 0.20303449034690857, 0.19747821986675262, 0.06685829162597656, -0.49675729870796204, -0.6425672769546509], [-0.9484432339668274, -0.31574010848999023, -1.1257774829864502, 1.300869345664978, 0.719301700592041, -0.1884566992521286, 1.7034447193145752, 0.7705270648002625, 0.19918082654476166, -2.2516582012176514, -0.7382844686508179, -0.3468594551086426, -0.7192599177360535, 1.498310923576355, -0.30318132042884827, 0.03015262447297573, -1.0680173635482788, 0.7899007797241211, -2.2088921070098877, -0.5178832411766052, -0.4521962106227875, -0.07322326302528381, 1.9801512956619263, -0.7095649242401123, -0.767460823059082, -0.2439851313829422, -0.2115451842546463, 0.35144051909446716, 0.18632517755031586, -0.17533151805400848, 0.34529033303260803, -0.33893218636512756, -1.4383066892623901, -1.482519268989563, 0.7009456157684326, 1.2124578952789307, 0.11519495397806168, 0.003842995734885335, 1.0539261102676392, -0.19908897578716278, -1.4036022424697876, 1.4888290166854858, 0.44665229320526123, -1.3592312335968018, -0.25796112418174744, -1.372245192527771, 0.6005613803863525, -0.8895357847213745, 0.43418583273887634, -1.3389630317687988, 1.7329802513122559, 0.07450199127197266, -1.7869654893875122, 0.4844887852668762, -0.29064300656318665, 0.7528735995292664, -0.17279654741287231, 1.4296964406967163, -1.2045050859451294, 1.1140390634536743, 1.467726707458496, 0.207523375749588, -0.7834553122520447, 0.6780737042427063, -0.5779068470001221, 1.1252411603927612, 1.1586782932281494, 1.2901089191436768, 0.9474480748176575, 1.6720410585403442, -0.589088499546051, 0.3894074857234955, 0.41236451268196106, 0.263814240694046, -1.6343153715133667, -1.0662108659744263, 1.3778877258300781, 1.5410739183425903, 0.30533143877983093, -0.7562986612319946, -0.4646036624908447, -0.5652210116386414, 2.1345224380493164, 1.5696526765823364, -0.14001798629760742, 0.3376155197620392, -1.464834213256836, -0.1908874362707138, 2.3161158561706543, 0.1903829425573349, -0.030229689553380013, 0.30084532499313354, 0.21399728953838348, -0.06101900339126587, 2.0097923278808594, 0.11809968203306198, 2.211779832839966, -0.9857469201087952, 2.1032843589782715, 0.7537989616394043, -0.9586586356163025, -0.8622395992279053, -0.34326767921447754, -1.3542160987854004, -0.8563635945320129, -0.17813251912593842, -1.765691876411438, 0.5134880542755127, -1.1203974485397339, -0.04838208481669426, 0.4038655161857605, -0.6499025821685791, 0.14014223217964172, 0.7252324819564819, -0.24210284650325775, 0.8131511807441711, -0.985933244228363, -0.6293664574623108, 1.4730148315429688, -0.2064424306154251, -1.9386723041534424, 0.33774250745773315, -0.8918871283531189, 0.43266239762306213, 1.1268742084503174, -1.3650155067443848, -0.6731773018836975, 1.3093299865722656, -0.17344136536121368], [0.7142011523246765, -0.17808324098587036, -0.8707358837127686, -0.012436390854418278, 0.7191213369369507, 0.47196969389915466, -1.996123194694519, 0.31692904233932495, -0.7523059844970703, 0.7520366907119751, -0.07815881818532944, 0.26052069664001465, 0.27445465326309204, -0.680361807346344, -0.05125517398118973, 0.048468563705682755, 1.7817341089248657, -0.7417618036270142, 1.086287021636963, -0.7149319052696228, 0.05085314065217972, -1.1867552995681763, 0.5562896728515625, 0.26208460330963135, 1.6222646236419678, -0.47599872946739197, -1.013307809829712, -0.1450847089290619, -1.2139942646026611, -0.8672139644622803, 0.7321414947509766, 0.10618945211172104, 0.6355161666870117, -0.710040807723999, 0.9708920121192932, -1.0972986221313477, -0.48251107335090637, -0.4543871581554413, -0.8514635562896729, 1.1161242723464966, -1.2566624879837036, 1.089316487312317, 0.7762694954872131, -0.31973978877067566, 0.016875075176358223, -0.3586727976799011, -0.5913486480712891, -0.22920392453670502, -1.1418582201004028, -0.36688435077667236, -0.4434433579444885, 0.1340114027261734, -0.5457882881164551, 0.5414448976516724, 0.06919899582862854, 2.4537365436553955, 1.8520166873931885, -0.7766470909118652, -0.1979498267173767, -0.37018030881881714, -0.708959698677063, 0.48961618542671204, 0.036181434988975525, 0.27179279923439026, 0.436365008354187, -0.38294774293899536, 0.0671931803226471, -0.38961631059646606, -1.4694936275482178, -0.7086357474327087, 0.9900455474853516, -0.6562450528144836, 1.7364856004714966, 1.6446635723114014, -0.29575562477111816, -0.09645569324493408, 0.1491634100675583, -0.883918821811676, -1.1589406728744507, 0.6875981092453003, -1.2967519760131836, -0.29265978932380676, 0.6939668655395508, 0.05414479225873947, 0.6200740337371826, -0.07069292664527893, 1.1711739301681519, -2.096553087234497, 0.10774249583482742, 0.2834668755531311, 0.5298148989677429, -0.6556863784790039, 0.8387434482574463, -1.2573959827423096, 0.06846866011619568, -0.6818385720252991, 1.49140202999115, 0.1365605741739273, -0.5078082084655762, 0.5235341191291809, 0.7338866591453552, -2.326411485671997, -1.0008617639541626, -0.30461493134498596, 0.16379334032535553, -1.2459391355514526, -1.8722013235092163, 1.2745743989944458, -0.4999643862247467, -1.717191219329834, 0.8777626752853394, 0.05330132693052292, -0.4979848861694336, -1.2400310039520264, -1.1875818967819214, 1.1753541231155396, 1.5113160610198975, -0.9883524179458618, 0.180337592959404, -0.9358401298522949, -0.67669278383255, 0.5853433609008789, -1.1023370027542114, -0.2926345765590668, -1.3668094873428345, 1.3472353219985962, -0.7235004901885986, -0.4909837543964386, -0.5365144610404968], [0.2649582624435425, -2.055755615234375, -1.0794289112091064, 0.9221311211585999, 1.6103489398956299, 0.33764809370040894, 0.05515022575855255, 1.2105984687805176, -0.3993164300918579, -0.74998539686203, 2.604970693588257, -0.5238223075866699, -1.8489338159561157, 1.0384769439697266, 2.879452705383301, -0.8019413948059082, 1.26679265499115, 0.041783396154642105, 0.8907755017280579, 2.1215827465057373, 2.2794508934020996, 1.284901738166809, 1.5038425922393799, -1.6044057607650757, -0.08056811988353729, 0.29975438117980957, -1.7338849306106567, 1.232980728149414, 0.990266740322113, 1.1349093914031982, -0.027135813608765602, -0.36561286449432373, -0.9965344071388245, 0.17891371250152588, -0.786250114440918, -0.2892095148563385, -0.7238718271255493, -0.24936792254447937, -1.4009736776351929, -0.48709389567375183, 2.0412299633026123, -0.27648603916168213, 1.005637288093567, -0.47010537981987, 0.7712163329124451, 0.49159690737724304, 0.1581955850124359, -1.4146513938903809, 0.5993651151657104, 0.5780994892120361, 0.4258963465690613, 0.7341670393943787, 0.37150219082832336, -0.05012022703886032, -0.663581132888794, 1.6440037488937378, 1.0406486988067627, 0.004749891813844442, 1.3081119060516357, 0.577144205570221, -2.0816187858581543, 1.5466891527175903, 1.575257420539856, -1.2996253967285156, 1.0621429681777954, -0.371421754360199, -1.5829130411148071, 0.21524643898010254, -0.8084354400634766, 1.4991705417633057, -0.6764938235282898, 2.3308990001678467, -0.7506880164146423, -0.5846746563911438, 0.21730440855026245, -0.09752712398767471, 0.37505194544792175, -1.1730539798736572, 0.7827728986740112, 0.4938628077507019, 0.6239229440689087, 1.3087495565414429, -0.16625764966011047, -0.08938252925872803, -1.1107304096221924, -1.2401100397109985, -0.06715469807386398, -0.6652594804763794, -1.108608603477478, -0.1607600748538971, -2.0086891651153564, 0.015278127044439316, 0.15691961348056793, 1.942014217376709, -0.4387947916984558, -0.4155944883823395, -0.1508626490831375, -1.1594433784484863, 0.3558579981327057, -0.39181363582611084, -0.764531672000885, 2.094435691833496, -0.06537102162837982, -0.06730342656373978, -0.21210679411888123, -0.7957877516746521, 0.4942571222782135, 0.31303685903549194, -1.7613213062286377, 0.36042916774749756, 1.2982839345932007, -0.9788447022438049, 0.672719419002533, 0.8696652054786682, 1.1732479333877563, 0.8882266879081726, 1.4402530193328857, 0.20410645008087158, 0.44167354702949524, -1.0053882598876953, 0.029469411820173264, 1.374400019645691, -1.3971295356750488, -0.3018321990966797, -0.3015446960926056, 1.2213789224624634, -0.16939133405685425, -0.8339543342590332, 0.22866284847259521], [-1.0688530206680298, 0.5044357776641846, 0.17124363780021667, -1.1279584169387817, 0.29243937134742737, 0.783223569393158, -0.8844744563102722, -1.194827675819397, 1.6885433197021484, -0.08907382190227509, 1.3473567962646484, 1.2169362306594849, 0.14620889723300934, 0.3358440399169922, 0.14398854970932007, -1.1378469467163086, -1.392740249633789, 0.5719523429870605, -1.53056001663208, 0.08860065788030624, -0.412578284740448, 0.48990756273269653, -0.5576788187026978, 0.2510119676589966, -0.8276716470718384, -0.19746385514736176, -1.8613954782485962, 1.3138574361801147, 0.784525454044342, 0.003947408404201269, 1.1835312843322754, -0.24261313676834106, -0.05015943944454193, -2.428116798400879, -1.0034148693084717, -0.10933101922273636, -1.0742263793945312, -0.8818836808204651, 1.5563987493515015, 0.8718290328979492, -1.1316893100738525, 0.005463855341076851, 0.09648183733224869, -1.1615028381347656, -2.862867832183838, 0.5346265435218811, -0.03322971239686012, -0.3163081109523773, 0.9420554041862488, 0.07289234548807144, -0.27477937936782837, 0.05558701232075691, 0.7920622229576111, -0.21843090653419495, 0.7217531204223633, -0.25998786091804504, 1.037091851234436, -1.7017927169799805, -0.19034115970134735, -0.04884345456957817, 1.3249329328536987, 0.4187254309654236, -1.8656392097473145, 0.8574029803276062, -0.6474536061286926, -0.13062989711761475, -0.5702369809150696, 0.7046213746070862, -0.7026412487030029, 0.8133348226547241, -1.527184247970581, -0.7249647378921509, 1.112671136856079, -0.9759721159934998, 1.027786374092102, 1.3706690073013306, -2.0467920303344727, 0.7008162140846252, 0.3520628809928894, 1.3268351554870605, -0.19709785282611847, 0.9150558710098267, 0.24362175166606903, -0.6452914476394653, 0.24300774931907654, 1.1739072799682617, 0.4318962097167969, 0.3976214826107025, 2.1934916973114014, -0.4625507891178131, -0.21350687742233276, 1.5875622034072876, -0.12296845018863678, 0.020301438868045807, 0.08139481395483017, 0.5212077498435974, 0.7243682146072388, 0.23913414776325226, -1.715575933456421, -0.5746215581893921, 0.09453770518302917, -1.2698084115982056, -0.22809791564941406, 0.8894363045692444, 0.7172605395317078, -0.3910525441169739, 0.6113847494125366, 0.8821619153022766, 1.1465224027633667, 0.821229875087738, -0.5243366360664368, -0.9603694081306458, -1.5657240152359009, -1.676927924156189, 0.7978973984718323, -0.2517630457878113, -0.1525861918926239, -1.2435451745986938, 1.835714340209961, -1.0937249660491943, -0.914937436580658, -1.7041053771972656, 0.344777911901474, -0.2831753194332123, 1.1003693342208862, -0.058992717415094376, -0.9475695490837097, 0.10502629727125168, 1.0350673198699951], [0.119473397731781, -1.5069091320037842, 0.3886968195438385, -0.4299852252006531, 0.25519686937332153, -0.7819617986679077, 0.38955986499786377, -1.0656901597976685, -0.40210068225860596, -0.3716990351676941, -0.5376731753349304, 0.2977067828178406, 2.6217234134674072, 0.8039020299911499, 0.7214305400848389, -1.327223539352417, -0.4982246458530426, -1.0979392528533936, 0.8815334439277649, 2.4781875610351562, 1.3392733335494995, 1.0892211198806763, -0.9576124548912048, -0.468484103679657, -1.6261940002441406, -0.7970814108848572, -1.0181750059127808, -1.1736291646957397, -0.7617676854133606, 0.21570651233196259, -0.590701699256897, -0.5569517016410828, -2.053136110305786, 0.09478078782558441, 0.34738099575042725, 0.6285642385482788, 0.6141191124916077, 0.18237069249153137, 2.338773488998413, -0.4657491147518158, -0.9813152551651001, 0.3043140470981598, 0.7573550343513489, -1.5707309246063232, -1.2602459192276, -0.19159191846847534, 0.008346086367964745, 2.332827091217041, -0.802648663520813, 0.3264240324497223, -0.5471076965332031, -0.17426355183124542, -1.017293095588684, -0.9558737277984619, -0.25632554292678833, 0.06376489996910095, 1.7417800426483154, 0.05682387948036194, 0.5458346009254456, 0.13300172984600067, -0.15649139881134033, -0.9732657074928284, 1.0311447381973267, -0.39582452178001404, 1.6640117168426514, -0.5551422238349915, -1.1913810968399048, 0.4552345275878906, -0.15465612709522247, 0.045523062348365784, -0.5005438327789307, -0.07404547929763794, -1.5745383501052856, 0.4148149788379669, 0.9649721384048462, -0.6419563293457031, -1.5584485530853271, -0.3969953656196594, -0.37603792548179626, -0.2567322552204132, 0.13169847428798676, 0.8629998564720154, -0.06232091411948204, 0.8501471877098083, 0.7788568139076233, -0.3506057560443878, -0.28286057710647583, -1.6859562397003174, -1.1050364971160889, -0.8051193356513977, 1.564483642578125, -0.9239394664764404, -0.1617061197757721, -0.6920360326766968, 0.9515920281410217, -1.4437693357467651, -0.5120442509651184, 0.938127338886261, 0.059310223907232285, 2.0257444381713867, -0.6287832856178284, -2.400528907775879, -0.9923806190490723, 0.31160464882850647, 0.3920089900493622, 0.018117135390639305, 0.2979592978954315, 1.2900363206863403, -1.3264203071594238, -0.9824737906455994, -0.9940371513366699, 1.7158105373382568, 0.5356016159057617, -0.4560166895389557, 1.0690003633499146, -1.1699492931365967, 0.17973776161670685, -1.863540530204773, -0.7054731249809265, 0.4057038128376007, 0.4878576695919037, 0.7544034123420715, -0.4272034764289856, 0.5652198791503906, 0.9945446848869324, 1.6453707218170166, -0.2968844473361969, 0.46875372529029846, -1.3666914701461792], [0.6826493740081787, 1.1232670545578003, 0.6640174388885498, 1.4089572429656982, -0.6793336272239685, 0.27148938179016113, 0.8013913631439209, -0.5541372299194336, 0.7079623937606812, -1.8795535564422607, -1.56508469581604, 0.4951975643634796, -0.5148199796676636, -0.34122636914253235, 0.22728902101516724, -0.3374083340167999, 0.6380822062492371, 1.2133030891418457, 0.4100487530231476, -1.5766289234161377, 0.6378423571586609, -1.8128756284713745, -1.1300665140151978, 1.5158439874649048, 0.3335033357143402, -0.31937339901924133, 0.8308054208755493, -0.12389209866523743, 0.6671229600906372, -1.788170337677002, 0.9240574836730957, -1.569994330406189, 1.6042890548706055, 1.2345342636108398, 1.0444144010543823, -1.4742779731750488, 0.65353924036026, -1.4734997749328613, 0.570338249206543, -0.2334711104631424, 0.8632473349571228, 1.771009087562561, 0.10883209854364395, 0.6595270037651062, -1.0186562538146973, 0.10768095403909683, 0.35527151823043823, -0.5578107833862305, 0.4235478341579437, -1.1262226104736328, -0.8583501577377319, 2.175340414047241, 0.14075234532356262, 0.10051555931568146, 0.384450227022171, -1.8840854167938232, -1.6241631507873535, 0.9456511735916138, 1.3951150178909302, 0.20524223148822784, 2.103243112564087, 0.03627778962254524, 0.9909123778343201, -0.9328184127807617, -0.12176604568958282, 0.8318547010421753, -0.5263632535934448, -0.8894359469413757, 0.2532278299331665, 0.2741204798221588, -0.7908159494400024, -1.7537492513656616, 0.7649413347244263, -1.5747225284576416, -1.9529939889907837, 2.6963582038879395, 1.1428171396255493, -0.6182432770729065, -0.21297436952590942, 0.760179340839386, -0.0844452977180481, -0.41255080699920654, 0.45680978894233704, -0.32005658745765686, -0.9649630784988403, -0.934590220451355, -1.0302306413650513, 0.14804670214653015, 0.6288118958473206, 0.1510835736989975, -1.037786841392517, -1.8288342952728271, -1.5790090560913086, 0.12280523777008057, -0.031806737184524536, 1.0805257558822632, -0.5401343703269958, -0.6211014986038208, 0.7963343262672424, 1.4480044841766357, -0.07312508672475815, -0.8471952080726624, 2.4830732345581055, 0.16286444664001465, -0.4837590157985687, 0.6363804340362549, 0.4195614457130432, 0.05716779828071594, 0.006472993642091751, -0.3862116038799286, 1.4084715843200684, -0.022712919861078262, -3.2589476108551025, -0.29934507608413696, -1.0256307125091553, -0.08569779992103577, 0.052219945937395096, -0.2507750689983368, 0.7173371911048889, 1.060030221939087, 0.10307218134403229, -0.12744224071502686, 1.4252923727035522, 0.26378852128982544, 1.6086424589157104, -2.0567269325256348, 0.7290652990341187, -0.6092175245285034, -0.021881533786654472], [0.9753017425537109, 0.9368959069252014, -1.305449366569519, -0.11493759602308273, 1.529416561126709, 0.042510345578193665, 0.25197821855545044, 1.4966161251068115, 0.27087053656578064, -0.6265267133712769, -0.27630314230918884, -1.184395432472229, 3.6275501251220703, 1.8061577081680298, 0.9785467386245728, 0.1771395355463028, -3.0901830196380615, -0.12208622694015503, 1.0981580018997192, 0.19796834886074066, 0.11028703302145004, 0.8786445260047913, -0.8166728615760803, -0.3949267566204071, -0.27170857787132263, 1.4008005857467651, -0.11109479516744614, 0.759848952293396, 0.21928095817565918, 0.8751353621482849, 0.20753400027751923, -0.11756015568971634, -0.5945625901222229, 1.326158881187439, -1.4700133800506592, -1.2206987142562866, 0.2891587018966675, 0.31302228569984436, -1.178076982498169, -0.4072026312351227, -0.17966265976428986, -0.43747836351394653, -1.230783224105835, 0.6106781363487244, -0.16036026179790497, 1.6681430339813232, 1.4100652933120728, 0.7091227173805237, -0.5851952433586121, -0.14897999167442322, -0.9247825145721436, -1.4860658645629883, 0.02311124838888645, -0.5130720138549805, 1.2635823488235474, -1.1450170278549194, -1.0370891094207764, 2.84944224357605, -1.9790788888931274, -0.084991954267025, -0.26536881923675537, -0.6049001216888428, 0.8356970548629761, 0.18106123805046082, 2.8764171600341797, 0.6790381073951721, 1.901969075202942, -1.2108491659164429, 0.6736418604850769, -0.2726448178291321, 0.4508388042449951, 0.018862321972846985, 0.025302134454250336, -0.1655808985233307, -0.5668801665306091, 0.06014016643166542, 1.8285789489746094, -1.7406857013702393, -0.9564709067344666, -0.569140613079071, 0.6199352741241455, 0.45382967591285706, -1.050439715385437, 1.5101304054260254, -0.25778141617774963, -0.47307026386260986, 0.978691041469574, 0.9096689820289612, -2.536046266555786, -0.9324464797973633, 0.41255512833595276, 1.5422219038009644, -1.8123910427093506, 0.30161574482917786, -0.38608235120773315, -0.6974183320999146, 0.20616717636585236, 0.4771304726600647, -1.4553273916244507, -1.1409730911254883, -0.9347898364067078, 0.3326363265514374, -0.6734811663627625, 0.15834900736808777, 0.2393820732831955, -0.21364644169807434, 1.5805126428604126, 0.3782632350921631, 0.044053927063941956, 0.13493528962135315, -0.22818762063980103, 0.7247470021247864, 0.1693432629108429, -1.0914413928985596, -0.9162303805351257, -0.050496120005846024, -0.00648609921336174, 0.8832139372825623, 0.21112625300884247, -0.37680038809776306, 1.3592149019241333, 1.7988989353179932, -0.26810014247894287, -1.3212614059448242, -0.7927382588386536, 0.8049821853637695, 1.619895339012146, -0.4538927674293518, 0.6392829418182373], [-0.4774358868598938, 0.4393359124660492, -0.2109169214963913, -0.5376250147819519, -0.8352881073951721, 0.9034553170204163, 0.46474191546440125, 0.5623422861099243, 1.0367006063461304, 0.0298526082187891, 0.002008062321692705, -0.46424809098243713, 0.5149343013763428, 0.9516754746437073, -0.09406907856464386, -0.6312841773033142, -0.7351751923561096, -0.6936865448951721, 1.461896538734436, -0.285552978515625, -0.7004497647285461, 2.4638094902038574, -0.9066042900085449, -0.6957716941833496, 1.4831345081329346, 0.749289870262146, -1.022141933441162, 1.4442931413650513, 2.2705142498016357, 0.09789343178272247, 0.8876031041145325, 0.26729637384414673, 0.3366243243217468, -0.01665090024471283, -0.47829321026802063, -1.5531615018844604, 0.30852481722831726, 1.4887782335281372, -2.5420799255371094, 0.014405157417058945, -1.805358648300171, 0.12528476119041443, -1.5158276557922363, -0.8930360078811646, -0.48639899492263794, -1.356650710105896, -0.24219034612178802, -0.5552197098731995, 0.7641322612762451, 1.0245709419250488, 1.2073343992233276, 0.309978187084198, 0.36344802379608154, 0.9490001201629639, 1.677786946296692, 0.02268097922205925, -0.8359565734863281, 1.6076287031173706, 0.2384532392024994, 1.0683366060256958, 0.4254607558250427, -0.4365794062614441, -0.5591135621070862, -0.38443484902381897, 0.8957608938217163, -1.2705814838409424, 1.2210623025894165, 0.05685006454586983, 1.1092807054519653, -0.8831892013549805, -0.07142698019742966, -0.012723629362881184, 1.3579009771347046, -1.265939474105835, 1.1549880504608154, -0.46806085109710693, -0.7999368906021118, -0.4601369798183441, -0.4337621331214905, 1.5747641324996948, 0.5865052938461304, -1.1066032648086548, -0.8348065614700317, 0.6077003479003906, -0.38351762294769287, 0.9121412634849548, -0.5530843138694763, -1.6321215629577637, -0.5014442801475525, 0.2847525179386139, 0.008630504831671715, 0.04807494953274727, 0.11304748058319092, -0.13202539086341858, 0.5854834914207458, -0.701580822467804, 0.7001774907112122, -0.07303830236196518, -0.2304667979478836, -0.29377448558807373, -1.076815128326416, -0.6022697687149048, 0.5387511253356934, 1.5275874137878418, -1.9304585456848145, 0.4025667607784271, 0.7566586136817932, 1.3117258548736572, -1.0616189241409302, 1.2717444896697998, -0.28616514801979065, 0.04992780089378357, -0.6051151752471924, -0.0277742650359869, 0.10083527117967606, 0.3647873103618622, -0.22794519364833832, -0.45767542719841003, -0.27697333693504333, 1.0271035432815552, -0.2659189701080322, 0.012715302407741547, 2.407665967941284, 0.7306926846504211, 1.6591980457305908, 1.9484509229660034, -0.11017865687608719, 0.6690950393676758, 0.6250657439231873], [-0.23153577744960785, -0.0725279301404953, 0.7862905859947205, -0.22839896380901337, 0.16177265346050262, -0.6884737014770508, 0.5060250163078308, 1.5235179662704468, 0.9606508016586304, 0.23998652398586273, 0.3735154867172241, -1.5188579559326172, 0.7558372616767883, 0.44434240460395813, 0.6254954934120178, -0.014293968677520752, 0.6061429381370544, -0.5114259719848633, 1.5563219785690308, 0.16049695014953613, -0.6314785480499268, 0.564877450466156, -3.251391887664795, 0.4536519944667816, 2.1358284950256348, 0.9711653590202332, -0.35220420360565186, 0.17155902087688446, 1.7252781391143799, 0.2325514554977417, 0.79243004322052, -0.9084246754646301, 1.3735445737838745, -1.2275354862213135, 0.8046613931655884, 0.022936731576919556, -0.08654080331325531, 1.4434893131256104, -1.6802524328231812, -0.3054322898387909, -0.09802619367837906, -1.255126953125, 1.600924015045166, -1.0267443656921387, -0.9933062195777893, 0.04930383712053299, 0.20252032577991486, 0.753237783908844, 0.3349950611591339, 0.5103099942207336, -1.2283045053482056, 0.24026243388652802, 0.5381581783294678, 1.4841262102127075, 0.5674092769622803, 0.6042594313621521, -0.6225710511207581, -0.13178172707557678, -0.3980165123939514, 0.35099735856056213, -0.9391394257545471, 0.35284313559532166, 0.22414857149124146, 1.6055887937545776, -1.2146639823913574, -1.0783076286315918, 0.6737262606620789, -0.8197326064109802, 0.7296794056892395, -1.8062101602554321, -1.337131142616272, -1.7720824480056763, -0.27188023924827576, 1.358095407485962, -1.5347577333450317, 0.7355602383613586, -1.208448052406311, -0.2333674132823944, -0.830427885055542, 0.4251408874988556, -2.0338823795318604, 2.495755195617676, 0.20693933963775635, 2.4027657508850098, 1.8975366353988647, -0.6572712659835815, 0.8591569066047668, -1.2593531608581543, 0.5579554438591003, 1.5371112823486328, 0.3035376965999603, -1.0611387491226196, 0.365467369556427, -1.0632699728012085, -0.7376583218574524, 0.3422897756099701, -0.7062124609947205, 0.3673497438430786, 0.5268774628639221, 0.23857666552066803, 0.9601544141769409, 0.6474320888519287, -0.417037695646286, 1.8720180988311768, 0.8096222281455994, -0.07126236706972122, 0.29862180352211, -1.081558108329773, -0.82023024559021, 0.5876938104629517, -1.7278696298599243, -1.9479364156723022, 0.16510134935379028, -1.059118390083313, -0.01923106424510479, -0.6438398957252502, 0.9262778759002686, 0.16511736810207367, 1.3479528427124023, 0.32457539439201355, 1.1847710609436035, 1.451032042503357, 1.9722306728363037, -0.9187341928482056, -0.27607882022857666, 1.745148777961731, 0.7107602953910828, 0.3352380692958832, 0.8452681303024292], [-0.7042903304100037, 2.3102169036865234, 0.12213359028100967, 1.577235221862793, -1.2455098628997803, -0.19974671304225922, 1.236022710800171, 0.2142563760280609, -0.1132279485464096, 1.3341519832611084, -0.10409244894981384, -0.8633210062980652, -0.1750313639640808, 0.3474763035774231, 0.010058460757136345, 1.141089916229248, -1.0028131008148193, 0.7029857635498047, -0.7890875935554504, 0.7208655476570129, -0.536790668964386, 0.5268129110336304, -0.8395102620124817, -0.0366651825606823, 0.4881289005279541, -0.5358692407608032, 2.0402231216430664, 1.6467862129211426, 0.154478520154953, 1.4611119031906128, 2.129211902618408, -0.26564210653305054, -0.12196885049343109, 0.9940349459648132, -0.9938041567802429, 0.31094101071357727, -0.9899638891220093, -1.0248994827270508, 1.9387224912643433, 0.7790466547012329, 1.1435669660568237, -0.1710282862186432, -1.2536184787750244, 0.33504146337509155, 0.47336387634277344, 1.2177327871322632, 0.9283552169799805, 0.9585338830947876, -0.9425897598266602, -1.265325903892517, -1.6023262739181519, -0.44101038575172424, 1.3404661417007446, 2.4853570461273193, -1.2619929313659668, 1.2322663068771362, -0.17715738713741302, 0.8738861083984375, -1.2422250509262085, -0.36906880140304565, 0.7789957523345947, 1.7707135677337646, -0.4594302177429199, -0.03341050446033478, -1.1404963731765747, 0.6711328625679016, 0.4659689664840698, 1.7513070106506348, -0.32201114296913147, -0.9385968446731567, -1.1566201448440552, 1.8890353441238403, 1.1092545986175537, 0.38619524240493774, 1.1714175939559937, -0.4703716039657593, 1.1978532075881958, 1.0469121932983398, 0.9973151087760925, -0.9584895372390747, -1.6209568977355957, 0.6085612177848816, -0.06652163714170456, -1.4511315822601318, 0.3433223068714142, 0.09356524050235748, 0.3315731883049011, -0.6871623992919922, 1.4288830757141113, -0.5026282668113708, 0.21955899894237518, -1.4396460056304932, -1.7317612171173096, -1.93580162525177, 0.3935920298099518, 0.2554294466972351, 0.8215645551681519, -0.10639843344688416, 0.8434629440307617, -0.5581797361373901, 1.2045456171035767, -1.606465458869934, 0.31802958250045776, 0.10101359337568283, -0.26423200964927673, -1.9785592555999756, 0.13352103531360626, -0.18643803894519806, -0.4383029043674469, 0.44353747367858887, -0.4972892105579376, -1.1846961975097656, 0.787163496017456, -1.095846176147461, 1.3637213706970215, -0.8201666474342346, 1.1731847524642944, -2.1970489025115967, -0.5763148069381714, 0.3202402889728546, -0.5073438286781311, -0.16042768955230713, -1.1682196855545044, 0.8207167387008667, -0.5986215472221375, -0.13149242103099823, 0.47213226556777954, 0.9919019937515259, -0.279384046792984], [-0.4410824775695801, -1.8438352346420288, 1.6604139804840088, -1.419061541557312, -1.6139923334121704, -0.6844088435173035, -0.06685622781515121, -0.1469423919916153, 0.926181972026825, -0.5178253650665283, 0.3247013986110687, -2.023345947265625, -0.23772463202476501, -0.3466958701610565, -0.908903956413269, 0.05385809764266014, -2.395277976989746, 1.100904941558838, -0.6868807077407837, -1.1252847909927368, -0.6450372338294983, -0.14783763885498047, -1.0652815103530884, 1.7087361812591553, 0.1301271617412567, -0.43169885873794556, 1.4643431901931763, 0.9794224500656128, -1.3627492189407349, -0.4457978904247284, 1.9622989892959595, -1.2563097476959229, -0.14719495177268982, -1.1722049713134766, -0.02056249789893627, -0.007088535465300083, 0.752297043800354, 0.5489342212677002, 1.8367220163345337, 0.488198459148407, -1.9645153284072876, -0.235646590590477, -1.4247137308120728, -0.4013373851776123, 1.8233656883239746, -3.382016181945801, -1.2399479150772095, -1.449475884437561, -1.159948706626892, 0.5323789715766907, -1.3902662992477417, 1.2114506959915161, -0.3370349705219269, -0.02613857574760914, 0.525543212890625, -1.3692151308059692, 0.29368317127227783, 0.07472732663154602, 1.0632715225219727, 0.4175914227962494, 0.15473313629627228, -0.9275524020195007, -0.13618528842926025, -0.015963150188326836, -0.47459203004837036, -0.1320507675409317, -0.5807825922966003, -0.7843719720840454, -1.1790790557861328, -0.494025319814682, -0.63944411277771, 2.4959957599639893, -1.0925573110580444, 0.019280510023236275, -0.8888875246047974, -0.15810434520244598, -0.405880331993103, 0.8711761832237244, 1.1413121223449707, -0.7441083192825317, 0.05446413531899452, 0.14615975320339203, -0.2272353619337082, 1.7090195417404175, 0.7934218049049377, -0.5766168236732483, 0.5416702032089233, 1.11550772190094, -1.5769665241241455, -0.6993358731269836, 0.6618317365646362, -0.6667434573173523, 0.0035804794169962406, -0.27355650067329407, -0.9486223459243774, -0.7926086783409119, 1.0069113969802856, -0.5582712292671204, -0.7165212035179138, 0.5409067869186401, 0.8644523620605469, 0.3552223742008209, -0.2831232249736786, -0.08703036606311798, 0.3205070197582245, 0.043878521770238876, 1.8781918287277222, -0.6810846924781799, 0.6754551529884338, 0.8744324445724487, 0.8582370281219482, -0.6064749956130981, -0.9241235256195068, -1.960342526435852, -0.12231212854385376, 0.6175857186317444, 0.3610444962978363, 0.5085133910179138, 0.9672448039054871, -2.225372791290283, -0.46094250679016113, 1.0661284923553467, -0.6644903421401978, 1.4153815507888794, -0.1549256145954132, 0.7403602004051208, 2.166086196899414, 0.7739607095718384, 1.8444353342056274], [-2.2650153636932373, 1.1210927963256836, -0.5579332709312439, -1.2200490236282349, 1.681307315826416, 1.2048611640930176, -0.7008392810821533, 0.47906845808029175, -0.95070880651474, -0.17286106944084167, -0.4233289062976837, -0.2726019024848938, 0.38697734475135803, 0.03261025622487068, 0.5543392300605774, 0.6370851993560791, 1.218505620956421, -1.0419620275497437, -1.0729689598083496, 1.8489748239517212, 0.8686848878860474, 1.5436750650405884, -0.594126284122467, 1.1933821439743042, 1.368924856185913, -0.6965278387069702, -0.7873873710632324, -1.4842740297317505, -0.47821757197380066, -1.284840703010559, -1.28227961063385, 0.08393169939517975, -0.10634321719408035, -0.42588913440704346, -0.0374968983232975, -0.5165245532989502, -1.1516438722610474, -1.1109504699707031, 0.0866597518324852, 0.973187267780304, 0.8611288666725159, 0.1791669726371765, -0.12802492082118988, 0.8630781769752502, 2.3126862049102783, 0.5745925307273865, -1.3292906284332275, -0.10980416089296341, 0.5684522986412048, 1.584532380104065, -1.0230953693389893, 0.3038058578968048, -1.0300196409225464, -1.1490246057510376, 0.8162515759468079, 1.7736179828643799, 1.681398868560791, 1.1393433809280396, 0.48791876435279846, -0.45309731364250183, 0.5261190533638, -0.8652490973472595, 0.729677140712738, -0.8138886094093323, 1.3093236684799194, 1.2016243934631348, -0.7723386883735657, -1.3245775699615479, -0.46705663204193115, 0.07913974672555923, 0.766860842704773, -1.0816552639007568, 0.04910578206181526, 0.6344655752182007, -0.7164945006370544, 0.24233099818229675, -0.7027043104171753, -1.598358392715454, -0.7501407861709595, -0.4022538661956787, -1.6984657049179077, 0.3513419032096863, -1.8241568803787231, 1.4586186408996582, 0.634459912776947, 1.4309078454971313, -0.25947457551956177, 0.4499295651912689, 1.81901216506958, -0.05306585878133774, 0.4901689291000366, -0.6278725862503052, 2.096252679824829, -0.3713214695453644, -1.6432240009307861, -1.6107081174850464, -1.6375298500061035, -0.7586376070976257, 0.0758991613984108, 1.2183544635772705, -0.9872817993164062, -1.9598290920257568, -0.7314902544021606, -2.4535624980926514, -0.07147449254989624, 0.6604222655296326, 0.1710342913866043, 0.7541255950927734, 2.458707094192505, -1.2153029441833496, 1.9794321060180664, -1.4017239809036255, -0.39690229296684265, -2.190288782119751, -0.25815245509147644, -0.05907606706023216, -0.2311834841966629, -0.1932581663131714, -0.1290583610534668, -1.0418983697891235, 0.7978467345237732, 0.12658186256885529, 1.0746958255767822, -0.37389522790908813, 0.27300211787223816, -1.4080060720443726, -0.9670973420143127, -0.9261570572853088, 0.9430019855499268], [0.6626257300376892, -0.24302376806735992, 0.10586214810609818, -0.35497620701789856, 2.0626273155212402, -0.9956274628639221, -1.392220377922058, -0.5245497822761536, 0.7943746447563171, -1.5807459354400635, 1.0468688011169434, -0.705682635307312, 0.05146538466215134, -0.20325778424739838, 1.1228556632995605, 1.423664927482605, 1.645855188369751, -0.8382120132446289, 0.7944139242172241, -0.99757319688797, -0.7358852028846741, 1.0639606714248657, -0.7672406435012817, -0.325224906206131, 0.03512139245867729, 0.2906728684902191, -0.6517512202262878, 1.3706210851669312, -1.2931878566741943, -0.23600468039512634, 0.6576246619224548, -0.6773623824119568, 0.8036735653877258, 0.9299245476722717, 0.22027909755706787, 0.1548614203929901, -1.4304463863372803, -0.9982934594154358, 0.19309864938259125, -1.0934900045394897, 1.0840057134628296, 0.7382954359054565, 1.8940082788467407, 0.616922914981842, -0.3239089548587799, 1.7080087661743164, -1.4083391427993774, 0.2023804783821106, -0.6190727353096008, 1.4407044649124146, 1.0953303575515747, -0.053550634533166885, -0.3019851744174957, 1.259925365447998, 1.1698991060256958, 0.09474552422761917, 1.0698531866073608, -0.016563400626182556, 2.35510516166687, 1.6351814270019531, -1.2783312797546387, -1.0429868698120117, 0.14974583685398102, 1.0506912469863892, 1.1572352647781372, 0.022060707211494446, 0.752001166343689, 1.6529344320297241, 1.1816496849060059, -0.36211028695106506, -0.5669309496879578, 0.21207883954048157, -0.6479759812355042, -0.7111843824386597, -0.22469596564769745, -1.1277563571929932, -0.5559848546981812, 1.1703451871871948, -1.9337854385375977, -0.6469800472259521, 1.3899483680725098, -0.6532682180404663, -1.0970799922943115, -0.6266646981239319, -0.10853225737810135, -1.4710997343063354, -0.9332003593444824, -1.1145821809768677, -0.368455171585083, -0.4669495224952698, -1.649635910987854, -0.19562393426895142, -0.6743202209472656, -0.2488640695810318, -1.1293307542800903, 2.231536388397217, 1.4777170419692993, 0.4753487706184387, 0.4847346246242523, -0.08833098411560059, 1.3339674472808838, -0.7561197280883789, -0.3469526767730713, -1.0676511526107788, 0.031427014619112015, -0.4224811792373657, 0.4985668957233429, -0.300354927778244, -0.2363707423210144, -0.8835757374763489, -0.12631836533546448, 0.16209188103675842, 1.1161900758743286, 1.688890814781189, -0.40788111090660095, -2.1091480255126953, -0.9114063382148743, -0.7941678166389465, 1.961172342300415, 0.08210915327072144, -0.8589422702789307, -1.2947821617126465, -1.0940581560134888, 0.6602985858917236, 0.8885259032249451, 0.1913897842168808, -0.8704214096069336, 2.6292238235473633, 1.6755067110061646], [0.09561356902122498, -0.5470620393753052, -1.4616566896438599, 0.3936085104942322, 0.7331811785697937, 1.281306266784668, 0.6692151427268982, 1.8332407474517822, 0.375785768032074, 0.9181393980979919, 0.5169808864593506, 0.2339307814836502, 1.88205885887146, 1.4002397060394287, -0.926738440990448, 1.0300074815750122, 0.033759795129299164, 1.0769851207733154, -0.4280247986316681, 2.567250967025757, -1.1835582256317139, 0.3629853427410126, -0.25199195742607117, 0.6019536256790161, -1.5743328332901, 0.11501257121562958, 0.3524293899536133, -0.599555253982544, 1.8360931873321533, 0.8901252150535583, 0.4897077977657318, -0.649884045124054, 0.8919274210929871, -0.07189887762069702, 1.7441500425338745, 0.1575648933649063, 0.08126532286405563, -0.25332820415496826, 0.908103346824646, 1.4016200304031372, -0.050622571259737015, 0.3183246850967407, -0.1634320318698883, -0.205221027135849, 1.6005949974060059, 0.046164512634277344, -0.8421409130096436, 0.14196988940238953, -2.096646547317505, -1.183616042137146, -1.938599705696106, -0.868701696395874, 1.9688329696655273, 0.17859132587909698, 0.7414045929908752, -1.5716705322265625, 1.9309139251708984, 0.3871903121471405, -2.599886655807495, -2.240154981613159, 0.5814136266708374, -2.485322952270508, 0.1257573366165161, -0.9264615774154663, 0.399966835975647, -0.12506146728992462, -1.031283974647522, -0.11912491172552109, 0.785797655582428, -1.0561046600341797, 0.5948116183280945, -0.6638191938400269, 0.8833242654800415, -0.274651437997818, 1.6166218519210815, -0.02425648830831051, -1.410151720046997, 0.9763143062591553, 1.5544947385787964, 1.8300694227218628, 0.3320017158985138, -2.053158760070801, 1.119099497795105, -0.20265892148017883, -0.33977770805358887, 1.4746935367584229, 0.06387665122747421, 0.061179373413324356, -1.2245063781738281, -0.9196543097496033, -1.00113844871521, -0.6324350237846375, -0.558211624622345, 0.7728943228721619, -1.4677320718765259, -0.6450733542442322, 1.863868236541748, -0.03483650088310242, 0.8366196751594543, -0.045245058834552765, -1.7411108016967773, 0.7730551362037659, 0.21071265637874603, -0.4155953526496887, -0.12023365497589111, -1.0799576044082642, -0.003645131830126047, 0.7523832321166992, -0.38333532214164734, -0.36634230613708496, 1.8416697978973389, 0.3586958050727844, 1.9400473833084106, -1.635699987411499, -1.8500546216964722, 0.20118969678878784, 0.5536969304084778, 0.24141648411750793, -2.3359482288360596, -0.8929417729377747, 2.0251195430755615, 0.3812704384326935, 1.879132866859436, -0.2123783528804779, -1.087110996246338, 0.5512032508850098, 0.26329782605171204, -0.1420314460992813, 0.8613144755363464], [-0.5638993382453918, 0.04100465774536133, 0.8867184519767761, -0.15807634592056274, 1.1624654531478882, -1.0805004835128784, 0.6238000392913818, -1.3005796670913696, 1.8375567197799683, -1.482500433921814, -0.06795518100261688, -0.00913893524557352, 0.9068246483802795, 0.257199764251709, -1.4570759534835815, -0.42583751678466797, -2.0001542568206787, -1.2988662719726562, -1.3497231006622314, -0.3283516466617584, -1.1922246217727661, -2.460538387298584, -0.8496191501617432, 1.5280638933181763, 1.0753639936447144, 0.5009452700614929, -0.027099953964352608, -0.6127195358276367, -0.39171966910362244, 0.5195019245147705, 0.5988913774490356, 0.9034652709960938, -0.6765578389167786, 1.0117684602737427, -0.7013312578201294, 1.659512996673584, 1.2298095226287842, -0.7597220540046692, 2.265451431274414, 0.25035393238067627, -0.5419577360153198, -1.0345641374588013, -0.6342340707778931, -0.5670722723007202, 0.372275710105896, 0.26306894421577454, 0.3383057415485382, -1.3136712312698364, -0.4670715630054474, -0.6724891066551208, 0.038376279175281525, 0.604224681854248, 0.6031133532524109, -1.9917842149734497, 0.1327424943447113, -1.057247281074524, -0.9054953455924988, -0.42717230319976807, 0.23770609498023987, 0.3858107924461365, -0.45541033148765564, -0.33463993668556213, -0.1906081587076187, 0.5983690023422241, -0.062210455536842346, 0.31981411576271057, 0.725398600101471, 0.35126471519470215, 0.1012895479798317, -0.2816980481147766, 0.3771703541278839, -1.1103782653808594, -0.5560681819915771, -0.2973167300224304, -1.6806340217590332, 0.9783070087432861, -1.307509422302246, -1.1829493045806885, 1.0203444957733154, -0.6197026968002319, -1.088365077972412, -1.0319619178771973, 0.9910743236541748, 2.151667356491089, 0.3173690736293793, -0.21809421479701996, -0.28527095913887024, 1.2509148120880127, 0.5301478505134583, 0.03865892067551613, 1.3531726598739624, 0.17527686059474945, 1.5345799922943115, -0.5341176390647888, -1.131661295890808, -0.7636744976043701, 0.5507205128669739, -1.6072343587875366, 0.3031718134880066, 0.6242004036903381, 0.30902665853500366, 1.1716793775558472, -0.9167978763580322, -0.4678899943828583, 0.637050986289978, -1.3414775133132935, 0.541424036026001, -0.6336165070533752, 0.9926760196685791, 0.4130043685436249, 0.9860410094261169, -0.5693602561950684, 0.07914909720420837, 2.385885238647461, -0.457283079624176, 1.3717150688171387, -0.9344755411148071, -0.3882787525653839, 1.1604905128479004, -1.2155134677886963, 0.8695643544197083, -0.007135747000575066, -0.008898247964680195, -0.23519179224967957, -1.1980713605880737, 1.503077507019043, 0.8915289044380188, -1.6871943473815918, -0.2538231313228607], [-0.036056745797395706, 0.45257383584976196, -3.012756586074829, -0.3318588137626648, -0.14005284011363983, 0.3682076632976532, 1.207998275756836, 0.3012309968471527, -2.5152573585510254, 2.2346513271331787, 0.8849549293518066, -1.007025122642517, 0.41440752148628235, -0.8963744640350342, -0.0722445547580719, -0.4828417897224426, -0.6034550666809082, -0.4313644468784332, -0.5497514009475708, -1.3832310438156128, 1.7730525732040405, 0.38863202929496765, 0.797878086566925, -1.0126550197601318, -0.041456907987594604, 2.241623640060425, 0.12439525872468948, 0.5566107630729675, 1.134477138519287, -0.20845063030719757, 0.8566754460334778, -0.9953914284706116, 1.1630268096923828, -1.256049633026123, 0.6554192900657654, 0.44921454787254333, -0.6148030161857605, -0.39201027154922485, 1.1157885789871216, 0.06501897424459457, 0.19712477922439575, -1.9528831243515015, 0.36893537640571594, -0.12131483107805252, 0.15961627662181854, -0.6271891593933105, -1.70357084274292, -1.1157606840133667, -0.6121329665184021, -0.3931739628314972, 0.9693095088005066, 1.123060941696167, 0.11307666450738907, 0.7472368478775024, 1.0113469362258911, -1.2010674476623535, -0.9322654008865356, 0.9198543429374695, 0.36244910955429077, -0.49646642804145813, -0.41921544075012207, 0.6610888838768005, -0.7673571705818176, -0.0029771472327411175, -0.920010507106781, -0.07118858397006989, 1.356863260269165, -0.357710063457489, -2.6572015285491943, 1.1226886510849, 0.1901172548532486, -1.1845405101776123, -0.16979250311851501, 1.0186457633972168, -0.514682948589325, 0.9194093942642212, -1.4195352792739868, 1.4726886749267578, -1.6702966690063477, -0.5830802917480469, 1.0893661975860596, 0.9450442790985107, 1.2211395502090454, -0.5286104679107666, -0.3164670467376709, 0.7382677793502808, 1.482080340385437, 0.9630604386329651, 1.1336528062820435, -1.4257879257202148, -0.7238615155220032, -0.12996169924736023, 0.30021244287490845, 0.8439998626708984, 0.3928583860397339, -0.43367084860801697, 0.7019885778427124, -0.4147515594959259, 0.14227180182933807, 0.422194242477417, -0.8380282521247864, 0.5291628241539001, -1.2188048362731934, -0.9333962202072144, 0.4270349144935608, -0.05043138191103935, 0.478148877620697, 1.2417423725128174, 0.017123330384492874, -1.0358028411865234, -0.5408667325973511, 1.0617737770080566, 0.9606322646141052, 0.38047051429748535, 0.05350729078054428, -0.4131423234939575, -0.6791713237762451, 1.0242953300476074, -1.3225419521331787, -0.8397429585456848, -0.6191229820251465, -0.055422019213438034, -1.1178697347640991, 1.9837064743041992, 1.6534205675125122, -0.7489569187164307, 0.1392275094985962, -0.6936818957328796, 0.030568651854991913], [0.12484758347272873, 0.5745647549629211, 1.0405805110931396, -1.0927975177764893, -0.4142930805683136, 1.4666638374328613, -0.9077726602554321, -1.2935212850570679, 0.34350669384002686, -0.58806312084198, -0.8718401789665222, -0.5533840656280518, -0.7603204846382141, 0.22067150473594666, -0.08284814655780792, 1.3729960918426514, -1.1276209354400635, -0.3010101020336151, -0.49685171246528625, 0.6281716823577881, 1.4336079359054565, 0.7869083285331726, -0.15127970278263092, 0.6878740191459656, -0.3723263442516327, 0.40952998399734497, -1.4332515001296997, 0.04215589165687561, -0.0824369490146637, -0.13751035928726196, -1.2184029817581177, 0.5660645365715027, 0.2387804090976715, 1.3570789098739624, 0.49754565954208374, 1.651658535003662, -0.22679319977760315, -0.4315527379512787, 1.0665045976638794, 0.4668799638748169, -0.8201695084571838, 1.0016674995422363, 0.5969970226287842, -0.6483302712440491, 1.8127942085266113, -0.6102392077445984, 0.22732508182525635, 0.15789884328842163, 1.8459104299545288, 0.3762844204902649, -0.3002013862133026, -1.575957179069519, 0.33423447608947754, 0.2528277635574341, -1.2996352910995483, -0.6821531057357788, 1.2363336086273193, -0.14733290672302246, 0.8311758041381836, 1.691691279411316, 0.6513829827308655, 0.8611754775047302, 0.7137773036956787, -0.9494109153747559, 0.18135175108909607, 0.34339895844459534, -0.3474549651145935, 1.1235519647598267, 0.5995149612426758, 0.8090317845344543, -0.06566490978002548, -0.650354266166687, -0.5903939604759216, 0.12194553762674332, -1.0134984254837036, 2.0170233249664307, -0.16296061873435974, 0.1864626258611679, 0.8810042142868042, -1.42145574092865, -0.4388566315174103, -0.47364354133605957, 1.0855975151062012, 0.10930033773183823, 0.2688222825527191, 0.17164841294288635, 0.6006789803504944, -1.7137832641601562, -0.6665256023406982, -0.43208521604537964, -0.8593096733093262, 1.3274210691452026, -0.8712154626846313, 0.2941378951072693, 0.5152938365936279, 0.5117392539978027, 0.11282767355442047, -2.493062734603882, 1.033128023147583, 0.34998828172683716, 1.0495964288711548, -1.1866872310638428, 0.6343660354614258, -1.0148597955703735, 0.5602541565895081, -1.6546530723571777, -0.19261117279529572, -1.354280948638916, -0.7850131392478943, 1.4823024272918701, -0.9923874139785767, -0.05168457701802254, -0.6878644824028015, -0.3595774471759796, 1.6359999179840088, -1.2480701208114624, -0.3402700424194336, -1.1095538139343262, 1.327660083770752, 0.3496609330177307, -1.127716064453125, 0.23793575167655945, 2.629007577896118, 0.28476008772850037, -1.4225877523422241, -0.5324673056602478, 0.8299337029457092, -0.4786261022090912, -0.2488659918308258], [-0.05128977447748184, -1.2878283262252808, -0.45107340812683105, -1.1559098958969116, 0.8749176859855652, 1.2293413877487183, -0.5237801671028137, -1.3936480283737183, 0.6348690986633301, 1.0092734098434448, 0.3201414942741394, 0.3072134256362915, 1.611345887184143, -0.589068591594696, -0.21811877191066742, -0.639528214931488, -0.07740303874015808, 0.2634996473789215, -0.2410692721605301, -0.5439742207527161, 0.8484364151954651, 1.0400642156600952, -0.09225241839885712, 1.6236107349395752, -0.3750753700733185, 0.130431666970253, 2.4941868782043457, -0.025451278313994408, 1.5282948017120361, 2.431575059890747, -0.25300103425979614, 0.9265164136886597, -0.7185924649238586, 1.5995925664901733, 0.4018568992614746, -0.027932271361351013, -0.7349417209625244, 0.8937773108482361, 1.3437186479568481, -1.589698314666748, 0.40943291783332825, 0.34730246663093567, 0.8157006502151489, -1.2880825996398926, 0.22001726925373077, -0.23580586910247803, 0.24485543370246887, -0.5965795516967773, 0.8596295118331909, -1.1603081226348877, -1.1806703805923462, 1.2850185632705688, -0.49346768856048584, -0.6518743634223938, 0.3153076469898224, 2.1192095279693604, 0.5882251858711243, 1.0596269369125366, -1.4963927268981934, -0.39085882902145386, 0.0061503127217292786, 0.14740076661109924, -2.3098080158233643, 1.168368935585022, 0.880259096622467, 1.9360404014587402, 0.10759555548429489, -1.1225311756134033, 1.1157691478729248, -1.1593893766403198, 0.08287830650806427, -0.9168362617492676, -0.8041950464248657, 0.3278367221355438, -0.279047429561615, -1.2957758903503418, -0.13937142491340637, -0.5616634488105774, -1.5045245885849, -1.2223833799362183, -0.30063822865486145, 0.09751330316066742, -0.24126316606998444, 0.7777209281921387, 0.7459421753883362, 0.5984804630279541, 0.15358708798885345, 1.3748892545700073, 1.2271134853363037, 1.270447850227356, 0.44349607825279236, 0.4710943102836609, -0.9602217674255371, -0.5360840559005737, 2.136220693588257, -0.08248436450958252, -0.19024567306041718, 0.6707772016525269, 0.28310391306877136, -1.5448256731033325, 0.8920562267303467, 0.3179793357849121, 0.8283109068870544, -0.14396759867668152, -0.8963225483894348, 0.6272872090339661, -0.29402369260787964, -0.16336503624916077, -0.6465712785720825, 0.9864774346351624, 0.6786867380142212, -0.5033509731292725, -0.4855493903160095, 0.7557814121246338, 0.8901591300964355, 0.8107510209083557, -0.583143949508667, -0.4524424076080322, -0.9031206369400024, 0.22145842015743256, 1.1609987020492554, 1.0674562454223633, -0.9477256536483765, -0.930880069732666, 2.2008297443389893, -0.8082956671714783, 0.16014479100704193, -0.4731091558933258, 1.516588807106018], [-0.7130748629570007, 0.8042666912078857, 0.5629559755325317, 0.7023048996925354, -0.4674825072288513, -1.7189449071884155, 0.28808486461639404, 0.15852877497673035, -0.9692413806915283, -0.36141830682754517, 1.1432349681854248, -1.3781459331512451, -0.2705112397670746, 0.5744125843048096, -1.986717939376831, 0.8997487425804138, 0.22469736635684967, -0.4795420467853546, 0.18487799167633057, 1.4340863227844238, 1.0277620553970337, -0.31319913268089294, -1.9101327657699585, -0.32263705134391785, -0.5871832966804504, -0.8055235147476196, 1.1465444564819336, -1.9661555290222168, 0.4194434583187103, -1.3281913995742798, 0.9876271486282349, -0.2416035532951355, 0.7137491703033447, 0.05956336110830307, 0.003483841195702553, -0.5704295039176941, 1.0644174814224243, -0.8678352236747742, 1.353991985321045, 0.15133975446224213, 0.30078887939453125, -1.3108303546905518, -0.9232642650604248, -0.034920867532491684, 1.0434753894805908, 1.3927879333496094, 0.21525916457176208, 0.5576146841049194, 1.951493740081787, 1.7190301418304443, -1.1255879402160645, 0.8431720733642578, -0.9883447289466858, -2.18815016746521, -2.226696014404297, -0.02560495212674141, -1.0547475814819336, 0.48759374022483826, -0.12504424154758453, 0.7049807906150818, -0.4883033335208893, -1.054824948310852, 0.10585435479879379, 1.0232093334197998, 0.07777758687734604, -0.8488919138908386, -0.5448299050331116, 1.0287611484527588, -1.208715796470642, 1.4995149374008179, -0.13381758332252502, -0.17796333134174347, 1.5307916402816772, -0.352322518825531, 1.3611582517623901, 0.20435155928134918, 0.2612735331058502, 0.1581430733203888, 0.5093845725059509, 1.5707032680511475, 0.8137957453727722, -1.188265323638916, -1.34067964553833, -0.7719027400016785, 1.816415786743164, -0.38547074794769287, 0.5830371379852295, -1.2679797410964966, -0.4607045650482178, -0.3939022719860077, 1.1820417642593384, 1.5629656314849854, 1.8494970798492432, 1.1620864868164062, 0.47105470299720764, 0.7503960728645325, 0.4688560664653778, -1.3192390203475952, -2.103883743286133, -0.1344333440065384, -0.28673481941223145, 1.8477195501327515, 1.3855268955230713, -2.308724880218506, -1.1056334972381592, -2.0886592864990234, -0.29711756110191345, 1.0403761863708496, -1.4491312503814697, 0.8801254034042358, -1.1525176763534546, -1.1036622524261475, -0.6601520776748657, -2.929809331893921, -0.3277112543582916, 0.5464692711830139, 1.3558149337768555, -0.2565363347530365, 0.3979693353176117, 0.8678461909294128, 1.128502607345581, -0.06134895980358124, 2.2011260986328125, -1.5388853549957275, -0.09844491630792618, -0.22388099133968353, -0.6121544241905212, 1.130094289779663, -0.2244148701429367], [-0.9237096905708313, -0.6612334251403809, 1.0130034685134888, 0.6424210667610168, -0.37778738141059875, -0.2516235411167145, 0.8348570466041565, -0.6030938029289246, 0.5401809811592102, 1.3957864046096802, -1.9759758710861206, -1.1084843873977661, 0.8483966588973999, -0.2278113216161728, 0.3806072473526001, 0.18004076182842255, -0.4816216230392456, 0.6303744316101074, -0.3828183710575104, -1.1224952936172485, -1.099951148033142, 0.15414835512638092, 0.3349960744380951, 0.31984180212020874, -0.9076077342033386, 0.8190905451774597, -0.5706100463867188, 0.8276326060295105, 0.7720265984535217, 0.9929504990577698, 1.5275174379348755, -1.5427592992782593, -0.9884625673294067, 0.4578753411769867, 1.6997957229614258, 0.8814504146575928, -0.13522736728191376, 0.045254185795784, 0.08873052150011063, -0.2937432527542114, 0.2035689651966095, 0.24061882495880127, -0.48993003368377686, 1.7188329696655273, 0.6485887169837952, 0.7888332009315491, -0.3467286229133606, 0.2734330892562866, -1.3610221147537231, -0.1374579519033432, -0.784805417060852, -0.2552714943885803, 0.4661387503147125, 0.987980306148529, 0.5788918137550354, -1.128818154335022, -0.10019354522228241, -0.39254260063171387, -0.09531945735216141, -0.0654015988111496, -2.5921688079833984, -0.25856536626815796, -0.5127455592155457, 0.22957967221736908, 0.47387272119522095, 1.2135610580444336, -1.4792590141296387, 1.4363569021224976, 0.7506622076034546, 1.3928627967834473, 0.06049215421080589, 0.7451928853988647, 0.5871548056602478, -0.1112971231341362, -0.7290078997612, 0.7987961769104004, 0.8431118726730347, 1.6546708345413208, 0.2316778004169464, 1.359245777130127, 0.12982046604156494, -1.3349028825759888, 0.6092426180839539, -1.4301321506500244, -0.17520996928215027, 1.0657382011413574, -0.17739734053611755, -0.2879118025302887, -1.5954852104187012, 1.0838415622711182, 0.5326504707336426, -1.9213905334472656, -0.3346301317214966, -0.07721967995166779, 0.29416805505752563, 1.2212902307510376, 0.9506024718284607, 0.09954485297203064, 2.1140339374542236, -0.08477070927619934, 0.20139449834823608, 0.20265427231788635, 0.9842507839202881, 1.1485038995742798, 0.40942153334617615, 2.160372495651245, -0.04500633850693703, -1.1113717555999756, -0.6900268793106079, 0.939242422580719, 2.3562750816345215, 1.2466894388198853, -0.28994619846343994, -0.6624544262886047, 1.3787800073623657, 1.1901227235794067, 0.19061364233493805, 1.3251826763153076, -1.4188591241836548, 0.3812274634838104, 0.2101995199918747, 0.8293268084526062, -0.3692548871040344, 0.3789574205875397, 0.08481016755104065, -0.6589205265045166, 0.17907124757766724, 1.7360588312149048, 0.969076931476593], [-0.6950851678848267, -0.5259456634521484, -1.4781831502914429, 0.14814014732837677, -2.154578924179077, -3.9982238376978785e-05, 0.4215197265148163, -0.7031887173652649, 0.8261222839355469, 0.10409235209226608, 1.2894055843353271, -1.0999913215637207, -2.0180702209472656, 0.12183742225170135, 0.22437025606632233, 0.8113405108451843, -0.3568171262741089, -1.1801376342773438, 0.05618605762720108, 1.4762500524520874, 2.0249791145324707, 1.455264687538147, 0.1695708930492401, -0.4528689682483673, 0.0383571982383728, -0.8755290508270264, -1.2794909477233887, -1.463957667350769, -0.3118517994880676, -1.0156316757202148, 0.24867622554302216, -0.8058414459228516, -1.1197224855422974, -1.9192450046539307, -0.9308123588562012, 0.5177412033081055, 1.3089348077774048, -0.19572660326957703, -0.3908902108669281, -1.764158010482788, 1.1009281873703003, 0.27949318289756775, -1.09702730178833, -0.016947194933891296, -0.2992247939109802, 0.5151562690734863, 0.41265133023262024, 0.6532880663871765, 1.3258256912231445, -0.6367472410202026, 1.3426681756973267, 0.552992582321167, -2.4123003482818604, -2.5988316535949707, -0.39110711216926575, 0.23034557700157166, -0.6095800995826721, -1.6552903652191162, 0.16699913144111633, 1.7751635313034058, -0.041581347584724426, 0.2827047109603882, -0.994118332862854, -0.05082155019044876, 0.9269047379493713, -0.2301573008298874, -0.3654709458351135, 1.2134703397750854, 0.8067225813865662, -0.31667396426200867, -1.1574877500534058, -0.11802281439304352, -1.833918571472168, -0.6087818741798401, 1.1096152067184448, 0.4700826406478882, -0.9077860713005066, -1.241500973701477, 0.4152568578720093, 1.194865345954895, -1.3261767625808716, 0.2640922963619232, 0.8637785911560059, 2.661975383758545, 0.5888833403587341, 0.38985732197761536, -1.152085542678833, 0.9164484739303589, -0.03034345805644989, -0.12769463658332825, -0.26232215762138367, -0.2023046612739563, -1.0743683576583862, 1.000378966331482, 0.4303191304206848, 0.8191303610801697, -1.0032167434692383, -0.9654257893562317, 0.4232548475265503, 1.2127240896224976, -0.3561660647392273, -0.8530420064926147, 1.2820234298706055, -1.510948657989502, -0.22532597184181213, 2.7302136421203613, -0.39012160897254944, -0.4014109671115875, 0.21135765314102173, 0.33392760157585144, 0.6346338391304016, -0.03912067785859108, -0.9649900794029236, 0.2861056327819824, 0.7537248730659485, -0.909432590007782, -0.6601461172103882, -0.7373361587524414, 0.05642211437225342, 0.684661328792572, -0.7381644248962402, -0.1775205135345459, 0.6122221350669861, -0.5552263259887695, -0.5764349102973938, -0.24435117840766907, 0.43685245513916016, -0.6270401477813721, -0.5969552993774414], [-0.07348249852657318, 0.23209501802921295, 0.8033743500709534, 2.3268606662750244, 0.04839673638343811, 0.47964349389076233, -0.010687526315450668, 1.0851614475250244, -0.16075414419174194, 0.25792786478996277, 0.02266443334519863, -0.4651736617088318, 0.06104804947972298, 0.16708172857761383, -1.6892045736312866, -0.7108321785926819, -0.5843530893325806, 0.9367321133613586, 0.5579400658607483, 1.523151159286499, 0.7622029781341553, 1.013185977935791, -1.2851618528366089, 0.07478569447994232, -1.04346764087677, -1.3039882183074951, -1.047316074371338, -1.1420013904571533, -0.5923140645027161, 0.7484226226806641, 0.7046452164649963, -3.3334593772888184, 0.9825025796890259, 0.6974020600318909, 0.813154399394989, -0.5239418745040894, -1.107452630996704, 1.2780373096466064, -0.24601732194423676, 0.5807015895843506, -0.3884967863559723, -0.4478696882724762, -0.21628284454345703, 0.38150858879089355, 1.5341418981552124, -1.0450843572616577, -0.6405475735664368, -0.4794524610042572, -0.9729302525520325, -0.14943605661392212, 1.282366156578064, 0.25136512517929077, -1.3190127611160278, 2.5683393478393555, 0.17081446945667267, -1.1950976848602295, 1.6567859649658203, -0.6060745716094971, 0.5178601741790771, 1.917377233505249, 1.1520886421203613, 0.8880141973495483, 0.12086760997772217, -0.042140502482652664, 0.3254331648349762, 1.0412753820419312, -0.6158915758132935, -1.500802755355835, -0.425392210483551, 0.48351454734802246, 0.5506682395935059, 1.5006098747253418, -1.441687822341919, -0.8867942690849304, -0.5439961552619934, 0.4137711226940155, -0.1463390588760376, -1.1395996809005737, -0.8290776014328003, 0.12638264894485474, 1.9829920530319214, 0.5436459183692932, 0.854788601398468, -0.08074139058589935, 1.1179111003875732, 0.07239208370447159, -1.3606905937194824, -0.5466980934143066, -0.8266592025756836, 0.8321982026100159, 0.0796269029378891, -0.22433282434940338, 1.3786799907684326, -1.7711337804794312, 1.5907831192016602, 0.6144980788230896, -1.1362472772598267, -1.6605806350708008, 0.3180968463420868, 0.09855969995260239, 0.7306694388389587, -0.8513286113739014, -0.49014797806739807, -0.5349704623222351, -0.016405262053012848, 1.8060520887374878, -0.8570418357849121, 0.056789834052324295, -0.879352331161499, 0.21272875368595123, 1.0049854516983032, -0.36380332708358765, 0.3602820932865143, 0.8736848831176758, 0.5884914398193359, -0.7804804444313049, -0.4646591246128082, -0.6526919007301331, 0.6598503589630127, -2.1371471881866455, 1.1135042905807495, -0.24842365086078644, 2.5446126461029053, -0.9594572186470032, 1.3330409526824951, -0.9464399218559265, 1.5363800525665283, 0.2620367705821991, 1.154252052307129], [1.128495454788208, 0.5706565380096436, -0.7801297903060913, -0.92637699842453, -0.7498297095298767, 0.06543096154928207, -0.25564366579055786, -1.2200517654418945, -0.45122864842414856, 3.171022415161133, -0.889926016330719, -1.0634512901306152, -0.6241495609283447, -1.426766276359558, -1.504544734954834, 0.10816217958927155, -0.4248770475387573, -0.9593272805213928, -0.23320624232292175, -0.7997189164161682, 0.5569157600402832, -0.13734497129917145, -0.14139041304588318, 0.6288456320762634, -1.0830694437026978, -0.5973303318023682, 2.452745199203491, -1.3400652408599854, 1.2277275323867798, 0.12327531725168228, 0.6431072354316711, 0.3712003231048584, 0.1072373017668724, -1.1278709173202515, 0.9201586246490479, -0.30745112895965576, -0.3950086534023285, -1.3931869268417358, 1.0552188158035278, -0.3843194246292114, -1.2799270153045654, -0.21279892325401306, 0.6701550483703613, 0.5800440907478333, -2.1125359535217285, 1.5212652683258057, 1.0012257099151611, 1.0412020683288574, -0.011149981990456581, -0.1945481300354004, -1.3217072486877441, -2.1765410900115967, -1.9068593978881836, 0.1896580159664154, 0.24388670921325684, 0.8542744517326355, -0.5911004543304443, -0.279016375541687, 1.1797090768814087, -2.533264636993408, -0.7535187005996704, -0.9681508541107178, -1.6641621589660645, -0.9963297247886658, -0.4483815133571625, -0.49694597721099854, -1.8630861043930054, -1.4308419227600098, -0.028155509382486343, -1.1241178512573242, 0.10628574341535568, -2.2301485538482666, 1.4146453142166138, 0.5033729672431946, 0.4659711718559265, -1.0942981243133545, -0.29200953245162964, 1.4717463254928589, -0.416786789894104, -0.38216516375541687, -0.625907301902771, 0.5752066373825073, 1.3078776597976685, 0.2604271173477173, -0.7329400181770325, 0.28869637846946716, 1.7989134788513184, -1.322568416595459, 1.5268384218215942, 0.10554095357656479, 0.9255445003509521, 0.7229470014572144, 1.1099317073822021, -0.0339837484061718, 1.6087085008621216, 1.370634913444519, 0.8054719567298889, -1.5269216299057007, 0.4375436007976532, -0.8635625839233398, -0.19585078954696655, 0.31209591031074524, -1.883863925933838, 1.639319658279419, 0.867883026599884, 0.7213341593742371, -0.27593356370925903, 0.1894826740026474, 1.8626667261123657, 0.20340904593467712, -1.7667617797851562, -1.6332883834838867, 0.1419382244348526, -1.3509254455566406, -0.39932748675346375, 0.3258580267429352, -0.06627258658409119, 0.2440890669822693, 0.4832600951194763, 0.26564422249794006, -1.102189540863037, 1.3717851638793945, 0.2875627279281616, -0.0012802394339814782, 1.2173864841461182, -0.13589735329151154, -0.6983069777488708, 0.3419676721096039, -1.224408745765686], [1.6292426586151123, 1.1863497495651245, -0.07275933027267456, -0.1213853508234024, -1.7778586149215698, 0.6963058114051819, 1.5678470134735107, -0.06312919408082962, 2.729072093963623, 2.9633939266204834, -0.15314249694347382, 1.2620766162872314, 1.4751142263412476, -0.3338654935359955, 0.8812016844749451, 0.3958837687969208, -0.300204336643219, 0.3432546854019165, 0.43316349387168884, -0.2993091344833374, 0.8080083727836609, -1.0276893377304077, -0.0865890309214592, -0.2894388437271118, 0.1092282384634018, -0.6398850083351135, 1.2367621660232544, 0.371599406003952, -0.542057991027832, -3.284454822540283, 0.05353491008281708, -0.7006506323814392, -0.9546424746513367, -0.14591947197914124, 0.291502445936203, 0.16004882752895355, 1.6182068586349487, 0.9921033978462219, 1.128420114517212, 0.49704471230506897, -0.4218906760215759, 0.1531505584716797, 1.05873703956604, 0.2629048824310303, 1.4216445684432983, 0.6505737900733948, -0.84328293800354, 1.6701728105545044, -0.9407266974449158, 0.43707504868507385, 0.19559156894683838, 1.5102438926696777, -0.7384624481201172, -0.03633785620331764, 2.0248782634735107, 1.1996136903762817, 0.8159898519515991, -0.9422184824943542, 0.3597460389137268, -0.40174299478530884, 0.543291449546814, 1.4579936265945435, 0.9138092398643494, -0.29794999957084656, 0.15045510232448578, 2.838768720626831, -1.8343861103057861, 0.9992955923080444, 0.6071851253509521, -0.9015598893165588, 1.8750483989715576, -0.45251962542533875, -0.11046701669692993, 1.6133334636688232, 1.3925589323043823, 0.2713887691497803, 0.5370676517486572, -2.043325901031494, 0.650087296962738, -1.5854270458221436, 0.45620158314704895, 0.42860502004623413, 0.23145632445812225, 0.4807797372341156, 1.1601989269256592, 0.39354056119918823, -1.1752220392227173, -1.6582262516021729, -0.22083687782287598, 1.0737804174423218, 1.2284165620803833, 0.1818322092294693, 0.27587535977363586, 1.8044090270996094, -0.4184759259223938, 1.4866799116134644, 0.5335529446601868, -1.3162801265716553, -0.8418068885803223, 0.8664396405220032, 0.7893922328948975, -1.7258105278015137, 0.4955301582813263, 1.5138211250305176, 1.4378135204315186, 1.0682123899459839, -0.435087114572525, -0.8234656453132629, 1.3498055934906006, 2.301344633102417, -0.1066538468003273, -1.7302186489105225, -0.7060843706130981, -0.12401247769594193, 0.06223384290933609, 0.9040377140045166, 1.368506908416748, 0.07547055929899216, 0.8084791898727417, 0.8759086728096008, -0.17380738258361816, 1.5863878726959229, 0.5905448794364929, 1.0324972867965698, -0.6418552994728088, 0.8403551578521729, 0.19780926406383514, -0.969352126121521, 0.5601333379745483], [0.8620921969413757, 1.845396876335144, 0.02096381224691868, 0.06613653898239136, -0.868121325969696, 1.319352626800537, 0.16008469462394714, -0.5316332578659058, -0.36182132363319397, -0.1967848688364029, 0.3270508646965027, 1.2504925727844238, 1.6217334270477295, -0.525624692440033, -0.7808817028999329, -0.5636646151542664, 1.180812954902649, 1.449508786201477, -1.7508575916290283, 1.0402721166610718, 0.19013169407844543, 1.7505098581314087, 0.764631986618042, -0.683212399482727, -2.4598934650421143, -2.38751220703125, -1.3616503477096558, -0.6372201442718506, 0.37142324447631836, -0.46607673168182373, -1.4847652912139893, -0.6440221667289734, -0.4294356405735016, -0.1903567761182785, 0.15972764790058136, -1.429998517036438, 1.0950210094451904, -0.792147159576416, 0.34416407346725464, 0.9461296796798706, 2.3196513652801514, 1.1457792520523071, -0.8086794018745422, -1.0075079202651978, -0.08698921650648117, -2.6894829273223877, 0.9887017607688904, 0.8643349409103394, -0.24911941587924957, 1.5581567287445068, -0.4725114107131958, -1.508165955543518, -0.790422260761261, -0.10401647537946701, 0.27522575855255127, 0.6492378115653992, 1.1502989530563354, 0.5998895764350891, 1.128949522972107, 0.9122388958930969, -0.8182283043861389, 0.602496862411499, -0.8089860081672668, 0.1266598105430603, -0.7243276238441467, -0.746905505657196, 0.6495808959007263, 1.7317609786987305, -1.84099543094635, 0.7506780624389648, -1.2811013460159302, 1.2650820016860962, -1.0769394636154175, 1.3109036684036255, 1.1090420484542847, 1.5061299800872803, -0.24498622119426727, 2.959564208984375, -0.9963585734367371, -1.2725532054901123, -0.8915334343910217, 1.2974165678024292, -0.35934507846832275, -0.2673477232456207, 0.7627050876617432, -0.02731378562748432, 0.7895861864089966, 0.6668977737426758, 0.26203805208206177, -1.5967943668365479, 0.10512931644916534, 0.2757863700389862, 0.48039934039115906, 0.7321460843086243, 0.44298097491264343, -2.0053763389587402, 0.15844310820102692, -0.9255543947219849, 0.7417294979095459, -0.674312949180603, -0.02095046453177929, 0.976081907749176, -0.7148285508155823, -0.4532288610935211, -1.7522605657577515, -0.8647482395172119, -1.2632683515548706, -1.0447900295257568, -0.6030564904212952, -0.4017895460128784, 0.046891141682863235, 0.945070207118988, -0.1889394372701645, 0.8226713538169861, -1.0188943147659302, 1.1192783117294312, -2.087008237838745, -0.0935714915394783, -0.42589685320854187, 0.8518770933151245, -1.0391051769256592, 1.354436993598938, -0.6958050727844238, 0.02048688568174839, -0.7498694062232971, -1.1774072647094727, 0.6083211302757263, -0.09308049082756042, 2.5653133392333984], [-0.13471387326717377, 3.1052603721618652, -0.9320502877235413, -0.5340566039085388, -1.9375483989715576, -0.8475695252418518, -1.19623863697052, 0.5715528726577759, -0.27854016423225403, -2.0267174243927, -1.0746872425079346, 0.47779184579849243, 1.649667501449585, -0.3573162853717804, -0.23302289843559265, -1.4051200151443481, -0.521921694278717, 0.86119145154953, -0.2502933144569397, -0.2650129497051239, -0.14327207207679749, -2.2497239112854004, -0.5491355061531067, -1.11988365650177, -0.23391607403755188, -1.4879635572433472, -0.9408263564109802, 1.4869011640548706, 0.9156480431556702, 1.352798342704773, 0.33119997382164, 0.4148455262184143, -0.17132030427455902, -1.1982340812683105, 1.6800302267074585, 1.2878390550613403, -0.15626055002212524, 0.7445055842399597, -0.29791897535324097, -0.28901493549346924, -1.2405338287353516, 0.03880704566836357, -1.157086968421936, 0.5204305052757263, -0.09946674108505249, 0.7844212651252747, 0.5297186970710754, 0.7304678559303284, 0.44241002202033997, -0.029527541249990463, -1.0114784240722656, 0.1914610117673874, -0.33585453033447266, 1.3181122541427612, 0.3567711412906647, 0.36733192205429077, -0.9433515071868896, 1.2652897834777832, -0.6770933270454407, 0.8688731789588928, -1.2978756427764893, -0.5215722322463989, 0.5547640323638916, 1.907407522201538, 1.2776830196380615, 0.7494766712188721, -0.9246189594268799, -0.785764217376709, -0.12155355513095856, -0.8219185471534729, -1.0311143398284912, -0.8101656436920166, -2.600780963897705, -0.2122170478105545, -1.0133278369903564, 0.3313552439212799, 0.48230037093162537, -1.2837265729904175, -0.18682055175304413, 0.9320133924484253, 0.5198205709457397, 1.1735734939575195, -0.07341544330120087, 0.3897348642349243, -0.11900878697633743, 0.14979611337184906, 0.04256761074066162, 0.19044704735279083, -0.7965036034584045, -0.7094820141792297, -0.198092982172966, -0.7162916660308838, -0.21227172017097473, -1.2250781059265137, -1.1742079257965088, 0.04386306181550026, 0.81339031457901, -2.9261884689331055, 0.2649073004722595, -0.7528672814369202, -2.576977252960205, -1.488013744354248, 0.1551988124847412, 0.5256516337394714, 0.02187853492796421, 0.15665477514266968, -0.9064615964889526, -0.481282114982605, -0.8120900988578796, -0.6417906880378723, -0.9762998223304749, 0.9048706889152527, -1.1180591583251953, 0.8006857633590698, 0.9323644042015076, 2.71280837059021, 1.2693564891815186, 0.3402509093284607, 0.5109425187110901, 0.08119550347328186, 1.1951733827590942, -0.20316068828105927, 1.1077258586883545, -1.3978064060211182, -0.05813862755894661, 0.7730449438095093, -1.4715286493301392, 0.9377687573432922, 0.6514263153076172], [0.25699830055236816, 0.23488876223564148, -1.0936390161514282, -1.8811064958572388, -0.42055776715278625, -0.3973628878593445, -0.18247748911380768, 0.13764823973178864, 1.2341487407684326, -0.841546356678009, -1.2971529960632324, -2.127108335494995, 0.47815263271331787, 0.018053516745567322, -0.029291771352291107, -0.5858962535858154, 0.3170892596244812, -0.4152098298072815, 0.8976376056671143, 0.21735583245754242, 0.021094413474202156, -0.6042321920394897, -0.39277443289756775, 0.0365944504737854, 2.4417271614074707, -0.3986082971096039, 1.0205141305923462, -2.1801862716674805, -0.34610238671302795, -0.5645467042922974, 1.5702965259552002, 1.243586540222168, -0.6076926589012146, -0.9220029711723328, 1.058211326599121, 0.3627243936061859, -0.7556823492050171, -1.4063838720321655, -0.07873788475990295, -0.29844963550567627, 0.1700076162815094, -0.18194231390953064, -1.3272508382797241, 1.4058905839920044, 1.3809096813201904, 0.47481027245521545, 1.0544934272766113, -2.7828705310821533, 0.15830373764038086, -2.1125383377075195, -0.09239199757575989, 0.34225550293922424, -0.10905718803405762, -0.9242133498191833, 1.1180020570755005, -1.1171445846557617, -1.4698917865753174, -0.40392404794692993, -1.0585956573486328, -1.8457562923431396, 1.0045766830444336, -0.7367802858352661, -0.0846225693821907, 0.17665772140026093, 0.7503447532653809, -1.3252023458480835, -0.3743496537208557, -1.24569833278656, 1.612746238708496, -0.33385175466537476, -0.582166850566864, -0.5509014129638672, -0.08493493497371674, -0.5331461429595947, -0.32115647196769714, 0.37114280462265015, -0.3446204662322998, 0.07616278529167175, 0.9135767221450806, -0.8557145595550537, -0.8994538187980652, 0.37374967336654663, 2.102881908416748, 1.548092007637024, -0.5919224619865417, -1.0558300018310547, 0.6129322648048401, -0.8573818802833557, 0.1585158407688141, 1.2436223030090332, -0.8447571992874146, -1.2324670553207397, 0.6597755551338196, 0.7243028283119202, 0.10533119738101959, 1.2088685035705566, -1.2899531126022339, 0.6052082777023315, -0.024909939616918564, -1.1319928169250488, 1.4686005115509033, -1.0816730260849, -0.47257137298583984, -0.2574792504310608, 0.432330459356308, 1.9372411966323853, 2.3131797313690186, 0.7866550087928772, -0.15462929010391235, -1.4565818309783936, 0.5353497266769409, 0.697373628616333, -0.712705135345459, 0.23824141919612885, 0.41786909103393555, -1.5327309370040894, -0.007947469130158424, -0.3271055519580841, 2.3708770275115967, 0.5581466555595398, 0.2417447715997696, 0.4804055690765381, -0.10770435631275177, -0.3363596796989441, -1.0734672546386719, -0.7655932903289795, -2.6189591884613037, 0.6303543448448181, 1.1563637256622314], [0.8576611280441284, 0.014605565927922726, -0.08476512134075165, -0.9173275232315063, -0.011268348433077335, 0.446056991815567, 0.28895631432533264, -0.3601560890674591, -0.9570170044898987, -1.1175358295440674, 0.2923327088356018, -0.3260377049446106, -0.6222869157791138, -1.5869840383529663, -0.3619510531425476, -0.666324257850647, -0.7151808142662048, -0.7124887704849243, 0.24298764765262604, 0.03206472471356392, -0.6338837742805481, -1.218555212020874, -0.9906500577926636, -0.09705904126167297, 0.5575168132781982, 1.237356424331665, 0.8944684267044067, -0.2603069543838501, 0.782136857509613, 1.133139967918396, -1.0333322286605835, -1.6686577796936035, 1.2036429643630981, -0.5495538115501404, 0.8189043402671814, 1.1504205465316772, -0.2150418609380722, -1.3810099363327026, 3.1035678386688232, -0.5484414100646973, 0.2626040577888489, -0.4734361469745636, 0.7680450677871704, -0.6275657415390015, 0.49743926525115967, 0.610053300857544, 0.3803490996360779, -1.1641120910644531, -0.3969089090824127, -1.1883342266082764, -2.3497583866119385, -1.611266016960144, 0.8690343499183655, -0.8312504291534424, -0.008238195441663265, -1.1644326448440552, 0.32457435131073, -0.35946303606033325, -0.33421963453292847, -0.5446890592575073, 1.9743096828460693, 1.8949824571609497, 0.2084425836801529, -0.5243620872497559, 0.9283992648124695, 0.8582333326339722, 1.1066606044769287, -0.03815734386444092, -1.0312626361846924, 0.8992280960083008, -1.2995946407318115, -0.5700811743736267, 0.9624509811401367, 0.3902983069419861, 0.5220616459846497, 1.1645534038543701, 1.3722896575927734, 0.5181922316551208, 1.159706473350525, -0.693131148815155, -0.30979278683662415, -1.1492844820022583, -1.0720012187957764, 1.3576459884643555, 0.6849204301834106, -0.27169787883758545, 0.7683672308921814, -0.7615012526512146, 0.8612222671508789, -0.6988521814346313, -1.170371413230896, -1.048162579536438, -1.3371591567993164, -0.03471541777253151, -0.41202476620674133, 1.3539162874221802, 1.1944196224212646, -0.23920458555221558, -0.6014357805252075, -1.121772050857544, 1.0630937814712524, -0.8328173160552979, -0.19612641632556915, -0.8079542517662048, 1.0272566080093384, 0.9019403457641602, 0.3667338490486145, 0.3060241639614105, 0.21735836565494537, 0.9111028909683228, 1.1846474409103394, 0.9810697436332703, -1.705763339996338, -0.8744531869888306, 1.40240478515625, 0.9003519415855408, 1.1654542684555054, 0.8853369951248169, 0.044073596596717834, -1.244317889213562, 1.1095104217529297, -1.8454147577285767, 1.4122998714447021, -0.8242314457893372, 0.14315563440322876, -2.196331739425659, -0.5533048510551453, -0.6730343699455261, 1.023525595664978], [-0.8807294964790344, 0.2583465874195099, -0.38261479139328003, -0.13158690929412842, 0.4527953267097473, -0.247569739818573, -1.094425082206726, -1.6637170314788818, 1.7497588396072388, -0.8124097585678101, -0.10936258733272552, -0.34074363112449646, -0.522997260093689, -2.148202896118164, -0.18649129569530487, 0.8458476066589355, 1.6102501153945923, -0.15929287672042847, 1.7649426460266113, 0.5773364901542664, 1.9919003248214722, -0.0760495662689209, -0.7423555850982666, -0.4105125367641449, 0.007315150462090969, 0.14122232794761658, 0.701516330242157, -0.2930464744567871, 0.677449643611908, -0.6484549641609192, -0.2827964425086975, 0.7001593112945557, -2.2879302501678467, -0.4875858426094055, -0.4398009777069092, -0.8090290427207947, 0.7277413606643677, 1.7100309133529663, -1.8441492319107056, -0.4766541123390198, 2.1596312522888184, -0.07851243019104004, 0.20049744844436646, 1.4466657638549805, -0.3540946841239929, 1.1881438493728638, -0.14403584599494934, 0.1323971450328827, -0.19280041754245758, 2.270158052444458, -0.7782729268074036, 0.3202517032623291, -0.2958093285560608, 1.8837835788726807, -0.8548331260681152, -0.09856649488210678, -0.5467389822006226, -0.4585224390029907, -0.7600008845329285, -0.6830580234527588, -0.04749453812837601, 0.16718073189258575, -0.3345660865306854, 0.18453139066696167, -0.08870519697666168, 0.6924297213554382, 0.4912523627281189, 0.6046538352966309, 1.2301970720291138, -0.5918862819671631, -0.07288683950901031, 0.6227616667747498, -0.9449819326400757, 0.07940391451120377, 0.6430226564407349, -2.8854422569274902, 1.1304818391799927, -0.7740575671195984, 0.23311640322208405, 0.568731427192688, 0.7465773820877075, -1.343741774559021, 0.4162284731864929, -1.2072242498397827, -1.4926068782806396, -1.4304752349853516, -1.840946912765503, 0.19699421525001526, 0.26053088903427124, -0.1821308135986328, 0.8578755855560303, 1.7316854000091553, -0.8695284724235535, 1.7398569583892822, -0.6382884979248047, -0.7122294902801514, 0.3812461197376251, 1.281409740447998, -0.2292899191379547, 0.1253635287284851, 2.082496404647827, 0.8005775809288025, 0.1995851993560791, -0.8001318573951721, -1.405491590499878, -0.46117138862609863, -1.0411808490753174, 1.6961406469345093, -0.49590298533439636, -0.072604238986969, 1.5652897357940674, 0.490829199552536, 0.7779805064201355, -0.6223993897438049, 1.0848519802093506, -0.4024832248687744, -2.403520345687866, -1.430422306060791, 0.08887051790952682, -0.8136553764343262, -0.873374342918396, 0.7483708262443542, 0.08734521269798279, 0.05901281163096428, 0.4933658838272095, -1.4877269268035889, 0.20183178782463074, 0.1390455961227417, 0.2638014554977417], [0.8910996913909912, -0.4662635624408722, -1.4957619905471802, 0.39946648478507996, -0.47391194105148315, -0.6628214716911316, 1.009200096130371, -0.7162541151046753, -0.24517785012722015, 0.9888204336166382, -1.6733293533325195, 0.8993068933486938, 0.2622026205062866, 0.4857213497161865, 1.1865193843841553, -0.7420729994773865, -0.6314541101455688, -0.6807661056518555, 0.511546790599823, -0.9576524496078491, -0.6385275721549988, 0.6593495607376099, -2.2464962005615234, -2.211181163787842, 0.6467936038970947, 0.10264509916305542, -0.12077774852514267, 0.4380599558353424, 0.965269148349762, 0.5390127897262573, 0.0599457211792469, 1.037790060043335, 0.10333584249019623, -0.5448898673057556, -0.9387337565422058, 0.48282963037490845, -0.9019659757614136, -1.8965418338775635, 0.06923986226320267, -0.015433663502335548, 1.516252040863037, -1.651923656463623, -0.015450081788003445, 0.6249148845672607, -0.7433260679244995, 0.6293718218803406, 1.1179002523422241, 1.2288750410079956, 0.3033965826034546, 1.0263710021972656, 0.022417353466153145, 0.9401997327804565, 0.6312059164047241, -0.5404523611068726, 1.3650271892547607, -1.0167200565338135, 0.9868568778038025, -1.4533659219741821, -1.1878429651260376, -1.5661548376083374, -2.04896879196167, 0.6099304556846619, -0.37030208110809326, -0.5136938095092773, 2.1704063415527344, 0.5800874829292297, 0.8016207814216614, 1.984114646911621, -0.22371722757816315, -1.3347535133361816, 0.7743260264396667, 0.3136395812034607, 1.4225094318389893, -0.2657052278518677, 0.5984861850738525, 0.9015564918518066, 1.29252028465271, -0.6004464626312256, -0.03506045415997505, -0.8315197825431824, 0.7146776914596558, 0.3184763193130493, 1.1999735832214355, -0.017325423657894135, -0.03033273108303547, 0.5912054181098938, 0.1450735330581665, 0.666002631187439, -1.0590001344680786, 0.10401010513305664, -1.3882973194122314, -1.120970606803894, -0.3197769522666931, -1.6606014966964722, -1.3556573390960693, -1.1953238248825073, -1.1521799564361572, 1.3599095344543457, -0.8413224816322327, 0.40947070717811584, -1.4890793561935425, 2.2851333618164062, 1.1687380075454712, -0.4634215533733368, -0.8133190274238586, 0.38405290246009827, -1.8943403959274292, -0.8875321745872498, 0.15692128241062164, -0.9321100115776062, 0.40968236327171326, -0.6360268592834473, -0.13811536133289337, 0.6188097596168518, -1.8142428398132324, 0.8478901386260986, 0.8151818513870239, -0.06589720398187637, -1.0101677179336548, 1.2799532413482666, -0.05864999070763588, -0.3267017900943756, -0.36619457602500916, -0.8518833518028259, 0.3353012502193451, 0.2805192470550537, 1.0042471885681152, 2.1310813426971436, -0.2128666639328003], [0.4882635176181793, -2.5234591960906982, -0.9550268650054932, -0.04229382053017616, 0.9986363649368286, 0.6348395347595215, -0.8365850448608398, 0.5202306509017944, -1.2264280319213867, -0.511720597743988, -0.290349543094635, 0.8357152342796326, 1.674606442451477, -0.27012741565704346, 0.8288801908493042, -0.3065668046474457, 0.4410603642463684, -1.0193829536437988, -0.21527183055877686, -2.0194222927093506, 1.5096490383148193, 1.9595221281051636, -1.3684617280960083, -0.40835368633270264, -1.592360019683838, -0.49456122517585754, -0.8625523447990417, 0.13092094659805298, -0.8889329433441162, 2.605700731277466, 0.1053939163684845, -0.7007243633270264, 1.4376144409179688, -0.17405642569065094, -1.5945249795913696, 0.5423160791397095, 2.117668390274048, 0.750447154045105, -1.0470572710037231, -1.1612040996551514, -0.9271770715713501, -0.05771893635392189, 1.6506257057189941, 0.6064151525497437, -1.5888339281082153, -0.8362584114074707, -0.9446710348129272, 0.6887989640235901, 0.19205395877361298, -1.1085678339004517, -0.9695850610733032, 1.15230393409729, 0.7510984539985657, 0.4326910972595215, 1.0376988649368286, 1.4414920806884766, 1.0986108779907227, -1.055994987487793, -0.09543764591217041, -0.41728562116622925, -1.2795755863189697, -1.8041486740112305, -1.2376388311386108, 0.868000864982605, 0.061030689626932144, -0.8286911249160767, -1.377419352531433, 0.5130761861801147, 1.8627164363861084, 0.14232678711414337, 1.8180959224700928, 1.0368229150772095, 0.6249648332595825, 0.928945004940033, -0.48834967613220215, -0.26803115010261536, 0.24076338112354279, -0.556246280670166, -0.07145130634307861, -0.9449045062065125, 2.5103461742401123, 1.4106522798538208, -0.7799676656723022, 0.9495749473571777, -0.6509125828742981, -0.377097487449646, -0.5926190614700317, -1.2454253435134888, -1.850149393081665, -0.27331840991973877, 0.04633130505681038, -0.07972954213619232, -1.2235456705093384, -1.416533350944519, -0.7091886401176453, 0.7643150687217712, -0.394881010055542, -0.6404888033866882, -0.3498361110687256, -2.1770825386047363, 0.37640002369880676, -1.620090126991272, -0.6620094180107117, -0.8115564584732056, -0.21376535296440125, -0.6928312182426453, 0.5114571452140808, 0.8551399111747742, 1.1532624959945679, 1.5552873611450195, 0.5509639978408813, 0.5246170163154602, 0.39168208837509155, 1.1092535257339478, 1.1763184070587158, -0.7833226919174194, 0.21367542445659637, 1.7834510803222656, -1.4123730659484863, 0.45418235659599304, 0.49678975343704224, 0.133707195520401, 1.2992396354675293, 0.2983768880367279, 1.1204931735992432, -1.848496913909912, -1.5925633907318115, 0.9839953780174255, -0.24944517016410828], [-1.2355048656463623, -0.508886992931366, -1.058349370956421, -0.7336030602455139, -1.4392956495285034, -0.32316192984580994, 0.09907501935958862, -0.04993477463722229, 0.029524443671107292, -0.4796169698238373, 1.048833966255188, 0.3014066815376282, 0.3178335130214691, 1.3339571952819824, -0.07727212458848953, -0.004439878277480602, -0.1434544175863266, 0.7643104791641235, 1.4877996444702148, -0.6463485360145569, -1.7873367071151733, 1.7695478200912476, -0.7857427000999451, 0.8906004428863525, 0.2331652194261551, 0.36277642846107483, 0.2806571424007416, -1.1849472522735596, -0.3763657212257385, -0.3197104334831238, 1.0609034299850464, 0.6168116927146912, 1.3660660982131958, -0.2656400501728058, 0.2365618646144867, -0.23270593583583832, 1.1719671487808228, 2.155529022216797, 0.6006574034690857, 0.5108769536018372, -1.7268106937408447, 1.4429723024368286, -0.19371356070041656, -1.0105066299438477, 0.06426336616277695, -0.657233476638794, 0.4941701591014862, 0.17557302117347717, 0.004503628239035606, 2.8912770748138428, -0.4009039103984833, 1.101047158241272, 1.2851364612579346, 1.310716986656189, -0.41218918561935425, -1.5399785041809082, -2.5957272052764893, -0.504368245601654, 0.5930175185203552, -0.5875793099403381, 0.9569105505943298, 2.424692153930664, 0.8135508894920349, -0.012857750989496708, 1.4854384660720825, 1.0258913040161133, 1.355006217956543, 1.5393744707107544, -1.2454984188079834, 1.4496110677719116, -0.2741931080818176, -0.11836516112089157, 1.2424826622009277, 1.1875066757202148, 0.4065346419811249, -1.3122937679290771, 1.5559228658676147, 0.1745893657207489, 0.8180602788925171, -0.33469358086586, -0.23258180916309357, -0.22028350830078125, 0.9997540712356567, -1.3753795623779297, -0.054350513964891434, -0.57346510887146, -1.0435079336166382, 1.1707332134246826, 0.031104078516364098, -0.0930875614285469, 1.0037662982940674, 1.508671760559082, 0.5914170742034912, 2.3398802280426025, -0.08803890645503998, 0.18125906586647034, -0.21143190562725067, 0.542618989944458, -1.0665181875228882, -1.4450031518936157, 0.9700129628181458, 0.09582651406526566, -0.8126970529556274, 1.0815736055374146, 1.355224370956421, -1.1417491436004639, 0.7155461311340332, 1.1034440994262695, 0.6071893572807312, 1.5305588245391846, 3.215959310531616, 0.07344327867031097, -1.366801381111145, -0.8823007345199585, -0.48363831639289856, -2.193303108215332, -1.2836164236068726, -0.25769609212875366, 0.34452807903289795, -0.6690048575401306, 0.894167423248291, -0.9119674563407898, 0.6727211475372314, -0.5057582259178162, 0.416358083486557, 1.3440243005752563, 0.9717559218406677, -0.8452897667884827, -0.11893623322248459], [-0.010139894671738148, 1.4952093362808228, 0.578406035900116, 1.122067928314209, 0.385152667760849, -1.5308219194412231, -0.002440808340907097, -1.527081847190857, 2.7512547969818115, 0.14282843470573425, -1.1527042388916016, 0.3479746878147125, 0.31021344661712646, 0.32446062564849854, 0.3171827793121338, 0.9768832325935364, -1.6541560888290405, 1.1519863605499268, 0.37429967522621155, 0.8277053236961365, -0.6720991730690002, -0.4937228560447693, 0.12019363790750504, -0.19579531252384186, -1.795778751373291, 1.450315237045288, 1.5667084455490112, 0.2525028586387634, -0.5391145944595337, 1.2396297454833984, -0.5478067398071289, 0.9507506489753723, 2.040181875228882, 1.9452133178710938, -0.07763253152370453, -0.5727421045303345, -1.6908419132232666, 1.2129098176956177, 1.5710563659667969, 0.5453896522521973, -1.541002869606018, 2.087175130844116, 0.3918974697589874, 0.32021066546440125, -1.935154676437378, 1.038201928138733, 1.5306843519210815, -0.15274226665496826, 2.0267655849456787, 0.23098628222942352, -0.38976776599884033, 1.896462321281433, 0.581260621547699, -0.2410627156496048, 1.1617200374603271, -0.43539339303970337, 1.3485771417617798, -0.5225789546966553, 0.005802182946354151, -0.35203349590301514, -1.2860122919082642, 0.1641356647014618, -0.7675758600234985, -1.0561600923538208, 0.6412257552146912, -1.0788503885269165, 1.3187274932861328, 0.6726728081703186, -0.04896942153573036, -0.013621916063129902, -1.4599512815475464, -0.4408135414123535, 1.9644062519073486, 0.2036992609500885, -0.17343293130397797, 0.4337114095687866, -0.6882206797599792, -1.563712477684021, -0.40046727657318115, 0.08922674506902695, -0.34366315603256226, 0.6781996488571167, -1.1437121629714966, 1.06508207321167, 1.196420431137085, -0.38940826058387756, 0.6275305151939392, 0.9887714982032776, -0.36295631527900696, 0.6461260914802551, -1.4033457040786743, 0.7520384788513184, 0.364164799451828, -0.25274500250816345, 0.3427124619483948, -0.2931664288043976, 1.1639785766601562, -0.3077080249786377, 0.03702492639422417, 1.0497934818267822, 0.8164302706718445, 2.4374167919158936, -3.3248655796051025, 1.5611913204193115, -0.1321256160736084, 0.056666404008865356, 1.0226924419403076, 0.17202496528625488, 1.2286081314086914, -0.06443826854228973, 0.13392983376979828, -0.2611244022846222, -1.425933599472046, 0.4257926940917969, -0.6407708525657654, -0.9387255907058716, -1.7576782703399658, 0.07462006062269211, 0.08437814563512802, 1.8449947834014893, 0.9756403565406799, -1.1520066261291504, -0.1855962723493576, -0.1213751956820488, -1.634916067123413, 1.0412946939468384, 2.06008243560791, 0.3394547700881958, -0.038639962673187256], [-0.4163302183151245, -1.300883412361145, 0.9096865653991699, -2.1642978191375732, 0.1770200878381729, -1.2140324115753174, -0.4533780515193939, -1.170285701751709, 0.7586944103240967, 0.7768293619155884, -0.306295782327652, -0.871063232421875, -1.8353383541107178, 1.5272459983825684, 0.2788918614387512, 1.1497269868850708, 1.2806886434555054, 1.427941918373108, -0.30891096591949463, -0.642873227596283, 0.49376532435417175, 0.29746636748313904, 0.29355886578559875, 0.700437068939209, 1.1134216785430908, -0.6806609630584717, 0.3928002715110779, -0.7070856690406799, 1.1108342409133911, 0.5048828125, -2.192610740661621, -1.8141931295394897, -1.253444790840149, 0.0447021909058094, 1.5739067792892456, -0.7312699556350708, -0.9064878821372986, 0.4716149568557739, -0.5376332998275757, -0.5201663970947266, 1.7285220623016357, 0.6764577031135559, 1.0168179273605347, 0.12187656760215759, 1.1617109775543213, 0.6518608331680298, 0.42814841866493225, -0.5482814311981201, 0.5908330678939819, 1.0805085897445679, -0.1868065893650055, 1.9298208951950073, -1.919319987297058, 3.0587692260742188, -0.9658706188201904, -0.2952001690864563, 0.5085131525993347, 0.708915650844574, -1.0764883756637573, -1.1671438217163086, -0.32707834243774414, -0.9228200912475586, -1.9731554985046387, 1.9916672706604004, -0.5914479494094849, 0.05385648086667061, 0.8630463480949402, 0.7560186982154846, 0.4169504642486572, 0.8269687294960022, 1.0517429113388062, 0.9672116637229919, -1.17240571975708, 0.009376647882163525, -1.984278917312622, 0.04322696849703789, 0.9408422112464905, -1.3728879690170288, 0.7048395872116089, 1.677017092704773, 1.3561580181121826, -1.0819482803344727, -0.7692757844924927, -0.644873857498169, 0.501118540763855, -0.5765076279640198, -1.6928598880767822, -2.0041778087615967, -2.8470258712768555, 1.2998205423355103, -2.517180919647217, 1.0299235582351685, 0.016495229676365852, -0.31116101145744324, -0.08329148590564728, 0.49557194113731384, 0.4331866204738617, -1.361689567565918, -1.0081228017807007, 1.1675041913986206, 0.7180668711662292, 2.498457670211792, 0.17547771334648132, 0.8765491247177124, -0.7064706087112427, -2.162541151046753, 0.3381110429763794, -0.7278127074241638, 0.7269962430000305, 0.3693089187145233, -0.2787744402885437, -2.04742431640625, -0.17763647437095642, -2.3227927684783936, 0.014629149809479713, -0.09199479967355728, 1.8137539625167847, 0.4290613532066345, -0.876726508140564, -0.05292452871799469, 0.5894220471382141, -0.03922775387763977, -0.30130380392074585, -0.7540006041526794, 0.2064833790063858, -0.5187020301818848, -1.0932540893554688, -0.1261925846338272, -0.5150795578956604], [0.8359863758087158, -1.380937933921814, 0.49831050634384155, -0.3545575737953186, -0.34035724401474, -1.341155767440796, 0.4614256024360657, -0.6390987038612366, 1.060433268547058, 1.1972920894622803, 0.990172266960144, 0.023342672735452652, 0.2071327269077301, 0.7910577654838562, -1.2773325443267822, 1.5495984554290771, -0.26311084628105164, 1.519059419631958, 0.5011513829231262, 0.05247892811894417, -0.5181642770767212, 1.9008883237838745, -0.4381016194820404, -1.2971431016921997, -0.9710901379585266, -0.5416361093521118, 2.2462480068206787, 0.07162129878997803, -1.0548152923583984, -1.4247064590454102, -2.522406816482544, 0.06295981258153915, -0.16183988749980927, -0.3523063063621521, -0.02414518967270851, -0.8992500901222229, 0.42573288083076477, -2.7381129264831543, 1.4512343406677246, -1.9617449045181274, 0.5561217665672302, 0.12929902970790863, 0.3840200901031494, -0.14863300323486328, -0.2801900804042816, -0.8020613789558411, 0.6958366632461548, -0.6522722840309143, -1.7845882177352905, 0.8233245015144348, 1.0703763961791992, -3.1979012489318848, 0.2685810625553131, 0.45592567324638367, -0.6993114352226257, -0.2718280851840973, 0.2123211771249771, -1.6560924053192139, 0.5427345037460327, -2.9634392261505127, -2.4290220737457275, 0.784719705581665, -0.020791208371520042, 0.08055006712675095, 0.41035860776901245, 0.6155051589012146, 0.13277976214885712, -0.8219906687736511, -0.4239904582500458, 0.06615376472473145, 0.47025391459465027, -1.5901867151260376, 0.2559059262275696, -0.06614987552165985, -0.49341097474098206, -0.6053796410560608, 0.2886357605457306, 1.0878169536590576, -0.15548744797706604, -1.0443575382232666, -0.13920044898986816, -1.8798710107803345, -1.556002140045166, 1.736402153968811, -0.04344719275832176, 0.020266342908143997, 0.7793504595756531, -0.5338797569274902, -0.8738304376602173, 0.09385466575622559, -0.5095292329788208, -0.6013039946556091, 0.6212668418884277, 0.527553915977478, 0.4095441997051239, -0.8364208340644836, -0.6852431297302246, -0.18574319779872894, 1.8096539974212646, -0.6639056205749512, 0.6573107242584229, 0.2158776819705963, 0.8000982999801636, 1.063919186592102, 0.737422525882721, 2.0412020683288574, 1.2498500347137451, -1.3929390907287598, -0.3083430230617523, -0.45140403509140015, -1.7201929092407227, 0.07427993416786194, -1.0752058029174805, -0.7272034287452698, -0.7246419787406921, -2.0065741539001465, 0.21766874194145203, 1.3761324882507324, 1.0822434425354004, -1.0502582788467407, 0.3453483283519745, -1.1719340085983276, -0.0835726261138916, 0.2840985357761383, 0.28901585936546326, 1.651547908782959, 1.4340497255325317, 1.1542657613754272, -1.8691797256469727], [1.0833534002304077, 0.22049714624881744, 0.953198254108429, -1.442177176475525, -0.6935399174690247, -0.5258642435073853, -1.5400876998901367, 0.6077755689620972, -1.1612296104431152, 0.8850347399711609, 0.2910436987876892, 0.11742180585861206, -0.466967910528183, 0.13282527029514313, -0.5380618572235107, 0.328620582818985, 1.1262187957763672, -0.16665156185626984, 0.4060732126235962, 0.8724951148033142, 0.5783786177635193, 0.7914848327636719, 1.8043345212936401, -2.0013599395751953, 1.196365237236023, 0.11733631044626236, 0.6370500326156616, -0.6835517883300781, -1.0577410459518433, -0.46968355774879456, -1.1117600202560425, 0.35103777050971985, -2.1230897903442383, 0.11745771020650864, -1.0720672607421875, 0.6175081729888916, -1.473895788192749, 0.32689663767814636, 0.7682324647903442, -0.9447293877601624, -1.709098219871521, -0.7338622808456421, 0.2532491087913513, -1.1815911531448364, -1.2797319889068604, 0.6768861413002014, 0.26825565099716187, -0.056379348039627075, -0.8215741515159607, 0.42261767387390137, -0.8536680340766907, 2.4790701866149902, -1.1790202856063843, 0.39839109778404236, -0.6487302780151367, -0.791374683380127, -0.37553247809410095, -0.16742928326129913, 0.12493763118982315, -1.021946907043457, -1.029022216796875, 0.7239935994148254, 2.063227415084839, 1.0308613777160645, 0.1330147385597229, -0.38053539395332336, -1.1260359287261963, -0.43474462628364563, -0.9208627939224243, -0.4478299021720886, 0.5208923816680908, -0.1666685789823532, 0.045443229377269745, -0.8655115962028503, 0.3687862753868103, 0.15121278166770935, -0.8686646819114685, -0.7512343525886536, -0.480097234249115, -0.601371705532074, -0.6329839825630188, -0.9901520609855652, 0.9290540218353271, 0.37683016061782837, -0.4499557614326477, -0.20764397084712982, 1.0976463556289673, 0.35498616099357605, -1.1391178369522095, -0.07397425174713135, 0.8018823862075806, -0.010720552876591682, 0.2705559730529785, 1.1647285223007202, -0.8662434816360474, 1.5752959251403809, 0.5020949840545654, 1.1208511590957642, -0.7366954684257507, 0.3558214604854584, 0.5635177493095398, 1.7994508743286133, -0.9272558689117432, -2.740607976913452, 1.1422817707061768, -0.40193819999694824, 1.2766903638839722, -1.4081238508224487, -0.8505874872207642, 0.4245617985725403, -0.9786903262138367, 0.9657420516014099, 0.563917875289917, -0.5406448245048523, -0.18294726312160492, -0.047146987169981, -0.3762068748474121, 0.1535797417163849, -0.8198685050010681, 0.7446469068527222, 1.6465433835983276, -1.4112383127212524, -6.501204916276038e-05, -0.11886629462242126, 2.15545392036438, 1.291445016860962, -0.40078121423721313, 1.5961583852767944, 0.004157414194196463], [-0.03340468555688858, 0.21589751541614532, 0.6398479342460632, -0.6550628542900085, -1.8640551567077637, 0.7988986968994141, -0.6892556548118591, 1.9263235330581665, -1.276771068572998, 0.7925539612770081, 0.2724400460720062, 0.18731839954853058, 0.14130710065364838, -0.14047162234783173, 0.13837440311908722, -0.379552960395813, -0.5324285626411438, -0.5072619915008545, 0.811676561832428, 0.2862972319126129, -0.1919042319059372, -0.6516860723495483, -0.4631688892841339, 1.0494147539138794, -0.2708650231361389, 0.8321598768234253, 3.1799325942993164, -1.3534431457519531, 1.1094800233840942, -1.3015486001968384, 1.4033538103103638, -0.42827585339546204, -0.7770565152168274, -0.5900310277938843, -1.8161511421203613, -0.7561963200569153, -2.240934133529663, -0.2567133605480194, 0.6548356413841248, -0.821133553981781, -0.9321056008338928, 0.7757388353347778, 0.040672581642866135, 0.41648411750793457, -0.1040678396821022, 1.1866563558578491, 0.08043690770864487, 0.5444843769073486, 0.5115967988967896, 0.5720095634460449, 0.3519023358821869, 0.18636032938957214, 0.5082873702049255, -0.2503696382045746, 0.6814813017845154, 0.1087176501750946, 2.409313678741455, -0.5532045960426331, 0.5791193842887878, -0.24459302425384521, 0.44443652033805847, 1.3490585088729858, 0.35463565587997437, -1.416250467300415, 0.7216426730155945, -1.3307201862335205, -0.879442572593689, -1.6161317825317383, -1.2811416387557983, 0.9712274074554443, -1.517578125, 0.6794564127922058, 0.1800963282585144, 0.6134698987007141, -0.5291522741317749, -0.4535863995552063, 0.3387172222137451, 1.0611629486083984, 0.8616043925285339, -0.24727456271648407, 1.100249171257019, 2.068737745285034, 1.5004292726516724, 0.34742358326911926, 0.5963472127914429, -0.3788105547428131, -0.17059046030044556, -1.342962622642517, 1.1703165769577026, 0.32888785004615784, -0.26976633071899414, -0.31317493319511414, 0.31656914949417114, -0.8208540678024292, -0.3448110818862915, 0.052261222153902054, 1.8869582414627075, -0.9736945033073425, -0.31930026412010193, 1.107003092765808, 2.0689589977264404, 1.344688057899475, 0.8121373653411865, -0.0010538334026932716, -0.10428620129823685, -0.8561356663703918, 1.0445291996002197, 0.5176385045051575, 0.7994566559791565, -0.08763691037893295, 0.0207215528935194, -0.5278851389884949, -0.35640278458595276, 0.7515433430671692, -1.7086492776870728, -0.5250983834266663, -1.6590380668640137, -0.4293559789657593, 1.2146857976913452, 0.6014537215232849, 0.5127989053726196, -0.13177335262298584, -0.1476820856332779, 0.44601795077323914, 0.4696958661079407, 0.7723305225372314, 1.2795445919036865, -2.0729472637176514, -0.7627959847450256], [-0.39986705780029297, -0.020746985450387, -0.9179854393005371, 0.07658509910106659, 0.3082068860530853, -0.9191382527351379, -0.5476077198982239, -0.22722291946411133, 0.3032059073448181, -1.6958202123641968, -0.9794674515724182, -0.8981302976608276, -1.7362576723098755, -0.5106009840965271, -0.24546660482883453, -1.0024651288986206, -1.233960747718811, -0.43447649478912354, -0.4617805778980255, -0.9181814193725586, 0.2616676688194275, 0.6400147676467896, 0.6935412883758545, -0.5048832297325134, 1.3433095216751099, 0.1377100795507431, 0.3502509295940399, 0.2798733115196228, -0.4581717252731323, 0.23235823214054108, -1.2102569341659546, -0.5294305682182312, 0.06379345059394836, 1.5215728282928467, -0.06891230493783951, 0.8930854201316833, -0.8609851002693176, 0.06393411010503769, 0.21468107402324677, -1.4501067399978638, 0.501715362071991, -1.7611879110336304, -2.547095775604248, 1.5268045663833618, -0.8981227278709412, -0.9811661839485168, 0.4075445234775543, 2.030172109603882, 0.09079813957214355, 0.5194119215011597, 0.1015746146440506, -0.6589378118515015, 0.16926294565200806, -1.1826188564300537, -0.8260392546653748, 0.011014405637979507, 1.6472145318984985, -0.3913077712059021, 0.8646438121795654, 0.837719738483429, -1.3012027740478516, -1.0766743421554565, -1.1301110982894897, 1.3115304708480835, 0.2912774085998535, -0.9434278607368469, 0.8564701676368713, -0.8631431460380554, 1.0036321878433228, 0.1385994404554367, 0.2608402371406555, -0.07416584342718124, -0.39815011620521545, 0.7985548973083496, -0.09984206408262253, -1.6348694562911987, 0.5510304570198059, -2.1261186599731445, -2.475578784942627, -0.5514132380485535, -1.2480111122131348, -0.44992727041244507, -0.3157641887664795, -0.009881202131509781, -0.8920581340789795, 1.4428153038024902, -2.2622416019439697, 1.1936578750610352, -0.5567309856414795, -0.347298264503479, 0.9281747937202454, -0.13467881083488464, -0.7586693167686462, -0.049514878541231155, 1.5156934261322021, 0.6037645936012268, -0.834606409072876, 0.062406640499830246, 0.7262446284294128, 1.525248408317566, -0.7006291151046753, 0.3689265549182892, -0.3018454611301422, 0.5718473196029663, -0.4966556429862976, 0.575934886932373, 1.2308039665222168, 2.20302414894104, 1.6017911434173584, 0.9472029209136963, -0.32610490918159485, -0.7769110798835754, -0.1322060078382492, 0.014054983854293823, -0.3707086145877838, -0.9972715973854065, -1.0138133764266968, 0.7966355085372925, 1.221218466758728, -1.172121286392212, -0.6206205487251282, -1.1748239994049072, 1.3111952543258667, -0.12638847529888153, 0.9799155592918396, 1.0299172401428223, -0.8490996956825256, 1.4114164113998413, 0.8872531056404114], [0.333967924118042, -0.7480714321136475, 1.6113260984420776, 0.7459124326705933, 1.8959027528762817, -1.3799338340759277, -0.4691944420337677, 0.8085874319076538, -0.9640553593635559, 0.8270590305328369, 0.5323801040649414, 0.08308281749486923, -0.4088824689388275, -0.8622428774833679, 0.799974799156189, -0.26413023471832275, -1.3295174837112427, -0.4736022651195526, -0.805376410484314, -0.7567440271377563, -0.6045808792114258, 0.1481017768383026, -1.3626620769500732, -0.06808505207300186, -0.07912787795066833, -1.4474551677703857, -0.11016768217086792, 1.4339264631271362, 0.5708624720573425, 1.299837350845337, 1.2511603832244873, 0.19418220221996307, 0.5114097595214844, 1.3680685758590698, -0.3917345404624939, -0.47883689403533936, 0.3560909926891327, 0.31938615441322327, -2.0002236366271973, 0.12656795978546143, 1.9533802270889282, -1.0110869407653809, -0.1747143417596817, -1.484090805053711, 0.9805435538291931, -0.18547706305980682, -1.3298832178115845, 0.08724205195903778, -0.8559548854827881, 1.2239127159118652, -0.9096139669418335, -0.00497426837682724, 0.12496955692768097, -1.126293420791626, -2.3020927906036377, -0.9648492932319641, 0.7835515737533569, 2.356902837753296, -0.12869790196418762, 1.614192247390747, 0.22019587457180023, 1.427476167678833, 1.2562956809997559, -0.6487465500831604, 1.07023024559021, 0.938358724117279, -1.385537028312683, 1.2000056505203247, -0.42436057329177856, -1.2665324211120605, -0.11402186751365662, 0.3780381977558136, -1.6114929914474487, 0.2092200219631195, -1.4503241777420044, -1.3604052066802979, -0.9521031379699707, -0.4696941077709198, 1.2037907838821411, -1.3527097702026367, -2.246032476425171, -0.42151808738708496, -1.4093221426010132, -1.030367374420166, 0.10803379863500595, -0.30682075023651123, -0.007415567059069872, 0.26516637206077576, -0.2637404203414917, -1.1107131242752075, 0.06874794512987137, 1.3674248456954956, -0.4820821285247803, -0.07876355201005936, -0.8668299913406372, 0.4276513159275055, 0.3918478488922119, -0.744817316532135, -1.4838913679122925, -0.8248240947723389, 0.38814783096313477, 1.3142002820968628, -0.5571244955062866, 0.23470546305179596, 1.786375641822815, 0.14771483838558197, 0.8736638426780701, 0.47825056314468384, -0.8403207659721375, 1.0771737098693848, 1.0603868961334229, 0.20112162828445435, -0.0808240994811058, 0.6162853240966797, 0.5704194903373718, 0.23547779023647308, -0.6026878952980042, 1.0318260192871094, 0.6205716729164124, 0.3966720998287201, 0.0732608214020729, 0.2633831799030304, 0.05830850079655647, 0.5469021797180176, 0.5065319538116455, -1.3192992210388184, 0.03013124316930771, 0.6910872459411621, 0.7945799231529236], [-2.021390199661255, 0.9256994128227234, 0.0002108213520841673, 1.5509570837020874, 0.9927077889442444, 1.5429271459579468, -0.948165237903595, 0.80830979347229, 0.6237647533416748, 1.1753344535827637, 0.4280378222465515, 0.928867518901825, -1.1756243705749512, -0.4568774998188019, 0.5987688899040222, 0.8119748830795288, 0.42864376306533813, -1.8814111948013306, -1.5998319387435913, -0.7176708579063416, 1.0164847373962402, -0.9814361333847046, 1.2573095560073853, 1.5875909328460693, 0.3177388608455658, 0.6488023996353149, -0.5924589037895203, -0.2219248265028, 1.0621154308319092, 0.10732343047857285, 0.3142722249031067, -2.4828054904937744, 0.36202970147132874, 0.19210302829742432, -0.7966189384460449, 0.026147030293941498, 0.2502807378768921, -1.524309754371643, -1.3317469358444214, -0.03169839829206467, 0.2395242154598236, 1.3063031435012817, -1.6130911111831665, 2.4973702430725098, -0.5010840892791748, 1.795961618423462, -0.12514154613018036, -1.6808815002441406, -0.2316974699497223, 0.5151517391204834, -0.7640206217765808, 0.2228529453277588, 0.09529103338718414, -0.179830864071846, 0.9134041666984558, 0.4577103555202484, -0.48653584718704224, -0.22837936878204346, 0.4683188199996948, 0.10510766506195068, 0.6224253177642822, 1.4155547618865967, -1.5445724725723267, -0.17222292721271515, 0.6921839714050293, 0.7878269553184509, -0.7109000086784363, -1.6187975406646729, -0.7141802906990051, 0.5541507005691528, 1.3988909721374512, -1.8103575706481934, 0.5812814235687256, 0.9237278699874878, 0.6987648010253906, 0.6329077482223511, 0.6339824199676514, -0.8023801445960999, -0.5476801991462708, 0.9202532172203064, -0.11756034195423126, 0.6752423644065857, 0.6350976228713989, -0.027503473684191704, 0.9631075263023376, -1.478434443473816, -0.4531392753124237, -0.44445332884788513, -0.5255256295204163, 0.7009962797164917, 0.5654976963996887, -1.2474873065948486, 0.8545300364494324, -0.311407208442688, 1.3879796266555786, -1.2904354333877563, -0.7568174004554749, -0.22110047936439514, -1.3241373300552368, 2.321674346923828, 0.055070336908102036, 0.6588159203529358, 0.6974970102310181, 0.16016364097595215, 0.2322075068950653, -0.75893634557724, 1.3708041906356812, 0.7522733807563782, 0.50262451171875, 0.009495428763329983, -1.60566246509552, -1.3700135946273804, 1.0426080226898193, 1.7273937463760376, -0.2740294933319092, -0.3670351207256317, 0.4921550452709198, -0.8869519233703613, -0.8205702304840088, 1.9194552898406982, 0.3373977839946747, 1.0183864831924438, 0.29045364260673523, -0.537593424320221, 1.7201800346374512, -0.27265599370002747, -0.056915055960416794, -0.10360325127840042, -1.3082765340805054], [-0.743466317653656, 0.0036293372977524996, -0.28562116622924805, 0.33310097455978394, -0.8385294079780579, -0.6986088156700134, -1.4633599519729614, -0.8646374344825745, -0.7212095260620117, -0.6745232343673706, 1.2745400667190552, 1.1390200853347778, 0.7767421007156372, 0.11287061125040054, 0.02295949123799801, 0.5298161506652832, 0.2032020390033722, -0.9711539149284363, 0.9986404180526733, -0.716035008430481, 0.9107227921485901, 0.32552024722099304, 0.37130799889564514, 0.16351290047168732, -1.090551495552063, -0.4136490225791931, -1.0253527164459229, 0.4251902103424072, 0.9478392004966736, -2.2353248596191406, -0.838969886302948, -0.7492038607597351, 0.055374473333358765, 0.7940636873245239, 0.721799910068512, -0.3891313970088959, 1.8000000715255737, -0.19172751903533936, 0.44978365302085876, 1.1944339275360107, 0.5918006896972656, 0.8362700343132019, -0.5760617852210999, -0.5663011074066162, 0.3114084303379059, -1.1962922811508179, 0.7556966543197632, 0.37151530385017395, 0.021620124578475952, -2.0499887466430664, 0.34441423416137695, -0.9660352468490601, -0.8444845676422119, 0.6820617914199829, -2.029575824737549, 0.13344208896160126, 0.33480530977249146, -0.27962371706962585, 0.08819124847650528, 0.251918762922287, 0.10961528867483139, 0.14221884310245514, 0.5961690545082092, -1.935590147972107, -0.1070924624800682, -1.1855723857879639, 0.4716944098472595, 1.6319900751113892, -0.07436274737119675, 1.1325126886367798, 1.8781964778900146, 0.256971150636673, 0.177265927195549, 1.107311725616455, 0.4501790404319763, 2.145312547683716, -1.8170101642608643, -1.1783441305160522, 0.4301665425300598, 0.1261318176984787, 1.9193079471588135, 0.4435964822769165, 0.924187958240509, -0.3955899477005005, -0.15227851271629333, 0.9322423934936523, 0.9926047325134277, -0.15209653973579407, -1.574019193649292, 0.23002727329730988, -1.2462886571884155, 0.5109497904777527, -0.2145923674106598, -0.052953869104385376, 1.454376459121704, -0.14352568984031677, -1.8104791641235352, 0.04090750589966774, 0.7280836701393127, 0.6394498944282532, 0.4045253396034241, -0.45992133021354675, -0.19122940301895142, -0.13797228038311005, -0.37382546067237854, 1.240061640739441, -1.3953516483306885, -1.0013971328735352, 0.979770839214325, -0.6082169413566589, -1.363735318183899, 0.5566012859344482, 1.3718369007110596, -1.3780289888381958, -0.5073663592338562, 0.5160377621650696, 0.5634018182754517, -1.7839313745498657, 0.7915396690368652, 0.14605043828487396, -0.5106052160263062, -0.788568377494812, -0.6046201586723328, -1.1133222579956055, 0.38727518916130066, -1.8228132724761963, 0.5095186233520508, -0.6406095623970032, -0.5357432961463928], [-0.3740324378013611, 0.2482128143310547, -1.1661430597305298, -0.4678311347961426, -0.7688093781471252, 1.2821625471115112, 0.20889486372470856, -0.7871504426002502, -0.4127404987812042, 1.0656499862670898, -1.009577989578247, 2.2650644779205322, -1.281458854675293, 0.35549160838127136, 1.7216168642044067, 1.3820992708206177, -0.1605641394853592, 0.7539194822311401, 2.408754825592041, 0.9903070330619812, 0.6367560625076294, 0.3177366256713867, -0.6325602531433105, -1.320261001586914, -1.5470495223999023, 2.044583320617676, -2.517601251602173, -1.2302988767623901, 0.002651857677847147, -1.8607280254364014, -1.4035497903823853, -0.9620527029037476, 1.1879825592041016, 0.8576151132583618, -0.2210211306810379, 1.793765664100647, 1.806827187538147, -0.33551695942878723, 0.7704347372055054, -0.9827138781547546, 1.897585391998291, 0.4069586992263794, -0.3596953749656677, -0.6765456199645996, -0.5983591079711914, -1.038055419921875, 1.117313265800476, -1.0678339004516602, 0.6764521598815918, 0.9593307971954346, -0.6588164567947388, 0.8565413355827332, -1.616817831993103, -0.04811352863907814, 0.678512692451477, -0.8177132606506348, 0.7013764381408691, 0.5583594441413879, -0.6984781622886658, 0.5386003851890564, -1.2349272966384888, 0.26978492736816406, -1.3683818578720093, -1.4181642532348633, -0.052095599472522736, -0.5993499159812927, -0.6717456579208374, -0.7120804786682129, -1.5501855611801147, -0.07777101546525955, 0.19154271483421326, -1.0765612125396729, 0.7159093618392944, 0.8200863003730774, 0.5701697468757629, -1.3950233459472656, -0.5744006633758545, -0.8354628682136536, -0.8962712287902832, -1.9634672403335571, 1.2387784719467163, -0.543282687664032, -1.0984665155410767, -2.0017683506011963, -0.7094261646270752, 1.4981907606124878, 0.4757860600948334, -0.10786062479019165, 0.4269682466983795, -1.6705061197280884, 0.7716857194900513, 1.2652548551559448, -0.6051433682441711, 0.9205684065818787, -1.2860437631607056, -1.4142314195632935, 0.7067199349403381, 0.316196084022522, 1.7421201467514038, 0.3384481966495514, -1.2495207786560059, -0.6635225415229797, -1.8940114974975586, -0.179128035902977, -0.449219286441803, -1.2051595449447632, 0.9523537755012512, 0.41745203733444214, -0.5429044961929321, 0.15564218163490295, 0.5048206448554993, -1.5271236896514893, -0.006677641067653894, -0.4024132192134857, 1.6923871040344238, 0.8696302175521851, -1.0081586837768555, 0.7697405815124512, -0.9380024671554565, -0.36785101890563965, -0.38968706130981445, 1.129204273223877, 0.7599141001701355, 1.1554620265960693, -0.6968084573745728, 0.6428133249282837, -0.4340314567089081, -1.865545630455017, 1.240377426147461], [-0.7930078506469727, -0.2979665696620941, -0.43731430172920227, 0.4534366726875305, 0.9400695562362671, -1.6207945346832275, 0.7741587162017822, -0.8192231059074402, 0.4492270052433014, 0.14062273502349854, 0.7266397476196289, -0.2500704824924469, -1.3708252906799316, 0.40156641602516174, 0.11897479742765427, -1.010865569114685, 0.9854317903518677, 1.6416462659835815, -0.29706066846847534, 0.24598370492458344, 0.02161424234509468, 0.19183388352394104, -1.1208299398422241, -0.6053432822227478, -0.6307616829872131, 1.3431154489517212, -1.3420127630233765, 0.8782814145088196, 0.13037753105163574, 1.22575843334198, -1.46025812625885, -2.1748909950256348, 0.10562369227409363, -1.2299567461013794, -1.4181804656982422, -1.844090461730957, -0.40448132157325745, -0.5211437344551086, -0.7505061626434326, -0.10479812324047089, -1.0141382217407227, -1.6461398601531982, 0.2217969298362732, 0.010945039801299572, -1.2541680335998535, 2.005730390548706, 0.024915222078561783, 1.4373900890350342, 0.544299840927124, -2.428004741668701, -0.025871386751532555, -2.3006625175476074, 0.3959125280380249, -0.9157298803329468, -0.9370493292808533, 1.0805631875991821, 0.6231626272201538, -0.6032808423042297, -0.09229281544685364, 0.12363595515489578, -0.9546651840209961, -0.1667926013469696, 0.5417232513427734, 0.5013796091079712, 0.07233574986457825, 2.000828266143799, -1.0023143291473389, -1.0836193561553955, -1.538219928741455, -1.023746371269226, 0.19345323741436005, -0.7475892901420593, 0.23574481904506683, -1.456362247467041, 0.9846783876419067, -0.2321552187204361, 0.5120836496353149, -2.4972832202911377, 0.3888430893421173, -1.6612955331802368, -0.7605029344558716, -0.5179673433303833, 0.5733559131622314, -1.8582510948181152, 0.8070730566978455, -0.5456281304359436, 0.6843917965888977, 0.22911305725574493, 0.010928316041827202, -0.55938321352005, 0.7620610594749451, -0.6613144874572754, 0.9050872325897217, -0.6912710666656494, -2.7170705795288086, -0.7767110466957092, 1.6380290985107422, -1.0676965713500977, 0.009329065680503845, 0.3100857138633728, -0.4741447865962982, -2.7226808071136475, 0.035201653838157654, -0.9255325794219971, -0.7122790217399597, 0.33392685651779175, 1.0913779735565186, 1.7542033195495605, -0.11921229958534241, 0.4317227602005005, 0.6590474843978882, 1.7620375156402588, -0.831648051738739, 0.1694362610578537, 0.006957349367439747, 1.1013379096984863, 1.0140509605407715, -1.6884936094284058, 0.8187386393547058, 0.3885815441608429, 0.007956575602293015, 1.0436298847198486, 0.5954083204269409, -0.11855616420507431, 2.26934552192688, -1.8152858018875122, 0.19418518245220184, -0.5022403001785278, 0.9957122802734375], [1.1185411214828491, 0.3427393138408661, 0.1048453152179718, -0.6318184733390808, 0.7676652669906616, -1.1997319459915161, 0.3039226233959198, 0.006859506946057081, -1.4094243049621582, -0.009506420232355595, 0.4846199154853821, -2.1961817741394043, 0.9718470573425293, 0.7383678555488586, 2.0393776893615723, 0.6931996941566467, 1.123970866203308, -0.8483695387840271, 0.5313724279403687, -0.2839019000530243, 0.23100155591964722, 0.3436320424079895, 1.8585413694381714, -0.0783592164516449, -1.4951331615447998, 1.948921799659729, 2.1193909645080566, 0.32445624470710754, -1.322266936302185, -0.7294233441352844, -0.7507398724555969, 1.0280046463012695, -1.0192804336547852, 0.006754183676093817, 1.0518450736999512, -0.2638567090034485, -1.026936411857605, 0.1097543016076088, -1.0876796245574951, 1.3020597696304321, 1.3291164636611938, 0.13476529717445374, 1.207234263420105, -1.4002563953399658, -1.3922252655029297, -0.5450801253318787, -2.150825262069702, 0.8175517916679382, -1.7821495532989502, -0.2791541516780853, -0.5979681015014648, -0.6564042568206787, 0.6391057968139648, -1.6199415922164917, 0.6503835320472717, -1.8231292963027954, 0.13260862231254578, 1.6205300092697144, -0.12514838576316833, -0.2885783314704895, -1.3827468156814575, -0.5394175052642822, 0.07228291034698486, -1.0695700645446777, 0.011939107440412045, -1.4493178129196167, -2.051321268081665, -0.6243593692779541, 1.9576647281646729, -0.9472294449806213, -1.1513981819152832, -1.568129539489746, 0.31893569231033325, 0.04255182668566704, 0.6166930198669434, 0.7430257201194763, -1.3325287103652954, 1.8293991088867188, 1.659716010093689, -1.161199927330017, 1.1458674669265747, -0.8205615282058716, 0.13025039434432983, 0.8990603685379028, 1.3147969245910645, -2.231544017791748, 0.5555250644683838, 0.3276442885398865, -0.39219385385513306, 0.2793455719947815, -1.302268385887146, -1.367438793182373, 0.8951628804206848, -1.2880470752716064, 1.1816245317459106, -0.5303683280944824, -0.03040371648967266, -0.8190852403640747, 0.4464162588119507, -0.9039377570152283, 1.2253751754760742, 1.5116924047470093, -0.8647879362106323, 0.3887837827205658, -0.46867984533309937, 0.6746968626976013, -1.6368135213851929, 1.4823681116104126, -0.18689127266407013, 0.5890922546386719, -1.1760752201080322, 0.2842719852924347, 1.4479998350143433, -2.2763826847076416, -1.8344331979751587, 1.1777760982513428, -1.0295488834381104, -0.6073259711265564, 0.7756621837615967, -0.8545498847961426, 0.1744205802679062, 0.4345756769180298, 0.38113030791282654, -0.10951719433069229, -1.0449110269546509, 0.39559897780418396, -1.0499279499053955, -0.7142850160598755, 0.31608766317367554], [-0.17556007206439972, 2.4254183769226074, 0.38728082180023193, -0.38046470284461975, -0.9257431030273438, -0.8418877124786377, 1.9662648439407349, 0.6255982518196106, -0.2796408236026764, 0.7021990418434143, -1.783642292022705, 0.46646949648857117, 0.27835938334465027, 1.530649185180664, -0.43030238151550293, 1.8020464181900024, -1.7230441570281982, -0.6289772391319275, 1.2173079252243042, 2.4624688625335693, -0.7513296008110046, -0.7913365960121155, 0.9774634838104248, 0.06728506833314896, -0.811630368232727, 1.0534052848815918, 0.351747989654541, 0.11321691423654556, -1.6109983921051025, -0.11938530206680298, 0.6943074464797974, 0.8633847832679749, -0.5517358183860779, -0.33501100540161133, -1.4620952606201172, 2.3188726902008057, -0.5060493350028992, 1.0212339162826538, 0.3954426646232605, -0.1336357593536377, -1.4869471788406372, 1.8959685564041138, 0.790473222732544, 1.363050937652588, -0.09233502298593521, 1.1809208393096924, 2.0713613033294678, -1.1999232769012451, -1.0500644445419312, -0.41711920499801636, -0.005823420826345682, -0.3792635500431061, 0.6631466150283813, -0.3831646144390106, -1.0379105806350708, -0.5098373293876648, -0.10009054094552994, 1.3468596935272217, 1.4799903631210327, 0.18452279269695282, -1.0300501585006714, 0.3574207127094269, -0.975638210773468, -2.733003854751587, -0.5936076641082764, -0.4518193006515503, 1.0888718366622925, -0.9123328328132629, 1.4506556987762451, 0.3561338186264038, 0.11206933856010437, 1.3316627740859985, -0.37236568331718445, 0.22288912534713745, -0.9646968245506287, -0.5905513763427734, -0.11603746563196182, 0.9742287993431091, 0.5786973834037781, 0.9846006631851196, -0.49825599789619446, 0.9398149847984314, 0.01283360831439495, -0.8089845776557922, 0.4785721004009247, -0.6929994821548462, -0.01816960610449314, -1.0253247022628784, -1.0585498809814453, 1.2592906951904297, -1.5803488492965698, 1.121146321296692, 0.9909021258354187, -0.4676901400089264, -0.03096107579767704, 2.10300350189209, 0.7790458798408508, -0.627258837223053, -0.5593007802963257, 0.5080137252807617, -0.12404866516590118, -0.11045041680335999, 0.3690970242023468, 0.42531469464302063, 0.23539359867572784, -0.09208065271377563, -0.5950751900672913, -0.3968086242675781, 0.2875933051109314, 0.38119083642959595, 0.0597434937953949, 0.34048134088516235, -0.46931493282318115, -2.2968194484710693, 0.7999714612960815, -0.8994255661964417, 0.7424878478050232, -0.9819220900535583, 0.2630453407764435, 0.7514167428016663, -0.017058532685041428, -0.6473592519760132, 0.10519929975271225, 0.704819917678833, -0.25595811009407043, -1.4386566877365112, -0.5509327054023743, 0.10462834686040878, 1.3495700359344482], [0.7428323030471802, -0.10063739120960236, 0.037601158022880554, -1.447087287902832, -0.8165813684463501, -0.4636095464229584, 0.9627528190612793, 0.8085278868675232, -0.023677371442317963, -1.7864247560501099, -0.656046450138092, 0.36457929015159607, 0.7654931545257568, 1.1256794929504395, 0.25900352001190186, 0.6703948974609375, -0.7922737002372742, 0.4679614007472992, -0.02834327332675457, -0.811126708984375, 1.0606369972229004, 0.2334325909614563, 0.9770823121070862, -0.6880629062652588, 1.1017895936965942, 1.0190623998641968, 0.441204309463501, -0.15478076040744781, -0.5419743657112122, -0.1530962735414505, -0.11642944812774658, 0.7764866948127747, -0.06585422158241272, -1.3141348361968994, 0.11971747875213623, -0.2610336244106293, 0.5353924632072449, -0.043662428855895996, 0.5460220575332642, 1.2278257608413696, -0.9755694270133972, 1.156082034111023, 0.7549436688423157, 0.7930372357368469, 0.23664413392543793, 0.498700886964798, 1.0200611352920532, 0.5081264972686768, -0.3029111325740814, -0.8778734803199768, -0.9028892517089844, -0.06727025657892227, -0.2452271580696106, -1.8623344898223877, 1.0645146369934082, 0.979799747467041, -0.35056138038635254, 0.2736527621746063, -1.1605181694030762, 1.6235452890396118, -0.7374861240386963, -1.5156190395355225, -1.5177524089813232, 1.9720016717910767, 1.7188091278076172, 0.6650002598762512, -1.0581960678100586, -1.1827548742294312, 0.04616234824061394, 1.2050347328186035, 1.6301138401031494, -0.9719106554985046, -0.288003534078598, -0.9094371199607849, 0.8443842530250549, 0.4048001766204834, 1.9609135389328003, -0.36801087856292725, -0.5564974546432495, -0.6501116752624512, 0.42652955651283264, 1.1313544511795044, -0.2721141576766968, 1.2708369493484497, 0.9640451073646545, -1.651809811592102, -0.011817938648164272, 0.5773869156837463, -1.5118873119354248, -0.9818702936172485, 1.1498517990112305, 0.8467201590538025, -0.2145024687051773, -0.37921571731567383, 0.40042418241500854, -1.1796956062316895, -0.9615359902381897, -0.31876981258392334, -2.2982826232910156, -0.37945953011512756, 1.8502728939056396, 0.36220261454582214, 0.23326632380485535, 0.15617366135120392, 0.3294861912727356, 0.7227759957313538, 0.030605755746364594, -0.4703914523124695, 0.8245878219604492, -1.5942659378051758, 0.7477511763572693, 2.111419916152954, 0.3160340189933777, -0.11660046130418777, -1.5807009935379028, -1.0108311176300049, -0.26922574639320374, 0.46525001525878906, 0.5556476712226868, 0.5628487467765808, 0.6715378761291504, -1.5972124338150024, -0.32068052887916565, 0.6687403917312622, 0.6508370637893677, -1.8437426090240479, -0.4683392345905304, -1.150336503982544, -1.3964791297912598], [0.5419353246688843, 0.70033198595047, -1.064470887184143, 0.6165246367454529, 0.8064563870429993, -0.14966538548469543, 1.4360616207122803, -1.4289255142211914, 0.2795632481575012, 0.19435003399848938, 0.3713036775588989, 2.2326128482818604, -0.983242392539978, -1.075884461402893, -0.5752225518226624, 1.051285982131958, 1.8038729429244995, -0.7681140303611755, 0.15697792172431946, 2.0468997955322266, -2.171201229095459, 0.07464113086462021, -0.20317178964614868, -1.6893556118011475, -0.7461008429527283, 0.3886605203151703, -0.02180088683962822, -0.5530136823654175, -0.8911799192428589, -1.3115026950836182, 0.6053337454795837, -0.7558557391166687, 0.49488091468811035, -2.5365118980407715, -0.5981248021125793, 0.7991756796836853, 1.2501327991485596, 0.9941149353981018, 0.33632802963256836, 0.2150474339723587, -1.166602611541748, 0.7358894348144531, 2.2090601921081543, -1.0281692743301392, 0.5713578462600708, -0.3876006603240967, -0.7868258953094482, -0.348239928483963, -0.8284193873405457, -1.031754970550537, 1.4132664203643799, 0.9352989196777344, 1.2803702354431152, -0.7109130620956421, 1.346258282661438, -1.0584007501602173, 1.1400845050811768, 1.3092334270477295, 0.3652329444885254, 1.583604097366333, -0.5153742432594299, -0.4505200982093811, 0.428739994764328, 0.0004955907352268696, 0.3056945502758026, -0.1849159449338913, 0.4857214689254761, 0.7170647382736206, -3.284458875656128, -0.32517918944358826, 0.6757814884185791, 0.8799925446510315, 1.018470287322998, 0.06898902356624603, -1.2832956314086914, -0.20447254180908203, 1.8421603441238403, 0.48498982191085815, -0.6404339671134949, 0.306876003742218, -1.7950907945632935, 0.3860469162464142, -2.0071840286254883, -0.19666330516338348, 0.897098958492279, -0.6119740605354309, 0.5393538475036621, -2.681401491165161, 0.258647084236145, -2.3599956035614014, 0.0917372778058052, -1.2406165599822998, 1.6595640182495117, 1.165460228919983, 1.2287395000457764, 0.9508688449859619, -1.9716768264770508, -0.12382828444242477, 0.2710331678390503, -0.47762125730514526, 1.1013281345367432, -0.03588390350341797, 0.5188159942626953, 0.5915169715881348, 0.5316265821456909, -0.7992939949035645, 2.1854817867279053, 1.0816218852996826, 0.05464324355125427, 1.5752695798873901, 0.46150442957878113, -0.027426566928625107, -0.3698574900627136, -1.0113084316253662, -1.8322049379348755, -0.11314762383699417, -0.15802854299545288, 1.6109209060668945, 0.6183558106422424, 0.40886229276657104, -0.5762622952461243, -0.7644045352935791, -0.18546417355537415, 0.9043897390365601, 0.17853385210037231, 0.4747425615787506, 0.5386658906936646, -0.5464981198310852, -0.03180816024541855], [-1.2713462114334106, 0.6526315212249756, 0.11867354065179825, 0.3341202735900879, -1.0753973722457886, 0.6394327282905579, -0.38437792658805847, 1.6813372373580933, -0.3377324938774109, 1.6940863132476807, -0.26829326152801514, -0.4084945023059845, -0.035794902592897415, -0.8995552659034729, 0.8052312135696411, -0.8512883186340332, 1.4247651100158691, 0.6075205206871033, 0.746853232383728, -0.2711779773235321, -0.476712703704834, 1.0247502326965332, 1.8239774703979492, -0.22464580833911896, -2.33855938911438, 0.044619832187891006, -0.1854172646999359, -0.032582107931375504, -0.8458611965179443, -0.31808754801750183, 0.8523920178413391, 0.5038536787033081, 1.3469196557998657, -0.3220253586769104, -0.13546383380889893, -0.6867350935935974, -1.757588267326355, 1.160843849182129, -0.02418540231883526, -0.27312174439430237, -1.1102341413497925, -0.2505187690258026, -0.6460400223731995, 0.25492212176322937, -0.6435072422027588, -0.6988806128501892, -0.30128973722457886, -0.07051476836204529, -0.5315312147140503, 0.7249616980552673, -1.9986059665679932, 0.19875481724739075, 0.223451629281044, -0.26997488737106323, -1.9794248342514038, -1.5207645893096924, -0.8700964450836182, -0.2892853915691376, -0.9482463598251343, -2.5824408531188965, 0.7510635852813721, -0.05254863575100899, 0.6355001330375671, -1.4614007472991943, -1.0552932024002075, -0.5168327689170837, 1.430922031402588, -0.12479018419981003, 0.049511782824993134, -0.6858425140380859, -0.4653501808643341, -1.247930884361267, 1.9136793613433838, 1.5375961065292358, -0.525058925151825, -0.8623875975608826, -2.0841994285583496, -0.2841307818889618, 1.007606029510498, 0.19742201268672943, 0.19030191004276276, -1.1421071290969849, -1.0771280527114868, -1.5942810773849487, -0.17845454812049866, 0.17275799810886383, -1.0965309143066406, 0.19995832443237305, -0.16279318928718567, 0.8519154787063599, 0.9501392245292664, 1.2540606260299683, 0.9981938600540161, 0.5189934968948364, 1.69961416721344, 1.3441952466964722, -0.5064857006072998, -0.2583281695842743, -2.7827601432800293, -0.639338493347168, -0.42909079790115356, 0.6970551609992981, -0.7209777235984802, -2.2547659873962402, 1.1535271406173706, 0.5951457023620605, 0.1183200404047966, -1.3563810586929321, 0.45941364765167236, -1.622960090637207, -0.6659412384033203, 0.024034269154071808, -0.3983023762702942, 0.29801908135414124, -2.0381429195404053, -0.48086580634117126, 1.1254502534866333, 1.161441445350647, -1.631450891494751, 0.006150469649583101, 0.5638052821159363, 0.23150262236595154, 0.7125227451324463, -2.608642816543579, -0.4420616328716278, -0.8416176438331604, 0.9570692777633667, -0.9502615332603455, -0.5102270841598511], [-1.2982231378555298, -0.9900697469711304, 0.7827579975128174, -0.15109296143054962, 1.0502307415008545, 1.3257039785385132, 1.9829375743865967, 0.34131693840026855, -0.5921612977981567, -0.9576100707054138, -1.3994557857513428, 0.07993979752063751, 0.1834583282470703, -0.40237465500831604, -0.6093796491622925, -1.660873532295227, -0.45596885681152344, -0.1635400354862213, 0.19679415225982666, -0.04960005357861519, -1.0866554975509644, -0.46456652879714966, -0.016596801578998566, 0.1571267545223236, 0.5044710040092468, 0.03144727647304535, 0.4972938895225525, -1.0865064859390259, -0.4091853201389313, -1.0246071815490723, -0.96406090259552, -1.3520174026489258, -0.2978077530860901, 0.8171247839927673, 1.2480984926223755, -1.8330607414245605, 2.4867193698883057, -0.7841284275054932, -1.9749517440795898, 0.9401133060455322, -1.7070690393447876, -0.7996205687522888, -2.4847021102905273, -0.10545781999826431, -0.578608512878418, 0.5298320651054382, -0.6973934173583984, -1.5835607051849365, 1.5858780145645142, -1.326721429824829, 1.6766812801361084, -1.2388572692871094, -0.29587581753730774, 0.16344328224658966, 0.45739471912384033, 0.8668060898780823, 1.2754524946212769, 1.434689998626709, 0.3835756182670593, -1.0053989887237549, -0.28753724694252014, 0.47045859694480896, -0.9602609872817993, 0.6588053107261658, 0.7343870997428894, -1.6657837629318237, 1.1533757448196411, -1.3442696332931519, 0.8977479338645935, 1.6261192560195923, -0.38762709498405457, -1.2328943014144897, -0.40161141753196716, 0.9062923789024353, 0.6566954851150513, -0.5020828247070312, -0.40361225605010986, -1.9824613332748413, -0.7338405847549438, 2.495373249053955, 1.2268242835998535, 2.3532278537750244, -0.9093971252441406, 0.9203450083732605, 0.7223081588745117, -1.1767053604125977, -1.4121768474578857, 0.004601611755788326, -0.6059492230415344, 0.7998008728027344, 0.29538124799728394, 0.7254581451416016, -0.7056890726089478, -0.7351387739181519, -0.74568110704422, -0.9698963761329651, -1.556695580482483, -1.1251721382141113, 0.6500334739685059, 2.2687666416168213, 0.683123767375946, 0.17225958406925201, 1.8198225498199463, 0.7840775847434998, -0.7176586985588074, 2.6990573406219482, -1.1385602951049805, 1.2449156045913696, -0.8667322993278503, 1.5874251127243042, -0.21662425994873047, 0.40567877888679504, 0.16612079739570618, -0.5985750555992126, 0.30284810066223145, -1.3887689113616943, -2.0687036514282227, -0.7616317272186279, 0.719168484210968, 1.308127522468567, 0.8673660159111023, 1.6745127439498901, 0.35288912057876587, 0.7820810675621033, 1.8122928142547607, 0.11946794390678406, 0.6951675415039062, -0.25743719935417175, 1.1226119995117188], [-0.7402028441429138, -0.9375749230384827, 0.123773492872715, -0.8292123079299927, -0.8728411197662354, -1.3797115087509155, 0.8609652519226074, 1.0018770694732666, 1.4594630002975464, -1.3185731172561646, -0.6351861357688904, 1.4006471633911133, -0.47328057885169983, -1.531920313835144, -0.19850684702396393, 1.0503900051116943, -0.39082327485084534, 0.6773822903633118, -1.0196409225463867, 0.0638200044631958, -0.20888543128967285, 0.7745059132575989, 0.9945470094680786, 0.22243840992450714, -0.638501763343811, -1.8961597681045532, -1.2074997425079346, 0.7026861310005188, 0.7612462043762207, 1.0400221347808838, -0.18071997165679932, 1.3994364738464355, 0.18298913538455963, -0.9264413118362427, -0.7569156289100647, 1.8025163412094116, 0.5379195809364319, 0.7489198446273804, -0.3527369499206543, -0.356372207403183, 1.4446508884429932, 0.12898564338684082, -1.2447036504745483, -0.714279294013977, -0.9461037516593933, 0.35330715775489807, -0.2457578033208847, -0.8456019759178162, -0.21585986018180847, -0.15942509472370148, -1.3853944540023804, -0.9746766090393066, -2.1280744075775146, 0.9948093891143799, 0.9097217917442322, -0.37299036979675293, 0.20520125329494476, 0.957146167755127, 0.5758368968963623, 2.5442943572998047, 2.2179818153381348, 0.13707448542118073, 0.28252726793289185, -0.8481671214103699, 1.0932923555374146, 0.19280102849006653, -0.4860287606716156, -0.004856566432863474, -0.8843992948532104, 0.16600799560546875, 1.6942553520202637, 0.03680787608027458, -0.10707516223192215, 0.8468098044395447, -0.01337269227951765, -1.259469985961914, -0.9454445242881775, 1.2310781478881836, -0.9101369976997375, 1.4284694194793701, -0.33403676748275757, 0.8854184746742249, -0.9821099638938904, -0.7266983389854431, 0.534260094165802, 0.7134975790977478, -0.7588427662849426, 0.6195409297943115, 0.6180381774902344, -0.8777148127555847, -1.7377629280090332, 0.7034488916397095, 0.9115410447120667, -1.3801745176315308, 0.5680091977119446, -0.3569091260433197, 1.4552021026611328, 0.9466157555580139, 1.7503693103790283, 0.09468778222799301, -1.1613130569458008, -0.7345200181007385, 0.3544921875, 0.8439797759056091, 1.3512154817581177, 0.08576333522796631, 0.14909756183624268, -1.3778046369552612, 1.6320470571517944, 0.23118549585342407, 0.8234440088272095, -1.803130030632019, 1.0549943447113037, 1.6768097877502441, 0.8771423101425171, -1.5483996868133545, 0.5002008080482483, -0.1946735829114914, -1.1266385316848755, -0.1731814444065094, 0.43110647797584534, 1.4662189483642578, 0.7718473076820374, -1.6515141725540161, -0.43965378403663635, -0.3250524699687958, -0.5580856800079346, -1.6855831146240234, -0.6556835770606995], [-2.347351551055908, 1.3251609802246094, 1.0856397151947021, -0.3166932761669159, 0.4424663484096527, 2.0469143390655518, 0.3778892159461975, 0.24974173307418823, -1.7441996335983276, -2.500549793243408, -0.594693660736084, 0.022056549787521362, 1.1382192373275757, 0.2537332773208618, 0.09528230130672455, 0.009908610954880714, -0.615466296672821, -0.8140292763710022, -1.8047105073928833, 0.9319881796836853, -0.8513878583908081, 0.45454299449920654, -1.0355751514434814, 0.2752959728240967, 1.4250223636627197, 0.2899932861328125, 0.8639974594116211, -1.6398253440856934, -0.6210325360298157, -1.178109884262085, -2.038466453552246, 0.10568065941333771, 2.033489465713501, 1.3218132257461548, -0.8551199436187744, 2.383113145828247, -2.319232225418091, 2.477334976196289, -0.48551639914512634, -0.09840262681245804, -1.3792189359664917, 1.30575692653656, 1.790935754776001, -0.21534442901611328, 0.20924216508865356, 0.9216720461845398, 0.14425329864025116, -0.11323164403438568, 0.3244415819644928, 1.6885735988616943, -1.8493428230285645, -0.9034428000450134, 0.5346775054931641, -0.31815123558044434, 0.7450074553489685, 0.34149786829948425, 0.40355822443962097, -0.014643870294094086, 0.3102804720401764, 0.913284957408905, 1.4958558082580566, -1.012591004371643, -0.45160576701164246, -0.12395371496677399, -0.6966358423233032, -1.3173351287841797, -0.9749839305877686, -0.5517587065696716, -0.7714519500732422, -1.1363388299942017, 2.018315076828003, -0.7604661583900452, -0.06896483153104782, -0.010375648736953735, 0.3786119818687439, 0.49744316935539246, 1.0726752281188965, 1.6157938241958618, 0.4202803075313568, -0.011499178595840931, 2.3072493076324463, -0.22978202998638153, -1.4429094791412354, -0.2948983907699585, 1.4342808723449707, 0.06008213385939598, 1.5791093111038208, -0.38217851519584656, -0.800744891166687, -1.6696648597717285, -0.4738486111164093, -0.31747063994407654, 0.5184289813041687, -0.11308880150318146, -1.3838560581207275, 0.679990828037262, 0.0880722850561142, -0.5637345314025879, -0.32231780886650085, 0.547514796257019, -1.5371975898742676, 0.727321982383728, -0.19308380782604218, 0.24938704073429108, -0.8766193985939026, 1.503588080406189, 1.0960137844085693, -0.988540530204773, -1.0556036233901978, 2.140454053878784, 1.5100411176681519, 0.6056551337242126, -0.37891533970832825, -1.3601276874542236, -1.2072010040283203, 1.0618088245391846, -0.2754959464073181, -0.7907466888427734, -0.5852039456367493, 1.138834834098816, 0.2505398988723755, -1.4766267538070679, -0.161950021982193, 0.1917741596698761, 1.1337802410125732, -1.2090545892715454, 0.8686115145683289, -0.5899166464805603, 0.11977028101682663], [0.5643442869186401, -0.4333064556121826, -0.8253152966499329, -1.7195932865142822, 0.8296968936920166, 0.6647970080375671, -1.0556443929672241, -0.08202970772981644, -1.4086543321609497, -0.8899011611938477, 1.1173343658447266, 0.47749388217926025, -0.8823668360710144, 0.4865887463092804, 1.880150318145752, 1.6496002674102783, 0.3343241214752197, 0.6183858513832092, 0.03430530056357384, 0.06159398704767227, 2.4536805152893066, 0.026959365233778954, 1.4160596132278442, -1.062239646911621, 0.45925119519233704, -1.0244561433792114, 0.7413032650947571, 2.78151273727417, 0.9203993082046509, 1.5992120504379272, 1.6452224254608154, 0.6025769710540771, 0.5976683497428894, -0.3194137215614319, 1.2055025100708008, 0.11850079894065857, 0.8012005686759949, -0.9098467230796814, -0.6179670691490173, -1.870046854019165, 2.646214008331299, -0.7877269983291626, -2.5917534828186035, 0.4260387122631073, 1.695267677307129, 0.41653144359588623, 0.6012580394744873, -0.17908155918121338, 1.7520670890808105, 1.4263938665390015, 0.3294278085231781, 1.58212149143219, -1.096921682357788, -1.056094765663147, 1.5674687623977661, -1.5090351104736328, 1.0431190729141235, 0.030965734273195267, 0.31361594796180725, 0.14665701985359192, 1.7300233840942383, 0.7176420092582703, -0.8383113145828247, -0.8694906234741211, -1.4523444175720215, 0.19528894126415253, 0.28011223673820496, -1.080918312072754, -1.045153021812439, -0.9909616112709045, -0.11884108930826187, -1.4691979885101318, -0.5244183540344238, 0.489287793636322, 1.4678819179534912, -1.4156932830810547, -0.1482769101858139, -0.6297036409378052, 0.0832703560590744, 0.29339319467544556, -0.7539788484573364, 1.1071741580963135, -1.4012560844421387, -1.1170988082885742, 0.9138510227203369, -1.218280553817749, 0.154056116938591, -0.4943098723888397, -0.44959595799446106, -0.7158993482589722, -1.7299892902374268, -0.061709627509117126, 1.1618205308914185, -0.3417239487171173, 0.019364967942237854, 2.7293314933776855, 0.5898537635803223, -1.708587884902954, -1.4324235916137695, -0.01734878495335579, -0.076043039560318, -1.3189984560012817, 0.09519191831350327, 0.09928129613399506, 0.4358055889606476, -0.3238834738731384, -1.6210310459136963, -0.662776529788971, -0.5058761239051819, 1.8447171449661255, -0.47558701038360596, 0.023790309205651283, -0.9678044319152832, -0.9313607811927795, -0.6007207632064819, -0.7288613319396973, 1.9001431465148926, -0.9693366885185242, 0.06167115271091461, -0.27326592803001404, -0.9970569014549255, -1.6821788549423218, -0.34253811836242676, -1.3840279579162598, -0.6591100692749023, 0.1632077693939209, 0.14372506737709045, 2.0500104427337646, 0.0701553225517273], [-0.21198607981204987, -1.080263614654541, -0.629267156124115, 1.5126073360443115, -1.6261972188949585, -0.23883964121341705, 0.42561110854148865, 0.2583233118057251, 0.692283570766449, 0.4069148898124695, 1.0901713371276855, -0.52623051404953, -1.7351269721984863, -0.09677576273679733, 1.0738539695739746, 1.1463783979415894, -0.3455648124217987, 1.9105224609375, 1.7386200428009033, 0.16609227657318115, -0.48079532384872437, 0.6220439076423645, -0.6521722078323364, 0.16389252245426178, 0.1717091202735901, 0.11804967373609543, 0.35352200269699097, 1.3278783559799194, -0.21300888061523438, 0.05794788524508476, -0.13067469000816345, -0.11606194078922272, -0.08038005232810974, 1.0025708675384521, 1.0724328756332397, 0.262024462223053, 0.39440464973449707, -0.17331553995609283, -0.26299014687538147, -0.1260920912027359, 1.2048357725143433, -0.7057826519012451, -1.3136389255523682, -1.804396629333496, 1.1734449863433838, 0.03027881309390068, -0.5789453387260437, 0.41342660784721375, 2.0346312522888184, -0.2753782570362091, -0.6331276893615723, -0.11849721521139145, 0.3339543342590332, -0.5935093760490417, 0.15020716190338135, -0.9540145993232727, -1.107100486755371, 1.9120560884475708, -0.8893743753433228, 1.6169154644012451, 1.4624054431915283, -3.654334545135498, -0.1889166533946991, -0.43396276235580444, 0.8027573227882385, 0.04369085654616356, 0.63036048412323, 0.7565122246742249, -0.7689397931098938, 0.7454649806022644, -1.8837884664535522, 1.017972469329834, 1.4246443510055542, 1.2667503356933594, -1.2847784757614136, 0.5738223791122437, -0.145847886800766, -1.193544864654541, -0.5772522687911987, -0.36908426880836487, 1.1118258237838745, 0.45566877722740173, -0.8798270225524902, -0.6424769163131714, 0.6176947355270386, 1.1502327919006348, -0.6184029579162598, -0.6496730446815491, -0.05551043897867203, -0.09250922501087189, -0.207641139626503, -0.4712055027484894, 0.6298806667327881, 0.2719707787036896, 1.0485671758651733, 0.7826606035232544, -1.5471910238265991, -0.018886573612689972, -0.7092488408088684, 0.04059014841914177, 1.6932812929153442, -0.8201895356178284, 0.3229946196079254, 0.1909627765417099, 1.0680840015411377, 0.5161378383636475, -0.5915219783782959, -0.5864126086235046, -0.6189871430397034, 1.3196051120758057, -1.704413652420044, 1.2537264823913574, 0.02689482644200325, 0.0547124408185482, -0.7363112568855286, 1.0555070638656616, -0.36952754855155945, -0.43836748600006104, -0.7060750126838684, -0.5493060350418091, -0.0018676409963518381, 0.1255558729171753, 0.9594805240631104, 0.10388477146625519, -0.23891352117061615, 0.09249286353588104, 0.6598992347717285, 0.15216179192066193, -0.5372763276100159], [0.07562772929668427, -1.1159262657165527, -0.15499697625637054, 0.3577394485473633, -0.1878383904695511, 0.5573351383209229, 0.1420096904039383, -0.19625596702098846, -1.9139386415481567, -0.8074736595153809, 0.05620888993144035, 0.2573137581348419, -1.2023289203643799, 1.9591625928878784, 0.9738208651542664, 1.1371526718139648, -0.20831437408924103, 0.7746308445930481, -0.13637717068195343, -1.8593028783798218, 0.503289520740509, 0.02560165897011757, 0.3698224723339081, 0.14013507962226868, 0.8073630928993225, -0.7096475958824158, 0.14977441728115082, -0.5986460447311401, -1.3236206769943237, -0.5886672735214233, -1.0249396562576294, 1.0105518102645874, 0.5514040589332581, 0.05146767571568489, 0.5956058502197266, -0.5706434845924377, 0.8143318891525269, 0.8418775796890259, 1.5295844078063965, -0.16429948806762695, -3.1252264976501465, 0.13754743337631226, 0.003757214406505227, -0.49966874718666077, -0.3489391505718231, -0.7260094881057739, 1.9193215370178223, -2.043509006500244, 0.5214387774467468, -0.9602575302124023, 0.5341887474060059, -0.6191895604133606, -0.5069474577903748, -1.414661169052124, 0.5238215923309326, -0.47819095849990845, -0.023024411872029305, 0.662915825843811, -0.30735844373703003, -0.6344882249832153, -1.124504804611206, 2.3032236099243164, -0.13477948307991028, 1.8807902336120605, 1.0913254022598267, 0.757635772228241, -0.42082446813583374, 0.3054860234260559, -1.698777675628662, -1.1337531805038452, -0.8514977693557739, -0.9267565608024597, 0.026340441778302193, 0.8392915725708008, -0.7051826119422913, -0.6142091751098633, -0.04824351891875267, -0.8258432149887085, 1.7252206802368164, -1.4581388235092163, -0.7173022031784058, 0.03428707644343376, -0.7036524415016174, -1.2622555494308472, -0.47028830647468567, 0.6877319812774658, 0.3968704640865326, 0.5835765600204468, -0.28446832299232483, -0.24609677493572235, -0.47749972343444824, 1.5484309196472168, 0.5555745363235474, 0.5881668925285339, 0.5606383681297302, -0.7955772876739502, 1.5853897333145142, -1.319846510887146, 0.632972240447998, 0.47514021396636963, 0.9098061919212341, 1.3181838989257812, -0.951172947883606, -0.2833895981311798, 0.17287345230579376, -1.075697898864746, 0.44228094816207886, 0.21177822351455688, 0.05645501986145973, 1.0793743133544922, 0.20735114812850952, 0.655012845993042, -0.28141218423843384, 1.2317042350769043, -0.6830506920814514, 0.3267629146575928, -1.420720100402832, 1.701197624206543, 1.1255691051483154, -1.4508024454116821, -0.300367146730423, -0.6866655945777893, 0.19373564422130585, -0.7087470889091492, 0.6593629717826843, 0.8974704146385193, 0.4657497704029083, -1.6342127323150635, -0.26375171542167664], [-1.230687141418457, -1.5305757522583008, -0.9903956651687622, 0.36747536063194275, -0.3795149326324463, 1.4826900959014893, -0.23414523899555206, -0.23830364644527435, 0.7366500496864319, -1.8231536149978638, 1.162632703781128, 0.3293362855911255, -1.1858702898025513, -0.34690022468566895, -0.03706054016947746, 1.4274479150772095, -2.220726251602173, 1.7746378183364868, -0.4633732736110687, -2.2713165283203125, 0.6383537650108337, 0.5254559516906738, 1.6509054899215698, 0.9371280074119568, 1.224137783050537, 0.6928437948226929, 0.37427595257759094, 0.3424653708934784, 1.9632004499435425, -2.07370662689209, -0.7914064526557922, 0.6841868758201599, -2.1846539974212646, 0.8603017330169678, -0.0959383100271225, -0.2943662405014038, -0.07604021579027176, -1.550616979598999, -0.4872330129146576, -0.16817477345466614, 0.9857761859893799, 0.1917528659105301, 0.3036683201789856, -0.6324155926704407, -0.3200398087501526, 1.2324204444885254, -1.572762131690979, -1.3681890964508057, 0.47984591126441956, 0.3635439872741699, -0.7379550337791443, 0.05422566086053848, -1.114234209060669, -1.1001629829406738, -1.0492901802062988, -0.566957950592041, 1.6432223320007324, -1.761748194694519, 1.1176754236221313, 1.0456136465072632, 1.050173044204712, 0.4176521301269531, 0.7190442681312561, 2.0064897537231445, 0.3819010555744171, 0.5951785445213318, 0.4580513834953308, 0.3305240869522095, 2.165982723236084, 1.1699167490005493, -2.7683873176574707, 0.700519859790802, -0.2346370816230774, 0.45491883158683777, 0.8151008486747742, -0.4805840849876404, -0.6640876531600952, 0.7855082154273987, 1.255042552947998, 0.41860973834991455, 2.3593742847442627, -0.9082339406013489, -0.5460793972015381, -1.1260675191879272, -0.13694557547569275, -0.0913538783788681, 0.044331710785627365, -0.6560739278793335, 0.3678814470767975, -0.3598890006542206, 0.3215152323246002, -0.034826815128326416, -1.0987275838851929, 1.0334571599960327, -1.2476801872253418, 0.220219686627388, 0.4776807129383087, 1.1587989330291748, -0.0076142107136547565, -1.5298148393630981, 1.1393647193908691, -1.3761773109436035, -0.022249722853302956, -0.08831895887851715, 1.0673575401306152, -0.3621360957622528, 0.7709683775901794, 1.2751197814941406, 0.24843545258045197, 2.4403774738311768, -1.0485951900482178, -0.4316455125808716, -1.3818418979644775, 0.39360833168029785, -0.3109665811061859, 1.5539674758911133, -1.455186128616333, -1.1222072839736938, -0.501853883266449, -0.658056378364563, -1.024553894996643, 1.4570019245147705, 0.8089599609375, 0.1334603726863861, 0.43090546131134033, 0.9021539092063904, 1.4782899618148804, -2.098743200302124, -1.7704704999923706], [0.05851586535573006, -0.7200953364372253, 0.7624605894088745, 0.6131711602210999, 0.8255956172943115, 1.7864489555358887, 0.6972428560256958, 0.4515397846698761, -0.06303201615810394, -0.12570694088935852, -2.272993564605713, -1.0692907571792603, -0.34339234232902527, 0.2669438123703003, 0.2614902853965759, -0.4449339509010315, -0.6776629686355591, 0.13569796085357666, -1.8236976861953735, 0.39678657054901123, -0.31996747851371765, 0.767653226852417, -0.1536213457584381, 1.7962456941604614, -1.7127699851989746, 0.394419401884079, 0.42040374875068665, -0.706151008605957, -0.061417851597070694, 0.9566283226013184, -1.4519580602645874, -1.9090656042099, -0.8287789225578308, 0.3917554020881653, -1.5766186714172363, 1.0924043655395508, 0.04053713008761406, 2.4175169467926025, 0.7086307406425476, 1.6313085556030273, 0.34006038308143616, -1.4678133726119995, -1.1427785158157349, 0.3229626715183258, 0.5819849967956543, -0.15150704979896545, -1.4907094240188599, -0.5770155191421509, 0.45157909393310547, 1.012001395225525, 0.6276417970657349, 0.6000660061836243, -1.119439959526062, -2.06620717048645, -1.469773769378662, -1.3736627101898193, -0.747002363204956, -1.7055495977401733, -0.27140292525291443, 1.2772241830825806, -0.4009550213813782, 0.07669690251350403, -0.2978653609752655, 0.10348659753799438, -1.0256867408752441, -0.9391708374023438, 0.8816319108009338, 2.3186728954315186, -0.5688369274139404, -0.4609416723251343, 0.007935665547847748, -1.1863034963607788, -0.10189361125230789, 0.24955613911151886, 1.557469129562378, 0.34632357954978943, -0.4912782609462738, 0.12813158333301544, 0.26491355895996094, 0.4541303515434265, 0.31976717710494995, 0.45725029706954956, -0.14667503535747528, -1.383285641670227, 0.05737588182091713, -2.5258002281188965, 0.08299709856510162, -0.6578260660171509, 0.3987676501274109, 0.8054530620574951, -1.6320900917053223, 0.28722530603408813, -1.2925328016281128, 1.4882093667984009, 0.37616026401519775, 2.195810556411743, -0.785645067691803, 0.2557108998298645, 0.26786962151527405, 0.6347501873970032, -0.5341073274612427, 0.7358734011650085, 1.8667936325073242, -0.3895840346813202, 0.5518529415130615, -0.024921834468841553, -1.5939829349517822, 0.37677904963493347, 0.25804486870765686, 0.8170871734619141, -1.2634118795394897, 0.06096139922738075, 0.16333073377609253, -0.5554659962654114, 1.010794758796692, -2.119234323501587, 0.6304668188095093, 0.324491947889328, -0.30879634618759155, 0.2985565662384033, -0.6108203530311584, 1.0358679294586182, -2.0506954193115234, 0.36862942576408386, 0.5086244940757751, -1.9135781526565552, -0.6242296695709229, 0.04928145557641983, 0.4582476317882538], [0.6622054576873779, -0.5116298794746399, 0.20015083253383636, 1.9557948112487793, 1.0871131420135498, -1.7923991680145264, -0.10432702302932739, -2.1612377166748047, -0.01493560429662466, 1.1233495473861694, 0.3933328688144684, -0.5736078023910522, -1.3773316144943237, 2.2630629539489746, 0.3020225167274475, -1.8432568311691284, 0.3490811884403229, 0.34714847803115845, -0.27680617570877075, -0.3842966556549072, 0.49962878227233887, 0.728483259677887, 0.5929016470909119, 0.907576858997345, -1.8689438104629517, 1.6661534309387207, 0.9687718152999878, 0.9483145475387573, 0.019206145778298378, 1.2270163297653198, 0.3163454830646515, 0.18041189014911652, -1.040027141571045, 0.2602497935295105, -0.06438075751066208, -2.6208064556121826, 0.27964383363723755, -0.6938385963439941, 0.771955132484436, -1.7779790163040161, -0.5015948414802551, -0.24401289224624634, 0.03830970078706741, 0.30082857608795166, -0.8352450728416443, -0.06804893165826797, -0.804915189743042, -1.360314130783081, 0.4489187002182007, 0.4308781027793884, -1.0817162990570068, 1.4576598405838013, -0.4907359182834625, 0.19463233649730682, -0.11455290019512177, -0.3153632879257202, -0.31411269307136536, 0.15224064886569977, 1.632590889930725, 0.3982935845851898, 0.5000444054603577, 0.3993080258369446, -0.5423200130462646, 0.3721711039543152, -1.6688826084136963, -0.6093178987503052, -1.964407205581665, 0.8404337763786316, -0.45916587114334106, 0.43524712324142456, 1.356305480003357, -0.5497546792030334, -1.2928401231765747, -0.4333941638469696, -0.10444848984479904, 0.46962428092956543, 1.7285066843032837, 0.989719808101654, -1.0283639430999756, -0.7228518128395081, -1.8439010381698608, -1.105217695236206, 0.26943376660346985, -0.8449047803878784, -1.696154236793518, 1.0207157135009766, -0.5311293601989746, -1.1426879167556763, 1.0922889709472656, -0.47380346059799194, 2.485004186630249, 0.7017367482185364, 0.2835056483745575, 0.17279835045337677, -0.55019611120224, 0.5586434602737427, -0.9382592439651489, -1.2823584079742432, -0.03219645097851753, 1.1833409070968628, -0.19141678512096405, 0.12532983720302582, -0.07407571375370026, -0.030812906101346016, 1.201258897781372, -1.7711342573165894, -3.365313768386841, 2.9066312313079834, 0.5725749135017395, 0.17304174602031708, -0.12538452446460724, 0.34063225984573364, -0.3328127861022949, 1.1608591079711914, -0.1312956064939499, 0.5079006552696228, 0.2938551902770996, -0.010499007999897003, 0.26430588960647583, -0.6571608185768127, 0.7017011642456055, -1.4841941595077515, 0.006467194762080908, 1.08559250831604, -0.5059903860092163, 0.15867049992084503, 0.2817445993423462, -1.8242501020431519, 0.8331071138381958], [0.706614077091217, -0.14906221628189087, 0.0007990558515302837, -0.9066634774208069, -1.1467080116271973, -1.44857656955719, 0.9283943176269531, -1.2952015399932861, 0.13152290880680084, 0.39764708280563354, 0.9689393639564514, -0.914897620677948, -0.19456292688846588, 0.6872125864028931, -2.061361789703369, -0.09614607691764832, -0.8771637082099915, -1.0315196514129639, -1.069583535194397, 0.9849862456321716, -1.505967140197754, 1.0836395025253296, -0.009853270836174488, -1.282610297203064, 1.2395055294036865, 1.3509888648986816, -0.12389107048511505, -0.6805802583694458, -0.6263591051101685, -0.2415435016155243, 2.0992610454559326, -0.1650664508342743, -1.3264681100845337, -1.1265785694122314, 0.400825172662735, -0.6202031970024109, -0.8549563884735107, 1.5440928936004639, 0.7475293278694153, -1.961897373199463, -0.39328521490097046, 0.848343014717102, 0.1018570140004158, 0.3285543620586395, 1.6879298686981201, -1.147457480430603, 1.8768858909606934, -1.6256263256072998, 0.5734531283378601, -0.9131581783294678, 0.20640411972999573, 2.605194568634033, -2.0804975032806396, 0.5093300342559814, -0.41273489594459534, -2.738490104675293, 0.37491559982299805, 0.36399024724960327, 0.5137808918952942, 1.3291887044906616, -1.400713324546814, 0.9947199821472168, -0.24705271422863007, 2.043790340423584, -0.1949504315853119, -0.464981347322464, 0.645708441734314, -0.6462641954421997, -2.0070855617523193, 0.7802315354347229, -0.1591368168592453, -0.6084201335906982, 1.403092384338379, -0.5522133708000183, -1.3590775728225708, -0.1825074404478073, -1.1685441732406616, -0.011599247343838215, -0.5285700559616089, 1.8665179014205933, -0.4649907350540161, -1.8708586692810059, -2.3470051288604736, 0.8547887802124023, 0.5824218988418579, 2.540212631225586, -0.9838818907737732, -0.23692192137241364, 1.2738211154937744, 0.22858871519565582, 1.4662026166915894, 1.4519376754760742, -0.7326679229736328, -2.1428589820861816, -0.5973864793777466, -0.22156259417533875, 0.5266990661621094, 0.7028701305389404, 0.35028913617134094, 0.3446844816207886, -0.33538851141929626, -0.45914191007614136, 0.47876349091529846, -0.7313007116317749, -0.7554642558097839, -1.3348544836044312, -1.0529179573059082, -1.458817720413208, 0.561844527721405, -0.4632710814476013, 0.22004230320453644, 0.18155519664287567, 1.087504506111145, 0.20183655619621277, -0.22054335474967957, 0.4630659222602844, -0.24727056920528412, 0.11965315788984299, -1.2409451007843018, 0.795453667640686, 1.0960112810134888, 1.8615410327911377, -0.251020222902298, 0.9931698441505432, -0.5645847320556641, -1.1991060972213745, 1.0259119272232056, -1.105695128440857, 0.30962419509887695], [-0.2913486659526825, -1.2328256368637085, 0.11600875854492188, 0.18088380992412567, 0.8871568441390991, -2.476884365081787, 0.10175830125808716, 1.3265407085418701, -0.7614359855651855, -0.18657058477401733, -1.1308658123016357, 1.6219682693481445, 1.2253731489181519, -2.948002815246582, 0.783153772354126, -0.6821920871734619, -1.5872050523757935, 1.6062002182006836, -0.5563532114028931, -0.2533571422100067, -0.7012293934822083, 0.20057648420333862, -1.5357282161712646, 0.8761130571365356, 0.2409093976020813, -0.40943509340286255, -0.8694388270378113, -0.8108084201812744, 1.8346867561340332, 0.20094050467014313, -0.30077263712882996, 0.15873609483242035, 0.7461795806884766, -1.1535484790802002, -0.8239145278930664, 0.8078416585922241, -0.6572535634040833, 0.45360448956489563, -0.9909483790397644, 0.09343942999839783, 0.8417022824287415, 0.42923665046691895, 0.6351965069770813, -0.02733239345252514, -0.7386745810508728, 1.1139636039733887, -0.4435640275478363, -0.7053020000457764, 0.749478816986084, -1.1117234230041504, 1.9370086193084717, -0.6624596118927002, 0.9850413203239441, 1.237006425857544, 1.106574296951294, 0.658150315284729, -0.7138460874557495, 2.7628607749938965, 0.47927403450012207, -0.8669674396514893, 0.001956970663741231, -1.1525858640670776, -1.3728755712509155, -0.2357218712568283, -0.2491753101348877, -2.241975784301758, 0.7104656100273132, 1.0664509534835815, 1.4179311990737915, -0.7446052432060242, 1.2812644243240356, -0.21194244921207428, 1.3301280736923218, 2.0092270374298096, 0.2613750100135803, 0.9434076547622681, 1.5227694511413574, -0.7211422324180603, -1.5570217370986938, 0.6244490146636963, 0.12171262502670288, 0.38203999400138855, -0.7901613712310791, 0.18380290269851685, -1.8754173517227173, 2.1185882091522217, 0.5614583492279053, 0.5750530362129211, 0.8013557195663452, 0.20663417875766754, 0.2728949189186096, 0.35746368765830994, -0.16903801262378693, 0.7294357419013977, 0.04220008850097656, 0.15046726167201996, -0.20091919600963593, 0.15105468034744263, 0.3199829161167145, 0.9733994603157043, -0.6627148389816284, 0.6084312796592712, -1.7357001304626465, 0.19117531180381775, -1.2082974910736084, -0.05083707347512245, 1.2393602132797241, 1.6671509742736816, 1.2410380840301514, -0.7333144545555115, 0.9984138011932373, -0.5099005103111267, -1.390941858291626, 0.8948430418968201, -0.870768129825592, 0.18576261401176453, -0.6746326088905334, 0.7238292694091797, -0.7421215176582336, -0.7823790907859802, 1.9707356691360474, 1.2838499546051025, 0.06636808067560196, -1.6385587453842163, -0.66474848985672, -1.180757761001587, 1.965822696685791, 0.8695624470710754, -0.5250752568244934], [0.5136865973472595, 1.5985037088394165, -0.6651182174682617, 1.1174815893173218, 1.4685801267623901, 0.8405603170394897, 1.2474496364593506, 0.7671910524368286, -0.11955884844064713, -0.10195314139127731, -0.5650558471679688, -1.260831356048584, 0.09947764128446579, -1.045420527458191, 1.7747684717178345, -0.455735981464386, -0.0820079818367958, 1.1520668268203735, 1.237196922302246, 0.9327325820922852, 0.004159760661423206, 0.4297010600566864, 1.3587491512298584, -1.0834625959396362, -0.950747013092041, 0.9344233274459839, -0.5697512626647949, 1.4904834032058716, -0.24662338197231293, 1.4066877365112305, 0.675294041633606, 0.32035425305366516, 0.7079713344573975, -0.2935621738433838, -0.9995477795600891, -0.20075303316116333, 0.49430182576179504, -0.19909054040908813, -0.2341533899307251, 0.8452143669128418, -0.2646733224391937, 1.8210761547088623, 2.0524330139160156, 0.900023877620697, -0.008854924701154232, 1.0634434223175049, -1.1572147607803345, 0.3843876123428345, 0.6071394085884094, 0.4468438923358917, -0.3187241852283478, 0.8757373094558716, -0.014248073101043701, 0.5943650603294373, 2.1326076984405518, 1.5427533388137817, 0.28034985065460205, -0.21147014200687408, -0.5276322960853577, 0.06674441695213318, 2.116659641265869, 1.1898603439331055, 1.1096104383468628, 0.07476145774126053, 1.6441878080368042, -0.5806038975715637, 0.10892044752836227, 0.0033366919960826635, 0.019976714625954628, 0.6993026733398438, 2.4407958984375, -1.754564881324768, -0.5396796464920044, 0.7220132946968079, -2.410869836807251, -0.9876823425292969, -0.2341149002313614, -0.4906112849712372, 0.1688334345817566, 1.653764247894287, -0.05647772550582886, 0.5347372889518738, -1.0566924810409546, 1.1709517240524292, -0.28793197870254517, -0.761002242565155, 0.7381833791732788, -1.7094480991363525, 0.0313502699136734, 1.062274694442749, -0.5613639950752258, -0.01397679839283228, 0.4468332529067993, -1.2357819080352783, 0.634246289730072, -1.3660378456115723, -0.8767610788345337, -0.6915218234062195, -0.5696983337402344, 0.5693368911743164, 1.1660337448120117, -0.2580488920211792, 0.6345683932304382, -0.3327224850654602, 0.1545877307653427, -0.11804499477148056, 0.99346923828125, 1.8593209981918335, 1.463602066040039, -0.4055039584636688, 0.7589492201805115, -0.7576766014099121, 0.6959670186042786, 1.4370946884155273, -1.2136605978012085, -0.34727707505226135, -1.5727264881134033, 0.8341544270515442, -0.030353009700775146, 0.4283893406391144, -0.5221997499465942, 0.7202970385551453, -0.6579747200012207, -0.7313382029533386, -1.5730607509613037, -0.22660858929157257, -0.9508872628211975, 0.6434693932533264, 0.30697163939476013], [-1.0350395441055298, 0.5251994132995605, -0.4827364981174469, -0.28509485721588135, 1.82557213306427, 1.389489769935608, -0.48344480991363525, 1.0791233777999878, 0.19992144405841827, -0.7134413123130798, 0.3898169696331024, 0.7135125398635864, 0.2040611207485199, 0.6606472730636597, 0.48002514243125916, 0.4831971228122711, 0.0836872085928917, 0.6236407160758972, 0.7076200842857361, 0.9776037931442261, 1.896236777305603, 0.7783964276313782, 1.7450047731399536, 0.027799192816019058, -0.23471790552139282, 0.4606827199459076, 0.007962062954902649, -0.2969057261943817, -1.456018328666687, -0.9185502529144287, -0.7089805603027344, -0.926115870475769, 0.7848734855651855, -2.403203248977661, 1.0547784566879272, 0.10233809053897858, 0.11459972709417343, -0.06773131340742111, 0.2901889681816101, -0.09870106726884842, -0.5698598027229309, 0.08429030328989029, 0.2772497534751892, 0.757556676864624, 1.4169174432754517, 1.1005924940109253, 0.2873161733150482, -0.26907917857170105, -1.1172783374786377, 0.9620614051818848, 0.976071834564209, -0.8942829966545105, -1.0132447481155396, -0.13042804598808289, 0.8155759572982788, 0.3923284709453583, -0.0008394612232223153, -0.6307276487350464, -1.1782647371292114, 0.1387639343738556, 0.2992725372314453, -0.20170412957668304, -0.25055524706840515, -2.0850048065185547, -1.045646071434021, -0.9493669271469116, -0.4716760814189911, 0.004011800512671471, 1.1090974807739258, -1.7512900829315186, -0.4113289415836334, -0.36700132489204407, 0.5480331778526306, 0.34917423129081726, -1.0864499807357788, -0.349147230386734, 0.3970858156681061, -0.43251195549964905, -0.10196974128484726, -0.06928344815969467, -0.5436312556266785, 0.3233284652233124, 1.2857491970062256, -0.6108642816543579, 0.35468801856040955, 1.6604194641113281, -0.06341345608234406, -0.34389591217041016, -0.12956075370311737, 1.339117169380188, 1.5651342868804932, -0.4760383367538452, -0.9967323541641235, -0.6316835880279541, 0.0843220204114914, -0.628685474395752, -0.26697272062301636, 1.046756386756897, -0.0023659891448915005, 0.4054366648197174, 0.4084187150001526, 0.44697368144989014, 0.3198738098144531, 0.18624581396579742, 0.3290150761604309, 0.33608362078666687, -0.5120386481285095, -0.059496596455574036, 0.5331565737724304, -0.3423103988170624, -1.3105069398880005, -0.7382206916809082, -0.27680808305740356, -0.9483623504638672, -1.4801801443099976, 0.8457183241844177, 0.30457019805908203, -0.7894953489303589, 0.10786344856023788, 0.6970884203910828, -0.14392520487308502, -1.0192315578460693, -0.8242146968841553, 0.7857455015182495, 0.31903088092803955, -0.13107505440711975, 0.17908930778503418, -1.4115642309188843, -0.231667622923851], [0.29545310139656067, 1.4786772727966309, 0.9822962284088135, 0.5540544986724854, -0.7110983729362488, -0.7466980218887329, -1.3789371252059937, 0.26154825091362, -0.9817718267440796, -0.9233635663986206, -0.5440648794174194, 0.6338315606117249, -1.0299122333526611, 0.5796487331390381, 0.9157742857933044, -0.3212708830833435, -0.18321417272090912, -0.5154844522476196, 2.163785934448242, 0.03261168301105499, -0.28430959582328796, 0.27814608812332153, 0.6594815850257874, 0.29446664452552795, -0.4728638827800751, 1.2119066715240479, -1.1255543231964111, 0.5620995163917542, -0.2923940122127533, -0.9506609439849854, -1.2623775005340576, 1.5341602563858032, 1.7214372158050537, 1.6054599285125732, -1.4387569427490234, -0.36817196011543274, -0.5912768840789795, -0.5809956192970276, -0.5920873880386353, 1.2092639207839966, -0.1713525354862213, 1.0282613039016724, -0.8068485260009766, 0.354854941368103, 1.5628471374511719, 0.6249343752861023, -0.946155309677124, -0.6486680507659912, 1.1241992712020874, 0.10710630565881729, 0.3606756925582886, 0.4018159508705139, -0.1577194482088089, 0.36335140466690063, -1.5463486909866333, 1.2025353908538818, 0.32175031304359436, 0.9916079640388489, -0.3480929732322693, 0.3176553547382355, -0.5266703367233276, -0.2501697242259979, -1.476051688194275, 0.493070125579834, 0.0622321292757988, -1.7455910444259644, 0.2892756164073944, 0.38285109400749207, -0.7628282904624939, -1.0068907737731934, -0.8850975632667542, -0.44401928782463074, -1.2081319093704224, -1.1486377716064453, -0.6238327026367188, -1.1715788841247559, 1.1832339763641357, 0.14705847203731537, 1.1550929546356201, 0.7464303970336914, -1.5207209587097168, -2.3147666454315186, -1.1855345964431763, -0.7333151698112488, 0.6389243602752686, -1.4937798976898193, 0.5621715188026428, 1.3609832525253296, -0.44623690843582153, 2.7069075107574463, 1.1962172985076904, 0.5511477589607239, 0.3721374571323395, -0.5169884562492371, 0.014742032624781132, 0.6813649535179138, 0.19907577335834503, -1.0127880573272705, 0.08108682185411453, 1.6036876440048218, -1.3203184604644775, 0.46320605278015137, 3.870405912399292, -1.20477294921875, -0.4982534945011139, 0.21678484976291656, 1.269914984703064, -0.09242726862430573, -1.2557305097579956, -0.08303319662809372, -0.24244624376296997, 1.7794759273529053, -0.24382349848747253, -0.422728568315506, 0.10245916247367859, -2.486711025238037, 5.9690437410608865e-06, 0.5590862035751343, 0.19069983065128326, 1.3737486600875854, 0.5658574104309082, -0.2868093252182007, -2.0258593559265137, 1.5698829889297485, 0.5338631868362427, 0.15657994151115417, -1.173546314239502, 1.0984501838684082, -0.22608502209186554]], "dtype": "float32"}, 32, 32, 32, 4], "e": {"__tup__": [{"__nd__": [[0, 4, 8, 12, 16], [1, 5, 9, 13, 17], [2, 6, 10, 14, 18], [3, 7, 11, 15, 19], [20, 21, 22, 23, 24]], "dtype": "int32"}, {"__nd__": [[3.102915048599243, 0.9498447775840759, -4.143826007843018, 7.7992844581604, -3.770907402038574, 1.3529865741729736, 18.310606002807617, -2.993445634841919, -1.0871531963348389, 12.02271842956543, -2.0380003452301025, 8.708945274353027, 7.154518127441406, 6.620322227478027, -3.231241226196289, -7.88021183013916, 5.559989929199219, 2.337207555770874, -6.63054084777832, 15.193706512451172, -7.52539587020874, 6.7758378982543945, -5.058922290802002, -11.41577434539795, -12.903090476989746, -3.5574252605438232, -8.299677848815918, -8.502945899963379, -1.5177161693572998, -0.684005856513977, -3.3629183769226074, -11.061094284057617, -4.950825214385986, -20.56419563293457, -4.07680606842041, -10.354004859924316, 5.9958038330078125, 4.717878341674805, -9.936635971069336, 0.9901384711265564, -14.306406021118164, 1.7142046689987183, 3.6168720722198486, -7.194216251373291, -16.97740364074707, -0.15258444845676422, 9.245397567749023, 6.188127517700195, 3.211341619491577, -10.389108657836914, 17.67461395263672, -19.502548217773438, -1.6402840614318848, -0.2694782018661499, 11.81297779083252, -4.758551597595215, -9.82457160949707, 14.916242599487305, -3.546623706817627, 2.28708553314209, -0.2560224235057831, -6.317954063415527, 0.26148393750190735, -9.07568359375, 4.000980854034424, 0.4923984706401825, -3.310481309890747, 0.4144010543823242, -14.704891204833984, 0.9514052867889404, 2.6808221340179443, -1.300245761871338, 15.835772514343262, -11.678019523620605, 12.464356422424316, 3.239952802658081, 9.933772087097168, 7.807294845581055, -5.88226842880249, 6.218191623687744, -0.5774359107017517, 1.4164345264434814, -1.9309148788452148, -5.739150047302246, 1.7572569847106934, 7.409146785736084, -3.1812310218811035, -1.5543216466903687, 4.252486228942871, -8.520630836486816, 2.702199935913086, 1.4792598485946655, 5.643813133239746, 2.444714307785034, 3.594724655151367, -8.635048866271973, 6.088909149169922, 1.4424158334732056, 2.9411420822143555, 1.2073591947555542, -14.50597858428955, -4.614567279815674, 2.6624579429626465, 10.324387550354004, -10.852911949157715, 12.623172760009766, -1.345970630645752, 2.9492597579956055, -3.866234302520752, 2.6048221588134766, 7.446411609649658, 13.609889030456543, -2.6422548294067383, -4.92008638381958, -0.7312247157096863, 3.9826266765594482, -5.9461774826049805, -2.9609534740448, -7.283791542053223, 7.959743976593018, -1.067375659942627, 4.09189510345459, 1.994582176208496, 4.30946159362793, 13.404740333557129, -9.376810073852539, 0.22599586844444275, -3.0896494388580322, 0.8498803973197937], [3.3785808086395264, 3.0166194438934326, -9.436504364013672, 3.98907470703125, 1.224619746208191, 4.008171081542969, 7.312544345855713, -2.058269500732422, -2.028273582458496, 8.340851783752441, 11.26959228515625, 4.456932544708252, -2.098633289337158, -0.05293061211705208, -7.866263389587402, -0.5011659264564514, 2.2324650287628174, -2.2138583660125732, -3.8204665184020996, 20.295961380004883, 0.4786040484905243, 0.6156018972396851, -1.3633654117584229, -3.23647403717041, -1.9114375114440918, 1.7242282629013062, -2.5454370975494385, -13.120542526245117, 0.9762120842933655, -7.684813499450684, -7.924893379211426, -1.353530764579773, 3.090043306350708, -8.973405838012695, 0.15627531707286835, -2.266497850418091, 7.907390594482422, -12.273323059082031, 7.295749664306641, 1.8091827630996704, 0.4574844539165497, -5.014082431793213, 0.3730838894844055, -4.692569732666016, 10.396818161010742, -2.6932928562164307, 5.108888149261475, -6.540633201599121, 5.878283500671387, -14.51672649383545, 11.133259773254395, -16.484050750732422, -0.3336511254310608, -6.526588439941406, 1.3408042192459106, 2.7305185794830322, -1.1551264524459839, -2.747755765914917, 2.2895469665527344, -4.1995062828063965, 8.539454460144043, 10.905418395996094, 13.715936660766602, -15.515830993652344, -5.723691940307617, -0.08760020136833191, 1.524662971496582, -6.718204975128174, -15.38697338104248, 1.6578344106674194, 2.0016567707061768, -4.878172874450684, 3.1234240531921387, 1.9967156648635864, 6.688655376434326, 10.435396194458008, -0.9460864067077637, 2.4920151233673096, -1.0749428272247314, -9.217362403869629, -17.55990982055664, 5.3804612159729, 5.611242294311523, 12.129619598388672, 3.754733085632324, 10.123504638671875, 14.634810447692871, 2.8477444648742676, 15.692601203918457, -2.587721109390259, 2.1589787006378174, -3.92584490776062, 10.558679580688477, 7.973953723907471, -3.645096778869629, -6.868240833282471, -9.099339485168457, -0.2952477037906647, 14.894231796264648, -0.9525193572044373, -6.316289901733398, -8.191529273986816, -4.99338960647583, -3.070570945739746, 3.2532122135162354, 6.202493190765381, 2.786747455596924, -1.8633509874343872, 4.730521202087402, 0.31980475783348083, -1.6123695373535156, -2.209913730621338, 0.9891980886459351, -0.22453774511814117, 3.1874046325683594, -1.1043205261230469, 0.6534327268600464, -2.368239164352417, 5.058136940002441, -1.6392887830734253, 1.3054884672164917, 5.408481121063232, 1.6694233417510986, 0.07990239560604095, -11.280474662780762, -0.725223958492279, 6.965331077575684, -12.533995628356934, 2.0945727825164795], [-4.584148406982422, -2.6275277137756348, 6.869563579559326, 6.021143436431885, 2.201569080352783, -0.9568166136741638, -5.258674621582031, 6.461230754852295, -1.3407740592956543, -11.283087730407715, 5.3928327560424805, 1.573106050491333, -11.042820930480957, -17.177682876586914, -4.383856773376465, -4.718983173370361, 6.450244903564453, -5.203884601593018, -4.11384916305542, -9.358057022094727, -13.032470703125, -5.060962200164795, 4.418546199798584, 2.5700457096099854, 11.806589126586914, -13.038148880004883, 2.7840352058410645, 2.8011128902435303, 4.639942169189453, 3.7415566444396973, -5.557572364807129, 11.254602432250977, 9.505252838134766, 8.649616241455078, -11.887786865234375, 5.0078253746032715, -6.493317604064941, -3.484461784362793, -10.986745834350586, -3.318873882293701, -0.6136289834976196, 2.6382453441619873, 8.617152214050293, -12.772780418395996, -14.94201946258545, -0.7764528393745422, -4.4592671394348145, -8.87573528289795, 10.11684799194336, -8.756595611572266, -0.008536320179700851, -5.5154900550842285, -2.104421615600586, 8.76089096069336, -2.437941312789917, 22.369190216064453, 11.504890441894531, -1.823375940322876, 9.011748313903809, 3.861581325531006, 7.80025053024292, 10.944692611694336, 0.3081933259963989, 14.525466918945312, 1.68844735622406, -11.997807502746582, 5.250706672668457, 1.6993945837020874, 3.1139440536499023, 3.3482398986816406, -7.870955944061279, 4.395594596862793, -1.3769043684005737, 3.599273920059204, -8.15929126739502, -10.968484878540039, -6.455811023712158, 8.853595733642578, 1.6937339305877686, 8.345879554748535, -1.40005624294281, 6.658984661102295, -11.740886688232422, 8.035809516906738, 1.1207853555679321, -4.332452774047852, 0.7321308255195618, 5.257369041442871, 16.359609603881836, 6.2157979011535645, -10.696526527404785, -1.7546502351760864, 2.8181209564208984, -7.524491310119629, -0.21356433629989624, 1.9158674478530884, 8.860523223876953, 0.3106195628643036, 10.545822143554688, -4.288943290710449, 3.108224868774414, 5.367446422576904, 2.7205076217651367, 0.9183668494224548, 1.756529450416565, 10.09980297088623, 12.897006034851074, -6.023708343505859, -0.3895476162433624, 7.297152519226074, 6.342012882232666, -12.008950233459473, 1.0718202590942383, 21.41338539123535, 5.6812310218811035, -4.555400848388672, -14.418991088867188, 4.783365249633789, 2.7095346450805664, 1.7837581634521484, 1.1845239400863647, -0.017501074820756912, -2.6092731952667236, -7.0064496994018555, 3.7592835426330566, 6.14252233505249, -8.169636726379395, 0.05452246218919754, -1.188017725944519], [-1.281126856803894, 1.5315196514129639, 2.6174206733703613, 16.100360870361328, 10.055119514465332, -17.78451156616211, -0.7008423209190369, -9.327428817749023, 9.835358619689941, 10.178742408752441, -3.6680095195770264, -13.378684043884277, -8.703374862670898, 8.010589599609375, -11.229216575622559, -6.551978588104248, -0.39324966073036194, 8.722638130187988, -8.12779426574707, 1.90321946144104, 1.3350636959075928, -5.340716361999512, 10.384476661682129, 0.07883764803409576, -7.704830169677734, -1.4277265071868896, 3.699800968170166, 13.61047077178955, -1.7181774377822876, 8.767361640930176, 1.6589208841323853, -6.2270941734313965, -15.92393684387207, -4.477822303771973, 4.417366027832031, -6.914444923400879, -12.349480628967285, -9.289608001708984, 4.185333251953125, -2.0784685611724854, -3.1162586212158203, -4.7599005699157715, 4.528914928436279, 2.667853832244873, -1.7614437341690063, -8.286171913146973, -4.936927795410156, 4.416123390197754, -6.471698760986328, -4.792969226837158, 11.870147705078125, -0.12017931789159775, -7.238198280334473, 9.95409870147705, -19.836166381835938, 7.086451530456543, 8.051290512084961, -1.922515869140625, -4.38117790222168, -2.384126901626587, -8.572599411010742, -3.0015835762023926, -5.450193881988525, 12.32461929321289, -11.01773738861084, 8.394659042358398, 5.33281946182251, 6.19277286529541, 12.39372444152832, -1.1213841438293457, -0.7047088146209717, 3.927377462387085, -12.596339225769043, -12.274243354797363, -7.752589702606201, -11.591048240661621, 2.109011650085449, 2.4114770889282227, -6.451276779174805, 2.6821999549865723, -15.14461898803711, -18.578157424926758, 11.8650484085083, -2.784029960632324, -4.213099479675293, 2.53943133354187, -12.539956092834473, 5.553379535675049, 4.350390911102295, 12.218335151672363, 8.493280410766602, -1.3809102773666382, -0.16970911622047424, -6.329690456390381, 8.101884841918945, -3.380234479904175, 7.358325481414795, -1.2920584678649902, 2.127159833908081, 13.460707664489746, -7.034635543823242, 1.4110815525054932, 7.108470916748047, -8.357316970825195, -4.946218967437744, 2.197316884994507, -14.742977142333984, 9.417798042297363, -7.084035396575928, -15.84847640991211, -2.6551520824432373, -1.141170620918274, 0.6549933552742004, -3.244422435760498, 2.302821397781372, 1.291420578956604, 0.524988055229187, -9.755919456481934, 3.4375498294830322, -9.645143508911133, -6.7898712158203125, -7.241085052490234, -9.177760124206543, 4.537954807281494, 7.291196823120117, -2.400909900665283, -3.326753854751587, 20.872968673706055, 5.342916965484619], [-3.0829567909240723, 2.894352912902832, 0.9971706867218018, 6.130041599273682, 2.4264867305755615, -20.401805877685547, 1.7017977237701416, -4.1288604736328125, 7.514357566833496, -1.743134617805481, 7.229402542114258, 4.980617046356201, -14.854143142700195, 8.328664779663086, -6.268961429595947, 3.210193634033203, 0.5454578399658203, -8.438570022583008, 13.84900951385498, 4.779191970825195, 0.703255295753479, -3.589170217514038, 4.488960266113281, -15.76047134399414, 6.439688682556152, 10.966498374938965, -7.373082160949707, 11.335732460021973, -10.027925491333008, -1.8598833084106445, -1.200862169265747, 3.1629106998443604, 2.651585817337036, -3.406642198562622, -11.442316055297852, 3.9402661323547363, 1.6964457035064697, 7.709607124328613, -8.665797233581543, -13.840384483337402, -17.45088005065918, -4.858128547668457, 1.288793921470642, 2.245518684387207, -6.385066032409668, 15.523650169372559, 10.500204086303711, 8.156732559204102, 5.2483229637146, 10.43334674835205, 11.265507698059082, 2.320260763168335, -5.455611228942871, 9.20048713684082, 4.692617416381836, 9.312490463256836, 0.12808644771575928, -2.485231399536133, 13.646627426147461, 8.380435943603516, -15.36326789855957, -2.2753963470458984, -6.363980770111084, 7.764375686645508, -1.604932427406311, 1.9866037368774414, -2.676365613937378, 0.6546868085861206, -3.018212080001831, -0.2623376250267029, 0.6407908201217651, 9.5329008102417, 4.339175701141357, -9.878883361816406, 5.409246444702148, -20.661779403686523, 16.863079071044922, -11.690136909484863, -4.8713860511779785, 1.5390037298202515, 5.174537181854248, -5.444951057434082, -0.27621057629585266, -9.514616012573242, -0.5657645463943481, 1.3642477989196777, -7.2511677742004395, 1.603488564491272, -3.078141689300537, 0.11375462263822556, 22.527725219726562, 7.509191513061523, 8.265311241149902, 2.1037509441375732, -13.198348045349121, 1.6269426345825195, 15.781258583068848, -0.6546083092689514, -4.096020698547363, 14.135948181152344, -9.029135704040527, -0.1789243370294571, -1.4420703649520874, 2.5613863468170166, 0.12576109170913696, -4.161890029907227, -2.8084447383880615, -1.1506807804107666, -12.577890396118164, 6.230193138122559, -3.639163017272949, 2.422001838684082, 6.947574138641357, 11.206786155700684, 1.4465399980545044, 0.7136176824569702, 2.0791449546813965, -6.406900405883789, -1.150368094444275, 10.682801246643066, 2.291510581970215, -6.4556403160095215, 0.8486583828926086, 13.815007209777832, 12.138307571411133, 20.10441017150879, 4.649860382080078, 8.657415390014648, -9.533498764038086], [1.1156216859817505, -12.644759178161621, -2.9617302417755127, 3.127753496170044, -6.372356414794922, -0.40419843792915344, -6.053338527679443, -4.9269633293151855, -0.3202761709690094, 4.23931884765625, 10.795379638671875, -0.4033060371875763, -9.332714080810547, 1.291609525680542, 0.756201982498169, -2.846085548400879, 10.052971839904785, 0.4341035485267639, 12.036080360412598, 3.8839473724365234, 3.8621985912323, 1.5303924083709717, 6.3243207931518555, 5.515450954437256, -3.2248687744140625, -2.956491231918335, -5.858450412750244, 0.8216857314109802, 5.372045993804932, 2.194897174835205, 13.294631004333496, -7.893888473510742, -11.260527610778809, -11.513957977294922, 16.210491180419922, -5.911527156829834, 15.495182991027832, 1.9495909214019775, 6.419873237609863, -9.950498580932617, -1.2940504550933838, 7.01670503616333, -2.136251926422119, -5.262165069580078, 3.1082887649536133, -6.932053565979004, 3.404047727584839, -10.237286567687988, 4.042421340942383, -0.3981668949127197, -2.6215431690216064, 8.546748161315918, -12.021799087524414, 8.968263626098633, -2.1428375244140625, -5.084219932556152, -6.008208751678467, -4.688945293426514, 7.923616409301758, 4.5777106285095215, 10.657448768615723, 14.666202545166016, 8.148983001708984, -2.2740957736968994, 0.7222801446914673, 0.6088191866874695, -5.005430221557617, 4.600433349609375, -6.80788516998291, 16.367525100708008, -4.132455348968506, -1.7571022510528564, 2.6967360973358154, 10.158432006835938, -3.0702507495880127, -0.06582815200090408, -2.4765944480895996, -0.3168847858905792, -1.505049228668213, 14.071852684020996, -6.996984958648682, 2.6726126670837402, 2.5417301654815674, -2.3077504634857178, -6.806555271148682, 9.785015106201172, -7.983574867248535, -16.04560089111328, 10.06861686706543, 6.474189758300781, 6.187854766845703, 0.5917931795120239, 0.34005510807037354, 4.4875664710998535, 11.134244918823242, 6.548137664794922, -5.428945541381836, 2.759660243988037, 5.097882270812988, 0.24225760996341705, 10.789079666137695, -15.117478370666504, -0.23823250830173492, 5.270860195159912, 12.970154762268066, -3.813253164291382, -19.752166748046875, 5.139930725097656, 2.112821102142334, 1.7550227642059326, 2.369173288345337, -2.9259262084960938, -1.753646969795227, -0.4402412176132202, 11.732318878173828, 3.457955837249756, 4.90567684173584, -3.000242233276367, -0.8461833596229553, -11.118961334228516, 3.163590908050537, 0.8173683881759644, 6.21173095703125, -8.108424186706543, -5.5190887451171875, 2.951777458190918, -4.102962970733643, -10.534421920776367, -0.9130057692527771], [1.1452500820159912, -3.0439293384552, -1.7047725915908813, -4.605614185333252, 13.066028594970703, -0.34247416257858276, -22.122135162353516, -2.4489235877990723, -6.520299911499023, -3.8384745121002197, 10.176848411560059, 4.803222179412842, -4.617180347442627, -1.0779387950897217, 5.578578948974609, -3.766757011413574, 17.629159927368164, -11.351329803466797, -1.7459756135940552, 1.3106275796890259, 17.05010414123535, 4.124673366546631, 4.46319580078125, -5.170943260192871, 10.309932708740234, -12.47141170501709, -25.958759307861328, 6.9765119552612305, 1.9857983589172363, 3.5580174922943115, -15.050923347473145, 12.072477340698242, -6.961346626281738, -1.8522250652313232, -15.15800952911377, -5.994213104248047, -3.116316318511963, -11.879463195800781, -19.39710235595703, 7.226297855377197, 18.43313980102539, -6.1207194328308105, 8.029257774353027, 2.5909523963928223, -3.0692219734191895, 1.4995567798614502, -4.900549411773682, 15.02096176147461, 5.3218278884887695, 3.61614990234375, 13.239864349365234, -9.409610748291016, -6.205613136291504, 9.78956127166748, -7.956282138824463, 32.02692794799805, 10.523100852966309, -9.670943260192871, -4.093003273010254, 3.689129114151001, -4.2038726806640625, -8.93314266204834, -1.1564655303955078, 10.915119171142578, -1.3973265886306763, -7.596925735473633, -2.966616153717041, 6.246728420257568, 1.2574366331100464, -0.7015445828437805, 14.873434066772461, 8.674500465393066, -17.870759963989258, -12.006917953491211, 2.9157447814941406, -4.884944915771484, -4.760040283203125, -3.8600406646728516, -7.063338756561279, 5.249411582946777, -2.0381662845611572, 2.074063777923584, 1.9000355005264282, -2.5218935012817383, -11.156890869140625, 1.5578231811523438, 1.7411487102508545, 8.711443901062012, 0.41978591680526733, -0.07091789692640305, -2.49820613861084, 7.870626926422119, -2.8452725410461426, -6.645017623901367, -8.858857154846191, -20.369346618652344, -5.324258804321289, 9.485093116760254, -1.5801798105239868, 4.898934841156006, -9.709158897399902, -1.563459038734436, 6.668797969818115, -8.10657787322998, -10.024922370910645, -8.075949668884277, -2.0939371585845947, -1.7563669681549072, 4.500402450561523, -23.57361602783203, -2.5573537349700928, -0.25064313411712646, 19.95901870727539, 7.584404468536377, 7.762531757354736, -8.453791618347168, -0.7878454923629761, -6.164206504821777, -1.7844444513320923, -4.40325927734375, -11.854819297790527, -7.854038238525391, -19.34888458251953, 2.6781558990478516, -2.4674324989318848, 1.0933126211166382, -21.453845977783203, 4.51433801651001, -10.803187370300293], [-2.280371904373169, 5.424172878265381, 6.697757244110107, 13.343082427978516, -0.9708294868469238, 6.370340824127197, 1.534813642501831, 13.409805297851562, -5.136771202087402, 12.663958549499512, -5.579249382019043, -0.6357768177986145, 5.850739479064941, -7.222431182861328, 6.258852958679199, -5.144686698913574, 3.924867630004883, -3.037567615509033, 1.6309584379196167, 8.584647178649902, -1.2883832454681396, 3.3792221546173096, -5.2085700035095215, -3.6019725799560547, -2.0307235717773438, -4.782505989074707, -8.842118263244629, -10.701915740966797, -2.489326238632202, 15.228408813476562, 15.251148223876953, -3.935523509979248, -2.8964498043060303, 11.854718208312988, -3.875591516494751, -6.1213603019714355, -3.5530903339385986, 11.742989540100098, -25.325584411621094, 5.2369704246521, 7.496042251586914, 2.62334942817688, -1.440806269645691, 9.653733253479004, -2.383099317550659, 11.669875144958496, -5.754001140594482, -0.07065131515264511, 1.2172174453735352, 4.332568645477295, -5.079184055328369, 9.190023422241211, -6.157890319824219, 19.774770736694336, -1.605362057685852, 12.115392684936523, -5.89138650894165, 11.185710906982422, -4.893095016479492, 10.157920837402344, 5.630229473114014, -1.5647348165512085, 5.137150764465332, -9.666802406311035, -3.883204221725464, -7.842733860015869, -2.3164472579956055, -3.7241344451904297, 4.472229957580566, -1.842366099357605, 8.851851463317871, 17.55131721496582, -0.9964969158172607, 5.574838161468506, 2.114105701446533, -8.379302024841309, -1.828179121017456, 0.33409395813941956, -8.933496475219727, 7.309185981750488, 2.9075887203216553, 9.88644790649414, -3.1595170497894287, -2.6878275871276855, 6.221375465393066, -2.931713819503784, -7.387422561645508, -10.109781265258789, -5.215398788452148, 7.228657245635986, 7.583095550537109, 5.2842488288879395, 5.18695068359375, -17.613449096679688, -2.2733802795410156, 2.845855712890625, -0.27920442819595337, 1.885425329208374, -4.388160228729248, -7.856793403625488, 2.676595449447632, 2.1706020832061768, -14.210965156555176, -1.039832592010498, -6.447798728942871, -6.455110549926758, -4.287795543670654, 0.6110442280769348, -0.14509317278862, 2.999372720718384, 7.527498245239258, -4.209529399871826, 14.317265510559082, 8.751693725585938, 13.488712310791016, 4.050584316253662, -2.6281521320343018, -6.307892322540283, -13.689935684204102, 10.37052059173584, 3.4171504974365234, 17.4841365814209, -2.565119504928589, -1.9987602233886719, -9.010437965393066, -7.602447509765625, -4.447389125823975, 1.8114407062530518, 7.763158321380615], [15.040620803833008, 3.2756409645080566, 0.9735000133514404, 7.523354530334473, 9.315169334411621, -19.361223220825195, 5.8098249435424805, 2.107293128967285, 0.8645362854003906, 4.7393879890441895, -8.406744003295898, -2.4763693809509277, 15.920051574707031, -3.4451639652252197, 1.0698059797286987, -12.378425598144531, -1.0449650287628174, -0.7579548358917236, 1.2634860277175903, -3.801511526107788, -8.62392807006836, -10.753424644470215, 2.0915825366973877, -10.006749153137207, -15.709595680236816, 7.308091163635254, -2.8107824325561523, -1.4106700420379639, -15.788215637207031, 12.055319786071777, 6.986328601837158, 7.905492782592773, 7.104286193847656, 5.3380818367004395, -0.42533767223358154, -5.758344650268555, -3.7116613388061523, 1.1800302267074585, 5.503999710083008, 5.01020622253418, -11.292271614074707, 5.982411861419678, 17.853370666503906, 3.88073992729187, -21.416378021240234, -5.172656059265137, 2.513814687728882, 15.707385063171387, -6.504299640655518, -10.52536392211914, 1.046567440032959, 6.503702163696289, 14.29611587524414, -1.4782174825668335, 4.505965232849121, -0.7526019215583801, -0.15224312245845795, 5.738533020019531, -2.5135338306427, -8.953752517700195, -11.660394668579102, -10.949098587036133, 9.021010398864746, -0.30243420600891113, 6.738977432250977, -7.675899028778076, 1.9492617845535278, -0.562424898147583, 9.269537925720215, 9.785419464111328, 4.301551342010498, 3.770487070083618, 6.09807014465332, 0.1934911161661148, -16.49727439880371, 4.631717205047607, -9.464069366455078, -2.3430051803588867, -7.437112331390381, -5.484321117401123, 3.637998580932617, 5.15081262588501, 2.1906321048736572, 10.42323112487793, -9.322673797607422, 3.780442714691162, -2.3821945190429688, 7.832718372344971, -4.35959005355835, -2.7403345108032227, 3.4962446689605713, 5.5046067237854, -4.386006832122803, -9.847709655761719, 2.7099223136901855, -17.32921600341797, 0.8767281174659729, 8.77304744720459, -6.349298477172852, -0.5763346552848816, 2.0589230060577393, 13.651086807250977, -25.27587127685547, 0.09481364488601685, -2.400559902191162, -10.52434253692627, 7.544946193695068, 9.077047348022461, 11.456988334655762, -27.158428192138672, -0.6380168199539185, 6.684051036834717, 11.26541805267334, 18.959457397460938, -9.353013038635254, 0.3140960931777954, -13.950395584106445, 16.062498092651367, -2.5928642749786377, 6.587075233459473, 5.207928657531738, 0.5137871503829956, -1.9105947017669678, 3.136902332305908, -14.342788696289062, 0.9222826957702637, -4.528805732727051, 4.511985778808594, -1.3916255235671997], [-11.68982219696045, -22.684202194213867, -2.052492141723633, -2.3773529529571533, -1.4347542524337769, 4.6616902351379395, -0.271355003118515, -5.664665699005127, -4.9816083908081055, -9.419778823852539, 4.237785816192627, -6.159935474395752, -19.022703170776367, 3.475266218185425, -9.598434448242188, 1.2357490062713623, 10.552587509155273, 2.0954020023345947, -7.623138427734375, -3.803093194961548, 2.4937753677368164, 5.098188400268555, 6.190491676330566, -0.8218371272087097, 6.315860748291016, 3.181387424468994, -9.047140121459961, -18.45018196105957, 1.8660041093826294, -6.646238803863525, -15.937496185302734, -7.34971284866333, 0.17769524455070496, 3.811208724975586, 20.665756225585938, 5.634986400604248, -6.048638820648193, -3.5337412357330322, 3.3554162979125977, -14.157217025756836, 1.3857837915420532, -5.089465141296387, -4.0706787109375, 2.0959346294403076, 9.178706169128418, -9.995458602905273, 9.968586921691895, -17.39347267150879, 7.23676872253418, -15.93193531036377, 9.911100387573242, -13.285501480102539, -5.142216205596924, -2.218599796295166, -11.602337837219238, -3.1060352325439453, 1.5642423629760742, -6.978104114532471, -1.6985583305358887, -6.378931522369385, -15.923810958862305, -11.866697311401367, -15.417997360229492, 11.707921028137207, -9.698251724243164, -1.0925352573394775, 1.0717768669128418, -6.124044895172119, 3.928118944168091, 11.214852333068848, -7.487430095672607, -0.40940842032432556, -0.6387411952018738, -3.7121405601501465, -2.069459915161133, 4.228118419647217, -4.146178722381592, 3.2174460887908936, -2.0045127868652344, 0.09629166126251221, 9.221805572509766, -15.666863441467285, 7.104313850402832, -0.28781962394714355, -6.487414836883545, 16.271188735961914, -7.3193359375, 10.072900772094727, -0.17156393826007843, 2.5022192001342773, -2.214928388595581, 1.5183591842651367, 0.1972757875919342, 10.048742294311523, 5.857531547546387, 3.819880485534668, -0.4412631690502167, -2.95257830619812, 23.4751033782959, 3.617097854614258, -4.190175533294678, 8.76775074005127, 1.1394858360290527, -12.736705780029297, -11.789670944213867, 15.222702980041504, -3.0291833877563477, -13.698616981506348, -14.554647445678711, -3.971728563308716, 4.848400115966797, 9.49711799621582, 5.565866947174072, 7.979226589202881, 5.231719493865967, 3.392962694168091, -4.563475608825684, 8.389748573303223, -0.6304645538330078, -8.315747261047363, -0.5976008772850037, -0.8055063486099243, 5.340092658996582, -1.6241554021835327, 8.858660697937012, -10.410801887512207, 2.3149421215057373, -2.059190034866333, 1.5461692810058594], [4.120632648468018, -19.472930908203125, -7.577242374420166, -21.82607650756836, 3.390268325805664, -16.72052001953125, 9.072017669677734, 0.32571911811828613, -6.753602504730225, -15.42626667022705, -3.30495548248291, -0.07003001868724823, -0.9694042801856995, 3.291269540786743, -15.68170166015625, 4.700658321380615, -5.572151184082031, -16.746286392211914, -3.7549290657043457, -6.617710590362549, -16.911653518676758, 7.492501735687256, 10.23114013671875, -4.7444000244140625, 6.8252973556518555, -0.21400809288024902, -6.018459320068359, -10.002511024475098, -0.01891830936074257, 12.252697944641113, 3.026076316833496, 25.46938133239746, 2.853703260421753, -22.111061096191406, 3.7267746925354004, 15.659764289855957, 14.529182434082031, 9.66816234588623, -0.5460405945777893, -1.8405152559280396, -9.844807624816895, 4.919283390045166, -1.2400537729263306, -7.951422691345215, -9.283833503723145, -7.315560340881348, 19.033519744873047, -8.907925605773926, -4.168328285217285, -13.17199993133545, 3.5796830654144287, -12.958152770996094, 0.6275841593742371, -21.8574275970459, 2.111297607421875, -13.200387001037598, -19.875852584838867, 3.7209293842315674, -8.635371208190918, -6.837282180786133, -3.9138007164001465, -16.1007137298584, -27.634984970092773, 11.614943504333496, 7.679327964782715, -12.875661849975586, 6.812351703643799, 7.014025688171387, 14.578958511352539, 8.251372337341309, 5.05375862121582, -11.252181053161621, 19.88291358947754, -2.97983455657959, 6.793323040008545, -9.093863487243652, -1.7469074726104736, 14.727127075195312, 10.25280475616455, 15.207865715026855, -1.900330662727356, -12.656023025512695, -1.4142205715179443, 9.287009239196777, -11.51889419555664, 5.720849514007568, 0.0901927500963211, 8.756952285766602, 5.2278828620910645, -3.0497992038726807, -1.451209545135498, 17.216678619384766, 1.215815782546997, 5.045165538787842, 16.810394287109375, 3.380399227142334, -1.1257063150405884, 14.529911994934082, -5.847996711730957, -16.902801513671875, -2.4161763191223145, 7.488883972167969, -4.755682945251465, 5.013540267944336, 0.5957267880439758, 6.09649658203125, -9.06962776184082, 1.016306757926941, 3.8461296558380127, -9.2938232421875, 6.34616231918335, 14.095958709716797, 11.364093780517578, 0.1785728931427002, -7.566484451293945, -10.755047798156738, -3.534250020980835, 16.516359329223633, -6.4695210456848145, 0.6177269816398621, 3.905710458755493, -11.308653831481934, 4.700667381286621, -6.585639476776123, 0.30207526683807373, -11.937210083007812, -20.50856590270996, 4.321049690246582, -16.535356521606445], [6.67264986038208, 2.9132955074310303, -1.4581446647644043, 9.98987102508545, -2.0316379070281982, -18.93438720703125, 8.09615707397461, 4.2251152992248535, 2.7569642066955566, 2.08734393119812, -2.956928014755249, -17.72686195373535, -1.7378759384155273, 9.419261932373047, 0.42303502559661865, -0.5875743627548218, 0.9068651795387268, -1.4691617488861084, 10.702258110046387, 5.952475070953369, 2.0895867347717285, 3.7382726669311523, -5.997823238372803, -11.576987266540527, 12.190037727355957, -3.835744857788086, -1.125793695449829, 12.96384334564209, -4.977193832397461, 0.9896954894065857, 4.371035099029541, -4.909595966339111, -13.606344223022461, 11.688608169555664, -3.7437098026275635, -9.558419227600098, -2.130459785461426, 1.0812524557113647, -18.408815383911133, -8.421401023864746, 12.686565399169922, -16.59571075439453, -14.843146324157715, -8.295475959777832, -1.879298210144043, -9.639548301696777, 7.637316703796387, 16.40995216369629, 4.303445339202881, -6.1868391036987305, 4.411048889160156, 7.053210735321045, -14.504752159118652, 1.8162931203842163, -2.6054205894470215, -8.370217323303223, -2.298837661743164, 27.333816528320312, -2.1982643604278564, 11.175926208496094, -5.778769493103027, -17.138887405395508, 10.59086799621582, -0.9238253831863403, 7.0674028396606445, 0.10595525801181793, 7.1002092361450195, -0.3667792081832886, 14.194116592407227, -8.47287654876709, -4.333755016326904, 12.642538070678711, 10.472030639648438, -12.08796215057373, -0.6230579614639282, -4.785618305206299, -2.6690399646759033, -23.824722290039062, -9.51908016204834, -2.2560644149780273, 6.345388889312744, -9.923867225646973, -7.512607574462891, 0.28808704018592834, -3.9414172172546387, 10.653315544128418, -19.15587043762207, 7.5141377449035645, -5.6014909744262695, 0.006654478143900633, 2.3645269870758057, -3.6927828788757324, -6.992321491241455, -14.300238609313965, -6.3354811668396, -7.591865539550781, 5.344544887542725, 4.151373863220215, -2.055187225341797, 9.564131736755371, -13.577166557312012, -3.731330394744873, 0.6449686884880066, -11.52749252319336, -16.875484466552734, 2.843785524368286, -13.73084545135498, -2.5922889709472656, -8.149003982543945, -4.63245964050293, -0.7498930096626282, -4.926839828491211, 7.186511039733887, -4.482976913452148, -3.8261313438415527, 17.000160217285156, 1.0800176858901978, -13.655548095703125, -9.74124813079834, -0.995356559753418, 4.921504974365234, 12.434723854064941, 14.581738471984863, -3.113133430480957, 4.15596342086792, 2.4471099376678467, 1.9984941482543945, 10.932171821594238, 10.332267761230469], [1.1493855714797974, 4.511474609375, -1.4456862211227417, -4.145103454589844, 0.23280426859855652, 2.8402891159057617, -12.618529319763184, 3.0683422088623047, -10.723335266113281, 10.338336944580078, -2.970682144165039, -2.9091928005218506, 19.14017677307129, 5.879213809967041, -1.0427255630493164, 4.448574542999268, -1.8541004657745361, -13.29133415222168, 1.8289039134979248, -5.571836948394775, 1.830062747001648, -3.324849843978882, 0.3986772894859314, 8.845598220825195, 10.015488624572754, 10.240159034729004, 4.1342082023620605, -7.111698627471924, -2.191833734512329, -4.313016891479492, -9.603079795837402, 2.185922622680664, 1.9241372346878052, 13.036696434020996, -1.722918152809143, 1.5397578477859497, -5.565922737121582, -0.25121963024139404, 1.923652172088623, 13.850217819213867, -10.288686752319336, 3.6835954189300537, 6.102160453796387, -0.42457613348960876, -4.380035400390625, -3.5604910850524902, 4.92376708984375, 7.920276165008545, -8.409605026245117, 2.0591094493865967, -20.904762268066406, -0.3194657862186432, 12.185127258300781, -4.813953399658203, -8.605032920837402, -0.6367287635803223, -3.2013916969299316, 1.1560463905334473, -4.960567951202393, -16.552244186401367, -10.150278091430664, -2.330869197845459, 4.490609169006348, 2.890294075012207, 7.507537841796875, 1.8852876424789429, 5.504271984100342, -1.0261099338531494, 0.7509292960166931, 4.735494136810303, 2.8140411376953125, -3.391746997833252, 11.403081893920898, 7.424301624298096, 9.645238876342773, 12.968791961669922, 2.6833744049072266, -0.4376065731048584, -0.6785587072372437, -10.32423210144043, -0.19080542027950287, -1.1028677225112915, 7.9027485847473145, 2.486891031265259, 3.1550323963165283, -7.980156898498535, 14.92892074584961, 14.693194389343262, -14.20322036743164, 0.42210477590560913, 5.99950647354126, -0.9846254587173462, -4.691365718841553, 11.2517671585083, -4.781643867492676, -8.203011512756348, -2.230217695236206, 16.093482971191406, -18.320322036743164, 0.7532932162284851, 0.6926661133766174, 8.615386962890625, -12.859465599060059, -5.804448127746582, 6.729771137237549, -2.1795523166656494, 6.317957401275635, -2.465120553970337, -0.26031211018562317, -7.734367370605469, 2.3094935417175293, 15.497373580932617, 3.6360630989074707, -21.221200942993164, -2.9355621337890625, -2.938222646713257, -0.5786792635917664, -1.6343106031417847, -13.718214988708496, 3.580508232116699, 11.604564666748047, -2.936124324798584, -2.333280563354492, 1.5814409255981445, 5.17337703704834, -2.6682016849517822, 0.5844355225563049, 4.4681172370910645, -2.8823981285095215], [6.314650535583496, -12.66850757598877, -6.251580715179443, 4.252158164978027, 7.108007907867432, 3.370786428451538, 9.17140007019043, -7.610975742340088, 0.6660328507423401, -4.402873992919922, 7.723442554473877, 3.262576103210449, -9.610569953918457, 4.057374477386475, 3.480597734451294, -3.312788963317871, -3.192758321762085, 5.663491249084473, 3.3828952312469482, 1.7090729475021362, 8.998215675354004, -8.414130210876465, 4.874867916107178, -6.604619979858398, 3.064598798751831, 8.370869636535645, -4.181212425231934, -1.0827808380126953, -5.081303119659424, 3.6525275707244873, 3.0142910480499268, -1.2769440412521362, 2.8581457138061523, 1.3136147260665894, 13.139628410339355, 1.2626675367355347, 3.2860779762268066, -9.52151107788086, 12.69504165649414, -9.442609786987305, 5.183445453643799, -5.628331661224365, -2.1701784133911133, -5.495546817779541, 5.574909210205078, 2.600968599319458, 5.931369781494141, -14.788809776306152, 5.287981986999512, -11.05087661743164, 15.161898612976074, 6.497685432434082, 0.9159098267555237, -20.02705955505371, 4.996795654296875, -12.308167457580566, -8.701643943786621, -4.963109970092773, -2.8901681900024414, 2.09005069732666, 4.325395107269287, 3.9274394512176514, -3.29093861579895, -5.327365875244141, 0.0688648670911789, 11.289112091064453, 2.8456692695617676, -4.567932605743408, -4.340646266937256, 3.7197065353393555, -9.615495681762695, -9.571760177612305, 1.6943058967590332, -9.328520774841309, -0.18766917288303375, 8.768874168395996, -2.428225517272949, -10.348990440368652, -1.1897655725479126, -4.831110000610352, -0.4999528229236603, -6.48399019241333, 6.7140703201293945, 2.5135650634765625, -1.4328200817108154, 3.7708001136779785, -2.0258963108062744, 7.190593719482422, 7.004637241363525, -1.2357069253921509, -1.617445468902588, -1.5476577281951904, -0.7313300371170044, 6.3245134353637695, 4.647702217102051, 5.047745227813721, -2.808790445327759, -0.9912266135215759, -0.11543627083301544, 0.31276917457580566, 3.3314828872680664, -2.8338170051574707, -1.1781052350997925, -5.292781352996826, 2.1109797954559326, 10.19729995727539, -9.449893951416016, -0.9995466470718384, -13.781913757324219, -3.525451898574829, 5.190462589263916, 13.925409317016602, -10.52316665649414, 15.790745735168457, 1.2348581552505493, 6.815656661987305, 1.1386122703552246, -0.604450523853302, 2.683023691177368, -5.705930233001709, -0.9626215696334839, -11.146734237670898, -2.6343672275543213, 1.646230697631836, 0.14754019677639008, -1.7772282361984253, 7.074870586395264, -6.1761250495910645, -3.4555301666259766], [10.151866912841797, -6.538353443145752, 9.164477348327637, -4.2719316482543945, 7.4833455085754395, -14.71290397644043, -3.729353666305542, -15.549995422363281, 10.0574951171875, -9.94494342803955, 1.1930172443389893, 10.022173881530762, 10.253595352172852, 5.380590438842773, -10.223536491394043, 5.452091217041016, -11.652937889099121, -3.07016921043396, -9.11201000213623, -10.592658996582031, -12.121586799621582, -2.167438268661499, 1.4031074047088623, 1.912170171737671, -9.640676498413086, -5.014582633972168, -2.9761886596679688, 12.354029655456543, 5.833805084228516, 6.0830583572387695, -2.28090238571167, 13.258502960205078, -2.724393367767334, 10.196288108825684, -0.9252777695655823, 1.5050666332244873, 11.98129653930664, -7.485597133636475, 9.618246078491211, -5.597764015197754, 1.8161084651947021, 3.1820271015167236, -3.0851879119873047, -17.569759368896484, -20.665740966796875, -12.075599670410156, 3.435317039489746, 7.844091892242432, -5.00590705871582, -1.7585644721984863, -4.099481582641602, -3.5366101264953613, -2.1801068782806396, -12.315357208251953, 7.244712829589844, -12.460505485534668, 12.66455364227295, -3.115356922149658, 8.514859199523926, 6.762842655181885, 1.1225478649139404, -3.288909673690796, -6.366666316986084, 13.207242965698242, 0.9538694024085999, 0.2751889228820801, 0.019167285412549973, 7.1078081130981445, 7.200319766998291, -0.35572075843811035, -3.3217201232910156, -7.596276760101318, -3.966756582260132, -5.046060562133789, 7.477481842041016, 1.459564447402954, -13.268314361572266, 13.266011238098145, -0.5675730109214783, -4.1220479011535645, -4.6996259689331055, -8.249000549316406, -6.143228530883789, 7.465424537658691, -0.08853781968355179, 1.185999870300293, 6.236334323883057, 7.163877964019775, 2.65486478805542, -14.616609573364258, 1.5469651222229004, 2.24338436126709, 0.383035808801651, -4.455706596374512, 1.4086025953292847, -3.579301595687866, 9.816262245178223, 5.771337985992432, 6.574623107910156, -2.537233829498291, -3.0012831687927246, -4.337485313415527, -12.137688636779785, 9.781669616699219, 5.960300445556641, -5.340106010437012, -5.01665735244751, -1.7011678218841553, -0.7329588532447815, 3.1353394985198975, -7.572700500488281, 2.689725875854492, 1.5496615171432495, 11.32152271270752, -5.506206512451172, -0.2627559006214142, -9.231056213378906, 1.3072876930236816, 1.8419792652130127, -1.1759833097457886, -0.9665451645851135, -8.4785737991333, 7.763790130615234, -2.8958752155303955, -5.5092949867248535, 17.47739601135254, 4.759191036224365, 6.519577503204346, -4.192893981933594], [1.5675584077835083, -3.9156689643859863, -7.737096786499023, 9.114137649536133, -12.029183387756348, 5.224304676055908, -3.798842430114746, -12.337870597839355, -11.984869956970215, 7.702376842498779, 15.776865005493164, -6.258312225341797, -5.750468730926514, 2.022632598876953, 2.357750415802002, -12.229425430297852, 7.008578300476074, 2.3269152641296387, -8.91380500793457, -5.088977813720703, 22.702329635620117, 16.39957618713379, -2.0336661338806152, -9.764410972595215, -0.20103642344474792, -9.461474418640137, 1.214725375175476, -6.624845027923584, 7.632241725921631, -5.578422546386719, -9.670591354370117, -12.687980651855469, -14.8983154296875, 5.420943737030029, 3.8330883979797363, -22.3132266998291, 6.542934894561768, -0.27646371722221375, -3.298617362976074, -8.35201358795166, 11.409032821655273, -14.034795761108398, -12.194364547729492, -5.338269233703613, 5.192310810089111, -18.71931266784668, 4.213417053222656, 6.500319004058838, 15.83730411529541, 2.8709182739257812, 19.635848999023438, 0.5227795839309692, -4.519154071807861, 11.69627571105957, -18.292848587036133, -6.820566177368164, -8.052932739257812, 7.2150163650512695, -1.281041145324707, 0.36344727873802185, -2.849158525466919, 22.575254440307617, 9.876724243164062, 9.201987266540527, -3.9233269691467285, 3.140634059906006, 14.140231132507324, 15.55916690826416, -16.80781364440918, 9.469369888305664, -6.275976181030273, 6.457601547241211, -14.9050931930542, 4.605625152587891, 2.9175939559936523, 10.629451751708984, 8.001999855041504, -10.044909477233887, -5.67899751663208, -13.648122787475586, -4.456942081451416, 2.595968723297119, 4.9971137046813965, -0.8944470286369324, -5.443666934967041, 4.210702419281006, -14.558685302734375, 9.830048561096191, -3.854365587234497, -7.601590156555176, 4.00921106338501, 10.632867813110352, -10.709146499633789, 15.52967357635498, 13.607810020446777, -8.446995735168457, -14.780096054077148, -14.282506942749023, 1.7706279754638672, 5.756252288818359, 5.220602035522461, 15.288400650024414, 3.961313009262085, -11.114460945129395, -11.621294021606445, -3.2630062103271484, -3.0584330558776855, -0.6913774013519287, -6.217141151428223, 7.817528247833252, -4.225149154663086, 8.91209888458252, -1.7330056428909302, 8.467554092407227, 6.07949161529541, -5.95816707611084, -16.77527618408203, -1.7644110918045044, 1.9493416547775269, -4.487124919891357, -5.929368495941162, -0.017263224348425865, -0.6914492845535278, -2.302809238433838, 8.541389465332031, -5.2596845626831055, 17.96079444885254, -5.627434253692627, -1.0804047584533691], [-4.822743892669678, 0.9132526516914368, -1.3710886240005493, 4.668757915496826, 5.878929138183594, 15.138213157653809, 7.658944606781006, 2.257091760635376, -3.9349145889282227, -9.174023628234863, -0.16089792549610138, -2.482520580291748, -7.839906215667725, -3.6783530712127686, 1.1978347301483154, 13.041297912597656, -4.432748794555664, 2.33784556388855, -3.097537040710449, 7.327376365661621, -0.8975257277488708, 6.572412967681885, 5.333914279937744, -2.01772403717041, 4.028946399688721, 9.782658576965332, -5.48197603225708, 0.1350138634443283, -7.509299278259277, 1.5428013801574707, -8.542459487915039, -8.1912260055542, -1.004844069480896, 3.264740228652954, -8.171699523925781, 12.794852256774902, 0.10176409035921097, 13.03012466430664, -1.355865716934204, 14.276701927185059, -1.811812162399292, -0.13827383518218994, 7.235574245452881, -5.93404483795166, 4.242801666259766, 2.869408130645752, -6.685396671295166, -1.909622073173523, 1.4856939315795898, -6.092613220214844, 8.205146789550781, -4.68896484375, 5.685744762420654, -10.58022403717041, -11.54690170288086, 2.9885902404785156, 2.308819532394409, 0.179001122713089, -5.991260051727295, 20.39708137512207, -7.032693386077881, 0.3440670669078827, -3.330465316772461, 8.545746803283691, 0.5929276347160339, 0.19717402756214142, 12.963142395019531, 9.302122116088867, -2.765547037124634, 3.1596291065216064, -11.54640007019043, 5.13284969329834, 3.318554162979126, -0.8597815632820129, 4.263220310211182, -3.3046188354492188, -0.971538245677948, -0.5806865692138672, 0.9551944136619568, 2.023221969604492, 0.8902194499969482, 1.4796664714813232, -8.580827713012695, -8.455026626586914, 9.487186431884766, -14.501434326171875, 3.8117315769195557, -1.607357382774353, -0.4785517454147339, -2.5027027130126953, -8.035407066345215, 1.5111602544784546, -9.770652770996094, 7.2376861572265625, 5.678145408630371, 6.921375751495361, -0.22631381452083588, 5.064584255218506, 1.9974653720855713, -7.06715726852417, 14.394219398498535, -2.6458992958068848, 6.573231220245361, -5.011194705963135, -2.615741729736328, 0.6881968975067139, 9.623993873596191, 3.803051710128784, -9.306233406066895, 14.92507266998291, 0.4097927212715149, 11.634397506713867, 2.045654535293579, -10.313558578491211, 9.7411527633667, -8.763548851013184, -11.121475219726562, -3.080139636993408, -0.5506002902984619, -0.2995181679725647, -9.935772895812988, -0.5862992405891418, -8.069145202636719, 10.487749099731445, 8.025032997131348, -11.640997886657715, -2.5885725021362305, 1.1494834423065186, 2.0360257625579834], [-2.3200907707214355, 4.5129804611206055, -1.5208967924118042, -3.8301968574523926, 3.6480929851531982, 3.3665289878845215, 9.441315650939941, 0.3353336453437805, 1.3222308158874512, -6.104183197021484, 7.841256141662598, 5.948019504547119, -10.62336254119873, 8.92732048034668, 8.328773498535156, 4.245325088500977, 9.706303596496582, 10.270712852478027, 3.312861919403076, 10.010035514831543, -4.242314338684082, 8.152711868286133, 8.447800636291504, -5.732266426086426, 1.7885633707046509, 5.2093610763549805, 0.6798407435417175, -2.182260751724243, -16.15332794189453, -11.529389381408691, -11.478656768798828, 2.7042276859283447, 7.944541931152344, -5.281210899353027, 3.7308132648468018, -3.796251058578491, 0.013530952855944633, 1.9387987852096558, 8.659106254577637, -4.540989398956299, 2.0898075103759766, 6.315258979797363, 6.073286056518555, -3.8566701412200928, 1.6802685260772705, -3.34295654296875, 0.41132333874702454, -4.344394683837891, -0.7316389083862305, 7.8090901374816895, 2.42231822013855, 3.084122657775879, 4.764238357543945, -3.142167091369629, 5.165575981140137, 5.238454341888428, 2.7617876529693604, 0.09908860921859741, 2.227916955947876, 0.961727499961853, 7.195662975311279, 16.40552520751953, 3.50338077545166, 0.9701781272888184, -4.93808126449585, 4.0293192863464355, 3.6135222911834717, -1.9182302951812744, -2.6788549423217773, 2.8832032680511475, -7.894068241119385, 1.6604368686676025, -0.9014764428138733, 4.187118053436279, 1.165849208831787, -3.304351329803467, 6.783852577209473, 2.020156145095825, -3.564725160598755, -4.776571750640869, -12.806716918945312, 2.4848735332489014, -15.8689546585083, -1.0917619466781616, 3.882848024368286, 7.830273628234863, -4.775437831878662, 2.112502336502075, 11.84168815612793, -3.614224910736084, -0.3538377583026886, 0.8985514044761658, -4.620246887207031, 6.951784133911133, -2.0430943965911865, -3.2721121311187744, -7.394822597503662, 9.494565963745117, 4.559756755828857, -0.9787985682487488, 3.557051420211792, -4.475554466247559, -0.08926905691623688, 2.711883306503296, 2.1975162029266357, -4.22226095199585, 1.6853892803192139, 6.805663108825684, 2.533050298690796, 3.2876274585723877, 1.1602165699005127, -11.54092025756836, -8.836585998535156, -8.673325538635254, 0.7809427380561829, -6.590976238250732, 1.7634170055389404, 7.422052383422852, 6.544991493225098, -3.052777051925659, 0.26141664385795593, -11.757863998413086, 2.0903265476226807, 3.390387773513794, -10.237017631530762, 8.931257247924805, -1.9504272937774658, -12.514924049377441, 10.040406227111816], [11.621435165405273, 5.986903667449951, -12.617570877075195, -0.6185114979743958, 1.463028907775879, 9.599071502685547, -1.1537361145019531, -3.6534297466278076, -12.122607231140137, 0.30626729130744934, -0.5545472502708435, 4.2703986167907715, 5.1123433113098145, -3.677471876144409, -4.5237274169921875, 2.153085231781006, 2.488861322402954, -5.04235315322876, 3.302405834197998, 8.488581657409668, 12.210853576660156, 9.882861137390137, 10.114916801452637, -7.401627063751221, -5.136881351470947, 3.830369234085083, -2.373055934906006, -10.102166175842285, -2.9279191493988037, -11.477991104125977, -14.933494567871094, 6.701492786407471, 6.940579414367676, -11.787888526916504, -9.144774436950684, 8.4528169631958, 3.4025814533233643, 2.3471338748931885, 7.230584621429443, 0.1716180145740509, -12.115056991577148, 7.390038967132568, 23.415082931518555, -6.776057720184326, 4.51148796081543, -17.808536529541016, 0.9515281319618225, -6.040572166442871, 3.6121397018432617, -1.7605855464935303, 23.879886627197266, -0.4261647164821625, -6.383725643157959, -2.311466932296753, 3.185722589492798, 4.768887042999268, 1.093505859375, -0.5947597622871399, 6.065439224243164, -0.41706547141075134, -11.583781242370605, 5.200523376464844, 1.5398263931274414, -2.4719791412353516, -4.037162780761719, -18.071035385131836, 11.578938484191895, 8.950151443481445, -3.7683331966400146, -8.996137619018555, 6.084918022155762, -9.596282005310059, -3.34614634513855, -0.8259227275848389, -7.0310959815979, 1.926098108291626, 1.4014081954956055, 9.154876708984375, -4.940566062927246, -5.3099141120910645, -4.783381938934326, -5.174990177154541, 1.597036361694336, 11.732754707336426, -7.509939193725586, 1.5835323333740234, 12.072348594665527, 2.455801010131836, 2.288397789001465, 1.7666239738464355, -1.324681043624878, 3.4196066856384277, -1.1737010478973389, 13.169739723205566, 7.308737754821777, -5.469261169433594, -2.5739006996154785, -6.591084003448486, 10.08428955078125, 1.9434312582015991, -7.788323402404785, 13.42874813079834, 3.3371078968048096, -12.70103645324707, -8.93394947052002, 1.0838361978530884, 7.9278998374938965, -2.917634963989258, 1.0062110424041748, -5.030122756958008, 0.42176002264022827, 3.183189630508423, -4.064864158630371, 5.348911285400391, -9.577608108520508, -6.560590744018555, -17.201175689697266, 9.559988975524902, -2.754035472869873, -0.5617529153823853, -4.464723587036133, -4.436604976654053, -8.593506813049316, 7.014451026916504, 0.3633913993835449, 0.5521634817123413, 5.739163398742676, -4.137057304382324, -3.483384847640991], [-2.401432514190674, -21.066083908081055, 6.472467422485352, 14.672473907470703, 17.43524742126465, -1.9240999221801758, -3.361088275909424, 12.873852729797363, -13.062336921691895, -2.5616512298583984, 11.282617568969727, -15.71462345123291, -8.612547874450684, 6.33039665222168, 8.184934616088867, -9.935325622558594, -5.463334560394287, -1.1428111791610718, -4.810742378234863, -4.492062091827393, 6.318873882293701, 2.528528928756714, -3.3261070251464844, 4.712268352508545, 10.866000175476074, -2.7693445682525635, 2.0927250385284424, 8.055168151855469, 6.31941556930542, 17.40819549560547, 8.225031852722168, -1.2102328538894653, -6.770592212677002, 23.024065017700195, -7.692805767059326, -7.976147174835205, 0.2885318994522095, -12.229018211364746, -2.977368116378784, 1.760622501373291, 9.759462356567383, -19.568429946899414, -14.13646125793457, 1.0717787742614746, 11.358054161071777, 1.9415823221206665, -6.100809574127197, -9.441512107849121, -4.837225437164307, -1.3616244792938232, -11.829717636108398, 3.9817426204681396, 16.939319610595703, -9.185166358947754, -19.91329574584961, 8.441322326660156, -1.7981904745101929, 4.247244834899902, -0.6482388973236084, -9.079610824584961, -12.594493865966797, 6.836613655090332, 11.04533863067627, 8.431241989135742, 5.687747955322266, 2.2751951217651367, -12.013256072998047, 1.7128318548202515, -0.8283976912498474, 6.2649030685424805, -2.8458869457244873, 16.967308044433594, -5.625107288360596, -2.283956289291382, -1.5055670738220215, 1.961828589439392, 4.645580768585205, -10.92531681060791, 11.866769790649414, -3.312258720397949, -6.673736572265625, -1.7137737274169922, -0.24016480147838593, -1.8112386465072632, -13.349867820739746, -7.0521240234375, 5.818619728088379, 4.336092948913574, -11.626741409301758, -1.4477336406707764, 4.961843013763428, -5.27000093460083, -5.469820022583008, 9.511049270629883, -3.585177183151245, -4.6824822425842285, -8.590007781982422, -3.3516814708709717, -3.8783438205718994, -3.927175283432007, 1.1087883710861206, 18.662202835083008, 4.99355936050415, -9.825518608093262, 3.354526996612549, -14.06464672088623, 12.086888313293457, 10.408345222473145, -16.187461853027344, 5.337399482727051, 3.6038074493408203, -1.6276482343673706, 6.820230960845947, 4.813701629638672, 8.19076919555664, 5.532734394073486, 11.75721549987793, 10.514090538024902, 0.41734546422958374, -12.088919639587402, 8.179326057434082, 9.379936218261719, -6.903354644775391, -2.2620224952697754, 1.5736110210418701, 12.108697891235352, 2.9820713996887207, 8.098567962646484, -2.0903477668762207], [-7.298579216003418, 11.895545959472656, 20.97842788696289, 11.524130821228027, 7.6253204345703125, -6.8392205238342285, 0.5335803627967834, 3.8717658519744873, 3.9086320400238037, -7.275617599487305, 0.46063923835754395, -5.445012092590332, 3.400113821029663, 3.7051899433135986, -21.34824562072754, -3.1070542335510254, 2.68316912651062, 13.829375267028809, -0.7259075045585632, 18.079505920410156, -13.65548324584961, -1.9530221223831177, 0.8304305076599121, 13.553901672363281, -3.335930824279785, -4.6570868492126465, -0.9576870799064636, 4.234353542327881, -10.686503410339355, 7.9266791343688965, -5.945721626281738, 0.0019141602097079158, 10.865070343017578, 1.572473406791687, 1.1480438709259033, -1.6677570343017578, -3.377434015274048, 24.387405395507812, 3.0907223224639893, 11.501331329345703, -20.242658615112305, 7.057200908660889, 12.53943157196045, -7.318548202514648, 19.873323440551758, -7.060138702392578, 1.5427967309951782, -0.22738465666770935, 5.021917819976807, -1.2226781845092773, -1.3885769844055176, 0.30167514085769653, 2.676541328430176, 14.557008743286133, -12.483141899108887, 9.28649616241455, -2.5137712955474854, -1.501174807548523, 2.367849349975586, 4.838313579559326, 10.485970497131348, -10.651657104492188, 2.0896079540252686, 8.918146133422852, -22.459182739257812, -6.841034412384033, 7.717445373535156, 5.031865119934082, -0.22731444239616394, 2.6420552730560303, 5.323520660400391, 3.5803146362304688, -0.8259644508361816, 1.9056531190872192, -21.422218322753906, 7.311284065246582, 0.0233346838504076, -5.180301189422607, 5.269029140472412, 13.294522285461426, -4.256027698516846, -1.1378624439239502, -10.220571517944336, 0.3795252740383148, 3.9770421981811523, 6.181957721710205, 4.651505947113037, -5.649529457092285, 6.293616771697998, 14.600822448730469, 13.553742408752441, 6.753596305847168, 13.109626770019531, -4.528863430023193, -0.6664490103721619, 5.736623764038086, -11.339082717895508, -11.761910438537598, -4.267782688140869, 3.363546371459961, 6.1889166831970215, -4.492185115814209, 1.4897816181182861, -2.62306547164917, 5.234488487243652, -2.7354116439819336, -15.779296875, -13.115860939025879, -1.7258950471878052, -2.5849905014038086, -1.716554045677185, 3.9373788833618164, -0.7595614194869995, -3.5994441509246826, -8.222119331359863, 3.1318702697753906, 5.09476900100708, -0.49525025486946106, -2.696345329284668, -5.349999904632568, 13.748531341552734, -7.301815509796143, 3.1588118076324463, -4.374267101287842, -10.964332580566406, -8.001472473144531, 12.245218276977539, -4.7027435302734375, -0.7315434813499451], [15.503677368164062, 0.08101380616426468, 4.2667036056518555, -5.799381732940674, 2.279933214187622, -8.473511695861816, 11.710810661315918, 12.004104614257812, -3.4603934288024902, 20.6300106048584, -14.813962936401367, -6.661543369293213, 8.96878719329834, 4.554625034332275, 7.296259880065918, 5.163140773773193, 2.170008897781372, 5.482110023498535, 0.6282011866569519, -4.455409526824951, -2.0575623512268066, -4.379978656768799, 13.087799072265625, 0.6625250577926636, -15.207451820373535, 10.809892654418945, 16.705909729003906, 7.080634117126465, 1.1340155601501465, 7.192620277404785, 9.385384559631348, 7.107882976531982, -15.57327651977539, -6.966638088226318, -3.358670711517334, -2.7057855129241943, 3.4498586654663086, 10.312702178955078, -5.9057536125183105, 5.697174072265625, 1.9483983516693115, -2.486027717590332, 4.23603630065918, 5.794229507446289, 10.118563652038574, 0.17777550220489502, -11.763315200805664, 1.2969876527786255, -13.0504732131958, 4.967288017272949, -1.4336930513381958, 15.909778594970703, 6.282176971435547, -7.229612827301025, 4.735013008117676, -12.477829933166504, -11.95743465423584, 20.86802101135254, -11.52627182006836, -9.408676147460938, -11.421883583068848, -10.484516143798828, -2.4765374660491943, 1.8398066759109497, -2.1278278827667236, 6.516358375549316, -6.397629737854004, 1.80954110622406, 7.0760579109191895, 0.6122056841850281, 4.915450572967529, 2.3901455402374268, -3.552525281906128, -0.4896349608898163, -7.642689228057861, -6.674298286437988, 11.467645645141602, 10.612916946411133, 9.487561225891113, -4.223302364349365, -2.366522789001465, -4.326537132263184, -0.13342545926570892, -5.0333170890808105, -6.182317733764648, -16.56171989440918, -2.3159377574920654, -3.4776997566223145, -8.615633010864258, 4.258114337921143, 3.2696526050567627, 4.407354354858398, 2.9873201847076416, 12.56838607788086, 3.87335205078125, 5.839656352996826, 3.4729576110839844, -3.6193675994873047, -13.962791442871094, -5.152430534362793, 5.610393524169922, 21.413076400756836, 7.45758056640625, -2.2768993377685547, 5.585143089294434, -12.46923542022705, -1.5031211376190186, 3.6244218349456787, -1.0322011709213257, -4.937248706817627, -2.951108932495117, 7.284340858459473, 7.5194010734558105, -3.9145686626434326, -8.476232528686523, -1.4070549011230469, 3.025512933731079, 24.44820785522461, -10.13931941986084, 5.375793933868408, 5.681851863861084, 10.833137512207031, -8.639701843261719, 9.546953201293945, -5.008497714996338, -6.13256311416626, -7.204251289367676, 4.409158229827881, -0.3174613118171692], [5.951672077178955, 1.9548406600952148, 2.1906702518463135, -9.655531883239746, 13.930276870727539, -6.166873455047607, 9.759605407714844, 1.8877710103988647, -0.639130711555481, -11.322916984558105, -2.945521831512451, -10.15804386138916, 13.934144973754883, -1.2736625671386719, 7.716188907623291, -8.460901260375977, 3.3493282794952393, -1.9619853496551514, 10.103684425354004, 12.137359619140625, 1.1803927421569824, 10.194968223571777, -6.106711387634277, -11.333353996276855, 2.9140255451202393, 6.044331073760986, -7.5000457763671875, 2.0648913383483887, 0.6639557480812073, 2.082298994064331, -0.6788145899772644, 13.52783489227295, 1.9860142469406128, -1.4769476652145386, 10.412108421325684, 2.5511317253112793, -0.37676870822906494, 4.333477020263672, -4.344995498657227, 0.16208243370056152, 1.993709683418274, -9.121201515197754, 11.074300765991211, -10.770365715026855, -1.2698097229003906, 7.598556995391846, -6.555420875549316, 2.121765375137329, 0.7733595371246338, 12.379193305969238, -9.436164855957031, -2.8606975078582764, -1.773476243019104, -0.8186182379722595, 5.603145599365234, -0.6993647813796997, -9.436389923095703, 15.046870231628418, 3.0106873512268066, 6.637505054473877, -2.3111352920532227, -2.5299692153930664, 15.248053550720215, -0.6787334084510803, 6.403483867645264, -4.703825950622559, -1.955318808555603, -2.0088818073272705, 2.226924419403076, -5.23414945602417, 5.103120803833008, -16.134599685668945, -6.89760684967041, -0.5588907599449158, -7.341506004333496, 1.8071906566619873, -3.387106418609619, 4.397385120391846, -13.009195327758789, 4.508790493011475, -11.379707336425781, 2.638335943222046, -7.434695720672607, 10.320467948913574, 4.789177417755127, -3.7747111320495605, 9.420317649841309, -4.211726665496826, -1.6989896297454834, -2.2570486068725586, 3.6617844104766846, 4.956568717956543, 5.6205925941467285, 2.2405991554260254, -2.1186423301696777, 3.356349468231201, -8.71603012084961, 1.103076457977295, 3.074277639389038, -4.447079181671143, -6.124148368835449, 6.256999969482422, 1.4395396709442139, 0.5143663883209229, 6.341078281402588, 4.8342108726501465, -4.753971099853516, -2.0886831283569336, -12.05453872680664, 2.664799213409424, 0.22291573882102966, -0.5508652925491333, 0.3570372760295868, -10.044092178344727, 0.03556973487138748, -1.9191187620162964, 3.9349982738494873, 2.2634854316711426, 8.300507545471191, 0.40421590209007263, 7.638491153717041, 8.365988731384277, 5.664302349090576, -6.995867729187012, -4.0764594078063965, 8.620588302612305, -1.5884230136871338, 3.5736804008483887, 11.687031745910645], [-0.6797927021980286, 7.513136863708496, 6.767811298370361, -7.795107841491699, -19.36018943786621, 6.625112533569336, 1.113465666770935, -0.7619678378105164, 1.9009747505187988, -7.017899513244629, -7.458377361297607, -9.914215087890625, 8.729549407958984, 10.141287803649902, -12.338763236999512, 4.997109413146973, -11.637397766113281, 10.554511070251465, 9.748601913452148, -6.96030855178833, -3.3492021560668945, -10.71330738067627, 14.844311714172363, 14.645495414733887, 4.267480850219727, 6.785019874572754, 19.899280548095703, -8.614197731018066, -9.683500289916992, -22.358108520507812, 2.680661678314209, 16.348543167114258, -9.377842903137207, 12.051661491394043, 13.823075294494629, -5.798938751220703, -1.5327420234680176, 10.782852172851562, 6.3153395652771, 12.257857322692871, -13.7185640335083, 11.701091766357422, -4.249493598937988, -3.4996135234832764, 10.27489948272705, -16.045148849487305, 2.528712749481201, -6.916092872619629, -2.5812063217163086, -2.606851816177368, -4.5587849617004395, 7.635185241699219, -0.20195645093917847, -2.726910352706909, -7.100443363189697, -10.648933410644531, -4.901648998260498, 4.68049430847168, -3.204817056655884, -1.9820326566696167, 0.6179484128952026, -7.716374397277832, 0.9267131686210632, 5.371711254119873, -4.078497409820557, -4.390674114227295, 0.8318342566490173, -11.218449592590332, 13.016725540161133, 13.838739395141602, -21.23993492126465, 9.527859687805176, 0.11510371416807175, -3.9894607067108154, -9.1296968460083, 2.9214582443237305, -3.5145416259765625, 9.24351978302002, 12.460936546325684, 3.960785388946533, 14.836089134216309, -4.229748725891113, 2.6235709190368652, -2.6565728187561035, -1.6655877828598022, 1.2615773677825928, -0.8670446276664734, 10.292229652404785, -0.02394624426960945, 7.218328475952148, 8.048213958740234, 7.108522891998291, -6.334631443023682, 5.842238903045654, 6.514346599578857, -6.1740264892578125, -0.8121064305305481, 4.3588433265686035, 1.496424913406372, -8.78592300415039, 12.27346134185791, 3.5370523929595947, 2.309467315673828, 8.605427742004395, -0.10766802728176117, 2.282553195953369, -2.311514139175415, -2.8645107746124268, -2.2832601070404053, -1.270385980606079, 0.2289435714483261, 6.440056800842285, -6.513865947723389, -18.91793441772461, -3.077165126800537, 6.633159637451172, -14.621079444885254, 2.57989764213562, 1.5503997802734375, -7.40998649597168, 0.8881087899208069, 3.0136821269989014, 0.604788601398468, 12.271178245544434, -10.78890609741211, 8.027326583862305, 3.8984339237213135, -22.383180618286133, -3.311591863632202], [10.41301441192627, -0.5970936417579651, 7.327302932739258, -3.1741864681243896, -7.288954734802246, 6.949748992919922, -4.09425163269043, 11.927292823791504, -19.202579498291016, -3.011575937271118, -2.5938093662261963, 10.961804389953613, 0.43505704402923584, -11.029838562011719, 1.6772938966751099, -1.9785524606704712, 21.180179595947266, 0.2408544421195984, 1.5615010261535645, -6.896777153015137, -7.885818958282471, -2.437981605529785, -4.048811912536621, -9.144111633300781, 5.338716506958008, -11.412392616271973, -7.65552282333374, -17.820371627807617, -1.599730134010315, -8.424356460571289, -11.649730682373047, -9.817486763000488, 2.195272922515869, 16.928794860839844, 5.122936725616455, -8.530435562133789, -3.2638564109802246, 8.551815032958984, -12.167353630065918, 2.7503106594085693, 9.296725273132324, 1.2040421962738037, -0.9341467022895813, -0.1565150022506714, -1.3364238739013672, 5.360125541687012, 1.0253664255142212, 13.823932647705078, -4.275777816772461, -8.582761764526367, 4.321347236633301, -0.05445484817028046, 1.0941131114959717, 5.376355171203613, -5.512264728546143, 0.8563316464424133, 2.822282314300537, 7.124510765075684, -5.468517780303955, -0.021523885428905487, -0.7494185566902161, 4.924435615539551, 12.58636474609375, 8.202739715576172, 8.51656436920166, 3.6381373405456543, 1.4002059698104858, -2.268148422241211, -5.1689910888671875, 1.4575084447860718, 3.8519210815429688, 8.222125053405762, -2.9314632415771484, 2.546013355255127, 4.987017631530762, 8.796758651733398, 0.747197687625885, -10.56847858428955, -3.4874110221862793, -1.2254408597946167, 15.446663856506348, 2.3701555728912354, -3.789228916168213, -9.450191497802734, -0.5963413119316101, -1.624488115310669, -2.811184883117676, -3.526089668273926, -12.299118041992188, -7.0378313064575195, -11.64533519744873, -0.582793116569519, -10.899251937866211, -1.451311707496643, 0.12326856702566147, 4.699147701263428, -3.3085105419158936, 6.395288467407227, 2.7033493518829346, -3.1066555976867676, 5.732552528381348, 4.908178806304932, 1.6336815357208252, -10.111770629882812, -12.27704906463623, -3.902954578399658, 3.2328546047210693, -9.079198837280273, -8.412172317504883, -5.2942986488342285, -2.4608209133148193, 8.534186363220215, 10.216570854187012, 11.166289329528809, -5.904487609863281, 2.442666530609131, 0.18355736136436462, 10.141505241394043, -5.313605308532715, 12.946840286254883, 4.862722396850586, 6.164066314697266, 3.5461716651916504, -11.0685396194458, 4.006106853485107, -6.988564968109131, -1.4144434928894043, -0.015626609325408936, 0.2460450977087021], [5.039387226104736, 1.8911093473434448, 5.746490478515625, 0.04928136616945267, 6.502932071685791, -11.543951988220215, 1.5243666172027588, 17.726186752319336, -18.027568817138672, 5.615904808044434, -5.3587965965271, 10.140344619750977, 21.07390022277832, 1.3732678890228271, -2.1856637001037598, -7.119908332824707, 8.523073196411133, -4.099316596984863, -1.1753880977630615, 1.7607187032699585, -6.155836582183838, -2.711897134780884, -6.019434928894043, -5.623899936676025, -15.563647270202637, -3.8358888626098633, -14.021663665771484, -6.492536544799805, -16.033573150634766, 9.761116981506348, 10.294882774353027, -5.006596565246582, 10.474146842956543, -12.84414005279541, -6.285177707672119, -3.870388984680176, -1.6096739768981934, 9.513999938964844, -5.47791051864624, 18.891847610473633, -12.933396339416504, -0.2774623930454254, 7.8754730224609375, 7.566389083862305, -11.077424049377441, -4.347190856933594, 0.6733623743057251, 8.180750846862793, -21.416183471679688, -8.839256286621094, 1.3738969564437866, -4.457159519195557, 13.994499206542969, 0.38041311502456665, 3.3859283924102783, 9.248390197753906, 3.2402565479278564, 8.956491470336914, -3.9317660331726074, -1.9389233589172363, -22.24894905090332, -9.34119701385498, -6.47632360458374, -4.1556715965271, 0.6706567406654358, -3.8575522899627686, -9.200421333312988, -6.767448902130127, -5.903061866760254, 1.1629639863967896, 10.678862571716309, 18.959760665893555, 2.1902480125427246, -0.25050294399261475, 3.55517578125, 4.376931190490723, -2.6730422973632812, 8.403349876403809, -5.964871883392334, -3.8425049781799316, 5.064000606536865, 18.054725646972656, 7.427523136138916, -7.706306457519531, 4.991701602935791, -4.846147060394287, 7.495643138885498, 9.428125381469727, -12.728353500366211, -2.8098766803741455, 1.8833457231521606, 3.3316946029663086, 10.814535140991211, -13.48360538482666, -1.0368812084197998, -11.461649894714355, 4.8762431144714355, 7.284573078155518, -11.379176139831543, -6.97766637802124, -0.12340454012155533, 10.379558563232422, -13.757859230041504, -4.5684919357299805, -9.698929786682129, -6.800594806671143, -1.4816144704818726, 4.7148895263671875, 0.7663740515708923, -21.81251335144043, 7.2905354499816895, 13.19106674194336, 15.8075532913208, -3.872897148132324, -5.25960111618042, 0.6123296618461609, 3.3754642009735107, 8.70528793334961, -0.5292900204658508, 0.508036196231842, 6.845176696777344, -1.026526689529419, -7.145709991455078, 1.2931568622589111, 5.822045803070068, -14.52928352355957, -10.436217308044434, 9.727170944213867, 14.205251693725586], [3.071183204650879, 3.911008358001709, 1.3261754512786865, -12.343241691589355, -3.1780917644500732, 8.554165840148926, -5.893960952758789, -2.272195339202881, -6.011057376861572, -2.397585868835449, 3.1259641647338867, 9.241056442260742, -8.62279224395752, 12.019742012023926, -4.899860858917236, 3.2590839862823486, -8.124051094055176, 6.687947750091553, 8.877854347229004, -3.4665122032165527, -11.235957145690918, 4.128580093383789, 10.318317413330078, -5.6113691329956055, -3.2777903079986572, 14.946637153625488, 17.423580169677734, -0.655885636806488, 6.446566104888916, -4.564235210418701, -17.76123809814453, 13.87834358215332, 15.452329635620117, 17.98368263244629, 3.7687277793884277, -4.683258056640625, -17.683515548706055, 8.68771743774414, 14.911806106567383, -1.8386882543563843, -19.85240364074707, 14.653252601623535, 7.083665370941162, -7.387836933135986, -0.37117886543273926, 7.613528251647949, 10.089380264282227, -0.3261624872684479, 5.380111217498779, 2.591654062271118, -8.491178512573242, 3.9135701656341553, 8.730862617492676, -3.305083751678467, -8.392454147338867, 1.3249092102050781, 15.560406684875488, -5.159383773803711, -10.615080833435059, -16.31673812866211, 0.8739032745361328, 1.079268217086792, -11.232500076293945, 0.85798579454422, 2.9304685592651367, -3.963047504425049, 6.057938575744629, -4.607758522033691, -5.2010579109191895, -0.5747822523117065, -4.337582588195801, -6.215194225311279, 1.7375595569610596, -3.9859955310821533, -5.877678871154785, -6.931724548339844, -3.4278104305267334, -1.5458049774169922, 10.434944152832031, 2.092041015625, 7.69509220123291, -13.09887409210205, -1.193031907081604, -5.349348545074463, -0.8154911994934082, 7.153327941894531, 12.425347328186035, -0.13350056111812592, -7.142996311187744, 0.5077417492866516, -11.016924858093262, 17.992338180541992, -8.61438274383545, 7.289770603179932, 1.4034392833709717, 9.018298149108887, 11.106762886047363, -1.1703548431396484, -4.2653279304504395, 0.614506721496582, 1.6769042015075684, 22.256330490112305, 2.588515281677246, 3.5170397758483887, -0.5698686838150024, -4.489328861236572, 8.251830101013184, -1.8757023811340332, -5.685875415802002, 5.806040287017822, -8.151269912719727, 9.676139831542969, -2.609175443649292, -7.467337131500244, -17.261390686035156, -3.4072229862213135, -4.089344501495361, 11.328123092651367, 1.8643560409545898, 15.006150245666504, 1.5211710929870605, -22.04902458190918, -5.0433735847473145, 8.407079696655273, 11.65993595123291, 6.869521617889404, 4.185870170593262, -2.7914326190948486, -4.610689163208008], [-10.213543891906738, -9.772745132446289, 3.6372125148773193, -5.194963455200195, 19.05509376525879, -7.122189521789551, -6.599236965179443, 4.416412830352783, -0.6267578601837158, -11.395238876342773, 5.496214389801025, 3.7859058380126953, -16.64917755126953, -9.670223236083984, 7.778017044067383, 10.898101806640625, 1.028944969177246, -1.3945902585983276, -0.4226059019565582, -11.786829948425293, 6.3240203857421875, -12.093218803405762, 6.911596775054932, 10.356189727783203, 4.109350681304932, 10.389986991882324, 1.2689638137817383, 7.707691192626953, 2.7989392280578613, -1.6874457597732544, -0.4694678485393524, -8.043107032775879, 5.28196382522583, -6.830027103424072, 6.572398662567139, 11.348289489746094, 4.801520824432373, -0.5040523409843445, 7.241345405578613, -5.800883769989014, 3.5958967208862305, -0.5967461466789246, 1.4951252937316895, -1.6576662063598633, -8.633190155029297, 14.344873428344727, -6.398910045623779, 0.75013667345047, -9.194748878479004, -1.8891762495040894, -5.222470283508301, -1.7498176097869873, -4.993435859680176, -8.101279258728027, -1.2953180074691772, 7.937906742095947, -6.441948413848877, 2.472444772720337, -1.3901301622390747, 3.001868963241577, 8.87062931060791, 0.10259515792131424, -8.262080192565918, -1.7252492904663086, -9.30207633972168, 10.368419647216797, -9.955658912658691, -9.186841011047363, 7.689764499664307, -9.5051908493042, -3.2466793060302734, -14.86990737915039, 1.4406739473342896, 4.966447830200195, 0.0855628028512001, -15.052847862243652, -0.29701903462409973, -4.898632526397705, -1.4820139408111572, -6.3313212394714355, -10.900410652160645, -4.953548431396484, 6.625710964202881, -3.981461763381958, -2.093919038772583, -0.6764617562294006, 3.2664401531219482, 5.618626117706299, 16.29818344116211, -0.28702783584594727, 9.37379264831543, -4.92056131362915, -5.831976413726807, 3.07474946975708, 3.2801501750946045, 6.503673553466797, 14.115238189697266, 12.088420867919922, 7.218619346618652, 8.899454116821289, 4.388213157653809, -8.697443008422852, -8.328948020935059, 0.3504190146923065, 6.402471542358398, -3.573883295059204, 4.59318733215332, 12.193440437316895, 0.6499162316322327, 0.9408986568450928, -8.052324295043945, -10.7197847366333, 3.0494589805603027, -5.0956501960754395, 3.333613157272339, 4.384227275848389, 6.339379787445068, -2.629668951034546, 9.430219650268555, -4.683894157409668, -6.112800598144531, -3.0596494674682617, -4.470912933349609, 2.1920952796936035, 14.64730453491211, -1.8407796621322632, -7.298556804656982, 4.966296672821045, -5.374436378479004], [9.079489707946777, -8.353887557983398, 0.512007474899292, -6.747526168823242, 1.3109091520309448, 6.970152854919434, -2.0444581508636475, 4.412177562713623, -5.97450065612793, -0.09357189387083054, -6.225287437438965, 0.8135582804679871, 22.18126678466797, -7.195534706115723, -2.237154722213745, -4.618050575256348, -11.451576232910156, 1.2689896821975708, -2.5200607776641846, -2.6445531845092773, -6.447037696838379, -5.11764669418335, -1.0276315212249756, -4.668554782867432, 0.7828950881958008, 0.35851559042930603, 10.303339004516602, 1.9462300539016724, 0.8466168642044067, 3.028075933456421, 4.966325283050537, -8.410035133361816, 3.7865681648254395, 1.0720329284667969, 7.152516841888428, 7.986616611480713, -6.294370651245117, -0.9807337522506714, 11.611738204956055, 5.377025604248047, 1.0706987380981445, -9.358336448669434, -6.4911417961120605, 1.0257824659347534, -2.8858025074005127, -14.147512435913086, -4.812898635864258, 6.898403167724609, -19.885013580322266, 2.736950159072876, 8.101763725280762, -5.7975592613220215, 6.624491214752197, 0.4300405979156494, 11.296215057373047, -18.598485946655273, 13.025279998779297, -8.285643577575684, -8.092767715454102, -6.831384658813477, -12.61359977722168, 0.26847630739212036, 2.799112319946289, 0.17949044704437256, 5.932723045349121, -8.89161491394043, 5.987717628479004, -7.425873756408691, -10.557741165161133, -7.3346076011657715, 1.3798058032989502, -8.562795639038086, 5.27970027923584, -3.698167562484741, 11.682302474975586, -6.3430585861206055, -8.06077766418457, 9.313772201538086, -8.102478981018066, 7.448241233825684, 7.730661869049072, -5.394183158874512, -3.56270694732666, -0.3591548800468445, 3.089245319366455, -2.0917816162109375, 3.287062168121338, 12.937082290649414, -11.619780540466309, -5.7381696701049805, -6.584042072296143, -13.560446739196777, -8.986952781677246, -7.203116416931152, 4.761765956878662, -8.073561668395996, 11.503411293029785, 14.327651023864746, 5.776437282562256, 0.4818859398365021, -6.236073970794678, 11.47792911529541, -0.43890219926834106, 14.428086280822754, -2.786956548690796, 12.37561321258545, 13.929805755615234, -3.205014705657959, -0.07924593985080719, -5.4290289878845215, 9.370835304260254, 6.778519153594971, 3.06006121635437, -8.382054328918457, -6.603967666625977, -7.037152290344238, -5.968720436096191, 6.320972919464111, -8.000589370727539, -15.19407844543457, 2.911104440689087, -7.648589134216309, -14.359318733215332, 2.173527956008911, 2.263951539993286, 16.28534507751465, 6.473496437072754, 11.829538345336914, -0.6905221939086914], [-0.6410174369812012, 6.492247581481934, -13.7935791015625, -1.5845129489898682, 7.72495174407959, 7.6570024490356445, 3.865798234939575, 6.01779317855835, -5.183935165405273, -2.105111837387085, -9.421422958374023, -4.275885581970215, 5.560055255889893, 8.454733848571777, -2.969745397567749, 0.7143945693969727, 10.390649795532227, -12.97954273223877, 5.530386447906494, 5.447455406188965, -7.88400936126709, 0.27463582158088684, 4.895459175109863, -2.199585437774658, 0.9847534894943237, 1.3188045024871826, -9.674022674560547, -12.917318344116211, -5.645757675170898, -7.585055828094482, 3.974133253097534, -0.9583151936531067, -0.4658692479133606, -1.6784439086914062, 10.08249568939209, 2.690002202987671, 0.9551578164100647, 9.271449089050293, 4.268089771270752, 5.096212387084961, -8.970184326171875, 0.40029284358024597, -4.2932844161987305, -3.516540765762329, 10.12392807006836, -6.079987049102783, -2.031903028488159, -10.1038818359375, -7.370347499847412, -10.568486213684082, 3.9532926082611084, 10.98183822631836, -7.064932346343994, -7.243023872375488, 13.311117172241211, -13.456934928894043, 1.572288155555725, 14.15554141998291, -14.853394508361816, 2.013972759246826, 7.679110527038574, -4.80696964263916, 0.8826425075531006, 6.473814010620117, 17.618595123291016, 4.497232437133789, 6.266887187957764, 2.912653923034668, -1.41569185256958, 5.139676094055176, 0.8120023608207703, -2.451852321624756, -0.9654395580291748, -1.3054211139678955, -9.564506530761719, 1.5991846323013306, -8.274209976196289, -4.325206756591797, -1.5238463878631592, 2.163400888442993, 2.6022446155548096, -2.2146551609039307, 5.883941173553467, 2.8318021297454834, -5.753582000732422, 7.342097759246826, -5.944568157196045, -1.8756723403930664, 5.241030693054199, -5.593552589416504, -6.6614990234375, 1.4529969692230225, 0.03798259422183037, 6.649677753448486, 9.3438720703125, -1.3756327629089355, -5.125306129455566, 6.377490520477295, 4.012584209442139, 4.384512901306152, -2.0894157886505127, -2.6725573539733887, -1.9614027738571167, -16.236528396606445, 2.58644962310791, 3.512305498123169, -4.652629375457764, 8.05917739868164, -4.807229042053223, -16.759206771850586, 7.209191799163818, 4.950892448425293, 14.56273078918457, 5.618255138397217, -6.09476375579834, -4.170230865478516, -8.702900886535645, 12.349710464477539, 0.0544540099799633, -1.7997652292251587, 4.171083927154541, -2.6046645641326904, 6.243135929107666, -1.810491919517517, 1.3767722845077515, -1.9493441581726074, -15.264213562011719, -7.986267566680908, 10.473957061767578], [4.728018283843994, 0.686706006526947, 10.356193542480469, -11.207799911499023, -3.750274419784546, 1.2608970403671265, -4.556550025939941, 5.026976108551025, -7.828235626220703, 6.4171905517578125, 0.5351527333259583, 10.790936470031738, 9.840364456176758, -0.03488754481077194, -5.649301052093506, 1.763612151145935, 11.88092041015625, 6.773842811584473, -7.54590368270874, -1.1675158739089966, -5.279369831085205, 1.4670460224151611, -3.859050989151001, -2.04799485206604, -5.202701568603516, -15.847877502441406, -0.5326197147369385, -0.8054091930389404, 11.925714492797852, 6.519773960113525, -8.355271339416504, 0.2510143518447876, -2.500270366668701, -5.053205490112305, 7.5917067527771, -6.539449214935303, 5.973978042602539, 12.785577774047852, -6.773431301116943, 10.999248504638672, -6.2636895179748535, -0.9640445113182068, 9.201959609985352, -16.498159408569336, 0.589439332485199, -6.930058002471924, 3.494875907897949, 20.628915786743164, 2.4834630489349365, 9.289657592773438, -8.964234352111816, -9.824434280395508, -3.033424139022827, 10.361226081848145, -4.693983554840088, 12.169785499572754, 5.499941825866699, 12.460685729980469, -4.198568820953369, 0.7264371514320374, 0.20648135244846344, -8.553613662719727, 5.589381694793701, 3.9935357570648193, -9.58493709564209, 3.3544139862060547, -4.097482681274414, 8.88627815246582, -5.603670120239258, -1.0679363012313843, -5.398667812347412, 15.110518455505371, 5.134968280792236, 7.318657875061035, 9.342668533325195, -5.84663200378418, 3.902745008468628, 11.143280982971191, 7.075985908508301, -10.92616081237793, -0.021839870139956474, 7.7525482177734375, -5.095431804656982, -12.147872924804688, 8.421112060546875, -0.6720571517944336, 5.814560890197754, -12.67404842376709, -1.6985414028167725, -7.837949752807617, 6.42389440536499, 7.632226467132568, 9.358139991760254, 3.0374248027801514, -5.859872817993164, -1.6147212982177734, 18.138872146606445, -4.174981594085693, -7.567482948303223, -16.355920791625977, 0.8289328217506409, 1.4898701906204224, -18.790813446044922, 8.972315788269043, -2.36221981048584, -18.354990005493164, -2.6162209510803223, -8.902774810791016, -3.008538007736206, 10.008150100708008, -5.66605281829834, 1.9992549419403076, 3.5376179218292236, -9.794116020202637, 1.743204116821289, 15.96340560913086, 5.119858741760254, 2.9400861263275146, -8.21581745147705, 9.132074356079102, -1.423628807067871, 6.170882225036621, 6.364378929138184, 0.21623595058918, 3.3713080883026123, -3.282942771911621, -7.576388835906982, 3.0812463760375977, 1.0090957880020142], [-0.19636499881744385, -8.070757865905762, 7.817574501037598, -2.1248414516448975, -4.633538246154785, 8.475251197814941, -13.726394653320312, 8.172588348388672, -3.916865587234497, -9.773064613342285, -5.626979827880859, 1.3465831279754639, -12.99432373046875, -15.472853660583496, 5.229475021362305, -4.705290794372559, 10.720415115356445, -1.1165698766708374, 9.934511184692383, -12.97913646697998, -1.0619288682937622, 1.346799373626709, -6.882138729095459, 7.122252941131592, 8.789328575134277, -8.618910789489746, -7.731328964233398, -8.04931926727295, 0.9086948037147522, -4.444227695465088, -5.236112594604492, -0.6841152906417847, 17.149131774902344, 12.608317375183105, 1.02825927734375, -2.839585065841675, -4.035595417022705, 8.685964584350586, -14.066164016723633, 13.284400939941406, 2.393092393875122, 7.613372802734375, 4.386139392852783, 3.8510220050811768, 14.502809524536133, -7.335379123687744, -13.518704414367676, 3.174551486968994, 3.2699437141418457, 8.66782283782959, -3.3299293518066406, -0.4040590822696686, 7.043169975280762, 9.196673393249512, -9.58299732208252, 1.2247289419174194, 3.311824083328247, -5.4948015213012695, -1.8323506116867065, 4.975933074951172, 24.772001266479492, -2.359395742416382, 8.090553283691406, -6.19973611831665, 11.209300994873047, -1.7593060731887817, 2.2984421253204346, 4.928664207458496, 2.9894492626190186, 4.056798934936523, 20.233078002929688, 5.987078666687012, -2.3856382369995117, 3.2702136039733887, -4.664890766143799, -0.3883250653743744, 7.109766483306885, -11.336828231811523, 7.296651363372803, -1.5844752788543701, 14.918800354003906, -7.658449172973633, 2.2545166015625, -7.976416110992432, -5.318811893463135, -9.659260749816895, -11.51351261138916, -11.848339080810547, -14.989763259887695, 0.9284219145774841, -10.756590843200684, 7.659836769104004, -1.2978570461273193, 2.829946279525757, 6.501248836517334, 3.743006944656372, -12.79725456237793, -3.391533136367798, -17.49515724182129, -0.5416390299797058, 14.009442329406738, 1.4146177768707275, 6.095032215118408, -6.472248077392578, 0.3090110421180725, -0.2281067967414856, 7.734643459320068, 3.502368688583374, 5.371894359588623, -5.017348289489746, 13.765264511108398, 1.6782511472702026, -0.08222411572933197, 9.206754684448242, -6.887424945831299, -13.618058204650879, 0.38598912954330444, 3.9525585174560547, 4.440751552581787, -1.750339150428772, -4.1419501304626465, -7.865538597106934, 12.422740936279297, -8.474664688110352, -1.8190693855285645, -14.943361282348633, -4.179159164428711, -2.585817575454712, -7.012261390686035], [7.516685962677002, -4.548935890197754, 7.621583461761475, -8.317802429199219, -0.5622900128364563, -5.389669418334961, -15.437992095947266, -1.4253394603729248, -3.8881869316101074, -4.472346782684326, -11.69722843170166, -1.7416956424713135, -4.590984344482422, -2.886873960494995, -0.7823160886764526, 9.144304275512695, 7.061916828155518, 4.079648017883301, -11.791728019714355, -14.767447471618652, 10.898458480834961, -4.600451469421387, -11.727227210998535, 6.296877861022949, 13.42750358581543, -15.723423957824707, 5.20793342590332, 4.772805690765381, 7.947495460510254, 18.320938110351562, -1.7346444129943848, 1.2324318885803223, 8.784348487854004, 13.670441627502441, 10.447383880615234, 11.690855026245117, -16.461528778076172, 6.4280853271484375, 8.686408042907715, -10.190382957458496, 11.171062469482422, -9.24731731414795, -5.51881217956543, 10.961790084838867, 7.687557220458984, -0.006912665441632271, 13.42777156829834, -3.499154567718506, -1.1407530307769775, 5.445376396179199, -8.75074291229248, 7.91364860534668, -2.234100103378296, 15.6662015914917, -2.4478042125701904, -3.98714542388916, -8.479595184326172, -4.183772563934326, 8.8447847366333, -5.05231237411499, -7.462826728820801, 1.2584563493728638, -1.1954982280731201, 7.0729570388793945, -14.826659202575684, 12.802120208740234, -0.1716526448726654, -0.7795450687408447, -4.091967582702637, -15.42533016204834, -1.0105435848236084, 5.823328971862793, -8.817132949829102, 0.15928713977336884, 1.9400314092636108, 6.322239875793457, 8.232406616210938, 2.7125048637390137, -8.922722816467285, -19.18157386779785, -5.929293155670166, 7.859954357147217, 6.772163391113281, -10.748344421386719, -8.798138618469238, -6.797789573669434, 10.037001609802246, 2.419719696044922, -3.649231195449829, -4.527358055114746, -18.00772476196289, -8.77676773071289, -2.351254940032959, 4.5235724449157715, -1.5898957252502441, 22.568960189819336, 6.346919536590576, 0.4516548812389374, 10.539811134338379, -3.7750625610351562, -5.309778213500977, 2.198030471801758, -14.585700988769531, 0.6337473392486572, 3.0686328411102295, -11.473950386047363, -1.4515057802200317, -23.299087524414062, -8.1492338180542, -7.788416385650635, -8.10645866394043, -1.914046049118042, -2.0051612854003906, 9.54152774810791, 13.56159782409668, -9.01684284210205, 23.82074737548828, 5.859258651733398, 0.5285751819610596, -2.406242609024048, -1.570027232170105, -5.363804340362549, -4.918492317199707, -7.112661838531494, 4.121891498565674, -4.046876907348633, 2.467210292816162, 22.152576446533203, -1.0309653282165527], [1.036714792251587, 12.219956398010254, -1.0966304540634155, 0.7561795115470886, 6.952552795410156, 14.215766906738281, -2.299328327178955, 11.572488784790039, -5.524105072021484, 14.936019897460938, 3.128222703933716, -4.413882732391357, 12.16655445098877, 0.77056485414505, -4.380369663238525, 6.356464385986328, 6.757201194763184, -4.931560039520264, -5.376448154449463, 6.597166061401367, 3.0607903003692627, 1.435452938079834, 17.7719669342041, 11.3765230178833, -2.1552340984344482, -10.648715019226074, 7.470142841339111, -5.900190353393555, -9.93233871459961, -13.026437759399414, -10.469554901123047, -4.876303672790527, -3.3061935901641846, -13.820459365844727, 2.741953134536743, -3.1918580532073975, 4.521373271942139, -6.737820625305176, 1.71526038646698, 17.48868179321289, -3.0577774047851562, 9.782422065734863, 6.234308242797852, -5.758016586303711, -2.8280317783355713, -9.888672828674316, -2.8264262676239014, 0.9767603874206543, -19.74276351928711, -2.5058863162994385, 3.6487224102020264, -18.62274169921875, -8.857549667358398, -3.5736005306243896, 1.0494592189788818, 9.361043930053711, -8.292181968688965, -4.5171799659729, -4.834504127502441, -14.458708763122559, 15.755475997924805, 10.867595672607422, 4.547604560852051, -8.18574333190918, -4.29587459564209, 0.09920402616262436, 0.09113773703575134, -4.887598037719727, 4.6472697257995605, -4.061081409454346, -0.14547234773635864, -15.878277778625488, 0.07024787366390228, 8.80675220489502, 4.154725074768066, 4.154179096221924, -13.446073532104492, 14.264436721801758, 3.288113594055176, -3.4177486896514893, -12.472183227539062, 10.690412521362305, 9.812202453613281, 12.184234619140625, 4.162132740020752, -6.854391574859619, 14.716378211975098, 4.349005222320557, 14.77023696899414, 10.834174156188965, -1.271186351776123, -10.146446228027344, 1.4371709823608398, 3.6275880336761475, 16.18390655517578, -12.804699897766113, -12.69973373413086, 16.293386459350586, -0.97913658618927, 0.8281688690185547, 5.422039031982422, -9.752522468566895, -4.920480251312256, 5.098063945770264, 11.890652656555176, 18.917116165161133, -3.5597641468048096, -1.4559643268585205, 7.899921417236328, -17.435678482055664, -17.477643966674805, -2.697965621948242, 9.027904510498047, -14.22718620300293, -5.431719779968262, -10.1904878616333, -4.750214576721191, -5.732408046722412, 0.6630319952964783, -0.8180151581764221, -5.97789192199707, -0.9152924418449402, -13.869961738586426, 2.0624961853027344, 2.568155527114868, -5.235831260681152, -11.939714431762695, -1.2533445358276367, -6.830318450927734], [-8.437947273254395, -4.438193321228027, -3.019591808319092, -1.1525816917419434, -7.186976909637451, -1.6268329620361328, 7.525887966156006, -0.7603798508644104, -0.26704782247543335, 14.929281234741211, 10.50488567352295, -4.140890121459961, -0.38426560163497925, 0.7909294366836548, -3.5719492435455322, 1.6140424013137817, -3.289746046066284, -4.624179840087891, -5.564741611480713, 1.540021538734436, 7.949014663696289, 9.630721092224121, -3.3150267601013184, 17.965150833129883, -10.697450637817383, 1.9215909242630005, 17.59496307373047, -4.584354877471924, 9.939045906066895, -5.367850303649902, 6.671411514282227, -14.955366134643555, -2.4261786937713623, -15.199000358581543, 3.758565664291382, -2.2234065532684326, -3.435270309448242, -1.7530028820037842, 19.28122329711914, -2.1131012439727783, -2.0072288513183594, 9.992311477661133, 10.851215362548828, -2.352203130722046, -0.7414487600326538, 6.814877033233643, -2.4211435317993164, -0.8895528316497803, -2.954598903656006, -5.403650760650635, -10.788175582885742, -6.998592853546143, -0.19219668209552765, 8.699121475219727, 1.6924654245376587, 5.736879348754883, 8.436742782592773, -4.513740539550781, 12.96012020111084, -4.551846981048584, 18.625099182128906, 7.508305072784424, -13.373929023742676, -16.844467163085938, -7.926428318023682, 1.7049626111984253, -10.252374649047852, 7.186034679412842, 0.19361868500709534, 17.42766571044922, 0.8981364965438843, 2.5438458919525146, 6.476863384246826, 19.118852615356445, 0.14125318825244904, 13.2163724899292, -1.3841991424560547, 15.730393409729004, 5.2063446044921875, -1.1048667430877686, 4.547585964202881, 16.74334716796875, 16.859643936157227, 4.487949848175049, 5.978513717651367, 9.568731307983398, -2.4895074367523193, -19.221088409423828, 17.15180206298828, 3.987821340560913, -5.747067451477051, -0.30104008316993713, 4.005257606506348, -0.7410715818405151, 11.071136474609375, 11.37695598602295, 3.095155715942383, -10.930825233459473, 1.5248452425003052, 8.286600112915039, 19.338254928588867, -11.443402290344238, -3.4278886318206787, 6.720655918121338, 10.525796890258789, 1.9743338823318481, -16.897371292114258, 9.071974754333496, 16.192968368530273, 7.143901824951172, -0.865524411201477, -8.233309745788574, -0.21560069918632507, -12.124604225158691, 0.8384742736816406, -6.158162593841553, 3.537851095199585, -14.430530548095703, 6.534862518310547, -2.7419073581695557, -0.3422456383705139, -3.148820638656616, 7.945767402648926, 4.728447914123535, 8.932358741760254, -21.536792755126953, 2.730557918548584, -17.03343391418457, -1.3932640552520752], [3.5950186252593994, 14.604832649230957, -2.5525519847869873, 12.871557235717773, -2.0809054374694824, 4.21814489364624, 0.07715918868780136, -3.491813898086548, -1.839138150215149, 18.198129653930664, 12.274995803833008, 5.8474440574646, -9.010411262512207, 12.498927116394043, -14.075530052185059, -7.013121604919434, 8.589690208435059, -3.655205011367798, 8.777548789978027, 13.298276901245117, -0.602500319480896, -7.531973838806152, -1.5240823030471802, 2.3848350048065186, -7.356501579284668, 7.720129489898682, -6.39420747756958, -11.658790588378906, -0.49683117866516113, 0.4192717373371124, 0.06469201296567917, -23.41900634765625, 2.626108407974243, 0.125708669424057, -8.721912384033203, -21.40226173400879, 3.2796313762664795, -7.466689586639404, 9.22204303741455, 0.7802788615226746, -20.336273193359375, -0.13973651826381683, -4.897724151611328, -3.8438589572906494, -17.98224639892578, 1.9234567880630493, 17.1895809173584, 3.6556947231292725, 0.1756981462240219, -8.336289405822754, 9.36393928527832, -1.7392405271530151, 1.9196382761001587, -6.765618801116943, -4.8366289138793945, 4.857348442077637, -4.8791069984436035, -3.3543941974639893, 10.923537254333496, -5.395845413208008, -7.553770065307617, 12.430912971496582, -0.9669392704963684, -2.2348947525024414, -2.895742177963257, 5.305548667907715, 8.878236770629883, -16.00929069519043, -21.116907119750977, 4.397096157073975, -17.757104873657227, -0.620920717716217, 5.999361991882324, -8.848838806152344, 9.091242790222168, 6.837294101715088, -3.1989986896514893, -14.47208309173584, -6.159947872161865, -6.369010925292969, -11.088713645935059, 8.52379035949707, 12.146576881408691, -1.9591161012649536, 3.2722721099853516, -7.543387413024902, 5.659660816192627, -0.910182774066925, 7.814912796020508, -1.625802755355835, 14.360737800598145, -1.8407272100448608, 11.592182159423828, 6.366976261138916, 3.5390586853027344, -8.69416332244873, 3.7383360862731934, -6.9518141746521, -8.054295539855957, 6.9679694175720215, 1.6662551164627075, -2.572299003601074, -14.583895683288574, -2.3255581855773926, -4.805671215057373, -13.47940444946289, 6.866913318634033, 11.444202423095703, -5.707910537719727, -7.873070240020752, -11.467288970947266, 4.2544331550598145, -5.191387176513672, 6.633574962615967, -3.1035573482513428, 4.579442977905273, -2.4273107051849365, -8.135175704956055, 4.274130821228027, 3.41990065574646, 2.8214526176452637, -2.0232458114624023, 6.392439365386963, 1.8241515159606934, 15.962130546569824, 13.907472610473633, 7.967010498046875, -10.725414276123047, 1.7401694059371948], [3.830960512161255, -10.172043800354004, -4.601187229156494, 1.979129433631897, 5.512380599975586, -7.8727312088012695, -3.435138702392578, -5.681320667266846, 6.514980792999268, 9.47907829284668, 9.896562576293945, -17.94791603088379, -3.1677911281585693, -5.3705549240112305, 0.8909698128700256, 8.574753761291504, 3.815028429031372, -5.815995216369629, 0.23539936542510986, -15.540310859680176, -3.213935613632202, 8.357244491577148, 1.6900229454040527, 4.125820636749268, -4.336019515991211, 6.812613487243652, 15.643163681030273, 5.455349922180176, 2.3071141242980957, -13.88177490234375, 10.319091796875, -2.022991180419922, -11.727140426635742, -8.472585678100586, -4.509875774383545, -10.270691871643066, 0.06483566761016846, 6.034533977508545, -11.869363784790039, -5.2428388595581055, 2.3445324897766113, 2.373906135559082, 14.691789627075195, -3.891045093536377, -12.769579887390137, -5.149698257446289, -11.874313354492188, -0.7565053105354309, -7.263856410980225, 8.426444053649902, -1.7443678379058838, 3.437757730484009, -6.269486904144287, -2.952632427215576, -5.408627510070801, -3.0856239795684814, -5.040748119354248, 5.4640398025512695, 7.890450954437256, 1.2997887134552002, -16.84811782836914, -10.315683364868164, -1.019776701927185, 0.9672828912734985, -2.888657569885254, -11.760924339294434, -0.8353662490844727, 14.076473236083984, 10.520559310913086, -4.243021488189697, -1.2895771265029907, 3.732465982437134, 16.023984909057617, 4.098525047302246, -5.1753458976745605, -7.114352226257324, -3.7195723056793213, 14.547239303588867, -1.9025479555130005, 5.647647857666016, 7.179201602935791, -14.233683586120605, -4.345856666564941, -0.8193032741546631, -7.234686374664307, -1.0226181745529175, 0.6547477841377258, -11.441100120544434, -0.7559496760368347, -1.1820544004440308, -2.4865148067474365, 6.177733421325684, -6.725617408752441, 11.23249626159668, 5.028371810913086, 6.706701278686523, 12.036288261413574, 7.169107913970947, -1.6957250833511353, -8.579950332641602, 7.734447479248047, 17.516321182250977, -1.663634181022644, 2.6946139335632324, -6.104401588439941, -7.0683817863464355, -1.135912299156189, 11.929595947265625, 6.0761518478393555, 8.664122581481934, -4.7510857582092285, -13.379293441772461, 1.6938064098358154, -17.813373565673828, -12.963033676147461, 14.544589042663574, -6.854837417602539, 3.512054443359375, 4.769372940063477, -3.380861282348633, -13.334932327270508, 14.53518295288086, -6.862189292907715, 4.433294773101807, 2.1189963817596436, 0.0689053162932396, 0.48087257146835327, 2.43497896194458, 5.156583309173584], [2.2000603675842285, 7.705297946929932, 5.732945442199707, -14.93616771697998, 4.5264105796813965, -4.61874532699585, -0.5172943472862244, 5.060572624206543, -8.360810279846191, -12.682695388793945, -3.5047128200531006, -11.077845573425293, 0.969362735748291, 8.50827693939209, 1.6873399019241333, 5.079326152801514, -12.316818237304688, 0.1324182152748108, -6.7480621337890625, -1.6346958875656128, -11.831893920898438, 4.910882472991943, -12.27742862701416, -2.7638514041900635, 12.14802360534668, -1.4389349222183228, 3.659280776977539, 3.834669589996338, -0.9184336066246033, 4.594689846038818, -3.243520736694336, -1.3942408561706543, 3.282015800476074, 7.059661865234375, -9.336668968200684, 5.675784587860107, -18.441509246826172, 7.414105415344238, 10.660076141357422, -9.832664489746094, -9.405264854431152, -4.80848503112793, -3.343013048171997, -3.2342512607574463, 5.016415596008301, 11.569087028503418, 7.897380828857422, -5.615847587585449, -3.151427984237671, 2.805816650390625, -9.926985740661621, -9.281648635864258, -10.041451454162598, -2.3150999546051025, 1.8168567419052124, -8.159406661987305, 6.483062267303467, 10.025193214416504, 2.4862823486328125, 6.943490505218506, 9.119751930236816, 17.61467742919922, 8.792115211486816, 9.611275672912598, 10.683541297912598, 12.191856384277344, 6.186417102813721, -4.187228202819824, -13.241729736328125, -15.827958106994629, 1.8678780794143677, -4.559886455535889, -10.918342590332031, -3.224865674972534, 0.34591758251190186, -10.375197410583496, 2.3362905979156494, 1.2854793071746826, 3.711061716079712, -16.12973976135254, -9.041471481323242, -6.339448928833008, -10.898240089416504, -2.56477689743042, 7.350432872772217, -4.9217658042907715, 12.391145706176758, -4.415930271148682, -1.4220010042190552, -11.060543060302734, -14.538022994995117, 7.254526615142822, 0.9335082173347473, -3.232492446899414, -14.41850757598877, 18.163244247436523, 26.945159912109375, -7.241065502166748, 0.6069490313529968, -1.3594602346420288, -5.1664276123046875, -8.735699653625488, -6.612378120422363, -10.430853843688965, 4.5049147605896, -3.6831469535827637, 14.83400821685791, -3.722475290298462, -3.9030842781066895, 10.477084159851074, 4.840954303741455, 0.21994401514530182, -4.732968807220459, -2.300131320953369, 1.832672119140625, -5.851876735687256, 7.456851959228516, 0.1640498787164688, 17.807472229003906, -4.155445098876953, 0.2977038025856018, -2.7775063514709473, -0.18977296352386475, -9.604082107543945, 7.478837013244629, -1.9281809329986572, 0.35407590866088867, 7.761347770690918, 19.77992820739746], [-3.893167495727539, 5.584774017333984, 2.4784462451934814, -6.973666667938232, -2.8392720222473145, 6.468620777130127, 2.1504359245300293, 5.6227827072143555, 3.8307950496673584, -2.285734176635742, -4.985080718994141, -11.752799034118652, 6.4761576652526855, 7.594186782836914, -2.7041335105895996, 1.935827374458313, -2.9381110668182373, -4.393119812011719, 1.7618454694747925, 14.697677612304688, 1.7452669143676758, 15.461163520812988, 1.3563286066055298, -0.7665387392044067, 6.518099784851074, -0.11218500137329102, -0.09684070199728012, -5.7427897453308105, 3.66025447845459, 5.453636169433594, 3.503995656967163, 0.8542804718017578, -11.062362670898438, -8.591383934020996, 4.134310245513916, 3.5412564277648926, -14.865342140197754, 1.3296409845352173, -0.5630211234092712, -5.152388572692871, 4.050876140594482, 1.466380000114441, -3.8752195835113525, 0.7420176863670349, -1.1266238689422607, 3.475334882736206, 9.949111938476562, -2.6677002906799316, -4.808971405029297, 7.055942535400391, -0.5811748504638672, -3.8056650161743164, -1.6014295816421509, 0.6983318328857422, 2.3790767192840576, -7.910884857177734, 2.6945178508758545, -3.552152395248413, -3.2929558753967285, 1.2202240228652954, -10.341872215270996, 2.708401679992676, -9.551286697387695, 4.067528247833252, 2.9937000274658203, -11.432110786437988, 6.312075138092041, 2.8342032432556152, 14.838164329528809, -3.6118454933166504, -19.760221481323242, -2.7544376850128174, -3.8282508850097656, -2.089752197265625, -1.7959144115447998, 7.320010185241699, -9.117730140686035, 8.69233512878418, -1.5016635656356812, 12.575361251831055, -6.616334915161133, 6.285046577453613, 5.639002323150635, 12.056739807128906, 2.840106248855591, 2.0115396976470947, -4.252003192901611, -4.991713523864746, 0.48791101574897766, 6.601738452911377, -4.348200798034668, -5.732137680053711, -8.81094741821289, -7.706872940063477, 7.03969144821167, 0.052731577306985855, -6.055176734924316, 2.544541835784912, 3.6740784645080566, 1.220955729484558, 7.568427085876465, 9.13684368133545, 4.506016254425049, 0.1480250358581543, -1.8747040033340454, 5.671968936920166, -9.895471572875977, -9.260275840759277, -9.148820877075195, -8.550479888916016, -7.109758377075195, 0.3300258219242096, 0.5108170509338379, -1.984352469444275, -2.1999058723449707, -7.745291233062744, -6.073579788208008, -6.580970764160156, -1.8424594402313232, 2.836484670639038, 6.095068454742432, -3.342635154724121, 6.7802557945251465, -4.9622368812561035, -2.9063663482666016, 6.279968738555908, 4.734807968139648, -0.5928975343704224, 0.6299611926078796], [6.540127277374268, -3.8003504276275635, -3.441615581512451, -8.700946807861328, -4.730373382568359, -4.375922203063965, -2.266433000564575, -5.492106914520264, 14.025793075561523, -0.73600172996521, -13.271899223327637, 6.1812357902526855, -0.0796232745051384, -6.5154595375061035, -12.099081039428711, 10.549574851989746, -12.55274486541748, 2.045369863510132, -2.449920892715454, -8.027501106262207, -10.736176490783691, -16.919431686401367, -2.750912666320801, 6.226654052734375, 5.866939544677734, -4.349186897277832, -0.7828125357627869, -6.074240207672119, -9.782219886779785, -7.549977779388428, 2.403456211090088, 7.456583023071289, 2.603466510772705, -9.12666130065918, -5.583480358123779, 18.948471069335938, 4.378508567810059, -0.5702248215675354, 12.284640312194824, -2.494886636734009, -6.518011569976807, -2.882183790206909, 3.4329636096954346, -0.23529334366321564, -9.708475112915039, -11.49351978302002, 11.222283363342285, -6.8330278396606445, -3.5478813648223877, -8.427375793457031, 2.7458603382110596, 4.652485370635986, -10.376012802124023, -5.470135688781738, 7.942385673522949, -16.699817657470703, 3.132450580596924, -10.377605438232422, -1.0660088062286377, -4.281589508056641, -12.362935066223145, -14.240520477294922, -8.100944519042969, 15.588088989257812, -1.2761995792388916, -5.823668479919434, 8.098983764648438, -9.541926383972168, -5.866044044494629, -7.670605659484863, -4.177548885345459, -12.329168319702148, 4.760288238525391, 1.3377264738082886, -19.094493865966797, -7.6293158531188965, -13.366642951965332, -3.4776947498321533, -4.374790668487549, 0.09080954641103745, -2.2314634323120117, 4.404365062713623, -4.279674053192139, 5.6887526512146, -7.336945056915283, 2.6014821529388428, 1.223497748374939, 13.36578369140625, 7.311731815338135, -3.7003390789031982, 0.5296270251274109, 5.768086910247803, 0.15823033452033997, -1.294385313987732, -5.083190441131592, -3.1575069427490234, 3.3561415672302246, 7.008522987365723, 5.427495956420898, 1.4037405252456665, -2.0392849445343018, 2.251124858856201, -10.982502937316895, 7.744103908538818, 13.862203598022461, -1.8018869161605835, 17.123544692993164, -16.063196182250977, 19.345762252807617, -12.071329116821289, -1.746621012687683, -11.541956901550293, -0.36571186780929565, 7.230832099914551, -7.371156215667725, -8.693378448486328, -12.46103572845459, 15.586584091186523, 1.2497317790985107, -15.059597969055176, -1.5464602708816528, 5.82896089553833, -5.783443927764893, 1.3435059785842896, -9.22586441040039, 11.493314743041992, 2.5002753734588623, -8.630884170532227, -10.5269136428833], [1.2398548126220703, 24.183786392211914, -11.505260467529297, 30.76658058166504, 14.80671215057373, 3.1209096908569336, 16.141489028930664, 8.115036964416504, 16.043785095214844, 9.225915908813477, 13.423332214355469, -6.087215423583984, 16.21882438659668, 8.012384414672852, -3.9790990352630615, 5.592165946960449, -11.074576377868652, 11.26750659942627, -7.446318626403809, 9.116081237792969, -2.463815927505493, -15.31554889678955, -1.7856289148330688, 6.088702201843262, -4.078255653381348, 1.4586851596832275, 7.1006293296813965, 14.424864768981934, 10.101966857910156, -13.882503509521484, 8.165141105651855, -17.371374130249023, 4.563418865203857, 1.043925404548645, -6.00181245803833, -12.7899808883667, 13.10869312286377, -16.164226531982422, 2.80208420753479, 18.30363655090332, -4.446416854858398, 1.9448648691177368, 5.508249282836914, -10.5255708694458, -6.314372539520264, 3.052704334259033, -1.0308151245117188, 0.4280271828174591, 10.308523178100586, -4.772789478302002, 11.183018684387207, -7.642571926116943, -1.8636431694030762, -3.4997358322143555, 11.471227645874023, 8.248991966247559, -2.6112937927246094, 11.576666831970215, 4.657585620880127, 4.4661407470703125, 12.744114875793457, 2.944580554962158, 10.849600791931152, 0.9802186489105225, -1.7398144006729126, -0.05648435652256012, 15.161551475524902, 8.30223274230957, -3.5334765911102295, 7.222605228424072, 1.6825590133666992, -20.483327865600586, 23.894779205322266, 7.5240583419799805, 9.645572662353516, 9.625092506408691, -5.758243083953857, -6.955770492553711, -5.570065498352051, 3.15421986579895, 6.1622114181518555, 7.6067986488342285, -4.089744567871094, 4.118690013885498, 8.756388664245605, -6.435326099395752, 6.382329940795898, -8.421977996826172, 16.345821380615234, -0.7095080018043518, 0.37166452407836914, 2.191270351409912, 12.054414749145508, 10.718727111816406, -2.722046136856079, -12.161693572998047, 14.544777870178223, -15.698293685913086, -10.313543319702148, 9.480430603027344, -3.3195602893829346, -6.88104772567749, -9.711119651794434, 5.2172064781188965, -1.6594891548156738, -8.051697731018066, -2.6875061988830566, 14.724430084228516, 4.515533924102783, 5.0467047691345215, -9.532890319824219, -18.369653701782227, -3.3663930892944336, -2.2135112285614014, -17.926511764526367, 14.56049633026123, -8.860114097595215, -21.394899368286133, 9.777674674987793, -3.3010873794555664, -1.6457306146621704, 14.08366870880127, 4.213522434234619, -4.240219593048096, -5.844348430633545, 6.604557991027832, 10.310654640197754, -2.311534881591797, 5.831725120544434], [-0.49814504384994507, 14.479622840881348, -2.1243796348571777, 2.896135091781616, -3.6678154468536377, -0.7622127532958984, 1.3884837627410889, -12.488103866577148, 1.3001383543014526, 1.7417491674423218, -6.24180269241333, 2.9457571506500244, 0.5301907658576965, -0.6790028810501099, -9.868217468261719, 4.1739821434021, -7.211498737335205, -5.247389793395996, 3.4004831314086914, -1.2223082780838013, -7.507606029510498, -8.971061706542969, 9.3468656539917, -3.2454185485839844, -5.245755672454834, 13.335826873779297, -2.123302936553955, -7.450751304626465, -17.294153213500977, -18.476396560668945, -1.6850242614746094, 4.664607524871826, -1.3932214975357056, 0.413634717464447, -4.040364742279053, 6.921515464782715, 8.324302673339844, -0.3944470286369324, 10.70732593536377, 12.562652587890625, -23.803316116333008, 4.1573100090026855, 2.212618350982666, -6.000887870788574, -14.955881118774414, -17.650863647460938, -3.5084381103515625, -10.029340744018555, -8.737067222595215, -8.42552661895752, 12.795705795288086, 2.272523880004883, 2.1651229858398438, 0.45081934332847595, -4.178195953369141, -11.387500762939453, -10.049139022827148, 8.749341011047363, 1.4700530767440796, 1.9325624704360962, -5.876566410064697, 0.11842915415763855, 6.012762546539307, -6.711397171020508, -6.226899147033691, -13.556955337524414, 10.418705940246582, 0.622202455997467, -2.4716053009033203, -3.5229461193084717, -3.6459667682647705, 1.0674561262130737, 5.151406764984131, -6.223432540893555, -4.1161699295043945, -3.1541783809661865, -13.004470825195312, 10.337922096252441, -2.9376790523529053, -6.944218635559082, -5.119958877563477, -12.832958221435547, 8.67713737487793, -2.9103593826293945, -3.993739128112793, 1.3619993925094604, 6.704738616943359, 11.042302131652832, 1.6980297565460205, -2.9447121620178223, 8.576769828796387, 3.2450246810913086, 0.23804175853729248, 4.071667671203613, 5.7134690284729, -13.142839431762695, 6.513056755065918, 12.641369819641113, 10.51157283782959, 3.3329098224639893, -1.3506935834884644, 6.8402228355407715, -10.936856269836426, -3.172245502471924, -6.293821334838867, -7.541832447052002, 19.890472412109375, 12.415449142456055, 3.1522903442382812, 0.04646298289299011, -7.055224895477295, 2.699068784713745, 4.058042049407959, -8.411839485168457, 3.58601450920105, 0.06740380078554153, -24.300230026245117, -3.208954095840454, 5.544628143310547, -1.4936304092407227, -11.714347839355469, -3.284128189086914, -5.226027011871338, 19.487022399902344, 4.119907379150391, 10.703712463378906, -4.891098976135254, 0.09879254549741745, 7.873628616333008], [-3.927510976791382, -5.952702522277832, -0.06486701965332031, 7.460507869720459, -4.724550724029541, 2.6070973873138428, -1.6304852962493896, -12.448297500610352, 11.633000373840332, 7.4219841957092285, 6.941089153289795, -4.621325969696045, -10.040107727050781, 5.118271827697754, -5.295877456665039, 6.5166473388671875, -7.215340614318848, -8.146842956542969, -6.7292022705078125, 2.1457982063293457, 10.750743865966797, 3.316474437713623, -1.5457273721694946, 10.976022720336914, -2.5057590007781982, -0.9775928258895874, -7.698675155639648, -0.7687172889709473, 0.5389397740364075, 3.3721072673797607, -5.107532501220703, -4.160670280456543, -9.51058292388916, -0.6172245740890503, -8.486410140991211, 4.3109259605407715, 10.024383544921875, 6.5247931480407715, -2.105748414993286, 1.6056183576583862, 0.05742369592189789, 1.322983980178833, 3.758845329284668, -10.242826461791992, 0.13632531464099884, -9.645265579223633, -6.091024875640869, -0.8594977855682373, 7.863194942474365, 5.819052696228027, -3.7759857177734375, 2.3711540699005127, -22.958139419555664, -2.3127167224884033, -5.560243606567383, 5.352158546447754, 2.287428379058838, -4.576454162597656, 2.2163989543914795, 24.607648849487305, 6.404117107391357, -5.592911720275879, -5.888134956359863, -1.110952377319336, -2.2349929809570312, -0.08680514246225357, -6.536542892456055, 6.04960298538208, -4.929981708526611, -5.8696441650390625, 8.026700019836426, -1.1073654890060425, -0.4848436713218689, 0.2796069085597992, -0.46895357966423035, -6.681003093719482, -10.773530960083008, -1.5920740365982056, 2.8249804973602295, 1.3352975845336914, 1.4731582403182983, -10.901522636413574, -0.28691935539245605, 3.087364435195923, 12.082911491394043, 3.2348387241363525, -6.370466709136963, -8.651869773864746, 3.303063154220581, 3.9803059101104736, 7.0939412117004395, 7.530350685119629, 8.270069122314453, -13.168194770812988, 2.4277634620666504, -13.421065330505371, 12.228293418884277, -7.665515899658203, 1.383683204650879, 6.277781009674072, -5.072685718536377, -7.407058238983154, -0.11338173598051071, 2.0584404468536377, -2.5666158199310303, -14.170610427856445, -10.725200653076172, 2.584291696548462, 7.893017768859863, 13.830016136169434, -4.511160850524902, -13.682674407958984, 7.610808849334717, 2.3419883251190186, 10.267143249511719, 12.528100967407227, -1.5168391466140747, -16.72366714477539, -8.684494018554688, -4.4004387855529785, -9.427634239196777, 15.300228118896484, 5.426342487335205, 6.031251907348633, -1.887123703956604, 1.6161775588989258, -4.783725738525391, -16.630435943603516, -6.76350212097168], [-9.245657920837402, -1.7902113199234009, -17.929698944091797, -12.672327041625977, -2.262953519821167, 4.778265476226807, -1.1428054571151733, -1.6979284286499023, -8.863851547241211, -1.5752078294754028, 10.501530647277832, 3.357046365737915, -9.502883911132812, -4.637202739715576, -7.860199928283691, 11.376425743103027, 17.746137619018555, -4.697150230407715, 0.2537311613559723, 11.035966873168945, 2.0068039894104004, 5.5724005699157715, -1.6428145170211792, -12.99239730834961, 11.96883773803711, -6.905168533325195, -4.662941932678223, -17.132410049438477, 0.8670876026153564, -21.339311599731445, -0.41630345582962036, -1.7172704935073853, 3.410938262939453, -21.564945220947266, 3.880647659301758, 0.1445188820362091, 0.5779648423194885, 2.4425323009490967, -6.346762180328369, -5.055180072784424, 8.059889793395996, 1.8103936910629272, 3.5338547229766846, -6.337226867675781, -1.3534966707229614, -8.565921783447266, 1.9083501100540161, 0.3859008848667145, 2.923093795776367, 11.94766902923584, 4.766031265258789, 11.265425682067871, -10.44572639465332, -0.9608100652694702, 5.111944198608398, -4.197749614715576, 4.091705322265625, 0.769610583782196, -4.6665778160095215, -5.634131908416748, 2.4572947025299072, -15.69622802734375, -7.194417953491211, 2.667259454727173, -5.111752986907959, -11.308938980102539, 11.114828109741211, -7.072572708129883, -21.814720153808594, 2.752218723297119, 4.420846939086914, -4.017117500305176, 11.788908958435059, 5.2754902839660645, -4.409411907196045, -7.028439998626709, -8.257110595703125, 1.00179123878479, -9.337140083312988, 16.096773147583008, 3.4314897060394287, -1.7658556699752808, -7.8770647048950195, 4.198287010192871, 2.3015830516815186, 9.144540786743164, -0.0504172146320343, -5.655669689178467, 5.895400524139404, -18.23812484741211, -6.475166320800781, 3.533978223800659, 14.209549903869629, 7.9459638595581055, -0.18389929831027985, -7.188672065734863, -1.4039239883422852, 5.202337741851807, -4.13873815536499, 6.01145076751709, -6.5235724449157715, 4.020800590515137, -9.021499633789062, -8.312623977661133, -11.31175422668457, -6.611542701721191, 4.547537803649902, 3.9715051651000977, 2.4558298587799072, -8.457781791687012, 6.136744976043701, -8.551005363464355, 1.2598469257354736, -22.11486053466797, -19.290786743164062, 2.4620988368988037, -4.080924987792969, 2.841097354888916, -6.632135391235352, -6.809420108795166, -9.253870010375977, -4.870691299438477, 1.3145145177841187, -2.296298027038574, 1.268460988998413, 3.2516283988952637, -2.2741122245788574, -2.413255214691162, -7.958852291107178], [-8.428504943847656, -1.025044560432434, 5.386782169342041, -4.7668561935424805, -12.43299388885498, -0.5189911723136902, 16.46522331237793, -7.427833557128906, 4.950742244720459, -9.687496185302734, -13.985807418823242, -1.9049975872039795, 2.4843108654022217, 14.841578483581543, -4.896965980529785, -1.1937391757965088, 2.740415096282959, -0.5332043170928955, 4.130971908569336, -7.973963737487793, -11.88267707824707, -4.349257946014404, -8.542269706726074, 12.219637870788574, 2.1712093353271484, -0.5612530708312988, 8.299042701721191, -8.50539779663086, -9.352909088134766, -5.9881110191345215, 0.9853563904762268, 2.5355398654937744, 6.061605453491211, -8.73473072052002, -1.675621509552002, -7.1966962814331055, 8.106800079345703, 11.87032413482666, 11.819808006286621, 3.504703998565674, -37.888641357421875, 16.56216812133789, 7.378446102142334, 0.9898759126663208, -16.858552932739258, 1.1529029607772827, 14.432547569274902, 6.609092712402344, 6.2509236335754395, 5.3010101318359375, -6.667633533477783, 1.2507741451263428, 3.0458080768585205, -11.196395874023438, 8.907329559326172, 3.469896078109741, -3.1037204265594482, -11.938385009765625, 12.909416198730469, -7.962648391723633, -2.0572710037231445, 7.432276725769043, -3.455904245376587, 7.009503364562988, 11.827234268188477, 7.874131679534912, 2.8940365314483643, 1.289451241493225, -0.6902609467506409, 4.6739091873168945, -4.928016185760498, -9.55038070678711, 13.357353210449219, 1.3465511798858643, 2.925147294998169, -1.9034593105316162, 7.122755527496338, -2.892169237136841, 12.884637832641602, 8.565659523010254, 4.832536220550537, 7.8709516525268555, -0.19441263377666473, -0.9386768341064453, 8.550935745239258, -12.288337707519531, -7.743174076080322, -7.7785725593566895, -1.1076083183288574, 4.419373035430908, 0.7074896097183228, -0.5488283634185791, 7.820120811462402, -3.0306456089019775, -3.5301249027252197, -7.665262222290039, 9.769502639770508, -5.710770130157471, -8.096735000610352, 11.338651657104492, 3.7861177921295166, -8.737590789794922, 6.401636600494385, 10.314308166503906, -3.675403356552124, -2.158207654953003, -7.597349166870117, 4.977783679962158, 5.348212242126465, 2.6026082038879395, 11.868536949157715, 1.4106659889221191, -17.50045394897461, -4.933487892150879, -11.689789772033691, -4.301696300506592, 0.6686621308326721, -6.066394329071045, 3.4162182807922363, 10.108039855957031, 0.008062688633799553, -0.5454846024513245, 11.060494422912598, -3.6920175552368164, 9.572091102600098, 6.315929889678955, 5.026495456695557, -9.829835891723633, -7.706536769866943], [6.721698760986328, -0.4405415654182434, 13.17992877960205, 8.928438186645508, 4.635299205780029, 5.050352573394775, -0.3483765423297882, -0.01334813516587019, -1.6045615673065186, -6.505055904388428, 2.5487732887268066, 6.9535369873046875, -10.317850112915039, -6.494100570678711, 7.306301116943359, 4.335134506225586, 17.574974060058594, -5.347187042236328, -4.550535202026367, -2.400641441345215, -10.700024604797363, -13.57518196105957, 1.9586992263793945, 2.3718833923339844, 9.374627113342285, 1.2080786228179932, -3.824794054031372, 6.1769022941589355, -17.666635513305664, -13.190237998962402, -4.738968372344971, -5.292712211608887, 9.155257225036621, 9.42533016204834, 2.3911819458007812, 0.6438475251197815, 10.725781440734863, 3.8010683059692383, -13.040422439575195, 8.68558406829834, 1.9107394218444824, 4.146833896636963, 4.502022743225098, 1.7566837072372437, 19.269006729125977, -20.53416633605957, -17.859203338623047, -9.413418769836426, -8.823131561279297, -12.629037857055664, 17.305442810058594, -2.5810606479644775, 19.32651710510254, 4.042191982269287, -0.06002667546272278, -0.731797456741333, 7.553938388824463, 6.923180103302002, 6.99290657043457, 14.667841911315918, 0.0417754240334034, 6.438400745391846, 14.903189659118652, -2.503708600997925, -11.035740852355957, -3.3660714626312256, -2.6305978298187256, -0.9409019351005554, -10.547287940979004, -6.6443939208984375, 12.132926940917969, 3.7246925830841064, -14.774739265441895, 0.14509741961956024, -10.55286979675293, 6.5323944091796875, 3.7476730346679688, 0.12556138634681702, 1.9573243856430054, -7.968207359313965, -14.658307075500488, -3.4561142921447754, -9.162322998046875, 1.545035481452942, 1.399315357208252, -10.377796173095703, -5.972087383270264, -2.504093885421753, -5.795080661773682, -5.199108600616455, -7.319817543029785, -13.125782012939453, -0.7963816523551941, 0.6556360721588135, -5.941701412200928, -1.4956128597259521, -9.752704620361328, -8.149483680725098, 15.779268264770508, 11.35783576965332, 10.997485160827637, -1.8606228828430176, 13.950401306152344, 3.3279006481170654, -0.14697252213954926, 0.5652710199356079, 10.389811515808105, -4.850851058959961, -2.8581366539001465, 4.989139556884766, 7.749220371246338, 8.106080055236816, 8.21218490600586, 11.138253211975098, 0.09409579634666443, -9.37172794342041, -4.691321849822998, 8.976584434509277, 12.592694282531738, -3.950071096420288, -7.3719048500061035, -6.1866631507873535, -6.633882522583008, 18.051145553588867, -13.439682960510254, 0.6371558904647827, 1.8055533170700073, -5.809887886047363, 7.828522682189941], [-1.3817944526672363, -10.353777885437012, 2.1284117698669434, 2.490643262863159, 3.4424045085906982, -2.321803092956543, 7.259965419769287, -7.024467468261719, -1.8993303775787354, 7.477253437042236, 9.352310180664062, 8.194141387939453, 8.796278953552246, -11.531411170959473, -10.517169952392578, -8.746460914611816, -5.3051910400390625, 10.398462295532227, -10.365452766418457, -8.612785339355469, -15.162076950073242, 2.491222381591797, -0.5301936864852905, 5.870162487030029, -18.9174747467041, 1.1357076168060303, 7.755983352661133, -8.336256980895996, -1.3377056121826172, -3.370230197906494, -2.5004334449768066, -13.380511283874512, 6.307166576385498, -2.2898716926574707, 7.8726887702941895, -11.336816787719727, -0.6358206868171692, -2.2951858043670654, 25.17988395690918, 1.681944489479065, -10.70045280456543, 1.5274758338928223, 3.3261282444000244, -4.024840831756592, -9.493138313293457, 5.087617874145508, -4.636141300201416, 6.427978515625, -15.379857063293457, -2.6287407875061035, -6.635711193084717, -11.819245338439941, 6.191854000091553, -5.993996620178223, 7.828902244567871, -4.556933403015137, 4.70670747756958, -1.1318671703338623, 5.574937343597412, -13.596506118774414, -3.222156286239624, -10.219635009765625, -8.776244163513184, 1.4984360933303833, 1.0836962461471558, 3.3975725173950195, -2.2933902740478516, 1.9129858016967773, -18.870492935180664, 4.512345790863037, -5.689099311828613, 1.9230293035507202, 11.880697250366211, 9.324955940246582, -0.1961365044116974, 3.6617698669433594, -3.84631085395813, 11.6983642578125, -3.0575456619262695, 0.4441341161727905, 7.245150566101074, -5.909135341644287, -5.068268299102783, -6.791681289672852, 2.4035983085632324, 15.286859512329102, 7.66387414932251, 5.321035861968994, 2.4998245239257812, -9.370654106140137, 14.547998428344727, -1.937676191329956, 4.166986465454102, 3.2531604766845703, 6.435255527496338, -5.658879280090332, 9.071556091308594, 5.905941486358643, 0.8960718512535095, 0.7782186269760132, 4.263463497161865, 8.403855323791504, -12.276999473571777, 6.9477667808532715, 1.1108942031860352, 0.31573542952537537, 11.466382026672363, 6.116942405700684, 9.520390510559082, 1.4717451333999634, 0.45396217703819275, 10.238414764404297, 0.10061031579971313, 3.5765011310577393, -19.611465454101562, 13.113187789916992, -7.955979824066162, 12.342485427856445, -8.064019203186035, -2.988065481185913, 3.1390929222106934, 4.585906505584717, -10.884455680847168, 4.40475606918335, 12.376347541809082, 1.0962188243865967, 16.23587417602539, -6.5253005027771, -1.6003921031951904], [-2.1484434604644775, 7.327090263366699, 3.182713270187378, 5.638715744018555, -10.37950325012207, -8.18706226348877, 14.127923011779785, 7.49957799911499, 10.035667419433594, -6.746682643890381, 2.7107560634613037, -3.550412893295288, -9.547333717346191, 9.278133392333984, 11.208332061767578, 7.721600532531738, -5.932056427001953, 7.8079938888549805, 2.4986228942871094, 3.1028261184692383, -4.754221439361572, -6.674042224884033, 9.835692405700684, 7.671843528747559, 1.0543042421340942, 8.154379844665527, -11.866469383239746, 7.545907020568848, -8.827062606811523, -5.140048980712891, 11.976253509521484, 3.8573031425476074, -8.14924430847168, -13.93287467956543, -6.661470890045166, 3.0665531158447266, 8.945331573486328, -4.208262920379639, 3.7165627479553223, 10.007451057434082, -11.92422866821289, 11.699029922485352, -0.3967974781990051, -4.78141975402832, -5.976260185241699, 13.469475746154785, -1.3183963298797607, -15.5679349899292, -1.8319169282913208, -22.46820068359375, -3.2713570594787598, 2.015972852706909, 4.7504777908325195, -7.242043495178223, 2.6028473377227783, -4.168133735656738, -18.576465606689453, 8.251251220703125, 1.925774097442627, 13.322539329528809, 11.263094902038574, 10.076900482177734, 8.7139892578125, -3.0222935676574707, 3.728503465652466, 2.4259557723999023, -7.742591857910156, 5.467050552368164, 2.5233259201049805, 3.2510948181152344, -8.699639320373535, -2.766724109649658, -4.130156517028809, -1.2760576009750366, -10.983853340148926, 19.194671630859375, -5.490520477294922, 0.43207621574401855, 15.536932945251465, 6.36378812789917, -18.02953338623047, 8.402710914611816, 12.910789489746094, 6.596710681915283, 2.4484214782714844, -6.832065105438232, 3.3867175579071045, -1.0138746500015259, 4.388034343719482, 5.993931770324707, -4.692566871643066, -3.402794361114502, 0.750495970249176, -0.8183845281600952, -3.743443489074707, 3.756842851638794, 1.3748931884765625, -7.443465232849121, 4.0027666091918945, 6.779850959777832, 13.52847957611084, -10.4876070022583, 9.87215518951416, -4.176056385040283, 14.329586029052734, -4.8897223472595215, 1.9837859869003296, -0.025433730334043503, 4.128622531890869, 2.759410858154297, -1.2387017011642456, 3.216118574142456, -3.2561278343200684, 4.449795722961426, 1.1997603178024292, -1.9404383897781372, 0.9676817059516907, -8.176968574523926, 13.538581848144531, 1.4905307292938232, 1.5379220247268677, 5.9336652755737305, 2.4529433250427246, -0.5230769515037537, -11.769841194152832, -0.710781991481781, -0.590175211429596, -17.348953247070312, 1.5133037567138672], [-3.6143152713775635, 1.494747281074524, 14.650677680969238, -8.642148971557617, -7.5126237869262695, -8.342187881469727, -4.709100246429443, -17.113378524780273, 7.768050193786621, 5.2032647132873535, -15.970168113708496, 9.194775581359863, -1.5902775526046753, -16.85430908203125, -13.111486434936523, -9.738524436950684, 1.68087637424469, -3.3459250926971436, -0.37486138939857483, -11.32358455657959, -6.873669147491455, -10.4895601272583, -15.144730567932129, 13.9761962890625, -11.253791809082031, -0.40279680490493774, 9.831572532653809, -15.919342994689941, 13.173158645629883, 7.3478498458862305, -3.7435688972473145, 1.3339840173721313, 16.633394241333008, -1.6253786087036133, 1.7905038595199585, 8.270258903503418, -3.3643267154693604, 10.173999786376953, 12.684992790222168, -6.511271953582764, -11.625861167907715, 5.895303249359131, 11.32289981842041, -8.542784690856934, -24.057680130004883, -7.732276439666748, 12.824322700500488, -6.251993179321289, 11.963689804077148, -1.7712653875350952, -20.527101516723633, 7.137779235839844, -6.4144110679626465, -3.561098575592041, 4.591710090637207, -4.539597988128662, -3.1723861694335938, -2.4426491260528564, 5.326272010803223, -5.566238880157471, 3.3679397106170654, -6.279453277587891, -25.120742797851562, 2.0701799392700195, -7.427856922149658, -7.021386623382568, -6.609955310821533, -7.330985069274902, -1.4138844013214111, -5.049006462097168, -1.904787540435791, -15.472383499145508, 7.577853202819824, 3.371337652206421, -5.608901023864746, 1.8863657712936401, -6.351743698120117, 7.595312595367432, -1.7998830080032349, -9.384510040283203, 2.4985599517822266, 4.8053154945373535, 2.136786699295044, 0.29958033561706543, 4.955554962158203, -1.097320795059204, 6.828396320343018, 2.9485793113708496, 13.441572189331055, 4.12861442565918, 6.199202537536621, 6.565500259399414, 13.425711631774902, -3.081725597381592, 11.586525917053223, 4.234150409698486, 2.391589879989624, -3.0795891284942627, -6.2291059494018555, -2.0618555545806885, 5.880784511566162, 15.434247016906738, -16.720657348632812, 8.723020553588867, -0.006145843304693699, 6.981479167938232, 5.658810138702393, -4.687053680419922, 5.946019172668457, 0.8757913708686829, 0.6515137553215027, -6.68941068649292, -13.161979675292969, 9.272298812866211, 0.9000880122184753, 6.817135810852051, -10.252931594848633, 6.238112926483154, 0.021755460649728775, 10.573104858398438, 8.6820068359375, -1.8106443881988525, 15.930773735046387, -9.620560646057129, 16.85240936279297, -16.217994689941406, -1.816752314567566, -4.338035583496094, 6.056576251983643], [7.695140361785889, -7.93356466293335, 0.9847233891487122, 3.5907559394836426, -0.04190127179026604, -5.66476583480835, 11.682241439819336, -0.1682274043560028, 4.874827861785889, -5.390199661254883, 2.904839515686035, 4.02163028717041, 11.225927352905273, 12.800134658813477, -1.9237747192382812, -3.0415587425231934, -4.777707576751709, -1.7720074653625488, -2.8673434257507324, 7.444756031036377, -14.000917434692383, 7.796779632568359, -0.890509843826294, -11.803906440734863, -1.995191216468811, 3.9085845947265625, -9.261404037475586, -4.419674396514893, -7.290284156799316, -12.305147171020508, -6.00439977645874, 0.49261564016342163, -8.244555473327637, 0.15957708656787872, -2.0462419986724854, 2.816911458969116, 4.451542377471924, 3.0034596920013428, 3.4275972843170166, 5.536752223968506, -8.904205322265625, 2.849966526031494, 6.823903560638428, -11.642898559570312, 2.1367082595825195, -6.180274963378906, -5.563668251037598, -0.5690522193908691, -5.998417854309082, 0.97482830286026, 14.62431526184082, 4.602453708648682, 1.4543427228927612, -10.971147537231445, 9.667908668518066, -7.524597644805908, 5.736278057098389, -4.702315807342529, 7.007299900054932, 9.830533027648926, -17.8922061920166, -5.257785797119141, 8.229610443115234, 10.469732284545898, 13.51980209350586, -2.4540398120880127, -5.9303879737854, 10.463516235351562, -8.700831413269043, 9.315041542053223, 3.033891201019287, -0.38435569405555725, -3.8207521438598633, -6.171331405639648, 1.7789607048034668, 11.394158363342285, -4.781035423278809, 3.0059499740600586, -0.9926748871803284, 0.35188162326812744, 7.566551208496094, -1.5093016624450684, -0.975029468536377, 13.738240242004395, 4.01524019241333, -2.6061503887176514, 3.5436933040618896, -2.163437604904175, -8.325900077819824, -7.094096660614014, 1.0920618772506714, -4.642652988433838, 7.414012432098389, 8.84473705291748, -6.76715612411499, -5.534359931945801, -6.929805278778076, -0.8553851246833801, 8.036294937133789, 5.306730270385742, -4.1012701988220215, 4.429596900939941, 1.1917738914489746, 2.712695837020874, 4.516225337982178, 8.532914161682129, 1.1137505769729614, -4.877773761749268, -9.564300537109375, -0.5300661325454712, 1.4620000123977661, 14.064082145690918, 6.203003883361816, 10.948451042175293, -10.018844604492188, -1.9576176404953003, -8.346234321594238, 6.801076412200928, 4.476886749267578, 4.303279876708984, 5.371270656585693, 0.9365594387054443, -5.96345853805542, 11.983805656433105, -0.4524765610694885, 15.93592357635498, 6.421200752258301, 1.4055025577545166, -3.796380043029785], [1.9765944480895996, -0.8571080565452576, 7.018780708312988, 3.4192633628845215, 6.020632743835449, 3.030107259750366, -4.936483860015869, 18.611400604248047, 5.202421188354492, 0.5483124256134033, -4.67158317565918, 7.860016822814941, 9.501057624816895, -0.7053271532058716, 5.983351230621338, -6.729400634765625, -8.662576675415039, 7.645410537719727, 2.0162782669067383, 4.38029146194458, 1.5187863111495972, 11.615927696228027, 5.542356967926025, -1.7375597953796387, -13.825556755065918, -2.2461929321289062, -6.940685272216797, -3.105682849884033, -9.373106002807617, 7.190450191497803, -1.1050039529800415, -0.6959006190299988, 13.35302734375, -1.6419957876205444, -8.98431396484375, 2.0996334552764893, -15.809433937072754, 21.791515350341797, -5.925987720489502, 7.485565662384033, -10.201920509338379, 4.84105920791626, 1.0398703813552856, 5.079223155975342, -8.874905586242676, -0.3945494294166565, 8.784623146057129, 0.8713310956954956, -6.8802690505981445, 7.794276714324951, 14.666421890258789, -1.4606510400772095, 2.36649227142334, 12.980477333068848, 1.2493360042572021, -0.40920504927635193, -3.9618289470672607, -11.990365028381348, -0.8623586297035217, -8.718276977539062, -4.958524703979492, 13.838634490966797, -0.520223081111908, -19.182188034057617, -7.413440227508545, -6.894292831420898, 5.898001670837402, -0.7283162474632263, 9.865942001342773, -12.033722877502441, 0.7437795996665955, 4.331094741821289, 7.015432357788086, 3.6534504890441895, 3.19633150100708, -0.3423457741737366, -2.8762831687927246, -6.80078125, -1.7430264949798584, -4.783993244171143, 4.188805103302002, 12.870318412780762, 15.10776424407959, 1.0939435958862305, -2.7402520179748535, -2.7036566734313965, 0.8808962106704712, 15.219037055969238, -4.180583477020264, 17.429439544677734, -3.8670778274536133, 10.786134719848633, -2.6595325469970703, 0.013782186433672905, 10.224482536315918, 6.2542829513549805, -0.8775399923324585, 7.525081157684326, -1.0252248048782349, -2.682450532913208, -3.2192904949188232, 16.88749122619629, -15.100303649902344, 5.52836275100708, -0.430166095495224, 7.808091163635254, -0.8877535462379456, -12.767398834228516, 2.9144957065582275, -6.7324910163879395, -3.03680157661438, -7.7007341384887695, 1.1048927307128906, 9.230504989624023, -5.438323497772217, -15.158575057983398, 0.013926678337156773, 0.4104914665222168, 5.353530406951904, 2.124314546585083, 3.3355116844177246, 2.9570603370666504, -9.902240753173828, -9.610377311706543, -0.005372569430619478, 0.8681790232658386, 17.411623001098633, 7.4036102294921875, -0.3858582377433777], [-1.192787528038025, 3.4033524990081787, 2.141564130783081, 9.968932151794434, 12.24316120147705, 2.902513265609741, -2.079087495803833, -0.28266483545303345, -9.415375709533691, -1.210761308670044, -0.9471477270126343, 22.26724624633789, -0.04912608116865158, -5.72324275970459, -6.359156131744385, -1.1918014287948608, -0.042755816131830215, -1.3769830465316772, -10.523202896118164, 1.69280207157135, -7.708792209625244, -6.3909759521484375, 6.000190734863281, 1.1412017345428467, -17.78635597229004, -8.632277488708496, -1.6216633319854736, -3.2613348960876465, -11.600852966308594, -0.034202300012111664, -2.19696044921875, -8.333561897277832, 6.632652282714844, -7.054056167602539, 1.4301387071609497, 6.259155750274658, 13.031623840332031, -5.9521074295043945, 17.850358963012695, 8.01582145690918, -10.413475036621094, 3.949296712875366, 2.5704565048217773, -1.8938443660736084, 4.513443946838379, -4.183183670043945, -4.8557538986206055, -1.5302486419677734, -9.367212295532227, -8.295353889465332, 19.466781616210938, -5.575522422790527, 7.348228931427002, 0.266493558883667, -5.38828706741333, 7.23626184463501, 10.6240816116333, -1.7111492156982422, 1.0750582218170166, -1.6721510887145996, -4.363424301147461, 15.466364860534668, -0.5999716520309448, -9.564239501953125, -5.776332855224609, 4.676855087280273, 2.7174108028411865, 12.495942115783691, -7.581635475158691, 3.857146739959717, 2.496124744415283, 2.635589361190796, -2.725461959838867, 9.476295471191406, -10.492817878723145, 1.8305678367614746, 5.517967224121094, -0.7936492562294006, 1.0275030136108398, -7.12727689743042, -17.194456100463867, 5.593577861785889, 6.336910247802734, 2.7460720539093018, -0.7322671413421631, 5.1543965339660645, -2.954763174057007, 3.032062530517578, 10.144719123840332, 1.267864465713501, 4.837551116943359, -0.8571052551269531, 3.8575406074523926, -0.6864126324653625, 8.518298149108887, 6.268507480621338, -4.5315141677856445, -13.464693069458008, 13.275856971740723, 1.6996519565582275, 6.864719867706299, 0.6203830242156982, -1.470723271369934, -5.815469264984131, 6.838928699493408, 0.3474562168121338, -6.199667453765869, 2.239131450653076, 9.706156730651855, 4.011570930480957, 5.851381778717041, -1.1806899309158325, -3.2797980308532715, 4.8500566482543945, -1.2912530899047852, -3.689983367919922, -8.429154396057129, 8.620567321777344, 11.963203430175781, -6.964799880981445, -1.8846454620361328, -18.39781951904297, -8.089123725891113, 2.8919386863708496, 2.5874016284942627, -5.880616188049316, 3.0700995922088623, 2.177448034286499, 1.8554149866104126], [0.14509817957878113, -3.197190999984741, 11.862010955810547, -2.5600743293762207, 16.842105865478516, 13.022164344787598, 10.043941497802734, 11.149447441101074, -4.41732120513916, -16.845428466796875, -10.468599319458008, 6.258213043212891, 10.732621192932129, 7.604034423828125, 6.7003607749938965, -9.548606872558594, -7.018800735473633, -2.1406338214874268, 5.426901817321777, 1.9471254348754883, -4.686021327972412, 0.5775169134140015, -2.34771466255188, -0.10121090710163116, 10.410726547241211, 12.131817817687988, -7.640775680541992, -0.5157520174980164, -8.64653491973877, -7.8976922035217285, -3.104250192642212, 5.656876087188721, 2.2094597816467285, 6.491006374359131, -3.982501268386841, -5.9754509925842285, 11.294695854187012, -9.257299423217773, -8.071805000305176, 10.554003715515137, -14.527620315551758, -1.0356724262237549, -0.6584993600845337, 3.920182943344116, 0.496433824300766, 10.692902565002441, -12.602312088012695, -4.0009541511535645, 3.593210220336914, -2.068455934524536, 10.72303295135498, 0.2007099837064743, 15.845307350158691, -3.4451916217803955, 5.634295463562012, 11.261649131774902, 3.4363937377929688, 0.6249140501022339, -1.2142727375030518, -8.396827697753906, 6.2029948234558105, -0.7137997150421143, 2.7477056980133057, 1.7636886835098267, -5.885400772094727, -9.926207542419434, 3.240203619003296, -5.496572971343994, 21.383052825927734, 1.912927508354187, 3.7996666431427, -10.386518478393555, 1.373340368270874, -4.485565662384033, 2.2266271114349365, 3.3479866981506348, 4.54398775100708, -6.717678070068359, 1.9679561853408813, 12.0597505569458, -4.279567718505859, -1.2875995635986328, -3.049269199371338, 11.785404205322266, -0.3193869888782501, -6.306548118591309, -4.115481376647949, -4.6372389793396, -1.392156958580017, 14.288948059082031, 13.906481742858887, 1.1859465837478638, -7.715537071228027, -4.987875461578369, -15.090579986572266, 3.6490745544433594, -8.629833221435547, 1.2313023805618286, -0.4863215684890747, 9.52778148651123, 1.1096093654632568, -10.219402313232422, 21.266700744628906, -4.577441215515137, -3.5250396728515625, 1.4663447141647339, -2.1910457611083984, 10.496676445007324, -6.076573848724365, 11.055166244506836, 0.6228798627853394, 6.508182525634766, -5.5983428955078125, -9.316851615905762, 0.8282804489135742, -14.20966911315918, -1.4793049097061157, -1.9688290357589722, 8.241911888122559, 7.263060092926025, 4.973231315612793, -1.8214643001556396, 3.4472837448120117, 12.52221393585205, 1.673694372177124, 5.858853340148926, -2.0660486221313477, 7.982846736907959, 2.5427401065826416], [3.5435616970062256, -5.547640800476074, 10.082520484924316, -3.1055779457092285, 1.1341238021850586, -9.897028923034668, 11.160380363464355, -1.301963448524475, 2.3953139781951904, -9.79660415649414, 6.909389495849609, -7.7470574378967285, -3.911445140838623, 11.604780197143555, -0.6852959990501404, 4.419947147369385, -6.877836227416992, -4.262973785400391, 5.988080024719238, 6.524055480957031, -8.278470993041992, 2.0546791553497314, -0.27077528834342957, -1.6417728662490845, 4.781222343444824, 10.119566917419434, -2.5549490451812744, 7.554215908050537, -14.67725658416748, 7.255558490753174, 7.140798568725586, -3.922078847885132, 6.8147358894348145, 2.4478464126586914, -3.9713633060455322, 1.3683140277862549, 4.85599422454834, 9.893501281738281, 9.360466003417969, 2.53233003616333, -9.97677993774414, -0.14382177591323853, 0.501614511013031, -6.755202770233154, 13.206488609313965, -6.676618576049805, 2.26170015335083, -9.319731712341309, -1.3121479749679565, -17.908706665039062, 4.574489593505859, 6.319155693054199, 0.39330795407295227, -1.6577506065368652, 6.956939220428467, -12.201607704162598, 0.44165074825286865, 9.339877128601074, -1.019278883934021, 18.349258422851562, 3.8739662170410156, -4.3207879066467285, 0.5673711895942688, 7.665281772613525, 5.21038293838501, -4.455957889556885, 1.5524747371673584, -4.6079607009887695, -12.419544219970703, 5.3425822257995605, -1.4680424928665161, 8.949715614318848, -2.6299514770507812, -13.127521514892578, -10.557320594787598, 0.597217857837677, -5.132538795471191, -5.945709705352783, 8.0513334274292, 2.200207233428955, 4.022839069366455, 2.7411396503448486, -19.430191040039062, 5.518622398376465, 8.256354331970215, -5.7760396003723145, -0.6164765357971191, -1.8589168787002563, -6.930862903594971, -4.175665378570557, -2.156778573989868, -0.6402804851531982, -1.6607062816619873, -13.292617797851562, 2.278118133544922, -3.9183011054992676, -4.948426246643066, -0.5956379771232605, -1.3741166591644287, 1.1922518014907837, 11.009572982788086, -4.8134894371032715, 2.813302993774414, 5.1599345207214355, 6.014120578765869, -1.3594167232513428, 16.764490127563477, -8.025662422180176, 0.18580052256584167, -0.4683893918991089, -1.2655997276306152, 7.52166223526001, 12.555697441101074, 5.676520824432373, 3.6827683448791504, -13.559382438659668, 2.602794647216797, 6.523357391357422, 1.9500281810760498, -2.656877040863037, 7.150146961212158, -5.589303016662598, -4.717236042022705, 11.631443977355957, -3.820580244064331, 3.13421368598938, 0.9233525991439819, -12.528925895690918, -4.113587856292725], [1.6194645166397095, -17.945068359375, -1.742730975151062, -7.142122268676758, 15.53466510772705, -3.586456060409546, 5.560147285461426, -2.8399174213409424, 2.667402744293213, -2.0848400592803955, 4.88024377822876, -1.473580002784729, -2.483440399169922, -7.8216552734375, -2.1539993286132812, -5.908483982086182, 8.063942909240723, -10.122739791870117, -11.057504653930664, -2.3948724269866943, 14.585419654846191, 3.024433135986328, -17.139728546142578, -1.389560341835022, 5.959169864654541, -13.288358688354492, -3.7126643657684326, -11.069452285766602, 12.139679908752441, 7.587109565734863, 11.635395050048828, -4.334743022918701, -3.684650421142578, -11.604496002197266, 0.46809110045433044, -2.4227821826934814, 14.198423385620117, -8.696751594543457, -7.331905841827393, -11.771007537841797, 28.165374755859375, -17.609710693359375, -5.675650119781494, 5.877608299255371, 3.3400638103485107, 3.5148541927337646, 4.58864688873291, -5.923041820526123, 11.795140266418457, 8.710989952087402, 18.869659423828125, 9.287581443786621, -2.6786956787109375, -13.420668601989746, -3.230567455291748, -7.8172125816345215, -14.513565063476562, -9.956413269042969, 5.549562454223633, -2.075777053833008, -0.9911454916000366, 12.753064155578613, -0.11319494992494583, -4.081740379333496, -4.902660369873047, 4.310638427734375, 5.1523756980896, 7.70015811920166, 4.291610240936279, 6.236715793609619, 0.325311541557312, -6.438686847686768, -1.2849514484405518, -0.30476170778274536, -0.8929626941680908, 7.193355560302734, -6.281918525695801, -3.411344528198242, -3.6832823753356934, 1.9563515186309814, -7.834402561187744, 12.584867477416992, 11.142504692077637, 5.950903415679932, -2.460681438446045, -6.126219272613525, -7.976433753967285, -0.5972261428833008, 10.635466575622559, 5.561527729034424, -11.366086959838867, 4.491110801696777, 1.4892929792404175, 8.77531623840332, 2.048624038696289, 7.072366714477539, -12.973014831542969, 4.798398017883301, -1.4330824613571167, -4.489550590515137, 3.9963035583496094, 11.111530303955078, 8.935064315795898, -4.288808345794678, -5.5348358154296875, -1.8817447423934937, -6.793744087219238, -0.2781091034412384, -1.7571454048156738, -0.6353214383125305, -0.5220842361450195, -11.418755531311035, -5.456201553344727, 11.309696197509766, 9.419717788696289, -11.888557434082031, 3.4193320274353027, 0.24127048254013062, 10.97221851348877, 9.218894958496094, -5.051843166351318, 8.1482515335083, 2.795700788497925, -9.349003791809082, 5.205886363983154, -16.54084014892578, -4.799459934234619, 11.49970531463623, 1.7269481420516968], [0.7015773057937622, -0.3805711567401886, -6.862071990966797, -0.8094345927238464, 6.783407688140869, 15.526174545288086, 5.937147617340088, 3.166635513305664, 5.178442001342773, -5.64506196975708, -4.217570781707764, 11.698907852172852, 9.087799072265625, 1.9456112384796143, 15.964954376220703, 0.5391800999641418, 17.440092086791992, 7.062419891357422, -0.3070511817932129, 6.137356758117676, 1.179714560508728, 5.252364635467529, -5.350462913513184, -3.4961395263671875, 12.985397338867188, -13.477897644042969, -16.786033630371094, -3.9847970008850098, -6.369165897369385, -2.212805986404419, -4.189677715301514, -7.780360221862793, 1.2306214570999146, -3.6056270599365234, 7.5513529777526855, 0.6556217670440674, 6.738434314727783, 2.9041531085968018, -10.129274368286133, 8.742792129516602, 14.411052703857422, 4.098591327667236, 4.044818878173828, 2.354874610900879, 5.668434143066406, 5.017329216003418, -2.1321585178375244, -4.342315673828125, -11.504329681396484, -0.6786969900131226, 1.1882489919662476, 6.107043743133545, -0.5524616837501526, 15.608805656433105, 13.126559257507324, 9.750709533691406, -7.113338470458984, 7.790652751922607, -12.10509204864502, 1.437656044960022, 0.9628480672836304, -1.520060420036316, 3.398665189743042, 1.0851200819015503, 3.2831828594207764, 5.681585311889648, 1.1082243919372559, 5.830725193023682, -5.622982978820801, -16.382192611694336, 10.81442928314209, 3.8194520473480225, -5.774359226226807, 0.6525330543518066, -1.8313672542572021, 4.061633110046387, 8.26284122467041, -5.580097675323486, -10.047157287597656, 7.166643142700195, -8.889910697937012, 6.602970123291016, -1.8537746667861938, -4.488564968109131, 2.223431348800659, -2.7038769721984863, -4.430794715881348, -0.5795566439628601, -10.31791877746582, 0.7543556094169617, -9.799500465393066, -11.48865032196045, -8.678723335266113, -1.574997901916504, -7.3113017082214355, -3.850435733795166, -8.077436447143555, 11.337770462036133, 9.102709770202637, 0.991722822189331, -6.26971435546875, -4.041514873504639, 11.22287654876709, 6.647953033447266, -5.701986312866211, -0.4936933219432831, 3.842618465423584, -7.94464635848999, -3.6689374446868896, -0.49500373005867004, 15.702998161315918, -3.518148183822632, 1.8074241876602173, -6.636780738830566, 2.324291229248047, -1.5645321607589722, 10.634042739868164, -3.1950817108154297, -5.995529651641846, 1.4014617204666138, -1.5058939456939697, 8.971784591674805, -18.085491180419922, 1.1261361837387085, 0.3026218116283417, 0.10611592233181, -8.384383201599121, -0.23405104875564575, 11.157316207885742], [7.388622283935547, 5.144890308380127, 7.010231971740723, -9.848889350891113, -0.36771056056022644, -0.00796184130012989, -4.013627529144287, -4.830956935882568, 12.320013999938965, 9.065322875976562, -7.459194183349609, -4.1612162590026855, 16.565837860107422, 1.810385823249817, -10.77781867980957, -4.410130023956299, -5.628085136413574, -9.298879623413086, -4.113454818725586, -5.387072563171387, -2.966073513031006, -7.653259754180908, 0.2958850562572479, -2.926884412765503, 8.341148376464844, -2.060137987136841, 10.597843170166016, 8.085829734802246, -6.220054626464844, -0.1298818141222, 0.6862972378730774, -2.008650302886963, -12.749104499816895, 8.89645004272461, -4.795926094055176, 0.8485231399536133, -0.8794070482254028, 15.632048606872559, -9.407905578613281, -14.869786262512207, -11.25167179107666, -4.744598865509033, 0.7002987265586853, 10.776402473449707, 6.872755527496338, -1.5779407024383545, 4.52329158782959, 10.414557456970215, -8.302021980285645, 2.899261474609375, -10.838508605957031, 12.711311340332031, -5.531961441040039, 8.649693489074707, 5.636210918426514, -5.273489475250244, 7.1419572830200195, -2.264049768447876, -0.8387572765350342, -6.853405475616455, -6.789095401763916, -8.96633529663086, 4.462512016296387, 15.741801261901855, -0.6967835426330566, 9.587520599365234, -3.8061962127685547, -13.715167999267578, 6.752749443054199, -6.127769470214844, 19.12646484375, 4.182565689086914, -8.77951717376709, -3.2612080574035645, -12.045827865600586, -4.747418403625488, 5.6030683517456055, -15.089909553527832, -3.2384440898895264, 3.0983383655548096, 11.357118606567383, -3.4744532108306885, -10.407191276550293, 6.882189750671387, -0.7425817251205444, 2.641559600830078, -5.711224555969238, -5.722152233123779, -21.020965576171875, 1.2022274732589722, 11.896525382995605, 2.5539603233337402, -7.512718677520752, -4.38786506652832, -9.211660385131836, 6.621711254119873, -3.165971279144287, 0.7863411903381348, -12.261497497558594, 1.4414602518081665, -7.881481170654297, -2.3771893978118896, 6.570960521697998, 8.033373832702637, 5.765439987182617, 2.7313499450683594, 5.288675308227539, -16.249286651611328, 3.1336052417755127, 9.00594711303711, -7.798516273498535, -2.766921043395996, 4.819801330566406, -2.8854644298553467, -1.7417871952056885, 8.751951217651367, 12.69888687133789, 9.776554107666016, -8.83946418762207, 8.432713508605957, 7.379349708557129, 10.766681671142578, 0.8862983584403992, 3.2068536281585693, 2.334174871444702, 7.59996223449707, 3.7281174659729004, 3.5246856212615967, 0.6507203578948975], [6.713420867919922, 3.6024281978607178, 3.9548983573913574, -2.950777769088745, -7.335292816162109, -2.280472993850708, -0.36347007751464844, 0.44484108686447144, 4.0577006340026855, -2.7215774059295654, -8.269487380981445, 4.945984363555908, 3.9612820148468018, 8.276383399963379, 7.6011810302734375, -3.5695533752441406, -5.807158470153809, -5.158721923828125, 5.113988399505615, -8.756118774414062, -9.546289443969727, 5.536221981048584, -6.4510579109191895, -12.461878776550293, 0.11819225549697876, -3.410346508026123, -22.928443908691406, -5.2257866859436035, -4.512650489807129, 5.502688884735107, -7.3985209465026855, 17.5274600982666, 3.2575762271881104, 4.88002872467041, -9.085856437683105, 3.916060447692871, 10.977243423461914, 6.928632736206055, -7.121270656585693, -1.0993515253067017, -1.8598068952560425, 0.8883141875267029, -4.113901138305664, 0.8226835131645203, -8.106963157653809, -7.08032751083374, 18.03129768371582, -6.260129451751709, 7.018745422363281, 15.603279113769531, 10.672135353088379, -5.533219814300537, -7.937986850738525, 9.453543663024902, 4.506253719329834, 1.4013822078704834, -14.568302154541016, -2.104501247406006, -5.413119792938232, 6.3807373046875, -18.37925910949707, 21.152812957763672, 3.6784920692443848, 5.110840320587158, 12.02950382232666, -8.690133094787598, 11.701780319213867, 15.563265800476074, -7.518458366394043, -3.620865821838379, 4.587608814239502, -0.3239029049873352, -9.599812507629395, 4.445407867431641, -13.287433624267578, -4.760700225830078, -4.691214084625244, 3.2436540126800537, -0.33476585149765015, -6.245810508728027, -7.961946487426758, 4.129390239715576, 0.8913986682891846, 6.035871982574463, -4.251976013183594, 3.7390642166137695, 0.97872394323349, 8.939885139465332, -7.003418922424316, 6.577579021453857, -3.5337979793548584, 20.88939666748047, -0.13551321625709534, -5.554481029510498, -4.878687381744385, -18.665010452270508, 6.929379940032959, 1.4755463600158691, 12.886089324951172, -3.748255491256714, -6.542454719543457, 26.08933448791504, -6.206241607666016, 7.434786796569824, -2.7765045166015625, -11.94485855102539, 3.678697347640991, -18.43242645263672, 3.0866031646728516, -5.231109142303467, -1.619752049446106, -6.267365455627441, 11.02747631072998, 31.313474655151367, 9.348315238952637, -5.920822620391846, -15.980718612670898, 16.654998779296875, 6.646812438964844, 8.756475448608398, -3.952291488647461, 8.41003131866455, -9.249005317687988, -3.6818349361419678, -17.8727970123291, 17.315528869628906, -4.610435485839844, -3.461000442504883, 4.226390361785889], [6.22177791595459, -11.576751708984375, -1.6311484575271606, -0.159703329205513, -12.378597259521484, -18.132898330688477, -2.7604317665100098, -1.5871778726577759, 4.446633815765381, 6.799382209777832, 9.093117713928223, -13.219318389892578, -7.049637794494629, 6.590054512023926, -12.318166732788086, 2.294130563735962, -1.5987547636032104, 4.989875316619873, 4.716864109039307, -8.021709442138672, -13.876575469970703, 8.002091407775879, -8.17142105102539, 15.431992530822754, 9.659061431884766, -1.9404981136322021, 26.52706527709961, 11.602093696594238, 6.123800754547119, 3.7312352657318115, 12.478657722473145, -0.37540361285209656, 1.509010672569275, -4.925873756408691, 0.766970694065094, -19.10089874267578, -6.899739742279053, 5.399819850921631, -1.6617501974105835, -7.786398410797119, -6.141847610473633, 2.1513924598693848, -7.82964563369751, -3.37065052986145, 4.6569647789001465, -22.963451385498047, 4.5268144607543945, -5.461857318878174, -3.1165287494659424, -3.0559957027435303, 1.9617247581481934, -2.6714768409729004, 6.728713512420654, 13.741626739501953, 4.067435264587402, -9.370955467224121, -1.4501988887786865, 16.303653717041016, 12.339492797851562, -1.6031795740127563, -3.083280563354492, -11.561759948730469, -8.068023681640625, 2.6056222915649414, 1.550676941871643, -8.556510925292969, 8.570805549621582, -4.532181739807129, -5.745814323425293, -1.6817474365234375, -2.3089451789855957, 12.2615327835083, 11.207046508789062, -2.4553794860839844, -2.3917205333709717, -4.6827287673950195, 12.824124336242676, 5.024455547332764, -8.098746299743652, 4.135098934173584, -1.977330207824707, -4.005110263824463, -9.441305160522461, 1.4578198194503784, -11.715417861938477, 8.576058387756348, -9.442166328430176, 2.1928622722625732, 1.4525684118270874, -6.119006633758545, -4.906516075134277, -2.3514091968536377, -1.926572561264038, -1.0122215747833252, 5.556014537811279, -4.281021595001221, 2.394721746444702, -2.081502914428711, -1.4378646612167358, 1.2674598693847656, 5.271158695220947, 6.952025890350342, 1.1063693761825562, 15.675986289978027, -4.264659404754639, -5.001275062561035, -2.693326711654663, 5.265949726104736, 11.47814655303955, -6.117313861846924, 7.848574638366699, -4.2997918128967285, -3.0566179752349854, -7.228578567504883, -16.691341400146484, -4.591256141662598, -0.09560755640268326, 2.6569831371307373, 0.0661882683634758, -20.69867515563965, 7.414726257324219, -7.271683216094971, 4.5183234214782715, 2.7299270629882812, 0.9181487560272217, 8.953344345092773, 16.459365844726562, -6.21282958984375, 0.2978793978691101], [-7.317852973937988, -6.599215507507324, 8.764050483703613, -5.00355863571167, -2.598841905593872, 10.695097923278809, -4.142911911010742, -7.3008809089660645, 0.6888206005096436, -13.903727531433105, 1.5751086473464966, 2.7444097995758057, -2.251103401184082, -14.098262786865234, 0.6099938154220581, -1.6869492530822754, 12.177728652954102, 5.975467681884766, 1.3726404905319214, -3.1724469661712646, 4.028518199920654, -3.806973457336426, -7.814998626708984, 2.2641899585723877, -0.8218879699707031, -5.457042694091797, 2.846487522125244, -11.274340629577637, -12.257009506225586, -9.84070873260498, -7.438177585601807, -3.6464431285858154, 9.080111503601074, -0.07910741865634918, 12.59386157989502, 4.148711681365967, 0.12908890843391418, 1.8942941427230835, 4.73158597946167, 0.5743688941001892, -1.0202280282974243, -3.2946059703826904, 6.311410427093506, -5.889577388763428, -2.0728390216827393, -1.1736867427825928, -11.943825721740723, 5.567617416381836, 9.646516799926758, 4.413948059082031, 0.11678439378738403, 1.459036946296692, 1.1077704429626465, -15.800838470458984, -0.04381021484732628, 3.039824962615967, 5.397778034210205, -14.912038803100586, 1.9833086729049683, 8.075063705444336, 0.8070529103279114, -8.654521942138672, 1.1026877164840698, 3.3743412494659424, -2.991981267929077, -0.34562087059020996, -6.998566150665283, -4.1279754638671875, 3.1132564544677734, -2.906207799911499, -12.275047302246094, -6.5061540603637695, 1.0218758583068848, 1.1539275646209717, 2.492783546447754, 4.880761623382568, -10.105067253112793, -3.3386380672454834, -1.800929307937622, -1.5777912139892578, 12.22906494140625, 3.3321545124053955, -6.861245155334473, -1.2079259157180786, 16.7006893157959, -4.116389751434326, -2.1795835494995117, -2.076028347015381, 1.5318771600723267, 1.4474073648452759, 0.8037113547325134, -8.568517684936523, 3.5703775882720947, -3.947575330734253, 2.6753900051116943, 5.167972564697266, -8.776341438293457, 0.18765592575073242, -1.2939071655273438, -6.376584529876709, 13.22601318359375, -6.922417640686035, 4.946023464202881, -3.7094063758850098, -4.639355659484863, 13.446226119995117, -0.31374090909957886, -2.5954508781433105, -5.944464206695557, 10.213553428649902, -3.371208906173706, 1.732682704925537, -15.045154571533203, -4.27617073059082, 0.042779263108968735, 7.616577625274658, -3.4465901851654053, -4.513195991516113, 7.122585296630859, 1.1832128763198853, -3.0662949085235596, -5.779574394226074, 10.529654502868652, -5.878448009490967, 4.246356964111328, -2.6123504638671875, -0.6682745814323425, -0.31906092166900635, 3.4907076358795166], [-6.212762355804443, 21.24073028564453, -1.2791781425476074, 7.449983596801758, -8.464942932128906, 12.941014289855957, -2.8533213138580322, 4.483211994171143, -0.24269534647464752, -1.6250585317611694, -4.452519416809082, 8.211877822875977, -2.224203109741211, -8.553128242492676, 5.145297527313232, 6.031946659088135, 1.989143967628479, -1.0786054134368896, 18.949752807617188, -8.539453506469727, 1.1836048364639282, -13.156041145324707, -1.0320156812667847, 6.270837783813477, 3.387532949447632, 5.452689170837402, 8.822044372558594, 5.758027076721191, 2.371166944503784, -20.3978214263916, -3.656285047531128, 6.661545753479004, 17.014503479003906, 4.015352249145508, 2.8512535095214844, -5.251640796661377, 11.81359577178955, -10.440213203430176, -9.582062721252441, 12.333063125610352, -6.692533016204834, 0.9911434650421143, -4.696850299835205, 1.3673452138900757, 2.2928617000579834, 7.158143997192383, -4.792043209075928, -5.238478183746338, 13.392240524291992, -6.533535957336426, -4.256754398345947, -6.035225868225098, 2.241377353668213, -0.1473851054906845, 1.334480881690979, 2.5429015159606934, -0.3287501037120819, 1.8971319198608398, -3.4215102195739746, -7.305752754211426, 12.896805763244629, 7.68020486831665, 3.262895345687866, -6.115421772003174, 0.22106900811195374, -3.1417758464813232, 6.466381549835205, -7.996172904968262, 8.24472713470459, -10.205885887145996, -9.335375785827637, -23.668964385986328, 5.7894463539123535, 8.21529769897461, -0.7168387174606323, -0.12568438053131104, -3.468083143234253, -14.074478149414062, 11.028387069702148, -0.7721143960952759, 3.9458167552948, 8.178610801696777, -6.128586292266846, -1.6599031686782837, 7.805751800537109, -10.993570327758789, 1.1888604164123535, 1.6792079210281372, 7.61468505859375, 13.464707374572754, 4.484949588775635, -0.9585085511207581, 2.1435534954071045, -5.216620445251465, -5.048797130584717, 0.7782297134399414, -13.042353630065918, -11.717266082763672, -7.375949382781982, 3.1809351444244385, 9.427648544311523, -9.275651931762695, 14.266240119934082, -1.4816012382507324, 7.974724769592285, 16.4473819732666, -4.037238121032715, -4.165853023529053, 4.9706268310546875, 7.140984535217285, -12.77989673614502, -2.6801280975341797, -7.411427021026611, -2.280430316925049, -8.945093154907227, -3.1332080364227295, -8.268207550048828, -8.279932975769043, 6.638463497161865, 8.335530281066895, 3.2992467880249023, -0.8693610429763794, 14.383504867553711, -3.99623703956604, 2.2358896732330322, -3.6649329662323, 5.46928071975708, -11.219076156616211, -4.107358932495117], [1.766526699066162, 0.9121584892272949, -5.256335735321045, -1.7343981266021729, 4.840324401855469, 4.420254707336426, 6.2643256187438965, 1.2467410564422607, 8.14232349395752, 5.109635353088379, -11.089760780334473, -14.110719680786133, 9.20684814453125, -6.373533725738525, 5.833712100982666, -6.287046909332275, -10.521373748779297, -1.027219295501709, 8.393448829650879, -4.315859317779541, -0.7102984189987183, -9.183608055114746, 5.224036693572998, 8.177227973937988, 13.542952537536621, 17.67730712890625, 7.966446399688721, 7.255856037139893, 6.342010498046875, 6.674404621124268, 9.711536407470703, -3.7106213569641113, -8.840176582336426, 3.3099358081817627, 4.855527877807617, 5.960020542144775, -5.417530536651611, 9.548213005065918, -2.8450682163238525, 7.275974750518799, -6.027588367462158, -0.5087767839431763, 4.608969211578369, -6.244244575500488, -5.9010725021362305, -7.641663074493408, -5.789352893829346, -0.6129162907600403, -7.345441818237305, 1.379109263420105, -2.2990527153015137, 8.253312110900879, 6.185388088226318, 13.470459938049316, 19.357444763183594, -15.065669059753418, -1.5689215660095215, 7.233855724334717, 1.6641908884048462, 2.1674818992614746, 12.987611770629883, 0.10010486096143723, 9.080463409423828, -9.635785102844238, 3.724669933319092, -8.859124183654785, 9.725188255310059, -4.362480640411377, 4.225252151489258, 3.064438819885254, 0.8538082838058472, -3.987462043762207, 6.22607421875, 10.285223960876465, -14.24687671661377, 2.5648417472839355, -7.430567741394043, -2.5947887897491455, -5.707874774932861, -2.4384665489196777, 6.472503185272217, 10.313179969787598, 19.594942092895508, 5.072811126708984, -6.4705491065979, -3.135221004486084, -1.9040448665618896, -8.197818756103516, 1.9321569204330444, -0.41335320472717285, -2.2135610580444336, -10.7107572555542, -2.987220525741577, -5.129251003265381, 2.0871198177337646, -14.921838760375977, 5.465560436248779, 7.878547668457031, 0.5578836798667908, 11.2633056640625, 14.928336143493652, 2.0565996170043945, -20.97255516052246, 17.085887908935547, 7.051845550537109, -6.563007354736328, 4.481021404266357, 13.516925811767578, 6.618846893310547, -2.448363780975342, 4.565147399902344, -1.546486496925354, 7.651322364807129, 8.719686508178711, -5.6116943359375, 5.3289923667907715, -18.52623748779297, -11.372905731201172, -4.799419403076172, -7.424149990081787, -2.3487517833709717, 10.389779090881348, 0.5396552681922913, 10.029413223266602, -4.474758148193359, 13.297615051269531, 6.673682689666748, -12.830428123474121, -1.6338653564453125], [6.1445746421813965, -9.150275230407715, -10.287351608276367, 1.020820140838623, 3.186875343322754, 5.689502716064453, 0.7000795602798462, 5.036371231079102, 26.655855178833008, 15.071757316589355, -0.40407538414001465, -8.909119606018066, 3.597187042236328, 8.818772315979004, 6.37883996963501, 6.412214756011963, 8.327796936035156, 5.893876075744629, 7.832627773284912, -5.023999214172363, 5.420923709869385, 8.334877967834473, 16.17938995361328, -3.3018808364868164, -18.63712501525879, 8.686280250549316, 14.39286994934082, 6.547221660614014, 4.714780807495117, 9.341235160827637, 0.261724591255188, -5.779908657073975, -6.053116321563721, -2.562037467956543, 2.653038501739502, -12.25745964050293, 4.689866542816162, 1.007144570350647, -14.029747009277344, -11.764932632446289, 8.122196197509766, 6.267649173736572, -0.6505485773086548, 10.116562843322754, -12.778043746948242, -4.374057292938232, 5.215434551239014, -2.2475359439849854, -2.6759493350982666, -1.469288945198059, -2.97519850730896, 7.255282878875732, -11.982527732849121, 9.65816879272461, 15.163453102111816, -0.5848245620727539, -5.0810933113098145, -7.81995153427124, -16.26700210571289, -18.502391815185547, -11.696702003479004, -14.502744674682617, -12.445134162902832, -2.1706206798553467, 0.9787401556968689, 0.5324159264564514, 0.7831006646156311, -13.834498405456543, 20.806108474731445, 0.6062751412391663, 4.541547775268555, -9.215165138244629, 8.351006507873535, -3.661384105682373, -3.4501237869262695, -13.37115478515625, -2.4833779335021973, -5.1074419021606445, -5.168067455291748, 4.266359329223633, 15.40447998046875, 1.193320870399475, 2.2143688201904297, 18.778099060058594, -5.849029541015625, -10.436578750610352, -13.454015731811523, 4.717970371246338, -8.854047775268555, 13.450514793395996, -6.083892822265625, 1.5849010944366455, -6.88057279586792, 1.3475124835968018, 9.6571683883667, -4.620479583740234, -2.347639799118042, 4.10571813583374, 3.2189362049102783, -8.405409812927246, 0.5300235748291016, 16.756437301635742, -3.91614031791687, 11.17006778717041, -0.4114760756492615, 14.0435209274292, -2.7775700092315674, -11.767535209655762, 9.596155166625977, -9.224684715270996, -6.855252742767334, -15.011240005493164, 12.946707725524902, 6.2615814208984375, -7.906998634338379, -5.177064418792725, -2.5952134132385254, 12.068638801574707, -8.217826843261719, 4.29415225982666, 2.8736090660095215, 13.145791053771973, -5.140289783477783, -4.164830684661865, -2.0090491771698, 4.02374792098999, -1.2168283462524414, -0.09366514533758163, -3.482656955718994], [3.8512187004089355, 17.9193058013916, 1.636293888092041, 13.376529693603516, -5.013911247253418, -6.425007343292236, -3.5982825756073, -6.615213871002197, 3.470762014389038, 7.8491315841674805, -1.9510077238082886, 15.163688659667969, 11.80026912689209, 4.400939464569092, 4.919891357421875, 8.421616554260254, 1.4312645196914673, -3.0086100101470947, 12.826895713806152, 1.423901081085205, -6.777432441711426, -3.622555732727051, -1.0926344394683838, -0.6611443758010864, -8.268974304199219, 6.0890984535217285, -8.71644401550293, 13.55731201171875, -0.2852748930454254, 4.285529613494873, -4.641871452331543, 1.3620555400848389, 10.377264022827148, 15.507718086242676, 2.4514477252960205, 4.124931812286377, 6.971285343170166, 7.345974445343018, 1.6931992769241333, 12.310450553894043, -2.0442402362823486, 18.834505081176758, 10.584955215454102, -8.384700775146484, -0.4036515951156616, -6.0686354637146, 5.642553329467773, 6.784420490264893, -11.913370132446289, -1.1703414916992188, -14.303019523620605, -6.823375225067139, 11.294930458068848, 18.17384910583496, -8.817659378051758, 2.7111620903015137, 0.6872694492340088, 12.113543510437012, -3.157132625579834, 11.250201225280762, -2.2109720706939697, 15.420753479003906, 6.495676040649414, -6.55910587310791, 9.792744636535645, 11.102108001708984, 4.021781921386719, 11.843576431274414, -11.652677536010742, -0.5510530471801758, 6.988105773925781, 0.03020285815000534, 1.1114528179168701, 8.639638900756836, -1.889365553855896, 1.1344329118728638, 8.937559127807617, 1.733384370803833, -0.26066356897354126, -11.489636421203613, -9.744328498840332, -4.001479148864746, -3.3658194541931152, -12.708589553833008, 8.362099647521973, -7.013563632965088, 5.773551940917969, -4.762613296508789, -8.911426544189453, -3.0090956687927246, 2.4896299839019775, -1.6109954118728638, -1.2240962982177734, -4.805542945861816, 6.834197044372559, -10.648646354675293, 3.5716254711151123, 1.3610646724700928, 1.6341793537139893, -1.5816460847854614, 4.655107498168945, -10.378473281860352, -5.584625244140625, 14.211609840393066, 9.203309059143066, -9.37139892578125, -0.7812413573265076, -2.6243503093719482, -8.131512641906738, -1.0200852155685425, 3.9661803245544434, 12.561458587646484, 4.832947254180908, -0.2668156325817108, 8.40997314453125, -2.9663023948669434, 2.0551817417144775, -8.28665828704834, -6.640536785125732, 6.620866775512695, 3.816993474960327, -4.1554412841796875, -1.108858346939087, 5.798191070556641, -3.766998767852783, 0.6923865675926208, 0.3869662284851074, -4.834265232086182, -2.512627124786377], [8.142668724060059, 4.566887855529785, 1.2987267971038818, 5.577367782592773, -6.921492576599121, -4.075344562530518, 3.9362475872039795, 15.024907112121582, -2.2019615173339844, 0.9815536141395569, 4.150312900543213, -1.759294033050537, 8.184287071228027, 1.5644373893737793, 1.0757505893707275, 0.4266365170478821, -8.732866287231445, 2.720613956451416, 13.565893173217773, -10.371254920959473, -8.991799354553223, -11.852415084838867, -2.487624168395996, -6.142154216766357, 11.934496879577637, 3.7902472019195557, 2.2934563159942627, 2.8682448863983154, -4.299278259277344, -4.975935459136963, 4.113640785217285, -2.6654958724975586, 5.206770896911621, -5.4189019203186035, 4.081580638885498, 7.741898059844971, 2.891666889190674, 4.2740254402160645, 5.8384785652160645, -2.9294090270996094, -17.38100242614746, 0.29047685861587524, 13.465304374694824, -9.316207885742188, 4.439218521118164, 3.816124439239502, 3.823970079421997, -1.232747197151184, -5.995476722717285, 0.6699272394180298, 10.453104019165039, -6.103211879730225, -6.2399091720581055, 11.11164379119873, 7.532273769378662, 0.722841203212738, -12.9026460647583, 3.6705892086029053, -9.127474784851074, 12.223628997802734, 3.865192174911499, 12.933141708374023, 14.4535551071167, 8.44281005859375, 12.332298278808594, 1.2739286422729492, 13.051860809326172, 0.45638424158096313, -15.723287582397461, -1.145871877670288, 12.316963195800781, -13.40131950378418, -4.332701683044434, 13.066235542297363, -29.917497634887695, 0.8402595520019531, 2.8745954036712646, -4.7305588722229, -6.065089225769043, -2.997727155685425, 1.4254953861236572, 5.861881256103516, 5.958434104919434, 13.000448226928711, 10.258049964904785, 0.5876504778862, 4.943539142608643, -14.755187034606934, -6.788217544555664, 6.718860149383545, 3.155256986618042, 2.164198875427246, 8.647523880004883, -11.097699165344238, 8.63869857788086, -2.8010737895965576, 6.3387932777404785, -8.584556579589844, -3.079197883605957, 3.0690112113952637, 14.380735397338867, 9.15432357788086, -1.2431665658950806, 7.091366291046143, 15.734490394592285, -2.0831987857818604, 4.452511787414551, -15.266910552978516, -0.7275616526603699, -3.434839963912964, -11.154452323913574, -1.1966471672058105, 4.135494232177734, 21.9316463470459, -15.628376960754395, 6.337024211883545, -2.628051519393921, 7.257955074310303, 11.989948272705078, -4.559041500091553, 4.405123710632324, -4.458380222320557, -5.602017879486084, -8.429171562194824, -6.2397613525390625, 18.40580177307129, 8.655266761779785, -10.7855224609375, -12.817815780639648], [10.26427936553955, -12.589397430419922, 0.2572614550590515, -7.5179548263549805, 5.88914680480957, -2.0038206577301025, -1.7327425479888916, -18.88113784790039, 2.8291170597076416, 12.135601043701172, 5.130454063415527, 6.473186492919922, 13.059306144714355, -4.823853492736816, 4.759228706359863, -0.31549757719039917, -10.27015209197998, -10.463092803955078, 15.0427885055542, 2.7989449501037598, 5.301156997680664, 4.913614749908447, -10.635976791381836, -6.933796405792236, -5.238053798675537, 5.9212493896484375, -3.9826998710632324, -3.0144565105438232, -6.261747360229492, -5.364055156707764, -3.2742085456848145, 7.354730129241943, -3.140235185623169, 7.822082042694092, -9.695798873901367, -4.1629838943481445, 23.200305938720703, -5.857264518737793, 5.92455530166626, -0.23851139843463898, 2.5887904167175293, -6.961402416229248, 4.118739604949951, -13.378616333007812, -2.1364285945892334, -8.054740905761719, 1.154390811920166, 5.9175190925598145, 5.410562038421631, 11.724663734436035, 4.099255561828613, -1.3844419717788696, 0.2797197699546814, -21.974416732788086, -3.2924323081970215, -3.039759397506714, 10.721940994262695, -7.822128772735596, 7.453157901763916, -1.908605694770813, -18.83976173400879, 6.773976802825928, 6.386176586151123, -8.356095314025879, 10.591756820678711, -9.730018615722656, 5.875356674194336, 13.813360214233398, -8.725203514099121, 0.06650395691394806, -1.7476578950881958, -4.968050003051758, 7.661983489990234, 10.019290924072266, 7.982413291931152, -5.996301651000977, -13.006203651428223, -12.051740646362305, 2.4983139038085938, -13.288414001464844, -2.8246874809265137, -1.2816635370254517, -15.276235580444336, 2.964320659637451, 4.318286418914795, -4.5085835456848145, 8.867202758789062, -6.9663496017456055, -8.721883773803711, -6.8665666580200195, 14.068689346313477, 12.362648963928223, 1.226869821548462, 9.296465873718262, -8.369629859924316, -12.564912796020508, -1.6200473308563232, 1.685014247894287, -11.767240524291992, -5.115079402923584, 11.909539222717285, -0.062242649495601654, -15.504751205444336, 5.948744297027588, 6.183518409729004, -4.379237651824951, 9.663480758666992, 6.342662334442139, 2.1876227855682373, 9.573553085327148, 1.9588017463684082, 7.920690536499023, -7.91131067276001, -0.7462069988250732, 4.666327476501465, -0.8493386507034302, -12.7987642288208, -0.018077345564961433, -1.2710812091827393, 1.861122727394104, -2.5188217163085938, 7.259815692901611, -4.96258544921875, 9.006239891052246, -12.001895904541016, 14.809497833251953, 8.88029670715332, -8.404388427734375, -4.508744239807129], [-14.25674057006836, -0.03395538404583931, 7.507707595825195, -2.5262250900268555, -2.7698121070861816, 10.697073936462402, 0.567811906337738, 6.810964107513428, -2.9017014503479004, -2.891244649887085, -8.824786186218262, -3.517472743988037, -0.3036552667617798, 3.266324996948242, 7.111660480499268, 7.173707008361816, -4.823606014251709, -8.630531311035156, -2.2527458667755127, -1.200924277305603, -5.1007208824157715, -4.45078182220459, 6.167264461517334, 8.652910232543945, 8.180475234985352, 1.1564481258392334, 12.279675483703613, -6.629900932312012, -13.58047866821289, -6.979953765869141, -3.9381604194641113, 5.9468817710876465, -0.4930930733680725, 9.644720077514648, 1.4788933992385864, 13.734249114990234, -5.322909355163574, 9.601173400878906, -3.2832486629486084, -5.231571197509766, -18.30097198486328, 9.452128410339355, 0.5029894113540649, 4.9132561683654785, 4.50892972946167, 2.819490432739258, -1.0704810619354248, -1.166927695274353, -1.0485272407531738, 1.9737178087234497, 3.388991117477417, -7.071031093597412, 7.121972560882568, -2.729017496109009, -4.373940944671631, 0.5554132461547852, -1.3752118349075317, 3.0282294750213623, -0.9547918438911438, -3.55712890625, 4.790278434753418, 0.42255282402038574, 8.746241569519043, -13.650052070617676, 1.517706274986267, -5.375184535980225, 0.8710476756095886, -4.683480739593506, 15.429244041442871, -7.089596271514893, 6.345363616943359, -4.601949691772461, -0.16462428867816925, 9.470686912536621, -11.453652381896973, -5.7049994468688965, -3.337780475616455, -6.307880878448486, 8.919822692871094, 0.07242107391357422, 3.4047062397003174, -0.829582154750824, -0.9754012823104858, 4.966382026672363, 1.476516604423523, 2.940854787826538, -9.724820137023926, -10.300990104675293, -9.799646377563477, 5.279220104217529, -3.7740607261657715, -3.6052868366241455, -2.6599009037017822, -5.802585124969482, 4.572469711303711, 2.8066022396087646, -2.7385804653167725, -6.11814546585083, 3.3547730445861816, 8.849712371826172, 2.7266788482666016, 0.39455270767211914, 0.5361137390136719, 4.654923915863037, 2.913424253463745, 3.8307762145996094, 7.929075717926025, -5.369928359985352, 4.580630779266357, 8.296792984008789, -4.897215843200684, -6.055935859680176, -0.008596750907599926, 0.70960533618927, -3.0255064964294434, -2.014812707901001, -0.9368704557418823, -3.144232749938965, 1.0477299690246582, 5.622162818908691, -1.7513636350631714, -4.788136005401611, 4.003096103668213, -0.9109236001968384, -5.715051174163818, 9.594953536987305, 4.306266784667969, -8.326424598693848, -12.516756057739258], [-8.575828552246094, 4.608354568481445, -4.879404544830322, 3.3600597381591797, -1.656436562538147, -0.021139223128557205, 11.90117073059082, -6.149937152862549, 3.636705160140991, -2.2226247787475586, 1.2152845859527588, -5.349688529968262, -8.279383659362793, -1.6732923984527588, -17.374635696411133, 4.634252071380615, -6.283812522888184, 1.6682344675064087, -7.250645160675049, 1.2970885038375854, -3.618710517883301, -1.1250282526016235, 4.825334548950195, -4.936577796936035, -2.915822982788086, -0.538417398929596, 10.673084259033203, -3.7918853759765625, 3.071854829788208, -2.5497450828552246, 5.079539775848389, -3.0120692253112793, -5.90394401550293, -12.864413261413574, 5.115368843078613, -4.244889259338379, 0.7294614315032959, 0.1375257521867752, -1.859732985496521, -2.9049198627471924, -2.125735282897949, -3.7723371982574463, -7.636424541473389, 1.8527840375900269, -7.463172435760498, 0.07492929697036743, 8.429677963256836, -16.006437301635742, 2.1026291847229004, -4.368595600128174, 18.17179298400879, -13.591825485229492, -3.700641393661499, 7.292624473571777, -12.153888702392578, -11.288146018981934, -8.801597595214844, 5.134225368499756, -1.8084744215011597, -1.4323015213012695, -1.5782630443572998, -0.1763305962085724, -14.43665599822998, -1.6992237567901611, -16.517162322998047, -12.387068748474121, 24.99303436279297, 4.566606044769287, 0.36259782314300537, 5.177057266235352, -21.086198806762695, 5.1002302169799805, 5.898806571960449, -2.184828519821167, -2.3783884048461914, -11.06470012664795, -6.205774307250977, 17.914194107055664, 0.4498743712902069, 14.609740257263184, 0.4600026309490204, -1.6227656602859497, 0.5731325745582581, -7.2850751876831055, -0.17374178767204285, -1.5708434581756592, -8.097079277038574, -3.2845821380615234, 10.720491409301758, 0.3487597703933716, -8.235030174255371, 10.654048919677734, -1.6621540784835815, 1.1972243785858154, 10.985127449035645, 5.199473857879639, 11.578758239746094, 0.6493650674819946, 6.655718803405762, -3.2042810916900635, 5.189512729644775, 10.16649341583252, 0.6095967292785645, 6.825838565826416, -9.804689407348633, 2.3607888221740723, -2.6996572017669678, 4.33944034576416, -4.835695743560791, 2.7649595737457275, -10.634110450744629, 4.7137885093688965, -0.35636281967163086, -6.08054780960083, 4.1260576248168945, -7.247062683105469, -14.491166114807129, -6.875857353210449, 4.502120494842529, 1.4828470945358276, -14.971724510192871, -3.926206588745117, -7.104576587677002, 10.107553482055664, 9.241381645202637, -10.61593246459961, 0.4428184926509857, 0.7923343777656555, -1.296057105064392], [-1.7741400003433228, 15.30377197265625, 0.6148524284362793, 5.947954177856445, 5.438456058502197, 15.024590492248535, -6.453083515167236, -0.09172628819942474, -0.3177645802497864, -15.664837837219238, -1.9470189809799194, -3.550041675567627, -0.2928908169269562, 7.081531047821045, 2.757904052734375, -10.11402416229248, -3.5685033798217773, -5.478664875030518, -2.1123416423797607, 2.174307346343994, 2.1363365650177, -12.645927429199219, 10.540779113769531, -2.607828140258789, -5.455935001373291, 11.865297317504883, -5.311654567718506, -2.1341941356658936, -4.011012554168701, 7.288006782531738, 9.992831230163574, 0.8776370882987976, -5.997143745422363, 5.703328609466553, -1.4213756322860718, -6.404187202453613, -13.440482139587402, 2.4352805614471436, 2.29866361618042, 7.133655071258545, -4.933971881866455, 6.004260063171387, -5.447218418121338, 15.678413391113281, -3.524585485458374, 11.861858367919922, 2.4280669689178467, -6.017697811126709, 1.1725891828536987, 1.6180529594421387, -6.692173480987549, 14.727712631225586, 2.0907821655273438, -6.936919212341309, -1.3529752492904663, -5.618019104003906, 0.3732682466506958, -11.924173355102539, -13.297255516052246, 3.5765573978424072, 7.208714962005615, 4.50242280960083, -3.395514726638794, 4.504654884338379, 0.9656406044960022, 1.7964675426483154, 6.705885410308838, -12.18426513671875, -2.379241943359375, 6.6379570960998535, -4.003142356872559, -0.6095204949378967, -9.406846046447754, -10.924560546875, -6.214988708496094, 6.063742637634277, -1.1542103290557861, -4.310323238372803, 3.931424856185913, 12.999589920043945, 6.017650127410889, -0.04493323713541031, 8.224703788757324, 5.641643047332764, 4.5257487297058105, -13.880237579345703, -2.6521878242492676, 2.614649534225464, -3.883436918258667, 4.637948989868164, -0.6422961950302124, 7.162769317626953, -0.3786691725254059, -11.932016372680664, 7.3100762367248535, 1.2126562595367432, -5.05095100402832, -2.669386148452759, -14.684107780456543, 0.37149783968925476, 3.4960389137268066, 11.042352676391602, 4.687063694000244, -11.794806480407715, -6.120952129364014, -8.90175724029541, -11.269890785217285, 4.213810443878174, -1.9994219541549683, -8.73825740814209, -3.256293535232544, 6.003373622894287, 5.201696872711182, 4.417295455932617, -6.958871364593506, 6.6696577072143555, -0.99805748462677, -3.1821303367614746, 8.0743408203125, -0.5713440179824829, -4.807781219482422, -7.337454319000244, -9.341835021972656, -8.866692543029785, -5.59359884262085, -3.549708604812622, -1.4610087871551514, -3.9226558208465576, 2.9275686740875244], [-6.204349040985107, -0.8170701265335083, 1.556333065032959, -2.840299129486084, 4.261030673980713, -6.505624771118164, 2.948979139328003, -5.985473155975342, 6.9900336265563965, -7.356197834014893, 8.952791213989258, 0.3250214457511902, 6.775041580200195, -6.637396335601807, -6.81755256652832, 4.242910385131836, 5.218659400939941, 1.439961314201355, 1.8941322565078735, 14.2604341506958, 2.2901322841644287, -2.227926731109619, -3.8479115962982178, 5.1996750831604, 10.603796005249023, -15.992680549621582, 8.92933464050293, -10.392535209655762, -7.785569667816162, -20.35126304626465, -6.117860317230225, 8.161429405212402, -10.18934154510498, -11.822254180908203, -13.951683044433594, -8.059011459350586, 15.427671432495117, 10.800464630126953, 3.8178346157073975, 4.884749412536621, -10.680801391601562, 14.0243501663208, 20.418006896972656, -18.33418083190918, 2.183750867843628, -9.394975662231445, 8.369026184082031, 2.077265739440918, 8.163637161254883, 0.8673981428146362, -3.2138428688049316, 2.481351613998413, -13.370380401611328, -4.793015003204346, -3.3466293811798096, 19.40622329711914, 7.366019248962402, 3.858875274658203, 3.9068057537078857, -0.8035115003585815, 13.571682929992676, 17.583356857299805, 20.0282039642334, 6.997017860412598, 12.062775611877441, 6.600417613983154, 1.8228886127471924, 5.604096412658691, -11.069075584411621, 6.157307147979736, 6.317744255065918, 1.3074023723602295, 2.1104576587677, 10.682855606079102, -6.963381767272949, -12.28360652923584, 4.49012565612793, -12.912276268005371, 8.673547744750977, -3.5299596786499023, -12.64730167388916, 9.10304069519043, -12.824760437011719, 13.908011436462402, 3.8207759857177734, -2.9221744537353516, 3.0293467044830322, 3.2861175537109375, 15.219683647155762, -4.599870681762695, 9.941466331481934, 12.640942573547363, 6.549499988555908, 15.491554260253906, -7.71087646484375, -21.9029541015625, 1.013646125793457, 4.627548694610596, -6.395843029022217, 5.742430686950684, 13.742793083190918, -7.747813701629639, -8.063833236694336, -4.96295690536499, 2.9141693115234375, 2.0907034873962402, 10.644512176513672, 4.557425022125244, 10.147477149963379, 1.2906972169876099, 24.970008850097656, -10.60899543762207, -9.632816314697266, -11.832056045532227, -3.9949891567230225, 0.596511721611023, -18.305484771728516, -7.884640693664551, 8.409897804260254, -6.072583198547363, 2.358133554458618, 11.732582092285156, -4.934038162231445, -6.191251754760742, -3.962529420852661, -3.495863437652588, 8.613753318786621, -18.43050765991211, -2.109757900238037], [-2.134571075439453, 6.924145698547363, -5.482408046722412, -4.102558135986328, -6.31379508972168, 2.833925724029541, -8.588892936706543, -5.737580299377441, 1.1181095838546753, -3.308506488800049, 5.225925922393799, 16.248775482177734, 16.62567901611328, -2.026151657104492, -11.129880905151367, 1.3937596082687378, -0.2871019244194031, -5.64321231842041, -2.3647139072418213, 18.893312454223633, -5.645102500915527, -5.609823703765869, -10.575124740600586, 0.07072895765304565, 0.664741039276123, -9.334758758544922, 0.5815482139587402, -0.5939954519271851, 1.755601167678833, 1.8704062700271606, -1.637341022491455, 17.523969650268555, -3.1398067474365234, -10.031339645385742, -7.718664169311523, 16.53895378112793, 0.4472782611846924, -5.493295669555664, 16.483930587768555, -0.7245148420333862, -1.3293801546096802, 0.3443928360939026, 6.952070713043213, -8.695372581481934, -1.8667882680892944, -1.9082801342010498, 5.170165538787842, -1.0669996738433838, 1.032457709312439, 3.0381579399108887, -8.412062644958496, -3.2697558403015137, 6.847537040710449, 6.633295059204102, -1.1512172222137451, 4.151363849639893, 3.9893431663513184, -7.091205596923828, -4.656101226806641, -5.346463203430176, 3.956839084625244, -5.736958980560303, 7.583579063415527, -6.647984981536865, -4.238897800445557, -9.977705955505371, 4.732630252838135, 8.325721740722656, -8.668037414550781, -6.683600902557373, 1.2321043014526367, 1.42539381980896, 6.13932466506958, 1.7767261266708374, 3.237199544906616, 1.210240125656128, -7.881010055541992, 13.403108596801758, -1.1828820705413818, 10.186131477355957, -4.506272792816162, -1.4388102293014526, -3.196934223175049, 2.1772279739379883, -1.4261863231658936, 16.480072021484375, 7.794469833374023, 1.7769180536270142, 12.134910583496094, -5.309505462646484, -1.5238380432128906, -1.664963960647583, -0.052570194005966187, -6.2651519775390625, -14.708706855773926, -7.906158924102783, 5.935254096984863, 12.152897834777832, 5.409881591796875, -4.454041481018066, -13.882806777954102, -6.372249126434326, 4.435943603515625, -2.6744139194488525, 0.9125756621360779, -13.38916301727295, -7.231781005859375, -9.43746566772461, 5.097818374633789, -4.286503791809082, 1.6369966268539429, -2.4014785289764404, 0.7438628673553467, -8.198588371276855, 5.889737606048584, 5.744197845458984, 3.8481178283691406, -11.871781349182129, -12.272058486938477, -7.245584487915039, 6.989976406097412, -11.395092010498047, -3.5995845794677734, -2.2098374366760254, -10.715883255004883, 8.698386192321777, -4.151933670043945, -3.44415545463562, -2.5988776683807373], [0.19160039722919464, -18.812490463256836, -10.476757049560547, -4.193365573883057, 17.777915954589844, 2.96551775932312, 1.2494592666625977, 4.708764553070068, -5.79278039932251, 0.8285318613052368, 9.539447784423828, -8.515413284301758, 2.869352340698242, -0.3313003480434418, -1.1878043413162231, 7.4249114990234375, 7.72384786605835, -12.48414134979248, 4.726059436798096, 0.392804354429245, 10.331092834472656, 11.186768531799316, 8.384931564331055, -5.783451080322266, -2.2622692584991455, -2.860856294631958, 18.51700210571289, 4.347984313964844, 13.828012466430664, 18.872177124023438, 4.566946029663086, -0.09669769555330276, -4.247916221618652, 3.315178155899048, 4.445392608642578, -4.7504119873046875, 7.21671724319458, 6.905076503753662, -19.20895767211914, -16.491296768188477, 12.8527193069458, 0.2680213153362274, 0.2265302985906601, -0.6167157888412476, -1.9295806884765625, -1.0388076305389404, 0.3305998742580414, -7.987038612365723, 8.040367126464844, 1.5145188570022583, 14.965603828430176, 4.890096664428711, -14.757651329040527, 12.890985488891602, 10.137895584106445, 4.487071990966797, 8.445215225219727, 6.949746608734131, -4.042664527893066, -4.80012321472168, 0.7383272647857666, 3.3325743675231934, -14.036240577697754, 4.139817237854004, 6.911983013153076, -5.197855472564697, 12.415060043334961, -13.12078857421875, 7.617522239685059, 8.804010391235352, 9.795997619628906, -9.108903884887695, -6.448681831359863, 4.315609455108643, -3.352649211883545, -28.863361358642578, -7.723312854766846, -0.189066544175148, -10.13459587097168, 17.117151260375977, 17.120399475097656, 12.377861022949219, -7.611537456512451, 14.937055587768555, 2.338778257369995, -15.060654640197754, -11.808488845825195, -3.0254037380218506, -0.7288646101951599, 7.373288154602051, -20.63263702392578, 8.820394515991211, 3.473558187484741, -8.105997085571289, 18.220212936401367, 0.1970997452735901, -0.09481758624315262, -1.2305104732513428, 6.655002117156982, -0.7617141604423523, 6.760936737060547, 14.168171882629395, 2.4880528450012207, 6.981486797332764, -7.2354736328125, 11.895923614501953, -9.71218490600586, 1.274388074874878, -4.362595558166504, 3.543972969055176, -0.5798566937446594, -7.880374908447266, 23.082338333129883, 8.808218955993652, 1.0303623676300049, -21.194618225097656, -14.017343521118164, 0.2816731631755829, -4.843031406402588, 0.7886123061180115, 1.3715696334838867, 3.8412997722625732, -7.328171730041504, -6.698441982269287, 14.98431396484375, -1.2461564540863037, -4.358944892883301, 10.938796997070312, -6.514186382293701], [-3.2527990341186523, -9.462639808654785, 10.091046333312988, -21.348224639892578, -1.7531633377075195, -2.8021576404571533, -3.4991726875305176, -8.07073974609375, 12.543808937072754, 12.451680183410645, 12.699942588806152, -1.620383381843567, 0.008600708097219467, -9.430045127868652, -3.020538568496704, -1.8086955547332764, 2.418826103210449, -8.178887367248535, -4.558730125427246, -4.194529056549072, -8.555903434753418, -2.0989890098571777, -27.553279876708984, 11.989901542663574, 7.52945613861084, -9.243071556091309, -2.660209894180298, 4.778783321380615, 6.784063339233398, -4.680723667144775, -0.5777570009231567, -10.774006843566895, 9.728025436401367, -7.000287055969238, 3.9136006832122803, 0.4567391276359558, 6.578490257263184, -6.4287543296813965, 7.091381549835205, -5.8128485679626465, 4.677286624908447, -12.398420333862305, 0.45637771487236023, -10.859675407409668, -11.839330673217773, -3.6206889152526855, -3.3167266845703125, 14.877949714660645, -8.703720092773438, 9.06518268585205, -20.35175323486328, -17.548500061035156, 8.855658531188965, -8.192137718200684, 5.693806171417236, 7.530050754547119, 12.136310577392578, -14.403071403503418, 16.252161026000977, -7.970767974853516, -2.5416553020477295, 1.661393642425537, -5.704872131347656, 3.49566912651062, -2.009756565093994, 11.565828323364258, -12.304516792297363, -7.78429651260376, -6.682215690612793, -7.850310325622559, -4.2218523025512695, -9.434794425964355, -11.251970291137695, 9.158112525939941, 10.714035987854004, 6.230390548706055, -18.8397274017334, -0.600460946559906, -2.386169195175171, -9.503901481628418, -13.483213424682617, 16.624120712280273, -3.6443355083465576, 11.906404495239258, 8.277849197387695, 2.272087335586548, 9.328871726989746, 0.8226138949394226, 6.009442329406738, -17.119205474853516, -1.1916007995605469, -4.041965007781982, 1.6873940229415894, 7.140467643737793, -17.6304931640625, 0.6139777302742004, 1.6014595031738281, -3.1082262992858887, -3.352693557739258, 2.3508806228637695, 4.11350679397583, -13.245866775512695, -12.82201099395752, 18.157686233520508, 16.146408081054688, 10.273964881896973, 8.164013862609863, -7.280285358428955, 6.53897762298584, 4.900708198547363, -0.8498890399932861, -12.984567642211914, -6.2493977546691895, -11.042384147644043, 7.283667087554932, -2.149625301361084, 11.616293907165527, 2.010455846786499, 15.334464073181152, -10.088417053222656, -1.6892940998077393, -0.12023276090621948, 14.94481372833252, -1.2198045253753662, 1.1723219156265259, 13.250357627868652, 4.258507251739502, -1.4339584112167358, 13.241290092468262], [0.06673365831375122, 10.149751663208008, -23.126483917236328, 5.843645095825195, 1.5872910022735596, 4.672384738922119, -0.931600034236908, 12.412041664123535, 3.306267023086548, -3.9518754482269287, 12.715913772583008, 3.5167415142059326, 12.855734825134277, -2.2249906063079834, 8.7274169921875, -2.4954779148101807, 0.8426364064216614, -7.73597526550293, 8.411633491516113, -6.338580131530762, 5.229512691497803, 4.585646152496338, -2.8532917499542236, -15.444864273071289, 11.952150344848633, 12.337956428527832, -12.301640510559082, 23.76425552368164, 12.753966331481934, -1.3439925909042358, 9.821114540100098, 4.85513162612915, 3.0711324214935303, -7.405765056610107, -7.720181465148926, 3.5368924140930176, 3.6209311485290527, -12.693849563598633, -6.458693027496338, -5.30004358291626, 7.525201797485352, -13.039225578308105, -2.9684112071990967, 2.0036611557006836, 0.6417025327682495, 14.425063133239746, -12.105920791625977, 9.155861854553223, 1.7996177673339844, 2.4565446376800537, 5.523797988891602, 0.7415685653686523, 4.743816375732422, 5.279589653015137, 14.983312606811523, 17.618024826049805, -8.301676750183105, 1.2616099119186401, -8.791311264038086, 10.514120101928711, -1.9080277681350708, -1.9488215446472168, 7.828186511993408, 6.293038845062256, 9.249361038208008, 7.422245979309082, 7.660022258758545, 8.496830940246582, 2.411482572555542, -3.5429933071136475, 2.872572422027588, -7.61114501953125, 10.075931549072266, -3.354318618774414, 8.707327842712402, -3.0262269973754883, 5.3332905769348145, -1.5586977005004883, -16.31967544555664, 4.881242275238037, 1.8901928663253784, -6.577959060668945, 0.26114463806152344, 3.7814629077911377, -2.1755902767181396, 12.691895484924316, -0.10370831191539764, 3.132251501083374, 11.610408782958984, -0.14564631879329681, 9.344962120056152, -10.160146713256836, -9.920995712280273, -2.4230332374572754, -12.617411613464355, -2.625394105911255, 12.044906616210938, 10.47584342956543, 4.716909408569336, -5.8267011642456055, -17.289024353027344, -19.109167098999023, 18.457258224487305, -9.475638389587402, -3.4081006050109863, -0.614615797996521, -3.8005452156066895, 2.8615877628326416, -5.346617698669434, -1.5370018482208252, -2.32515025138855, 0.581359326839447, 4.796409606933594, 9.93067741394043, 3.6768057346343994, 8.422076225280762, 8.278363227844238, -4.658577919006348, -5.9107537269592285, 3.590742349624634, -0.018029509112238884, -1.409071922302246, -1.85688054561615, 4.579239845275879, 4.9752020835876465, 12.374052047729492, -10.32868480682373, 9.112385749816895, 1.222388505935669], [-1.6159026622772217, -5.66348934173584, -9.611254692077637, -5.07207727432251, 1.24773108959198, 1.06024968624115, 0.9115447402000427, 3.0983588695526123, -0.08813150227069855, 9.445058822631836, -0.32900866866111755, -3.4878671169281006, -1.634470820426941, 4.962751388549805, 2.4618265628814697, 5.54417610168457, 9.648056030273438, 4.067959785461426, -6.047770977020264, -4.827236175537109, 4.768361568450928, 6.694712162017822, 14.942595481872559, -1.896825909614563, -6.630410671234131, -1.9361815452575684, 20.54022216796875, 7.122089385986328, -3.303884983062744, 12.145543098449707, 11.453394889831543, 4.812681674957275, -3.01474666595459, -16.12358856201172, 10.76120376586914, 1.0287995338439941, -4.765285968780518, 4.314043045043945, 3.483720541000366, -9.96926498413086, -3.666297197341919, -0.014456202276051044, 0.7081982493400574, 3.6516294479370117, 2.2001965045928955, 4.381305694580078, -2.149230480194092, -4.481624603271484, -4.801560878753662, 0.13434168696403503, -8.192389488220215, 14.58997631072998, -9.030376434326172, 7.248898506164551, 3.2330167293548584, -13.393487930297852, -3.3474788665771484, -1.5602169036865234, -13.682019233703613, -16.59579086303711, 5.418303966522217, -4.322904109954834, -6.977078914642334, -1.0706406831741333, -7.641520023345947, 4.830661773681641, -4.66857385635376, -0.14972956478595734, 3.3126723766326904, -13.583976745605469, 3.415930986404419, -19.12880516052246, 6.521487712860107, 4.046483993530273, -8.608129501342773, -17.8278865814209, -7.0469279289245605, 3.7130401134490967, 6.667032718658447, 4.363785743713379, -4.318619251251221, -5.934300899505615, 7.345905780792236, -4.88399600982666, 0.2200738489627838, -0.06446744501590729, -14.670357704162598, 1.5311766862869263, 5.14905309677124, 9.816957473754883, -1.0278218984603882, 1.2336509227752686, 0.7484268546104431, -9.700157165527344, 2.0662193298339844, 12.8397855758667, 7.550653457641602, -3.1879663467407227, -4.787762641906738, 0.8142878413200378, 2.2838566303253174, 3.930257797241211, 6.558703899383545, -7.12281608581543, 10.341540336608887, 1.4324102401733398, -16.310070037841797, -7.938597679138184, 7.950951099395752, -0.47904112935066223, 0.5013033151626587, 0.856371283531189, 8.75990104675293, 0.39803725481033325, -13.889488220214844, -8.531486511230469, 1.1583513021469116, -3.2095582485198975, -12.201506614685059, 4.262145042419434, 3.2952468395233154, -5.165604591369629, -6.256124973297119, -0.03845714032649994, 12.037518501281738, -12.687904357910156, -5.6413350105285645, 4.295041561126709, -5.825692176818848], [10.946452140808105, 12.927935600280762, -9.60273551940918, -27.00600814819336, -6.243377685546875, -9.054147720336914, -15.940533638000488, 6.368204593658447, 7.011336803436279, -1.889758825302124, -9.477659225463867, 5.413024425506592, -0.15326015651226044, 3.4969050884246826, -0.11164508014917374, 3.6928014755249023, 2.311253547668457, 8.339256286621094, 18.102088928222656, -4.822051525115967, 2.497375965118408, 1.4359617233276367, 21.680604934692383, -11.330313682556152, 3.3693325519561768, 1.0685652494430542, -5.56566858291626, 3.0782470703125, -6.407625198364258, -8.837761878967285, -18.64103126525879, 21.506126403808594, 3.505577325820923, -2.000471830368042, 7.087373733520508, 4.600013732910156, -8.011281967163086, 0.19090791046619415, 2.3345398902893066, 1.1738649606704712, -6.669219017028809, 2.716787815093994, -10.804774284362793, 0.3993856608867645, -3.4835665225982666, -0.9590092897415161, 5.836886405944824, 0.46251219511032104, -2.0815417766571045, 13.427412033081055, 4.074224948883057, 0.398317813873291, -6.240645885467529, 9.74688720703125, 1.4990264177322388, 9.186841011047363, 0.602509617805481, 1.1176598072052002, -4.236393928527832, -16.076091766357422, -15.428430557250977, 3.112022638320923, -9.06752872467041, 7.653034687042236, 6.165359020233154, -3.133840322494507, 8.589479446411133, -9.127313613891602, 5.847423076629639, 1.1273343563079834, -1.4604742527008057, -10.895503044128418, 3.0588560104370117, 6.409620761871338, -1.9356831312179565, -12.72690200805664, 2.925931930541992, -10.837910652160645, -9.752693176269531, -0.08545876294374466, -6.949469089508057, -8.964679718017578, 2.1964123249053955, 0.5751026272773743, -5.614409446716309, 1.9511518478393555, 1.8627852201461792, 4.657930374145508, -4.3057050704956055, 12.890762329101562, -3.278704881668091, 15.215802192687988, 6.472225189208984, 10.456048965454102, 9.274720191955566, 4.8421101570129395, 12.022303581237793, -2.4341208934783936, 2.609281063079834, -4.644235134124756, -8.716617584228516, 0.130630761384964, -5.142114162445068, 4.732422828674316, 3.795487880706787, -7.755392551422119, 4.190659046173096, 3.4576051235198975, 2.3455820083618164, -7.582040786743164, -12.623348236083984, -9.306099891662598, -9.137925148010254, -13.653458595275879, 1.3305532932281494, -12.59505558013916, 8.694070816040039, 13.778962135314941, 11.871694564819336, -6.131844520568848, 3.233668565750122, -4.538559436798096, 3.2045466899871826, -6.0006256103515625, -7.685229778289795, 26.87139320373535, -22.71314239501953, 8.175580024719238, 0.8597228527069092], [6.062152862548828, -7.434415817260742, 15.635854721069336, -2.098031759262085, -6.1146135330200195, -7.476301193237305, 2.74249005317688, -7.231517791748047, -9.328734397888184, 20.009124755859375, -1.3104428052902222, -11.605368614196777, -10.288687705993652, 8.710179328918457, -12.76780891418457, -2.862668752670288, -7.680612087249756, 3.901228427886963, 8.77554702758789, -2.233250856399536, -0.683018684387207, 0.7297907471656799, -8.149856567382812, -3.241616725921631, 6.808538436889648, 2.460082530975342, 24.108871459960938, -7.999661445617676, -0.13106240332126617, 3.030593156814575, 4.543047904968262, -8.243142127990723, -1.7948161363601685, 18.870044708251953, 5.260492324829102, -6.590156555175781, -5.182684421539307, 8.98585033416748, 8.381758689880371, -11.295126914978027, -12.266526222229004, -3.9161646366119385, 1.892351508140564, 2.6529781818389893, 18.393051147460938, 1.6895374059677124, 7.566815376281738, -2.7695541381835938, 4.373888969421387, 4.6004557609558105, 5.851090908050537, 16.68740463256836, 0.21011242270469666, 14.975373268127441, -9.451077461242676, 0.19490307569503784, 0.745206356048584, 0.4975305497646332, 2.9427478313446045, -11.455782890319824, -6.472139358520508, 9.204432487487793, -6.970834255218506, 11.103169441223145, -4.859797954559326, 5.176355361938477, -7.438823223114014, -2.2149875164031982, -5.2077956199646, 12.980570793151855, 4.838878631591797, 19.40245819091797, -8.612513542175293, -6.509314060211182, -6.848802089691162, -6.5385870933532715, 17.750139236450195, -4.909767150878906, 5.950323104858398, -6.227542400360107, 9.91774845123291, -3.9101943969726562, 1.6172629594802856, -6.45263147354126, -2.844485282897949, 0.5251532793045044, 2.905653238296509, -13.465380668640137, -13.620128631591797, 6.402733325958252, 8.154414176940918, 8.556197166442871, 0.27132588624954224, 10.230419158935547, 5.298578262329102, 17.770774841308594, -3.025805711746216, -10.591814994812012, -6.0706963539123535, -0.07706602662801743, 15.879950523376465, 16.215688705444336, 0.4219318628311157, 6.231867790222168, 1.412069320678711, 0.4078145921230316, 9.713240623474121, -0.5453441739082336, -16.34087371826172, 15.224447250366211, 0.651283860206604, 2.730093002319336, -3.774942398071289, -6.71863317489624, 11.712346076965332, -5.200191974639893, 3.047475576400757, 13.111605644226074, 3.5246565341949463, 5.964820384979248, 6.600142002105713, -9.13107967376709, -1.7545143365859985, 14.307106018066406, 11.965024948120117, 4.353670597076416, 6.492640018463135, 13.967351913452148, -0.2687202990055084], [-0.0756019726395607, 2.5082828998565674, 5.4848833084106445, 5.7881951332092285, -2.832777976989746, 8.91528606414795, -1.3982685804367065, 2.1644461154937744, -0.45764434337615967, -14.280830383300781, -4.807531356811523, 9.229789733886719, -7.0743632316589355, 0.892402172088623, 8.65485668182373, 11.07569694519043, -7.120940685272217, 6.470520496368408, -3.195451259613037, -15.976639747619629, 5.269077301025391, -10.429224014282227, 7.436061859130859, 15.333406448364258, 10.373723030090332, 0.07458337396383286, -2.7579195499420166, 7.405426502227783, 2.5602025985717773, -6.648548126220703, 4.044536113739014, 1.0958744287490845, 9.789508819580078, 3.477402448654175, 3.9649906158447266, 5.162165641784668, 2.1025776863098145, 4.340031147003174, -4.2283525466918945, 6.502773761749268, -4.339775562286377, 19.599580764770508, 4.401538372039795, 4.969656944274902, -6.178153038024902, 3.519646167755127, -0.13115526735782623, -4.980508804321289, 3.415070056915283, -9.043633460998535, 3.2317895889282227, 0.9442498087882996, 1.101341724395752, 2.0126166343688965, 11.358819007873535, 7.620217323303223, 1.8459159135818481, -6.868365287780762, 10.818676948547363, 15.77365493774414, 21.625049591064453, 7.412562370300293, -8.730550765991211, 0.00934771541506052, -13.727456092834473, -2.6526520252227783, -8.211993217468262, -4.763749122619629, 9.797276496887207, -1.875405192375183, 0.4088728129863739, -3.5174267292022705, 4.448559284210205, 5.913303375244141, 4.022006511688232, 8.592551231384277, 5.420519828796387, 8.045258522033691, 3.521503448486328, 4.730135917663574, 2.4703476428985596, 8.593829154968262, 4.049983501434326, -9.199860572814941, 3.487276792526245, -2.3244757652282715, 6.163732051849365, -14.349154472351074, 7.5932230949401855, 7.1763105392456055, -7.689991474151611, -8.828798294067383, -1.0686484575271606, -9.3733491897583, 10.440801620483398, 10.8909912109375, 6.43212366104126, 0.903443455696106, 7.918141841888428, 3.70294189453125, 8.870105743408203, -24.14183807373047, 0.8855796456336975, 20.912784576416016, -0.48359811305999756, -6.176666736602783, -4.123000621795654, 9.160091400146484, 4.2038655281066895, 13.843681335449219, -7.561930179595947, -5.560595512390137, -9.689287185668945, -0.04760443791747093, 1.97702956199646, 0.8461757898330688, 6.372351169586182, -10.736640930175781, 8.647221565246582, 2.027803421020508, -14.459945678710938, -2.22092866897583, 3.1277832984924316, 2.8108925819396973, -5.238375663757324, -1.122167706489563, -3.7353909015655518, -5.096975326538086, -9.920632362365723], [6.269276142120361, 10.500410079956055, 5.141615390777588, -4.143524169921875, 1.7116258144378662, -9.165904998779297, -11.78783130645752, -2.113584041595459, 1.602339267730713, 3.5627615451812744, 8.512990951538086, -13.86455249786377, 15.866018295288086, -0.21231791377067566, -3.0077974796295166, 0.8662821650505066, -1.6202484369277954, -8.964346885681152, -1.962187647819519, -3.541773557662964, -7.958415985107422, -9.335273742675781, -4.790475845336914, 6.232659816741943, 6.829667091369629, 10.79997444152832, 14.187824249267578, 10.53232192993164, -4.893588542938232, 1.7725838422775269, 5.466033935546875, 10.561601638793945, 0.2396012395620346, 5.35838508605957, 2.623307704925537, 5.726443767547607, -12.621484756469727, -4.968934535980225, 8.127055168151855, 10.098794937133789, 5.200520992279053, -12.390450477600098, 1.5882796049118042, -3.6200110912323, 4.543521881103516, -4.611147403717041, -6.3674492835998535, 7.668307781219482, -14.93252944946289, -2.1814937591552734, -13.158228874206543, -6.448555946350098, 23.208845138549805, -2.508104085922241, -9.249704360961914, -8.284285545349121, -9.832481384277344, -3.285836935043335, -5.550013065338135, -15.376901626586914, -10.318747520446777, -9.00760269165039, 1.411585807800293, 6.1367645263671875, -15.926593780517578, -1.434212565422058, 1.7455077171325684, 2.4220128059387207, 11.26822280883789, -11.829160690307617, -16.938865661621094, -2.2731776237487793, 7.70085334777832, -3.2297873497009277, 11.735007286071777, 9.562374114990234, -2.7150001525878906, 11.445221900939941, 11.269072532653809, -16.505722045898438, -6.210077285766602, -12.900751113891602, 1.2262202501296997, 6.237669944763184, 1.9586516618728638, -9.543631553649902, 12.331439971923828, 13.323053359985352, 6.01732873916626, 1.1441763639450073, 11.153916358947754, -11.724568367004395, -2.4378130435943604, 2.7786755561828613, -5.993404388427734, 7.00266695022583, -0.9719056487083435, 4.885124206542969, -8.916034698486328, -17.13697624206543, 4.505428791046143, -1.865877389907837, -10.12937068939209, -1.4792145490646362, 6.725064277648926, -2.3773670196533203, -0.12547077238559723, -6.7957587242126465, -5.233397006988525, -4.80785608291626, -10.64201545715332, 9.122710227966309, 4.040721893310547, -15.660282135009766, 0.3137947618961334, 4.0109663009643555, 9.414738655090332, -3.4981460571289062, 2.8204641342163086, -12.739217758178711, 6.531737804412842, -8.673881530761719, -4.207149505615234, 3.866941452026367, -13.204523086547852, 6.933387756347656, -0.5075858235359192, 10.89664363861084, 0.1219693273305893], [5.584654808044434, 1.2430830001831055, -6.3222527503967285, 3.4181783199310303, -7.564517974853516, -1.7310609817504883, 9.773612022399902, -8.846643447875977, 3.3014845848083496, -8.291498184204102, 2.703977584838867, -10.459412574768066, -15.261812210083008, -1.951865315437317, -8.185990333557129, -5.015578746795654, 9.96158504486084, -0.2637806236743927, -9.693753242492676, -16.630226135253906, -9.153804779052734, -0.8876270055770874, -4.967015743255615, -6.3720550537109375, 18.77539825439453, -2.6580302715301514, 6.142879962921143, -1.348737120628357, 4.296451568603516, -4.840816974639893, 0.5221619009971619, -7.3316121101379395, -3.663156509399414, 4.9782915115356445, 0.9101117253303528, -9.956815719604492, -1.7884032726287842, -12.423416137695312, -1.6980080604553223, -3.063272476196289, 16.472536087036133, -7.948709487915039, -9.075484275817871, 8.144011497497559, 7.7021284103393555, 4.431412220001221, -6.101170539855957, -2.7647435665130615, -2.2201552391052246, -5.459427833557129, 11.450621604919434, 2.1253201961517334, 5.203224182128906, 16.123878479003906, -4.290304660797119, -6.503113269805908, -12.286993026733398, -4.859316349029541, -2.693636417388916, -1.8346014022827148, 13.543375015258789, 8.026725769042969, -3.5931639671325684, 23.101482391357422, -4.366485595703125, 11.127385139465332, 21.31965446472168, 12.150531768798828, -11.3401460647583, 8.252908706665039, -1.2281098365783691, 7.543671607971191, -12.977301597595215, -10.371879577636719, -1.6132266521453857, 2.458944082260132, 20.833532333374023, 12.722737312316895, -1.20200514793396, 1.1573996543884277, -6.405394077301025, -12.918462753295898, 10.180078506469727, -0.11985927075147629, -1.8475468158721924, -15.663610458374023, -11.911816596984863, 2.612215995788574, 2.724747657775879, -3.269164562225342, -10.482293128967285, -8.084680557250977, -8.205363273620605, 6.341235160827637, -1.1565778255462646, 13.186586380004883, -1.0137684345245361, 1.226997971534729, 7.265403747558594, 4.276191711425781, 6.723026275634766, 0.2204824835062027, 26.67501449584961, -7.201076507568359, -6.0272536277771, 1.172544002532959, 5.468932628631592, -0.3566799759864807, -9.097827911376953, 1.6271426677703857, 7.68696403503418, 12.961461067199707, 2.2747910022735596, 10.99830436706543, 0.07536891847848892, -22.475666046142578, -2.6928067207336426, 1.2750883102416992, 8.54309368133545, 4.417076587677002, -13.240642547607422, -6.930363655090332, -14.773406982421875, 11.554282188415527, 9.554816246032715, -8.62811279296875, -4.484078884124756, 13.710116386413574, 0.8631508946418762], [-9.130446434020996, -9.115365028381348, -5.183705806732178, -3.9439315795898438, 14.807252883911133, 4.732080936431885, -5.593532562255859, -7.121017932891846, 2.9847569465637207, -25.648731231689453, 7.296643257141113, -0.4446781575679779, -6.573705196380615, -2.6107468605041504, -4.615438461303711, -2.673624277114868, -6.480315685272217, -2.4979147911071777, -19.47420883178711, 2.4325037002563477, 10.715953826904297, -7.657840728759766, 4.5120744705200195, -0.7537192106246948, 17.757667541503906, -15.44112777709961, -7.1366190910339355, -3.682142496109009, -10.35096263885498, 4.805136680603027, 1.3046417236328125, -8.546673774719238, -6.745916366577148, -6.1414794921875, 8.072041511535645, 6.871238708496094, -2.920058012008667, -6.495057106018066, 4.694613933563232, -4.492401123046875, 10.598803520202637, -8.571097373962402, -9.948121070861816, 6.45766544342041, 15.265829086303711, 10.35942554473877, -3.676088333129883, -2.4176437854766846, -0.017153875902295113, 2.3930554389953613, 8.859298706054688, 12.217397689819336, -11.493098258972168, -13.068510055541992, 0.7566321492195129, -2.2004241943359375, 14.331963539123535, -15.80543327331543, -2.5998826026916504, 10.153691291809082, 2.5144670009613037, 2.3794515132904053, -3.8760242462158203, 13.658629417419434, 5.2594218254089355, 13.045294761657715, 3.844315767288208, 2.7372262477874756, 1.5924359560012817, -2.6274139881134033, -4.607429027557373, 6.068901538848877, -15.773476600646973, -8.083250999450684, -0.5710674524307251, 1.7551839351654053, 4.363559246063232, -19.750394821166992, 0.1308850646018982, 13.128788948059082, -4.491978645324707, -3.2690489292144775, 1.1718463897705078, 11.448845863342285, 9.266749382019043, 0.9104315042495728, -15.2493314743042, 16.34316062927246, -5.876965045928955, -2.8944506645202637, -2.2975783348083496, -5.59027099609375, -6.362590789794922, -6.962417125701904, -10.26901912689209, 5.942477703094482, -1.758858561515808, -5.704580783843994, -5.46671199798584, 7.410824298858643, -4.582894802093506, -2.354717254638672, 7.401584148406982, -14.274643898010254, -1.3094922304153442, 9.8716402053833, -9.285737037658691, -3.4301607608795166, 1.6761997938156128, 5.913510799407959, 20.396352767944336, -1.97468900680542, -10.646557807922363, -1.0546305179595947, -1.9294019937515259, 7.130270004272461, 4.074590682983398, -8.717833518981934, 2.4809858798980713, -12.196625709533691, -0.47750744223594666, -11.811747550964355, -7.907780170440674, -8.829793930053711, 0.19158272445201874, -1.9171557426452637, 6.225358009338379, 3.882267475128174, 1.5052142143249512], [-5.636087417602539, -0.8038557171821594, 6.004497528076172, 5.762248992919922, -14.591864585876465, 0.6922029852867126, -7.434032440185547, -1.7384426593780518, 11.599101066589355, 0.611002504825592, 2.7068123817443848, 6.796231746673584, -8.813460350036621, -8.919057846069336, -5.388094425201416, -8.108624458312988, -8.621874809265137, 11.291365623474121, -10.229086875915527, -19.140043258666992, -8.0222749710083, 0.9356774091720581, 0.17534063756465912, -0.6223292946815491, 5.387430191040039, -12.182854652404785, 0.5302370190620422, 3.3017029762268066, 12.830000877380371, -11.905364990234375, -6.042530059814453, 1.922817349433899, -10.064847946166992, 10.568626403808594, -1.9003853797912598, -11.957752227783203, -9.225955963134766, -17.09963035583496, 0.25216999650001526, 3.8170509338378906, 2.037282705307007, -6.113338947296143, -13.034049987792969, -1.9250741004943848, -17.58643341064453, 4.0356669425964355, 4.27056360244751, 6.180734634399414, 1.149803638458252, -5.673349857330322, -9.02387809753418, -14.7550048828125, 2.4086287021636963, 2.0165576934814453, -18.946128845214844, 5.208724021911621, 10.206069946289062, -16.39815330505371, 2.9060163497924805, -8.593442916870117, -2.32100510597229, 16.44767951965332, 7.069703102111816, 13.900479316711426, 7.070650100708008, 11.177596092224121, -1.7710785865783691, 5.204992771148682, 7.317486763000488, 5.756075859069824, -13.696428298950195, 3.7952353954315186, 1.1348825693130493, 1.1177642345428467, 13.930671691894531, 7.060804843902588, -4.90099573135376, -5.675692558288574, 3.926091194152832, -6.414911270141602, 0.8830053806304932, 1.3306167125701904, 6.614975929260254, -6.744219779968262, -5.434072494506836, 6.0291523933410645, -2.0992045402526855, 11.173294067382812, 3.755030393600464, -2.167965888977051, 3.929004192352295, 3.255974292755127, -11.317774772644043, 4.699289321899414, -1.830825924873352, -1.637634515762329, -4.375833511352539, 5.463571548461914, -0.51992267370224, -5.9132184982299805, 5.517516136169434, 0.21926014125347137, -0.2483569234609604, 1.6269501447677612, 6.760251522064209, 11.203595161437988, 4.591172695159912, -6.744017601013184, 3.147270917892456, 2.492647171020508, -9.034208297729492, -5.5185770988464355, -7.866447925567627, 9.15721607208252, 0.16287384927272797, 3.83838152885437, -5.642958641052246, -5.897854804992676, 10.16576099395752, 1.4839105606079102, -3.1513710021972656, 3.363532543182373, 7.8010993003845215, -11.209856986999512, 9.194581031799316, 3.9145500659942627, 12.123476028442383, 0.41596776247024536, -3.8688628673553467], [-8.865568161010742, 2.5379905700683594, -0.3486878573894501, -3.2644271850585938, -20.935420989990234, 9.182202339172363, -1.7239171266555786, -21.370166778564453, 12.038775444030762, -8.278176307678223, -1.4100241661071777, 8.922972679138184, -4.191814422607422, -11.812600135803223, -11.765506744384766, -15.249857902526855, -14.909157752990723, 5.668257713317871, -23.519149780273438, -15.374672889709473, 0.8891745209693909, -14.627527236938477, -10.408148765563965, 10.36767864227295, 12.07686710357666, -4.913393020629883, -7.783163547515869, -4.707360744476318, 17.418058395385742, -0.795551061630249, 0.6866531372070312, -1.5821338891983032, -10.494842529296875, 1.8898407220840454, 6.528102397918701, -5.154475212097168, 0.3960912823677063, -4.506322383880615, 12.669355392456055, 0.926158607006073, -8.92762279510498, 2.9415218830108643, -9.924514770507812, -9.652440071105957, -11.551803588867188, -5.868536472320557, 8.089550971984863, -16.78877067565918, 15.81491756439209, -25.997432708740234, -16.266820907592773, 3.9431324005126953, -5.074854373931885, 3.264749050140381, -11.877643585205078, 6.322396755218506, -1.8398240804672241, 1.0164215564727783, -1.4530717134475708, 6.233859062194824, 12.349590301513672, 11.53308391571045, -4.934877872467041, 12.494146347045898, -7.152553081512451, 11.686201095581055, 2.0472123622894287, -1.9242095947265625, -7.37002420425415, 0.8376327157020569, -17.893016815185547, 1.8271090984344482, 0.15428981184959412, -0.6620845794677734, 5.945228099822998, 19.265470504760742, -5.2051873207092285, -5.466881275177002, 2.805232286453247, -2.3954663276672363, -8.228889465332031, -1.9480801820755005, 4.51745080947876, 5.015774250030518, 1.5114601850509644, 14.624833106994629, 1.8033524751663208, 6.506379127502441, 19.12944984436035, -8.390823364257812, 6.433338165283203, 2.805302381515503, -15.167834281921387, -6.569916725158691, -7.364248275756836, -7.441347122192383, 14.368389129638672, -6.479454040527344, 9.077155113220215, -3.690934419631958, -5.911768436431885, -22.87694549560547, -5.512626647949219, -0.5882596373558044, -8.981277465820312, -7.8173017501831055, -0.6022428274154663, 0.24029506742954254, 0.46540650725364685, 10.597126007080078, 3.709306478500366, 0.7252885103225708, -21.535192489624023, 4.330815315246582, 24.8033504486084, 26.80851173400879, 2.0573322772979736, -15.807374000549316, 0.19435441493988037, 0.2534066140651703, -5.410856246948242, 16.996990203857422, 7.177621841430664, 1.133530616760254, 5.210339069366455, -12.259190559387207, 6.069536209106445, -21.259035110473633, 6.1277337074279785], [-0.6100267171859741, 6.928249835968018, -8.578028678894043, 1.6226484775543213, -5.440343856811523, 0.6393937468528748, -0.013763673603534698, -7.107004642486572, 7.356630802154541, -12.535028457641602, -0.7585724592208862, 1.7276763916015625, 15.379730224609375, -8.690483093261719, 11.076507568359375, 12.444087982177734, -8.520076751708984, -13.26848030090332, 10.606978416442871, -3.236150026321411, -3.490236520767212, -4.313355922698975, -2.07631516456604, -0.6599923372268677, -1.8712023496627808, 0.1093711256980896, -7.103621959686279, 9.219643592834473, -6.557031154632568, -9.571667671203613, 5.816409587860107, 10.452009201049805, 1.9903368949890137, 7.000448226928711, 0.7098616361618042, 16.94465446472168, 15.933669090270996, 1.4404348134994507, -9.964170455932617, 9.791909217834473, 5.927964687347412, 11.403831481933594, 2.2497875690460205, -4.648144245147705, -3.7188243865966797, -6.004537582397461, -3.323488235473633, 5.432178020477295, -2.9483513832092285, -12.002391815185547, 1.848406434059143, -7.748308181762695, 9.512823104858398, -5.557862758636475, 9.644251823425293, -0.8141189813613892, 3.1937572956085205, 6.984975337982178, 2.5376646518707275, 10.682793617248535, -7.502820014953613, -8.65436840057373, -1.9142451286315918, -11.200665473937988, 11.913933753967285, -8.207427978515625, 4.879234313964844, 12.888851165771484, 23.099210739135742, -5.899087429046631, -3.6488327980041504, 7.205118179321289, -0.9992474913597107, 7.431560039520264, 3.9578490257263184, 6.869029998779297, -7.675111770629883, -1.0986326932907104, 6.489080905914307, -1.470041036605835, 12.210055351257324, 6.746687889099121, -1.3455369472503662, 13.31857681274414, 3.7926173210144043, -6.838162899017334, 1.4182360172271729, 5.989527702331543, -13.080290794372559, -3.8847787380218506, -12.127264022827148, -7.120940208435059, -7.201625347137451, -3.1973910331726074, 2.3366310596466064, -0.29865801334381104, -6.8026652336120605, -3.331150531768799, 7.488195896148682, -7.491419315338135, 5.890082359313965, -8.616689682006836, -4.458123207092285, 6.691129684448242, -4.58689546585083, 16.92509651184082, -1.4364131689071655, -4.63735818862915, 1.8123141527175903, 10.344769477844238, 6.684657573699951, 3.3290815353393555, 3.7905640602111816, -4.166551113128662, 0.30976957082748413, 6.779974460601807, -12.579763412475586, -10.320759773254395, 5.6976518630981445, -1.2245124578475952, -7.899667263031006, 4.146657466888428, 2.440012216567993, -2.1640825271606445, -16.15471649169922, 2.926501750946045, 3.768070936203003, 3.563586950302124, 6.460494518280029], [-3.206287384033203, 17.038982391357422, -3.720447540283203, 7.807709217071533, 0.9245863556861877, -11.031533241271973, 1.569701075553894, -7.202540874481201, 12.474943161010742, -7.001862525939941, -10.649867057800293, 0.019374877214431763, 15.032851219177246, 1.5765540599822998, -14.023714065551758, 4.3918585777282715, -8.980416297912598, -0.692396879196167, 0.978621780872345, -4.961869716644287, 1.2054029703140259, -12.906106948852539, 0.15277695655822754, 6.112509250640869, 2.8246612548828125, 3.1357996463775635, 10.024378776550293, 6.475174903869629, -1.4545409679412842, -6.884754657745361, -1.9817959070205688, -2.4556920528411865, 1.1946957111358643, 6.907785892486572, -3.99491024017334, -5.302027702331543, 9.332337379455566, -8.747672080993652, 5.7299981117248535, 1.161475419998169, -4.831751823425293, 6.542686939239502, -12.513951301574707, 11.418703079223633, 0.6484635472297668, -4.097077369689941, 24.476423263549805, 6.80675745010376, 3.0896427631378174, -20.559988021850586, 10.527663230895996, -15.73828411102295, 6.685018539428711, -2.342869997024536, -2.817429304122925, -0.4051816165447235, -8.78443717956543, 6.794305801391602, -4.066997528076172, -5.861808776855469, -3.241788148880005, -1.5621360540390015, -1.8674222230911255, -7.643290042877197, 4.630008220672607, 4.9601216316223145, 19.929752349853516, 4.270500659942627, 14.280407905578613, 7.006233215332031, 9.015938758850098, -5.436127662658691, 17.766643524169922, -2.713860034942627, 0.19557440280914307, 12.116628646850586, 14.064122200012207, -17.51999282836914, -6.412940502166748, -7.293979167938232, 1.5638645887374878, -3.8781309127807617, 9.947318077087402, 2.4989516735076904, -9.419901847839355, 7.0628485679626465, -4.221358776092529, 15.883234024047852, -1.0416436195373535, 2.909827470779419, 3.3710849285125732, -0.49341633915901184, -12.698481559753418, 1.3500148057937622, 4.594494342803955, -12.493983268737793, -1.1213581562042236, 2.3976759910583496, 1.9622879028320312, 9.616597175598145, 2.9034833908081055, -6.783328533172607, 3.7465531826019287, -2.988217830657959, -13.156627655029297, 11.115835189819336, -5.715063095092773, -3.0298564434051514, 7.620240688323975, -16.070533752441406, 5.159315586090088, 8.73188304901123, -7.257972717285156, 0.001237306627444923, -9.276559829711914, 1.1091662645339966, -8.527053833007812, -14.575077056884766, 4.914785385131836, 6.255605697631836, 2.7530055046081543, -7.586470127105713, 2.446815013885498, -4.534485340118408, 0.7307475805282593, -17.637514114379883, 13.084338188171387, 1.2568219900131226, -0.9231999516487122], [-9.511754035949707, 7.473851680755615, 5.078402519226074, 4.385950565338135, -2.660841703414917, 0.6708714962005615, -2.0410189628601074, 13.929807662963867, -4.404516696929932, -17.33925437927246, 1.9842101335525513, -4.963129997253418, 13.914981842041016, 3.7299070358276367, 3.6089701652526855, -7.215012550354004, -0.1176418885588646, -20.18600082397461, 8.829011917114258, 8.767400741577148, -11.863064765930176, 2.121992349624634, -10.897215843200684, 8.056293487548828, 10.923962593078613, 1.0125784873962402, -13.548110008239746, -17.48996925354004, -11.486520767211914, -8.47968578338623, 11.630497932434082, 2.1935815811157227, 10.56743049621582, -0.68893963098526, -7.481385231018066, 9.143588066101074, -3.671025276184082, 20.743844985961914, -13.390249252319336, 10.606122016906738, -11.222453117370605, 14.526247024536133, 10.525668144226074, -11.371963500976562, -4.753529071807861, -14.690760612487793, -4.133893013000488, -0.8591717481613159, 0.7369101047515869, 13.591108322143555, 6.179147243499756, 11.3591947555542, 7.236735820770264, 4.364797592163086, 2.0742945671081543, -5.624149322509766, -15.797142028808594, 10.331862449645996, 5.875555992126465, 20.452960968017578, 12.081229209899902, 0.499077171087265, 12.439976692199707, -5.372199535369873, 7.5792646408081055, -19.593027114868164, 13.21578311920166, 0.3254130482673645, -1.0632286071777344, 9.445755004882812, -7.0272626876831055, 7.679787635803223, 2.8391575813293457, 3.931638717651367, -16.82522964477539, 7.952414035797119, -3.546518564224243, 7.024624824523926, -2.7004051208496094, 7.813102722167969, 0.9598488807678223, 13.009310722351074, -0.2305680364370346, 18.907678604125977, 10.751995086669922, -10.578990936279297, -5.882268905639648, -0.9942126870155334, 2.254523515701294, 3.8914241790771484, 2.0724446773529053, -3.040675163269043, 9.797175407409668, -4.573156833648682, 11.71848201751709, -7.851896286010742, -14.377537727355957, 8.084986686706543, -7.781047821044922, -0.38456350564956665, 1.710418939590454, 3.8406054973602295, -8.740307807922363, 5.301547527313232, 0.14485213160514832, -3.3769023418426514, 13.9440279006958, 2.5786166191101074, -2.08943510055542, -4.765106201171875, 8.255363464355469, -4.254402160644531, -0.32820242643356323, -0.8798526525497437, -1.888817310333252, -11.24795150756836, -12.132391929626465, -2.663172960281372, 6.718780994415283, -0.6832423210144043, 5.342142581939697, -1.3014229536056519, 5.03759765625, -2.9040334224700928, -7.938661575317383, 11.743673324584961, -2.045187473297119, -8.364990234375, -4.537261486053467], [-3.2307262420654297, 1.1850841045379639, -2.2527289390563965, 0.661456286907196, 5.451966285705566, -4.80262565612793, 5.146591663360596, -7.586308002471924, -5.775448799133301, -8.629903793334961, 0.7008863091468811, 1.6820600032806396, 6.871870994567871, -1.8810299634933472, -11.081796646118164, -1.117258906364441, -8.676957130432129, 0.6591517925262451, -4.365336894989014, 10.980934143066406, -3.3896636962890625, 4.576013088226318, 2.1036717891693115, 4.221221446990967, 3.9240293502807617, -5.934450149536133, 2.806082248687744, -7.865057945251465, -3.6482021808624268, -5.981181621551514, -12.445268630981445, 2.9389023780822754, 0.3100570738315582, 0.368914395570755, 4.782959461212158, 0.9044248461723328, 8.858097076416016, -4.415403842926025, 11.837725639343262, 4.867535591125488, -11.722312927246094, 8.838727951049805, 1.4868135452270508, -7.134646892547607, 10.875182151794434, -9.291316986083984, -2.5582776069641113, -8.123108863830566, -3.2276694774627686, -14.862568855285645, 10.443285942077637, -15.177876472473145, 2.4698922634124756, 1.1055649518966675, 5.270266532897949, 10.763738632202148, 8.859648704528809, 10.211624145507812, 8.971976280212402, -1.127667784690857, 10.313616752624512, -3.628795862197876, -1.3314273357391357, -1.686574935913086, -0.775310218334198, -12.58336067199707, 2.0707173347473145, 2.8882200717926025, 0.02594362385571003, 2.407421112060547, 7.495526313781738, 4.476954936981201, -8.06031608581543, 7.549576282501221, -0.8970092535018921, 4.173771858215332, 1.9960449934005737, 4.070568561553955, -0.7924920916557312, -2.5349748134613037, -9.103511810302734, -0.9095638394355774, -4.747498989105225, 13.7705078125, -2.8004064559936523, 12.38173770904541, 10.324673652648926, -7.849359035491943, 3.076669454574585, -6.078164577484131, 4.588509559631348, -7.8495001792907715, 7.065441608428955, 1.7428436279296875, 5.029590129852295, -9.57412052154541, -8.090839385986328, -7.0730719566345215, 15.895601272583008, 7.860054969787598, 2.3596572875976562, -15.470441818237305, -10.564672470092773, 4.5209784507751465, -2.7979605197906494, 6.74689245223999, 8.776077270507812, 16.597347259521484, 4.332042694091797, 2.598179340362549, 3.071687698364258, 8.877678871154785, -2.0391976833343506, -1.8074257373809814, -0.5099791884422302, -1.9131956100463867, -9.917769432067871, -2.824387788772583, 4.472630500793457, -6.99542236328125, 1.555375099182129, -6.8052802085876465, -0.39821213483810425, 8.985283851623535, -5.178735733032227, 3.30682110786438, 2.499864101409912, -4.213184356689453, -1.156388759613037], [-8.419102668762207, -4.343413352966309, 0.40639829635620117, 2.0517578125, -1.6180964708328247, 3.4391725063323975, -5.136502265930176, -5.190728664398193, 2.6433262825012207, 24.383344650268555, 15.025080680847168, -1.9166544675827026, -8.882781028747559, 6.350392818450928, -2.5839014053344727, 4.079524040222168, 6.028324604034424, -1.2720075845718384, -3.361654043197632, 5.677730560302734, -3.0241429805755615, -2.528594493865967, -3.7358615398406982, -0.28329798579216003, 4.566648960113525, 4.161502838134766, -2.631073474884033, 1.9690803289413452, 8.187233924865723, 8.131756782531738, 2.367927312850952, -10.771455764770508, 4.220617294311523, -3.3735477924346924, 16.57984161376953, 0.22964437305927277, -6.064400672912598, -14.562100410461426, 8.454407691955566, -7.277643203735352, -0.19189254939556122, -8.120010375976562, -4.141160011291504, 1.1757245063781738, 7.7330145835876465, 10.503111839294434, 4.994847297668457, -3.1934876441955566, -0.9651508331298828, -14.070540428161621, -1.0330559015274048, -8.176008224487305, 8.695548057556152, -0.0481475405395031, -10.970926284790039, 3.324183702468872, -6.755908489227295, -10.47914981842041, -15.311043739318848, -18.263887405395508, -9.147526741027832, -5.825767993927002, -7.161927223205566, -1.9616609811782837, -12.939430236816406, 5.788759231567383, -3.8463542461395264, -2.3295257091522217, -4.512683391571045, -8.648701667785645, -8.911616325378418, -8.798901557922363, 8.205435752868652, -4.252260208129883, 2.0201687812805176, 9.843982696533203, -14.059242248535156, -3.9592745304107666, 6.395262241363525, 6.457226276397705, -13.933391571044922, -12.561375617980957, 3.3967814445495605, 0.21027258038520813, 5.94126033782959, 17.507715225219727, 9.76058578491211, -5.099157810211182, 0.507431149482727, 9.605966567993164, 2.8949058055877686, -13.8287992477417, 1.6822316646575928, -2.615572214126587, 1.2191503047943115, 7.215415954589844, -1.1286700963974, 0.04965461045503616, 3.9838664531707764, -1.9167530536651611, -4.651159286499023, -4.320889949798584, 0.8325397968292236, 3.557110071182251, 4.369228363037109, 6.403069019317627, 4.93693733215332, -13.8434476852417, -8.760883331298828, -2.0379066467285156, -15.934905052185059, 3.7604329586029053, 0.7418277859687805, -8.254120826721191, 9.014163970947266, 10.468079566955566, 23.587373733520508, -0.2529452443122864, -9.05693531036377, 4.199079990386963, 3.591836452484131, -9.352179527282715, -1.928078055381775, -0.05952563136816025, 0.7958548665046692, 5.783457279205322, -2.4746646881103516, 8.159200668334961, -7.61251974105835], [-10.248396873474121, 3.4261505603790283, 3.56278133392334, -1.776515245437622, -2.6451375484466553, 3.79325270652771, -10.016853332519531, 0.6328758001327515, -0.3649493157863617, -24.48006820678711, -8.200928688049316, -2.9138240814208984, -4.502156734466553, -11.550437927246094, 2.0701744556427, -17.37148666381836, -3.5982654094696045, 12.625521659851074, -9.505181312561035, -15.948344230651855, 5.010552406311035, 6.0317864418029785, 3.7234768867492676, -5.132070064544678, 11.761502265930176, -12.268573760986328, 0.3583042025566101, 9.842486381530762, -15.285178184509277, -0.8225700259208679, -1.1571394205093384, -6.795958042144775, 5.272621154785156, -3.5142433643341064, 14.813640594482422, -3.541403293609619, -8.49726390838623, 1.9298616647720337, 5.235737323760986, 1.1187586784362793, -17.830093383789062, -8.751555442810059, -19.596717834472656, 3.5969433784484863, -4.699443817138672, -10.619949340820312, -8.416047096252441, -0.14462241530418396, 18.169124603271484, 5.652862071990967, 11.450701713562012, 15.013309478759766, -9.442262649536133, 7.095419883728027, 5.605881214141846, 6.349480152130127, -4.4235148429870605, 4.166100978851318, 4.934816360473633, 12.074581146240234, 4.595441818237305, -4.974299907684326, 0.44796204566955566, 19.310855865478516, 0.5408145189285278, -1.7997210025787354, 11.256183624267578, -4.103496551513672, -2.1190547943115234, 3.0582852363586426, -15.736837387084961, -0.8276031613349915, -7.31020450592041, 1.2068661451339722, -7.74282169342041, 0.43391844630241394, 0.16181409358978271, -13.61914348602295, -29.326114654541016, 0.5185302495956421, -5.292102813720703, 20.16050148010254, -6.4004411697387695, 6.098923206329346, 4.430362224578857, 3.689565896987915, -11.96530818939209, 20.310001373291016, 5.735892295837402, 6.935550689697266, 7.578830242156982, -5.754025459289551, -8.274696350097656, -7.063955307006836, 12.260896682739258, 6.386955261230469, -18.789884567260742, 6.831502914428711, -2.148585796356201, 10.820998191833496, 1.448265552520752, -6.942461013793945, 4.753873348236084, -5.447150230407715, -0.6492490768432617, 22.819482803344727, 3.7798514366149902, -1.1841329336166382, 2.99177622795105, 0.7164533138275146, -13.812310218811035, 5.109856128692627, -14.828641891479492, -0.10342247784137726, 5.262025833129883, -12.178189277648926, -11.555739402770996, -0.8459616303443909, 14.951977729797363, 1.490099310874939, -4.548527717590332, -15.4114990234375, -6.62127161026001, 2.853990077972412, 27.789939880371094, -3.3327279090881348, -2.9273934364318848, 29.960412979125977, 14.626973152160645], [-0.5346353650093079, 5.101819038391113, 2.271198034286499, 18.96316909790039, -10.940598487854004, -7.534122467041016, -6.889894962310791, -1.3113452196121216, -9.002490997314453, -12.000565528869629, 12.885235786437988, 8.465170860290527, -5.251646518707275, -11.311516761779785, -9.420790672302246, -10.027838706970215, -0.508206844329834, -1.8785704374313354, 2.2530176639556885, 0.3523620069026947, 5.843899726867676, -1.3139046430587769, -3.281059741973877, -4.955535888671875, -2.3348963260650635, -13.825810432434082, -10.614744186401367, -6.0099101066589355, -2.544625759124756, -6.502079010009766, -8.210153579711914, -0.8753998875617981, 13.146101951599121, -2.9038896560668945, 1.191827654838562, -8.506734848022461, 5.559131622314453, -0.10655160993337631, -3.481440544128418, 0.31902816891670227, -1.0700480937957764, 1.991599678993225, -5.196145534515381, -4.886641502380371, -13.240103721618652, -1.272956132888794, 5.710311412811279, 17.93524742126465, 8.60073184967041, -2.971497058868408, 9.10226821899414, -19.081212997436523, -1.5818873643875122, -3.0569639205932617, -3.6655337810516357, 11.33425521850586, 4.382492542266846, -12.899828910827637, 7.074954986572266, 15.468838691711426, 8.440730094909668, 6.720717430114746, -2.57454252243042, 4.52606201171875, 0.4378422498703003, -12.822036743164062, -2.6471691131591797, 4.9430670738220215, -11.646526336669922, 6.017650604248047, -4.386078834533691, -0.7744631171226501, -2.390200614929199, 7.430453300476074, 4.5859479904174805, 12.229629516601562, 1.2449060678482056, -2.7692954540252686, -6.204909801483154, 9.315645217895508, -1.1496527194976807, 9.01095962524414, 1.2303920984268188, 4.804720878601074, 6.288920879364014, 13.651082038879395, 2.353571891784668, 3.8842859268188477, 6.647942543029785, -3.6826181411743164, 10.65146541595459, -0.7297924757003784, 3.798030376434326, -10.218039512634277, 15.471480369567871, -10.039102554321289, -9.725274085998535, -4.238417625427246, 3.096128225326538, 9.192672729492188, 4.0081939697265625, -5.155398368835449, 0.03703447803854942, 6.627782344818115, -3.9415578842163086, 11.6875638961792, -13.472601890563965, 5.041242599487305, -0.9710961580276489, -2.4406015872955322, -8.852341651916504, 10.738719940185547, -2.990156650543213, 14.805624008178711, -8.143335342407227, 3.1370184421539307, -10.769475936889648, -4.560424327850342, 4.825723171234131, 0.21418778598308563, -1.2515164613723755, -13.793994903564453, 10.048718452453613, -10.016624450683594, -5.43002986907959, 4.016115665435791, 10.0971040725708, -7.195560455322266, -17.199140548706055], [5.4961748123168945, -17.036178588867188, -2.7549827098846436, -9.675719261169434, 2.739832878112793, -7.461272716522217, -3.2119357585906982, -7.498778343200684, -4.403602123260498, -11.449224472045898, 5.250032424926758, 11.558817863464355, -10.428112983703613, -6.019990921020508, -5.759026050567627, 0.8022426962852478, 8.415092468261719, 6.778200149536133, -14.441887855529785, -6.393085479736328, -6.6366496086120605, 1.09380042552948, 15.396265983581543, -7.168089389801025, -7.696805953979492, -15.046546936035156, -10.371100425720215, 2.4780430793762207, -0.9427958130836487, 6.44651460647583, -6.1771159172058105, 8.398388862609863, -8.455230712890625, -10.607481002807617, 6.275206089019775, -0.4244356155395508, 15.237984657287598, -7.135277271270752, -5.792534828186035, 1.4888355731964111, 5.268072605133057, 2.6752352714538574, -2.8426239490509033, -11.664637565612793, -2.219318151473999, -14.685462951660156, -3.99835467338562, -15.736091613769531, 3.5803287029266357, 1.9846651554107666, 1.2102947235107422, -2.198776960372925, -8.458647727966309, -2.401089668273926, -3.891540050506592, -3.9285433292388916, -1.8011157512664795, 12.762370109558105, 8.666582107543945, 8.921554565429688, -6.244319915771484, -9.288558959960938, -1.8246504068374634, 11.013725280761719, 5.845154762268066, -3.7045185565948486, -1.0671379566192627, 5.31682014465332, -13.07642650604248, 17.30552101135254, 5.685283660888672, 8.951451301574707, -0.29925888776779175, -0.11115729063749313, 2.4197278022766113, -3.247922897338867, 6.0584211349487305, 11.466764450073242, -5.381775856018066, 7.763515949249268, 4.8912835121154785, -1.6458736658096313, -18.00857162475586, -5.973015308380127, -6.606106758117676, -5.7037553787231445, -1.3539029359817505, -1.2530914545059204, -6.719682216644287, -19.84258460998535, -7.581180572509766, 11.695172309875488, 7.4373321533203125, 8.260270118713379, 3.405970811843872, -5.745920658111572, -3.695279359817505, -4.023853302001953, -3.930778980255127, -15.44145393371582, -5.8153815269470215, 7.217734336853027, -4.231490135192871, 0.5067894458770752, 2.540363311767578, -12.657403945922852, -4.231349468231201, -0.2317873239517212, 1.091399073600769, 0.7995848059654236, 1.102035403251648, 0.6171091198921204, 0.7900214195251465, 2.544416904449463, 0.6566230654716492, 6.791962623596191, 2.1857430934906006, 12.274797439575195, -5.01806640625, -8.453399658203125, -2.384417772293091, 2.8240628242492676, -4.205732345581055, -5.344723701477051, -3.4019970893859863, -1.2887238264083862, -9.69020938873291, -1.5321251153945923, -3.3708293437957764], [3.3828468322753906, -16.473642349243164, -0.8011242747306824, -8.297201156616211, 14.593500137329102, 6.671711444854736, -5.834847450256348, -6.1775126457214355, -6.854504585266113, 11.01379108428955, 14.445194244384766, -2.5214366912841797, -4.43510627746582, -4.413757801055908, -0.2020624428987503, 5.44706392288208, 22.494266510009766, -13.880982398986816, -0.4319285452365875, 11.401763916015625, -7.420064449310303, 6.319858074188232, -6.536044120788574, -12.983246803283691, 13.412491798400879, -5.570858001708984, -7.4042649269104, -9.474027633666992, -18.873743057250977, -5.85101318359375, 11.470617294311523, -2.0418694019317627, -7.259502410888672, -4.575074195861816, 9.984183311462402, 6.470151901245117, 1.511361837387085, -7.365346908569336, -0.6997285485267639, -5.52789831161499, 17.047767639160156, -13.866671562194824, 8.612109184265137, -10.267593383789062, 14.104564666748047, -9.892827033996582, -6.945779323577881, -6.717243194580078, -20.305824279785156, 10.098185539245605, -8.67879581451416, 6.5182204246521, 4.183877944946289, -8.27795696258545, -7.71755313873291, -8.467287063598633, 1.6877878904342651, -8.040586471557617, 1.7210310697555542, -1.3183832168579102, -11.557954788208008, -1.2541923522949219, 18.558195114135742, -3.5558207035064697, -4.676427364349365, 4.175631523132324, -8.236284255981445, 3.166017770767212, -15.62916088104248, -1.7781890630722046, -2.5887820720672607, 15.960474967956543, 5.143176078796387, 2.296416759490967, -0.9992150664329529, 0.9205555319786072, -8.916857719421387, 10.898813247680664, -5.253857612609863, -13.682662010192871, -12.195263862609863, -0.8928636312484741, -5.424095630645752, -2.7781689167022705, 0.7838941216468811, 1.5818817615509033, 11.718206405639648, -14.366806983947754, 0.6128994822502136, -20.525299072265625, 5.9537272453308105, -12.44051456451416, 2.274662494659424, 9.66224479675293, -9.916840553283691, 4.917681694030762, -18.923423767089844, 24.456506729125977, -2.250842571258545, -21.262630462646484, 10.548410415649414, -5.44558572769165, -16.401153564453125, -7.720244407653809, 15.23364543914795, -5.731537342071533, 2.583143472671509, -9.192656517028809, -15.647200584411621, -4.349274635314941, 2.009937047958374, -1.2737617492675781, 9.169391632080078, -12.962324142456055, 6.699619293212891, 8.344812393188477, 13.922163009643555, 5.108880996704102, -1.3291741609573364, -7.222690105438232, -7.32334566116333, 0.1426297128200531, -10.744786262512207, 8.38374137878418, -10.843323707580566, 8.836873054504395, -5.7127461433410645, 4.610116481781006, 6.5838303565979], [-2.602071523666382, -1.546766996383667, -11.7985200881958, -5.546640396118164, -19.27642059326172, 3.933274030685425, -4.261285781860352, -9.004318237304688, 2.938851833343506, 19.647850036621094, 2.341219186782837, -1.2913442850112915, -13.051031112670898, 9.069552421569824, -1.6565618515014648, 13.33371639251709, 8.427591323852539, 5.805418491363525, 14.029682159423828, -4.461191177368164, 5.8692626953125, 12.006394386291504, -2.2444608211517334, -6.354175090789795, -10.082929611206055, 8.805007934570312, 6.92755126953125, 3.7195546627044678, 1.9702671766281128, -3.7249932289123535, 7.9323530197143555, -5.64965295791626, 2.055846929550171, -5.587238311767578, 2.8203799724578857, -6.832719326019287, -1.5573443174362183, 0.21478988230228424, 5.600306510925293, -16.226926803588867, -12.602252006530762, 0.9566176533699036, -7.368251323699951, 1.8477298021316528, -4.522958755493164, 0.04647381603717804, 1.6240320205688477, -2.922562599182129, 3.7865872383117676, 1.9211609363555908, -4.172852993011475, 9.232281684875488, -7.548328399658203, 9.383703231811523, 8.4357271194458, -11.892680168151855, 1.1139453649520874, -7.737348556518555, 2.60933256149292, -7.356619834899902, 0.04139310121536255, -6.739750385284424, -7.084957599639893, -5.507660388946533, -3.4546139240264893, 6.408065319061279, -5.857882976531982, -6.1712846755981445, -11.181679725646973, -3.652022361755371, -5.876540660858154, -1.8626322746276855, 6.459348201751709, -3.7821414470672607, 6.465270042419434, -10.693619728088379, -6.667454719543457, 2.977658748626709, -0.6403905153274536, -10.05099105834961, 10.86013412475586, -3.9715747833251953, 8.059109687805176, -14.354963302612305, -3.5152747631073, 3.718745708465576, 0.6932312250137329, -14.536954879760742, -0.10250503569841385, -9.44255256652832, -0.7941039800643921, -1.6109997034072876, 5.245173454284668, 4.632817268371582, -4.517321586608887, 11.049080848693848, 12.754287719726562, -4.880282402038574, 3.293002128601074, -0.5438183546066284, 4.402325630187988, -18.519428253173828, -7.588914394378662, -4.22423791885376, 4.201296806335449, -1.9554857015609741, 1.146529197692871, 1.7266713380813599, -4.29095458984375, 2.0113184452056885, -7.889157772064209, 7.919722557067871, -1.3617782592773438, 0.19112642109394073, 1.4521338939666748, -7.061104774475098, 0.8882505297660828, -4.21859073638916, -4.442148685455322, 4.834825038909912, -8.01381778717041, -6.942782402038574, 10.773853302001953, 11.852200508117676, 13.727754592895508, 4.935891628265381, 2.0334692001342773, 1.246198058128357, -1.7815282344818115], [3.8245692253112793, 1.1216875314712524, -6.564224720001221, 1.845837950706482, 7.433550834655762, 9.298426628112793, -2.186797618865967, -3.275541067123413, 4.326701641082764, -3.2300288677215576, 1.887330174446106, -0.5083377361297607, 6.584170818328857, 15.907963752746582, 1.1215866804122925, 3.7146756649017334, 2.853470802307129, 2.7065653800964355, 6.196134567260742, 7.07165002822876, 9.780471801757812, 5.757287502288818, 18.772790908813477, -4.375284194946289, -8.561927795410156, 4.23547887802124, -5.222080230712891, 8.809441566467285, -4.212692737579346, 9.522896766662598, -5.652493953704834, 0.2664682865142822, -19.48647117614746, 18.355188369750977, -6.803696632385254, -6.080014705657959, 8.627681732177734, 14.202427864074707, -9.259408950805664, -4.0819501876831055, -1.9280695915222168, 2.326260805130005, -2.295138359069824, 4.690469264984131, 5.107375144958496, -1.8558825254440308, 7.8735127449035645, 8.205096244812012, 5.025174140930176, 1.6324002742767334, 6.028146266937256, 2.1592471599578857, -8.077008247375488, -0.015942521393299103, -11.111855506896973, 7.942378520965576, 9.168923377990723, -0.6452823281288147, 0.6244373917579651, 9.832293510437012, -6.6145524978637695, 2.380293130874634, 5.647185802459717, 6.595700263977051, 1.3918092250823975, 10.442609786987305, 12.988594055175781, 1.9374403953552246, 12.554445266723633, 10.060702323913574, 5.153224945068359, 12.995370864868164, -11.84869384765625, -2.3518826961517334, 10.986074447631836, -19.93842124938965, 2.842730760574341, -12.624744415283203, -4.820206642150879, -4.196678638458252, 9.661094665527344, -1.049802303314209, -2.049027919769287, -14.555753707885742, -3.2279796600341797, -11.060724258422852, -13.857438087463379, -0.935987651348114, -14.423023223876953, 2.640920639038086, 3.304988384246826, 19.003177642822266, -6.092641353607178, 10.051734924316406, -1.7508639097213745, 5.220914363861084, 7.082805156707764, 0.034413717687129974, -5.5075249671936035, -6.6210856437683105, 1.788465142250061, -8.742793083190918, -0.8609206676483154, -6.366745948791504, -5.596896171569824, -9.172855377197266, -11.226207733154297, 2.9796981811523438, -5.3180251121521, 9.029199600219727, 4.559441089630127, 10.491423606872559, 10.163269996643066, 4.505965232849121, 9.830133438110352, -3.0384719371795654, -6.122198104858398, -10.92354965209961, -3.7230422496795654, 1.1792211532592773, -5.613790512084961, 5.957233905792236, -6.058232307434082, 0.0756259337067604, 1.2969505786895752, -2.4916319847106934, -3.981205701828003, -1.1554224491119385, 12.389671325683594], [-1.4905521869659424, -12.064360618591309, -7.277585983276367, -6.7123541831970215, -4.0312604904174805, 6.32775354385376, 10.85378646850586, 5.034660339355469, 0.04377710819244385, 3.0689101219177246, 4.762655735015869, -6.084875583648682, -2.8558077812194824, 5.779440879821777, 7.161812782287598, 7.416231155395508, -7.566318511962891, 12.161845207214355, -6.664536476135254, 2.699540853500366, 3.4231903553009033, 5.134980201721191, 11.646085739135742, -7.8046417236328125, -0.6617962718009949, -3.801908254623413, 15.002288818359375, 6.8581013679504395, 7.058192253112793, 0.5332722663879395, -8.980382919311523, 0.1885921210050583, -16.647506713867188, 6.774893760681152, 15.093868255615234, 1.4192359447479248, 0.6493138074874878, -1.5272401571273804, -1.2721260786056519, -2.6576459407806396, 6.250714302062988, -12.134422302246094, -16.597820281982422, -4.242583751678467, 4.881009101867676, 10.12425422668457, -13.909905433654785, 8.236440658569336, -1.681661605834961, 2.694681406021118, 0.5210046768188477, -7.448646545410156, -0.08607953786849976, -9.50667953491211, 4.148495197296143, -2.2810819149017334, 11.237286567687988, 6.578771591186523, -8.186126708984375, -1.8629533052444458, -1.2875248193740845, -15.364123344421387, -1.0040178298950195, 5.074182987213135, 6.474761486053467, 3.4683358669281006, -6.814462661743164, -5.955077171325684, 1.4360867738723755, 0.8842298984527588, -2.1651952266693115, -1.4075143337249756, -12.296761512756348, 6.181988716125488, 11.113738059997559, -9.146514892578125, -8.30764389038086, -3.661487579345703, 0.5949870944023132, 9.252586364746094, 10.021683692932129, 2.160381317138672, -6.305022239685059, -0.33289316296577454, 7.316826820373535, 0.17579519748687744, -4.745673179626465, -10.006670951843262, -16.56344223022461, -10.172028541564941, -12.731070518493652, -4.056197643280029, 2.3228759765625, 5.53506326675415, 5.607100486755371, 5.3700079917907715, 2.4305129051208496, -10.325553894042969, 5.0810065269470215, 3.863551616668701, -2.0205001831054688, 2.931061029434204, 4.273547172546387, 12.87395191192627, 3.791604995727539, 5.472968101501465, 1.418701410293579, 6.9097900390625, -5.832252502441406, 19.789886474609375, -16.086408615112305, 2.4254679679870605, 7.7478861808776855, -9.739227294921875, -3.8081135749816895, 2.3779115676879883, 5.163082599639893, 2.1255297660827637, -3.185804605484009, -3.69608211517334, -3.544579267501831, 3.1157076358795166, 1.5027872323989868, 1.7180413007736206, 1.0366729497909546, 20.924612045288086, -1.41661536693573, 3.5228519439697266, -6.991115093231201], [-3.086467742919922, 3.3456497192382812, -25.39902114868164, -2.2663230895996094, 9.000346183776855, 12.283787727355957, -5.285923004150391, -1.708902359008789, 3.994863986968994, -6.429933547973633, 5.511261463165283, -3.347193956375122, -2.2730019092559814, 2.0054914951324463, -3.083651065826416, -7.123311519622803, 18.37514877319336, -5.733986854553223, 7.019664287567139, 10.763509750366211, 0.3328365683555603, -1.1836799383163452, -5.769386291503906, -4.747386932373047, 19.47243309020996, -1.0610463619232178, -10.785900115966797, -15.722932815551758, 11.422378540039062, 0.0035067859571427107, 12.455814361572266, -0.6326492428779602, 0.36300790309906006, -12.008184432983398, 13.265069007873535, -8.406437873840332, 2.2395408153533936, -4.293854236602783, -10.297557830810547, -6.267141819000244, -0.5116424560546875, 2.0198731422424316, 4.610174655914307, 1.7276039123535156, 0.11971080303192139, 12.95318603515625, -1.7183572053909302, -6.304498672485352, 12.907862663269043, 9.159565925598145, 0.6984632611274719, 22.06062126159668, -3.0904457569122314, 2.6004397869110107, 8.47800064086914, -1.0834038257598877, -9.34088134765625, -10.60804271697998, -0.8084400296211243, -6.357683181762695, 11.769449234008789, 3.840994358062744, 15.9210844039917, 8.804228782653809, 10.109352111816406, 1.7344287633895874, 8.110450744628906, -3.7135844230651855, -0.9694057106971741, 9.049419403076172, -1.5431733131408691, -16.6402645111084, 11.845980644226074, -2.329728841781616, -1.2340389490127563, 3.423887014389038, 3.4324958324432373, -16.30479621887207, -9.058624267578125, 23.0422420501709, -5.880779266357422, -1.0796501636505127, 2.96830415725708, 9.98979663848877, -7.6591715812683105, 1.961240291595459, -6.14602518081665, -19.830387115478516, 12.410995483398438, -3.8641412258148193, -0.3022522032260895, -5.060789108276367, 4.099320411682129, 12.143875122070312, -9.60936450958252, 6.424758434295654, -11.714390754699707, 0.3100278675556183, -5.177856922149658, -9.741183280944824, -7.726263999938965, -17.82636070251465, -5.940274238586426, -4.012390613555908, 4.251625061035156, -9.738025665283203, -5.510503768920898, 19.66319465637207, 0.6686894297599792, -3.677947998046875, 14.70720100402832, -11.535293579101562, -15.422924995422363, -4.431362152099609, -1.7124053239822388, 3.9998114109039307, 7.582558631896973, -2.4736039638519287, -1.0895782709121704, -2.4389894008636475, 6.406068801879883, -0.8183006644248962, 17.769399642944336, -15.551722526550293, -9.298962593078613, 14.481000900268555, -10.859774589538574, -7.541172981262207, -5.165104389190674], [-2.1827337741851807, 2.792957067489624, -8.69215202331543, -3.0468029975891113, -0.7606879472732544, -1.3612456321716309, 11.553840637207031, 3.447252035140991, -5.126704692840576, 20.870145797729492, -11.741642951965332, 2.0872251987457275, -3.1954445838928223, 5.686117172241211, 5.957827568054199, -5.205339431762695, 1.9641244411468506, 0.045409541577100754, 1.050624132156372, 5.8555521965026855, -7.056212425231934, 5.707997798919678, -0.9614420533180237, -4.0063629150390625, -0.2232312262058258, -0.37994185090065, -14.793536186218262, -5.106510639190674, 13.465903282165527, -3.6578316688537598, 7.640097141265869, -3.242988348007202, -9.399901390075684, -13.47567367553711, -3.854726552963257, 0.23356464505195618, 0.9140759706497192, 9.933038711547852, -18.09929656982422, 8.313077926635742, 11.628130912780762, -1.5937254428863525, -0.07129427790641785, 1.323517918586731, -5.97378396987915, 1.8441288471221924, 3.3759539127349854, 0.7017583250999451, -0.48378556966781616, 12.950233459472656, 3.2522826194763184, 13.919486999511719, -6.233668327331543, 13.866656303405762, 5.943620681762695, -13.539440155029297, -9.409896850585938, 6.877310752868652, -1.338343620300293, -8.435415267944336, -0.6051864624023438, 3.7648186683654785, -4.861814498901367, 0.184538796544075, 1.649308681488037, -1.513444423675537, -3.4596948623657227, 2.9287405014038086, -11.668173789978027, 14.122627258300781, 10.885331153869629, -1.8545362949371338, 4.779817581176758, 3.0875205993652344, -5.586002349853516, 1.4405572414398193, 2.1118149757385254, 0.5870391130447388, -0.9657895565032959, 16.345054626464844, 3.995612859725952, 13.870772361755371, 4.671936988830566, -15.338804244995117, -13.39291000366211, 3.027229070663452, -1.250801682472229, -5.320576190948486, 5.846096992492676, -2.159832000732422, -5.534059524536133, 10.060136795043945, 7.5002217292785645, 10.690664291381836, 0.25001081824302673, -2.91619610786438, -15.561649322509766, 5.489957809448242, -5.465494155883789, 4.474356651306152, -12.40511417388916, 7.614388942718506, -2.797342538833618, 15.773863792419434, 7.48629093170166, -4.314167499542236, -5.161323070526123, 6.520168781280518, 6.678694248199463, -6.630016803741455, 6.068774223327637, -13.3028564453125, 7.456299781799316, -12.71187973022461, 0.07365430891513824, -5.8281941413879395, 1.1435097455978394, 9.579781532287598, -9.23403549194336, -0.12139002978801727, -1.1423325538635254, 14.674206733703613, -3.6490790843963623, 0.5997934937477112, 1.1215755939483643, -1.9518715143203735, -8.66947078704834, -3.9899184703826904, -4.9324727058410645], [0.9467967748641968, -9.119356155395508, -9.434929847717285, 0.02273457869887352, -2.8753674030303955, -7.391363143920898, 6.052943706512451, 4.737865924835205, -8.763321876525879, 15.889589309692383, -5.324293613433838, 1.9000539779663086, -9.536622047424316, -8.015090942382812, -0.26220422983169556, -3.7436904907226562, -3.652862787246704, 7.330850601196289, 6.4930195808410645, -1.1499805450439453, -17.313756942749023, -12.495758056640625, -1.2291107177734375, -3.612334728240967, 1.9247336387634277, 4.535881996154785, 0.9554130434989929, -3.439784526824951, 5.639046669006348, -0.4561448097229004, -5.447878360748291, -1.7930227518081665, 2.404658079147339, -0.3143581449985504, 2.6043901443481445, 9.355010032653809, -4.936983585357666, -3.7579843997955322, 2.355895757675171, 9.248517990112305, 4.680151462554932, -7.680467128753662, 5.825088977813721, -6.838771820068359, 8.457846641540527, -7.6211371421813965, -2.6899797916412354, -15.111542701721191, -5.520604610443115, -14.509441375732422, 3.8333961963653564, -1.3288832902908325, 7.286777496337891, 20.221506118774414, -8.621432304382324, -10.404589653015137, 4.818545818328857, 5.611898422241211, -9.32706356048584, -23.645580291748047, -0.5528687238693237, 7.3387579917907715, 7.553772449493408, -5.8580708503723145, 2.6244940757751465, -0.8615579605102539, 10.247987747192383, 0.3723413944244385, -12.081033706665039, 11.129340171813965, -1.7173796892166138, 12.522074699401855, 5.005395889282227, 7.829533100128174, -12.779363632202148, -5.609656810760498, -0.4623427093029022, -1.9402804374694824, 13.095175743103027, -3.0323617458343506, 9.643527030944824, -0.16404984891414642, 2.578639030456543, -8.66265869140625, -13.161072731018066, 4.338973045349121, 4.577551364898682, 4.698863506317139, 3.7206313610076904, 0.3112616539001465, -16.892940521240234, 0.007224258966743946, -0.05551586672663689, 6.1449875831604, -10.337484359741211, -6.638834476470947, 1.3196978569030762, 5.35530424118042, 7.763216495513916, -2.9448370933532715, -4.8038763999938965, 7.273367881774902, -11.761034965515137, 3.825253486633301, 10.44375991821289, 1.4610567092895508, 14.124256134033203, 1.2516539096832275, 4.384119510650635, -9.30718994140625, 10.87649917602539, -4.646059989929199, 7.22053337097168, -2.1055893898010254, 8.594286918640137, -1.3152320384979248, -0.06676040589809418, 15.08400821685791, -8.657663345336914, -14.963088035583496, 5.173598766326904, 8.481335639953613, -10.4371919631958, 5.1543378829956055, -6.054837226867676, -1.6040544509887695, -2.7258386611938477, -5.693239212036133, -3.2519352436065674], [-8.487874984741211, -6.981845855712891, -4.321373462677002, 2.441420793533325, 7.813905715942383, 13.531944274902344, -6.700195789337158, 3.0627293586730957, -5.1895647048950195, 0.16658064723014832, -1.1718839406967163, 7.536083698272705, -4.964585781097412, -17.29555320739746, 7.01566743850708, 12.75458812713623, -3.8129918575286865, -13.07654857635498, 8.428629875183105, 4.144959449768066, 2.5459694862365723, -5.818962574005127, -6.319825172424316, -3.6109790802001953, 8.64923095703125, 12.73134708404541, -8.579667091369629, -10.843737602233887, 2.263566732406616, 1.1014835834503174, 0.5886626839637756, 1.1163618564605713, 5.1758646965026855, 9.93128490447998, -10.635235786437988, 18.58941650390625, 2.345519542694092, -8.158974647521973, 2.3973026275634766, -7.256186485290527, 13.385499954223633, -3.4167897701263428, 3.846795082092285, 10.515876770019531, 8.141555786132812, 16.483739852905273, 1.1357311010360718, -18.73719024658203, 8.195758819580078, -3.416699171066284, -6.815434455871582, 8.230188369750977, 7.6306023597717285, -11.46879768371582, -5.68992805480957, -4.553163051605225, 0.3953465521335602, -7.642535209655762, -11.056164741516113, -5.777432441711426, -2.7423412799835205, 2.035944700241089, 3.281259059906006, -6.229257583618164, 7.685521125793457, -4.093804359436035, 13.487010955810547, 5.605424880981445, 4.4689812660217285, 6.5757670402526855, -2.6527457237243652, 3.2425696849823, 14.188189506530762, 6.336419582366943, -1.6585698127746582, 0.186909019947052, -3.996493339538574, -9.766681671142578, 4.017978191375732, 5.131671905517578, 7.759183406829834, -7.3731794357299805, -15.025649070739746, 0.9172801375389099, -5.843082904815674, 4.512461185455322, 4.084226608276367, -0.7612272500991821, -1.6879467964172363, 0.5959331393241882, -4.230635643005371, 2.318967819213867, -12.521846771240234, 9.030594825744629, -10.976042747497559, 7.674464702606201, -5.59691858291626, 16.929323196411133, -6.0095133781433105, -7.896622180938721, 7.746013641357422, 6.403633117675781, -2.962707757949829, -21.53544807434082, 4.1794514656066895, 0.3239514231681824, 17.7100887298584, -5.673496246337891, 1.3406481742858887, 0.5525819659233093, 8.969815254211426, 1.723593831062317, -0.3966008424758911, 5.351032733917236, 15.423627853393555, -0.25287318229675293, -3.656938076019287, 11.925621032714844, -17.115768432617188, 2.5331594944000244, 7.6011061668396, -2.9977684020996094, 0.4336216151714325, -5.74415397644043, -9.533113479614258, -12.138082504272461, 0.5967824459075928, -1.5738393068313599, -4.474969387054443], [-1.5868109464645386, -0.4342036247253418, 8.180644035339355, 0.3479195237159729, -3.694854259490967, 4.426145076751709, -1.3211792707443237, -9.70869255065918, 7.124431610107422, -12.607392311096191, -1.9282349348068237, 18.750131607055664, -12.677249908447266, -4.665441989898682, -6.901246547698975, 8.643574714660645, -6.966902732849121, -5.48591423034668, -17.234514236450195, 3.123385190963745, 17.260831832885742, -9.352034568786621, 23.47989845275879, -0.5231687426567078, -16.694616317749023, -5.335386753082275, -8.468870162963867, -3.9970595836639404, -4.127397537231445, -3.5425658226013184, -16.078140258789062, 5.898068428039551, -9.692289352416992, 1.9059597253799438, -7.8108015060424805, 19.02643585205078, 8.76899528503418, 3.9795284271240234, 14.736751556396484, 2.611887216567993, -6.179678916931152, 9.374348640441895, -1.3967374563217163, 6.0633158683776855, -13.17498779296875, 9.560835838317871, 10.805720329284668, 5.489538192749023, -0.6028463244438171, -10.537107467651367, 1.5694807767868042, -13.897377014160156, -8.924537658691406, -11.488555908203125, -3.747527837753296, 8.143165588378906, 2.712228775024414, -17.81498908996582, 9.17258071899414, 14.71026611328125, 3.3665802478790283, 2.9276416301727295, -4.515059471130371, -4.6041388511657715, -3.7432525157928467, 7.033143997192383, -26.764331817626953, -3.1747937202453613, -0.3222777843475342, 12.785154342651367, 8.333741188049316, 4.51154899597168, -6.750492095947266, -8.391217231750488, 15.858697891235352, 3.359828233718872, 0.6080164313316345, 4.831217288970947, 6.880153656005859, -1.3957439661026, 11.020551681518555, 7.448625087738037, 10.019213676452637, -8.531180381774902, -5.952364444732666, -0.20556451380252838, 5.640403747558594, 3.058983564376831, 2.459810733795166, -10.46540355682373, 1.4289422035217285, 7.443624496459961, -3.9569506645202637, 7.205334663391113, 17.63237190246582, -2.4318032264709473, 0.25202682614326477, 3.2573254108428955, 15.756996154785156, 3.498303174972534, -3.5120038986206055, -4.824721813201904, -1.0056076049804688, 9.931597709655762, 4.2498602867126465, -1.7907602787017822, 1.794265866279602, -6.855894565582275, 4.38440465927124, 4.552639007568359, -18.792034149169922, -1.7358481884002686, 13.920633316040039, 13.84272575378418, 17.274700164794922, -2.8031976222991943, 1.4844346046447754, -2.262356758117676, 2.993041515350342, 8.77562141418457, -13.31856918334961, -1.0979679822921753, -3.0124473571777344, 5.482575416564941, 3.9980008602142334, -2.7038862705230713, -7.0640645027160645, -12.240957260131836, -25.31320571899414], [-6.484209060668945, 0.11346292495727539, 0.8672743439674377, -8.76052188873291, 2.737518072128296, 6.298590660095215, -3.1554505825042725, 1.5104807615280151, -0.8256386518478394, -20.821758270263672, -4.280898094177246, -6.21658182144165, -12.608628273010254, -3.8782036304473877, 11.77615737915039, -6.7668023109436035, 5.202617168426514, -2.9992918968200684, 1.5161714553833008, -7.810765266418457, -8.398700714111328, 3.6252784729003906, -0.9999420046806335, 7.816997051239014, 20.15532875061035, 9.66174030303955, -5.800919532775879, 0.17628714442253113, 10.61848258972168, 6.749200344085693, -12.353021621704102, 3.3457109928131104, -5.598479270935059, 1.9525964260101318, 8.927597045898438, -1.1348867416381836, -8.5919771194458, 14.90475082397461, -21.228605270385742, 0.4325902462005615, -1.5910134315490723, 6.953365325927734, -0.46128273010253906, 0.4846249520778656, 4.730936050415039, 14.765052795410156, -4.832457542419434, -7.190558433532715, 11.796225547790527, 6.603227615356445, -4.3294148445129395, -3.6666905879974365, 13.331582069396973, -1.8912732601165771, -0.025391658768057823, 11.286304473876953, -8.249083518981934, 13.431178092956543, 3.7439167499542236, 13.198369979858398, -7.422691345214844, 2.0728399753570557, -8.603078842163086, 17.665807723999023, 12.575778007507324, 1.6064597368240356, 5.012166500091553, -0.5288878083229065, 5.882411479949951, 5.76766300201416, -16.064308166503906, -0.44340622425079346, -10.947043418884277, 7.957219123840332, -11.872395515441895, 9.466686248779297, 21.697900772094727, -8.523651123046875, -3.1192259788513184, 9.336021423339844, -9.343400001525879, 8.74990177154541, -5.849467754364014, 7.938643455505371, 1.0866754055023193, -11.019805908203125, -2.4284541606903076, -12.525728225708008, -1.5225580930709839, 3.87566876411438, -6.585400581359863, -3.357785701751709, -5.320560455322266, 7.56063985824585, 4.702271938323975, 9.112543106079102, -14.109453201293945, -11.238408088684082, 2.134005546569824, -3.1723809242248535, 11.369019508361816, 2.845287799835205, 11.574182510375977, 11.590753555297852, -0.996819257736206, -2.0317015647888184, -2.9064383506774902, 10.668575286865234, -10.391219139099121, 14.8853759765625, -2.379847526550293, 7.689949989318848, -11.995723724365234, 0.32109880447387695, 4.469738006591797, 5.400094509124756, 1.8521884679794312, 8.624397277832031, 12.476451873779297, 4.797818183898926, 2.609876871109009, 5.7461676597595215, 5.921994686126709, -4.050016403198242, 2.2661080360412598, 2.537442684173584, -4.68043327331543, -6.7631707191467285, 2.8032379150390625], [-12.715288162231445, 2.8539915084838867, 1.4713479280471802, 1.4255948066711426, -5.701602935791016, -7.493374347686768, -4.113745212554932, 12.844808578491211, -7.062066078186035, -3.1473498344421387, 2.2940776348114014, 12.80486011505127, 8.889486312866211, -2.4272878170013428, 2.958041191101074, -0.2969140410423279, 0.9601892232894897, 13.140145301818848, -3.5122838020324707, -9.89413833618164, -3.0130937099456787, 3.6648173332214355, 2.33282208442688, 3.230189800262451, 4.365194320678711, -5.956733703613281, -0.13956135511398315, -2.467395067214966, 1.2346818447113037, 0.4720875918865204, -1.3889120817184448, 4.145007133483887, 17.62566566467285, -7.925620079040527, -1.7436174154281616, -1.811672568321228, -5.893784999847412, 6.422398090362549, -10.07977294921875, 7.884422302246094, -10.277414321899414, 13.804826736450195, 1.0847289562225342, 0.15562008321285248, -6.7182183265686035, 0.786273717880249, 2.154919385910034, -12.773123741149902, 0.40147218108177185, 9.231694221496582, 2.01786470413208, -0.5775371193885803, 8.780731201171875, 18.544095993041992, 2.2008132934570312, 7.790809631347656, -13.94037914276123, 14.691852569580078, -1.2358934879302979, -6.143336296081543, -5.631381988525391, 3.9307236671447754, -10.583303451538086, 0.567142128944397, -0.09436360001564026, -10.229914665222168, 3.1322319507598877, 2.571204662322998, -9.923287391662598, -1.5965044498443604, 6.085771083831787, -2.735325813293457, 16.056188583374023, 9.250080108642578, -1.4946949481964111, -2.6829545497894287, 11.306452751159668, 7.180909156799316, -1.9607256650924683, 8.724907875061035, 1.866131067276001, -2.8926055431365967, -3.7561275959014893, -12.486510276794434, 5.491652488708496, 1.1645008325576782, -0.13378222286701202, 11.91834545135498, -6.559847831726074, 9.117244720458984, 1.3061338663101196, 4.672873497009277, 13.163847923278809, -1.5785061120986938, 1.9693094491958618, -9.727313995361328, 15.691367149353027, -4.207494735717773, -7.553942680358887, -0.4106766879558563, -4.274127960205078, 17.277280807495117, -2.0065596103668213, 7.146020889282227, -9.123748779296875, -5.59689474105835, 14.133566856384277, 1.9350073337554932, 1.0990493297576904, 2.908689498901367, 15.395336151123047, 3.135019540786743, -11.481547355651855, -2.433391809463501, -8.796420097351074, -0.6166777610778809, -4.888460159301758, -0.7776622176170349, -8.712309837341309, 2.681288242340088, 12.480949401855469, -3.731759786605835, -5.383396625518799, -4.5225090980529785, 11.303162574768066, -2.8683300018310547, 1.3290225267410278, 2.4492435455322266, -3.577824115753174], [-0.8095866441726685, -16.339372634887695, -6.955868244171143, 4.049904823303223, 13.981252670288086, -10.954229354858398, 2.9271023273468018, 3.370516538619995, -2.716641664505005, -1.9377813339233398, -2.412384033203125, -4.81762170791626, 3.870941638946533, 3.654754638671875, -5.808981418609619, -1.973443627357483, -0.8746911883354187, -3.3918020725250244, 15.567681312561035, 5.829921722412109, -1.6091234683990479, -4.043015480041504, -13.56222915649414, 2.8467788696289062, 11.017850875854492, 0.48886173963546753, 6.373997211456299, -8.919134140014648, 2.2566943168640137, -1.674148678779602, -5.098269462585449, -0.9502927660942078, 9.388474464416504, -2.6940553188323975, 0.2927004396915436, -8.129695892333984, 4.785794734954834, 4.290033340454102, -4.299953937530518, -0.7391654849052429, -9.84970474243164, -11.889625549316406, -0.07781869918107986, -1.4583935737609863, -2.4322962760925293, 0.3380827307701111, 6.64677619934082, -3.626007318496704, 7.345922470092773, -5.549732685089111, 17.189722061157227, -3.598072052001953, -0.010854867286980152, -1.3473985195159912, 1.201200008392334, -3.1121246814727783, -0.6286804676055908, 1.1771397590637207, -5.409420013427734, -16.00775146484375, -4.280915260314941, -7.02204704284668, 4.30005407333374, -2.9888739585876465, 7.186758518218994, -14.221481323242188, 20.5828914642334, -4.526328086853027, -2.845447540283203, -2.1338183879852295, 9.826440811157227, -10.207701683044434, 15.382139205932617, 2.2748162746429443, -4.8159098625183105, -5.0023932456970215, 0.1821947544813156, -23.367589950561523, -7.541181564331055, 18.375076293945312, 3.0838818550109863, 0.8688753843307495, -5.818150520324707, 6.892613410949707, -5.525477886199951, 3.634122610092163, -0.8106651306152344, 0.6582233309745789, -5.57469367980957, 7.286784648895264, -8.44719409942627, 4.192387104034424, 5.45628023147583, 3.2564828395843506, 1.242713451385498, -6.298617839813232, -5.780613899230957, 7.364530563354492, -1.6589795351028442, 13.592535972595215, -6.382389545440674, 11.449566841125488, -1.1738581657409668, 5.57975435256958, -3.5225112438201904, 7.355316638946533, 0.7097687721252441, 10.1757230758667, 5.109355449676514, -9.27218246459961, 6.067251205444336, -4.511791229248047, -3.430598020553589, -11.775707244873047, -12.257349967956543, -11.009706497192383, 2.3887321949005127, 3.892895221710205, -0.8766398429870605, -12.97216796875, 12.080504417419434, -4.166066646575928, 1.1630909442901611, -13.451623916625977, -0.34619250893592834, 11.553696632385254, 10.298300743103027, -2.1064529418945312, -12.08043098449707], [6.7637858390808105, -12.76297378540039, -2.2713499069213867, -7.9746599197387695, -3.059532880783081, 0.30706092715263367, -7.763003349304199, 10.166814804077148, -0.1185450330376625, 5.2977399826049805, -0.26895666122436523, 7.360944747924805, -1.0100334882736206, -3.970104932785034, 8.650321960449219, -1.6142401695251465, 7.400120258331299, 10.352888107299805, 13.299156188964844, -10.710650444030762, -6.733678817749023, -0.49343329668045044, 8.626629829406738, 5.961990833282471, -4.7825093269348145, -10.858697891235352, -7.64648962020874, 14.911554336547852, 5.156567573547363, 9.37147331237793, 2.5325357913970947, 14.043506622314453, 9.577860832214355, 3.3535029888153076, 10.399892807006836, -1.2681407928466797, 5.520989894866943, 7.683619976043701, -18.96692657470703, -6.697425842285156, 13.750848770141602, 3.172851085662842, -12.538410186767578, 3.3155195713043213, 11.182619094848633, -10.59058666229248, 1.0755494832992554, -3.8490211963653564, -6.087273120880127, -1.058914065361023, 1.1309788227081299, -5.537734031677246, 7.572882652282715, 6.528690814971924, -1.0752899646759033, -1.3758021593093872, -2.3754305839538574, -0.6799297332763672, -5.225590229034424, -19.781490325927734, 3.006596565246582, -4.400865077972412, -13.177356719970703, -5.32371187210083, -7.539143085479736, 0.4038223922252655, 4.62148904800415, -7.837937355041504, 12.753121376037598, -12.29906940460205, 5.5255231857299805, -4.822686672210693, -6.178447246551514, 12.490715980529785, -6.406589984893799, -10.895979881286621, -5.3064799308776855, -6.130260944366455, 7.951696872711182, 9.431360244750977, 0.728782057762146, -5.280102729797363, -3.673431634902954, -9.127758979797363, -12.144869804382324, 4.063131809234619, -5.174593925476074, 2.157238721847534, -7.871090412139893, 14.486029624938965, -9.990397453308105, 4.868916034698486, 2.0748953819274902, -0.1772940307855606, 7.996782302856445, 10.558083534240723, -5.620042324066162, -3.648603916168213, -1.4120056629180908, -7.406963348388672, -7.438347339630127, -7.091282844543457, 10.147456169128418, 12.076001167297363, 2.824897289276123, 4.895082473754883, -13.735150337219238, -9.510961532592773, -3.1793830394744873, -5.809557914733887, -1.6098774671554565, 0.8935458660125732, 0.19607725739479065, 8.43192195892334, -7.777095317840576, -2.3464314937591553, 19.162817001342773, 13.49366283416748, -6.471951484680176, -8.9338960647583, 0.5302947163581848, 8.671404838562012, -2.0686228275299072, -15.342562675476074, -15.575751304626465, 5.237045764923096, -1.299972653388977, 3.8553924560546875, -2.3725545406341553], [-2.63787841796875, -1.7481892108917236, 11.581731796264648, 14.513778686523438, -4.83105993270874, -6.351847171783447, 11.012277603149414, -3.107325315475464, 10.117135047912598, 2.355072021484375, -1.669105052947998, -1.8512165546417236, 0.5555152893066406, 3.016597270965576, -1.1166114807128906, 5.739214897155762, -7.887409687042236, 10.360965728759766, -0.49901658296585083, -10.329268455505371, -13.902166366577148, -1.0337144136428833, 8.9199857711792, 4.630425930023193, -15.279263496398926, 2.3392770290374756, 10.040382385253906, 20.520254135131836, -0.3171016275882721, 4.1900763511657715, -0.11968117952346802, -2.8433005809783936, 4.293776512145996, 15.670098304748535, 5.87839412689209, 3.574831008911133, 12.625528335571289, 2.8353142738342285, 5.168533802032471, 5.422262668609619, -7.372927188873291, 3.5109236240386963, -10.272150993347168, -15.450055122375488, -5.548580169677734, -7.725221157073975, -2.1063215732574463, 4.834400653839111, -3.0126700401306152, -4.699161052703857, -8.78369426727295, -10.76594352722168, 8.421456336975098, 9.45264720916748, -1.1921964883804321, -7.730524063110352, -3.8656747341156006, 7.083577632904053, -0.30182331800460815, 2.861708164215088, 6.494936943054199, 10.548766136169434, -2.3603341579437256, 4.954043865203857, 0.7977373003959656, 13.555505752563477, 1.115002155303955, 11.312300682067871, -3.3091678619384766, 12.105865478515625, -5.837861061096191, -1.2103058099746704, 4.995232105255127, 6.770262241363525, 2.2453620433807373, -0.2954311668872833, 10.913640975952148, 11.364551544189453, 6.823885440826416, -5.130051136016846, -0.9836052656173706, -0.20017141103744507, 1.615614891052246, -6.274109840393066, 9.304774284362793, -12.506608963012695, 0.899063229560852, 8.71243953704834, -2.8129055500030518, 0.254429429769516, 3.7245633602142334, -4.31528902053833, 0.0015409047482535243, 5.873020648956299, 16.926252365112305, 5.1113080978393555, 3.504883050918579, -4.0115275382995605, 9.573519706726074, 2.33480167388916, 9.486397743225098, -0.9336230754852295, 9.017518997192383, 14.19091796875, 15.025554656982422, 12.49715805053711, -3.981013536453247, -10.508661270141602, -16.61121368408203, 20.849470138549805, -2.041782855987549, 6.397275924682617, 1.7940516471862793, -0.4461776912212372, 3.2451558113098145, -9.957740783691406, -4.319328784942627, -0.5968596339225769, 0.5859977602958679, 2.8083651065826416, 1.1529343128204346, 2.461998224258423, 1.0033684968948364, 1.4055885076522827, 5.096073150634766, 8.419543266296387, 14.494800567626953, 10.204158782958984, -2.60901141166687], [-0.9015036225318909, 7.1790080070495605, -11.112855911254883, -4.559463977813721, -10.407448768615723, -1.5933234691619873, -3.0199644565582275, 1.3968368768692017, 5.4851975440979, -4.583856582641602, 0.6168332099914551, 15.756758689880371, 7.254539966583252, -6.359231948852539, -5.944999694824219, 11.100357055664062, -11.168855667114258, 9.543916702270508, 1.7576605081558228, -6.088751316070557, 6.460552215576172, -8.48410701751709, -2.6356658935546875, -1.7850487232208252, 0.49415597319602966, -6.206841945648193, -8.991927146911621, -3.4851183891296387, 3.108957052230835, 1.110800862312317, 6.737593650817871, 4.909907817840576, 12.93532657623291, -9.919804573059082, -0.8381345272064209, -4.214519500732422, 11.157157897949219, -6.5430169105529785, -1.9189423322677612, 9.949881553649902, 2.0819828510284424, 10.765131950378418, -8.200748443603516, 11.395222663879395, -7.835887432098389, 3.0394539833068848, 11.827579498291016, -5.066782474517822, 5.976689338684082, 0.19148816168308258, 4.860530853271484, 0.5303351879119873, -0.38198408484458923, 8.550995826721191, 11.261950492858887, 0.1866418868303299, -9.457292556762695, -3.0501418113708496, -4.938357830047607, 0.2719421982765198, 10.561712265014648, -0.2968471646308899, -8.479181289672852, -6.100027561187744, 3.9688661098480225, -0.7444822192192078, 20.124767303466797, -7.255405426025391, -7.8582987785339355, 8.726617813110352, 9.663392066955566, 3.9950737953186035, 5.406789302825928, 5.645409107208252, 5.257585048675537, 6.678682804107666, 2.8265178203582764, -3.3238525390625, -10.961512565612793, 9.28740406036377, 7.571738243103027, 15.863106727600098, 3.3869469165802, 3.9651174545288086, 4.320705890655518, -1.9015837907791138, 5.6422014236450195, 10.559521675109863, -9.058758735656738, -6.516448974609375, -10.511537551879883, 12.192870140075684, 5.790483474731445, -4.9329681396484375, 1.5086913108825684, -13.48891544342041, 0.5657123923301697, 5.171866416931152, -13.356612205505371, -4.94616174697876, 3.0805633068084717, 0.9291154742240906, -15.463601112365723, 5.885767459869385, -5.130541801452637, -8.265751838684082, 3.726318597793579, -0.7946338057518005, 17.384784698486328, -12.56337833404541, 2.791011333465576, -1.5497182607650757, -5.906360626220703, -0.26478832960128784, -4.990100860595703, -6.918573379516602, -2.584933042526245, -4.281052112579346, -2.260211944580078, -2.1409404277801514, -3.000462532043457, -6.019366264343262, -0.03444207087159157, -11.387857437133789, -12.719820022583008, -4.052334308624268, 8.763343811035156, -2.3173439502716064, -8.047369956970215], [1.0470141172409058, 18.455270767211914, -4.178015232086182, 3.5253138542175293, -19.498504638671875, 4.068421840667725, -5.140857696533203, -4.009243011474609, 2.8053934574127197, 1.663736343383789, -14.136798858642578, -13.487637519836426, -23.019058227539062, 4.517780303955078, -15.265569686889648, 12.643244743347168, -7.383098125457764, 4.938670635223389, 8.727137565612793, 5.470178604125977, 4.602156162261963, -10.618009567260742, 12.502659797668457, -4.44740104675293, -14.62778377532959, -2.159285545349121, 10.259841918945312, 1.9809398651123047, 4.773776054382324, -2.6303629875183105, 8.65544605255127, -10.10737133026123, -4.834166526794434, -3.2030086517333984, 13.493202209472656, 9.18720531463623, -3.974492311477661, -0.4700632393360138, 20.486003875732422, -3.8249049186706543, -16.80270004272461, 7.851644039154053, -16.153127670288086, 14.044671058654785, -2.0473408699035645, 7.673823833465576, 16.40288543701172, 0.7360123991966248, 1.997085690498352, 0.23258699476718903, -9.6649808883667, -3.5636978149414062, -16.89626121520996, 13.575875282287598, -8.382691383361816, -15.547874450683594, -0.5432525277137756, -16.456443786621094, 1.147148609161377, -6.746969223022461, 11.858747482299805, 7.103457927703857, -0.5322229266166687, -10.222585678100586, -3.5831565856933594, 4.5704522132873535, -4.546680450439453, -5.677789688110352, -4.768394947052002, 14.329370498657227, -4.967939376831055, -3.1160120964050293, 6.869683742523193, -2.674160957336426, -2.7533140182495117, -5.174610137939453, 3.0744528770446777, 3.536616563796997, 10.312169075012207, 13.75126838684082, 6.336980819702148, -5.420817852020264, 20.383020401000977, -8.91469669342041, -8.339990615844727, -0.8500573635101318, 2.6925930976867676, -23.559829711914062, 0.582237720489502, 3.0610954761505127, -13.090702056884766, 2.955441951751709, 3.7341525554656982, 4.44298791885376, 21.177087783813477, 28.03730010986328, 13.564016342163086, -16.174531936645508, 12.359809875488281, 4.880561351776123, 1.461449146270752, -11.34992504119873, 0.08283045142889023, 1.8411099910736084, 9.074715614318848, 0.8861333727836609, -13.790972709655762, 1.2719509601593018, -0.19144007563591003, 5.151333808898926, -15.447300910949707, -7.893139839172363, -2.5180225372314453, -20.728044509887695, 8.223786354064941, -3.5553786754608154, 10.384976387023926, -2.8210291862487793, 13.787997245788574, -5.615490913391113, -15.55956745147705, -10.050329208374023, 15.442811965942383, -7.783123970031738, 0.8952315449714661, -0.25726333260536194, -8.880492210388184, 14.683998107910156, -6.245511531829834], [-12.677663803100586, 1.724987506866455, -6.657865524291992, -7.440523147583008, -9.609189987182617, 7.461182594299316, 0.4598808288574219, 0.4889126420021057, 0.286967933177948, 7.039445400238037, 7.2551398277282715, 0.30468684434890747, 3.7277934551239014, 2.8091259002685547, -3.3315324783325195, 3.7495779991149902, 5.563311576843262, -3.3359627723693848, -3.0346546173095703, 5.554378986358643, 14.713042259216309, 7.426514148712158, -8.608585357666016, -2.203829050064087, 10.261669158935547, -11.561165809631348, -10.27525806427002, 2.107469081878662, 9.86133098602295, 0.8946525454521179, 5.070786952972412, -14.260876655578613, -1.840470790863037, -6.083600044250488, 0.7777686715126038, 13.982551574707031, 14.261906623840332, 10.27504825592041, 7.895711421966553, -18.00720977783203, 10.788431167602539, -11.830041885375977, -13.345609664916992, -1.949987530708313, 17.30838394165039, -14.45479965209961, 17.957998275756836, 10.486854553222656, 2.263389825820923, 14.528985023498535, -2.8892128467559814, 10.4524507522583, -3.3343052864074707, -0.9548677802085876, -0.5031221508979797, -1.9710040092468262, -15.090612411499023, -2.045625686645508, -0.9457004070281982, -0.7900780439376831, -0.44774746894836426, 6.664198875427246, -0.8962849378585815, 1.3103564977645874, -8.811534881591797, 18.652965545654297, 6.75351619720459, 11.832412719726562, -3.7114615440368652, 1.257124900817871, -1.7669062614440918, 9.520231246948242, 0.5732632279396057, 7.111513137817383, 7.09451150894165, 2.9909000396728516, 4.334254264831543, -10.258164405822754, -0.41164806485176086, -11.9420166015625, -3.6504769325256348, 2.81679630279541, 3.7492260932922363, -4.996091365814209, -4.2398858070373535, 16.8604736328125, -6.529197692871094, 6.005990982055664, 1.200260043144226, -7.9086456298828125, 11.412324905395508, -4.944880962371826, -4.186314105987549, 21.931764602661133, -1.9764716625213623, 2.7452807426452637, -0.5033450126647949, 1.2875670194625854, -4.408505439758301, -1.1425189971923828, -10.378157615661621, -8.978589057922363, -3.615471839904785, -9.217132568359375, -3.337953805923462, -2.0123722553253174, 0.8211711049079895, -17.608137130737305, -10.302168846130371, 4.9497480392456055, 7.593868255615234, -5.421517848968506, -7.727458953857422, -0.8808013200759888, 13.72342300415039, 11.921804428100586, 13.914648056030273, 1.5572049617767334, -14.44793701171875, 0.04107183963060379, 0.7963188886642456, -1.5552871227264404, 11.600812911987305, -6.400257587432861, 6.586709976196289, -1.861413598060608, 5.295105457305908, 6.686988353729248, 6.800136566162109], [-3.6308422088623047, -16.332611083984375, -0.9770179390907288, -1.140594244003296, -0.7085369229316711, -11.34950065612793, 1.3591450452804565, -9.189785957336426, -11.92856502532959, 5.076207160949707, 4.332995891571045, 8.304752349853516, 1.8105016946792603, -5.574012756347656, -7.702552795410156, -0.48445209860801697, 0.8947111368179321, -1.7102279663085938, -9.298287391662598, -3.029024362564087, -4.637125015258789, -10.584224700927734, -21.16619110107422, 12.783670425415039, 7.938199520111084, -14.128682136535645, -15.602336883544922, -9.73228645324707, 3.0374300479888916, -1.2978941202163696, 19.871801376342773, -2.422708749771118, 3.298666000366211, -7.048094749450684, 13.19665813446045, 0.3308810889720917, 6.7862629890441895, -7.708177089691162, 6.686691761016846, 0.7844064831733704, 6.696056842803955, -2.4374911785125732, 5.854312896728516, -3.952414035797119, 6.242362976074219, -0.6801119446754456, 2.417100667953491, 1.0617594718933105, -0.6040500998497009, -11.491714477539062, 3.3372204303741455, -3.7535033226013184, 3.6626079082489014, -4.573003768920898, -11.60444450378418, -0.4930822551250458, 3.776118516921997, 0.7786400318145752, 14.633747100830078, 6.958632469177246, 1.908980131149292, -6.365270614624023, 4.065034866333008, -8.531205177307129, -9.952201843261719, 0.27568677067756653, -0.6455310583114624, 14.986513137817383, -6.246711254119873, 3.2421436309814453, -2.259047031402588, 15.862775802612305, 0.8354358077049255, 8.780076026916504, 2.1410698890686035, 14.205035209655762, -10.940077781677246, 8.843401908874512, -3.9168827533721924, 0.8464589715003967, -8.570030212402344, 8.743821144104004, 9.171178817749023, -17.575458526611328, -3.8507299423217773, 21.355571746826172, 4.841888427734375, -7.209932327270508, 7.805334568023682, -14.703577995300293, -0.9799744486808777, -0.31646785140037537, -0.3437187075614929, -2.6230292320251465, -16.74034881591797, 3.29292368888855, 6.358509540557861, 6.600225448608398, 1.465873122215271, 0.9052098393440247, -0.6672486662864685, -20.03108787536621, -15.208635330200195, -0.4038862884044647, 1.1402256488800049, -14.158906936645508, -13.796577453613281, 3.104978084564209, 2.4946162700653076, -11.869114875793457, 14.916295051574707, 6.522556781768799, -6.066681861877441, -1.1833001375198364, 5.591107368469238, 11.0550537109375, 10.216313362121582, -12.997483253479004, -3.690885543823242, -4.4496965408325195, -6.59214448928833, 1.9837040901184082, -1.5315290689468384, 2.695809841156006, -11.751334190368652, -13.011127471923828, 5.780970096588135, -7.358804225921631, 2.9987077713012695], [-9.82779598236084, -9.35850715637207, 10.135896682739258, -7.666903972625732, -1.9443484544754028, -7.60796594619751, -14.976303100585938, -9.290578842163086, -10.287928581237793, -6.937182426452637, 3.6563925743103027, 7.739045143127441, -15.290205001831055, 1.1581038236618042, 1.672113299369812, -1.7086564302444458, 24.55025291442871, -2.3201189041137695, -3.028315544128418, -18.287029266357422, 4.1712541580200195, 6.762796878814697, -1.6297993659973145, 8.051007270812988, 9.748273849487305, -11.222643852233887, 1.7372899055480957, -1.1943731307983398, 7.397096157073975, 7.530148029327393, 1.2921768426895142, 14.593979835510254, 3.374215602874756, 12.716754913330078, 9.062166213989258, -6.631474494934082, -10.36882209777832, 8.996179580688477, -8.252968788146973, -7.808696746826172, 15.090041160583496, 5.240249156951904, 1.8703522682189941, -1.4140651226043701, -3.4797112941741943, 0.7953252792358398, 0.5967792272567749, 0.9888959527015686, 13.172422409057617, 16.70812225341797, -6.0620927810668945, 27.260042190551758, -1.1233044862747192, 22.840986251831055, -12.15582275390625, 8.814775466918945, -3.111001968383789, -6.16286039352417, 10.2232084274292, -1.3882222175598145, 7.918601989746094, 12.993264198303223, -7.9063801765441895, 9.528584480285645, -13.794416427612305, 5.567966938018799, -0.9309367537498474, 7.847039222717285, 3.4533941745758057, 8.2543306350708, 3.2175421714782715, 8.523153305053711, -7.396027565002441, 4.973934650421143, -4.930291175842285, 2.187251329421997, 3.302966833114624, 3.5124826431274414, -4.52239465713501, 2.7314834594726562, 3.5692269802093506, 7.798824310302734, 7.312955379486084, -15.5137300491333, -2.532923460006714, 0.3791801929473877, -10.642884254455566, 2.0180981159210205, 8.849165916442871, 6.777004241943359, -7.904794692993164, 6.537117958068848, 3.8292346000671387, -8.763813972473145, 4.232335090637207, 9.851706504821777, -12.180950164794922, 5.097318172454834, 0.40003225207328796, -0.12638325989246368, 7.83466100692749, 7.549285411834717, 3.059711456298828, -3.316312789916992, -4.6944499015808105, -18.903257369995117, -27.105701446533203, -7.297563552856445, -3.9876039028167725, -1.4236356019973755, 0.48752081394195557, -12.04129695892334, 2.3710107803344727, 6.94556188583374, 4.154200077056885, -7.951927185058594, 7.78567361831665, -8.485396385192871, -2.342914581298828, 8.196771621704102, 0.15081706643104553, -11.742027282714844, 0.9939714074134827, -10.41453742980957, 7.842081069946289, -4.6175456047058105, -2.515441656112671, 4.194716453552246, -6.140310287475586], [1.4104801416397095, 8.204130172729492, 9.058881759643555, -16.619384765625, 4.134732723236084, 0.5559893250465393, -1.9655940532684326, -7.976398468017578, 2.2431275844573975, -7.085397720336914, -1.307328462600708, -1.303627371788025, 8.705784797668457, -7.194059371948242, -13.999876976013184, 5.3885369300842285, -6.053036689758301, -15.707440376281738, -1.889258623123169, -4.447688102722168, -9.707913398742676, -14.19864273071289, 6.563072681427002, 7.983561992645264, 12.515321731567383, 1.5407530069351196, 11.236388206481934, 7.94363260269165, -7.118574619293213, -6.009860515594482, -0.5907467007637024, 7.994263648986816, 8.956814765930176, 0.9435274004936218, 5.690575122833252, 12.411445617675781, 8.385710716247559, -3.473834753036499, 7.946235179901123, -1.2512216567993164, -11.188087463378906, 4.024219036102295, 4.199326992034912, -6.225769519805908, 5.569260120391846, -6.649148464202881, -0.05199776217341423, -1.7850768566131592, -3.1207592487335205, -9.0610933303833, -8.31143569946289, -7.290333271026611, 4.517867565155029, -6.178284168243408, 7.841963291168213, 2.0119922161102295, 9.614830017089844, -4.301813125610352, 9.581740379333496, -1.0147885084152222, 14.074840545654297, 2.6700994968414307, -4.129124641418457, 4.5982346534729, -7.9645280838012695, -3.4425978660583496, 5.0555267333984375, -5.434467792510986, 11.570920944213867, 5.3122124671936035, 0.8671420812606812, -4.789466381072998, 1.807722568511963, 4.575787544250488, 0.766507625579834, 5.957045555114746, -0.9977317452430725, 8.93371868133545, 4.3998894691467285, 2.009622097015381, -2.9539525508880615, 3.495577096939087, -12.009419441223145, 14.704716682434082, 9.112611770629883, -1.322725534439087, 11.236950874328613, -5.23058557510376, 4.9421281814575195, -1.8511735200881958, 1.717343807220459, -6.765592575073242, -2.633319854736328, -3.7539620399475098, -0.2644004821777344, 10.570805549621582, -3.598410129547119, 2.3151915073394775, -4.237244129180908, -4.921792984008789, 8.247246742248535, -19.461421966552734, -0.3905879855155945, 10.00246524810791, 13.45531940460205, 13.35493278503418, -0.2620665431022644, -8.002991676330566, 3.303790330886841, 9.841670036315918, -5.617307186126709, -1.5472474098205566, -6.582053184509277, -13.921967506408691, 4.470517635345459, -1.8756744861602783, 7.43030309677124, 0.8879982829093933, 4.210179328918457, 2.2492902278900146, 1.692196011543274, -13.652080535888672, 7.479325771331787, 3.2270307540893555, -9.845623016357422, 3.2524876594543457, -4.308403491973877, -1.0669993162155151, 0.715600848197937], [-4.777228355407715, -9.834981918334961, -3.5323877334594727, -6.597258567810059, -1.2197484970092773, 1.1060408353805542, 10.549247741699219, -2.0448265075683594, 4.525294780731201, -8.302826881408691, -3.331488609313965, 3.6797420978546143, -4.79335880279541, 1.240617275238037, 6.669559001922607, -1.4625904560089111, -2.6026816368103027, -5.272723197937012, 5.354191780090332, -0.6509239673614502, 14.480587005615234, 2.521834135055542, -2.7574596405029297, -7.780261039733887, 10.720670700073242, 9.064653396606445, 1.867577314376831, -10.644862174987793, 6.159668445587158, -9.189319610595703, -3.512049674987793, 1.4636884927749634, -7.736027717590332, 6.212285041809082, 3.546119451522827, 7.081352233886719, 5.644794464111328, 10.323287963867188, -0.155755415558815, -6.110876560211182, 10.763361930847168, -5.58982515335083, -9.265521049499512, 9.320636749267578, 2.747039318084717, 13.834796905517578, 4.121006488800049, -1.2416571378707886, 2.3337228298187256, -4.006828308105469, 8.294163703918457, 14.84816837310791, -2.8130197525024414, -7.638149261474609, 9.241337776184082, 7.7628936767578125, -5.39430570602417, -10.325072288513184, -5.079967021942139, -2.261181354522705, -2.954190254211426, -1.6692423820495605, -16.9841251373291, 0.7105415463447571, -3.4327611923217773, 3.7470428943634033, -11.06450080871582, -4.405308723449707, 14.379453659057617, -0.7265684604644775, 11.910360336303711, -2.435701847076416, -8.82675552368164, -0.8202628493309021, 16.224096298217773, 0.7395172119140625, 2.5631747245788574, -18.009159088134766, -1.599437952041626, -3.124236583709717, 14.384612083435059, 5.4699296951293945, 15.487078666687012, 8.030632019042969, -4.32705020904541, 1.2718498706817627, -4.959033012390137, 1.4550411701202393, -1.1003748178482056, 2.887810707092285, 3.028149366378784, 0.6678733825683594, -3.8475701808929443, 8.606165885925293, 2.7974259853363037, 7.322403907775879, -13.39336109161377, 2.8574819564819336, 9.231318473815918, 10.242079734802246, 4.150454521179199, 2.258910655975342, 1.0587477684020996, 1.4977538585662842, -10.783321380615234, 12.963020324707031, -10.232218742370605, 4.7380900382995605, -7.883078098297119, 4.274498462677002, -0.967404305934906, -0.16151967644691467, 8.996895790100098, 7.440306186676025, 5.27527379989624, -1.1131540536880493, 2.7277379035949707, -4.451619625091553, 7.193918228149414, 8.494439125061035, -1.1989611387252808, 0.5779720544815063, -1.9633928537368774, 5.448129653930664, 8.604141235351562, -1.8513847589492798, 0.657072126865387, 1.2485032081604004, -4.066642761230469], [8.132217407226562, -4.90805721282959, -0.39323675632476807, -10.228951454162598, 9.778526306152344, 0.4149695038795471, 7.709693431854248, 6.3059258460998535, -6.261267185211182, -7.697959899902344, -2.296356201171875, 0.869448184967041, 7.078053951263428, 0.7087125182151794, -5.785336971282959, -12.077911376953125, -1.8922863006591797, -3.736725330352783, -11.020933151245117, -2.8110337257385254, 6.959341526031494, 3.4695017337799072, 4.218612194061279, -6.991074085235596, 0.5589568614959717, -15.492888450622559, -5.7281718254089355, -3.3853938579559326, -1.002633810043335, -1.8095707893371582, -9.243992805480957, 3.939204692840576, -7.886389255523682, -5.663474082946777, 3.537766456604004, -6.4066619873046875, 11.3759183883667, 0.3377990126609802, -4.403412342071533, -0.8576482534408569, -1.6058950424194336, -3.3783938884735107, -0.4933576285839081, 1.2238348722457886, 3.0264484882354736, 0.07533928006887436, -1.945641040802002, 6.5310564041137695, 0.3128035366535187, -4.210763454437256, 17.95171546936035, -7.153997421264648, -8.429329872131348, -0.9289262890815735, 1.9627660512924194, 23.751821517944336, 6.675368309020996, 1.022535800933838, 1.9370787143707275, -5.431101322174072, 0.23760168254375458, 2.3311758041381836, -8.16856861114502, 12.625801086425781, -3.1642208099365234, 1.3754801750183105, -2.3275675773620605, -0.21750129759311676, 13.791926383972168, 6.763227939605713, 8.995567321777344, 1.347122073173523, -19.72408103942871, 4.017765045166016, -1.4388608932495117, -1.6880815029144287, 0.5299215316772461, -4.263846397399902, -2.5217185020446777, 4.478408336639404, 0.02620278298854828, 11.525768280029297, 1.8140920400619507, 17.695905685424805, -5.488073348999023, 3.8389480113983154, -4.095922470092773, -6.2244791984558105, 3.737243175506592, 3.3198368549346924, -0.07047713547945023, 11.419182777404785, -4.231725692749023, 0.17228464782238007, 4.396413803100586, 0.6265473365783691, -4.263490676879883, -10.94215202331543, 6.852120399475098, 3.2209911346435547, -6.416244029998779, -7.853728294372559, 6.300599575042725, 1.0401815176010132, -6.9099860191345215, 3.8577771186828613, -13.813201904296875, 5.810173988342285, -1.078851580619812, -2.3877859115600586, -10.821240425109863, -2.9951131343841553, 5.331548690795898, 6.032654762268066, 4.92911434173584, -5.518692493438721, 4.150268077850342, 10.094782829284668, 12.723421096801758, 3.994762897491455, -6.5619425773620605, 6.813303470611572, -3.796611785888672, 3.8748061656951904, 1.3355352878570557, -1.9484429359436035, -12.948019027709961, 6.6635942459106445, -0.9503147602081299], [3.4747345447540283, 16.4316349029541, -3.944972038269043, -6.8415679931640625, -17.22060203552246, -1.0059164762496948, 6.113486289978027, -6.683532238006592, 0.6925272941589355, 14.55835247039795, -4.727059364318848, -1.1711750030517578, -6.037479400634766, 10.65422248840332, -0.9529120922088623, 12.724658012390137, 6.089132785797119, -4.934475421905518, 10.762590408325195, 6.660473346710205, 8.076024055480957, -4.3349609375, -2.262594699859619, -1.687975525856018, 7.470648765563965, 7.140500545501709, 20.622772216796875, -9.8389310836792, -6.224347114562988, -15.937341690063477, -1.0492668151855469, 3.767350435256958, -9.422659873962402, -15.104109764099121, 10.095065116882324, 4.177857875823975, -9.662029266357422, -0.9731051921844482, 6.273620128631592, -12.92438793182373, -1.0617519617080688, 6.343565464019775, 10.827527046203613, 1.0145890712738037, 2.400845527648926, 9.692246437072754, 14.335735321044922, 5.026446342468262, 0.394432008266449, 5.962734222412109, 0.3410429358482361, -2.0786194801330566, -6.651595592498779, -1.9127593040466309, 5.44298791885376, 0.5303774476051331, -7.074094772338867, 8.00814437866211, -2.0290770530700684, 4.441628456115723, -5.531071186065674, 4.320530414581299, -9.053566932678223, -3.2618541717529297, 0.285931259393692, 12.073848724365234, -9.369550704956055, -8.155258178710938, 2.1789517402648926, -3.941133737564087, -17.46542739868164, -1.8051413297653198, 1.4982150793075562, -1.3915430307388306, -4.0204176902771, -0.02767726592719555, 19.151552200317383, 9.864398956298828, -3.7088816165924072, -13.640715599060059, -6.995264530181885, 6.453257083892822, 8.024877548217773, -1.048115611076355, 9.375128746032715, 5.112076282501221, -3.277982234954834, -11.630497932434082, 14.261777877807617, 2.415114402770996, 6.6077961921691895, -5.6432013511657715, -1.1420797109603882, -1.6101489067077637, 7.630229949951172, 3.255336284637451, -0.004256108775734901, 0.4282650947570801, 1.5385655164718628, -0.4086754024028778, 10.470779418945312, -4.880998134613037, 6.123998641967773, 9.229479789733887, 2.049797534942627, 1.9221690893173218, -13.94260311126709, -4.088505744934082, -7.438517093658447, -1.4531928300857544, -13.30465030670166, -3.9666857719421387, -9.792591094970703, -14.379161834716797, 3.1750447750091553, 6.063948631286621, 2.5411009788513184, -6.729820251464844, 6.591118335723877, 9.527676582336426, -2.2989959716796875, -7.60805082321167, 8.055707931518555, 11.409089088439941, 3.735440731048584, 0.32638493180274963, -5.255690574645996, -11.963631629943848, -4.598384380340576], [-4.30197811126709, -9.196968078613281, 7.673153877258301, 9.44805908203125, 9.627828598022461, -6.433152198791504, 14.022979736328125, 2.587272882461548, -0.23610563576221466, -5.895321846008301, -1.7813106775283813, -4.908384323120117, 7.665680408477783, 16.383413314819336, 2.5684571266174316, 0.4059196412563324, -4.587933540344238, -2.0816409587860107, 6.272705078125, 4.707314968109131, -1.5778182744979858, 11.603657722473145, 6.160749912261963, -8.199421882629395, -6.416744709014893, -1.1550365686416626, 0.9939001202583313, 0.22993184626102448, -14.404644966125488, -0.4414560794830322, -11.656814575195312, 2.593972682952881, -5.635465145111084, 12.496334075927734, -0.5992229580879211, -2.9241323471069336, 3.679623603820801, 2.179403305053711, -2.326345205307007, -8.136191368103027, -11.679969787597656, 5.594071865081787, 6.361824035644531, -10.430815696716309, -11.339722633361816, 4.214892387390137, 2.6496102809906006, 5.922011375427246, -10.929842948913574, 2.0869364738464355, -5.8823957443237305, -7.302372455596924, -0.9686048626899719, -4.769372463226318, 2.9994304180145264, 1.2815910577774048, 5.345635414123535, 2.9133002758026123, 4.508083343505859, -6.7137603759765625, 1.3771470785140991, -4.264188289642334, 6.234285354614258, -6.697115898132324, 8.357759475708008, 2.670046806335449, -8.189828872680664, 5.935991287231445, 19.876577377319336, 4.993679523468018, -3.2101259231567383, 11.776535987854004, 1.6669007539749146, -2.0188865661621094, -2.5026330947875977, -2.5645816326141357, -0.37935736775398254, 0.6670588254928589, 11.820566177368164, -3.229806661605835, 11.90121841430664, -6.811119556427002, -2.600649356842041, 8.518487930297852, 0.2521980404853821, 2.940419912338257, -4.743775367736816, -6.862149238586426, -12.846634864807129, -0.5005546808242798, 5.152945518493652, -5.650078296661377, 0.7478430271148682, -2.245025634765625, -2.628901481628418, -3.4188778400421143, 3.208932876586914, 3.078202724456787, 3.404979705810547, 3.8233749866485596, -1.4541761875152588, -3.8464033603668213, -9.733073234558105, 0.6911987066268921, 3.463193655014038, 15.096878051757812, -5.382384300231934, 2.009394407272339, -13.930463790893555, 7.72783088684082, 2.491488456726074, 10.641520500183105, 1.4558353424072266, 2.093902826309204, -2.2309751510620117, 6.277854919433594, -3.056217908859253, -0.871239960193634, -7.724508285522461, 8.301207542419434, 4.997430801391602, -3.854391098022461, 8.86119270324707, 5.124361038208008, 6.120271682739258, 8.930120468139648, 4.359464168548584, 2.306246042251587, -7.8852009773254395], [10.859789848327637, -6.309556007385254, -1.8386670351028442, 6.451749324798584, 4.894865036010742, -3.2108523845672607, 1.4587759971618652, -10.805469512939453, 2.6886491775512695, -2.5860800743103027, 7.858004093170166, 3.910292625427246, -18.77280044555664, -8.739400863647461, 3.1741676330566406, -2.244364023208618, 1.0450282096862793, 2.487257719039917, -10.122010231018066, -6.581370830535889, 1.0058026313781738, -0.5749668478965759, 7.257749080657959, -19.83490562438965, 0.1474706381559372, -3.675475835800171, -4.549768924713135, 17.33219337463379, -5.904565334320068, 9.689349174499512, -10.738852500915527, 8.440873146057129, -0.7898222208023071, 4.1339335441589355, -10.859554290771484, 5.0260138511657715, 1.0259120464324951, -3.911041498184204, 4.15928316116333, -8.607458114624023, 11.222392082214355, -18.902660369873047, -2.5632641315460205, -10.444002151489258, -6.400835037231445, -2.30672550201416, -3.370565414428711, 5.802079677581787, 2.8463621139526367, 1.4997724294662476, 2.6381797790527344, -3.3268165588378906, -4.980553150177002, -9.702890396118164, 2.1143031120300293, 6.3439483642578125, 5.576126575469971, -11.50775146484375, 1.7385966777801514, 11.104488372802734, -2.2126314640045166, 7.38108491897583, 4.325514316558838, 9.09284782409668, -4.7314453125, 11.277668952941895, -2.612314224243164, 9.26388168334961, -19.883420944213867, -4.546701908111572, -2.8737640380859375, 7.792179584503174, -2.7867400646209717, -5.67032527923584, 6.089334487915039, -8.780704498291016, 5.568999767303467, 1.7822036743164062, -2.441744565963745, -16.755918502807617, -10.382351875305176, -5.454517841339111, -11.105962753295898, -2.3260841369628906, -1.6087599992752075, 0.6189656853675842, 7.141062259674072, 8.032623291015625, 11.330183982849121, -16.310007095336914, 0.9740092158317566, -1.7857609987258911, 1.6452313661575317, 4.272971153259277, -4.072261810302734, 3.820169448852539, 8.735185623168945, 1.8359997272491455, 8.774394989013672, -8.339471817016602, -5.686543941497803, -2.562647581100464, -5.783346176147461, -1.4434870481491089, 7.258930206298828, 2.8564226627349854, 0.33963173627853394, -6.761837482452393, -5.50679349899292, 10.934249877929688, -1.7240378856658936, 2.6946463584899902, 1.2658004760742188, 19.49289894104004, 9.546272277832031, 7.538152694702148, 2.1177151203155518, 12.419569969177246, 0.5770701766014099, -5.669276237487793, -15.049138069152832, -6.815764427185059, -13.295991897583008, 6.344679832458496, -3.6368446350097656, 9.925414085388184, -1.6785626411437988, 5.125111103057861, -3.781511068344116], [-3.7914011478424072, -4.153215408325195, -9.675183296203613, 3.855807065963745, 1.8470234870910645, 8.433710098266602, -3.1617519855499268, -0.13752365112304688, -7.81273078918457, -15.408259391784668, 0.6380284428596497, 19.17378044128418, -1.4546470642089844, 3.911193370819092, 4.142597675323486, -7.133031368255615, 4.4959893226623535, -4.324544906616211, -5.270867347717285, 1.9257014989852905, -3.455059289932251, 3.4859461784362793, -10.422361373901367, -0.002488392172381282, -2.138625383377075, -3.1175081729888916, -24.84550666809082, -6.74790096282959, 1.5363308191299438, 14.579323768615723, -10.328018188476562, -0.3957049548625946, 10.488170623779297, -4.57753324508667, -4.834685802459717, 8.59196949005127, 9.11332893371582, -3.850637435913086, -3.775893449783325, 6.653559684753418, -1.4994184970855713, 2.283191442489624, 6.544529914855957, 4.758419990539551, -4.2326436042785645, 0.09971705079078674, 3.1900768280029297, 4.597177028656006, 7.391583442687988, -4.475046157836914, -5.346467971801758, -12.008477210998535, 11.720507621765137, -7.095456600189209, -1.2637478113174438, 7.124509811401367, 2.065448760986328, 0.050877008587121964, -5.398990631103516, 9.019736289978027, -3.236607551574707, -16.21045684814453, -8.54342269897461, 6.280765533447266, 11.614699363708496, 1.2851860523223877, -4.48075008392334, 1.0488836765289307, -1.6040583848953247, 1.6785110235214233, 8.492937088012695, -1.1376689672470093, 0.8584437966346741, -4.796876907348633, 4.833055019378662, -1.3167041540145874, 8.010297775268555, -4.6276421546936035, -0.6369747519493103, 1.6856948137283325, 2.4090123176574707, -2.3420939445495605, -6.424820899963379, -7.885218620300293, -2.168982982635498, 4.921088218688965, -12.155217170715332, 7.347600936889648, -4.827127456665039, -4.751491546630859, 3.331167221069336, 5.600966930389404, 1.2925958633422852, -7.19834566116333, -8.608757019042969, -9.16364860534668, -6.60174560546875, -4.23452615737915, -6.678835391998291, -8.399356842041016, -12.865321159362793, -4.609864711761475, 7.112467288970947, -3.6561012268066406, -6.660751819610596, -8.044933319091797, 2.111973762512207, 6.394606590270996, 11.379979133605957, 2.781423568725586, 16.333524703979492, 10.056674003601074, 3.585517168045044, 8.31796646118164, 3.2279551029205322, 2.402629852294922, -3.4323623180389404, 2.8847389221191406, -7.575207710266113, -4.851593017578125, 5.1954264640808105, -9.01037883758545, 12.397750854492188, -4.2337822914123535, 8.929129600524902, -16.762317657470703, -18.20612907409668, -4.025716781616211, 3.4964675903320312], [-2.8731319904327393, -1.7715353965759277, 2.1914844512939453, -4.292675495147705, 15.13867473602295, 16.808300018310547, 2.3242390155792236, 5.277191638946533, -14.688913345336914, -6.383114337921143, -8.95785140991211, -0.2804390490055084, -3.3408255577087402, 17.417726516723633, 10.974444389343262, 3.8468313217163086, -7.509629726409912, -0.6512570977210999, -2.9725325107574463, -6.120382308959961, 3.3097479343414307, 2.6668872833251953, 2.026384115219116, -2.040355920791626, 2.917282819747925, 9.31114673614502, -6.772790431976318, -3.251211166381836, -3.9619433879852295, 0.8498148322105408, -10.26122760772705, -11.908785820007324, 13.975811004638672, 1.431395173072815, 15.408867835998535, -1.6327015161514282, -3.5828609466552734, -2.5710599422454834, 6.719597816467285, 5.126274585723877, -21.979595184326172, 0.8989659547805786, -0.8315911889076233, 4.394484519958496, 4.077726364135742, 14.403468132019043, 1.5609980821609497, -2.5519566535949707, 6.233180046081543, -1.209186315536499, 10.448711395263672, -3.479360580444336, -0.06535159051418304, -13.025093078613281, 11.245877265930176, -6.051797389984131, 7.0031657218933105, -4.712718963623047, -6.078139305114746, -4.027274131774902, 0.39198461174964905, 4.476323127746582, -1.1213587522506714, -4.701290130615234, 4.837128639221191, 1.0472588539123535, -1.7533454895019531, -2.9337291717529297, 0.1841777116060257, -2.3690178394317627, -2.4515299797058105, -18.0157413482666, 6.973003387451172, -0.41884517669677734, -6.427756309509277, 14.586324691772461, -4.6593017578125, -13.311595916748047, 7.459234714508057, 2.110508680343628, 3.51424241065979, 3.5942494869232178, 2.6930136680603027, 1.613319754600525, 7.955824851989746, 3.8200361728668213, 8.797608375549316, -16.816438674926758, -10.681062698364258, 5.977205753326416, -11.87342357635498, -4.155190467834473, -0.1334969401359558, -8.24640941619873, 2.000013828277588, 8.180730819702148, 0.7334639430046082, -11.867780685424805, 1.0347626209259033, 9.683215141296387, -0.2746424376964569, -2.017160177230835, -2.9251396656036377, 2.3356270790100098, 2.8020944595336914, 2.592095136642456, -1.7536380290985107, 3.724466323852539, -1.0445481538772583, 8.535473823547363, -15.423860549926758, 4.755940914154053, -7.199310302734375, -7.225423336029053, -7.6730055809021, 0.8397372364997864, 8.668213844299316, 6.708297252655029, 8.148581504821777, 10.136491775512695, -3.369663953781128, -13.674432754516602, 7.632777690887451, 0.19248399138450623, 7.7979960441589355, 2.135052442550659, -1.368061900138855, 3.453798532485962, -6.536901950836182], [2.3055882453918457, -17.393564224243164, 0.19589997828006744, -7.437225818634033, 3.774475336074829, 0.2777770459651947, -4.978277683258057, 3.1772446632385254, 15.075384140014648, -3.315100908279419, 0.7660567760467529, 9.25489616394043, -10.418596267700195, -1.8220608234405518, 6.444944858551025, 5.546555995941162, 4.219050884246826, 11.245452880859375, 11.505888938903809, -0.9890787601470947, 2.8361072540283203, 6.875192642211914, 6.217467784881592, -5.8778557777404785, -0.507188618183136, -5.729333877563477, -8.393209457397461, 10.213960647583008, 10.262810707092285, 6.870046138763428, 4.5134196281433105, 4.385756015777588, 5.0359697341918945, -3.726473331451416, 16.39141273498535, -8.19555377960205, 3.691361904144287, -2.7355897426605225, -5.347312927246094, -7.273257732391357, 15.796270370483398, 8.04421615600586, -10.56957721710205, 4.9240193367004395, 10.179913520812988, -3.8898637294769287, 9.329017639160156, -13.672588348388672, 9.453958511352539, 10.15784740447998, 0.8862178921699524, 9.22038459777832, -7.269058704376221, 8.221853256225586, 12.052861213684082, -7.314191818237305, 9.931100845336914, -7.3712477684021, -4.623006820678711, -7.74699068069458, 8.926329612731934, -2.6673357486724854, -11.919219017028809, -8.457236289978027, 4.149224758148193, -1.2762097120285034, 7.43945837020874, 0.7689046263694763, 8.087762832641602, 7.877620697021484, 5.790863513946533, -3.68559193611145, 11.934478759765625, 7.228893280029297, -1.1556026935577393, -4.258241653442383, -5.680696964263916, -13.88135814666748, -1.9958909749984741, 15.560445785522461, 9.778379440307617, -1.9641528129577637, -7.8587236404418945, 1.995456576347351, -7.506585121154785, 10.309767723083496, -15.09637451171875, -9.253326416015625, -3.985408306121826, 7.804049491882324, -12.717916488647461, 12.020256996154785, 1.3728840351104736, -2.5781664848327637, 5.525102615356445, 0.6751636266708374, -4.465670585632324, 2.0801987648010254, -7.8847975730896, 3.399225950241089, 7.806943893432617, 4.196093559265137, -4.0299296379089355, 4.488440036773682, 0.7644502520561218, 1.2453182935714722, -21.010974884033203, -9.332715034484863, 4.775543212890625, -11.70705509185791, 2.162177801132202, -8.700948715209961, 0.11245986074209213, 10.5279541015625, -8.805909156799316, -3.2824316024780273, 7.592236518859863, 4.060051918029785, -4.276237964630127, -3.0299105644226074, 2.773704767227173, -4.434084415435791, 9.313729286193848, -14.317267417907715, -12.483016967773438, -2.5942280292510986, 5.580995559692383, 3.514573097229004, -4.226040363311768], [14.609114646911621, 1.7485209703445435, -2.6653695106506348, -14.877220153808594, 3.003462076187134, -8.828265190124512, -6.140291690826416, 2.0415589809417725, -7.4911274909973145, 12.72823715209961, 12.401710510253906, -9.273755073547363, 14.957612037658691, 11.961220741271973, -11.47711181640625, 1.2575637102127075, -2.336245059967041, -4.232913017272949, 5.246770858764648, 13.180523872375488, -7.245901584625244, 4.450340270996094, -10.663711547851562, -3.799250602722168, 3.4356305599212646, 13.3011474609375, 17.06785011291504, -12.563675880432129, 3.7071428298950195, 6.172882080078125, -10.422758102416992, 7.377956390380859, 3.9294371604919434, -2.514723300933838, 0.15363965928554535, -0.12111710011959076, -18.967761993408203, 8.430075645446777, 14.179699897766113, -0.03583243861794472, -10.480722427368164, -3.062086343765259, 23.423187255859375, -15.07988166809082, 5.951225280761719, 10.627849578857422, 6.3463921546936035, 4.541772365570068, -0.6377008557319641, 4.28831148147583, -10.802163124084473, 2.9384143352508545, 6.688355922698975, -11.108036041259766, -9.994078636169434, -9.748719215393066, -3.986095666885376, -5.006619930267334, -12.679098129272461, -15.572325706481934, -9.299036979675293, -0.41337209939956665, 6.790546894073486, 8.590462684631348, 8.644509315490723, 0.9224175214767456, 3.448082447052002, 1.5715010166168213, -6.885912895202637, -3.361942768096924, -2.9461560249328613, -15.377337455749512, 7.829991340637207, -5.106495380401611, 2.3823564052581787, 9.102503776550293, -3.901740074157715, -0.8378763198852539, 7.834293842315674, -5.607346057891846, 0.6736271381378174, -11.729368209838867, 0.3768582046031952, 11.407190322875977, 8.2660551071167, -2.1412580013275146, 20.964250564575195, -1.2748363018035889, -2.662383556365967, 0.5727443099021912, -4.674139499664307, 11.032153129577637, 0.299218088388443, 11.93411636352539, -10.742429733276367, 8.745490074157715, 4.762606620788574, 2.45570707321167, -20.588245391845703, -15.440935134887695, 2.7610116004943848, 17.18975257873535, -9.54842758178711, -8.125864028930664, 10.679496765136719, -5.622705459594727, 1.1092101335525513, -5.187743663787842, -5.854182720184326, -5.447624206542969, -4.703904628753662, 12.862811088562012, -2.481849193572998, -15.810505867004395, -10.41037654876709, -0.4437519609928131, 7.05525016784668, 6.383714199066162, -3.6333088874816895, 3.2989675998687744, 23.409626007080078, -12.889763832092285, 3.1214962005615234, -5.637132167816162, -0.8958093523979187, -0.3172604739665985, 0.8839029669761658, -2.071946144104004, -3.9392988681793213], [-1.6226087808609009, -19.792972564697266, 4.386811256408691, -3.1950132846832275, -0.18331432342529297, -1.1217001676559448, -5.691976070404053, 8.84184741973877, 5.627349376678467, -16.97874641418457, 6.746292591094971, 1.4493898153305054, -8.166921615600586, -2.512904167175293, 4.384586811065674, -5.942126274108887, -2.9732449054718018, 1.1673662662506104, 4.419365406036377, -2.3225605487823486, 0.9215778112411499, 6.465107440948486, -3.051586151123047, 3.814973831176758, 7.07354736328125, -1.8582980632781982, -22.025495529174805, -4.24881649017334, 2.800872564315796, 6.2494049072265625, 1.8951107263565063, 4.799327850341797, 3.187483787536621, 10.489897727966309, -3.4611406326293945, -3.342768907546997, 4.458022117614746, 6.612738609313965, -10.298858642578125, 4.9953227043151855, 0.769018828868866, -1.6201890707015991, -14.195221900939941, -0.6408332586288452, 5.16918420791626, 12.79470443725586, -7.074544906616211, -9.504302978515625, 14.69727611541748, -11.734819412231445, 8.731627464294434, 4.955563545227051, -3.832418918609619, -1.0288177728652954, 2.7272419929504395, 8.309322357177734, -5.614230155944824, -5.177046775817871, -0.26624441146850586, 15.179737091064453, 11.500738143920898, -5.014520645141602, 1.9699792861938477, 3.5006163120269775, 11.845797538757324, -3.5330822467803955, -6.80734395980835, 3.281323194503784, 5.1579461097717285, 9.442678451538086, 5.569035053253174, 7.80253267288208, -6.640228271484375, -8.546512603759766, 9.696520805358887, 6.2242021560668945, -7.512681007385254, -15.908523559570312, -0.6636940240859985, 11.830839157104492, 9.9749116897583, 3.7259275913238525, 6.570992946624756, 3.870570659637451, -6.107769012451172, 2.407627582550049, -2.1904456615448, 8.038312911987305, -3.4471280574798584, 4.437759876251221, -6.103204250335693, 11.207798957824707, -6.386318206787109, 3.7535531520843506, -6.485058784484863, 6.966062068939209, -3.058582305908203, 4.5714921951293945, 5.779385089874268, 0.6137133836746216, 2.0346193313598633, -6.624073505401611, 5.727868556976318, -6.988583564758301, 0.30802375078201294, 5.756477355957031, 1.4083672761917114, -0.8923617005348206, -2.192777156829834, -0.04881368577480316, 8.710201263427734, 7.070887565612793, 8.610116958618164, 21.077871322631836, 13.10107135772705, -13.37425708770752, 0.08945796638727188, -1.2461955547332764, 5.565914154052734, 2.8401758670806885, 6.5846967697143555, 5.293823719024658, 0.17631739377975464, -4.197687149047852, -3.16827654838562, -0.5780867338180542, -3.261472463607788, -3.464280128479004, -0.25405368208885193], [4.377997875213623, -0.5950068831443787, -6.736384868621826, 1.9481005668640137, -8.11484146118164, -7.282901763916016, -1.3798457384109497, 6.6618146896362305, 4.219252109527588, -11.970240592956543, 3.2657341957092285, -8.552090644836426, 10.45800495147705, -1.9234566688537598, 3.5847291946411133, 9.24441146850586, -18.30580711364746, -3.007014513015747, 3.862405776977539, 0.7246997952461243, -0.07339078187942505, -0.11208163946866989, -8.305336952209473, -0.9190934300422668, 4.459003925323486, 5.0398054122924805, 5.028079032897949, 3.3382389545440674, 4.323783874511719, -10.810546875, 14.114364624023438, 3.9363291263580322, -4.3647780418396, 0.9608743190765381, -11.71184253692627, 3.958587408065796, 0.8518645763397217, -11.55893325805664, 7.39989709854126, 7.055803298950195, 6.148130893707275, -4.765737533569336, -9.974064826965332, 2.6973350048065186, 8.352508544921875, 7.024497032165527, -11.337727546691895, 4.938820838928223, -3.269425630569458, -8.388826370239258, 3.145533323287964, 1.2751556634902954, 5.9403557777404785, -3.871912717819214, 3.6598293781280518, -13.45860481262207, 4.500748157501221, 0.16932313144207, -6.4968791007995605, 5.481766700744629, 4.195492267608643, -2.610710382461548, 15.1135835647583, 0.666993260383606, 12.94406509399414, -7.446767807006836, 3.650358200073242, 13.019841194152832, 5.027474880218506, -2.9495575428009033, -10.457639694213867, 10.23150634765625, 2.218395948410034, -8.390164375305176, 13.090917587280273, 1.86175537109375, 3.244018077850342, 5.633075714111328, 11.080368995666504, 2.163919687271118, 2.2602901458740234, -9.368975639343262, 2.4759418964385986, 8.068050384521484, -0.699557363986969, -5.1352458000183105, 3.575606107711792, 8.169322967529297, -6.088626384735107, -6.7838592529296875, -2.972161054611206, -9.93421745300293, -16.360551834106445, 3.842989921569824, -11.218735694885254, 0.9817088842391968, 11.526535987854004, 8.766416549682617, 5.419005393981934, 0.7326508164405823, -0.5943167209625244, -4.544775009155273, 10.940964698791504, -10.651082038879395, -0.4214136600494385, -2.2043468952178955, 21.16559600830078, 5.624321937561035, 1.6176085472106934, 1.7340048551559448, 6.470912933349609, 3.8806824684143066, 1.463902473449707, -6.6855292320251465, -0.9561686515808105, -13.899567604064941, -5.137754917144775, -4.523861885070801, 2.7606706619262695, -3.7796523571014404, 2.067064046859741, 3.3966660499572754, -3.383222818374634, 4.509891510009766, -13.235004425048828, 10.980870246887207, 8.496284484863281, 7.180825233459473, -4.862011432647705], [0.03439747169613838, -0.9975814819335938, 1.3070454597473145, 5.954164981842041, -4.706101894378662, 2.6765103340148926, -7.0567755699157715, 11.24665641784668, -6.013934135437012, -7.962252616882324, 10.165124893188477, -3.253518581390381, 3.4318888187408447, -8.625710487365723, 1.2000433206558228, 0.7112724184989929, 5.544282913208008, -4.542934417724609, 3.679060220718384, -2.181666851043701, 12.91041374206543, -12.329205513000488, -4.939520359039307, 6.530974388122559, 11.256993293762207, -7.551995277404785, -3.5932931900024414, -10.620333671569824, 3.6738908290863037, -9.825559616088867, 6.836491107940674, -5.800463676452637, 5.084750175476074, -7.278492450714111, 0.8159856796264648, -2.4588124752044678, 0.1789485216140747, 2.818373441696167, -22.389738082885742, 12.301011085510254, 9.517367362976074, 2.2761595249176025, 9.061665534973145, -0.15935559570789337, 4.880119800567627, 8.12760066986084, -8.534708976745605, 10.06159782409668, 5.558685302734375, -1.0763212442398071, 11.430673599243164, -1.0410723686218262, -7.495614051818848, -0.06353605538606644, 3.6656110286712646, 16.360822677612305, -3.5940301418304443, -6.627321243286133, -0.6443243026733398, 21.886266708374023, 8.89822006225586, -11.902298927307129, 1.0510039329528809, 0.8932873606681824, -4.938793182373047, -14.179000854492188, -4.652342319488525, -2.2936999797821045, -1.648744821548462, -0.11015703529119492, -2.1081643104553223, 13.257509231567383, -10.860682487487793, -0.20535984635353088, 2.2295289039611816, 10.935737609863281, 3.7913155555725098, 6.988631248474121, -2.631195306777954, 7.634122371673584, 13.67232894897461, 22.496126174926758, 15.161733627319336, 5.655502796173096, 16.32373046875, -15.680152893066406, 11.38966178894043, -5.230245113372803, -6.581080436706543, 2.661456823348999, -2.241377115249634, -9.973160743713379, 5.021421432495117, -6.407220840454102, 1.718505620956421, -4.252480506896973, -4.657666206359863, 3.415738821029663, -8.127490997314453, 0.6664478182792664, 8.963077545166016, 3.308171510696411, 2.230013370513916, 7.2530622482299805, -6.324686527252197, 4.518179893493652, -0.011532225646078587, -3.3373258113861084, -8.022072792053223, -6.304372787475586, -7.1366729736328125, 2.566065549850464, 4.960659027099609, 2.886655569076538, -7.741396903991699, 5.781073093414307, 1.0639177560806274, -13.091756820678711, 4.135473728179932, 9.61996078491211, -7.821431636810303, 7.274562358856201, -13.175736427307129, 0.7347524762153625, -7.613563060760498, -0.37464603781700134, 1.7917861938476562, -5.239132404327393, -13.391054153442383], [-12.9669828414917, 20.372936248779297, -12.356866836547852, 4.21184778213501, 4.479360580444336, 4.9917731285095215, -1.7058745622634888, 2.3359694480895996, 5.780086040496826, -5.8147172927856445, 6.0479912757873535, 2.4114739894866943, -6.411106109619141, -2.6947245597839355, 5.305008411407471, 0.5262911319732666, -5.981788635253906, 5.594348430633545, -3.2976646423339844, 7.153088092803955, -4.007136344909668, 0.5061227083206177, -3.34177303314209, 4.435962677001953, 7.763031005859375, 11.913579940795898, -9.616573333740234, 1.6065669059753418, 8.171871185302734, -7.30376672744751, 9.710341453552246, -7.110958576202393, 11.344902992248535, -0.7099495530128479, -2.8287343978881836, 0.2268841415643692, -10.253454208374023, 5.471321105957031, -6.465819358825684, 5.020118236541748, 11.634339332580566, 1.576156735420227, -4.0139851570129395, 2.5292928218841553, 2.7228386402130127, 11.051669120788574, -5.203983783721924, 5.15202522277832, 6.507963180541992, 9.932036399841309, -7.326658725738525, 9.271807670593262, 2.6285321712493896, 1.1515651941299438, -1.0304524898529053, -9.677664756774902, -6.933023929595947, 7.406023025512695, -0.45631298422813416, 15.979703903198242, 22.95842170715332, -0.4950536787509918, 7.363652229309082, -6.943931579589844, -5.643418312072754, 2.7092459201812744, 6.891877174377441, -0.629523754119873, -4.653818607330322, -1.8847413063049316, -5.937204837799072, -3.4706969261169434, 10.626240730285645, -1.6245989799499512, 5.250797748565674, -0.10549186170101166, 4.176558971405029, -5.878507137298584, -7.742682933807373, 0.6500162482261658, -7.812564373016357, -3.682129383087158, -3.4777798652648926, -16.886152267456055, -0.6237260103225708, 10.481549263000488, -2.6196398735046387, 5.883885860443115, 17.343040466308594, -11.459856986999512, 7.932868480682373, 3.0014097690582275, -6.488976001739502, 8.29604721069336, -11.74626636505127, 5.473343372344971, 0.2817240357398987, 8.988468170166016, -9.005967140197754, -2.9013516902923584, -10.336183547973633, -14.794695854187012, -9.591347694396973, 0.5408381819725037, 2.478499412536621, -10.743634223937988, 11.070291519165039, 16.79288101196289, 10.477255821228027, 5.086287021636963, 10.072168350219727, -10.902817726135254, -9.82805347442627, -12.588140487670898, -2.256336212158203, 4.39138650894165, -0.9076727032661438, -11.778107643127441, -0.2852065861225128, 0.008747818879783154, -2.6396493911743164, -0.4532572627067566, 7.3258466720581055, 1.4270797967910767, -3.732187032699585, -6.890126705169678, 1.625563144683838, -10.97781753540039, 9.604856491088867], [4.36391544342041, 4.300568103790283, 0.2872943580150604, -5.291393280029297, -5.44854736328125, 13.067344665527344, -7.188455581665039, 6.339685440063477, 0.5046274065971375, -6.812082290649414, 1.883710265159607, 1.2816503047943115, -7.179804801940918, -1.7854365110397339, 1.5290576219558716, 8.141951560974121, 7.9460577964782715, -9.183279991149902, 7.083730220794678, 9.110390663146973, 10.179729461669922, -2.966407060623169, 10.583930969238281, -3.4079370498657227, 6.847314834594727, 4.443039894104004, -10.745959281921387, 1.574092984199524, -14.118199348449707, 0.7702571153640747, 6.305455684661865, 4.206528186798096, 0.3531286120414734, -4.252549648284912, -5.542543411254883, 4.30391263961792, 1.5019407272338867, 11.57563591003418, -11.331844329833984, 2.792114496231079, -6.347370147705078, 8.796489715576172, 2.0605580806732178, 15.793442726135254, 19.741622924804688, -2.6965887546539307, -2.3581290245056152, -1.6414899826049805, -2.191140651702881, -8.435750007629395, 9.836945533752441, 4.222201824188232, 4.382543087005615, -0.45450830459594727, 1.3730547428131104, 5.0018768310546875, -0.02643345482647419, -7.645190238952637, -2.3258988857269287, 15.14031982421875, 4.178043842315674, -1.5384384393692017, 9.528419494628906, -8.068401336669922, 0.19056521356105804, 2.4848692417144775, -5.415544509887695, -14.641764640808105, 2.57039475440979, -1.470500111579895, 9.493722915649414, 11.198432922363281, -8.201048851013184, -11.895403861999512, -3.6631994247436523, -1.4621555805206299, 2.193366050720215, -9.060770988464355, 4.3312530517578125, 4.902342319488525, 1.1772820949554443, 0.708622932434082, -0.6712353825569153, 3.903998851776123, -1.7275547981262207, -4.737879276275635, -3.944260358810425, -6.524045467376709, -8.933823585510254, 2.146819829940796, -2.3939921855926514, -2.22314715385437, -5.297619342803955, -6.094838619232178, 1.5387094020843506, -0.08384548872709274, -8.723419189453125, 1.3832907676696777, -0.6499994397163391, -5.337991714477539, 5.841408729553223, -15.436601638793945, 11.68104076385498, -5.114516258239746, -3.3286900520324707, -2.823699474334717, 5.752520561218262, -6.571659564971924, 2.7781999111175537, -6.6014909744262695, 0.8507257699966431, 6.621917247772217, 8.783641815185547, 5.368251323699951, 8.45950698852539, -7.359971046447754, 8.353343963623047, 2.1614441871643066, 0.4562300741672516, 1.489780068397522, -6.92412805557251, -3.283067464828491, 0.5796025395393372, 7.347321510314941, -12.430251121520996, -3.900912046432495, -6.914517879486084, -8.811800956726074, -7.005866527557373], [-1.9396653175354004, 7.637606620788574, 4.424022674560547, 8.473779678344727, -2.839299440383911, 3.235826253890991, 10.900830268859863, 6.008129596710205, -5.047995567321777, -3.706007242202759, -4.177803039550781, 7.493170261383057, 2.235637664794922, -0.150247260928154, 4.247030735015869, -11.09206485748291, 10.220843315124512, -3.6803641319274902, -5.387904167175293, 10.164207458496094, -7.977607727050781, -2.596980333328247, -10.744940757751465, -9.23936939239502, 11.510218620300293, -0.1781633198261261, -6.862630844116211, -7.022110939025879, -7.105505466461182, 3.6788275241851807, 3.6934266090393066, -5.818417549133301, 10.808366775512695, 1.5127174854278564, -8.530102729797363, 1.5404237508773804, -6.398464679718018, 0.3405294716358185, 2.1665074825286865, 11.083489418029785, 2.678642988204956, 0.9611334204673767, 1.9445666074752808, 8.098938941955566, 9.931066513061523, -2.7915124893188477, 6.260257720947266, -2.047821521759033, -5.994652271270752, -5.401331901550293, -5.874626636505127, 6.859704971313477, 22.604516983032227, 0.3710915744304657, -5.392419815063477, 5.020634174346924, -7.126440525054932, 1.178989291191101, -8.233074188232422, -0.6764795780181885, -0.7413417100906372, 6.503408432006836, 7.513967990875244, -3.1951308250427246, 0.17186526954174042, 1.3331254720687866, 3.3257813453674316, 2.204984188079834, -17.619312286376953, 0.9608417749404907, 8.949259757995605, 14.17591381072998, 11.067791938781738, -4.898586273193359, 4.230738639831543, 13.238495826721191, 17.324914932250977, -2.142552137374878, 2.5764946937561035, -2.691472291946411, -7.20492696762085, 1.4903693199157715, -6.653796195983887, -4.639739990234375, 6.369542598724365, -6.701300144195557, 1.0645722150802612, -3.2637484073638916, -6.921858787536621, -3.6886329650878906, 0.8118855357170105, -15.092137336730957, -0.6096085906028748, -4.824145317077637, -1.2857248783111572, -9.627180099487305, -14.50256633758545, 9.131518363952637, -10.5661039352417, -1.5539056062698364, -3.457759141921997, 3.2331385612487793, 10.298113822937012, -3.7727930545806885, -10.502352714538574, -8.996471405029297, 14.396310806274414, -2.96940279006958, -8.729470252990723, -9.021617889404297, 11.20812702178955, 6.1781792640686035, -2.037205457687378, -0.5495330095291138, 1.335109829902649, -0.38278403878211975, 19.31584358215332, 1.5440315008163452, -7.59197998046875, 11.576577186584473, 5.282342910766602, -2.8369510173797607, -8.255875587463379, 2.7938597202301025, -1.639206051826477, -10.062027931213379, -2.7017221450805664, 2.393052339553833, 0.25246039032936096], [-2.245680570602417, -3.233738899230957, 0.30483004450798035, 11.520852088928223, -12.606645584106445, 5.859168529510498, 4.043699741363525, 0.6284728646278381, 10.099183082580566, -15.642888069152832, -1.276865839958191, -2.3912601470947266, -23.93102264404297, -6.624354839324951, 1.168967604637146, -3.2733421325683594, -7.964381694793701, 8.772866249084473, -11.94098949432373, -13.08480167388916, 0.6873044967651367, 0.6823634505271912, 5.727025985717773, 3.551513910293579, 12.754683494567871, -3.0304505825042725, -10.890828132629395, 10.115377426147461, -0.12388629466295242, -1.2174843549728394, 13.6073579788208, -3.2215356826782227, -10.568061828613281, -7.924408912658691, 2.924243450164795, -0.9366381764411926, 1.478265643119812, 15.332358360290527, -10.059715270996094, 9.811485290527344, 7.021601676940918, -12.356925964355469, -19.199596405029297, 10.90523624420166, 18.264375686645508, 0.6363104581832886, -9.778129577636719, -4.619511127471924, 11.308321952819824, 1.9489731788635254, 21.99088478088379, 4.653611183166504, -8.631658554077148, 1.3917980194091797, -17.856992721557617, -7.858372211456299, -23.058082580566406, -3.077899932861328, -1.5421032905578613, 38.645408630371094, 14.595959663391113, 0.6805219054222107, 4.256438732147217, 6.746834754943848, -13.171380043029785, 4.698903560638428, 19.270301818847656, 21.42765998840332, -1.3546881675720215, -0.940355122089386, -7.217534065246582, 15.784445762634277, -4.004945278167725, -5.509291648864746, 7.592954158782959, -0.8299528956413269, 13.488179206848145, -1.2327336072921753, -8.193102836608887, 10.540672302246094, -8.367301940917969, -2.521385908126831, 2.119715452194214, -21.43623161315918, -1.6785632371902466, -1.171147346496582, -21.222257614135742, 0.5007094740867615, 6.761951923370361, 5.317315101623535, 3.4343440532684326, 9.196328163146973, -15.499173164367676, 3.824251890182495, 11.706177711486816, 21.348384857177734, -9.774045944213867, 9.354299545288086, -4.712474822998047, -2.9159529209136963, 3.34335994720459, -6.993464946746826, 24.60359001159668, 0.5874726176261902, -0.45130839943885803, -4.799910068511963, -17.395383834838867, -3.3847475051879883, 3.2775979042053223, 7.894268989562988, -8.785391807556152, 3.4403724670410156, -2.3896048069000244, 7.867648124694824, 9.909517288208008, -6.67214822769165, 8.657296180725098, -9.367218971252441, 7.096701622009277, -1.6666061878204346, -22.113428115844727, 0.6613724231719971, -13.994470596313477, 5.17063045501709, -2.300799608230591, -15.897022247314453, 2.5926718711853027, -1.0655643939971924, -1.9681967496871948], [-5.525822639465332, 3.515855312347412, 6.537642955780029, 1.832831859588623, 3.553070068359375, -5.248510837554932, -4.1271772384643555, -3.071361541748047, -0.16358549892902374, -7.643804550170898, -2.015230655670166, -8.44433307647705, -16.019411087036133, 3.661034107208252, -2.3692731857299805, 3.2972159385681152, 3.0993072986602783, 2.7986764907836914, -5.649084568023682, -8.442631721496582, 0.3996659517288208, 3.7313730716705322, -3.1765544414520264, -1.512831687927246, 1.3226209878921509, -4.2396674156188965, -5.34220027923584, 0.8855192065238953, -8.05662727355957, -0.8194864988327026, -4.907255172729492, -8.363502502441406, -2.745137929916382, 8.182564735412598, -6.3122968673706055, -4.1214599609375, -11.987452507019043, -9.016844749450684, 12.840158462524414, -3.1103286743164062, 8.041753768920898, -4.096526622772217, -4.386153221130371, 10.473211288452148, 1.4631203413009644, 7.952033519744873, 3.0705299377441406, 2.010890483856201, -9.853717803955078, 2.224964141845703, -9.609596252441406, -1.3632866144180298, -2.2142012119293213, -0.5555508732795715, -13.8840913772583, -1.3354746103286743, 3.4587957859039307, -14.413970947265625, 2.8954102993011475, -4.31727409362793, 1.9774404764175415, 7.4054274559021, 0.4958878755569458, 10.840950965881348, 0.38318750262260437, 20.158597946166992, -0.33939966559410095, 3.0231564044952393, -3.256424903869629, 7.350222587585449, -6.130791664123535, 10.570152282714844, -11.146583557128906, -6.890429496765137, -7.6952080726623535, 1.7626228332519531, 1.909680724143982, -2.8609397411346436, 7.721519470214844, -15.64908504486084, -2.678004503250122, -5.769403457641602, 3.5388216972351074, -5.686736106872559, -5.979673385620117, -5.97800350189209, -1.1891611814498901, 0.6435855031013489, 2.752664089202881, -5.691282272338867, -3.263925075531006, -1.5494800806045532, -1.2508857250213623, 1.7651852369308472, -4.386285305023193, 8.753999710083008, -1.9536844491958618, -0.4552474915981293, 0.04159744456410408, 0.26799020171165466, 9.633153915405273, -9.782591819763184, -4.217299938201904, -18.21147346496582, 6.218636989593506, -0.2732684910297394, 4.989952087402344, -1.6961597204208374, -0.01807323470711708, -11.673256874084473, -0.27172181010246277, 4.3778557777404785, -0.0943039208650589, 5.505352020263672, 4.6159515380859375, -9.267719268798828, 4.177381992340088, 5.014969825744629, 13.280988693237305, -5.7557196617126465, -9.255921363830566, -11.512139320373535, -1.9765561819076538, -0.1661432534456253, 8.31031608581543, -13.547237396240234, -3.945735454559326, 8.390963554382324, 0.7996336817741394], [-5.615322589874268, 6.568789958953857, -5.12987756729126, 7.823490142822266, -3.7262768745422363, 8.070609092712402, 8.834341049194336, -2.9204604625701904, 2.9613678455352783, -13.295550346374512, 2.808521270751953, 11.246459007263184, 8.985394477844238, -1.829800009727478, 1.3854413032531738, -1.34820556640625, -1.77561354637146, 7.233558177947998, -3.3323421478271484, 6.218561172485352, -0.7033019661903381, -3.0341482162475586, -7.100259780883789, -6.481841087341309, 3.816910743713379, -14.174792289733887, -6.199450492858887, -1.8737409114837646, 5.764112949371338, 2.9822795391082764, 1.7950763702392578, -4.164148330688477, 5.542498588562012, -1.3436743021011353, 14.500946998596191, 0.5688402056694031, 16.2775936126709, -4.088027477264404, 4.0456223487854, 4.641952991485596, 9.038426399230957, 8.366643905639648, -4.251593112945557, -3.5610694885253906, -1.4431153535842896, -3.514118194580078, 17.670501708984375, 4.519383907318115, 0.6899963021278381, -3.717288017272949, -4.236663818359375, -6.343925476074219, 4.291290283203125, 5.925018787384033, 4.318769931793213, 8.554060935974121, -5.644421577453613, 4.224861145019531, -8.512775421142578, 0.6799611449241638, 0.45302146673202515, 2.08439040184021, -2.645587921142578, -0.36354199051856995, 0.6741083264350891, -0.8132314682006836, 7.488430976867676, 5.308033466339111, -4.731501579284668, -1.0379295349121094, -7.47067928314209, 3.387711763381958, 0.5698457360267639, 8.85892105102539, -0.997757613658905, 7.928056716918945, 4.404759407043457, -3.536691188812256, -1.6260493993759155, 4.921815395355225, -5.540961742401123, 15.671217918395996, -3.5824384689331055, 8.62916374206543, 4.583527088165283, 4.983046054840088, 1.0022224187850952, 1.6220881938934326, -2.101881742477417, -8.202620506286621, 1.5725653171539307, -8.813896179199219, -6.1513776779174805, -5.967245578765869, 5.065727233886719, -16.164587020874023, -10.681304931640625, -1.6852648258209229, 7.713160991668701, -3.2629287242889404, -0.03632320091128349, 0.7352445125579834, -7.36191987991333, 15.039497375488281, -8.41694450378418, 0.638319194316864, -2.0664172172546387, -8.838103294372559, -4.201250076293945, -0.020806550979614258, 2.7487900257110596, -1.6997992992401123, -2.880878210067749, 1.043265461921692, -2.362492322921753, 13.870218276977539, -1.2140663862228394, -5.135859489440918, -3.047081232070923, 3.668407440185547, -2.0666754245758057, 1.9377325773239136, -3.9832160472869873, -5.776244163513184, -7.279286861419678, 3.2534737586975098, 9.403119087219238, -1.6372411251068115, -3.3277435302734375], [-5.867336750030518, -1.872591495513916, 9.383493423461914, -5.383724212646484, 2.7115378379821777, 13.48293685913086, 12.921516418457031, 10.819735527038574, -5.0376739501953125, -6.807984352111816, -10.076589584350586, -19.809770584106445, 0.5708997845649719, 8.556334495544434, -11.697283744812012, -8.948068618774414, -14.419877052307129, 2.5169644355773926, -8.3165864944458, 1.7058578729629517, -21.836349487304688, -2.0734379291534424, 8.81999683380127, 11.144454956054688, 5.638612270355225, -0.1416914314031601, 10.237236976623535, 9.042926788330078, -9.601553916931152, 13.322189331054688, 16.646377563476562, 2.3938183784484863, -8.43938159942627, 5.189528942108154, 3.079228401184082, 4.1060333251953125, 0.30867743492126465, 21.05295753479004, -0.28571584820747375, 4.291968822479248, -33.80396270751953, -3.3167388439178467, -9.848357200622559, -6.9262847900390625, 14.409943580627441, -8.124670028686523, -10.369025230407715, -9.662557601928711, -2.0525095462799072, -6.182072162628174, -0.9157096743583679, -4.603062629699707, 0.7709057927131653, -3.3506221771240234, -5.673615455627441, 7.013060569763184, 5.184723854064941, 6.368007183074951, 6.548185348510742, 10.447988510131836, 7.977151393890381, 5.02871036529541, 7.166031837463379, 5.983945369720459, -5.475471019744873, 4.9066925048828125, -1.2294105291366577, -10.209718704223633, 3.0018515586853027, 8.1707181930542, -7.542507171630859, 7.633254051208496, -3.6150596141815186, 0.4003565311431885, -9.30573558807373, -18.84790802001953, -3.5845468044281006, 6.771210670471191, 10.173399925231934, 9.053132057189941, -9.353501319885254, 2.5444304943084717, -6.555301666259766, -0.18209028244018555, 3.1918296813964844, -3.1172678470611572, -2.4688262939453125, -15.822066307067871, 10.782185554504395, 5.16218900680542, 4.9136738777160645, 2.8318676948547363, 5.758880615234375, -4.355461120605469, -0.5251522064208984, 20.82942771911621, 16.069883346557617, -16.924060821533203, 10.07606315612793, -12.537148475646973, 1.8217116594314575, -26.264204025268555, 5.245882511138916, 8.580052375793457, 10.632286071777344, 4.729613780975342, -5.1694536209106445, 8.49205207824707, -6.409387588500977, 26.29248809814453, 12.434661865234375, 1.6376569271087646, -5.5738091468811035, -11.235443115234375, 16.718280792236328, 2.434784173965454, 4.029412746429443, 4.078774452209473, 11.75987720489502, -11.90845012664795, -6.105679512023926, 8.424756050109863, 1.200648546218872, 10.386847496032715, -0.10015246272087097, 7.270156383514404, -10.651124000549316, -2.794924020767212, 15.967705726623535]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-0.0607568733394146, 1.8255975246429443, 0.10331789404153824, 0.30314844846725464, -0.9500860571861267, 0.4888887405395508, 0.13834570348262787, -0.4106406569480896, 1.3470858335494995, 1.6704076528549194, -1.0952130556106567, -2.003145694732666, -0.18646827340126038, -0.694010853767395, 0.1823398470878601, -0.02246456779539585, -1.0390714406967163], [-0.6639001965522766, 0.6858952045440674, -0.05207805335521698, 0.5304419994354248, -1.0937563180923462, -0.45575442910194397, 1.8885730504989624, -0.2016231119632721, -0.7236751317977905, -0.8762997984886169, -0.03379439935088158, 0.023537205532193184, -1.0643410682678223, 1.7927364110946655, -1.0941402912139893, -0.9173477292060852, 0.2659219205379486], [-0.4515153765678406, 0.8509543538093567, 2.1239311695098877, 1.4365341663360596, 0.5537005066871643, 1.3775137662887573, -0.1506115347146988, -2.2822883129119873, 2.2510738372802734, -0.8169445991516113, 0.47647544741630554, -0.8759957551956177, -1.0526570081710815, 0.017163893207907677, 2.4718167781829834, 0.02875535935163498, 0.17038151621818542], [-1.4724397659301758, -0.35610318183898926, -1.2062855958938599, 0.1459490805864334, -0.2355540245771408, 1.521655559539795, 0.8877913355827332, -0.8063844442367554, -0.7552749514579773, -1.0918915271759033, 0.3335055708885193, -0.14111189544200897, -0.67001873254776, -0.27667590975761414, -0.1888647973537445, 0.33094552159309387, 0.8388763666152954], [-0.2706209123134613, -0.7597128748893738, -0.04787440970540047, 0.8615742921829224, -0.2693452537059784, -0.4031304717063904, -0.03937511891126633, 0.40867745876312256, -0.8752546310424805, 0.1091424748301506, 0.2861444056034088, 0.8425743579864502, -1.4702407121658325, -2.1506168842315674, -0.3723105490207672, -0.3060724437236786, -1.7324814796447754], [1.265772819519043, 0.854358434677124, 0.5778688192367554, -0.48784205317497253, -0.4117920696735382, -0.3843728303909302, 0.6844402551651001, -0.8309915065765381, 0.4572714567184448, 0.3276892900466919, 0.9315605163574219, -0.04566749930381775, 0.9438450336456299, 0.30548620223999023, 0.1169438585639, -0.0548657551407814, 0.09607680141925812], [0.5345564484596252, 0.890234649181366, -0.6088855862617493, 0.38072434067726135, -1.3629499673843384, 0.4642902612686157, -0.5255164504051208, -0.31749144196510315, -0.2879544794559479, 0.17479203641414642, 1.0397149324417114, 1.1304196119308472, 0.16476508975028992, -0.6521852016448975, 0.16991522908210754, 1.0502879619598389, -0.1435241997241974], [-0.34325915575027466, -0.10156209021806717, -1.9768610000610352, -0.28303107619285583, -0.15365783870220184, 0.4248865842819214, 0.298361212015152, -0.6053568720817566, 0.09203363209962845, -1.1130683422088623, 1.259798288345337, -1.01030433177948, -0.7249224185943604, -0.2667410373687744, 1.201827883720398, -0.47926661372184753, -1.240332007408142], [-1.1374810934066772, -1.8335527181625366, -0.1378920078277588, 0.5770726203918457, 0.7470935583114624, -0.23382778465747833, -0.6753941774368286, -1.6921740770339966, -0.3513394296169281, 0.3471517562866211, -0.6253460645675659, -0.2543829083442688, -1.629138469696045, -0.05937865749001503, -0.3711102604866028, 0.37423640489578247, 0.4307647943496704], [-1.1772955656051636, 0.6543543934822083, 0.5052213668823242, 1.4784408807754517, 0.4712459444999695, -0.17832770943641663, -1.4354438781738281, 0.8180949091911316, -0.4161248207092285, 0.2937124967575073, 1.3920649290084839, 0.5703022480010986, -0.47494128346443176, -2.31211519241333, 1.4729549884796143, 0.5008083581924438, -0.02139255218207836], [-1.6789659261703491, -0.04680849611759186, 0.6826497316360474, 0.7753415703773499, -0.7717992663383484, -0.7394113540649414, 0.04274723306298256, -0.40308234095573425, -1.6611539125442505, -0.9107741713523865, 0.6161928772926331, 2.2982325553894043, -0.971381664276123, -0.6325045824050903, 0.06409595161676407, -0.3778343200683594, -0.6024200916290283], [-2.390388250350952, 0.426219642162323, -0.7692163586616516, 2.561875581741333, -2.0991604328155518, 1.1334564685821533, -0.702192485332489, 0.12515266239643097, 0.12845823168754578, -0.33476364612579346, -0.050687406212091446, 0.5637518167495728, 0.21651622653007507, -0.9171918034553528, -0.17217007279396057, 0.4399755001068115, -0.24785351753234863], [2.1239535808563232, -0.28964126110076904, 0.01903735287487507, 0.7095628976821899, -0.7299510836601257, 0.6436019539833069, 0.7258095145225525, -1.0711584091186523, -0.9866315722465515, -1.548243761062622, -0.4349367320537567, 1.3439807891845703, -0.03349423035979271, -0.7765108942985535, 0.4915529787540436, -0.2872221767902374, -0.8719706535339355], [-1.3066308498382568, -0.7242279648780823, -0.7166008949279785, 0.6362035274505615, 0.232876718044281, 0.08368736505508423, 2.008683919906616, -1.6079057455062866, 0.4394652843475342, -0.5846236348152161, 2.6966564655303955, 1.67134690284729, 0.2365298718214035, -0.654965877532959, 1.5394513607025146, 0.8643099069595337, 0.0340871624648571], [-0.5817399621009827, -0.552066445350647, -1.6899720430374146, -0.23874184489250183, -0.7872176766395569, 0.9429493546485901, -0.5008513927459717, -2.3030803203582764, 1.1114970445632935, 0.14883218705654144, 0.6685011386871338, -0.8035268783569336, -0.8548474907875061, 1.6068178415298462, 1.1092647314071655, 0.21453212201595306, -0.2857627868652344], [0.061357531696558, 0.645149827003479, -1.8516566753387451, 2.431964159011841, -0.295399934053421, -0.1532403826713562, 1.999849557876587, -0.4724399149417877, 1.3103913068771362, -0.10241811722517014, 0.058578696101903915, -0.9425411820411682, -0.9602614045143127, -0.6552204489707947, -1.1258631944656372, -0.4782004654407501, -0.6017186045646667], [0.14766891300678253, 1.2938566207885742, -0.3204207718372345, -2.3482108116149902, -1.0785075426101685, 0.2691779136657715, -0.6648780703544617, -1.2177355289459229, 0.38519102334976196, 0.4952297806739807, 0.19586414098739624, -0.5506905913352966, -0.4772413969039917, -1.3407188653945923, 0.2379865050315857, 0.6098827719688416, -0.8184846043586731], [-1.2201648950576782, -1.9694972038269043, 1.6768516302108765, 0.147661954164505, -0.43933647871017456, 1.0256274938583374, 1.1577496528625488, -0.01971604861319065, 0.4817203879356384, -0.6940860152244568, -0.8851453065872192, 2.86942458152771, -0.309166818857193, -0.6507968306541443, 2.781287670135498, 0.4118425250053406, 0.5051686763763428], [0.9027851819992065, 0.5487924218177795, -0.5465127825737, 0.6978732943534851, -1.8897894620895386, -0.7263308167457581, -1.2625941038131714, 1.1276401281356812, 0.46181342005729675, -0.36440733075141907, 0.8625097274780273, -0.8243597149848938, -0.017276812344789505, -1.4261831045150757, 0.14176851511001587, -1.547491431236267, -1.0724865198135376], [-0.6705822348594666, 0.9865386486053467, 1.1064772605895996, 1.317382574081421, -0.07593011111021042, 0.40454086661338806, -0.8865888118743896, 1.0467153787612915, -0.10895334184169769, 0.17817765474319458, 0.5815300941467285, -0.3555227816104889, 2.37725830078125, 1.2161519527435303, 0.8368165493011475, -0.6218254566192627, -0.07044338434934616], [-1.4620816707611084, -0.0882827490568161, 1.0393823385238647, -1.2680912017822266, -0.03902667015790939, -2.0567145347595215, -1.819357991218567, -0.43094319105148315, -1.657853126525879, -0.5018191933631897, -0.4630128741264343, -0.012615286745131016, 0.2526683211326599, -0.5391683578491211, 1.1013076305389404, 0.059245068579912186, 1.1298850774765015], [0.3548392653465271, -0.015127202495932579, -0.1151316836476326, 0.10947999358177185, -0.7981447577476501, 0.1567077785730362, 0.23736347258090973, 0.6787797212600708, 1.2077401876449585, 2.040555000305176, -0.4432376027107239, 0.7194688320159912, -0.21171791851520538, 0.4477440118789673, -0.24960152804851532, 0.7947109937667847, 0.4381580352783203], [0.870208203792572, 0.5248897075653076, -1.2984979152679443, 0.6990167498588562, -1.025480031967163, -0.23119403421878815, -2.033083200454712, 1.2804630994796753, -1.5251675844192505, -0.4310556948184967, -0.11460869014263153, 0.3103921711444855, 0.9997122883796692, 1.7067914009094238, -0.1717693954706192, 0.018028870224952698, 1.459058403968811], [0.3913995027542114, -0.45415326952934265, 1.0200175046920776, 0.8541727066040039, -0.28446224331855774, 0.18655073642730713, -0.2854902446269989, 0.4713113605976105, 0.4781055450439453, -0.27911755442619324, -0.1814665049314499, 1.1977179050445557, 0.32449662685394287, 0.32623767852783203, 0.2922140657901764, 1.5014311075210571, -0.25711309909820557], [0.10321509838104248, -0.8277804851531982, -0.6233527660369873, 1.517699956893921, 0.9035778045654297, -0.04053059220314026, 0.905082643032074, 0.029436232522130013, 0.3471825420856476, 0.36840304732322693, 0.013978871516883373, -0.3524388372898102, -0.9958552122116089, -0.4204815626144409, -0.9019445776939392, -0.7174028158187866, 0.6028656959533691], [0.6872023344039917, -0.04167195037007332, 0.17158107459545135, -0.2527415454387665, -0.4450693428516388, -0.13203652203083038, 0.1461954116821289, 0.25639307498931885, 1.4599539041519165, 0.09530110657215118, 1.295276165008545, -0.05028287693858147, 0.3454333245754242, 0.34904348850250244, 1.2173004150390625, -0.8929253816604614, 1.4073206186294556], [0.11584486812353134, 0.0820331871509552, 1.14109468460083, -0.720770001411438, -0.10678564012050629, 0.2908538579940796, 1.0279958248138428, 0.3267623484134674, -0.8698082566261292, -1.0520679950714111, -0.015361189842224121, 0.1684114784002304, -2.1470999717712402, -0.06876381486654282, 0.6503675580024719, -0.1711365282535553, 0.7449855208396912], [1.213368535041809, -1.1679933071136475, -0.18796129524707794, -2.197384834289551, 1.8077619075775146, -1.1424764394760132, -1.743833303451538, -1.5928726196289062, 0.08869562298059464, -0.9176753759384155, -1.6999077796936035, -0.02168644778430462, -0.8518138527870178, 0.6649923324584961, -0.7091413736343384, -0.18608888983726501, -1.688607096672058], [-0.6243956685066223, -0.23141713440418243, -0.9749464988708496, 0.381845623254776, 0.0967179387807846, -0.031576886773109436, -0.697048544883728, -0.8218953013420105, 0.22149883210659027, 1.723185658454895, -0.9592244625091553, -0.10856591910123825, 1.2492947578430176, 0.3095908463001251, -0.8520262241363525, -1.6986677646636963, 1.335121750831604], [-0.17679287493228912, -1.009504795074463, -0.7778589725494385, 0.17256686091423035, 0.3774632215499878, 0.20623144507408142, 0.41876697540283203, 0.7973225116729736, 0.019005566835403442, 1.7197884321212769, 0.7597585320472717, 0.29659876227378845, -1.2154276371002197, 0.9696151614189148, -0.06975213438272476, 0.7513947486877441, 0.61479252576828], [-0.7387226223945618, 1.5515656471252441, -0.9288998246192932, -0.166176900267601, -0.2191774547100067, 0.7729468941688538, 0.19280706346035004, 0.5955904722213745, -0.5284374356269836, 0.18126708269119263, 0.13632039725780487, -0.2669333815574646, -0.17258219420909882, 0.5155118703842163, -1.1070917844772339, 1.3896740674972534, -0.34327784180641174], [-1.1744389533996582, 1.7688703536987305, 1.72054123878479, 1.848118782043457, -0.1345832496881485, -0.39702296257019043, 1.028037190437317, 0.8352905511856079, -1.1427476406097412, -0.29332852363586426, -0.5951890349388123, 0.350485622882843, 1.059340238571167, -0.3882712125778198, 0.31463196873664856, 0.6325123906135559, -0.1402343362569809], [-0.7080163359642029, -0.0011417034547775984, 1.0793801546096802, -0.7123672366142273, 0.6878343224525452, -0.9396439790725708, 0.5209329128265381, 0.5662364363670349, -0.730952799320221, 0.4650927484035492, 0.42195308208465576, -1.140578031539917, -1.0124691724777222, -1.60924232006073, 0.5052259564399719, -1.2233831882476807, -0.8301295042037964], [1.2154690027236938, -0.45997482538223267, -1.123019814491272, -0.2082214057445526, 1.3212236166000366, 1.713086485862732, -1.1543833017349243, 0.20945721864700317, -1.7182693481445312, 2.238046884536743, -0.895232081413269, -1.092700481414795, -1.4026153087615967, -1.9372518062591553, 1.0612328052520752, -0.4860459864139557, -1.1184266805648804], [1.5789345502853394, 0.9058794975280762, 1.4471231698989868, -0.6903323531150818, -1.6442725658416748, -1.6493743658065796, 0.3397224247455597, 0.19148606061935425, -1.6937135457992554, -1.1162470579147339, 1.5014818906784058, -1.23134446144104, 0.4272690415382385, -0.8425061702728271, -0.27572375535964966, -1.7243990898132324, 1.7873198986053467], [1.2894295454025269, 1.2202295064926147, -0.2935009300708771, -1.150292158126831, -0.6031349301338196, 1.579471230506897, 0.27175697684288025, -0.48348161578178406, 2.162606954574585, -0.8791545033454895, -0.43696868419647217, 0.5643594861030579, -1.0565307140350342, 0.06815771013498306, 1.4900577068328857, 0.6282668113708496, -1.742181658744812], [1.0955413579940796, -0.2758026421070099, -0.08886592835187912, 0.0509820356965065, 2.0009727478027344, -0.7142388820648193, 1.2970000505447388, -0.8179230690002441, -0.7984793782234192, -1.5210057497024536, 0.30898380279541016, 0.29942432045936584, 0.18643690645694733, -1.800873875617981, -0.21934784948825836, 0.3240046501159668, 0.10531829297542572], [2.376688003540039, 1.2785924673080444, -2.655843734741211, -1.5613682270050049, 0.052222006022930145, 0.2972569167613983, 0.6570395231246948, -0.21455790102481842, -0.04835902526974678, -1.054966926574707, -0.48112666606903076, -1.470342993736267, 0.7631430625915527, -0.3600097596645355, -1.3827602863311768, -0.17127399146556854, -0.027088478207588196], [-0.4667913615703583, -0.525249183177948, 1.4325590133666992, 0.2112220823764801, 1.0852235555648804, -1.0358350276947021, -2.009706974029541, 0.3518963158130646, -0.5605688691139221, -0.7808293104171753, -0.8306739926338196, -0.3382963240146637, -0.5117329955101013, -0.19359274208545685, -0.5920574069023132, -0.6164735555648804, -0.8543530106544495], [-2.0108139514923096, 1.16713285446167, 0.5211403369903564, -0.339397668838501, -0.205704465508461, -0.2212473303079605, 0.3158700466156006, -0.03332868218421936, 1.49581778049469, 1.8621503114700317, -0.43533438444137573, 0.005406119395047426, 0.8351921439170837, 1.0559451580047607, -1.4763787984848022, 0.6739111542701721, -0.5319231748580933], [-2.43192720413208, -1.1521189212799072, -1.0728410482406616, 0.1841963678598404, -0.8359073400497437, 1.628190279006958, 1.2377275228500366, -2.262761354446411, 1.342943787574768, -0.8103327751159668, -1.193191647529602, 1.4648385047912598, 0.8262515664100647, 0.4365202784538269, -0.246406689286232, 0.14283370971679688, -1.156184196472168], [2.1081340312957764, 1.0431772470474243, 0.9130862951278687, 1.4038091897964478, -0.3569074869155884, -0.19058437645435333, 0.03155287355184555, 0.7675982713699341, 0.9532870054244995, -0.8018227219581604, -0.1461823433637619, 1.0311226844787598, 1.2171140909194946, 0.20023903250694275, -0.8163345456123352, -1.0630836486816406, -0.20812730491161346], [1.187369704246521, 0.6332225203514099, -0.5478215217590332, -2.29367733001709, -0.004969841800630093, 0.5653712749481201, 0.29540297389030457, 1.0335222482681274, -0.23576930165290833, -1.0566074848175049, 1.2811880111694336, -0.08715003728866577, -1.5320069789886475, 0.0305654164403677, 0.19338686764240265, 0.24455736577510834, 0.5903634428977966], [-0.3292432129383087, -0.19684633612632751, 1.010338544845581, 0.03647014498710632, -0.8492467999458313, -1.626407265663147, 0.038007091730833054, -0.18616844713687897, 0.16532385349273682, 0.02824455499649048, 0.07680830359458923, -0.515903651714325, 0.5303344130516052, 1.1166471242904663, -0.20358514785766602, -1.7798320055007935, -1.0071700811386108], [-1.2636102437973022, 0.795203685760498, 0.2331790328025818, 0.12406880408525467, -1.197574496269226, -0.873348593711853, -0.5505883097648621, 0.17256726324558258, 0.4580818712711334, -0.23269833624362946, 1.0528441667556763, -0.7050731778144836, -0.8426900506019592, -0.894250750541687, 0.13723962008953094, -0.015459218062460423, -2.265214204788208], [1.0620183944702148, 0.2541928291320801, 0.47679439187049866, -0.3140919506549835, 0.2329506278038025, -0.13292814791202545, -0.2787180542945862, 0.6418682336807251, -0.6937391757965088, 0.03153371438384056, -1.9808214902877808, 1.1330946683883667, 0.3440456688404083, 1.3717992305755615, 0.30027154088020325, 1.0412957668304443, 1.2189568281173706], [0.07374751567840576, -0.7211993336677551, 2.0797009468078613, -0.7674718499183655, 0.40043312311172485, -0.7897769808769226, -0.37709176540374756, 1.399548053741455, 0.9594228267669678, -0.12923647463321686, 0.41229864954948425, -1.1018978357315063, 0.7036407589912415, 1.1609764099121094, 2.4913153648376465, 0.9289519786834717, -3.142446517944336], [-1.174099087715149, 0.215431809425354, 1.8812756538391113, -0.1297011822462082, -2.257324695587158, -0.6509461998939514, 0.6683498620986938, 0.748893141746521, -1.513954758644104, 0.5067983269691467, -0.5179545283317566, 0.20056022703647614, 0.9327617883682251, -0.8384921550750732, 0.34487026929855347, -0.7924221754074097, 0.11689654737710953], [1.4263761043548584, -0.0021641631610691547, -0.3518741726875305, -0.15757280588150024, 0.6350029706954956, 1.5134145021438599, 2.113875150680542, 0.08837643265724182, -0.9712628126144409, -0.014153169468045235, 1.019189476966858, 2.314485788345337, -2.9448482990264893, 0.045779719948768616, 0.7095686793327332, 0.023625588044524193, -0.105087049305439], [0.8448766469955444, 0.04530424624681473, -0.2455160766839981, -0.7361897826194763, -0.3247828483581543, 0.09409549087285995, 1.3260592222213745, -0.08034860342741013, -0.36301836371421814, 0.7146816253662109, -1.0858590602874756, 0.7373117804527283, -0.1553204506635666, 0.45488277077674866, -0.09750929474830627, -0.08358236402273178, 0.33548328280448914], [0.12273474037647247, -1.2487835884094238, -0.5890515446662903, 0.21539613604545593, -1.4750181436538696, 0.4665755331516266, -0.09924469143152237, 1.0688912868499756, 1.0088236331939697, -0.3473338484764099, -1.064194679260254, 1.186141848564148, 0.1531716138124466, 0.7869256734848022, -0.8139356374740601, -0.2127772718667984, -0.0018204232910647988], [0.592352032661438, -0.9511141777038574, -0.07405100762844086, 1.1085959672927856, -1.69887375831604, 0.4956910312175751, -0.6065864562988281, -0.6031466126441956, 0.46986427903175354, -0.10596451163291931, -0.49169206619262695, 0.17353370785713196, 0.18547911942005157, -0.2525077164173126, -0.6006685495376587, 0.1207730770111084, -0.4648703932762146], [-0.515649676322937, 0.33723533153533936, -0.6803679466247559, 0.39719292521476746, 1.7036123275756836, -0.8283073306083679, 1.1808561086654663, -1.9841638803482056, -0.3197345435619354, 0.005779807455837727, -0.7866586446762085, -0.6672114729881287, -0.4606526792049408, -0.5401111841201782, -1.9602165222167969, -0.3312872648239136, -1.3533656597137451], [0.4625494182109833, 1.4185799360275269, -1.8716013431549072, 1.2532113790512085, -2.6725120544433594, 0.13258302211761475, 0.25415992736816406, 0.6019459962844849, -0.9032628536224365, -0.3224353492259979, 0.6748717427253723, 0.7174187302589417, 0.18699350953102112, -1.2047003507614136, -0.15827317535877228, 0.024025006219744682, -0.3762587606906891], [-0.20986975729465485, -0.28324586153030396, 2.4259960651397705, 0.7517513036727905, 0.05989212915301323, -0.5443370342254639, -1.5685948133468628, 0.7460247278213501, 0.06252943724393845, -0.43457159399986267, -0.6235505938529968, -0.1340562403202057, -0.6172282695770264, 0.40293270349502563, 0.8742372989654541, 0.0457548163831234, 0.2706126272678375], [-0.9283052086830139, 0.510671079158783, 2.141568183898926, -1.7678635120391846, 0.7105457186698914, -0.2323724776506424, -0.8198704719543457, 0.0022183305118232965, -0.24359650909900665, 1.231634259223938, -0.0970495268702507, -0.35361990332603455, 1.1553115844726562, 0.7476928234100342, 0.350987046957016, 0.24784207344055176, 1.3252224922180176], [-0.38416287302970886, -0.05104135349392891, -0.8822731375694275, 0.9365904927253723, -0.3069625496864319, -0.2065611481666565, 0.36412543058395386, -0.4247918426990509, -2.6281137466430664, 1.0298229455947876, 1.619065523147583, -0.3887825906276703, 0.290250301361084, -0.46440088748931885, 0.16429780423641205, 0.6708344221115112, -0.9831477403640747], [0.15798619389533997, -0.25106415152549744, 0.5122841000556946, 0.6686651110649109, 0.4496587812900543, 0.5029097199440002, -1.9411619901657104, 0.4558161795139313, -2.0103533267974854, -0.35807526111602783, -0.6210422515869141, -2.5817365646362305, -0.44816356897354126, 0.8033974170684814, -0.9049484729766846, 1.5864967107772827, -1.5692696571350098], [1.2645703554153442, -0.41103559732437134, -0.10446583479642868, 1.0189049243927002, -0.41669705510139465, 0.06982956826686859, -0.14945033192634583, -1.1317944526672363, 0.47484052181243896, 0.008270379155874252, 1.3536851406097412, -0.6771520376205444, -0.4058530032634735, 0.3392905592918396, 1.8736239671707153, 0.18446038663387299, 1.510979175567627], [-0.181865394115448, 1.0932987928390503, 0.9613538384437561, 1.621229887008667, -1.8299264907836914, 0.4390215575695038, -0.7714962363243103, 0.46940767765045166, 0.056403983384370804, -0.2882467210292816, -0.2409178614616394, 0.13442911207675934, -1.391026496887207, 0.33160415291786194, -0.7932620048522949, -0.7796826958656311, 0.34952038526535034], [-0.42466625571250916, 1.9196847677230835, 1.62521231174469, 0.6534788608551025, -0.4749578535556793, 0.23974019289016724, -0.27131107449531555, 1.8445974588394165, -0.35494059324264526, 0.5384731888771057, -1.275043249130249, 0.20200742781162262, 0.4604227840900421, 0.4818231463432312, 0.19790776073932648, -0.3172798454761505, 1.3430514335632324], [0.21227706968784332, -0.32630541920661926, 0.9407180547714233, -0.36973249912261963, -0.47145920991897583, 0.8099803924560547, 2.144897699356079, -0.37876567244529724, -1.2111400365829468, -0.46123334765434265, 0.15463733673095703, -0.132207989692688, -0.41305869817733765, -1.1155256032943726, -0.22469083964824677, -0.4883621037006378, -0.002282700501382351], [0.012652464210987091, -1.2435003519058228, -0.30815958976745605, 0.00580196687951684, -1.6087322235107422, 0.8035106658935547, -0.4942459762096405, -0.6708781719207764, -0.35698944330215454, -0.04980028048157692, 1.1163543462753296, 1.4087920188903809, 0.6983590722084045, 0.3588976263999939, 2.546539068222046, 0.12217948585748672, 1.1053217649459839], [2.0919086933135986, -1.1786471605300903, -0.7382145524024963, -1.294899582862854, -0.21331679821014404, 1.4789730310440063, -0.1212603822350502, -0.8219693303108215, -0.4511314034461975, -0.8135418891906738, -1.0796514749526978, 0.2720441520214081, 1.129926323890686, -0.9000542759895325, 0.2790018916130066, -0.7487542033195496, 0.43901026248931885], [0.6444357633590698, 0.5500593781471252, -0.3588794469833374, 0.4612025022506714, 0.12456898391246796, 1.2491754293441772, 0.0880340114235878, 0.9182196855545044, 0.12133069336414337, 1.397518515586853, -1.0605450868606567, -0.5121601819992065, -0.7256587147712708, 0.26477184891700745, 0.3264026641845703, 0.5064160227775574, -1.0375714302062988], [1.952884316444397, -2.3895349502563477, 0.9118102788925171, 0.8088541030883789, -1.0475720167160034, -0.3617286682128906, 0.5421380400657654, 0.12167680263519287, -0.3940192759037018, 1.116212248802185, 1.1114445924758911, -0.5339808464050293, -0.35225263237953186, -1.7761412858963013, -0.5376853346824646, 1.18040931224823, 0.5607871413230896], [-0.988955557346344, -2.1235272884368896, 0.03609558567404747, -0.04691481962800026, -0.7536934018135071, -0.5965023040771484, 0.24336454272270203, 1.2202881574630737, -0.9403705596923828, 0.3610267639160156, 0.06486041843891144, -0.933431088924408, -0.3108633756637573, -0.18237584829330444, 0.659094512462616, -0.06758174300193787, -0.784675657749176], [0.1512213945388794, -0.6323046088218689, -0.9769165515899658, -0.4781627357006073, 0.24749404191970825, 0.853443443775177, 0.7945619225502014, -1.5880869626998901, 0.8422228693962097, 0.5927249193191528, 0.6802836656570435, -2.6819944381713867, 0.2764381170272827, 0.22241471707820892, 1.2934958934783936, 1.2384663820266724, 0.9393656253814697], [-0.9939084053039551, 0.36966073513031006, 0.7728206515312195, 1.8980399370193481, 1.599255084991455, -0.03391386941075325, 0.8033441305160522, -0.3598046898841858, 0.7147300243377686, 0.3407779633998871, 0.7026494145393372, 1.4831866025924683, 0.12587377429008484, 2.2669782638549805, 0.28043949604034424, 0.691950261592865, -2.2791635990142822], [0.13435323536396027, -0.40482842922210693, -0.08910949528217316, -1.5820763111114502, -0.7510983943939209, -0.6466275453567505, -0.47012150287628174, 0.896767258644104, 0.4754062294960022, -0.5132167935371399, 0.8062945008277893, 0.6722843647003174, -0.30773621797561646, 0.027238158509135246, -0.8013927340507507, 0.4680086672306061, -1.3719747066497803], [1.0740605592727661, -0.8720754981040955, 0.06911690533161163, -0.8594086170196533, 0.907784640789032, 0.36910614371299744, 1.5380762815475464, -0.08070003986358643, 1.2662992477416992, 0.17630240321159363, 0.049694862216711044, 0.69822096824646, 0.08751518279314041, -0.0807170644402504, -0.8050137758255005, 0.030631253495812416, 0.4858413338661194], [0.37939712405204773, 1.106433391571045, 0.062470823526382446, -0.2375156581401825, 2.014540910720825, -0.8627719879150391, 0.02500808611512184, 0.8057927489280701, -1.3361085653305054, -1.3190972805023193, 0.5439447164535522, -0.633139967918396, 0.057110436260700226, -0.21942000091075897, -0.9350017309188843, 0.5344023108482361, -1.3034067153930664], [-0.013564086519181728, -0.4971645474433899, -1.0990172624588013, 0.6573183536529541, 0.14057326316833496, 0.15285611152648926, -1.6147387027740479, 3.3688607215881348, -0.33631888031959534, -0.27641215920448303, -1.4176453351974487, -0.37608540058135986, 1.1516042947769165, 0.45909473299980164, 1.4011023044586182, 0.8935244679450989, 0.2893924415111542], [1.8407292366027832, 0.648952066898346, -0.3221132755279541, -1.4545350074768066, 0.3314827084541321, -0.19883061945438385, -0.07244736701250076, -0.8029950857162476, 0.011342370882630348, 0.3072107136249542, 0.3191632330417633, -0.7394288778305054, 0.0013199198292568326, -0.8608534932136536, 1.4812721014022827, 1.0065250396728516, -1.733702540397644], [-0.539385199546814, 0.8133668899536133, 0.4393976330757141, -1.0697318315505981, 0.5392512083053589, -0.5575554966926575, -0.42453432083129883, -1.4382333755493164, -0.16926805675029755, -1.38547682762146, -1.9632611274719238, -1.5148426294326782, 0.33484700322151184, -0.07706093043088913, -0.11778723448514938, -1.0299687385559082, -1.2627187967300415], [-0.9329664707183838, -0.6784767508506775, -0.08748193085193634, 0.9366878271102905, -1.2509130239486694, -1.0068293809890747, -1.0325794219970703, 1.9956823587417603, 0.6949044466018677, -0.6222553849220276, -1.3506443500518799, 1.165627360343933, -1.5360528230667114, 2.8350911140441895, -2.177469491958618, -0.8788658380508423, -0.03671732172369957], [-0.1569620966911316, 0.3435729146003723, 1.0883501768112183, -1.1386010646820068, -0.181303009390831, -0.7738745808601379, 0.004272807389497757, 0.17403262853622437, -0.04078790917992592, 0.900735080242157, -1.403592824935913, 1.6086732149124146, 1.8982954025268555, 1.6707735061645508, -0.7648230195045471, 0.1651775985956192, 1.8870726823806763], [0.04971809312701225, -1.7102185487747192, -2.0760622024536133, -1.214787244796753, 0.912612795829773, 1.5679230690002441, 1.253433346748352, 1.9109606742858887, -0.7229638695716858, 0.779552698135376, -1.3684475421905518, 0.9077648520469666, 0.16508592665195465, -0.11667994409799576, -1.0636311769485474, -0.456136018037796, -0.7191912531852722], [0.5825619697570801, -0.22069673240184784, 0.6659705638885498, 1.4870024919509888, 2.1634044647216797, 0.8791139125823975, -0.14604994654655457, 2.1084249019622803, -0.03680219501256943, 0.1283821314573288, -0.034476183354854584, -0.8696637153625488, 0.47413507103919983, 1.0700104236602783, -0.4144277572631836, -0.9692097902297974, 0.5705729722976685], [-2.409356117248535, -0.5283063650131226, -0.0718822181224823, 0.42013368010520935, 0.17508657276630402, -0.8129932284355164, 1.643509864807129, -0.02535644918680191, -1.0495651960372925, -0.49611544609069824, 0.37474727630615234, -0.13117475807666779, -1.5825586318969727, -0.7639747262001038, 0.3140711188316345, -0.9607661962509155, -0.14608126878738403], [0.771125853061676, 0.4809774160385132, -0.12767821550369263, 0.5858632326126099, 1.3260449171066284, -0.7473101019859314, -0.15956874191761017, -1.0265239477157593, -0.4361524283885956, 0.12566594779491425, -0.6491872072219849, -0.8643304109573364, 0.18746142089366913, 1.7671558856964111, 1.3672187328338623, -0.6287955641746521, -0.01086853165179491], [-2.1038565635681152, 0.7716619968414307, 0.3798363208770752, 2.4303030967712402, -0.8702312707901001, -0.4774262011051178, 0.012387339025735855, 0.2250160127878189, -1.7231820821762085, 0.761377215385437, 0.5673854947090149, 0.05755501985549927, -0.6526382565498352, 1.554000973701477, -0.003918012138456106, 0.05845918506383896, -0.7035043835639954], [-1.2733761072158813, -0.3728545904159546, -0.46156004071235657, 0.5939894318580627, -0.43115586042404175, 0.21621525287628174, 0.644717812538147, 1.0869758129119873, -0.7242885828018188, 0.7399866580963135, 0.5617531538009644, 0.08685054630041122, 0.6551619172096252, 0.1502140909433365, -0.7397257089614868, 1.8278939723968506, -1.942323088645935], [0.833093523979187, 1.4469969272613525, -0.3474952280521393, 1.124419927597046, -0.3598032593727112, -0.1429138332605362, 1.0456913709640503, -1.6198437213897705, -1.083904504776001, 0.9925855994224548, 0.8675583600997925, -1.4019110202789307, -0.8546690344810486, -0.02936575748026371, -1.235738754272461, -0.20610518753528595, 1.6081377267837524], [0.00010021633352153003, -1.3102883100509644, 0.014421685598790646, -2.3726885318756104, -0.17170631885528564, -0.5945119261741638, -0.01908191107213497, -0.8805809617042542, 0.1380959451198578, 1.935854196548462, -0.2447325885295868, 0.0724913701415062, 3.0348317623138428, -1.358420491218567, -0.8753867745399475, 0.5677648186683655, -0.9820725917816162], [-0.8766088485717773, 1.4604487419128418, 0.8584657311439514, 0.5862916111946106, 0.06839405745267868, -0.3874436914920807, -0.3491268754005432, 1.983627200126648, 0.09016074985265732, 0.20965458452701569, 0.25247153639793396, -0.2012881338596344, 0.251772940158844, -0.29984089732170105, 1.4091905355453491, -0.6206459403038025, -0.8322041630744934], [1.4687079191207886, 0.3779790699481964, -1.1665546894073486, -1.4468876123428345, -0.5545916557312012, -1.2701209783554077, -0.1042533665895462, 1.0924209356307983, 0.6095781326293945, 0.4214732348918915, -0.12967705726623535, 1.1282070875167847, 1.096721887588501, -0.9506237506866455, 1.90459144115448, -0.8929479718208313, 0.874467134475708], [-0.21728190779685974, -0.5981661081314087, 1.2554619312286377, -0.6295818090438843, -0.3911243975162506, -2.5045278072357178, 0.7943639755249023, -0.7148756384849548, -0.2757374048233032, 0.15118680894374847, -0.0756528452038765, -0.03376047685742378, 0.9128110408782959, 0.8772501349449158, 1.001647710800171, -1.8014781475067139, -0.27363473176956177], [0.8701488375663757, 1.2984843254089355, -1.2267347574234009, -1.0181374549865723, -0.7995090484619141, 0.9131824374198914, -0.6300493478775024, 0.854108989238739, -1.0640240907669067, 0.8936111330986023, 1.164452075958252, 0.5878745317459106, 0.2856481373310089, -0.36126989126205444, -0.6400194764137268, 0.22227515280246735, 1.2512922286987305], [-1.8212062120437622, -0.2850123345851898, -0.7015559673309326, 0.10227572172880173, 2.2383618354797363, 0.43668437004089355, 0.9042666554450989, -0.2961733341217041, 0.3517228364944458, 0.4454186260700226, 0.7668154835700989, 0.23270989954471588, 1.2189167737960815, -0.28459376096725464, -1.1310737133026123, 1.0716065168380737, 0.46912452578544617], [0.9237110018730164, -0.27928638458251953, -1.5270253419876099, 1.4258793592453003, -1.3720242977142334, -0.9700472354888916, -0.08651220798492432, -0.22653041779994965, -0.22724470496177673, 1.2986257076263428, -0.7229905724525452, 2.046630859375, 3.50754714012146, 0.18396775424480438, -0.9652897715568542, 0.15248054265975952, 0.1827632337808609], [-0.7977417707443237, 0.8944817185401917, 0.2508276402950287, -0.8078786730766296, -0.17690983414649963, -0.7333763837814331, 0.6350793838500977, -0.7667399048805237, 0.4650766849517822, -2.267385721206665, 0.7499201893806458, -1.0843050479888916, 0.29742181301116943, -0.31464114785194397, -1.0863405466079712, 0.9306862354278564, 0.36934301257133484], [-0.8774603605270386, -0.7649234533309937, 0.8795430660247803, -1.0292282104492188, 0.6203081607818604, 1.141474962234497, -0.3263441026210785, -0.04969743266701698, 1.3189321756362915, 0.864878237247467, -0.5448203682899475, 1.7180463075637817, 3.130524158477783, -1.3875004053115845, -0.11238131672143936, 0.3713441491127014, 1.7811706066131592], [-0.376189261674881, -0.18946197628974915, -0.5179789662361145, 1.4397282600402832, 0.10184110701084137, -1.849751591682434, -0.6997126936912537, 0.13876046240329742, 1.6613006591796875, -0.4984957277774811, 0.9224717617034912, -0.8618929982185364, 1.133378267288208, -0.16440829634666443, 1.2397865056991577, 0.8995611071586609, 2.4255154132843018], [-0.8368654847145081, -0.176443412899971, 1.1254361867904663, 1.5771114826202393, 0.381393164396286, -0.6970039010047913, 0.017378907650709152, -0.3306562602519989, -0.9524853825569153, 0.7588728070259094, 2.3278205394744873, -1.1940940618515015, 1.7209464311599731, 0.4240579307079315, -1.352251648902893, 0.5744659900665283, -0.46009787917137146], [-0.4404343068599701, 1.3079718351364136, -3.19085693359375, 0.5028396844863892, -0.04166191816329956, -0.6689162850379944, -1.312137246131897, -0.26546710729599, 1.0243653059005737, -2.1320736408233643, 0.6278902888298035, 1.2368847131729126, 0.9810196757316589, 0.5459458827972412, 1.0353164672851562, 1.3162671327590942, 1.2477794885635376], [-1.9156343936920166, 0.6416242718696594, -1.472494125366211, 1.191532850265503, 0.6262957453727722, -0.1776169091463089, 1.7552536725997925, -1.1369487047195435, 0.29167574644088745, -0.8128662109375, -0.004446420352905989, 0.853273332118988, -1.5229779481887817, 0.8489828109741211, -0.7155492305755615, -0.036424241960048676, 0.8256188631057739], [-0.09890338778495789, 0.6856327652931213, -0.8349494338035583, -0.11719784140586853, 1.3082507848739624, 0.7207825183868408, 1.8688105344772339, -1.3080947399139404, 0.5001748204231262, 0.20564189553260803, 0.39973005652427673, -0.5590124130249023, -0.7518155574798584, -0.3968774378299713, -0.03898623585700989, 1.3394864797592163, -1.1183762550354004], [2.5139260292053223, -0.16763930022716522, 0.7840338945388794, 0.09862610697746277, -1.7523763179779053, 0.17157314717769623, -1.7342661619186401, -0.35277700424194336, -0.9847797155380249, 0.8459171056747437, -0.009550088085234165, 0.796086311340332, 0.4789818823337555, 2.8670167922973633, 0.8292577266693115, -0.6056097745895386, 1.8138957023620605], [0.017223147675395012, -1.8655608892440796, -1.1119956970214844, -0.40847739577293396, -0.4825104773044586, -0.9924548268318176, -1.0719319581985474, 0.37112343311309814, -1.97383713722229, -0.285041868686676, 0.463162362575531, 0.14149905741214752, -0.19299814105033875, 0.7891638278961182, 1.6794363260269165, 0.24455566704273224, -2.240265130996704], [-2.3568477630615234, 1.2308964729309082, -0.4152188301086426, 0.09724406152963638, 0.22586780786514282, 0.39242827892303467, 1.60316002368927, 0.028617296367883682, -0.2070702612400055, 0.3841556906700134, 1.6406633853912354, 1.2340878248214722, -0.7229782938957214, -1.9003130197525024, -0.2911190688610077, -0.14926433563232422, 1.1857210397720337], [-0.7447620034217834, -1.752382516860962, 1.1296331882476807, 0.1795102208852768, -1.1687783002853394, 1.8901009559631348, 0.7663275599479675, 1.1914403438568115, -1.0850154161453247, 1.3221768140792847, 0.3860202729701996, -0.7598376274108887, 0.5910763740539551, -0.2407076358795166, 0.6169119477272034, -0.4252161681652069, 0.07155503332614899], [-0.1757723093032837, -1.3321696519851685, 0.5265426635742188, -0.083835668861866, -1.5833321809768677, -0.07332393527030945, -2.135444402694702, -0.03166508674621582, 0.08744506537914276, 0.0006818842957727611, 0.6638262271881104, 0.31049299240112305, -0.037194713950157166, -0.8542172908782959, 0.1661999523639679, -0.9803377389907837, -0.7738566994667053], [-0.4881350100040436, 2.7100648880004883, -0.7570801973342896, -1.1202226877212524, -0.37564322352409363, -0.9253199100494385, 0.5487583875656128, -0.9687950611114502, -0.27769070863723755, 0.5112515687942505, -1.0806772708892822, 0.3820967674255371, -0.05813765525817871, -0.1322949230670929, 0.689346432685852, 1.6630641222000122, 0.08413521945476532], [0.31475192308425903, 0.4466995298862457, -0.31765666604042053, -1.541496753692627, -0.29700902104377747, 0.054614074528217316, 0.6985260248184204, 1.039924144744873, 0.6971381306648254, -0.07306655496358871, 0.1974080353975296, 0.22025345265865326, -0.22995272278785706, -1.4056633710861206, -0.24121679365634918, 1.4112086296081543, -0.007551269605755806], [0.18773193657398224, -0.8534470200538635, 1.9397058486938477, 1.3051830530166626, -0.0241862703114748, 1.9085252285003662, 1.7747914791107178, 0.22764019668102264, 2.535921573638916, 0.35300856828689575, 0.8281842470169067, -1.9758919477462769, -1.4079378843307495, -0.32371020317077637, 1.0217145681381226, 0.8496138453483582, -1.5917948484420776], [-1.4832854270935059, -0.7973127365112305, 0.9766104817390442, -1.899107813835144, 0.7581436634063721, -1.3011094331741333, -0.9803263545036316, 0.029946299269795418, -1.3968263864517212, -0.5437248349189758, 1.632850170135498, 0.029086759313941002, 0.7829283475875854, 0.43817704916000366, -1.5163016319274902, 1.0547888278961182, 0.4550527334213257], [0.861361026763916, 0.012331102974712849, 0.6496002078056335, -1.4289095401763916, 0.5346763730049133, 1.2270077466964722, 0.47771313786506653, -1.3652859926223755, -0.804260790348053, -0.2498723566532135, 0.31632715463638306, -0.6716439723968506, -0.31461119651794434, -1.1870161294937134, -0.03003891184926033, 0.41168761253356934, -0.15116754174232483], [0.6663179397583008, 1.4090927839279175, -0.7783311605453491, -1.8582302331924438, 0.3162248432636261, 0.7136398553848267, 0.7606456875801086, -0.0969168096780777, -1.778254747390747, 0.5856579542160034, -0.2357817143201828, 0.7133811712265015, -0.27996188402175903, -0.11202249675989151, -0.062149569392204285, -0.6658243536949158, -0.4165601134300232], [-0.20347586274147034, -1.1841896772384644, -0.7773958444595337, -0.9200253486633301, 1.5318987369537354, -0.06271268427371979, -1.1725833415985107, 0.0632200539112091, -0.05450217053294182, 0.23186710476875305, -2.0117146968841553, 1.9389458894729614, -1.3430393934249878, 0.6064058542251587, 0.014133777469396591, 1.2852892875671387, 0.8359373211860657], [-1.4021830558776855, 0.5416314005851746, 1.6554572582244873, 0.5634120106697083, 2.138014554977417, -0.6250085234642029, -0.7853149771690369, -1.3574308156967163, -2.7606449127197266, 0.1852370649576187, 0.8281280994415283, 0.5236588716506958, -0.39400285482406616, 0.913386881351471, -0.30323582887649536, 0.28486698865890503, 0.5854309797286987], [2.186095714569092, 0.4420097768306732, 0.47297337651252747, -0.35180675983428955, 0.23673474788665771, 1.5870757102966309, -1.3213952779769897, -0.29316529631614685, -0.5600780844688416, -0.1524772047996521, -0.36891114711761475, -0.32723715901374817, -1.6639524698257446, -1.9640260934829712, 0.35887017846107483, 1.8452367782592773, -0.18392927944660187], [0.32334038615226746, -0.11687145382165909, -1.2086256742477417, 0.445069283246994, -0.26238206028938293, -0.9632891416549683, 0.08353716135025024, -0.30476275086402893, -0.8601678609848022, -0.7115743160247803, -0.04697059094905853, 0.2792225778102875, 2.006934404373169, -1.4137591123580933, 0.7881448268890381, 0.07649324089288712, 2.335597515106201], [-0.25093966722488403, 0.5313761234283447, 2.0578536987304688, 0.16642023622989655, -0.4554475247859955, 1.7143810987472534, 0.06485015153884888, -0.4451848268508911, 1.1550273895263672, 0.7549257874488831, -2.8598697185516357, -0.45393434166908264, -0.015635527670383453, -0.5480533242225647, -1.5483981370925903, -1.1044296026229858, -0.6542211771011353], [1.2761503458023071, 1.3800276517868042, -0.9304360747337341, 1.1350857019424438, 1.3331100940704346, 0.15999656915664673, -0.8675675988197327, -1.7548837661743164, -2.5123825073242188, 0.6115716695785522, 0.45093658566474915, -1.0585544109344482, 1.2520921230316162, -1.8989936113357544, -0.22709113359451294, -2.1834073066711426, 0.6024608016014099], [0.5339161157608032, 1.4659961462020874, -0.3571798801422119, 0.2518408000469208, -1.0175652503967285, -0.00011165442992933095, -0.38993948698043823, 1.617219090461731, 1.293511986732483, 0.3447559177875519, -0.8280248641967773, 0.36552250385284424, 0.5893420577049255, 0.6932491660118103, -0.382779061794281, 0.06418110430240631, 0.17804881930351257], [-0.8265806436538696, 0.8822168707847595, 1.729993462562561, -0.5634770393371582, -0.05811340734362602, -0.2746487259864807, 1.1388241052627563, -1.0160552263259888, -0.09434328228235245, -1.4252278804779053, 0.3366079032421112, 0.6763116121292114, 0.9637736082077026, 0.04151638597249985, 0.4737449586391449, -0.3648182451725006, 1.963637113571167], [-0.2634654939174652, -0.313190758228302, -1.7426872253417969, 0.48700135946273804, 1.642155647277832, 0.9344305992126465, 0.2958102226257324, 0.007219544146209955, 0.5510737299919128, 1.143412709236145, -0.4561559855937958, -1.2107239961624146, 0.33600661158561707, 0.5140392780303955, 1.45905339717865, 0.19631865620613098, -0.7282319068908691], [0.10342710465192795, 0.9559435248374939, -0.8086644411087036, -0.6926156878471375, -0.772763729095459, 1.2365305423736572, -0.7599909901618958, -0.009451404213905334, -1.0146459341049194, -0.8975378274917603, 0.1590937227010727, 0.2153690606355667, 0.4315764605998993, 0.06510352343320847, -0.5933701992034912, 0.965718686580658, 1.1620187759399414], [-0.12782864272594452, -1.5009326934814453, 1.2137763500213623, -1.1103932857513428, -0.8838528990745544, -0.17788314819335938, -0.8862730264663696, -0.03791067749261856, 0.9196889400482178, -0.6589508652687073, 0.051232483237981796, 0.36309266090393066, -0.684579074382782, 0.07406499981880188, -0.5323559641838074, 0.8769583702087402, 0.03612363338470459], [0.23817746341228485, -0.14916527271270752, 0.6498920321464539, -1.0469138622283936, -0.6061874628067017, 1.1077802181243896, -0.8362034559249878, 0.13362452387809753, -0.7096979022026062, 1.8496417999267578, -0.49390628933906555, -0.06590906530618668, -1.0806901454925537, 1.0586798191070557, 0.1860658973455429, 1.8122235536575317, 0.6720791459083557], [-0.34788477420806885, 0.30437344312667847, -0.004530320875346661, -0.6931992173194885, -0.7113840579986572, -0.17052505910396576, 0.37093600630760193, -0.2109820693731308, -0.10212061554193497, 0.0883064717054367, 0.47982436418533325, -0.7464056611061096, 1.2562450170516968, 0.08261837810277939, -1.061501145362854, 0.6040266752243042, -0.30632284283638], [0.5638190507888794, -1.1378602981567383, 0.45761099457740784, -0.06923023611307144, -0.12429165095090866, -0.08779381215572357, -0.32239845395088196, -0.9233136177062988, 1.1484391689300537, -1.5931556224822998, -1.237263798713684, -0.8138220310211182, -0.3121040165424347, -0.2132168412208557, 0.7111421823501587, 0.39017200469970703, 0.8746827244758606], [-1.2819335460662842, -1.1565706729888916, -0.8941420912742615, 1.4870485067367554, -0.7825596332550049, -1.1883500814437866, -0.49135541915893555, -0.07514245063066483, -1.5546234846115112, 0.7419178485870361, -0.20508448779582977, -1.094720482826233, -0.23288863897323608, -0.5862424969673157, 1.198104977607727, 0.020494384691119194, 1.0568082332611084], [1.2502199411392212, -0.787230372428894, 0.22392341494560242, -0.04819020256400108, 1.3595049381256104, 1.6301932334899902, -0.7506672143936157, 0.09690805524587631, 2.814575433731079, 0.009510235860943794, -0.46354642510414124, 0.6262606382369995, 0.5535620450973511, 1.4760208129882812, 0.890417754650116, 0.1021440252661705, -0.4169364273548126], [1.5489356517791748, -2.126917600631714, -1.496310830116272, 0.9742264151573181, 0.3060210943222046, 0.767873227596283, -0.17234359681606293, 0.09576359391212463, 0.34197095036506653, -1.0646491050720215, -0.24116356670856476, 0.6020494699478149, 1.6524200439453125, -0.07577241212129593, -1.263839602470398, -0.04156671091914177, 0.13840892910957336], [-0.6367340683937073, 0.9358275532722473, -0.5262043476104736, -1.531691312789917, 0.8823389410972595, -0.27222147583961487, -0.1845715194940567, -0.9966833591461182, -1.3447270393371582, 0.5916457176208496, 0.1336068958044052, -0.7375980019569397, 1.900020956993103, -0.9795507788658142, 1.4210357666015625, 3.39583683013916, -2.2205722332000732], [-0.5564044117927551, -0.46089789271354675, -0.45291122794151306, -1.8505682945251465, -0.482321172952652, 0.19620223343372345, -1.167248010635376, 1.0273550748825073, -1.042362928390503, 0.9481615424156189, 0.40844085812568665, -0.567091166973114, 1.3945189714431763, 0.18393269181251526, 0.5231418609619141, 0.2898521423339844, 1.9070329666137695], [-1.188514232635498, -0.5892805457115173, 0.21681031584739685, -0.42798787355422974, -0.3484264314174652, 0.541312575340271, -2.2012922763824463, -0.9897902607917786, 0.6010698080062866, 1.6759519577026367, 0.22185546159744263, 0.43824535608291626, -0.3741481304168701, -1.1459286212921143, 1.2948898077011108, 0.3328898847103119, -0.08719480037689209], [-0.34271085262298584, 0.9468286633491516, 1.2348190546035767, 0.10456424951553345, -0.5080775618553162, -1.8638683557510376, 0.07151734828948975, 0.1964605152606964, 2.026050567626953, -0.7096524834632874, -0.84027498960495, -0.6466071009635925, -0.3449493944644928, 0.2602926790714264, 0.2729673385620117, -0.3359576463699341, 0.5889419317245483], [-1.4591776132583618, 1.533945918083191, 1.2089340686798096, 0.3647613227367401, -0.8901666402816772, -1.00780189037323, -0.33231908082962036, -1.2572568655014038, -0.056458890438079834, -1.1079365015029907, -0.035551898181438446, -1.1572597026824951, -0.17201955616474152, -0.1459505707025528, -0.08075971901416779, 1.05091392993927, -1.5085891485214233], [-0.2362286001443863, 1.0685254335403442, 0.1581648737192154, 0.044751446694135666, -0.5026317834854126, 0.5527536869049072, -0.11738285422325134, 0.9788142442703247, -0.1714184433221817, -1.2332408428192139, -0.36158818006515503, -1.6690874099731445, 2.5996670722961426, 0.28160232305526733, 0.42442283034324646, 1.7837706804275513, -2.228633165359497], [-0.1736503541469574, -0.08491484820842743, -0.4160989820957184, 1.1336227655410767, 1.3922264575958252, -0.6193355321884155, 0.3088110089302063, 1.5837894678115845, 0.0003781006671488285, -0.04852079227566719, 0.7178764939308167, -0.13566800951957703, -0.3255574405193329, -0.5542929768562317, -1.341863751411438, 0.7600593566894531, -0.6969107985496521], [-1.3024941682815552, 2.779130458831787, 0.5558731555938721, 0.10713133960962296, 1.154832363128662, -0.7480404376983643, 0.36456596851348877, 0.2956486940383911, -1.4212841987609863, -3.266430616378784, 0.38078930974006653, -1.4100229740142822, -0.4145768880844116, -0.2978580594062805, 0.8429321646690369, -0.905677855014801, -0.3277958035469055], [-2.6044704914093018, -0.051875434815883636, 1.4282864332199097, 0.5041796565055847, 0.08984063565731049, 1.2718791961669922, -0.815757155418396, -0.9365329146385193, -1.0589243173599243, 0.3841485381126404, 0.3725281059741974, 0.3933146595954895, -0.9823645949363708, -1.3136489391326904, 0.6453107595443726, 0.6556123495101929, -0.6804744005203247], [0.7106640934944153, 0.3293512165546417, -0.3935435712337494, 1.1160751581192017, 0.3622385859489441, 0.76301109790802, 0.19618985056877136, -1.247331976890564, 0.045587506145238876, -0.29136595129966736, -3.014782428741455, -0.9505956768989563, -1.1708823442459106, 1.0573303699493408, 1.043352484703064, -1.6560425758361816, -0.36777088046073914], [1.0514708757400513, 0.9620435237884521, 0.9601574540138245, 0.666227400302887, -1.696520209312439, 0.800494909286499, 1.1663178205490112, -0.9377752542495728, 0.8734560608863831, -0.1961708664894104, -3.0353429317474365, 0.47375205159187317, 0.16332821547985077, -0.7011796236038208, 0.2867605984210968, 1.3389313220977783, -0.129135400056839], [0.9462199211120605, 1.046858549118042, -0.4851895868778229, -0.8546726703643799, 0.9083194136619568, 0.5060709714889526, 0.6000659465789795, 0.043945133686065674, 0.16667619347572327, -0.40342891216278076, -1.666593074798584, -0.23399800062179565, -0.18682508170604706, -0.9830226898193359, 1.0253783464431763, -0.09313801676034927, 0.4362567663192749], [0.9027559161186218, 0.9185585379600525, 1.7172430753707886, 0.8285260796546936, -0.4313686192035675, -1.4152321815490723, 0.5882902145385742, -0.046937495470047, 0.9813225269317627, -0.6142476201057434, 0.8347190022468567, 0.2503683865070343, 0.6029084324836731, 0.8934450149536133, 0.7179871797561646, -0.7435137629508972, 2.352984666824341], [-0.5963243246078491, -0.10348326712846756, -0.5102642178535461, 0.363407701253891, 0.3988744914531708, 1.923423409461975, -0.2608776390552521, -1.8137050867080688, -0.8706943392753601, -0.3592754006385803, -0.9424593448638916, 2.0943920612335205, -0.3146860897541046, -1.2026335000991821, 0.9078708291053772, 0.05818253383040428, -0.9505605697631836], [-1.087989091873169, 0.7620225548744202, -0.9736064672470093, 0.7146031260490417, -0.6621243357658386, -0.8199167251586914, -0.9060863256454468, 0.08349400013685226, 1.2720458507537842, 0.43183568120002747, -1.252341628074646, 0.3619893789291382, -0.9651728272438049, -0.6010984182357788, -1.0373626947402954, -2.2541849613189697, -0.31297147274017334], [0.2396879643201828, 0.8998527526855469, -1.1637600660324097, -0.099242202937603, 0.47697770595550537, 0.7124998569488525, -0.7660337686538696, -1.1183706521987915, 1.7073349952697754, 0.34588178992271423, 0.3179640769958496, 0.9328866600990295, -0.9649426341056824, -0.5445596575737, 0.4521101117134094, 0.9675278663635254, -1.6620453596115112], [-1.0011398792266846, 1.8661394119262695, 1.8535971641540527, 0.8220295310020447, 1.3947093486785889, 1.2690644264221191, 1.7087514400482178, -0.5627294778823853, 0.7720924019813538, -0.20871707797050476, 0.9396753907203674, 0.17567920684814453, 1.2783825397491455, 0.7975015044212341, 1.2430018186569214, 1.130467414855957, -0.7665237784385681], [-2.8170721530914307, -0.5445161461830139, 0.547359049320221, 1.053244709968567, -1.100658893585205, 0.4100615978240967, -0.587201714515686, -0.4641391932964325, 0.3533548414707184, 0.05494050309062004, -2.070197820663452, -1.0019887685775757, 1.0937047004699707, 0.38132137060165405, 0.3303987681865692, -0.6597543954849243, 0.30661067366600037], [0.7486066222190857, 0.7366723418235779, -1.1370527744293213, 0.28713199496269226, 0.25598740577697754, 1.6184556484222412, 2.1166419982910156, 0.6010643839836121, -1.286515474319458, 0.27948257327079773, 0.04440204054117203, 0.4407474994659424, -1.2827973365783691, 1.6770368814468384, 0.1501399278640747, 0.05636530742049217, -0.28540167212486267], [-0.5621939301490784, -0.29874804615974426, -0.20682547986507416, 0.951789915561676, 0.27709493041038513, 1.4141641855239868, -0.5367159843444824, -0.785183846950531, -0.5959543585777283, 1.3391518592834473, -1.4829546213150024, 0.8398464322090149, 0.9093202352523804, 0.7899539470672607, -0.8504437804222107, 0.23687544465065002, -0.3794712424278259], [0.42915746569633484, -0.423377126455307, 1.3686699867248535, -0.4182635247707367, 2.0953962802886963, -1.2546700239181519, 0.1382787823677063, -1.5253934860229492, -1.3363354206085205, 0.6567021608352661, -1.388023018836975, 1.8553828001022339, 0.15325693786144257, 0.0056481873616576195, 0.6869458556175232, -0.4109478294849396, 0.4866477847099304], [-0.5671637058258057, 0.3013017475605011, 0.40655142068862915, 0.08603723347187042, 2.0716710090637207, 0.17649513483047485, 0.7116835117340088, 0.6263860464096069, -1.7075375318527222, 0.12446234375238419, 1.5106201171875, -0.4097953736782074, 1.6995245218276978, -1.0562161207199097, 0.6954320669174194, -0.8842695951461792, 0.5704014897346497], [-1.8505494594573975, 1.8499770164489746, 0.7946015000343323, 0.8739621043205261, -1.728566288948059, -0.12445513904094696, 0.3897711932659149, -0.6930869817733765, -0.9794094562530518, 2.101825714111328, 0.043900277465581894, -0.9959725737571716, 0.7956855893135071, -0.0726483091711998, 0.483447402715683, 0.17984582483768463, 1.1210200786590576], [-0.4944019615650177, 1.6941442489624023, 0.8802629113197327, 0.766071081161499, -1.2656512260437012, 0.34810107946395874, 0.5169592499732971, -0.21464183926582336, -0.24461284279823303, 1.021445631980896, -1.2076891660690308, 0.45469236373901367, -1.806484341621399, -0.5260049104690552, 0.030542992055416107, -0.19688260555267334, 0.7900124788284302], [2.0272374153137207, -0.7099835276603699, -0.485886812210083, 0.29497554898262024, -1.2286015748977661, -0.37677061557769775, -0.8382651805877686, -0.33219853043556213, -1.3567373752593994, 0.014523995108902454, 0.39039915800094604, 0.9896479249000549, 0.48922407627105713, -1.4303834438323975, -0.9160318374633789, 1.1372277736663818, -1.334017276763916], [0.4601871073246002, -0.31360924243927, -0.27730944752693176, 1.1291598081588745, 0.5917037725448608, -0.7103298902511597, 1.0317304134368896, 0.4753546118736267, -0.16631031036376953, 1.2313649654388428, -0.37274816632270813, -0.8402473330497742, 1.0707781314849854, -0.07231316715478897, 0.13136127591133118, 0.2929700016975403, 0.8034980297088623], [0.35118648409843445, 0.3171759247779846, 1.304513692855835, 1.6132463216781616, 0.5836790204048157, 1.509366750717163, -0.13522639870643616, 1.0569417476654053, -0.2362254112958908, -1.621878981590271, -0.7235310077667236, -0.7507264018058777, -0.3760102093219757, -1.0096708536148071, -1.672743558883667, -1.9618823528289795, 0.3139989376068115], [0.6286670565605164, -1.187983751296997, 0.12777677178382874, -1.53147292137146, -0.08539470285177231, 0.5614787340164185, 1.5721473693847656, -0.4930596351623535, -0.7387358546257019, -0.4709518849849701, 1.0822919607162476, -0.273980975151062, -0.44920337200164795, -1.4240094423294067, 1.7662699222564697, -0.4712650179862976, 0.4748515188694], [0.37047645449638367, -0.5087930560112, -1.4388867616653442, -0.6760165691375732, 0.32779231667518616, -1.8743622303009033, -0.35112228989601135, 0.7781631350517273, 0.398459792137146, 0.9868377447128296, -0.04877983778715134, -0.023086750879883766, -1.1213911771774292, 0.0522795170545578, 0.31767207384109497, -1.4148844480514526, -0.6568583846092224], [-0.040375106036663055, -1.0508687496185303, 1.2216180562973022, -0.05090264976024628, 0.16112209856510162, -0.1113695427775383, 1.0970053672790527, -0.8247791528701782, 0.03609194606542587, -2.1847972869873047, -0.5405998826026917, 0.6334793567657471, 0.31613242626190186, 0.7922373414039612, 0.11171437799930573, -0.20130205154418945, -0.07739194482564926], [-0.5164373517036438, -0.034120943397283554, -1.8267076015472412, 2.3299341201782227, -0.8995358347892761, 0.1098465770483017, -0.30706650018692017, -0.8324835896492004, -0.5029832720756531, 1.0302069187164307, -0.2701612412929535, 0.1305629014968872, -1.093937873840332, 1.225455641746521, 0.2938806414604187, 1.0100311040878296, 0.43844324350357056], [-0.4228293299674988, -0.7897989153862, 0.5437244176864624, -1.9019591808319092, 1.9121079444885254, 1.8347288370132446, 0.5452291965484619, 1.9845155477523804, -2.186718463897705, 1.000476360321045, -0.05098545178771019, -0.7362506985664368, -1.9097721576690674, -0.21900860965251923, 0.9752036929130554, 0.32109156250953674, 0.993744432926178], [-1.0601791143417358, 0.33418238162994385, 0.3407956659793854, 1.8432402610778809, -0.648684561252594, -1.474676489830017, 0.9525296688079834, -1.0732107162475586, 0.28316646814346313, -2.1323344707489014, -0.34745919704437256, 1.164841890335083, -0.6108936667442322, -1.4641298055648804, -0.283700555562973, -0.16863258183002472, 0.805459201335907], [0.36565324664115906, 0.34340232610702515, -1.2132599353790283, 1.0695205926895142, -0.07347878068685532, -0.09571492671966553, 0.7050536870956421, 0.5134024620056152, 0.8524616956710815, -0.45918965339660645, 0.36666908860206604, -1.1259167194366455, -0.9667643904685974, -0.3168831169605255, 0.7524615526199341, 0.7217524647712708, 0.551105797290802], [0.6415954232215881, 0.10167370736598969, -0.39390039443969727, -0.47827816009521484, 1.240782618522644, -0.7003089785575867, 2.246331214904785, -0.5802127122879028, 0.7290546894073486, 0.33946898579597473, 2.498847246170044, -1.085432529449463, 0.17778615653514862, -1.144879937171936, 0.04428098350763321, 0.1240796223282814, -1.7182899713516235], [-0.9635859131813049, 0.13823740184307098, 1.1614183187484741, 0.3858298659324646, -0.18339432775974274, -0.07097230851650238, -0.9623476266860962, -1.1020466089248657, 0.8557361960411072, -0.13860423862934113, -0.5764410495758057, -1.1297146081924438, -2.006310224533081, -1.2586110830307007, -0.14249098300933838, 1.6966396570205688, 0.388582319021225], [0.34703758358955383, -0.4785759150981903, 1.1591861248016357, 1.8447247743606567, 0.9833380579948425, -0.0016217799857258797, -0.8924182653427124, -1.653160810470581, 0.8789411783218384, 0.09191270172595978, 0.08473234623670578, -0.3352673053741455, 0.6562419533729553, -0.0439629964530468, -1.5040855407714844, -1.2403703927993774, -2.2243261337280273], [-1.9393912553787231, -0.10894718766212463, 1.798661708831787, 2.2870872020721436, 1.3286734819412231, 0.47421616315841675, 0.7537203431129456, -1.3215184211730957, 0.4610652029514313, 0.120980404317379, -0.9196357131004333, 1.1008886098861694, -0.023558981716632843, -0.33528760075569153, 1.1506961584091187, -0.5468580722808838, -0.6317218542098999], [-0.5421469807624817, 0.7480573654174805, -0.19892291724681854, 0.8121473789215088, -0.015740135684609413, -0.20431570708751678, 0.5208083987236023, 0.5879275798797607, 0.37034735083580017, -0.9713485240936279, 0.7404657006263733, 1.7312270402908325, 0.2783231735229492, -0.21934042870998383, 0.67764812707901, 0.42027556896209717, 0.011324948631227016], [1.3790128231048584, 1.199615716934204, -0.6728701591491699, -0.7612230777740479, 0.6996892690658569, 0.1947343796491623, -0.11165227741003036, 0.7181466221809387, -0.536177933216095, 0.1978117674589157, -0.10899566113948822, 0.28442540764808655, 0.1901901811361313, -0.8358999490737915, 1.0120781660079956, -0.03266850486397743, 0.395323246717453], [0.724564790725708, -1.4788326025009155, -1.0742071866989136, -0.10431648790836334, 1.8848005533218384, 1.583187222480774, -0.5712703466415405, 0.3527471125125885, 0.14175933599472046, -0.5981260538101196, -0.7332779765129089, -0.33501332998275757, -1.4648007154464722, 0.27121827006340027, -0.557052493095398, -1.3942221403121948, -0.18854498863220215], [0.9820349812507629, -0.39964962005615234, 0.06267049908638, -0.4729765057563782, 1.3941328525543213, -0.3498889207839966, -0.9852607846260071, 0.9124800562858582, 1.9229408502578735, 0.5981247425079346, -1.3240883350372314, 0.8875723481178284, -1.8023912906646729, 0.058220960199832916, -0.6401271224021912, 0.6528476476669312, 0.5435072779655457], [1.5096635818481445, -0.14437665045261383, 1.740875005722046, 2.0911405086517334, -0.13950499892234802, -1.5750077962875366, -0.1548669934272766, -0.02710164524614811, 0.9930633902549744, -1.7390680313110352, 1.554893970489502, 0.5582031011581421, 1.4335390329360962, -0.6269014477729797, 0.547672688961029, 0.8683976531028748, 0.7574360370635986], [-1.1423814296722412, -3.023632287979126, 1.0602034330368042, 1.3118910789489746, -1.024818778038025, -2.624298095703125, 0.07024577260017395, -0.04408826306462288, -0.567263126373291, 0.47355833649635315, -0.5331898927688599, 0.17679211497306824, -1.6647543907165527, -1.0879864692687988, -1.4686678647994995, -1.8075345754623413, -0.8204652070999146], [-0.5000324845314026, -0.5942423343658447, 0.25559988617897034, -2.361877918243408, -0.9186246991157532, -1.5680913925170898, -0.8503826260566711, 0.4579164683818817, 0.11031205952167511, -0.24566231667995453, 1.8655767440795898, 0.07860836386680603, -0.8186231851577759, 0.42051270604133606, 0.49421823024749756, -0.3922417163848877, 0.8937388062477112], [0.828564465045929, 0.10333125293254852, -1.812193512916565, -0.8524054288864136, -0.26332888007164, 0.03378647193312645, -0.20887787640094757, 1.0855555534362793, 0.07417749613523483, -2.02063250541687, -0.44199249148368835, -0.28248530626296997, 0.26545509696006775, 0.6332933306694031, -0.5101439952850342, -0.8355097770690918, -0.905256450176239], [1.8837026357650757, -1.0421416759490967, -0.9798193573951721, 0.3784632682800293, 0.4161500632762909, -1.213224172592163, -1.5355727672576904, 1.0979218482971191, -0.654373288154602, -0.12574590742588043, 0.11617663502693176, 0.011995933018624783, 0.9226637482643127, 0.2301388382911682, 0.674088180065155, -0.016787342727184296, -0.08820203691720963], [3.1873619556427, 1.5014879703521729, -0.35851016640663147, -0.26154670119285583, -2.7630832195281982, -0.8186841607093811, 0.7534160017967224, -0.34990933537483215, -0.9758512377738953, -0.6146430969238281, -0.17098140716552734, -0.2800523042678833, -1.3667943477630615, -1.3045989274978638, -1.6763861179351807, -1.0633114576339722, -0.8199712634086609], [0.6859561204910278, 0.9306933879852295, -1.0908912420272827, 0.8800855278968811, -0.7371138334274292, 0.3064427077770233, -0.19573266804218292, 0.2491283267736435, -0.06589788943529129, -2.74979829788208, -1.0796817541122437, 2.5533862113952637, 1.1320642232894897, -1.8291529417037964, 0.8903322815895081, 2.797473669052124, -1.0726642608642578], [-1.1004196405410767, -1.2280147075653076, 0.34919971227645874, 0.8711294531822205, -0.5613874197006226, 1.7638511657714844, 2.6525931358337402, 1.0882617235183716, -0.18943530321121216, -0.13161134719848633, 0.9814509153366089, 0.7899640202522278, -0.8829993009567261, 0.5227462649345398, -0.8049383163452148, 0.48704585433006287, 0.387190580368042], [0.9966776371002197, 0.0398738756775856, -0.8985860347747803, 0.2435232251882553, 0.9678515791893005, -1.7659766674041748, -0.4802946448326111, 0.849565863609314, -0.34831881523132324, 1.1311100721359253, -0.36374998092651367, -1.5867706537246704, 0.6475204229354858, -0.34577587246894836, 0.8178255558013916, 0.7922643423080444, -0.6756525635719299], [-0.995785117149353, 0.8156706690788269, 1.646633505821228, -0.8684008717536926, -0.3337424397468567, 0.24517324566841125, 1.5434203147888184, -0.11051125824451447, 0.7668682932853699, 0.5719707608222961, -0.24384529888629913, 0.3529928922653198, -0.016306651756167412, -0.5125163793563843, -0.17643393576145172, 0.9791708588600159, 0.48600679636001587], [-0.8757725954055786, -1.2937513589859009, 2.0300965309143066, 2.062396287918091, -1.2243839502334595, -1.877354383468628, 0.5078569054603577, 1.2364355325698853, -0.6753401160240173, 1.0625767707824707, 0.8849847316741943, 1.2198302745819092, -1.8845922946929932, 0.2728787958621979, -0.5400642156600952, -0.05726555362343788, -0.5522294640541077], [0.4598604440689087, 0.016822675243020058, 0.8199974894523621, 0.12318162620067596, -1.4334137439727783, -0.6659080386161804, 1.2125855684280396, 0.6104195713996887, 0.5141969323158264, -1.4284679889678955, 0.8431618213653564, 0.5942990183830261, 0.5801973938941956, 0.4508373439311981, 0.02120261639356613, -1.8237417936325073, 0.36157920956611633], [0.05009079724550247, 0.9591216444969177, 1.0166329145431519, -0.5965908765792847, -1.526330828666687, 1.0560599565505981, 1.9268995523452759, -1.570486307144165, -2.378887414932251, -0.5636863708496094, -0.032382797449827194, 1.659374713897705, -0.8694242835044861, 0.5442964434623718, 0.2639044523239136, -0.4847487807273865, 2.3764808177948], [0.34509700536727905, 0.46572378277778625, -0.2556964159011841, -0.20604091882705688, -1.9538815021514893, 0.6666141748428345, 0.42856794595718384, -0.8860538601875305, -0.9214760065078735, -0.2368575930595398, 0.21019193530082703, 0.49492549896240234, -0.18270990252494812, 0.12265077233314514, -0.17727798223495483, -0.668476402759552, -1.0732403993606567], [0.8703846335411072, 1.1073315143585205, 1.6339138746261597, -0.09518502652645111, 0.08010563999414444, -1.5460304021835327, 1.9511393308639526, -0.11704789102077484, 1.5208967924118042, -1.03775155544281, -0.48239317536354065, 0.039551325142383575, -0.4038962125778198, 0.20022188127040863, 0.7525282502174377, 1.2971912622451782, 0.40982186794281006], [-0.6474066376686096, 0.2344254106283188, -0.9996204376220703, -1.3679035902023315, 0.5414385199546814, 0.4049404263496399, 1.879442572593689, -1.5874409675598145, 1.686141848564148, 0.7422274947166443, 0.31104618310928345, 1.4377892017364502, -1.4594011306762695, 1.4489312171936035, 0.26196566224098206, -0.5905067324638367, 0.5104670524597168], [-3.0121309757232666, 0.9862467646598816, -0.021770142018795013, 0.9047086834907532, 1.4483141899108887, -0.12229011207818985, 0.3921491205692291, 0.33826932311058044, 2.70982027053833, 0.4271456003189087, 0.5551141500473022, 0.23358336091041565, 0.4287177622318268, 0.44906240701675415, -0.7504926919937134, -2.8131942749023438, 0.5812124609947205], [0.08019186556339264, 1.5837854146957397, 0.1705075353384018, -0.775012731552124, -0.6465869545936584, 0.5288921594619751, 1.0800553560256958, -1.5859594345092773, -0.817516565322876, -0.29511621594429016, -0.4241887032985687, 0.5514153838157654, 0.6455872058868408, 1.8543164730072021, -0.35544535517692566, 1.3485528230667114, -0.3535281717777252], [-1.3992623090744019, -0.8117607831954956, 0.43454188108444214, 1.0625001192092896, 0.7566335797309875, 0.5742968320846558, -0.38413146138191223, -1.2025492191314697, -0.8737190365791321, -1.0769734382629395, 0.3937142789363861, -2.196984052658081, 1.2720146179199219, 0.07912710309028625, 0.29862579703330994, -0.1991659700870514, -0.15021593868732452], [0.06019241735339165, 1.1240792274475098, 1.2383919954299927, -2.1221506595611572, -0.8294493556022644, -1.3617594242095947, -0.7498956322669983, -0.17401544749736786, -0.8120313286781311, 1.2370609045028687, -1.946742057800293, -0.2914017140865326, -1.3074707984924316, 1.0561891794204712, -0.25146007537841797, -0.5005087852478027, 0.3470454216003418], [0.5835900902748108, -0.6876113414764404, 0.4476785659790039, -1.0158600807189941, -0.6121467351913452, -2.043618679046631, 0.8257550597190857, -1.4998711347579956, 2.2985949516296387, 0.6378998160362244, -0.5100966691970825, -0.3020640015602112, 0.4272266626358032, -0.07100407034158707, 0.0648968443274498, 1.7932943105697632, 0.29757413268089294], [1.6995885372161865, -2.97125506401062, -0.9660574197769165, 0.6724646687507629, 0.8885934948921204, -0.2827633321285248, 1.021067500114441, 1.1392383575439453, 0.35912731289863586, -0.31822630763053894, 1.2822933197021484, 0.6171690821647644, 2.3324897289276123, -1.315466284751892, 0.9088564515113831, 0.48690953850746155, 0.05427729710936546], [0.7482836842536926, 0.08296801149845123, -0.6176902055740356, -1.0975725650787354, 0.6693191528320312, -1.9725028276443481, -0.30156081914901733, -0.4183388650417328, 0.09341540187597275, -0.8098875880241394, 0.08519833534955978, -0.7125077843666077, -2.098665237426758, -1.340131402015686, 0.40313205122947693, 0.86199951171875, -2.629337787628174], [0.5394948124885559, -0.23796024918556213, -1.7168766260147095, 0.8870434761047363, -0.341335654258728, -1.5023850202560425, -0.31379935145378113, -1.6061314344406128, 1.7430975437164307, 0.00017613853560760617, 0.2708771824836731, -0.6521514058113098, -0.957858681678772, -0.3048776090145111, -0.28780823945999146, 1.520934820175171, 1.215153455734253], [-0.5366827845573425, 1.6863298416137695, -0.004544130060821772, -0.036361854523420334, 0.18555717170238495, 0.09183841943740845, -0.5572168231010437, 1.7695064544677734, -0.3967791497707367, 0.607943594455719, 0.0226224847137928, -0.11345823854207993, 0.5728527903556824, 1.0117615461349487, 0.18335892260074615, -0.881329357624054, 0.09578634798526764], [0.44701048731803894, 0.05608534812927246, 1.5261482000350952, 0.2596197724342346, -0.7826589941978455, 1.583983302116394, 0.5192088484764099, 1.9684789180755615, -1.6757076978683472, 0.45884954929351807, 0.08293543756008148, 1.107521414756775, -0.17992734909057617, -0.0809159204363823, -0.5486364960670471, 1.2324389219284058, 1.3516318798065186], [0.14528299868106842, 0.05187186971306801, -0.17134180665016174, 0.3495630621910095, -0.7187727093696594, -0.3527957499027252, 1.2489384412765503, -0.875653862953186, -0.40565264225006104, 0.2251836061477661, -0.4751400649547577, -1.0411771535873413, -2.5611319541931152, -1.3543645143508911, 1.6103501319885254, 1.6770634651184082, 0.17313286662101746], [-0.07132947444915771, -0.5160271525382996, 0.9442589282989502, -0.04248267412185669, -0.7129702568054199, -0.29739540815353394, -0.49190202355384827, 0.0018870464991778135, -0.1982094794511795, -1.4634637832641602, -0.1905258446931839, 1.8178354501724243, 0.03177650272846222, 0.2573449909687042, -1.1398000717163086, -2.1073920726776123, -0.18662472069263458], [-0.37959542870521545, -0.4000737965106964, 0.5492790341377258, 0.547432541847229, -0.20927399396896362, -0.432855486869812, 0.6594722867012024, 1.415682315826416, -0.3611992299556732, 0.50111985206604, -1.7031562328338623, -0.7050609588623047, -1.814765214920044, 0.8262677788734436, 0.427245169878006, 0.2362724393606186, -1.2050715684890747], [0.15079139173030853, -0.630036473274231, -0.15933489799499512, 0.5617012977600098, -0.48732441663742065, 1.2018431425094604, -0.8632159233093262, 0.13902662694454193, -0.6286166310310364, 0.8593059182167053, -1.1793075799942017, -0.0017393749440088868, 0.12904572486877441, 0.036720048636198044, -1.194754719734192, -0.8778355717658997, 1.6015169620513916], [-2.2770540714263916, -0.6935442090034485, 2.0665969848632812, -0.3110836148262024, 0.37891995906829834, 0.5129480361938477, 1.0496710538864136, -0.16826486587524414, -0.06768542528152466, -0.38495761156082153, -1.148364782333374, -0.03487805649638176, 0.8674243688583374, 0.27482834458351135, -0.5969306230545044, -0.5391346216201782, -0.28997039794921875], [-1.172318935394287, -1.000313639640808, -0.24089209735393524, 1.002629280090332, 1.8011285066604614, 0.19252370297908783, -0.7454013228416443, 0.8908112645149231, 1.7136791944503784, -0.010891996324062347, -1.0387681722640991, 0.46622464060783386, 1.4807618856430054, -1.0057204961776733, -0.772781491279602, 0.7627590298652649, 0.9789528250694275]], "dtype": "float32"}, {"__nd__": [[0.7517682313919067, -1.2447307109832764, -0.8898921012878418, 1.2150115966796875, -0.5142170190811157, 2.2211403846740723, -1.0908058881759644, -1.5053073167800903, -1.0318922996520996, 0.3491758406162262, -0.6653367280960083, 0.1471964716911316, -1.1731070280075073, -0.27952781319618225, -0.5720765590667725, 0.4910784065723419, 1.4725590944290161, -0.012309049256145954, 1.6379963159561157, 0.7347962260246277, 0.758620023727417, 0.03304809704422951, 0.5337004065513611, 0.8673864603042603, -0.7567200064659119, 0.19734911620616913, 1.0351046323776245, -2.110926628112793, 1.6424603462219238, -0.4976000487804413, -0.7973206639289856, 0.5829237699508667, 0.8403589725494385, -0.4266032874584198, 1.28932523727417, 1.0251457691192627, 1.2684584856033325, 0.16484493017196655, -0.2596849501132965, -1.777059555053711], [-0.95025634765625, 1.43221116065979, -0.964673638343811, 0.9347403049468994, 0.7698957920074463, -0.28790283203125, 0.05853165313601494, -0.5344265699386597, -1.0292763710021973, 0.3211336135864258, 0.43854430317878723, 0.18404948711395264, 1.0244399309158325, -1.4244093894958496, -1.3405500650405884, -0.03254639357328415, 0.19492436945438385, -1.1707994937896729, -0.753931999206543, 1.3019226789474487, -1.0493613481521606, 0.7233384251594543, 0.517725944519043, 0.42599061131477356, 1.7731999158859253, 0.4995504319667816, 0.1424192637205124, -0.44563600420951843, 0.8100824356079102, 0.4006313979625702, -1.3007198572158813, -0.42406484484672546, -2.3218178749084473, -0.04252740740776062, -0.9520106911659241, 0.45670071244239807, -2.9140281677246094, -0.2837013006210327, -0.2151675671339035, 1.3661670684814453], [-0.535663366317749, 1.460194706916809, -0.3511468470096588, -1.0329399108886719, 0.133297398686409, -1.030542016029358, -0.8667909502983093, 0.8347142338752747, 0.23100508749485016, -0.2029881775379181, 0.8901842832565308, -0.027692362666130066, -0.7128937244415283, 1.320441484451294, 0.9009165167808533, 0.7840867638587952, 1.700610876083374, 0.9033493399620056, -0.7180024981498718, -1.3931742906570435, -0.8490121364593506, -1.5012277364730835, -0.09286174178123474, 0.15120790898799896, 0.057842545211315155, -0.18431176245212555, -0.511423647403717, 0.7180240154266357, -0.241222083568573, -1.0372542142868042, -0.322317510843277, -2.230609893798828, -0.7782315015792847, 0.8266834020614624, 1.4985167980194092, 0.24925728142261505, -0.3823336362838745, -0.9270389080047607, -1.3306472301483154, -0.7501299977302551], [1.8827691078186035, 0.3245971202850342, -0.24760808050632477, -0.9311960935592651, -0.8507178425788879, -0.3345405161380768, 0.9643553495407104, 0.3867676556110382, 1.8094518184661865, 0.4833623766899109, -0.2447044402360916, 0.24632765352725983, 0.8893489837646484, 0.15559369325637817, 0.9532855749130249, 0.638704240322113, 1.0219870805740356, -0.8531901240348816, -0.3162374496459961, -0.42113977670669556, 0.0008179505239240825, 0.24989908933639526, 0.06656362116336823, -0.06369805335998535, -1.0483936071395874, -1.0705859661102295, -0.7609732747077942, -0.7317548394203186, 0.9951313734054565, -0.6694193482398987, -0.43658146262168884, 0.7917978763580322, -0.160880908370018, 0.45095300674438477, 0.33016541600227356, 1.5482699871063232, -0.9666051864624023, 0.6166700720787048, -1.2867722511291504, -1.492270588874817], [0.17069830000400543, 2.2247209548950195, 1.5552517175674438, -0.34294623136520386, -1.4077839851379395, 0.5559893846511841, 0.7475802302360535, 0.5514954924583435, 1.0132665634155273, 0.38288071751594543, -0.25512009859085083, -1.4752535820007324, -0.6299599409103394, -0.16062739491462708, 0.6344116926193237, -1.0492109060287476, 1.4476304054260254, 0.23843178153038025, 0.4084795415401459, 0.09500810503959656, -1.262266755104065, 1.770185112953186, -1.6739310026168823, -0.31919601559638977, -2.0634725093841553, -0.1374158412218094, 0.6731351017951965, -1.9101927280426025, -0.9117056727409363, -0.3860248029232025, -2.111910581588745, -1.6824201345443726, -0.7008106112480164, 0.021083058789372444, -0.6691625118255615, 0.9634902477264404, 0.6333982348442078, -0.03714606538414955, 0.553451657295227, -0.25355541706085205], [1.8581668138504028, -0.12145669013261795, -1.7927768230438232, -0.04020436480641365, 1.3378883600234985, 0.30711114406585693, -0.1101938933134079, 1.3411767482757568, 1.1533117294311523, -0.9200854301452637, 0.9909682273864746, 1.3303532600402832, 0.7522832751274109, -0.36893224716186523, 1.2016836404800415, -0.9705439209938049, -0.10877209156751633, -0.06888999044895172, -1.1070811748504639, 1.029558777809143, -0.8691768646240234, -1.4136576652526855, 1.4800903797149658, 1.3106932640075684, -0.07047997415065765, -0.5400931239128113, -0.9717991948127747, 1.2804069519042969, -0.10544468462467194, -0.35209545493125916, -0.5623354911804199, -0.33395928144454956, 1.0048351287841797, 0.8929377198219299, -0.4302785098552704, -0.13845232129096985, 0.37100204825401306, 0.48302820324897766, -0.7047296166419983, 0.30742737650871277], [-0.5098802447319031, 1.3314995765686035, -0.5509357452392578, -0.6298122406005859, -2.4266161918640137, -0.15320052206516266, 1.1510547399520874, -0.17854717373847961, -0.9982208609580994, 1.8500299453735352, -0.3150753080844879, 0.4810599982738495, -0.8052583932876587, 0.33619558811187744, -1.6346383094787598, 0.20366689562797546, 1.5944386720657349, 0.045480720698833466, 1.668235182762146, -0.5302495360374451, -1.3062695264816284, -1.2875937223434448, -0.3959469795227051, 1.3472445011138916, -1.046093225479126, -0.20433761179447174, -0.4717649221420288, 0.27199602127075195, 1.4740989208221436, -1.0990955829620361, -0.41483187675476074, 0.4727809429168701, -0.06640761345624924, -0.4091591238975525, 0.8210264444351196, 0.3561722934246063, -1.3513628244400024, 0.24141332507133484, -0.5812827944755554, -0.13328059017658234], [1.6344308853149414, 1.1624752283096313, -0.6044178605079651, 0.5023573637008667, -0.9941309690475464, -0.8033850789070129, -0.38828569650650024, 0.837668240070343, 0.13840463757514954, 1.476904273033142, -0.4418814480304718, -0.1259845495223999, -1.316207766532898, 0.7370789647102356, -0.9395331740379333, 0.055169932544231415, -0.4491601288318634, 0.6677548289299011, -1.0913208723068237, -0.03293924406170845, -0.5220287442207336, 0.5145071148872375, 0.6976012587547302, -0.6982389092445374, 0.4634469151496887, -0.5594741106033325, -1.1726458072662354, -0.743934690952301, 0.34395739436149597, 1.0118162631988525, 1.4337130784988403, -0.082705058157444, -0.2510279417037964, 0.8834314942359924, -0.43572527170181274, -0.7329265475273132, 0.5032098889350891, -0.030072122812271118, 1.5960147380828857, -2.089318037033081], [-2.9059367179870605, 0.28975436091423035, -1.782890796661377, -0.24818243086338043, -1.170291543006897, 1.0221331119537354, -0.563974142074585, -0.22793124616146088, -1.2237768173217773, -2.0263333320617676, 0.08210679143667221, 0.31562891602516174, -0.1239808052778244, 1.204364538192749, 0.7143197655677795, -1.4666293859481812, 0.4835571050643921, -0.5686311721801758, 0.24154528975486755, 0.029125381261110306, -1.5711300373077393, 0.08902251720428467, -1.2714359760284424, 1.4241218566894531, -0.7352908253669739, -0.7340415716171265, -0.03903748095035553, -0.42654475569725037, -0.07462894916534424, -1.0957121849060059, -0.6948946714401245, 0.03532792255282402, 2.045464038848877, -0.6938930153846741, 0.8748745322227478, -0.15228436887264252, 1.3337570428848267, 1.0949639081954956, 1.0004736185073853, -0.9377421140670776], [-1.5831209421157837, -0.7252787351608276, 0.6686571836471558, -0.9029943346977234, 1.4252021312713623, -1.2456364631652832, 0.5118481516838074, -0.5652053356170654, -2.0555739402770996, -0.29686757922172546, 0.6744588613510132, 0.40070560574531555, -0.9092170000076294, -0.38942471146583557, 1.2000524997711182, 1.0610132217407227, 0.3241012394428253, -0.7535954713821411, 1.3991838693618774, -0.493593692779541, -0.322172075510025, -0.6934340596199036, 1.895156979560852, -0.10916696488857269, 1.3745641708374023, -0.012899326160550117, -0.008819276466965675, -1.369749903678894, 0.05985714867711067, 0.6414411664009094, 0.5768036246299744, -0.20150721073150635, -0.9771310687065125, -0.8963820338249207, 1.9728773832321167, 0.826106607913971, -1.9501069784164429, -0.6222324967384338, 0.49729716777801514, -0.5907728672027588], [-1.6805990934371948, 0.816647469997406, 1.9612983465194702, 0.40736863017082214, 0.8494654297828674, -0.8290639519691467, 1.063817024230957, -1.2562403678894043, 0.46280133724212646, -0.34901556372642517, 0.42850032448768616, 1.853275179862976, 1.1271367073059082, -0.4390190839767456, 1.1627466678619385, -1.7173196077346802, 0.8137496113777161, 0.059942349791526794, 0.3306439220905304, 0.8627902865409851, -1.5143529176712036, -0.033146459609270096, 1.4329791069030762, 0.16519615054130554, 0.1401076316833496, 1.9281225204467773, -0.020764518529176712, 0.41768959164619446, -0.6394497156143188, 0.9370629191398621, 0.5484355688095093, 1.6333012580871582, -0.6283282041549683, -0.9960830807685852, -0.5659677386283875, -0.9558935165405273, 1.1236941814422607, 1.8769450187683105, 1.7389942407608032, 1.6290265321731567], [0.12071507424116135, -0.6198928952217102, -0.8609408736228943, 1.0849963426589966, 0.7044486999511719, -0.4581029713153839, -0.3203001320362091, 0.9576126337051392, -0.9079229831695557, 1.3346474170684814, 1.2942150831222534, 0.9547840356826782, 0.12223608046770096, -1.5767041444778442, 0.4521196782588959, -1.0230106115341187, 2.1732864379882812, -0.1813662350177765, -0.41468966007232666, 0.6623515486717224, 0.7421074509620667, -0.9050238728523254, 0.930346667766571, -1.129090428352356, -0.055100347846746445, -0.6387676000595093, 2.2660534381866455, -2.1689493656158447, 0.37894901633262634, 0.44021326303482056, -0.37602654099464417, 0.8812097311019897, -0.04046272858977318, 0.7117900848388672, -1.3519580364227295, 0.3010736107826233, 1.5836087465286255, 1.6326043605804443, 0.5159464478492737, 0.7088121175765991], [-0.1332467794418335, 0.10341636091470718, 0.18687179684638977, 1.1185892820358276, -0.22455213963985443, -0.6354383230209351, 1.6113710403442383, 1.152276635169983, -0.3319331109523773, 0.2677118182182312, -0.6402392983436584, 0.5074594616889954, -0.12489902228116989, -1.6406947374343872, 0.4175894856452942, 0.23919837176799774, 1.4191025495529175, 0.13936790823936462, -0.8173190951347351, 0.4123702347278595, -0.4888336956501007, 1.73726487159729, 1.4160045385360718, -1.0068484544754028, 1.7501453161239624, 0.6854535341262817, -0.021333931013941765, 0.21963386237621307, -1.1433004140853882, -0.8089001178741455, 1.3061096668243408, -0.6749663352966309, -0.9190307259559631, 1.0762149095535278, 0.8723868131637573, -0.35821324586868286, -0.876663863658905, -0.583652675151825, 0.39182502031326294, -0.46733492612838745], [0.7955909371376038, -0.7148467898368835, 1.3547099828720093, 0.3233489990234375, 1.1417407989501953, 1.055384635925293, -1.1295506954193115, -1.4587688446044922, 1.1646870374679565, 1.7893818616867065, 0.5767728686332703, 0.4661194682121277, 0.3038395941257477, 0.31415584683418274, 0.3156762719154358, -0.6237136125564575, -2.04947829246521, -1.6609511375427246, -0.39215564727783203, -0.24805080890655518, -0.4391050636768341, 0.685634195804596, -0.834585964679718, 0.887144923210144, 1.4531430006027222, 0.283429890871048, 0.39635372161865234, -1.302340030670166, 0.21244512498378754, -0.29878485202789307, 1.6475231647491455, -0.2204553782939911, 0.5759599208831787, 0.23781746625900269, -1.3947638273239136, -0.410592645406723, 0.06710582226514816, -0.48140615224838257, -0.4231131672859192, -0.24736525118350983], [0.3309309482574463, 1.097326636314392, 0.13352598249912262, -0.3272302448749542, -0.6194488406181335, 1.4039030075073242, -0.04961134493350983, -1.0418431758880615, 1.2441500425338745, 1.6274161338806152, -1.0924192667007446, 0.19094553589820862, 0.3645937442779541, 0.901493489742279, 0.42436307668685913, -0.8246992826461792, 0.46861422061920166, 0.2786760926246643, 0.1658175140619278, -0.3596189320087433, -0.04152418300509453, 1.6234877109527588, -0.364244669675827, 1.0646692514419556, -0.8653087019920349, -1.7273313999176025, 1.850407361984253, -1.149965524673462, 1.0023164749145508, 0.9470067620277405, -0.5524682402610779, 0.7590875625610352, -0.23423095047473907, 1.0497157573699951, 1.2675483226776123, -0.27345097064971924, 0.38057249784469604, -0.2194739729166031, 1.3808811902999878, -0.8942959308624268], [-1.2334091663360596, 1.4045621156692505, 0.39678671956062317, -0.004193175118416548, -2.86064076423645, 0.8372690081596375, 1.2027589082717896, -0.26466426253318787, 0.3167757987976074, -0.7117801308631897, -0.6620122790336609, -0.05002504214644432, -0.04734671115875244, 0.15326784551143646, -0.34804606437683105, -0.8046130537986755, -0.36925166845321655, -0.6188872456550598, 0.6009299755096436, 0.0012333375634625554, -0.9237218499183655, -0.17566634714603424, 1.0286248922348022, 0.3013192415237427, 0.6578823328018188, -0.8079103827476501, -0.014199456200003624, -0.35741230845451355, 0.12730902433395386, -0.194102481007576, -1.4798909425735474, 0.36098790168762207, 0.15524974465370178, 1.106751561164856, 0.003967560827732086, -0.03465474396944046, 0.3966637849807739, 1.0381531715393066, -0.16841629147529602, 0.6373521685600281], [-0.1477232128381729, -0.31424480676651, 0.6803494095802307, -0.6260019540786743, -0.5740088820457458, 0.5085625648498535, -0.08084635436534882, -0.5528470277786255, -1.5339844226837158, 0.9959069490432739, -0.4990726709365845, 0.4705316126346588, -1.1735233068466187, -1.3243939876556396, -0.7853416204452515, -0.9231297969818115, 1.0866128206253052, -0.42658770084381104, -2.411956310272217, 0.8183722496032715, 1.055841326713562, 0.8500057458877563, -0.421146422624588, 0.5686438679695129, -0.4563005268573761, -0.07017387449741364, -0.9298604726791382, -1.8095778226852417, 0.13664349913597107, 1.0248678922653198, -1.9729903936386108, -0.353422611951828, 0.9443642497062683, -0.24351713061332703, 2.469163656234741, 0.2984708845615387, 1.5109694004058838, -1.479568362236023, 0.38635390996932983, 0.8515580296516418]], "dtype": "float32"}, 32, 16, 16, 2], "e": {"__tup__": [{"__nd__": [[0, 2, 4], [1, 3, 5], [6, 8, 10], [7, 9, 11], [12, 14, 16], [13, 15, 17], [18, 19, 20]], "dtype": "int32"}, {"__nd__": [[-6.5064473152160645, 0.9713544249534607, -7.375246524810791, -3.047973871231079, 1.5167202949523926, -0.7212042808532715, 0.033633723855018616, -1.5232560634613037, -4.540825843811035, -8.004156112670898, -0.047680482268333435, -1.1497045755386353, 1.4670753479003906, 3.5523147583007812, -0.4940871298313141, 5.6551313400268555, -4.201069355010986, -2.922548294067383, 4.182301044464111, -1.4639084339141846, -4.411927700042725, -2.0954160690307617, 1.5792176723480225, 5.123884677886963, 4.791504383087158, -1.7665560245513916, -4.335779666900635, 5.517137050628662, 2.577552080154419, -2.7637271881103516, -0.6307165026664734, -2.3320953845977783, -2.531357765197754, -2.5455706119537354, 5.429275989532471, 2.2448768615722656, -13.799731254577637, -3.0929908752441406, -3.195728302001953, -2.9545557498931885], [3.3977465629577637, -1.7734192609786987, 0.531781017780304, -0.9357729554176331, 1.9799253940582275, -1.902758240699768, -2.0887913703918457, -2.806715726852417, 0.4112488925457001, 7.281982421875, 2.6386141777038574, 1.7396962642669678, 1.9424587488174438, 0.33021897077560425, -6.049797534942627, 2.100816488265991, -4.502322673797607, -3.4836292266845703, -0.935584545135498, -1.1893364191055298, 0.6308757066726685, -4.628360748291016, -4.1187028884887695, 2.5164363384246826, 1.665306568145752, 2.629971981048584, -3.3499820232391357, 2.8198468685150146, 4.218136310577393, -1.44368577003479, 3.8869407176971436, 1.9442721605300903, -0.18950502574443817, -2.540278673171997, -5.382969856262207, -0.2481415718793869, -5.431275367736816, -1.2657972574234009, -5.927814960479736, 3.5032498836517334], [-5.823775291442871, 8.314300537109375, -4.253663539886475, -7.246607780456543, -1.8201276063919067, 6.262991428375244, -1.656054139137268, -2.582501173019409, 7.797027587890625, -6.059490203857422, 1.0763204097747803, 2.070345401763916, 6.707845211029053, 7.55295467376709, 8.05410385131836, -6.3128461837768555, 5.129315376281738, -1.8452658653259277, -1.6284927129745483, -2.0501067638397217, -7.633053779602051, -0.29675790667533875, -7.312478542327881, 11.39503002166748, -8.350600242614746, -6.290926933288574, 1.6245219707489014, 2.253936767578125, 2.065958023071289, -5.01504373550415, -11.747940063476562, -2.7146646976470947, 3.0423529148101807, 1.375535249710083, 6.056349754333496, 2.6482608318328857, 0.5410131812095642, 1.5568608045578003, -3.682840347290039, 0.2671404480934143], [3.7894387245178223, 0.19995033740997314, 0.5140904784202576, -2.103902578353882, -0.752291202545166, -0.17691676318645477, 3.6024062633514404, 2.0454037189483643, 4.324216365814209, -0.18392066657543182, 0.35837388038635254, 2.368152379989624, 4.367563724517822, -2.0972485542297363, -1.389512300491333, -4.009268760681152, -3.3060619831085205, -0.3894868791103363, -3.484215021133423, 2.432011604309082, 0.4824599325656891, -2.7837729454040527, -0.5403532981872559, 2.271001100540161, -3.1309564113616943, -0.15814857184886932, -3.696674346923828, 6.590831279754639, -0.7860549092292786, 1.125889778137207, -3.1608011722564697, 3.280125141143799, 3.0047061443328857, 0.2370832860469818, -3.9140331745147705, -1.7813241481781006, 1.4688961505889893, 1.9650087356567383, -2.0588204860687256, 8.183510780334473], [3.045830011367798, -0.19916661083698273, -2.213108777999878, -1.4360251426696777, 0.3358481526374817, -5.866415977478027, 1.1966930627822876, 4.221826076507568, 2.2619986534118652, -2.7032597064971924, 1.4841818809509277, -1.6927258968353271, 1.5766111612319946, 2.8063971996307373, 1.2122209072113037, 4.2813873291015625, 1.713478684425354, 4.820801258087158, 5.380538463592529, -2.548828601837158, 2.825679302215576, -6.894895076751709, 2.4559333324432373, -5.465126037597656, -5.233284950256348, -1.2631409168243408, 0.772257387638092, 4.594945430755615, 0.9629478454589844, 1.4370945692062378, 1.365607738494873, 4.331365585327148, -2.475862741470337, -1.0690053701400757, -4.15506649017334, 1.3889888525009155, -0.4658564031124115, 5.6471967697143555, -0.6881414651870728, -1.009721279144287], [-6.789239406585693, -0.006388053297996521, 0.060802314430475235, 2.478980302810669, 0.5868821144104004, 1.5464835166931152, 0.5546863079071045, -4.651137351989746, -5.3309006690979, 0.31258243322372437, 0.0731591135263443, 3.3875932693481445, -0.3454073965549469, -2.454763412475586, -0.6088308691978455, -0.3478333353996277, 5.303769111633301, -1.4662355184555054, 3.2339437007904053, 1.6473690271377563, -2.7505533695220947, 0.13023927807807922, 2.5755257606506348, 3.0145022869110107, 3.0395829677581787, 3.7361884117126465, 2.3406991958618164, -1.816987156867981, 1.766946792602539, -1.9320757389068604, -0.5208790898323059, 0.6757614612579346, -1.8306794166564941, -2.1290485858917236, 4.252096652984619, 0.41772857308387756, -2.073647975921631, 0.19759510457515717, 0.3303893208503723, 1.1319663524627686], [-1.8316612243652344, -2.185126304626465, -2.5003373622894287, 3.9432177543640137, 2.5928587913513184, -0.48845258355140686, 2.0766921043395996, -1.433715581893921, -1.8866478204727173, -1.6694470643997192, 1.225071907043457, 5.446062088012695, 4.255279064178467, -5.149942874908447, 0.5720298290252686, -1.8865516185760498, 1.921971321105957, -2.06249737739563, 0.09686824679374695, 4.4809699058532715, 0.8952370285987854, -2.2808175086975098, 8.878809928894043, -0.5786128044128418, 4.407421588897705, 0.7642413973808289, 2.674011468887329, -0.5093393921852112, 2.187641143798828, 2.857450008392334, -0.9522044658660889, 6.749858856201172, -1.6669938564300537, 0.9328662157058716, -1.6311464309692383, -0.32189032435417175, 0.11778831481933594, 5.6485514640808105, 1.2400974035263062, 4.517870903015137], [0.29820671677589417, 0.24359770119190216, 1.4304250478744507, 1.2244586944580078, 0.5992783904075623, 3.617654323577881, 1.8654145002365112, -4.091556549072266, 6.294382095336914, -1.9336082935333252, -2.450146436691284, 1.0583431720733643, 6.652169227600098, 2.4507126808166504, -0.22211188077926636, -4.033297061920166, -6.273846626281738, 0.6701260805130005, 4.548787593841553, 2.0626707077026367, -1.613577961921692, 2.4353158473968506, -2.2987749576568604, 3.333195924758911, -3.854600429534912, 1.4652241468429565, 1.8938097953796387, 6.351327896118164, 0.36035680770874023, 1.8863965272903442, 1.7314468622207642, 7.456855297088623, 1.7086468935012817, -2.742360830307007, -6.236983776092529, -3.7411277294158936, 1.3622746467590332, 5.06534481048584, 3.879610300064087, 3.646796226501465], [0.33268529176712036, -3.2497034072875977, 5.520719051361084, -6.910756587982178, 0.026162272319197655, 0.5843314528465271, -0.058926329016685486, 0.28647416830062866, 4.1262054443359375, -4.422657012939453, 0.7865496873855591, -3.9987735748291016, 0.7109976410865784, 3.253192901611328, 4.419848918914795, 0.4461192786693573, -4.040362358093262, -0.12301990389823914, 1.4984933137893677, -4.090026378631592, 4.34986686706543, -2.554701089859009, -5.961608409881592, -0.902752161026001, -6.511135101318359, -2.2393698692321777, -0.40823987126350403, 1.243541955947876, -3.0451841354370117, -0.32513850927352905, -3.735124349594116, -0.9932671189308167, 4.387063503265381, -2.175405502319336, 0.378009557723999, 2.294004440307617, 3.6739180088043213, -0.814196765422821, -4.018903732299805, 2.7615413665771484], [-0.6055387854576111, 8.10982608795166, 1.559315800666809, -2.2824387550354004, -0.6893523931503296, -5.917291641235352, 4.772943019866943, 3.274794340133667, 4.671905517578125, -1.9054878950119019, -0.5088631510734558, 0.6094279289245605, 3.9727554321289062, -0.2353912740945816, 4.794363021850586, -2.149170160293579, 5.622361183166504, 2.9280898571014404, -3.723675489425659, 0.5085088610649109, -1.4239572286605835, 2.4726994037628174, 4.704100608825684, -4.554148197174072, -2.801725387573242, -3.0210182666778564, 0.8086483478546143, -0.4606678783893585, -1.33603835105896, 6.171195030212402, -4.782016754150391, 2.4393887519836426, -6.977736473083496, 2.808985471725464, 0.6139741539955139, 0.7147528529167175, -0.11087071895599365, 4.112926006317139, 5.295154571533203, 1.3748712539672852], [3.403482437133789, 0.5708389282226562, 1.4458534717559814, -0.3999347388744354, 4.324302673339844, -7.727670192718506, 0.8252203464508057, 4.1492414474487305, 4.18158483505249, 4.373648166656494, 4.368961334228516, 1.5495513677597046, 5.237480163574219, -1.6752984523773193, 0.8105188012123108, 0.7514182329177856, 2.5146310329437256, 2.3052449226379395, -2.581998109817505, -1.7213420867919922, 4.106544017791748, -5.583324432373047, 1.368589997291565, -6.536803722381592, -0.8918090462684631, -0.24555708467960358, 3.6899967193603516, 3.317429304122925, -0.02628900483250618, 3.009471893310547, 1.7954096794128418, 3.8080546855926514, -5.164198398590088, 2.173236608505249, -7.874836444854736, -0.6756426095962524, -0.36167094111442566, 4.288671493530273, -2.0853631496429443, 6.583321571350098], [4.332167148590088, -1.228861689567566, -5.890494346618652, -1.9008493423461914, 2.9476232528686523, -7.081314563751221, 4.683036804199219, 6.862832546234131, 5.545194625854492, -4.078437328338623, 2.1359164714813232, 4.491650104522705, 8.845137596130371, -1.7652772665023804, 3.261193037033081, 0.41906043887138367, -3.652628183364868, -2.464197874069214, -7.586478233337402, 0.8020038604736328, 0.9480537176132202, -3.4105591773986816, 6.0894999504089355, -2.6978442668914795, 3.8833932876586914, -3.8100390434265137, -5.485756874084473, 8.497127532958984, -0.4200437068939209, 1.194398045539856, 2.930131673812866, 5.638738632202148, -0.44043126702308655, 3.930896043777466, -4.150539398193359, -0.8234198689460754, -5.210443019866943, 4.580788612365723, -2.8665313720703125, 3.0403201580047607], [8.3919677734375, -4.171023845672607, -6.8206963539123535, 3.8803255558013916, -1.0599737167358398, 4.958075523376465, -1.636675477027893, 0.7551443576812744, 2.4579360485076904, 3.0188145637512207, -0.5384895205497742, 1.715125322341919, 1.7869516611099243, -1.8598082065582275, -1.7295289039611816, 2.0338432788848877, 6.861270427703857, 2.2538790702819824, 4.0666608810424805, 1.6404082775115967, 5.2592267990112305, -4.377655029296875, 1.6096782684326172, 0.9393231868743896, -5.59956169128418, -2.0367934703826904, 5.928577423095703, -0.08321782201528549, 6.233193397521973, -3.3353052139282227, -2.690321445465088, 4.847670555114746, 2.0213465690612793, 2.410022020339966, -1.4146238565444946, 3.037339925765991, 3.1773951053619385, 3.935957908630371, -5.5309367179870605, -1.7925139665603638], [-7.96534538269043, 6.165342807769775, 4.437039375305176, -1.4179649353027344, -6.140138149261475, -0.7920718789100647, 9.586575508117676, -1.6433862447738647, 2.8741672039031982, 2.505678653717041, 0.019546465948224068, 7.366941928863525, 6.255306243896484, -2.209390878677368, 3.875985860824585, -9.915833473205566, 9.586265563964844, -0.20912300050258636, 4.069933891296387, 1.3939520120620728, -5.7733564376831055, -1.271194577217102, 1.8016172647476196, 2.6211700439453125, -6.494616508483887, -0.3232404291629791, 5.699382305145264, 0.38436660170555115, 0.3493296802043915, 0.4590010643005371, -4.499474048614502, 9.774428367614746, 0.14926865696907043, -0.4942851662635803, -1.0969709157943726, -1.6380935907363892, 4.899117946624756, 10.933096885681152, 5.030036449432373, 8.744608879089355], [-4.497106075286865, -6.192311763763428, 2.632538080215454, -1.4957716464996338, 5.529561996459961, 7.062349796295166, -1.0862983465194702, -7.671725273132324, 4.231556415557861, -5.2376837730407715, 0.7107905149459839, 3.529733180999756, 7.45601224899292, 2.165754795074463, 5.096319675445557, -5.892122745513916, -10.08128547668457, -5.530866622924805, 3.1662466526031494, 1.166103482246399, -0.6333199143409729, 0.6299083828926086, -2.6854400634765625, 7.394102573394775, 0.3041034936904907, -0.04922451078891754, 2.9468533992767334, 2.9520089626312256, -0.5320123434066772, 0.2507694661617279, 1.0154324769973755, 6.075136184692383, 7.21240758895874, -3.4463021755218506, -2.433964490890503, -2.6956377029418945, 2.720501184463501, 3.108038902282715, 0.6453840136528015, 5.564290523529053], [-1.0684783458709717, 0.14542177319526672, -4.924802780151367, -2.834212064743042, -6.438363552093506, 0.38890090584754944, 4.497411727905273, -1.898843765258789, -0.8761199116706848, -2.614020347595215, -1.571720004081726, 0.3459097146987915, 3.2746946811676025, 1.5821653604507446, -3.4380128383636475, 1.8312945365905762, 0.11748144030570984, -4.088257312774658, 6.677810192108154, 0.4031842350959778, -3.1197292804718018, -2.6021575927734375, -3.4475631713867188, 4.462617874145508, -5.99254035949707, -0.6650863885879517, -5.131195068359375, 2.5843451023101807, 5.749680042266846, -4.466561317443848, -2.5498600006103516, 6.660886287689209, 2.5503556728363037, -6.16686487197876, -1.048552393913269, 4.590417385101318, -6.837971210479736, 5.063846588134766, -3.7222249507904053, -1.6599375009536743], [-10.557838439941406, -1.7559974193572998, -3.4291160106658936, 2.4766037464141846, 4.765199661254883, 0.8179741501808167, -1.2462347745895386, -2.9166347980499268, -6.651336669921875, -9.411697387695312, 1.0618906021118164, 0.39297008514404297, 2.6384706497192383, -0.9127182960510254, -1.718753457069397, -0.07474270462989807, -4.201995372772217, 0.7960390448570251, 4.095296382904053, 2.747789144515991, -0.4294136166572571, -3.207183361053467, 3.7852957248687744, 1.8434146642684937, 5.20406436920166, 2.955258369445801, 2.442194938659668, 6.794120788574219, -1.2807523012161255, 2.060870885848999, -1.9348459243774414, 0.9600268602371216, -1.7411158084869385, -3.034572124481201, -0.021801963448524475, -3.005577325820923, -3.1680588722229004, 0.850523054599762, 1.7656974792480469, 7.5473222732543945], [2.936462640762329, 3.6574935913085938, -6.37791109085083, -4.347121715545654, -5.738215923309326, 0.780906081199646, -1.1307704448699951, 6.70552921295166, 3.502960681915283, 6.530849456787109, 1.3423372507095337, 3.2809736728668213, -1.3413435220718384, 4.369493007659912, 4.15443229675293, -5.0435309410095215, 10.04820728302002, 4.592000961303711, -3.4656898975372314, -4.516922473907471, 1.2895656824111938, -5.828070163726807, -1.2590069770812988, 1.8870404958724976, -8.456199645996094, -11.86388874053955, 6.598602294921875, -2.296839475631714, 2.835453510284424, -0.5271053314208984, -4.146042346954346, 1.119330644607544, 5.387697219848633, 8.449396133422852, 4.761261940002441, -0.8250325322151184, 9.693323135375977, 3.232041358947754, 0.847586989402771, -2.6258559226989746], [1.2460020780563354, -4.445303440093994, -4.862302303314209, 3.385876178741455, 5.345524787902832, -2.3570525646209717, -2.493680953979492, -1.5417366027832031, -0.6884788870811462, -4.30735445022583, -1.6672334671020508, 1.2305543422698975, 2.7878313064575195, 1.8416216373443604, -0.6401841044425964, 4.237315654754639, -2.5697524547576904, 2.586644172668457, 0.4144400656223297, 1.3578603267669678, 3.0507164001464844, 0.9028370380401611, 3.4033477306365967, -2.3497464656829834, 1.8925670385360718, 2.604849338531494, -1.7159439325332642, 5.079006195068359, 1.6749606132507324, 2.865399122238159, 7.18529748916626, 5.82310676574707, -1.1068804264068604, -2.8213419914245605, -0.08690577000379562, -2.033518075942993, -1.3830859661102295, 1.9620367288589478, 3.4703874588012695, -4.480415344238281], [4.067018508911133, 4.087130546569824, 1.6116695404052734, 1.374305248260498, 4.670678615570068, -4.238853454589844, 2.1306562423706055, 2.6010916233062744, 5.673943996429443, 4.036526203155518, 0.3908575475215912, 3.174834966659546, 2.821843385696411, -1.87665593624115, 4.872509479522705, 0.31101033091545105, 1.527123212814331, -1.6721194982528687, -8.07568073272705, 0.03468674421310425, -4.387888431549072, 6.874301910400391, 4.360121726989746, -1.7771062850952148, 7.733194828033447, 0.5957690477371216, -2.136334180831909, -0.4083111584186554, -2.3758673667907715, -0.2035183161497116, 6.457625865936279, -3.5876221656799316, -6.26265811920166, 5.538595199584961, 0.6516589522361755, -1.2118881940841675, -6.034426689147949, -3.068103551864624, 1.110585331916809, -4.105831623077393], [-1.5140886306762695, 0.9005736708641052, 8.203683853149414, -1.0236680507659912, 2.6979808807373047, -3.136420249938965, -1.1324832439422607, 0.3301311433315277, 0.69147127866745, 2.1436173915863037, -2.0631682872772217, -5.050729751586914, -0.7186062335968018, -1.1111844778060913, -1.2709071636199951, 2.037511110305786, -2.421637535095215, 3.91848087310791, -6.729930400848389, -2.6873810291290283, 7.2023773193359375, 6.004251480102539, -3.092686176300049, -7.1696271896362305, 2.9626097679138184, 0.892051637172699, 3.061005115509033, 0.7206757664680481, -5.315903663635254, 5.967140197753906, 0.004803425632417202, -4.527807235717773, -5.7726664543151855, 2.5838303565979004, 1.9400748014450073, -2.8868374824523926, 0.8043426871299744, -7.512315273284912, 1.99955415725708, 5.228944301605225], [-4.959137439727783, -2.860802412033081, -3.481628656387329, -0.9373937845230103, -0.406522661447525, -0.2273533046245575, -1.0396109819412231, -1.5560671091079712, -7.620114326477051, -0.5419438481330872, 1.7505265474319458, 2.7785871028900146, -3.613206148147583, -0.037487518042325974, 0.9218548536300659, 0.17645302414894104, 0.6326698064804077, -3.8241755962371826, 1.8751115798950195, -0.3343389630317688, -1.1142377853393555, -3.5669360160827637, 4.563225269317627, 2.02785062789917, 3.9194693565368652, -2.9236581325531006, -0.3809625506401062, -6.095559120178223, 2.8379154205322266, 0.5251895785331726, 0.5808572173118591, 1.4316166639328003, 2.7099273204803467, -0.8949822187423706, 5.021315574645996, 1.4995933771133423, -0.6234636306762695, 0.9142208695411682, 1.5584914684295654, -3.38661789894104], [10.9664888381958, -7.626119613647461, 3.5827863216400146, 6.3937506675720215, 6.217179298400879, 2.1405274868011475, -2.683981418609619, -2.7818186283111572, 2.0551743507385254, 7.178990364074707, -1.8399159908294678, 1.8843200206756592, 0.5959063768386841, -7.3829827308654785, -2.381459951400757, 0.4091043770313263, -6.263194561004639, -4.108730792999268, -9.372437477111816, 5.057626724243164, 7.926937103271484, 8.071784973144531, 4.680471897125244, -4.205798149108887, 8.537189483642578, 1.9443573951721191, -0.1509137749671936, -7.2899980545043945, 0.7941554188728333, 6.098053932189941, 5.650562286376953, 2.8987162113189697, -0.44323447346687317, 3.3291513919830322, -3.197161912918091, -1.2261377573013306, 2.6051836013793945, -3.591200828552246, 1.5238770246505737, -0.8774341940879822], [1.0028109550476074, 1.902279019355774, -2.8479881286621094, 0.5951308608055115, -4.563943386077881, 1.3402338027954102, -0.28142231702804565, 2.1364495754241943, 2.444979667663574, 0.6813705563545227, 0.497420072555542, 1.6500592231750488, -0.43068042397499084, 1.5254393815994263, 2.5646231174468994, -1.610966682434082, 3.687530040740967, -0.5837448239326477, -0.986358642578125, -1.2558871507644653, -1.2379926443099976, -1.8528871536254883, 2.5808804035186768, -0.22348952293395996, 0.22050155699253082, -4.366250514984131, 1.9605432748794556, -3.418715238571167, 1.5246044397354126, -2.1556060314178467, -1.090944528579712, 0.4600728154182434, 1.8673607110977173, 5.184126853942871, 0.3384747803211212, 0.7953474521636963, 4.046746253967285, 3.6326258182525635, -1.148726224899292, -3.6587653160095215], [2.3584587574005127, -0.4030665457248688, 0.7188354730606079, -4.405925750732422, -3.7256507873535156, -0.3955785632133484, 1.6771419048309326, 0.2804415822029114, 0.02807200327515602, 0.06755584478378296, -0.5455822348594666, -1.308693766593933, -1.9532805681228638, 1.3636237382888794, 0.3451419174671173, 0.842308521270752, 2.702531576156616, -0.5262558460235596, 2.346673011779785, -1.091986894607544, 0.23567327857017517, -1.5239731073379517, -3.8985238075256348, 1.0537232160568237, -8.015952110290527, -0.9208322167396545, -3.841583013534546, -2.3116607666015625, 1.6743073463439941, -1.5039299726486206, -3.1974949836730957, 1.1806366443634033, 3.2060811519622803, -4.096447944641113, 2.190487861633301, 4.211997032165527, 0.9935336709022522, 0.876740038394928, -1.6536457538604736, -2.873326539993286], [-5.036709308624268, -0.3595341444015503, 0.08761239796876907, 0.29718291759490967, 0.49823230504989624, 3.0783796310424805, -1.8602890968322754, -5.183626174926758, -3.770359516143799, 1.8991912603378296, -1.2243390083312988, 4.67420768737793, -1.3864575624465942, 0.2842610478401184, 1.2726341485977173, -4.981748580932617, 4.4111504554748535, -0.5635281205177307, -2.1709866523742676, 1.9023481607437134, -1.614335536956787, 3.2495429515838623, -0.6400516629219055, 4.735885143280029, -1.3548479080200195, 0.6252589821815491, 1.1579346656799316, -4.703048229217529, 1.5528278350830078, 1.9873864650726318, -0.4492829442024231, 2.574068069458008, 3.556723117828369, -2.2469534873962402, 7.091376781463623, -1.7553004026412964, 5.865720272064209, 0.03285875916481018, 6.1956610679626465, -1.570650577545166], [3.3599395751953125, 3.3416547775268555, -1.1522438526153564, -2.5085079669952393, -1.8242743015289307, 1.706252932548523, -4.659877300262451, -1.5675697326660156, 1.313520073890686, 5.0919904708862305, 0.7946431636810303, -0.38127145171165466, -1.7922581434249878, 3.9195704460144043, -4.534350872039795, -0.9978217482566833, 0.5599812269210815, 2.8080434799194336, -1.237842321395874, -1.261336088180542, 1.252179503440857, -5.143573760986328, -4.082334041595459, 3.7747316360473633, -5.535597801208496, -1.9239072799682617, -0.16930736601352692, 0.7021015286445618, 4.206291198730469, 1.9216136932373047, -4.715879917144775, -0.6042033433914185, 1.0615733861923218, 0.3574713170528412, 0.021609898656606674, -0.5152804851531982, 3.3077409267425537, -1.3117485046386719, -1.9094688892364502, 1.9182575941085815], [-0.60689377784729, -5.939784049987793, 2.777510404586792, 2.989622116088867, 2.0832746028900146, 8.047201156616211, -7.756496429443359, -1.384094476699829, 1.6821227073669434, -5.895086288452148, 0.012214582413434982, -9.951213836669922, -1.9926775693893433, 3.5249409675598145, 0.6485173106193542, 1.3646416664123535, -6.685122489929199, 2.3821253776550293, 6.993603706359863, -2.137967586517334, 5.381792068481445, 2.3116543292999268, -10.514845848083496, -2.998713970184326, -4.086509704589844, 1.4405633211135864, 8.408720016479492, -1.3324893712997437, -4.573568820953369, -3.5603911876678467, -0.6972770690917969, -5.658456802368164, 4.187582492828369, -2.148871421813965, -7.267703056335449, 0.2830047607421875, 7.274310111999512, -1.561540126800537, -2.25944447517395, -0.5953586101531982], [-0.11531171202659607, -8.252608299255371, 1.8574086427688599, -1.5011041164398193, 8.017239570617676, -3.1136934757232666, 0.7994776964187622, 1.5080965757369995, -5.867016315460205, -1.0771441459655762, 1.27603280544281, -0.22209852933883667, -1.4591991901397705, -6.128928184509277, 2.950381278991699, 3.126711368560791, -0.3649407923221588, -3.041316032409668, -3.7928307056427, 0.9024235606193542, 4.57854700088501, 3.448983669281006, 0.1768370270729065, -2.8009016513824463, 3.4805526733398438, 1.8676152229309082, -2.0306506156921387, -2.939493179321289, -3.5844295024871826, 0.5860235095024109, 2.7176055908203125, -2.8476078510284424, 0.46624821424484253, -3.3261241912841797, 4.966109752655029, 2.4438347816467285, -3.906014919281006, -6.297817230224609, -0.4745737910270691, 0.02137770876288414], [-1.0209707021713257, -0.844818115234375, 5.010621547698975, -2.854179859161377, -0.7971491813659668, -0.03873270004987717, 0.7928734421730042, -3.626842975616455, -1.0684802532196045, 2.7176501750946045, 1.0443122386932373, 2.1306402683258057, -2.597554922103882, 1.1676371097564697, 2.091308355331421, -2.841980218887329, -2.2947192192077637, -2.8802411556243896, 2.839155673980713, -0.6422485113143921, -1.4212324619293213, -1.9313809871673584, 2.139392852783203, 1.6609132289886475, -0.7480942606925964, -1.1241756677627563, -0.7998903393745422, -6.318278789520264, 1.210307240486145, 3.539463996887207, 0.6931740045547485, 3.3889272212982178, 2.745739698410034, -2.2909610271453857, 0.8779951930046082, 0.32559794187545776, 3.3250162601470947, 2.2367982864379883, 3.6454858779907227, -0.6859330534934998], [-0.18028418719768524, 2.5497238636016846, -0.2681725323200226, 2.167210340499878, 0.21399910748004913, -2.1353797912597656, 2.798226833343506, 0.4390823245048523, 0.035985857248306274, -0.8535499572753906, 1.544042944908142, 1.1448642015457153, 2.884197235107422, -3.2334225177764893, -3.4529244899749756, -0.9405695796012878, -6.268725395202637, -3.885807514190674, -1.233734369277954, 3.250251531600952, -3.732825756072998, -1.1869347095489502, 4.5993571281433105, 0.04478103667497635, 7.050301551818848, 1.878846287727356, -3.7600080966949463, 2.64166259765625, -0.1011776253581047, 1.2882314920425415, 0.24810625612735748, 0.722034752368927, -3.4321086406707764, 0.8139165043830872, -7.099462509155273, -1.2017439603805542, -6.286195278167725, 1.781618356704712, -1.0400502681732178, 5.3582587242126465], [3.796846628189087, 10.039259910583496, -2.050105571746826, -1.7095431089401245, -4.988243103027344, -7.187241077423096, 5.128604888916016, 5.761923313140869, 3.0387587547302246, 6.7405009269714355, -0.10335920006036758, -0.30762988328933716, 1.2451720237731934, -1.3495802879333496, -2.7738521099090576, 4.562090873718262, 6.232700347900391, -0.1590999960899353, -4.860766410827637, -2.593402624130249, -3.511775255203247, 0.2075207233428955, 2.85489821434021, -2.8530313968658447, 3.636618137359619, -3.0457210540771484, -3.0384018421173096, 1.2654908895492554, 2.5595736503601074, -1.8706451654434204, -1.3698053359985352, -3.7905080318450928, -10.308990478515625, 6.651895523071289, 0.5365188717842102, 2.8096694946289062, -12.15185260772705, -2.3644461631774902, -5.092532634735107, -1.3879766464233398], [-2.2249772548675537, 5.8658552169799805, 2.625354528427124, -4.152261734008789, 0.1870681345462799, -5.833150386810303, 0.3685583770275116, 0.9109116196632385, 0.009321419522166252, -0.4938846826553345, -0.564595103263855, -4.243492603302002, -2.020834445953369, 5.4172587394714355, -1.0041269063949585, 4.474534511566162, 1.0485724210739136, 6.304553508758545, 3.759855270385742, -4.445590972900391, -1.533169150352478, -1.5582045316696167, -2.70864200592041, -1.9959306716918945, -3.967533588409424, 1.5096749067306519, -1.7375826835632324, 5.551480770111084, -1.2651889324188232, 1.9323149919509888, 0.8326101303100586, -3.1899642944335938, -6.177853584289551, -2.8999805450439453, 2.1238534450531006, -0.5682014226913452, -4.932506561279297, -2.8739776611328125, 1.8207162618637085, -0.9833777546882629], [8.484288215637207, -2.5431461334228516, -0.6526565551757812, -2.274172067642212, 5.221689224243164, 1.3879421949386597, -0.2977840304374695, 0.8620582222938538, 2.2501893043518066, -3.7039384841918945, -1.2555705308914185, -4.439788818359375, -1.826835036277771, 0.9532228708267212, 3.5220513343811035, 5.296928882598877, -2.3981831073760986, 3.5019733905792236, 7.059804916381836, 0.39471644163131714, 3.8836379051208496, -0.7410959005355835, 4.769947052001953, -0.7405544519424438, -4.847494125366211, -3.2162506580352783, 1.3114744424819946, -0.9593730568885803, 1.1342101097106934, 4.547550201416016, -3.1715614795684814, -0.6518957614898682, -2.4853787422180176, -1.1280478239059448, 2.549283742904663, 4.232769012451172, -3.121171236038208, -2.3327996730804443, 0.7334957718849182, -4.747653484344482], [0.10687404870986938, -1.893196702003479, 3.1923677921295166, 2.313743829727173, 3.5782339572906494, -2.744856357574463, -2.5093040466308594, -5.510664939880371, -5.931447505950928, 4.775567054748535, -2.6536436080932617, 1.8836272954940796, -2.959339141845703, -2.9688308238983154, -7.600707530975342, 5.418853759765625, 4.659573078155518, 4.485554218292236, -3.7808916568756104, 1.4998422861099243, 5.4690351486206055, 0.6715815663337708, 1.3676928281784058, -0.921040415763855, 2.2627766132354736, 8.914011001586914, -3.5763895511627197, 3.2259459495544434, 2.2444770336151123, 3.390380859375, 2.098886728286743, 0.18990829586982727, -5.027687072753906, -3.874293804168701, 7.855051040649414, -1.6596906185150146, -1.4035065174102783, -6.993344306945801, -0.4315323233604431, 2.339993476867676], [-4.215042591094971, 1.0714198350906372, -12.192450523376465, 3.9429593086242676, -1.163360595703125, 9.581034660339355, -5.586524486541748, -2.9298064708709717, -0.2619788646697998, -4.683431625366211, 1.2138304710388184, 2.310115098953247, 3.7276859283447266, 4.3347978591918945, -0.6099737882614136, -5.595586776733398, -1.3666350841522217, -1.4779257774353027, 5.329863548278809, 2.9504306316375732, -4.7972517013549805, -3.475752830505371, -1.059497356414795, 9.009231567382812, -1.7201976776123047, -4.848762035369873, 6.6608567237854, 0.7423095107078552, 5.487178325653076, -2.9461214542388916, -4.483870506286621, 3.342663288116455, 5.424831867218018, 1.183292031288147, -3.9464333057403564, -1.3903740644454956, 3.0135674476623535, 6.797534942626953, 0.9903867840766907, -0.8344129323959351], [0.5452556610107422, 5.869052410125732, 2.0029144287109375, 0.885441243648529, -10.646932601928711, 2.6218016147613525, 4.931384086608887, 1.5393812656402588, 0.5046831369400024, 1.6575037240982056, -3.963935613632202, -3.805964708328247, -2.1473255157470703, -2.3838231563568115, -3.9091877937316895, -0.9662260413169861, 10.74072551727295, 4.499814987182617, 4.829620838165283, 1.2237021923065186, 0.04328111931681633, 1.911852478981018, -4.047402381896973, -1.5597728490829468, -10.603665351867676, 1.0809307098388672, 2.980154275894165, -1.8305505514144897, 0.6541978716850281, -2.826444149017334, -9.598479270935059, -0.2217252254486084, -1.7395682334899902, -0.6570733785629272, 0.3270757496356964, 3.5548572540283203, 3.658571243286133, 2.41876482963562, -0.9662472009658813, 1.6896615028381348], [0.7415503263473511, -4.949923515319824, -3.6262917518615723, 8.109375953674316, -2.43216872215271, 7.735996723175049, 0.07211871445178986, -4.242148399353027, -6.562705993652344, -2.3085453510284424, -4.717038154602051, -1.8466428518295288, -1.4552788734436035, -6.01072359085083, -10.262518882751465, 0.8168275356292725, -3.9216232299804688, -1.217780590057373, 4.565648078918457, 7.747501373291016, 1.6696724891662598, 4.231075763702393, -0.25942978262901306, 2.712019920349121, 1.5442578792572021, 6.0973639488220215, -1.124166488647461, 1.0585973262786865, 2.1725292205810547, -1.4700286388397217, -1.68942129611969, 2.2640843391418457, 2.275726318359375, -4.646673202514648, -2.694039821624756, -0.2550978362560272, -1.2435609102249146, -0.9714204668998718, 0.13785453140735626, 1.9637267589569092], [4.463992595672607, 1.0360335111618042, 2.682659864425659, -1.3876768350601196, 2.574066400527954, -2.5397942066192627, -4.191877841949463, 4.35743522644043, 6.232077121734619, -3.002976655960083, 0.8089622855186462, -7.3295674324035645, -0.5087937116622925, 4.40376091003418, 2.764583110809326, 4.060422420501709, -3.059321165084839, 3.958254098892212, -2.8867251873016357, -4.242591381072998, 3.1192758083343506, 1.8853211402893066, -5.842373371124268, -6.509580612182617, -2.0568199157714844, -0.009522518143057823, -0.25753527879714966, 2.1681456565856934, -4.895846366882324, -0.7535649538040161, 1.6787277460098267, -7.035974979400635, -2.330918550491333, 1.7151267528533936, -4.1114654541015625, 0.5045552849769592, 1.3633617162704468, -3.2556064128875732, -2.6139841079711914, -3.2487900257110596], [-11.275544166564941, 2.32551908493042, -0.49787959456443787, -1.8459701538085938, 2.398512601852417, -7.057210922241211, 2.4543771743774414, 2.446868658065796, -6.123112678527832, -4.660916328430176, 4.850795269012451, 0.7528517246246338, 0.5358418226242065, -0.5715804100036621, 1.8885139226913452, 0.8910101056098938, -3.4625244140625, -5.143100261688232, -0.5492219924926758, -2.023516893386841, -7.417739391326904, -2.5056560039520264, 2.0934247970581055, -0.6485285758972168, 10.429566383361816, 1.5053654909133911, -3.856497049331665, 2.412909984588623, -4.072610855102539, -2.984039783477783, 3.937617540359497, -5.040833473205566, -3.5392932891845703, -1.0687147378921509, -1.2329015731811523, -0.7373934388160706, -10.355432510375977, -0.8912042379379272, -0.9893417358398438, 2.7322773933410645], [-1.5458896160125732, -4.008196830749512, -6.481171131134033, -1.4192368984222412, 0.6772876977920532, -0.5408741235733032, 3.3969902992248535, 7.285486221313477, 3.14937162399292, -4.543552875518799, 3.9720096588134766, 3.09580397605896, 7.028369903564453, -0.525536835193634, 4.172041893005371, -3.654122829437256, -2.386659622192383, -2.277529001235962, 0.16316118836402893, -1.09983491897583, -1.413464069366455, -6.074325084686279, -2.1029016971588135, 1.7026962041854858, -0.6252037286758423, -3.936598300933838, 1.7230714559555054, 9.242499351501465, -2.918243169784546, -6.871725559234619, 2.1935935020446777, 2.4167282581329346, 6.8535895347595215, 2.4131510257720947, -6.508116245269775, -1.983219861984253, -0.3608013391494751, 5.565749168395996, -4.921483993530273, 5.699418067932129], [3.5092015266418457, -1.4192196130752563, -7.8292012214660645, 5.156660556793213, 0.01974732056260109, 1.0109858512878418, -3.103820323944092, 1.5233736038208008, -2.73089599609375, 2.4336488246917725, 0.9222224950790405, 2.2177951335906982, -1.2832307815551758, -2.605526924133301, -0.8983739614486694, 2.0563805103302, 8.928502082824707, -0.9554791450500488, -1.8658281564712524, 2.3263001441955566, -0.23481924831867218, 0.4450361728668213, 1.3781920671463013, -0.4065978229045868, 0.550872266292572, 0.5267223715782166, 0.8863738179206848, -5.315300464630127, 4.174417495727539, -4.637350559234619, 0.7650681138038635, -0.45355185866355896, -0.11316972225904465, 1.2431938648223877, 0.7394510507583618, 3.4921507835388184, 0.1783776730298996, 1.5049842596054077, -2.4534215927124023, -7.347562789916992], [-1.0480225086212158, 1.3877781629562378, -0.13462962210178375, 4.350722789764404, -0.20806020498275757, 4.917331695556641, -4.405819416046143, -5.346099853515625, -2.5695548057556152, 1.7724031209945679, -0.3920406103134155, 1.845542311668396, -1.5732158422470093, -0.005238870158791542, -6.068758487701416, -5.931460380554199, -2.663935899734497, 1.7788585424423218, 0.24280987679958344, 5.153879165649414, -0.8801640868186951, -1.029779314994812, 0.09868820756673813, 4.056406497955322, -1.2696765661239624, 3.262082576751709, 1.0853891372680664, -0.9410772919654846, 2.2144081592559814, 5.00402307510376, -2.834962844848633, 2.977299690246582, 2.339918613433838, -2.2628424167633057, -3.6713273525238037, -4.353506565093994, 7.87034273147583, 2.3021461963653564, 5.574845790863037, 4.629802703857422], [-1.64585280418396, -2.9828100204467773, 2.317803144454956, -0.35857364535331726, 5.733977317810059, -3.696202516555786, -3.12034273147583, -1.7761646509170532, 0.1377265453338623, 1.6148240566253662, 1.3041051626205444, -0.8936513662338257, 0.4650125205516815, 3.650264024734497, 0.922866940498352, 5.270695686340332, -2.714155912399292, 0.5795834064483643, 1.3657644987106323, -4.786409854888916, 0.7037070989608765, 0.2627066373825073, -3.539609432220459, -1.9399975538253784, 3.6333680152893066, 3.599140167236328, 0.13495786488056183, 3.5669596195220947, -1.162540316581726, -2.2774152755737305, 10.005895614624023, -1.7021604776382446, -2.347524404525757, -2.3354198932647705, -0.3230518698692322, -1.624508023262024, -4.812436103820801, -3.346466302871704, -1.4948681592941284, -3.143235206604004], [-5.878722190856934, 2.993009328842163, -1.0512194633483887, -0.3673741817474365, 2.827147960662842, -5.478122711181641, 0.6209822297096252, -0.39292603731155396, 3.0405707359313965, -6.606906414031982, 1.3378126621246338, -0.2512580156326294, 6.352633476257324, 5.519221782684326, 1.0756304264068604, 1.9582037925720215, -6.060420989990234, 1.5419896841049194, 3.4369287490844727, -2.456254243850708, -3.4319169521331787, -2.844877004623413, 1.0448145866394043, -1.7983627319335938, 3.098395347595215, 1.874731183052063, -0.9197900891304016, 10.299574851989746, -1.3213839530944824, 0.9572605490684509, 5.572302341461182, 3.362677812576294, -4.537649631500244, -1.774278163909912, -5.954084396362305, -3.5622544288635254, -5.947388172149658, 4.610074043273926, 1.4100651741027832, 2.196561813354492], [5.806337356567383, -1.534123182296753, -1.3110390901565552, 2.280292510986328, -2.890291690826416, 5.03240442276001, -4.1599249839782715, -0.07364995777606964, -2.210153818130493, 7.955939292907715, -0.9535838961601257, -2.0255730152130127, -5.50511360168457, -2.769538640975952, -3.8417723178863525, 0.859149694442749, 1.0051747560501099, -2.3961775302886963, -3.9501731395721436, 0.28941062092781067, 4.4468464851379395, 2.549591064453125, -0.4471861720085144, -0.3515118956565857, 2.7997188568115234, -4.5324883460998535, 3.374810218811035, -10.617965698242188, 3.38207745552063, 0.2681024372577667, -2.989914655685425, -3.9626104831695557, 1.0076260566711426, 5.455974578857422, 1.956687092781067, 2.4359254837036133, 2.266923666000366, -4.989035606384277, -2.425524950027466, -3.6915059089660645], [-3.174231767654419, 9.434954643249512, 1.7182711362838745, -0.02154902182519436, -4.721991062164307, 2.158425807952881, -2.0020151138305664, -1.590880036354065, 9.43864631652832, 0.259503573179245, -2.517568826675415, -2.5557689666748047, -0.22981251776218414, 13.287803649902344, 5.04667329788208, -0.03503561019897461, -3.4520652294158936, 3.850274085998535, 6.460076808929443, -7.826827049255371, -9.051648139953613, 3.5851612091064453, -1.7919896841049194, 1.6314669847488403, 1.905946135520935, -3.467541217803955, 4.489922046661377, 2.762784481048584, -1.2172110080718994, -3.205543041229248, 8.485322952270508, -3.2332918643951416, -2.669713258743286, 5.763291835784912, -0.9651320576667786, -4.947409152984619, -0.4041740596294403, 1.4893842935562134, 4.485552787780762, -10.60991096496582], [1.7237285375595093, -0.13647520542144775, -2.256951093673706, -1.7777950763702393, 4.0859856605529785, -10.162919044494629, 0.36877337098121643, 4.547596454620361, -3.3101096153259277, 4.648597717285156, 1.2352206707000732, 1.6738919019699097, -1.6275945901870728, -0.0040147509425878525, -2.5859878063201904, 8.420085906982422, 1.9820820093154907, 4.080324172973633, -4.34458589553833, -4.457276344299316, 3.338900566101074, -5.097064971923828, 5.734897136688232, -4.400574684143066, 7.134129047393799, 0.1902211755514145, -3.420116662979126, 7.20345401763916, 0.5994147062301636, 1.6128443479537964, 7.60920524597168, -2.1408159732818604, -6.942775249481201, 3.2636308670043945, 5.4736247062683105, -2.049835681915283, -9.061004638671875, -6.000272750854492, -2.7464935779571533, -0.6421359181404114], [5.014760494232178, 1.4367300271987915, -2.7754833698272705, 0.41011640429496765, -0.8551478385925293, 4.008846282958984, -2.6079843044281006, -3.406137466430664, 0.1252560168504715, 8.416988372802734, 3.9243619441986084, 4.6610918045043945, -0.3172527253627777, -0.4018036425113678, -1.9164294004440308, -5.324383735656738, 6.747836589813232, -0.05037815496325493, 6.55145263671875, 2.9207262992858887, -0.5795081853866577, -9.388469696044922, 1.6078362464904785, 5.642156600952148, -9.453605651855469, -2.915191173553467, 6.384795188903809, -7.414626598358154, 9.050450325012207, 2.7050580978393555, -7.52966833114624, 7.145449161529541, 2.1801438331604004, -1.4353643655776978, -4.693702697753906, 2.888676881790161, 6.502011775970459, 8.037789344787598, 0.24799348413944244, 2.6751418113708496], [0.8907285928726196, -3.28436279296875, -2.7529804706573486, 0.8380593657493591, -0.6592854857444763, 1.916274070739746, -1.819368600845337, -1.298008918762207, -6.140512466430664, 5.020957946777344, 0.5105305910110474, 0.3505144715309143, -4.117359638214111, -1.8840080499649048, -4.180069446563721, 2.184089183807373, 1.7249494791030884, -1.1610954999923706, 3.1423587799072266, 0.01818975619971752, 2.282468795776367, -3.187018871307373, 0.7046074867248535, 1.8463599681854248, 0.7597635388374329, -1.281937837600708, 2.118952989578247, -4.248341083526611, 4.219593048095703, -0.6806179881095886, -0.510373592376709, 0.08775192499160767, 1.1115738153457642, -0.5515344142913818, 2.3804116249084473, 1.7952321767807007, -1.5851930379867554, -2.0865232944488525, -1.8602149486541748, -1.1027216911315918], [4.56181001663208, -7.7575225830078125, -6.711850643157959, 2.087151288986206, 0.8265645503997803, 0.8609201312065125, -4.288260459899902, 3.1726276874542236, -1.5987319946289062, 0.5540565252304077, 1.5056582689285278, 2.1103270053863525, -1.786910891532898, 0.9986723065376282, -0.2934798002243042, 0.06614220887422562, -3.2818360328674316, -0.900795042514801, -2.7320406436920166, -0.22084027528762817, 3.758523941040039, -3.7414958477020264, 0.7087744474411011, -0.7542741298675537, 1.5859296321868896, -2.988286018371582, -1.0855039358139038, -0.6379497647285461, 1.2461575269699097, -1.392479658126831, 6.382327079772949, 2.717900276184082, 8.159100532531738, 1.7595263719558716, -2.030402660369873, -1.7771265506744385, 5.949537754058838, 2.0294923782348633, -0.40185412764549255, -4.815483570098877], [2.5755436420440674, -7.636174201965332, -5.368505954742432, 0.09651760011911392, 2.1037142276763916, 0.6576078534126282, -1.4583741426467896, 1.4841125011444092, 0.6349428296089172, -5.113328456878662, 0.7587061524391174, 2.1344051361083984, 1.8209435939788818, 0.9638644456863403, 2.7834486961364746, 3.01092791557312, -1.3961987495422363, -0.6228675842285156, 0.5658479332923889, -0.7447352409362793, 3.9440815448760986, -4.785425662994385, 2.9720938205718994, -0.05196443945169449, 0.7238050103187561, -1.478554606437683, -1.3588237762451172, 3.693979501724243, 1.1413310766220093, -2.9138660430908203, 3.1747374534606934, 3.317570686340332, 5.236841201782227, 0.48675045371055603, 1.4635275602340698, 0.8130709528923035, 1.596713662147522, 2.193178653717041, -4.374762535095215, -2.888564109802246], [-2.936235189437866, 1.2385036945343018, 2.431060791015625, -2.439934492111206, -2.633535623550415, -1.939281940460205, 4.28563117980957, 1.6915439367294312, -0.258683443069458, -4.434728145599365, 0.833956241607666, -6.367899417877197, 1.601804494857788, -0.6357243657112122, -1.8264135122299194, 4.471256256103516, -0.17092680931091309, -0.8296481370925903, 8.28809928894043, -2.002805471420288, -1.4868507385253906, -1.8548463582992554, -6.529250621795654, -1.8892275094985962, -4.405091762542725, 3.2726964950561523, -0.7701478004455566, 4.312398910522461, -1.9275327920913696, -6.161832332611084, -3.651423454284668, -3.06317138671875, -3.253769636154175, -5.311058521270752, -5.221085548400879, 4.836592674255371, -8.5160493850708, 0.2744113802909851, -7.026118278503418, 3.876544713973999], [4.08408784866333, -4.856332778930664, -6.095031261444092, 5.383298873901367, 2.76729416847229, -3.0326766967773438, 3.0173897743225098, -0.802435576915741, -2.4757347106933594, 1.7097004652023315, -0.7349879145622253, 5.701447486877441, 5.392622947692871, -6.232468605041504, -5.1802754402160645, 2.6186318397521973, -0.8832769989967346, -1.8638492822647095, -0.36410072445869446, 5.3097124099731445, 4.122406482696533, -2.382138729095459, 9.91057014465332, -1.9903188943862915, 5.438783645629883, 1.2874284982681274, -0.8442570567131042, 3.043689727783203, 5.823443412780762, 4.154881954193115, 3.7650299072265625, 11.285956382751465, -2.977724552154541, -0.12887011468410492, -3.5317070484161377, -0.548227071762085, -5.272457122802734, 5.124969005584717, 0.3876609802246094, 2.769186019897461], [3.378859519958496, 3.113882303237915, 0.1441841423511505, -3.400033712387085, 0.2800801992416382, -0.2247503101825714, -5.494019985198975, 1.5058132410049438, 5.54186487197876, 0.6867688894271851, 0.3794548511505127, -2.8975539207458496, -1.4042242765426636, 6.301863670349121, 3.398576498031616, 1.7242649793624878, -0.0847453773021698, 1.9864126443862915, -5.790977954864502, -4.824778079986572, 1.3545231819152832, 0.9073247313499451, -3.715265989303589, -1.0883125066757202, -1.1577123403549194, -4.277839183807373, -0.38701823353767395, -1.604770541191101, -0.43654415011405945, 0.10342652350664139, -0.48710882663726807, -5.675239086151123, -0.39273250102996826, 4.468019485473633, 2.8714256286621094, 0.5346482992172241, 2.581897497177124, -3.834766149520874, -1.976147174835205, -5.7339348793029785], [-6.613537788391113, 4.037947654724121, 5.638790130615234, -1.8196115493774414, 3.9520626068115234, -3.488466739654541, -0.937996506690979, 0.6760649085044861, -3.98896861076355, 0.5746473670005798, 1.8205052614212036, -0.7773775458335876, -4.050241947174072, -1.4517624378204346, 2.3719301223754883, -0.3190581500530243, 1.367432713508606, 0.5729726552963257, -6.154533863067627, -1.6768077611923218, -2.5517754554748535, 2.9450011253356934, 1.0019314289093018, -1.1848098039627075, 7.235223293304443, 2.0700013637542725, -0.6650957465171814, -1.7830933332443237, -5.795615196228027, 1.634346842765808, -0.34428519010543823, -10.224126815795898, -5.075396537780762, 2.1435606479644775, 6.325693607330322, -1.887628436088562, -2.4268429279327393, -8.317347526550293, 1.7022984027862549, 2.571807384490967], [2.8824193477630615, 0.6022577881813049, 8.34682846069336, -0.11099891364574432, 2.5313262939453125, -5.706806659698486, 8.188031196594238, -1.8183703422546387, 4.45778226852417, 2.593292236328125, -0.751977264881134, 2.2744834423065186, 4.335598468780518, -3.637500286102295, 1.3953760862350464, 3.448498010635376, -1.1092357635498047, -0.10992588102817535, 5.170065879821777, -0.008332507684826851, 0.27954334020614624, -0.5806884765625, 8.774772644042969, -3.968585968017578, 3.4146015644073486, 3.9721076488494873, -0.14865076541900635, 3.500633478164673, -0.07480496913194656, 3.628087282180786, 4.177130222320557, 5.922288417816162, -8.3589448928833, -0.48880910873413086, -2.655226469039917, 0.29827433824539185, -7.635812759399414, 2.413782835006714, -0.2923946678638458, 4.371126651763916], [9.903220176696777, 1.171570062637329, 5.142951011657715, -1.1190063953399658, 0.959006667137146, 0.48844581842422485, -0.5498071312904358, 0.9040688872337341, 12.58132553100586, -4.911018371582031, -1.6711864471435547, -6.095451831817627, 2.141021490097046, 5.07443904876709, 1.5871527194976807, 6.170760154724121, -13.167304039001465, 0.35545381903648376, 0.2781798839569092, -3.0998997688293457, 0.3218817412853241, 1.1576025485992432, 0.9323965311050415, -2.5782039165496826, 4.009634017944336, 1.147720456123352, -6.113518238067627, 7.431673049926758, -3.2587907314300537, -1.310249924659729, 3.777052640914917, -5.0171051025390625, -3.0024874210357666, 3.717489004135132, -6.028926849365234, -0.07013098150491714, -4.9949822425842285, -3.53818678855896, -7.01883602142334, -2.908978223800659], [-1.6557989120483398, -1.2221705913543701, 3.180804491043091, -2.2355828285217285, -1.627702236175537, 7.248042583465576, -0.019171981140971184, -8.6952486038208, 1.8333016633987427, 1.2317088842391968, -3.360443115234375, 4.1697893142700195, 1.4327486753463745, 0.940004825592041, 2.930825710296631, -4.236940383911133, 3.3475708961486816, -2.293721914291382, 0.7027900815010071, 1.2999718189239502, 0.8269162774085999, 3.355496644973755, -0.6550264358520508, 6.937255382537842, -4.790735721588135, -1.3235502243041992, 2.199141025543213, -5.285644054412842, 4.032969951629639, 1.7571371793746948, -5.193205833435059, 5.435258865356445, 4.3157782554626465, -1.7544128894805908, 8.681142807006836, 1.4002785682678223, 5.76509428024292, 0.6656942963600159, 1.9098715782165527, -0.6296877861022949], [4.836028575897217, -3.394742012023926, -5.993336200714111, -1.4808136224746704, 6.878498554229736, -3.1932835578918457, -5.515567302703857, 0.5639764666557312, 1.0729917287826538, -0.8946771025657654, 4.168201446533203, 2.4854042530059814, 3.188316822052002, 1.396675705909729, -0.10418085008859634, 3.3340249061584473, -3.2283918857574463, -2.466714859008789, -6.319265365600586, -0.12100422382354736, 2.7163002490997314, -5.830089569091797, 1.0886404514312744, 0.8766608238220215, 2.7843539714813232, -0.9651726484298706, -4.913544178009033, 2.698807716369629, 3.6123907566070557, 0.6919527649879456, 1.5599263906478882, 0.7418541312217712, 0.3639068901538849, -0.0032950211316347122, -1.4880826473236084, 1.3543134927749634, -3.3304812908172607, -0.6437616944313049, -5.6288251876831055, -0.704218327999115], [4.660548686981201, 3.95331072807312, -4.4691033363342285, -0.6898090839385986, 1.8738688230514526, -4.178020000457764, -2.8606185913085938, 3.4105618000030518, -2.61059832572937, 6.418970584869385, 1.3961843252182007, 0.1409362256526947, -3.100565195083618, -1.7606936693191528, -3.0606906414031982, 3.469595193862915, 1.9453152418136597, -1.6166675090789795, -10.241711616516113, -0.21188566088676453, -0.202052503824234, 1.3200770616531372, 2.3222758769989014, -0.6944103240966797, 6.560618877410889, -3.2262444496154785, -4.2860236167907715, -4.1299238204956055, 2.62618088722229, 1.8582146167755127, -0.45452991127967834, -6.612747669219971, -5.487348556518555, 4.999483108520508, 3.9600391387939453, 1.6923257112503052, -7.3316216468811035, -7.353527545928955, -2.014582872390747, -3.929725170135498], [2.651230573654175, 1.8051979541778564, -2.7967236042022705, -2.195662260055542, -1.846472144126892, -2.6893820762634277, 1.4238678216934204, 2.354973316192627, -1.0429633855819702, 2.3162877559661865, 0.7336411476135254, 1.5508286952972412, -1.520898461341858, 0.723312258720398, -3.4118247032165527, 3.0747313499450684, 5.166347503662109, 4.326480865478516, 2.555356502532959, -1.3364989757537842, -0.1395828127861023, -8.106708526611328, 1.3978708982467651, 2.1161558628082275, -3.984996795654297, 0.9633885025978088, -2.790212392807007, 6.7450456619262695, 2.7267651557922363, -2.1434926986694336, -1.5951240062713623, -0.19602838158607483, -1.0463593006134033, -0.4445585310459137, 2.7898054122924805, 0.50525963306427, -2.6163389682769775, -0.6755232810974121, -4.603112697601318, 1.8202388286590576], [1.872478723526001, -4.953968524932861, 1.3673853874206543, 1.1479415893554688, 4.175212383270264, 2.9695303440093994, 0.14988908171653748, -2.8519198894500732, 2.377747058868408, 4.62605619430542, -0.5864244103431702, 7.875236511230469, 3.0024240016937256, -2.641387462615967, 4.9660468101501465, -5.58405876159668, 2.7413034439086914, -0.021300461143255234, -3.714233160018921, 1.8741912841796875, 4.285516262054443, 1.0749963521957397, 5.6936750411987305, 1.8861370086669922, 0.6196577548980713, -2.7839934825897217, 6.138857364654541, -2.886718511581421, 1.3567036390304565, 4.887814998626709, 1.9297804832458496, 7.584939956665039, 3.86396861076355, 3.6477410793304443, 4.476780891418457, -3.366288661956787, 8.567272186279297, 2.3804564476013184, 4.757800579071045, 2.131598949432373], [6.610787391662598, -8.275594711303711, -5.625041961669922, 4.397294521331787, 1.4892096519470215, 6.8371405601501465, -2.612771987915039, 2.022150993347168, -2.4713330268859863, -0.9931551814079285, -2.2945756912231445, 0.36156943440437317, -3.0409767627716064, -4.002862453460693, -1.1127278804779053, 0.5511515140533447, 3.5413308143615723, 3.846846342086792, 0.47479140758514404, 3.7545294761657715, 6.814947128295898, -0.11686138808727264, 1.5415183305740356, 1.0461499691009521, -3.493773937225342, -0.2406456023454666, 3.2021422386169434, 0.24501027166843414, 0.4873146414756775, -1.7224578857421875, -1.6103631258010864, 0.06266714632511139, 6.34308385848999, 1.2737140655517578, 3.9697611331939697, 0.12237299233675003, 7.409793376922607, -2.001554012298584, -1.1313339471817017, -1.919885277748108], [3.920184373855591, 0.525515615940094, -4.90079927444458, -0.9140217900276184, 0.019380636513233185, 1.7825114727020264, -1.3235304355621338, 0.013721403665840626, 0.786963164806366, -0.4424436092376709, 0.43131428956985474, -0.8278569579124451, -1.0046244859695435, 2.7545676231384277, 0.4405500590801239, 2.629775047302246, -3.5087759494781494, -2.2224178314208984, 3.7621378898620605, -0.5548993945121765, -2.37388014793396, -2.0928585529327393, 3.027315855026245, 3.0835609436035156, 1.3221012353897095, -4.3964409828186035, -1.1440136432647705, -1.9794225692749023, 3.9443180561065674, -0.4928566813468933, -0.06205878406763077, -0.5851059556007385, -0.1874813288450241, 1.29893159866333, -0.019559772685170174, 2.5722334384918213, -5.363587856292725, -0.11845636367797852, -1.5226480960845947, -5.999632358551025], [-1.7925820350646973, -3.242586612701416, 1.0884222984313965, -3.6475210189819336, -6.550617218017578, -0.6610034108161926, 2.539767265319824, -1.972305178642273, -3.80452036857605, -3.8279497623443604, -2.8164689540863037, 2.2997233867645264, -6.062692642211914, 3.179517984390259, 1.900591492652893, 4.2349534034729, 7.096817493438721, 4.115991115570068, 7.141628265380859, -3.058499813079834, 2.736748218536377, -7.605113983154297, 6.732377529144287, 0.2526504695415497, -5.036136627197266, 0.26796552538871765, -3.6099767684936523, 0.27909401059150696, 2.5875444412231445, -0.7511285543441772, -0.9452190399169922, 4.442781448364258, 4.219735145568848, -2.482741355895996, 12.503040313720703, 2.422837495803833, 5.8423333168029785, 2.307487726211548, -0.9858466982841492, -5.341121196746826], [4.053235054016113, -0.5641959309577942, 4.0240864753723145, -3.6477837562561035, -1.5127009153366089, -3.78896164894104, 0.9241800308227539, 1.1064093112945557, 4.783203601837158, 1.9150667190551758, -2.4481945037841797, -1.673201084136963, -1.9637035131454468, 6.9098711013793945, 0.8079655170440674, 3.9487245082855225, -5.942787170410156, 4.514004230499268, 2.3701865673065186, -6.10696268081665, 1.9711369276046753, -1.5526633262634277, 0.3372349143028259, -2.7470200061798096, -0.8751662969589233, -1.4057825803756714, -2.9807839393615723, 5.742133617401123, -1.3133776187896729, 2.154381513595581, 8.961050987243652, 1.868417501449585, 0.6806557774543762, 0.980924665927887, 1.2098969221115112, -3.626004934310913, 0.4867287278175354, -1.3021442890167236, 1.9790687561035156, -4.672743320465088], [-6.986069679260254, 1.7488173246383667, 4.508033752441406, -4.3827972412109375, -5.987689018249512, 7.415316104888916, 4.760703086853027, -7.231678009033203, 0.8547331094741821, -4.817523956298828, -5.357630252838135, 1.24207603931427, 0.5218202471733093, 2.6776986122131348, 1.2581279277801514, -4.024200439453125, -3.374729871749878, -2.296199321746826, 4.541690826416016, 0.2898930311203003, -4.629226207733154, 3.6684939861297607, -1.6162407398223877, 9.44227123260498, -2.4164493083953857, -0.012158315628767014, -2.7904837131500244, 2.699761390686035, -0.35963302850723267, -1.2999467849731445, -4.918619632720947, 1.481752634048462, 4.937474727630615, -2.873859167098999, 9.028642654418945, -0.9532883763313293, 0.13068626821041107, -1.7471284866333008, 1.4424017667770386, 2.041041851043701], [-0.9820078611373901, 7.727199554443359, 3.0201938152313232, -1.0452969074249268, -1.5983924865722656, -1.5301090478897095, 3.218541383743286, 0.8742989301681519, 9.053951263427734, 4.099647045135498, 4.620304107666016, 1.542777419090271, 5.467475414276123, 2.7916762828826904, 7.420740604400635, -3.6617166996002197, 2.806898832321167, -5.177631378173828, 4.596188068389893, -3.456540107727051, -9.957573890686035, -0.1607612520456314, -1.6012377738952637, 0.5874559283256531, -0.11566128581762314, -2.6265885829925537, 5.17638635635376, -4.57811975479126, 0.45060181617736816, -5.518884181976318, 1.7710412740707397, -0.1321990042924881, -4.011351108551025, 3.104179859161377, -9.51463794708252, 2.5673773288726807, -4.328042984008789, 7.48931360244751, -2.0723750591278076, -1.9579384326934814], [-4.489624977111816, -1.6571441888809204, -0.6614239811897278, 4.567311763763428, 1.3108813762664795, -1.286452054977417, -2.6997363567352295, -0.2306833118200302, -1.8930611610412598, -3.300091028213501, 1.7151316404342651, 0.5818240642547607, 0.010207125917077065, 2.0756282806396484, -1.0304656028747559, -1.485767126083374, -4.169775009155273, 2.211526393890381, 2.8185927867889404, 0.28517672419548035, -0.7556542158126831, -3.201078414916992, 1.9734370708465576, -2.8870906829833984, 3.4861059188842773, 3.2633254528045654, 1.920117974281311, 3.3150553703308105, -2.192284345626831, 1.0504710674285889, 6.23246955871582, 2.6560230255126953, 1.2627650499343872, -0.9441854953765869, -6.763005256652832, -5.1499176025390625, 4.9691572189331055, 5.1281023025512695, 4.072884559631348, 1.447708010673523], [-4.504225254058838, 0.08546984195709229, -2.475735664367676, 0.21860630810260773, -5.595861434936523, 3.0156941413879395, -0.19660675525665283, 0.22653190791606903, -6.3339762687683105, -0.27298295497894287, 0.5219014883041382, 0.9951044321060181, -5.4699788093566895, -0.12455324828624725, -0.8544862866401672, -3.1485023498535156, 7.069800853729248, 0.7297043204307556, 4.207721710205078, 0.77128666639328, -2.85581111907959, -3.2963438034057617, -1.8241362571716309, 3.2205374240875244, -5.189055919647217, 0.17723926901817322, 0.7821140289306641, -4.098489761352539, 0.8537855744361877, -3.870283365249634, -3.838959217071533, -0.8627786040306091, 5.548323154449463, -2.547088861465454, 3.418417453765869, 1.3013203144073486, 6.165172100067139, 2.336190938949585, 1.0826570987701416, -1.606056809425354], [2.820301055908203, 8.386320114135742, 4.836859703063965, 3.554654836654663, -4.749969005584717, -0.936294436454773, 2.678922414779663, 1.1219037771224976, 5.091196537017822, 1.6485116481781006, -1.8170603513717651, -5.448563098907471, 0.4240577518939972, -0.5728791356086731, -4.222846031188965, 0.7892283797264099, -0.20204974710941315, 2.276353120803833, 2.3811774253845215, 1.0059272050857544, -3.704742908477783, 4.407657623291016, -2.2507708072662354, -4.259642124176025, -1.0045781135559082, 4.306303024291992, -0.05927514284849167, 0.8975255489349365, -1.889795184135437, -0.7941511273384094, -1.5681688785552979, -3.228700637817383, -7.611174583435059, 0.7630330324172974, -9.125094413757324, 0.34178444743156433, -2.9887635707855225, 1.1403518915176392, -0.2625602185726166, 1.2459768056869507], [12.208690643310547, 1.6446795463562012, -0.9533517360687256, 2.837667465209961, -5.299639701843262, 2.654883861541748, -0.5268667340278625, 3.4705047607421875, 6.257968902587891, 6.096345901489258, -6.259732723236084, -3.152278184890747, -3.458998203277588, 1.413818597793579, -1.650260329246521, -0.004416937008500099, -6.103727340698242, 0.722504198551178, -7.378994464874268, 0.26252660155296326, 2.6280174255371094, 10.527454376220703, 1.9184006452560425, -3.8888819217681885, 2.9410552978515625, -6.894497871398926, -1.6046109199523926, -6.091723442077637, 0.2394808977842331, 4.661373615264893, 4.873076915740967, -0.3524307310581207, 0.9871163964271545, 8.554397583007812, -0.5979205369949341, -2.291151762008667, 3.4875051975250244, -3.0318450927734375, 6.032334804534912, -10.809759140014648], [-5.712426662445068, 2.126390218734741, -0.6941015720367432, 3.433839797973633, -2.6211259365081787, 6.333962440490723, 0.3837486803531647, -5.571803569793701, -1.2141033411026, -3.5815422534942627, -3.3035740852355957, -1.617884874343872, 0.45218202471733093, 1.7462152242660522, -0.9733486771583557, 0.11122605949640274, 0.5120006203651428, 1.5955402851104736, 10.541433334350586, 1.0485354661941528, -2.3373303413391113, 1.811344027519226, 2.1542983055114746, 3.0133626461029053, -0.27997782826423645, 0.4120837152004242, 6.973702907562256, -0.1448899358510971, 2.239835500717163, -0.29402807354927063, -2.9638469219207764, 2.0030570030212402, -2.4734814167022705, -0.8250865340232849, 1.1843746900558472, -0.11932337284088135, -1.3088419437408447, 2.3310012817382812, 2.6752517223358154, -0.5736525058746338], [0.6078100204467773, 0.36307182908058167, -1.7489196062088013, 0.06222096085548401, 1.8966857194900513, 1.7323217391967773, -2.9634437561035156, 2.1799488067626953, 2.798508882522583, -4.070120334625244, -1.1205723285675049, -7.890196800231934, 1.9821885824203491, 2.742971658706665, -3.023920774459839, 5.0986328125, -6.0058770179748535, 2.166583776473999, 0.3882714807987213, -2.6027660369873047, 1.5270743370056152, 3.0199472904205322, -9.121322631835938, -0.7204550504684448, 0.20445430278778076, 1.1387736797332764, 0.4984920918941498, 8.482931137084961, -3.160752773284912, -4.8092498779296875, 0.18868793547153473, -7.781538486480713, -2.435026168823242, 0.11564884334802628, -3.6013095378875732, -0.3239973485469818, -6.992660999298096, -6.550059795379639, -5.967669486999512, 1.031280517578125], [7.663128852844238, -8.320409774780273, -1.912607192993164, 0.6143478155136108, 5.52417516708374, -2.3220486640930176, -9.974756240844727, 1.4941800832748413, 1.8399876356124878, 3.9223151206970215, 4.945681571960449, -1.5385159254074097, -1.026012659072876, 4.149811267852783, -1.5628167390823364, 2.131781816482544, -12.099234580993652, -4.204421043395996, -6.83666467666626, -3.165741205215454, 5.228871822357178, -3.24430251121521, -5.924354553222656, -3.9000539779663086, 4.307570457458496, -1.325940489768982, -3.625426769256592, -2.8231008052825928, 0.5575112104415894, 0.20092269778251648, 10.96303653717041, -0.10606914013624191, 6.355615139007568, 0.05933923274278641, -10.405522346496582, -1.7109214067459106, 3.9280457496643066, -0.3842135965824127, -2.9340503215789795, -4.911106109619141], [-2.771530866622925, -3.201138973236084, 1.1084730625152588, 2.346841335296631, 2.167198419570923, -1.8228052854537964, -2.231678009033203, 1.7917453050613403, -9.446795463562012, 6.658873558044434, 2.3210322856903076, 0.7060539722442627, -6.382803440093994, -6.814352035522461, -2.087982177734375, 0.9377020597457886, 3.8003406524658203, -2.981598138809204, -7.00417947769165, 0.06990224123001099, 2.703763246536255, 1.9211406707763672, 1.386591911315918, -3.1500356197357178, 9.131881713867188, 0.6180939078330994, 1.7377692461013794, -8.625079154968262, -1.8010116815567017, -0.47880786657333374, 1.7505139112472534, -6.910671710968018, -1.230116605758667, 3.020542860031128, 4.484799385070801, 0.2878287732601166, -0.602882444858551, -7.1839494705200195, -0.8176720142364502, 0.5305379629135132], [9.519087791442871, -4.143257141113281, -3.1965973377227783, 2.3820629119873047, -1.0735875368118286, -1.049926996231079, 0.7042497396469116, 6.9189653396606445, -1.952845811843872, 3.9859375953674316, 0.4524060785770416, -1.5219093561172485, -5.959539890289307, -1.7918758392333984, -2.60276460647583, 0.8870266079902649, -2.617206573486328, 2.6918482780456543, 3.17033314704895, 1.0923981666564941, 1.751685619354248, -3.0863797664642334, 2.7152607440948486, -3.118062973022461, -1.9067190885543823, -1.8202786445617676, -1.1834259033203125, -1.1850224733352661, -0.6205316185951233, 0.5632532238960266, 5.360311031341553, 0.3774197995662689, 4.059484958648682, 0.8979337811470032, -5.275085926055908, -1.0040701627731323, 3.032585859298706, 1.0580826997756958, 2.599443197250366, -4.332353591918945], [10.232760429382324, 5.072558879852295, 2.0061213970184326, -1.6027562618255615, -1.855726957321167, -0.07066906243562698, -0.5655484199523926, 3.0410401821136475, 6.05985689163208, 5.092374801635742, -0.7553807497024536, -1.842172384262085, -4.187057971954346, 1.6678451299667358, 2.6104519367218018, -0.09130147844552994, 2.692671060562134, 0.050284724682569504, -4.776439189910889, -0.4200066030025482, -4.0588555335998535, 5.326826095581055, -2.4340932369232178, 0.18288445472717285, -3.6568479537963867, -1.0303294658660889, -5.541522026062012, -6.143674850463867, -0.530654788017273, -1.5305867195129395, 0.3063339591026306, -6.32357120513916, -0.38578081130981445, 2.178858518600464, 0.8028534650802612, 2.920879602432251, 0.905604362487793, -3.452115535736084, 0.18718594312667847, -9.680732727050781], [1.2633806467056274, 4.787491321563721, 4.032684803009033, -6.11915397644043, -1.9639689922332764, -6.593508243560791, 3.1189630031585693, 1.925156831741333, 3.9120235443115234, 4.2518439292907715, 0.35605525970458984, -1.973140001296997, 1.9826509952545166, 3.8594064712524414, -2.3945939540863037, 1.2350727319717407, -1.3553158044815063, 3.190131664276123, 0.7622917890548706, -4.4720354080200195, 0.11323614418506622, -3.2806081771850586, -5.524816513061523, -1.5951621532440186, -6.160739421844482, -0.5567243099212646, -2.487905740737915, 6.515054225921631, -0.030890898779034615, 2.134993076324463, 0.7261245250701904, 1.807803750038147, -3.0906577110290527, -2.113359212875366, -3.1926417350769043, -1.0567805767059326, -3.6849582195281982, 0.16667553782463074, -0.5021381378173828, 4.232139587402344], [3.2011489868164062, 0.5325976610183716, 5.547228813171387, -0.7313929200172424, 0.46226227283477783, 6.331969738006592, -1.4694926738739014, -5.997809886932373, 4.512991428375244, 5.910399913787842, -2.6546471118927, -3.457432270050049, 0.43921124935150146, 0.7460665106773376, 0.6634147763252258, 0.9851315021514893, -1.2530813217163086, -3.375958204269409, 2.273045778274536, -1.4707497358322144, 0.5406071543693542, 8.328530311584473, -6.558618545532227, 2.788926601409912, -1.5903220176696777, -0.9874929189682007, 4.700183391571045, -7.102787971496582, 2.12912654876709, -1.2670788764953613, -1.5434101819992065, -2.5418307781219482, -2.0302088260650635, -0.08634353429079056, 1.0936853885650635, 3.3445663452148438, -3.3954875469207764, -5.029841423034668, -1.8060925006866455, -3.384589672088623], [5.592458724975586, 1.9641387462615967, 5.9242353439331055, -3.8198606967926025, 5.775759696960449, -8.378767013549805, 2.9505488872528076, -0.2590254545211792, 8.321761131286621, 6.074000835418701, 3.447640895843506, 1.972898006439209, 6.575087070465088, 0.6389999985694885, 2.5876755714416504, 3.965250015258789, -6.005716323852539, -4.611318111419678, -2.652867317199707, -3.8093247413635254, -0.8609176874160767, -1.6751813888549805, 2.942512273788452, -2.9411797523498535, 6.223828315734863, -0.13825732469558716, -3.0478603839874268, 2.3722870349884033, 1.2527042627334595, 2.314296245574951, 6.851694583892822, 2.2620158195495605, -7.960457801818848, 1.9118261337280273, -6.258732318878174, 0.8713873028755188, -11.462716102600098, -0.08763948082923889, -4.5749053955078125, 1.9768621921539307], [0.35808229446411133, 3.6911683082580566, 1.881579875946045, 0.37347155809402466, -3.1387391090393066, -6.5106329917907715, 6.949278354644775, 3.828989267349243, 4.1273651123046875, -0.576738178730011, 0.7955731153488159, 1.447740912437439, 2.4012696743011475, 0.9913604259490967, 1.249259114265442, 1.6272742748260498, -4.2084174156188965, -0.628963828086853, 3.8890411853790283, -2.1567494869232178, -5.217147350311279, -3.41686749458313, 7.733572006225586, -3.8255112171173096, 6.074237823486328, 0.19289089739322662, -3.0024805068969727, 5.557258129119873, -1.7157639265060425, -0.740986704826355, 7.314180850982666, 2.9126129150390625, -4.1908698081970215, 3.1276981830596924, -6.504556179046631, -2.2098429203033447, -6.139376163482666, 5.35520076751709, 0.008280215784907341, 0.08005708456039429], [-2.3038840293884277, -2.0554773807525635, 3.9584479331970215, -3.0284461975097656, 2.0938093662261963, -0.6858360171318054, 2.9715583324432373, -7.0515899658203125, -5.051779747009277, 0.08923406153917313, 0.1841801404953003, 2.1318955421447754, 0.9701854586601257, -5.05457067489624, -3.534923553466797, 3.548980712890625, 1.505119800567627, -4.886262893676758, 2.1888010501861572, 2.537672281265259, 0.8668957352638245, -2.390645742416382, 1.7208901643753052, 4.127628803253174, 0.6659718751907349, 5.5118513107299805, -5.240350723266602, -0.5409703850746155, 4.050775527954102, 0.4119575619697571, -5.862327575683594, 1.7647292613983154, -3.2549867630004883, -7.034550189971924, 5.526915073394775, 5.278890609741211, -8.624436378479004, -3.2861573696136475, -5.982004642486572, 5.9708757400512695], [-11.169270515441895, -4.163112163543701, 1.8607858419418335, 2.6732800006866455, 1.0695340633392334, -5.126986503601074, 5.739448070526123, 4.202694416046143, -10.223498344421387, -7.522538185119629, -0.4814797341823578, -0.5776723027229309, -4.657104015350342, -4.294210433959961, 2.7534213066101074, 4.185672760009766, 3.581979751586914, 3.9202442169189453, 6.493235111236572, -1.3303583860397339, -0.24934865534305573, -0.8567176461219788, 7.920598983764648, -6.257078170776367, 7.610610008239746, 4.897218227386475, 1.9444806575775146, 4.5935211181640625, -7.945777416229248, -2.6477251052856445, 6.343794345855713, -3.3863637447357178, -2.505922794342041, -0.9509983062744141, 6.215006351470947, -2.4816200733184814, -2.7350170612335205, -1.6407839059829712, 3.038140296936035, 1.0484797954559326], [1.349230408668518, 8.048442840576172, -1.6165392398834229, -0.004279752261936665, 0.6986882090568542, -4.796497821807861, 0.16950708627700806, 1.959171175956726, 3.06477952003479, 3.773918867111206, -0.37435993552207947, -0.48905426263809204, 1.2250343561172485, 2.8698177337646484, 0.15743553638458252, 1.127416729927063, 0.5830476880073547, 1.4351836442947388, -3.3672566413879395, -1.4849956035614014, -4.351158618927002, 3.9818100929260254, 1.3443965911865234, -1.7357416152954102, 2.884028911590576, -2.308154821395874, -0.5241401791572571, -0.040932539850473404, 0.9020530581474304, 2.9108595848083496, 3.2180192470550537, -1.6639328002929688, -7.589327812194824, 3.386059522628784, -1.0821136236190796, -1.2903825044631958, -6.424929618835449, -0.9381266832351685, 4.180498123168945, -4.547976493835449], [-3.354919672012329, -2.9886956214904785, -2.710625410079956, 5.937715530395508, -1.0274715423583984, 3.6390485763549805, -1.8084790706634521, -3.7366223335266113, -9.98627758026123, 5.93774938583374, -4.500306606292725, 1.033724308013916, -4.933302402496338, -4.214561462402344, -4.620563983917236, -1.4587774276733398, 5.9071431159973145, 1.9655534029006958, 0.6020898222923279, 3.6275713443756104, 4.567171096801758, 7.2127203941345215, 2.096698522567749, -1.2049998044967651, 0.8117914199829102, -1.1560643911361694, 7.756285190582275, -9.768646240234375, 3.370828866958618, 5.728048801422119, 0.57430499792099, 4.372592449188232, -0.10113993287086487, -0.5138493776321411, 6.429421901702881, -1.5613930225372314, 4.772953510284424, -1.1444543600082397, 10.469881057739258, -3.6150524616241455], [-3.8916802406311035, -1.4049324989318848, 5.426797389984131, -1.1380155086517334, 2.2737720012664795, -2.215033531188965, -1.8689533472061157, -3.467458724975586, -2.3496758937835693, 6.69044303894043, -1.0029003620147705, -1.7887699604034424, -2.417818069458008, 2.6622414588928223, -0.6212614178657532, 4.735010623931885, 2.4643964767456055, 2.237698554992676, 2.898404121398926, -6.033586025238037, 2.3787155151367188, 2.776607036590576, -5.494536399841309, -1.9581862688064575, 1.0048401355743408, 2.439926862716675, 4.55728816986084, -0.627094030380249, -0.2798449993133545, -1.2452216148376465, 7.241477012634277, -2.039111614227295, -3.554532527923584, -1.9066741466522217, 4.570936679840088, -1.1322418451309204, -3.3033835887908936, -5.633556842803955, 0.4881858825683594, -2.6148524284362793], [0.23305097222328186, -3.946204900741577, 0.2965059280395508, 5.746817111968994, 6.7697649002075195, -1.6642816066741943, 0.6273653507232666, -2.1998462677001953, -7.060347080230713, 1.172589659690857, 1.3890529870986938, 5.307467460632324, -0.3016570210456848, -9.511658668518066, -2.8909947872161865, -1.9606815576553345, -0.11345157772302628, -1.797387957572937, -3.6194827556610107, 8.021759033203125, 2.1489555835723877, 0.048685960471630096, 10.620720863342285, -1.4010916948318481, 7.526271343231201, 4.9747843742370605, -0.8007920384407043, -3.2949607372283936, 0.5269548296928406, 7.184074401855469, 0.1241750493645668, 3.7997231483459473, -2.7693307399749756, -1.4019763469696045, -0.6254408359527588, -1.5932819843292236, 0.4249555766582489, 0.5287644863128662, 4.615624904632568, 5.96904182434082], [-5.423583984375, 7.360813140869141, 5.905736446380615, -1.9345067739486694, -6.124480724334717, -4.260992527008057, 9.82122802734375, 5.141056537628174, 0.2978076934814453, -1.9660478830337524, 0.45140522718429565, -0.15494060516357422, 0.20747680962085724, -4.449751377105713, 2.6146955490112305, -5.465953350067139, 4.282953262329102, -1.0857945680618286, -0.728762149810791, 1.3889206647872925, -7.321476936340332, 2.6237881183624268, 0.27972012758255005, -2.3088502883911133, -1.3701937198638916, 1.8313547372817993, -2.8311474323272705, -0.09486505389213562, -6.5957560539245605, -1.8380680084228516, -4.186090469360352, -2.7719223499298096, -2.071612596511841, -0.05220721662044525, -2.281487464904785, 0.36551037430763245, -0.6550137996673584, 2.868450880050659, 2.332791805267334, 5.701523780822754], [1.3106080293655396, -10.16787052154541, -1.5255173444747925, 6.554652214050293, 1.385854721069336, -2.9940345287323, 5.773166179656982, 2.627180576324463, -7.8595194816589355, 3.9341986179351807, -1.0102356672286987, 4.021155834197998, -1.183740258216858, -11.747493743896484, 0.5449574589729309, 4.614949703216553, 6.710327625274658, -4.217879772186279, 0.21164071559906006, 2.748164176940918, 5.946861267089844, 3.4467403888702393, 10.18496322631836, -7.573215484619141, 9.103898048400879, 1.091049313545227, 3.62785267829895, -7.6087646484375, 0.6642661094665527, -1.5482581853866577, 7.222888469696045, 5.004823684692383, -1.6244168281555176, 2.0387587547302246, 3.0396454334259033, 2.989938974380493, -3.5351321697235107, 0.9861882328987122, -0.14400418102741241, -3.320706605911255], [-7.08012580871582, 5.346441268920898, 1.276064157485962, 1.0774084329605103, -6.221419811248779, 0.2730558216571808, 2.603642702102661, -0.5390012264251709, 0.06815903633832932, -4.307285785675049, -1.7575663328170776, -0.9544628262519836, 2.7226927280426025, 0.012562848627567291, -5.655306339263916, -2.948676586151123, -1.8594986200332642, 0.6172640323638916, -2.263767957687378, 1.4421119689941406, -3.6983871459960938, 0.8114883899688721, -4.556698799133301, 1.2779759168624878, 0.6015515327453613, 4.867430686950684, -3.9874465465545654, 8.494451522827148, -2.926692008972168, -2.827406167984009, -3.2800443172454834, -1.1452053785324097, -0.24240000545978546, -1.2145200967788696, -3.269423723220825, -3.467866897583008, -0.1621786653995514, 0.9061436057090759, -1.6355677843093872, 8.168279647827148], [-6.387872219085693, 1.095051884651184, -3.678492546081543, 0.697854220867157, -1.515305519104004, -3.9646332263946533, 4.62791109085083, 10.160940170288086, -8.315998077392578, -3.482182502746582, 1.2459877729415894, 3.038347005844116, -4.815822124481201, -7.524516582489014, 5.100831985473633, -4.27646017074585, 12.51888370513916, 2.5741653442382812, -8.137117385864258, 2.389336585998535, -1.563057780265808, 1.453559160232544, 5.774576187133789, -3.618943929672241, 2.4349205493927, -1.8283058404922485, 0.1958855539560318, -2.630601167678833, -7.2243876457214355, -1.541499376296997, -3.5886921882629395, -6.4304375648498535, 1.4815428256988525, 4.765326499938965, 9.191669464111328, -0.7138763666152954, 5.146286487579346, -1.6227213144302368, 4.168493270874023, 1.0480144023895264], [-6.983003616333008, 3.8157618045806885, 5.536930084228516, -2.124716281890869, -9.958415985107422, 3.5379881858825684, 4.5590033531188965, -4.797502517700195, -0.5036824345588684, 0.3717583119869232, -7.246941089630127, 0.35639411211013794, -0.6247929334640503, -0.27088651061058044, 0.23116956651210785, -5.391138076782227, 3.5193021297454834, -2.4295613765716553, -5.385756969451904, 0.6317704916000366, -0.614225447177887, 11.7711820602417, -4.031985282897949, 1.1555585861206055, -2.447256565093994, -1.7564418315887451, -1.309647560119629, -6.847865581512451, -0.8076251745223999, 2.168787717819214, -4.805132865905762, 2.8467166423797607, 2.7208259105682373, -0.20234932005405426, 9.382524490356445, -0.15970657765865326, 5.623417854309082, -0.766339898109436, 6.548158168792725, -1.4718749523162842], [-3.344146728515625, 4.673429489135742, 10.59186840057373, -2.0985782146453857, 0.8085293769836426, -8.833626747131348, 8.83199405670166, 0.49483874440193176, 4.573516368865967, -2.03351092338562, 0.8029562830924988, 2.506747245788574, 2.897409677505493, -1.619139552116394, 5.756442546844482, 2.3970730304718018, 2.3750197887420654, -0.8678465485572815, 0.527167558670044, -2.2263028621673584, -6.2766642570495605, 1.0028849840164185, 5.789069652557373, -4.103618144989014, 5.344302654266357, 7.5005412101745605, -6.028377056121826, 5.066015720367432, -5.628942012786865, -2.329188346862793, 5.5270304679870605, -1.396332859992981, -7.7790327072143555, -0.8141216039657593, 0.5220123529434204, 0.10559319704771042, -7.1966705322265625, 0.7745482325553894, -1.8977047204971313, 2.0910074710845947], [-1.6018846035003662, -0.24856118857860565, 1.4909104108810425, 7.666713714599609, -4.560588359832764, 7.709531307220459, 3.4408042430877686, -3.828794240951538, 2.138185977935791, 2.1560564041137695, -4.6407365798950195, 2.2467846870422363, 7.072235584259033, -8.375789642333984, -3.287034749984741, -11.30734634399414, -2.6349287033081055, -6.02913761138916, -6.855123519897461, 8.987536430358887, 1.873267650604248, 13.290448188781738, -1.7217960357666016, -0.6814450025558472, 1.5704647302627563, -1.0160971879959106, 6.232832908630371, -7.600062370300293, -0.07743001729249954, 4.831418514251709, -4.776692867279053, 9.018238067626953, 2.793264865875244, 2.3060271739959717, -7.919368267059326, -2.76826810836792, 7.895236015319824, 6.2979536056518555, 7.757270336151123, 7.3278584480285645], [-0.8797553777694702, 2.2516896724700928, 1.8307561874389648, -3.326592445373535, -3.496971607208252, -0.21055105328559875, 3.155473232269287, -0.8651726841926575, 1.515701413154602, 3.969078540802002, 2.0037317276000977, 0.4471237361431122, 4.591166973114014, -2.0746028423309326, -3.265941858291626, -4.737167835235596, -1.344536542892456, -5.451963424682617, -0.961959183216095, 1.026120662689209, -1.4605331420898438, -1.2758384943008423, -7.3870463371276855, 2.5623106956481934, -4.982644557952881, -1.1275252103805542, -0.7386223077774048, -1.2337543964385986, 2.12675404548645, -0.6344908475875854, -5.698796272277832, 3.680203676223755, 1.5955790281295776, -3.0496323108673096, -7.180910110473633, 2.0790388584136963, -1.8283966779708862, 3.538637161254883, -2.8763866424560547, 8.161097526550293], [-6.261194705963135, 6.703248023986816, -0.14942654967308044, -1.8350356817245483, -7.341246128082275, 2.93257999420166, 5.68406867980957, -1.951704502105713, 0.6381920576095581, -3.3406381607055664, -0.3843929171562195, -0.5212583541870117, 2.5857043266296387, 1.1067057847976685, -1.0837610960006714, -3.139622688293457, 1.2682842016220093, -2.1043920516967773, 9.27781867980957, 0.7276161313056946, -7.8688859939575195, -2.49838924407959, -1.4674609899520874, 5.563838005065918, -4.411872386932373, -0.422914057970047, 0.8550528883934021, 2.1208372116088867, 1.7753024101257324, -3.871978759765625, -7.501130104064941, 1.5518802404403687, -0.7777969837188721, -2.4961791038513184, -2.6864421367645264, 2.2132301330566406, -4.489408016204834, 5.222057342529297, -1.646748423576355, 4.337646007537842], [6.747519016265869, -12.741813659667969, 2.4558441638946533, 3.9040815830230713, 12.10861873626709, 6.302575588226318, -9.184293746948242, -8.540420532226562, -1.9806923866271973, 7.809906005859375, 1.699682354927063, 5.757481098175049, -2.0450282096862793, -4.4373955726623535, 2.7514758110046387, 1.2162904739379883, -1.1514443159103394, -5.0402140617370605, -5.268683910369873, 2.1811654567718506, 8.756905555725098, 1.914325475692749, 4.995266914367676, 2.3133134841918945, 7.899425983428955, 1.1177016496658325, 4.805487632751465, -11.50091552734375, 4.299291610717773, 3.44881010055542, 4.811946392059326, 1.2610344886779785, 3.248706102371216, 1.5985496044158936, 6.119984149932861, 0.8618765473365784, 5.27749490737915, -5.9412841796875, -1.6805789470672607, -3.5099990367889404], [7.503357410430908, -4.166783809661865, 7.61628532409668, 2.853616237640381, 3.2250497341156006, 1.4184080362319946, -0.5830978155136108, -2.9948372840881348, 10.068283081054688, 4.528501987457275, -2.6786396503448486, -1.8010677099227905, 2.9670052528381348, 3.867840051651001, 2.410598039627075, 1.8081892728805542, -9.104463577270508, 2.9543328285217285, 5.966687202453613, -3.342222213745117, 5.276066303253174, 3.202099084854126, 1.5576579570770264, -5.152848243713379, 1.0573281049728394, -0.267282634973526, 7.2366557121276855, 1.4627633094787598, -0.8447994589805603, 4.403797626495361, 11.089818954467773, 6.543951034545898, -1.2987805604934692, 2.662210464477539, -7.708410263061523, -4.451523780822754, 3.4807400703430176, 2.6287481784820557, 3.814779758453369, -2.4354984760284424], [-6.656547546386719, 7.362689018249512, 0.5593963265419006, -2.820955991744995, -0.7178703546524048, -9.308938980102539, 7.440485000610352, 3.5467019081115723, -4.768823623657227, 1.036597490310669, 3.2621848583221436, 4.436152458190918, 2.9314687252044678, -5.580179691314697, -1.7416163682937622, -4.841489791870117, 7.107439041137695, 0.5124390721321106, -3.3636698722839355, 3.268126964569092, -4.239688873291016, -4.1978654861450195, 3.518237829208374, -1.1045904159545898, -1.8892955780029297, 1.563831090927124, -2.754882335662842, 2.6723616123199463, -0.8201390504837036, 4.748119831085205, -6.183775901794434, 3.2473177909851074, -5.553666114807129, -2.5678329467773438, -1.2592591047286987, -0.46352526545524597, -3.4746038913726807, 4.394309997558594, 3.812513828277588, 11.323653221130371], [6.474541187286377, -0.49603962898254395, -0.4599532186985016, -4.689543724060059, 3.494199514389038, -6.328948020935059, 1.8604263067245483, 4.571473121643066, 3.0785255432128906, 1.7881441116333008, 1.1312373876571655, 4.5758256912231445, -2.305570602416992, 3.407811164855957, 4.563173294067383, 3.472752332687378, -0.1219865083694458, 4.017983913421631, -2.0422346591949463, -3.8035666942596436, -1.0857996940612793, -6.3974504470825195, 7.961150169372559, 0.8106786608695984, 1.9120874404907227, -1.4872779846191406, -6.843139171600342, 7.224994659423828, -0.8585398197174072, 0.9236684441566467, 7.763217449188232, -0.869961142539978, 0.35530874133110046, 3.189934253692627, 6.815327167510986, -2.5781657695770264, -2.0738816261291504, -2.895112991333008, -0.5506187081336975, -4.5516815185546875], [0.6949055790901184, -7.3113555908203125, -1.0362602472305298, 0.6516826748847961, 9.410493850708008, -3.5346877574920654, -3.7068703174591064, 1.927304983139038, 1.9785118103027344, -6.508338928222656, 2.1187682151794434, 1.7628819942474365, 2.630047082901001, 2.3102192878723145, 6.4184889793396, 1.9169178009033203, -2.8290655612945557, 3.9483375549316406, -1.9453989267349243, -1.3714778423309326, 5.248194217681885, -2.908412218093872, 3.5747575759887695, -4.475880146026611, 1.6511784791946411, 1.3128244876861572, 0.8625082969665527, 6.006955146789551, -3.802011251449585, 2.51900315284729, 7.582155704498291, 2.522891044616699, 2.1035494804382324, -0.19074220955371857, -0.108085036277771, -3.4721100330352783, 5.211818695068359, 1.6898564100265503, 1.950549602508545, -1.0324681997299194], [-8.365710258483887, 3.4848132133483887, -1.2712851762771606, 2.2875871658325195, -2.854059934616089, 1.5922837257385254, 2.257265329360962, -3.7174782752990723, -7.054772853851318, 1.6716771125793457, -1.0341291427612305, -1.6427416801452637, 1.7053086757659912, -4.868461608886719, -7.527782440185547, 0.21249181032180786, -1.7845137119293213, -4.565712928771973, 2.0414063930511475, 2.3407931327819824, -0.7027791142463684, 2.25645112991333, 1.085518479347229, 1.1736258268356323, 7.022505760192871, -1.1705185174942017, 4.6610283851623535, -2.7402384281158447, 3.216921329498291, 2.0417864322662354, -6.256087303161621, 0.2696433961391449, -6.643460750579834, 0.5799137949943542, -1.1088811159133911, 0.9526656866073608, -9.839807510375977, -1.5880510807037354, -0.39668166637420654, 7.614960670471191], [-6.648024082183838, 3.7358925342559814, -4.309871673583984, 2.3240208625793457, -6.8201985359191895, 0.3634193539619446, 1.279844045639038, 0.5948401093482971, -6.652405738830566, -2.910334348678589, -1.4216766357421875, 0.34826454520225525, -3.074000597000122, 0.023992525413632393, -5.164178848266602, -2.4074923992156982, 1.5106836557388306, 2.148815870285034, 2.3484973907470703, 2.236679792404175, -2.868563413619995, -2.644505500793457, 3.2061550617218018, 0.828141450881958, 0.6098664999008179, -0.2689286470413208, -0.6202524304389954, 1.0643398761749268, 0.8758401274681091, 1.3307477235794067, -3.3165032863616943, 1.5577168464660645, 0.7127496600151062, -0.1650037318468094, 0.7886447906494141, -2.3608593940734863, 2.327249526977539, 3.322802782058716, 4.328254222869873, 1.8468472957611084], [-4.723231792449951, 9.422607421875, -7.193480491638184, -8.636103630065918, -8.907116889953613, 2.6051125526428223, 0.5099388360977173, -0.5519537925720215, 6.351568698883057, -7.770040035247803, 0.22029753029346466, 2.6665472984313965, 0.7399960160255432, 15.028026580810547, 6.194063186645508, -1.9156818389892578, 2.0488576889038086, 1.2104203701019287, 7.731728553771973, -6.07505464553833, -13.169286727905273, -8.87551212310791, -1.8716239929199219, 12.260507583618164, -8.435052871704102, -6.065008640289307, -6.224034309387207, 7.805539608001709, 3.972426414489746, -7.943077087402344, -3.081462860107422, -0.3018772602081299, 6.078115463256836, -0.17230314016342163, 6.654023170471191, 0.8569045662879944, -0.8404574394226074, 5.500375270843506, -2.78086519241333, -8.649227142333984], [-5.642117977142334, 2.835587739944458, 12.819964408874512, 1.3956769704818726, 1.6882187128067017, -6.454837322235107, 2.7744317054748535, 1.5270555019378662, 0.9472350478172302, -1.4829593896865845, 1.5226079225540161, -1.3161777257919312, -0.7582350373268127, -2.5929713249206543, 0.8374177813529968, -2.3468101024627686, -2.8094329833984375, 3.108562707901001, -3.944328546524048, -0.6485291123390198, -0.7678254842758179, 1.4652342796325684, 1.0936963558197021, -7.776916980743408, 5.635012149810791, 8.502236366271973, -1.496670126914978, 4.278317928314209, -10.283164024353027, 2.7362778186798096, 3.9792237281799316, -4.318960666656494, -4.502425193786621, 0.16193878650665283, -5.062896728515625, -5.777129650115967, 4.191665172576904, -0.981245219707489, 2.867457151412964, 9.377058029174805], [-1.7840713262557983, 1.551306962966919, -0.6037623286247253, -0.30937978625297546, -0.02056494541466236, 2.1234240531921387, 0.6846392154693604, 0.05688628554344177, -0.7450185418128967, -4.578310966491699, 0.276769757270813, -0.21493616700172424, -0.3062485158443451, -0.5789037942886353, -0.3243159353733063, -0.1541213095188141, 2.9144556522369385, 3.4493746757507324, 3.702869176864624, 1.4317563772201538, -1.0618677139282227, -4.311498641967773, 1.510853886604309, 2.9384188652038574, -2.997028112411499, 2.326573610305786, 0.3120478093624115, 5.295583248138428, -0.8387261629104614, -1.6182472705841064, -6.083308219909668, -2.4540741443634033, -0.4500507712364197, -0.9410969614982605, 2.2057132720947266, 0.31467992067337036, 0.47682681679725647, -0.4309234023094177, -2.793130874633789, 4.7039079666137695], [2.4471020698547363, -1.4301421642303467, -0.6414698958396912, 4.521139144897461, 6.314716339111328, -0.6209803819656372, -0.956069827079773, -0.8442092537879944, -4.780605316162109, 4.819909572601318, 2.2686007022857666, 0.33096548914909363, -0.5751221179962158, -6.213300704956055, -4.975727081298828, 1.4883567094802856, -0.04555482789874077, 0.3662668764591217, 2.77254581451416, 4.076878547668457, 1.579162359237671, -1.8363996744155884, 4.0987772941589355, -0.6562144756317139, 3.547361373901367, 3.2764856815338135, 3.905738353729248, -1.1867178678512573, 1.9800012111663818, 3.2974181175231934, -0.35832032561302185, -0.21098853647708893, -5.717503070831299, -1.155588984489441, -4.543466567993164, 0.060186367481946945, -5.107142448425293, -1.6281375885009766, 0.2991101145744324, 5.216272830963135], [2.8662731647491455, -2.991713762283325, 1.2153894901275635, -0.19812937080860138, -2.7060933113098145, 5.408802509307861, -4.266092777252197, 1.9735889434814453, -0.5632274746894836, 1.8278083801269531, 0.7631376385688782, -5.2368388175964355, -4.308873176574707, -0.8298050165176392, 0.18989181518554688, -3.7601277828216553, -2.549034833908081, -1.1283776760101318, -1.4609440565109253, -0.11117199063301086, 5.948407173156738, 1.1531023979187012, -4.872560977935791, -2.7915775775909424, -3.974250316619873, -6.42641544342041, 6.224802494049072, -10.713354110717773, -1.122855544090271, 2.5352704524993896, -6.243192195892334, -2.400484561920166, 5.673635005950928, 2.7380547523498535, -3.5954675674438477, 1.792845606803894, 10.224112510681152, -0.1443958878517151, 1.1064887046813965, 0.3237341344356537], [2.632038116455078, 5.937150478363037, 13.556985855102539, -3.317864418029785, 5.023793697357178, -5.611350059509277, 2.968679189682007, 0.8811253905296326, 7.2478718757629395, 4.921201705932617, 3.7491116523742676, -2.539397716522217, 2.8096210956573486, -4.141188144683838, 4.155150890350342, 0.18422174453735352, 2.4472811222076416, -0.689128041267395, -4.013620376586914, -1.7590306997299194, 0.11310745030641556, 2.003467559814453, -1.3524926900863647, -5.486186981201172, 0.5433499217033386, 3.5768790245056152, 1.895660400390625, -2.547776937484741, -4.867957592010498, 2.158386707305908, -4.286713600158691, -6.531445026397705, -10.361385345458984, 1.751847505569458, -4.653627395629883, 2.9099254608154297, -4.028531074523926, -3.544553279876709, -4.507384300231934, 8.464694023132324], [2.4703314304351807, 0.8471909761428833, -5.884543418884277, 0.7132843732833862, -2.5032732486724854, 6.428629398345947, -2.809196949005127, -0.467702716588974, 0.8138704895973206, -7.6082048416137695, -1.014998435974121, -1.6374258995056152, -0.6443836688995361, 1.0709471702575684, -0.07991144061088562, -0.02464328706264496, 1.5642200708389282, 2.411602735519409, 2.881723165512085, 3.085448980331421, 2.000725746154785, -4.450857162475586, 4.317483901977539, 2.901499032974243, -4.438470363616943, -3.398158073425293, 1.2910817861557007, -0.31961557269096375, 3.2517683506011963, 1.0918259620666504, -11.33462905883789, -0.09583273530006409, 1.7646914720535278, 1.850502371788025, 2.3481180667877197, 3.1676480770111084, 4.556196212768555, 2.3077893257141113, -2.6804842948913574, -0.14991621673107147], [1.6320736408233643, -1.3880760669708252, 2.5059659481048584, 2.247938394546509, -5.578304290771484, 1.1479681730270386, 5.918173789978027, 0.19909803569316864, -3.8402583599090576, 4.931169509887695, -6.61739444732666, 0.5813519954681396, -1.869669795036316, -8.536003112792969, -4.398497104644775, -0.49458175897598267, 7.394379615783691, 1.2411531209945679, -5.101302623748779, 3.8078019618988037, 6.47426700592041, 8.419066429138184, 1.788973093032837, -3.3888447284698486, -1.214714527130127, 0.220993310213089, 1.0308756828308105, -4.46852445602417, 0.21137836575508118, 3.2774505615234375, -4.094444274902344, 3.1540822982788086, -0.9727532863616943, 1.5466272830963135, 7.637062072753906, 0.8122397661209106, 2.6275954246520996, -3.1162474155426025, 3.1857192516326904, 1.997214913368225], [1.5770182609558105, -2.4419212341308594, -13.209796905517578, -4.1657562255859375, 4.7378034591674805, -3.4462318420410156, -5.367079734802246, 9.056138038635254, -3.9046778678894043, -8.086007118225098, 5.416630268096924, -2.793321371078491, -2.387547492980957, 3.4138755798339844, 1.7893153429031372, 7.97222900390625, 0.42311254143714905, 0.7013219594955444, -2.3686747550964355, -3.6850974559783936, -0.6844027638435364, -9.407837867736816, -1.4948076009750366, 1.8786653280258179, 1.5222499370574951, -3.604111909866333, -6.323351860046387, 7.39546537399292, -0.40038201212882996, -8.264808654785156, -0.8020378351211548, -10.962737083435059, 1.7887215614318848, 1.1739414930343628, 4.0240936279296875, 3.9711010456085205, -8.681575775146484, -6.002892971038818, -10.865707397460938, -5.29918098449707], [6.6807050704956055, -3.0163097381591797, 4.762510299682617, 1.5451276302337646, 8.544431686401367, -2.5096263885498047, 5.299378395080566, -0.020315594971179962, -0.2783212959766388, -0.14641833305358887, -1.9572899341583252, -1.8262138366699219, 2.5929627418518066, -11.013812065124512, 0.8675187230110168, 6.2636942863464355, 7.116311073303223, 1.2015377283096313, -0.05034450814127922, 5.2341132164001465, 5.704413414001465, 6.764194488525391, 3.6377370357513428, -4.554961204528809, -1.7038182020187378, 6.6029462814331055, 0.2943739891052246, -0.6619152426719666, -1.2703945636749268, 2.212557792663574, -4.650976181030273, -1.4366918802261353, -9.80398941040039, -3.7979679107666016, 3.519965410232544, 6.490332126617432, -8.931526184082031, -5.685708522796631, -2.9127414226531982, 3.303011178970337], [-0.28612807393074036, -1.506273627281189, -7.0408759117126465, 3.8843190670013428, 0.5052857398986816, 0.6181702613830566, -3.084221363067627, -0.07322322577238083, -5.228743076324463, 1.089381217956543, 0.3421398997306824, 1.520555853843689, -1.3965225219726562, -1.554844617843628, -3.1995816230773926, 0.4978196322917938, -1.83549964427948, -3.843855142593384, -3.723273277282715, 2.4988975524902344, -0.9462636113166809, 1.8007479906082153, 2.6282684803009033, 0.6773442029953003, 6.885579586029053, -1.6379634141921997, -1.465463638305664, -4.427656173706055, 3.024878978729248, 0.2937942147254944, 2.6876113414764404, 0.1445693075656891, 0.5551614165306091, 1.3982305526733398, -0.2149958461523056, -0.13762333989143372, -2.625394105911255, -0.4925684332847595, 1.623044729232788, -4.5413312911987305], [-3.941235065460205, 5.197264671325684, 0.5892073512077332, -0.748566746711731, -1.9292882680892944, -1.2794474363327026, 0.9527616500854492, 1.1632411479949951, -2.5511538982391357, 4.9258317947387695, 0.6489759683609009, 2.4599711894989014, -0.06322148442268372, -3.6546950340270996, -2.841341018676758, -3.283611297607422, 8.548088073730469, 1.0099105834960938, -7.661005020141602, 0.7720645666122437, -1.1663997173309326, 0.7309459447860718, -3.165762424468994, 1.7493685483932495, -0.32096099853515625, 1.3697235584259033, 0.2896876037120819, 0.6872016191482544, -0.643373966217041, -1.27085542678833, -5.5471510887146, -4.045238018035889, -2.2687203884124756, 2.0326852798461914, 4.266834735870361, -0.9275746941566467, 0.33821266889572144, -3.6138947010040283, -1.7088459730148315, 6.6321258544921875], [1.7489558458328247, 2.916229724884033, 2.2055211067199707, -2.198729991912842, -3.330524206161499, 4.5300421714782715, 4.0720133781433105, -1.4805933237075806, 4.86531400680542, 0.3299655318260193, -3.4469194412231445, -1.999302625656128, 0.805856466293335, 2.0383920669555664, 3.252519130706787, -2.246110677719116, -3.2930760383605957, -2.468383312225342, 4.748082637786865, -0.2707914710044861, -4.509145736694336, 6.7298808097839355, -1.8762238025665283, 3.705411911010742, -3.0246081352233887, -3.9806268215179443, 1.057725191116333, -3.168896198272705, 0.03457479923963547, -0.4524192810058594, -1.0317720174789429, 0.389289915561676, 0.8840981125831604, 0.25436729192733765, 0.5077922940254211, 1.4370206594467163, -2.887227773666382, -0.0005631942767649889, 3.234503746032715, -4.440334796905518], [3.399500608444214, -2.3659274578094482, -0.9541552066802979, 4.51596736907959, 2.8705661296844482, 1.305008888244629, 0.3230736553668976, 0.6180073022842407, -0.4478231966495514, -0.3904576897621155, 0.3146176040172577, 2.9186668395996094, 2.347609281539917, -7.056075096130371, -3.30184268951416, -2.856858015060425, -3.282097339630127, -1.653279423713684, -6.930454254150391, 6.362563133239746, 2.5818283557891846, 0.5745437145233154, 5.291481971740723, -0.35178473591804504, 5.585473537445068, 2.390453815460205, -1.8117274045944214, 1.6475166082382202, -0.8900421261787415, 3.175544261932373, -2.053877830505371, 1.2154515981674194, 0.16511808335781097, 2.471177816390991, -3.4850082397460938, -2.290714740753174, 1.7716459035873413, -0.2608342468738556, -0.6028799414634705, 7.270900726318359], [-4.286738395690918, -2.6738364696502686, -1.1526925563812256, -0.7409518361091614, -0.4662728011608124, 0.9594818949699402, -4.744169235229492, 0.8305745720863342, -0.47668787837028503, -5.886689186096191, 1.711167335510254, 0.06995218992233276, -1.6366938352584839, 5.007694244384766, 2.097972869873047, -1.8713349103927612, -2.803041458129883, 2.5245046615600586, -1.1507874727249146, -2.5091614723205566, 1.4059184789657593, -5.298638343811035, -1.448571801185608, -0.16142086684703827, -0.24870707094669342, -0.4296000897884369, -0.2768937349319458, 3.6012208461761475, -2.9403722286224365, -1.0420902967453003, 1.453035593032837, -0.9797977209091187, 6.565507411956787, 0.6164343357086182, 0.3752555251121521, -3.6312954425811768, 9.322820663452148, 1.772574543952942, -0.03885023668408394, 0.5249673128128052], [-0.7291045188903809, -2.582005262374878, 1.4376623630523682, -1.9118572473526, 3.7386443614959717, 1.6110411882400513, -3.409550189971924, -2.8895742893218994, -2.055809259414673, -0.9429954886436462, 2.5240931510925293, 1.425703525543213, -2.6223857402801514, 0.7539752721786499, 1.7511279582977295, 0.1288193166255951, -5.878288269042969, -2.791062116622925, -0.4002476930618286, -0.5881303548812866, 0.9189164042472839, -4.697270393371582, 5.736572742462158, 2.6674180030822754, 6.170206069946289, -2.1318774223327637, -0.5486039519309998, -2.9422388076782227, 0.744985044002533, 3.2899932861328125, -1.0165989398956299, -2.0416135787963867, 1.2984920740127563, 1.8120486736297607, 3.1415138244628906, -0.2883712351322174, 0.3352067768573761, -2.8101818561553955, -1.3073501586914062, 1.299861192703247], [-4.717977046966553, -0.064356729388237, 1.3726954460144043, 1.7229641675949097, 0.13015905022621155, -3.0806002616882324, 3.3157830238342285, 0.32337453961372375, -2.663630485534668, -3.060009717941284, -0.033869482576847076, 1.3405264616012573, 0.5995309948921204, -1.9887681007385254, -1.5542309284210205, 1.4529404640197754, -2.186537504196167, -0.39207732677459717, 0.6041295528411865, 0.5083819627761841, -2.5444793701171875, -0.8236011862754822, 3.707780599594116, -1.0249305963516235, 6.602750301361084, 4.785793781280518, -3.407808542251587, 5.97047758102417, -2.8971683979034424, -1.710435152053833, 4.067030429840088, -0.6650622487068176, -2.14874267578125, -0.7033663988113403, -0.324430376291275, -2.567964792251587, -4.451977252960205, -0.47148415446281433, -0.7405855059623718, 3.353274345397949], [0.263400673866272, -1.2982100248336792, -3.188959836959839, -2.4347870349884033, -6.75929069519043, 8.126705169677734, -3.658353805541992, -0.750235915184021, 2.094231605529785, -3.4446654319763184, -3.954051971435547, -3.204209089279175, -1.7271101474761963, 5.137506008148193, -0.8574455380439758, -1.339381456375122, -0.5565840601921082, 1.6581326723098755, -1.4509861469268799, -1.978294849395752, 3.1279215812683105, 1.461058497428894, -8.273239135742188, 4.233974456787109, -6.949934959411621, -4.387982368469238, 0.04660077765583992, 1.030124306678772, 0.6499735713005066, -3.6533188819885254, -5.509422779083252, -1.849178433418274, 8.623000144958496, 1.253208041191101, 5.481587886810303, 0.21859894692897797, 7.748465538024902, -2.3014957904815674, -2.70717453956604, -2.9917960166931152], [4.5353593826293945, -2.2929935455322266, 7.2892279624938965, -5.571134090423584, -0.9297561645507812, -2.830566167831421, 4.104919910430908, -1.7307308912277222, 3.8609962463378906, 3.271803379058838, -4.774864673614502, -1.9489808082580566, 0.5723505020141602, 0.9363769292831421, 0.28060537576675415, 4.396159648895264, -4.351393222808838, 0.4389829635620117, -1.6668657064437866, -3.693206310272217, 6.82333517074585, 4.101557731628418, -0.03831489756703377, -3.2475216388702393, -1.0842270851135254, -2.4167709350585938, -2.6336581707000732, 0.7372910976409912, 0.10800953954458237, 5.373538970947266, 2.342576742172241, 3.7282118797302246, -1.6642849445343018, 0.05623214691877365, 5.706000804901123, 0.5550332069396973, -2.9203388690948486, -4.521803855895996, 0.5363171696662903, -0.7898969054222107], [-0.7289344072341919, -0.13712163269519806, -5.35430383682251, 1.4356566667556763, -1.4621845483779907, 9.374588966369629, -4.6544365882873535, 0.1539815366268158, 2.573134183883667, -3.0679914951324463, 1.0104645490646362, 1.3795393705368042, -1.2438454627990723, 3.587472915649414, 7.5863118171691895, -7.55403995513916, 2.6043593883514404, -2.368546485900879, 0.3519376516342163, 1.0974596738815308, -5.367297172546387, 2.9816200733184814, -3.3962173461914062, 6.1270976066589355, -3.9237945079803467, -5.257782936096191, 4.601455211639404, -7.750329494476318, -0.317390501499176, -5.042419910430908, -2.467275619506836, -2.6162428855895996, 9.191452980041504, 2.5825884342193604, 0.831949770450592, 0.6841913461685181, 9.939383506774902, 3.467097043991089, 2.784660577774048, -8.005476951599121], [8.046600341796875, -7.336047649383545, -2.103868007659912, 4.1355485916137695, -3.8762292861938477, 4.35542106628418, 1.8935837745666504, 3.911550521850586, 2.066089630126953, -1.2060538530349731, -2.6155548095703125, 1.0313743352890015, -1.78850257396698, -3.628244400024414, 1.8484086990356445, -1.259695053100586, 3.365220546722412, 0.8192513585090637, 0.16419413685798645, 2.945364475250244, 3.8798184394836426, 1.853546380996704, 1.2130463123321533, -2.3678438663482666, -4.509263038635254, 0.23210693895816803, -0.4136217534542084, -2.2433619499206543, -1.2949150800704956, -4.2046661376953125, 2.0288774967193604, 3.2945876121520996, 8.517227172851562, 1.219008207321167, -0.808912992477417, 1.116621971130371, 10.025521278381348, 3.8303518295288086, -0.6271799206733704, -5.198005676269531], [-7.625213623046875, 9.130088806152344, 5.536919593811035, 3.1679275035858154, -6.740936756134033, 0.1903892606496811, 10.39360523223877, 0.3108840584754944, 3.0665879249572754, -3.8224539756774902, -4.338260650634766, -2.9910454750061035, 3.5359137058258057, -2.354414701461792, -0.13314935564994812, -0.1520259529352188, -2.004956007003784, 0.8928467035293579, 7.8717451095581055, 0.060836005955934525, -6.001932621002197, 5.320008754730225, 7.344303607940674, -2.9601635932922363, 7.427232265472412, 0.3903926908969879, 5.613572597503662, 4.609586238861084, -4.131988525390625, 0.03253888711333275, -1.8692245483398438, -0.7810787558555603, -10.277261734008789, 5.4239654541015625, -2.928762674331665, -2.2791128158569336, -8.811275482177734, 2.0615687370300293, 2.3238797187805176, 5.853250503540039], [-0.39782917499542236, -2.596325397491455, 6.251062870025635, 1.6997475624084473, 4.36549186706543, -1.9421491622924805, 0.9604024887084961, -0.6357390880584717, -3.760504961013794, 2.0258331298828125, -2.0578505992889404, 1.7141293287277222, -3.6242244243621826, -4.853661060333252, -0.48778483271598816, -1.4535003900527954, -3.897273063659668, 1.2678906917572021, -7.241701602935791, 2.5234460830688477, 3.125732660293579, 5.48361349105835, 5.965430736541748, -2.9007041454315186, 7.9302496910095215, 2.9877095222473145, -2.1985321044921875, -1.0432559251785278, -5.042741775512695, 7.334370136260986, 4.214850425720215, -1.6893278360366821, -1.1370351314544678, 1.6112231016159058, 5.0912885665893555, -5.197657585144043, 2.767857551574707, -6.310776233673096, 6.865851879119873, 2.976461172103882], [-6.355313301086426, -2.5449612140655518, 0.6847909092903137, -3.0896637439727783, 7.344504356384277, -2.2228314876556396, -0.15915131568908691, 1.125465989112854, -0.3957987427711487, -8.596019744873047, 2.4154672622680664, 1.0615030527114868, 2.9200029373168945, 0.6485005021095276, 9.09573745727539, -1.8932464122772217, -1.3886991739273071, 0.7564045190811157, -1.0945059061050415, -0.7729772329330444, 1.6315059661865234, -0.9335246682167053, 4.049200057983398, -1.6714140176773071, 1.5275360345840454, -2.747384548187256, 3.3156750202178955, 1.2866445779800415, -4.640102386474609, 4.313353538513184, -1.1500948667526245, 0.2522808611392975, 0.2368984967470169, 0.26591646671295166, 3.6499152183532715, -1.139631748199463, 2.2181103229522705, 0.9867420792579651, 3.906506299972534, 2.9556825160980225], [-7.6466593742370605, 3.2765307426452637, -3.2608227729797363, -2.2655632495880127, -4.8896074295043945, 1.4780583381652832, -4.216399192810059, -2.4697065353393555, -3.93017315864563, -0.8940691351890564, -1.6178027391433716, -3.132216215133667, -1.899200439453125, 5.4991936683654785, -3.6900391578674316, 0.8400895595550537, -0.09066145122051239, -0.9344024062156677, -2.3445630073547363, -3.3624963760375977, -1.4232466220855713, 2.4578981399536133, -8.725167274475098, 2.6211206912994385, -0.40266722440719604, -2.071484088897705, -1.1736654043197632, -0.946061372756958, 1.7026454210281372, -2.447035074234009, -1.9010515213012695, -3.452489137649536, 1.2637453079223633, -1.287885308265686, 4.513868808746338, 0.044005755335092545, -1.4797451496124268, -3.4522478580474854, -0.29306283593177795, -3.211669921875], [-5.780657768249512, 5.558526992797852, 0.10565361380577087, -1.5328383445739746, -0.8334417939186096, -2.957430839538574, 1.6118040084838867, -0.07922298461198807, 4.774058818817139, -5.7490949630737305, 0.4256111681461334, -2.470824718475342, 6.971922397613525, 3.8787076473236084, -1.1036908626556396, 3.0051755905151367, -5.505690097808838, -0.7306907176971436, 0.39872056245803833, -3.308471441268921, -3.424093246459961, -1.2176047563552856, -2.351177930831909, -0.30141934752464294, 4.6983771324157715, 1.2663708925247192, -1.6019774675369263, 10.889017105102539, -1.5526760816574097, -3.201294183731079, 0.21778729557991028, -1.9849352836608887, -5.970729827880859, 1.297601580619812, -4.525551795959473, -1.4084229469299316, -9.796573638916016, 0.11529325693845749, -5.768399715423584, 5.227629661560059], [2.411437511444092, 7.189242362976074, -2.862504720687866, 4.668201923370361, -5.70254373550415, 0.13818873465061188, 5.065340995788574, 3.7334625720977783, 7.7120184898376465, -2.208974599838257, -4.159138202667236, -0.9527430534362793, 3.814931631088257, 1.8136426210403442, -1.7879689931869507, 2.0124309062957764, -5.413027286529541, 0.5144158005714417, -0.3133978843688965, -0.2561796307563782, -7.0756516456604, 5.075625419616699, 4.67893648147583, -0.6155823469161987, 8.864694595336914, 0.011616998352110386, -3.04388427734375, 9.354066848754883, -1.9791325330734253, -4.621759414672852, 6.149989604949951, -2.2655789852142334, -5.144932746887207, 7.825716495513916, -4.698878288269043, -4.004958152770996, -8.610833168029785, 0.34603217244148254, -1.424564242362976, -3.3457300662994385], [1.0020309686660767, 6.162059783935547, 2.578225612640381, -0.24985890090465546, -7.1664719581604, -3.4997401237487793, 4.40856409072876, 2.6178081035614014, 2.4865658283233643, -0.3579789102077484, -1.0369986295700073, -2.543393850326538, -1.2531946897506714, 1.180404782295227, -0.8170216083526611, -0.13389752805233002, 1.3235175609588623, 0.6369476318359375, 2.385378360748291, -0.38726598024368286, -4.290066242218018, 0.8807157278060913, 0.08862197399139404, -3.750152111053467, -3.0049309730529785, 0.8239495754241943, -3.686009645462036, -1.4272236824035645, -0.9319217205047607, -0.4532894194126129, -0.49170538783073425, 0.5468432903289795, -2.466702461242676, -0.6541997194290161, -4.741819381713867, 1.5205682516098022, -0.20481671392917633, 4.484396934509277, 1.7249577045440674, -2.5987372398376465], [5.14222526550293, 13.005382537841797, 2.373835325241089, 1.1948883533477783, -2.6647486686706543, -1.0291028022766113, 1.0056711435317993, 0.6380942463874817, 9.75574779510498, 5.499076843261719, -2.7729439735412598, -4.9251580238342285, 6.200190544128418, 0.8452286124229431, -7.884157657623291, -0.7908276915550232, -2.2249183654785156, 2.6343648433685303, -6.437564849853516, 1.2740360498428345, -3.363072156906128, 7.670445442199707, -8.867697715759277, -0.3542249798774719, -2.3890507221221924, 2.569932222366333, -1.68318772315979, 7.0893354415893555, -0.02211545966565609, 1.0899317264556885, -4.2161784172058105, -3.8764092922210693, -9.992246627807617, 2.3902587890625, -9.424834251403809, -1.9288368225097656, -7.477728366851807, -3.1434483528137207, -1.6692004203796387, 5.8860578536987305], [0.02107812836766243, 5.442549228668213, 1.073361873626709, -6.013564109802246, 4.762913227081299, -7.795865535736084, 3.2143654823303223, 6.699592590332031, 7.397768974304199, -5.2816362380981445, 4.679210186004639, 0.5218365788459778, 6.1984639167785645, 1.8920071125030518, 7.196379661560059, -0.3105296194553375, -0.3999292254447937, 2.5972025394439697, -3.420046091079712, -2.9079749584198, -1.8241236209869385, -6.194430828094482, 3.2911899089813232, -2.515244245529175, -0.5303950905799866, -2.7544453144073486, -1.270584225654602, 9.087350845336914, -4.4090256690979, 1.8251943588256836, -2.8633711338043213, -2.407175064086914, -5.093668460845947, 3.6637794971466064, -2.666459560394287, -0.5207981467247009, -3.8955299854278564, 1.708856225013733, -3.0579006671905518, 6.8643622398376465], [10.594414710998535, -4.69149923324585, -6.172008514404297, -3.4463586807250977, 2.234070062637329, 8.019158363342285, -6.614066123962402, -1.3050849437713623, 4.703526973724365, 2.7972593307495117, -1.01407790184021, -5.4062652587890625, 0.7093101739883423, 4.21089506149292, -1.2402714490890503, 5.238872528076172, -4.82285737991333, -2.753734827041626, 1.445113182067871, -2.7226719856262207, 4.891409397125244, 1.3289672136306763, -10.047882080078125, 5.837433338165283, -5.872457504272461, -6.798401355743408, 1.6564106941223145, -1.874266505241394, 5.983789443969727, -4.3633341789245605, -3.102295160293579, -3.4522159099578857, 3.919106960296631, 0.2667131721973419, 0.883499801158905, 5.73187255859375, -5.349059104919434, -6.692237377166748, -8.770940780639648, -6.932224750518799], [0.5004010796546936, -1.3444998264312744, -15.002459526062012, -0.9161510467529297, -6.9948410987854, 4.774203777313232, -2.242274045944214, 2.914302110671997, -5.130173683166504, -1.6560570001602173, -0.7053014636039734, -0.5765411853790283, -1.707866907119751, 1.2457716464996338, -4.8535261154174805, 5.372931480407715, 3.8743021488189697, -1.996140956878662, 1.5283334255218506, -1.6663745641708374, 1.8672449588775635, -6.854093551635742, 0.6756120324134827, 5.31924295425415, 0.6522225141525269, -8.5635404586792, 0.31140780448913574, 0.8092925548553467, 7.944830417633057, -7.199333190917969, -6.775567054748535, -1.8721377849578857, 3.277899742126465, 4.731840133666992, 6.880552291870117, 4.960484027862549, -6.16450309753418, -1.9043041467666626, -10.069135665893555, -4.5164923667907715], [1.7881028652191162, 2.8105857372283936, -5.925685882568359, 0.9925863146781921, -5.169783115386963, 5.945886611938477, -1.4649261236190796, 0.3834646940231323, -3.24079966545105, 1.8209913969039917, -3.3309085369110107, -3.817166805267334, -3.1543219089508057, -0.8392949104309082, -5.706423282623291, 0.008724012412130833, 2.9193458557128906, 1.2077276706695557, 0.764946460723877, 2.1478271484375, 0.8826383352279663, 3.0606703758239746, -3.6227426528930664, 3.2969086170196533, -4.309837818145752, -4.145505428314209, 2.4208486080169678, -3.546642541885376, 3.7075955867767334, -0.3449772596359253, -8.302886962890625, -2.811645030975342, 0.25535276532173157, 1.2013179063796997, 3.0562093257904053, 2.478091239929199, -0.9884051084518433, -3.3255603313446045, 0.03190605342388153, -1.9315983057022095], [-4.448474884033203, 2.593416452407837, 1.0589489936828613, -0.5978123545646667, -2.822187662124634, 2.0702996253967285, -2.309464693069458, -5.5223541259765625, -4.534683704376221, 6.855946063995361, -1.3180360794067383, 3.354473829269409, -2.6701197624206543, -1.0413416624069214, -2.005786657333374, -2.1271560192108154, 9.455166816711426, -2.0860726833343506, -5.582034587860107, 0.22073666751384735, -0.6735315918922424, 3.994096040725708, -3.6771459579467773, 3.873075246810913, -0.7642715573310852, 0.7621649503707886, 0.4285619854927063, -8.101968765258789, 3.9287984371185303, -0.5420570969581604, -3.786176919937134, -0.9738490581512451, -0.019026018679142, -0.7503498196601868, 9.370039939880371, 1.6495577096939087, 2.30025577545166, -3.4776158332824707, 0.6995310187339783, -1.8150734901428223], [5.808256149291992, -1.6908975839614868, -5.398168563842773, 0.3922960162162781, 3.993811845779419, 0.5691452026367188, 1.5142199993133545, 6.541619300842285, 4.669612884521484, -1.4921445846557617, 3.4410240650177, 0.9074063301086426, 5.819460391998291, -4.073809623718262, 4.459917068481445, -1.7674288749694824, 4.008822917938232, 1.0027763843536377, 0.2442532330751419, 1.9914582967758179, 3.2540571689605713, -4.256121635437012, 2.4286673069000244, -1.3806867599487305, -4.3820343017578125, -5.106308937072754, 6.897414207458496, 1.731405258178711, 0.1781659573316574, -0.5877794623374939, -5.811458110809326, 1.7070140838623047, -0.22365841269493103, 4.211163520812988, -5.395333766937256, 2.57106876373291, 0.7195504307746887, 4.484742164611816, -3.8352630138397217, 4.911392688751221], [-0.8112825155258179, -6.294894695281982, -5.700674057006836, -0.9489587545394897, 7.478384971618652, -4.309121608734131, -4.17091178894043, 2.4621098041534424, -4.550955772399902, -4.1592230796813965, 3.002040147781372, -2.780614137649536, 1.9226734638214111, 0.08149126917123795, -0.4659833014011383, 3.919611930847168, -4.339432239532471, -1.540912389755249, -2.2488880157470703, -0.33286234736442566, 4.448686599731445, -0.7633342146873474, -4.6671600341796875, -3.3284788131713867, 0.23726783692836761, -0.5295288562774658, -1.4042248725891113, 1.3349682092666626, -0.36534783244132996, 0.9665804505348206, 3.05092453956604, 0.25756141543388367, 0.7289756536483765, -4.806365013122559, -3.25170636177063, 1.5180778503417969, -4.3117995262146, -1.0771937370300293, -0.43536293506622314, -0.9066867232322693], [-7.3039445877075195, 0.5055170059204102, -4.682466506958008, 2.1939053535461426, 0.6745787262916565, 4.266962051391602, 0.12428371608257294, -1.6493908166885376, -0.10264348238706589, -8.250021934509277, 2.259249687194824, 0.895127534866333, 5.6772308349609375, 0.15802685916423798, 3.986940860748291, -5.4427971839904785, -0.6390658020973206, -3.1155202388763428, 6.211772918701172, 3.2195165157318115, -3.9509940147399902, -1.2119975090026855, 0.9014346599578857, 2.6611826419830322, -1.5636581182479858, -2.3528616428375244, 6.5591936111450195, -1.4985994100570679, 0.6609607934951782, -0.6698845624923706, -5.360010147094727, 4.3485798835754395, 1.7117462158203125, -1.5409388542175293, -5.966012477874756, 1.0393872261047363, 1.5527976751327515, 8.94077205657959, 2.428558111190796, 3.1178081035614014], [-5.042782306671143, 15.250399589538574, -0.8831304907798767, -1.7139123678207397, -5.57276725769043, -1.069948434829712, 6.395719528198242, 2.13861083984375, 5.4656195640563965, 2.5953662395477295, 2.0158631801605225, 3.780769109725952, 4.4931182861328125, 0.4483772814273834, 3.664672613143921, -5.518002033233643, 8.704116821289062, -2.6823630332946777, -0.49919775128364563, -0.37403687834739685, -14.411384582519531, 0.8835538625717163, 1.0992369651794434, 5.795405864715576, 0.9778371453285217, -2.1671934127807617, 0.851153552532196, 0.6190816760063171, 0.5467202663421631, -5.782062530517578, -5.831557273864746, -4.43184232711792, -6.7927374839782715, 5.365551948547363, -0.6648710370063782, 1.2212519645690918, -7.982241630554199, 3.015409231185913, -1.892872929573059, 1.9119089841842651], [3.484191417694092, -1.7790056467056274, -2.342710256576538, -5.9986572265625, 3.0875444412231445, -4.797027111053467, 0.6629645228385925, 6.949244976043701, 4.598292350769043, -2.390383243560791, 0.26700150966644287, -2.011993169784546, 2.6005070209503174, 2.9445745944976807, 3.1934688091278076, 4.4672393798828125, -6.784308910369873, -0.6553200483322144, -8.537956237792969, -5.319686412811279, 2.0486230850219727, 0.9869635701179504, -3.3495943546295166, -1.387023687362671, 3.924359083175659, -4.560079097747803, -6.171446800231934, 8.751890182495117, -4.176302433013916, -2.6092939376831055, 5.2633442878723145, -5.503429889678955, 0.9750115871429443, 4.189730644226074, 2.8738176822662354, -1.2111878395080566, -7.35378885269165, -7.258442401885986, -5.315678119659424, -1.7737399339675903], [8.768802642822266, 0.6357879042625427, -1.0128414630889893, 1.1211541891098022, 0.22199757397174835, 3.1546530723571777, -0.39823171496391296, -3.3578033447265625, 2.464221715927124, 9.954134941101074, 1.750557541847229, 3.297297954559326, 0.5692276954650879, -1.4180861711502075, -4.154606819152832, -1.3419328927993774, -2.150430679321289, -4.088564872741699, 3.634467840194702, 2.4431748390197754, -2.3259241580963135, -3.710747003555298, 2.000277280807495, 5.685387134552002, -0.6635993123054504, -1.2041678428649902, 0.41898876428604126, -4.1368608474731445, 7.502110481262207, 0.7228131890296936, -0.3829486072063446, 3.9579381942749023, -0.18863599002361298, 0.17376889288425446, -5.904470920562744, 1.7607195377349854, -3.462883710861206, 2.0202510356903076, -2.4474880695343018, 0.2646089196205139], [5.6245927810668945, -4.73136568069458, -1.4198867082595825, -0.8100715279579163, 5.509350299835205, -2.3317952156066895, 1.5369848012924194, 5.663671493530273, 1.6459249258041382, -0.9628972411155701, 3.8135955333709717, 0.29464226961135864, 1.2402925491333008, -4.251676082611084, 5.253840446472168, 2.9318244457244873, -0.9834815859794617, -3.5408501625061035, -1.5842446088790894, -0.3489418923854828, 1.5937604904174805, -2.2239694595336914, 4.215348720550537, -2.569295883178711, 4.241130352020264, -2.318094491958618, -0.24189311265945435, -1.0757282972335815, -1.986079216003418, -2.7917683124542236, 1.6190195083618164, -3.3001110553741455, -0.4442427456378937, 3.2217326164245605, -1.8266335725784302, 3.4932525157928467, -4.943102836608887, -1.4775127172470093, -5.938302040100098, -0.6824899911880493], [0.7136743068695068, 0.8354748487472534, 4.801477432250977, -1.5882799625396729, -0.38543379306793213, 0.6818301677703857, -0.9346379041671753, 1.4775681495666504, -2.2284796237945557, 6.913417816162109, 1.0256507396697998, -5.155808448791504, -4.740075588226318, -3.2453529834747314, 1.3617117404937744, 2.48405385017395, 10.202787399291992, 1.7528306245803833, 2.24546217918396, -3.201474666595459, 4.82307243347168, 1.7324798107147217, -4.6133503913879395, -4.200466632843018, -4.86660099029541, -2.3874735832214355, 9.458539962768555, -10.296051979064941, -0.2249947041273117, -1.080367922782898, -6.687651634216309, -6.584433078765869, -4.49045467376709, 1.653342604637146, 3.385326623916626, 5.84586238861084, 0.5816451907157898, -4.7780938148498535, -2.977780342102051, -0.10767142474651337], [2.9205377101898193, 9.19757080078125, 7.644809246063232, -0.29789999127388, -1.1713508367538452, -5.8341383934021, 8.024566650390625, 3.564208984375, 3.024523973464966, 5.973508358001709, -2.099900007247925, 0.15870055556297302, -0.9651693105697632, -4.923633575439453, 1.175394892692566, -0.9419100880622864, 9.238082885742188, 4.4114603996276855, -2.6563704013824463, 1.4526276588439941, -4.042440414428711, 6.514071941375732, 2.845104932785034, -3.7990500926971436, -2.1880526542663574, 3.760615348815918, -1.245157241821289, -0.4011746048927307, -3.8341596126556396, 2.4549527168273926, -1.4603358507156372, -3.443220853805542, -9.464091300964355, 1.5346769094467163, 2.161628246307373, 0.18779772520065308, -3.361555576324463, -2.520402193069458, 4.250489234924316, 1.576871633529663], [-4.853775501251221, 1.6898726224899292, 2.1497116088867188, -4.861413955688477, 5.506793975830078, -8.638588905334473, 5.17995548248291, -1.0939592123031616, -4.216857433319092, 1.2334702014923096, 1.3461294174194336, 3.5649056434631348, 2.7394556999206543, -3.7854576110839844, -0.10950960963964462, 5.455212116241455, -0.9408650994300842, -4.086653709411621, -4.440195560455322, -1.500402808189392, -1.1799291372299194, -1.045968770980835, 7.114964008331299, 0.49342429637908936, 10.802842140197754, 0.9681461453437805, -5.571510314941406, 4.115053176879883, 0.5913559198379517, 2.8080968856811523, 1.4627705812454224, -1.5017781257629395, -9.21914005279541, 0.15793392062187195, 7.952780246734619, 0.9821940064430237, -16.589595794677734, -6.839868545532227, -3.2603893280029297, 5.037130355834961], [-0.0576041005551815, -0.225305438041687, -6.668380260467529, -4.2079010009765625, 3.1629323959350586, -3.109699010848999, -3.0804977416992188, 0.06693058460950851, -5.092063903808594, 1.1700996160507202, 3.563560962677002, 0.7628969550132751, 0.24984081089496613, -0.03234375640749931, -3.0999011993408203, 4.316367149353027, 0.61078941822052, -2.651672840118408, -2.320845365524292, -0.9668130278587341, 2.1825308799743652, -7.274394989013672, 1.1357901096343994, 2.6856255531311035, 1.5200283527374268, -4.022395610809326, -2.2753074169158936, -0.3677152395248413, 6.006844520568848, 1.5664341449737549, -5.057010650634766, -0.8664396405220032, -2.5933566093444824, -0.5717093348503113, 3.5531113147735596, 4.015923500061035, -8.399685859680176, -3.059589385986328, -5.689121246337891, 1.672218918800354], [2.6599860191345215, -6.957692623138428, -1.6769635677337646, 5.595451831817627, 0.6594173312187195, -0.30890953540802, 1.5359296798706055, 0.5243462920188904, -1.4770286083221436, -3.5719587802886963, -0.692004382610321, 1.171746850013733, 0.5039878487586975, -3.8495781421661377, 0.05119768902659416, 4.6951904296875, 2.5412323474884033, 2.247178554534912, 6.312307834625244, 1.810053825378418, 6.080556869506836, -4.368338584899902, 9.562679290771484, -5.7747368812561035, 2.0992536544799805, 2.3519742488861084, 3.5846779346466064, 0.9625259637832642, 1.3838008642196655, 0.07908987998962402, 2.5436313152313232, 6.2548418045043945, -0.912609338760376, 0.6361018419265747, -1.646185278892517, 1.2712419033050537, 2.318654775619507, 5.451545715332031, -1.8996821641921997, -0.5123458504676819], [0.37253379821777344, 2.0093812942504883, 2.943570375442505, -2.591376543045044, -6.897804260253906, -0.3369271159172058, 5.0172343254089355, -1.0459767580032349, -2.509981155395508, 4.095792770385742, -4.252922534942627, -1.2395939826965332, -4.709224700927734, -0.8798695802688599, -1.3349244594573975, 3.310455083847046, 4.347729206085205, -1.2194889783859253, 2.383701801300049, -1.8177913427352905, -0.9246814250946045, 4.014100074768066, 0.786342203617096, 0.14853936433792114, -0.8437331318855286, -1.2803285121917725, -2.7017297744750977, -5.299205780029297, 1.9573932886123657, -1.1548124551773071, -0.8339053392410278, -0.7844715118408203, -1.6946600675582886, -0.3405354619026184, 8.216197967529297, 3.68094539642334, -4.467511177062988, -3.5354654788970947, 0.10789819061756134, -5.725642204284668], [12.448040008544922, 1.5848816633224487, -7.500838756561279, -1.7454309463500977, 1.5614688396453857, -3.1710691452026367, -3.229199171066284, 8.558984756469727, 4.8793559074401855, -1.8247077465057373, 1.8272011280059814, -2.0970518589019775, -0.6454178690910339, 1.4301886558532715, -0.5460153222084045, 4.013001918792725, 3.393839120864868, 3.849213123321533, -7.687008380889893, 0.4441584050655365, 0.6761900186538696, -3.339369773864746, -3.353668212890625, -0.7358533143997192, -6.226168155670166, -0.011641092598438263, -9.149816513061523, 5.80543327331543, 0.028008706867694855, -4.085063934326172, -2.159792184829712, -6.337075710296631, 0.8695983290672302, 1.1473579406738281, -1.2277888059616089, 3.188441276550293, -0.725242555141449, -2.8654427528381348, -7.21303129196167, -5.043665885925293], [-0.8662123680114746, 1.747209072113037, 0.9329647421836853, -1.3675938844680786, -2.995274782180786, 2.5681846141815186, 1.1389477252960205, -2.3905858993530273, -0.8119733333587646, 2.610611915588379, -2.8073956966400146, 2.1993370056152344, -2.109790325164795, 1.849912166595459, -1.6527798175811768, -2.4226012229919434, 5.463921070098877, 6.2618937492370605, 4.246311664581299, 0.01889156363904476, 0.6156171560287476, -2.56585693359375, 0.18122698366641998, 4.066617012023926, -7.553738594055176, 0.204829603433609, 2.4832797050476074, 3.1316044330596924, 1.8992619514465332, 2.152892589569092, -3.032837152481079, 3.3278417587280273, 1.9113298654556274, -1.3254718780517578, 6.134693145751953, -2.448976755142212, 5.688685417175293, 0.48303163051605225, 3.6450436115264893, 2.0907108783721924], [-2.2187507152557373, -4.327786445617676, 3.196040153503418, 0.7061147689819336, 2.3320376873016357, 0.637684166431427, -3.387878894805908, -4.938015460968018, -3.915597677230835, 1.830130934715271, -1.4462674856185913, -3.723449230194092, -2.7841646671295166, 2.6595585346221924, -1.5791573524475098, 1.8371697664260864, -4.2407636642456055, 0.6113011240959167, 6.164562225341797, -1.5007466077804565, 3.5420870780944824, 3.3851256370544434, -4.27189826965332, -2.4560904502868652, -2.108067035675049, 0.9136578440666199, 4.04440450668335, -5.271630764007568, 0.913046658039093, 4.197814464569092, 4.858465671539307, 3.373276472091675, 0.46470481157302856, -5.862268447875977, -1.379258632659912, -0.4373701810836792, 1.6038486957550049, -0.14764878153800964, 6.500524044036865, -3.7902493476867676], [3.354985237121582, 1.2777295112609863, -1.138487696647644, -0.3482949435710907, -4.5844197273254395, 3.0214154720306396, -1.9532674551010132, 2.3757760524749756, 4.772216796875, 3.71125864982605, -0.044712912291288376, -0.7363705635070801, -0.13602039217948914, 2.9247658252716064, -0.9439312815666199, -1.31125807762146, 3.2398195266723633, 2.318617105484009, -1.1901047229766846, -2.869474411010742, 0.4590233266353607, -1.7961461544036865, -6.927150249481201, 1.4458588361740112, -5.192857265472412, -1.457265019416809, 1.9422295093536377, 2.4281649589538574, 0.27022871375083923, -5.792749404907227, -0.600621223449707, -2.221935749053955, 3.9308090209960938, 3.1400301456451416, -1.7463784217834473, -0.7178454399108887, 5.08262300491333, 0.09107708185911179, -4.826530456542969, -0.5149812698364258], [4.063823699951172, -5.288538455963135, 2.770786762237549, -2.8389716148376465, 1.240844964981079, 2.0517067909240723, 2.1764063835144043, -4.686570167541504, 3.8261778354644775, 2.140519142150879, -0.1592293530702591, 2.0785555839538574, 4.828455924987793, -1.5448020696640015, 1.7427672147750854, 0.2074156254529953, -6.590514183044434, -7.831967353820801, 0.849962055683136, 0.1871892511844635, 3.5512430667877197, 0.24598385393619537, 1.7423452138900757, 1.7458823919296265, 1.3829926252365112, -3.6891989707946777, -0.026569005101919174, -4.437647819519043, 4.268441200256348, 2.4487104415893555, -0.38596111536026, 7.3827056884765625, 2.0043222904205322, -0.5470423698425293, -1.5718082189559937, 3.355328321456909, -3.2028095722198486, 1.990357518196106, -3.086747407913208, 1.2844864130020142], [7.889134407043457, 6.8591227531433105, 4.984535217285156, -4.448441505432129, -0.7244563698768616, -0.6631902456283569, -1.4517252445220947, 1.617518424987793, 3.15106201171875, 6.166859149932861, 0.00909289252012968, -2.5791256427764893, -6.707450866699219, 3.1640443801879883, -1.4590760469436646, -1.7275584936141968, -2.467062473297119, 5.089972496032715, -1.6276861429214478, -0.9387369155883789, -1.3256535530090332, -2.0835397243499756, 0.825086772441864, 1.6716481447219849, -4.510762691497803, -2.4379255771636963, -3.22204852104187, -1.6354175806045532, -0.54084712266922, 6.7835798263549805, -3.9156272411346436, -5.535048961639404, -1.669257402420044, 2.1913936138153076, 1.323003888130188, -1.496006965637207, 3.118091583251953, -5.080960273742676, 3.4813995361328125, -0.05692192167043686], [-0.7283059358596802, 2.4558053016662598, -3.465635061264038, -2.491419792175293, -7.208122253417969, -2.1030309200286865, 2.2705349922180176, 3.029496908187866, 0.38040369749069214, 1.2567574977874756, -0.9075305461883545, -1.0174120664596558, 3.1723549365997314, -0.5840890407562256, -4.155229568481445, 0.2801249623298645, 6.822384834289551, 0.8728398084640503, -3.218954086303711, -1.2398592233657837, 3.565991163253784, -2.0368566513061523, -6.392139911651611, -1.8514741659164429, -7.251061916351318, -2.6453583240509033, -0.005438783206045628, 2.2233121395111084, 2.76981520652771, -2.2588229179382324, -6.686764240264893, 3.135374069213867, 0.3033781349658966, 0.08658324182033539, -0.44707348942756653, 2.7315821647644043, 0.4537930488586426, 2.7684662342071533, -4.660590171813965, 3.8943605422973633], [-0.4469349682331085, 3.3662400245666504, -1.1234216690063477, -1.7818268537521362, -8.366703033447266, 4.891042709350586, 1.8933347463607788, -4.6026811599731445, 1.4704898595809937, 0.3358696401119232, -4.637974739074707, 0.0878140926361084, 0.6751099824905396, 3.0010366439819336, -3.432781219482422, -2.7168421745300293, -1.489978313446045, -2.044095277786255, 1.860029697418213, 1.1426639556884766, -2.151611328125, 2.7783050537109375, -2.7272789478302, 5.32149076461792, -4.925271511077881, -2.914858102798462, -2.4806365966796875, -1.8106471300125122, 4.735866546630859, 0.9380941390991211, -4.75018310546875, 5.155843734741211, 3.2314698696136475, -1.6211071014404297, 2.1956117153167725, 0.9139595031738281, 1.0097960233688354, 2.4131276607513428, 2.0726327896118164, -1.986267328262329], [-11.292640686035156, 8.056741714477539, 4.017036437988281, -0.1346757411956787, -7.209616184234619, -0.7226728200912476, 7.636193752288818, -4.466634750366211, -1.7064489126205444, -3.103701114654541, -2.170140027999878, 1.108981728553772, 0.7947624325752258, 2.511378765106201, -0.11451994627714157, -2.1827638149261475, 6.364118576049805, 2.25528621673584, 13.528521537780762, -0.19507460296154022, -10.2341947555542, -0.5954657196998596, 0.32187965512275696, 3.2405834197998047, -5.14099645614624, 5.548484802246094, 1.0578091144561768, 3.609510898590088, 0.14518830180168152, -3.100417137145996, -1.3024872541427612, 3.9874989986419678, -3.69340181350708, -6.400735378265381, 0.47697803378105164, -0.2856346368789673, -3.034498929977417, 6.936509609222412, 4.51367712020874, 1.5843849182128906], [-6.591284275054932, 3.6315062046051025, -1.678672432899475, -6.885013580322266, -4.436683654785156, 0.8104588389396667, -0.9844209551811218, 0.08203695714473724, 1.3196581602096558, -10.595460891723633, 0.07263533025979996, -3.6678953170776367, 1.5867605209350586, 6.218050479888916, 1.2595851421356201, 0.1261947900056839, -3.413816213607788, 0.09505626559257507, -0.5548551678657532, -3.148601770401001, -0.619097113609314, -4.464894771575928, -4.095485210418701, 2.7107126712799072, -3.5999886989593506, -3.7715044021606445, -3.858819007873535, 5.42872953414917, -0.9687987565994263, -1.0207600593566895, -9.166669845581055, -2.8173046112060547, 2.3775265216827393, -0.6322304010391235, 3.456712245941162, 1.5579465627670288, 0.0165074672549963, -0.1719411313533783, -4.456447124481201, 3.2379701137542725], [-0.15841875970363617, -1.8774571418762207, -0.8818662762641907, -1.5671632289886475, 4.958149433135986, -2.7695300579071045, 0.3361755609512329, 3.9946353435516357, 5.354610919952393, -9.059165000915527, 3.764289379119873, -2.0966241359710693, 4.077996730804443, 2.549995183944702, 9.365306854248047, 4.452098369598389, 3.1703646183013916, 0.0780438706278801, 4.591458797454834, -3.308654546737671, -2.657747268676758, -2.2620315551757812, -2.4548914432525635, -2.5145461559295654, -2.7442612648010254, 2.538613796234131, -0.20776638388633728, 4.73101806640625, -4.012407302856445, -8.825806617736816, 2.5456535816192627, -4.0399699211120605, -0.9135117530822754, -2.1430611610412598, -2.834737777709961, 4.457221031188965, -3.841423749923706, 2.541417121887207, -7.029977798461914, -4.18333625793457], [1.5839602947235107, 7.535191059112549, -2.2115566730499268, -7.576563835144043, -2.0799622535705566, -3.988300323486328, 3.2253222465515137, 7.283874034881592, 7.231907367706299, 1.389844536781311, 3.153620958328247, -1.2697709798812866, 3.5861449241638184, 3.218780517578125, 7.4405975341796875, 0.33144330978393555, 8.8010835647583, 0.13496294617652893, -1.3353089094161987, -5.74617338180542, -3.8707916736602783, -2.092625141143799, -4.968481540679932, 0.20916421711444855, -7.153258800506592, -7.499613285064697, 2.0310962200164795, 0.3302133083343506, -0.1052045226097107, -5.457767009735107, -5.50241756439209, -4.764499664306641, -3.0818052291870117, 4.248543739318848, 1.1730002164840698, 5.305080413818359, -5.353340148925781, 0.6459091305732727, -5.895552158355713, -2.0383222103118896], [-0.2510012686252594, 4.945319175720215, -1.9990769624710083, 2.4457151889801025, -3.8728647232055664, -1.0277020931243896, 2.6670799255371094, 1.6905958652496338, 1.4483033418655396, 4.584682464599609, 0.2948944568634033, 3.096918821334839, 2.925367832183838, -2.4595963954925537, -0.8349257707595825, -4.895060062408447, 5.653397560119629, -0.736339271068573, -2.665632963180542, 2.2143337726593018, -2.366122007369995, 1.4019129276275635, 1.216199517250061, -1.0242239236831665, -1.0975267887115479, -2.208664655685425, 3.3282384872436523, -3.3296315670013428, 2.0437915325164795, 1.1718876361846924, -2.1352453231811523, 4.212236404418945, -1.8382877111434937, 3.0543060302734375, -4.08589506149292, -0.490131139755249, 2.0745856761932373, 5.833705425262451, 3.1099226474761963, 1.8216561079025269], [1.6248533725738525, 1.9846388101577759, -1.6692769527435303, 3.966614007949829, -1.13541579246521, 3.311796188354492, -0.16483809053897858, -1.7105586528778076, -3.499569892883301, 3.642794370651245, -2.546191453933716, -0.5950632691383362, -2.3082804679870605, -4.107808589935303, -3.7410354614257812, -0.9958529472351074, 4.066625595092773, 0.8942424058914185, 0.538790225982666, 4.362009525299072, 0.9109042882919312, 4.513201713562012, 2.5942771434783936, 0.269046425819397, -0.48937150835990906, -0.6268131136894226, 3.8501763343811035, -6.465348720550537, 2.9641261100769043, 3.7159109115600586, -4.721482276916504, 0.5254010558128357, -3.2059247493743896, 0.7715393304824829, 1.6087076663970947, 1.2486670017242432, 0.24802060425281525, -0.9390851855278015, 4.222872257232666, -0.8355262875556946], [10.161341667175293, -3.7439494132995605, -1.0852020978927612, -0.870209813117981, 2.1377387046813965, 5.3384108543396, -4.965944290161133, 2.19912052154541, 5.669768810272217, -1.6348488330841064, 0.631147563457489, -3.771094560623169, -1.684226632118225, 2.9265024662017822, 2.4516220092773438, -2.120756149291992, -3.7034690380096436, 1.9113881587982178, 0.49449682235717773, 1.0657129287719727, 2.361011505126953, 0.1326029896736145, -6.754190444946289, 1.4195829629898071, -9.892413139343262, -1.6023069620132446, -0.6869124174118042, -1.5721542835235596, -1.1924117803573608, -0.7890673279762268, -1.8334733247756958, -1.9339388608932495, 7.701664924621582, -1.8465886116027832, -4.888674736022949, 1.415006160736084, 8.724061012268066, 0.2989860475063324, -0.38344839215278625, -4.143614768981934], [-3.237783908843994, -0.6562668085098267, -3.7106049060821533, -0.9143486022949219, -4.558250904083252, 5.518185138702393, -6.680918216705322, -0.3866068720817566, -5.083093166351318, -3.417036294937134, 0.8737236261367798, -4.704854965209961, -6.401331424713135, 3.593893051147461, 0.08986691385507584, -2.8174538612365723, 0.4144858419895172, -1.1097253561019897, 1.2528762817382812, -0.297884464263916, 0.7686114311218262, -0.5545278787612915, -5.852969646453857, 0.996008038520813, -5.511449337005615, -5.04083251953125, 2.281534433364868, -10.631757736206055, 1.0948917865753174, -0.11467884480953217, -7.101645469665527, -3.2140166759490967, 6.887718677520752, -1.5646946430206299, 0.8869271278381348, 3.150075912475586, 9.123919486999512, 1.174633264541626, 2.2652971744537354, -5.273653030395508], [-3.008953332901001, 5.430008888244629, 0.3624310791492462, 1.4557019472122192, -10.164922714233398, 2.3304443359375, 2.7532310485839844, -1.4328107833862305, 2.130664825439453, 1.203214168548584, -4.027450084686279, 2.5859227180480957, -0.057749100029468536, 0.32441768050193787, 1.6079586744308472, -2.5699942111968994, 13.500395774841309, 1.259974479675293, -1.9225070476531982, -0.1672583371400833, -1.8090684413909912, 4.1982526779174805, -0.22981788218021393, -0.3249014616012573, -4.52557897567749, -0.3780362904071808, 1.8289194107055664, -4.957068920135498, 2.1091742515563965, -3.610210418701172, -4.57717752456665, 2.4526619911193848, 0.21917662024497986, 2.8379387855529785, 6.534761905670166, 2.0855889320373535, 6.917338848114014, 4.298347473144531, 0.5614868402481079, -4.4593915939331055], [1.7942863702774048, -7.1933441162109375, 3.2087175846099854, -7.664908409118652, 1.7833447456359863, -9.440977096557617, -2.513650894165039, 3.425323009490967, 0.22608524560928345, -1.7461787462234497, 1.6369848251342773, -5.107518196105957, -3.222938299179077, 8.31706714630127, 2.2701056003570557, 10.613303184509277, -1.4096481800079346, 5.838887691497803, 4.596747875213623, -10.795854568481445, 8.746231079101562, -8.62955093383789, -5.822454452514648, -8.200421333312988, -6.660248279571533, 0.10729840397834778, -3.169726848602295, 6.067614555358887, -1.6400083303451538, -1.1286811828613281, 8.9237060546875, 0.983185350894928, 2.0085043907165527, -5.115329742431641, 1.990609884262085, 1.4796077013015747, 0.8201529383659363, -1.1883710622787476, -4.381190299987793, -5.29911994934082], [-8.382057189941406, -2.0650999546051025, 7.253012180328369, 1.4564766883850098, 4.774348735809326, -1.0227506160736084, -4.250319480895996, -6.320681095123291, -3.8709604740142822, 0.4452339708805084, 0.4084964096546173, 1.8144409656524658, -1.1218076944351196, 1.6964586973190308, -1.0219199657440186, -4.278297424316406, -4.565404415130615, 2.510671377182007, -1.9033337831497192, 0.12906931340694427, 2.0595755577087402, 1.04513680934906, -1.2118927240371704, -1.2084017992019653, 2.846613645553589, 5.404271602630615, 2.244518756866455, 0.5324822068214417, -3.401158332824707, 7.132405757904053, 4.894455432891846, 2.696768283843994, 1.1024166345596313, -3.854933977127075, -0.10333552956581116, -7.3971757888793945, 8.47443675994873, -0.20876459777355194, 8.646735191345215, 5.801162242889404], [6.945104122161865, -3.9852468967437744, -3.2157998085021973, 7.093287944793701, 0.32878512144088745, 4.6993727684021, -3.6613235473632812, -0.25305548310279846, 2.462000608444214, 1.6572487354278564, -2.377995014190674, -1.7494078874588013, 0.9688133597373962, -0.25451815128326416, -5.780186653137207, -0.9085115194320679, -7.346277236938477, 0.8608310222625732, -0.5349762439727783, 3.536853075027466, 2.4906513690948486, 4.045742034912109, -3.326880931854248, -0.8110154867172241, 0.41895604133605957, 2.1960203647613525, 0.6022756695747375, 2.1304824352264404, 0.4742242991924286, -0.0714302808046341, 5.682568550109863, 2.8010027408599854, 3.634650707244873, 0.02423875778913498, -9.336337089538574, -3.803246021270752, 3.952744722366333, 1.3765236139297485, 2.0328335762023926, -1.786871314048767], [6.2617268562316895, -4.176514148712158, 4.391695976257324, 4.676485061645508, -1.36739182472229, 4.5091094970703125, -1.0457881689071655, -2.9897501468658447, 2.3271737098693848, 3.7798943519592285, -5.197392463684082, -2.265765905380249, -2.9930808544158936, -1.1318997144699097, 0.4234278202056885, 1.151800274848938, -0.08854260295629501, 1.2888929843902588, 1.3777774572372437, 0.9615123271942139, 5.560616493225098, 8.665058135986328, 1.1236928701400757, -4.515591621398926, -0.665786623954773, 0.17549356818199158, 4.3631510734558105, -8.307991027832031, 0.352270245552063, 2.959904909133911, 3.8707854747772217, 2.979332447052002, 0.7716121673583984, 1.342881202697754, 0.799901008605957, 0.2674175798892975, 6.721277713775635, -0.3421182632446289, 4.936816215515137, -7.45979642868042], [1.8044214248657227, -9.963711738586426, -9.119138717651367, 5.759571552276611, 4.297603607177734, 1.3513809442520142, -6.01739501953125, -5.724369525909424, -9.532622337341309, -1.5286427736282349, 0.2302214503288269, 0.9916658997535706, -0.570749819278717, -1.9774388074874878, -10.488140106201172, 9.736953735351562, -0.41183117032051086, 0.8760322332382202, 7.498802661895752, 3.045562505722046, 7.753739833831787, -9.12197208404541, 3.951347827911377, 1.0476704835891724, 2.0144004821777344, 5.667946815490723, -0.5138837695121765, 4.216192722320557, 9.270415306091309, -0.5994178652763367, 1.7350355386734009, 6.5279860496521, -0.6537688970565796, -6.563619136810303, -0.08872317522764206, 2.4306368827819824, -5.117135524749756, 0.6089218854904175, -6.4458327293396, 0.011809066869318485], [4.887279510498047, 3.702432632446289, -10.886018753051758, 8.372859954833984, -12.096636772155762, 5.079713821411133, 4.154453277587891, 6.416476726531982, 3.559364080429077, 0.7809298634529114, -3.878471612930298, 0.4503619372844696, 4.591019153594971, -5.038236141204834, -5.10342264175415, -3.699002981185913, 6.0329718589782715, 0.8783338665962219, -1.383150339126587, 5.046989440917969, 2.27588152885437, 1.576319932937622, 4.782008171081543, -3.4669554233551025, -1.1754934787750244, -7.398573875427246, 7.4283857345581055, -1.3745269775390625, 4.3738298416137695, -1.5299460887908936, -7.110374450683594, 6.902379989624023, 0.14394380152225494, 10.199891090393066, -7.547398567199707, 0.4381897747516632, 4.578487873077393, 9.820454597473145, -0.6039544343948364, 1.0782469511032104], [4.21466064453125, 3.303962230682373, -1.9419890642166138, -4.0360493659973145, -4.340135097503662, -4.410854816436768, 3.094189405441284, 3.7919230461120605, 2.7247586250305176, 4.811069965362549, 3.434901237487793, 6.390918254852295, -0.48501041531562805, 2.3826446533203125, 0.011915553361177444, -3.17097544670105, 2.2808566093444824, 0.055710144340991974, -0.8383587002754211, -1.3978455066680908, -5.065070152282715, -10.485865592956543, 3.139038324356079, 3.7021007537841797, -3.367435932159424, -1.8128345012664795, -6.627087593078613, 3.8599586486816406, 2.697521448135376, -1.619789719581604, 1.6003034114837646, 3.5373198986053467, 4.065911769866943, 1.2503482103347778, -1.7290455102920532, -1.250815987586975, 2.107805013656616, 5.240380764007568, -2.14615797996521, 0.8307415843009949], [-1.1584311723709106, 2.601168394088745, 5.398701190948486, 0.37778767943382263, -6.545492172241211, 2.3886451721191406, 3.2619075775146484, -4.274941444396973, -0.23468568921089172, 1.0504783391952515, -6.896062850952148, -5.756186008453369, -3.662428379058838, 1.8608393669128418, -1.9908546209335327, 4.227921962738037, -1.9203009605407715, 0.12144077569246292, 6.676660537719727, -1.992503046989441, -0.11321479827165604, 8.871763229370117, -0.20278961956501007, -2.2552435398101807, 0.6327115297317505, -0.541020393371582, 1.3045313358306885, -5.337228775024414, 0.7559152245521545, 1.5003509521484375, 0.9795816540718079, -0.02247183583676815, -4.254873275756836, -0.694198489189148, 4.025206089019775, 1.8909945487976074, -4.287537097930908, -2.821594715118408, 3.9278740882873535, -6.983077049255371], [-9.034747123718262, 6.4676289558410645, -3.774716377258301, -3.120377779006958, -4.332281112670898, -3.7103655338287354, 1.5997748374938965, 2.409379005432129, -6.4082489013671875, -1.0899935960769653, 2.4070868492126465, 1.5741572380065918, -2.326664924621582, 1.0206955671310425, -2.0600509643554688, -0.5940732955932617, 4.842191219329834, 0.22636915743350983, -0.5063197016716003, -1.8459049463272095, -5.669223308563232, -6.178225994110107, 1.2728290557861328, 2.9085774421691895, 2.1816699504852295, -1.4457529783248901, -2.382429599761963, 2.6690785884857178, 0.47009649872779846, -2.309258222579956, -4.295952320098877, -4.072111129760742, -1.9461630582809448, 0.9325336813926697, 4.694559574127197, -0.23643088340759277, -4.464018821716309, -0.7559932470321655, -1.6221601963043213, 3.2558610439300537], [0.8130256533622742, 0.5993220806121826, 3.3852202892303467, -6.099258899688721, 0.18668992817401886, -9.59107494354248, -1.3621283769607544, 0.17708714306354523, 1.7392207384109497, 4.8555426597595215, 3.9500162601470947, 1.310210943222046, -1.5520211458206177, 6.935039043426514, 2.9026169776916504, 5.529844284057617, 2.0278022289276123, 1.1056065559387207, 2.258861541748047, -8.459163665771484, 1.417210340499878, -9.38154125213623, 1.4933674335479736, -4.459652900695801, -1.1818034648895264, -1.8019810914993286, -1.9615439176559448, -1.0449162721633911, 3.1632471084594727, 1.4220999479293823, 6.599456310272217, 3.226325750350952, -2.556079626083374, -0.737006664276123, 0.6133290529251099, 1.6152591705322266, -1.0855644941329956, 2.933246612548828, -2.3080856800079346, -4.628059387207031], [0.9261013865470886, -1.5038822889328003, -3.3019003868103027, 2.393764019012451, 1.2368192672729492, -1.4198601245880127, -3.04475998878479, -0.8862693309783936, -2.041699171066284, 5.8035712242126465, 0.5567419528961182, 4.303524971008301, -0.38277995586395264, 0.6017221808433533, -3.0481581687927246, 0.02580522559583187, 4.596444129943848, 1.8164032697677612, -2.8364343643188477, -0.06470416486263275, 0.5211078524589539, -1.5664244890213013, -1.6447176933288574, 1.1652886867523193, -0.3495052456855774, 2.60746693611145, -0.4233628213405609, 1.5954986810684204, 2.9012980461120605, -1.369769811630249, 5.599325180053711, 2.6319668292999268, 1.4646378755569458, -0.8909543752670288, 0.9624254703521729, -2.543208122253418, 2.3279426097869873, 0.4088591933250427, 0.6623225212097168, -1.9709793329238892], [4.553949356079102, -3.1343772411346436, -1.1516836881637573, -0.5406362414360046, 5.600602626800537, -0.7333474159240723, -2.201568126678467, -1.8208783864974976, -4.352816104888916, 10.616270065307617, 4.236398220062256, 6.343230724334717, 0.5179575085639954, -7.418798446655273, -5.78079891204834, -0.028177421540021896, 5.615774631500244, -1.2847594022750854, -5.591040134429932, 2.7285661697387695, 5.811056137084961, -8.388951301574707, 2.8214423656463623, 3.847001552581787, 1.1946810483932495, 1.0230358839035034, 1.3798234462738037, -0.5626924633979797, 6.031286716461182, 2.480046272277832, -5.445865631103516, 1.2297028303146362, -1.6251137256622314, 0.96915602684021, 2.5757362842559814, 1.2086920738220215, -1.697130560874939, -3.9326393604278564, -7.087299823760986, 10.522717475891113], [2.6098873615264893, -6.223090171813965, -3.9664180278778076, 2.871992826461792, 7.892909526824951, -1.4272295236587524, -1.700900912284851, -0.9444381594657898, -0.34878212213516235, 0.1589060127735138, 3.0677478313446045, 4.115324020385742, 4.538372993469238, -1.963393211364746, -1.066701054573059, 3.051158905029297, -2.654940128326416, -0.47040364146232605, 2.1777091026306152, 1.2836253643035889, 2.8798251152038574, -6.416784763336182, 4.857075214385986, 0.6474300622940063, 4.3076958656311035, 2.4847960472106934, 1.4720574617385864, 6.309568405151367, 2.772026777267456, -0.18434685468673706, 5.195888042449951, 4.6457366943359375, -0.9688084721565247, -0.7127974629402161, -3.9604251384735107, -1.3299076557159424, -4.170783996582031, 1.8808099031448364, -3.888068675994873, 3.5876121520996094], [-8.657242774963379, 8.794294357299805, -3.796754837036133, -1.2487361431121826, -13.829465866088867, 5.375316143035889, -1.1731057167053223, -4.278310298919678, -4.399884223937988, 3.5788257122039795, -2.973099946975708, -1.5903915166854858, -3.7073612213134766, 4.89457368850708, -6.748948097229004, -1.3887656927108765, 7.054551601409912, -0.8894360661506653, 3.4816629886627197, -2.753218650817871, -5.116535186767578, -0.2790568768978119, -6.803086757659912, 6.302521705627441, -3.6676888465881348, -3.6386559009552, 2.198970079421997, -4.537361145019531, 6.391205310821533, -4.989009380340576, -8.172945022583008, -1.7355769872665405, 0.5194478631019592, 0.7311401963233948, 5.591694355010986, 2.0480246543884277, -0.5133901238441467, 0.1654331088066101, -1.7799543142318726, -2.557565212249756], [-9.794634819030762, -1.525639295578003, -0.2197510004043579, -1.4130204916000366, 1.9183934926986694, 3.866976022720337, -2.046339750289917, -5.349695682525635, -6.5822930335998535, 2.0885555744171143, 4.604074954986572, 3.5160341262817383, 0.8246133327484131, -0.8029201030731201, -0.028479497879743576, -7.804343223571777, 0.7324709892272949, -5.025313377380371, 4.588098526000977, 1.5873165130615234, -3.692918062210083, -3.798839807510376, -5.528327941894531, 7.366994857788086, -3.117640256881714, -0.3178522288799286, 5.66438102722168, -5.224146366119385, 2.1926348209381104, -0.46841537952423096, -4.20671272277832, 2.799483299255371, 5.185657978057861, -5.826053619384766, -2.1935906410217285, -0.10419756919145584, 2.7959327697753906, 3.630899667739868, 2.408524751663208, 6.969094276428223], [-7.13344669342041, 5.125766277313232, -0.22668923437595367, -4.1870527267456055, 4.714078426361084, -8.060111045837402, 1.647276759147644, 4.898210048675537, -1.7782210111618042, -1.6228317022323608, 4.859557628631592, 0.7843492031097412, 2.7889420986175537, -0.2117135226726532, 4.781240940093994, -4.797858715057373, 2.642374277114868, -2.5857365131378174, -7.3403239250183105, -0.17812751233577728, -7.937180519104004, 3.8161163330078125, -7.8153228759765625, -0.292936235666275, -1.5591317415237427, 1.4073339700698853, -4.454797744750977, 0.4834797978401184, -5.724757671356201, -1.4160223007202148, 1.3740655183792114, -4.704413890838623, -1.2788251638412476, -4.431797027587891, -2.4945995807647705, -0.3198980689048767, -4.142688274383545, -0.18413738906383514, 4.283854961395264, 1.540602684020996], [-1.9828171730041504, -0.0970287173986435, -0.2241002917289734, 3.395526647567749, 1.5355908870697021, 2.3454604148864746, 0.7234591245651245, -2.631364107131958, -1.0358796119689941, 3.2776825428009033, 2.6260974407196045, 2.7388627529144287, 3.4037156105041504, -4.746002674102783, -3.473937749862671, -0.4933888614177704, -2.20505690574646, -5.451505661010742, 1.1128826141357422, 1.9398975372314453, -2.275925874710083, -2.4749298095703125, 2.5438435077667236, 3.2786865234375, 8.288331031799316, 2.0262484550476074, 2.878068208694458, 0.8735118508338928, 2.12917160987854, -3.2770774364471436, -0.7400516271591187, -0.7448024749755859, -3.01305890083313, 2.2936227321624756, -4.765974044799805, -0.09652137756347656, -6.994819641113281, -0.36002564430236816, -6.284823894500732, 7.302729606628418], [4.491002559661865, 3.6441335678100586, 6.403079509735107, -4.130300998687744, -0.370533287525177, -1.4756948947906494, 5.436049461364746, 2.9468886852264404, 11.617110252380371, -3.068488359451294, -2.6306333541870117, -1.8878003358840942, 4.715053558349609, 1.9713568687438965, 4.639059066772461, 0.8804850578308105, -2.9659852981567383, 1.820923089981079, -3.4383628368377686, -2.566420078277588, -2.0529375076293945, 4.683656692504883, -3.1651954650878906, -0.3472180962562561, -1.8680058717727661, 1.805580496788025, -5.295903205871582, 9.883557319641113, -6.001191139221191, -3.2965247631073, 1.327838659286499, -3.92128586769104, -1.6855080127716064, 2.2264702320098877, 0.2902698516845703, -1.059948205947876, -3.400296926498413, -3.5202107429504395, -4.095114231109619, 1.4182275533676147], [-3.120643377304077, -4.746109962463379, 0.5204319953918457, -0.7337194085121155, 7.885636806488037, -0.6544455885887146, -9.189326286315918, -3.966393232345581, -7.984616756439209, 1.7971469163894653, 2.6853983402252197, -4.945623874664307, -4.60322380065918, 1.5106403827667236, -5.130422115325928, 7.101134777069092, -7.710339546203613, -1.0209745168685913, -0.3724789619445801, -3.335554838180542, 5.737574100494385, -2.4159905910491943, -3.0798678398132324, -0.8072164058685303, 7.772775173187256, 0.9473540782928467, 1.5933096408843994, -1.771437644958496, 1.0533831119537354, 2.6917507648468018, 2.1433427333831787, -6.9330902099609375, -3.3911473751068115, -1.9591399431228638, 1.912007212638855, -0.008960863575339317, -6.423447608947754, -10.080240249633789, -3.3368499279022217, 1.1257532835006714], [-17.051055908203125, -0.5794942378997803, -0.10403470695018768, -1.8737077713012695, -10.2356538772583, 4.996857166290283, 0.7384169101715088, -5.656310081481934, -9.756474494934082, -5.599860668182373, -3.2207441329956055, -1.5389869213104248, -4.4164204597473145, 3.7098965644836426, -1.7584855556488037, -0.41172653436660767, 2.4649600982666016, -1.7049477100372314, 7.954636096954346, -3.684814929962158, -1.7546398639678955, -0.6442175507545471, -3.89273738861084, 3.7684342861175537, 0.0425252690911293, -1.1601580381393433, 3.0048253536224365, -2.4474947452545166, 0.6663706302642822, -4.845926761627197, -3.553973436355591, 0.22772987186908722, 5.120948791503906, -3.258761167526245, 9.625383377075195, 0.36817479133605957, 2.114797353744507, 0.39127975702285767, 0.6094817519187927, -0.6669259667396545], [2.566864490509033, 0.9769463539123535, 2.30018949508667, 2.5357415676116943, -12.464407920837402, 2.9949748516082764, 7.468012809753418, 1.945056438446045, 1.1539198160171509, 3.5324270725250244, -7.221667766571045, 2.253051519393921, -5.525943756103516, -0.6669073104858398, 2.6430625915527344, -3.2945473194122314, 12.167800903320312, 5.489228248596191, 5.372542381286621, 0.3465442359447479, -0.5146914720535278, 5.00732421875, 3.887345552444458, -2.363611936569214, -8.545519828796387, -1.4285920858383179, 2.787649154663086, -5.690793037414551, -0.19431592524051666, -1.4934654235839844, 1.6071163415908813, 6.20322847366333, 4.937493324279785, 2.136051654815674, 6.947157859802246, -0.2924000024795532, 12.190673828125, 5.97927713394165, 7.947790145874023, -9.030197143554688], [-5.879315376281738, 3.11600923538208, 1.210707426071167, 1.1756960153579712, -5.86138391494751, 4.0203328132629395, -1.588371753692627, -4.694468021392822, 1.8407325744628906, -5.687271595001221, -3.140639543533325, -7.542165756225586, 1.3397413492202759, 7.4107136726379395, -3.693495035171509, 1.628389835357666, -5.078102111816406, 3.5999608039855957, 12.744858741760254, -2.6375393867492676, -0.032389093190431595, -0.20067504048347473, -5.986213207244873, -1.0943199396133423, -5.757756233215332, 0.2816982865333557, 6.017129421234131, 3.089311122894287, 1.2516469955444336, 0.14270862936973572, -2.2824041843414307, 3.2706680297851562, -1.5616265535354614, -3.812446355819702, -6.607229232788086, -0.5332361459732056, 0.83738112449646, 5.127366542816162, 1.9018316268920898, 0.30969294905662537], [-9.958776473999023, -3.4053189754486084, 2.2238028049468994, -1.9241937398910522, -7.887485980987549, 7.056494235992432, 1.9743422269821167, -8.063194274902344, -4.284464359283447, -6.4516072273254395, -4.2480669021606445, -0.8999592065811157, 1.0210564136505127, 0.2275664210319519, -1.5273520946502686, -3.3446309566497803, -1.8392506837844849, -5.12421989440918, 4.387526988983154, 1.377921223640442, 1.961687445640564, 2.903763771057129, -4.644084930419922, 3.6892130374908447, -4.0971598625183105, -0.7251179814338684, 0.6533238887786865, -4.517784118652344, 1.9959365129470825, -0.1957949697971344, -7.621403694152832, 6.552043437957764, 6.930123329162598, -5.840364456176758, 4.444232940673828, 2.470771551132202, 4.7642388343811035, 3.456886053085327, 0.9912701845169067, 2.83150053024292], [3.3076796531677246, 2.5836195945739746, 0.6964577436447144, 2.42258620262146, 5.708491325378418, -3.7231674194335938, -1.2481389045715332, 0.4792364537715912, -0.014053449966013432, 5.355659008026123, 1.3312251567840576, 0.5184286236763, 0.1399981826543808, -2.474743604660034, -1.2268447875976562, 0.634739339351654, -2.8992958068847656, -2.0022177696228027, -5.808147430419922, 1.8014805316925049, -2.1903860569000244, 4.894177436828613, 2.401553153991699, -1.8652851581573486, 7.275241374969482, 1.292548418045044, -1.657750129699707, -2.9690468311309814, -0.2889057695865631, 3.7437567710876465, 4.686945915222168, -2.8213324546813965, -6.250234603881836, 1.692171573638916, -3.161504030227661, -1.2715445756912231, -6.201247215270996, -3.609206199645996, 3.1365621089935303, -1.5191783905029297], [7.912038803100586, 2.1890571117401123, -2.889209270477295, -0.018231255933642387, -0.7478316426277161, -4.1750664710998535, -0.3679988384246826, 4.408314228057861, -1.1684786081314087, 6.192123889923096, 2.576343536376953, 4.568118572235107, -4.475645542144775, -2.7879421710968018, -1.8728574514389038, 0.9679169654846191, 4.648169040679932, 1.1565930843353271, -6.469592094421387, 1.162914514541626, 0.5062622427940369, -7.2500786781311035, 9.825800895690918, -0.7312589883804321, 3.4995357990264893, -2.0109105110168457, -4.875217437744141, -2.3336496353149414, 3.1582045555114746, 2.700591802597046, -1.1632957458496094, -1.441666603088379, -1.6303693056106567, 5.813343048095703, 2.8040823936462402, 0.5318417549133301, 1.1419073343276978, -0.949126660823822, -2.4144062995910645, -0.994875967502594], [-2.8386776447296143, 3.5935935974121094, -1.326744556427002, -6.260372638702393, -9.18231201171875, 4.498366832733154, 1.1074854135513306, -5.708005428314209, 0.1873878389596939, -0.6404746770858765, -4.014567852020264, -2.114429473876953, 0.04561224207282066, 6.263715744018555, -4.648655414581299, 1.5710018873214722, -1.6354267597198486, 0.2603742182254791, 7.320826053619385, -3.1772139072418213, 1.1662482023239136, -4.590624809265137, -2.4184412956237793, 6.351239204406738, -6.819620609283447, -6.008335113525391, 0.2532179653644562, 1.5649555921554565, 7.3469014167785645, 1.5738242864608765, -9.084752082824707, 4.75227689743042, 1.2761155366897583, -1.4252362251281738, 5.932292461395264, 2.5293192863464355, -2.422667980194092, 0.6701052188873291, -2.6230618953704834, 1.1998876333236694], [5.313146591186523, -6.099269390106201, -3.3902785778045654, 2.945220947265625, 7.99231481552124, -3.6452248096466064, -6.003765106201172, 4.814347267150879, 0.17386363446712494, 1.947536587715149, 4.84264612197876, 0.9468247294425964, 0.803223729133606, -1.4705358743667603, 0.7287629842758179, 1.4920967817306519, 2.3562474250793457, 2.8413238525390625, -5.221513748168945, -0.1749405860900879, 5.57452917098999, -4.060052394866943, -2.393387794494629, -4.927264213562012, -0.6324294209480286, 2.3726911544799805, 1.780617594718933, 2.075773239135742, -1.528315544128418, -1.263802170753479, 5.409668922424316, -0.9561166763305664, 1.5964806079864502, 0.28124216198921204, -5.654955863952637, -1.1989604234695435, 4.888960361480713, 0.49693480134010315, -2.6753649711608887, 0.39513805508613586], [5.929475784301758, 1.9691493511199951, -2.016309976577759, -4.498201370239258, -3.5019516944885254, -0.013213385827839375, -4.079868793487549, -0.04677514731884003, 4.03084135055542, 4.077327251434326, -0.49404481053352356, -4.913943290710449, -2.837677478790283, 9.42857551574707, -2.698090076446533, 5.787159442901611, -7.379352569580078, 0.17257513105869293, 3.683043956756592, -6.944623947143555, 0.18380604684352875, -3.5125255584716797, -4.340153694152832, 1.294595718383789, -1.4209240674972534, -6.0552520751953125, -2.117504119873047, 0.3608666956424713, 4.583065986633301, -0.8111617565155029, 3.5024962425231934, -1.790600299835205, 0.21940039098262787, 1.712729573249817, -1.4326828718185425, 1.1790817975997925, -4.9342570304870605, -2.664569139480591, -3.599120616912842, -8.07221508026123], [7.583220958709717, -8.185826301574707, -2.091076374053955, -1.5857924222946167, 6.491039276123047, -1.7641608715057373, -2.4851295948028564, 3.6319198608398438, -2.516908884048462, -0.7262166142463684, 1.915909767150879, 0.544665515422821, -2.7971746921539307, -3.9246459007263184, 1.3821580410003662, 3.587008476257324, -1.2492051124572754, -0.9091495275497437, -6.509608745574951, 1.2423906326293945, 6.504422187805176, -2.235797882080078, 2.5858867168426514, -1.6317799091339111, 1.5419299602508545, -0.40345871448516846, -5.169572830200195, -1.0590484142303467, -0.950770378112793, 1.0152549743652344, 0.7309455275535583, -2.8539302349090576, 3.3689098358154297, -0.005748352035880089, 4.093603610992432, 2.3577215671539307, 0.5830709338188171, -5.46724796295166, -3.9088659286499023, -1.5669293403625488], [0.8073192238807678, 4.655889987945557, -1.075645923614502, -4.859082221984863, -0.3219940662384033, -6.860716819763184, 0.9748129844665527, 9.137314796447754, 2.820329189300537, 0.2320452183485031, 3.236379384994507, -1.9394464492797852, -1.0826855897903442, 3.33754563331604, 1.84640371799469, 2.563605785369873, 1.0544177293777466, 3.1841979026794434, -4.531249046325684, -5.870797157287598, -3.3464741706848145, -4.34454345703125, -3.7326557636260986, -1.6523276567459106, 0.5181820392608643, -1.2709159851074219, -4.6032514572143555, 8.559539794921875, -4.956785202026367, -5.536768913269043, 2.9157795906066895, -9.210465431213379, -1.535043716430664, 3.8142049312591553, -0.043577346950769424, -1.3750184774398804, -4.735998153686523, -4.479276180267334, -5.535635948181152, 0.15162506699562073], [-0.9119366407394409, 4.181849956512451, -1.5487176179885864, -2.117454767227173, -8.728163719177246, -1.4819663763046265, 4.689543724060059, 9.219171524047852, 0.04909348115324974, -4.461209297180176, -1.9317721128463745, -3.360734701156616, -3.4217636585235596, -1.1964950561523438, 3.6585693359375, -3.3875057697296143, 5.267115116119385, 0.7484087944030762, -5.9882073402404785, -0.10720597207546234, -2.257441520690918, 5.283191204071045, -2.53222918510437, -4.167781829833984, -4.081624507904053, -4.6362199783325195, -3.4333908557891846, -3.727200746536255, -5.872702121734619, -2.7079050540924072, -4.748301982879639, -5.106632709503174, 3.554159641265869, 3.4337356090545654, 2.69321346282959, 1.8469204902648926, 5.218628883361816, 0.5790901780128479, 2.597836971282959, -4.152772426605225]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]], "dtype": "float32"}, {"__nd__": [[-0.49795278906822205, 0.9071056246757507, 0.26289376616477966, -1.401198148727417, 0.3639110326766968, -0.6602045893669128, -0.629126250743866, -1.6209625005722046, 2.453885316848755, -0.5064194798469543, -1.3996905088424683, -0.4106569290161133, 0.12188715487718582, 1.1586107015609741, -0.8062254190444946, -1.132196307182312, 1.3947203159332275, -0.9601513743400574, -0.24415872991085052, 1.248913049697876, 0.007314543705433607, -2.5862128734588623, -0.01915866881608963, -0.022707825526595116, -0.8542375564575195, -1.4517061710357666, -0.43094372749328613, 0.3329795002937317, 0.06830940395593643, 0.4425162672996521, 1.3015930652618408, 1.1641207933425903, 0.5729677081108093, 0.8242462277412415, -0.3695106506347656, -2.2801053524017334, -0.171730637550354, 1.1385548114776611, -1.1620961427688599, 0.626406192779541, -0.49618756771087646, 0.8453544974327087, -0.6755011677742004, -0.6902378797531128, -1.5114761590957642, -1.238778829574585, -0.4678708016872406, -1.8021366596221924, 1.369589924812317, -0.07560531049966812, 0.9458588361740112, -0.6035559773445129, 0.4349993169307709, 0.6739364266395569, 0.4373567998409271, 0.7256230711936951, 0.2646333575248718, -0.2882468104362488, -1.26893949508667, -0.8895449638366699, -0.3766125440597534, -0.09674873948097229, 0.5886800289154053, -0.39802706241607666, 0.8090369701385498, 0.35144859552383423, -1.072837471961975, -0.2736830711364746, -1.3187545537948608, -1.068756341934204, -0.17415501177310944, 0.260063499212265, -0.11198100447654724, -0.47000476717948914, -0.3185606598854065, -0.8431681394577026, -0.4066788852214813, 2.0687289237976074, 0.2711505591869354, -1.9579445123672485, -1.075190544128418, 0.9181632399559021, -1.5653542280197144, -0.14204582571983337, 1.7429618835449219, -0.1911378651857376, 0.4871656000614166, 0.09801697731018066, -0.846275269985199, 1.3442105054855347, -0.4451044797897339, 1.1930162906646729, -1.4092973470687866, 0.17644761502742767, 1.47214937210083, 0.7202909588813782], [-0.6945894956588745, -0.7533285617828369, 1.0080952644348145, 1.187495231628418, 0.37781980633735657, -1.7775661945343018, 0.9806830883026123, 2.1965959072113037, 0.49401697516441345, -1.740050196647644, -1.9020121097564697, 0.13613061606884003, 0.1635597050189972, -1.6070481538772583, -1.57585871219635, 0.6684826612472534, -0.2359996736049652, 0.6907364726066589, -0.8321084380149841, -0.2195291519165039, -0.4339107275009155, 0.5658369660377502, 2.2520196437835693, 0.0738578662276268, 0.18852777779102325, 0.7725536823272705, 0.059030093252658844, -0.6160983443260193, 0.12117034196853638, -0.7597494125366211, 0.9799633622169495, -1.1431952714920044, 0.47183752059936523, 0.0862109586596489, -0.6162278056144714, -0.11722469329833984, -0.61776202917099, 0.9345230460166931, 0.06085528805851936, -1.4126583337783813, 0.48712053894996643, -0.08574935048818588, -1.8016637563705444, -1.6464998722076416, 1.1172759532928467, -0.8870353698730469, -0.32910168170928955, 0.5005953907966614, -1.2827403545379639, -0.5848491787910461, -0.6972424983978271, -1.0842139720916748, -0.571546196937561, 0.2732192575931549, -1.132301688194275, -0.13160641491413116, -0.041324056684970856, 0.8564432859420776, -0.02753492072224617, -0.23107482492923737, 0.5181928873062134, 0.3622864782810211, 0.5048407316207886, 1.0502057075500488, -0.815879762172699, 0.4531157612800598, 0.0014811510918661952, -1.0134453773498535, 0.8385912775993347, 1.5005077123641968, 0.6084892153739929, -1.7855366468429565, 1.7991013526916504, 0.5920233130455017, 0.7708218097686768, -1.1749951839447021, 0.32176876068115234, 1.2176711559295654, -0.9063076972961426, -0.4035940170288086, -2.408651351928711, 1.3349159955978394, 1.713362455368042, 1.0888680219650269, -1.414018988609314, 0.6039206385612488, -0.3037649095058441, -1.8177177906036377, -0.8264444470405579, 0.8408046364784241, -1.0133486986160278, 0.12162350118160248, 1.2013206481933594, 0.2886587977409363, -0.8716387152671814, -0.15070563554763794], [-0.43582242727279663, -1.1686877012252808, -0.36938127875328064, 0.8399848937988281, -0.6851648092269897, 1.3787250518798828, 1.4485273361206055, -0.5881469249725342, 0.12648241221904755, 1.3206477165222168, 0.9335799813270569, -0.02970162034034729, 0.2248820811510086, -0.22568784654140472, -0.45399197936058044, -0.058496396988630295, 1.085605263710022, 0.048564840108156204, 0.6785559058189392, 0.7633587121963501, -0.08840077370405197, -0.4696447551250458, 0.29957279562950134, -0.9968749284744263, 0.7426887154579163, -0.22771291434764862, -1.4379513263702393, -0.33600133657455444, -1.6371142864227295, -0.213825523853302, 1.14036226272583, 1.5936816930770874, 0.021696502342820168, -0.4610954821109772, -0.7327889800071716, 0.6268059015274048, 0.1280675232410431, 0.4912046492099762, -0.42118901014328003, -0.4220081865787506, -0.39053329825401306, -0.5336024165153503, 1.4688233137130737, -0.2690778374671936, 0.10794682055711746, 1.6432377099990845, -1.460132122039795, 0.38496047258377075, 0.04505527764558792, 0.296792209148407, -0.9738928079605103, -0.16252650320529938, -0.6680938601493835, 1.541739583015442, -0.7827043533325195, 0.134311705827713, 0.296099454164505, -0.22313347458839417, 0.8949829936027527, -1.1263643503189087, -0.7466985583305359, 0.4610041379928589, -0.004559234715998173, -0.23520204424858093, -0.5330246686935425, 0.8827319145202637, 1.2857816219329834, 0.08222571760416031, -0.09779711812734604, -0.23515260219573975, -0.2916783392429352, 1.9158343076705933, 0.4543007016181946, 1.3150362968444824, -0.6457900404930115, 0.7055777311325073, -0.21141457557678223, -0.1997203230857849, 0.26317131519317627, -0.4246659576892853, 0.7075518369674683, -2.379399061203003, -1.5253639221191406, -0.1862526684999466, -1.227222204208374, 1.598136067390442, 0.361646831035614, -1.5977619886398315, 0.5935990214347839, 1.780019760131836, -0.20186029374599457, 0.1690872311592102, 0.3446178734302521, -1.1512935161590576, 0.45261135697364807, -0.5522788166999817], [-0.006241438910365105, 0.36090996861457825, -1.448960781097412, -0.6422763466835022, -0.026445189490914345, 0.4553539752960205, 1.7707860469818115, 0.9271524548530579, -0.6097609996795654, 1.5319663286209106, -0.6844172477722168, 1.6417683362960815, -1.7569941282272339, -0.3784109950065613, 1.3031738996505737, -1.5854904651641846, -2.264111280441284, 1.2650066614151, -0.3272346258163452, 0.9025905132293701, -0.43085330724716187, 1.3682122230529785, 2.30863356590271, 0.7841261029243469, -0.9586005210876465, -0.310547411441803, -0.5509255528450012, -0.021156935021281242, 0.4560758173465729, -0.37632080912590027, -0.7271192669868469, 1.174673318862915, 0.22328084707260132, 1.4489799737930298, -0.9835121035575867, 1.5408847332000732, 0.07550261169672012, -0.6448619961738586, 0.06367002427577972, -1.0998731851577759, -1.275716781616211, -1.2025408744812012, 0.6622570157051086, -1.126190423965454, -0.06942912936210632, 0.17846357822418213, -1.5971040725708008, 0.5545268058776855, -1.2929162979125977, 1.8170456886291504, -0.4157670736312866, 1.5833371877670288, 0.9139670729637146, 0.23237662017345428, -0.8756034970283508, 0.0017870597075670958, 0.8387735486030579, -0.014401113614439964, 0.10554665327072144, 0.08762677013874054, 0.09334404021501541, 2.2366292476654053, 0.9469573497772217, -0.03171543404459953, 0.5998403429985046, 0.5002793669700623, 0.384071409702301, 1.627465009689331, -0.407040536403656, 0.7329308986663818, 0.537267804145813, 0.17957834899425507, -0.7253226637840271, 1.7614502906799316, 0.8112240433692932, 1.3355698585510254, 0.48769351840019226, 0.36633187532424927, 0.057538505643606186, -0.6364378929138184, -1.8784000873565674, -1.6228797435760498, 0.9217084646224976, -0.5868649482727051, 0.12367870658636093, 0.729232907295227, -0.26214781403541565, -0.9549933075904846, -0.9674780964851379, -0.5430068969726562, -1.267194151878357, -0.6718016266822815, 0.20139093697071075, -0.014255517162382603, -0.5397071242332458, -0.9398757219314575], [-0.6240309476852417, 0.7226833701133728, 0.15261080861091614, 0.6606891751289368, 2.1143593788146973, -0.5842588543891907, -0.6557784080505371, 0.6054374575614929, -1.0069384574890137, -0.13090236485004425, -0.41022562980651855, -0.5714211463928223, -0.1739046424627304, -0.08973930776119232, 2.5498924255371094, 1.6900784969329834, -0.42611101269721985, 0.10486975312232971, -0.6722941398620605, 0.2071312963962555, 0.22452130913734436, -0.9934946894645691, 0.696973979473114, 0.9959542751312256, -0.33484312891960144, -1.8655940294265747, -0.175608828663826, 1.453181266784668, -0.05983706936240196, 0.7900193929672241, 0.16178911924362183, 0.24890387058258057, 0.35231679677963257, -0.510332465171814, 0.6793636083602905, 0.38870933651924133, -0.004874637350440025, -0.10136209428310394, -0.5875624418258667, 1.2588053941726685, 0.21309983730316162, 0.6395609378814697, 0.05369439348578453, -0.6469739079475403, -1.017188310623169, 1.5248262882232666, 0.9177829027175903, -0.24939103424549103, -1.7933933734893799, 0.5806452631950378, 0.584466278553009, -0.5310268998146057, 0.3817123472690582, 0.00683584064245224, -0.8949990272521973, 0.2626420855522156, 0.9742196202278137, -0.2341984361410141, -0.2099805772304535, -0.9160048365592957, -1.1067479848861694, 0.668129563331604, -0.47448521852493286, -0.8589116930961609, 0.5430090427398682, -0.6096816658973694, 0.2553333640098572, -1.1254719495773315, 0.3777589499950409, 0.06258653104305267, -0.695708692073822, 1.6627748012542725, -0.6883028745651245, 1.3721072673797607, 0.850386917591095, -0.818960964679718, -0.20149743556976318, 1.3135793209075928, -0.17503724992275238, -1.408103585243225, 1.3306992053985596, 0.6992471814155579, -0.08837329596281052, -1.0754303932189941, -1.0240864753723145, 1.5744622945785522, -1.269956111907959, 0.9128707051277161, -0.6733980774879456, 1.195202112197876, 0.9741679430007935, -0.27697810530662537, -0.3545531630516052, 1.299951195716858, 1.3261185884475708, 3.2254867553710938], [-2.1039438247680664, -0.998136579990387, 0.7497047781944275, -0.9163962602615356, -1.2616593837738037, -1.4050238132476807, -0.013596647419035435, 1.5041604042053223, -0.17687830328941345, 0.4454765021800995, 0.2859637141227722, -1.7895114421844482, 0.48649001121520996, -0.4119691550731659, -0.31454843282699585, 1.7022244930267334, 0.7589757442474365, 0.19594205915927887, -1.081047534942627, -0.28321337699890137, -0.9783759713172913, -0.2384425401687622, 0.8568532466888428, 0.09171908348798752, -0.8494375944137573, 0.2889663577079773, -1.5663468837738037, -0.2364378273487091, 0.8794224858283997, 0.4620170593261719, -0.42421481013298035, -0.38074371218681335, -1.0680180788040161, 0.5991125702857971, -1.4848897457122803, 0.6723499894142151, 0.05535924434661865, 0.28647127747535706, -0.32812175154685974, 0.8183506727218628, 0.7964516878128052, 0.22343838214874268, 0.26571741700172424, 0.9651152491569519, 0.9239421486854553, -0.6457239985466003, 0.02770305424928665, -1.4484840631484985, 0.4641319215297699, 1.3335696458816528, -0.4746023714542389, -0.5514082908630371, 0.5382747650146484, 1.136925220489502, -0.5828882455825806, -0.16359710693359375, 0.32138943672180176, -0.5065733194351196, -1.0848009586334229, 0.37880828976631165, -0.9948378801345825, 0.9257183074951172, 1.0687999725341797, -0.7620264887809753, 0.11565215140581131, 0.6456112861633301, 0.003915951121598482, -0.047814033925533295, 0.5712251663208008, 2.19088077545166, 0.7471486330032349, 0.8475247621536255, -0.6043049693107605, -0.06751503050327301, -0.6471095085144043, -2.99748158454895, 0.09809450060129166, 0.25149431824684143, -0.45680612325668335, -0.2259976714849472, -0.7079541683197021, -0.738388180732727, 0.26416561007499695, 0.3570207953453064, 0.4245726466178894, -0.34357792139053345, -0.6634505391120911, -1.142197847366333, 1.6037611961364746, -0.5756776928901672, -0.1969238519668579, 0.32143881916999817, -2.196554183959961, 0.08822394162416458, 1.0566025972366333, 0.07355085015296936], [-0.4703666865825653, 0.12382535636425018, -0.5646469593048096, 0.05780757591128349, -1.6681294441223145, -1.5445283651351929, -0.35648804903030396, 0.07682168483734131, -0.4228613078594208, -1.3330023288726807, -0.4401605427265167, -0.4427315592765808, -1.8809133768081665, 0.45930466055870056, -0.5592933893203735, -0.15558938682079315, -0.6364084482192993, -0.6859750151634216, 0.3067642152309418, 0.49030032753944397, -0.6907861828804016, -1.0022473335266113, 1.5126149654388428, -0.12749788165092468, -1.6945942640304565, -0.07712822407484055, 1.068579912185669, 0.0249099750071764, -0.07262753695249557, 0.18895794451236725, 0.28380587697029114, 0.7694535255432129, -1.2019699811935425, 0.4044361412525177, 0.7896705865859985, -0.9700336456298828, -1.3423179388046265, -0.14704692363739014, 0.8786119818687439, -1.631941318511963, -0.14761878550052643, 0.8228174448013306, -1.6508259773254395, 1.362518072128296, -0.17462164163589478, -0.8288095593452454, 1.470057725906372, 0.4656227231025696, 0.6624540090560913, 0.6886785626411438, -0.48696500062942505, 0.008323430083692074, -0.5344892740249634, -1.8060617446899414, -0.890058159828186, 0.9281018376350403, 2.2086164951324463, 1.5036368370056152, 0.31871500611305237, -0.7437292337417603, -0.36843109130859375, 1.0033177137374878, -0.508159339427948, -0.4354137182235718, -0.03879166021943092, -0.7792671322822571, -0.15405836701393127, 0.007017816416919231, 1.2630183696746826, 0.47429123520851135, -1.0363376140594482, 0.28819766640663147, -1.4129877090454102, 1.1624864339828491, 0.6377509236335754, 0.8318082094192505, 0.45608508586883545, 0.7285357713699341, 0.7142336368560791, 0.23692792654037476, -0.8879398107528687, 0.08670589327812195, -2.290158271789551, -0.42315399646759033, -0.879625678062439, -1.1374400854110718, 0.8487400412559509, -2.0932364463806152, 1.1188970804214478, 2.8721423149108887, 1.5181766748428345, -0.036856360733509064, 0.7465159296989441, -0.7712705135345459, 1.1443958282470703, 0.04872975870966911], [0.9012510776519775, 1.7253258228302002, -0.30104175209999084, -0.4010622501373291, 1.6385867595672607, 0.6049880385398865, -0.9316867589950562, 1.3972030878067017, -1.227165937423706, 0.787543535232544, -1.2334636449813843, -0.023279784247279167, -1.6198776960372925, 0.5609673261642456, 1.3864985704421997, -0.6123051047325134, -0.2365586757659912, -0.3211665451526642, 0.2902032732963562, -1.276986837387085, 0.8112168312072754, -0.3251607120037079, -1.1579283475875854, 1.2914843559265137, -1.4956941604614258, 1.421221137046814, -0.11259917169809341, -0.4156803488731384, -1.2119336128234863, -0.7617514729499817, -0.7773157954216003, -1.132198452949524, -0.4836583435535431, 0.14156924188137054, 1.129356026649475, -2.3238511085510254, -0.9831873774528503, -0.18155480921268463, 0.49274271726608276, 0.3799891471862793, 0.5499352216720581, -0.9764860272407532, -0.8042727112770081, 0.8056177496910095, 1.0286918878555298, -0.2873384952545166, 0.5874477028846741, -1.3359079360961914, 0.24328310787677765, 0.529800295829773, -0.38812193274497986, 2.697082281112671, 0.5666472911834717, 1.044291615486145, -0.16742639243602753, -1.2509218454360962, -0.030698098242282867, -0.25923267006874084, 0.4865227937698364, -0.9617146849632263, 0.2947951853275299, 0.4070988893508911, 0.5520862340927124, 0.2767235040664673, -0.25348371267318726, 0.8893678188323975, -0.49147483706474304, -0.5341799259185791, -0.6171811819076538, -1.1052651405334473, 1.00263249874115, -1.277819275856018, -1.3277851343154907, -0.9930272102355957, -0.7351406812667847, 0.03734526038169861, -0.8093864917755127, 0.6671777963638306, -1.0803413391113281, 0.5070992112159729, 1.3192802667617798, -0.39230379462242126, -2.6665611267089844, -2.1977272033691406, -0.09437486529350281, 1.1465519666671753, -0.011832315474748611, 0.5285620093345642, -2.1338062286376953, 0.9260376691818237, -0.14381931722164154, -1.925286054611206, 1.1013741493225098, 0.29036930203437805, -1.0934717655181885, -0.7287799715995789], [-0.3246835768222809, -0.8073686361312866, 0.48169630765914917, -0.10798284411430359, 0.015277745202183723, -0.1376291811466217, -1.0267504453659058, 0.8576993346214294, -0.36315757036209106, 0.331756591796875, 0.012063615024089813, -2.350733995437622, -0.3040810823440552, 0.04250214993953705, -1.408835768699646, -1.5109643936157227, 0.3606606423854828, -1.4970154762268066, 0.8479454517364502, -1.698587417602539, -1.2433760166168213, -0.2748022675514221, 0.6805523037910461, 0.26579973101615906, -0.617192804813385, 0.8394040465354919, 0.7016069293022156, -1.0413578748703003, -0.17175151407718658, 0.016139637678861618, -0.3929236829280853, 0.20455290377140045, 0.06373818963766098, -1.0208239555358887, -0.8078200221061707, 0.22596514225006104, -1.1204410791397095, 0.8613828420639038, 0.34936845302581787, -0.3047407269477844, -0.07092124223709106, -0.8487991094589233, -1.0285950899124146, 0.1835770606994629, 0.5604608654975891, -0.12774182856082916, -0.48920756578445435, -1.5892386436462402, -0.3584495484828949, 0.5757064819335938, -1.2478467226028442, -0.7473973035812378, -0.5660526156425476, 0.1010410338640213, 0.2550380825996399, 0.7779929637908936, 0.13250324130058289, 1.3464027643203735, -0.2793295979499817, -0.12232217192649841, 0.1268683224916458, 1.6820622682571411, -0.3707502484321594, 0.5658265948295593, 1.0263909101486206, 0.003699560882523656, 0.8180952072143555, -0.6974077820777893, 0.97198486328125, -1.2128368616104126, -0.49158820509910583, -0.8776080012321472, 0.7795112133026123, 0.14791645109653473, 1.5801769495010376, -0.13102227449417114, -0.5678715109825134, -0.9654333591461182, 1.1320037841796875, 0.3685697317123413, 1.1612547636032104, 1.946276068687439, 2.2819032669067383, 1.1950631141662598, -2.1831018924713135, 2.2635836601257324, 0.4715476930141449, -0.48182952404022217, -0.815723180770874, -0.3230040669441223, 1.0318104028701782, -1.1138206720352173, 0.6942121982574463, -0.8002138733863831, -0.7812758684158325, -0.7020620107650757], [0.04643898084759712, 0.5151606798171997, -0.7387146353721619, 0.024483809247612953, 1.3994388580322266, 0.05855577439069748, 1.6634920835494995, -1.5847493410110474, -0.31960195302963257, 0.3751198947429657, 1.0195950269699097, -0.567138671875, 0.1495487093925476, -1.0566599369049072, -0.15512177348136902, 0.007523552514612675, 0.5757823586463928, 0.3912578821182251, -0.3721323311328888, 1.890181064605713, 0.36495235562324524, 0.7951963543891907, -0.034490637481212616, 0.15630978345870972, 0.6978076696395874, -1.902329444885254, -0.47625479102134705, -0.38679414987564087, -0.8669521808624268, 0.5184411406517029, 0.842341959476471, 0.08557749539613724, 0.23271040618419647, 1.6818264722824097, 1.2052546739578247, 0.7703916430473328, -1.6565767526626587, -1.178814172744751, -0.08464693278074265, -2.58064341545105, 0.07295971363782883, 0.37662509083747864, 0.3882305920124054, -0.5180512070655823, 1.5769269466400146, 1.0518733263015747, -1.0784287452697754, 0.7108385562896729, -0.17098361253738403, 0.5813629031181335, 0.4344177544116974, 1.464686393737793, 1.8827143907546997, -0.3196091055870056, -0.49234533309936523, 0.9737237095832825, -0.9721001386642456, -1.3961256742477417, 0.057473745197057724, -0.9823413491249084, 0.24509906768798828, -0.759467601776123, 0.26916101574897766, -0.5070133805274963, -1.280462622642517, 1.1183925867080688, 0.3216203451156616, -0.17343533039093018, -2.2328834533691406, -0.5759223103523254, 0.22990986704826355, -0.10710801184177399, 0.6928749084472656, 0.931574821472168, 1.351051688194275, 1.4411931037902832, 0.48833438754081726, -0.07527845352888107, 0.3707844913005829, 0.2538926303386688, -0.39814651012420654, 0.003555546049028635, 0.31638726592063904, -0.12648239731788635, -0.8742060661315918, 0.27196913957595825, 0.604417085647583, 0.7950952649116516, 0.9055721163749695, 0.05032901465892792, -0.9801292419433594, 0.001739636529237032, -0.2504304051399231, -0.502291738986969, 0.5652589201927185, 0.23610611259937286], [-1.5724283456802368, -1.8300501108169556, -0.5965132117271423, 0.11768470704555511, -1.1415480375289917, -0.0865742564201355, -0.19856445491313934, 0.005678272806107998, -0.0924263671040535, 0.915302574634552, 0.1136927679181099, -0.02728560008108616, -0.8987405300140381, 0.9633285403251648, 0.60666424036026, 0.1401224285364151, 0.21648524701595306, 0.02239510603249073, -0.3170478940010071, -1.36604642868042, 0.6393730640411377, 0.9107131958007812, 0.2964033782482147, -2.876631259918213, -0.4176587462425232, 0.7840473651885986, -0.8420780897140503, 1.6168659925460815, 0.7128962278366089, -1.4725819826126099, -0.7036986947059631, -0.5574815273284912, -1.6339799165725708, -0.36831533908843994, -0.46124693751335144, -0.9009546041488647, 0.930688202381134, 0.6629396080970764, -0.47217822074890137, -1.1063238382339478, -1.4917078018188477, -0.36169955134391785, 1.1953033208847046, 0.38745495676994324, 1.5054527521133423, -0.20923392474651337, 0.8925318121910095, -0.24515056610107422, -0.9315056204795837, 1.5990756750106812, 1.655969262123108, 1.7582753896713257, 1.5199209451675415, -1.9596490859985352, 0.9032212495803833, 0.9243226051330566, -1.3951618671417236, -1.9160583019256592, 0.004576630890369415, 0.6437805891036987, -0.12184948474168777, -0.21526826918125153, -0.1832956224679947, 1.157203197479248, -0.4605427384376526, -0.6387475728988647, -0.20896625518798828, -0.5069522261619568, 0.2748439610004425, -0.33485352993011475, 0.3812362551689148, 0.021940505132079124, 1.9125508069992065, -0.5686619877815247, -0.9306972622871399, 0.060448601841926575, 0.36363279819488525, -0.9735159277915955, 1.8853760957717896, 0.9242631793022156, -0.5890610814094543, 0.6142690777778625, 1.2807704210281372, 0.37480178475379944, 0.17240436375141144, 1.1986325979232788, 0.7369639873504639, -0.7269115447998047, -0.7191872000694275, 0.4943925440311432, 1.1312313079833984, 0.6862639784812927, 0.32566285133361816, -0.5240622758865356, 0.5440499186515808, 0.021913956850767136], [-0.685411810874939, -0.3666370213031769, 0.9720906019210815, 0.36682894825935364, -0.5276739001274109, 1.1290518045425415, -0.1597057431936264, -1.2952959537506104, 0.5395038723945618, -1.0957354307174683, -0.4011271595954895, 0.8547888398170471, -0.6792475581169128, -0.5145795345306396, -0.06217261403799057, -0.6610673666000366, -0.4185587167739868, 0.9855679869651794, 0.6044508218765259, 0.29983872175216675, 0.06102975085377693, 0.4169212281703949, -0.29284432530403137, 0.5990751385688782, -0.48019200563430786, 0.541928768157959, 0.7858368158340454, 0.4563625752925873, -0.3925577402114868, -0.7969385385513306, 1.3956032991409302, -2.102741241455078, 1.3324054479599, -0.5313189625740051, 0.3529857099056244, 0.6633933782577515, 0.05991455167531967, 1.6647374629974365, -0.6155417561531067, 1.9155855178833008, -1.12447190284729, 0.509036660194397, -0.8695082664489746, -1.400797724723816, -0.9299245476722717, 0.6154340505599976, -0.1061665341258049, 1.175481915473938, 0.6393640041351318, 0.5816953182220459, -1.4969353675842285, 0.7776058912277222, 0.03015938028693199, 0.5900951623916626, 0.5435627102851868, -0.8636648654937744, 0.17364206910133362, -0.9014608860015869, 1.152024269104004, 0.15417979657649994, 0.9640173316001892, -0.9075795412063599, -0.3535751700401306, -0.08069099485874176, 0.4505394995212555, -1.7556829452514648, 0.7019816040992737, -0.7650355696678162, 0.8242113590240479, 0.15337318181991577, 0.642098605632782, -0.06801434606313705, -0.11221513897180557, -0.30613213777542114, -0.8773781657218933, 1.4536207914352417, -0.2918783724308014, -0.41455167531967163, 1.216259241104126, -0.02158910036087036, 0.07182838767766953, 1.0076780319213867, -0.7759730815887451, -0.6775530576705933, 0.8699404001235962, -0.4021363854408264, -0.17033787071704865, -0.7064419388771057, 2.13275146484375, 1.8255223035812378, -1.0593459606170654, 0.45547839999198914, -0.21642647683620453, -0.9906754493713379, -0.1306656301021576, -0.800820529460907], [0.28585678339004517, 0.2859191298484802, 0.8185268640518188, 0.0490758940577507, 2.1158759593963623, 1.6968071460723877, 1.8985238075256348, 0.010127409361302853, -0.7794955372810364, -0.8186898827552795, 2.0400023460388184, -1.3508161306381226, 0.4080314040184021, 0.03421156853437424, -0.8043286800384521, 0.0747670829296112, 1.5160232782363892, 0.869463324546814, -1.0396002531051636, -0.40371131896972656, 1.3372100591659546, -0.3457934856414795, 0.6263850927352905, 1.1839255094528198, 0.038307949900627136, -0.20167428255081177, 0.20398183166980743, -1.115766167640686, 0.49383819103240967, 1.395468831062317, -1.9461045265197754, 2.5765910148620605, -0.24014973640441895, -0.9004629254341125, 0.4366162419319153, 0.04422471672296524, -0.12152771651744843, 1.0269304513931274, 0.2561244070529938, -0.9829571843147278, -1.2870314121246338, 0.44657662510871887, 0.4258951246738434, -1.7090966701507568, -1.0679106712341309, -0.9928019642829895, -0.2963126301765442, -0.318449467420578, 1.4290415048599243, -0.03199300915002823, -1.1791231632232666, 0.9000323414802551, 0.24630244076251984, -0.28175923228263855, -2.083613634109497, -0.03800005093216896, -1.9071491956710815, 0.8014316558837891, -1.2413030862808228, -0.5322567820549011, -0.14966586232185364, 0.3945961892604828, -1.1376562118530273, 0.9355284571647644, -1.1610863208770752, 0.012084339745342731, -0.45458242297172546, -1.5106738805770874, 0.14944985508918762, 1.557963490486145, 0.05899422988295555, -0.5258286595344543, 0.3242796063423157, 0.14056873321533203, -0.20747317373752594, -0.8380531072616577, -1.4502580165863037, 1.0756433010101318, -2.384476900100708, 0.5278165340423584, 0.438652902841568, 1.119884967803955, -0.22993986308574677, 0.22405236959457397, 0.5435740351676941, -0.8538905382156372, -0.14084307849407196, -0.688944935798645, -0.21434062719345093, 0.24925760924816132, 0.8028906583786011, 1.4538745880126953, 0.11283445358276367, 0.09338792413473129, 0.432371586561203, -0.5835219025611877], [1.2392815351486206, 0.6846833825111389, -1.5795626640319824, -0.9202082753181458, 0.5468029379844666, -0.47567200660705566, -0.5391581654548645, -0.5761483907699585, 1.6092705726623535, 1.1979619264602661, -1.606908917427063, -1.1188017129898071, 0.1401531845331192, -1.7945579290390015, 0.5038855075836182, 1.0296522378921509, -0.6316300630569458, -0.8378907442092896, 0.6988952159881592, -1.2702510356903076, 2.503932237625122, 0.32456454634666443, 0.234671488404274, -0.6544645428657532, 1.8205598592758179, -0.11096768826246262, 0.7759998440742493, 0.2891191244125366, -0.7181664109230042, -0.5596407055854797, 0.1052095964550972, -1.3900433778762817, 0.5222809910774231, -0.6897312998771667, -0.04910663142800331, -0.9991747140884399, -0.006217981688678265, -1.769903540611267, -1.7835170030593872, -0.748916506767273, 0.6916004419326782, 1.3147494792938232, -0.45521295070648193, -0.25456786155700684, 0.5278615951538086, -1.4419602155685425, -0.8133334517478943, -0.008487074635922909, -0.33194732666015625, 0.23204471170902252, -1.1207101345062256, -0.20115700364112854, 0.9108837246894836, -1.1820052862167358, 0.06355730444192886, 0.1868634670972824, 0.9985330700874329, 0.8445019125938416, 1.2803058624267578, 1.6429332494735718, 3.405303716659546, -1.7110322713851929, 1.1567277908325195, 0.5268872380256653, -1.1201045513153076, 1.1873594522476196, -1.254646897315979, -0.9036970734596252, -1.871256709098816, -1.6066937446594238, 1.5137217044830322, 0.10179511457681656, 0.3475266396999359, 0.5033748745918274, -1.271618127822876, -0.5003936290740967, 0.09319432824850082, 0.054982028901576996, 1.8269116878509521, -0.27956801652908325, -0.6144214272499084, -0.9689426422119141, -1.1927934885025024, -0.6313275694847107, -0.8317966461181641, 0.3898947238922119, -0.39121997356414795, -1.140873670578003, -1.2031835317611694, -0.24174438416957855, -0.7453365325927734, 0.37908560037612915, 0.8646543025970459, -0.9562906622886658, 0.8287451863288879, -0.5965794324874878], [-0.3955351710319519, 0.5734788775444031, 1.0524930953979492, -0.31730547547340393, 1.7203187942504883, -2.0890989303588867, 0.06690102815628052, 1.4781148433685303, -0.5716235637664795, 0.5122388005256653, -1.7095575332641602, 1.088561773300171, -0.6571959257125854, 0.7982277870178223, 2.0577995777130127, 0.19464737176895142, -0.32361167669296265, 0.5603563189506531, 0.2714861035346985, 1.0290173292160034, 1.910738229751587, -0.08997954428195953, 2.349661111831665, -0.11920548230409622, -1.4017462730407715, 0.20398060977458954, -0.02734016813337803, -0.48220011591911316, 0.716255784034729, -0.18797940015792847, -0.4929267168045044, 0.2004004716873169, -0.40651950240135193, -0.09237947314977646, 0.14227840304374695, -0.7202656865119934, -0.12746375799179077, -0.4130651652812958, -0.15015430748462677, 0.3672732412815094, -1.2923767566680908, -2.1993448734283447, -0.06070692464709282, 0.24496866762638092, -0.22810505330562592, -0.25328588485717773, -0.9070162773132324, -1.5960850715637207, 0.08973094075918198, 0.8203177452087402, 0.12809628248214722, 0.764723539352417, -0.8503777384757996, 1.3450766801834106, 2.4880361557006836, -0.23070871829986572, -0.15031270682811737, -0.5832374691963196, 0.5601776838302612, 0.8116126656532288, -1.2162271738052368, 2.237157106399536, 0.590937614440918, 0.4985203444957733, -1.1771782636642456, 1.2256191968917847, -0.026174498721957207, -0.23298616707324982, 1.5426537990570068, 0.8871373534202576, -0.3720022737979889, 0.4259101450443268, -0.45164528489112854, 0.348984032869339, 0.30194947123527527, -0.27939072251319885, -0.09392989426851273, 0.7324493527412415, -0.9733847975730896, -0.7683209180831909, -0.12476533651351929, 0.4178542196750641, -0.6847026348114014, 2.036423444747925, 0.0666450783610344, 2.092726230621338, -1.0984166860580444, 0.654473602771759, 0.09883645176887512, 0.9443584084510803, -2.999506950378418, -0.8507953882217407, 1.704089641571045, -0.6237859725952148, 0.1424839049577713, 0.8811883330345154], [2.3324623107910156, -0.5583322644233704, -0.12539736926555634, 0.21287287771701813, -1.5264250040054321, -0.3968278765678406, -0.4857129454612732, 0.12501326203346252, 1.752380132675171, 1.649229645729065, -0.1531098634004593, -0.7762449979782104, 0.5312752723693848, 1.6591674089431763, -0.5195257663726807, -0.8778945207595825, 1.1954903602600098, 1.310590386390686, 0.9384272694587708, 2.099360942840576, -0.16894960403442383, -0.47527575492858887, 0.6582354307174683, 0.8320094347000122, -0.7213571667671204, 1.0252633094787598, -2.194885492324829, 0.3973413109779358, 0.7489414811134338, -1.861934781074524, 1.3176850080490112, 1.2850191593170166, 0.27440527081489563, -0.213458850979805, 0.5985321402549744, 0.5844821333885193, 0.5914268493652344, -0.8100637793540955, 0.9063379764556885, -1.3708833456039429, 1.759494662284851, 1.503660798072815, 0.19814714789390564, 1.5545710325241089, -2.5656354427337646, 0.6663287878036499, 0.5288040637969971, 0.690639317035675, 1.280657172203064, 0.2619255781173706, -0.8946475982666016, -1.3401330709457397, 0.5708303451538086, 0.1476958841085434, 1.0363194942474365, -0.885251522064209, 0.34965333342552185, 0.025859083980321884, 1.9344335794448853, -0.5309504866600037, -1.6646753549575806, -0.8853256702423096, -0.6318628787994385, -1.5664722919464111, 0.9518669247627258, 0.07425738871097565, 0.07470640540122986, -0.33626773953437805, 0.3714919090270996, -1.8586947917938232, 1.3426703214645386, 0.7857221364974976, 0.9243856072425842, -1.1715470552444458, 0.15134096145629883, -2.008882761001587, -1.9925429821014404, 1.8912769556045532, -0.30756670236587524, -1.6722570657730103, 1.0709437131881714, -0.7068649530410767, -0.3791760206222534, 0.732623279094696, -0.47651344537734985, 1.214637041091919, 0.2319381684064865, 0.21733877062797546, -0.15572944283485413, -0.4446440041065216, -0.6844009160995483, -0.8336769938468933, 0.7157617807388306, 0.3046664893627167, -1.690531849861145, -0.5141929388046265], [-0.6279045939445496, 1.6628501415252686, -0.7432134747505188, -0.3783602714538574, -0.32022884488105774, -0.9244564175605774, 0.41979461908340454, -1.1073209047317505, -0.3022196888923645, 1.3955481052398682, -0.5169890522956848, -0.12535834312438965, -1.7529770135879517, -0.7803651690483093, 0.8267956972122192, 0.47365620732307434, -0.4910582900047302, -0.2308797389268875, -1.0724270343780518, 0.317382276058197, 0.10350670665502548, -0.5603325366973877, -1.6891957521438599, -1.2809463739395142, -0.4188365042209625, -0.5810232162475586, -0.1893901824951172, -1.1182833909988403, -1.819093108177185, -1.0699598789215088, -2.9682440757751465, -0.6658071875572205, 0.807636022567749, -0.23341436684131622, -1.0948814153671265, 2.2664573192596436, -0.7005868554115295, 0.3730693757534027, 0.6684612035751343, -0.65716552734375, 1.0472699403762817, 0.8589797019958496, 2.2144484519958496, 0.29902684688568115, 0.48381537199020386, 0.3091640770435333, -0.5837397575378418, -0.3360377848148346, 2.197690725326538, 0.6600980758666992, -1.497046709060669, -1.3198035955429077, -0.3269340395927429, 0.215164452791214, -1.4944961071014404, -0.48918065428733826, -1.3160165548324585, -0.21793095767498016, -0.46274513006210327, 0.7080005407333374, -2.337639093399048, -0.2680083215236664, 2.195784330368042, 0.4769325256347656, 0.5492335557937622, -0.8568636775016785, -1.967490792274475, -1.493852972984314, 1.1616756916046143, -0.31288355588912964, 0.7197615504264832, -1.7119617462158203, 0.3697583079338074, 1.0203077793121338, 0.13527719676494598, -0.1958169788122177, -1.3259203433990479, 0.0934356227517128, 0.22838103771209717, 1.0153026580810547, 1.2289494276046753, -0.8840365409851074, 0.3534452021121979, -0.707243800163269, 0.4768940806388855, -1.1044549942016602, -0.24059079587459564, -0.33902889490127563, 1.272325038909912, -0.004284997936338186, 1.2635754346847534, -0.274840384721756, 1.0986310243606567, 0.6121232509613037, -0.11593925952911377, -0.03016742691397667], [-0.05201395973563194, 0.46859362721443176, 2.044081926345825, 0.32265400886535645, -1.1893590688705444, -0.1566428691148758, -0.025176698341965675, 0.8594664335250854, -1.6275966167449951, -0.3769434094429016, 0.4121229350566864, 0.9271199703216553, -0.3671708106994629, 0.10634112358093262, 1.9791978597640991, -0.9588684439659119, 1.0528357028961182, 0.5206893682479858, 0.44358953833580017, 0.41740313172340393, 0.9117162227630615, -0.22179196774959564, -0.18829596042633057, -0.9166949391365051, 1.1066263914108276, -0.20613406598567963, -1.2200499773025513, -0.48113560676574707, -0.3694378733634949, -1.8230689764022827, 1.121052622795105, 0.34646525979042053, 0.3315886855125427, -1.6497750282287598, 0.3361803889274597, -1.102662444114685, 0.16974912583827972, -0.9029931426048279, 0.5841989517211914, 0.7804110646247864, 0.0784507468342781, 1.6398471593856812, -0.4641357958316803, -2.067216157913208, -0.4885174632072449, -0.8705620765686035, -1.1456528902053833, 1.1905220746994019, -0.8703552484512329, -0.23480311036109924, -0.08241309970617294, 0.6856384873390198, -0.46188676357269287, -0.12125267833471298, 0.7053117156028748, -1.2074731588363647, 1.1435497999191284, -1.187959909439087, 0.2710602581501007, 0.36340367794036865, 0.6412844657897949, 1.0877004861831665, 2.1858787536621094, 0.7160791158676147, 0.7348071932792664, -0.6549115180969238, 0.2698763310909271, -0.7498493194580078, -0.48734086751937866, -0.5030022263526917, -0.17363417148590088, -0.9463402628898621, 0.22482262551784515, -0.488919734954834, 1.5020555257797241, -0.7268403768539429, -0.44346675276756287, 1.2103067636489868, -0.6876415610313416, 0.5403741002082825, -0.8014107942581177, -0.05563432350754738, -1.9085593223571777, -1.1740361452102661, -1.941710114479065, 0.013981903903186321, -0.401868999004364, -0.5601974129676819, -0.19805105030536652, 1.1618176698684692, 0.9084813594818115, 0.30781516432762146, -0.6624118089675903, 0.9735419154167175, -0.9068780541419983, -0.8198967576026917], [1.7862900495529175, -0.16515225172042847, 0.3431363105773926, -2.6145265102386475, 0.12453503906726837, 1.6674209833145142, -1.8860669136047363, 0.19521430134773254, -0.5935702323913574, -0.25518321990966797, 1.3888133764266968, 0.20762421190738678, 0.46552637219429016, 0.100124292075634, -2.0184991359710693, -0.9948804974555969, 2.2095789909362793, 0.7047573328018188, -1.0214308500289917, 1.7670649290084839, 1.0554169416427612, 0.2514609694480896, 0.1682947725057602, 0.6557000875473022, -0.32598036527633667, 0.9388819336891174, 0.4517594575881958, -0.1320050060749054, 0.5592151284217834, -0.6490228176116943, 1.1572248935699463, 0.027782369405031204, -0.9347316026687622, 1.4360324144363403, -0.5216611623764038, -0.800349771976471, -1.0953444242477417, -0.3809320330619812, -0.3454880714416504, 0.0500040277838707, 0.22025716304779053, 0.975481390953064, -0.5952023267745972, 0.03678641468286514, 0.09224629402160645, -1.9035547971725464, 0.2504534125328064, -0.9812834858894348, 0.05324632301926613, 0.07028897106647491, -0.6648560762405396, -1.9640206098556519, -1.004088282585144, 2.2529664039611816, -1.0412826538085938, 1.0263487100601196, -0.12200822681188583, 1.0846275091171265, -0.8698747754096985, 0.6236182451248169, 0.008058667182922363, -0.0274124126881361, -1.7003897428512573, 0.6741202473640442, 0.363444447517395, 0.5716289281845093, -0.6982159614562988, 0.21182478964328766, -0.36020609736442566, 0.4059554934501648, -1.2929909229278564, -1.430985689163208, -1.520788550376892, 0.7419686317443848, 1.2992891073226929, -0.5656936168670654, 1.4472209215164185, 0.9937295317649841, -0.6982289552688599, 0.6500522494316101, 1.3204302787780762, 1.3200348615646362, -1.0922601222991943, -0.8272475600242615, 0.6367392539978027, 1.377931833267212, 1.969064712524414, -0.8113541603088379, -1.1744250059127808, 1.2132718563079834, -1.6308205127716064, -0.4533792734146118, 0.23803338408470154, -1.6923165321350098, -1.120218276977539, -0.20526348054409027], [0.19757990539073944, -0.43597424030303955, 0.5902867913246155, -0.8061034083366394, -0.4013875424861908, 0.6059243083000183, 0.5887131094932556, -1.311471939086914, 2.096283435821533, -0.5726165771484375, 1.5638097524642944, 0.11530619114637375, -1.095418930053711, -1.630066990852356, -0.5930080413818359, 1.9413484334945679, -0.5616907477378845, 0.4448397755622864, 0.3906879127025604, 0.2823476493358612, -0.5388748049736023, 1.5227792263031006, 0.2906413972377777, 1.5566688776016235, 0.2548213005065918, 1.0570813417434692, 1.0468721389770508, 0.8508619070053101, -1.5719467401504517, 1.0342849493026733, 1.6548004150390625, -1.444663643836975, 1.6021363735198975, -1.6496329307556152, -0.6630157232284546, -0.9828504920005798, 0.6088881492614746, 0.22171080112457275, 0.7498743534088135, 0.10300558060407639, -0.5029668807983398, 0.5940257906913757, -0.9641515612602234, 0.3123440146446228, 0.1554117053747177, -0.4119802415370941, -0.36435607075691223, -1.234192132949829, 0.4686048626899719, 0.40290704369544983, -0.28997907042503357, 0.3047669231891632, -0.6487746238708496, 0.2551482617855072, 1.567063570022583, -2.0220470428466797, 0.28773438930511475, -0.5182284712791443, -0.26879188418388367, 0.5521560907363892, -1.5506837368011475, -0.20552141964435577, -0.7793723940849304, -0.9140578508377075, -0.8386213779449463, -1.808555245399475, -0.24995997548103333, -0.29227378964424133, 1.4473390579223633, -0.9740722179412842, 0.3027612864971161, -0.5681636333465576, 0.7508199214935303, -0.32506370544433594, -0.2871512174606323, -0.6226370334625244, -0.8161059021949768, 1.3605681657791138, -2.4599218368530273, 0.4082055985927582, 0.595725953578949, -1.0098904371261597, -0.4040449559688568, 0.9460636377334595, 1.3564786911010742, -0.480754554271698, 1.3129445314407349, 0.3145076334476471, -1.6477948427200317, -1.0008254051208496, 0.21199457347393036, -0.8701605796813965, -0.38984671235084534, 0.024559833109378815, 0.5297690033912659, 2.0037758350372314], [-0.5667341351509094, -1.0497028827667236, 0.20830486714839935, -0.18473181128501892, -0.7807669639587402, 0.0815635398030281, -1.948230266571045, 1.477319598197937, 1.0315104722976685, -0.42148903012275696, -0.5189439058303833, -0.18366460502147675, -0.5718197226524353, -0.7563415765762329, 1.1798549890518188, -1.100735068321228, 0.42272287607192993, -0.16214516758918762, 1.721588373184204, 1.2131627798080444, 0.2549281418323517, 0.040279947221279144, -1.3116397857666016, -0.12956340610980988, 0.5766858458518982, -1.4505250453948975, 0.11965876817703247, -1.463382363319397, -0.9490877985954285, 1.357965350151062, 1.1076902151107788, -1.2136738300323486, -1.028352975845337, 1.4955142736434937, -0.9011039733886719, 0.7847530245780945, -1.5163120031356812, 0.2321462780237198, -0.31234756112098694, -0.3914147615432739, -1.2773939371109009, -0.40380698442459106, 0.506599485874176, -0.4149025082588196, -0.7957959771156311, 0.016737166792154312, 1.1219431161880493, 1.1544427871704102, 1.3991419076919556, -0.18751955032348633, -1.1187587976455688, -0.7577018737792969, 0.9194114208221436, -0.8269535303115845, 1.8674484491348267, 0.8342657089233398, -0.5834754705429077, -0.33211207389831543, -0.27791866660118103, 2.0554263591766357, -2.795780658721924, -1.5324918031692505, 0.9660340547561646, -0.7094156742095947, -0.3869856297969818, -0.34588226675987244, 1.8558021783828735, -0.30031123757362366, -2.4041028022766113, 0.04826074838638306, -0.6801530122756958, -0.220456063747406, -0.2563649117946625, 0.9649374485015869, 1.4301930665969849, 1.0306402444839478, 1.0237709283828735, 1.6655176877975464, 0.6515273451805115, -0.7701049447059631, 0.9760303497314453, 0.3261389434337616, 1.684112787246704, 0.4299033284187317, 1.026389479637146, -0.9195461273193359, 0.033952876925468445, -0.04878930002450943, 0.9891034960746765, -0.07774807512760162, 0.6859171390533447, -0.9769221544265747, -0.3317824900150299, -0.06824656575918198, 0.7703296542167664, -1.7577496767044067], [1.0193893909454346, -0.11085127294063568, -3.0447895526885986, 0.7860326170921326, -0.8166168332099915, 1.4717164039611816, 2.1020052433013916, 0.31502199172973633, -0.30483996868133545, -0.6819478273391724, 0.9753008484840393, -0.7162512540817261, -0.38103359937667847, -1.5600157976150513, 0.4218350946903229, -1.6964772939682007, -0.3400215804576874, -0.24257595837116241, 0.4606928825378418, 0.4684973955154419, -0.7320452928543091, 0.4429984986782074, 0.07853472977876663, -0.5588974356651306, 0.3067416548728943, -1.0243124961853027, -1.2356008291244507, -0.6012117862701416, -0.8419360518455505, -1.081093430519104, 0.8015642762184143, -0.7467857599258423, 0.6978377103805542, -0.471113920211792, -1.000968098640442, 0.7296140193939209, 1.099386215209961, 0.5302827954292297, -0.2677222788333893, -1.509644865989685, -0.3934638798236847, 0.7876391410827637, -1.6234674453735352, -3.010364294052124, -1.3693721294403076, -0.4073706269264221, -1.0846149921417236, 0.18549473583698273, 0.9527456164360046, -0.38195616006851196, -0.13767366111278534, -0.08099234849214554, 0.5413526892662048, 0.8597134351730347, 1.2270351648330688, 0.585169792175293, -0.582355260848999, 0.12250224500894547, -0.5054716467857361, 0.8572505116462708, -2.3068907260894775, 0.045555148273706436, -1.2601734399795532, -0.8038526773452759, 0.44181832671165466, 0.5390911102294922, 0.4510105848312378, 1.4927231073379517, 1.2250208854675293, -0.9660512804985046, -0.5660885572433472, 0.026525935158133507, 0.22650162875652313, 1.5348472595214844, 0.8821831345558167, -0.8641007542610168, 1.071602463722229, 0.2121741771697998, -0.013697235845029354, -1.1378690004348755, -1.1835839748382568, 0.9212059378623962, 0.17129723727703094, 1.2282434701919556, 0.37214258313179016, 0.2453378140926361, 0.041308894753456116, 0.32109326124191284, -0.5182846188545227, -0.05159936472773552, 0.18564006686210632, -1.743804693222046, -0.4439171552658081, -0.77947598695755, 0.5089854001998901, 1.3371366262435913], [-0.35303065180778503, 0.6795167922973633, -1.30086088180542, -0.6188860535621643, 0.4172877073287964, -0.019691672176122665, -2.242832660675049, -0.282767653465271, -0.3757416009902954, 1.5724364519119263, 2.131714105606079, -3.4585776329040527, -1.2575911283493042, 0.7815226316452026, -0.6492013335227966, -1.0450469255447388, -0.22688551247119904, -0.5790548324584961, 0.3213343620300293, -0.8874518871307373, 0.8411651849746704, 0.4027147889137268, -0.44809654355049133, 0.5361310243606567, 0.6132180094718933, -1.7305272817611694, -0.49167758226394653, 0.5361326336860657, -1.4685463905334473, 0.6642612814903259, -0.27386537194252014, 0.561120867729187, 2.15043306350708, -0.8988370299339294, 0.12549051642417908, -0.0028597309719771147, 0.38439202308654785, -0.5276534557342529, 0.22078190743923187, -0.06614784896373749, -0.7011014223098755, 0.9092273712158203, -1.647939920425415, 0.3604230582714081, 0.1491110622882843, -1.3833069801330566, 0.0058711739256978035, -3.135430335998535, 0.007046212907880545, 0.9741501212120056, 0.1377106010913849, 0.4775298833847046, -0.18880105018615723, 1.9557485580444336, -1.4483156204223633, 0.21025308966636658, -0.6302441358566284, -0.009710807353258133, 0.6368236541748047, -1.003572702407837, -1.1214977502822876, -0.3611590266227722, 0.09735944122076035, -2.5091123580932617, 1.3491315841674805, 0.17646948993206024, -0.856568455696106, -0.22472406923770905, -1.4068856239318848, 1.2754302024841309, 0.6067955493927002, -0.9412782788276672, -1.7813482284545898, 0.3071501553058624, -0.497444212436676, 0.35709288716316223, 1.1731711626052856, 0.7561697959899902, -2.2738399505615234, 0.01197375450283289, 1.7914048433303833, -1.0692460536956787, -0.6856157183647156, 1.2684844732284546, -0.7504116296768188, -0.048787008970975876, 0.874568521976471, 0.5118349194526672, 0.2470596730709076, -0.13660521805286407, -0.22779855132102966, 0.8518056273460388, 0.6389352083206177, -0.3691159188747406, 0.42505142092704773, -0.08815581351518631], [-0.14357411861419678, 0.2593200206756592, 0.13795332610607147, -1.6706277132034302, -2.1486170291900635, 1.188962697982788, 0.09648267924785614, -0.1672120839357376, 0.743295431137085, 1.0601335763931274, -0.9729439616203308, -0.48462027311325073, -0.6545234322547913, -1.182112455368042, 0.8489335775375366, -0.10531666874885559, 0.3700152039527893, 0.5221086740493774, -0.2761205732822418, 1.0694489479064941, 1.0887621641159058, 0.09385716170072556, 0.3961428999900818, -0.05083194002509117, 0.06277336925268173, 0.1585947722196579, -0.5753196477890015, 1.207767367362976, -1.1659317016601562, 0.4496791958808899, 1.900596261024475, -1.3674484491348267, -0.03801451250910759, 0.5447686910629272, -0.8514190316200256, 0.02843344397842884, -0.9224952459335327, -0.18543240427970886, -1.3795478343963623, 1.1912678480148315, 1.2928870916366577, -0.00136575975921005, -1.7463077306747437, 0.4196956157684326, 0.46036380529403687, -1.2560323476791382, -0.9196177124977112, 0.5608704090118408, 1.0702930688858032, -0.8090700507164001, 0.4771040081977844, 1.8478152751922607, -1.3791513442993164, -0.9332589507102966, -1.1804956197738647, 0.13415655493736267, -0.8375223278999329, 0.6450154781341553, -1.5591744184494019, -0.6958550810813904, -0.2189057469367981, -0.6366201043128967, -1.542023777961731, 1.2666572332382202, -0.6609749794006348, -0.8186603784561157, -1.80343759059906, 0.5142561793327332, 0.7657366394996643, -0.01975322887301445, -0.21516935527324677, -1.529727816581726, -1.0816770792007446, -0.44677308201789856, 0.5984291434288025, -0.07047832757234573, -0.22073359787464142, 0.45810672640800476, 0.9010764360427856, 1.4345873594284058, 0.5084935426712036, -0.4483173191547394, 0.5074876546859741, 0.8681607842445374, -0.9301890134811401, -0.2484906166791916, 2.234375238418579, 2.184565305709839, 0.5057932734489441, -0.4123937785625458, 0.40708479285240173, 0.5901316404342651, -0.060596246272325516, -0.07416938245296478, -0.4147847890853882, -0.7313308119773865], [-0.9069517254829407, -0.4221491515636444, -0.782326340675354, -1.4874634742736816, 0.5278586745262146, -0.09809096902608871, 1.819502592086792, -0.773224949836731, 0.5293527245521545, 0.581737220287323, 0.28993454575538635, 1.2932112216949463, -1.4152880907058716, -0.07548883557319641, -0.8306808471679688, 0.11299218982458115, 0.10706431418657303, 0.157209575176239, -0.8547773957252502, 0.8168294429779053, 0.799977719783783, -0.14234763383865356, 0.6741513609886169, 1.1666173934936523, -0.6660174131393433, 1.3144478797912598, 0.04720839485526085, 0.8462415933609009, -1.5444282293319702, 0.7884528040885925, 0.6746900081634521, -0.836311399936676, 1.2915267944335938, -0.49616754055023193, 0.4860111474990845, -0.1463685929775238, -0.5705170631408691, -0.027959411963820457, 0.26601454615592957, -0.32725989818573, 1.0627537965774536, -1.2305874824523926, 0.11844469606876373, 0.3890112340450287, -1.0594565868377686, 1.3924837112426758, -2.059704542160034, 0.48966044187545776, 1.2661914825439453, 2.8952314853668213, 1.1107255220413208, -1.8615564107894897, -0.004165025893598795, 0.3384978175163269, -0.18804261088371277, 1.0722326040267944, -1.716792345046997, -1.547706961631775, 0.37860360741615295, 0.5176995396614075, 0.8837685585021973, 0.40438899397850037, -0.13458512723445892, -0.8808053731918335, -0.9534722566604614, 1.3976465463638306, 1.1073927879333496, 0.8635351657867432, -0.14090238511562347, 1.2138526439666748, 0.4834122657775879, -0.16493067145347595, 0.341194212436676, -2.573960065841675, -0.9415878653526306, -0.21361050009727478, 0.3285503387451172, -0.23177872598171234, -1.0969434976577759, 0.812777578830719, 1.2488267421722412, 0.0736398696899414, 1.7246532440185547, -0.17279919981956482, -0.5646412372589111, -0.03049515187740326, -0.15331877768039703, 2.1581437587738037, 1.7900609970092773, -0.6521338820457458, 0.4759838581085205, -1.332789421081543, 0.8834125995635986, 0.669013500213623, -0.15627911686897278, 0.7827135920524597], [0.3817436695098877, 0.9994505643844604, -0.5355928540229797, 0.6999657154083252, 0.7858275175094604, -1.66254460811615, 1.483963966369629, -0.7813084125518799, -0.07480692118406296, -2.1780028343200684, -0.496383935213089, 0.8912547826766968, 1.240716814994812, -0.11307484656572342, 1.1209192276000977, -0.14254437386989594, -0.9644230604171753, 0.466329425573349, 0.2888270914554596, 0.40217921137809753, -1.3828010559082031, 0.1653684377670288, 0.15249036252498627, 1.6012932062149048, 0.014461585320532322, -0.07342445850372314, 0.9765841364860535, 1.4181180000305176, 0.9308189749717712, 1.0821064710617065, 0.9841504096984863, -0.2134489268064499, -1.2638709545135498, -0.000102797886938788, -0.6020558476448059, -0.6911662220954895, -0.36921533942222595, -0.5316734313964844, -0.7268494367599487, -0.30842679738998413, -0.26645627617836, -1.2160046100616455, -0.903369665145874, -0.5305222868919373, -0.4912753999233246, -0.47329476475715637, -0.7158094048500061, 0.8391144871711731, 2.0621252059936523, 0.38777226209640503, 0.46286842226982117, -0.8474038243293762, -0.6271855235099792, 1.3426264524459839, -0.447915256023407, 1.0474717617034912, 0.054177768528461456, 0.27439063787460327, 0.007678018417209387, -0.6751158237457275, 0.5639544725418091, -0.6087969541549683, 1.3369420766830444, 0.12511660158634186, -0.6312776803970337, 0.25975120067596436, -0.2898254692554474, -0.5801841616630554, 0.5456026196479797, 1.1416765451431274, -1.2403432130813599, -0.3945172131061554, 1.6102322340011597, 0.368282675743103, 0.28753799200057983, -1.21392822265625, -0.07800678163766861, -0.3247714042663574, -0.7964730262756348, 0.06926785409450531, 0.03890419378876686, -1.3177944421768188, -0.32381635904312134, 0.4970547556877136, -0.44389739632606506, -1.4046213626861572, 1.302522897720337, -0.10936732590198517, 0.05862267687916756, -1.1965540647506714, 0.15232156217098236, -0.9658739566802979, -0.24613207578659058, 1.1673020124435425, 0.9292761087417603, 0.5435423254966736], [-0.962215006351471, 0.9584687352180481, -0.6645184755325317, 0.20324063301086426, 0.6162398457527161, 0.3220140039920807, 0.6228219270706177, -0.8335665464401245, -0.3088656961917877, 0.7972568273544312, 0.7461130023002625, -0.274189829826355, 0.6988563537597656, -1.6834510564804077, -0.8302759528160095, 0.13415351510047913, 0.5053418874740601, -0.06599289178848267, 1.4039140939712524, -2.0978000164031982, 1.0653631687164307, -0.9111176133155823, -0.014561320655047894, -0.30641257762908936, -1.1007764339447021, -1.0959975719451904, 1.7422915697097778, -2.084108591079712, -1.3710166215896606, 1.476579189300537, -0.2666740417480469, -0.8101701736450195, 0.24097296595573425, 0.42078080773353577, 1.4267429113388062, -0.4369831383228302, -0.1698029637336731, 0.9675244092941284, 0.15685045719146729, 0.9513485431671143, -1.639853835105896, -1.0968585014343262, 0.13319522142410278, -1.4112070798873901, 1.5770468711853027, -0.8744257092475891, 0.393404096364975, -0.6566958427429199, -0.05535572022199631, -0.00831593107432127, 0.49599596858024597, 0.8179537653923035, -0.4613282382488251, 0.9274072647094727, 0.26177823543548584, 1.9273622035980225, 0.4153956472873688, -1.6770426034927368, 0.748552143573761, -0.3441888391971588, 0.08819817006587982, 0.511333167552948, -0.3371727764606476, 1.305959939956665, -1.270375370979309, -0.4628431499004364, 0.6246037483215332, -0.05041103437542915, -0.7420594692230225, -0.0253619235008955, -0.28614431619644165, -0.3271730840206146, -0.05519193783402443, -0.03347672522068024, -0.6567130088806152, -2.514286518096924, -0.5846889615058899, 0.6553257703781128, -0.24109072983264923, -2.3960886001586914, -0.5118787288665771, -0.14936508238315582, -0.1549435704946518, 0.691512942314148, 1.426074743270874, -0.50022292137146, -0.14913132786750793, -1.5275112390518188, -1.8551249504089355, 0.3326970934867859, -0.9374268054962158, -0.8147692084312439, -0.37246114015579224, -0.11163213849067688, -0.020743969827890396, -1.0353556871414185], [-0.7116543650627136, 0.9261255264282227, 0.7659499049186707, -0.7757275104522705, 0.6917950510978699, -0.20220713317394257, 1.440359354019165, 0.4456760585308075, 0.4219350814819336, 0.8053045868873596, -0.025565749034285545, 0.18392542004585266, 0.17923228442668915, 0.8640058040618896, -1.0803345441818237, 1.6042810678482056, -0.19305945932865143, -1.1650564670562744, -0.2801208794116974, 0.1599956601858139, -0.7864968180656433, -0.13358986377716064, 1.0795680284500122, 1.5412405729293823, -0.013161447830498219, 0.38145995140075684, 1.449789047241211, 1.7803303003311157, 1.4078004360198975, -0.7117423415184021, 0.864936888217926, -0.21163585782051086, 0.13619893789291382, -0.8762661814689636, 1.1446254253387451, 0.5836678743362427, -1.062080979347229, 0.31873729825019836, -1.1323410272598267, -0.8638111352920532, -0.47319501638412476, 0.4160996377468109, -0.7513434886932373, 0.4429531991481781, -0.10448814928531647, 0.4897952973842621, 0.8109467625617981, 0.2079884111881256, -0.09356603771448135, 0.611203670501709, 0.5687808990478516, 0.7773984670639038, -0.5433752536773682, -0.3202870786190033, -1.538694977760315, -0.4214645326137543, -0.011314568109810352, 0.02201228030025959, 0.858564555644989, -0.5958220362663269, 0.05771017447113991, 0.5189977288246155, 0.2306043654680252, 0.6382242441177368, 0.7859182357788086, 1.1952431201934814, -0.660099446773529, -0.3086927831172943, -0.6928316950798035, -0.6966468095779419, -0.24432769417762756, -1.56099534034729, -0.5980384945869446, 0.6422666311264038, -1.4451963901519775, 0.37846869230270386, 1.0995053052902222, 3.071611166000366, 0.6997044682502747, 0.31445735692977905, 1.1270432472229004, -0.29238393902778625, 0.764668881893158, 0.31621620059013367, 0.6207066774368286, 0.42396214604377747, -2.1084132194519043, -1.975271224975586, -0.8768113851547241, 0.6265775561332703, -1.9587624073028564, -1.39266037940979, -0.20568320155143738, -0.1664123386144638, -0.680348813533783, 0.26824355125427246], [1.1501117944717407, -0.8495049476623535, -1.056780219078064, -2.175104856491089, -0.936796247959137, -1.0771538019180298, -1.1564449071884155, -1.7564449310302734, -0.847284734249115, -0.4508466124534607, 1.0386831760406494, -0.47023746371269226, -0.6514528393745422, 0.8654021620750427, -0.5172958970069885, -0.273419588804245, 0.3413431644439697, 0.09200538694858551, -1.8023486137390137, -0.03693443164229393, 0.7406934499740601, -1.360094666481018, 0.8649935126304626, -0.6704156994819641, -0.7922489643096924, -1.9774699211120605, 0.12068530172109604, 0.23923633992671967, 1.63359797000885, -0.8326672315597534, 0.1033831238746643, 0.7600420713424683, -0.3807853162288666, -0.38092121481895447, 0.7875370383262634, 1.928481101989746, -1.5113320350646973, -0.5858249664306641, -0.6137811541557312, 2.393562078475952, 0.11702807992696762, 0.5263461470603943, -1.1496487855911255, 1.3856921195983887, -0.022421598434448242, 0.3469158709049225, 0.3143947124481201, -1.3442790508270264, -1.6737381219863892, 0.39630240201950073, 2.006495237350464, 1.000561237335205, 0.01983688399195671, -0.9339159727096558, -1.438434362411499, 0.12320872396230698, 0.9027498364448547, -0.8989530205726624, -0.2487732172012329, -0.07474355399608612, 0.34407925605773926, -0.5249194502830505, -2.592921733856201, -0.5825926661491394, -0.7917681932449341, -0.6089293360710144, -0.23908743262290955, 1.3042610883712769, 0.6830242276191711, 0.5367076992988586, -0.4667334258556366, 1.0124149322509766, -1.2603120803833008, 1.0523234605789185, 0.7258375883102417, 0.6377598643302917, 0.4882049560546875, -0.24228093028068542, -1.6323022842407227, -0.685036838054657, -0.4276110827922821, 0.05126947537064552, -3.6084177494049072, 0.43774181604385376, -0.8868677616119385, -0.7787778973579407, -0.1767183095216751, 0.26246708631515503, 0.41549405455589294, -0.0416744127869606, 0.8241162300109863, -0.3066504895687103, -0.6759112477302551, 0.3676096200942993, 0.2639060914516449, -0.3398543894290924], [1.2063133716583252, -0.18257050216197968, -1.5293927192687988, -0.5935338735580444, -0.17705535888671875, 1.1380671262741089, 0.044655121862888336, 1.5023226737976074, -0.6413580179214478, -0.9340515732765198, 0.21444310247898102, 0.1560138612985611, 0.23673412203788757, 1.4190773963928223, 0.7003107070922852, -0.27507641911506653, 0.14577706158161163, -0.823479413986206, -0.3130197525024414, 1.3306357860565186, 0.05316660925745964, -0.1714491844177246, 0.8850457072257996, -0.7000218629837036, 0.3657872676849365, -0.8786322474479675, -0.48258307576179504, -0.885300874710083, 0.15591642260551453, -0.42221248149871826, -1.0914344787597656, 0.2696686089038849, -0.08055590093135834, 1.3912200927734375, 0.3442051112651825, -0.5359601378440857, 1.3607791662216187, -0.7761748433113098, -0.027745164930820465, 0.6483660340309143, 0.12045587599277496, 0.5808935761451721, 0.9516317844390869, 1.4082467555999756, 0.7814961075782776, -0.006050082854926586, 0.2126084715127945, -0.3208593726158142, 0.09569092094898224, 0.34866446256637573, -1.3174329996109009, 0.8646571040153503, 1.5594182014465332, 0.15763571858406067, 0.6295494437217712, -1.890397548675537, -1.8111720085144043, 1.7416324615478516, -0.5550214052200317, 2.4675772190093994, -1.653546690940857, -0.7833368182182312, -0.526477038860321, 1.011724829673767, 0.7815331816673279, 1.4499539136886597, 2.0375399589538574, -0.33776143193244934, -0.10673090815544128, 0.7279458045959473, 0.5164808034896851, -2.309231758117676, 0.7337759733200073, 0.29661229252815247, 0.9557740688323975, -0.6164835095405579, 0.11380364000797272, 0.6782709956169128, 0.7659735679626465, 0.9374932050704956, 0.4147920310497284, -0.14390678703784943, 0.42554521560668945, -0.6733494400978088, -1.3809117078781128, 1.4602220058441162, 0.22113066911697388, -0.17340464890003204, 1.002007007598877, -2.2380216121673584, -0.461833119392395, 0.843438982963562, -1.1689956188201904, -0.19299426674842834, 0.036139920353889465, -0.445611834526062], [0.38311994075775146, -0.3263680338859558, 1.1463189125061035, -0.4156348705291748, 0.3856383264064789, -0.5088514685630798, -1.3213953971862793, -1.6337449550628662, 0.05821079760789871, 0.18962158262729645, -0.1834496706724167, 0.1393369734287262, 0.40434640645980835, -0.050487615168094635, 0.6895710825920105, -0.7923316359519958, 1.00875723361969, 0.6412773728370667, 0.5129771828651428, -0.42160695791244507, 0.9734413623809814, -1.6512795686721802, -0.4790085256099701, -1.2892701625823975, 0.30455976724624634, 0.6688705682754517, 0.2335144430398941, 1.169468879699707, -0.6445558667182922, 0.520584762096405, 1.570959210395813, -1.3829594850540161, -0.5079212188720703, 1.0593541860580444, 1.3456883430480957, 1.6150870323181152, -1.4853239059448242, -0.7701627016067505, 0.34535953402519226, 1.9578697681427002, 1.675215482711792, -0.14149810373783112, 0.5372909307479858, -1.0282331705093384, 0.567091166973114, 0.5498799681663513, -0.5023356080055237, 0.1999998688697815, 0.19547845423221588, -2.272923231124878, 1.9048219919204712, -0.52579265832901, 1.114367961883545, -0.2935672998428345, 0.5569836497306824, 0.4476083517074585, 0.8014020323753357, 0.9093532562255859, -0.26373961567878723, -0.2663421332836151, -0.9302496910095215, 0.39553529024124146, -0.6840366125106812, 0.9838014245033264, -1.2099509239196777, -0.5775054693222046, 0.47067299485206604, 1.148979902267456, 0.18038281798362732, -1.5532366037368774, -0.30082452297210693, -0.619792640209198, -0.8821921944618225, 0.3924790024757385, -0.3553507328033447, 0.7976049184799194, -1.9106017351150513, 1.975823163986206, 0.7529715895652771, 0.3451124131679535, 0.6952996850013733, -0.42027854919433594, 1.5594134330749512, 0.6763558387756348, -2.2455461025238037, 0.18061456084251404, 0.39645326137542725, 0.3734230399131775, 0.8912795186042786, -1.454939842224121, -0.20781178772449493, -0.2203030288219452, 0.0923352986574173, 0.08686453104019165, -0.6850491166114807, -0.013092181645333767], [-2.251725673675537, -0.5304121971130371, 0.16790513694286346, 0.9979333877563477, -0.08500059694051743, 1.6477375030517578, 0.8191831707954407, -0.6186138391494751, 0.37520337104797363, -1.2089171409606934, -0.9164575338363647, 0.21543709933757782, 1.2855958938598633, 0.5951708555221558, -1.3070971965789795, -0.06683340668678284, 1.0673134326934814, -0.8341367840766907, -0.28387120366096497, 0.6292434930801392, -0.09448767453432083, 0.3593195974826813, -0.5726274251937866, 2.3339731693267822, -0.42173323035240173, 0.4309736490249634, 0.1808786690235138, -0.37215399742126465, 0.7917206287384033, 0.8690701127052307, 0.3658207654953003, 0.017142336815595627, -2.6947829723358154, -0.7413122057914734, 0.2836553156375885, 0.04904910922050476, 0.6438093185424805, -0.8270589113235474, -1.3438481092453003, 0.9225441217422485, -0.6304686069488525, 0.6484363079071045, 1.5543639659881592, -0.18270038068294525, -0.041272345930337906, 0.9311956167221069, 0.43373844027519226, 0.4199848175048828, -1.6214323043823242, 0.4605039656162262, -1.12044358253479, -0.5010558366775513, -0.730410099029541, -0.26492199301719666, -1.848785161972046, -0.4369930922985077, 1.1011390686035156, -0.25517815351486206, -1.7793980836868286, -0.24531038105487823, -0.5667499303817749, -0.7707213163375854, -0.9739289879798889, -0.669117271900177, 0.6509954929351807, -0.7677711248397827, 1.6378810405731201, 0.21803589165210724, -2.0148048400878906, -1.4725894927978516, 1.0573689937591553, -1.496757984161377, -1.5739511251449585, 2.6317005157470703, 1.751849889755249, -0.40057241916656494, 1.6504629850387573, -0.7013322710990906, 1.7457448244094849, -0.1854398548603058, 0.3553764224052429, 1.00471031665802, 0.6702052354812622, 0.03906994313001633, -0.40140190720558167, -1.2946792840957642, 0.9340654611587524, -1.3362141847610474, -1.3566436767578125, -0.07266481965780258, -0.9524196982383728, -0.026570262387394905, 1.5841199159622192, -2.2508342266082764, -0.5137391686439514, -0.23553819954395294], [-0.8838974237442017, 0.25057080388069153, 1.3047959804534912, -1.2358850240707397, 0.5652929544448853, -0.6948463320732117, 0.7218424677848816, 0.3157176971435547, 1.161043643951416, 0.4041418731212616, 0.9808416962623596, -1.7628002166748047, 0.7528685331344604, 0.15244384109973907, 0.5327108502388, 1.415444016456604, 0.11309932172298431, -0.06344231218099594, 0.18435442447662354, 0.3316571116447449, -1.6153801679611206, -0.3256738781929016, -0.7338551878929138, 0.33461466431617737, 0.9850144982337952, -1.839236855506897, 0.2811789810657501, -0.3239779770374298, -0.2507355511188507, -0.6989415287971497, -1.0108087062835693, -0.448892205953598, -1.149340271949768, 0.3875041604042053, -1.1536226272583008, -0.00816455390304327, 1.3615881204605103, 0.062421008944511414, 1.7683161497116089, -1.0470978021621704, 0.7018139362335205, -0.4530915319919586, 1.6438896656036377, 0.11595416814088821, 0.453133225440979, 0.026291027665138245, -0.32862260937690735, -1.0614145994186401, -0.9033656716346741, -0.05271304026246071, 1.6556130647659302, -0.4935513734817505, 0.20147168636322021, 0.0008848276920616627, 0.28261592984199524, -0.2799413502216339, 0.3922511041164398, 0.946479320526123, -0.43948543071746826, -0.5997015237808228, -1.524779200553894, 0.2545756697654724, 1.100687026977539, -1.1243386268615723, 0.507604718208313, -1.015286922454834, 1.9302430152893066, 0.43394753336906433, -0.20418709516525269, 1.6218868494033813, 1.5849347114562988, 0.27583205699920654, -0.8148980140686035, -1.062360167503357, 1.1513715982437134, 1.39816153049469, -0.5046792030334473, 0.5734890103340149, 0.963456392288208, -0.25850191712379456, -0.05201397091150284, 1.7323216199874878, 0.5848047733306885, -1.1871943473815918, -0.04812905192375183, -1.57721745967865, 0.5504494905471802, -0.4706505239009857, -0.5635346174240112, -0.4245762526988983, 0.06091871112585068, -1.544708251953125, 0.13171043992042542, -0.8204823136329651, -1.9538886547088623, -0.007212894968688488], [-0.9435106515884399, 0.2267240285873413, -1.4932984113693237, 1.6944406032562256, -1.0480166673660278, -0.2115667164325714, -0.45908546447753906, -1.3656178712844849, 0.18475976586341858, -0.273106187582016, -0.48150667548179626, 1.1963567733764648, -1.1654890775680542, -1.5813813209533691, -0.8157323002815247, 0.5132503509521484, 1.2800167798995972, 0.31548362970352173, 0.08374860882759094, -0.09128350764513016, 0.23265543580055237, 0.89250248670578, -2.8723645210266113, -0.8594179749488831, 0.05422728881239891, -0.30430305004119873, 2.1719443798065186, -1.0518323183059692, -0.7256535887718201, 1.0882279872894287, 0.49151745438575745, 0.8671954274177551, 1.536389946937561, 1.2836309671401978, -0.478039026260376, 1.4281320571899414, 2.0501155853271484, 0.037846874445676804, 0.5919142365455627, -1.1258724927902222, -1.1354610919952393, 0.6124895811080933, -0.6858066916465759, 0.39794954657554626, -0.14889150857925415, 0.6249262690544128, 0.7679793238639832, -1.2644999027252197, -1.4186303615570068, 0.27734601497650146, 0.3425191640853882, 0.020185744389891624, -0.14144080877304077, -0.6280799508094788, -0.08860397338867188, -0.6345564723014832, 0.5682106018066406, -0.05277588218450546, -0.018542105332016945, 2.2000582218170166, -0.4042501151561737, -1.5184451341629028, -0.04051589220762253, -0.538057804107666, -0.6199215650558472, -0.44607117772102356, -0.3611852526664734, -0.15702100098133087, -0.3616955280303955, -1.1742148399353027, -0.38358980417251587, 0.19682534039020538, 0.6305310726165771, 1.013053297996521, -0.6785938739776611, 0.3985733389854431, 0.5455343723297119, -0.5522795915603638, -0.7524938583374023, 1.1796317100524902, -0.7367955446243286, 1.0227504968643188, -0.1237124353647232, -0.5231980681419373, 0.8526924848556519, 1.3461169004440308, 0.2891434133052826, 0.20531779527664185, -1.995587944984436, -0.0563388466835022, 1.2065397500991821, -0.6961758136749268, 1.4617443084716797, 0.5925524234771729, -1.472674012184143, -0.3856077194213867], [0.04373819753527641, 0.2632509768009186, 0.3094489872455597, -0.1776198297739029, -0.7615205645561218, -0.08858296275138855, 0.3116486370563507, -0.12724314630031586, 1.2389031648635864, -0.27638015151023865, 0.29814913868904114, -1.298644781112671, -0.4040869474411011, -2.4128258228302, 2.9940028190612793, 1.1058475971221924, 0.7859205603599548, 0.4713252782821655, 0.41491758823394775, -0.029064973816275597, 0.7691761255264282, -0.11179250478744507, -0.2897246778011322, 0.4859531819820404, 0.5182814598083496, 0.648984432220459, -0.5574877858161926, 1.1740000247955322, -0.3224339783191681, -0.5111392736434937, 0.7427508234977722, -1.0429697036743164, 0.19321861863136292, -0.5279240012168884, -0.8483585119247437, 1.3440399169921875, 0.35698992013931274, 1.5544512271881104, -3.0492310523986816, -1.6650028228759766, 0.9213014841079712, 1.3203250169754028, 0.12474025040864944, -1.8574062585830688, 0.19299988448619843, 0.33256998658180237, 0.14420369267463684, -0.5240490436553955, -0.46186497807502747, 0.4055953323841095, 2.217477321624756, -0.8823539018630981, -0.3351759910583496, 0.6980709433555603, 0.7618764042854309, 1.0931488275527954, 0.9097055792808533, -0.05029686167836189, -0.6851878762245178, -0.4864301383495331, -0.3066488206386566, -1.4450445175170898, -0.30464547872543335, -0.5368209481239319, 0.19383804500102997, -0.21535615622997284, 1.0546659231185913, 1.9257558584213257, -0.6148634552955627, -1.6611626148223877, -0.1969919055700302, -1.0677014589309692, 0.11510463804006577, 1.718638300895691, -0.1478148251771927, 1.3667200803756714, -0.23022355139255524, 0.7322241067886353, -0.31656786799430847, 0.16171090304851532, -0.2658891975879669, -0.5833511352539062, -0.7402155995368958, 0.8034711480140686, -1.8815006017684937, 0.460496723651886, -0.6504440307617188, 1.422794222831726, 1.1457428932189941, 0.7145237922668457, -2.239293098449707, -0.9386631846427917, 1.5062587261199951, 0.3356887400150299, -1.5905468463897705, -1.2319607734680176], [-0.5377016067504883, 0.8255385160446167, 0.8676725625991821, -1.5619608163833618, 0.3171754777431488, -1.7135401964187622, -0.7150855660438538, -0.9660124778747559, 0.4768761992454529, 1.1605395078659058, -0.08396096527576447, -0.18034207820892334, -0.5433163046836853, 0.7018530964851379, 1.0612258911132812, 1.0917788743972778, 1.4215857982635498, 1.2958080768585205, -0.45310458540916443, 1.3890453577041626, 0.8282637000083923, -1.0576248168945312, 1.1530603170394897, 2.5015625953674316, -1.437030553817749, -0.16616354882717133, -0.5282045602798462, 2.415419340133667, 0.28833404183387756, 1.4568558931350708, -1.4074132442474365, -0.10636068135499954, 0.37431401014328003, -2.3209176063537598, 0.8968703746795654, -0.9090180397033691, -0.6903402805328369, -1.1403083801269531, 2.5356690883636475, 1.9642596244812012, -0.5283979773521423, 1.5969122648239136, 1.8982843160629272, -0.20542079210281372, 0.16314290463924408, 0.10168953239917755, -1.5225605964660645, 0.8088787794113159, -1.8330706357955933, 1.7831321954727173, 0.5159493088722229, 0.11562082916498184, 0.03305093199014664, -0.7351472973823547, -0.34855103492736816, 1.624650001525879, 1.1300699710845947, 1.5387789011001587, -0.28776952624320984, 0.6545646786689758, 1.167357087135315, -0.5820440053939819, 0.24418018758296967, -0.6418851613998413, 1.1414258480072021, -1.4563586711883545, -0.468501478433609, 0.009009058587253094, -0.3499367833137512, -0.3707493245601654, 0.6361979842185974, 0.886692225933075, 0.33126917481422424, -0.7834882736206055, -0.5381639003753662, 0.8531529307365417, -0.37190383672714233, -0.49860522150993347, -0.454604834318161, 0.18329347670078278, 0.37704703211784363, 1.8082551956176758, -0.12115342915058136, 0.4806080758571625, 0.9881883859634399, -0.8837894797325134, 1.2782973051071167, -1.0767525434494019, 0.9887375235557556, 0.5709531903266907, 0.07195114344358444, 0.1695416420698166, 0.40097469091415405, -1.847435712814331, -1.3539198637008667, -0.1357363611459732], [-0.8103162050247192, -0.5342683792114258, -1.1534643173217773, -1.3460338115692139, -0.08870598673820496, 0.0498206689953804, -0.6386463046073914, 0.6623319387435913, 0.6556603908538818, 0.7859649658203125, -0.2093518078327179, -0.5982308387756348, -0.9751598834991455, -0.5675485730171204, 0.3814844489097595, -0.04039205610752106, 1.960060477256775, -0.20323807001113892, -0.17433542013168335, -0.38637012243270874, -0.006088975351303816, -1.0792495012283325, -0.4383830726146698, -0.3423057198524475, 0.6779482960700989, 1.167412281036377, 0.18478479981422424, -1.8143408298492432, -0.2425355315208435, 1.3856513500213623, -0.10336332768201828, 0.03650211915373802, -1.9283134937286377, -1.146801471710205, -2.2871129512786865, 0.08661246299743652, 1.1137913465499878, 0.9322709441184998, -1.3764348030090332, -0.11416685581207275, 0.6951842904090881, 0.5628137588500977, 0.5774708986282349, 0.5900641679763794, 0.10364416241645813, -1.3811460733413696, 0.008377375081181526, -0.13140766322612762, 1.038960576057434, 1.9032065868377686, 0.7537388801574707, -0.23822109401226044, -0.1310311108827591, -0.4516051709651947, -1.5645434856414795, -0.7174042463302612, -1.658233880996704, -0.1952817589044571, -1.8759757280349731, 0.425491601228714, 0.2936440110206604, -0.9835094213485718, -2.1815922260284424, 1.1709275245666504, -3.2744901180267334, 0.8871188759803772, 1.8130801916122437, -1.0146641731262207, -0.047498710453510284, -0.18388721346855164, -0.7743898034095764, 0.2573585510253906, -2.2628026008605957, -0.6825293302536011, 1.5557061433792114, 0.4231564402580261, 2.569385528564453, 0.31920287013053894, 1.3115140199661255, 2.4504504203796387, 0.7226472496986389, 0.24139641225337982, 0.5499482750892639, 0.10208582878112793, 0.0024008448235690594, -0.9552030563354492, 1.9565874338150024, -0.928320050239563, 0.14327402412891388, -0.2308894693851471, -1.782402515411377, 0.45689520239830017, 0.8210524916648865, -0.47212278842926025, 0.8025510907173157, 0.9159857034683228], [1.0566034317016602, 0.1701750010251999, 0.44698822498321533, 0.36071693897247314, -0.8522779941558838, -0.888911247253418, -0.9937688112258911, 0.5900774002075195, 0.7385435700416565, 1.2984585762023926, 0.7790795564651489, -0.3461354374885559, 1.8146183490753174, -0.7056974172592163, 2.3213326930999756, -0.35754600167274475, -0.20378543436527252, -0.2553566098213196, -0.4717473089694977, -0.06928816437721252, 0.14673367142677307, -1.395706057548523, -0.08042984455823898, -0.4650433659553528, -0.26246362924575806, -0.49730587005615234, -0.014153599739074707, 0.41796639561653137, -0.21769045293331146, -0.6328072547912598, 1.968103289604187, 1.3253215551376343, -1.009203314781189, 0.5730755925178528, -0.42096081376075745, -1.2624765634536743, 0.1671493947505951, 0.6561166048049927, -0.7028771638870239, 0.3693476915359497, 0.763359010219574, 0.04569574445486069, 0.7228307723999023, 0.6818366646766663, -0.4191884696483612, 0.8625044822692871, 1.5672436952590942, -0.10620435327291489, -0.3578444719314575, -0.16089782118797302, -1.9883325099945068, 1.142279863357544, 0.8919044733047485, 0.3751540780067444, -2.2770726680755615, -0.6204289793968201, -0.6448021531105042, 0.8775901198387146, 1.484671711921692, -1.4587393999099731, -0.14456702768802643, 0.5081683397293091, 1.6599953174591064, 0.5587791204452515, -0.0718345195055008, 0.5257080793380737, -1.0886191129684448, -0.2865389287471771, -0.6696028113365173, 0.21244291961193085, -0.47365039587020874, 0.04946425184607506, -1.0695358514785767, 0.03472991660237312, 1.3061721324920654, -1.4679622650146484, 0.482686847448349, 0.49459704756736755, 0.6858492493629456, 1.0202778577804565, -0.6390907764434814, 1.254793643951416, -0.3387907147407532, 1.2007012367248535, -0.47894394397735596, -1.7259423732757568, -0.13836076855659485, 0.02320178784430027, 0.1844109296798706, 1.8283330202102661, 2.374171733856201, 1.269105076789856, -0.8498430252075195, 1.8819301128387451, 1.2404797077178955, 0.993632972240448], [1.5485867261886597, -0.33335641026496887, -0.1853480339050293, 0.5693660378456116, -0.9024481177330017, 0.34941115975379944, 0.9463838338851929, 0.39440301060676575, -0.9143736362457275, -0.1963748335838318, -0.1464744210243225, -1.114047646522522, -1.5120623111724854, 0.2257828414440155, 0.10310575366020203, 1.055733323097229, 0.26936742663383484, 1.3226221799850464, 0.8577879667282104, -1.226146936416626, -0.550774335861206, -0.15415175259113312, 0.46982601284980774, 0.15661780536174774, 0.5784059166908264, -1.404506802558899, 0.9541361927986145, 0.3223188817501068, 0.5717951059341431, 0.2612464129924774, 1.567609429359436, -1.2164231538772583, 0.9461120963096619, 1.117011547088623, -0.706851601600647, 0.9505046010017395, 1.4137952327728271, 0.18586918711662292, -0.9430649280548096, -0.17724238336086273, 0.07906132191419601, 0.2539098858833313, -0.5720933675765991, 0.967413067817688, -1.1440672874450684, -0.531126856803894, 0.9045716524124146, -0.14972171187400818, -1.3955299854278564, -0.2716272473335266, -0.6867768168449402, 0.5843239426612854, 1.1766127347946167, 0.33783286809921265, -0.39502477645874023, -0.8807882070541382, 0.11579450964927673, -0.7814978361129761, 0.8871289491653442, -0.11960747092962265, 1.268835186958313, -1.0331093072891235, 0.630230724811554, 0.7249488234519958, 1.8438050746917725, 0.2748256325721741, -0.5072131156921387, 0.2128370851278305, 0.08700411021709442, 0.7304851412773132, -1.320923089981079, 0.7721738219261169, 0.35745757818222046, 1.1238406896591187, -0.09161899238824844, 0.16591013967990875, -1.2021650075912476, -1.5463001728057861, 0.3682023882865906, -0.17876850068569183, -0.6488157510757446, -1.930001139640808, -0.3327190577983856, -0.6480181813240051, 0.026000220328569412, 0.7712462544441223, 1.7408109903335571, 0.9280327558517456, -1.1696548461914062, -2.002319574356079, 1.9733268022537231, 1.3691978454589844, -0.5230116844177246, 1.707319736480713, -1.1433802843093872, -0.217844158411026], [-1.9478328227996826, -0.17699690163135529, -0.9985218644142151, 0.6349549293518066, 0.7205275297164917, -0.5870364904403687, 0.4410703182220459, 0.3121061325073242, -0.0194545891135931, 0.5340055227279663, -0.9314376711845398, -0.5095782279968262, 0.26830631494522095, 1.385500192642212, -1.1150026321411133, 0.3847883343696594, 1.412628412246704, 0.13347211480140686, -2.012850046157837, -0.8445324897766113, -0.48673033714294434, -0.6505051255226135, -0.5923654437065125, -0.11362773180007935, -0.879462480545044, 1.0842868089675903, -0.6042198538780212, 0.8120744228363037, -0.3376082479953766, 1.217504620552063, 0.05363258346915245, -1.6836390495300293, 2.03800892829895, -0.6061850190162659, 0.4635803997516632, 1.4204097986221313, 0.16939249634742737, -0.8881286978721619, -2.27596116065979, -1.2604596614837646, 1.0816200971603394, 0.5460999011993408, 1.4426847696304321, 0.7883576154708862, -1.4609297513961792, -0.7029662132263184, -0.2682291269302368, 0.22999627888202667, 0.005383598152548075, 0.05438517779111862, 2.1354312896728516, -1.813987374305725, 1.2436463832855225, 1.7504198551177979, 0.7249513864517212, 0.019786907359957695, -0.2157789021730423, 0.15400545299053192, 0.4060143232345581, 0.30977797508239746, -1.363065481185913, 0.8140509724617004, 0.17753884196281433, -1.5299268960952759, -0.2401251196861267, -0.3137999176979065, -0.9372947216033936, -0.17545217275619507, 1.8397958278656006, 1.7164140939712524, 0.8504317998886108, -1.1502727270126343, -0.2725651264190674, 0.17966049909591675, 0.14129838347434998, -0.6524964570999146, -1.2929248809814453, 0.3998486399650574, 0.13127431273460388, 0.33548104763031006, 0.12715086340904236, 0.4658159911632538, -0.03695795312523842, 0.4658903181552887, -1.1782422065734863, -1.469955563545227, -0.3705461323261261, -1.0484764575958252, -1.2785117626190186, 1.1922500133514404, -0.4658333659172058, 0.4848025143146515, 0.7078414559364319, -0.3357287049293518, -0.01116577722132206, -2.3407657146453857], [1.3627015352249146, 0.3190830647945404, -1.221523642539978, -0.6994436979293823, 0.8633442521095276, -0.7005369663238525, 0.7865275144577026, -1.9784398078918457, 0.9659916162490845, 0.06151445582509041, -0.3915504813194275, 0.392553448677063, 1.3144807815551758, -0.7869659662246704, 0.3582571744918823, 0.442516952753067, 0.8567978143692017, 0.7789140939712524, -0.3766767680644989, -0.10436704754829407, 0.16682715713977814, 1.2304271459579468, -1.1581801176071167, -1.0183318853378296, -0.8873576521873474, -0.6222658157348633, 0.5881041884422302, 1.12208092212677, -0.9636226892471313, 1.2022452354431152, 1.0029176473617554, -0.2674325704574585, 1.4269770383834839, -1.0187263488769531, -0.23202288150787354, -0.012896898202598095, 1.4230785369873047, -1.8659414052963257, -0.3576095998287201, 1.7032396793365479, -0.08863727003335953, 0.00950927659869194, -0.11792276054620743, -1.7864400148391724, 0.3658001124858856, 0.989112377166748, -0.8893575668334961, -0.34246188402175903, -0.4087989330291748, -0.472871333360672, 2.7680108547210693, -0.5911054015159607, -0.9856207370758057, 0.5517484545707703, -0.309175580739975, -1.178154706954956, 0.062332846224308014, 0.5334135890007019, 0.6512893438339233, -0.19364158809185028, 0.38219332695007324, -1.2274761199951172, -0.5714280605316162, -0.3046581447124481, 1.692279577255249, -0.07929550856351852, 0.23577430844306946, 0.3008292019367218, 1.3262052536010742, -1.136543869972229, 0.010853073559701443, 0.005282540339976549, 0.08447014540433884, 1.0667901039123535, -1.8858011960983276, 0.9745416641235352, 1.553499460220337, -0.2709548771381378, -0.31071123480796814, 0.031125493347644806, 0.20718643069267273, 1.7742280960083008, -1.8216105699539185, 1.5566973686218262, -0.02394828200340271, -0.5624169111251831, -0.4324344992637634, -1.3310043811798096, -0.46129274368286133, 1.05144202709198, -1.0312602519989014, -2.3483479022979736, 0.5477317571640015, -0.47898513078689575, 0.5119223594665527, -1.8868248462677002], [-0.18516185879707336, -1.7718079090118408, -1.1326321363449097, -0.11131584644317627, 0.9252709746360779, -0.17254869639873505, 0.2808559834957123, 1.0780644416809082, -1.7410238981246948, 0.07736837863922119, -0.6990236639976501, 0.34232231974601746, 0.4639510214328766, -1.3949079513549805, 0.4409324526786804, 0.30074542760849, 1.126137614250183, 0.5714470148086548, 0.8719927668571472, 0.06386516988277435, -1.0871297121047974, 0.36844784021377563, -0.22226545214653015, 0.24848638474941254, 0.8698693513870239, 1.6052470207214355, 0.571180522441864, 1.111830711364746, 1.4039288759231567, 0.4098559617996216, -0.48380592465400696, -0.011509530246257782, -1.1047922372817993, 0.241936594247818, 0.7278138399124146, -0.6657859683036804, 0.17075563967227936, -0.532080888748169, 1.1754686832427979, -0.734165370464325, -1.5260692834854126, -1.3056433200836182, -0.07695195823907852, -1.2070847749710083, -0.10227043926715851, -1.0122077465057373, -0.27332085371017456, -0.8294552564620972, 0.822939395904541, -1.209756851196289, 2.1764917373657227, 0.02125278115272522, 1.7444952726364136, -0.20800548791885376, 2.5551788806915283, -0.7579319477081299, 0.8330135941505432, -0.06374562531709671, 1.0625102519989014, -0.8417258262634277, 0.6903514862060547, -0.013940354809165001, 0.11543109267950058, 1.8806966543197632, -0.10520142316818237, 0.7046653628349304, 0.09692699462175369, 0.8237718343734741, 0.8678525686264038, 0.3070613741874695, 1.1960629224777222, 0.9770752191543579, 0.07614892721176147, -2.5592544078826904, 1.0315734148025513, -0.001204191124998033, 0.8687754273414612, 0.6778147220611572, 0.8069542050361633, 0.7838097810745239, -0.8961957097053528, 0.6597112417221069, 0.44283929467201233, -0.31651031970977783, 0.15122036635875702, 0.39787793159484863, -0.7300472855567932, -0.36488285660743713, -1.0983080863952637, -0.8775525093078613, 1.04971182346344, -0.27499955892562866, 0.05027264356613159, 1.7777389287948608, 2.473214864730835, -1.4476943016052246], [-0.7255716919898987, 0.22030574083328247, 2.7667548656463623, -1.1544501781463623, -1.6813596487045288, 0.11156265437602997, -0.05281227082014084, -0.7831876873970032, 0.053462572395801544, -0.2965802252292633, -1.3647292852401733, 0.4726603329181671, -0.4030752182006836, 1.4507756233215332, -0.053744684904813766, -1.1495780944824219, -0.5032036900520325, 1.2344413995742798, -0.17470794916152954, -0.2936396300792694, 0.9171932935714722, 1.1889772415161133, 1.4866907596588135, 1.8868553638458252, -1.5385537147521973, 1.5003366470336914, 0.14738331735134125, 1.9095019102096558, -0.5858043432235718, 0.8633201122283936, -0.19928500056266785, 0.4621325135231018, -0.5887993574142456, -0.6921005249023438, 0.25514286756515503, -0.4735962450504303, 0.6616275310516357, -1.0178128480911255, 0.8530194163322449, 2.1367549896240234, 0.6303310990333557, -0.7707723379135132, 0.31540897488594055, -0.013187546283006668, 0.8527811765670776, 0.25693297386169434, -0.24702519178390503, 1.1562806367874146, -0.5844938158988953, -0.12647174298763275, 0.7353927493095398, -0.7005088925361633, -0.4683285355567932, -0.23412708938121796, -0.8892836570739746, -0.5357480645179749, 1.6993286609649658, 0.12404301017522812, -0.9372504353523254, -0.961843729019165, 1.1851532459259033, 1.1975148916244507, 2.090435266494751, 0.34772175550460815, -1.190915584564209, -0.21907760202884674, 0.354878693819046, 0.985205888748169, 0.1971563994884491, 0.31568121910095215, 1.1821056604385376, 0.25645166635513306, 1.2423359155654907, 0.5191553831100464, -0.39600062370300293, -1.021515965461731, 1.910326361656189, -2.3566136360168457, -0.2149811089038849, 2.570502996444702, 1.4285014867782593, 0.8702058792114258, -1.4680501222610474, -0.05822309851646423, 0.33251485228538513, 0.7943596839904785, -2.361778974533081, -0.650307834148407, -0.1995425820350647, 0.4013501703739166, 0.31086716055870056, -1.3800487518310547, -0.44857102632522583, -1.7717077732086182, 1.7659671306610107, 0.27240243554115295], [0.24125316739082336, -0.9925662875175476, 0.3386250436306, 1.6193135976791382, -1.0491912364959717, -1.354414463043213, -0.016411608085036278, -1.5909980535507202, -1.2669527530670166, -1.6404350996017456, 0.9511212706565857, -1.1375192403793335, -0.39257973432540894, -0.6451063752174377, 1.3251149654388428, 0.6053317189216614, -1.207554817199707, -0.31836920976638794, -0.7203599810600281, -0.1008673831820488, 1.3106297254562378, 0.8055686354637146, -0.14285635948181152, 1.7288278341293335, 1.772202491760254, -2.1090712547302246, 0.31893789768218994, 0.7473299503326416, 1.139237880706787, -0.14836052060127258, -0.7366706728935242, 1.5227922201156616, -0.10468937456607819, 0.1838354617357254, -1.2426022291183472, 1.496427297592163, -1.1695672273635864, -0.2013407200574875, 1.325911283493042, 0.6590368151664734, -2.3183391094207764, 0.8259611129760742, -1.174544334411621, 1.999474287033081, -0.6787278056144714, 1.010398507118225, -0.831548273563385, 3.2681686878204346, 1.0496551990509033, 0.0524434968829155, -1.4911096096038818, 1.8639121055603027, 1.1953206062316895, -0.6466176509857178, -0.7997339367866516, 0.9635571837425232, 2.5469064712524414, 0.1630072146654129, 2.613921880722046, -0.25027576088905334, -0.08792667090892792, -0.10625562071800232, 0.8345057368278503, -0.4642738997936249, -1.1700615882873535, -0.4597894549369812, 0.26257747411727905, -0.24041961133480072, 1.4413036108016968, 0.19786964356899261, 0.8781869411468506, -1.2455140352249146, -0.8625131249427795, 0.7346513271331787, -0.7164794206619263, -1.2513865232467651, -0.07392877340316772, 0.7409714460372925, 0.8465184569358826, 0.015809573233127594, -0.034431323409080505, -0.693299412727356, 0.027323488146066666, -0.45501288771629333, -0.56785649061203, 0.48792174458503723, 0.9981936812400818, 0.4809179902076721, -2.1823997497558594, -0.902606189250946, -0.747947633266449, -0.7225383520126343, 1.1554152965545654, 0.5582703948020935, 0.2897019684314728, -0.20577354729175568], [0.7542818784713745, -0.6785310506820679, 0.39368826150894165, 0.4067305624485016, 0.7055584192276001, -1.224312424659729, 0.8046392202377319, 0.2697583734989166, -0.630192756652832, 0.42772120237350464, 0.06629599630832672, 1.097737431526184, -1.2203179597854614, -1.7829257249832153, 1.6410670280456543, -1.042129635810852, 0.3238098621368408, 0.15180917084217072, -0.8799448013305664, -1.7956396341323853, 1.5300507545471191, -2.8842668533325195, -1.2887009382247925, -1.5243228673934937, 0.7538391351699829, 0.36510157585144043, -0.8272484540939331, 1.4306349754333496, -0.7941817045211792, 1.5045186281204224, -1.201220154762268, -0.8102702498435974, 0.39774391055107117, 1.600312352180481, -0.1905660480260849, 0.002959097735583782, 0.9524161219596863, -0.5193025469779968, -0.7661958932876587, -0.4403037428855896, -0.16337688267230988, 0.7255859375, 1.0126168727874756, -1.455654263496399, 0.24492931365966797, -0.4177689850330353, -0.028118517249822617, -0.6420842409133911, 1.534143328666687, 0.1412643939256668, 0.08933531492948532, -1.212306261062622, 0.6197423338890076, 1.1615681648254395, -0.24263238906860352, 0.0651162713766098, 0.8655267357826233, -1.2479830980300903, -0.2072889506816864, 0.4835937023162842, 1.1075464487075806, -0.9272027611732483, -0.047239795327186584, 0.8759022951126099, 2.2229185104370117, -2.3306021690368652, -0.22666415572166443, -0.7824850082397461, -1.426595687866211, -0.45379331707954407, 2.342115640640259, 0.5787918567657471, -1.6044731140136719, 0.9031794667243958, 0.7803446650505066, -0.8521198630332947, 2.438047170639038, -0.3291901648044586, -0.14095649123191833, 0.09459756314754486, -0.0807240828871727, -0.23307786881923676, 1.0857117176055908, -1.2259535789489746, -0.7470132112503052, -0.6149498224258423, -0.2674184739589691, -0.6845077276229858, -1.034843921661377, 0.6351592540740967, -0.4387744665145874, 1.5040212869644165, 0.47117501497268677, 1.5237468481063843, 0.2676217257976532, -0.1020439937710762], [-1.052130103111267, -0.04899299144744873, 1.2543439865112305, -2.2481086254119873, -0.09245967119932175, -0.2630916237831116, 0.8201330900192261, -0.48118793964385986, -0.5395132899284363, 0.2590722441673279, 1.7696142196655273, 1.0626636743545532, 0.029143720865249634, 0.08302000164985657, 0.10780200362205505, -1.2328886985778809, -0.6498956084251404, -2.2905337810516357, -1.6580296754837036, 1.596724271774292, -0.26588407158851624, 0.13252320885658264, 0.34128043055534363, 0.6577730774879456, -0.3257893919944763, 1.526710033416748, -1.410609245300293, -0.25719088315963745, -0.31829506158828735, -1.2236804962158203, -2.214632749557495, 0.4185909926891327, -0.05377182364463806, -0.9694523215293884, 0.026554957032203674, -0.016330359503626823, -0.1243242546916008, -0.16541346907615662, -0.41781753301620483, -1.0415666103363037, -0.1693073809146881, -0.2950073778629303, -0.19672371447086334, 1.4888596534729004, 0.8546295166015625, -0.2597123384475708, -0.25202012062072754, 1.1485642194747925, 0.9552196264266968, -0.35858097672462463, 0.6725996732711792, -0.34376680850982666, -1.7367252111434937, -1.211631178855896, -0.8409102559089661, 0.8251405954360962, 0.8811870217323303, -0.8953311443328857, -0.1642448753118515, -0.8155817985534668, -0.28214335441589355, 1.1314531564712524, -0.4359460473060608, -2.17513370513916, -0.9663355946540833, -1.5126532316207886, -0.5713685154914856, -0.05813547223806381, 1.9489893913269043, 0.10729558765888214, 1.7680506706237793, 0.13569006323814392, 0.10966996848583221, -0.32964691519737244, -1.4638628959655762, 1.3047764301300049, -0.41182419657707214, 0.41151702404022217, -0.11369743198156357, -0.06555869430303574, 0.7833410501480103, -1.125617265701294, -1.840407371520996, -0.5086259841918945, 0.6367419362068176, -0.1448633372783661, -1.055152177810669, 0.40754544734954834, 0.41561245918273926, -0.34237903356552124, -0.6097890138626099, -0.721576452255249, -0.05201129615306854, 0.6287444233894348, -1.6118993759155273, -0.3192315101623535], [-1.000478744506836, 1.6636900901794434, 0.06871362030506134, -1.2327309846878052, -0.9067776203155518, 0.4872322976589203, -0.8581846952438354, -0.4975232481956482, 0.7500845193862915, 1.3000091314315796, -0.17914564907550812, 2.1671581268310547, -0.2414468675851822, -0.36586618423461914, -0.5448182225227356, -0.39549028873443604, 0.7602397799491882, -0.6850574016571045, -1.052096962928772, 1.3689830303192139, 0.6548721194267273, 0.06112939119338989, 1.2257276773452759, -0.3748530149459839, 0.2159690260887146, 2.459528923034668, -0.07571205496788025, 0.4412076771259308, 0.6755626201629639, -0.43201813101768494, -0.49413561820983887, 0.40743309259414673, -2.5233991146087646, -1.0887234210968018, 0.3145248591899872, -1.5679032802581787, 1.0784510374069214, 0.7991921305656433, 0.6978957653045654, -1.335747241973877, -0.287269651889801, -2.818596601486206, 1.8868434429168701, 1.1452912092208862, 0.3319898247718811, 0.25505927205085754, -0.1090254932641983, -1.2375551462173462, -0.27252325415611267, 0.1649588793516159, -0.5819490551948547, -0.1530124545097351, 0.8338139653205872, 1.415043830871582, 0.15431305766105652, -0.3310454785823822, 1.8777226209640503, -1.9487619400024414, -1.3751686811447144, 0.5983136892318726, -0.5252707004547119, 0.6512371897697449, -0.10739373415708542, -0.4475949704647064, 1.0869884490966797, 0.032306794077157974, 0.6635265350341797, 0.5989348292350769, 1.6756229400634766, -0.11590651422739029, -0.12097030878067017, 0.22951640188694, 1.3076668977737427, -0.8572896718978882, 0.11756826192140579, -0.9059399366378784, 0.941121518611908, 0.4446295201778412, 0.4085625112056732, -1.105562448501587, 0.09022674709558487, -2.622140884399414, -0.3499405086040497, 0.16874302923679352, -0.7887124419212341, -0.4809935688972473, -0.5731713771820068, 1.6726891994476318, 0.33120059967041016, 0.4750478267669678, 1.0143793821334839, -0.008105170913040638, -0.4805814325809479, -0.1680455356836319, 0.9002379179000854, -0.5530370473861694], [1.2557209730148315, 0.982088565826416, -1.3377209901809692, -0.2431478053331375, 0.07848411798477173, 0.3893650770187378, -0.5288052558898926, 1.5176697969436646, -0.902926504611969, 0.4667855203151703, -1.298743486404419, -0.46899664402008057, 1.0632712841033936, -0.35316044092178345, 0.1444004774093628, -0.3941689431667328, -1.2749409675598145, 0.4616248607635498, -0.4227634370326996, -0.7176706790924072, 1.0334895849227905, -2.0801992416381836, -0.9899066090583801, -0.09188058227300644, -0.3270873725414276, 1.0498905181884766, -1.513893485069275, 0.11698516458272934, -0.05321485549211502, 1.0616718530654907, 0.3300793170928955, -1.4724388122558594, -0.592505693435669, -1.2502026557922363, -0.9449618458747864, -0.2663489282131195, 0.4820016622543335, 0.37914493680000305, -1.1959691047668457, -1.724187970161438, 0.2311234176158905, -0.10515276342630386, -0.36018484830856323, -0.598162055015564, 2.130762815475464, 1.305930733680725, -0.2633976340293884, -0.2704252600669861, 0.8644351959228516, 0.5620707869529724, 0.5258893966674805, -0.764086902141571, -0.44293951988220215, -0.841782808303833, -0.24425898492336273, 0.24746105074882507, 1.0975476503372192, 1.6813362836837769, -0.5624770522117615, -0.6416207551956177, 1.0903756618499756, -0.3960750699043274, -0.34257012605667114, 0.7386184930801392, -0.7125055193901062, -1.810420036315918, 0.6163699626922607, -0.4332277774810791, 0.18760353326797485, -0.26136186718940735, -0.26256367564201355, 0.5545255541801453, 0.36100029945373535, 0.013600938953459263, -0.06844398379325867, 0.5177666544914246, -1.4735058546066284, -0.6300014853477478, -2.1716952323913574, 0.5855305194854736, -0.12568287551403046, -1.4431158304214478, -0.2205512523651123, 1.3649203777313232, -0.26791852712631226, 0.9161984920501709, -1.2099055051803589, 0.5918372869491577, 0.03202449157834053, -0.09105376899242401, -0.45192596316337585, 1.6604554653167725, 1.3483846187591553, -0.14680640399456024, 0.31709232926368713, 0.2518453896045685], [-0.07356753945350647, 1.1241079568862915, 1.7500020265579224, 0.5362411141395569, 0.7285184860229492, -0.10088586062192917, 0.026012476533651352, -0.6613684296607971, 0.5998511910438538, -0.38078323006629944, 0.38226401805877686, 0.9976019263267517, -1.2675336599349976, -1.2877784967422485, -0.4816204309463501, -1.3588041067123413, -0.3899654746055603, -0.28298404812812805, -0.9567863941192627, -1.2431186437606812, 2.6135659217834473, 0.8003730773925781, 0.7759250402450562, -0.3146354556083679, -0.5192477703094482, 1.5408062934875488, 1.8579171895980835, -0.5159883499145508, -1.5539696216583252, -2.6741273403167725, -2.0221927165985107, -0.0233903918415308, -0.5381492376327515, -1.021759033203125, -1.2631868124008179, 1.768524169921875, 1.1367082595825195, -1.2501864433288574, -0.8490656018257141, 1.2855448722839355, 2.3559374809265137, -1.307024598121643, -0.6203823685646057, 1.6904311180114746, 0.6976683735847473, 0.3246552646160126, -0.7926210761070251, -3.8580145835876465, 0.8377923965454102, 0.45144766569137573, -0.32732826471328735, -1.8497565984725952, -1.3411593437194824, 0.9479410648345947, 1.3218071460723877, -0.7363241910934448, -0.4782921075820923, 1.0322540998458862, -0.5565536618232727, 0.6714609861373901, -0.8727023005485535, -0.2542192339897156, -0.8580556511878967, -0.9730688333511353, -2.1743404865264893, -1.214034080505371, -0.2540653944015503, -0.11555667221546173, -0.33568325638771057, -1.560246229171753, -1.3322099447250366, -1.4662253856658936, -0.3870331943035126, 1.7302190065383911, -1.1668323278427124, 0.6786954402923584, -0.6371580362319946, -0.5600716471672058, 0.8359249234199524, -1.576356053352356, -1.7267574071884155, 1.3333889245986938, 0.672618567943573, -0.0990133136510849, 0.941815197467804, -0.9370039701461792, -0.6153120994567871, 0.23997808992862701, 1.5267350673675537, -0.4886857271194458, -0.4009375274181366, 0.33074069023132324, 1.5980031490325928, 0.638735294342041, 0.6305623650550842, -0.1630246937274933], [-0.18161681294441223, 0.05504021793603897, 0.4480273723602295, 0.5560941696166992, 0.14199475944042206, 0.7975977063179016, -0.4624764025211334, -0.1765851080417633, 0.514901340007782, 0.7862944602966309, -1.2536412477493286, -0.7949573397636414, -0.44088879227638245, 1.211979866027832, -0.3284832835197449, -1.3388715982437134, 1.2492448091506958, -0.9273935556411743, -1.423996090888977, 2.620872974395752, 0.6196911334991455, -0.34418970346450806, -0.03477639704942703, 1.0453011989593506, -0.3224446177482605, -0.33569252490997314, -0.33236485719680786, -1.2604947090148926, 2.5120418071746826, -1.8373552560806274, -0.9545185565948486, 0.15559396147727966, -0.5570327639579773, 0.865862250328064, -1.132741928100586, -0.2098686695098877, 0.22019940614700317, -0.011274571530520916, 0.4165700674057007, -1.3415720462799072, -1.8129726648330688, 0.39415886998176575, -1.3748304843902588, -2.219076633453369, 0.1185072660446167, 0.7244240045547485, 0.1534215360879898, -0.290100634098053, -0.48049408197402954, -0.1715264916419983, -0.40563899278640747, -0.3690424859523773, 0.2248954027891159, 0.27614161372184753, 1.1476045846939087, -1.4326974153518677, 0.06841832399368286, 0.2586337924003601, 1.24750816822052, 0.2527741491794586, -0.8686320781707764, 1.3709954023361206, -0.6885780096054077, -0.11583936959505081, -1.4839390516281128, 0.8467212915420532, 0.31731003522872925, 1.097078561782837, 1.4400027990341187, -0.15349456667900085, 0.40677979588508606, -0.9167298674583435, 0.5526453256607056, -0.4370354115962982, -0.28495529294013977, 0.36100971698760986, 0.06732603907585144, 0.5548655390739441, -0.2639508545398712, -1.556047797203064, -0.45199546217918396, 0.5719805955886841, 1.4228452444076538, -0.20979562401771545, 0.36938947439193726, -1.6387532949447632, 0.7407510876655579, -1.0255990028381348, -0.22113612294197083, 0.2791156470775604, 0.9699791669845581, 1.6056904792785645, -0.23695547878742218, 1.309086561203003, -0.2982412576675415, -0.7100424766540527], [0.21873052418231964, 0.4942021071910858, -1.0258574485778809, -0.5915423631668091, 0.3731672763824463, -0.17020320892333984, 1.5671212673187256, 1.0094096660614014, -0.41793134808540344, 0.4723691940307617, -0.864765465259552, -1.474584937095642, -1.013981580734253, -1.772340178489685, -0.12903402745723724, 1.0547014474868774, 0.6751242280006409, 1.606561303138733, 0.0644102469086647, -0.3758912682533264, -0.499186247587204, 0.4356420636177063, -0.9385013580322266, 0.6251951456069946, 0.7467008829116821, 0.9142126441001892, 0.19347554445266724, -0.11936673521995544, -0.9992434978485107, 2.4639408588409424, 0.0777682512998581, -0.4535636901855469, 2.875425100326538, -0.5681242346763611, 1.2204153537750244, 1.247269868850708, 1.0109210014343262, 0.8463128209114075, -0.5247541069984436, -0.1745733767747879, -0.2825031578540802, -0.5629675984382629, 0.3987809717655182, 0.6823137402534485, -0.40287551283836365, 0.6782243251800537, -1.275475263595581, 0.725010871887207, -1.616767406463623, 1.1629105806350708, -0.04997559264302254, -0.6872217655181885, 0.3725147843360901, 0.24586480855941772, 0.8525930047035217, 0.3298909366130829, 0.6556268930435181, -1.2234138250350952, 0.4230719804763794, 1.7146596908569336, -0.8073396682739258, 0.6234700679779053, 1.3077588081359863, -2.157215118408203, 1.9954969882965088, -1.206986665725708, 1.4930284023284912, 0.16879230737686157, 0.26520851254463196, -0.5556225776672363, -0.4851975440979004, 0.11237434297800064, 0.6499543190002441, 0.4817877411842346, 0.2715439200401306, 0.9476683735847473, 0.15409919619560242, -0.2816934585571289, 1.048236608505249, 0.5124945640563965, -1.729303002357483, -1.4279228448867798, 1.4109045267105103, -1.398507833480835, -0.48454663157463074, 1.2457654476165771, 0.18577665090560913, -0.1805577129125595, -0.9542533159255981, -1.2882269620895386, -0.9683015942573547, 0.5273758769035339, 1.1019691228866577, -0.13409030437469482, 0.8933281898498535, 0.3717651069164276], [-0.25996026396751404, 0.28648582100868225, 0.033320095390081406, -1.262998342514038, 0.32470864057540894, 1.313332438468933, -0.11023566126823425, 0.4175333082675934, -0.0809088870882988, -0.657758891582489, -2.0822386741638184, -0.34440886974334717, 0.648453414440155, -2.8098766803741455, -1.9596513509750366, 0.8076561093330383, -1.1458473205566406, 0.31517693400382996, 0.37864717841148376, -0.5424290895462036, 0.502762496471405, 1.2659825086593628, 1.6847091913223267, -0.2232547402381897, 0.004045980051159859, 1.5531084537506104, -2.0993261337280273, 0.003561720484867692, -1.2816333770751953, 1.4390262365341187, -0.03257149085402489, 0.6851922273635864, 0.27649158239364624, -1.425117015838623, -0.7331514954566956, 1.6778000593185425, -0.13188059628009796, 0.6767988801002502, 0.21127989888191223, 0.031322184950113297, -1.494545578956604, -0.4331617057323456, 0.3254479467868805, 0.003693448379635811, -0.47624674439430237, -0.4902972877025604, 0.23444268107414246, 0.6679388284683228, 0.23855268955230713, -0.7918826937675476, 0.08552423864603043, 1.2424101829528809, 0.7909477353096008, 0.8186451196670532, 0.8834939002990723, 0.5234646201133728, 1.5988579988479614, -1.1510310173034668, -0.8852809071540833, -0.41303426027297974, 0.39029252529144287, -0.5636803507804871, -1.3035657405853271, 0.2004171907901764, 0.7666659951210022, 0.2278061956167221, -0.49125105142593384, -1.8662368059158325, 0.2808057963848114, -0.6052228212356567, 0.11592863500118256, 0.7033218741416931, 1.1384015083312988, 0.2338939756155014, -1.111465573310852, -1.4354597330093384, 0.22502532601356506, 0.052919067442417145, 0.23005586862564087, 0.6261300444602966, 1.3610974550247192, -0.32971519231796265, -2.7584381103515625, 0.15625318884849548, 0.7104386687278748, 0.45760220289230347, 0.8784875869750977, -1.6745615005493164, -0.70449298620224, -1.930677056312561, -1.2914918661117554, 1.1750541925430298, -0.6417241096496582, -1.8271788358688354, -0.3088666796684265, -0.22609210014343262], [1.564987301826477, -0.9942742586135864, 1.7127236127853394, -0.812287449836731, -0.4296739399433136, -0.3704548180103302, 0.15656955540180206, -0.8184484243392944, 0.3907223641872406, -0.6627305746078491, 0.9282946586608887, 0.6834993958473206, -0.3075072765350342, -1.6214245557785034, 0.3233512341976166, -0.6888303160667419, -0.6276222467422485, 0.5824449062347412, 0.27757009863853455, 0.5363104939460754, 1.132789969444275, -0.21668046712875366, 0.2762782871723175, -1.5468670129776, -0.9552311301231384, 1.0538434982299805, 1.0353527069091797, -0.9028458595275879, 0.8121612668037415, -1.2358918190002441, -1.7332127094268799, -1.9686321020126343, -0.24476072192192078, 2.759706497192383, 0.8877031207084656, 0.006349568720906973, 0.4903826415538788, 0.0007312198868021369, 0.3642219603061676, 0.182756245136261, 0.07098716497421265, 0.9091426730155945, 0.6214998960494995, -1.9449045658111572, -2.1391851902008057, 1.3981913328170776, -1.0342518091201782, -1.2096625566482544, -0.28387320041656494, -0.48468098044395447, 1.018036127090454, -1.7477856874465942, 0.3273864686489105, 1.0268990993499756, 0.9262863397598267, -0.19544482231140137, 0.4554249346256256, -0.34126901626586914, -1.6564929485321045, -0.020174361765384674, -1.2140148878097534, 0.7470558881759644, -0.7028835415840149, 1.8893024921417236, -1.2678474187850952, -1.918447732925415, 0.10429048538208008, -1.34916090965271, 0.5120405554771423, -0.4886321425437927, 0.48697206377983093, -0.4047168791294098, -0.2638929486274719, 0.35743317008018494, -0.3553614616394043, 0.3898380398750305, -1.6774929761886597, -0.27774113416671753, 2.089323043823242, 3.01046085357666, 1.7325299978256226, -0.6741759777069092, 0.7400872707366943, 0.7328580021858215, -0.5335443019866943, -0.8260652422904968, -1.27151358127594, -0.9333855509757996, -0.6411970853805542, 0.2253759801387787, 0.7783210873603821, 0.31560975313186646, 0.06852748245000839, -0.5652256608009338, 0.4565787613391876, -0.9492173194885254], [-1.4636667966842651, -1.0680452585220337, -1.6290769577026367, -0.20989923179149628, 0.2299347072839737, 0.584086537361145, -0.9386611580848694, 2.466700792312622, 0.4902009963989258, 1.3741388320922852, -0.13042724132537842, 0.4835336208343506, -0.3260371685028076, -0.43071842193603516, -0.18455757200717926, 0.22430847585201263, 1.4259655475616455, -0.05797345191240311, 1.7671825885772705, 0.12266324460506439, 0.0075119794346392155, 0.8732424378395081, 0.05042356997728348, -1.330010175704956, 0.20363251864910126, 0.3135520815849304, -0.8626662492752075, -1.1993488073349, 1.7838155031204224, 0.7645031809806824, 1.9385108947753906, -0.15809334814548492, 1.9068185091018677, -0.31031620502471924, -1.8166296482086182, -1.5889250040054321, 1.5728099346160889, -0.9636968374252319, -1.069871425628662, -0.4866698384284973, 0.9282272458076477, -1.0055538415908813, -0.2736036479473114, -0.4003250300884247, 0.011387371458113194, -1.6698095798492432, 0.15349406003952026, -1.2810920476913452, -0.9603993892669678, 1.3723106384277344, -0.1364683359861374, -2.367676019668579, -0.18931537866592407, -0.35565391182899475, -0.14191703498363495, -0.6721847057342529, -0.890005886554718, 0.07018603384494781, -0.06901853531599045, -0.719920814037323, -1.1853468418121338, 0.4151982069015503, -0.45389845967292786, 0.7156105637550354, -1.0555520057678223, -0.8377936482429504, 0.6204683780670166, -0.8453202247619629, -0.10212689638137817, 0.7728626728057861, -0.3361132740974426, -0.1143624410033226, -1.265325665473938, 0.29689422249794006, 0.8206124305725098, 0.21408669650554657, 1.7115143537521362, -0.06236867606639862, 0.8275309801101685, -0.5019171237945557, -0.3942002058029175, 1.7002179622650146, 0.357887327671051, 1.540031909942627, -1.2060260772705078, -1.3290430307388306, 0.30657848715782166, -0.5907264947891235, -0.2239723652601242, -0.7926075458526611, 0.1619139313697815, 0.333332896232605, -0.1659778207540512, 2.3003013134002686, -0.88548743724823, 0.6651337146759033], [0.9681976437568665, 1.7356350421905518, -2.5141518115997314, 0.15125463902950287, -0.27017149329185486, -1.1774237155914307, -0.7482132911682129, -0.7377604246139526, 1.1102092266082764, 1.999262809753418, 0.014960241504013538, 0.11538109928369522, 1.1498479843139648, 1.463085651397705, 1.051817774772644, -0.9977497458457947, 0.03283795714378357, 1.4253771305084229, -1.2067360877990723, 0.02220870740711689, -0.20942774415016174, 0.008302463218569756, -0.49217525124549866, -1.3295401334762573, 0.7432507276535034, -0.055428341031074524, -0.7726231217384338, -0.6661116480827332, -0.12880758941173553, -2.247313976287842, 1.3348801136016846, 0.9123289585113525, 0.4353311061859131, -0.6450116038322449, -1.2416415214538574, 2.6185760498046875, 0.11009228229522705, -0.23128294944763184, 1.8782193660736084, -0.5032833218574524, 0.9376694560050964, -1.2927464246749878, -1.3874232769012451, -1.7760916948318481, 0.23580506443977356, 0.5845791697502136, 0.12090510129928589, 0.19576382637023926, 1.0753631591796875, 0.09808965027332306, -0.7654810547828674, 0.37600991129875183, -0.741673469543457, 2.03475022315979, -0.8446844220161438, 1.2519525289535522, 0.6335713267326355, -1.2951867580413818, 1.381568193435669, 0.7832409739494324, 1.1124187707901, -0.986150860786438, -0.43666738271713257, -0.032457541674375534, 0.41667959094047546, -0.3142830729484558, -0.13189418613910675, -2.0519204139709473, 0.6086854934692383, -0.4123346209526062, 1.6546355485916138, 1.6152358055114746, 1.1443989276885986, -0.13083171844482422, -1.9801383018493652, 0.2442130446434021, 0.5067396759986877, 1.057828426361084, -0.839270293712616, -2.0024635791778564, -0.31322160363197327, -0.14178264141082764, -0.10574807971715927, -0.13532094657421112, 0.9170718789100647, -0.6254073977470398, -1.238312005996704, -0.48744410276412964, 1.3153828382492065, 1.3702857494354248, -0.053006138652563095, 1.1685452461242676, 2.1532201766967773, 0.1740640550851822, 0.5081782937049866, 0.607257604598999], [-0.6604056358337402, -0.1475464403629303, 0.00814431719481945, 0.46229416131973267, -0.9317179918289185, 0.720844566822052, 0.09445939213037491, 1.3805055618286133, -0.9340825080871582, -0.0181554164737463, -0.15292486548423767, 0.10858739167451859, -0.7928085923194885, 1.7587448358535767, -1.2710206508636475, 0.798991858959198, 0.7176530957221985, -1.4003556966781616, -0.13545116782188416, 0.44268906116485596, 1.1890079975128174, -0.9704640507698059, -0.45291364192962646, 2.0193920135498047, 0.7370357513427734, 1.4074819087982178, 2.0508933067321777, -0.3049745559692383, -1.6042667627334595, 1.3965916633605957, -0.36339378356933594, 0.21700426936149597, -0.8075446486473083, 0.5515861511230469, 0.10431144386529922, -0.413293719291687, -0.9217705130577087, 0.40707892179489136, 0.8641396164894104, -0.9384562969207764, 1.0453264713287354, 1.1107892990112305, -1.783648133277893, -0.09898069500923157, 0.794049859046936, 0.3829650580883026, -0.13931089639663696, -0.5282509326934814, 0.12381142377853394, 0.13782377541065216, -0.2423006147146225, -0.19790968298912048, -0.05132526159286499, 0.49908187985420227, -1.3882975578308105, -1.692427396774292, -0.5714029669761658, -2.117691993713379, 0.17611220479011536, 1.0209732055664062, -0.3510524034500122, -1.2890596389770508, -1.7304531335830688, -0.812927782535553, -0.6639413237571716, -0.3417429029941559, -0.3199867010116577, -0.3951396644115448, 0.17241571843624115, 0.2076999545097351, 0.8331297039985657, 1.3367094993591309, 0.4617210030555725, -0.9365255832672119, 2.2860822677612305, 0.7407549619674683, -0.644608199596405, -0.4620504677295685, 0.03763488680124283, 0.8092900514602661, -0.18058858811855316, 1.1265778541564941, -0.4029216766357422, 0.9176721572875977, -0.7886549830436707, 0.7072068452835083, -1.5580949783325195, -0.19844168424606323, 0.6743882894515991, 0.8476694226264954, -1.745545744895935, -2.024843454360962, 0.4234775900840759, 0.7378765344619751, 1.5668466091156006, -1.4563332796096802], [0.8899574875831604, -0.25469261407852173, -1.5240031480789185, -0.455651193857193, 0.12544894218444824, -0.28641659021377563, 0.32338476181030273, -0.05794905498623848, -2.793898344039917, -1.4938790798187256, 0.13303668797016144, 0.04592578485608101, 0.8971273899078369, 0.24353240430355072, -0.5955632328987122, -1.0423345565795898, -0.6371053457260132, -0.25503507256507874, 0.2740892767906189, 0.11412075906991959, -0.16069142520427704, -0.4913072884082794, 0.7915984988212585, -0.1567775309085846, 1.6858594417572021, 1.0883439779281616, 2.045542001724243, -0.1413751244544983, 1.0902549028396606, -1.2263704538345337, -1.2125816345214844, -0.4091687798500061, -0.1142609715461731, 1.0944753885269165, 2.420534372329712, -1.8232343196868896, -0.6351487636566162, -0.9465770125389099, -1.0749198198318481, 0.9505958557128906, -0.4432367980480194, -0.44007304310798645, 0.37468352913856506, -0.3705582916736603, 0.443509578704834, -0.01577567495405674, 0.18603914976119995, 0.5913456678390503, 1.687995195388794, 1.2389411926269531, -1.0940937995910645, 0.9058927297592163, -1.308509111404419, -1.3968883752822876, 0.8741016983985901, 0.8610697388648987, 0.10538861155509949, 0.8699228763580322, -0.838701069355011, -0.007314283400774002, -0.6596779227256775, -1.2052384614944458, -0.15581324696540833, 0.47040891647338867, -1.3382360935211182, 0.11763180047273636, 0.3583467900753021, 0.1847415566444397, 0.7761963605880737, -0.7432385683059692, 0.023620517924427986, 0.5546333193778992, 0.4248618185520172, -1.2467907667160034, -0.8264517188072205, -1.451870083808899, 1.0084093809127808, -0.8644049167633057, 0.6209363341331482, 0.4282633364200592, 0.07913389056921005, 0.1145382672548294, 1.1933130025863647, -0.36304929852485657, -0.07175032794475555, 0.030238263309001923, 1.3742613792419434, 0.12957431375980377, -1.311745285987854, -0.03441178426146507, -0.8371915817260742, 0.039200276136398315, 0.8658791780471802, -1.8577139377593994, -0.5554517507553101, -0.7995668053627014], [-0.2191181480884552, 1.2310981750488281, 0.9848439693450928, -0.9576236009597778, -0.873973548412323, -0.6795287132263184, 1.2610539197921753, 1.1713120937347412, 1.683556079864502, 0.7676793932914734, 0.9027124047279358, 1.1681993007659912, -0.49795252084732056, -0.17081022262573242, -0.0792534202337265, -0.5702829360961914, -0.8072507977485657, -0.4294024705886841, 0.7727571725845337, -0.46899235248565674, -1.735746145248413, -0.8109987378120422, 1.7422655820846558, 0.3290499448776245, 0.21241635084152222, 1.169465184211731, 1.0132302045822144, -0.29681628942489624, 1.187023639678955, -1.1945549249649048, -0.11048233509063721, 0.12430732697248459, -0.7042582035064697, -0.3171458840370178, -0.04108361527323723, -2.1075286865234375, 1.6857749223709106, 1.6750198602676392, -0.19249728322029114, -0.5891498327255249, 0.43128737807273865, 1.5087288618087769, 0.9767901301383972, -0.43588241934776306, 1.4142857789993286, -1.603981852531433, 1.5470693111419678, 0.6997631192207336, 0.2734391689300537, -0.3413408100605011, -1.3209682703018188, 0.8080760836601257, 0.40340274572372437, -1.0262120962142944, -0.5864109992980957, 0.15781253576278687, -1.727826476097107, -0.11705728620290756, -0.6446756720542908, 1.1193811893463135, -1.091556191444397, -1.4646453857421875, 1.402859091758728, 0.9839804172515869, 1.6078463792800903, 1.7543467283248901, 0.88652503490448, 0.1396055370569229, -0.42190420627593994, 1.1634100675582886, -1.2847967147827148, 1.1287235021591187, -0.1927889883518219, 0.08660081028938293, -0.9662380218505859, 0.38845568895339966, -0.06006813794374466, -0.3643740713596344, -1.8506547212600708, 0.8567308187484741, -0.5755185484886169, -0.3950391709804535, 0.06640598922967911, 1.0683979988098145, 1.4015744924545288, 0.5999293327331543, 0.4827441871166229, -0.25273004174232483, -0.060515329241752625, -1.7216832637786865, 1.152942419052124, -0.5800198316574097, -1.0924859046936035, -1.3736572265625, 1.2106729745864868, 0.48178234696388245], [-0.35516557097435, 0.9786068797111511, -0.28883200883865356, -0.5863690376281738, -0.19403783977031708, -1.4147379398345947, -1.4925209283828735, 0.8610363602638245, 1.0408482551574707, 1.7302254438400269, 1.107832670211792, 0.9696766138076782, 0.882532000541687, 0.2633247673511505, 2.0894808769226074, 0.6546575427055359, 1.195108413696289, -0.9147436022758484, -0.11353475600481033, -0.5963572859764099, 0.027800295501947403, 0.9400990009307861, 1.0318598747253418, -1.084558129310608, -0.8724017143249512, 1.086184024810791, 0.01914278417825699, 0.8261121511459351, -1.51710844039917, 0.913359522819519, 1.0924720764160156, 0.010512711480259895, -0.23048318922519684, -0.954312801361084, 1.0731598138809204, -0.19285990297794342, 0.28639763593673706, 1.2112642526626587, 0.4999014437198639, 1.4201322793960571, 0.6762259602546692, -0.09357679635286331, 0.7823204398155212, -2.067120313644409, 0.844227135181427, -0.08128854632377625, -1.1204888820648193, -0.28255534172058105, 1.9562079906463623, 0.20040437579154968, -0.058242857456207275, 0.5879780650138855, -0.4900647699832916, 0.07757211476564407, 0.4678782820701599, -1.4157977104187012, 1.0794919729232788, 0.12608981132507324, 0.26118093729019165, 2.518859386444092, -1.1333658695220947, 0.712955117225647, -0.9021412134170532, -0.01312093436717987, -0.345859557390213, 1.841666340827942, 0.46588966250419617, -0.6855356097221375, 1.3726834058761597, 0.2467688024044037, 2.3060803413391113, 0.04906342923641205, 0.06853656470775604, 0.5763769745826721, 0.5685081481933594, 0.864528238773346, 1.4394879341125488, -1.3905783891677856, 0.513824462890625, -0.14767634868621826, 0.39691171050071716, -1.2216269969940186, 1.8145437240600586, -2.27653431892395, 0.43990302085876465, 1.3318700790405273, 0.17028774321079254, -0.7733973264694214, 1.323383092880249, 0.9823853373527527, -0.054274845868349075, -0.46321186423301697, 1.3304141759872437, -1.1391572952270508, -1.0654592514038086, 0.8536874651908875], [-0.1917392760515213, -0.18849343061447144, 2.0788753032684326, -1.0725383758544922, -2.06459903717041, -0.24526356160640717, -0.7791774272918701, 0.6023486256599426, 3.107783794403076, -1.0740653276443481, -0.04974035173654556, 0.8068488240242004, 0.4902224540710449, -0.9441048502922058, 0.05773448944091797, -0.23419184982776642, -0.5043299198150635, 0.10323993861675262, -0.3274412155151367, 0.35700011253356934, -0.15578322112560272, -1.3838046789169312, -0.5451629757881165, -0.3743251860141754, 0.8163660764694214, -0.2402266561985016, -0.6904807090759277, -1.007139801979065, -0.2919868230819702, -0.612404465675354, 0.49611613154411316, -0.34848448634147644, 0.17660300433635712, -0.5763411521911621, -0.03506306931376457, -0.19644412398338318, -0.1172729954123497, -0.9111077189445496, 1.7269279956817627, 0.21237413585186005, 0.7464522123336792, -2.359962224960327, -0.34315240383148193, -2.0285677909851074, -0.23736310005187988, 1.8448501825332642, 0.5727092027664185, -0.9747412800788879, 0.013703139498829842, -0.28095000982284546, -0.010641781613230705, -0.12025271356105804, -0.5640828609466553, -0.0761793926358223, 0.07332754880189896, -0.404519259929657, 0.5230896472930908, 1.0140271186828613, -0.7089340090751648, 0.6065343022346497, -0.016876477748155594, 2.5193862915039062, 1.0610418319702148, -0.312694251537323, 0.5077597498893738, -0.5022372603416443, -1.062729835510254, -0.4075261354446411, 0.1825120896100998, -0.6532937288284302, -0.8231952786445618, 1.298553705215454, -2.5895204544067383, 0.05876101180911064, -0.5144893527030945, -0.3536340594291687, -0.12693603336811066, 0.2730085849761963, 0.43252357840538025, -1.2355045080184937, -1.9722431898117065, -0.27246174216270447, 0.11485602706670761, 1.0584059953689575, -0.9161413908004761, 0.0021713310852646828, 0.3654513359069824, -0.8701576590538025, -0.9416553974151611, -0.7513465285301208, -0.3337550759315491, -0.13325347006320953, -1.2691127061843872, 1.1220914125442505, 2.1846020221710205, 0.3504582345485687], [-0.3041995167732239, 1.0584524869918823, -1.589493751525879, 0.5140326619148254, 1.3087365627288818, 0.5061746835708618, -1.1388823986053467, -0.5726355314254761, 0.5636438727378845, 0.7901911735534668, 1.1867170333862305, 1.443634033203125, -0.12991870939731598, 0.5672705769538879, -0.5685860514640808, -0.33256179094314575, -0.07182648032903671, 0.3777908384799957, 0.9852798581123352, -0.38060063123703003, 0.8825406432151794, -0.33176058530807495, 0.7496548891067505, -1.002327799797058, -1.9416359663009644, -0.5709546804428101, -0.4273354113101959, 1.055516004562378, 0.21058285236358643, -0.8405593633651733, -0.6011897325515747, 0.42723068594932556, 0.2238156795501709, -0.5662238597869873, 0.8927026391029358, -0.5627083778381348, -0.23112893104553223, -0.668354332447052, -0.23861946165561676, -0.44591692090034485, -0.6869527697563171, -0.32977503538131714, -0.3761826157569885, 0.5681428909301758, -0.47361841797828674, -0.5806041359901428, -2.0095174312591553, 2.7320468425750732, -1.895460844039917, 0.16896802186965942, -0.4958837330341339, -0.4800437092781067, 1.0140495300292969, -0.20135602355003357, 0.44846442341804504, 0.1352730095386505, 0.4022878408432007, 1.9428846836090088, -0.2301350086927414, -0.2902214229106903, -1.2864652872085571, -2.0686309337615967, 0.49771153926849365, 1.091291069984436, -0.5989727973937988, -1.3335484266281128, -0.9884699583053589, 1.2305105924606323, 0.6207820177078247, 0.20288309454917908, 1.0087754726409912, 1.7640259265899658, -2.4899771213531494, 0.6998181939125061, -0.16874973475933075, -0.5339862108230591, 1.3156038522720337, 1.0676639080047607, 1.0735667943954468, 1.3700876235961914, -0.13347958028316498, -1.4040359258651733, 1.1529630422592163, 1.266303300857544, 1.518943428993225, 1.2712321281433105, 0.9052395224571228, 0.10928903520107269, 0.7777360677719116, 0.7528443932533264, -0.8903780579566956, 0.5255033373832703, 1.610079050064087, 0.1506221741437912, -1.2409477233886719, 1.3057390451431274], [1.5454567670822144, 0.01164236105978489, -1.9846757650375366, 1.5088106393814087, -0.7510050535202026, 1.6186904907226562, 0.09681206196546555, -2.24721360206604, 0.20848287642002106, -0.2039259821176529, -0.6347165703773499, 2.201878309249878, -1.5091544389724731, 0.6674294471740723, -0.12041456997394562, 0.25928637385368347, -0.7843758463859558, -0.6900697350502014, 1.0904585123062134, 1.05764639377594, 0.5699771642684937, -0.39152100682258606, -1.6728371381759644, -0.43190568685531616, 0.9345526099205017, -1.3783884048461914, 0.6585928201675415, -0.4809578061103821, 0.0926424190402031, -1.1693516969680786, 0.3869040608406067, -2.7369213104248047, -0.11660310626029968, -1.2038264274597168, -1.4035638570785522, 0.6938037872314453, -1.5259524583816528, -0.503929615020752, -0.3424050509929657, -0.08043396472930908, 0.005092478822916746, 0.3732878267765045, 2.6069676876068115, 0.9559135437011719, -0.027830801904201508, 1.2623798847198486, -0.4030139446258545, -1.0628714561462402, 1.1036992073059082, -1.1846446990966797, 1.4880635738372803, -0.1503058671951294, -0.44058990478515625, -0.10342293232679367, 1.7770891189575195, -0.662487804889679, -0.725542426109314, 0.9350367188453674, -0.7621080875396729, 0.34406936168670654, -0.9539306163787842, 0.823951244354248, -0.5337828397750854, -1.1555103063583374, 1.9812414646148682, 1.2218571901321411, -0.41009974479675293, 0.08871171623468399, -0.584248423576355, -0.7452601194381714, -1.0512902736663818, 1.7589102983474731, -0.6071892380714417, 0.09567611664533615, -0.42124003171920776, 1.0548003911972046, 1.5171114206314087, 0.6965346932411194, -0.4429984390735626, 0.2509828805923462, -1.9018168449401855, 1.1197854280471802, 1.073634386062622, -0.16226889193058014, 0.9459161758422852, -1.3119103908538818, -0.4186396300792694, 1.782127022743225, -1.2979168891906738, 0.5270909070968628, 0.3966837525367737, -0.1695953756570816, -0.459185928106308, 0.2406337410211563, 0.16401252150535583, -0.9874309301376343], [1.1453816890716553, 0.734766960144043, 0.6104494333267212, 2.082494020462036, 1.1813937425613403, -0.07560397684574127, -0.8817757964134216, -3.1466598510742188, -0.8858622908592224, 1.0289815664291382, -0.9871195554733276, -1.1186972856521606, -0.1662103682756424, -1.2000412940979004, -0.16166086494922638, -1.2760142087936401, -0.9789595007896423, -0.6710243225097656, 0.6636791825294495, 0.411289244890213, -0.09029223769903183, 0.12057869136333466, -0.49746784567832947, -0.12469532340765, -1.0656355619430542, -0.28221458196640015, 0.7636414170265198, 0.45799294114112854, 0.29260554909706116, 0.3194305896759033, 0.947566568851471, 1.1908957958221436, -1.4559364318847656, 0.9206664562225342, -0.3453752398490906, -0.5347960591316223, -1.271444320678711, 1.7648441791534424, -0.8379240036010742, -1.5564674139022827, -0.38589853048324585, -1.1540111303329468, -1.2141163349151611, 0.7464174032211304, -0.5339120626449585, -1.0320041179656982, -0.860295295715332, -0.010480405762791634, -0.5422960519790649, 2.4298644065856934, -0.11258145421743393, -0.2687399685382843, -1.4565984010696411, 2.0773751735687256, -1.6708050966262817, -0.4724671244621277, -0.9418349266052246, 1.671130895614624, 1.165168285369873, -0.6054382920265198, 1.3081530332565308, 0.23123544454574585, -0.518902063369751, 1.4333726167678833, 0.05821870639920235, -0.62473064661026, 0.7527378797531128, 1.4364521503448486, -0.18063728511333466, 0.18839940428733826, 1.3971621990203857, -0.5835351347923279, -0.1869777888059616, -1.2968380451202393, -0.20759204030036926, -1.0680851936340332, -0.2993607223033905, -0.7210599184036255, 0.3355618417263031, -0.6099351644515991, 1.59575355052948, -0.08912382274866104, -0.8969129323959351, 0.014807152561843395, 0.3262758255004883, 1.1439039707183838, -0.6136911511421204, -0.8461293578147888, 0.10197614133358002, 0.7837309837341309, 0.7672555446624756, -2.1253840923309326, -1.1342546939849854, 0.36561015248298645, 0.1604306548833847, -2.56056547164917], [-0.056981898844242096, -0.8989623188972473, 0.26137545704841614, 0.19691334664821625, 0.5220058560371399, -1.7225233316421509, -0.4561885595321655, -0.2664068937301636, -0.24733103811740875, -0.9156325459480286, -1.7000316381454468, -0.6991307139396667, -1.1000937223434448, -0.38814395666122437, 1.6932766437530518, 0.07204420119524002, -1.3634966611862183, -0.5598084330558777, -0.1916348785161972, 0.6584640145301819, -2.2638556957244873, 1.3245530128479004, -0.9030329585075378, -1.6532491445541382, 1.0284614562988281, -0.7597441077232361, -0.46362820267677307, -0.6478586792945862, 0.9978489875793457, 1.1269285678863525, -0.6982783079147339, -0.4782845973968506, 0.4388904273509979, -0.5935690999031067, 0.0875258520245552, 0.21667088568210602, -0.5177256464958191, -0.2438361793756485, 0.029949193820357323, -0.6790105104446411, 2.191842794418335, -1.251941442489624, 0.9860514402389526, 0.3243027925491333, 0.18755698204040527, -2.62406325340271, -0.8528687357902527, -0.04782133176922798, 0.774192750453949, -0.0185908954590559, 0.7008641362190247, -0.9118286967277527, -1.0143035650253296, 0.5756906270980835, 0.49544215202331543, -1.6127263307571411, 0.5909994840621948, -1.1297663450241089, -0.2453281432390213, 0.8604313731193542, -0.1497000753879547, 2.436599016189575, 1.5552908182144165, 0.736656665802002, -2.202146053314209, -0.4107207953929901, -0.5410231947898865, -0.22496727108955383, -0.2131582349538803, -1.788259506225586, 0.39242851734161377, -0.9396747350692749, -0.3589814007282257, 0.36776211857795715, 1.4122905731201172, 0.7162569165229797, -1.2820333242416382, 0.6469306349754333, 0.4665064215660095, -0.12645873427391052, 1.6139189004898071, -0.38792577385902405, 0.3476412296295166, 2.2179946899414062, -0.5633615851402283, -0.7440720796585083, 0.6617629528045654, 0.5196205973625183, 0.10005571693181992, -0.11638688296079636, 0.19965289533138275, 0.7700883150100708, -1.5564850568771362, 0.01765761710703373, 1.0768712759017944, -1.016036033630371], [1.116678237915039, 0.047591693699359894, 0.07614673674106598, -0.31156569719314575, 0.9570826292037964, -0.33231475949287415, 1.0664643049240112, -0.0008378541097044945, -0.11842000484466553, 1.4740031957626343, -1.5257118940353394, -0.4282357692718506, 0.35725417733192444, 1.319219946861267, -0.6006031036376953, -0.12284963577985764, -0.6059670448303223, 0.23678524792194366, -0.262746661901474, -1.246826171875, 1.1695795059204102, 0.05385966598987579, 0.014546552672982216, -0.506261944770813, 1.2441356182098389, -0.24092641472816467, 0.4608844518661499, 0.6067044138908386, 1.7701457738876343, 0.7296796441078186, -0.2808989882469177, -0.270981103181839, 0.3926229476928711, -1.215714693069458, 0.1744968742132187, 1.0081275701522827, 1.5194822549819946, 0.17232617735862732, 0.8211826086044312, 0.4051341116428375, -0.011022062040865421, -1.776801347732544, -0.3141663074493408, 0.9622313380241394, 0.07745322585105896, 1.172461748123169, -0.7530659437179565, 1.5240888595581055, -1.9178162813186646, -0.5176776051521301, -1.9229907989501953, -0.002468975493684411, -0.24473528563976288, 0.5576306581497192, 1.3157676458358765, 0.06086180731654167, 0.18811377882957458, -1.2256499528884888, 0.12379588931798935, -0.6472005844116211, -0.8264755606651306, -0.39033910632133484, 0.37530672550201416, 0.9636746048927307, -0.3431013822555542, -0.223097562789917, 0.7137653827667236, 0.6503947377204895, 0.5620886087417603, 0.4386025369167328, -0.7352032661437988, -0.4363519251346588, -1.108488917350769, 1.2213160991668701, 1.3290592432022095, -1.8124815225601196, 0.4415249228477478, 0.6406148076057434, -0.5844061970710754, -0.5133653879165649, -0.18134260177612305, 0.4237145483493805, 2.6910715103149414, -0.047748394310474396, 0.8732319474220276, -1.0782917737960815, -0.3211009204387665, -1.6105316877365112, 1.1395436525344849, -0.2618025243282318, 0.08036229014396667, -1.5690066814422607, 1.9989314079284668, 0.30566489696502686, -1.5769333839416504, -0.11856072396039963], [-1.2162057161331177, -0.21589191257953644, -0.8434592485427856, 1.0307356119155884, -1.4058009386062622, -0.5928440690040588, 0.520419180393219, 0.8039630055427551, -0.7299857139587402, -0.9572217464447021, -1.982008695602417, -1.6003128290176392, -1.3278263807296753, 0.4680219292640686, -1.7258135080337524, -1.0082404613494873, -0.014185260981321335, 0.5836926102638245, 0.16168320178985596, 0.844058096408844, -0.14939925074577332, -0.20324909687042236, -1.0786627531051636, -0.9309016466140747, -0.05427689850330353, -1.2953743934631348, -0.35218825936317444, 0.9631378054618835, 0.9175106883049011, -0.5476603507995605, -1.1384398937225342, -0.18927301466464996, 1.3832950592041016, 1.9208158254623413, -2.2480506896972656, -1.6174275875091553, 0.6361578702926636, 0.8953825235366821, 0.8245815634727478, 0.21398064494132996, 1.8340368270874023, -1.157168984413147, -1.7369612455368042, -0.027750683948397636, -1.124093770980835, 0.19305051863193512, -0.44359588623046875, 1.306749939918518, 1.4711357355117798, 1.5404287576675415, -0.37925490736961365, 0.4585639238357544, 0.6676915287971497, 0.5389013886451721, 0.06528017669916153, -0.3478536903858185, -0.5631411075592041, -1.3879187107086182, 0.7161977291107178, 0.2106562852859497, -0.8118163347244263, -0.11516343802213669, -2.1463098526000977, 1.552724003791809, 0.40055716037750244, -0.49238649010658264, 0.20636701583862305, -1.6110725402832031, 1.6659443378448486, 0.19110015034675598, -0.052433181554079056, -0.007781213615089655, 0.15386290848255157, -0.7547163367271423, 1.6354819536209106, -0.06567654758691788, -0.3610784113407135, -0.49826768040657043, 0.16343295574188232, -0.6577292680740356, -0.19707919657230377, -1.9408637285232544, -0.06280718743801117, -0.6550964117050171, 1.2965925931930542, 0.2934047281742096, -1.4543368816375732, 0.668190598487854, 1.0215225219726562, -0.5874514579772949, 0.3261151909828186, 0.015007497742772102, 1.4877780675888062, 0.788846492767334, -0.07596994936466217, -1.168887972831726], [-0.6502446532249451, 1.1840548515319824, 0.7842321991920471, -2.1922783851623535, -1.9957058429718018, 0.5814277529716492, -0.43778547644615173, -0.6248124241828918, 0.3516988158226013, -0.4291982352733612, -0.8997917175292969, 1.071770429611206, 1.7061861753463745, 0.31433698534965515, 0.034751418977975845, 1.2400672435760498, 1.7263201475143433, 0.0810059905052185, 0.6811189651489258, 0.0011371758300811052, 3.344388008117676, -1.3332659006118774, 0.031535662710666656, 0.15179793536663055, 0.005437117535620928, -0.12955552339553833, -1.7614359855651855, -0.5818546414375305, -0.3004626929759979, 1.64681875705719, -1.5276379585266113, -0.4239612817764282, 2.5664923191070557, 0.42612388730049133, 0.22923396527767181, 1.8603442907333374, 0.21620315313339233, 0.06621083617210388, 0.6071948409080505, -1.5294071435928345, -0.011058350093662739, -0.9015923738479614, -1.248058795928955, -0.6142850518226624, -0.3311147093772888, -1.016517162322998, 0.2871362268924713, -1.296924352645874, 2.3812410831451416, 0.4074687957763672, 1.1020818948745728, -0.12933598458766937, -0.5079724788665771, -0.3449902832508087, -0.4045672118663788, -0.28741195797920227, -1.0007603168487549, 0.25883403420448303, 1.4148401021957397, 0.4808637499809265, 0.2479233592748642, 0.4350014626979828, -1.7150620222091675, 1.5571359395980835, 1.3179504871368408, 0.34214940667152405, -1.0506510734558105, 2.300102472305298, 0.26571646332740784, -1.6198740005493164, 1.2816413640975952, -1.0848976373672485, -0.6614949107170105, 1.3528193235397339, 0.6658744215965271, 1.1343703269958496, 0.06273481994867325, -0.6700582504272461, 1.936523675918579, -0.019374577328562737, -0.16685548424720764, 0.5134979486465454, -0.30303558707237244, 0.42717206478118896, -0.9054185748100281, -0.24917329847812653, -1.1516404151916504, -1.380237102508545, 1.0663114786148071, 1.3079726696014404, -1.0684562921524048, -0.5496293902397156, 0.3630337417125702, 0.853280246257782, 0.5380480289459229, 0.8273037075996399], [-0.525283694267273, -1.7699366807937622, 1.624145746231079, 0.1813424676656723, 3.046755313873291, 0.5796144604682922, 0.41450151801109314, -0.553464412689209, -1.7484891414642334, 0.17373061180114746, -1.1558681726455688, -2.0499043464660645, -0.7482508420944214, 0.8413780927658081, -0.45720115303993225, 0.7028550505638123, -1.6765708923339844, 1.0677447319030762, 0.7062992453575134, -1.0135787725448608, -0.8064015507698059, 0.017229586839675903, 0.3688495457172394, 0.1742350161075592, -0.028306743130087852, -0.13243047893047333, 0.5343894362449646, 0.5309028625488281, 0.3799341917037964, -1.2253541946411133, -0.23956215381622314, -0.5478346943855286, 0.794349193572998, -0.6767521500587463, -0.5989671945571899, -0.5634809732437134, -0.4106055498123169, -1.7575582265853882, 0.004828039091080427, -0.7015771269798279, 0.1571141481399536, 0.9323076605796814, 1.0924698114395142, -1.2178367376327515, -0.1880410760641098, -1.0052355527877808, 1.1908066272735596, -0.7929452657699585, -0.6814943552017212, 0.10062377899885178, 1.0006498098373413, -0.545518159866333, -1.7627440690994263, -0.24540852010250092, -0.662222146987915, -1.130203366279602, -0.3943079710006714, -0.13446566462516785, -0.4559563100337982, -1.0863538980484009, -0.9784676432609558, -0.7018921971321106, -0.9066882133483887, -0.8903847932815552, 2.4535906314849854, -0.3453730642795563, -0.21886134147644043, 1.3404536247253418, -1.567399024963379, -0.6476058959960938, -2.3738791942596436, 1.5224395990371704, 0.9601825475692749, -0.6835514307022095, -1.5720824003219604, -1.042453646659851, 0.7008545398712158, -0.14991340041160583, -1.2133253812789917, -0.1883023977279663, 0.47110387682914734, -0.4625089168548584, 0.5738282799720764, -1.6848902702331543, -0.055467721074819565, 0.9301708340644836, 0.8795902132987976, 0.4483359158039093, -0.33449211716651917, -1.5253719091415405, 2.1827220916748047, -1.0960532426834106, -0.3564727008342743, 2.093109607696533, -0.1444970816373825, -1.3983060121536255], [0.024050015956163406, -0.07426247000694275, -0.1144498661160469, -1.2108012437820435, 1.5561842918395996, 0.1559082716703415, 0.07479938864707947, -1.22428297996521, 1.0254606008529663, 2.3202786445617676, -0.3033042848110199, -1.0776646137237549, 0.35173559188842773, -2.021010160446167, 0.2591525912284851, -1.4896081686019897, -0.12004587799310684, 1.933388590812683, -0.6535043716430664, -2.1790053844451904, -0.5469116568565369, 0.7717640995979309, -1.2765153646469116, 0.6561295390129089, -0.22698941826820374, 0.039231158792972565, -1.3527864217758179, -0.36793065071105957, -0.6834731698036194, -0.01069632638245821, -0.8546491265296936, -1.3505593538284302, -2.1301329135894775, -0.31335821747779846, -0.526177704334259, 0.5824574828147888, -0.37669074535369873, -1.2081959247589111, 0.5385795831680298, -1.7501685619354248, -1.20253324508667, -0.19278135895729065, -0.43067988753318787, -0.6090670824050903, -0.11799473315477371, 1.0391197204589844, 1.3575583696365356, -1.0063668489456177, -1.1320756673812866, 1.3884273767471313, -0.4066166877746582, 0.27829092741012573, -1.147624135017395, 0.6435712575912476, 1.6621463298797607, 0.8378666639328003, -0.9726304411888123, -0.011624407023191452, 0.46135255694389343, -0.7714359164237976, 0.1471574902534485, -1.7545037269592285, -1.0076630115509033, 1.0221365690231323, -0.5755795240402222, 0.45351988077163696, -0.07058651745319366, 0.5801573395729065, 0.4405611753463745, 0.3441402316093445, 0.22851000726222992, -0.3207760751247406, -0.992706298828125, -0.3704187273979187, 0.5139621496200562, -0.7615489959716797, -0.1208299919962883, 0.5086022019386292, -0.4070299565792084, 0.23149003088474274, -0.7281211614608765, -0.06739875674247742, 0.4206794798374176, -0.36677974462509155, 1.7184710502624512, 0.2753216028213501, -0.15969125926494598, 0.2870340347290039, 0.6516790986061096, -1.4571529626846313, 1.169602394104004, 0.16548509895801544, 1.4559463262557983, -0.0028213136829435825, -0.49178174138069153, 1.2055660486221313], [-0.20520715415477753, 0.12850138545036316, -0.7254571914672852, -1.064810872077942, -0.28542274236679077, -1.2894688844680786, -0.3382706642150879, -0.6067303419113159, -0.14077548682689667, -0.8311799764633179, 0.43464797735214233, -0.3539678454399109, 3.7365529537200928, 1.2157316207885742, 0.9461154937744141, 0.9932973980903625, 1.8681622743606567, 2.2451171875, 0.13728247582912445, -0.25935375690460205, 0.20394866168498993, 1.8638538122177124, -0.16371652483940125, 0.27633729577064514, -0.4995567500591278, -1.0520782470703125, -0.48760274052619934, 0.7204124331474304, -0.0699893906712532, -1.7267529964447021, 0.3298541307449341, -0.8987631797790527, -1.086361289024353, -0.308270126581192, -2.8965060710906982, 1.375853419303894, -0.6037130951881409, -1.1432232856750488, 0.24301715195178986, 1.6098188161849976, 0.16390977799892426, 0.8184814453125, -1.3736025094985962, -0.1896800696849823, -0.09796332567930222, -0.9444130063056946, -1.165950059890747, -0.5886985659599304, 0.17903362214565277, 2.3118293285369873, -0.23341305553913116, -0.23619376122951508, -1.468105435371399, 0.8212952017784119, -0.33283737301826477, 1.9637447595596313, -1.103440523147583, 1.0334333181381226, 0.5412747859954834, -0.2856195569038391, 0.9225008487701416, -1.4528729915618896, -0.32379889488220215, 0.5572443604469299, 0.6838892698287964, -1.4385675191879272, 2.687242269515991, -0.9632949829101562, 0.9682213068008423, -1.0175975561141968, 1.4071842432022095, 0.7952014803886414, -0.05950813367962837, -0.3531583249568939, -0.8467565774917603, -1.112862229347229, 1.1163535118103027, 0.9745959043502808, -0.4764440059661865, -2.0271120071411133, 0.5990593433380127, 0.6737715601921082, -0.20700226724147797, 0.7377405166625977, -0.058727771043777466, 0.24714075028896332, -0.35541823506355286, 0.8678978085517883, 1.551814079284668, -0.6724025011062622, -1.0713915824890137, 1.4681740999221802, 1.4797050952911377, 0.7655118703842163, 0.20890618860721588, -2.2720398902893066], [2.0305073261260986, -1.2027153968811035, 0.2405005693435669, 0.2000628560781479, -2.0200910568237305, 0.5939080119132996, -0.7518969774246216, -0.1845633089542389, -0.7199134230613708, -0.6830205321311951, 0.1469334214925766, -0.3747645914554596, -0.7185173034667969, 0.039685215801000595, -0.5876916646957397, 1.4181854724884033, -0.2028535157442093, 0.023662641644477844, -1.046823501586914, 1.6661335229873657, 0.28673648834228516, -0.40326988697052, 0.9509480595588684, 0.8940321207046509, 0.5980756878852844, 0.5126367211341858, 1.5047781467437744, -0.8666093349456787, -0.2994614541530609, 0.26563388109207153, -0.5243228077888489, 1.063114047050476, -0.7430933117866516, 1.765916109085083, -1.7189221382141113, -0.15883013606071472, -0.48319822549819946, -0.3016045391559601, -0.25223061442375183, -0.49157771468162537, -1.1259714365005493, -1.5392550230026245, -0.14377306401729584, -0.8182833194732666, 0.5948428511619568, 1.035590410232544, 0.7544728517532349, -1.0098013877868652, 0.029994428157806396, -2.169816493988037, -0.8953929543495178, -0.8249944448471069, 0.6042911410331726, 0.7055587768554688, -0.0409061498939991, -1.4412424564361572, 1.190119743347168, -2.3065853118896484, 1.3927873373031616, 1.3691476583480835, -0.4240562319755554, -1.108775019645691, -0.47786185145378113, -2.038276433944702, 0.1767989695072174, 0.48518943786621094, 0.668807327747345, -0.4069374203681946, 0.5704875588417053, 0.7279192209243774, -1.1173574924468994, 1.3009315729141235, 1.3785898685455322, -0.13541264832019806, -2.465502977371216, 0.9972277283668518, 0.12151435762643814, 0.7274248600006104, -0.6664616465568542, 0.29693371057510376, -0.37806716561317444, -1.5576362609863281, 0.40157270431518555, 0.8615834712982178, -1.2312263250350952, 2.42618989944458, 0.7705395817756653, 1.2575478553771973, -1.587593674659729, -1.4083075523376465, -0.06424251198768616, -0.3795948028564453, -1.754629373550415, -0.33867624402046204, -0.19072137773036957, 0.548020601272583], [-0.5875810384750366, 0.02746378444135189, 1.050307035446167, -0.07679152488708496, -0.9129804372787476, 0.9584649205207825, -0.15585821866989136, -0.09679211676120758, 1.4254542589187622, 0.0316004678606987, -0.37551766633987427, -0.03184415400028229, 0.39404061436653137, 0.9707001447677612, -0.8867321610450745, -2.203582286834717, 0.33376792073249817, -2.27298903465271, -0.5715442299842834, 0.39358100295066833, 1.0231274366378784, -0.26081469655036926, 0.46860411763191223, 0.07369931787252426, 0.38885626196861267, -0.7215287089347839, 0.577692985534668, -0.04572542384266853, -1.0716233253479004, 1.0251407623291016, -1.2538905143737793, 0.10678960382938385, -0.7154750227928162, 0.45096835494041443, 0.6070055365562439, 0.8951442837715149, -3.0059866905212402, 1.118963599205017, 0.17502470314502716, 0.4587636888027191, 0.9050478339195251, 0.8181913495063782, 0.9067307114601135, -0.19220934808254242, 1.316817283630371, -0.29953014850616455, 0.57820063829422, -1.034675121307373, 0.585539698600769, 0.2067999690771103, -1.2911285161972046, -0.37866953015327454, -0.5291301012039185, -0.5546607375144958, -1.1178758144378662, 1.924458384513855, 1.4934171438217163, -1.3802285194396973, 0.6666709184646606, -0.27376115322113037, -1.798061490058899, 0.4515683650970459, 1.5786356925964355, -0.25416114926338196, 0.2113751620054245, -0.4769642949104309, -0.071415476500988, 0.018445611000061035, -0.7896828651428223, 0.1303723156452179, -0.8383824825286865, 1.1136020421981812, 0.22666779160499573, 0.15028725564479828, -1.2228869199752808, -0.33228278160095215, -0.06735877692699432, -2.5992825031280518, -1.8401433229446411, 1.1259626150131226, 2.3409583568573, 0.39950937032699585, -0.3389744460582733, -1.4944816827774048, 0.4437982439994812, -0.2093687355518341, -2.0569159984588623, 0.2313779890537262, -1.3450218439102173, 0.7946959733963013, 0.233688086271286, -0.344667911529541, 0.6892362833023071, 0.48721352219581604, -0.005661176051944494, 1.4840826988220215], [0.272768497467041, -0.16012588143348694, -0.6348145604133606, 0.4405023157596588, -1.325445532798767, 0.3375944197177887, -0.6609233617782593, -0.7721601128578186, 0.13831818103790283, 1.1740853786468506, 1.7876594066619873, 2.357309341430664, -0.2540733814239502, 1.0459219217300415, -0.3328624367713928, -0.5443784594535828, 1.1949615478515625, 0.2697087228298187, 0.3565402030944824, -0.6806763410568237, 1.301712989807129, -0.6210419535636902, 0.042457424104213715, -0.34066617488861084, 0.12665419280529022, -1.0187618732452393, 1.2153831720352173, -0.1655302792787552, -0.8906795382499695, -1.0480815172195435, -2.624455213546753, -0.0936279296875, -0.5554296970367432, -0.8134810924530029, -0.9334595203399658, -0.6585236191749573, -2.5090816020965576, -0.04625615477561951, 0.39290058612823486, 0.2418978065252304, -1.3734997510910034, -0.4632795751094818, 0.7979562878608704, 0.5145686864852905, 1.8528645038604736, -0.055973030626773834, -0.40847721695899963, -0.8246744871139526, -0.1992262303829193, -2.0062060356140137, 0.06971905380487442, 0.34842416644096375, 0.4385640025138855, -0.6547000408172607, 0.5527017712593079, -0.23333419859409332, -2.015831708908081, 0.8311728835105896, -1.7128617763519287, 0.3168938159942627, -0.269853413105011, -1.6016438007354736, -0.5724551677703857, 0.22400963306427002, 1.1570823192596436, -0.7718369364738464, 0.15585055947303772, 1.3969889879226685, -0.8173588514328003, -0.3999935984611511, -1.0309330224990845, -0.981657087802887, -0.49410244822502136, 0.07942772656679153, 0.32312434911727905, -0.017330214381217957, -1.9460724592208862, -1.4377458095550537, 2.179955244064331, -1.0108877420425415, -0.9969037771224976, 0.3039849102497101, 0.6852980852127075, -0.342941552400589, 0.6172073483467102, -2.0148298740386963, 0.3218078017234802, 1.6122227907180786, 0.3358592391014099, -1.9253053665161133, 0.34769323468208313, 0.003918359521776438, -0.12894774973392487, 1.856748342514038, -1.8383821249008179, 0.5937923192977905], [-1.6788893938064575, 2.18109130859375, 1.7699685096740723, 1.1618784666061401, -0.8504844903945923, -0.28312644362449646, 0.8677034378051758, 0.9815848469734192, 1.0132761001586914, 1.4406393766403198, -1.7303169965744019, 1.020591378211975, -0.7980299592018127, 0.6926171183586121, -0.5333484411239624, -0.3444004952907562, -1.352749228477478, -0.4949721097946167, -1.9798740148544312, -0.6048232913017273, -1.073073387145996, -0.8795639872550964, 0.30430397391319275, 1.0596299171447754, -1.2683923244476318, 1.245994210243225, 1.7227234840393066, -1.3516380786895752, 0.1869441270828247, -0.41785916686058044, 1.087266206741333, -0.047337356954813004, -1.0193865299224854, 1.0994023084640503, -0.5394179224967957, -0.5037531852722168, 0.20834527909755707, 0.13064995408058167, -0.4699561595916748, 2.0027852058410645, 0.8589077591896057, -0.5309668183326721, -0.2646041512489319, -1.2126028537750244, -0.6282755732536316, -0.030415114015340805, 1.0719408988952637, -0.1956489235162735, -0.38984596729278564, -0.2204776406288147, 0.9763778448104858, -0.8529828190803528, 1.0751609802246094, -0.42420727014541626, 0.1651744544506073, 0.48264947533607483, -0.06710823625326157, 0.5079697370529175, 1.8900227546691895, -0.691509485244751, 1.8339314460754395, -1.4819403886795044, -0.7274031639099121, 0.4721861481666565, 0.6261669397354126, -1.826695442199707, 0.8979287147521973, 0.5749494433403015, 1.3069045543670654, -1.7893558740615845, -0.14207150042057037, -0.7761998176574707, -0.6644382476806641, 2.712294340133667, 0.3535405993461609, -0.5252808928489685, 1.0148801803588867, -0.3468506336212158, 1.0109859704971313, 1.3427125215530396, 0.4754645526409149, -1.9957003593444824, 1.0261061191558838, -1.0228599309921265, 1.6614863872528076, 0.3104143738746643, -1.1170804500579834, 1.5777140855789185, -0.01517704501748085, 1.7217168807983398, 0.15462613105773926, 0.2867431342601776, -0.07223450392484665, 1.280221700668335, -0.1966591328382492, -0.29577696323394775], [-0.7517362236976624, 0.5661882162094116, 0.6043401956558228, -0.009317469783127308, -0.6708378791809082, -2.0002760887145996, 0.026179654523730278, 0.966320812702179, 0.8855729699134827, -0.4442782998085022, -0.10245900601148605, -0.36523154377937317, 0.5373997092247009, -1.20217764377594, -1.2663779258728027, -0.5057381391525269, 0.7959527969360352, 0.07003208249807358, 1.2082902193069458, 0.36401253938674927, 1.4111645221710205, -0.3952288031578064, 1.1202036142349243, -0.2810516059398651, -1.6854645013809204, -0.533873975276947, -0.30895429849624634, -0.6466382145881653, -0.07815756648778915, 0.6864622235298157, 0.47576597332954407, 1.0314100980758667, 0.04632531479001045, -1.110945224761963, -0.6810529232025146, -1.485124945640564, -0.1088743731379509, -2.4302845001220703, -0.9249282479286194, 0.42798954248428345, 0.7537715435028076, 1.5733147859573364, -0.35580888390541077, -0.585422694683075, -0.8156148195266724, 0.11092951893806458, 0.36423587799072266, 1.1786720752716064, -1.3639788627624512, 0.9304026961326599, 1.5915857553482056, 0.6065329313278198, 0.5823027491569519, -1.1227478981018066, -1.666643738746643, 0.4550555944442749, 0.5779927968978882, 0.29503852128982544, 1.129899024963379, 0.06375010311603546, 0.4395408034324646, -1.8451433181762695, 1.0763291120529175, 0.44867947697639465, 1.1370370388031006, -1.2475614547729492, -0.688624382019043, 0.8494996428489685, 1.1701806783676147, 0.8218140006065369, 0.5047253966331482, 0.21454495191574097, 0.03694426640868187, 0.9952777028083801, -1.8725577592849731, 0.7414180636405945, 0.6937347650527954, -1.1759552955627441, -0.24828603863716125, -1.8944758176803589, -0.37614113092422485, -1.0689926147460938, -1.5753194093704224, 1.273140788078308, -0.6583883762359619, -0.8466948866844177, 0.4728873074054718, -0.7505086660385132, -0.2768947184085846, -1.0842307806015015, -1.2543423175811768, 0.9681221842765808, 0.42014816403388977, 0.10787276178598404, 0.9058806300163269, 0.5039677619934082], [1.4408588409423828, 0.013707969337701797, -0.6324089765548706, 0.9459042549133301, 1.395884394645691, 0.3992070257663727, -0.985650897026062, -2.1609725952148438, -1.3614344596862793, -0.9323062300682068, -0.4232613146305084, 0.3491286039352417, 1.2904038429260254, -1.6677321195602417, 0.5341277718544006, 0.8752820491790771, 0.7371194958686829, -0.5994114279747009, -0.5559695959091187, 1.1604517698287964, -3.852165460586548, 1.3093336820602417, 0.29037967324256897, -1.0597506761550903, -1.7991392612457275, -0.4783659279346466, -0.3370274603366852, 0.5182023644447327, 1.3727651834487915, -0.3392285406589508, -0.17671997845172882, 0.5287436842918396, 0.2528584897518158, 0.29694125056266785, 0.24357782304286957, -0.3000488877296448, 0.5040662884712219, -0.9376864433288574, 1.8292279243469238, -0.17040951550006866, 1.653876781463623, -0.9429283738136292, 0.6515690088272095, -1.1887876987457275, 0.1735764890909195, 0.5660556554794312, -0.10186631977558136, -1.0872204303741455, 0.35246357321739197, -1.31636643409729, 0.6815548539161682, -0.37226173281669617, 0.26908543705940247, -1.0928266048431396, -0.9216421246528625, 0.3235531747341156, -0.0995873212814331, -1.9049451351165771, 2.125230312347412, -2.2919535636901855, 0.7555210590362549, 0.13403533399105072, -0.8860841393470764, 0.45170342922210693, 1.2238543033599854, 0.7742140293121338, -0.4754756987094879, -1.5112160444259644, -0.010829566046595573, -2.456770420074463, 1.5067026615142822, -0.445852130651474, 0.43434804677963257, 0.8050114512443542, -1.8417140245437622, -0.9680891633033752, 0.22601155936717987, 0.42046889662742615, 0.12707507610321045, -1.728732943534851, 0.2547750174999237, 0.9706703424453735, -0.8063328266143799, -0.40588390827178955, -0.2173093855381012, -0.9521093368530273, 2.761732339859009, 0.6426706910133362, 0.8597468137741089, -0.9690319299697876, -1.2927618026733398, 0.4828975200653076, 0.5975548028945923, -0.43114736676216125, 0.5635680556297302, -0.458122581243515], [1.0139974355697632, -0.4966607093811035, -1.1107200384140015, 0.08650165796279907, -1.32437264919281, 0.4861702024936676, 0.6825398206710815, -0.5045374631881714, 2.6863551139831543, -0.3961365818977356, -0.40115928649902344, 0.19902515411376953, -1.0830756425857544, -0.6221067905426025, -0.3435022830963135, 0.7430158257484436, 2.200406312942505, -2.95154070854187, 1.4495793581008911, -0.39129894971847534, 0.4397670328617096, -0.9592656493186951, 0.8998770713806152, 0.859887957572937, 2.2495779991149902, 0.6174808740615845, -0.40784671902656555, -0.4957192838191986, -3.4715161323547363, -2.5907235145568848, 0.937604546546936, 1.4286670684814453, 0.0009908821666613221, -1.2931498289108276, -0.08814699202775955, 0.7992043495178223, 1.484811782836914, -0.7685871720314026, 0.28280892968177795, -1.0126272439956665, -1.464430570602417, -0.8121821284294128, -0.03718011453747749, 0.3593577444553375, -0.4054108262062073, 0.22251710295677185, 0.26203468441963196, 0.001970029668882489, -1.1788541078567505, 0.7318255305290222, 0.856705367565155, 0.4074380695819855, 0.296223521232605, -0.7322273254394531, -0.3025667369365692, 1.2357423305511475, -1.154700517654419, 0.9814739227294922, 2.5413103103637695, -1.2359219789505005, 0.38869795203208923, -1.622696042060852, 0.8523505926132202, -0.1095070019364357, -0.0927577093243599, -0.6556758284568787, -0.05396425724029541, 1.4777947664260864, -0.8745120167732239, 0.8328232765197754, -1.1805419921875, -0.17480000853538513, -0.5500271320343018, 2.552555561065674, 0.0697975605726242, -1.454836368560791, 0.05165252834558487, -0.7944586277008057, 1.120506763458252, -1.9049474000930786, -0.1925811618566513, 0.9170861840248108, -1.0458675622940063, -1.0233147144317627, 1.3199056386947632, 0.19758287072181702, -0.6528744101524353, 0.24552659690380096, -0.535018265247345, 0.4512055814266205, -1.5123308897018433, -0.5321947336196899, -1.3416694402694702, -0.7012051343917847, 2.5942392349243164, -1.7452130317687988], [-1.4883275032043457, -0.7899966239929199, -1.3510252237319946, -1.307806134223938, 0.6534383296966553, 1.1191167831420898, 0.5105189085006714, 1.2082023620605469, -3.593231201171875, 0.043784432113170624, -0.7302325367927551, 0.42270562052726746, 0.03294806554913521, -0.3631855845451355, 0.16129247844219208, 0.015703294426202774, 1.5872166156768799, -1.3909494876861572, 2.2693138122558594, -0.2708352506160736, -0.2146226167678833, 0.14034157991409302, -0.38632750511169434, 0.3575234115123749, -0.0814099833369255, 0.41292834281921387, -1.717434287071228, -0.9078443050384521, -0.8624849319458008, -1.0391765832901, 0.5023599863052368, 0.766029953956604, -1.2133806943893433, 0.6713598966598511, -1.2586373090744019, -1.9028033018112183, -1.007853388786316, 1.243224859237671, 0.2805533707141876, 0.5916670560836792, 0.10244731605052948, 0.4051184356212616, 0.07962394505739212, -1.3571388721466064, 0.10906961560249329, -1.1434558629989624, 0.03537662327289581, 0.5436141490936279, -1.297612190246582, 1.505203366279602, -0.21498987078666687, 1.5337871313095093, -0.47894686460494995, 0.24056243896484375, -0.2057839184999466, 0.6749655604362488, 1.0473824739456177, 0.7499487400054932, -0.5096343755722046, 1.608832597732544, 1.5401197671890259, -0.5491004586219788, 0.7778387069702148, 0.7469237446784973, -0.6298142075538635, 0.8087716102600098, -1.3297430276870728, 1.1867575645446777, 0.45768484473228455, 1.1262437105178833, 0.12763474881649017, -0.017926551401615143, -0.3530576825141907, 1.4212040901184082, 0.9942249655723572, 0.6118272542953491, -0.21171268820762634, 2.2210452556610107, 0.16899892687797546, -1.6011916399002075, 1.4953556060791016, 0.7553986310958862, -0.41110363602638245, -2.0462911128997803, 0.3894886076450348, 0.16611385345458984, -0.13779784739017487, 1.1228810548782349, -1.5705713033676147, -0.6220701932907104, -1.23415207862854, -1.291395902633667, -0.6318552494049072, -2.0285675525665283, -0.9112683534622192, 0.7239915728569031], [-0.0852171778678894, 0.4336502254009247, -0.15598247945308685, -0.07932554185390472, -0.6699812412261963, -0.40100598335266113, 0.16931389272212982, 1.0857677459716797, 0.02301022782921791, 0.07995103299617767, 0.14352214336395264, -0.5273150205612183, 0.4071812033653259, 1.2149837017059326, -0.7750453352928162, -0.7644844055175781, 0.9737403988838196, -0.9493513703346252, -0.9060689806938171, 1.0783030986785889, -0.8110003471374512, -0.705752432346344, -1.3064159154891968, -1.1783757209777832, 1.9244091510772705, 2.547081232070923, -0.5272589325904846, 0.41792312264442444, 1.4297020435333252, 0.12222035974264145, 0.4840848445892334, -0.747627854347229, -0.7296427488327026, -0.14617189764976501, -1.222357153892517, -0.47637543082237244, -1.987004280090332, 0.9591853618621826, 1.3860478401184082, -0.7952947020530701, -0.5587637424468994, -0.5162286758422852, 1.4274758100509644, 2.107025146484375, -0.3872077465057373, -1.1040316820144653, 0.10271059721708298, 0.8737886548042297, 1.4845690727233887, 0.4299362003803253, -0.7883785367012024, 0.5007784962654114, 0.5770206451416016, 1.3936458826065063, -0.6977832317352295, 0.0265607051551342, -0.5841132998466492, 0.5223527550697327, -0.9362531304359436, 0.4478360414505005, 0.7252258062362671, -1.4669462442398071, -0.8279346823692322, 0.2522854208946228, -0.898431658744812, 0.42326590418815613, 0.21465036273002625, 0.5063678026199341, 0.7057658433914185, -1.0203328132629395, -0.4804917871952057, -2.2685084342956543, 0.030024411156773567, -1.6848641633987427, 1.5774226188659668, -0.1593824177980423, 2.0216877460479736, -0.3776370882987976, -0.30200478434562683, -0.3120516836643219, 0.03043493814766407, 0.6135185956954956, 0.19806717336177826, -2.2764439582824707, -0.32032862305641174, -0.043870244175195694, -0.03735256940126419, 0.7728177905082703, -0.9668079614639282, 0.0012415562523528934, 0.4089376628398895, 0.4710054397583008, -0.031530581414699554, 0.8570906519889832, 0.05495024099946022, 0.29885333776474], [-0.6115185618400574, -0.09780479967594147, -1.014026165008545, -0.41996362805366516, -1.2665055990219116, -1.2986027002334595, 0.9324052929878235, 0.7175065279006958, 0.2144274264574051, 0.6426364779472351, 0.13218723237514496, -0.29449015855789185, 1.4104893207550049, -1.1794698238372803, -0.14191962778568268, -0.1810327023267746, -0.043297991156578064, -0.32860589027404785, 0.40444833040237427, 1.7863118648529053, -0.9585207104682922, 1.0313242673873901, -0.1157739907503128, 0.6009297966957092, -0.1627255082130432, 1.2224969863891602, -0.9568083882331848, 2.3674991130828857, -0.6063680052757263, -0.2789236605167389, -0.5561293363571167, 0.5756704211235046, -3.2069828510284424, 0.30262109637260437, -0.5918357372283936, 1.3952909708023071, -0.6696699261665344, 1.4124274253845215, 1.1638720035552979, 1.8625670671463013, -3.085963010787964, 0.6771887540817261, -0.42593252658843994, -0.34061458706855774, -0.12647853791713715, 0.7732170224189758, -0.20819033682346344, 1.49164879322052, 2.5338592529296875, 1.2048717737197876, 0.20973928272724152, 0.8432213068008423, 1.0989936590194702, -0.0963946282863617, 0.38830286264419556, 0.6554950475692749, 0.02727774903178215, 0.8499489426612854, -1.6584618091583252, -1.4275116920471191, -0.8758272528648376, -3.861528158187866, 0.6214864253997803, -0.5151300430297852, 0.6534560322761536, -0.6389850378036499, 0.5287356972694397, -0.8509553074836731, -1.0773664712905884, 0.6545868515968323, -0.5417629480361938, 0.8711471557617188, -0.3155210018157959, 0.37225791811943054, 0.4786057472229004, -0.45227929949760437, -0.6878065466880798, -1.0537070035934448, -0.28734496235847473, -1.0825185775756836, -0.13261768221855164, -0.799029529094696, 0.3476433753967285, -0.6466544270515442, -0.9858987927436829, 0.5468904972076416, 1.3043254613876343, 0.0062436675652861595, 0.1637030392885208, 0.8277908563613892, 1.1951402425765991, -0.6131725311279297, -2.0227510929107666, -0.06204375624656677, 1.3807554244995117, -1.252343773841858], [2.2228455543518066, 2.273552179336548, -1.485344409942627, 1.1522469520568848, -2.2119956016540527, 0.6275163888931274, 0.6195469498634338, 0.6837811470031738, 1.2472679615020752, -0.17574253678321838, 1.3128334283828735, -0.9412449598312378, -0.7163158655166626, -0.49799710512161255, -0.0156287532299757, 1.4153916835784912, -0.03028310090303421, -0.6953220963478088, -0.38512009382247925, -0.679520845413208, -0.5235272645950317, 0.33952584862709045, -0.12060526013374329, 0.7741318345069885, 1.1227566003799438, -0.07544009387493134, 0.7368286848068237, -0.04652564972639084, -0.9751104712486267, -1.1630452871322632, 1.1543407440185547, 0.1180194541811943, 0.5627546310424805, 0.5370272994041443, -0.19775395095348358, 1.4393131732940674, 1.376475214958191, 0.2972905933856964, 0.12489701807498932, -0.28188079595565796, -0.7735673785209656, -1.0343670845031738, 0.036076582968235016, -1.747153401374817, 0.03698471188545227, -0.14466746151447296, -0.920242965221405, 0.6402252316474915, 0.8059341907501221, 0.23519587516784668, 0.8183708190917969, -0.07189983129501343, -0.9231435656547546, 0.8890631198883057, -1.1426451206207275, 1.5816192626953125, 0.8203930854797363, 1.4381998777389526, 1.2364603281021118, -0.3074013590812683, 1.1877943277359009, 0.9809601902961731, -1.5408499240875244, -0.639244794845581, 1.3752998113632202, -0.6477894186973572, -1.7849172353744507, 0.09426312893629074, -0.3014419972896576, 1.870874285697937, 0.3021349608898163, -1.1383541822433472, 0.19344209134578705, 1.215081810951233, 0.1032227873802185, 1.3569469451904297, -1.4538414478302002, 0.4371402859687805, 0.33793190121650696, 0.5737248063087463, -0.18572615087032318, -0.09815976768732071, -0.3793196976184845, -1.5732396841049194, -0.05663437396287918, 0.1658540517091751, 1.0645209550857544, 0.47955116629600525, -0.03649505227804184, 0.5192960500717163, -0.12194181233644485, 0.489276260137558, 0.709184467792511, 1.1387637853622437, 0.08939504623413086, -0.1094549149274826], [-0.9098944664001465, 1.7517004013061523, 1.256609559059143, -0.5294193625450134, 0.042129501700401306, 0.2919818162918091, -0.4195388853549957, -0.1811223030090332, -0.07920384407043457, -0.9996359348297119, 0.7971836924552917, -2.075632095336914, -0.2194206416606903, 0.45403704047203064, 0.29554852843284607, -0.4131593108177185, -0.2655014991760254, -0.08448304980993271, -0.14776545763015747, 1.5615253448486328, 0.15088872611522675, 1.8607981204986572, 0.2940182089805603, 1.3974615335464478, -0.17616978287696838, 0.5638839602470398, 0.14238667488098145, -1.023450493812561, -0.9095591306686401, 0.5865501165390015, -0.9108880162239075, -1.800642728805542, 1.6283010244369507, -0.6919599175453186, 2.3160643577575684, 0.18728840351104736, 0.08085546642541885, 1.7866138219833374, 1.1308014392852783, 0.5426053404808044, 0.2861367166042328, 0.43517017364501953, 2.1466963291168213, 0.5041483640670776, 0.5160337090492249, 1.4131507873535156, -0.4259520471096039, -0.8461704254150391, 0.4418134093284607, -0.4196498990058899, 1.1584481000900269, 0.015664823353290558, -1.7099733352661133, 0.25845685601234436, -0.8187965154647827, 0.8600371479988098, 0.0717795118689537, 1.5834866762161255, -0.3716849088668823, 0.47662878036499023, -0.871040940284729, 0.4927894175052643, 1.6262503862380981, -0.007911463268101215, -0.6501232385635376, -0.0936901643872261, 1.1484854221343994, -1.3685110807418823, 0.26807907223701477, 0.46174299716949463, 0.0830775797367096, 1.4589520692825317, -0.11645497381687164, 0.33213019371032715, 0.3003397285938263, -0.5123298764228821, 0.24896694719791412, -0.23852886259555817, -0.22484846413135529, 0.05754859372973442, -0.9367002248764038, 0.6619271039962769, 0.8107920289039612, -0.23397304117679596, 0.886955738067627, -2.3252267837524414, -0.3569696247577667, 1.2198233604431152, 0.6210957169532776, 0.24114525318145752, -0.02735585905611515, 0.31329289078712463, 0.972241997718811, 2.3968796730041504, -1.3396834135055542, -0.7489110231399536], [0.3713013231754303, -0.5900385975837708, 1.0924243927001953, 1.1671051979064941, 0.9894277453422546, 0.4379468262195587, 1.0725069046020508, 1.2735958099365234, 0.1724964678287506, 0.14328932762145996, -0.6244006156921387, -1.5364868640899658, -0.2557394504547119, 1.0492868423461914, 2.4906673431396484, -0.728358268737793, -0.9757213592529297, 0.7367355823516846, 1.1029763221740723, 0.7581115365028381, 1.4191936254501343, -0.23130813241004944, -0.10406361520290375, 1.240319848060608, 0.7151231169700623, 0.9024981260299683, 2.4264743328094482, 0.5760591626167297, 0.9827942252159119, -0.3535975217819214, -0.12807993590831757, -0.186215341091156, -0.42308661341667175, -0.5955061912536621, 0.05380682647228241, -1.452799677848816, -0.3199688196182251, -0.35189542174339294, -0.3627212345600128, -0.2719605565071106, -1.5358564853668213, -0.07827621698379517, 1.5593914985656738, 0.48936721682548523, 0.290479451417923, 0.3536887466907501, 0.5473164916038513, -0.3865486681461334, -0.6328052282333374, 0.8377465605735779, -0.06961629539728165, -1.291856050491333, -0.22728466987609863, 1.234125018119812, -0.23532308638095856, 0.975365161895752, 0.9564276933670044, 0.34627771377563477, -2.2146313190460205, -0.03353509679436684, -0.40943875908851624, 0.7391632199287415, -0.7188723087310791, -0.3766115605831146, -0.39523112773895264, -0.2479410320520401, 0.822365939617157, -0.3092300295829773, -0.1340097337961197, -0.8737068772315979, 0.08002358675003052, -0.11881175637245178, -0.6192734241485596, 1.9134643077850342, 0.32769864797592163, -0.2850843369960785, -0.5637667179107666, -0.927241325378418, -0.287007600069046, 1.5293529033660889, 0.0037260903045535088, -1.5013322830200195, 0.6037412881851196, -0.22050189971923828, 1.261490821838379, 1.5799901485443115, 1.5769654512405396, -0.7398552298545837, -0.5939614176750183, -0.7272220849990845, 0.9186690449714661, 1.1261898279190063, -0.6469132900238037, -0.13113559782505035, 0.71834397315979, -0.5323953628540039], [1.1783013343811035, -0.7157368063926697, 1.4613398313522339, -1.7235227823257446, -0.41076454520225525, -0.9286868572235107, -0.08376222103834152, 1.2266608476638794, 0.389724463224411, 0.06311219930648804, 0.1771399974822998, 0.6665477156639099, 0.502915620803833, 0.408482164144516, -0.17121174931526184, 0.6437785029411316, 0.3304116129875183, -0.4500385820865631, 0.011109787039458752, 0.4708574116230011, -1.052047610282898, -0.45511651039123535, 0.11259864270687103, -0.9760291576385498, 0.7852494716644287, -1.2712072134017944, -0.22346746921539307, 0.8611302971839905, 0.38284316658973694, 1.4768925905227661, -0.17922888696193695, -0.70545893907547, -0.17678192257881165, 0.36692407727241516, 2.311479330062866, 0.07552164793014526, -0.23154762387275696, 0.4181171953678131, -0.8387276530265808, -0.9894188642501831, 0.41336238384246826, -1.412053108215332, 2.090955972671509, -1.566011905670166, 0.5779523253440857, 1.0677614212036133, -0.9286702275276184, 0.3463270366191864, -0.09445691108703613, -0.5405825972557068, 0.335199236869812, -0.49747124314308167, -1.2568124532699585, 0.051758140325546265, 0.32888105511665344, -0.23408645391464233, -0.37015730142593384, 0.39850470423698425, 0.07856196165084839, -0.6666789054870605, -0.08690184354782104, 0.32905739545822144, -0.8525896668434143, 1.3921210765838623, -2.073239803314209, -1.3870760202407837, 0.5775946974754333, 0.01960648223757744, -1.693972110748291, -1.2425490617752075, -0.6217936277389526, 0.8794918656349182, -0.1127345934510231, -1.5275307893753052, -0.9724931120872498, -1.3822314739227295, -1.4136182069778442, 1.0018227100372314, -0.10619427263736725, -0.9769010543823242, -0.985069990158081, -1.075402855873108, 0.3761135935783386, -0.0814087763428688, 1.146906852722168, -0.3153947591781616, 0.20130670070648193, 0.32868021726608276, 0.5697015523910522, -0.4716803729534149, 0.33434879779815674, 0.06871239840984344, -1.0319424867630005, -1.082664132118225, -0.529662013053894, -1.471641182899475], [-0.2035883665084839, -1.2170473337173462, 2.5340678691864014, 0.20585498213768005, 0.4012100398540497, -1.2831186056137085, -0.5520204305648804, 0.046599481254816055, 0.4908493161201477, 1.2990407943725586, 1.7581371068954468, 0.8031694293022156, 0.38816937804222107, 1.12028169631958, 0.9253832697868347, 0.2676035463809967, 0.24376265704631805, 0.1474730223417282, 1.6243202686309814, -0.2563697397708893, 0.023662561550736427, -2.46006178855896, -0.16513486206531525, -0.8699630498886108, 0.23192310333251953, 0.8246321082115173, -2.1015758514404297, -0.42451155185699463, 0.7245345115661621, -1.6757768392562866, -0.8847615122795105, -0.9334514737129211, 0.9422733783721924, -0.6798949241638184, -0.4004492461681366, -0.56546550989151, -1.2128242254257202, 1.2169032096862793, 1.8478540182113647, -0.7246072292327881, 2.6204960346221924, -0.4142071604728699, -0.8010064363479614, -0.9921383857727051, 0.23949263989925385, -0.429319828748703, 0.04442442208528519, 0.8016964793205261, -0.09276513010263443, 0.567646324634552, -0.16016942262649536, 1.1765556335449219, -0.09734000265598297, 1.3681102991104126, -0.16900566220283508, 1.1298831701278687, 0.45471957325935364, 0.5466572046279907, -0.18127040565013885, 0.39725232124328613, 0.48690181970596313, 2.1273787021636963, -1.4585137367248535, -0.6210675239562988, 0.8455579280853271, -1.0455950498580933, -0.2825852632522583, 0.5204891562461853, 0.5119758248329163, -2.4153976440429688, 1.3199357986450195, 2.0501821041107178, -1.336485743522644, 1.0445612668991089, 0.7308616638183594, -1.0568093061447144, 0.3113090693950653, 1.7831214666366577, -0.08361973613500595, -0.38414108753204346, 0.3681349456310272, 0.8133945465087891, 0.14369173347949982, -0.731062114238739, 1.3286839723587036, 0.2941541075706482, -0.8075625896453857, -0.9184131622314453, 0.4670698642730713, 2.1980109214782715, 1.941051721572876, -0.6123737096786499, 0.5900260806083679, 2.684424877166748, -1.5450189113616943, -0.2800537049770355], [0.5309045910835266, -0.287803590297699, 0.5087596774101257, 0.8249565958976746, -0.1612735241651535, 0.5139696002006531, -0.07452704757452011, 0.015393373556435108, -0.33170565962791443, -2.253751754760742, -0.5030736327171326, 1.8258527517318726, 0.7890871167182922, -0.08221794664859772, -0.23714008927345276, 0.5818670988082886, -0.3885795772075653, -0.8864502310752869, -0.4066607654094696, -0.14601856470108032, 0.2894427180290222, -0.24933698773384094, -0.1695265918970108, -0.6202830076217651, -1.021627426147461, 0.3849540650844574, -2.369291305541992, -0.7806940078735352, -0.2908630073070526, 0.8642311692237854, -0.2260676771402359, -0.8041931986808777, -0.37940365076065063, 1.2356343269348145, -0.8597760796546936, 0.9251825213432312, 0.19381621479988098, 1.6601425409317017, -1.057517647743225, 0.1510278880596161, 0.35400086641311646, 1.7294962406158447, 0.5111709833145142, -1.2578387260437012, 0.7429134845733643, -1.3661662340164185, 1.9299137592315674, 2.1436996459960938, 0.3427228331565857, 0.3802759051322937, -0.4798987805843353, 0.1806703805923462, 0.21409453451633453, 0.6207169890403748, -0.9888989925384521, -0.07231676578521729, -0.8319811224937439, -0.8048908710479736, -0.1984459012746811, 0.03340764716267586, -0.4659271538257599, -1.635723352432251, -0.04962639883160591, 1.8731353282928467, -0.3292291462421417, 1.0790352821350098, -0.0573759526014328, 0.3559338450431824, -0.08669356256723404, 0.16839846968650818, 0.6721559166908264, -0.5985718965530396, -0.5317187905311584, -2.0430350303649902, -1.0053895711898804, 0.696922779083252, -0.7350214123725891, 0.48989439010620117, 0.8284916281700134, 0.8882510662078857, -0.6975094079971313, 1.0988045930862427, -0.06658901274204254, -0.26199132204055786, -1.1985422372817993, 1.191752552986145, -0.03285451605916023, 0.3957318365573883, -2.50736665725708, 1.5132358074188232, -1.5934025049209595, -1.6537333726882935, -1.3288229703903198, -0.2157277911901474, 0.14644481241703033, -1.1380233764648438], [-0.18326126039028168, -0.36514467000961304, -0.09070077538490295, -0.5703496336936951, -0.4056451916694641, 1.2168443202972412, 0.8712696433067322, -1.778927206993103, 1.5358482599258423, -0.28607064485549927, 0.872226893901825, -0.4115388095378876, 0.6795371770858765, 0.47221505641937256, -0.6379454135894775, 0.2363225370645523, 0.018584301695227623, -0.5428816080093384, -1.0850468873977661, 0.9579108357429504, 0.4703206717967987, -1.0008087158203125, -0.013752459548413754, -1.1430068016052246, -0.7275763154029846, 1.540229082107544, 1.1410995721817017, 0.11099319159984589, 1.6849050521850586, 0.8828701972961426, -1.5481233596801758, 0.33093154430389404, 0.22196970880031586, 0.6427333950996399, 0.3026657998561859, -1.244848370552063, 0.4924420118331909, 0.977157711982727, 0.025496458634734154, -0.13312391936779022, 1.9180302619934082, -2.6898515224456787, -0.5238210558891296, 0.6439749002456665, 1.6743932962417603, -1.7847659587860107, 0.2658836841583252, 1.2802854776382446, -0.48952245712280273, -1.8746259212493896, 1.1393073797225952, -0.3857823610305786, 1.2079803943634033, 1.0296354293823242, 1.0314991474151611, 0.2996855676174164, 1.198563575744629, 1.6802889108657837, -1.4301447868347168, 1.2327758073806763, -0.6056979894638062, -0.785290539264679, -0.3923731744289398, -2.127715587615967, -0.246408149600029, 0.013118655420839787, -0.3024488091468811, -1.1272404193878174, 0.4268093407154083, -0.02915291301906109, -2.7470414638519287, 0.5655549764633179, -0.013078478164970875, -0.2222258597612381, 0.6974546313285828, 1.1710413694381714, -2.3086984157562256, -0.07468753308057785, 1.6273553371429443, -0.901035487651825, -0.13815870881080627, -0.02482195384800434, -0.6808879971504211, 0.9238810539245605, 0.26202264428138733, 0.6137514114379883, 0.6401759386062622, -0.5999636054039001, -0.010856172069907188, 0.2563398778438568, 0.18207165598869324, 1.8272300958633423, -0.12568214535713196, 2.6713736057281494, 1.027462363243103, 1.6200716495513916], [-0.4281514585018158, -0.22345873713493347, -1.1083849668502808, 0.8192078471183777, -0.26875099539756775, 0.4146176278591156, 0.4246055781841278, 1.451230764389038, -0.1779828667640686, 1.0104122161865234, -0.11928772926330566, -0.45852911472320557, 1.6029685735702515, -0.032214704900979996, 0.21687771379947662, 1.1918309926986694, -0.7843013405799866, -1.0141806602478027, 0.21641318500041962, 0.5373764038085938, 0.5139298439025879, 2.3947012424468994, 0.8485471606254578, 1.0249526500701904, -1.2757962942123413, -0.6529989838600159, -1.110474944114685, 1.0904349088668823, -1.125797152519226, -0.5717002153396606, -1.3523995876312256, 1.4895271062850952, -0.5428513884544373, 0.8224903345108032, 1.5795625448226929, 0.48940950632095337, -0.7568188309669495, -1.0209496021270752, 0.029585029929876328, -0.413979709148407, 0.24831941723823547, 0.5827746391296387, -0.294399231672287, -0.6007898449897766, -0.7401251792907715, -1.755629062652588, 0.5031570792198181, -1.2903459072113037, 0.11658673733472824, 0.032174695283174515, -0.32422924041748047, 0.8692352175712585, 0.2591751217842102, 1.8037229776382446, 0.4057011604309082, 1.0489753484725952, 1.0272091627120972, 1.3815054893493652, -0.429364949464798, 2.6313610076904297, -2.266775131225586, 0.545799732208252, 0.45161172747612, 0.5841052532196045, 2.1819908618927, -1.708406925201416, 0.7613571286201477, -0.8410559892654419, 1.5681875944137573, 0.8740506172180176, 1.1209259033203125, -0.534134566783905, -1.1794919967651367, 0.5287038087844849, -0.3707989454269409, -0.6911197304725647, 1.4464582204818726, -1.1199291944503784, -0.7573873996734619, -0.5065471529960632, 0.7462963461875916, -1.0918614864349365, 0.04770991578698158, 0.20966538786888123, 0.7215152382850647, -0.6848055124282837, -1.9325976371765137, -0.5897459387779236, -2.412213087081909, -0.341886967420578, -1.8336589336395264, -1.9492294788360596, -0.9016523957252502, 1.62649405002594, -1.879201054573059, -0.21493951976299286], [-0.4454900324344635, 1.4752871990203857, -0.4335225224494934, -0.2482975721359253, 0.2065650373697281, 1.4194836616516113, 0.15209156274795532, -1.6393558979034424, -0.17635098099708557, 0.9750550389289856, 1.2269902229309082, -0.7543753981590271, -0.1959836184978485, -0.18712998926639557, 0.03907274082303047, 0.9746937155723572, 0.17471511662006378, -0.346910685300827, -0.5975297689437866, 0.857040524482727, -0.6581949591636658, 1.1586425304412842, 0.24953308701515198, -1.7784240245819092, 0.32281166315078735, -1.974778652191162, 0.2121279388666153, -0.522205114364624, 0.563452422618866, -0.5358158349990845, 1.318186640739441, -0.3699449598789215, 1.5988128185272217, 0.2077072113752365, 0.19548740983009338, 0.15945695340633392, -0.5694707632064819, 1.0248478651046753, -2.522148370742798, -0.4532361626625061, -0.5236812829971313, 1.5229361057281494, -1.4245482683181763, 0.097941093146801, 1.1143397092819214, -0.8157839179039001, 1.5110790729522705, -0.11773961782455444, -0.6739119291305542, -0.8132210969924927, -1.1860966682434082, -0.15363147854804993, -0.3408355116844177, -0.05433463305234909, -1.463974118232727, 0.7922946810722351, 0.11091314256191254, 0.3421538174152374, -0.6279222965240479, 0.01541126612573862, -0.05580738186836243, 1.046299934387207, 0.9935141205787659, -0.10741913318634033, -0.6551379561424255, 1.8107448816299438, -0.5755385756492615, -0.9738866090774536, -0.4941185712814331, 1.7451536655426025, -0.5932671427726746, 0.948929488658905, -1.2454243898391724, 1.1736489534378052, 1.7043527364730835, -0.6079495549201965, -0.8313103914260864, -0.5643629431724548, 0.21282215416431427, -0.32670724391937256, 0.9178809523582458, 0.7009218335151672, 2.158442497253418, 0.9253513216972351, 1.2789552211761475, 0.9332144856452942, -1.8252723217010498, -0.44741135835647583, 0.7448534965515137, 1.91130793094635, 0.2600846290588379, -0.8286184072494507, 0.9799556732177734, -0.7287525534629822, 0.36664924025535583, -1.1295546293258667], [0.6522035598754883, -1.0761879682540894, 1.3712772130966187, -0.6643219590187073, 0.5387807488441467, -0.6528935432434082, 0.30651286244392395, -0.11676305532455444, 0.615185022354126, 1.081640362739563, -0.4277217388153076, 1.0038564205169678, -2.328233242034912, -0.12203668057918549, 0.40886110067367554, 0.7146164178848267, -1.4731543064117432, 0.8560152053833008, 1.123231291770935, -1.3815147876739502, -0.3021200895309448, 0.6578772664070129, -0.4827588200569153, 1.0642459392547607, 1.0078343152999878, -0.48650550842285156, -0.6462295651435852, 0.8186724781990051, -1.5178494453430176, -0.3575875461101532, 0.6502118110656738, -0.7815431356430054, -0.07314636558294296, 0.6321502327919006, -1.699236512184143, -0.6940560340881348, -0.5395640730857849, 0.31468719244003296, 1.435336709022522, -1.2701407670974731, -0.675449788570404, -0.3010879158973694, 0.8106183409690857, 0.4583727717399597, 1.0269927978515625, -1.0853252410888672, 0.24746796488761902, -0.3092373013496399, -0.036779992282390594, -1.3103994131088257, 1.2270950078964233, -0.9369661211967468, -0.16083768010139465, 1.148123860359192, 0.595869243144989, 0.14446164667606354, 1.1146793365478516, 0.12244226038455963, -0.2059326320886612, -0.016820799559354782, 0.19859741628170013, -3.1537973880767822, -0.12243146449327469, -0.525398313999176, 0.21830910444259644, -1.101082682609558, 1.2499816417694092, -0.24557586014270782, 1.0036838054656982, 0.6317331790924072, 0.8358795046806335, 0.09938672930002213, 0.42441311478614807, 0.3267926275730133, -1.7990740537643433, -0.523358166217804, -0.3884701430797577, -1.4391491413116455, -1.3091166019439697, 0.5991506576538086, -0.8455089330673218, 0.4984554052352905, -0.548332929611206, 0.06761269271373749, 0.28905948996543884, 0.17626258730888367, 0.6833481788635254, -0.129084512591362, 0.0784216970205307, -0.5042449235916138, 1.9813764095306396, -0.8769988417625427, 1.2035584449768066, 0.635132372379303, 1.030734658241272, -0.7989813685417175], [0.4313599169254303, -0.7314174175262451, -1.0041497945785522, 1.6557368040084839, 0.37529733777046204, -0.6712630391120911, 0.16515658795833588, -0.4775511920452118, 0.8079532980918884, -0.9598753452301025, -1.0362251996994019, 0.24284908175468445, -1.5312554836273193, 0.5579637289047241, -0.8621180057525635, 0.26592621207237244, -0.5958254933357239, 0.7654720544815063, 0.12616291642189026, 0.41756394505500793, -2.0300631523132324, 0.08272265642881393, 0.11030115187168121, 0.6832738518714905, 1.1367918252944946, 0.47129884362220764, -0.4429212212562561, 0.6416391134262085, 0.6455983519554138, 0.12326265871524811, 1.6881775856018066, 1.592786431312561, 0.8814534544944763, -0.02078264020383358, -0.8382565975189209, 0.0810975432395935, 0.702067494392395, -2.267228841781616, 0.9159107804298401, 0.5544248819351196, -0.04865832254290581, -0.14346933364868164, -1.0526635646820068, 0.22152747213840485, -0.8935034275054932, 0.7424785494804382, 0.008954011835157871, -0.05147486552596092, 1.2162021398544312, 0.4409787952899933, 1.3694064617156982, -0.09007863700389862, -0.8596698641777039, 0.10622230917215347, -0.37181058526039124, -0.0067849718034267426, 0.4637286961078644, 0.3742583692073822, 0.35876843333244324, -1.0117900371551514, -0.12225867062807083, -0.08242329955101013, 0.1649305820465088, 0.22167372703552246, 1.7772246599197388, 0.19024117290973663, -0.5213483572006226, -0.5007416009902954, -0.9087387919425964, 0.18545086681842804, 0.07837074995040894, 2.575859785079956, -0.2388215959072113, -0.27764075994491577, -0.3535749316215515, -0.09585890918970108, 1.4663360118865967, -1.2548325061798096, 0.4808584749698639, 0.9099956750869751, -0.13804693520069122, 0.30973708629608154, -0.8437023758888245, 0.025666818022727966, 0.3268410563468933, -0.49772170186042786, -0.8732519149780273, 0.5803821086883545, -0.023419557139277458, 1.0713202953338623, 0.3002952039241791, -0.25077322125434875, 0.6513085961341858, 1.1936020851135254, 0.5336251258850098, -0.8338047862052917], [-0.2888035476207733, 1.0317529439926147, 0.4897615313529968, -0.08238628506660461, -0.30182936787605286, -1.398168921470642, -0.04159785434603691, -0.8062137365341187, -0.5881291627883911, -0.1431388109922409, -1.459970235824585, -0.2443370372056961, 1.4043248891830444, 0.18161466717720032, 1.5659633874893188, -0.020557019859552383, -0.07763616740703583, 0.6709205508232117, -0.5517421960830688, -0.3340725898742676, -0.706313967704773, 1.0473276376724243, -0.7874857783317566, 0.21872058510780334, -0.2560131847858429, -0.7327775359153748, 1.9032816886901855, -0.6738521456718445, -1.245646357536316, 0.24192066490650177, 0.7013838291168213, 0.19666871428489685, -0.7218449115753174, 0.40868470072746277, 2.0638840198516846, -0.11951308697462082, 0.00864381156861782, -0.36727645993232727, -0.19187991321086884, -0.06467342376708984, 1.4514813423156738, 1.3750653266906738, 0.09106782078742981, -1.2653459310531616, -0.3919163942337036, 0.2767956852912903, -0.14456544816493988, 1.8270870447158813, -0.8589468598365784, 1.0971696376800537, -1.484216570854187, -0.7610009908676147, 0.7177760004997253, 0.6873390674591064, -0.21670465171337128, 0.022728731855750084, 2.2587945461273193, 1.391963005065918, 2.2993311882019043, 0.2690458595752716, 1.59824538230896, 0.10012780129909515, 0.2960290312767029, 1.358752965927124, 0.16819502413272858, 1.113067388534546, -0.05485493317246437, 1.4640400409698486, -0.6390298008918762, 0.36274453997612, -0.25804364681243896, -0.6212397217750549, 0.24566885828971863, 0.5895450711250305, -0.17846594750881195, 0.49103862047195435, 1.1856166124343872, 0.24166448414325714, -0.4082431197166443, -0.8687567114830017, -0.14272440969944, -2.123016357421875, -0.4159659445285797, -3.1293909549713135, -0.7236876487731934, -1.1947126388549805, 0.789694607257843, 0.10050775110721588, 0.29632532596588135, -0.18706932663917542, 0.6922820210456848, 0.5008264183998108, 0.8714815378189087, 1.14119291305542, 0.7178844809532166, -0.8933191299438477], [0.8368484973907471, -1.3398247957229614, -1.8061504364013672, -0.18836891651153564, 1.4185948371887207, 1.2832452058792114, 0.3677048683166504, -1.6883639097213745, -0.9516795873641968, -1.1694034337997437, 0.22139430046081543, 1.1544846296310425, 0.5028183460235596, -1.3080333471298218, -1.2819175720214844, -1.1831895112991333, 1.2033358812332153, 0.8006065487861633, 2.005485773086548, -1.0998603105545044, -1.248868465423584, 1.3479106426239014, -0.9345016479492188, -0.06759605556726456, 0.3217306435108185, 0.8012588620185852, -1.2593815326690674, 0.7788556218147278, -0.5463085174560547, 1.778515338897705, 0.6009320020675659, 1.4363138675689697, 2.5825252532958984, -1.127663254737854, 0.47929030656814575, 1.3639869689941406, 1.472808837890625, -0.6110773086547852, -1.3345640897750854, 2.146145820617676, -0.5197583436965942, -0.3927359879016876, 1.4680440425872803, -0.5486389398574829, 1.754269003868103, -0.3235055208206177, 1.3740041255950928, -0.5537219643592834, -1.51162588596344, -0.5362884998321533, 0.41070955991744995, -1.4173372983932495, 0.24721960723400116, 0.11723609268665314, -0.7864993810653687, 0.6788805723190308, -0.4492645859718323, -1.054115891456604, 0.14649923145771027, 0.8472549319267273, 0.14238408207893372, -0.5082601308822632, 0.67425537109375, -1.2222965955734253, -0.868867039680481, 0.5524448156356812, 0.6864125728607178, 0.6022374629974365, 0.717625617980957, -1.0443170070648193, -1.5154836177825928, 1.9397687911987305, -0.820692777633667, 0.24244938790798187, -1.170475959777832, 1.067208170890808, -0.1686364859342575, 1.3600637912750244, 0.812681257724762, -0.002263353206217289, -0.37979891896247864, -0.9653412103652954, -1.2738571166992188, 0.18830065429210663, 2.0014889240264893, 2.242016553878784, 0.8194200396537781, -0.3995521366596222, 0.11809799075126648, -1.0257375240325928, 0.1958630383014679, 1.0231744050979614, -0.08406811207532883, -0.7359532713890076, 0.35951918363571167, -0.12638013064861298], [-0.24639557301998138, -0.726830244064331, 1.4539259672164917, 0.47749999165534973, -0.028364907950162888, -1.9263206720352173, 0.2811618447303772, -0.20882964134216309, -0.3347802758216858, 1.6947227716445923, 0.9052469730377197, 0.8959938883781433, -0.9673714637756348, -0.19678033888339996, 0.7661756873130798, 0.7843418121337891, -0.11792730540037155, -1.7430217266082764, 1.309441328048706, 0.11607937514781952, 0.5808667540550232, -0.42641884088516235, 0.3328387141227722, -2.5058016777038574, 0.4584812521934509, 0.39457640051841736, 0.28238973021507263, -0.33561745285987854, -0.21939659118652344, -0.833486020565033, -0.3478662669658661, -0.9157987236976624, 0.8123764991760254, 0.9878158569335938, -0.3819883465766907, -1.3826439380645752, 0.38181090354919434, -0.6783491969108582, 0.18158209323883057, -1.1899142265319824, 0.2433209866285324, -0.35297635197639465, 0.39098992943763733, 0.6301499605178833, -0.05611829087138176, 0.9302265048027039, -0.05724726617336273, -1.7672193050384521, -0.2269570231437683, 0.2600974142551422, -0.47086089849472046, -0.3326047658920288, -0.5229357481002808, 0.18921653926372528, 1.160836100578308, -1.1355215311050415, -0.7230517864227295, 1.3626011610031128, 1.1382701396942139, -0.2695334851741791, -2.0516197681427, -1.1375582218170166, -0.9698330163955688, 0.9823732972145081, 0.5492860078811646, 1.9401445388793945, 0.17804796993732452, 0.5234114527702332, 0.08092544227838516, -0.2931528091430664, -0.4559359848499298, -0.4466361403465271, 0.8029927015304565, -1.1496448516845703, 0.791154682636261, -1.201806664466858, -0.11787691712379456, 0.6789433360099792, 0.2411700338125229, 0.5042451024055481, 0.25143128633499146, 0.8939381837844849, -1.0619251728057861, 0.7658842206001282, -0.573061466217041, 0.4661228656768799, 0.5762974619865417, 0.48036691546440125, 2.398899793624878, -0.43035006523132324, -0.42723608016967773, -0.3835670053958893, -0.0966557040810585, 0.8381685018539429, 1.0226414203643799, 0.07697395980358124], [0.7046387791633606, 0.098325714468956, 0.6763911843299866, -0.24026308953762054, -0.17723448574543, 0.3342551290988922, 0.9047608971595764, 1.3746827840805054, -1.2101725339889526, -0.9581463932991028, -0.7406882047653198, 0.268962562084198, -0.15268629789352417, -0.5292180776596069, -1.920449137687683, -0.08047974854707718, -1.959286093711853, 1.3717433214187622, -1.5333260297775269, -1.1491360664367676, -0.08697979152202606, 0.7622974514961243, -0.43601343035697937, 1.4110827445983887, 1.6954354047775269, -1.0850461721420288, -0.8526296615600586, -0.7954747676849365, -0.6667974591255188, -0.37305110692977905, 0.20644444227218628, 0.5915655493736267, 1.6283643245697021, -0.14129124581813812, -1.0716094970703125, 0.8485081195831299, 0.3027704954147339, -0.2125084400177002, 1.5054794549942017, -0.09947014600038528, -1.3120487928390503, -1.1476207971572876, 1.1831071376800537, -0.5634925365447998, 0.025322340428829193, 1.209451675415039, 1.6164602041244507, -0.5842701196670532, -1.4168277978897095, -1.7110114097595215, 0.6394097805023193, 1.1354931592941284, 0.8121371865272522, 0.4312763512134552, -1.081053376197815, 0.11851374059915543, 1.9509077072143555, 1.260549545288086, 0.10464516282081604, -1.3257083892822266, -0.7909702062606812, -0.9365396499633789, 0.37720879912376404, -1.1806049346923828, -0.19112439453601837, 0.6515341997146606, -0.3180253505706787, 0.762978196144104, -0.3476245105266571, -1.3016949892044067, -1.6048040390014648, 0.4633590579032898, 0.46974194049835205, 0.9826306700706482, 0.2041357010602951, 0.043402694165706635, 0.6475814580917358, 0.48774686455726624, -2.151947021484375, -0.06890644878149033, -0.4557236433029175, -1.2362053394317627, -0.1509179025888443, -1.0467331409454346, 1.0218311548233032, 1.0860764980316162, -1.1432567834854126, 0.7472706437110901, -2.3071751594543457, -0.06041533872485161, -0.6973950862884521, 0.13536588847637177, -0.6736600399017334, -1.9224909543991089, -0.4261334240436554, -0.40235209465026855], [-0.6747071146965027, 1.7395212650299072, -0.663000762462616, 1.0376231670379639, -0.6448155641555786, -0.4513171911239624, -0.23989561200141907, -0.4358505606651306, 1.8364640474319458, -0.07068081200122833, 0.44347816705703735, 1.8340870141983032, 0.15985946357250214, 1.1091420650482178, 1.2526143789291382, 0.910757839679718, 0.8464482426643372, -0.02491104044020176, 0.8724640011787415, -1.0720843076705933, 1.0065780878067017, -0.1842627078294754, 2.383958101272583, 0.4395841658115387, -0.8507684469223022, -0.36804184317588806, 1.1307743787765503, -1.1397888660430908, 1.5231562852859497, -0.4367790222167969, 0.757314920425415, -0.12141481041908264, 1.5850731134414673, 0.6584046483039856, 0.4139045178890228, -0.4987398386001587, -1.0788395404815674, 0.23618854582309723, -0.4145885109901428, 0.42266109585762024, 0.7758591771125793, -0.3660653233528137, 1.1417149305343628, 0.9035656452178955, -0.758561372756958, 1.2365494966506958, -0.5001250505447388, -0.25485149025917053, -2.733701467514038, -2.8623523712158203, 0.43974804878234863, -0.06805554777383804, -1.0561566352844238, 0.08166555315256119, 0.9901110529899597, 1.0725799798965454, 0.7992321848869324, 0.2544039487838745, 1.8944414854049683, -1.1802771091461182, 1.2626688480377197, -1.6594834327697754, 0.5493480563163757, 1.0247780084609985, -0.05851038545370102, 0.17078952491283417, 1.6561864614486694, 1.2892006635665894, 1.3248454332351685, -1.0886235237121582, 1.5667269229888916, -0.2836121618747711, 0.2048460990190506, 0.9516554474830627, -0.9563331007957458, -0.5770044922828674, -0.24824930727481842, 0.8085756301879883, 1.384291410446167, 2.236612558364868, 0.41165435314178467, -1.8915046453475952, -1.2279672622680664, -0.6354721188545227, -1.0396908521652222, 0.5578834414482117, 0.11137112975120544, 1.9864165782928467, -0.21034705638885498, 0.6874715685844421, -0.2309582382440567, 0.08744042366743088, 1.436599850654602, 0.25633057951927185, -0.581882894039154, 1.1030749082565308]], "dtype": "float32"}, 32, 32, 32, 2], "e": {"__tup__": [{"__nd__": [[0, 2, 4], [1, 3, 5], [6, 7, 8]], "dtype": "int32"}, {"__nd__": [[-0.49795278906822205, 0.9071056246757507, 0.26289376616477966, -1.401198148727417, 0.3639110326766968, -0.6602045893669128, -0.629126250743866, -1.6209625005722046, 2.453885316848755, -0.5064194798469543, -1.3996905088424683, -0.4106569290161133, 0.12188715487718582, 1.1586107015609741, -0.8062254190444946, -1.132196307182312, 1.3947203159332275, -0.9601513743400574, -0.24415872991085052, 1.248913049697876, 0.007314543705433607, -2.5862128734588623, -0.01915866881608963, -0.022707825526595116, -0.8542375564575195, -1.4517061710357666, -0.43094372749328613, 0.3329795002937317, 0.06830940395593643, 0.4425162672996521, 1.3015930652618408, 1.1641207933425903, 0.5729677081108093, 0.8242462277412415, -0.3695106506347656, -2.2801053524017334, -0.171730637550354, 1.1385548114776611, -1.1620961427688599, 0.626406192779541, -0.49618756771087646, 0.8453544974327087, -0.6755011677742004, -0.6902378797531128, -1.5114761590957642, -1.238778829574585, -0.4678708016872406, -1.8021366596221924, 1.369589924812317, -0.07560531049966812, 0.9458588361740112, -0.6035559773445129, 0.4349993169307709, 0.6739364266395569, 0.4373567998409271, 0.7256230711936951, 0.2646333575248718, -0.2882468104362488, -1.26893949508667, -0.8895449638366699, -0.3766125440597534, -0.09674873948097229, 0.5886800289154053, -0.39802706241607666, 0.8090369701385498, 0.35144859552383423, -1.072837471961975, -0.2736830711364746, -1.3187545537948608, -1.068756341934204, -0.17415501177310944, 0.260063499212265, -0.11198100447654724, -0.47000476717948914, -0.3185606598854065, -0.8431681394577026, -0.4066788852214813, 2.0687289237976074, 0.2711505591869354, -1.9579445123672485, -1.075190544128418, 0.9181632399559021, -1.5653542280197144, -0.14204582571983337, 1.7429618835449219, -0.1911378651857376, 0.4871656000614166, 0.09801697731018066, -0.846275269985199, 1.3442105054855347, -0.4451044797897339, 1.1930162906646729, -1.4092973470687866, 0.17644761502742767, 1.47214937210083, 0.7202909588813782], [-0.6945894956588745, -0.7533285617828369, 1.0080952644348145, 1.187495231628418, 0.37781980633735657, -1.7775661945343018, 0.9806830883026123, 2.1965959072113037, 0.49401697516441345, -1.740050196647644, -1.9020121097564697, 0.13613061606884003, 0.1635597050189972, -1.6070481538772583, -1.57585871219635, 0.6684826612472534, -0.2359996736049652, 0.6907364726066589, -0.8321084380149841, -0.2195291519165039, -0.4339107275009155, 0.5658369660377502, 2.2520196437835693, 0.0738578662276268, 0.18852777779102325, 0.7725536823272705, 0.059030093252658844, -0.6160983443260193, 0.12117034196853638, -0.7597494125366211, 0.9799633622169495, -1.1431952714920044, 0.47183752059936523, 0.0862109586596489, -0.6162278056144714, -0.11722469329833984, -0.61776202917099, 0.9345230460166931, 0.06085528805851936, -1.4126583337783813, 0.48712053894996643, -0.08574935048818588, -1.8016637563705444, -1.6464998722076416, 1.1172759532928467, -0.8870353698730469, -0.32910168170928955, 0.5005953907966614, -1.2827403545379639, -0.5848491787910461, -0.6972424983978271, -1.0842139720916748, -0.571546196937561, 0.2732192575931549, -1.132301688194275, -0.13160641491413116, -0.041324056684970856, 0.8564432859420776, -0.02753492072224617, -0.23107482492923737, 0.5181928873062134, 0.3622864782810211, 0.5048407316207886, 1.0502057075500488, -0.815879762172699, 0.4531157612800598, 0.0014811510918661952, -1.0134453773498535, 0.8385912775993347, 1.5005077123641968, 0.6084892153739929, -1.7855366468429565, 1.7991013526916504, 0.5920233130455017, 0.7708218097686768, -1.1749951839447021, 0.32176876068115234, 1.2176711559295654, -0.9063076972961426, -0.4035940170288086, -2.408651351928711, 1.3349159955978394, 1.713362455368042, 1.0888680219650269, -1.414018988609314, 0.6039206385612488, -0.3037649095058441, -1.8177177906036377, -0.8264444470405579, 0.8408046364784241, -1.0133486986160278, 0.12162350118160248, 1.2013206481933594, 0.2886587977409363, -0.8716387152671814, -0.15070563554763794], [-0.43582242727279663, -1.1686877012252808, -0.36938127875328064, 0.8399848937988281, -0.6851648092269897, 1.3787250518798828, 1.4485273361206055, -0.5881469249725342, 0.12648241221904755, 1.3206477165222168, 0.9335799813270569, -0.02970162034034729, 0.2248820811510086, -0.22568784654140472, -0.45399197936058044, -0.058496396988630295, 1.085605263710022, 0.048564840108156204, 0.6785559058189392, 0.7633587121963501, -0.08840077370405197, -0.4696447551250458, 0.29957279562950134, -0.9968749284744263, 0.7426887154579163, -0.22771291434764862, -1.4379513263702393, -0.33600133657455444, -1.6371142864227295, -0.213825523853302, 1.14036226272583, 1.5936816930770874, 0.021696502342820168, -0.4610954821109772, -0.7327889800071716, 0.6268059015274048, 0.1280675232410431, 0.4912046492099762, -0.42118901014328003, -0.4220081865787506, -0.39053329825401306, -0.5336024165153503, 1.4688233137130737, -0.2690778374671936, 0.10794682055711746, 1.6432377099990845, -1.460132122039795, 0.38496047258377075, 0.04505527764558792, 0.296792209148407, -0.9738928079605103, -0.16252650320529938, -0.6680938601493835, 1.541739583015442, -0.7827043533325195, 0.134311705827713, 0.296099454164505, -0.22313347458839417, 0.8949829936027527, -1.1263643503189087, -0.7466985583305359, 0.4610041379928589, -0.004559234715998173, -0.23520204424858093, -0.5330246686935425, 0.8827319145202637, 1.2857816219329834, 0.08222571760416031, -0.09779711812734604, -0.23515260219573975, -0.2916783392429352, 1.9158343076705933, 0.4543007016181946, 1.3150362968444824, -0.6457900404930115, 0.7055777311325073, -0.21141457557678223, -0.1997203230857849, 0.26317131519317627, -0.4246659576892853, 0.7075518369674683, -2.379399061203003, -1.5253639221191406, -0.1862526684999466, -1.227222204208374, 1.598136067390442, 0.361646831035614, -1.5977619886398315, 0.5935990214347839, 1.780019760131836, -0.20186029374599457, 0.1690872311592102, 0.3446178734302521, -1.1512935161590576, 0.45261135697364807, -0.5522788166999817], [-0.006241438910365105, 0.36090996861457825, -1.448960781097412, -0.6422763466835022, -0.026445189490914345, 0.4553539752960205, 1.7707860469818115, 0.9271524548530579, -0.6097609996795654, 1.5319663286209106, -0.6844172477722168, 1.6417683362960815, -1.7569941282272339, -0.3784109950065613, 1.3031738996505737, -1.5854904651641846, -2.264111280441284, 1.2650066614151, -0.3272346258163452, 0.9025905132293701, -0.43085330724716187, 1.3682122230529785, 2.30863356590271, 0.7841261029243469, -0.9586005210876465, -0.310547411441803, -0.5509255528450012, -0.021156935021281242, 0.4560758173465729, -0.37632080912590027, -0.7271192669868469, 1.174673318862915, 0.22328084707260132, 1.4489799737930298, -0.9835121035575867, 1.5408847332000732, 0.07550261169672012, -0.6448619961738586, 0.06367002427577972, -1.0998731851577759, -1.275716781616211, -1.2025408744812012, 0.6622570157051086, -1.126190423965454, -0.06942912936210632, 0.17846357822418213, -1.5971040725708008, 0.5545268058776855, -1.2929162979125977, 1.8170456886291504, -0.4157670736312866, 1.5833371877670288, 0.9139670729637146, 0.23237662017345428, -0.8756034970283508, 0.0017870597075670958, 0.8387735486030579, -0.014401113614439964, 0.10554665327072144, 0.08762677013874054, 0.09334404021501541, 2.2366292476654053, 0.9469573497772217, -0.03171543404459953, 0.5998403429985046, 0.5002793669700623, 0.384071409702301, 1.627465009689331, -0.407040536403656, 0.7329308986663818, 0.537267804145813, 0.17957834899425507, -0.7253226637840271, 1.7614502906799316, 0.8112240433692932, 1.3355698585510254, 0.48769351840019226, 0.36633187532424927, 0.057538505643606186, -0.6364378929138184, -1.8784000873565674, -1.6228797435760498, 0.9217084646224976, -0.5868649482727051, 0.12367870658636093, 0.729232907295227, -0.26214781403541565, -0.9549933075904846, -0.9674780964851379, -0.5430068969726562, -1.267194151878357, -0.6718016266822815, 0.20139093697071075, -0.014255517162382603, -0.5397071242332458, -0.9398757219314575], [-0.6240309476852417, 0.7226833701133728, 0.15261080861091614, 0.6606891751289368, 2.1143593788146973, -0.5842588543891907, -0.6557784080505371, 0.6054374575614929, -1.0069384574890137, -0.13090236485004425, -0.41022562980651855, -0.5714211463928223, -0.1739046424627304, -0.08973930776119232, 2.5498924255371094, 1.6900784969329834, -0.42611101269721985, 0.10486975312232971, -0.6722941398620605, 0.2071312963962555, 0.22452130913734436, -0.9934946894645691, 0.696973979473114, 0.9959542751312256, -0.33484312891960144, -1.8655940294265747, -0.175608828663826, 1.453181266784668, -0.05983706936240196, 0.7900193929672241, 0.16178911924362183, 0.24890387058258057, 0.35231679677963257, -0.510332465171814, 0.6793636083602905, 0.38870933651924133, -0.004874637350440025, -0.10136209428310394, -0.5875624418258667, 1.2588053941726685, 0.21309983730316162, 0.6395609378814697, 0.05369439348578453, -0.6469739079475403, -1.017188310623169, 1.5248262882232666, 0.9177829027175903, -0.24939103424549103, -1.7933933734893799, 0.5806452631950378, 0.584466278553009, -0.5310268998146057, 0.3817123472690582, 0.00683584064245224, -0.8949990272521973, 0.2626420855522156, 0.9742196202278137, -0.2341984361410141, -0.2099805772304535, -0.9160048365592957, -1.1067479848861694, 0.668129563331604, -0.47448521852493286, -0.8589116930961609, 0.5430090427398682, -0.6096816658973694, 0.2553333640098572, -1.1254719495773315, 0.3777589499950409, 0.06258653104305267, -0.695708692073822, 1.6627748012542725, -0.6883028745651245, 1.3721072673797607, 0.850386917591095, -0.818960964679718, -0.20149743556976318, 1.3135793209075928, -0.17503724992275238, -1.408103585243225, 1.3306992053985596, 0.6992471814155579, -0.08837329596281052, -1.0754303932189941, -1.0240864753723145, 1.5744622945785522, -1.269956111907959, 0.9128707051277161, -0.6733980774879456, 1.195202112197876, 0.9741679430007935, -0.27697810530662537, -0.3545531630516052, 1.299951195716858, 1.3261185884475708, 3.2254867553710938], [-2.1039438247680664, -0.998136579990387, 0.7497047781944275, -0.9163962602615356, -1.2616593837738037, -1.4050238132476807, -0.013596647419035435, 1.5041604042053223, -0.17687830328941345, 0.4454765021800995, 0.2859637141227722, -1.7895114421844482, 0.48649001121520996, -0.4119691550731659, -0.31454843282699585, 1.7022244930267334, 0.7589757442474365, 0.19594205915927887, -1.081047534942627, -0.28321337699890137, -0.9783759713172913, -0.2384425401687622, 0.8568532466888428, 0.09171908348798752, -0.8494375944137573, 0.2889663577079773, -1.5663468837738037, -0.2364378273487091, 0.8794224858283997, 0.4620170593261719, -0.42421481013298035, -0.38074371218681335, -1.0680180788040161, 0.5991125702857971, -1.4848897457122803, 0.6723499894142151, 0.05535924434661865, 0.28647127747535706, -0.32812175154685974, 0.8183506727218628, 0.7964516878128052, 0.22343838214874268, 0.26571741700172424, 0.9651152491569519, 0.9239421486854553, -0.6457239985466003, 0.02770305424928665, -1.4484840631484985, 0.4641319215297699, 1.3335696458816528, -0.4746023714542389, -0.5514082908630371, 0.5382747650146484, 1.136925220489502, -0.5828882455825806, -0.16359710693359375, 0.32138943672180176, -0.5065733194351196, -1.0848009586334229, 0.37880828976631165, -0.9948378801345825, 0.9257183074951172, 1.0687999725341797, -0.7620264887809753, 0.11565215140581131, 0.6456112861633301, 0.003915951121598482, -0.047814033925533295, 0.5712251663208008, 2.19088077545166, 0.7471486330032349, 0.8475247621536255, -0.6043049693107605, -0.06751503050327301, -0.6471095085144043, -2.99748158454895, 0.09809450060129166, 0.25149431824684143, -0.45680612325668335, -0.2259976714849472, -0.7079541683197021, -0.738388180732727, 0.26416561007499695, 0.3570207953453064, 0.4245726466178894, -0.34357792139053345, -0.6634505391120911, -1.142197847366333, 1.6037611961364746, -0.5756776928901672, -0.1969238519668579, 0.32143881916999817, -2.196554183959961, 0.08822394162416458, 1.0566025972366333, 0.07355085015296936], [-0.4703666865825653, 0.12382535636425018, -0.5646469593048096, 0.05780757591128349, -1.6681294441223145, -1.5445283651351929, -0.35648804903030396, 0.07682168483734131, -0.4228613078594208, -1.3330023288726807, -0.4401605427265167, -0.4427315592765808, -1.8809133768081665, 0.45930466055870056, -0.5592933893203735, -0.15558938682079315, -0.6364084482192993, -0.6859750151634216, 0.3067642152309418, 0.49030032753944397, -0.6907861828804016, -1.0022473335266113, 1.5126149654388428, -0.12749788165092468, -1.6945942640304565, -0.07712822407484055, 1.068579912185669, 0.0249099750071764, -0.07262753695249557, 0.18895794451236725, 0.28380587697029114, 0.7694535255432129, -1.2019699811935425, 0.4044361412525177, 0.7896705865859985, -0.9700336456298828, -1.3423179388046265, -0.14704692363739014, 0.8786119818687439, -1.631941318511963, -0.14761878550052643, 0.8228174448013306, -1.6508259773254395, 1.362518072128296, -0.17462164163589478, -0.8288095593452454, 1.470057725906372, 0.4656227231025696, 0.6624540090560913, 0.6886785626411438, -0.48696500062942505, 0.008323430083692074, -0.5344892740249634, -1.8060617446899414, -0.890058159828186, 0.9281018376350403, 2.2086164951324463, 1.5036368370056152, 0.31871500611305237, -0.7437292337417603, -0.36843109130859375, 1.0033177137374878, -0.508159339427948, -0.4354137182235718, -0.03879166021943092, -0.7792671322822571, -0.15405836701393127, 0.007017816416919231, 1.2630183696746826, 0.47429123520851135, -1.0363376140594482, 0.28819766640663147, -1.4129877090454102, 1.1624864339828491, 0.6377509236335754, 0.8318082094192505, 0.45608508586883545, 0.7285357713699341, 0.7142336368560791, 0.23692792654037476, -0.8879398107528687, 0.08670589327812195, -2.290158271789551, -0.42315399646759033, -0.879625678062439, -1.1374400854110718, 0.8487400412559509, -2.0932364463806152, 1.1188970804214478, 2.8721423149108887, 1.5181766748428345, -0.036856360733509064, 0.7465159296989441, -0.7712705135345459, 1.1443958282470703, 0.04872975870966911], [0.9012510776519775, 1.7253258228302002, -0.30104175209999084, -0.4010622501373291, 1.6385867595672607, 0.6049880385398865, -0.9316867589950562, 1.3972030878067017, -1.227165937423706, 0.787543535232544, -1.2334636449813843, -0.023279784247279167, -1.6198776960372925, 0.5609673261642456, 1.3864985704421997, -0.6123051047325134, -0.2365586757659912, -0.3211665451526642, 0.2902032732963562, -1.276986837387085, 0.8112168312072754, -0.3251607120037079, -1.1579283475875854, 1.2914843559265137, -1.4956941604614258, 1.421221137046814, -0.11259917169809341, -0.4156803488731384, -1.2119336128234863, -0.7617514729499817, -0.7773157954216003, -1.132198452949524, -0.4836583435535431, 0.14156924188137054, 1.129356026649475, -2.3238511085510254, -0.9831873774528503, -0.18155480921268463, 0.49274271726608276, 0.3799891471862793, 0.5499352216720581, -0.9764860272407532, -0.8042727112770081, 0.8056177496910095, 1.0286918878555298, -0.2873384952545166, 0.5874477028846741, -1.3359079360961914, 0.24328310787677765, 0.529800295829773, -0.38812193274497986, 2.697082281112671, 0.5666472911834717, 1.044291615486145, -0.16742639243602753, -1.2509218454360962, -0.030698098242282867, -0.25923267006874084, 0.4865227937698364, -0.9617146849632263, 0.2947951853275299, 0.4070988893508911, 0.5520862340927124, 0.2767235040664673, -0.25348371267318726, 0.8893678188323975, -0.49147483706474304, -0.5341799259185791, -0.6171811819076538, -1.1052651405334473, 1.00263249874115, -1.277819275856018, -1.3277851343154907, -0.9930272102355957, -0.7351406812667847, 0.03734526038169861, -0.8093864917755127, 0.6671777963638306, -1.0803413391113281, 0.5070992112159729, 1.3192802667617798, -0.39230379462242126, -2.6665611267089844, -2.1977272033691406, -0.09437486529350281, 1.1465519666671753, -0.011832315474748611, 0.5285620093345642, -2.1338062286376953, 0.9260376691818237, -0.14381931722164154, -1.925286054611206, 1.1013741493225098, 0.29036930203437805, -1.0934717655181885, -0.7287799715995789], [-0.3246835768222809, -0.8073686361312866, 0.48169630765914917, -0.10798284411430359, 0.015277745202183723, -0.1376291811466217, -1.0267504453659058, 0.8576993346214294, -0.36315757036209106, 0.331756591796875, 0.012063615024089813, -2.350733995437622, -0.3040810823440552, 0.04250214993953705, -1.408835768699646, -1.5109643936157227, 0.3606606423854828, -1.4970154762268066, 0.8479454517364502, -1.698587417602539, -1.2433760166168213, -0.2748022675514221, 0.6805523037910461, 0.26579973101615906, -0.617192804813385, 0.8394040465354919, 0.7016069293022156, -1.0413578748703003, -0.17175151407718658, 0.016139637678861618, -0.3929236829280853, 0.20455290377140045, 0.06373818963766098, -1.0208239555358887, -0.8078200221061707, 0.22596514225006104, -1.1204410791397095, 0.8613828420639038, 0.34936845302581787, -0.3047407269477844, -0.07092124223709106, -0.8487991094589233, -1.0285950899124146, 0.1835770606994629, 0.5604608654975891, -0.12774182856082916, -0.48920756578445435, -1.5892386436462402, -0.3584495484828949, 0.5757064819335938, -1.2478467226028442, -0.7473973035812378, -0.5660526156425476, 0.1010410338640213, 0.2550380825996399, 0.7779929637908936, 0.13250324130058289, 1.3464027643203735, -0.2793295979499817, -0.12232217192649841, 0.1268683224916458, 1.6820622682571411, -0.3707502484321594, 0.5658265948295593, 1.0263909101486206, 0.003699560882523656, 0.8180952072143555, -0.6974077820777893, 0.97198486328125, -1.2128368616104126, -0.49158820509910583, -0.8776080012321472, 0.7795112133026123, 0.14791645109653473, 1.5801769495010376, -0.13102227449417114, -0.5678715109825134, -0.9654333591461182, 1.1320037841796875, 0.3685697317123413, 1.1612547636032104, 1.946276068687439, 2.2819032669067383, 1.1950631141662598, -2.1831018924713135, 2.2635836601257324, 0.4715476930141449, -0.48182952404022217, -0.815723180770874, -0.3230040669441223, 1.0318104028701782, -1.1138206720352173, 0.6942121982574463, -0.8002138733863831, -0.7812758684158325, -0.7020620107650757], [0.04643898084759712, 0.5151606798171997, -0.7387146353721619, 0.024483809247612953, 1.3994388580322266, 0.05855577439069748, 1.6634920835494995, -1.5847493410110474, -0.31960195302963257, 0.3751198947429657, 1.0195950269699097, -0.567138671875, 0.1495487093925476, -1.0566599369049072, -0.15512177348136902, 0.007523552514612675, 0.5757823586463928, 0.3912578821182251, -0.3721323311328888, 1.890181064605713, 0.36495235562324524, 0.7951963543891907, -0.034490637481212616, 0.15630978345870972, 0.6978076696395874, -1.902329444885254, -0.47625479102134705, -0.38679414987564087, -0.8669521808624268, 0.5184411406517029, 0.842341959476471, 0.08557749539613724, 0.23271040618419647, 1.6818264722824097, 1.2052546739578247, 0.7703916430473328, -1.6565767526626587, -1.178814172744751, -0.08464693278074265, -2.58064341545105, 0.07295971363782883, 0.37662509083747864, 0.3882305920124054, -0.5180512070655823, 1.5769269466400146, 1.0518733263015747, -1.0784287452697754, 0.7108385562896729, -0.17098361253738403, 0.5813629031181335, 0.4344177544116974, 1.464686393737793, 1.8827143907546997, -0.3196091055870056, -0.49234533309936523, 0.9737237095832825, -0.9721001386642456, -1.3961256742477417, 0.057473745197057724, -0.9823413491249084, 0.24509906768798828, -0.759467601776123, 0.26916101574897766, -0.5070133805274963, -1.280462622642517, 1.1183925867080688, 0.3216203451156616, -0.17343533039093018, -2.2328834533691406, -0.5759223103523254, 0.22990986704826355, -0.10710801184177399, 0.6928749084472656, 0.931574821472168, 1.351051688194275, 1.4411931037902832, 0.48833438754081726, -0.07527845352888107, 0.3707844913005829, 0.2538926303386688, -0.39814651012420654, 0.003555546049028635, 0.31638726592063904, -0.12648239731788635, -0.8742060661315918, 0.27196913957595825, 0.604417085647583, 0.7950952649116516, 0.9055721163749695, 0.05032901465892792, -0.9801292419433594, 0.001739636529237032, -0.2504304051399231, -0.502291738986969, 0.5652589201927185, 0.23610611259937286], [-1.5724283456802368, -1.8300501108169556, -0.5965132117271423, 0.11768470704555511, -1.1415480375289917, -0.0865742564201355, -0.19856445491313934, 0.005678272806107998, -0.0924263671040535, 0.915302574634552, 0.1136927679181099, -0.02728560008108616, -0.8987405300140381, 0.9633285403251648, 0.60666424036026, 0.1401224285364151, 0.21648524701595306, 0.02239510603249073, -0.3170478940010071, -1.36604642868042, 0.6393730640411377, 0.9107131958007812, 0.2964033782482147, -2.876631259918213, -0.4176587462425232, 0.7840473651885986, -0.8420780897140503, 1.6168659925460815, 0.7128962278366089, -1.4725819826126099, -0.7036986947059631, -0.5574815273284912, -1.6339799165725708, -0.36831533908843994, -0.46124693751335144, -0.9009546041488647, 0.930688202381134, 0.6629396080970764, -0.47217822074890137, -1.1063238382339478, -1.4917078018188477, -0.36169955134391785, 1.1953033208847046, 0.38745495676994324, 1.5054527521133423, -0.20923392474651337, 0.8925318121910095, -0.24515056610107422, -0.9315056204795837, 1.5990756750106812, 1.655969262123108, 1.7582753896713257, 1.5199209451675415, -1.9596490859985352, 0.9032212495803833, 0.9243226051330566, -1.3951618671417236, -1.9160583019256592, 0.004576630890369415, 0.6437805891036987, -0.12184948474168777, -0.21526826918125153, -0.1832956224679947, 1.157203197479248, -0.4605427384376526, -0.6387475728988647, -0.20896625518798828, -0.5069522261619568, 0.2748439610004425, -0.33485352993011475, 0.3812362551689148, 0.021940505132079124, 1.9125508069992065, -0.5686619877815247, -0.9306972622871399, 0.060448601841926575, 0.36363279819488525, -0.9735159277915955, 1.8853760957717896, 0.9242631793022156, -0.5890610814094543, 0.6142690777778625, 1.2807704210281372, 0.37480178475379944, 0.17240436375141144, 1.1986325979232788, 0.7369639873504639, -0.7269115447998047, -0.7191872000694275, 0.4943925440311432, 1.1312313079833984, 0.6862639784812927, 0.32566285133361816, -0.5240622758865356, 0.5440499186515808, 0.021913956850767136], [-0.685411810874939, -0.3666370213031769, 0.9720906019210815, 0.36682894825935364, -0.5276739001274109, 1.1290518045425415, -0.1597057431936264, -1.2952959537506104, 0.5395038723945618, -1.0957354307174683, -0.4011271595954895, 0.8547888398170471, -0.6792475581169128, -0.5145795345306396, -0.06217261403799057, -0.6610673666000366, -0.4185587167739868, 0.9855679869651794, 0.6044508218765259, 0.29983872175216675, 0.06102975085377693, 0.4169212281703949, -0.29284432530403137, 0.5990751385688782, -0.48019200563430786, 0.541928768157959, 0.7858368158340454, 0.4563625752925873, -0.3925577402114868, -0.7969385385513306, 1.3956032991409302, -2.102741241455078, 1.3324054479599, -0.5313189625740051, 0.3529857099056244, 0.6633933782577515, 0.05991455167531967, 1.6647374629974365, -0.6155417561531067, 1.9155855178833008, -1.12447190284729, 0.509036660194397, -0.8695082664489746, -1.400797724723816, -0.9299245476722717, 0.6154340505599976, -0.1061665341258049, 1.175481915473938, 0.6393640041351318, 0.5816953182220459, -1.4969353675842285, 0.7776058912277222, 0.03015938028693199, 0.5900951623916626, 0.5435627102851868, -0.8636648654937744, 0.17364206910133362, -0.9014608860015869, 1.152024269104004, 0.15417979657649994, 0.9640173316001892, -0.9075795412063599, -0.3535751700401306, -0.08069099485874176, 0.4505394995212555, -1.7556829452514648, 0.7019816040992737, -0.7650355696678162, 0.8242113590240479, 0.15337318181991577, 0.642098605632782, -0.06801434606313705, -0.11221513897180557, -0.30613213777542114, -0.8773781657218933, 1.4536207914352417, -0.2918783724308014, -0.41455167531967163, 1.216259241104126, -0.02158910036087036, 0.07182838767766953, 1.0076780319213867, -0.7759730815887451, -0.6775530576705933, 0.8699404001235962, -0.4021363854408264, -0.17033787071704865, -0.7064419388771057, 2.13275146484375, 1.8255223035812378, -1.0593459606170654, 0.45547839999198914, -0.21642647683620453, -0.9906754493713379, -0.1306656301021576, -0.800820529460907], [0.28585678339004517, 0.2859191298484802, 0.8185268640518188, 0.0490758940577507, 2.1158759593963623, 1.6968071460723877, 1.8985238075256348, 0.010127409361302853, -0.7794955372810364, -0.8186898827552795, 2.0400023460388184, -1.3508161306381226, 0.4080314040184021, 0.03421156853437424, -0.8043286800384521, 0.0747670829296112, 1.5160232782363892, 0.869463324546814, -1.0396002531051636, -0.40371131896972656, 1.3372100591659546, -0.3457934856414795, 0.6263850927352905, 1.1839255094528198, 0.038307949900627136, -0.20167428255081177, 0.20398183166980743, -1.115766167640686, 0.49383819103240967, 1.395468831062317, -1.9461045265197754, 2.5765910148620605, -0.24014973640441895, -0.9004629254341125, 0.4366162419319153, 0.04422471672296524, -0.12152771651744843, 1.0269304513931274, 0.2561244070529938, -0.9829571843147278, -1.2870314121246338, 0.44657662510871887, 0.4258951246738434, -1.7090966701507568, -1.0679106712341309, -0.9928019642829895, -0.2963126301765442, -0.318449467420578, 1.4290415048599243, -0.03199300915002823, -1.1791231632232666, 0.9000323414802551, 0.24630244076251984, -0.28175923228263855, -2.083613634109497, -0.03800005093216896, -1.9071491956710815, 0.8014316558837891, -1.2413030862808228, -0.5322567820549011, -0.14966586232185364, 0.3945961892604828, -1.1376562118530273, 0.9355284571647644, -1.1610863208770752, 0.012084339745342731, -0.45458242297172546, -1.5106738805770874, 0.14944985508918762, 1.557963490486145, 0.05899422988295555, -0.5258286595344543, 0.3242796063423157, 0.14056873321533203, -0.20747317373752594, -0.8380531072616577, -1.4502580165863037, 1.0756433010101318, -2.384476900100708, 0.5278165340423584, 0.438652902841568, 1.119884967803955, -0.22993986308574677, 0.22405236959457397, 0.5435740351676941, -0.8538905382156372, -0.14084307849407196, -0.688944935798645, -0.21434062719345093, 0.24925760924816132, 0.8028906583786011, 1.4538745880126953, 0.11283445358276367, 0.09338792413473129, 0.432371586561203, -0.5835219025611877], [1.2392815351486206, 0.6846833825111389, -1.5795626640319824, -0.9202082753181458, 0.5468029379844666, -0.47567200660705566, -0.5391581654548645, -0.5761483907699585, 1.6092705726623535, 1.1979619264602661, -1.606908917427063, -1.1188017129898071, 0.1401531845331192, -1.7945579290390015, 0.5038855075836182, 1.0296522378921509, -0.6316300630569458, -0.8378907442092896, 0.6988952159881592, -1.2702510356903076, 2.503932237625122, 0.32456454634666443, 0.234671488404274, -0.6544645428657532, 1.8205598592758179, -0.11096768826246262, 0.7759998440742493, 0.2891191244125366, -0.7181664109230042, -0.5596407055854797, 0.1052095964550972, -1.3900433778762817, 0.5222809910774231, -0.6897312998771667, -0.04910663142800331, -0.9991747140884399, -0.006217981688678265, -1.769903540611267, -1.7835170030593872, -0.748916506767273, 0.6916004419326782, 1.3147494792938232, -0.45521295070648193, -0.25456786155700684, 0.5278615951538086, -1.4419602155685425, -0.8133334517478943, -0.008487074635922909, -0.33194732666015625, 0.23204471170902252, -1.1207101345062256, -0.20115700364112854, 0.9108837246894836, -1.1820052862167358, 0.06355730444192886, 0.1868634670972824, 0.9985330700874329, 0.8445019125938416, 1.2803058624267578, 1.6429332494735718, 3.405303716659546, -1.7110322713851929, 1.1567277908325195, 0.5268872380256653, -1.1201045513153076, 1.1873594522476196, -1.254646897315979, -0.9036970734596252, -1.871256709098816, -1.6066937446594238, 1.5137217044830322, 0.10179511457681656, 0.3475266396999359, 0.5033748745918274, -1.271618127822876, -0.5003936290740967, 0.09319432824850082, 0.054982028901576996, 1.8269116878509521, -0.27956801652908325, -0.6144214272499084, -0.9689426422119141, -1.1927934885025024, -0.6313275694847107, -0.8317966461181641, 0.3898947238922119, -0.39121997356414795, -1.140873670578003, -1.2031835317611694, -0.24174438416957855, -0.7453365325927734, 0.37908560037612915, 0.8646543025970459, -0.9562906622886658, 0.8287451863288879, -0.5965794324874878], [-0.3955351710319519, 0.5734788775444031, 1.0524930953979492, -0.31730547547340393, 1.7203187942504883, -2.0890989303588867, 0.06690102815628052, 1.4781148433685303, -0.5716235637664795, 0.5122388005256653, -1.7095575332641602, 1.088561773300171, -0.6571959257125854, 0.7982277870178223, 2.0577995777130127, 0.19464737176895142, -0.32361167669296265, 0.5603563189506531, 0.2714861035346985, 1.0290173292160034, 1.910738229751587, -0.08997954428195953, 2.349661111831665, -0.11920548230409622, -1.4017462730407715, 0.20398060977458954, -0.02734016813337803, -0.48220011591911316, 0.716255784034729, -0.18797940015792847, -0.4929267168045044, 0.2004004716873169, -0.40651950240135193, -0.09237947314977646, 0.14227840304374695, -0.7202656865119934, -0.12746375799179077, -0.4130651652812958, -0.15015430748462677, 0.3672732412815094, -1.2923767566680908, -2.1993448734283447, -0.06070692464709282, 0.24496866762638092, -0.22810505330562592, -0.25328588485717773, -0.9070162773132324, -1.5960850715637207, 0.08973094075918198, 0.8203177452087402, 0.12809628248214722, 0.764723539352417, -0.8503777384757996, 1.3450766801834106, 2.4880361557006836, -0.23070871829986572, -0.15031270682811737, -0.5832374691963196, 0.5601776838302612, 0.8116126656532288, -1.2162271738052368, 2.237157106399536, 0.590937614440918, 0.4985203444957733, -1.1771782636642456, 1.2256191968917847, -0.026174498721957207, -0.23298616707324982, 1.5426537990570068, 0.8871373534202576, -0.3720022737979889, 0.4259101450443268, -0.45164528489112854, 0.348984032869339, 0.30194947123527527, -0.27939072251319885, -0.09392989426851273, 0.7324493527412415, -0.9733847975730896, -0.7683209180831909, -0.12476533651351929, 0.4178542196750641, -0.6847026348114014, 2.036423444747925, 0.0666450783610344, 2.092726230621338, -1.0984166860580444, 0.654473602771759, 0.09883645176887512, 0.9443584084510803, -2.999506950378418, -0.8507953882217407, 1.704089641571045, -0.6237859725952148, 0.1424839049577713, 0.8811883330345154], [2.3324623107910156, -0.5583322644233704, -0.12539736926555634, 0.21287287771701813, -1.5264250040054321, -0.3968278765678406, -0.4857129454612732, 0.12501326203346252, 1.752380132675171, 1.649229645729065, -0.1531098634004593, -0.7762449979782104, 0.5312752723693848, 1.6591674089431763, -0.5195257663726807, -0.8778945207595825, 1.1954903602600098, 1.310590386390686, 0.9384272694587708, 2.099360942840576, -0.16894960403442383, -0.47527575492858887, 0.6582354307174683, 0.8320094347000122, -0.7213571667671204, 1.0252633094787598, -2.194885492324829, 0.3973413109779358, 0.7489414811134338, -1.861934781074524, 1.3176850080490112, 1.2850191593170166, 0.27440527081489563, -0.213458850979805, 0.5985321402549744, 0.5844821333885193, 0.5914268493652344, -0.8100637793540955, 0.9063379764556885, -1.3708833456039429, 1.759494662284851, 1.503660798072815, 0.19814714789390564, 1.5545710325241089, -2.5656354427337646, 0.6663287878036499, 0.5288040637969971, 0.690639317035675, 1.280657172203064, 0.2619255781173706, -0.8946475982666016, -1.3401330709457397, 0.5708303451538086, 0.1476958841085434, 1.0363194942474365, -0.885251522064209, 0.34965333342552185, 0.025859083980321884, 1.9344335794448853, -0.5309504866600037, -1.6646753549575806, -0.8853256702423096, -0.6318628787994385, -1.5664722919464111, 0.9518669247627258, 0.07425738871097565, 0.07470640540122986, -0.33626773953437805, 0.3714919090270996, -1.8586947917938232, 1.3426703214645386, 0.7857221364974976, 0.9243856072425842, -1.1715470552444458, 0.15134096145629883, -2.008882761001587, -1.9925429821014404, 1.8912769556045532, -0.30756670236587524, -1.6722570657730103, 1.0709437131881714, -0.7068649530410767, -0.3791760206222534, 0.732623279094696, -0.47651344537734985, 1.214637041091919, 0.2319381684064865, 0.21733877062797546, -0.15572944283485413, -0.4446440041065216, -0.6844009160995483, -0.8336769938468933, 0.7157617807388306, 0.3046664893627167, -1.690531849861145, -0.5141929388046265], [-0.6279045939445496, 1.6628501415252686, -0.7432134747505188, -0.3783602714538574, -0.32022884488105774, -0.9244564175605774, 0.41979461908340454, -1.1073209047317505, -0.3022196888923645, 1.3955481052398682, -0.5169890522956848, -0.12535834312438965, -1.7529770135879517, -0.7803651690483093, 0.8267956972122192, 0.47365620732307434, -0.4910582900047302, -0.2308797389268875, -1.0724270343780518, 0.317382276058197, 0.10350670665502548, -0.5603325366973877, -1.6891957521438599, -1.2809463739395142, -0.4188365042209625, -0.5810232162475586, -0.1893901824951172, -1.1182833909988403, -1.819093108177185, -1.0699598789215088, -2.9682440757751465, -0.6658071875572205, 0.807636022567749, -0.23341436684131622, -1.0948814153671265, 2.2664573192596436, -0.7005868554115295, 0.3730693757534027, 0.6684612035751343, -0.65716552734375, 1.0472699403762817, 0.8589797019958496, 2.2144484519958496, 0.29902684688568115, 0.48381537199020386, 0.3091640770435333, -0.5837397575378418, -0.3360377848148346, 2.197690725326538, 0.6600980758666992, -1.497046709060669, -1.3198035955429077, -0.3269340395927429, 0.215164452791214, -1.4944961071014404, -0.48918065428733826, -1.3160165548324585, -0.21793095767498016, -0.46274513006210327, 0.7080005407333374, -2.337639093399048, -0.2680083215236664, 2.195784330368042, 0.4769325256347656, 0.5492335557937622, -0.8568636775016785, -1.967490792274475, -1.493852972984314, 1.1616756916046143, -0.31288355588912964, 0.7197615504264832, -1.7119617462158203, 0.3697583079338074, 1.0203077793121338, 0.13527719676494598, -0.1958169788122177, -1.3259203433990479, 0.0934356227517128, 0.22838103771209717, 1.0153026580810547, 1.2289494276046753, -0.8840365409851074, 0.3534452021121979, -0.707243800163269, 0.4768940806388855, -1.1044549942016602, -0.24059079587459564, -0.33902889490127563, 1.272325038909912, -0.004284997936338186, 1.2635754346847534, -0.274840384721756, 1.0986310243606567, 0.6121232509613037, -0.11593925952911377, -0.03016742691397667], [-0.05201395973563194, 0.46859362721443176, 2.044081926345825, 0.32265400886535645, -1.1893590688705444, -0.1566428691148758, -0.025176698341965675, 0.8594664335250854, -1.6275966167449951, -0.3769434094429016, 0.4121229350566864, 0.9271199703216553, -0.3671708106994629, 0.10634112358093262, 1.9791978597640991, -0.9588684439659119, 1.0528357028961182, 0.5206893682479858, 0.44358953833580017, 0.41740313172340393, 0.9117162227630615, -0.22179196774959564, -0.18829596042633057, -0.9166949391365051, 1.1066263914108276, -0.20613406598567963, -1.2200499773025513, -0.48113560676574707, -0.3694378733634949, -1.8230689764022827, 1.121052622795105, 0.34646525979042053, 0.3315886855125427, -1.6497750282287598, 0.3361803889274597, -1.102662444114685, 0.16974912583827972, -0.9029931426048279, 0.5841989517211914, 0.7804110646247864, 0.0784507468342781, 1.6398471593856812, -0.4641357958316803, -2.067216157913208, -0.4885174632072449, -0.8705620765686035, -1.1456528902053833, 1.1905220746994019, -0.8703552484512329, -0.23480311036109924, -0.08241309970617294, 0.6856384873390198, -0.46188676357269287, -0.12125267833471298, 0.7053117156028748, -1.2074731588363647, 1.1435497999191284, -1.187959909439087, 0.2710602581501007, 0.36340367794036865, 0.6412844657897949, 1.0877004861831665, 2.1858787536621094, 0.7160791158676147, 0.7348071932792664, -0.6549115180969238, 0.2698763310909271, -0.7498493194580078, -0.48734086751937866, -0.5030022263526917, -0.17363417148590088, -0.9463402628898621, 0.22482262551784515, -0.488919734954834, 1.5020555257797241, -0.7268403768539429, -0.44346675276756287, 1.2103067636489868, -0.6876415610313416, 0.5403741002082825, -0.8014107942581177, -0.05563432350754738, -1.9085593223571777, -1.1740361452102661, -1.941710114479065, 0.013981903903186321, -0.401868999004364, -0.5601974129676819, -0.19805105030536652, 1.1618176698684692, 0.9084813594818115, 0.30781516432762146, -0.6624118089675903, 0.9735419154167175, -0.9068780541419983, -0.8198967576026917], [1.7862900495529175, -0.16515225172042847, 0.3431363105773926, -2.6145265102386475, 0.12453503906726837, 1.6674209833145142, -1.8860669136047363, 0.19521430134773254, -0.5935702323913574, -0.25518321990966797, 1.3888133764266968, 0.20762421190738678, 0.46552637219429016, 0.100124292075634, -2.0184991359710693, -0.9948804974555969, 2.2095789909362793, 0.7047573328018188, -1.0214308500289917, 1.7670649290084839, 1.0554169416427612, 0.2514609694480896, 0.1682947725057602, 0.6557000875473022, -0.32598036527633667, 0.9388819336891174, 0.4517594575881958, -0.1320050060749054, 0.5592151284217834, -0.6490228176116943, 1.1572248935699463, 0.027782369405031204, -0.9347316026687622, 1.4360324144363403, -0.5216611623764038, -0.800349771976471, -1.0953444242477417, -0.3809320330619812, -0.3454880714416504, 0.0500040277838707, 0.22025716304779053, 0.975481390953064, -0.5952023267745972, 0.03678641468286514, 0.09224629402160645, -1.9035547971725464, 0.2504534125328064, -0.9812834858894348, 0.05324632301926613, 0.07028897106647491, -0.6648560762405396, -1.9640206098556519, -1.004088282585144, 2.2529664039611816, -1.0412826538085938, 1.0263487100601196, -0.12200822681188583, 1.0846275091171265, -0.8698747754096985, 0.6236182451248169, 0.008058667182922363, -0.0274124126881361, -1.7003897428512573, 0.6741202473640442, 0.363444447517395, 0.5716289281845093, -0.6982159614562988, 0.21182478964328766, -0.36020609736442566, 0.4059554934501648, -1.2929909229278564, -1.430985689163208, -1.520788550376892, 0.7419686317443848, 1.2992891073226929, -0.5656936168670654, 1.4472209215164185, 0.9937295317649841, -0.6982289552688599, 0.6500522494316101, 1.3204302787780762, 1.3200348615646362, -1.0922601222991943, -0.8272475600242615, 0.6367392539978027, 1.377931833267212, 1.969064712524414, -0.8113541603088379, -1.1744250059127808, 1.2132718563079834, -1.6308205127716064, -0.4533792734146118, 0.23803338408470154, -1.6923165321350098, -1.120218276977539, -0.20526348054409027], [0.19757990539073944, -0.43597424030303955, 0.5902867913246155, -0.8061034083366394, -0.4013875424861908, 0.6059243083000183, 0.5887131094932556, -1.311471939086914, 2.096283435821533, -0.5726165771484375, 1.5638097524642944, 0.11530619114637375, -1.095418930053711, -1.630066990852356, -0.5930080413818359, 1.9413484334945679, -0.5616907477378845, 0.4448397755622864, 0.3906879127025604, 0.2823476493358612, -0.5388748049736023, 1.5227792263031006, 0.2906413972377777, 1.5566688776016235, 0.2548213005065918, 1.0570813417434692, 1.0468721389770508, 0.8508619070053101, -1.5719467401504517, 1.0342849493026733, 1.6548004150390625, -1.444663643836975, 1.6021363735198975, -1.6496329307556152, -0.6630157232284546, -0.9828504920005798, 0.6088881492614746, 0.22171080112457275, 0.7498743534088135, 0.10300558060407639, -0.5029668807983398, 0.5940257906913757, -0.9641515612602234, 0.3123440146446228, 0.1554117053747177, -0.4119802415370941, -0.36435607075691223, -1.234192132949829, 0.4686048626899719, 0.40290704369544983, -0.28997907042503357, 0.3047669231891632, -0.6487746238708496, 0.2551482617855072, 1.567063570022583, -2.0220470428466797, 0.28773438930511475, -0.5182284712791443, -0.26879188418388367, 0.5521560907363892, -1.5506837368011475, -0.20552141964435577, -0.7793723940849304, -0.9140578508377075, -0.8386213779449463, -1.808555245399475, -0.24995997548103333, -0.29227378964424133, 1.4473390579223633, -0.9740722179412842, 0.3027612864971161, -0.5681636333465576, 0.7508199214935303, -0.32506370544433594, -0.2871512174606323, -0.6226370334625244, -0.8161059021949768, 1.3605681657791138, -2.4599218368530273, 0.4082055985927582, 0.595725953578949, -1.0098904371261597, -0.4040449559688568, 0.9460636377334595, 1.3564786911010742, -0.480754554271698, 1.3129445314407349, 0.3145076334476471, -1.6477948427200317, -1.0008254051208496, 0.21199457347393036, -0.8701605796813965, -0.38984671235084534, 0.024559833109378815, 0.5297690033912659, 2.0037758350372314], [-0.5667341351509094, -1.0497028827667236, 0.20830486714839935, -0.18473181128501892, -0.7807669639587402, 0.0815635398030281, -1.948230266571045, 1.477319598197937, 1.0315104722976685, -0.42148903012275696, -0.5189439058303833, -0.18366460502147675, -0.5718197226524353, -0.7563415765762329, 1.1798549890518188, -1.100735068321228, 0.42272287607192993, -0.16214516758918762, 1.721588373184204, 1.2131627798080444, 0.2549281418323517, 0.040279947221279144, -1.3116397857666016, -0.12956340610980988, 0.5766858458518982, -1.4505250453948975, 0.11965876817703247, -1.463382363319397, -0.9490877985954285, 1.357965350151062, 1.1076902151107788, -1.2136738300323486, -1.028352975845337, 1.4955142736434937, -0.9011039733886719, 0.7847530245780945, -1.5163120031356812, 0.2321462780237198, -0.31234756112098694, -0.3914147615432739, -1.2773939371109009, -0.40380698442459106, 0.506599485874176, -0.4149025082588196, -0.7957959771156311, 0.016737166792154312, 1.1219431161880493, 1.1544427871704102, 1.3991419076919556, -0.18751955032348633, -1.1187587976455688, -0.7577018737792969, 0.9194114208221436, -0.8269535303115845, 1.8674484491348267, 0.8342657089233398, -0.5834754705429077, -0.33211207389831543, -0.27791866660118103, 2.0554263591766357, -2.795780658721924, -1.5324918031692505, 0.9660340547561646, -0.7094156742095947, -0.3869856297969818, -0.34588226675987244, 1.8558021783828735, -0.30031123757362366, -2.4041028022766113, 0.04826074838638306, -0.6801530122756958, -0.220456063747406, -0.2563649117946625, 0.9649374485015869, 1.4301930665969849, 1.0306402444839478, 1.0237709283828735, 1.6655176877975464, 0.6515273451805115, -0.7701049447059631, 0.9760303497314453, 0.3261389434337616, 1.684112787246704, 0.4299033284187317, 1.026389479637146, -0.9195461273193359, 0.033952876925468445, -0.04878930002450943, 0.9891034960746765, -0.07774807512760162, 0.6859171390533447, -0.9769221544265747, -0.3317824900150299, -0.06824656575918198, 0.7703296542167664, -1.7577496767044067], [1.0193893909454346, -0.11085127294063568, -3.0447895526885986, 0.7860326170921326, -0.8166168332099915, 1.4717164039611816, 2.1020052433013916, 0.31502199172973633, -0.30483996868133545, -0.6819478273391724, 0.9753008484840393, -0.7162512540817261, -0.38103359937667847, -1.5600157976150513, 0.4218350946903229, -1.6964772939682007, -0.3400215804576874, -0.24257595837116241, 0.4606928825378418, 0.4684973955154419, -0.7320452928543091, 0.4429984986782074, 0.07853472977876663, -0.5588974356651306, 0.3067416548728943, -1.0243124961853027, -1.2356008291244507, -0.6012117862701416, -0.8419360518455505, -1.081093430519104, 0.8015642762184143, -0.7467857599258423, 0.6978377103805542, -0.471113920211792, -1.000968098640442, 0.7296140193939209, 1.099386215209961, 0.5302827954292297, -0.2677222788333893, -1.509644865989685, -0.3934638798236847, 0.7876391410827637, -1.6234674453735352, -3.010364294052124, -1.3693721294403076, -0.4073706269264221, -1.0846149921417236, 0.18549473583698273, 0.9527456164360046, -0.38195616006851196, -0.13767366111278534, -0.08099234849214554, 0.5413526892662048, 0.8597134351730347, 1.2270351648330688, 0.585169792175293, -0.582355260848999, 0.12250224500894547, -0.5054716467857361, 0.8572505116462708, -2.3068907260894775, 0.045555148273706436, -1.2601734399795532, -0.8038526773452759, 0.44181832671165466, 0.5390911102294922, 0.4510105848312378, 1.4927231073379517, 1.2250208854675293, -0.9660512804985046, -0.5660885572433472, 0.026525935158133507, 0.22650162875652313, 1.5348472595214844, 0.8821831345558167, -0.8641007542610168, 1.071602463722229, 0.2121741771697998, -0.013697235845029354, -1.1378690004348755, -1.1835839748382568, 0.9212059378623962, 0.17129723727703094, 1.2282434701919556, 0.37214258313179016, 0.2453378140926361, 0.041308894753456116, 0.32109326124191284, -0.5182846188545227, -0.05159936472773552, 0.18564006686210632, -1.743804693222046, -0.4439171552658081, -0.77947598695755, 0.5089854001998901, 1.3371366262435913], [-0.35303065180778503, 0.6795167922973633, -1.30086088180542, -0.6188860535621643, 0.4172877073287964, -0.019691672176122665, -2.242832660675049, -0.282767653465271, -0.3757416009902954, 1.5724364519119263, 2.131714105606079, -3.4585776329040527, -1.2575911283493042, 0.7815226316452026, -0.6492013335227966, -1.0450469255447388, -0.22688551247119904, -0.5790548324584961, 0.3213343620300293, -0.8874518871307373, 0.8411651849746704, 0.4027147889137268, -0.44809654355049133, 0.5361310243606567, 0.6132180094718933, -1.7305272817611694, -0.49167758226394653, 0.5361326336860657, -1.4685463905334473, 0.6642612814903259, -0.27386537194252014, 0.561120867729187, 2.15043306350708, -0.8988370299339294, 0.12549051642417908, -0.0028597309719771147, 0.38439202308654785, -0.5276534557342529, 0.22078190743923187, -0.06614784896373749, -0.7011014223098755, 0.9092273712158203, -1.647939920425415, 0.3604230582714081, 0.1491110622882843, -1.3833069801330566, 0.0058711739256978035, -3.135430335998535, 0.007046212907880545, 0.9741501212120056, 0.1377106010913849, 0.4775298833847046, -0.18880105018615723, 1.9557485580444336, -1.4483156204223633, 0.21025308966636658, -0.6302441358566284, -0.009710807353258133, 0.6368236541748047, -1.003572702407837, -1.1214977502822876, -0.3611590266227722, 0.09735944122076035, -2.5091123580932617, 1.3491315841674805, 0.17646948993206024, -0.856568455696106, -0.22472406923770905, -1.4068856239318848, 1.2754302024841309, 0.6067955493927002, -0.9412782788276672, -1.7813482284545898, 0.3071501553058624, -0.497444212436676, 0.35709288716316223, 1.1731711626052856, 0.7561697959899902, -2.2738399505615234, 0.01197375450283289, 1.7914048433303833, -1.0692460536956787, -0.6856157183647156, 1.2684844732284546, -0.7504116296768188, -0.048787008970975876, 0.874568521976471, 0.5118349194526672, 0.2470596730709076, -0.13660521805286407, -0.22779855132102966, 0.8518056273460388, 0.6389352083206177, -0.3691159188747406, 0.42505142092704773, -0.08815581351518631], [-0.14357411861419678, 0.2593200206756592, 0.13795332610607147, -1.6706277132034302, -2.1486170291900635, 1.188962697982788, 0.09648267924785614, -0.1672120839357376, 0.743295431137085, 1.0601335763931274, -0.9729439616203308, -0.48462027311325073, -0.6545234322547913, -1.182112455368042, 0.8489335775375366, -0.10531666874885559, 0.3700152039527893, 0.5221086740493774, -0.2761205732822418, 1.0694489479064941, 1.0887621641159058, 0.09385716170072556, 0.3961428999900818, -0.05083194002509117, 0.06277336925268173, 0.1585947722196579, -0.5753196477890015, 1.207767367362976, -1.1659317016601562, 0.4496791958808899, 1.900596261024475, -1.3674484491348267, -0.03801451250910759, 0.5447686910629272, -0.8514190316200256, 0.02843344397842884, -0.9224952459335327, -0.18543240427970886, -1.3795478343963623, 1.1912678480148315, 1.2928870916366577, -0.00136575975921005, -1.7463077306747437, 0.4196956157684326, 0.46036380529403687, -1.2560323476791382, -0.9196177124977112, 0.5608704090118408, 1.0702930688858032, -0.8090700507164001, 0.4771040081977844, 1.8478152751922607, -1.3791513442993164, -0.9332589507102966, -1.1804956197738647, 0.13415655493736267, -0.8375223278999329, 0.6450154781341553, -1.5591744184494019, -0.6958550810813904, -0.2189057469367981, -0.6366201043128967, -1.542023777961731, 1.2666572332382202, -0.6609749794006348, -0.8186603784561157, -1.80343759059906, 0.5142561793327332, 0.7657366394996643, -0.01975322887301445, -0.21516935527324677, -1.529727816581726, -1.0816770792007446, -0.44677308201789856, 0.5984291434288025, -0.07047832757234573, -0.22073359787464142, 0.45810672640800476, 0.9010764360427856, 1.4345873594284058, 0.5084935426712036, -0.4483173191547394, 0.5074876546859741, 0.8681607842445374, -0.9301890134811401, -0.2484906166791916, 2.234375238418579, 2.184565305709839, 0.5057932734489441, -0.4123937785625458, 0.40708479285240173, 0.5901316404342651, -0.060596246272325516, -0.07416938245296478, -0.4147847890853882, -0.7313308119773865], [-0.9069517254829407, -0.4221491515636444, -0.782326340675354, -1.4874634742736816, 0.5278586745262146, -0.09809096902608871, 1.819502592086792, -0.773224949836731, 0.5293527245521545, 0.581737220287323, 0.28993454575538635, 1.2932112216949463, -1.4152880907058716, -0.07548883557319641, -0.8306808471679688, 0.11299218982458115, 0.10706431418657303, 0.157209575176239, -0.8547773957252502, 0.8168294429779053, 0.799977719783783, -0.14234763383865356, 0.6741513609886169, 1.1666173934936523, -0.6660174131393433, 1.3144478797912598, 0.04720839485526085, 0.8462415933609009, -1.5444282293319702, 0.7884528040885925, 0.6746900081634521, -0.836311399936676, 1.2915267944335938, -0.49616754055023193, 0.4860111474990845, -0.1463685929775238, -0.5705170631408691, -0.027959411963820457, 0.26601454615592957, -0.32725989818573, 1.0627537965774536, -1.2305874824523926, 0.11844469606876373, 0.3890112340450287, -1.0594565868377686, 1.3924837112426758, -2.059704542160034, 0.48966044187545776, 1.2661914825439453, 2.8952314853668213, 1.1107255220413208, -1.8615564107894897, -0.004165025893598795, 0.3384978175163269, -0.18804261088371277, 1.0722326040267944, -1.716792345046997, -1.547706961631775, 0.37860360741615295, 0.5176995396614075, 0.8837685585021973, 0.40438899397850037, -0.13458512723445892, -0.8808053731918335, -0.9534722566604614, 1.3976465463638306, 1.1073927879333496, 0.8635351657867432, -0.14090238511562347, 1.2138526439666748, 0.4834122657775879, -0.16493067145347595, 0.341194212436676, -2.573960065841675, -0.9415878653526306, -0.21361050009727478, 0.3285503387451172, -0.23177872598171234, -1.0969434976577759, 0.812777578830719, 1.2488267421722412, 0.0736398696899414, 1.7246532440185547, -0.17279919981956482, -0.5646412372589111, -0.03049515187740326, -0.15331877768039703, 2.1581437587738037, 1.7900609970092773, -0.6521338820457458, 0.4759838581085205, -1.332789421081543, 0.8834125995635986, 0.669013500213623, -0.15627911686897278, 0.7827135920524597], [0.3817436695098877, 0.9994505643844604, -0.5355928540229797, 0.6999657154083252, 0.7858275175094604, -1.66254460811615, 1.483963966369629, -0.7813084125518799, -0.07480692118406296, -2.1780028343200684, -0.496383935213089, 0.8912547826766968, 1.240716814994812, -0.11307484656572342, 1.1209192276000977, -0.14254437386989594, -0.9644230604171753, 0.466329425573349, 0.2888270914554596, 0.40217921137809753, -1.3828010559082031, 0.1653684377670288, 0.15249036252498627, 1.6012932062149048, 0.014461585320532322, -0.07342445850372314, 0.9765841364860535, 1.4181180000305176, 0.9308189749717712, 1.0821064710617065, 0.9841504096984863, -0.2134489268064499, -1.2638709545135498, -0.000102797886938788, -0.6020558476448059, -0.6911662220954895, -0.36921533942222595, -0.5316734313964844, -0.7268494367599487, -0.30842679738998413, -0.26645627617836, -1.2160046100616455, -0.903369665145874, -0.5305222868919373, -0.4912753999233246, -0.47329476475715637, -0.7158094048500061, 0.8391144871711731, 2.0621252059936523, 0.38777226209640503, 0.46286842226982117, -0.8474038243293762, -0.6271855235099792, 1.3426264524459839, -0.447915256023407, 1.0474717617034912, 0.054177768528461456, 0.27439063787460327, 0.007678018417209387, -0.6751158237457275, 0.5639544725418091, -0.6087969541549683, 1.3369420766830444, 0.12511660158634186, -0.6312776803970337, 0.25975120067596436, -0.2898254692554474, -0.5801841616630554, 0.5456026196479797, 1.1416765451431274, -1.2403432130813599, -0.3945172131061554, 1.6102322340011597, 0.368282675743103, 0.28753799200057983, -1.21392822265625, -0.07800678163766861, -0.3247714042663574, -0.7964730262756348, 0.06926785409450531, 0.03890419378876686, -1.3177944421768188, -0.32381635904312134, 0.4970547556877136, -0.44389739632606506, -1.4046213626861572, 1.302522897720337, -0.10936732590198517, 0.05862267687916756, -1.1965540647506714, 0.15232156217098236, -0.9658739566802979, -0.24613207578659058, 1.1673020124435425, 0.9292761087417603, 0.5435423254966736], [-0.962215006351471, 0.9584687352180481, -0.6645184755325317, 0.20324063301086426, 0.6162398457527161, 0.3220140039920807, 0.6228219270706177, -0.8335665464401245, -0.3088656961917877, 0.7972568273544312, 0.7461130023002625, -0.274189829826355, 0.6988563537597656, -1.6834510564804077, -0.8302759528160095, 0.13415351510047913, 0.5053418874740601, -0.06599289178848267, 1.4039140939712524, -2.0978000164031982, 1.0653631687164307, -0.9111176133155823, -0.014561320655047894, -0.30641257762908936, -1.1007764339447021, -1.0959975719451904, 1.7422915697097778, -2.084108591079712, -1.3710166215896606, 1.476579189300537, -0.2666740417480469, -0.8101701736450195, 0.24097296595573425, 0.42078080773353577, 1.4267429113388062, -0.4369831383228302, -0.1698029637336731, 0.9675244092941284, 0.15685045719146729, 0.9513485431671143, -1.639853835105896, -1.0968585014343262, 0.13319522142410278, -1.4112070798873901, 1.5770468711853027, -0.8744257092475891, 0.393404096364975, -0.6566958427429199, -0.05535572022199631, -0.00831593107432127, 0.49599596858024597, 0.8179537653923035, -0.4613282382488251, 0.9274072647094727, 0.26177823543548584, 1.9273622035980225, 0.4153956472873688, -1.6770426034927368, 0.748552143573761, -0.3441888391971588, 0.08819817006587982, 0.511333167552948, -0.3371727764606476, 1.305959939956665, -1.270375370979309, -0.4628431499004364, 0.6246037483215332, -0.05041103437542915, -0.7420594692230225, -0.0253619235008955, -0.28614431619644165, -0.3271730840206146, -0.05519193783402443, -0.03347672522068024, -0.6567130088806152, -2.514286518096924, -0.5846889615058899, 0.6553257703781128, -0.24109072983264923, -2.3960886001586914, -0.5118787288665771, -0.14936508238315582, -0.1549435704946518, 0.691512942314148, 1.426074743270874, -0.50022292137146, -0.14913132786750793, -1.5275112390518188, -1.8551249504089355, 0.3326970934867859, -0.9374268054962158, -0.8147692084312439, -0.37246114015579224, -0.11163213849067688, -0.020743969827890396, -1.0353556871414185], [-0.7116543650627136, 0.9261255264282227, 0.7659499049186707, -0.7757275104522705, 0.6917950510978699, -0.20220713317394257, 1.440359354019165, 0.4456760585308075, 0.4219350814819336, 0.8053045868873596, -0.025565749034285545, 0.18392542004585266, 0.17923228442668915, 0.8640058040618896, -1.0803345441818237, 1.6042810678482056, -0.19305945932865143, -1.1650564670562744, -0.2801208794116974, 0.1599956601858139, -0.7864968180656433, -0.13358986377716064, 1.0795680284500122, 1.5412405729293823, -0.013161447830498219, 0.38145995140075684, 1.449789047241211, 1.7803303003311157, 1.4078004360198975, -0.7117423415184021, 0.864936888217926, -0.21163585782051086, 0.13619893789291382, -0.8762661814689636, 1.1446254253387451, 0.5836678743362427, -1.062080979347229, 0.31873729825019836, -1.1323410272598267, -0.8638111352920532, -0.47319501638412476, 0.4160996377468109, -0.7513434886932373, 0.4429531991481781, -0.10448814928531647, 0.4897952973842621, 0.8109467625617981, 0.2079884111881256, -0.09356603771448135, 0.611203670501709, 0.5687808990478516, 0.7773984670639038, -0.5433752536773682, -0.3202870786190033, -1.538694977760315, -0.4214645326137543, -0.011314568109810352, 0.02201228030025959, 0.858564555644989, -0.5958220362663269, 0.05771017447113991, 0.5189977288246155, 0.2306043654680252, 0.6382242441177368, 0.7859182357788086, 1.1952431201934814, -0.660099446773529, -0.3086927831172943, -0.6928316950798035, -0.6966468095779419, -0.24432769417762756, -1.56099534034729, -0.5980384945869446, 0.6422666311264038, -1.4451963901519775, 0.37846869230270386, 1.0995053052902222, 3.071611166000366, 0.6997044682502747, 0.31445735692977905, 1.1270432472229004, -0.29238393902778625, 0.764668881893158, 0.31621620059013367, 0.6207066774368286, 0.42396214604377747, -2.1084132194519043, -1.975271224975586, -0.8768113851547241, 0.6265775561332703, -1.9587624073028564, -1.39266037940979, -0.20568320155143738, -0.1664123386144638, -0.680348813533783, 0.26824355125427246], [1.1501117944717407, -0.8495049476623535, -1.056780219078064, -2.175104856491089, -0.936796247959137, -1.0771538019180298, -1.1564449071884155, -1.7564449310302734, -0.847284734249115, -0.4508466124534607, 1.0386831760406494, -0.47023746371269226, -0.6514528393745422, 0.8654021620750427, -0.5172958970069885, -0.273419588804245, 0.3413431644439697, 0.09200538694858551, -1.8023486137390137, -0.03693443164229393, 0.7406934499740601, -1.360094666481018, 0.8649935126304626, -0.6704156994819641, -0.7922489643096924, -1.9774699211120605, 0.12068530172109604, 0.23923633992671967, 1.63359797000885, -0.8326672315597534, 0.1033831238746643, 0.7600420713424683, -0.3807853162288666, -0.38092121481895447, 0.7875370383262634, 1.928481101989746, -1.5113320350646973, -0.5858249664306641, -0.6137811541557312, 2.393562078475952, 0.11702807992696762, 0.5263461470603943, -1.1496487855911255, 1.3856921195983887, -0.022421598434448242, 0.3469158709049225, 0.3143947124481201, -1.3442790508270264, -1.6737381219863892, 0.39630240201950073, 2.006495237350464, 1.000561237335205, 0.01983688399195671, -0.9339159727096558, -1.438434362411499, 0.12320872396230698, 0.9027498364448547, -0.8989530205726624, -0.2487732172012329, -0.07474355399608612, 0.34407925605773926, -0.5249194502830505, -2.592921733856201, -0.5825926661491394, -0.7917681932449341, -0.6089293360710144, -0.23908743262290955, 1.3042610883712769, 0.6830242276191711, 0.5367076992988586, -0.4667334258556366, 1.0124149322509766, -1.2603120803833008, 1.0523234605789185, 0.7258375883102417, 0.6377598643302917, 0.4882049560546875, -0.24228093028068542, -1.6323022842407227, -0.685036838054657, -0.4276110827922821, 0.05126947537064552, -3.6084177494049072, 0.43774181604385376, -0.8868677616119385, -0.7787778973579407, -0.1767183095216751, 0.26246708631515503, 0.41549405455589294, -0.0416744127869606, 0.8241162300109863, -0.3066504895687103, -0.6759112477302551, 0.3676096200942993, 0.2639060914516449, -0.3398543894290924], [1.2063133716583252, -0.18257050216197968, -1.5293927192687988, -0.5935338735580444, -0.17705535888671875, 1.1380671262741089, 0.044655121862888336, 1.5023226737976074, -0.6413580179214478, -0.9340515732765198, 0.21444310247898102, 0.1560138612985611, 0.23673412203788757, 1.4190773963928223, 0.7003107070922852, -0.27507641911506653, 0.14577706158161163, -0.823479413986206, -0.3130197525024414, 1.3306357860565186, 0.05316660925745964, -0.1714491844177246, 0.8850457072257996, -0.7000218629837036, 0.3657872676849365, -0.8786322474479675, -0.48258307576179504, -0.885300874710083, 0.15591642260551453, -0.42221248149871826, -1.0914344787597656, 0.2696686089038849, -0.08055590093135834, 1.3912200927734375, 0.3442051112651825, -0.5359601378440857, 1.3607791662216187, -0.7761748433113098, -0.027745164930820465, 0.6483660340309143, 0.12045587599277496, 0.5808935761451721, 0.9516317844390869, 1.4082467555999756, 0.7814961075782776, -0.006050082854926586, 0.2126084715127945, -0.3208593726158142, 0.09569092094898224, 0.34866446256637573, -1.3174329996109009, 0.8646571040153503, 1.5594182014465332, 0.15763571858406067, 0.6295494437217712, -1.890397548675537, -1.8111720085144043, 1.7416324615478516, -0.5550214052200317, 2.4675772190093994, -1.653546690940857, -0.7833368182182312, -0.526477038860321, 1.011724829673767, 0.7815331816673279, 1.4499539136886597, 2.0375399589538574, -0.33776143193244934, -0.10673090815544128, 0.7279458045959473, 0.5164808034896851, -2.309231758117676, 0.7337759733200073, 0.29661229252815247, 0.9557740688323975, -0.6164835095405579, 0.11380364000797272, 0.6782709956169128, 0.7659735679626465, 0.9374932050704956, 0.4147920310497284, -0.14390678703784943, 0.42554521560668945, -0.6733494400978088, -1.3809117078781128, 1.4602220058441162, 0.22113066911697388, -0.17340464890003204, 1.002007007598877, -2.2380216121673584, -0.461833119392395, 0.843438982963562, -1.1689956188201904, -0.19299426674842834, 0.036139920353889465, -0.445611834526062], [0.38311994075775146, -0.3263680338859558, 1.1463189125061035, -0.4156348705291748, 0.3856383264064789, -0.5088514685630798, -1.3213953971862793, -1.6337449550628662, 0.05821079760789871, 0.18962158262729645, -0.1834496706724167, 0.1393369734287262, 0.40434640645980835, -0.050487615168094635, 0.6895710825920105, -0.7923316359519958, 1.00875723361969, 0.6412773728370667, 0.5129771828651428, -0.42160695791244507, 0.9734413623809814, -1.6512795686721802, -0.4790085256099701, -1.2892701625823975, 0.30455976724624634, 0.6688705682754517, 0.2335144430398941, 1.169468879699707, -0.6445558667182922, 0.520584762096405, 1.570959210395813, -1.3829594850540161, -0.5079212188720703, 1.0593541860580444, 1.3456883430480957, 1.6150870323181152, -1.4853239059448242, -0.7701627016067505, 0.34535953402519226, 1.9578697681427002, 1.675215482711792, -0.14149810373783112, 0.5372909307479858, -1.0282331705093384, 0.567091166973114, 0.5498799681663513, -0.5023356080055237, 0.1999998688697815, 0.19547845423221588, -2.272923231124878, 1.9048219919204712, -0.52579265832901, 1.114367961883545, -0.2935672998428345, 0.5569836497306824, 0.4476083517074585, 0.8014020323753357, 0.9093532562255859, -0.26373961567878723, -0.2663421332836151, -0.9302496910095215, 0.39553529024124146, -0.6840366125106812, 0.9838014245033264, -1.2099509239196777, -0.5775054693222046, 0.47067299485206604, 1.148979902267456, 0.18038281798362732, -1.5532366037368774, -0.30082452297210693, -0.619792640209198, -0.8821921944618225, 0.3924790024757385, -0.3553507328033447, 0.7976049184799194, -1.9106017351150513, 1.975823163986206, 0.7529715895652771, 0.3451124131679535, 0.6952996850013733, -0.42027854919433594, 1.5594134330749512, 0.6763558387756348, -2.2455461025238037, 0.18061456084251404, 0.39645326137542725, 0.3734230399131775, 0.8912795186042786, -1.454939842224121, -0.20781178772449493, -0.2203030288219452, 0.0923352986574173, 0.08686453104019165, -0.6850491166114807, -0.013092181645333767], [-2.251725673675537, -0.5304121971130371, 0.16790513694286346, 0.9979333877563477, -0.08500059694051743, 1.6477375030517578, 0.8191831707954407, -0.6186138391494751, 0.37520337104797363, -1.2089171409606934, -0.9164575338363647, 0.21543709933757782, 1.2855958938598633, 0.5951708555221558, -1.3070971965789795, -0.06683340668678284, 1.0673134326934814, -0.8341367840766907, -0.28387120366096497, 0.6292434930801392, -0.09448767453432083, 0.3593195974826813, -0.5726274251937866, 2.3339731693267822, -0.42173323035240173, 0.4309736490249634, 0.1808786690235138, -0.37215399742126465, 0.7917206287384033, 0.8690701127052307, 0.3658207654953003, 0.017142336815595627, -2.6947829723358154, -0.7413122057914734, 0.2836553156375885, 0.04904910922050476, 0.6438093185424805, -0.8270589113235474, -1.3438481092453003, 0.9225441217422485, -0.6304686069488525, 0.6484363079071045, 1.5543639659881592, -0.18270038068294525, -0.041272345930337906, 0.9311956167221069, 0.43373844027519226, 0.4199848175048828, -1.6214323043823242, 0.4605039656162262, -1.12044358253479, -0.5010558366775513, -0.730410099029541, -0.26492199301719666, -1.848785161972046, -0.4369930922985077, 1.1011390686035156, -0.25517815351486206, -1.7793980836868286, -0.24531038105487823, -0.5667499303817749, -0.7707213163375854, -0.9739289879798889, -0.669117271900177, 0.6509954929351807, -0.7677711248397827, 1.6378810405731201, 0.21803589165210724, -2.0148048400878906, -1.4725894927978516, 1.0573689937591553, -1.496757984161377, -1.5739511251449585, 2.6317005157470703, 1.751849889755249, -0.40057241916656494, 1.6504629850387573, -0.7013322710990906, 1.7457448244094849, -0.1854398548603058, 0.3553764224052429, 1.00471031665802, 0.6702052354812622, 0.03906994313001633, -0.40140190720558167, -1.2946792840957642, 0.9340654611587524, -1.3362141847610474, -1.3566436767578125, -0.07266481965780258, -0.9524196982383728, -0.026570262387394905, 1.5841199159622192, -2.2508342266082764, -0.5137391686439514, -0.23553819954395294], [-0.8838974237442017, 0.25057080388069153, 1.3047959804534912, -1.2358850240707397, 0.5652929544448853, -0.6948463320732117, 0.7218424677848816, 0.3157176971435547, 1.161043643951416, 0.4041418731212616, 0.9808416962623596, -1.7628002166748047, 0.7528685331344604, 0.15244384109973907, 0.5327108502388, 1.415444016456604, 0.11309932172298431, -0.06344231218099594, 0.18435442447662354, 0.3316571116447449, -1.6153801679611206, -0.3256738781929016, -0.7338551878929138, 0.33461466431617737, 0.9850144982337952, -1.839236855506897, 0.2811789810657501, -0.3239779770374298, -0.2507355511188507, -0.6989415287971497, -1.0108087062835693, -0.448892205953598, -1.149340271949768, 0.3875041604042053, -1.1536226272583008, -0.00816455390304327, 1.3615881204605103, 0.062421008944511414, 1.7683161497116089, -1.0470978021621704, 0.7018139362335205, -0.4530915319919586, 1.6438896656036377, 0.11595416814088821, 0.453133225440979, 0.026291027665138245, -0.32862260937690735, -1.0614145994186401, -0.9033656716346741, -0.05271304026246071, 1.6556130647659302, -0.4935513734817505, 0.20147168636322021, 0.0008848276920616627, 0.28261592984199524, -0.2799413502216339, 0.3922511041164398, 0.946479320526123, -0.43948543071746826, -0.5997015237808228, -1.524779200553894, 0.2545756697654724, 1.100687026977539, -1.1243386268615723, 0.507604718208313, -1.015286922454834, 1.9302430152893066, 0.43394753336906433, -0.20418709516525269, 1.6218868494033813, 1.5849347114562988, 0.27583205699920654, -0.8148980140686035, -1.062360167503357, 1.1513715982437134, 1.39816153049469, -0.5046792030334473, 0.5734890103340149, 0.963456392288208, -0.25850191712379456, -0.05201397091150284, 1.7323216199874878, 0.5848047733306885, -1.1871943473815918, -0.04812905192375183, -1.57721745967865, 0.5504494905471802, -0.4706505239009857, -0.5635346174240112, -0.4245762526988983, 0.06091871112585068, -1.544708251953125, 0.13171043992042542, -0.8204823136329651, -1.9538886547088623, -0.007212894968688488], [-0.9435106515884399, 0.2267240285873413, -1.4932984113693237, 1.6944406032562256, -1.0480166673660278, -0.2115667164325714, -0.45908546447753906, -1.3656178712844849, 0.18475976586341858, -0.273106187582016, -0.48150667548179626, 1.1963567733764648, -1.1654890775680542, -1.5813813209533691, -0.8157323002815247, 0.5132503509521484, 1.2800167798995972, 0.31548362970352173, 0.08374860882759094, -0.09128350764513016, 0.23265543580055237, 0.89250248670578, -2.8723645210266113, -0.8594179749488831, 0.05422728881239891, -0.30430305004119873, 2.1719443798065186, -1.0518323183059692, -0.7256535887718201, 1.0882279872894287, 0.49151745438575745, 0.8671954274177551, 1.536389946937561, 1.2836309671401978, -0.478039026260376, 1.4281320571899414, 2.0501155853271484, 0.037846874445676804, 0.5919142365455627, -1.1258724927902222, -1.1354610919952393, 0.6124895811080933, -0.6858066916465759, 0.39794954657554626, -0.14889150857925415, 0.6249262690544128, 0.7679793238639832, -1.2644999027252197, -1.4186303615570068, 0.27734601497650146, 0.3425191640853882, 0.020185744389891624, -0.14144080877304077, -0.6280799508094788, -0.08860397338867188, -0.6345564723014832, 0.5682106018066406, -0.05277588218450546, -0.018542105332016945, 2.2000582218170166, -0.4042501151561737, -1.5184451341629028, -0.04051589220762253, -0.538057804107666, -0.6199215650558472, -0.44607117772102356, -0.3611852526664734, -0.15702100098133087, -0.3616955280303955, -1.1742148399353027, -0.38358980417251587, 0.19682534039020538, 0.6305310726165771, 1.013053297996521, -0.6785938739776611, 0.3985733389854431, 0.5455343723297119, -0.5522795915603638, -0.7524938583374023, 1.1796317100524902, -0.7367955446243286, 1.0227504968643188, -0.1237124353647232, -0.5231980681419373, 0.8526924848556519, 1.3461169004440308, 0.2891434133052826, 0.20531779527664185, -1.995587944984436, -0.0563388466835022, 1.2065397500991821, -0.6961758136749268, 1.4617443084716797, 0.5925524234771729, -1.472674012184143, -0.3856077194213867], [0.04373819753527641, 0.2632509768009186, 0.3094489872455597, -0.1776198297739029, -0.7615205645561218, -0.08858296275138855, 0.3116486370563507, -0.12724314630031586, 1.2389031648635864, -0.27638015151023865, 0.29814913868904114, -1.298644781112671, -0.4040869474411011, -2.4128258228302, 2.9940028190612793, 1.1058475971221924, 0.7859205603599548, 0.4713252782821655, 0.41491758823394775, -0.029064973816275597, 0.7691761255264282, -0.11179250478744507, -0.2897246778011322, 0.4859531819820404, 0.5182814598083496, 0.648984432220459, -0.5574877858161926, 1.1740000247955322, -0.3224339783191681, -0.5111392736434937, 0.7427508234977722, -1.0429697036743164, 0.19321861863136292, -0.5279240012168884, -0.8483585119247437, 1.3440399169921875, 0.35698992013931274, 1.5544512271881104, -3.0492310523986816, -1.6650028228759766, 0.9213014841079712, 1.3203250169754028, 0.12474025040864944, -1.8574062585830688, 0.19299988448619843, 0.33256998658180237, 0.14420369267463684, -0.5240490436553955, -0.46186497807502747, 0.4055953323841095, 2.217477321624756, -0.8823539018630981, -0.3351759910583496, 0.6980709433555603, 0.7618764042854309, 1.0931488275527954, 0.9097055792808533, -0.05029686167836189, -0.6851878762245178, -0.4864301383495331, -0.3066488206386566, -1.4450445175170898, -0.30464547872543335, -0.5368209481239319, 0.19383804500102997, -0.21535615622997284, 1.0546659231185913, 1.9257558584213257, -0.6148634552955627, -1.6611626148223877, -0.1969919055700302, -1.0677014589309692, 0.11510463804006577, 1.718638300895691, -0.1478148251771927, 1.3667200803756714, -0.23022355139255524, 0.7322241067886353, -0.31656786799430847, 0.16171090304851532, -0.2658891975879669, -0.5833511352539062, -0.7402155995368958, 0.8034711480140686, -1.8815006017684937, 0.460496723651886, -0.6504440307617188, 1.422794222831726, 1.1457428932189941, 0.7145237922668457, -2.239293098449707, -0.9386631846427917, 1.5062587261199951, 0.3356887400150299, -1.5905468463897705, -1.2319607734680176], [-0.5377016067504883, 0.8255385160446167, 0.8676725625991821, -1.5619608163833618, 0.3171754777431488, -1.7135401964187622, -0.7150855660438538, -0.9660124778747559, 0.4768761992454529, 1.1605395078659058, -0.08396096527576447, -0.18034207820892334, -0.5433163046836853, 0.7018530964851379, 1.0612258911132812, 1.0917788743972778, 1.4215857982635498, 1.2958080768585205, -0.45310458540916443, 1.3890453577041626, 0.8282637000083923, -1.0576248168945312, 1.1530603170394897, 2.5015625953674316, -1.437030553817749, -0.16616354882717133, -0.5282045602798462, 2.415419340133667, 0.28833404183387756, 1.4568558931350708, -1.4074132442474365, -0.10636068135499954, 0.37431401014328003, -2.3209176063537598, 0.8968703746795654, -0.9090180397033691, -0.6903402805328369, -1.1403083801269531, 2.5356690883636475, 1.9642596244812012, -0.5283979773521423, 1.5969122648239136, 1.8982843160629272, -0.20542079210281372, 0.16314290463924408, 0.10168953239917755, -1.5225605964660645, 0.8088787794113159, -1.8330706357955933, 1.7831321954727173, 0.5159493088722229, 0.11562082916498184, 0.03305093199014664, -0.7351472973823547, -0.34855103492736816, 1.624650001525879, 1.1300699710845947, 1.5387789011001587, -0.28776952624320984, 0.6545646786689758, 1.167357087135315, -0.5820440053939819, 0.24418018758296967, -0.6418851613998413, 1.1414258480072021, -1.4563586711883545, -0.468501478433609, 0.009009058587253094, -0.3499367833137512, -0.3707493245601654, 0.6361979842185974, 0.886692225933075, 0.33126917481422424, -0.7834882736206055, -0.5381639003753662, 0.8531529307365417, -0.37190383672714233, -0.49860522150993347, -0.454604834318161, 0.18329347670078278, 0.37704703211784363, 1.8082551956176758, -0.12115342915058136, 0.4806080758571625, 0.9881883859634399, -0.8837894797325134, 1.2782973051071167, -1.0767525434494019, 0.9887375235557556, 0.5709531903266907, 0.07195114344358444, 0.1695416420698166, 0.40097469091415405, -1.847435712814331, -1.3539198637008667, -0.1357363611459732], [-0.8103162050247192, -0.5342683792114258, -1.1534643173217773, -1.3460338115692139, -0.08870598673820496, 0.0498206689953804, -0.6386463046073914, 0.6623319387435913, 0.6556603908538818, 0.7859649658203125, -0.2093518078327179, -0.5982308387756348, -0.9751598834991455, -0.5675485730171204, 0.3814844489097595, -0.04039205610752106, 1.960060477256775, -0.20323807001113892, -0.17433542013168335, -0.38637012243270874, -0.006088975351303816, -1.0792495012283325, -0.4383830726146698, -0.3423057198524475, 0.6779482960700989, 1.167412281036377, 0.18478479981422424, -1.8143408298492432, -0.2425355315208435, 1.3856513500213623, -0.10336332768201828, 0.03650211915373802, -1.9283134937286377, -1.146801471710205, -2.2871129512786865, 0.08661246299743652, 1.1137913465499878, 0.9322709441184998, -1.3764348030090332, -0.11416685581207275, 0.6951842904090881, 0.5628137588500977, 0.5774708986282349, 0.5900641679763794, 0.10364416241645813, -1.3811460733413696, 0.008377375081181526, -0.13140766322612762, 1.038960576057434, 1.9032065868377686, 0.7537388801574707, -0.23822109401226044, -0.1310311108827591, -0.4516051709651947, -1.5645434856414795, -0.7174042463302612, -1.658233880996704, -0.1952817589044571, -1.8759757280349731, 0.425491601228714, 0.2936440110206604, -0.9835094213485718, -2.1815922260284424, 1.1709275245666504, -3.2744901180267334, 0.8871188759803772, 1.8130801916122437, -1.0146641731262207, -0.047498710453510284, -0.18388721346855164, -0.7743898034095764, 0.2573585510253906, -2.2628026008605957, -0.6825293302536011, 1.5557061433792114, 0.4231564402580261, 2.569385528564453, 0.31920287013053894, 1.3115140199661255, 2.4504504203796387, 0.7226472496986389, 0.24139641225337982, 0.5499482750892639, 0.10208582878112793, 0.0024008448235690594, -0.9552030563354492, 1.9565874338150024, -0.928320050239563, 0.14327402412891388, -0.2308894693851471, -1.782402515411377, 0.45689520239830017, 0.8210524916648865, -0.47212278842926025, 0.8025510907173157, 0.9159857034683228], [1.0566034317016602, 0.1701750010251999, 0.44698822498321533, 0.36071693897247314, -0.8522779941558838, -0.888911247253418, -0.9937688112258911, 0.5900774002075195, 0.7385435700416565, 1.2984585762023926, 0.7790795564651489, -0.3461354374885559, 1.8146183490753174, -0.7056974172592163, 2.3213326930999756, -0.35754600167274475, -0.20378543436527252, -0.2553566098213196, -0.4717473089694977, -0.06928816437721252, 0.14673367142677307, -1.395706057548523, -0.08042984455823898, -0.4650433659553528, -0.26246362924575806, -0.49730587005615234, -0.014153599739074707, 0.41796639561653137, -0.21769045293331146, -0.6328072547912598, 1.968103289604187, 1.3253215551376343, -1.009203314781189, 0.5730755925178528, -0.42096081376075745, -1.2624765634536743, 0.1671493947505951, 0.6561166048049927, -0.7028771638870239, 0.3693476915359497, 0.763359010219574, 0.04569574445486069, 0.7228307723999023, 0.6818366646766663, -0.4191884696483612, 0.8625044822692871, 1.5672436952590942, -0.10620435327291489, -0.3578444719314575, -0.16089782118797302, -1.9883325099945068, 1.142279863357544, 0.8919044733047485, 0.3751540780067444, -2.2770726680755615, -0.6204289793968201, -0.6448021531105042, 0.8775901198387146, 1.484671711921692, -1.4587393999099731, -0.14456702768802643, 0.5081683397293091, 1.6599953174591064, 0.5587791204452515, -0.0718345195055008, 0.5257080793380737, -1.0886191129684448, -0.2865389287471771, -0.6696028113365173, 0.21244291961193085, -0.47365039587020874, 0.04946425184607506, -1.0695358514785767, 0.03472991660237312, 1.3061721324920654, -1.4679622650146484, 0.482686847448349, 0.49459704756736755, 0.6858492493629456, 1.0202778577804565, -0.6390907764434814, 1.254793643951416, -0.3387907147407532, 1.2007012367248535, -0.47894394397735596, -1.7259423732757568, -0.13836076855659485, 0.02320178784430027, 0.1844109296798706, 1.8283330202102661, 2.374171733856201, 1.269105076789856, -0.8498430252075195, 1.8819301128387451, 1.2404797077178955, 0.993632972240448], [1.5485867261886597, -0.33335641026496887, -0.1853480339050293, 0.5693660378456116, -0.9024481177330017, 0.34941115975379944, 0.9463838338851929, 0.39440301060676575, -0.9143736362457275, -0.1963748335838318, -0.1464744210243225, -1.114047646522522, -1.5120623111724854, 0.2257828414440155, 0.10310575366020203, 1.055733323097229, 0.26936742663383484, 1.3226221799850464, 0.8577879667282104, -1.226146936416626, -0.550774335861206, -0.15415175259113312, 0.46982601284980774, 0.15661780536174774, 0.5784059166908264, -1.404506802558899, 0.9541361927986145, 0.3223188817501068, 0.5717951059341431, 0.2612464129924774, 1.567609429359436, -1.2164231538772583, 0.9461120963096619, 1.117011547088623, -0.706851601600647, 0.9505046010017395, 1.4137952327728271, 0.18586918711662292, -0.9430649280548096, -0.17724238336086273, 0.07906132191419601, 0.2539098858833313, -0.5720933675765991, 0.967413067817688, -1.1440672874450684, -0.531126856803894, 0.9045716524124146, -0.14972171187400818, -1.3955299854278564, -0.2716272473335266, -0.6867768168449402, 0.5843239426612854, 1.1766127347946167, 0.33783286809921265, -0.39502477645874023, -0.8807882070541382, 0.11579450964927673, -0.7814978361129761, 0.8871289491653442, -0.11960747092962265, 1.268835186958313, -1.0331093072891235, 0.630230724811554, 0.7249488234519958, 1.8438050746917725, 0.2748256325721741, -0.5072131156921387, 0.2128370851278305, 0.08700411021709442, 0.7304851412773132, -1.320923089981079, 0.7721738219261169, 0.35745757818222046, 1.1238406896591187, -0.09161899238824844, 0.16591013967990875, -1.2021650075912476, -1.5463001728057861, 0.3682023882865906, -0.17876850068569183, -0.6488157510757446, -1.930001139640808, -0.3327190577983856, -0.6480181813240051, 0.026000220328569412, 0.7712462544441223, 1.7408109903335571, 0.9280327558517456, -1.1696548461914062, -2.002319574356079, 1.9733268022537231, 1.3691978454589844, -0.5230116844177246, 1.707319736480713, -1.1433802843093872, -0.217844158411026], [-1.9478328227996826, -0.17699690163135529, -0.9985218644142151, 0.6349549293518066, 0.7205275297164917, -0.5870364904403687, 0.4410703182220459, 0.3121061325073242, -0.0194545891135931, 0.5340055227279663, -0.9314376711845398, -0.5095782279968262, 0.26830631494522095, 1.385500192642212, -1.1150026321411133, 0.3847883343696594, 1.412628412246704, 0.13347211480140686, -2.012850046157837, -0.8445324897766113, -0.48673033714294434, -0.6505051255226135, -0.5923654437065125, -0.11362773180007935, -0.879462480545044, 1.0842868089675903, -0.6042198538780212, 0.8120744228363037, -0.3376082479953766, 1.217504620552063, 0.05363258346915245, -1.6836390495300293, 2.03800892829895, -0.6061850190162659, 0.4635803997516632, 1.4204097986221313, 0.16939249634742737, -0.8881286978721619, -2.27596116065979, -1.2604596614837646, 1.0816200971603394, 0.5460999011993408, 1.4426847696304321, 0.7883576154708862, -1.4609297513961792, -0.7029662132263184, -0.2682291269302368, 0.22999627888202667, 0.005383598152548075, 0.05438517779111862, 2.1354312896728516, -1.813987374305725, 1.2436463832855225, 1.7504198551177979, 0.7249513864517212, 0.019786907359957695, -0.2157789021730423, 0.15400545299053192, 0.4060143232345581, 0.30977797508239746, -1.363065481185913, 0.8140509724617004, 0.17753884196281433, -1.5299268960952759, -0.2401251196861267, -0.3137999176979065, -0.9372947216033936, -0.17545217275619507, 1.8397958278656006, 1.7164140939712524, 0.8504317998886108, -1.1502727270126343, -0.2725651264190674, 0.17966049909591675, 0.14129838347434998, -0.6524964570999146, -1.2929248809814453, 0.3998486399650574, 0.13127431273460388, 0.33548104763031006, 0.12715086340904236, 0.4658159911632538, -0.03695795312523842, 0.4658903181552887, -1.1782422065734863, -1.469955563545227, -0.3705461323261261, -1.0484764575958252, -1.2785117626190186, 1.1922500133514404, -0.4658333659172058, 0.4848025143146515, 0.7078414559364319, -0.3357287049293518, -0.01116577722132206, -2.3407657146453857], [1.3627015352249146, 0.3190830647945404, -1.221523642539978, -0.6994436979293823, 0.8633442521095276, -0.7005369663238525, 0.7865275144577026, -1.9784398078918457, 0.9659916162490845, 0.06151445582509041, -0.3915504813194275, 0.392553448677063, 1.3144807815551758, -0.7869659662246704, 0.3582571744918823, 0.442516952753067, 0.8567978143692017, 0.7789140939712524, -0.3766767680644989, -0.10436704754829407, 0.16682715713977814, 1.2304271459579468, -1.1581801176071167, -1.0183318853378296, -0.8873576521873474, -0.6222658157348633, 0.5881041884422302, 1.12208092212677, -0.9636226892471313, 1.2022452354431152, 1.0029176473617554, -0.2674325704574585, 1.4269770383834839, -1.0187263488769531, -0.23202288150787354, -0.012896898202598095, 1.4230785369873047, -1.8659414052963257, -0.3576095998287201, 1.7032396793365479, -0.08863727003335953, 0.00950927659869194, -0.11792276054620743, -1.7864400148391724, 0.3658001124858856, 0.989112377166748, -0.8893575668334961, -0.34246188402175903, -0.4087989330291748, -0.472871333360672, 2.7680108547210693, -0.5911054015159607, -0.9856207370758057, 0.5517484545707703, -0.309175580739975, -1.178154706954956, 0.062332846224308014, 0.5334135890007019, 0.6512893438339233, -0.19364158809185028, 0.38219332695007324, -1.2274761199951172, -0.5714280605316162, -0.3046581447124481, 1.692279577255249, -0.07929550856351852, 0.23577430844306946, 0.3008292019367218, 1.3262052536010742, -1.136543869972229, 0.010853073559701443, 0.005282540339976549, 0.08447014540433884, 1.0667901039123535, -1.8858011960983276, 0.9745416641235352, 1.553499460220337, -0.2709548771381378, -0.31071123480796814, 0.031125493347644806, 0.20718643069267273, 1.7742280960083008, -1.8216105699539185, 1.5566973686218262, -0.02394828200340271, -0.5624169111251831, -0.4324344992637634, -1.3310043811798096, -0.46129274368286133, 1.05144202709198, -1.0312602519989014, -2.3483479022979736, 0.5477317571640015, -0.47898513078689575, 0.5119223594665527, -1.8868248462677002], [-0.18516185879707336, -1.7718079090118408, -1.1326321363449097, -0.11131584644317627, 0.9252709746360779, -0.17254869639873505, 0.2808559834957123, 1.0780644416809082, -1.7410238981246948, 0.07736837863922119, -0.6990236639976501, 0.34232231974601746, 0.4639510214328766, -1.3949079513549805, 0.4409324526786804, 0.30074542760849, 1.126137614250183, 0.5714470148086548, 0.8719927668571472, 0.06386516988277435, -1.0871297121047974, 0.36844784021377563, -0.22226545214653015, 0.24848638474941254, 0.8698693513870239, 1.6052470207214355, 0.571180522441864, 1.111830711364746, 1.4039288759231567, 0.4098559617996216, -0.48380592465400696, -0.011509530246257782, -1.1047922372817993, 0.241936594247818, 0.7278138399124146, -0.6657859683036804, 0.17075563967227936, -0.532080888748169, 1.1754686832427979, -0.734165370464325, -1.5260692834854126, -1.3056433200836182, -0.07695195823907852, -1.2070847749710083, -0.10227043926715851, -1.0122077465057373, -0.27332085371017456, -0.8294552564620972, 0.822939395904541, -1.209756851196289, 2.1764917373657227, 0.02125278115272522, 1.7444952726364136, -0.20800548791885376, 2.5551788806915283, -0.7579319477081299, 0.8330135941505432, -0.06374562531709671, 1.0625102519989014, -0.8417258262634277, 0.6903514862060547, -0.013940354809165001, 0.11543109267950058, 1.8806966543197632, -0.10520142316818237, 0.7046653628349304, 0.09692699462175369, 0.8237718343734741, 0.8678525686264038, 0.3070613741874695, 1.1960629224777222, 0.9770752191543579, 0.07614892721176147, -2.5592544078826904, 1.0315734148025513, -0.001204191124998033, 0.8687754273414612, 0.6778147220611572, 0.8069542050361633, 0.7838097810745239, -0.8961957097053528, 0.6597112417221069, 0.44283929467201233, -0.31651031970977783, 0.15122036635875702, 0.39787793159484863, -0.7300472855567932, -0.36488285660743713, -1.0983080863952637, -0.8775525093078613, 1.04971182346344, -0.27499955892562866, 0.05027264356613159, 1.7777389287948608, 2.473214864730835, -1.4476943016052246], [-0.7255716919898987, 0.22030574083328247, 2.7667548656463623, -1.1544501781463623, -1.6813596487045288, 0.11156265437602997, -0.05281227082014084, -0.7831876873970032, 0.053462572395801544, -0.2965802252292633, -1.3647292852401733, 0.4726603329181671, -0.4030752182006836, 1.4507756233215332, -0.053744684904813766, -1.1495780944824219, -0.5032036900520325, 1.2344413995742798, -0.17470794916152954, -0.2936396300792694, 0.9171932935714722, 1.1889772415161133, 1.4866907596588135, 1.8868553638458252, -1.5385537147521973, 1.5003366470336914, 0.14738331735134125, 1.9095019102096558, -0.5858043432235718, 0.8633201122283936, -0.19928500056266785, 0.4621325135231018, -0.5887993574142456, -0.6921005249023438, 0.25514286756515503, -0.4735962450504303, 0.6616275310516357, -1.0178128480911255, 0.8530194163322449, 2.1367549896240234, 0.6303310990333557, -0.7707723379135132, 0.31540897488594055, -0.013187546283006668, 0.8527811765670776, 0.25693297386169434, -0.24702519178390503, 1.1562806367874146, -0.5844938158988953, -0.12647174298763275, 0.7353927493095398, -0.7005088925361633, -0.4683285355567932, -0.23412708938121796, -0.8892836570739746, -0.5357480645179749, 1.6993286609649658, 0.12404301017522812, -0.9372504353523254, -0.961843729019165, 1.1851532459259033, 1.1975148916244507, 2.090435266494751, 0.34772175550460815, -1.190915584564209, -0.21907760202884674, 0.354878693819046, 0.985205888748169, 0.1971563994884491, 0.31568121910095215, 1.1821056604385376, 0.25645166635513306, 1.2423359155654907, 0.5191553831100464, -0.39600062370300293, -1.021515965461731, 1.910326361656189, -2.3566136360168457, -0.2149811089038849, 2.570502996444702, 1.4285014867782593, 0.8702058792114258, -1.4680501222610474, -0.05822309851646423, 0.33251485228538513, 0.7943596839904785, -2.361778974533081, -0.650307834148407, -0.1995425820350647, 0.4013501703739166, 0.31086716055870056, -1.3800487518310547, -0.44857102632522583, -1.7717077732086182, 1.7659671306610107, 0.27240243554115295], [0.24125316739082336, -0.9925662875175476, 0.3386250436306, 1.6193135976791382, -1.0491912364959717, -1.354414463043213, -0.016411608085036278, -1.5909980535507202, -1.2669527530670166, -1.6404350996017456, 0.9511212706565857, -1.1375192403793335, -0.39257973432540894, -0.6451063752174377, 1.3251149654388428, 0.6053317189216614, -1.207554817199707, -0.31836920976638794, -0.7203599810600281, -0.1008673831820488, 1.3106297254562378, 0.8055686354637146, -0.14285635948181152, 1.7288278341293335, 1.772202491760254, -2.1090712547302246, 0.31893789768218994, 0.7473299503326416, 1.139237880706787, -0.14836052060127258, -0.7366706728935242, 1.5227922201156616, -0.10468937456607819, 0.1838354617357254, -1.2426022291183472, 1.496427297592163, -1.1695672273635864, -0.2013407200574875, 1.325911283493042, 0.6590368151664734, -2.3183391094207764, 0.8259611129760742, -1.174544334411621, 1.999474287033081, -0.6787278056144714, 1.010398507118225, -0.831548273563385, 3.2681686878204346, 1.0496551990509033, 0.0524434968829155, -1.4911096096038818, 1.8639121055603027, 1.1953206062316895, -0.6466176509857178, -0.7997339367866516, 0.9635571837425232, 2.5469064712524414, 0.1630072146654129, 2.613921880722046, -0.25027576088905334, -0.08792667090892792, -0.10625562071800232, 0.8345057368278503, -0.4642738997936249, -1.1700615882873535, -0.4597894549369812, 0.26257747411727905, -0.24041961133480072, 1.4413036108016968, 0.19786964356899261, 0.8781869411468506, -1.2455140352249146, -0.8625131249427795, 0.7346513271331787, -0.7164794206619263, -1.2513865232467651, -0.07392877340316772, 0.7409714460372925, 0.8465184569358826, 0.015809573233127594, -0.034431323409080505, -0.693299412727356, 0.027323488146066666, -0.45501288771629333, -0.56785649061203, 0.48792174458503723, 0.9981936812400818, 0.4809179902076721, -2.1823997497558594, -0.902606189250946, -0.747947633266449, -0.7225383520126343, 1.1554152965545654, 0.5582703948020935, 0.2897019684314728, -0.20577354729175568], [0.7542818784713745, -0.6785310506820679, 0.39368826150894165, 0.4067305624485016, 0.7055584192276001, -1.224312424659729, 0.8046392202377319, 0.2697583734989166, -0.630192756652832, 0.42772120237350464, 0.06629599630832672, 1.097737431526184, -1.2203179597854614, -1.7829257249832153, 1.6410670280456543, -1.042129635810852, 0.3238098621368408, 0.15180917084217072, -0.8799448013305664, -1.7956396341323853, 1.5300507545471191, -2.8842668533325195, -1.2887009382247925, -1.5243228673934937, 0.7538391351699829, 0.36510157585144043, -0.8272484540939331, 1.4306349754333496, -0.7941817045211792, 1.5045186281204224, -1.201220154762268, -0.8102702498435974, 0.39774391055107117, 1.600312352180481, -0.1905660480260849, 0.002959097735583782, 0.9524161219596863, -0.5193025469779968, -0.7661958932876587, -0.4403037428855896, -0.16337688267230988, 0.7255859375, 1.0126168727874756, -1.455654263496399, 0.24492931365966797, -0.4177689850330353, -0.028118517249822617, -0.6420842409133911, 1.534143328666687, 0.1412643939256668, 0.08933531492948532, -1.212306261062622, 0.6197423338890076, 1.1615681648254395, -0.24263238906860352, 0.0651162713766098, 0.8655267357826233, -1.2479830980300903, -0.2072889506816864, 0.4835937023162842, 1.1075464487075806, -0.9272027611732483, -0.047239795327186584, 0.8759022951126099, 2.2229185104370117, -2.3306021690368652, -0.22666415572166443, -0.7824850082397461, -1.426595687866211, -0.45379331707954407, 2.342115640640259, 0.5787918567657471, -1.6044731140136719, 0.9031794667243958, 0.7803446650505066, -0.8521198630332947, 2.438047170639038, -0.3291901648044586, -0.14095649123191833, 0.09459756314754486, -0.0807240828871727, -0.23307786881923676, 1.0857117176055908, -1.2259535789489746, -0.7470132112503052, -0.6149498224258423, -0.2674184739589691, -0.6845077276229858, -1.034843921661377, 0.6351592540740967, -0.4387744665145874, 1.5040212869644165, 0.47117501497268677, 1.5237468481063843, 0.2676217257976532, -0.1020439937710762], [-1.052130103111267, -0.04899299144744873, 1.2543439865112305, -2.2481086254119873, -0.09245967119932175, -0.2630916237831116, 0.8201330900192261, -0.48118793964385986, -0.5395132899284363, 0.2590722441673279, 1.7696142196655273, 1.0626636743545532, 0.029143720865249634, 0.08302000164985657, 0.10780200362205505, -1.2328886985778809, -0.6498956084251404, -2.2905337810516357, -1.6580296754837036, 1.596724271774292, -0.26588407158851624, 0.13252320885658264, 0.34128043055534363, 0.6577730774879456, -0.3257893919944763, 1.526710033416748, -1.410609245300293, -0.25719088315963745, -0.31829506158828735, -1.2236804962158203, -2.214632749557495, 0.4185909926891327, -0.05377182364463806, -0.9694523215293884, 0.026554957032203674, -0.016330359503626823, -0.1243242546916008, -0.16541346907615662, -0.41781753301620483, -1.0415666103363037, -0.1693073809146881, -0.2950073778629303, -0.19672371447086334, 1.4888596534729004, 0.8546295166015625, -0.2597123384475708, -0.25202012062072754, 1.1485642194747925, 0.9552196264266968, -0.35858097672462463, 0.6725996732711792, -0.34376680850982666, -1.7367252111434937, -1.211631178855896, -0.8409102559089661, 0.8251405954360962, 0.8811870217323303, -0.8953311443328857, -0.1642448753118515, -0.8155817985534668, -0.28214335441589355, 1.1314531564712524, -0.4359460473060608, -2.17513370513916, -0.9663355946540833, -1.5126532316207886, -0.5713685154914856, -0.05813547223806381, 1.9489893913269043, 0.10729558765888214, 1.7680506706237793, 0.13569006323814392, 0.10966996848583221, -0.32964691519737244, -1.4638628959655762, 1.3047764301300049, -0.41182419657707214, 0.41151702404022217, -0.11369743198156357, -0.06555869430303574, 0.7833410501480103, -1.125617265701294, -1.840407371520996, -0.5086259841918945, 0.6367419362068176, -0.1448633372783661, -1.055152177810669, 0.40754544734954834, 0.41561245918273926, -0.34237903356552124, -0.6097890138626099, -0.721576452255249, -0.05201129615306854, 0.6287444233894348, -1.6118993759155273, -0.3192315101623535], [-1.000478744506836, 1.6636900901794434, 0.06871362030506134, -1.2327309846878052, -0.9067776203155518, 0.4872322976589203, -0.8581846952438354, -0.4975232481956482, 0.7500845193862915, 1.3000091314315796, -0.17914564907550812, 2.1671581268310547, -0.2414468675851822, -0.36586618423461914, -0.5448182225227356, -0.39549028873443604, 0.7602397799491882, -0.6850574016571045, -1.052096962928772, 1.3689830303192139, 0.6548721194267273, 0.06112939119338989, 1.2257276773452759, -0.3748530149459839, 0.2159690260887146, 2.459528923034668, -0.07571205496788025, 0.4412076771259308, 0.6755626201629639, -0.43201813101768494, -0.49413561820983887, 0.40743309259414673, -2.5233991146087646, -1.0887234210968018, 0.3145248591899872, -1.5679032802581787, 1.0784510374069214, 0.7991921305656433, 0.6978957653045654, -1.335747241973877, -0.287269651889801, -2.818596601486206, 1.8868434429168701, 1.1452912092208862, 0.3319898247718811, 0.25505927205085754, -0.1090254932641983, -1.2375551462173462, -0.27252325415611267, 0.1649588793516159, -0.5819490551948547, -0.1530124545097351, 0.8338139653205872, 1.415043830871582, 0.15431305766105652, -0.3310454785823822, 1.8777226209640503, -1.9487619400024414, -1.3751686811447144, 0.5983136892318726, -0.5252707004547119, 0.6512371897697449, -0.10739373415708542, -0.4475949704647064, 1.0869884490966797, 0.032306794077157974, 0.6635265350341797, 0.5989348292350769, 1.6756229400634766, -0.11590651422739029, -0.12097030878067017, 0.22951640188694, 1.3076668977737427, -0.8572896718978882, 0.11756826192140579, -0.9059399366378784, 0.941121518611908, 0.4446295201778412, 0.4085625112056732, -1.105562448501587, 0.09022674709558487, -2.622140884399414, -0.3499405086040497, 0.16874302923679352, -0.7887124419212341, -0.4809935688972473, -0.5731713771820068, 1.6726891994476318, 0.33120059967041016, 0.4750478267669678, 1.0143793821334839, -0.008105170913040638, -0.4805814325809479, -0.1680455356836319, 0.9002379179000854, -0.5530370473861694], [1.2557209730148315, 0.982088565826416, -1.3377209901809692, -0.2431478053331375, 0.07848411798477173, 0.3893650770187378, -0.5288052558898926, 1.5176697969436646, -0.902926504611969, 0.4667855203151703, -1.298743486404419, -0.46899664402008057, 1.0632712841033936, -0.35316044092178345, 0.1444004774093628, -0.3941689431667328, -1.2749409675598145, 0.4616248607635498, -0.4227634370326996, -0.7176706790924072, 1.0334895849227905, -2.0801992416381836, -0.9899066090583801, -0.09188058227300644, -0.3270873725414276, 1.0498905181884766, -1.513893485069275, 0.11698516458272934, -0.05321485549211502, 1.0616718530654907, 0.3300793170928955, -1.4724388122558594, -0.592505693435669, -1.2502026557922363, -0.9449618458747864, -0.2663489282131195, 0.4820016622543335, 0.37914493680000305, -1.1959691047668457, -1.724187970161438, 0.2311234176158905, -0.10515276342630386, -0.36018484830856323, -0.598162055015564, 2.130762815475464, 1.305930733680725, -0.2633976340293884, -0.2704252600669861, 0.8644351959228516, 0.5620707869529724, 0.5258893966674805, -0.764086902141571, -0.44293951988220215, -0.841782808303833, -0.24425898492336273, 0.24746105074882507, 1.0975476503372192, 1.6813362836837769, -0.5624770522117615, -0.6416207551956177, 1.0903756618499756, -0.3960750699043274, -0.34257012605667114, 0.7386184930801392, -0.7125055193901062, -1.810420036315918, 0.6163699626922607, -0.4332277774810791, 0.18760353326797485, -0.26136186718940735, -0.26256367564201355, 0.5545255541801453, 0.36100029945373535, 0.013600938953459263, -0.06844398379325867, 0.5177666544914246, -1.4735058546066284, -0.6300014853477478, -2.1716952323913574, 0.5855305194854736, -0.12568287551403046, -1.4431158304214478, -0.2205512523651123, 1.3649203777313232, -0.26791852712631226, 0.9161984920501709, -1.2099055051803589, 0.5918372869491577, 0.03202449157834053, -0.09105376899242401, -0.45192596316337585, 1.6604554653167725, 1.3483846187591553, -0.14680640399456024, 0.31709232926368713, 0.2518453896045685], [-0.07356753945350647, 1.1241079568862915, 1.7500020265579224, 0.5362411141395569, 0.7285184860229492, -0.10088586062192917, 0.026012476533651352, -0.6613684296607971, 0.5998511910438538, -0.38078323006629944, 0.38226401805877686, 0.9976019263267517, -1.2675336599349976, -1.2877784967422485, -0.4816204309463501, -1.3588041067123413, -0.3899654746055603, -0.28298404812812805, -0.9567863941192627, -1.2431186437606812, 2.6135659217834473, 0.8003730773925781, 0.7759250402450562, -0.3146354556083679, -0.5192477703094482, 1.5408062934875488, 1.8579171895980835, -0.5159883499145508, -1.5539696216583252, -2.6741273403167725, -2.0221927165985107, -0.0233903918415308, -0.5381492376327515, -1.021759033203125, -1.2631868124008179, 1.768524169921875, 1.1367082595825195, -1.2501864433288574, -0.8490656018257141, 1.2855448722839355, 2.3559374809265137, -1.307024598121643, -0.6203823685646057, 1.6904311180114746, 0.6976683735847473, 0.3246552646160126, -0.7926210761070251, -3.8580145835876465, 0.8377923965454102, 0.45144766569137573, -0.32732826471328735, -1.8497565984725952, -1.3411593437194824, 0.9479410648345947, 1.3218071460723877, -0.7363241910934448, -0.4782921075820923, 1.0322540998458862, -0.5565536618232727, 0.6714609861373901, -0.8727023005485535, -0.2542192339897156, -0.8580556511878967, -0.9730688333511353, -2.1743404865264893, -1.214034080505371, -0.2540653944015503, -0.11555667221546173, -0.33568325638771057, -1.560246229171753, -1.3322099447250366, -1.4662253856658936, -0.3870331943035126, 1.7302190065383911, -1.1668323278427124, 0.6786954402923584, -0.6371580362319946, -0.5600716471672058, 0.8359249234199524, -1.576356053352356, -1.7267574071884155, 1.3333889245986938, 0.672618567943573, -0.0990133136510849, 0.941815197467804, -0.9370039701461792, -0.6153120994567871, 0.23997808992862701, 1.5267350673675537, -0.4886857271194458, -0.4009375274181366, 0.33074069023132324, 1.5980031490325928, 0.638735294342041, 0.6305623650550842, -0.1630246937274933], [-0.18161681294441223, 0.05504021793603897, 0.4480273723602295, 0.5560941696166992, 0.14199475944042206, 0.7975977063179016, -0.4624764025211334, -0.1765851080417633, 0.514901340007782, 0.7862944602966309, -1.2536412477493286, -0.7949573397636414, -0.44088879227638245, 1.211979866027832, -0.3284832835197449, -1.3388715982437134, 1.2492448091506958, -0.9273935556411743, -1.423996090888977, 2.620872974395752, 0.6196911334991455, -0.34418970346450806, -0.03477639704942703, 1.0453011989593506, -0.3224446177482605, -0.33569252490997314, -0.33236485719680786, -1.2604947090148926, 2.5120418071746826, -1.8373552560806274, -0.9545185565948486, 0.15559396147727966, -0.5570327639579773, 0.865862250328064, -1.132741928100586, -0.2098686695098877, 0.22019940614700317, -0.011274571530520916, 0.4165700674057007, -1.3415720462799072, -1.8129726648330688, 0.39415886998176575, -1.3748304843902588, -2.219076633453369, 0.1185072660446167, 0.7244240045547485, 0.1534215360879898, -0.290100634098053, -0.48049408197402954, -0.1715264916419983, -0.40563899278640747, -0.3690424859523773, 0.2248954027891159, 0.27614161372184753, 1.1476045846939087, -1.4326974153518677, 0.06841832399368286, 0.2586337924003601, 1.24750816822052, 0.2527741491794586, -0.8686320781707764, 1.3709954023361206, -0.6885780096054077, -0.11583936959505081, -1.4839390516281128, 0.8467212915420532, 0.31731003522872925, 1.097078561782837, 1.4400027990341187, -0.15349456667900085, 0.40677979588508606, -0.9167298674583435, 0.5526453256607056, -0.4370354115962982, -0.28495529294013977, 0.36100971698760986, 0.06732603907585144, 0.5548655390739441, -0.2639508545398712, -1.556047797203064, -0.45199546217918396, 0.5719805955886841, 1.4228452444076538, -0.20979562401771545, 0.36938947439193726, -1.6387532949447632, 0.7407510876655579, -1.0255990028381348, -0.22113612294197083, 0.2791156470775604, 0.9699791669845581, 1.6056904792785645, -0.23695547878742218, 1.309086561203003, -0.2982412576675415, -0.7100424766540527], [0.21873052418231964, 0.4942021071910858, -1.0258574485778809, -0.5915423631668091, 0.3731672763824463, -0.17020320892333984, 1.5671212673187256, 1.0094096660614014, -0.41793134808540344, 0.4723691940307617, -0.864765465259552, -1.474584937095642, -1.013981580734253, -1.772340178489685, -0.12903402745723724, 1.0547014474868774, 0.6751242280006409, 1.606561303138733, 0.0644102469086647, -0.3758912682533264, -0.499186247587204, 0.4356420636177063, -0.9385013580322266, 0.6251951456069946, 0.7467008829116821, 0.9142126441001892, 0.19347554445266724, -0.11936673521995544, -0.9992434978485107, 2.4639408588409424, 0.0777682512998581, -0.4535636901855469, 2.875425100326538, -0.5681242346763611, 1.2204153537750244, 1.247269868850708, 1.0109210014343262, 0.8463128209114075, -0.5247541069984436, -0.1745733767747879, -0.2825031578540802, -0.5629675984382629, 0.3987809717655182, 0.6823137402534485, -0.40287551283836365, 0.6782243251800537, -1.275475263595581, 0.725010871887207, -1.616767406463623, 1.1629105806350708, -0.04997559264302254, -0.6872217655181885, 0.3725147843360901, 0.24586480855941772, 0.8525930047035217, 0.3298909366130829, 0.6556268930435181, -1.2234138250350952, 0.4230719804763794, 1.7146596908569336, -0.8073396682739258, 0.6234700679779053, 1.3077588081359863, -2.157215118408203, 1.9954969882965088, -1.206986665725708, 1.4930284023284912, 0.16879230737686157, 0.26520851254463196, -0.5556225776672363, -0.4851975440979004, 0.11237434297800064, 0.6499543190002441, 0.4817877411842346, 0.2715439200401306, 0.9476683735847473, 0.15409919619560242, -0.2816934585571289, 1.048236608505249, 0.5124945640563965, -1.729303002357483, -1.4279228448867798, 1.4109045267105103, -1.398507833480835, -0.48454663157463074, 1.2457654476165771, 0.18577665090560913, -0.1805577129125595, -0.9542533159255981, -1.2882269620895386, -0.9683015942573547, 0.5273758769035339, 1.1019691228866577, -0.13409030437469482, 0.8933281898498535, 0.3717651069164276], [-0.25996026396751404, 0.28648582100868225, 0.033320095390081406, -1.262998342514038, 0.32470864057540894, 1.313332438468933, -0.11023566126823425, 0.4175333082675934, -0.0809088870882988, -0.657758891582489, -2.0822386741638184, -0.34440886974334717, 0.648453414440155, -2.8098766803741455, -1.9596513509750366, 0.8076561093330383, -1.1458473205566406, 0.31517693400382996, 0.37864717841148376, -0.5424290895462036, 0.502762496471405, 1.2659825086593628, 1.6847091913223267, -0.2232547402381897, 0.004045980051159859, 1.5531084537506104, -2.0993261337280273, 0.003561720484867692, -1.2816333770751953, 1.4390262365341187, -0.03257149085402489, 0.6851922273635864, 0.27649158239364624, -1.425117015838623, -0.7331514954566956, 1.6778000593185425, -0.13188059628009796, 0.6767988801002502, 0.21127989888191223, 0.031322184950113297, -1.494545578956604, -0.4331617057323456, 0.3254479467868805, 0.003693448379635811, -0.47624674439430237, -0.4902972877025604, 0.23444268107414246, 0.6679388284683228, 0.23855268955230713, -0.7918826937675476, 0.08552423864603043, 1.2424101829528809, 0.7909477353096008, 0.8186451196670532, 0.8834939002990723, 0.5234646201133728, 1.5988579988479614, -1.1510310173034668, -0.8852809071540833, -0.41303426027297974, 0.39029252529144287, -0.5636803507804871, -1.3035657405853271, 0.2004171907901764, 0.7666659951210022, 0.2278061956167221, -0.49125105142593384, -1.8662368059158325, 0.2808057963848114, -0.6052228212356567, 0.11592863500118256, 0.7033218741416931, 1.1384015083312988, 0.2338939756155014, -1.111465573310852, -1.4354597330093384, 0.22502532601356506, 0.052919067442417145, 0.23005586862564087, 0.6261300444602966, 1.3610974550247192, -0.32971519231796265, -2.7584381103515625, 0.15625318884849548, 0.7104386687278748, 0.45760220289230347, 0.8784875869750977, -1.6745615005493164, -0.70449298620224, -1.930677056312561, -1.2914918661117554, 1.1750541925430298, -0.6417241096496582, -1.8271788358688354, -0.3088666796684265, -0.22609210014343262], [1.564987301826477, -0.9942742586135864, 1.7127236127853394, -0.812287449836731, -0.4296739399433136, -0.3704548180103302, 0.15656955540180206, -0.8184484243392944, 0.3907223641872406, -0.6627305746078491, 0.9282946586608887, 0.6834993958473206, -0.3075072765350342, -1.6214245557785034, 0.3233512341976166, -0.6888303160667419, -0.6276222467422485, 0.5824449062347412, 0.27757009863853455, 0.5363104939460754, 1.132789969444275, -0.21668046712875366, 0.2762782871723175, -1.5468670129776, -0.9552311301231384, 1.0538434982299805, 1.0353527069091797, -0.9028458595275879, 0.8121612668037415, -1.2358918190002441, -1.7332127094268799, -1.9686321020126343, -0.24476072192192078, 2.759706497192383, 0.8877031207084656, 0.006349568720906973, 0.4903826415538788, 0.0007312198868021369, 0.3642219603061676, 0.182756245136261, 0.07098716497421265, 0.9091426730155945, 0.6214998960494995, -1.9449045658111572, -2.1391851902008057, 1.3981913328170776, -1.0342518091201782, -1.2096625566482544, -0.28387320041656494, -0.48468098044395447, 1.018036127090454, -1.7477856874465942, 0.3273864686489105, 1.0268990993499756, 0.9262863397598267, -0.19544482231140137, 0.4554249346256256, -0.34126901626586914, -1.6564929485321045, -0.020174361765384674, -1.2140148878097534, 0.7470558881759644, -0.7028835415840149, 1.8893024921417236, -1.2678474187850952, -1.918447732925415, 0.10429048538208008, -1.34916090965271, 0.5120405554771423, -0.4886321425437927, 0.48697206377983093, -0.4047168791294098, -0.2638929486274719, 0.35743317008018494, -0.3553614616394043, 0.3898380398750305, -1.6774929761886597, -0.27774113416671753, 2.089323043823242, 3.01046085357666, 1.7325299978256226, -0.6741759777069092, 0.7400872707366943, 0.7328580021858215, -0.5335443019866943, -0.8260652422904968, -1.27151358127594, -0.9333855509757996, -0.6411970853805542, 0.2253759801387787, 0.7783210873603821, 0.31560975313186646, 0.06852748245000839, -0.5652256608009338, 0.4565787613391876, -0.9492173194885254], [-1.4636667966842651, -1.0680452585220337, -1.6290769577026367, -0.20989923179149628, 0.2299347072839737, 0.584086537361145, -0.9386611580848694, 2.466700792312622, 0.4902009963989258, 1.3741388320922852, -0.13042724132537842, 0.4835336208343506, -0.3260371685028076, -0.43071842193603516, -0.18455757200717926, 0.22430847585201263, 1.4259655475616455, -0.05797345191240311, 1.7671825885772705, 0.12266324460506439, 0.0075119794346392155, 0.8732424378395081, 0.05042356997728348, -1.330010175704956, 0.20363251864910126, 0.3135520815849304, -0.8626662492752075, -1.1993488073349, 1.7838155031204224, 0.7645031809806824, 1.9385108947753906, -0.15809334814548492, 1.9068185091018677, -0.31031620502471924, -1.8166296482086182, -1.5889250040054321, 1.5728099346160889, -0.9636968374252319, -1.069871425628662, -0.4866698384284973, 0.9282272458076477, -1.0055538415908813, -0.2736036479473114, -0.4003250300884247, 0.011387371458113194, -1.6698095798492432, 0.15349406003952026, -1.2810920476913452, -0.9603993892669678, 1.3723106384277344, -0.1364683359861374, -2.367676019668579, -0.18931537866592407, -0.35565391182899475, -0.14191703498363495, -0.6721847057342529, -0.890005886554718, 0.07018603384494781, -0.06901853531599045, -0.719920814037323, -1.1853468418121338, 0.4151982069015503, -0.45389845967292786, 0.7156105637550354, -1.0555520057678223, -0.8377936482429504, 0.6204683780670166, -0.8453202247619629, -0.10212689638137817, 0.7728626728057861, -0.3361132740974426, -0.1143624410033226, -1.265325665473938, 0.29689422249794006, 0.8206124305725098, 0.21408669650554657, 1.7115143537521362, -0.06236867606639862, 0.8275309801101685, -0.5019171237945557, -0.3942002058029175, 1.7002179622650146, 0.357887327671051, 1.540031909942627, -1.2060260772705078, -1.3290430307388306, 0.30657848715782166, -0.5907264947891235, -0.2239723652601242, -0.7926075458526611, 0.1619139313697815, 0.333332896232605, -0.1659778207540512, 2.3003013134002686, -0.88548743724823, 0.6651337146759033], [0.9681976437568665, 1.7356350421905518, -2.5141518115997314, 0.15125463902950287, -0.27017149329185486, -1.1774237155914307, -0.7482132911682129, -0.7377604246139526, 1.1102092266082764, 1.999262809753418, 0.014960241504013538, 0.11538109928369522, 1.1498479843139648, 1.463085651397705, 1.051817774772644, -0.9977497458457947, 0.03283795714378357, 1.4253771305084229, -1.2067360877990723, 0.02220870740711689, -0.20942774415016174, 0.008302463218569756, -0.49217525124549866, -1.3295401334762573, 0.7432507276535034, -0.055428341031074524, -0.7726231217384338, -0.6661116480827332, -0.12880758941173553, -2.247313976287842, 1.3348801136016846, 0.9123289585113525, 0.4353311061859131, -0.6450116038322449, -1.2416415214538574, 2.6185760498046875, 0.11009228229522705, -0.23128294944763184, 1.8782193660736084, -0.5032833218574524, 0.9376694560050964, -1.2927464246749878, -1.3874232769012451, -1.7760916948318481, 0.23580506443977356, 0.5845791697502136, 0.12090510129928589, 0.19576382637023926, 1.0753631591796875, 0.09808965027332306, -0.7654810547828674, 0.37600991129875183, -0.741673469543457, 2.03475022315979, -0.8446844220161438, 1.2519525289535522, 0.6335713267326355, -1.2951867580413818, 1.381568193435669, 0.7832409739494324, 1.1124187707901, -0.986150860786438, -0.43666738271713257, -0.032457541674375534, 0.41667959094047546, -0.3142830729484558, -0.13189418613910675, -2.0519204139709473, 0.6086854934692383, -0.4123346209526062, 1.6546355485916138, 1.6152358055114746, 1.1443989276885986, -0.13083171844482422, -1.9801383018493652, 0.2442130446434021, 0.5067396759986877, 1.057828426361084, -0.839270293712616, -2.0024635791778564, -0.31322160363197327, -0.14178264141082764, -0.10574807971715927, -0.13532094657421112, 0.9170718789100647, -0.6254073977470398, -1.238312005996704, -0.48744410276412964, 1.3153828382492065, 1.3702857494354248, -0.053006138652563095, 1.1685452461242676, 2.1532201766967773, 0.1740640550851822, 0.5081782937049866, 0.607257604598999], [-0.6604056358337402, -0.1475464403629303, 0.00814431719481945, 0.46229416131973267, -0.9317179918289185, 0.720844566822052, 0.09445939213037491, 1.3805055618286133, -0.9340825080871582, -0.0181554164737463, -0.15292486548423767, 0.10858739167451859, -0.7928085923194885, 1.7587448358535767, -1.2710206508636475, 0.798991858959198, 0.7176530957221985, -1.4003556966781616, -0.13545116782188416, 0.44268906116485596, 1.1890079975128174, -0.9704640507698059, -0.45291364192962646, 2.0193920135498047, 0.7370357513427734, 1.4074819087982178, 2.0508933067321777, -0.3049745559692383, -1.6042667627334595, 1.3965916633605957, -0.36339378356933594, 0.21700426936149597, -0.8075446486473083, 0.5515861511230469, 0.10431144386529922, -0.413293719291687, -0.9217705130577087, 0.40707892179489136, 0.8641396164894104, -0.9384562969207764, 1.0453264713287354, 1.1107892990112305, -1.783648133277893, -0.09898069500923157, 0.794049859046936, 0.3829650580883026, -0.13931089639663696, -0.5282509326934814, 0.12381142377853394, 0.13782377541065216, -0.2423006147146225, -0.19790968298912048, -0.05132526159286499, 0.49908187985420227, -1.3882975578308105, -1.692427396774292, -0.5714029669761658, -2.117691993713379, 0.17611220479011536, 1.0209732055664062, -0.3510524034500122, -1.2890596389770508, -1.7304531335830688, -0.812927782535553, -0.6639413237571716, -0.3417429029941559, -0.3199867010116577, -0.3951396644115448, 0.17241571843624115, 0.2076999545097351, 0.8331297039985657, 1.3367094993591309, 0.4617210030555725, -0.9365255832672119, 2.2860822677612305, 0.7407549619674683, -0.644608199596405, -0.4620504677295685, 0.03763488680124283, 0.8092900514602661, -0.18058858811855316, 1.1265778541564941, -0.4029216766357422, 0.9176721572875977, -0.7886549830436707, 0.7072068452835083, -1.5580949783325195, -0.19844168424606323, 0.6743882894515991, 0.8476694226264954, -1.745545744895935, -2.024843454360962, 0.4234775900840759, 0.7378765344619751, 1.5668466091156006, -1.4563332796096802], [0.8899574875831604, -0.25469261407852173, -1.5240031480789185, -0.455651193857193, 0.12544894218444824, -0.28641659021377563, 0.32338476181030273, -0.05794905498623848, -2.793898344039917, -1.4938790798187256, 0.13303668797016144, 0.04592578485608101, 0.8971273899078369, 0.24353240430355072, -0.5955632328987122, -1.0423345565795898, -0.6371053457260132, -0.25503507256507874, 0.2740892767906189, 0.11412075906991959, -0.16069142520427704, -0.4913072884082794, 0.7915984988212585, -0.1567775309085846, 1.6858594417572021, 1.0883439779281616, 2.045542001724243, -0.1413751244544983, 1.0902549028396606, -1.2263704538345337, -1.2125816345214844, -0.4091687798500061, -0.1142609715461731, 1.0944753885269165, 2.420534372329712, -1.8232343196868896, -0.6351487636566162, -0.9465770125389099, -1.0749198198318481, 0.9505958557128906, -0.4432367980480194, -0.44007304310798645, 0.37468352913856506, -0.3705582916736603, 0.443509578704834, -0.01577567495405674, 0.18603914976119995, 0.5913456678390503, 1.687995195388794, 1.2389411926269531, -1.0940937995910645, 0.9058927297592163, -1.308509111404419, -1.3968883752822876, 0.8741016983985901, 0.8610697388648987, 0.10538861155509949, 0.8699228763580322, -0.838701069355011, -0.007314283400774002, -0.6596779227256775, -1.2052384614944458, -0.15581324696540833, 0.47040891647338867, -1.3382360935211182, 0.11763180047273636, 0.3583467900753021, 0.1847415566444397, 0.7761963605880737, -0.7432385683059692, 0.023620517924427986, 0.5546333193778992, 0.4248618185520172, -1.2467907667160034, -0.8264517188072205, -1.451870083808899, 1.0084093809127808, -0.8644049167633057, 0.6209363341331482, 0.4282633364200592, 0.07913389056921005, 0.1145382672548294, 1.1933130025863647, -0.36304929852485657, -0.07175032794475555, 0.030238263309001923, 1.3742613792419434, 0.12957431375980377, -1.311745285987854, -0.03441178426146507, -0.8371915817260742, 0.039200276136398315, 0.8658791780471802, -1.8577139377593994, -0.5554517507553101, -0.7995668053627014], [-0.2191181480884552, 1.2310981750488281, 0.9848439693450928, -0.9576236009597778, -0.873973548412323, -0.6795287132263184, 1.2610539197921753, 1.1713120937347412, 1.683556079864502, 0.7676793932914734, 0.9027124047279358, 1.1681993007659912, -0.49795252084732056, -0.17081022262573242, -0.0792534202337265, -0.5702829360961914, -0.8072507977485657, -0.4294024705886841, 0.7727571725845337, -0.46899235248565674, -1.735746145248413, -0.8109987378120422, 1.7422655820846558, 0.3290499448776245, 0.21241635084152222, 1.169465184211731, 1.0132302045822144, -0.29681628942489624, 1.187023639678955, -1.1945549249649048, -0.11048233509063721, 0.12430732697248459, -0.7042582035064697, -0.3171458840370178, -0.04108361527323723, -2.1075286865234375, 1.6857749223709106, 1.6750198602676392, -0.19249728322029114, -0.5891498327255249, 0.43128737807273865, 1.5087288618087769, 0.9767901301383972, -0.43588241934776306, 1.4142857789993286, -1.603981852531433, 1.5470693111419678, 0.6997631192207336, 0.2734391689300537, -0.3413408100605011, -1.3209682703018188, 0.8080760836601257, 0.40340274572372437, -1.0262120962142944, -0.5864109992980957, 0.15781253576278687, -1.727826476097107, -0.11705728620290756, -0.6446756720542908, 1.1193811893463135, -1.091556191444397, -1.4646453857421875, 1.402859091758728, 0.9839804172515869, 1.6078463792800903, 1.7543467283248901, 0.88652503490448, 0.1396055370569229, -0.42190420627593994, 1.1634100675582886, -1.2847967147827148, 1.1287235021591187, -0.1927889883518219, 0.08660081028938293, -0.9662380218505859, 0.38845568895339966, -0.06006813794374466, -0.3643740713596344, -1.8506547212600708, 0.8567308187484741, -0.5755185484886169, -0.3950391709804535, 0.06640598922967911, 1.0683979988098145, 1.4015744924545288, 0.5999293327331543, 0.4827441871166229, -0.25273004174232483, -0.060515329241752625, -1.7216832637786865, 1.152942419052124, -0.5800198316574097, -1.0924859046936035, -1.3736572265625, 1.2106729745864868, 0.48178234696388245], [-0.35516557097435, 0.9786068797111511, -0.28883200883865356, -0.5863690376281738, -0.19403783977031708, -1.4147379398345947, -1.4925209283828735, 0.8610363602638245, 1.0408482551574707, 1.7302254438400269, 1.107832670211792, 0.9696766138076782, 0.882532000541687, 0.2633247673511505, 2.0894808769226074, 0.6546575427055359, 1.195108413696289, -0.9147436022758484, -0.11353475600481033, -0.5963572859764099, 0.027800295501947403, 0.9400990009307861, 1.0318598747253418, -1.084558129310608, -0.8724017143249512, 1.086184024810791, 0.01914278417825699, 0.8261121511459351, -1.51710844039917, 0.913359522819519, 1.0924720764160156, 0.010512711480259895, -0.23048318922519684, -0.954312801361084, 1.0731598138809204, -0.19285990297794342, 0.28639763593673706, 1.2112642526626587, 0.4999014437198639, 1.4201322793960571, 0.6762259602546692, -0.09357679635286331, 0.7823204398155212, -2.067120313644409, 0.844227135181427, -0.08128854632377625, -1.1204888820648193, -0.28255534172058105, 1.9562079906463623, 0.20040437579154968, -0.058242857456207275, 0.5879780650138855, -0.4900647699832916, 0.07757211476564407, 0.4678782820701599, -1.4157977104187012, 1.0794919729232788, 0.12608981132507324, 0.26118093729019165, 2.518859386444092, -1.1333658695220947, 0.712955117225647, -0.9021412134170532, -0.01312093436717987, -0.345859557390213, 1.841666340827942, 0.46588966250419617, -0.6855356097221375, 1.3726834058761597, 0.2467688024044037, 2.3060803413391113, 0.04906342923641205, 0.06853656470775604, 0.5763769745826721, 0.5685081481933594, 0.864528238773346, 1.4394879341125488, -1.3905783891677856, 0.513824462890625, -0.14767634868621826, 0.39691171050071716, -1.2216269969940186, 1.8145437240600586, -2.27653431892395, 0.43990302085876465, 1.3318700790405273, 0.17028774321079254, -0.7733973264694214, 1.323383092880249, 0.9823853373527527, -0.054274845868349075, -0.46321186423301697, 1.3304141759872437, -1.1391572952270508, -1.0654592514038086, 0.8536874651908875], [-0.1917392760515213, -0.18849343061447144, 2.0788753032684326, -1.0725383758544922, -2.06459903717041, -0.24526356160640717, -0.7791774272918701, 0.6023486256599426, 3.107783794403076, -1.0740653276443481, -0.04974035173654556, 0.8068488240242004, 0.4902224540710449, -0.9441048502922058, 0.05773448944091797, -0.23419184982776642, -0.5043299198150635, 0.10323993861675262, -0.3274412155151367, 0.35700011253356934, -0.15578322112560272, -1.3838046789169312, -0.5451629757881165, -0.3743251860141754, 0.8163660764694214, -0.2402266561985016, -0.6904807090759277, -1.007139801979065, -0.2919868230819702, -0.612404465675354, 0.49611613154411316, -0.34848448634147644, 0.17660300433635712, -0.5763411521911621, -0.03506306931376457, -0.19644412398338318, -0.1172729954123497, -0.9111077189445496, 1.7269279956817627, 0.21237413585186005, 0.7464522123336792, -2.359962224960327, -0.34315240383148193, -2.0285677909851074, -0.23736310005187988, 1.8448501825332642, 0.5727092027664185, -0.9747412800788879, 0.013703139498829842, -0.28095000982284546, -0.010641781613230705, -0.12025271356105804, -0.5640828609466553, -0.0761793926358223, 0.07332754880189896, -0.404519259929657, 0.5230896472930908, 1.0140271186828613, -0.7089340090751648, 0.6065343022346497, -0.016876477748155594, 2.5193862915039062, 1.0610418319702148, -0.312694251537323, 0.5077597498893738, -0.5022372603416443, -1.062729835510254, -0.4075261354446411, 0.1825120896100998, -0.6532937288284302, -0.8231952786445618, 1.298553705215454, -2.5895204544067383, 0.05876101180911064, -0.5144893527030945, -0.3536340594291687, -0.12693603336811066, 0.2730085849761963, 0.43252357840538025, -1.2355045080184937, -1.9722431898117065, -0.27246174216270447, 0.11485602706670761, 1.0584059953689575, -0.9161413908004761, 0.0021713310852646828, 0.3654513359069824, -0.8701576590538025, -0.9416553974151611, -0.7513465285301208, -0.3337550759315491, -0.13325347006320953, -1.2691127061843872, 1.1220914125442505, 2.1846020221710205, 0.3504582345485687], [-0.3041995167732239, 1.0584524869918823, -1.589493751525879, 0.5140326619148254, 1.3087365627288818, 0.5061746835708618, -1.1388823986053467, -0.5726355314254761, 0.5636438727378845, 0.7901911735534668, 1.1867170333862305, 1.443634033203125, -0.12991870939731598, 0.5672705769538879, -0.5685860514640808, -0.33256179094314575, -0.07182648032903671, 0.3777908384799957, 0.9852798581123352, -0.38060063123703003, 0.8825406432151794, -0.33176058530807495, 0.7496548891067505, -1.002327799797058, -1.9416359663009644, -0.5709546804428101, -0.4273354113101959, 1.055516004562378, 0.21058285236358643, -0.8405593633651733, -0.6011897325515747, 0.42723068594932556, 0.2238156795501709, -0.5662238597869873, 0.8927026391029358, -0.5627083778381348, -0.23112893104553223, -0.668354332447052, -0.23861946165561676, -0.44591692090034485, -0.6869527697563171, -0.32977503538131714, -0.3761826157569885, 0.5681428909301758, -0.47361841797828674, -0.5806041359901428, -2.0095174312591553, 2.7320468425750732, -1.895460844039917, 0.16896802186965942, -0.4958837330341339, -0.4800437092781067, 1.0140495300292969, -0.20135602355003357, 0.44846442341804504, 0.1352730095386505, 0.4022878408432007, 1.9428846836090088, -0.2301350086927414, -0.2902214229106903, -1.2864652872085571, -2.0686309337615967, 0.49771153926849365, 1.091291069984436, -0.5989727973937988, -1.3335484266281128, -0.9884699583053589, 1.2305105924606323, 0.6207820177078247, 0.20288309454917908, 1.0087754726409912, 1.7640259265899658, -2.4899771213531494, 0.6998181939125061, -0.16874973475933075, -0.5339862108230591, 1.3156038522720337, 1.0676639080047607, 1.0735667943954468, 1.3700876235961914, -0.13347958028316498, -1.4040359258651733, 1.1529630422592163, 1.266303300857544, 1.518943428993225, 1.2712321281433105, 0.9052395224571228, 0.10928903520107269, 0.7777360677719116, 0.7528443932533264, -0.8903780579566956, 0.5255033373832703, 1.610079050064087, 0.1506221741437912, -1.2409477233886719, 1.3057390451431274], [1.5454567670822144, 0.01164236105978489, -1.9846757650375366, 1.5088106393814087, -0.7510050535202026, 1.6186904907226562, 0.09681206196546555, -2.24721360206604, 0.20848287642002106, -0.2039259821176529, -0.6347165703773499, 2.201878309249878, -1.5091544389724731, 0.6674294471740723, -0.12041456997394562, 0.25928637385368347, -0.7843758463859558, -0.6900697350502014, 1.0904585123062134, 1.05764639377594, 0.5699771642684937, -0.39152100682258606, -1.6728371381759644, -0.43190568685531616, 0.9345526099205017, -1.3783884048461914, 0.6585928201675415, -0.4809578061103821, 0.0926424190402031, -1.1693516969680786, 0.3869040608406067, -2.7369213104248047, -0.11660310626029968, -1.2038264274597168, -1.4035638570785522, 0.6938037872314453, -1.5259524583816528, -0.503929615020752, -0.3424050509929657, -0.08043396472930908, 0.005092478822916746, 0.3732878267765045, 2.6069676876068115, 0.9559135437011719, -0.027830801904201508, 1.2623798847198486, -0.4030139446258545, -1.0628714561462402, 1.1036992073059082, -1.1846446990966797, 1.4880635738372803, -0.1503058671951294, -0.44058990478515625, -0.10342293232679367, 1.7770891189575195, -0.662487804889679, -0.725542426109314, 0.9350367188453674, -0.7621080875396729, 0.34406936168670654, -0.9539306163787842, 0.823951244354248, -0.5337828397750854, -1.1555103063583374, 1.9812414646148682, 1.2218571901321411, -0.41009974479675293, 0.08871171623468399, -0.584248423576355, -0.7452601194381714, -1.0512902736663818, 1.7589102983474731, -0.6071892380714417, 0.09567611664533615, -0.42124003171920776, 1.0548003911972046, 1.5171114206314087, 0.6965346932411194, -0.4429984390735626, 0.2509828805923462, -1.9018168449401855, 1.1197854280471802, 1.073634386062622, -0.16226889193058014, 0.9459161758422852, -1.3119103908538818, -0.4186396300792694, 1.782127022743225, -1.2979168891906738, 0.5270909070968628, 0.3966837525367737, -0.1695953756570816, -0.459185928106308, 0.2406337410211563, 0.16401252150535583, -0.9874309301376343], [1.1453816890716553, 0.734766960144043, 0.6104494333267212, 2.082494020462036, 1.1813937425613403, -0.07560397684574127, -0.8817757964134216, -3.1466598510742188, -0.8858622908592224, 1.0289815664291382, -0.9871195554733276, -1.1186972856521606, -0.1662103682756424, -1.2000412940979004, -0.16166086494922638, -1.2760142087936401, -0.9789595007896423, -0.6710243225097656, 0.6636791825294495, 0.411289244890213, -0.09029223769903183, 0.12057869136333466, -0.49746784567832947, -0.12469532340765, -1.0656355619430542, -0.28221458196640015, 0.7636414170265198, 0.45799294114112854, 0.29260554909706116, 0.3194305896759033, 0.947566568851471, 1.1908957958221436, -1.4559364318847656, 0.9206664562225342, -0.3453752398490906, -0.5347960591316223, -1.271444320678711, 1.7648441791534424, -0.8379240036010742, -1.5564674139022827, -0.38589853048324585, -1.1540111303329468, -1.2141163349151611, 0.7464174032211304, -0.5339120626449585, -1.0320041179656982, -0.860295295715332, -0.010480405762791634, -0.5422960519790649, 2.4298644065856934, -0.11258145421743393, -0.2687399685382843, -1.4565984010696411, 2.0773751735687256, -1.6708050966262817, -0.4724671244621277, -0.9418349266052246, 1.671130895614624, 1.165168285369873, -0.6054382920265198, 1.3081530332565308, 0.23123544454574585, -0.518902063369751, 1.4333726167678833, 0.05821870639920235, -0.62473064661026, 0.7527378797531128, 1.4364521503448486, -0.18063728511333466, 0.18839940428733826, 1.3971621990203857, -0.5835351347923279, -0.1869777888059616, -1.2968380451202393, -0.20759204030036926, -1.0680851936340332, -0.2993607223033905, -0.7210599184036255, 0.3355618417263031, -0.6099351644515991, 1.59575355052948, -0.08912382274866104, -0.8969129323959351, 0.014807152561843395, 0.3262758255004883, 1.1439039707183838, -0.6136911511421204, -0.8461293578147888, 0.10197614133358002, 0.7837309837341309, 0.7672555446624756, -2.1253840923309326, -1.1342546939849854, 0.36561015248298645, 0.1604306548833847, -2.56056547164917], [-0.056981898844242096, -0.8989623188972473, 0.26137545704841614, 0.19691334664821625, 0.5220058560371399, -1.7225233316421509, -0.4561885595321655, -0.2664068937301636, -0.24733103811740875, -0.9156325459480286, -1.7000316381454468, -0.6991307139396667, -1.1000937223434448, -0.38814395666122437, 1.6932766437530518, 0.07204420119524002, -1.3634966611862183, -0.5598084330558777, -0.1916348785161972, 0.6584640145301819, -2.2638556957244873, 1.3245530128479004, -0.9030329585075378, -1.6532491445541382, 1.0284614562988281, -0.7597441077232361, -0.46362820267677307, -0.6478586792945862, 0.9978489875793457, 1.1269285678863525, -0.6982783079147339, -0.4782845973968506, 0.4388904273509979, -0.5935690999031067, 0.0875258520245552, 0.21667088568210602, -0.5177256464958191, -0.2438361793756485, 0.029949193820357323, -0.6790105104446411, 2.191842794418335, -1.251941442489624, 0.9860514402389526, 0.3243027925491333, 0.18755698204040527, -2.62406325340271, -0.8528687357902527, -0.04782133176922798, 0.774192750453949, -0.0185908954590559, 0.7008641362190247, -0.9118286967277527, -1.0143035650253296, 0.5756906270980835, 0.49544215202331543, -1.6127263307571411, 0.5909994840621948, -1.1297663450241089, -0.2453281432390213, 0.8604313731193542, -0.1497000753879547, 2.436599016189575, 1.5552908182144165, 0.736656665802002, -2.202146053314209, -0.4107207953929901, -0.5410231947898865, -0.22496727108955383, -0.2131582349538803, -1.788259506225586, 0.39242851734161377, -0.9396747350692749, -0.3589814007282257, 0.36776211857795715, 1.4122905731201172, 0.7162569165229797, -1.2820333242416382, 0.6469306349754333, 0.4665064215660095, -0.12645873427391052, 1.6139189004898071, -0.38792577385902405, 0.3476412296295166, 2.2179946899414062, -0.5633615851402283, -0.7440720796585083, 0.6617629528045654, 0.5196205973625183, 0.10005571693181992, -0.11638688296079636, 0.19965289533138275, 0.7700883150100708, -1.5564850568771362, 0.01765761710703373, 1.0768712759017944, -1.016036033630371], [1.116678237915039, 0.047591693699359894, 0.07614673674106598, -0.31156569719314575, 0.9570826292037964, -0.33231475949287415, 1.0664643049240112, -0.0008378541097044945, -0.11842000484466553, 1.4740031957626343, -1.5257118940353394, -0.4282357692718506, 0.35725417733192444, 1.319219946861267, -0.6006031036376953, -0.12284963577985764, -0.6059670448303223, 0.23678524792194366, -0.262746661901474, -1.246826171875, 1.1695795059204102, 0.05385966598987579, 0.014546552672982216, -0.506261944770813, 1.2441356182098389, -0.24092641472816467, 0.4608844518661499, 0.6067044138908386, 1.7701457738876343, 0.7296796441078186, -0.2808989882469177, -0.270981103181839, 0.3926229476928711, -1.215714693069458, 0.1744968742132187, 1.0081275701522827, 1.5194822549819946, 0.17232617735862732, 0.8211826086044312, 0.4051341116428375, -0.011022062040865421, -1.776801347732544, -0.3141663074493408, 0.9622313380241394, 0.07745322585105896, 1.172461748123169, -0.7530659437179565, 1.5240888595581055, -1.9178162813186646, -0.5176776051521301, -1.9229907989501953, -0.002468975493684411, -0.24473528563976288, 0.5576306581497192, 1.3157676458358765, 0.06086180731654167, 0.18811377882957458, -1.2256499528884888, 0.12379588931798935, -0.6472005844116211, -0.8264755606651306, -0.39033910632133484, 0.37530672550201416, 0.9636746048927307, -0.3431013822555542, -0.223097562789917, 0.7137653827667236, 0.6503947377204895, 0.5620886087417603, 0.4386025369167328, -0.7352032661437988, -0.4363519251346588, -1.108488917350769, 1.2213160991668701, 1.3290592432022095, -1.8124815225601196, 0.4415249228477478, 0.6406148076057434, -0.5844061970710754, -0.5133653879165649, -0.18134260177612305, 0.4237145483493805, 2.6910715103149414, -0.047748394310474396, 0.8732319474220276, -1.0782917737960815, -0.3211009204387665, -1.6105316877365112, 1.1395436525344849, -0.2618025243282318, 0.08036229014396667, -1.5690066814422607, 1.9989314079284668, 0.30566489696502686, -1.5769333839416504, -0.11856072396039963], [-1.2162057161331177, -0.21589191257953644, -0.8434592485427856, 1.0307356119155884, -1.4058009386062622, -0.5928440690040588, 0.520419180393219, 0.8039630055427551, -0.7299857139587402, -0.9572217464447021, -1.982008695602417, -1.6003128290176392, -1.3278263807296753, 0.4680219292640686, -1.7258135080337524, -1.0082404613494873, -0.014185260981321335, 0.5836926102638245, 0.16168320178985596, 0.844058096408844, -0.14939925074577332, -0.20324909687042236, -1.0786627531051636, -0.9309016466140747, -0.05427689850330353, -1.2953743934631348, -0.35218825936317444, 0.9631378054618835, 0.9175106883049011, -0.5476603507995605, -1.1384398937225342, -0.18927301466464996, 1.3832950592041016, 1.9208158254623413, -2.2480506896972656, -1.6174275875091553, 0.6361578702926636, 0.8953825235366821, 0.8245815634727478, 0.21398064494132996, 1.8340368270874023, -1.157168984413147, -1.7369612455368042, -0.027750683948397636, -1.124093770980835, 0.19305051863193512, -0.44359588623046875, 1.306749939918518, 1.4711357355117798, 1.5404287576675415, -0.37925490736961365, 0.4585639238357544, 0.6676915287971497, 0.5389013886451721, 0.06528017669916153, -0.3478536903858185, -0.5631411075592041, -1.3879187107086182, 0.7161977291107178, 0.2106562852859497, -0.8118163347244263, -0.11516343802213669, -2.1463098526000977, 1.552724003791809, 0.40055716037750244, -0.49238649010658264, 0.20636701583862305, -1.6110725402832031, 1.6659443378448486, 0.19110015034675598, -0.052433181554079056, -0.007781213615089655, 0.15386290848255157, -0.7547163367271423, 1.6354819536209106, -0.06567654758691788, -0.3610784113407135, -0.49826768040657043, 0.16343295574188232, -0.6577292680740356, -0.19707919657230377, -1.9408637285232544, -0.06280718743801117, -0.6550964117050171, 1.2965925931930542, 0.2934047281742096, -1.4543368816375732, 0.668190598487854, 1.0215225219726562, -0.5874514579772949, 0.3261151909828186, 0.015007497742772102, 1.4877780675888062, 0.788846492767334, -0.07596994936466217, -1.168887972831726], [-0.6502446532249451, 1.1840548515319824, 0.7842321991920471, -2.1922783851623535, -1.9957058429718018, 0.5814277529716492, -0.43778547644615173, -0.6248124241828918, 0.3516988158226013, -0.4291982352733612, -0.8997917175292969, 1.071770429611206, 1.7061861753463745, 0.31433698534965515, 0.034751418977975845, 1.2400672435760498, 1.7263201475143433, 0.0810059905052185, 0.6811189651489258, 0.0011371758300811052, 3.344388008117676, -1.3332659006118774, 0.031535662710666656, 0.15179793536663055, 0.005437117535620928, -0.12955552339553833, -1.7614359855651855, -0.5818546414375305, -0.3004626929759979, 1.64681875705719, -1.5276379585266113, -0.4239612817764282, 2.5664923191070557, 0.42612388730049133, 0.22923396527767181, 1.8603442907333374, 0.21620315313339233, 0.06621083617210388, 0.6071948409080505, -1.5294071435928345, -0.011058350093662739, -0.9015923738479614, -1.248058795928955, -0.6142850518226624, -0.3311147093772888, -1.016517162322998, 0.2871362268924713, -1.296924352645874, 2.3812410831451416, 0.4074687957763672, 1.1020818948745728, -0.12933598458766937, -0.5079724788665771, -0.3449902832508087, -0.4045672118663788, -0.28741195797920227, -1.0007603168487549, 0.25883403420448303, 1.4148401021957397, 0.4808637499809265, 0.2479233592748642, 0.4350014626979828, -1.7150620222091675, 1.5571359395980835, 1.3179504871368408, 0.34214940667152405, -1.0506510734558105, 2.300102472305298, 0.26571646332740784, -1.6198740005493164, 1.2816413640975952, -1.0848976373672485, -0.6614949107170105, 1.3528193235397339, 0.6658744215965271, 1.1343703269958496, 0.06273481994867325, -0.6700582504272461, 1.936523675918579, -0.019374577328562737, -0.16685548424720764, 0.5134979486465454, -0.30303558707237244, 0.42717206478118896, -0.9054185748100281, -0.24917329847812653, -1.1516404151916504, -1.380237102508545, 1.0663114786148071, 1.3079726696014404, -1.0684562921524048, -0.5496293902397156, 0.3630337417125702, 0.853280246257782, 0.5380480289459229, 0.8273037075996399], [-0.525283694267273, -1.7699366807937622, 1.624145746231079, 0.1813424676656723, 3.046755313873291, 0.5796144604682922, 0.41450151801109314, -0.553464412689209, -1.7484891414642334, 0.17373061180114746, -1.1558681726455688, -2.0499043464660645, -0.7482508420944214, 0.8413780927658081, -0.45720115303993225, 0.7028550505638123, -1.6765708923339844, 1.0677447319030762, 0.7062992453575134, -1.0135787725448608, -0.8064015507698059, 0.017229586839675903, 0.3688495457172394, 0.1742350161075592, -0.028306743130087852, -0.13243047893047333, 0.5343894362449646, 0.5309028625488281, 0.3799341917037964, -1.2253541946411133, -0.23956215381622314, -0.5478346943855286, 0.794349193572998, -0.6767521500587463, -0.5989671945571899, -0.5634809732437134, -0.4106055498123169, -1.7575582265853882, 0.004828039091080427, -0.7015771269798279, 0.1571141481399536, 0.9323076605796814, 1.0924698114395142, -1.2178367376327515, -0.1880410760641098, -1.0052355527877808, 1.1908066272735596, -0.7929452657699585, -0.6814943552017212, 0.10062377899885178, 1.0006498098373413, -0.545518159866333, -1.7627440690994263, -0.24540852010250092, -0.662222146987915, -1.130203366279602, -0.3943079710006714, -0.13446566462516785, -0.4559563100337982, -1.0863538980484009, -0.9784676432609558, -0.7018921971321106, -0.9066882133483887, -0.8903847932815552, 2.4535906314849854, -0.3453730642795563, -0.21886134147644043, 1.3404536247253418, -1.567399024963379, -0.6476058959960938, -2.3738791942596436, 1.5224395990371704, 0.9601825475692749, -0.6835514307022095, -1.5720824003219604, -1.042453646659851, 0.7008545398712158, -0.14991340041160583, -1.2133253812789917, -0.1883023977279663, 0.47110387682914734, -0.4625089168548584, 0.5738282799720764, -1.6848902702331543, -0.055467721074819565, 0.9301708340644836, 0.8795902132987976, 0.4483359158039093, -0.33449211716651917, -1.5253719091415405, 2.1827220916748047, -1.0960532426834106, -0.3564727008342743, 2.093109607696533, -0.1444970816373825, -1.3983060121536255], [0.024050015956163406, -0.07426247000694275, -0.1144498661160469, -1.2108012437820435, 1.5561842918395996, 0.1559082716703415, 0.07479938864707947, -1.22428297996521, 1.0254606008529663, 2.3202786445617676, -0.3033042848110199, -1.0776646137237549, 0.35173559188842773, -2.021010160446167, 0.2591525912284851, -1.4896081686019897, -0.12004587799310684, 1.933388590812683, -0.6535043716430664, -2.1790053844451904, -0.5469116568565369, 0.7717640995979309, -1.2765153646469116, 0.6561295390129089, -0.22698941826820374, 0.039231158792972565, -1.3527864217758179, -0.36793065071105957, -0.6834731698036194, -0.01069632638245821, -0.8546491265296936, -1.3505593538284302, -2.1301329135894775, -0.31335821747779846, -0.526177704334259, 0.5824574828147888, -0.37669074535369873, -1.2081959247589111, 0.5385795831680298, -1.7501685619354248, -1.20253324508667, -0.19278135895729065, -0.43067988753318787, -0.6090670824050903, -0.11799473315477371, 1.0391197204589844, 1.3575583696365356, -1.0063668489456177, -1.1320756673812866, 1.3884273767471313, -0.4066166877746582, 0.27829092741012573, -1.147624135017395, 0.6435712575912476, 1.6621463298797607, 0.8378666639328003, -0.9726304411888123, -0.011624407023191452, 0.46135255694389343, -0.7714359164237976, 0.1471574902534485, -1.7545037269592285, -1.0076630115509033, 1.0221365690231323, -0.5755795240402222, 0.45351988077163696, -0.07058651745319366, 0.5801573395729065, 0.4405611753463745, 0.3441402316093445, 0.22851000726222992, -0.3207760751247406, -0.992706298828125, -0.3704187273979187, 0.5139621496200562, -0.7615489959716797, -0.1208299919962883, 0.5086022019386292, -0.4070299565792084, 0.23149003088474274, -0.7281211614608765, -0.06739875674247742, 0.4206794798374176, -0.36677974462509155, 1.7184710502624512, 0.2753216028213501, -0.15969125926494598, 0.2870340347290039, 0.6516790986061096, -1.4571529626846313, 1.169602394104004, 0.16548509895801544, 1.4559463262557983, -0.0028213136829435825, -0.49178174138069153, 1.2055660486221313], [-0.20520715415477753, 0.12850138545036316, -0.7254571914672852, -1.064810872077942, -0.28542274236679077, -1.2894688844680786, -0.3382706642150879, -0.6067303419113159, -0.14077548682689667, -0.8311799764633179, 0.43464797735214233, -0.3539678454399109, 3.7365529537200928, 1.2157316207885742, 0.9461154937744141, 0.9932973980903625, 1.8681622743606567, 2.2451171875, 0.13728247582912445, -0.25935375690460205, 0.20394866168498993, 1.8638538122177124, -0.16371652483940125, 0.27633729577064514, -0.4995567500591278, -1.0520782470703125, -0.48760274052619934, 0.7204124331474304, -0.0699893906712532, -1.7267529964447021, 0.3298541307449341, -0.8987631797790527, -1.086361289024353, -0.308270126581192, -2.8965060710906982, 1.375853419303894, -0.6037130951881409, -1.1432232856750488, 0.24301715195178986, 1.6098188161849976, 0.16390977799892426, 0.8184814453125, -1.3736025094985962, -0.1896800696849823, -0.09796332567930222, -0.9444130063056946, -1.165950059890747, -0.5886985659599304, 0.17903362214565277, 2.3118293285369873, -0.23341305553913116, -0.23619376122951508, -1.468105435371399, 0.8212952017784119, -0.33283737301826477, 1.9637447595596313, -1.103440523147583, 1.0334333181381226, 0.5412747859954834, -0.2856195569038391, 0.9225008487701416, -1.4528729915618896, -0.32379889488220215, 0.5572443604469299, 0.6838892698287964, -1.4385675191879272, 2.687242269515991, -0.9632949829101562, 0.9682213068008423, -1.0175975561141968, 1.4071842432022095, 0.7952014803886414, -0.05950813367962837, -0.3531583249568939, -0.8467565774917603, -1.112862229347229, 1.1163535118103027, 0.9745959043502808, -0.4764440059661865, -2.0271120071411133, 0.5990593433380127, 0.6737715601921082, -0.20700226724147797, 0.7377405166625977, -0.058727771043777466, 0.24714075028896332, -0.35541823506355286, 0.8678978085517883, 1.551814079284668, -0.6724025011062622, -1.0713915824890137, 1.4681740999221802, 1.4797050952911377, 0.7655118703842163, 0.20890618860721588, -2.2720398902893066], [2.0305073261260986, -1.2027153968811035, 0.2405005693435669, 0.2000628560781479, -2.0200910568237305, 0.5939080119132996, -0.7518969774246216, -0.1845633089542389, -0.7199134230613708, -0.6830205321311951, 0.1469334214925766, -0.3747645914554596, -0.7185173034667969, 0.039685215801000595, -0.5876916646957397, 1.4181854724884033, -0.2028535157442093, 0.023662641644477844, -1.046823501586914, 1.6661335229873657, 0.28673648834228516, -0.40326988697052, 0.9509480595588684, 0.8940321207046509, 0.5980756878852844, 0.5126367211341858, 1.5047781467437744, -0.8666093349456787, -0.2994614541530609, 0.26563388109207153, -0.5243228077888489, 1.063114047050476, -0.7430933117866516, 1.765916109085083, -1.7189221382141113, -0.15883013606071472, -0.48319822549819946, -0.3016045391559601, -0.25223061442375183, -0.49157771468162537, -1.1259714365005493, -1.5392550230026245, -0.14377306401729584, -0.8182833194732666, 0.5948428511619568, 1.035590410232544, 0.7544728517532349, -1.0098013877868652, 0.029994428157806396, -2.169816493988037, -0.8953929543495178, -0.8249944448471069, 0.6042911410331726, 0.7055587768554688, -0.0409061498939991, -1.4412424564361572, 1.190119743347168, -2.3065853118896484, 1.3927873373031616, 1.3691476583480835, -0.4240562319755554, -1.108775019645691, -0.47786185145378113, -2.038276433944702, 0.1767989695072174, 0.48518943786621094, 0.668807327747345, -0.4069374203681946, 0.5704875588417053, 0.7279192209243774, -1.1173574924468994, 1.3009315729141235, 1.3785898685455322, -0.13541264832019806, -2.465502977371216, 0.9972277283668518, 0.12151435762643814, 0.7274248600006104, -0.6664616465568542, 0.29693371057510376, -0.37806716561317444, -1.5576362609863281, 0.40157270431518555, 0.8615834712982178, -1.2312263250350952, 2.42618989944458, 0.7705395817756653, 1.2575478553771973, -1.587593674659729, -1.4083075523376465, -0.06424251198768616, -0.3795948028564453, -1.754629373550415, -0.33867624402046204, -0.19072137773036957, 0.548020601272583], [-0.5875810384750366, 0.02746378444135189, 1.050307035446167, -0.07679152488708496, -0.9129804372787476, 0.9584649205207825, -0.15585821866989136, -0.09679211676120758, 1.4254542589187622, 0.0316004678606987, -0.37551766633987427, -0.03184415400028229, 0.39404061436653137, 0.9707001447677612, -0.8867321610450745, -2.203582286834717, 0.33376792073249817, -2.27298903465271, -0.5715442299842834, 0.39358100295066833, 1.0231274366378784, -0.26081469655036926, 0.46860411763191223, 0.07369931787252426, 0.38885626196861267, -0.7215287089347839, 0.577692985534668, -0.04572542384266853, -1.0716233253479004, 1.0251407623291016, -1.2538905143737793, 0.10678960382938385, -0.7154750227928162, 0.45096835494041443, 0.6070055365562439, 0.8951442837715149, -3.0059866905212402, 1.118963599205017, 0.17502470314502716, 0.4587636888027191, 0.9050478339195251, 0.8181913495063782, 0.9067307114601135, -0.19220934808254242, 1.316817283630371, -0.29953014850616455, 0.57820063829422, -1.034675121307373, 0.585539698600769, 0.2067999690771103, -1.2911285161972046, -0.37866953015327454, -0.5291301012039185, -0.5546607375144958, -1.1178758144378662, 1.924458384513855, 1.4934171438217163, -1.3802285194396973, 0.6666709184646606, -0.27376115322113037, -1.798061490058899, 0.4515683650970459, 1.5786356925964355, -0.25416114926338196, 0.2113751620054245, -0.4769642949104309, -0.071415476500988, 0.018445611000061035, -0.7896828651428223, 0.1303723156452179, -0.8383824825286865, 1.1136020421981812, 0.22666779160499573, 0.15028725564479828, -1.2228869199752808, -0.33228278160095215, -0.06735877692699432, -2.5992825031280518, -1.8401433229446411, 1.1259626150131226, 2.3409583568573, 0.39950937032699585, -0.3389744460582733, -1.4944816827774048, 0.4437982439994812, -0.2093687355518341, -2.0569159984588623, 0.2313779890537262, -1.3450218439102173, 0.7946959733963013, 0.233688086271286, -0.344667911529541, 0.6892362833023071, 0.48721352219581604, -0.005661176051944494, 1.4840826988220215], [0.272768497467041, -0.16012588143348694, -0.6348145604133606, 0.4405023157596588, -1.325445532798767, 0.3375944197177887, -0.6609233617782593, -0.7721601128578186, 0.13831818103790283, 1.1740853786468506, 1.7876594066619873, 2.357309341430664, -0.2540733814239502, 1.0459219217300415, -0.3328624367713928, -0.5443784594535828, 1.1949615478515625, 0.2697087228298187, 0.3565402030944824, -0.6806763410568237, 1.301712989807129, -0.6210419535636902, 0.042457424104213715, -0.34066617488861084, 0.12665419280529022, -1.0187618732452393, 1.2153831720352173, -0.1655302792787552, -0.8906795382499695, -1.0480815172195435, -2.624455213546753, -0.0936279296875, -0.5554296970367432, -0.8134810924530029, -0.9334595203399658, -0.6585236191749573, -2.5090816020965576, -0.04625615477561951, 0.39290058612823486, 0.2418978065252304, -1.3734997510910034, -0.4632795751094818, 0.7979562878608704, 0.5145686864852905, 1.8528645038604736, -0.055973030626773834, -0.40847721695899963, -0.8246744871139526, -0.1992262303829193, -2.0062060356140137, 0.06971905380487442, 0.34842416644096375, 0.4385640025138855, -0.6547000408172607, 0.5527017712593079, -0.23333419859409332, -2.015831708908081, 0.8311728835105896, -1.7128617763519287, 0.3168938159942627, -0.269853413105011, -1.6016438007354736, -0.5724551677703857, 0.22400963306427002, 1.1570823192596436, -0.7718369364738464, 0.15585055947303772, 1.3969889879226685, -0.8173588514328003, -0.3999935984611511, -1.0309330224990845, -0.981657087802887, -0.49410244822502136, 0.07942772656679153, 0.32312434911727905, -0.017330214381217957, -1.9460724592208862, -1.4377458095550537, 2.179955244064331, -1.0108877420425415, -0.9969037771224976, 0.3039849102497101, 0.6852980852127075, -0.342941552400589, 0.6172073483467102, -2.0148298740386963, 0.3218078017234802, 1.6122227907180786, 0.3358592391014099, -1.9253053665161133, 0.34769323468208313, 0.003918359521776438, -0.12894774973392487, 1.856748342514038, -1.8383821249008179, 0.5937923192977905], [-1.6788893938064575, 2.18109130859375, 1.7699685096740723, 1.1618784666061401, -0.8504844903945923, -0.28312644362449646, 0.8677034378051758, 0.9815848469734192, 1.0132761001586914, 1.4406393766403198, -1.7303169965744019, 1.020591378211975, -0.7980299592018127, 0.6926171183586121, -0.5333484411239624, -0.3444004952907562, -1.352749228477478, -0.4949721097946167, -1.9798740148544312, -0.6048232913017273, -1.073073387145996, -0.8795639872550964, 0.30430397391319275, 1.0596299171447754, -1.2683923244476318, 1.245994210243225, 1.7227234840393066, -1.3516380786895752, 0.1869441270828247, -0.41785916686058044, 1.087266206741333, -0.047337356954813004, -1.0193865299224854, 1.0994023084640503, -0.5394179224967957, -0.5037531852722168, 0.20834527909755707, 0.13064995408058167, -0.4699561595916748, 2.0027852058410645, 0.8589077591896057, -0.5309668183326721, -0.2646041512489319, -1.2126028537750244, -0.6282755732536316, -0.030415114015340805, 1.0719408988952637, -0.1956489235162735, -0.38984596729278564, -0.2204776406288147, 0.9763778448104858, -0.8529828190803528, 1.0751609802246094, -0.42420727014541626, 0.1651744544506073, 0.48264947533607483, -0.06710823625326157, 0.5079697370529175, 1.8900227546691895, -0.691509485244751, 1.8339314460754395, -1.4819403886795044, -0.7274031639099121, 0.4721861481666565, 0.6261669397354126, -1.826695442199707, 0.8979287147521973, 0.5749494433403015, 1.3069045543670654, -1.7893558740615845, -0.14207150042057037, -0.7761998176574707, -0.6644382476806641, 2.712294340133667, 0.3535405993461609, -0.5252808928489685, 1.0148801803588867, -0.3468506336212158, 1.0109859704971313, 1.3427125215530396, 0.4754645526409149, -1.9957003593444824, 1.0261061191558838, -1.0228599309921265, 1.6614863872528076, 0.3104143738746643, -1.1170804500579834, 1.5777140855789185, -0.01517704501748085, 1.7217168807983398, 0.15462613105773926, 0.2867431342601776, -0.07223450392484665, 1.280221700668335, -0.1966591328382492, -0.29577696323394775], [-0.7517362236976624, 0.5661882162094116, 0.6043401956558228, -0.009317469783127308, -0.6708378791809082, -2.0002760887145996, 0.026179654523730278, 0.966320812702179, 0.8855729699134827, -0.4442782998085022, -0.10245900601148605, -0.36523154377937317, 0.5373997092247009, -1.20217764377594, -1.2663779258728027, -0.5057381391525269, 0.7959527969360352, 0.07003208249807358, 1.2082902193069458, 0.36401253938674927, 1.4111645221710205, -0.3952288031578064, 1.1202036142349243, -0.2810516059398651, -1.6854645013809204, -0.533873975276947, -0.30895429849624634, -0.6466382145881653, -0.07815756648778915, 0.6864622235298157, 0.47576597332954407, 1.0314100980758667, 0.04632531479001045, -1.110945224761963, -0.6810529232025146, -1.485124945640564, -0.1088743731379509, -2.4302845001220703, -0.9249282479286194, 0.42798954248428345, 0.7537715435028076, 1.5733147859573364, -0.35580888390541077, -0.585422694683075, -0.8156148195266724, 0.11092951893806458, 0.36423587799072266, 1.1786720752716064, -1.3639788627624512, 0.9304026961326599, 1.5915857553482056, 0.6065329313278198, 0.5823027491569519, -1.1227478981018066, -1.666643738746643, 0.4550555944442749, 0.5779927968978882, 0.29503852128982544, 1.129899024963379, 0.06375010311603546, 0.4395408034324646, -1.8451433181762695, 1.0763291120529175, 0.44867947697639465, 1.1370370388031006, -1.2475614547729492, -0.688624382019043, 0.8494996428489685, 1.1701806783676147, 0.8218140006065369, 0.5047253966331482, 0.21454495191574097, 0.03694426640868187, 0.9952777028083801, -1.8725577592849731, 0.7414180636405945, 0.6937347650527954, -1.1759552955627441, -0.24828603863716125, -1.8944758176803589, -0.37614113092422485, -1.0689926147460938, -1.5753194093704224, 1.273140788078308, -0.6583883762359619, -0.8466948866844177, 0.4728873074054718, -0.7505086660385132, -0.2768947184085846, -1.0842307806015015, -1.2543423175811768, 0.9681221842765808, 0.42014816403388977, 0.10787276178598404, 0.9058806300163269, 0.5039677619934082], [1.4408588409423828, 0.013707969337701797, -0.6324089765548706, 0.9459042549133301, 1.395884394645691, 0.3992070257663727, -0.985650897026062, -2.1609725952148438, -1.3614344596862793, -0.9323062300682068, -0.4232613146305084, 0.3491286039352417, 1.2904038429260254, -1.6677321195602417, 0.5341277718544006, 0.8752820491790771, 0.7371194958686829, -0.5994114279747009, -0.5559695959091187, 1.1604517698287964, -3.852165460586548, 1.3093336820602417, 0.29037967324256897, -1.0597506761550903, -1.7991392612457275, -0.4783659279346466, -0.3370274603366852, 0.5182023644447327, 1.3727651834487915, -0.3392285406589508, -0.17671997845172882, 0.5287436842918396, 0.2528584897518158, 0.29694125056266785, 0.24357782304286957, -0.3000488877296448, 0.5040662884712219, -0.9376864433288574, 1.8292279243469238, -0.17040951550006866, 1.653876781463623, -0.9429283738136292, 0.6515690088272095, -1.1887876987457275, 0.1735764890909195, 0.5660556554794312, -0.10186631977558136, -1.0872204303741455, 0.35246357321739197, -1.31636643409729, 0.6815548539161682, -0.37226173281669617, 0.26908543705940247, -1.0928266048431396, -0.9216421246528625, 0.3235531747341156, -0.0995873212814331, -1.9049451351165771, 2.125230312347412, -2.2919535636901855, 0.7555210590362549, 0.13403533399105072, -0.8860841393470764, 0.45170342922210693, 1.2238543033599854, 0.7742140293121338, -0.4754756987094879, -1.5112160444259644, -0.010829566046595573, -2.456770420074463, 1.5067026615142822, -0.445852130651474, 0.43434804677963257, 0.8050114512443542, -1.8417140245437622, -0.9680891633033752, 0.22601155936717987, 0.42046889662742615, 0.12707507610321045, -1.728732943534851, 0.2547750174999237, 0.9706703424453735, -0.8063328266143799, -0.40588390827178955, -0.2173093855381012, -0.9521093368530273, 2.761732339859009, 0.6426706910133362, 0.8597468137741089, -0.9690319299697876, -1.2927618026733398, 0.4828975200653076, 0.5975548028945923, -0.43114736676216125, 0.5635680556297302, -0.458122581243515], [1.0139974355697632, -0.4966607093811035, -1.1107200384140015, 0.08650165796279907, -1.32437264919281, 0.4861702024936676, 0.6825398206710815, -0.5045374631881714, 2.6863551139831543, -0.3961365818977356, -0.40115928649902344, 0.19902515411376953, -1.0830756425857544, -0.6221067905426025, -0.3435022830963135, 0.7430158257484436, 2.200406312942505, -2.95154070854187, 1.4495793581008911, -0.39129894971847534, 0.4397670328617096, -0.9592656493186951, 0.8998770713806152, 0.859887957572937, 2.2495779991149902, 0.6174808740615845, -0.40784671902656555, -0.4957192838191986, -3.4715161323547363, -2.5907235145568848, 0.937604546546936, 1.4286670684814453, 0.0009908821666613221, -1.2931498289108276, -0.08814699202775955, 0.7992043495178223, 1.484811782836914, -0.7685871720314026, 0.28280892968177795, -1.0126272439956665, -1.464430570602417, -0.8121821284294128, -0.03718011453747749, 0.3593577444553375, -0.4054108262062073, 0.22251710295677185, 0.26203468441963196, 0.001970029668882489, -1.1788541078567505, 0.7318255305290222, 0.856705367565155, 0.4074380695819855, 0.296223521232605, -0.7322273254394531, -0.3025667369365692, 1.2357423305511475, -1.154700517654419, 0.9814739227294922, 2.5413103103637695, -1.2359219789505005, 0.38869795203208923, -1.622696042060852, 0.8523505926132202, -0.1095070019364357, -0.0927577093243599, -0.6556758284568787, -0.05396425724029541, 1.4777947664260864, -0.8745120167732239, 0.8328232765197754, -1.1805419921875, -0.17480000853538513, -0.5500271320343018, 2.552555561065674, 0.0697975605726242, -1.454836368560791, 0.05165252834558487, -0.7944586277008057, 1.120506763458252, -1.9049474000930786, -0.1925811618566513, 0.9170861840248108, -1.0458675622940063, -1.0233147144317627, 1.3199056386947632, 0.19758287072181702, -0.6528744101524353, 0.24552659690380096, -0.535018265247345, 0.4512055814266205, -1.5123308897018433, -0.5321947336196899, -1.3416694402694702, -0.7012051343917847, 2.5942392349243164, -1.7452130317687988], [-1.4883275032043457, -0.7899966239929199, -1.3510252237319946, -1.307806134223938, 0.6534383296966553, 1.1191167831420898, 0.5105189085006714, 1.2082023620605469, -3.593231201171875, 0.043784432113170624, -0.7302325367927551, 0.42270562052726746, 0.03294806554913521, -0.3631855845451355, 0.16129247844219208, 0.015703294426202774, 1.5872166156768799, -1.3909494876861572, 2.2693138122558594, -0.2708352506160736, -0.2146226167678833, 0.14034157991409302, -0.38632750511169434, 0.3575234115123749, -0.0814099833369255, 0.41292834281921387, -1.717434287071228, -0.9078443050384521, -0.8624849319458008, -1.0391765832901, 0.5023599863052368, 0.766029953956604, -1.2133806943893433, 0.6713598966598511, -1.2586373090744019, -1.9028033018112183, -1.007853388786316, 1.243224859237671, 0.2805533707141876, 0.5916670560836792, 0.10244731605052948, 0.4051184356212616, 0.07962394505739212, -1.3571388721466064, 0.10906961560249329, -1.1434558629989624, 0.03537662327289581, 0.5436141490936279, -1.297612190246582, 1.505203366279602, -0.21498987078666687, 1.5337871313095093, -0.47894686460494995, 0.24056243896484375, -0.2057839184999466, 0.6749655604362488, 1.0473824739456177, 0.7499487400054932, -0.5096343755722046, 1.608832597732544, 1.5401197671890259, -0.5491004586219788, 0.7778387069702148, 0.7469237446784973, -0.6298142075538635, 0.8087716102600098, -1.3297430276870728, 1.1867575645446777, 0.45768484473228455, 1.1262437105178833, 0.12763474881649017, -0.017926551401615143, -0.3530576825141907, 1.4212040901184082, 0.9942249655723572, 0.6118272542953491, -0.21171268820762634, 2.2210452556610107, 0.16899892687797546, -1.6011916399002075, 1.4953556060791016, 0.7553986310958862, -0.41110363602638245, -2.0462911128997803, 0.3894886076450348, 0.16611385345458984, -0.13779784739017487, 1.1228810548782349, -1.5705713033676147, -0.6220701932907104, -1.23415207862854, -1.291395902633667, -0.6318552494049072, -2.0285675525665283, -0.9112683534622192, 0.7239915728569031], [-0.0852171778678894, 0.4336502254009247, -0.15598247945308685, -0.07932554185390472, -0.6699812412261963, -0.40100598335266113, 0.16931389272212982, 1.0857677459716797, 0.02301022782921791, 0.07995103299617767, 0.14352214336395264, -0.5273150205612183, 0.4071812033653259, 1.2149837017059326, -0.7750453352928162, -0.7644844055175781, 0.9737403988838196, -0.9493513703346252, -0.9060689806938171, 1.0783030986785889, -0.8110003471374512, -0.705752432346344, -1.3064159154891968, -1.1783757209777832, 1.9244091510772705, 2.547081232070923, -0.5272589325904846, 0.41792312264442444, 1.4297020435333252, 0.12222035974264145, 0.4840848445892334, -0.747627854347229, -0.7296427488327026, -0.14617189764976501, -1.222357153892517, -0.47637543082237244, -1.987004280090332, 0.9591853618621826, 1.3860478401184082, -0.7952947020530701, -0.5587637424468994, -0.5162286758422852, 1.4274758100509644, 2.107025146484375, -0.3872077465057373, -1.1040316820144653, 0.10271059721708298, 0.8737886548042297, 1.4845690727233887, 0.4299362003803253, -0.7883785367012024, 0.5007784962654114, 0.5770206451416016, 1.3936458826065063, -0.6977832317352295, 0.0265607051551342, -0.5841132998466492, 0.5223527550697327, -0.9362531304359436, 0.4478360414505005, 0.7252258062362671, -1.4669462442398071, -0.8279346823692322, 0.2522854208946228, -0.898431658744812, 0.42326590418815613, 0.21465036273002625, 0.5063678026199341, 0.7057658433914185, -1.0203328132629395, -0.4804917871952057, -2.2685084342956543, 0.030024411156773567, -1.6848641633987427, 1.5774226188659668, -0.1593824177980423, 2.0216877460479736, -0.3776370882987976, -0.30200478434562683, -0.3120516836643219, 0.03043493814766407, 0.6135185956954956, 0.19806717336177826, -2.2764439582824707, -0.32032862305641174, -0.043870244175195694, -0.03735256940126419, 0.7728177905082703, -0.9668079614639282, 0.0012415562523528934, 0.4089376628398895, 0.4710054397583008, -0.031530581414699554, 0.8570906519889832, 0.05495024099946022, 0.29885333776474], [-0.6115185618400574, -0.09780479967594147, -1.014026165008545, -0.41996362805366516, -1.2665055990219116, -1.2986027002334595, 0.9324052929878235, 0.7175065279006958, 0.2144274264574051, 0.6426364779472351, 0.13218723237514496, -0.29449015855789185, 1.4104893207550049, -1.1794698238372803, -0.14191962778568268, -0.1810327023267746, -0.043297991156578064, -0.32860589027404785, 0.40444833040237427, 1.7863118648529053, -0.9585207104682922, 1.0313242673873901, -0.1157739907503128, 0.6009297966957092, -0.1627255082130432, 1.2224969863891602, -0.9568083882331848, 2.3674991130828857, -0.6063680052757263, -0.2789236605167389, -0.5561293363571167, 0.5756704211235046, -3.2069828510284424, 0.30262109637260437, -0.5918357372283936, 1.3952909708023071, -0.6696699261665344, 1.4124274253845215, 1.1638720035552979, 1.8625670671463013, -3.085963010787964, 0.6771887540817261, -0.42593252658843994, -0.34061458706855774, -0.12647853791713715, 0.7732170224189758, -0.20819033682346344, 1.49164879322052, 2.5338592529296875, 1.2048717737197876, 0.20973928272724152, 0.8432213068008423, 1.0989936590194702, -0.0963946282863617, 0.38830286264419556, 0.6554950475692749, 0.02727774903178215, 0.8499489426612854, -1.6584618091583252, -1.4275116920471191, -0.8758272528648376, -3.861528158187866, 0.6214864253997803, -0.5151300430297852, 0.6534560322761536, -0.6389850378036499, 0.5287356972694397, -0.8509553074836731, -1.0773664712905884, 0.6545868515968323, -0.5417629480361938, 0.8711471557617188, -0.3155210018157959, 0.37225791811943054, 0.4786057472229004, -0.45227929949760437, -0.6878065466880798, -1.0537070035934448, -0.28734496235847473, -1.0825185775756836, -0.13261768221855164, -0.799029529094696, 0.3476433753967285, -0.6466544270515442, -0.9858987927436829, 0.5468904972076416, 1.3043254613876343, 0.0062436675652861595, 0.1637030392885208, 0.8277908563613892, 1.1951402425765991, -0.6131725311279297, -2.0227510929107666, -0.06204375624656677, 1.3807554244995117, -1.252343773841858], [2.2228455543518066, 2.273552179336548, -1.485344409942627, 1.1522469520568848, -2.2119956016540527, 0.6275163888931274, 0.6195469498634338, 0.6837811470031738, 1.2472679615020752, -0.17574253678321838, 1.3128334283828735, -0.9412449598312378, -0.7163158655166626, -0.49799710512161255, -0.0156287532299757, 1.4153916835784912, -0.03028310090303421, -0.6953220963478088, -0.38512009382247925, -0.679520845413208, -0.5235272645950317, 0.33952584862709045, -0.12060526013374329, 0.7741318345069885, 1.1227566003799438, -0.07544009387493134, 0.7368286848068237, -0.04652564972639084, -0.9751104712486267, -1.1630452871322632, 1.1543407440185547, 0.1180194541811943, 0.5627546310424805, 0.5370272994041443, -0.19775395095348358, 1.4393131732940674, 1.376475214958191, 0.2972905933856964, 0.12489701807498932, -0.28188079595565796, -0.7735673785209656, -1.0343670845031738, 0.036076582968235016, -1.747153401374817, 0.03698471188545227, -0.14466746151447296, -0.920242965221405, 0.6402252316474915, 0.8059341907501221, 0.23519587516784668, 0.8183708190917969, -0.07189983129501343, -0.9231435656547546, 0.8890631198883057, -1.1426451206207275, 1.5816192626953125, 0.8203930854797363, 1.4381998777389526, 1.2364603281021118, -0.3074013590812683, 1.1877943277359009, 0.9809601902961731, -1.5408499240875244, -0.639244794845581, 1.3752998113632202, -0.6477894186973572, -1.7849172353744507, 0.09426312893629074, -0.3014419972896576, 1.870874285697937, 0.3021349608898163, -1.1383541822433472, 0.19344209134578705, 1.215081810951233, 0.1032227873802185, 1.3569469451904297, -1.4538414478302002, 0.4371402859687805, 0.33793190121650696, 0.5737248063087463, -0.18572615087032318, -0.09815976768732071, -0.3793196976184845, -1.5732396841049194, -0.05663437396287918, 0.1658540517091751, 1.0645209550857544, 0.47955116629600525, -0.03649505227804184, 0.5192960500717163, -0.12194181233644485, 0.489276260137558, 0.709184467792511, 1.1387637853622437, 0.08939504623413086, -0.1094549149274826], [-0.9098944664001465, 1.7517004013061523, 1.256609559059143, -0.5294193625450134, 0.042129501700401306, 0.2919818162918091, -0.4195388853549957, -0.1811223030090332, -0.07920384407043457, -0.9996359348297119, 0.7971836924552917, -2.075632095336914, -0.2194206416606903, 0.45403704047203064, 0.29554852843284607, -0.4131593108177185, -0.2655014991760254, -0.08448304980993271, -0.14776545763015747, 1.5615253448486328, 0.15088872611522675, 1.8607981204986572, 0.2940182089805603, 1.3974615335464478, -0.17616978287696838, 0.5638839602470398, 0.14238667488098145, -1.023450493812561, -0.9095591306686401, 0.5865501165390015, -0.9108880162239075, -1.800642728805542, 1.6283010244369507, -0.6919599175453186, 2.3160643577575684, 0.18728840351104736, 0.08085546642541885, 1.7866138219833374, 1.1308014392852783, 0.5426053404808044, 0.2861367166042328, 0.43517017364501953, 2.1466963291168213, 0.5041483640670776, 0.5160337090492249, 1.4131507873535156, -0.4259520471096039, -0.8461704254150391, 0.4418134093284607, -0.4196498990058899, 1.1584481000900269, 0.015664823353290558, -1.7099733352661133, 0.25845685601234436, -0.8187965154647827, 0.8600371479988098, 0.0717795118689537, 1.5834866762161255, -0.3716849088668823, 0.47662878036499023, -0.871040940284729, 0.4927894175052643, 1.6262503862380981, -0.007911463268101215, -0.6501232385635376, -0.0936901643872261, 1.1484854221343994, -1.3685110807418823, 0.26807907223701477, 0.46174299716949463, 0.0830775797367096, 1.4589520692825317, -0.11645497381687164, 0.33213019371032715, 0.3003397285938263, -0.5123298764228821, 0.24896694719791412, -0.23852886259555817, -0.22484846413135529, 0.05754859372973442, -0.9367002248764038, 0.6619271039962769, 0.8107920289039612, -0.23397304117679596, 0.886955738067627, -2.3252267837524414, -0.3569696247577667, 1.2198233604431152, 0.6210957169532776, 0.24114525318145752, -0.02735585905611515, 0.31329289078712463, 0.972241997718811, 2.3968796730041504, -1.3396834135055542, -0.7489110231399536], [0.3713013231754303, -0.5900385975837708, 1.0924243927001953, 1.1671051979064941, 0.9894277453422546, 0.4379468262195587, 1.0725069046020508, 1.2735958099365234, 0.1724964678287506, 0.14328932762145996, -0.6244006156921387, -1.5364868640899658, -0.2557394504547119, 1.0492868423461914, 2.4906673431396484, -0.728358268737793, -0.9757213592529297, 0.7367355823516846, 1.1029763221740723, 0.7581115365028381, 1.4191936254501343, -0.23130813241004944, -0.10406361520290375, 1.240319848060608, 0.7151231169700623, 0.9024981260299683, 2.4264743328094482, 0.5760591626167297, 0.9827942252159119, -0.3535975217819214, -0.12807993590831757, -0.186215341091156, -0.42308661341667175, -0.5955061912536621, 0.05380682647228241, -1.452799677848816, -0.3199688196182251, -0.35189542174339294, -0.3627212345600128, -0.2719605565071106, -1.5358564853668213, -0.07827621698379517, 1.5593914985656738, 0.48936721682548523, 0.290479451417923, 0.3536887466907501, 0.5473164916038513, -0.3865486681461334, -0.6328052282333374, 0.8377465605735779, -0.06961629539728165, -1.291856050491333, -0.22728466987609863, 1.234125018119812, -0.23532308638095856, 0.975365161895752, 0.9564276933670044, 0.34627771377563477, -2.2146313190460205, -0.03353509679436684, -0.40943875908851624, 0.7391632199287415, -0.7188723087310791, -0.3766115605831146, -0.39523112773895264, -0.2479410320520401, 0.822365939617157, -0.3092300295829773, -0.1340097337961197, -0.8737068772315979, 0.08002358675003052, -0.11881175637245178, -0.6192734241485596, 1.9134643077850342, 0.32769864797592163, -0.2850843369960785, -0.5637667179107666, -0.927241325378418, -0.287007600069046, 1.5293529033660889, 0.0037260903045535088, -1.5013322830200195, 0.6037412881851196, -0.22050189971923828, 1.261490821838379, 1.5799901485443115, 1.5769654512405396, -0.7398552298545837, -0.5939614176750183, -0.7272220849990845, 0.9186690449714661, 1.1261898279190063, -0.6469132900238037, -0.13113559782505035, 0.71834397315979, -0.5323953628540039], [1.1783013343811035, -0.7157368063926697, 1.4613398313522339, -1.7235227823257446, -0.41076454520225525, -0.9286868572235107, -0.08376222103834152, 1.2266608476638794, 0.389724463224411, 0.06311219930648804, 0.1771399974822998, 0.6665477156639099, 0.502915620803833, 0.408482164144516, -0.17121174931526184, 0.6437785029411316, 0.3304116129875183, -0.4500385820865631, 0.011109787039458752, 0.4708574116230011, -1.052047610282898, -0.45511651039123535, 0.11259864270687103, -0.9760291576385498, 0.7852494716644287, -1.2712072134017944, -0.22346746921539307, 0.8611302971839905, 0.38284316658973694, 1.4768925905227661, -0.17922888696193695, -0.70545893907547, -0.17678192257881165, 0.36692407727241516, 2.311479330062866, 0.07552164793014526, -0.23154762387275696, 0.4181171953678131, -0.8387276530265808, -0.9894188642501831, 0.41336238384246826, -1.412053108215332, 2.090955972671509, -1.566011905670166, 0.5779523253440857, 1.0677614212036133, -0.9286702275276184, 0.3463270366191864, -0.09445691108703613, -0.5405825972557068, 0.335199236869812, -0.49747124314308167, -1.2568124532699585, 0.051758140325546265, 0.32888105511665344, -0.23408645391464233, -0.37015730142593384, 0.39850470423698425, 0.07856196165084839, -0.6666789054870605, -0.08690184354782104, 0.32905739545822144, -0.8525896668434143, 1.3921210765838623, -2.073239803314209, -1.3870760202407837, 0.5775946974754333, 0.01960648223757744, -1.693972110748291, -1.2425490617752075, -0.6217936277389526, 0.8794918656349182, -0.1127345934510231, -1.5275307893753052, -0.9724931120872498, -1.3822314739227295, -1.4136182069778442, 1.0018227100372314, -0.10619427263736725, -0.9769010543823242, -0.985069990158081, -1.075402855873108, 0.3761135935783386, -0.0814087763428688, 1.146906852722168, -0.3153947591781616, 0.20130670070648193, 0.32868021726608276, 0.5697015523910522, -0.4716803729534149, 0.33434879779815674, 0.06871239840984344, -1.0319424867630005, -1.082664132118225, -0.529662013053894, -1.471641182899475], [-0.2035883665084839, -1.2170473337173462, 2.5340678691864014, 0.20585498213768005, 0.4012100398540497, -1.2831186056137085, -0.5520204305648804, 0.046599481254816055, 0.4908493161201477, 1.2990407943725586, 1.7581371068954468, 0.8031694293022156, 0.38816937804222107, 1.12028169631958, 0.9253832697868347, 0.2676035463809967, 0.24376265704631805, 0.1474730223417282, 1.6243202686309814, -0.2563697397708893, 0.023662561550736427, -2.46006178855896, -0.16513486206531525, -0.8699630498886108, 0.23192310333251953, 0.8246321082115173, -2.1015758514404297, -0.42451155185699463, 0.7245345115661621, -1.6757768392562866, -0.8847615122795105, -0.9334514737129211, 0.9422733783721924, -0.6798949241638184, -0.4004492461681366, -0.56546550989151, -1.2128242254257202, 1.2169032096862793, 1.8478540182113647, -0.7246072292327881, 2.6204960346221924, -0.4142071604728699, -0.8010064363479614, -0.9921383857727051, 0.23949263989925385, -0.429319828748703, 0.04442442208528519, 0.8016964793205261, -0.09276513010263443, 0.567646324634552, -0.16016942262649536, 1.1765556335449219, -0.09734000265598297, 1.3681102991104126, -0.16900566220283508, 1.1298831701278687, 0.45471957325935364, 0.5466572046279907, -0.18127040565013885, 0.39725232124328613, 0.48690181970596313, 2.1273787021636963, -1.4585137367248535, -0.6210675239562988, 0.8455579280853271, -1.0455950498580933, -0.2825852632522583, 0.5204891562461853, 0.5119758248329163, -2.4153976440429688, 1.3199357986450195, 2.0501821041107178, -1.336485743522644, 1.0445612668991089, 0.7308616638183594, -1.0568093061447144, 0.3113090693950653, 1.7831214666366577, -0.08361973613500595, -0.38414108753204346, 0.3681349456310272, 0.8133945465087891, 0.14369173347949982, -0.731062114238739, 1.3286839723587036, 0.2941541075706482, -0.8075625896453857, -0.9184131622314453, 0.4670698642730713, 2.1980109214782715, 1.941051721572876, -0.6123737096786499, 0.5900260806083679, 2.684424877166748, -1.5450189113616943, -0.2800537049770355], [0.5309045910835266, -0.287803590297699, 0.5087596774101257, 0.8249565958976746, -0.1612735241651535, 0.5139696002006531, -0.07452704757452011, 0.015393373556435108, -0.33170565962791443, -2.253751754760742, -0.5030736327171326, 1.8258527517318726, 0.7890871167182922, -0.08221794664859772, -0.23714008927345276, 0.5818670988082886, -0.3885795772075653, -0.8864502310752869, -0.4066607654094696, -0.14601856470108032, 0.2894427180290222, -0.24933698773384094, -0.1695265918970108, -0.6202830076217651, -1.021627426147461, 0.3849540650844574, -2.369291305541992, -0.7806940078735352, -0.2908630073070526, 0.8642311692237854, -0.2260676771402359, -0.8041931986808777, -0.37940365076065063, 1.2356343269348145, -0.8597760796546936, 0.9251825213432312, 0.19381621479988098, 1.6601425409317017, -1.057517647743225, 0.1510278880596161, 0.35400086641311646, 1.7294962406158447, 0.5111709833145142, -1.2578387260437012, 0.7429134845733643, -1.3661662340164185, 1.9299137592315674, 2.1436996459960938, 0.3427228331565857, 0.3802759051322937, -0.4798987805843353, 0.1806703805923462, 0.21409453451633453, 0.6207169890403748, -0.9888989925384521, -0.07231676578521729, -0.8319811224937439, -0.8048908710479736, -0.1984459012746811, 0.03340764716267586, -0.4659271538257599, -1.635723352432251, -0.04962639883160591, 1.8731353282928467, -0.3292291462421417, 1.0790352821350098, -0.0573759526014328, 0.3559338450431824, -0.08669356256723404, 0.16839846968650818, 0.6721559166908264, -0.5985718965530396, -0.5317187905311584, -2.0430350303649902, -1.0053895711898804, 0.696922779083252, -0.7350214123725891, 0.48989439010620117, 0.8284916281700134, 0.8882510662078857, -0.6975094079971313, 1.0988045930862427, -0.06658901274204254, -0.26199132204055786, -1.1985422372817993, 1.191752552986145, -0.03285451605916023, 0.3957318365573883, -2.50736665725708, 1.5132358074188232, -1.5934025049209595, -1.6537333726882935, -1.3288229703903198, -0.2157277911901474, 0.14644481241703033, -1.1380233764648438], [-0.18326126039028168, -0.36514467000961304, -0.09070077538490295, -0.5703496336936951, -0.4056451916694641, 1.2168443202972412, 0.8712696433067322, -1.778927206993103, 1.5358482599258423, -0.28607064485549927, 0.872226893901825, -0.4115388095378876, 0.6795371770858765, 0.47221505641937256, -0.6379454135894775, 0.2363225370645523, 0.018584301695227623, -0.5428816080093384, -1.0850468873977661, 0.9579108357429504, 0.4703206717967987, -1.0008087158203125, -0.013752459548413754, -1.1430068016052246, -0.7275763154029846, 1.540229082107544, 1.1410995721817017, 0.11099319159984589, 1.6849050521850586, 0.8828701972961426, -1.5481233596801758, 0.33093154430389404, 0.22196970880031586, 0.6427333950996399, 0.3026657998561859, -1.244848370552063, 0.4924420118331909, 0.977157711982727, 0.025496458634734154, -0.13312391936779022, 1.9180302619934082, -2.6898515224456787, -0.5238210558891296, 0.6439749002456665, 1.6743932962417603, -1.7847659587860107, 0.2658836841583252, 1.2802854776382446, -0.48952245712280273, -1.8746259212493896, 1.1393073797225952, -0.3857823610305786, 1.2079803943634033, 1.0296354293823242, 1.0314991474151611, 0.2996855676174164, 1.198563575744629, 1.6802889108657837, -1.4301447868347168, 1.2327758073806763, -0.6056979894638062, -0.785290539264679, -0.3923731744289398, -2.127715587615967, -0.246408149600029, 0.013118655420839787, -0.3024488091468811, -1.1272404193878174, 0.4268093407154083, -0.02915291301906109, -2.7470414638519287, 0.5655549764633179, -0.013078478164970875, -0.2222258597612381, 0.6974546313285828, 1.1710413694381714, -2.3086984157562256, -0.07468753308057785, 1.6273553371429443, -0.901035487651825, -0.13815870881080627, -0.02482195384800434, -0.6808879971504211, 0.9238810539245605, 0.26202264428138733, 0.6137514114379883, 0.6401759386062622, -0.5999636054039001, -0.010856172069907188, 0.2563398778438568, 0.18207165598869324, 1.8272300958633423, -0.12568214535713196, 2.6713736057281494, 1.027462363243103, 1.6200716495513916], [-0.4281514585018158, -0.22345873713493347, -1.1083849668502808, 0.8192078471183777, -0.26875099539756775, 0.4146176278591156, 0.4246055781841278, 1.451230764389038, -0.1779828667640686, 1.0104122161865234, -0.11928772926330566, -0.45852911472320557, 1.6029685735702515, -0.032214704900979996, 0.21687771379947662, 1.1918309926986694, -0.7843013405799866, -1.0141806602478027, 0.21641318500041962, 0.5373764038085938, 0.5139298439025879, 2.3947012424468994, 0.8485471606254578, 1.0249526500701904, -1.2757962942123413, -0.6529989838600159, -1.110474944114685, 1.0904349088668823, -1.125797152519226, -0.5717002153396606, -1.3523995876312256, 1.4895271062850952, -0.5428513884544373, 0.8224903345108032, 1.5795625448226929, 0.48940950632095337, -0.7568188309669495, -1.0209496021270752, 0.029585029929876328, -0.413979709148407, 0.24831941723823547, 0.5827746391296387, -0.294399231672287, -0.6007898449897766, -0.7401251792907715, -1.755629062652588, 0.5031570792198181, -1.2903459072113037, 0.11658673733472824, 0.032174695283174515, -0.32422924041748047, 0.8692352175712585, 0.2591751217842102, 1.8037229776382446, 0.4057011604309082, 1.0489753484725952, 1.0272091627120972, 1.3815054893493652, -0.429364949464798, 2.6313610076904297, -2.266775131225586, 0.545799732208252, 0.45161172747612, 0.5841052532196045, 2.1819908618927, -1.708406925201416, 0.7613571286201477, -0.8410559892654419, 1.5681875944137573, 0.8740506172180176, 1.1209259033203125, -0.534134566783905, -1.1794919967651367, 0.5287038087844849, -0.3707989454269409, -0.6911197304725647, 1.4464582204818726, -1.1199291944503784, -0.7573873996734619, -0.5065471529960632, 0.7462963461875916, -1.0918614864349365, 0.04770991578698158, 0.20966538786888123, 0.7215152382850647, -0.6848055124282837, -1.9325976371765137, -0.5897459387779236, -2.412213087081909, -0.341886967420578, -1.8336589336395264, -1.9492294788360596, -0.9016523957252502, 1.62649405002594, -1.879201054573059, -0.21493951976299286], [-0.4454900324344635, 1.4752871990203857, -0.4335225224494934, -0.2482975721359253, 0.2065650373697281, 1.4194836616516113, 0.15209156274795532, -1.6393558979034424, -0.17635098099708557, 0.9750550389289856, 1.2269902229309082, -0.7543753981590271, -0.1959836184978485, -0.18712998926639557, 0.03907274082303047, 0.9746937155723572, 0.17471511662006378, -0.346910685300827, -0.5975297689437866, 0.857040524482727, -0.6581949591636658, 1.1586425304412842, 0.24953308701515198, -1.7784240245819092, 0.32281166315078735, -1.974778652191162, 0.2121279388666153, -0.522205114364624, 0.563452422618866, -0.5358158349990845, 1.318186640739441, -0.3699449598789215, 1.5988128185272217, 0.2077072113752365, 0.19548740983009338, 0.15945695340633392, -0.5694707632064819, 1.0248478651046753, -2.522148370742798, -0.4532361626625061, -0.5236812829971313, 1.5229361057281494, -1.4245482683181763, 0.097941093146801, 1.1143397092819214, -0.8157839179039001, 1.5110790729522705, -0.11773961782455444, -0.6739119291305542, -0.8132210969924927, -1.1860966682434082, -0.15363147854804993, -0.3408355116844177, -0.05433463305234909, -1.463974118232727, 0.7922946810722351, 0.11091314256191254, 0.3421538174152374, -0.6279222965240479, 0.01541126612573862, -0.05580738186836243, 1.046299934387207, 0.9935141205787659, -0.10741913318634033, -0.6551379561424255, 1.8107448816299438, -0.5755385756492615, -0.9738866090774536, -0.4941185712814331, 1.7451536655426025, -0.5932671427726746, 0.948929488658905, -1.2454243898391724, 1.1736489534378052, 1.7043527364730835, -0.6079495549201965, -0.8313103914260864, -0.5643629431724548, 0.21282215416431427, -0.32670724391937256, 0.9178809523582458, 0.7009218335151672, 2.158442497253418, 0.9253513216972351, 1.2789552211761475, 0.9332144856452942, -1.8252723217010498, -0.44741135835647583, 0.7448534965515137, 1.91130793094635, 0.2600846290588379, -0.8286184072494507, 0.9799556732177734, -0.7287525534629822, 0.36664924025535583, -1.1295546293258667], [0.6522035598754883, -1.0761879682540894, 1.3712772130966187, -0.6643219590187073, 0.5387807488441467, -0.6528935432434082, 0.30651286244392395, -0.11676305532455444, 0.615185022354126, 1.081640362739563, -0.4277217388153076, 1.0038564205169678, -2.328233242034912, -0.12203668057918549, 0.40886110067367554, 0.7146164178848267, -1.4731543064117432, 0.8560152053833008, 1.123231291770935, -1.3815147876739502, -0.3021200895309448, 0.6578772664070129, -0.4827588200569153, 1.0642459392547607, 1.0078343152999878, -0.48650550842285156, -0.6462295651435852, 0.8186724781990051, -1.5178494453430176, -0.3575875461101532, 0.6502118110656738, -0.7815431356430054, -0.07314636558294296, 0.6321502327919006, -1.699236512184143, -0.6940560340881348, -0.5395640730857849, 0.31468719244003296, 1.435336709022522, -1.2701407670974731, -0.675449788570404, -0.3010879158973694, 0.8106183409690857, 0.4583727717399597, 1.0269927978515625, -1.0853252410888672, 0.24746796488761902, -0.3092373013496399, -0.036779992282390594, -1.3103994131088257, 1.2270950078964233, -0.9369661211967468, -0.16083768010139465, 1.148123860359192, 0.595869243144989, 0.14446164667606354, 1.1146793365478516, 0.12244226038455963, -0.2059326320886612, -0.016820799559354782, 0.19859741628170013, -3.1537973880767822, -0.12243146449327469, -0.525398313999176, 0.21830910444259644, -1.101082682609558, 1.2499816417694092, -0.24557586014270782, 1.0036838054656982, 0.6317331790924072, 0.8358795046806335, 0.09938672930002213, 0.42441311478614807, 0.3267926275730133, -1.7990740537643433, -0.523358166217804, -0.3884701430797577, -1.4391491413116455, -1.3091166019439697, 0.5991506576538086, -0.8455089330673218, 0.4984554052352905, -0.548332929611206, 0.06761269271373749, 0.28905948996543884, 0.17626258730888367, 0.6833481788635254, -0.129084512591362, 0.0784216970205307, -0.5042449235916138, 1.9813764095306396, -0.8769988417625427, 1.2035584449768066, 0.635132372379303, 1.030734658241272, -0.7989813685417175], [0.4313599169254303, -0.7314174175262451, -1.0041497945785522, 1.6557368040084839, 0.37529733777046204, -0.6712630391120911, 0.16515658795833588, -0.4775511920452118, 0.8079532980918884, -0.9598753452301025, -1.0362251996994019, 0.24284908175468445, -1.5312554836273193, 0.5579637289047241, -0.8621180057525635, 0.26592621207237244, -0.5958254933357239, 0.7654720544815063, 0.12616291642189026, 0.41756394505500793, -2.0300631523132324, 0.08272265642881393, 0.11030115187168121, 0.6832738518714905, 1.1367918252944946, 0.47129884362220764, -0.4429212212562561, 0.6416391134262085, 0.6455983519554138, 0.12326265871524811, 1.6881775856018066, 1.592786431312561, 0.8814534544944763, -0.02078264020383358, -0.8382565975189209, 0.0810975432395935, 0.702067494392395, -2.267228841781616, 0.9159107804298401, 0.5544248819351196, -0.04865832254290581, -0.14346933364868164, -1.0526635646820068, 0.22152747213840485, -0.8935034275054932, 0.7424785494804382, 0.008954011835157871, -0.05147486552596092, 1.2162021398544312, 0.4409787952899933, 1.3694064617156982, -0.09007863700389862, -0.8596698641777039, 0.10622230917215347, -0.37181058526039124, -0.0067849718034267426, 0.4637286961078644, 0.3742583692073822, 0.35876843333244324, -1.0117900371551514, -0.12225867062807083, -0.08242329955101013, 0.1649305820465088, 0.22167372703552246, 1.7772246599197388, 0.19024117290973663, -0.5213483572006226, -0.5007416009902954, -0.9087387919425964, 0.18545086681842804, 0.07837074995040894, 2.575859785079956, -0.2388215959072113, -0.27764075994491577, -0.3535749316215515, -0.09585890918970108, 1.4663360118865967, -1.2548325061798096, 0.4808584749698639, 0.9099956750869751, -0.13804693520069122, 0.30973708629608154, -0.8437023758888245, 0.025666818022727966, 0.3268410563468933, -0.49772170186042786, -0.8732519149780273, 0.5803821086883545, -0.023419557139277458, 1.0713202953338623, 0.3002952039241791, -0.25077322125434875, 0.6513085961341858, 1.1936020851135254, 0.5336251258850098, -0.8338047862052917], [-0.2888035476207733, 1.0317529439926147, 0.4897615313529968, -0.08238628506660461, -0.30182936787605286, -1.398168921470642, -0.04159785434603691, -0.8062137365341187, -0.5881291627883911, -0.1431388109922409, -1.459970235824585, -0.2443370372056961, 1.4043248891830444, 0.18161466717720032, 1.5659633874893188, -0.020557019859552383, -0.07763616740703583, 0.6709205508232117, -0.5517421960830688, -0.3340725898742676, -0.706313967704773, 1.0473276376724243, -0.7874857783317566, 0.21872058510780334, -0.2560131847858429, -0.7327775359153748, 1.9032816886901855, -0.6738521456718445, -1.245646357536316, 0.24192066490650177, 0.7013838291168213, 0.19666871428489685, -0.7218449115753174, 0.40868470072746277, 2.0638840198516846, -0.11951308697462082, 0.00864381156861782, -0.36727645993232727, -0.19187991321086884, -0.06467342376708984, 1.4514813423156738, 1.3750653266906738, 0.09106782078742981, -1.2653459310531616, -0.3919163942337036, 0.2767956852912903, -0.14456544816493988, 1.8270870447158813, -0.8589468598365784, 1.0971696376800537, -1.484216570854187, -0.7610009908676147, 0.7177760004997253, 0.6873390674591064, -0.21670465171337128, 0.022728731855750084, 2.2587945461273193, 1.391963005065918, 2.2993311882019043, 0.2690458595752716, 1.59824538230896, 0.10012780129909515, 0.2960290312767029, 1.358752965927124, 0.16819502413272858, 1.113067388534546, -0.05485493317246437, 1.4640400409698486, -0.6390298008918762, 0.36274453997612, -0.25804364681243896, -0.6212397217750549, 0.24566885828971863, 0.5895450711250305, -0.17846594750881195, 0.49103862047195435, 1.1856166124343872, 0.24166448414325714, -0.4082431197166443, -0.8687567114830017, -0.14272440969944, -2.123016357421875, -0.4159659445285797, -3.1293909549713135, -0.7236876487731934, -1.1947126388549805, 0.789694607257843, 0.10050775110721588, 0.29632532596588135, -0.18706932663917542, 0.6922820210456848, 0.5008264183998108, 0.8714815378189087, 1.14119291305542, 0.7178844809532166, -0.8933191299438477], [0.8368484973907471, -1.3398247957229614, -1.8061504364013672, -0.18836891651153564, 1.4185948371887207, 1.2832452058792114, 0.3677048683166504, -1.6883639097213745, -0.9516795873641968, -1.1694034337997437, 0.22139430046081543, 1.1544846296310425, 0.5028183460235596, -1.3080333471298218, -1.2819175720214844, -1.1831895112991333, 1.2033358812332153, 0.8006065487861633, 2.005485773086548, -1.0998603105545044, -1.248868465423584, 1.3479106426239014, -0.9345016479492188, -0.06759605556726456, 0.3217306435108185, 0.8012588620185852, -1.2593815326690674, 0.7788556218147278, -0.5463085174560547, 1.778515338897705, 0.6009320020675659, 1.4363138675689697, 2.5825252532958984, -1.127663254737854, 0.47929030656814575, 1.3639869689941406, 1.472808837890625, -0.6110773086547852, -1.3345640897750854, 2.146145820617676, -0.5197583436965942, -0.3927359879016876, 1.4680440425872803, -0.5486389398574829, 1.754269003868103, -0.3235055208206177, 1.3740041255950928, -0.5537219643592834, -1.51162588596344, -0.5362884998321533, 0.41070955991744995, -1.4173372983932495, 0.24721960723400116, 0.11723609268665314, -0.7864993810653687, 0.6788805723190308, -0.4492645859718323, -1.054115891456604, 0.14649923145771027, 0.8472549319267273, 0.14238408207893372, -0.5082601308822632, 0.67425537109375, -1.2222965955734253, -0.868867039680481, 0.5524448156356812, 0.6864125728607178, 0.6022374629974365, 0.717625617980957, -1.0443170070648193, -1.5154836177825928, 1.9397687911987305, -0.820692777633667, 0.24244938790798187, -1.170475959777832, 1.067208170890808, -0.1686364859342575, 1.3600637912750244, 0.812681257724762, -0.002263353206217289, -0.37979891896247864, -0.9653412103652954, -1.2738571166992188, 0.18830065429210663, 2.0014889240264893, 2.242016553878784, 0.8194200396537781, -0.3995521366596222, 0.11809799075126648, -1.0257375240325928, 0.1958630383014679, 1.0231744050979614, -0.08406811207532883, -0.7359532713890076, 0.35951918363571167, -0.12638013064861298], [-0.24639557301998138, -0.726830244064331, 1.4539259672164917, 0.47749999165534973, -0.028364907950162888, -1.9263206720352173, 0.2811618447303772, -0.20882964134216309, -0.3347802758216858, 1.6947227716445923, 0.9052469730377197, 0.8959938883781433, -0.9673714637756348, -0.19678033888339996, 0.7661756873130798, 0.7843418121337891, -0.11792730540037155, -1.7430217266082764, 1.309441328048706, 0.11607937514781952, 0.5808667540550232, -0.42641884088516235, 0.3328387141227722, -2.5058016777038574, 0.4584812521934509, 0.39457640051841736, 0.28238973021507263, -0.33561745285987854, -0.21939659118652344, -0.833486020565033, -0.3478662669658661, -0.9157987236976624, 0.8123764991760254, 0.9878158569335938, -0.3819883465766907, -1.3826439380645752, 0.38181090354919434, -0.6783491969108582, 0.18158209323883057, -1.1899142265319824, 0.2433209866285324, -0.35297635197639465, 0.39098992943763733, 0.6301499605178833, -0.05611829087138176, 0.9302265048027039, -0.05724726617336273, -1.7672193050384521, -0.2269570231437683, 0.2600974142551422, -0.47086089849472046, -0.3326047658920288, -0.5229357481002808, 0.18921653926372528, 1.160836100578308, -1.1355215311050415, -0.7230517864227295, 1.3626011610031128, 1.1382701396942139, -0.2695334851741791, -2.0516197681427, -1.1375582218170166, -0.9698330163955688, 0.9823732972145081, 0.5492860078811646, 1.9401445388793945, 0.17804796993732452, 0.5234114527702332, 0.08092544227838516, -0.2931528091430664, -0.4559359848499298, -0.4466361403465271, 0.8029927015304565, -1.1496448516845703, 0.791154682636261, -1.201806664466858, -0.11787691712379456, 0.6789433360099792, 0.2411700338125229, 0.5042451024055481, 0.25143128633499146, 0.8939381837844849, -1.0619251728057861, 0.7658842206001282, -0.573061466217041, 0.4661228656768799, 0.5762974619865417, 0.48036691546440125, 2.398899793624878, -0.43035006523132324, -0.42723608016967773, -0.3835670053958893, -0.0966557040810585, 0.8381685018539429, 1.0226414203643799, 0.07697395980358124], [0.7046387791633606, 0.098325714468956, 0.6763911843299866, -0.24026308953762054, -0.17723448574543, 0.3342551290988922, 0.9047608971595764, 1.3746827840805054, -1.2101725339889526, -0.9581463932991028, -0.7406882047653198, 0.268962562084198, -0.15268629789352417, -0.5292180776596069, -1.920449137687683, -0.08047974854707718, -1.959286093711853, 1.3717433214187622, -1.5333260297775269, -1.1491360664367676, -0.08697979152202606, 0.7622974514961243, -0.43601343035697937, 1.4110827445983887, 1.6954354047775269, -1.0850461721420288, -0.8526296615600586, -0.7954747676849365, -0.6667974591255188, -0.37305110692977905, 0.20644444227218628, 0.5915655493736267, 1.6283643245697021, -0.14129124581813812, -1.0716094970703125, 0.8485081195831299, 0.3027704954147339, -0.2125084400177002, 1.5054794549942017, -0.09947014600038528, -1.3120487928390503, -1.1476207971572876, 1.1831071376800537, -0.5634925365447998, 0.025322340428829193, 1.209451675415039, 1.6164602041244507, -0.5842701196670532, -1.4168277978897095, -1.7110114097595215, 0.6394097805023193, 1.1354931592941284, 0.8121371865272522, 0.4312763512134552, -1.081053376197815, 0.11851374059915543, 1.9509077072143555, 1.260549545288086, 0.10464516282081604, -1.3257083892822266, -0.7909702062606812, -0.9365396499633789, 0.37720879912376404, -1.1806049346923828, -0.19112439453601837, 0.6515341997146606, -0.3180253505706787, 0.762978196144104, -0.3476245105266571, -1.3016949892044067, -1.6048040390014648, 0.4633590579032898, 0.46974194049835205, 0.9826306700706482, 0.2041357010602951, 0.043402694165706635, 0.6475814580917358, 0.48774686455726624, -2.151947021484375, -0.06890644878149033, -0.4557236433029175, -1.2362053394317627, -0.1509179025888443, -1.0467331409454346, 1.0218311548233032, 1.0860764980316162, -1.1432567834854126, 0.7472706437110901, -2.3071751594543457, -0.06041533872485161, -0.6973950862884521, 0.13536588847637177, -0.6736600399017334, -1.9224909543991089, -0.4261334240436554, -0.40235209465026855], [-0.6747071146965027, 1.7395212650299072, -0.663000762462616, 1.0376231670379639, -0.6448155641555786, -0.4513171911239624, -0.23989561200141907, -0.4358505606651306, 1.8364640474319458, -0.07068081200122833, 0.44347816705703735, 1.8340870141983032, 0.15985946357250214, 1.1091420650482178, 1.2526143789291382, 0.910757839679718, 0.8464482426643372, -0.02491104044020176, 0.8724640011787415, -1.0720843076705933, 1.0065780878067017, -0.1842627078294754, 2.383958101272583, 0.4395841658115387, -0.8507684469223022, -0.36804184317588806, 1.1307743787765503, -1.1397888660430908, 1.5231562852859497, -0.4367790222167969, 0.757314920425415, -0.12141481041908264, 1.5850731134414673, 0.6584046483039856, 0.4139045178890228, -0.4987398386001587, -1.0788395404815674, 0.23618854582309723, -0.4145885109901428, 0.42266109585762024, 0.7758591771125793, -0.3660653233528137, 1.1417149305343628, 0.9035656452178955, -0.758561372756958, 1.2365494966506958, -0.5001250505447388, -0.25485149025917053, -2.733701467514038, -2.8623523712158203, 0.43974804878234863, -0.06805554777383804, -1.0561566352844238, 0.08166555315256119, 0.9901110529899597, 1.0725799798965454, 0.7992321848869324, 0.2544039487838745, 1.8944414854049683, -1.1802771091461182, 1.2626688480377197, -1.6594834327697754, 0.5493480563163757, 1.0247780084609985, -0.05851038545370102, 0.17078952491283417, 1.6561864614486694, 1.2892006635665894, 1.3248454332351685, -1.0886235237121582, 1.5667269229888916, -0.2836121618747711, 0.2048460990190506, 0.9516554474830627, -0.9563331007957458, -0.5770044922828674, -0.24824930727481842, 0.8085756301879883, 1.384291410446167, 2.236612558364868, 0.41165435314178467, -1.8915046453475952, -1.2279672622680664, -0.6354721188545227, -1.0396908521652222, 0.5578834414482117, 0.11137112975120544, 1.9864165782928467, -0.21034705638885498, 0.6874715685844421, -0.2309582382440567, 0.08744042366743088, 1.436599850654602, 0.25633057951927185, -0.581882894039154, 1.1030749082565308]], "dtype": "float32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(grouped_matmul, json.loads(_TESTS), 'triton', *[0.0001, 0.0001])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")